# Выбираем отдых по историческим данным

Проект решает задачу сравнения горнолыжных курортов по историческим погодным данным и инфраструктуре. Логика работы инструмента проста: **параметры → веса → даты → рейтинг → карточка лидера**. Сравнивается погода за выбранный период прошлого (даты поездки выбираются в прошлом), а не прогноз.

### Какие данные уже внутри

- **Дневная погода:** 1 января 1996 - 31 декабря 2025 года, 43 832 строки для четырёх курортов: Роза Хутор, Красная Поляна, Газпром Поляна и Шерегеш.
- Остальные 28 курортов остаются в справочнике, но без дневной истории в расчёт не попадают. Месячные значения вместо дневных не подставляются - смешивать разную детализацию нельзя.

### Правила выбора периода

Можно выбрать один день, неделю, месяц или несколько лет; обе даты включаются в расчёт. По умолчанию сравнивается 15-21 января 2025 года. Результат отвечает на вопрос «какие условия были в этот период по архиву модели», а не «куда ехать в будущем».

### Важные методологические оговорки

- **Погода — исторический реанализ ERA5**, а не прямые измерения на каждой трассе: он объединяет наблюдения и расчёты модели. Для первых трёх курортов выбрана одна ячейка сетки - это не три независимые метеостанции. Запрошенные точки и высоты отличаются; эти сведения сохранены ниже в метаданных.
- **Инфраструктура - снимок OpenStreetMap на 14 сентября 2026 года.** Поэтому расчёт смешанный: погода выбранных прошлых дат плюс более поздние сведения о местах питания и проживания. Утверждать, что все эти объекты существовали в выбранном году, нельзя.
- В архиве есть среднесуточная температура и солнечное сияние, но нет минимума и максимума за день. Поэтому вместо перепада температуры используется параметр «Больше солнца». 

### Источники

- Open-Meteo Historical Weather API - https://open-meteo.com/en/docs/historical-weather-api (CC BY 4.0)
- OpenStreetMap contributors - https://www.openstreetmap.org/copyright (ODbL)


## Подготовка: библиотеки и скачанная история

В этой ячейке подключаются необходимые библиотеки и используется сохранённая дневная история. Запросы к API не выполняются: данные уже содержатся в ноутбуке.


In [1]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from io import StringIO
from html import escape
import os

In [2]:
# @title
# Дневная история, уже скачанная в проекте. Это не прогноз и не месячные средние.
weather = pd.read_csv(StringIO('resort_id,date,temperature_c,precipitation_mm,sunshine_seconds\nR001,1996-01-01,2.1,0.9,27974.96\nR001,1996-01-02,2.3,0.0,31626.87\nR001,1996-01-03,3.0,0.2,1888.06\nR001,1996-01-04,3.4,0.0,31737.93\nR001,1996-01-05,3.9,0.2,16920.48\nR001,1996-01-06,2.7,0.0,31857.7\nR001,1996-01-07,2.7,3.9,15718.35\nR001,1996-01-08,1.8,2.0,23787.38\nR001,1996-01-09,-0.4,0.0,32051.21\nR001,1996-01-10,-2.4,0.0,32118.84\nR001,1996-01-11,-3.8,0.0,32187.78\nR001,1996-01-12,-2.9,0.0,32257.91\nR001,1996-01-13,-3.6,0.0,32329.08\nR001,1996-01-14,-3.8,0.0,32400.0\nR001,1996-01-15,-2.6,0.0,32400.0\nR001,1996-01-16,-3.9,0.0,32400.0\nR001,1996-01-17,-3.8,0.0,32400.0\nR001,1996-01-18,-4.6,0.0,32400.0\nR001,1996-01-19,-4.6,2.5,30456.52\nR001,1996-01-20,-2.9,6.2,16980.8\nR001,1996-01-21,-2.9,5.7,6137.44\nR001,1996-01-22,-5.4,0.3,32400.0\nR001,1996-01-23,-5.8,0.5,26761.33\nR001,1996-01-24,-6.1,0.3,31481.38\nR001,1996-01-25,-2.7,0.3,28800.0\nR001,1996-01-26,-0.2,0.0,32400.0\nR001,1996-01-27,1.2,0.0,30351.34\nR001,1996-01-28,2.6,2.0,3395.71\nR001,1996-01-29,2.2,5.3,15213.75\nR001,1996-01-30,1.1,8.1,3191.67\nR001,1996-01-31,0.3,11.8,1.56\nR001,1996-02-01,-6.5,10.5,30631.22\nR001,1996-02-02,-10.1,0.0,32700.41\nR001,1996-02-03,-6.3,0.0,33089.41\nR001,1996-02-04,-3.5,0.0,33704.15\nR001,1996-02-05,-2.8,0.0,33619.37\nR001,1996-02-06,-0.9,6.4,2540.12\nR001,1996-02-07,2.3,28.4,15962.83\nR001,1996-02-08,4.0,3.2,19393.09\nR001,1996-02-09,5.9,5.8,19952.24\nR001,1996-02-10,3.1,5.1,5618.7\nR001,1996-02-11,1.6,14.2,0.0\nR001,1996-02-12,-0.3,3.4,21352.66\nR001,1996-02-13,0.3,0.0,26548.84\nR001,1996-02-14,3.0,0.0,27074.19\nR001,1996-02-15,3.1,3.3,27978.59\nR001,1996-02-16,1.5,5.6,8988.85\nR001,1996-02-17,2.1,2.9,17089.18\nR001,1996-02-18,4.1,0.4,37654.32\nR001,1996-02-19,4.7,0.3,37828.42\nR001,1996-02-20,3.2,0.0,38003.31\nR001,1996-02-21,3.8,0.0,38178.82\nR001,1996-02-22,5.0,0.0,38354.79\nR001,1996-02-23,6.4,0.0,35630.05\nR001,1996-02-24,7.3,0.0,27277.01\nR001,1996-02-25,3.4,6.3,17519.13\nR001,1996-02-26,1.8,3.1,21434.34\nR001,1996-02-27,1.5,0.5,37270.78\nR001,1996-02-28,1.1,1.7,27036.78\nR001,1996-02-29,0.2,6.0,27736.69\nR001,1996-03-01,0.0,1.7,35982.65\nR001,1996-03-02,0.6,12.1,0.0\nR001,1996-03-03,1.2,0.9,39600.0\nR001,1996-03-04,0.6,13.2,13736.15\nR001,1996-03-05,0.7,2.9,36075.25\nR001,1996-03-06,2.8,14.8,0.0\nR001,1996-03-07,3.8,1.1,35028.6\nR001,1996-03-08,1.9,4.1,19192.79\nR001,1996-03-09,0.5,4.7,16926.55\nR001,1996-03-10,0.7,0.7,30223.94\nR001,1996-03-11,2.8,0.1,37596.1\nR001,1996-03-12,1.7,9.4,8170.99\nR001,1996-03-13,1.5,5.3,37825.09\nR001,1996-03-14,1.9,0.0,39363.12\nR001,1996-03-15,4.3,0.0,39839.5\nR001,1996-03-16,5.2,0.0,40089.51\nR001,1996-03-17,3.8,0.0,40035.22\nR001,1996-03-18,3.1,1.1,37029.06\nR001,1996-03-19,2.7,2.9,18256.69\nR001,1996-03-20,1.9,0.3,35612.17\nR001,1996-03-21,3.4,0.2,39183.51\nR001,1996-03-22,4.5,2.2,29650.3\nR001,1996-03-23,4.2,0.8,15297.18\nR001,1996-03-24,3.9,3.2,5603.46\nR001,1996-03-25,2.1,7.1,857.46\nR001,1996-03-26,1.6,11.0,19972.43\nR001,1996-03-27,1.2,6.4,21506.13\nR001,1996-03-28,3.6,0.0,39400.97\nR001,1996-03-29,4.6,9.9,35200.8\nR001,1996-03-30,5.0,11.0,13851.49\nR001,1996-03-31,4.4,1.7,40663.09\nR001,1996-04-01,6.9,0.7,43408.06\nR001,1996-04-02,6.6,4.8,8387.42\nR001,1996-04-03,6.6,0.0,35648.67\nR001,1996-04-04,7.7,0.1,45688.81\nR001,1996-04-05,7.5,0.0,45962.75\nR001,1996-04-06,7.6,0.4,44024.23\nR001,1996-04-07,6.7,3.4,44632.1\nR001,1996-04-08,6.4,11.9,9112.6\nR001,1996-04-09,4.9,4.1,41147.91\nR001,1996-04-10,3.9,3.2,39856.54\nR001,1996-04-11,3.1,0.6,46389.12\nR001,1996-04-12,4.3,1.3,30569.82\nR001,1996-04-13,4.3,7.1,0.0\nR001,1996-04-14,3.4,2.6,36552.6\nR001,1996-04-15,5.0,0.0,45816.35\nR001,1996-04-16,7.6,0.1,35200.78\nR001,1996-04-17,6.4,17.3,0.0\nR001,1996-04-18,4.4,31.3,17852.29\nR001,1996-04-19,1.7,7.3,9809.37\nR001,1996-04-20,2.5,2.5,44583.79\nR001,1996-04-21,5.1,1.2,20258.72\nR001,1996-04-22,4.5,11.5,0.0\nR001,1996-04-23,5.7,1.7,18746.26\nR001,1996-04-24,6.8,1.1,43618.01\nR001,1996-04-25,7.4,0.9,36598.84\nR001,1996-04-26,8.8,0.1,46800.0\nR001,1996-04-27,10.8,0.0,47650.5\nR001,1996-04-28,10.6,0.0,47944.42\nR001,1996-04-29,12.1,0.0,48574.8\nR001,1996-04-30,12.5,5.3,39408.16\nR001,1996-05-01,13.0,4.4,46800.0\nR001,1996-05-02,14.1,0.6,48805.49\nR001,1996-05-03,14.8,2.2,48435.1\nR001,1996-05-04,14.2,3.6,46414.3\nR001,1996-05-05,13.7,2.1,45132.83\nR001,1996-05-06,13.5,4.1,48164.17\nR001,1996-05-07,14.0,2.8,47474.35\nR001,1996-05-08,15.2,0.6,49637.68\nR001,1996-05-09,17.0,0.3,49952.37\nR001,1996-05-10,16.9,2.0,47529.71\nR001,1996-05-11,16.8,5.1,46058.39\nR001,1996-05-12,17.0,4.4,49190.36\nR001,1996-05-13,14.9,7.5,41981.7\nR001,1996-05-14,15.4,3.1,49226.4\nR001,1996-05-15,15.0,2.9,45823.98\nR001,1996-05-16,12.4,6.3,40558.97\nR001,1996-05-17,13.3,0.0,46589.41\nR001,1996-05-18,14.2,0.0,50400.0\nR001,1996-05-19,15.0,1.3,48892.66\nR001,1996-05-20,14.7,0.5,44619.46\nR001,1996-05-21,17.5,0.1,47329.03\nR001,1996-05-22,16.2,0.0,48577.64\nR001,1996-05-23,13.8,2.0,38731.65\nR001,1996-05-24,11.6,4.7,44506.15\nR001,1996-05-25,10.1,5.9,40098.25\nR001,1996-05-26,12.4,0.0,49126.45\nR001,1996-05-27,15.7,2.3,47688.33\nR001,1996-05-28,13.6,22.8,19122.03\nR001,1996-05-29,15.4,0.5,48769.81\nR001,1996-05-30,15.4,3.1,41885.38\nR001,1996-05-31,15.0,16.0,10237.6\nR001,1996-06-01,11.9,25.3,6154.43\nR001,1996-06-02,9.0,6.6,45995.95\nR001,1996-06-03,10.0,3.1,49937.74\nR001,1996-06-04,10.7,8.2,26583.11\nR001,1996-06-05,10.5,3.2,24963.01\nR001,1996-06-06,11.1,11.7,1747.63\nR001,1996-06-07,11.5,5.3,45028.63\nR001,1996-06-08,11.6,3.9,48952.27\nR001,1996-06-09,11.6,17.6,29660.02\nR001,1996-06-10,13.1,0.0,52177.96\nR001,1996-06-11,14.7,0.0,52777.32\nR001,1996-06-12,17.5,0.8,52397.99\nR001,1996-06-13,18.6,1.3,49299.26\nR001,1996-06-14,18.3,1.6,49276.49\nR001,1996-06-15,18.2,0.0,53097.85\nR001,1996-06-16,17.1,0.8,51447.81\nR001,1996-06-17,16.2,11.4,4651.51\nR001,1996-06-18,14.6,33.0,12617.7\nR001,1996-06-19,14.0,3.3,44142.67\nR001,1996-06-20,15.4,1.8,50598.99\nR001,1996-06-21,14.1,17.8,0.0\nR001,1996-06-22,15.7,3.5,35661.1\nR001,1996-06-23,17.6,0.4,51360.52\nR001,1996-06-24,19.1,3.2,52185.79\nR001,1996-06-25,20.1,2.8,51861.97\nR001,1996-06-26,20.9,0.3,52630.07\nR001,1996-06-27,22.9,0.0,53967.67\nR001,1996-06-28,22.0,0.2,53964.35\nR001,1996-06-29,15.8,31.0,10593.67\nR001,1996-06-30,15.2,0.1,53381.9\nR001,1996-07-01,17.5,0.3,53016.49\nR001,1996-07-02,19.0,0.1,52287.31\nR001,1996-07-03,21.2,0.0,53893.57\nR001,1996-07-04,23.3,0.2,53900.03\nR001,1996-07-05,22.6,3.1,53255.7\nR001,1996-07-06,21.6,1.4,53393.35\nR001,1996-07-07,20.2,2.6,51618.54\nR001,1996-07-08,21.0,2.2,52678.21\nR001,1996-07-09,21.7,3.8,53047.01\nR001,1996-07-10,23.7,0.9,53759.92\nR001,1996-07-11,23.8,0.7,53729.22\nR001,1996-07-12,22.9,2.8,52351.41\nR001,1996-07-13,22.8,4.4,51240.73\nR001,1996-07-14,22.7,2.1,44606.57\nR001,1996-07-15,22.7,0.2,53585.55\nR001,1996-07-16,21.8,2.3,51930.31\nR001,1996-07-17,20.6,3.3,51303.11\nR001,1996-07-18,18.4,12.4,20748.65\nR001,1996-07-19,19.1,8.4,46863.09\nR001,1996-07-20,18.3,19.0,41894.54\nR001,1996-07-21,19.7,5.7,44557.86\nR001,1996-07-22,19.4,4.5,43537.41\nR001,1996-07-23,18.7,11.0,26032.62\nR001,1996-07-24,16.5,42.2,837.6\nR001,1996-07-25,17.4,0.4,49085.02\nR001,1996-07-26,17.4,2.0,49389.74\nR001,1996-07-27,18.5,5.3,42042.38\nR001,1996-07-28,17.2,0.0,49235.59\nR001,1996-07-29,17.9,0.0,48572.81\nR001,1996-07-30,19.3,0.3,48303.61\nR001,1996-07-31,22.0,2.6,49153.63\nR001,1996-08-01,22.2,3.5,48143.71\nR001,1996-08-02,21.8,1.6,46800.0\nR001,1996-08-03,22.4,6.0,47852.53\nR001,1996-08-04,21.7,8.6,38240.06\nR001,1996-08-05,20.8,15.6,43325.87\nR001,1996-08-06,20.9,6.7,41647.56\nR001,1996-08-07,18.6,6.7,39108.77\nR001,1996-08-08,17.8,7.2,45570.17\nR001,1996-08-09,16.2,0.2,45115.09\nR001,1996-08-10,17.1,4.3,45949.74\nR001,1996-08-11,17.3,8.7,43792.26\nR001,1996-08-12,17.8,0.6,46800.0\nR001,1996-08-13,19.3,0.5,46800.0\nR001,1996-08-14,20.0,0.6,45028.06\nR001,1996-08-15,20.9,0.5,46846.75\nR001,1996-08-16,19.8,2.0,43000.28\nR001,1996-08-17,19.8,4.5,41310.19\nR001,1996-08-18,19.6,1.3,44168.45\nR001,1996-08-19,19.2,10.8,43548.6\nR001,1996-08-20,19.7,1.9,46070.05\nR001,1996-08-21,17.5,26.2,25121.78\nR001,1996-08-22,16.9,5.4,44428.54\nR001,1996-08-23,17.5,0.6,45656.05\nR001,1996-08-24,19.4,0.1,46800.0\nR001,1996-08-25,18.3,4.4,43218.71\nR001,1996-08-26,16.7,1.9,44350.81\nR001,1996-08-27,16.0,1.7,44358.52\nR001,1996-08-28,17.2,3.8,43231.02\nR001,1996-08-29,18.3,2.2,44129.1\nR001,1996-08-30,17.5,7.1,42631.33\nR001,1996-08-31,18.7,1.1,41468.42\nR001,1996-09-01,19.1,3.1,43000.35\nR001,1996-09-02,20.4,0.0,44056.51\nR001,1996-09-03,19.1,9.3,34909.24\nR001,1996-09-04,16.9,4.9,38607.77\nR001,1996-09-05,17.2,1.1,40729.11\nR001,1996-09-06,16.5,14.5,29978.03\nR001,1996-09-07,15.3,10.3,30407.26\nR001,1996-09-08,14.7,17.6,34791.79\nR001,1996-09-09,11.4,19.8,23854.27\nR001,1996-09-10,11.2,14.0,16615.91\nR001,1996-09-11,11.4,4.6,35898.08\nR001,1996-09-12,12.0,0.0,41884.07\nR001,1996-09-13,14.6,0.0,43159.04\nR001,1996-09-14,18.6,0.1,37834.12\nR001,1996-09-15,15.1,3.4,33282.89\nR001,1996-09-16,11.0,0.3,40468.25\nR001,1996-09-17,11.1,0.1,41456.51\nR001,1996-09-18,10.6,0.5,40216.94\nR001,1996-09-19,8.6,3.6,39396.35\nR001,1996-09-20,9.2,0.0,39639.49\nR001,1996-09-21,10.2,4.6,21226.27\nR001,1996-09-22,10.2,0.5,38140.27\nR001,1996-09-23,10.9,0.2,38353.91\nR001,1996-09-24,13.0,3.6,21939.96\nR001,1996-09-25,15.6,8.0,35859.85\nR001,1996-09-26,14.3,1.3,39856.86\nR001,1996-09-27,15.3,0.0,40797.84\nR001,1996-09-28,17.2,5.5,38619.02\nR001,1996-09-29,11.9,27.6,6190.75\nR001,1996-09-30,11.8,2.4,31463.02\nR001,1996-10-01,12.1,0.0,39746.5\nR001,1996-10-02,16.1,0.0,39995.12\nR001,1996-10-03,13.7,7.4,4075.48\nR001,1996-10-04,12.2,2.6,37784.29\nR001,1996-10-05,9.2,4.6,38430.58\nR001,1996-10-06,9.9,0.0,38786.88\nR001,1996-10-07,11.4,0.0,36618.44\nR001,1996-10-08,11.0,0.1,36981.67\nR001,1996-10-09,11.4,0.0,36478.42\nR001,1996-10-10,12.9,0.1,36767.54\nR001,1996-10-11,13.1,3.8,28200.02\nR001,1996-10-12,12.0,6.0,381.26\nR001,1996-10-13,11.2,1.0,29909.82\nR001,1996-10-14,10.8,4.0,24559.25\nR001,1996-10-15,11.1,6.0,24308.4\nR001,1996-10-16,11.2,2.0,28492.66\nR001,1996-10-17,10.3,0.0,36000.0\nR001,1996-10-18,11.5,0.0,36000.0\nR001,1996-10-19,13.2,0.0,33419.15\nR001,1996-10-20,14.1,0.2,15146.47\nR001,1996-10-21,13.4,0.3,36000.0\nR001,1996-10-22,13.8,7.8,15234.54\nR001,1996-10-23,9.1,42.3,655.65\nR001,1996-10-24,7.0,3.7,29472.05\nR001,1996-10-25,8.1,6.3,11855.98\nR001,1996-10-26,7.4,16.7,18340.19\nR001,1996-10-27,5.7,7.3,26774.54\nR001,1996-10-28,3.4,8.4,32400.0\nR001,1996-10-29,2.6,0.0,36000.0\nR001,1996-10-30,4.4,0.1,36000.0\nR001,1996-10-31,6.0,0.0,29891.49\nR001,1996-11-01,4.5,9.1,19090.1\nR001,1996-11-02,2.3,0.0,35920.86\nR001,1996-11-03,4.1,0.6,27013.13\nR001,1996-11-04,5.7,2.2,17804.35\nR001,1996-11-05,5.6,0.0,35648.87\nR001,1996-11-06,6.2,0.0,35492.41\nR001,1996-11-07,7.2,0.0,35337.7\nR001,1996-11-08,7.9,0.0,35184.91\nR001,1996-11-09,7.4,0.0,32622.28\nR001,1996-11-10,4.0,8.6,28408.0\nR001,1996-11-11,1.0,0.0,34740.11\nR001,1996-11-12,5.8,0.0,31229.02\nR001,1996-11-13,6.1,0.0,34456.85\nR001,1996-11-14,7.0,0.0,34319.84\nR001,1996-11-15,5.7,0.0,34186.0\nR001,1996-11-16,4.9,0.0,34054.05\nR001,1996-11-17,4.5,0.0,33923.67\nR001,1996-11-18,5.3,0.0,33693.06\nR001,1996-11-19,4.9,0.0,33325.1\nR001,1996-11-20,8.0,0.0,32590.76\nR001,1996-11-21,9.3,0.2,31202.83\nR001,1996-11-22,7.5,2.0,27683.92\nR001,1996-11-23,6.6,0.0,30085.61\nR001,1996-11-24,7.3,0.0,30549.56\nR001,1996-11-25,7.5,0.0,30828.13\nR001,1996-11-26,8.1,0.0,28800.0\nR001,1996-11-27,9.7,0.0,29873.39\nR001,1996-11-28,8.1,0.0,28800.0\nR001,1996-11-29,6.7,1.8,25733.15\nR001,1996-11-30,4.8,0.0,26926.52\nR001,1996-12-01,7.1,0.0,28116.11\nR001,1996-12-02,8.5,0.7,29964.47\nR001,1996-12-03,7.8,0.0,29978.42\nR001,1996-12-04,9.8,0.0,26138.04\nR001,1996-12-05,8.3,0.9,28800.0\nR001,1996-12-06,7.4,4.0,13756.65\nR001,1996-12-07,6.1,0.9,22430.25\nR001,1996-12-08,5.0,3.7,5595.64\nR001,1996-12-09,5.3,0.2,27601.43\nR001,1996-12-10,4.6,0.2,24463.08\nR001,1996-12-11,4.6,1.1,17255.22\nR001,1996-12-12,4.0,0.0,27956.17\nR001,1996-12-13,3.8,1.6,5078.86\nR001,1996-12-14,3.5,1.4,24344.49\nR001,1996-12-15,3.5,0.3,12156.93\nR001,1996-12-16,3.5,32.4,0.0\nR001,1996-12-17,-4.3,4.7,30400.76\nR001,1996-12-18,-1.7,0.0,26096.48\nR001,1996-12-19,3.0,0.6,27973.04\nR001,1996-12-20,3.9,19.5,5678.86\nR001,1996-12-21,4.8,5.2,15786.91\nR001,1996-12-22,2.8,35.8,0.0\nR001,1996-12-23,-0.6,10.3,0.0\nR001,1996-12-24,2.5,57.5,0.0\nR001,1996-12-25,3.2,52.7,0.0\nR001,1996-12-26,3.7,26.3,0.0\nR001,1996-12-27,3.3,42.6,0.0\nR001,1996-12-28,3.7,33.3,4351.96\nR001,1996-12-29,2.1,23.5,0.0\nR001,1996-12-30,-3.0,8.2,25562.53\nR001,1996-12-31,3.2,3.7,4003.25\nR001,1997-01-01,3.0,12.8,5866.58\nR001,1997-01-02,3.1,18.9,5427.23\nR001,1997-01-03,1.0,8.0,1155.42\nR001,1997-01-04,-2.5,2.6,25200.0\nR001,1997-01-05,3.4,3.0,0.0\nR001,1997-01-06,2.3,34.9,16242.17\nR001,1997-01-07,-0.3,3.6,31976.54\nR001,1997-01-08,0.4,0.3,26672.51\nR001,1997-01-09,4.2,3.0,20553.04\nR001,1997-01-10,1.6,5.0,18419.92\nR001,1997-01-11,1.4,0.0,22761.71\nR001,1997-01-12,3.0,0.0,28800.0\nR001,1997-01-13,2.4,0.1,32380.25\nR001,1997-01-14,0.9,0.0,32400.0\nR001,1997-01-15,0.4,0.1,26980.72\nR001,1997-01-16,-2.1,0.8,32400.0\nR001,1997-01-17,-1.1,3.7,16165.29\nR001,1997-01-18,-1.7,0.0,32400.0\nR001,1997-01-19,0.1,0.0,32400.0\nR001,1997-01-20,1.1,0.0,32400.0\nR001,1997-01-21,-0.8,9.2,0.0\nR001,1997-01-22,-1.1,0.9,31844.15\nR001,1997-01-23,-1.9,0.6,32400.0\nR001,1997-01-24,-0.7,0.0,28800.0\nR001,1997-01-25,-2.0,12.0,5967.47\nR001,1997-01-26,-9.1,0.0,31141.96\nR001,1997-01-27,-11.6,0.2,32400.0\nR001,1997-01-28,-10.6,0.2,32400.0\nR001,1997-01-29,-6.3,22.4,0.0\nR001,1997-01-30,-10.5,0.8,32400.0\nR001,1997-01-31,-9.6,3.1,28800.0\nR001,1997-02-01,-4.6,30.0,0.0\nR001,1997-02-02,-5.8,11.1,19098.53\nR001,1997-02-03,-7.5,3.4,21051.5\nR001,1997-02-04,-7.1,0.0,25169.5\nR001,1997-02-05,-8.0,0.0,33562.48\nR001,1997-02-06,-7.3,0.0,32691.85\nR001,1997-02-07,-3.3,7.8,0.0\nR001,1997-02-08,-5.2,0.0,34669.39\nR001,1997-02-09,-2.8,0.2,33733.58\nR001,1997-02-10,-2.5,3.3,27435.06\nR001,1997-02-11,-1.8,1.2,19090.44\nR001,1997-02-12,0.4,0.3,34304.86\nR001,1997-02-13,1.4,0.0,31612.35\nR001,1997-02-14,3.4,0.3,29188.37\nR001,1997-02-15,2.4,11.9,20765.45\nR001,1997-02-16,3.0,0.0,37331.35\nR001,1997-02-17,4.2,28.5,4715.23\nR001,1997-02-18,-3.5,1.1,35407.8\nR001,1997-02-19,-5.4,0.0,37965.81\nR001,1997-02-20,-5.4,0.0,38138.71\nR001,1997-02-21,-0.4,0.0,31939.19\nR001,1997-02-22,-0.1,7.2,7437.98\nR001,1997-02-23,-1.0,2.4,32395.65\nR001,1997-02-24,-0.6,0.0,35768.18\nR001,1997-02-25,-1.1,0.0,39005.17\nR001,1997-02-26,-0.9,0.0,39181.56\nR001,1997-02-27,0.5,0.0,39354.99\nR001,1997-02-28,2.3,9.5,0.0\nR001,1997-03-01,-0.6,8.1,21956.73\nR001,1997-03-02,-3.3,0.0,39600.0\nR001,1997-03-03,-1.9,0.0,39600.0\nR001,1997-03-04,-1.9,2.9,7200.0\nR001,1997-03-05,-2.7,13.3,29269.72\nR001,1997-03-06,-4.8,3.7,39600.0\nR001,1997-03-07,-6.0,0.0,39360.58\nR001,1997-03-08,-5.8,0.0,39600.0\nR001,1997-03-09,-5.1,0.0,38960.7\nR001,1997-03-10,-2.2,0.0,39600.0\nR001,1997-03-11,-1.2,0.0,39600.0\nR001,1997-03-12,0.8,0.0,39600.0\nR001,1997-03-13,2.0,0.0,39600.0\nR001,1997-03-14,2.5,0.0,39600.0\nR001,1997-03-15,2.7,0.0,39841.96\nR001,1997-03-16,1.7,0.0,39788.79\nR001,1997-03-17,-0.9,10.2,16555.52\nR001,1997-03-18,-4.2,5.2,18557.16\nR001,1997-03-19,-6.3,0.0,35571.63\nR001,1997-03-20,-3.3,0.0,36065.61\nR001,1997-03-21,4.1,0.0,38875.92\nR001,1997-03-22,3.8,1.5,11441.47\nR001,1997-03-23,-0.2,10.6,8719.68\nR001,1997-03-24,-3.4,2.4,39999.45\nR001,1997-03-25,-2.9,0.0,41497.83\nR001,1997-03-26,-3.8,3.3,29942.97\nR001,1997-03-27,-3.6,0.0,35752.09\nR001,1997-03-28,-3.6,0.0,39836.64\nR001,1997-03-29,-0.4,0.0,43972.15\nR001,1997-03-30,6.3,0.0,43091.29\nR001,1997-03-31,9.6,0.0,44438.33\nR001,1997-04-01,8.3,0.0,45149.03\nR001,1997-04-02,8.2,0.0,44514.84\nR001,1997-04-03,8.3,0.0,40973.21\nR001,1997-04-04,6.8,11.8,40456.4\nR001,1997-04-05,5.3,0.6,35781.76\nR001,1997-04-06,5.5,0.1,44352.97\nR001,1997-04-07,3.9,9.2,22856.7\nR001,1997-04-08,2.9,0.2,42606.58\nR001,1997-04-09,1.9,13.4,1472.23\nR001,1997-04-10,-0.9,10.9,16932.8\nR001,1997-04-11,-2.8,3.2,46372.24\nR001,1997-04-12,1.9,0.2,43512.59\nR001,1997-04-13,5.5,0.1,46515.12\nR001,1997-04-14,6.2,2.1,23063.32\nR001,1997-04-15,4.1,13.9,21908.31\nR001,1997-04-16,3.9,0.3,45637.05\nR001,1997-04-17,7.2,0.0,46800.0\nR001,1997-04-18,6.1,10.8,40116.12\nR001,1997-04-19,1.3,4.1,44917.74\nR001,1997-04-20,4.6,0.9,30898.99\nR001,1997-04-21,7.4,3.4,40127.13\nR001,1997-04-22,10.9,0.0,38068.05\nR001,1997-04-23,14.9,0.6,33699.01\nR001,1997-04-24,11.5,7.2,22240.3\nR001,1997-04-25,8.8,13.7,31123.12\nR001,1997-04-26,7.8,12.4,34016.56\nR001,1997-04-27,7.5,1.6,41026.4\nR001,1997-04-28,5.0,14.5,2951.24\nR001,1997-04-29,4.2,10.3,1750.94\nR001,1997-04-30,5.7,1.6,38461.99\nR001,1997-05-01,6.3,0.0,47507.08\nR001,1997-05-02,5.9,0.4,46488.67\nR001,1997-05-03,6.1,5.8,15713.54\nR001,1997-05-04,5.9,6.4,34830.61\nR001,1997-05-05,6.6,0.5,45046.58\nR001,1997-05-06,9.7,0.0,47324.99\nR001,1997-05-07,12.4,0.0,49897.99\nR001,1997-05-08,14.7,0.0,47257.19\nR001,1997-05-09,15.2,0.0,50049.12\nR001,1997-05-10,16.5,0.0,50122.14\nR001,1997-05-11,17.4,0.0,50193.31\nR001,1997-05-12,17.1,0.5,48163.46\nR001,1997-05-13,15.4,0.6,50204.64\nR001,1997-05-14,14.6,0.7,48801.99\nR001,1997-05-15,13.8,0.1,48353.15\nR001,1997-05-16,14.8,0.0,50400.0\nR001,1997-05-17,15.7,0.0,50400.0\nR001,1997-05-18,13.4,9.6,15912.51\nR001,1997-05-19,12.8,0.7,43554.77\nR001,1997-05-20,14.4,0.6,49392.42\nR001,1997-05-21,15.7,2.6,50153.72\nR001,1997-05-22,15.1,3.9,45424.63\nR001,1997-05-23,16.6,0.9,46335.23\nR001,1997-05-24,17.1,2.1,50400.0\nR001,1997-05-25,14.5,11.5,42524.46\nR001,1997-05-26,12.3,10.8,16436.05\nR001,1997-05-27,13.2,2.3,46117.49\nR001,1997-05-28,10.6,25.7,36882.46\nR001,1997-05-29,13.3,0.0,51543.75\nR001,1997-05-30,16.9,2.4,50439.07\nR001,1997-05-31,13.5,21.0,27439.35\nR001,1997-06-01,13.4,9.7,38264.99\nR001,1997-06-02,10.3,12.2,28941.72\nR001,1997-06-03,11.3,0.0,50400.0\nR001,1997-06-04,13.9,0.0,51063.46\nR001,1997-06-05,14.6,6.9,42169.16\nR001,1997-06-06,13.9,7.8,19072.63\nR001,1997-06-07,16.1,1.2,44817.32\nR001,1997-06-08,17.8,3.1,38321.41\nR001,1997-06-09,15.3,19.3,4599.96\nR001,1997-06-10,17.3,2.7,50223.86\nR001,1997-06-11,18.9,6.1,49556.76\nR001,1997-06-12,17.5,20.0,29032.38\nR001,1997-06-13,15.8,16.7,44558.99\nR001,1997-06-14,16.9,5.4,49736.01\nR001,1997-06-15,16.0,8.5,42355.81\nR001,1997-06-16,16.5,3.6,47403.54\nR001,1997-06-17,16.6,1.2,45018.09\nR001,1997-06-18,17.9,0.1,49680.6\nR001,1997-06-19,17.6,0.6,51048.2\nR001,1997-06-20,17.3,0.2,50358.78\nR001,1997-06-21,19.8,0.8,53414.44\nR001,1997-06-22,17.9,1.6,51324.22\nR001,1997-06-23,16.5,0.2,50347.63\nR001,1997-06-24,18.3,1.0,51367.85\nR001,1997-06-25,19.4,0.0,53762.13\nR001,1997-06-26,21.4,0.7,53391.28\nR001,1997-06-27,20.3,7.9,45010.69\nR001,1997-06-28,20.6,3.3,50868.79\nR001,1997-06-29,18.3,6.2,27012.25\nR001,1997-06-30,18.8,11.4,47053.7\nR001,1997-07-01,19.1,14.1,21781.76\nR001,1997-07-02,19.0,14.3,38581.77\nR001,1997-07-03,19.8,4.1,46770.64\nR001,1997-07-04,17.2,5.2,38606.85\nR001,1997-07-05,16.4,8.3,33936.31\nR001,1997-07-06,17.0,5.6,44880.79\nR001,1997-07-07,19.8,0.0,49294.9\nR001,1997-07-08,21.4,0.4,52006.14\nR001,1997-07-09,21.9,0.8,53767.74\nR001,1997-07-10,18.8,8.8,49361.54\nR001,1997-07-11,17.6,0.4,51229.76\nR001,1997-07-12,17.8,1.8,48112.15\nR001,1997-07-13,17.5,0.0,53078.15\nR001,1997-07-14,17.4,2.7,50761.44\nR001,1997-07-15,15.7,6.9,44095.96\nR001,1997-07-16,14.9,4.8,46033.48\nR001,1997-07-17,14.6,8.9,32697.19\nR001,1997-07-18,14.8,1.2,49541.06\nR001,1997-07-19,16.3,0.0,52800.81\nR001,1997-07-20,19.9,0.0,52783.73\nR001,1997-07-21,19.2,2.0,51682.46\nR001,1997-07-22,18.0,0.4,49639.79\nR001,1997-07-23,16.9,0.1,50385.8\nR001,1997-07-24,16.1,3.8,47183.33\nR001,1997-07-25,16.1,2.0,47674.06\nR001,1997-07-26,18.7,0.0,50796.12\nR001,1997-07-27,21.5,0.2,51110.2\nR001,1997-07-28,21.7,1.6,49570.93\nR001,1997-07-29,18.4,60.3,25371.41\nR001,1997-07-30,18.5,1.6,46675.08\nR001,1997-07-31,19.7,0.5,45824.29\nR001,1997-08-01,20.7,6.5,42853.05\nR001,1997-08-02,19.9,12.6,35259.7\nR001,1997-08-03,21.2,0.0,47489.07\nR001,1997-08-04,19.4,0.3,48213.39\nR001,1997-08-05,16.9,4.4,40136.5\nR001,1997-08-06,18.2,8.3,32551.77\nR001,1997-08-07,19.2,3.0,46406.94\nR001,1997-08-08,20.3,1.5,46800.0\nR001,1997-08-09,19.8,5.7,18454.36\nR001,1997-08-10,19.8,19.1,33107.6\nR001,1997-08-11,21.4,9.9,40469.71\nR001,1997-08-12,18.9,16.2,41111.97\nR001,1997-08-13,18.4,3.7,46800.0\nR001,1997-08-14,17.3,0.4,41570.64\nR001,1997-08-15,17.5,6.1,41475.61\nR001,1997-08-16,18.1,0.0,46180.28\nR001,1997-08-17,18.0,1.2,46800.0\nR001,1997-08-18,18.1,0.9,44789.44\nR001,1997-08-19,17.3,2.0,40141.28\nR001,1997-08-20,16.8,10.2,33116.11\nR001,1997-08-21,17.4,8.5,41975.5\nR001,1997-08-22,16.4,9.8,38245.01\nR001,1997-08-23,18.2,8.2,44023.01\nR001,1997-08-24,19.4,1.8,40927.55\nR001,1997-08-25,22.3,1.3,46008.55\nR001,1997-08-26,19.2,16.2,19581.97\nR001,1997-08-27,18.2,18.1,38373.26\nR001,1997-08-28,18.6,2.8,44013.74\nR001,1997-08-29,18.9,4.4,43959.16\nR001,1997-08-30,19.7,0.0,44868.11\nR001,1997-08-31,19.5,0.9,45504.65\nR001,1997-09-01,15.6,35.0,11080.68\nR001,1997-09-02,12.6,4.1,26451.77\nR001,1997-09-03,11.8,6.5,25362.47\nR001,1997-09-04,9.5,8.0,23601.63\nR001,1997-09-05,9.5,5.8,27582.91\nR001,1997-09-06,10.9,0.0,43560.88\nR001,1997-09-07,12.0,0.4,43932.62\nR001,1997-09-08,14.3,0.0,43346.21\nR001,1997-09-09,14.3,0.0,43200.0\nR001,1997-09-10,14.6,0.0,42119.61\nR001,1997-09-11,10.9,17.1,26282.12\nR001,1997-09-12,9.3,0.5,41504.08\nR001,1997-09-13,10.0,0.0,41636.03\nR001,1997-09-14,13.9,0.0,43108.37\nR001,1997-09-15,16.6,0.0,43041.03\nR001,1997-09-16,15.6,1.0,42973.6\nR001,1997-09-17,13.1,5.0,30320.42\nR001,1997-09-18,13.5,0.0,42838.37\nR001,1997-09-19,16.1,0.0,39924.36\nR001,1997-09-20,17.2,1.5,42186.44\nR001,1997-09-21,12.7,29.9,597.58\nR001,1997-09-22,10.2,0.5,37014.56\nR001,1997-09-23,10.5,0.0,40061.85\nR001,1997-09-24,11.4,0.0,41371.07\nR001,1997-09-25,10.2,3.7,27980.64\nR001,1997-09-26,7.5,1.8,40167.23\nR001,1997-09-27,8.1,0.0,39600.0\nR001,1997-09-28,8.9,3.1,1378.84\nR001,1997-09-29,8.1,6.0,28349.49\nR001,1997-09-30,8.0,3.7,37569.61\nR001,1997-10-01,7.9,2.7,20393.86\nR001,1997-10-02,8.3,14.9,0.0\nR001,1997-10-03,10.4,0.0,36454.85\nR001,1997-10-04,10.7,28.1,15462.05\nR001,1997-10-05,8.5,0.0,31833.41\nR001,1997-10-06,8.1,8.1,22270.41\nR001,1997-10-07,9.3,1.3,37882.47\nR001,1997-10-08,10.4,0.0,38630.01\nR001,1997-10-09,12.7,0.0,37754.83\nR001,1997-10-10,14.6,0.0,37525.12\nR001,1997-10-11,16.3,0.0,37137.23\nR001,1997-10-12,17.4,0.0,36541.95\nR001,1997-10-13,16.0,1.4,36000.0\nR001,1997-10-14,14.9,3.7,12586.25\nR001,1997-10-15,17.2,0.0,36050.15\nR001,1997-10-16,19.8,0.0,36102.24\nR001,1997-10-17,15.7,7.5,23582.97\nR001,1997-10-18,13.2,10.7,16582.82\nR001,1997-10-19,12.3,23.0,0.0\nR001,1997-10-20,12.6,0.0,33026.5\nR001,1997-10-21,11.6,8.0,10263.56\nR001,1997-10-22,8.7,12.3,70.6\nR001,1997-10-23,8.3,0.1,33505.77\nR001,1997-10-24,10.3,0.0,36000.0\nR001,1997-10-25,11.1,14.4,11799.3\nR001,1997-10-26,8.8,70.0,0.0\nR001,1997-10-27,6.5,14.8,2728.9\nR001,1997-10-28,6.0,6.7,17962.39\nR001,1997-10-29,5.8,7.8,20053.38\nR001,1997-10-30,2.9,8.5,29949.24\nR001,1997-10-31,-0.4,0.0,36000.0\nR001,1997-11-01,1.9,0.0,36000.0\nR001,1997-11-02,3.9,0.0,35937.36\nR001,1997-11-03,4.7,0.0,35857.42\nR001,1997-11-04,5.6,13.6,7961.33\nR001,1997-11-05,1.3,4.7,33195.07\nR001,1997-11-06,0.1,0.0,35524.97\nR001,1997-11-07,4.4,0.0,35372.37\nR001,1997-11-08,5.9,0.0,35222.31\nR001,1997-11-09,7.3,0.0,35074.97\nR001,1997-11-10,7.0,0.0,34930.36\nR001,1997-11-11,6.4,0.0,34787.12\nR001,1997-11-12,6.0,0.0,30854.01\nR001,1997-11-13,6.8,0.0,34504.07\nR001,1997-11-14,7.1,0.0,34364.59\nR001,1997-11-15,8.7,0.0,29316.0\nR001,1997-11-16,9.5,0.3,21712.43\nR001,1997-11-17,9.2,0.0,32061.19\nR001,1997-11-18,6.4,7.1,2847.38\nR001,1997-11-19,8.7,1.0,25285.65\nR001,1997-11-20,8.2,6.8,15767.24\nR001,1997-11-21,8.3,0.0,28322.46\nR001,1997-11-22,8.5,0.0,29003.45\nR001,1997-11-23,7.4,0.5,8199.27\nR001,1997-11-24,5.6,0.0,30545.68\nR001,1997-11-25,5.5,0.0,31561.94\nR001,1997-11-26,5.2,0.0,31256.79\nR001,1997-11-27,4.4,0.0,30246.83\nR001,1997-11-28,4.4,3.8,18914.25\nR001,1997-11-29,4.7,2.6,21600.0\nR001,1997-11-30,4.1,0.0,19633.64\nR001,1997-12-01,5.2,0.0,30325.17\nR001,1997-12-02,6.0,1.7,28818.74\nR001,1997-12-03,6.6,2.8,11941.39\nR001,1997-12-04,6.5,2.9,28800.0\nR001,1997-12-05,4.8,30.8,22187.72\nR001,1997-12-06,2.9,7.9,12722.98\nR001,1997-12-07,3.8,2.8,3037.52\nR001,1997-12-08,5.2,17.5,8888.31\nR001,1997-12-09,3.5,20.8,0.0\nR001,1997-12-10,-1.5,0.9,28436.08\nR001,1997-12-11,-2.7,0.0,28800.0\nR001,1997-12-12,0.5,0.0,24192.44\nR001,1997-12-13,3.3,0.1,11983.98\nR001,1997-12-14,4.6,8.3,4860.41\nR001,1997-12-15,3.9,16.3,0.0\nR001,1997-12-16,1.8,23.3,0.0\nR001,1997-12-17,-0.8,11.1,0.0\nR001,1997-12-18,-5.1,3.5,17133.82\nR001,1997-12-19,-5.7,0.1,23350.16\nR001,1997-12-20,-3.6,0.0,30769.55\nR001,1997-12-21,0.7,5.4,10191.98\nR001,1997-12-22,3.7,3.4,18998.38\nR001,1997-12-23,4.4,0.0,22844.31\nR001,1997-12-24,2.6,2.3,7469.97\nR001,1997-12-25,0.8,3.2,10799.04\nR001,1997-12-26,-0.1,1.5,25765.6\nR001,1997-12-27,0.4,0.0,28800.0\nR001,1997-12-28,1.8,0.0,31407.54\nR001,1997-12-29,2.9,0.0,29818.13\nR001,1997-12-30,4.1,0.0,24088.4\nR001,1997-12-31,0.9,6.8,3151.15\nR001,1998-01-01,-2.1,0.1,31608.83\nR001,1998-01-02,-3.8,0.0,31663.59\nR001,1998-01-03,-5.4,0.0,31719.86\nR001,1998-01-04,-4.3,0.0,31777.53\nR001,1998-01-05,-0.4,0.0,29126.67\nR001,1998-01-06,-0.0,6.5,20077.72\nR001,1998-01-07,-0.5,5.5,20420.87\nR001,1998-01-08,-1.7,4.1,30256.7\nR001,1998-01-09,-1.2,11.4,0.0\nR001,1998-01-10,-1.6,34.1,639.95\nR001,1998-01-11,-5.7,0.0,32214.38\nR001,1998-01-12,-6.1,5.9,0.0\nR001,1998-01-13,-3.0,10.6,0.0\nR001,1998-01-14,0.8,3.5,15530.02\nR001,1998-01-15,0.5,0.0,32400.0\nR001,1998-01-16,0.4,0.0,23181.74\nR001,1998-01-17,0.2,0.0,26972.93\nR001,1998-01-18,-0.3,0.0,29242.54\nR001,1998-01-19,-0.4,2.0,20406.15\nR001,1998-01-20,-0.6,0.0,32335.2\nR001,1998-01-21,0.3,0.0,32400.0\nR001,1998-01-22,-1.0,0.0,32400.0\nR001,1998-01-23,-1.2,0.0,32400.0\nR001,1998-01-24,0.9,0.0,32400.0\nR001,1998-01-25,1.3,0.0,29884.99\nR001,1998-01-26,0.6,5.2,0.0\nR001,1998-01-27,0.1,2.7,18371.63\nR001,1998-01-28,-0.2,0.0,29730.09\nR001,1998-01-29,1.1,12.2,0.0\nR001,1998-01-30,-2.6,12.4,17421.69\nR001,1998-01-31,-7.8,0.0,32417.59\nR001,1998-02-01,-4.2,7.2,32400.0\nR001,1998-02-02,-2.6,10.3,442.83\nR001,1998-02-03,0.1,12.3,266.0\nR001,1998-02-04,2.5,16.2,4421.12\nR001,1998-02-05,4.6,19.8,0.0\nR001,1998-02-06,5.5,24.4,0.0\nR001,1998-02-07,1.1,22.1,15204.86\nR001,1998-02-08,-7.0,1.1,32413.95\nR001,1998-02-09,-7.6,0.0,34947.56\nR001,1998-02-10,-4.5,0.0,10347.31\nR001,1998-02-11,-4.9,0.0,35460.13\nR001,1998-02-12,-2.7,2.7,32441.82\nR001,1998-02-13,-0.6,22.7,0.0\nR001,1998-02-14,-7.1,3.3,35569.38\nR001,1998-02-15,-9.3,4.7,0.0\nR001,1998-02-16,-7.1,1.0,36312.59\nR001,1998-02-17,-1.9,22.4,15223.52\nR001,1998-02-18,-4.3,21.3,31032.84\nR001,1998-02-19,-8.8,0.0,37745.08\nR001,1998-02-20,-1.8,0.3,34909.51\nR001,1998-02-21,0.4,0.0,38130.83\nR001,1998-02-22,2.4,0.0,38430.72\nR001,1998-02-23,5.3,0.0,38606.58\nR001,1998-02-24,4.9,0.0,38783.58\nR001,1998-02-25,3.2,0.2,38961.56\nR001,1998-02-26,-2.5,13.3,36266.41\nR001,1998-02-27,-5.6,0.0,37609.27\nR001,1998-02-28,-0.6,0.0,39437.86\nR001,1998-03-01,1.4,0.0,39539.85\nR001,1998-03-02,1.0,0.0,39600.0\nR001,1998-03-03,-0.2,6.5,39600.0\nR001,1998-03-04,-0.1,1.8,28800.0\nR001,1998-03-05,2.7,6.5,5526.37\nR001,1998-03-06,4.9,0.0,39600.0\nR001,1998-03-07,1.5,13.8,22947.43\nR001,1998-03-08,1.5,0.0,39600.0\nR001,1998-03-09,6.2,0.0,39455.99\nR001,1998-03-10,7.0,0.0,34512.94\nR001,1998-03-11,8.5,0.0,34397.77\nR001,1998-03-12,5.8,7.5,18936.03\nR001,1998-03-13,-0.3,5.2,36541.05\nR001,1998-03-14,-0.1,3.5,12885.04\nR001,1998-03-15,1.2,0.0,39600.0\nR001,1998-03-16,2.1,3.9,13940.39\nR001,1998-03-17,2.4,7.2,1236.52\nR001,1998-03-18,1.5,0.4,33783.65\nR001,1998-03-19,-1.3,14.3,23920.92\nR001,1998-03-20,-1.5,15.2,21850.68\nR001,1998-03-21,0.1,1.3,36456.38\nR001,1998-03-22,0.0,11.4,10997.46\nR001,1998-03-23,-0.1,10.1,5448.98\nR001,1998-03-24,-2.0,3.9,26615.57\nR001,1998-03-25,-3.3,0.0,40881.3\nR001,1998-03-26,-1.8,0.0,36904.43\nR001,1998-03-27,4.0,0.0,42306.8\nR001,1998-03-28,5.9,1.7,30086.6\nR001,1998-03-29,4.0,9.6,0.0\nR001,1998-03-30,2.4,5.5,163.77\nR001,1998-03-31,2.2,2.6,42731.64\nR001,1998-04-01,3.3,0.1,42497.34\nR001,1998-04-02,6.1,0.0,45195.24\nR001,1998-04-03,6.2,0.0,44311.41\nR001,1998-04-04,7.2,0.2,44703.87\nR001,1998-04-05,7.8,0.0,45926.98\nR001,1998-04-06,9.5,0.0,45998.49\nR001,1998-04-07,10.5,0.0,46069.98\nR001,1998-04-08,11.9,0.0,41790.09\nR001,1998-04-09,12.0,0.0,46212.86\nR001,1998-04-10,13.3,0.0,44229.58\nR001,1998-04-11,14.0,1.8,46355.56\nR001,1998-04-12,16.2,0.0,46426.81\nR001,1998-04-13,16.5,0.0,46498.0\nR001,1998-04-14,12.3,2.8,39600.0\nR001,1998-04-15,9.7,7.8,38286.19\nR001,1998-04-16,7.5,2.1,41090.54\nR001,1998-04-17,7.2,3.1,14485.56\nR001,1998-04-18,10.5,3.7,37598.1\nR001,1998-04-19,11.5,4.2,36081.01\nR001,1998-04-20,8.4,18.7,6835.13\nR001,1998-04-21,6.6,3.5,4076.47\nR001,1998-04-22,5.9,7.9,34030.03\nR001,1998-04-23,5.8,2.2,34170.72\nR001,1998-04-24,6.3,4.6,37197.53\nR001,1998-04-25,8.0,0.0,45528.87\nR001,1998-04-26,11.1,0.1,47382.89\nR001,1998-04-27,11.7,2.4,25161.93\nR001,1998-04-28,9.0,14.2,17401.65\nR001,1998-04-29,8.5,6.0,13557.15\nR001,1998-04-30,8.9,3.4,29896.04\nR001,1998-05-01,9.6,3.7,42941.61\nR001,1998-05-02,10.3,7.5,45629.36\nR001,1998-05-03,12.1,0.0,48785.81\nR001,1998-05-04,16.2,0.2,36570.35\nR001,1998-05-05,18.2,0.5,39089.85\nR001,1998-05-06,18.6,3.8,33452.43\nR001,1998-05-07,14.5,4.7,43370.82\nR001,1998-05-08,15.9,0.6,42847.01\nR001,1998-05-09,16.4,1.0,41914.6\nR001,1998-05-10,18.2,3.7,14560.35\nR001,1998-05-11,19.3,6.2,34413.77\nR001,1998-05-12,15.4,26.8,12330.12\nR001,1998-05-13,14.6,9.6,35868.65\nR001,1998-05-14,12.4,11.1,22255.42\nR001,1998-05-15,11.5,12.3,2327.61\nR001,1998-05-16,12.5,2.3,42129.72\nR001,1998-05-17,15.7,0.0,37481.14\nR001,1998-05-18,17.3,4.6,41217.61\nR001,1998-05-19,16.2,10.0,36499.82\nR001,1998-05-20,12.9,18.2,32397.01\nR001,1998-05-21,11.2,10.7,4921.67\nR001,1998-05-22,10.5,10.5,32883.32\nR001,1998-05-23,12.6,1.7,46783.31\nR001,1998-05-24,10.8,4.8,36368.08\nR001,1998-05-25,12.8,0.1,49153.61\nR001,1998-05-26,16.4,0.0,50539.78\nR001,1998-05-27,20.3,0.3,50874.78\nR001,1998-05-28,17.6,12.2,38992.2\nR001,1998-05-29,15.9,20.1,723.37\nR001,1998-05-30,16.1,16.9,14827.16\nR001,1998-05-31,14.7,15.1,26702.44\nR001,1998-06-01,14.8,15.2,44457.24\nR001,1998-06-02,14.5,13.2,29989.97\nR001,1998-06-03,15.2,13.4,33160.06\nR001,1998-06-04,15.7,12.1,42941.16\nR001,1998-06-05,16.0,9.7,42591.89\nR001,1998-06-06,18.3,4.7,47682.12\nR001,1998-06-07,19.4,13.0,37917.13\nR001,1998-06-08,20.3,1.7,52091.21\nR001,1998-06-09,20.1,2.5,52075.74\nR001,1998-06-10,20.6,5.7,51328.46\nR001,1998-06-11,20.5,8.3,46685.62\nR001,1998-06-12,21.7,2.5,50733.44\nR001,1998-06-13,22.5,3.4,50919.36\nR001,1998-06-14,22.7,2.0,53477.46\nR001,1998-06-15,21.8,2.0,50788.24\nR001,1998-06-16,21.7,0.9,44455.1\nR001,1998-06-17,22.3,1.4,48139.24\nR001,1998-06-18,22.1,6.1,48799.46\nR001,1998-06-19,22.0,2.0,53432.4\nR001,1998-06-20,22.4,0.5,51234.97\nR001,1998-06-21,23.3,1.3,53418.07\nR001,1998-06-22,18.9,24.8,13880.9\nR001,1998-06-23,17.2,3.2,43280.49\nR001,1998-06-24,17.3,0.7,50844.8\nR001,1998-06-25,18.6,0.0,53959.55\nR001,1998-06-26,18.5,2.1,52362.44\nR001,1998-06-27,18.1,6.5,51175.84\nR001,1998-06-28,17.8,3.4,45315.19\nR001,1998-06-29,19.0,0.1,51275.31\nR001,1998-06-30,19.0,2.3,50326.88\nR001,1998-07-01,19.6,2.1,48151.23\nR001,1998-07-02,19.5,4.2,43322.33\nR001,1998-07-03,20.3,0.0,49912.84\nR001,1998-07-04,17.6,5.3,36577.68\nR001,1998-07-05,16.5,4.9,41120.09\nR001,1998-07-06,15.7,21.4,48552.06\nR001,1998-07-07,14.9,0.9,50059.41\nR001,1998-07-08,16.7,0.1,52747.4\nR001,1998-07-09,20.6,0.0,53806.34\nR001,1998-07-10,18.5,15.0,44652.13\nR001,1998-07-11,16.4,0.1,52476.71\nR001,1998-07-12,17.4,0.0,51495.35\nR001,1998-07-13,17.9,0.0,51944.61\nR001,1998-07-14,19.3,0.0,53642.87\nR001,1998-07-15,22.4,0.0,53603.85\nR001,1998-07-16,23.7,0.0,53468.36\nR001,1998-07-17,23.2,5.5,51338.87\nR001,1998-07-18,19.8,6.2,42293.31\nR001,1998-07-19,19.7,0.0,48419.78\nR001,1998-07-20,21.6,0.0,52380.6\nR001,1998-07-21,22.9,1.3,51684.16\nR001,1998-07-22,21.5,1.0,50688.02\nR001,1998-07-23,21.9,1.0,49675.52\nR001,1998-07-24,23.0,0.0,52275.36\nR001,1998-07-25,23.4,0.0,50112.31\nR001,1998-07-26,22.9,0.5,50859.75\nR001,1998-07-27,22.6,1.2,51463.54\nR001,1998-07-28,22.4,3.4,50034.3\nR001,1998-07-29,22.6,1.9,49954.19\nR001,1998-07-30,22.9,4.8,50005.61\nR001,1998-07-31,21.4,5.6,45386.31\nR001,1998-08-01,22.2,0.5,48040.9\nR001,1998-08-02,23.2,0.0,49132.18\nR001,1998-08-03,23.3,0.0,48846.99\nR001,1998-08-04,23.0,0.6,47879.1\nR001,1998-08-05,22.5,1.5,47550.44\nR001,1998-08-06,22.9,0.3,47665.9\nR001,1998-08-07,21.0,4.2,43252.33\nR001,1998-08-08,20.9,0.2,44503.06\nR001,1998-08-09,20.5,0.0,46986.3\nR001,1998-08-10,20.4,1.3,47492.57\nR001,1998-08-11,19.0,0.8,46800.0\nR001,1998-08-12,16.4,0.0,47412.85\nR001,1998-08-13,16.2,0.0,46800.0\nR001,1998-08-14,17.7,0.0,47471.87\nR001,1998-08-15,18.7,2.7,47167.64\nR001,1998-08-16,17.0,0.0,45607.54\nR001,1998-08-17,17.1,0.0,43247.52\nR001,1998-08-18,17.7,0.6,46800.0\nR001,1998-08-19,19.4,0.1,46800.0\nR001,1998-08-20,19.5,1.2,45967.48\nR001,1998-08-21,19.6,0.7,46800.0\nR001,1998-08-22,20.6,0.8,46800.0\nR001,1998-08-23,21.0,0.0,46800.0\nR001,1998-08-24,20.1,0.0,45137.51\nR001,1998-08-25,21.8,0.0,46800.0\nR001,1998-08-26,21.8,0.0,46800.0\nR001,1998-08-27,20.3,0.0,46126.97\nR001,1998-08-28,22.1,0.0,46651.37\nR001,1998-08-29,23.8,0.2,46547.69\nR001,1998-08-30,23.0,0.1,46443.55\nR001,1998-08-31,19.2,5.1,37846.79\nR001,1998-09-01,18.1,3.9,37074.28\nR001,1998-09-02,13.6,3.8,36972.42\nR001,1998-09-03,11.7,1.1,43712.99\nR001,1998-09-04,10.9,0.0,44083.85\nR001,1998-09-05,10.2,1.2,42131.58\nR001,1998-09-06,10.7,1.6,42733.62\nR001,1998-09-07,12.1,0.0,43915.49\nR001,1998-09-08,14.3,4.0,37407.64\nR001,1998-09-09,15.2,1.4,39414.69\nR001,1998-09-10,15.1,0.0,40683.42\nR001,1998-09-11,17.1,0.0,43200.0\nR001,1998-09-12,18.3,0.0,43200.0\nR001,1998-09-13,19.6,0.5,42736.02\nR001,1998-09-14,21.2,0.0,43123.72\nR001,1998-09-15,22.7,0.0,43056.31\nR001,1998-09-16,20.9,2.6,42988.8\nR001,1998-09-17,17.5,4.1,36271.39\nR001,1998-09-18,17.5,3.1,39195.07\nR001,1998-09-19,19.7,0.0,42781.28\nR001,1998-09-20,20.7,0.2,42717.77\nR001,1998-09-21,17.7,7.5,38540.29\nR001,1998-09-22,18.2,0.5,39208.33\nR001,1998-09-23,15.4,40.2,30024.32\nR001,1998-09-24,12.5,12.4,159.78\nR001,1998-09-25,14.1,1.2,33875.76\nR001,1998-09-26,12.7,0.0,38877.43\nR001,1998-09-27,13.4,0.0,39875.59\nR001,1998-09-28,14.9,0.0,41187.8\nR001,1998-09-29,18.8,0.0,41217.54\nR001,1998-09-30,20.0,0.0,39483.86\nR001,1998-10-01,13.8,7.1,6008.03\nR001,1998-10-02,11.8,0.0,38655.59\nR001,1998-10-03,13.0,0.0,39648.29\nR001,1998-10-04,15.6,0.0,37054.1\nR001,1998-10-05,15.3,1.1,37433.72\nR001,1998-10-06,13.9,0.0,38040.36\nR001,1998-10-07,9.8,1.8,29422.24\nR001,1998-10-08,9.0,0.0,37929.41\nR001,1998-10-09,10.5,0.0,37736.0\nR001,1998-10-10,12.0,0.0,37505.87\nR001,1998-10-11,16.3,0.0,36476.18\nR001,1998-10-12,18.3,0.0,36855.75\nR001,1998-10-13,21.1,0.0,36258.0\nR001,1998-10-14,20.8,0.0,32958.51\nR001,1998-10-15,17.8,0.0,36362.31\nR001,1998-10-16,15.6,0.0,36000.0\nR001,1998-10-17,12.7,3.1,24303.61\nR001,1998-10-18,11.7,3.0,28282.76\nR001,1998-10-19,10.8,6.5,24707.9\nR001,1998-10-20,11.6,0.0,36000.0\nR001,1998-10-21,12.4,2.1,36000.0\nR001,1998-10-22,6.5,1.4,36000.0\nR001,1998-10-23,5.1,0.0,36000.0\nR001,1998-10-24,7.6,0.0,36000.0\nR001,1998-10-25,12.8,0.0,32757.95\nR001,1998-10-26,14.9,0.1,34334.42\nR001,1998-10-27,14.8,0.0,13169.2\nR001,1998-10-28,11.3,24.1,1348.08\nR001,1998-10-29,10.3,21.0,9389.45\nR001,1998-10-30,10.9,0.0,33408.45\nR001,1998-10-31,10.1,0.0,34586.74\nR001,1998-11-01,7.5,10.1,24849.39\nR001,1998-11-02,7.3,3.2,19327.7\nR001,1998-11-03,8.0,8.3,21051.75\nR001,1998-11-04,8.7,0.0,35800.27\nR001,1998-11-05,12.3,0.0,29692.95\nR001,1998-11-06,10.2,7.3,22891.36\nR001,1998-11-07,8.7,4.3,8332.58\nR001,1998-11-08,5.5,15.3,0.0\nR001,1998-11-09,-0.9,0.0,25680.04\nR001,1998-11-10,-2.5,16.7,18046.09\nR001,1998-11-11,-6.0,0.0,34825.49\nR001,1998-11-12,2.6,0.0,31968.11\nR001,1998-11-13,5.0,0.0,32062.83\nR001,1998-11-14,7.0,17.2,0.0\nR001,1998-11-15,7.3,13.6,8843.11\nR001,1998-11-16,6.2,0.4,25067.37\nR001,1998-11-17,4.4,18.0,20878.03\nR001,1998-11-18,4.8,17.7,9652.49\nR001,1998-11-19,1.5,3.7,22322.4\nR001,1998-11-20,1.3,0.4,28928.64\nR001,1998-11-21,5.7,0.0,31999.71\nR001,1998-11-22,7.0,0.0,29849.94\nR001,1998-11-23,5.6,0.2,29919.71\nR001,1998-11-24,6.4,0.0,31475.65\nR001,1998-11-25,6.1,2.9,23522.09\nR001,1998-11-26,6.1,0.0,28800.0\nR001,1998-11-27,5.6,0.1,25878.94\nR001,1998-11-28,6.4,0.1,12972.03\nR001,1998-11-29,5.2,3.9,13809.95\nR001,1998-11-30,4.8,1.4,1898.74\nR001,1998-12-01,2.7,7.8,11783.12\nR001,1998-12-02,2.1,3.2,16788.44\nR001,1998-12-03,0.1,0.0,29968.77\nR001,1998-12-04,0.3,2.7,28488.25\nR001,1998-12-05,3.7,2.6,20509.58\nR001,1998-12-06,5.9,11.1,12153.95\nR001,1998-12-07,6.3,14.6,0.0\nR001,1998-12-08,5.9,2.1,26985.89\nR001,1998-12-09,5.9,3.6,3600.0\nR001,1998-12-10,5.2,56.6,0.0\nR001,1998-12-11,-1.3,11.4,5379.72\nR001,1998-12-12,1.4,4.9,1300.31\nR001,1998-12-13,1.4,7.4,1568.88\nR001,1998-12-14,-0.5,0.0,29644.74\nR001,1998-12-15,-0.0,2.8,14305.43\nR001,1998-12-16,2.1,1.5,26036.32\nR001,1998-12-17,0.4,2.8,936.03\nR001,1998-12-18,2.3,0.0,25873.53\nR001,1998-12-19,1.3,0.0,30782.75\nR001,1998-12-20,1.7,4.3,14400.0\nR001,1998-12-21,3.5,5.8,15815.15\nR001,1998-12-22,3.5,0.0,29180.35\nR001,1998-12-23,2.4,0.9,23557.12\nR001,1998-12-24,1.7,6.3,4352.73\nR001,1998-12-25,2.3,4.2,20180.12\nR001,1998-12-26,1.7,0.2,27539.97\nR001,1998-12-27,-0.1,0.0,31042.71\nR001,1998-12-28,-1.5,0.0,31401.56\nR001,1998-12-29,1.7,0.0,31447.34\nR001,1998-12-30,2.7,0.3,10926.67\nR001,1998-12-31,0.5,3.9,22072.4\nR001,1999-01-01,-4.8,0.0,31594.47\nR001,1999-01-02,-2.2,0.0,31646.39\nR001,1999-01-03,-1.2,0.0,31699.57\nR001,1999-01-04,-0.9,0.0,28800.0\nR001,1999-01-05,2.0,0.0,31811.09\nR001,1999-01-06,-0.2,0.0,31870.41\nR001,1999-01-07,1.1,0.0,30200.09\nR001,1999-01-08,2.7,0.0,29137.34\nR001,1999-01-09,3.7,0.0,30807.02\nR001,1999-01-10,3.1,10.6,7301.47\nR001,1999-01-11,0.5,12.1,32197.52\nR001,1999-01-12,-3.2,0.0,21874.32\nR001,1999-01-13,1.6,0.0,20752.19\nR001,1999-01-14,4.2,0.0,29692.56\nR001,1999-01-15,4.2,0.0,25766.88\nR001,1999-01-16,2.8,1.5,32400.0\nR001,1999-01-17,0.0,18.0,20594.31\nR001,1999-01-18,-0.5,0.0,32400.0\nR001,1999-01-19,0.5,0.0,32400.0\nR001,1999-01-20,0.3,0.0,32400.0\nR001,1999-01-21,-0.1,0.0,32400.0\nR001,1999-01-22,-0.3,0.0,32400.0\nR001,1999-01-23,-0.4,0.0,32400.0\nR001,1999-01-24,-0.8,3.8,19057.77\nR001,1999-01-25,-2.2,0.0,32113.47\nR001,1999-01-26,-0.9,0.0,32400.0\nR001,1999-01-27,-0.6,0.0,32400.0\nR001,1999-01-28,1.4,0.0,32400.0\nR001,1999-01-29,3.4,0.0,29897.77\nR001,1999-01-30,5.2,0.0,27591.76\nR001,1999-01-31,5.1,0.0,22921.59\nR001,1999-02-01,2.9,3.5,21554.98\nR001,1999-02-02,2.2,26.3,0.0\nR001,1999-02-03,-3.9,18.6,321.29\nR001,1999-02-04,-6.5,0.0,27087.99\nR001,1999-02-05,-2.6,1.3,32037.72\nR001,1999-02-06,-0.1,0.0,33552.76\nR001,1999-02-07,4.2,5.3,24658.58\nR001,1999-02-08,1.5,22.5,11168.17\nR001,1999-02-09,-0.0,3.9,33269.13\nR001,1999-02-10,1.0,4.7,34588.92\nR001,1999-02-11,4.1,0.0,35110.56\nR001,1999-02-12,4.7,0.0,36170.51\nR001,1999-02-13,6.6,0.0,36544.9\nR001,1999-02-14,4.2,0.0,35561.03\nR001,1999-02-15,3.6,0.8,29975.7\nR001,1999-02-16,0.9,13.2,18521.7\nR001,1999-02-17,0.5,0.0,37008.63\nR001,1999-02-18,2.2,9.6,12923.25\nR001,1999-02-19,3.5,4.8,33475.24\nR001,1999-02-20,-1.1,22.0,5624.07\nR001,1999-02-21,-4.1,5.2,23891.75\nR001,1999-02-22,-0.3,14.4,13434.99\nR001,1999-02-23,4.9,1.9,3787.46\nR001,1999-02-24,6.4,1.1,28530.82\nR001,1999-02-25,0.2,23.3,18884.89\nR001,1999-02-26,-0.9,0.0,39104.72\nR001,1999-02-27,-0.1,1.0,26550.5\nR001,1999-02-28,0.5,4.8,29632.18\nR001,1999-03-01,3.4,0.1,39519.35\nR001,1999-03-02,5.7,0.0,39600.0\nR001,1999-03-03,6.5,0.0,39600.0\nR001,1999-03-04,4.2,0.0,39600.0\nR001,1999-03-05,6.3,0.0,37624.02\nR001,1999-03-06,8.7,0.0,39600.0\nR001,1999-03-07,6.6,0.0,39600.0\nR001,1999-03-08,7.0,0.0,39600.0\nR001,1999-03-09,6.4,0.1,36639.47\nR001,1999-03-10,5.8,2.8,7493.06\nR001,1999-03-11,4.1,1.7,1337.54\nR001,1999-03-12,4.5,0.0,33785.7\nR001,1999-03-13,3.6,1.9,39600.0\nR001,1999-03-14,2.8,5.3,27948.82\nR001,1999-03-15,1.5,22.6,0.0\nR001,1999-03-16,-0.4,9.9,0.0\nR001,1999-03-17,-0.5,2.9,32742.44\nR001,1999-03-18,1.1,0.0,37704.75\nR001,1999-03-19,4.5,5.3,520.27\nR001,1999-03-20,5.6,0.1,37926.93\nR001,1999-03-21,3.8,10.7,0.0\nR001,1999-03-22,1.8,3.9,28484.75\nR001,1999-03-23,1.9,6.5,12873.2\nR001,1999-03-24,2.4,0.0,40359.67\nR001,1999-03-25,3.2,0.6,29943.44\nR001,1999-03-26,3.4,4.6,5195.76\nR001,1999-03-27,3.3,0.6,24136.37\nR001,1999-03-28,5.7,0.0,41276.54\nR001,1999-03-29,6.8,0.0,42655.18\nR001,1999-03-30,7.1,0.2,41694.99\nR001,1999-03-31,8.8,0.0,43437.92\nR001,1999-04-01,8.0,7.4,28374.16\nR001,1999-04-02,6.6,1.1,34926.39\nR001,1999-04-03,4.9,3.1,37738.76\nR001,1999-04-04,5.1,2.7,44176.72\nR001,1999-04-05,2.8,8.9,30982.98\nR001,1999-04-06,2.2,0.8,45968.62\nR001,1999-04-07,3.0,0.0,43615.78\nR001,1999-04-08,5.1,0.0,44944.32\nR001,1999-04-09,6.3,0.0,43949.47\nR001,1999-04-10,7.1,0.0,43301.49\nR001,1999-04-11,7.8,0.0,46336.99\nR001,1999-04-12,9.0,2.3,29451.68\nR001,1999-04-13,5.8,16.9,4143.05\nR001,1999-04-14,6.6,0.4,46549.51\nR001,1999-04-15,6.9,0.7,33432.64\nR001,1999-04-16,8.0,5.5,30921.19\nR001,1999-04-17,9.1,0.1,46103.12\nR001,1999-04-18,12.3,0.0,44528.18\nR001,1999-04-19,13.1,1.8,46800.0\nR001,1999-04-20,9.5,0.6,35029.06\nR001,1999-04-21,10.1,1.1,44846.91\nR001,1999-04-22,8.4,9.6,8762.32\nR001,1999-04-23,7.7,8.2,10866.5\nR001,1999-04-24,9.3,0.0,42997.46\nR001,1999-04-25,11.3,0.3,46800.0\nR001,1999-04-26,13.5,0.0,47390.53\nR001,1999-04-27,14.7,0.0,46997.26\nR001,1999-04-28,12.7,4.0,34722.11\nR001,1999-04-29,9.0,20.0,1763.57\nR001,1999-04-30,9.2,9.1,29379.06\nR001,1999-05-01,12.4,0.6,44397.21\nR001,1999-05-02,12.9,3.3,46719.89\nR001,1999-05-03,11.7,6.7,43240.56\nR001,1999-05-04,8.7,2.0,45161.12\nR001,1999-05-05,6.5,10.5,32400.0\nR001,1999-05-06,6.4,18.8,998.48\nR001,1999-05-07,4.1,8.2,23799.18\nR001,1999-05-08,4.5,0.9,47275.33\nR001,1999-05-09,7.1,0.0,50007.31\nR001,1999-05-10,9.9,0.0,50080.48\nR001,1999-05-11,12.3,0.0,47865.02\nR001,1999-05-12,12.9,0.0,50222.75\nR001,1999-05-13,11.2,5.4,18871.4\nR001,1999-05-14,10.7,4.6,37898.0\nR001,1999-05-15,12.5,1.2,44128.51\nR001,1999-05-16,15.1,2.1,49623.03\nR001,1999-05-17,13.1,25.2,27821.57\nR001,1999-05-18,7.1,11.5,44187.73\nR001,1999-05-19,5.4,16.6,24445.16\nR001,1999-05-20,5.4,6.8,28829.1\nR001,1999-05-21,8.5,6.7,46112.03\nR001,1999-05-22,11.2,0.0,48641.9\nR001,1999-05-23,14.6,0.0,50400.0\nR001,1999-05-24,16.7,0.6,50400.0\nR001,1999-05-25,14.6,7.2,34390.82\nR001,1999-05-26,13.3,16.6,26401.8\nR001,1999-05-27,12.6,2.7,41601.54\nR001,1999-05-28,14.0,0.0,50511.38\nR001,1999-05-29,14.3,0.1,48705.4\nR001,1999-05-30,14.3,13.4,10929.19\nR001,1999-05-31,14.1,4.5,32703.9\nR001,1999-06-01,14.4,4.0,45645.94\nR001,1999-06-02,14.6,3.5,44336.62\nR001,1999-06-03,14.9,3.3,47260.9\nR001,1999-06-04,14.9,10.8,44817.03\nR001,1999-06-05,14.5,17.6,40975.53\nR001,1999-06-06,14.1,9.1,49729.13\nR001,1999-06-07,16.6,1.7,48727.98\nR001,1999-06-08,17.2,8.7,38281.17\nR001,1999-06-09,18.6,3.8,47784.35\nR001,1999-06-10,19.4,8.5,50599.19\nR001,1999-06-11,20.3,2.1,49201.34\nR001,1999-06-12,18.3,8.8,27667.59\nR001,1999-06-13,16.9,9.5,19344.72\nR001,1999-06-14,18.1,4.0,43079.05\nR001,1999-06-15,18.5,1.4,53472.44\nR001,1999-06-16,18.3,0.7,51014.08\nR001,1999-06-17,18.1,4.2,45103.52\nR001,1999-06-18,17.9,6.4,50574.7\nR001,1999-06-19,19.4,0.8,52921.75\nR001,1999-06-20,20.6,0.9,53426.83\nR001,1999-06-21,18.1,7.3,50844.96\nR001,1999-06-22,18.1,4.4,48718.27\nR001,1999-06-23,19.9,0.9,50819.71\nR001,1999-06-24,15.7,23.4,35673.77\nR001,1999-06-25,14.9,16.6,33650.47\nR001,1999-06-26,15.3,11.0,39459.12\nR001,1999-06-27,15.2,3.2,28231.12\nR001,1999-06-28,15.9,3.9,47949.81\nR001,1999-06-29,16.4,2.9,34102.92\nR001,1999-06-30,17.2,2.2,47170.76\nR001,1999-07-01,17.8,1.8,50216.01\nR001,1999-07-02,18.9,2.3,53942.74\nR001,1999-07-03,18.5,1.7,53930.21\nR001,1999-07-04,19.0,5.7,53383.47\nR001,1999-07-05,19.2,1.5,49313.56\nR001,1999-07-06,20.6,1.5,49870.67\nR001,1999-07-07,21.5,0.0,52667.19\nR001,1999-07-08,21.7,0.2,51947.56\nR001,1999-07-09,22.0,0.9,50868.72\nR001,1999-07-10,22.5,0.3,53734.99\nR001,1999-07-11,22.1,1.0,53417.73\nR001,1999-07-12,21.4,0.0,50373.97\nR001,1999-07-13,19.1,1.3,52180.83\nR001,1999-07-14,19.1,0.0,53650.67\nR001,1999-07-15,20.4,0.5,53612.85\nR001,1999-07-16,21.2,0.3,52749.34\nR001,1999-07-17,20.7,3.1,52269.78\nR001,1999-07-18,20.4,6.5,51091.8\nR001,1999-07-19,19.6,10.7,41671.06\nR001,1999-07-20,19.0,1.6,42669.99\nR001,1999-07-21,18.4,0.0,49224.96\nR001,1999-07-22,17.9,0.0,49618.2\nR001,1999-07-23,19.0,0.0,47799.58\nR001,1999-07-24,20.2,0.5,49353.75\nR001,1999-07-25,21.7,0.0,51899.83\nR001,1999-07-26,22.5,0.0,51863.55\nR001,1999-07-27,23.7,0.1,51477.64\nR001,1999-07-28,24.4,0.1,51093.32\nR001,1999-07-29,20.6,8.9,22696.5\nR001,1999-07-30,20.8,4.9,49300.0\nR001,1999-07-31,21.9,3.1,49699.95\nR001,1999-08-01,20.9,4.2,48965.55\nR001,1999-08-02,21.1,5.8,44418.74\nR001,1999-08-03,23.0,3.1,46800.0\nR001,1999-08-04,23.0,5.8,43094.15\nR001,1999-08-05,21.4,17.1,31134.73\nR001,1999-08-06,19.5,6.1,44954.73\nR001,1999-08-07,19.0,0.0,47717.66\nR001,1999-08-08,19.9,0.0,47968.3\nR001,1999-08-09,21.5,0.8,47657.11\nR001,1999-08-10,22.0,1.5,47347.52\nR001,1999-08-11,22.0,2.0,46800.0\nR001,1999-08-12,19.9,14.3,39144.9\nR001,1999-08-13,20.9,2.2,46764.53\nR001,1999-08-14,22.2,0.0,46800.0\nR001,1999-08-15,21.5,0.6,47158.92\nR001,1999-08-16,21.4,0.3,46800.0\nR001,1999-08-17,21.9,0.0,46312.33\nR001,1999-08-18,23.1,0.9,46734.39\nR001,1999-08-19,23.9,0.0,45974.76\nR001,1999-08-20,23.8,0.1,46800.0\nR001,1999-08-21,23.3,0.3,46800.0\nR001,1999-08-22,23.1,0.8,46800.0\nR001,1999-08-23,22.0,2.7,46800.0\nR001,1999-08-24,19.5,10.5,37522.87\nR001,1999-08-25,19.2,5.6,29696.68\nR001,1999-08-26,16.9,12.1,38153.53\nR001,1999-08-27,17.3,1.1,46775.85\nR001,1999-08-28,14.6,21.6,12677.58\nR001,1999-08-29,14.6,8.1,34601.15\nR001,1999-08-30,15.9,2.6,38569.64\nR001,1999-08-31,16.9,0.7,44515.03\nR001,1999-09-01,17.7,0.0,43032.46\nR001,1999-09-02,18.7,0.4,44289.37\nR001,1999-09-03,18.7,0.3,44017.73\nR001,1999-09-04,18.3,1.4,42819.17\nR001,1999-09-05,19.4,0.3,44439.4\nR001,1999-09-06,19.2,6.1,33519.55\nR001,1999-09-07,17.2,28.3,13719.02\nR001,1999-09-08,16.7,18.7,14472.84\nR001,1999-09-09,16.5,7.8,37071.18\nR001,1999-09-10,16.5,0.4,40658.78\nR001,1999-09-11,16.0,3.5,39141.55\nR001,1999-09-12,14.7,0.7,39942.17\nR001,1999-09-13,12.9,0.0,42876.25\nR001,1999-09-14,13.0,1.3,36773.17\nR001,1999-09-15,13.0,0.0,41048.09\nR001,1999-09-16,11.1,0.0,41080.99\nR001,1999-09-17,12.6,0.0,42938.32\nR001,1999-09-18,13.3,0.0,42739.07\nR001,1999-09-19,14.3,0.0,39337.35\nR001,1999-09-20,14.7,0.1,39942.1\nR001,1999-09-21,13.1,4.0,32316.04\nR001,1999-09-22,13.2,0.0,41283.92\nR001,1999-09-23,13.4,0.0,33926.37\nR001,1999-09-24,13.4,0.0,40398.36\nR001,1999-09-25,12.8,0.2,39688.41\nR001,1999-09-26,13.2,0.7,38493.01\nR001,1999-09-27,13.3,0.0,40185.64\nR001,1999-09-28,16.2,0.0,41178.77\nR001,1999-09-29,17.7,0.0,40896.07\nR001,1999-09-30,19.6,0.0,40837.92\nR001,1999-10-01,20.2,0.0,40764.48\nR001,1999-10-02,19.7,0.0,40371.63\nR001,1999-10-03,18.5,0.0,39978.96\nR001,1999-10-04,17.8,0.0,38208.58\nR001,1999-10-05,16.6,1.3,33719.11\nR001,1999-10-06,17.4,0.2,38399.2\nR001,1999-10-07,17.0,0.0,38170.62\nR001,1999-10-08,16.3,0.0,38264.7\nR001,1999-10-09,15.0,0.0,36028.73\nR001,1999-10-10,12.2,26.4,0.0\nR001,1999-10-11,9.0,19.6,22728.76\nR001,1999-10-12,5.4,3.8,33228.18\nR001,1999-10-13,8.3,3.1,22125.03\nR001,1999-10-14,8.4,4.4,31809.06\nR001,1999-10-15,7.6,25.9,0.0\nR001,1999-10-16,6.9,19.2,28692.35\nR001,1999-10-17,7.5,8.3,25076.53\nR001,1999-10-18,5.7,9.7,33886.66\nR001,1999-10-19,3.9,0.0,34631.86\nR001,1999-10-20,7.0,3.0,22240.97\nR001,1999-10-21,9.0,8.1,17812.67\nR001,1999-10-22,4.3,7.0,32400.0\nR001,1999-10-23,2.4,0.0,36000.0\nR001,1999-10-24,4.1,0.0,36000.0\nR001,1999-10-25,5.8,0.0,36000.0\nR001,1999-10-26,5.9,0.0,34324.89\nR001,1999-10-27,7.4,0.0,34464.48\nR001,1999-10-28,7.7,26.6,19378.07\nR001,1999-10-29,3.8,7.3,34506.09\nR001,1999-10-30,2.9,4.0,4542.18\nR001,1999-10-31,1.4,2.5,35101.13\nR001,1999-11-01,3.5,0.8,25859.63\nR001,1999-11-02,3.2,0.0,35982.01\nR001,1999-11-03,5.8,0.0,35902.65\nR001,1999-11-04,8.0,0.0,35822.82\nR001,1999-11-05,5.4,0.0,35742.62\nR001,1999-11-06,4.5,0.0,35614.35\nR001,1999-11-07,5.8,0.0,35207.61\nR001,1999-11-08,5.0,0.2,33399.61\nR001,1999-11-09,0.3,4.8,22727.45\nR001,1999-11-10,-0.4,0.0,35001.16\nR001,1999-11-11,0.7,0.0,34852.05\nR001,1999-11-12,1.5,2.1,29669.46\nR001,1999-11-13,2.5,0.0,18323.96\nR001,1999-11-14,4.8,0.0,34091.81\nR001,1999-11-15,7.1,1.2,32219.8\nR001,1999-11-16,3.8,39.5,16.29\nR001,1999-11-17,-1.3,5.6,29500.1\nR001,1999-11-18,5.6,14.8,11012.52\nR001,1999-11-19,6.0,25.0,14516.85\nR001,1999-11-20,3.3,0.0,29917.68\nR001,1999-11-21,3.6,10.9,0.0\nR001,1999-11-22,5.5,0.0,26792.89\nR001,1999-11-23,5.5,0.4,27728.69\nR001,1999-11-24,5.2,20.9,514.74\nR001,1999-11-25,0.3,29.5,18390.74\nR001,1999-11-26,-4.3,11.9,13961.85\nR001,1999-11-27,-6.5,8.2,14400.0\nR001,1999-11-28,-3.9,5.0,26836.41\nR001,1999-11-29,-1.9,5.9,521.65\nR001,1999-11-30,-0.8,0.4,29538.74\nR001,1999-12-01,-0.8,0.0,29555.52\nR001,1999-12-02,2.3,0.0,30711.67\nR001,1999-12-03,2.1,1.2,28800.0\nR001,1999-12-04,-0.7,12.4,30361.89\nR001,1999-12-05,-0.4,0.0,30757.81\nR001,1999-12-06,3.8,0.0,30386.61\nR001,1999-12-07,3.8,4.1,27408.91\nR001,1999-12-08,2.8,1.2,22813.49\nR001,1999-12-09,2.7,3.7,11203.0\nR001,1999-12-10,1.4,0.0,30417.66\nR001,1999-12-11,3.6,0.0,22171.64\nR001,1999-12-12,2.7,0.0,30423.2\nR001,1999-12-13,1.8,0.0,29259.26\nR001,1999-12-14,2.3,0.0,30422.05\nR001,1999-12-15,0.9,8.7,0.0\nR001,1999-12-16,0.7,1.0,19500.9\nR001,1999-12-17,6.9,0.1,28848.75\nR001,1999-12-18,5.7,0.0,29619.99\nR001,1999-12-19,3.4,0.3,21474.2\nR001,1999-12-20,3.8,0.0,30380.33\nR001,1999-12-21,7.3,0.0,27743.59\nR001,1999-12-22,3.9,0.0,30354.47\nR001,1999-12-23,2.8,0.0,29948.17\nR001,1999-12-24,3.7,0.0,27114.49\nR001,1999-12-25,2.7,0.0,16696.6\nR001,1999-12-26,1.1,8.2,0.0\nR001,1999-12-27,2.0,8.6,0.0\nR001,1999-12-28,3.8,31.1,10594.36\nR001,1999-12-29,7.8,22.1,0.0\nR001,1999-12-30,5.1,19.5,0.0\nR001,1999-12-31,4.5,0.0,6398.06\nR001,2000-01-01,6.2,0.0,11855.68\nR001,2000-01-02,2.3,1.2,8165.53\nR001,2000-01-03,1.9,20.0,0.0\nR001,2000-01-04,0.3,6.1,11305.47\nR001,2000-01-05,-0.1,1.4,1700.6\nR001,2000-01-06,0.4,0.0,19771.76\nR001,2000-01-07,-1.0,0.0,30556.74\nR001,2000-01-08,-1.8,0.0,31986.23\nR001,2000-01-09,-1.4,0.0,32052.5\nR001,2000-01-10,-1.9,0.0,32120.15\nR001,2000-01-11,-2.3,3.3,26324.2\nR001,2000-01-12,-1.5,8.6,16983.75\nR001,2000-01-13,-3.2,0.5,32329.98\nR001,2000-01-14,-4.0,4.2,32076.27\nR001,2000-01-15,-3.9,2.9,32400.0\nR001,2000-01-16,-3.8,4.7,31911.78\nR001,2000-01-17,-4.1,0.6,29223.18\nR001,2000-01-18,-2.9,15.6,0.0\nR001,2000-01-19,-3.1,25.1,3600.0\nR001,2000-01-20,-5.1,11.3,18000.0\nR001,2000-01-21,-8.5,1.5,31451.62\nR001,2000-01-22,-7.5,23.7,8302.45\nR001,2000-01-23,-10.5,0.4,29181.68\nR001,2000-01-24,-5.6,0.0,32400.0\nR001,2000-01-25,-0.7,34.8,0.0\nR001,2000-01-26,-6.2,4.1,30167.32\nR001,2000-01-27,-10.0,0.0,32400.0\nR001,2000-01-28,-6.7,1.7,32400.0\nR001,2000-01-29,-4.7,0.5,32523.64\nR001,2000-01-30,-2.4,0.0,32480.01\nR001,2000-01-31,-0.8,18.5,13258.56\nR001,2000-02-01,-1.7,13.2,8500.5\nR001,2000-02-02,-3.3,0.0,32699.52\nR001,2000-02-03,0.5,0.0,32400.0\nR001,2000-02-04,0.1,25.5,0.0\nR001,2000-02-05,-5.1,0.1,33609.81\nR001,2000-02-06,-5.4,4.1,25200.0\nR001,2000-02-07,-5.4,0.3,33448.63\nR001,2000-02-08,-1.3,0.0,33363.38\nR001,2000-02-09,0.8,0.0,34112.46\nR001,2000-02-10,3.1,0.0,35406.71\nR001,2000-02-11,1.7,0.6,35496.51\nR001,2000-02-12,0.7,2.1,28454.48\nR001,2000-02-13,-0.1,0.0,34600.72\nR001,2000-02-14,-0.7,0.2,35207.97\nR001,2000-02-15,-0.6,0.0,32182.32\nR001,2000-02-16,1.5,0.6,26253.15\nR001,2000-02-17,2.4,0.0,29683.55\nR001,2000-02-18,2.8,0.0,36694.01\nR001,2000-02-19,4.5,0.0,35386.68\nR001,2000-02-20,0.1,6.7,27625.0\nR001,2000-02-21,2.5,0.5,24361.19\nR001,2000-02-22,2.1,14.4,7440.72\nR001,2000-02-23,-1.7,18.8,1058.88\nR001,2000-02-24,-3.8,0.2,36861.61\nR001,2000-02-25,-4.6,3.2,22938.85\nR001,2000-02-26,-3.6,0.9,20764.58\nR001,2000-02-27,-4.3,16.9,2488.69\nR001,2000-02-28,-5.3,0.4,39387.63\nR001,2000-02-29,-4.7,0.0,39485.65\nR001,2000-03-01,-3.0,0.0,39585.17\nR001,2000-03-02,-1.9,0.0,39600.0\nR001,2000-03-03,-0.5,0.0,39600.0\nR001,2000-03-04,4.6,0.0,33826.1\nR001,2000-03-05,4.2,0.0,36000.0\nR001,2000-03-06,1.9,19.6,0.0\nR001,2000-03-07,-2.2,6.1,20527.58\nR001,2000-03-08,-2.9,0.0,39600.0\nR001,2000-03-09,0.7,4.1,25200.0\nR001,2000-03-10,1.7,29.4,0.0\nR001,2000-03-11,-1.9,25.9,21663.46\nR001,2000-03-12,-3.4,3.9,30623.51\nR001,2000-03-13,-3.1,19.3,22054.49\nR001,2000-03-14,-6.6,0.0,39600.0\nR001,2000-03-15,-3.9,0.0,39837.67\nR001,2000-03-16,2.0,0.0,39600.0\nR001,2000-03-17,3.8,3.9,22857.39\nR001,2000-03-18,5.5,3.2,25005.49\nR001,2000-03-19,0.5,12.5,37369.14\nR001,2000-03-20,1.1,21.7,0.0\nR001,2000-03-21,-0.2,4.6,32400.0\nR001,2000-03-22,1.2,0.2,39840.64\nR001,2000-03-23,0.8,0.0,39379.63\nR001,2000-03-24,2.1,0.0,40295.18\nR001,2000-03-25,-0.5,6.4,24348.83\nR001,2000-03-26,0.7,0.0,41635.36\nR001,2000-03-27,4.9,0.4,40342.99\nR001,2000-03-28,6.5,0.5,29669.48\nR001,2000-03-29,6.4,0.4,36436.97\nR001,2000-03-30,5.4,0.0,40979.47\nR001,2000-03-31,4.6,0.0,45092.77\nR001,2000-04-01,7.6,0.0,43574.09\nR001,2000-04-02,11.1,0.0,42665.85\nR001,2000-04-03,11.1,0.2,44678.76\nR001,2000-04-04,9.9,0.0,45892.42\nR001,2000-04-05,11.2,0.1,45963.69\nR001,2000-04-06,9.4,3.0,35144.88\nR001,2000-04-07,6.9,17.5,27902.66\nR001,2000-04-08,5.9,3.3,42441.27\nR001,2000-04-09,3.8,14.6,3093.71\nR001,2000-04-10,1.1,6.9,14797.32\nR001,2000-04-11,2.4,0.4,46224.32\nR001,2000-04-12,4.9,0.8,31178.85\nR001,2000-04-13,9.2,0.0,46532.9\nR001,2000-04-14,9.2,3.8,29482.06\nR001,2000-04-15,8.0,2.4,0.0\nR001,2000-04-16,8.3,0.0,41877.64\nR001,2000-04-17,8.6,0.0,46800.0\nR001,2000-04-18,11.2,0.0,46800.0\nR001,2000-04-19,12.5,0.1,46800.0\nR001,2000-04-20,9.7,4.4,43702.87\nR001,2000-04-21,9.7,1.7,44217.09\nR001,2000-04-22,9.6,0.5,46800.0\nR001,2000-04-23,9.8,1.4,43107.19\nR001,2000-04-24,10.9,0.3,46517.46\nR001,2000-04-25,11.8,1.3,47067.76\nR001,2000-04-26,13.0,2.0,47357.48\nR001,2000-04-27,14.0,0.9,47316.69\nR001,2000-04-28,15.0,0.7,41924.94\nR001,2000-04-29,15.0,0.4,45565.09\nR001,2000-04-30,12.8,2.2,36641.47\nR001,2000-05-01,9.9,10.6,41490.65\nR001,2000-05-02,9.1,5.3,34665.37\nR001,2000-05-03,6.2,12.6,10438.73\nR001,2000-05-04,4.8,0.0,44864.3\nR001,2000-05-05,8.6,0.0,47179.98\nR001,2000-05-06,10.5,15.3,23964.47\nR001,2000-05-07,8.9,4.4,36303.43\nR001,2000-05-08,5.4,18.6,10079.47\nR001,2000-05-09,4.2,5.6,42096.29\nR001,2000-05-10,6.3,0.1,38761.8\nR001,2000-05-11,9.9,0.1,49214.31\nR001,2000-05-12,10.6,2.3,47273.75\nR001,2000-05-13,10.7,10.7,27279.41\nR001,2000-05-14,10.4,0.0,43190.62\nR001,2000-05-15,8.1,0.0,50400.0\nR001,2000-05-16,8.9,0.2,48542.71\nR001,2000-05-17,8.9,0.3,48999.64\nR001,2000-05-18,11.0,1.1,47744.13\nR001,2000-05-19,12.0,1.2,45136.7\nR001,2000-05-20,12.4,0.0,49068.71\nR001,2000-05-21,15.2,0.0,50659.41\nR001,2000-05-22,16.8,0.2,50632.72\nR001,2000-05-23,17.2,0.0,50962.95\nR001,2000-05-24,17.7,0.3,50937.54\nR001,2000-05-25,17.3,9.3,49579.66\nR001,2000-05-26,16.7,1.8,47355.56\nR001,2000-05-27,15.3,3.8,11194.44\nR001,2000-05-28,15.3,9.4,44911.42\nR001,2000-05-29,15.9,7.5,49764.4\nR001,2000-05-30,16.5,2.9,50794.0\nR001,2000-05-31,15.8,16.0,45691.64\nR001,2000-06-01,15.6,6.8,40848.3\nR001,2000-06-02,17.6,3.6,50729.85\nR001,2000-06-03,18.4,7.0,39796.25\nR001,2000-06-04,16.2,19.7,34383.49\nR001,2000-06-05,16.0,13.0,36522.68\nR001,2000-06-06,17.9,0.2,48436.97\nR001,2000-06-07,19.0,9.7,49502.6\nR001,2000-06-08,16.7,20.4,40044.3\nR001,2000-06-09,16.5,11.2,41344.11\nR001,2000-06-10,14.5,0.0,51217.64\nR001,2000-06-11,13.6,0.0,52776.45\nR001,2000-06-12,13.8,0.0,53497.75\nR001,2000-06-13,15.5,0.0,52751.89\nR001,2000-06-14,15.1,3.4,37931.88\nR001,2000-06-15,16.0,0.0,53464.23\nR001,2000-06-16,18.2,0.0,53821.65\nR001,2000-06-17,19.6,0.0,53444.39\nR001,2000-06-18,16.7,34.5,37326.77\nR001,2000-06-19,10.6,10.5,49030.84\nR001,2000-06-20,9.5,5.8,37368.36\nR001,2000-06-21,10.5,0.0,51311.83\nR001,2000-06-22,12.3,0.0,32499.94\nR001,2000-06-23,13.8,0.6,53767.43\nR001,2000-06-24,14.3,0.1,50914.42\nR001,2000-06-25,14.0,0.0,52289.27\nR001,2000-06-26,16.7,0.0,52621.7\nR001,2000-06-27,18.2,0.0,52678.25\nR001,2000-06-28,18.7,0.3,53963.25\nR001,2000-06-29,18.6,1.2,48352.14\nR001,2000-06-30,20.4,1.7,49782.28\nR001,2000-07-01,20.0,4.1,49319.48\nR001,2000-07-02,18.5,5.6,26162.2\nR001,2000-07-03,18.7,1.3,47939.56\nR001,2000-07-04,19.7,0.0,52394.68\nR001,2000-07-05,20.1,0.0,51934.18\nR001,2000-07-06,21.4,0.5,53861.38\nR001,2000-07-07,20.2,2.5,53839.2\nR001,2000-07-08,16.8,0.0,52617.68\nR001,2000-07-09,18.9,0.0,53173.74\nR001,2000-07-10,21.9,1.5,53759.98\nR001,2000-07-11,20.1,0.0,51980.46\nR001,2000-07-12,20.0,0.0,52446.38\nR001,2000-07-13,21.5,0.0,51615.19\nR001,2000-07-14,23.1,0.1,53450.28\nR001,2000-07-15,22.4,0.4,44015.39\nR001,2000-07-16,23.4,0.5,52758.4\nR001,2000-07-17,23.3,0.7,53129.18\nR001,2000-07-18,20.0,11.0,7835.91\nR001,2000-07-19,19.5,6.4,26920.72\nR001,2000-07-20,22.0,2.0,45330.3\nR001,2000-07-21,22.8,2.1,43869.79\nR001,2000-07-22,24.3,1.0,48945.98\nR001,2000-07-23,23.8,2.2,51099.08\nR001,2000-07-24,19.5,15.7,2417.93\nR001,2000-07-25,20.2,0.0,45482.01\nR001,2000-07-26,21.4,1.0,49833.26\nR001,2000-07-27,23.3,0.2,50928.66\nR001,2000-07-28,22.8,1.1,50975.77\nR001,2000-07-29,23.2,0.0,49637.6\nR001,2000-07-30,26.8,0.0,50996.64\nR001,2000-07-31,27.2,0.0,50270.6\nR001,2000-08-01,23.3,4.5,45992.07\nR001,2000-08-02,20.8,21.8,9688.26\nR001,2000-08-03,16.6,11.7,19427.31\nR001,2000-08-04,19.9,0.0,46229.94\nR001,2000-08-05,20.3,0.0,48316.83\nR001,2000-08-06,21.5,0.3,47933.81\nR001,2000-08-07,21.7,1.3,47620.98\nR001,2000-08-08,22.9,0.3,47648.7\nR001,2000-08-09,23.1,0.0,48014.71\nR001,2000-08-10,21.6,1.2,47030.38\nR001,2000-08-11,22.2,1.4,45082.17\nR001,2000-08-12,23.1,1.1,47090.41\nR001,2000-08-13,21.0,4.6,46800.0\nR001,2000-08-14,20.0,5.2,29195.62\nR001,2000-08-15,19.5,0.6,46800.0\nR001,2000-08-16,19.2,0.5,46879.42\nR001,2000-08-17,20.4,1.1,46800.0\nR001,2000-08-18,19.8,3.3,45504.01\nR001,2000-08-19,20.2,1.0,45785.19\nR001,2000-08-20,21.5,0.4,46800.0\nR001,2000-08-21,20.8,0.0,46800.0\nR001,2000-08-22,21.3,0.1,46800.0\nR001,2000-08-23,21.1,2.3,46800.0\nR001,2000-08-24,20.6,1.1,46800.0\nR001,2000-08-25,18.2,2.5,39452.54\nR001,2000-08-26,18.2,0.9,44707.05\nR001,2000-08-27,15.9,17.6,21367.87\nR001,2000-08-28,14.1,3.1,35308.12\nR001,2000-08-29,13.2,6.2,16684.55\nR001,2000-08-30,14.1,1.2,33998.58\nR001,2000-08-31,14.0,1.7,34199.31\nR001,2000-09-01,13.4,2.4,40022.02\nR001,2000-09-02,15.4,0.0,45612.08\nR001,2000-09-03,16.8,0.8,42307.13\nR001,2000-09-04,17.2,0.0,43788.97\nR001,2000-09-05,18.0,0.0,43031.03\nR001,2000-09-06,19.4,0.2,43893.64\nR001,2000-09-07,17.5,6.0,30590.27\nR001,2000-09-08,15.7,3.0,40546.52\nR001,2000-09-09,15.2,0.0,43414.59\nR001,2000-09-10,15.3,0.0,41317.2\nR001,2000-09-11,15.6,0.0,43203.4\nR001,2000-09-12,16.3,0.4,43200.0\nR001,2000-09-13,16.0,5.9,35737.62\nR001,2000-09-14,16.1,0.1,38215.99\nR001,2000-09-15,14.9,7.9,39728.41\nR001,2000-09-16,14.5,2.3,37139.35\nR001,2000-09-17,14.6,0.0,42730.34\nR001,2000-09-18,15.2,0.0,41172.43\nR001,2000-09-19,16.4,0.0,41842.88\nR001,2000-09-20,17.2,1.2,41558.84\nR001,2000-09-21,16.2,0.6,41275.06\nR001,2000-09-22,16.1,1.1,36873.35\nR001,2000-09-23,15.1,0.0,41320.36\nR001,2000-09-24,17.3,0.0,41380.52\nR001,2000-09-25,17.1,0.2,40800.26\nR001,2000-09-26,13.1,28.7,18864.06\nR001,2000-09-27,10.4,0.6,40148.42\nR001,2000-09-28,10.9,0.5,37669.27\nR001,2000-09-29,11.9,0.0,32158.34\nR001,2000-09-30,12.9,2.8,24899.93\nR001,2000-10-01,12.5,5.7,35190.49\nR001,2000-10-02,12.2,0.0,39199.93\nR001,2000-10-03,13.3,0.0,39604.38\nR001,2000-10-04,13.2,0.0,39535.86\nR001,2000-10-05,13.6,0.0,38430.57\nR001,2000-10-06,14.5,0.0,38142.39\nR001,2000-10-07,14.8,0.0,31585.21\nR001,2000-10-08,13.7,1.9,25682.93\nR001,2000-10-09,12.3,4.4,19878.66\nR001,2000-10-10,12.8,0.1,35942.64\nR001,2000-10-11,13.5,5.4,30167.17\nR001,2000-10-12,11.6,4.1,25865.51\nR001,2000-10-13,10.6,0.7,33910.76\nR001,2000-10-14,11.1,0.0,36335.64\nR001,2000-10-15,10.7,0.0,36061.82\nR001,2000-10-16,10.5,0.0,33778.63\nR001,2000-10-17,8.4,0.0,36000.0\nR001,2000-10-18,5.7,7.7,17217.28\nR001,2000-10-19,4.0,2.7,29852.26\nR001,2000-10-20,4.3,0.0,36000.0\nR001,2000-10-21,4.9,0.3,148.04\nR001,2000-10-22,7.2,0.3,36000.0\nR001,2000-10-23,7.7,1.1,12398.29\nR001,2000-10-24,8.4,1.3,31892.27\nR001,2000-10-25,7.9,0.2,35665.65\nR001,2000-10-26,8.1,0.1,33583.95\nR001,2000-10-27,8.3,0.0,36000.0\nR001,2000-10-28,9.1,0.0,36000.0\nR001,2000-10-29,7.7,4.0,10336.01\nR001,2000-10-30,6.9,0.0,30065.66\nR001,2000-10-31,6.8,0.0,36000.0\nR001,2000-11-01,8.1,0.0,35999.46\nR001,2000-11-02,8.0,2.8,35919.29\nR001,2000-11-03,7.5,0.0,35838.93\nR001,2000-11-04,7.8,0.0,32603.42\nR001,2000-11-05,7.6,0.0,35643.12\nR001,2000-11-06,7.7,0.0,35486.82\nR001,2000-11-07,9.0,0.0,33337.1\nR001,2000-11-08,8.9,0.0,34443.84\nR001,2000-11-09,7.5,0.0,35029.89\nR001,2000-11-10,8.7,0.0,31122.74\nR001,2000-11-11,9.3,0.1,26702.54\nR001,2000-11-12,5.8,16.0,23289.58\nR001,2000-11-13,1.5,0.0,33011.58\nR001,2000-11-14,2.1,0.0,34319.85\nR001,2000-11-15,4.7,0.0,33709.99\nR001,2000-11-16,5.2,0.0,34052.84\nR001,2000-11-17,5.6,0.0,33696.22\nR001,2000-11-18,5.2,0.0,33692.14\nR001,2000-11-19,5.4,0.0,33324.32\nR001,2000-11-20,6.1,0.0,31950.81\nR001,2000-11-21,5.4,0.0,32030.09\nR001,2000-11-22,4.0,0.0,27739.76\nR001,2000-11-23,3.5,10.9,0.0\nR001,2000-11-24,2.1,5.7,14975.64\nR001,2000-11-25,2.2,0.0,25002.56\nR001,2000-11-26,2.3,0.1,30227.93\nR001,2000-11-27,2.6,0.0,27195.41\nR001,2000-11-28,4.5,0.0,28800.0\nR001,2000-11-29,4.9,0.0,22457.87\nR001,2000-11-30,5.7,0.0,29933.47\nR001,2000-12-01,4.4,0.0,29190.08\nR001,2000-12-02,4.9,0.0,20576.3\nR001,2000-12-03,4.6,1.5,29491.14\nR001,2000-12-04,4.2,0.0,29991.3\nR001,2000-12-05,4.0,0.0,30385.19\nR001,2000-12-06,2.6,0.0,29626.83\nR001,2000-12-07,0.7,0.6,28864.04\nR001,2000-12-08,0.6,1.6,29638.87\nR001,2000-12-09,0.7,0.0,30414.86\nR001,2000-12-10,1.3,0.0,30418.42\nR001,2000-12-11,2.3,0.0,25056.86\nR001,2000-12-12,1.2,0.0,27944.32\nR001,2000-12-13,1.0,1.4,27356.55\nR001,2000-12-14,0.7,0.6,29250.64\nR001,2000-12-15,2.1,2.0,14878.39\nR001,2000-12-16,3.1,0.0,10353.76\nR001,2000-12-17,3.3,20.8,5271.48\nR001,2000-12-18,-0.4,0.5,19740.11\nR001,2000-12-19,-0.5,0.0,21629.88\nR001,2000-12-20,-2.5,0.0,12542.22\nR001,2000-12-21,-5.1,0.5,26824.81\nR001,2000-12-22,-8.1,3.7,29413.96\nR001,2000-12-23,-9.5,0.0,31218.38\nR001,2000-12-24,-6.6,0.0,18884.82\nR001,2000-12-25,-1.0,0.0,25381.73\nR001,2000-12-26,-0.3,0.0,23117.12\nR001,2000-12-27,2.4,0.2,29259.32\nR001,2000-12-28,5.2,0.0,21644.4\nR001,2000-12-29,5.9,0.0,27343.76\nR001,2000-12-30,8.1,0.0,31511.25\nR001,2000-12-31,4.6,4.0,10877.91\nR001,2001-01-01,4.4,1.4,30700.08\nR001,2001-01-02,2.5,0.0,14617.65\nR001,2001-01-03,1.6,0.0,31730.28\nR001,2001-01-04,1.3,0.0,31789.57\nR001,2001-01-05,3.0,0.0,28996.8\nR001,2001-01-06,2.8,0.0,31912.68\nR001,2001-01-07,2.3,0.0,23913.48\nR001,2001-01-08,3.1,0.0,32040.86\nR001,2001-01-09,5.8,0.0,32106.52\nR001,2001-01-10,5.3,4.3,18503.49\nR001,2001-01-11,0.7,2.3,28640.3\nR001,2001-01-12,0.5,13.3,3671.68\nR001,2001-01-13,-1.6,19.8,10877.01\nR001,2001-01-14,-5.0,1.6,12169.7\nR001,2001-01-15,-10.4,0.0,32400.0\nR001,2001-01-16,-7.3,0.0,32400.0\nR001,2001-01-17,-5.8,0.0,32400.0\nR001,2001-01-18,-4.7,0.0,32400.0\nR001,2001-01-19,-4.8,1.1,32400.0\nR001,2001-01-20,-2.0,0.4,30700.18\nR001,2001-01-21,-2.5,0.0,30131.13\nR001,2001-01-22,-4.4,2.2,30085.53\nR001,2001-01-23,-5.9,0.0,29697.99\nR001,2001-01-24,-7.8,0.0,32400.0\nR001,2001-01-25,-7.6,0.0,32400.0\nR001,2001-01-26,-6.5,0.0,32400.0\nR001,2001-01-27,-3.1,0.0,32400.0\nR001,2001-01-28,1.2,0.0,30939.11\nR001,2001-01-29,3.5,0.0,23843.43\nR001,2001-01-30,5.3,0.0,32445.6\nR001,2001-01-31,5.3,0.0,26448.13\nR001,2001-02-01,6.5,0.0,29450.76\nR001,2001-02-02,2.5,5.5,22954.63\nR001,2001-02-03,1.4,0.9,28800.0\nR001,2001-02-04,-0.2,7.6,15828.27\nR001,2001-02-05,-2.1,0.2,27033.44\nR001,2001-02-06,1.1,9.7,0.0\nR001,2001-02-07,0.8,6.2,24630.88\nR001,2001-02-08,-1.0,0.0,34333.05\nR001,2001-02-09,1.4,0.0,35394.76\nR001,2001-02-10,3.4,0.0,35774.74\nR001,2001-02-11,3.3,0.0,36152.2\nR001,2001-02-12,1.4,0.0,36344.76\nR001,2001-02-13,0.7,0.0,36221.11\nR001,2001-02-14,1.4,0.3,32916.34\nR001,2001-02-15,-0.8,0.6,26277.44\nR001,2001-02-16,-1.2,0.1,34594.74\nR001,2001-02-17,-2.1,0.0,30258.86\nR001,2001-02-18,-2.1,15.6,0.0\nR001,2001-02-19,-1.0,4.4,25181.22\nR001,2001-02-20,-2.6,20.3,0.0\nR001,2001-02-21,-7.7,1.7,36470.2\nR001,2001-02-22,-5.7,22.4,1187.83\nR001,2001-02-23,-3.1,30.7,0.0\nR001,2001-02-24,-1.0,13.0,19200.35\nR001,2001-02-25,2.8,42.7,0.0\nR001,2001-02-26,4.0,18.4,8568.22\nR001,2001-02-27,6.4,2.7,36625.42\nR001,2001-02-28,3.5,11.2,19272.36\nR001,2001-03-01,4.8,1.0,20882.38\nR001,2001-03-02,5.2,13.5,9916.8\nR001,2001-03-03,1.6,5.8,24767.38\nR001,2001-03-04,6.5,5.1,22143.73\nR001,2001-03-05,8.8,0.0,39373.48\nR001,2001-03-06,7.6,4.4,33468.11\nR001,2001-03-07,3.9,34.8,0.0\nR001,2001-03-08,3.3,1.3,32890.99\nR001,2001-03-09,1.5,0.0,39600.0\nR001,2001-03-10,2.9,0.0,39600.0\nR001,2001-03-11,6.3,0.0,39600.0\nR001,2001-03-12,7.4,0.0,38357.24\nR001,2001-03-13,7.3,0.2,34703.45\nR001,2001-03-14,5.8,6.0,32400.0\nR001,2001-03-15,6.1,6.2,39600.0\nR001,2001-03-16,3.0,11.2,34594.07\nR001,2001-03-17,2.2,3.6,30657.04\nR001,2001-03-18,6.2,0.0,40545.7\nR001,2001-03-19,8.5,2.9,39600.0\nR001,2001-03-20,7.6,6.4,11228.16\nR001,2001-03-21,5.4,17.2,29015.37\nR001,2001-03-22,5.3,0.0,36879.12\nR001,2001-03-23,7.5,4.6,18362.01\nR001,2001-03-24,4.0,24.1,0.0\nR001,2001-03-25,2.6,0.0,38849.71\nR001,2001-03-26,7.2,1.7,17958.44\nR001,2001-03-27,9.0,3.1,42335.65\nR001,2001-03-28,9.5,1.4,41544.35\nR001,2001-03-29,3.9,10.0,29900.83\nR001,2001-03-30,3.7,0.0,36841.85\nR001,2001-03-31,5.7,0.0,39998.57\nR001,2001-04-01,10.4,0.6,30815.34\nR001,2001-04-02,10.6,17.1,19281.22\nR001,2001-04-03,7.7,2.8,26286.72\nR001,2001-04-04,9.8,0.2,41579.88\nR001,2001-04-05,11.1,4.5,30893.21\nR001,2001-04-06,6.7,19.3,6491.07\nR001,2001-04-07,9.3,0.0,45280.9\nR001,2001-04-08,9.8,0.9,43110.39\nR001,2001-04-09,9.7,0.6,45538.79\nR001,2001-04-10,10.4,0.1,46301.66\nR001,2001-04-11,7.9,17.5,21778.06\nR001,2001-04-12,5.8,8.8,40346.21\nR001,2001-04-13,7.1,0.2,46515.86\nR001,2001-04-14,10.9,0.0,38773.79\nR001,2001-04-15,9.0,1.7,15418.07\nR001,2001-04-16,8.8,1.6,46300.68\nR001,2001-04-17,5.9,9.1,43415.58\nR001,2001-04-18,8.9,0.3,39600.0\nR001,2001-04-19,8.7,19.9,16897.41\nR001,2001-04-20,8.7,0.0,46017.15\nR001,2001-04-21,9.7,0.0,46926.04\nR001,2001-04-22,13.2,0.0,45464.84\nR001,2001-04-23,14.9,2.1,22894.17\nR001,2001-04-24,10.3,23.6,119.87\nR001,2001-04-25,8.6,17.2,2715.61\nR001,2001-04-26,6.9,4.2,39989.27\nR001,2001-04-27,6.8,4.8,36792.97\nR001,2001-04-28,9.0,0.0,46800.0\nR001,2001-04-29,10.3,0.0,43725.17\nR001,2001-04-30,11.1,2.6,48212.84\nR001,2001-05-01,10.5,2.1,46834.4\nR001,2001-05-02,10.0,2.4,15068.05\nR001,2001-05-03,10.8,4.2,23216.98\nR001,2001-05-04,10.7,11.4,15939.89\nR001,2001-05-05,10.7,5.5,29247.79\nR001,2001-05-06,11.3,4.6,45654.23\nR001,2001-05-07,12.7,0.5,48308.47\nR001,2001-05-08,12.3,3.0,47254.08\nR001,2001-05-09,10.4,2.5,34682.86\nR001,2001-05-10,10.5,4.9,31099.47\nR001,2001-05-11,9.0,9.5,7912.32\nR001,2001-05-12,10.4,15.9,42701.5\nR001,2001-05-13,11.8,5.0,43424.08\nR001,2001-05-14,12.5,16.8,25307.55\nR001,2001-05-15,9.9,20.5,82.87\nR001,2001-05-16,9.7,8.5,32229.02\nR001,2001-05-17,9.6,4.9,40775.35\nR001,2001-05-18,11.5,1.1,47593.62\nR001,2001-05-19,11.5,7.8,33244.65\nR001,2001-05-20,12.9,3.6,36692.51\nR001,2001-05-21,10.8,15.2,27904.16\nR001,2001-05-22,11.5,0.0,46055.22\nR001,2001-05-23,11.2,6.1,11378.34\nR001,2001-05-24,7.9,23.0,38885.93\nR001,2001-05-25,9.2,0.0,48164.77\nR001,2001-05-26,13.7,0.0,50400.0\nR001,2001-05-27,14.0,2.1,42407.42\nR001,2001-05-28,12.5,4.2,40603.27\nR001,2001-05-29,10.3,8.3,37224.97\nR001,2001-05-30,10.0,25.9,27242.53\nR001,2001-05-31,8.9,12.5,32191.66\nR001,2001-06-01,11.3,2.1,30000.0\nR001,2001-06-02,19.1,1.0,51100.69\nR001,2001-06-03,15.7,7.8,29475.87\nR001,2001-06-04,18.2,0.0,52518.6\nR001,2001-06-05,15.6,8.4,39064.93\nR001,2001-06-06,10.4,8.7,36626.68\nR001,2001-06-07,10.5,24.2,33061.54\nR001,2001-06-08,11.7,0.0,53186.07\nR001,2001-06-09,12.4,0.0,51342.08\nR001,2001-06-10,14.1,0.2,48618.19\nR001,2001-06-11,16.7,0.0,51391.17\nR001,2001-06-12,19.7,0.0,53132.08\nR001,2001-06-13,21.0,0.7,52385.73\nR001,2001-06-14,19.6,0.2,48371.9\nR001,2001-06-15,22.3,0.1,53464.67\nR001,2001-06-16,17.7,14.0,12202.99\nR001,2001-06-17,15.9,29.8,6709.13\nR001,2001-06-18,16.4,13.4,24818.15\nR001,2001-06-19,17.4,4.8,46555.55\nR001,2001-06-20,19.6,0.0,52386.87\nR001,2001-06-21,23.4,1.6,53046.5\nR001,2001-06-22,19.1,14.0,50382.76\nR001,2001-06-23,17.7,0.2,50351.23\nR001,2001-06-24,17.5,0.0,53957.19\nR001,2001-06-25,17.9,0.0,53961.2\nR001,2001-06-26,15.5,0.0,49395.17\nR001,2001-06-27,14.7,2.5,50950.73\nR001,2001-06-28,14.3,0.4,50361.25\nR001,2001-06-29,12.9,3.2,51952.92\nR001,2001-06-30,15.1,0.2,53164.17\nR001,2001-07-01,17.3,0.4,51127.5\nR001,2001-07-02,18.1,4.6,52921.1\nR001,2001-07-03,18.6,0.6,51455.77\nR001,2001-07-04,19.2,2.4,51615.21\nR001,2001-07-05,19.5,1.6,53888.31\nR001,2001-07-06,19.5,0.8,53869.21\nR001,2001-07-07,19.4,1.3,53757.81\nR001,2001-07-08,17.8,25.7,19719.06\nR001,2001-07-09,17.3,17.1,28970.16\nR001,2001-07-10,17.6,24.1,34370.33\nR001,2001-07-11,18.6,14.1,44499.83\nR001,2001-07-12,19.0,6.4,48598.08\nR001,2001-07-13,20.5,2.0,51005.42\nR001,2001-07-14,20.7,1.6,53089.37\nR001,2001-07-15,21.7,0.4,52125.91\nR001,2001-07-16,22.2,0.0,51274.46\nR001,2001-07-17,21.6,0.2,53395.57\nR001,2001-07-18,21.1,0.8,53141.4\nR001,2001-07-19,20.3,1.8,51529.85\nR001,2001-07-20,20.3,8.0,48293.01\nR001,2001-07-21,21.7,0.0,52387.47\nR001,2001-07-22,22.8,0.5,52348.21\nR001,2001-07-23,24.1,0.4,52308.97\nR001,2001-07-24,24.0,0.1,51918.3\nR001,2001-07-25,24.1,1.0,51529.38\nR001,2001-07-26,24.4,0.0,51221.92\nR001,2001-07-27,24.5,0.2,51455.57\nR001,2001-07-28,24.1,2.8,47113.54\nR001,2001-07-29,23.9,4.6,49623.75\nR001,2001-07-30,23.7,5.2,47598.68\nR001,2001-07-31,24.3,0.5,48845.25\nR001,2001-08-01,23.5,0.1,47368.84\nR001,2001-08-02,23.1,1.9,47554.01\nR001,2001-08-03,22.5,0.3,31877.61\nR001,2001-08-04,22.3,0.0,45097.03\nR001,2001-08-05,22.4,0.0,46851.47\nR001,2001-08-06,23.5,0.0,48362.27\nR001,2001-08-07,22.3,0.5,47953.24\nR001,2001-08-08,22.8,0.1,47980.6\nR001,2001-08-09,22.9,0.5,47669.95\nR001,2001-08-10,23.2,0.0,47361.02\nR001,2001-08-11,23.3,0.0,47390.49\nR001,2001-08-12,24.9,0.0,47420.03\nR001,2001-08-13,24.9,0.2,47449.69\nR001,2001-08-14,23.7,7.9,46800.0\nR001,2001-08-15,20.9,1.2,43958.69\nR001,2001-08-16,20.6,0.7,45301.61\nR001,2001-08-17,21.9,0.6,46800.0\nR001,2001-08-18,21.5,2.6,46800.0\nR001,2001-08-19,19.5,7.8,46173.91\nR001,2001-08-20,19.4,13.6,37687.99\nR001,2001-08-21,20.4,0.6,46190.92\nR001,2001-08-22,21.6,0.8,46800.0\nR001,2001-08-23,20.4,3.6,46800.0\nR001,2001-08-24,19.0,2.2,46250.14\nR001,2001-08-25,18.6,1.4,42801.89\nR001,2001-08-26,16.8,7.4,30266.19\nR001,2001-08-27,16.3,0.0,46732.15\nR001,2001-08-28,16.7,0.0,44673.09\nR001,2001-08-29,16.5,0.0,45724.89\nR001,2001-08-30,15.7,7.9,32439.53\nR001,2001-08-31,11.6,3.8,45146.78\nR001,2001-09-01,13.2,0.0,46197.33\nR001,2001-09-02,14.6,0.0,45602.83\nR001,2001-09-03,18.6,0.0,45651.74\nR001,2001-09-04,15.1,10.7,27271.01\nR001,2001-09-05,15.0,0.0,39680.25\nR001,2001-09-06,17.0,0.0,43881.72\nR001,2001-09-07,20.0,0.4,43933.45\nR001,2001-09-08,16.3,14.3,16007.66\nR001,2001-09-09,14.9,2.0,20544.05\nR001,2001-09-10,16.3,0.0,42745.52\nR001,2001-09-11,18.4,0.0,43200.0\nR001,2001-09-12,19.8,0.0,43200.0\nR001,2001-09-13,18.6,4.2,43173.35\nR001,2001-09-14,16.2,14.0,38453.68\nR001,2001-09-15,17.0,0.0,40746.58\nR001,2001-09-16,16.5,4.5,39226.32\nR001,2001-09-17,16.3,2.0,40789.56\nR001,2001-09-18,17.8,0.0,42835.76\nR001,2001-09-19,17.5,0.0,42767.88\nR001,2001-09-20,16.5,0.2,38276.34\nR001,2001-09-21,15.4,0.0,35517.91\nR001,2001-09-22,14.9,0.0,41939.01\nR001,2001-09-23,16.1,0.1,40700.33\nR001,2001-09-24,15.8,0.0,35206.6\nR001,2001-09-25,17.0,0.0,41727.07\nR001,2001-09-26,15.6,3.5,28377.95\nR001,2001-09-27,14.7,0.1,25382.58\nR001,2001-09-28,14.7,7.0,28856.77\nR001,2001-09-29,11.5,3.1,36041.37\nR001,2001-09-30,9.7,0.0,39843.19\nR001,2001-10-01,11.0,0.0,39203.17\nR001,2001-10-02,12.7,0.0,39236.09\nR001,2001-10-03,13.6,0.0,39310.52\nR001,2001-10-04,16.0,0.0,39560.5\nR001,2001-10-05,17.4,0.0,39314.16\nR001,2001-10-06,15.4,1.0,35353.51\nR001,2001-10-07,12.7,8.4,17737.35\nR001,2001-10-08,13.3,0.0,37618.18\nR001,2001-10-09,14.7,0.0,37434.78\nR001,2001-10-10,14.7,0.0,37204.61\nR001,2001-10-11,14.0,0.0,36814.11\nR001,2001-10-12,11.1,13.4,19409.84\nR001,2001-10-13,8.3,0.0,36000.0\nR001,2001-10-14,8.3,0.0,32114.94\nR001,2001-10-15,9.4,2.1,32134.21\nR001,2001-10-16,8.4,0.0,34652.35\nR001,2001-10-17,8.8,0.4,34459.19\nR001,2001-10-18,8.5,3.0,28698.73\nR001,2001-10-19,6.6,1.6,31457.57\nR001,2001-10-20,7.2,8.2,23617.7\nR001,2001-10-21,7.8,2.2,36000.0\nR001,2001-10-22,6.5,1.1,35116.76\nR001,2001-10-23,6.8,1.4,21008.07\nR001,2001-10-24,6.8,42.3,1784.28\nR001,2001-10-25,-1.4,0.2,36000.0\nR001,2001-10-26,-1.3,0.0,36000.0\nR001,2001-10-27,1.5,0.0,36000.0\nR001,2001-10-28,3.6,0.0,36000.0\nR001,2001-10-29,4.0,0.6,31571.17\nR001,2001-10-30,5.9,4.8,30423.71\nR001,2001-10-31,4.6,19.6,293.83\nR001,2001-11-01,8.1,2.4,25722.01\nR001,2001-11-02,8.5,0.0,35934.56\nR001,2001-11-03,7.6,0.7,33331.2\nR001,2001-11-04,8.2,0.0,35775.37\nR001,2001-11-05,8.7,0.0,35675.13\nR001,2001-11-06,9.4,3.1,20943.64\nR001,2001-11-07,8.1,3.4,25670.65\nR001,2001-11-08,7.8,0.0,32879.96\nR001,2001-11-09,10.8,0.0,30721.67\nR001,2001-11-10,11.5,0.5,23324.2\nR001,2001-11-11,8.9,7.1,15166.83\nR001,2001-11-12,3.8,2.4,31149.11\nR001,2001-11-13,6.2,3.3,14578.47\nR001,2001-11-14,10.5,5.6,6470.04\nR001,2001-11-15,8.7,24.1,0.0\nR001,2001-11-16,1.1,10.6,22425.7\nR001,2001-11-17,-4.1,22.0,31884.65\nR001,2001-11-18,-9.3,0.6,32142.78\nR001,2001-11-19,-2.0,3.9,22917.62\nR001,2001-11-20,1.6,3.9,18341.0\nR001,2001-11-21,-0.1,37.0,0.0\nR001,2001-11-22,-4.4,3.0,14152.01\nR001,2001-11-23,-0.4,0.0,31851.41\nR001,2001-11-24,0.3,0.0,22227.88\nR001,2001-11-25,2.6,2.3,9209.87\nR001,2001-11-26,2.2,18.9,0.0\nR001,2001-11-27,0.7,2.5,28800.0\nR001,2001-11-28,-0.0,4.4,28957.74\nR001,2001-11-29,-0.6,0.0,30667.46\nR001,2001-11-30,1.4,0.0,24971.22\nR001,2001-12-01,3.8,0.0,30325.38\nR001,2001-12-02,3.9,0.0,30341.2\nR001,2001-12-03,3.7,0.0,30355.56\nR001,2001-12-04,3.4,0.0,29221.7\nR001,2001-12-05,1.5,4.8,1610.03\nR001,2001-12-06,-1.8,5.9,20910.2\nR001,2001-12-07,-2.1,0.0,29629.61\nR001,2001-12-08,1.8,2.3,6444.11\nR001,2001-12-09,3.2,0.3,28800.0\nR001,2001-12-10,2.3,12.8,4173.54\nR001,2001-12-11,-1.3,4.8,0.0\nR001,2001-12-12,-0.3,2.5,18610.4\nR001,2001-12-13,0.5,10.9,3969.61\nR001,2001-12-14,1.3,12.9,12212.95\nR001,2001-12-15,-2.6,10.5,24035.5\nR001,2001-12-16,-4.2,0.3,25536.26\nR001,2001-12-17,0.3,29.8,0.0\nR001,2001-12-18,2.2,35.6,0.0\nR001,2001-12-19,0.9,30.2,0.0\nR001,2001-12-20,-4.4,1.7,28542.31\nR001,2001-12-21,-6.3,0.5,29192.9\nR001,2001-12-22,-1.9,1.8,25200.0\nR001,2001-12-23,0.4,7.2,2339.2\nR001,2001-12-24,2.9,1.7,29476.79\nR001,2001-12-25,4.0,32.9,0.0\nR001,2001-12-26,2.7,47.0,0.0\nR001,2001-12-27,1.5,0.7,31364.49\nR001,2001-12-28,4.7,21.3,3422.02\nR001,2001-12-29,3.5,23.8,2650.55\nR001,2001-12-30,0.7,0.0,31321.85\nR001,2001-12-31,2.9,16.7,0.0\nR001,2002-01-01,4.1,27.0,0.0\nR001,2002-01-02,-0.4,39.5,0.0\nR001,2002-01-03,-1.7,21.9,0.0\nR001,2002-01-04,-5.3,1.4,25200.0\nR001,2002-01-05,-4.1,5.2,0.0\nR001,2002-01-06,-1.3,0.9,22484.0\nR001,2002-01-07,-0.8,5.5,22380.75\nR001,2002-01-08,-5.4,1.9,21076.2\nR001,2002-01-09,-9.7,0.0,32080.72\nR001,2002-01-10,-7.3,3.2,23607.8\nR001,2002-01-11,-4.4,6.3,20395.5\nR001,2002-01-12,-3.8,1.8,30327.36\nR001,2002-01-13,-2.5,0.2,32354.56\nR001,2002-01-14,-1.1,0.0,32400.0\nR001,2002-01-15,-1.2,0.0,32400.0\nR001,2002-01-16,-1.7,0.0,32400.0\nR001,2002-01-17,-1.5,0.0,32400.0\nR001,2002-01-18,-3.1,0.0,28236.59\nR001,2002-01-19,-0.1,3.4,10563.32\nR001,2002-01-20,-0.1,3.2,7670.38\nR001,2002-01-21,-0.7,16.5,1632.86\nR001,2002-01-22,-1.6,2.7,29859.93\nR001,2002-01-23,-3.1,0.0,32400.0\nR001,2002-01-24,-1.0,0.0,32400.0\nR001,2002-01-25,2.2,0.0,32400.0\nR001,2002-01-26,2.0,0.1,26236.04\nR001,2002-01-27,2.7,0.0,32400.0\nR001,2002-01-28,1.5,0.0,32400.0\nR001,2002-01-29,0.1,5.5,32400.0\nR001,2002-01-30,0.9,16.3,0.0\nR001,2002-01-31,-0.5,17.3,28006.33\nR001,2002-02-01,1.6,0.7,17111.35\nR001,2002-02-02,-0.3,0.2,29939.39\nR001,2002-02-03,-1.0,0.0,32400.0\nR001,2002-02-04,2.1,0.0,32940.01\nR001,2002-02-05,5.2,0.0,33927.11\nR001,2002-02-06,7.0,0.0,24432.87\nR001,2002-02-07,6.8,0.0,34440.65\nR001,2002-02-08,4.0,0.0,35038.01\nR001,2002-02-09,3.6,0.0,28198.35\nR001,2002-02-10,1.6,3.7,33607.33\nR001,2002-02-11,3.3,0.0,19500.3\nR001,2002-02-12,3.1,2.7,970.06\nR001,2002-02-13,1.5,12.9,4421.33\nR001,2002-02-14,1.2,21.1,17657.12\nR001,2002-02-15,-2.4,15.2,10622.02\nR001,2002-02-16,-5.2,0.0,36651.11\nR001,2002-02-17,-4.3,0.6,28807.01\nR001,2002-02-18,-1.0,0.0,37047.77\nR001,2002-02-19,2.5,0.0,37913.72\nR001,2002-02-20,6.0,0.0,36577.54\nR001,2002-02-21,4.6,2.8,16923.53\nR001,2002-02-22,7.0,0.0,38435.14\nR001,2002-02-23,7.4,0.0,38611.49\nR001,2002-02-24,5.8,0.0,30331.01\nR001,2002-02-25,6.1,0.4,34420.23\nR001,2002-02-26,1.7,14.1,2903.08\nR001,2002-02-27,-1.7,0.0,39325.64\nR001,2002-02-28,1.6,0.0,38809.29\nR001,2002-03-01,4.7,2.7,7816.81\nR001,2002-03-02,5.9,5.2,24974.25\nR001,2002-03-03,6.9,0.0,36000.0\nR001,2002-03-04,4.0,17.1,0.0\nR001,2002-03-05,-0.9,3.7,37949.26\nR001,2002-03-06,3.7,0.0,39600.0\nR001,2002-03-07,8.6,0.0,36402.14\nR001,2002-03-08,7.2,11.9,21600.0\nR001,2002-03-09,-1.0,5.5,39600.0\nR001,2002-03-10,3.4,0.2,39600.0\nR001,2002-03-11,6.3,6.3,4133.14\nR001,2002-03-12,8.0,0.5,36019.97\nR001,2002-03-13,7.1,1.4,30594.26\nR001,2002-03-14,5.9,1.5,39158.73\nR001,2002-03-15,5.0,2.5,8974.47\nR001,2002-03-16,3.5,1.5,37592.74\nR001,2002-03-17,1.8,8.4,33860.92\nR001,2002-03-18,3.7,0.0,39690.27\nR001,2002-03-19,4.2,0.0,32369.08\nR001,2002-03-20,3.8,0.0,21199.51\nR001,2002-03-21,3.9,2.8,4431.72\nR001,2002-03-22,2.8,7.7,29867.93\nR001,2002-03-23,4.3,0.4,37544.24\nR001,2002-03-24,5.5,4.9,12130.41\nR001,2002-03-25,5.8,14.7,22662.96\nR001,2002-03-26,5.0,7.7,26166.93\nR001,2002-03-27,3.0,4.8,28980.64\nR001,2002-03-28,1.7,3.7,40100.36\nR001,2002-03-29,1.7,1.3,39759.42\nR001,2002-03-30,2.4,0.0,40239.15\nR001,2002-03-31,3.6,0.0,43218.07\nR001,2002-04-01,5.5,0.0,42840.73\nR001,2002-04-02,7.5,0.0,44879.93\nR001,2002-04-03,6.8,0.0,40163.71\nR001,2002-04-04,3.2,0.2,38477.96\nR001,2002-04-05,4.2,0.9,38620.42\nR001,2002-04-06,6.7,13.6,27156.11\nR001,2002-04-07,4.1,9.8,394.28\nR001,2002-04-08,0.1,8.9,26060.23\nR001,2002-04-09,-1.5,10.1,36054.43\nR001,2002-04-10,0.3,8.7,42685.12\nR001,2002-04-11,1.8,0.0,46356.45\nR001,2002-04-12,3.4,0.0,45776.42\nR001,2002-04-13,5.1,0.0,41918.55\nR001,2002-04-14,6.6,0.0,21767.85\nR001,2002-04-15,7.2,2.9,22127.9\nR001,2002-04-16,8.9,0.0,45567.42\nR001,2002-04-17,7.4,7.3,28782.3\nR001,2002-04-18,8.0,1.2,19680.25\nR001,2002-04-19,8.0,1.0,45363.66\nR001,2002-04-20,7.2,4.1,0.0\nR001,2002-04-21,5.6,11.7,1414.82\nR001,2002-04-22,5.1,8.1,14079.39\nR001,2002-04-23,4.8,7.5,4164.65\nR001,2002-04-24,5.6,1.5,41087.78\nR001,2002-04-25,6.0,1.1,39455.27\nR001,2002-04-26,5.6,0.0,41111.12\nR001,2002-04-27,4.4,0.8,45521.86\nR001,2002-04-28,5.4,0.5,46800.0\nR001,2002-04-29,7.3,0.2,47923.96\nR001,2002-04-30,8.6,2.8,42751.57\nR001,2002-05-01,9.8,0.7,47176.74\nR001,2002-05-02,10.1,1.7,46850.56\nR001,2002-05-03,9.7,4.3,44332.87\nR001,2002-05-04,8.5,1.5,46150.39\nR001,2002-05-05,7.5,6.0,42184.26\nR001,2002-05-06,7.6,12.0,11485.96\nR001,2002-05-07,8.1,6.1,40115.9\nR001,2002-05-08,8.3,9.8,14492.32\nR001,2002-05-09,9.0,6.3,41040.26\nR001,2002-05-10,9.6,6.4,49594.89\nR001,2002-05-11,9.5,2.7,44180.67\nR001,2002-05-12,10.7,0.6,46093.69\nR001,2002-05-13,12.3,0.1,45307.12\nR001,2002-05-14,12.9,0.0,50377.75\nR001,2002-05-15,12.1,0.0,50400.0\nR001,2002-05-16,12.1,3.4,47616.7\nR001,2002-05-17,12.3,3.8,47543.72\nR001,2002-05-18,13.0,0.3,47782.49\nR001,2002-05-19,12.8,0.9,47410.03\nR001,2002-05-20,11.3,1.6,45757.27\nR001,2002-05-21,11.4,0.0,39794.6\nR001,2002-05-22,12.4,0.0,48372.85\nR001,2002-05-23,13.3,3.6,47419.76\nR001,2002-05-24,12.5,7.0,30491.16\nR001,2002-05-25,14.1,1.0,48280.48\nR001,2002-05-26,15.7,0.1,50540.6\nR001,2002-05-27,18.0,0.0,51234.97\nR001,2002-05-28,16.4,5.5,50300.56\nR001,2002-05-29,14.4,4.6,46871.96\nR001,2002-05-30,17.4,0.4,51169.43\nR001,2002-05-31,15.2,10.7,24970.0\nR001,2002-06-01,12.9,6.8,43718.64\nR001,2002-06-02,12.4,4.1,46264.69\nR001,2002-06-03,13.2,4.1,46842.42\nR001,2002-06-04,15.7,0.4,49695.38\nR001,2002-06-05,14.0,14.8,37366.94\nR001,2002-06-06,11.1,5.2,33184.54\nR001,2002-06-07,11.9,2.9,43599.19\nR001,2002-06-08,15.0,0.0,50829.43\nR001,2002-06-09,17.9,0.9,51673.85\nR001,2002-06-10,18.6,0.0,53525.24\nR001,2002-06-11,21.7,0.5,50580.02\nR001,2002-06-12,16.1,24.4,8284.28\nR001,2002-06-13,15.4,13.4,28491.77\nR001,2002-06-14,15.8,7.8,48189.86\nR001,2002-06-15,16.3,0.9,43127.04\nR001,2002-06-16,17.5,0.6,42090.39\nR001,2002-06-17,17.8,0.2,49011.85\nR001,2002-06-18,13.0,19.9,10618.76\nR001,2002-06-19,12.8,6.2,28363.45\nR001,2002-06-20,13.0,37.2,0.0\nR001,2002-06-21,18.6,4.9,48180.84\nR001,2002-06-22,18.7,0.0,51480.52\nR001,2002-06-23,20.0,0.0,48216.31\nR001,2002-06-24,20.9,0.0,48938.3\nR001,2002-06-25,22.8,0.0,53960.26\nR001,2002-06-26,20.6,7.6,46439.65\nR001,2002-06-27,18.0,23.3,25145.69\nR001,2002-06-28,17.7,11.0,34088.52\nR001,2002-06-29,17.1,5.4,41283.99\nR001,2002-06-30,17.5,7.5,38056.55\nR001,2002-07-01,18.0,0.8,47650.07\nR001,2002-07-02,18.8,1.9,51603.31\nR001,2002-07-03,19.0,6.9,50789.06\nR001,2002-07-04,18.6,5.4,43090.05\nR001,2002-07-05,17.8,12.5,51188.33\nR001,2002-07-06,18.8,1.3,50549.01\nR001,2002-07-07,20.6,0.6,51136.49\nR001,2002-07-08,22.0,0.5,51585.81\nR001,2002-07-09,22.4,6.0,53804.52\nR001,2002-07-10,22.8,1.1,53411.92\nR001,2002-07-11,21.9,6.1,53745.42\nR001,2002-07-12,21.2,7.8,51283.06\nR001,2002-07-13,21.1,5.7,49769.37\nR001,2002-07-14,20.7,3.2,53640.84\nR001,2002-07-15,19.9,5.9,44445.26\nR001,2002-07-16,19.7,4.0,49180.07\nR001,2002-07-17,20.2,1.7,51306.26\nR001,2002-07-18,20.4,0.7,52133.19\nR001,2002-07-19,22.3,0.1,50880.78\nR001,2002-07-20,24.3,0.5,53141.01\nR001,2002-07-21,23.9,0.0,52746.18\nR001,2002-07-22,23.4,0.3,52353.06\nR001,2002-07-23,18.6,25.5,1877.97\nR001,2002-07-24,17.9,4.8,7497.74\nR001,2002-07-25,17.2,10.1,22742.1\nR001,2002-07-26,18.5,2.0,47109.07\nR001,2002-07-27,19.7,0.2,50555.23\nR001,2002-07-28,21.4,1.1,50733.17\nR001,2002-07-29,21.7,2.6,50649.74\nR001,2002-07-30,21.1,1.8,49660.05\nR001,2002-07-31,22.0,0.3,50634.48\nR001,2002-08-01,22.0,2.1,49766.26\nR001,2002-08-02,21.8,0.7,49477.71\nR001,2002-08-03,19.9,2.0,48505.56\nR001,2002-08-04,19.0,8.0,39802.77\nR001,2002-08-05,18.9,0.7,46844.77\nR001,2002-08-06,18.1,11.7,5979.82\nR001,2002-08-07,18.0,0.9,41355.48\nR001,2002-08-08,19.0,0.0,47296.64\nR001,2002-08-09,18.4,7.6,38909.78\nR001,2002-08-10,17.8,6.1,36494.97\nR001,2002-08-11,18.6,4.9,45631.44\nR001,2002-08-12,20.2,0.1,45354.16\nR001,2002-08-13,22.8,0.0,47443.58\nR001,2002-08-14,21.6,1.7,47138.56\nR001,2002-08-15,18.9,9.7,25605.83\nR001,2002-08-16,18.5,2.1,46743.28\nR001,2002-08-17,18.1,3.2,44099.37\nR001,2002-08-18,17.5,4.4,44760.53\nR001,2002-08-19,17.5,3.1,46503.12\nR001,2002-08-20,18.5,1.3,44938.74\nR001,2002-08-21,18.0,12.9,37493.77\nR001,2002-08-22,16.6,9.7,43815.04\nR001,2002-08-23,17.4,6.3,38218.38\nR001,2002-08-24,17.1,0.0,46262.67\nR001,2002-08-25,17.1,0.1,46725.93\nR001,2002-08-26,16.7,1.3,44234.59\nR001,2002-08-27,16.2,0.2,45634.29\nR001,2002-08-28,15.7,1.5,44095.88\nR001,2002-08-29,15.7,0.1,45513.18\nR001,2002-08-30,17.0,0.6,46440.62\nR001,2002-08-31,17.9,0.3,46140.16\nR001,2002-09-01,19.6,0.9,45865.05\nR001,2002-09-02,18.9,9.0,44306.06\nR001,2002-09-03,16.7,9.4,44355.33\nR001,2002-09-04,16.7,5.7,42670.21\nR001,2002-09-05,18.0,2.1,43813.95\nR001,2002-09-06,18.7,4.8,40440.16\nR001,2002-09-07,17.0,12.7,25670.67\nR001,2002-09-08,16.9,1.5,36525.61\nR001,2002-09-09,15.7,6.2,19825.84\nR001,2002-09-10,15.4,2.6,39007.85\nR001,2002-09-11,14.3,0.0,43200.0\nR001,2002-09-12,14.3,3.8,25516.76\nR001,2002-09-13,15.1,0.0,39600.0\nR001,2002-09-14,15.4,0.0,41024.48\nR001,2002-09-15,15.1,2.7,40273.05\nR001,2002-09-16,17.7,0.0,41408.49\nR001,2002-09-17,15.1,31.2,22339.61\nR001,2002-09-18,11.9,1.1,34528.09\nR001,2002-09-19,11.3,1.6,40235.83\nR001,2002-09-20,11.8,0.0,39423.93\nR001,2002-09-21,13.8,0.0,41577.38\nR001,2002-09-22,15.8,0.0,42567.2\nR001,2002-09-23,19.0,0.0,42511.84\nR001,2002-09-24,20.4,0.0,41682.38\nR001,2002-09-25,20.4,0.0,41718.01\nR001,2002-09-26,19.1,0.0,40797.53\nR001,2002-09-27,20.5,0.0,41152.36\nR001,2002-09-28,18.5,2.1,40231.22\nR001,2002-09-29,16.7,0.4,36754.06\nR001,2002-09-30,16.1,4.2,30312.51\nR001,2002-10-01,14.5,2.8,38589.43\nR001,2002-10-02,13.9,12.8,20770.13\nR001,2002-10-03,11.3,0.3,31041.51\nR001,2002-10-04,9.5,0.0,38582.22\nR001,2002-10-05,9.6,0.0,39294.64\nR001,2002-10-06,11.8,0.0,38785.1\nR001,2002-10-07,14.7,0.0,35017.66\nR001,2002-10-08,14.7,3.5,25805.34\nR001,2002-10-09,12.0,2.1,32470.08\nR001,2002-10-10,11.6,0.8,36747.82\nR001,2002-10-11,13.2,0.0,35443.31\nR001,2002-10-12,15.3,0.6,36281.75\nR001,2002-10-13,16.4,0.0,34869.99\nR001,2002-10-14,17.0,0.0,32380.77\nR001,2002-10-15,15.2,3.5,20367.71\nR001,2002-10-16,12.3,13.8,1527.28\nR001,2002-10-17,12.9,0.0,34829.07\nR001,2002-10-18,14.4,0.0,36000.0\nR001,2002-10-19,16.4,0.0,36000.0\nR001,2002-10-20,14.5,0.5,36000.0\nR001,2002-10-21,6.7,23.0,11730.04\nR001,2002-10-22,3.6,0.0,36000.0\nR001,2002-10-23,10.7,1.9,19489.68\nR001,2002-10-24,12.9,0.3,34335.63\nR001,2002-10-25,13.6,0.0,28699.18\nR001,2002-10-26,10.4,2.7,28488.24\nR001,2002-10-27,9.2,1.8,16173.07\nR001,2002-10-28,6.9,2.9,35786.76\nR001,2002-10-29,3.8,30.2,513.34\nR001,2002-10-30,0.2,4.9,35335.32\nR001,2002-10-31,1.3,1.0,32133.06\nR001,2002-11-01,3.1,3.1,35070.76\nR001,2002-11-02,2.9,0.0,32367.17\nR001,2002-11-03,4.8,0.0,35877.14\nR001,2002-11-04,5.9,22.0,1749.23\nR001,2002-11-05,8.8,0.0,35720.29\nR001,2002-11-06,9.8,0.0,19793.74\nR001,2002-11-07,10.3,0.0,27835.35\nR001,2002-11-08,10.1,0.0,35266.79\nR001,2002-11-09,11.7,0.0,35116.26\nR001,2002-11-10,9.6,15.4,1985.45\nR001,2002-11-11,8.0,23.7,10637.96\nR001,2002-11-12,3.4,2.7,32683.39\nR001,2002-11-13,2.4,9.2,29227.93\nR001,2002-11-14,2.6,0.0,29259.74\nR001,2002-11-15,3.1,0.0,31870.4\nR001,2002-11-16,6.0,0.0,34071.59\nR001,2002-11-17,8.8,0.0,33701.32\nR001,2002-11-18,9.2,0.0,33330.78\nR001,2002-11-19,9.0,0.0,33325.3\nR001,2002-11-20,9.4,0.0,32955.5\nR001,2002-11-21,8.5,0.0,30890.1\nR001,2002-11-22,6.6,0.0,29650.95\nR001,2002-11-23,9.0,0.0,29769.44\nR001,2002-11-24,8.5,0.0,28287.98\nR001,2002-11-25,6.8,0.0,29071.79\nR001,2002-11-26,5.4,3.4,14419.14\nR001,2002-11-27,4.7,0.0,26906.96\nR001,2002-11-28,3.5,0.1,30261.88\nR001,2002-11-29,5.3,0.0,29536.53\nR001,2002-11-30,4.2,0.0,30725.88\nR001,2002-12-01,2.6,0.0,29560.15\nR001,2002-12-02,3.9,0.0,30335.67\nR001,2002-12-03,5.5,0.0,29206.95\nR001,2002-12-04,6.8,0.0,26376.52\nR001,2002-12-05,6.2,0.0,29611.05\nR001,2002-12-06,6.7,0.0,7637.66\nR001,2002-12-07,5.0,14.9,3557.91\nR001,2002-12-08,-0.2,15.5,264.49\nR001,2002-12-09,-3.6,3.4,11704.4\nR001,2002-12-10,-4.2,0.2,26565.84\nR001,2002-12-11,-4.4,3.2,26470.5\nR001,2002-12-12,-7.6,1.5,30423.24\nR001,2002-12-13,-8.1,0.0,27424.15\nR001,2002-12-14,-7.2,0.0,29645.24\nR001,2002-12-15,-8.2,0.0,28863.11\nR001,2002-12-16,-8.4,0.0,29636.15\nR001,2002-12-17,-9.5,0.0,30799.05\nR001,2002-12-18,-9.9,0.0,28800.0\nR001,2002-12-19,-7.6,0.0,28513.27\nR001,2002-12-20,-8.7,0.0,4430.8\nR001,2002-12-21,-5.6,1.9,28624.44\nR001,2002-12-22,-3.8,8.6,22260.14\nR001,2002-12-23,-4.2,16.2,28315.56\nR001,2002-12-24,-5.1,22.4,0.0\nR001,2002-12-25,-12.6,0.9,29517.99\nR001,2002-12-26,-12.3,0.9,26892.32\nR001,2002-12-27,-8.7,0.6,31359.64\nR001,2002-12-28,-4.1,0.0,24791.9\nR001,2002-12-29,0.5,0.4,29291.7\nR001,2002-12-30,4.4,8.9,20560.81\nR001,2002-12-31,4.8,12.1,962.15\nR001,2003-01-01,4.8,10.0,0.0\nR001,2003-01-02,4.1,14.5,3161.06\nR001,2003-01-03,3.9,3.1,15694.1\nR001,2003-01-04,3.2,17.2,0.0\nR001,2003-01-05,1.6,0.3,1880.1\nR001,2003-01-06,3.2,21.1,9080.89\nR001,2003-01-07,0.3,14.8,31109.19\nR001,2003-01-08,4.4,4.0,19110.13\nR001,2003-01-09,1.3,10.3,24976.56\nR001,2003-01-10,0.5,0.0,32131.49\nR001,2003-01-11,3.9,3.9,13666.74\nR001,2003-01-12,5.1,0.0,0.0\nR001,2003-01-13,0.3,13.7,0.0\nR001,2003-01-14,-3.3,3.4,30702.07\nR001,2003-01-15,-4.0,4.5,22307.12\nR001,2003-01-16,-2.4,2.5,30578.07\nR001,2003-01-17,-2.1,1.4,23362.88\nR001,2003-01-18,-0.9,0.3,28247.68\nR001,2003-01-19,-3.3,0.0,32400.0\nR001,2003-01-20,-2.3,0.0,32400.0\nR001,2003-01-21,-3.6,0.0,32400.0\nR001,2003-01-22,-3.6,0.0,32400.0\nR001,2003-01-23,-2.3,0.5,21252.4\nR001,2003-01-24,0.3,0.0,30336.0\nR001,2003-01-25,1.7,0.0,23169.23\nR001,2003-01-26,-0.2,1.1,29200.84\nR001,2003-01-27,-1.1,1.8,32400.0\nR001,2003-01-28,-2.7,0.1,32051.39\nR001,2003-01-29,-0.8,1.5,32400.0\nR001,2003-01-30,2.1,11.1,2302.36\nR001,2003-01-31,5.3,26.5,2321.6\nR001,2003-02-01,3.3,21.8,7795.27\nR001,2003-02-02,3.2,4.3,24238.98\nR001,2003-02-03,2.5,14.0,317.24\nR001,2003-02-04,-0.2,1.2,30947.35\nR001,2003-02-05,1.3,0.2,33597.93\nR001,2003-02-06,3.6,0.0,29348.15\nR001,2003-02-07,3.7,0.0,29602.39\nR001,2003-02-08,3.6,1.0,23730.61\nR001,2003-02-09,0.3,12.8,0.0\nR001,2003-02-10,-1.8,9.6,27763.33\nR001,2003-02-11,-2.9,1.7,30400.17\nR001,2003-02-12,-4.1,0.0,35492.93\nR001,2003-02-13,-3.2,0.0,36853.08\nR001,2003-02-14,-3.3,1.1,34500.37\nR001,2003-02-15,-2.2,3.8,24488.04\nR001,2003-02-16,-3.8,3.6,30454.22\nR001,2003-02-17,-5.5,1.6,35665.67\nR001,2003-02-18,-6.5,0.0,37699.58\nR001,2003-02-19,-7.3,0.0,36034.04\nR001,2003-02-20,-5.1,3.5,22400.14\nR001,2003-02-21,-6.1,4.2,27807.33\nR001,2003-02-22,-6.1,0.9,29704.07\nR001,2003-02-23,-6.6,0.0,38577.75\nR001,2003-02-24,-3.4,0.0,29438.53\nR001,2003-02-25,-1.5,1.3,35365.42\nR001,2003-02-26,-1.0,6.2,21097.68\nR001,2003-02-27,-1.7,1.8,28842.89\nR001,2003-02-28,-2.1,1.7,37250.08\nR001,2003-03-01,-2.3,0.0,39165.36\nR001,2003-03-02,-1.2,0.0,39600.0\nR001,2003-03-03,1.5,0.2,6253.43\nR001,2003-03-04,2.7,0.0,19508.01\nR001,2003-03-05,3.1,0.0,39600.0\nR001,2003-03-06,4.2,0.0,38516.45\nR001,2003-03-07,3.7,8.9,5735.2\nR001,2003-03-08,2.4,1.2,39014.17\nR001,2003-03-09,3.2,0.0,39494.49\nR001,2003-03-10,1.9,5.0,3996.41\nR001,2003-03-11,1.9,3.9,24586.08\nR001,2003-03-12,1.6,6.0,27158.65\nR001,2003-03-13,1.8,0.3,24179.98\nR001,2003-03-14,2.2,0.9,21040.54\nR001,2003-03-15,1.8,8.2,34754.83\nR001,2003-03-16,0.5,0.1,39600.0\nR001,2003-03-17,-0.7,2.3,38810.13\nR001,2003-03-18,-0.1,0.0,40018.61\nR001,2003-03-19,0.3,12.0,423.73\nR001,2003-03-20,-2.2,7.1,39600.0\nR001,2003-03-21,-2.5,10.3,18841.64\nR001,2003-03-22,-4.9,17.7,0.0\nR001,2003-03-23,-6.2,0.0,41018.85\nR001,2003-03-24,-2.3,0.0,40956.72\nR001,2003-03-25,1.3,0.0,41230.88\nR001,2003-03-26,0.9,0.0,41177.92\nR001,2003-03-27,-1.1,8.9,39874.12\nR001,2003-03-28,-3.5,3.5,39044.45\nR001,2003-03-29,-3.2,0.0,41821.43\nR001,2003-03-30,-0.7,0.0,44639.73\nR001,2003-03-31,2.9,0.0,45032.6\nR001,2003-04-01,4.3,0.1,42816.02\nR001,2003-04-02,0.2,11.7,34764.03\nR001,2003-04-03,3.6,1.7,14393.03\nR001,2003-04-04,6.3,4.3,13888.64\nR001,2003-04-05,6.9,0.7,43979.88\nR001,2003-04-06,4.7,25.9,15859.23\nR001,2003-04-07,5.7,1.3,34174.85\nR001,2003-04-08,10.0,0.0,46127.11\nR001,2003-04-09,3.5,11.9,5445.87\nR001,2003-04-10,3.4,0.0,43593.56\nR001,2003-04-11,4.4,0.0,46077.07\nR001,2003-04-12,5.0,0.0,46411.45\nR001,2003-04-13,7.1,0.0,46482.28\nR001,2003-04-14,8.2,0.0,46553.25\nR001,2003-04-15,7.8,0.0,46624.34\nR001,2003-04-16,7.5,0.0,46695.54\nR001,2003-04-17,6.8,1.4,33146.56\nR001,2003-04-18,6.8,0.3,44994.12\nR001,2003-04-19,6.3,2.5,16980.2\nR001,2003-04-20,4.6,10.4,2136.77\nR001,2003-04-21,5.1,5.0,1899.57\nR001,2003-04-22,4.6,0.6,18375.2\nR001,2003-04-23,5.6,2.2,37855.67\nR001,2003-04-24,5.5,4.5,33128.7\nR001,2003-04-25,6.7,9.0,4629.55\nR001,2003-04-26,5.3,13.2,16239.84\nR001,2003-04-27,4.1,0.1,39993.68\nR001,2003-04-28,4.6,0.0,47305.29\nR001,2003-04-29,6.4,0.0,47199.71\nR001,2003-04-30,8.2,1.0,45506.99\nR001,2003-05-01,9.5,0.0,48190.97\nR001,2003-05-02,12.0,0.0,48826.67\nR001,2003-05-03,13.4,0.0,48120.16\nR001,2003-05-04,15.7,0.0,48423.21\nR001,2003-05-05,14.2,0.0,46800.0\nR001,2003-05-06,7.6,2.6,42007.56\nR001,2003-05-07,7.3,0.0,47647.33\nR001,2003-05-08,9.0,0.0,49656.74\nR001,2003-05-09,10.4,1.3,47923.98\nR001,2003-05-10,11.4,0.0,48919.23\nR001,2003-05-11,14.9,0.0,50155.26\nR001,2003-05-12,16.1,0.0,50225.92\nR001,2003-05-13,16.2,0.5,50295.08\nR001,2003-05-14,16.6,0.3,50362.69\nR001,2003-05-15,16.6,0.5,47853.83\nR001,2003-05-16,15.1,4.4,49425.72\nR001,2003-05-17,13.7,5.9,44943.64\nR001,2003-05-18,13.8,0.8,46051.57\nR001,2003-05-19,14.0,8.8,42515.43\nR001,2003-05-20,14.2,3.6,41587.37\nR001,2003-05-21,16.4,0.5,47596.54\nR001,2003-05-22,18.8,0.5,50400.0\nR001,2003-05-23,18.6,0.5,50400.0\nR001,2003-05-24,18.8,0.7,50955.77\nR001,2003-05-25,20.3,1.0,50931.1\nR001,2003-05-26,18.2,2.6,49084.25\nR001,2003-05-27,16.0,7.8,24821.01\nR001,2003-05-28,15.0,9.3,21582.67\nR001,2003-05-29,15.1,4.7,48267.37\nR001,2003-05-30,14.3,12.7,40210.11\nR001,2003-05-31,12.7,4.1,29666.52\nR001,2003-06-01,12.2,12.0,44233.92\nR001,2003-06-02,14.4,0.0,50277.81\nR001,2003-06-03,13.6,0.5,49740.17\nR001,2003-06-04,10.1,4.7,40769.58\nR001,2003-06-05,9.4,0.0,51777.58\nR001,2003-06-06,11.5,0.0,51286.23\nR001,2003-06-07,14.3,0.9,51743.23\nR001,2003-06-08,15.9,0.2,45728.82\nR001,2003-06-09,16.9,0.0,49662.26\nR001,2003-06-10,17.1,0.0,52795.85\nR001,2003-06-11,18.4,0.0,53149.58\nR001,2003-06-12,17.5,0.6,50570.44\nR001,2003-06-13,15.8,3.8,41659.01\nR001,2003-06-14,16.2,0.0,51205.36\nR001,2003-06-15,16.5,0.0,47823.52\nR001,2003-06-16,16.7,0.0,50851.67\nR001,2003-06-17,17.8,2.4,52716.74\nR001,2003-06-18,14.8,0.0,52372.31\nR001,2003-06-19,14.2,0.0,53893.85\nR001,2003-06-20,15.3,0.0,52690.17\nR001,2003-06-21,16.6,0.8,51579.23\nR001,2003-06-22,15.5,1.4,50474.4\nR001,2003-06-23,13.5,0.0,51495.01\nR001,2003-06-24,12.5,1.3,50040.63\nR001,2003-06-25,15.1,0.0,53964.29\nR001,2003-06-26,17.7,0.0,53968.47\nR001,2003-06-27,15.9,1.1,53018.49\nR001,2003-06-28,15.5,0.0,52452.94\nR001,2003-06-29,15.1,0.0,50903.67\nR001,2003-06-30,14.4,0.9,48345.73\nR001,2003-07-01,14.9,1.3,50739.57\nR001,2003-07-02,16.6,0.3,50420.65\nR001,2003-07-03,20.3,0.0,53930.32\nR001,2003-07-04,23.5,0.0,53915.47\nR001,2003-07-05,24.5,0.0,53898.46\nR001,2003-07-06,23.4,0.7,53879.29\nR001,2003-07-07,18.4,9.1,47179.6\nR001,2003-07-08,16.8,0.3,50728.62\nR001,2003-07-09,16.4,0.3,52570.23\nR001,2003-07-10,15.9,2.2,48778.81\nR001,2003-07-11,17.7,0.0,52334.54\nR001,2003-07-12,19.1,0.2,53065.1\nR001,2003-07-13,18.1,0.0,53686.17\nR001,2003-07-14,18.3,0.0,52026.89\nR001,2003-07-15,20.3,0.6,52378.5\nR001,2003-07-16,18.6,6.4,25555.41\nR001,2003-07-17,17.7,0.5,47059.02\nR001,2003-07-18,16.2,4.8,48741.81\nR001,2003-07-19,16.5,5.6,45567.26\nR001,2003-07-20,17.7,2.1,48827.46\nR001,2003-07-21,19.0,2.1,49575.99\nR001,2003-07-22,19.3,4.5,49970.81\nR001,2003-07-23,18.2,9.4,34304.71\nR001,2003-07-24,16.8,18.6,42655.61\nR001,2003-07-25,16.4,6.8,37636.36\nR001,2003-07-26,16.5,13.4,38743.08\nR001,2003-07-27,18.0,7.4,47749.94\nR001,2003-07-28,18.1,17.2,30117.07\nR001,2003-07-29,16.8,48.5,16726.61\nR001,2003-07-30,17.0,18.7,29021.36\nR001,2003-07-31,17.5,9.4,45076.52\nR001,2003-08-01,18.6,5.0,48031.42\nR001,2003-08-02,20.3,0.0,49811.54\nR001,2003-08-03,20.3,0.2,48863.67\nR001,2003-08-04,20.2,1.9,46833.02\nR001,2003-08-05,18.9,6.5,43055.02\nR001,2003-08-06,17.6,0.7,46800.0\nR001,2003-08-07,18.8,0.4,47261.72\nR001,2003-08-08,18.9,6.6,43522.15\nR001,2003-08-09,19.1,14.0,27637.11\nR001,2003-08-10,19.6,3.2,43335.03\nR001,2003-08-11,19.7,2.7,33509.47\nR001,2003-08-12,16.8,7.6,46800.0\nR001,2003-08-13,14.5,0.0,46012.15\nR001,2003-08-14,15.7,0.0,45928.64\nR001,2003-08-15,17.4,0.0,46825.91\nR001,2003-08-16,18.8,0.1,46616.7\nR001,2003-08-17,19.9,0.9,46479.35\nR001,2003-08-18,19.3,5.0,43070.75\nR001,2003-08-19,19.6,0.1,43660.17\nR001,2003-08-20,19.4,0.2,46233.07\nR001,2003-08-21,19.2,1.7,45794.63\nR001,2003-08-22,18.5,3.5,46800.0\nR001,2003-08-23,17.0,5.0,44873.33\nR001,2003-08-24,17.3,1.5,43591.7\nR001,2003-08-25,18.0,0.0,45769.22\nR001,2003-08-26,18.8,0.4,46800.0\nR001,2003-08-27,18.9,0.0,45895.96\nR001,2003-08-28,19.8,0.0,46673.02\nR001,2003-08-29,19.7,1.5,46359.26\nR001,2003-08-30,19.8,0.3,44790.48\nR001,2003-08-31,20.6,0.0,45803.56\nR001,2003-09-01,22.0,0.2,46265.71\nR001,2003-09-02,21.8,0.2,45254.05\nR001,2003-09-03,19.9,3.2,39981.75\nR001,2003-09-04,18.0,23.5,28872.85\nR001,2003-09-05,14.7,74.9,1326.87\nR001,2003-09-06,11.6,65.9,0.0\nR001,2003-09-07,10.2,0.2,43265.69\nR001,2003-09-08,10.1,0.0,41194.07\nR001,2003-09-09,11.9,0.0,41688.38\nR001,2003-09-10,13.4,0.0,43427.05\nR001,2003-09-11,14.0,0.0,43200.0\nR001,2003-09-12,14.6,9.9,16648.62\nR001,2003-09-13,14.9,8.5,12421.65\nR001,2003-09-14,15.2,0.4,41972.17\nR001,2003-09-15,17.4,0.0,43071.42\nR001,2003-09-16,15.3,5.0,35898.06\nR001,2003-09-17,14.0,11.1,2947.24\nR001,2003-09-18,11.8,6.4,36715.38\nR001,2003-09-19,10.8,9.7,32830.31\nR001,2003-09-20,11.7,0.3,41215.98\nR001,2003-09-21,12.2,0.0,40931.87\nR001,2003-09-22,13.1,0.0,41284.71\nR001,2003-09-23,14.2,0.0,41319.56\nR001,2003-09-24,15.3,0.3,41991.81\nR001,2003-09-25,15.1,1.1,40590.22\nR001,2003-09-26,14.2,0.0,41426.15\nR001,2003-09-27,14.8,0.0,39147.96\nR001,2003-09-28,14.1,7.3,31103.97\nR001,2003-09-29,12.8,4.9,34071.42\nR001,2003-09-30,12.2,0.1,38416.2\nR001,2003-10-01,12.8,4.7,38570.17\nR001,2003-10-02,12.9,0.5,38343.4\nR001,2003-10-03,12.0,3.5,36033.82\nR001,2003-10-04,12.6,0.0,38656.65\nR001,2003-10-05,14.2,0.0,38630.25\nR001,2003-10-06,17.8,0.0,38450.1\nR001,2003-10-07,17.5,0.0,37850.15\nR001,2003-10-08,17.1,0.0,36000.0\nR001,2003-10-09,19.5,0.7,37325.34\nR001,2003-10-10,14.1,18.9,24992.54\nR001,2003-10-11,12.9,51.8,7200.0\nR001,2003-10-12,10.4,6.8,24064.3\nR001,2003-10-13,11.1,0.0,33721.92\nR001,2003-10-14,11.9,2.0,31389.84\nR001,2003-10-15,12.5,0.0,36021.03\nR001,2003-10-16,11.8,5.9,2806.37\nR001,2003-10-17,13.3,6.7,23669.79\nR001,2003-10-18,11.8,21.7,7855.48\nR001,2003-10-19,7.2,6.3,29765.82\nR001,2003-10-20,7.1,5.9,18611.02\nR001,2003-10-21,12.2,0.4,32312.85\nR001,2003-10-22,13.2,8.6,20780.92\nR001,2003-10-23,16.7,0.1,33947.22\nR001,2003-10-24,14.0,22.2,3600.0\nR001,2003-10-25,11.9,46.7,0.0\nR001,2003-10-26,7.3,0.5,34597.19\nR001,2003-10-27,4.1,2.5,29128.25\nR001,2003-10-28,3.3,17.0,837.29\nR001,2003-10-29,2.5,0.8,27859.24\nR001,2003-10-30,3.9,15.1,0.0\nR001,2003-10-31,5.4,0.5,26236.49\nR001,2003-11-01,7.8,10.7,1185.92\nR001,2003-11-02,6.6,0.0,35978.51\nR001,2003-11-03,8.1,0.0,35898.91\nR001,2003-11-04,11.4,0.0,32153.67\nR001,2003-11-05,12.3,0.0,35738.67\nR001,2003-11-06,10.4,0.0,29446.85\nR001,2003-11-07,11.0,0.0,32450.68\nR001,2003-11-08,8.1,1.2,33712.52\nR001,2003-11-09,5.0,14.0,4665.76\nR001,2003-11-10,3.5,0.4,13142.6\nR001,2003-11-11,7.0,4.2,30952.85\nR001,2003-11-12,4.8,25.9,3659.2\nR001,2003-11-13,1.2,0.0,32665.54\nR001,2003-11-14,-1.4,0.0,32648.62\nR001,2003-11-15,-0.9,0.0,33716.95\nR001,2003-11-16,-1.1,0.0,34068.86\nR001,2003-11-17,2.3,11.4,6713.48\nR001,2003-11-18,3.6,6.3,11530.64\nR001,2003-11-19,0.6,9.7,28569.83\nR001,2003-11-20,0.5,37.5,0.0\nR001,2003-11-21,1.0,8.8,22805.55\nR001,2003-11-22,-0.2,0.0,32215.98\nR001,2003-11-23,3.1,0.0,32215.93\nR001,2003-11-24,3.3,0.0,31827.13\nR001,2003-11-25,2.6,0.0,31851.3\nR001,2003-11-26,5.1,0.0,21600.0\nR001,2003-11-27,5.4,0.0,14541.95\nR001,2003-11-28,4.4,0.0,29879.54\nR001,2003-11-29,4.0,0.0,30658.55\nR001,2003-11-30,2.2,0.0,29959.47\nR001,2003-12-01,1.0,0.1,30316.01\nR001,2003-12-02,1.3,0.0,30333.35\nR001,2003-12-03,0.8,0.0,30349.29\nR001,2003-12-04,0.4,0.0,30363.78\nR001,2003-12-05,-0.3,0.4,25228.27\nR001,2003-12-06,2.7,3.6,20047.8\nR001,2003-12-07,3.1,0.1,28860.92\nR001,2003-12-08,2.3,18.6,0.0\nR001,2003-12-09,-1.7,3.5,20721.94\nR001,2003-12-10,-5.2,0.0,30804.98\nR001,2003-12-11,-6.4,0.0,30422.0\nR001,2003-12-12,-2.8,0.0,29261.09\nR001,2003-12-13,-0.2,0.0,28169.72\nR001,2003-12-14,1.4,3.5,12509.92\nR001,2003-12-15,3.5,0.0,29251.91\nR001,2003-12-16,5.0,2.2,20295.57\nR001,2003-12-17,5.5,7.2,18000.0\nR001,2003-12-18,3.2,26.1,0.0\nR001,2003-12-19,-1.6,0.3,18272.26\nR001,2003-12-20,-2.0,2.7,1208.38\nR001,2003-12-21,-0.6,0.0,25493.32\nR001,2003-12-22,-1.0,0.0,30745.76\nR001,2003-12-23,-2.5,0.0,25858.22\nR001,2003-12-24,-2.8,0.0,31233.12\nR001,2003-12-25,-1.6,0.0,31268.75\nR001,2003-12-26,1.9,0.0,19106.86\nR001,2003-12-27,2.2,1.4,5751.77\nR001,2003-12-28,0.7,0.0,30245.82\nR001,2003-12-29,-1.3,0.0,31429.13\nR001,2003-12-30,-2.3,0.0,31474.58\nR001,2003-12-31,-2.6,0.0,30563.99\nR001,2004-01-01,-1.6,0.0,29755.52\nR001,2004-01-02,-0.9,0.0,28946.84\nR001,2004-01-03,0.2,0.0,31681.96\nR001,2004-01-04,1.7,1.6,18437.66\nR001,2004-01-05,3.5,7.0,0.0\nR001,2004-01-06,1.8,8.1,27163.83\nR001,2004-01-07,-0.6,4.1,4349.72\nR001,2004-01-08,-2.7,11.8,12673.33\nR001,2004-01-09,-6.2,1.1,29491.63\nR001,2004-01-10,-6.5,0.0,32121.96\nR001,2004-01-11,-8.0,0.0,32190.57\nR001,2004-01-12,-3.5,0.0,28800.0\nR001,2004-01-13,1.1,0.0,32330.72\nR001,2004-01-14,-0.1,0.0,32400.0\nR001,2004-01-15,0.9,5.2,6151.0\nR001,2004-01-16,5.3,13.2,4100.34\nR001,2004-01-17,0.9,23.0,5280.36\nR001,2004-01-18,-2.4,0.0,32400.0\nR001,2004-01-19,-1.4,0.0,32400.0\nR001,2004-01-20,1.2,0.0,27919.95\nR001,2004-01-21,2.6,0.0,17027.6\nR001,2004-01-22,4.1,0.0,18938.05\nR001,2004-01-23,4.9,6.6,29453.75\nR001,2004-01-24,1.9,14.2,21600.0\nR001,2004-01-25,-0.0,5.4,31862.26\nR001,2004-01-26,1.2,0.0,32400.0\nR001,2004-01-27,0.8,0.0,24501.43\nR001,2004-01-28,0.6,0.0,32400.0\nR001,2004-01-29,3.7,0.0,26094.71\nR001,2004-01-30,4.0,0.0,32400.0\nR001,2004-01-31,3.0,2.0,18378.37\nR001,2004-02-01,1.5,13.3,1122.17\nR001,2004-02-02,0.0,0.9,10226.03\nR001,2004-02-03,-1.6,1.9,15360.21\nR001,2004-02-04,-2.1,12.1,17938.0\nR001,2004-02-05,-2.8,0.6,26939.87\nR001,2004-02-06,-0.6,15.2,1563.89\nR001,2004-02-07,1.6,14.1,11121.27\nR001,2004-02-08,5.0,5.6,28875.83\nR001,2004-02-09,3.3,8.3,24414.3\nR001,2004-02-10,-0.1,31.6,0.0\nR001,2004-02-11,-5.2,4.2,34760.7\nR001,2004-02-12,-5.4,0.0,29048.15\nR001,2004-02-13,0.2,1.9,19596.0\nR001,2004-02-14,-0.5,16.9,3800.88\nR001,2004-02-15,-4.3,6.6,19210.41\nR001,2004-02-16,-6.8,2.3,33655.85\nR001,2004-02-17,-4.4,7.6,26016.52\nR001,2004-02-18,-3.4,3.1,35351.41\nR001,2004-02-19,-3.7,0.0,36389.46\nR001,2004-02-20,-2.9,12.4,955.51\nR001,2004-02-21,-4.4,16.7,2175.94\nR001,2004-02-22,-4.3,0.5,2900.3\nR001,2004-02-23,-2.1,0.0,34804.37\nR001,2004-02-24,-0.1,0.1,38715.35\nR001,2004-02-25,3.2,7.1,17080.41\nR001,2004-02-26,4.0,4.3,32270.75\nR001,2004-02-27,6.2,0.0,32348.99\nR001,2004-02-28,8.6,2.2,31589.06\nR001,2004-02-29,8.8,0.3,32312.43\nR001,2004-03-01,6.3,10.0,5057.54\nR001,2004-03-02,4.8,0.2,39600.0\nR001,2004-03-03,4.8,2.8,18889.5\nR001,2004-03-04,5.3,13.2,0.0\nR001,2004-03-05,2.6,59.5,0.0\nR001,2004-03-06,-3.0,9.4,27937.01\nR001,2004-03-07,-6.8,0.0,38813.61\nR001,2004-03-08,-4.0,0.0,39600.0\nR001,2004-03-09,1.2,0.0,39600.0\nR001,2004-03-10,3.1,0.0,31621.59\nR001,2004-03-11,3.3,6.9,870.91\nR001,2004-03-12,1.9,4.3,29333.06\nR001,2004-03-13,1.0,4.6,23965.03\nR001,2004-03-14,-0.5,1.2,37878.95\nR001,2004-03-15,-0.4,0.0,39600.0\nR001,2004-03-16,-1.2,4.1,34366.15\nR001,2004-03-17,-0.6,2.0,39600.0\nR001,2004-03-18,0.1,0.0,39600.0\nR001,2004-03-19,0.4,0.5,34204.36\nR001,2004-03-20,2.9,0.3,21742.2\nR001,2004-03-21,5.8,0.6,40136.67\nR001,2004-03-22,8.5,0.0,39764.69\nR001,2004-03-23,9.9,0.0,42594.08\nR001,2004-03-24,10.2,0.0,40534.09\nR001,2004-03-25,10.8,0.0,43058.52\nR001,2004-03-26,6.1,6.8,40500.67\nR001,2004-03-27,7.3,0.7,2538.81\nR001,2004-03-28,9.8,0.8,22545.2\nR001,2004-03-29,7.5,13.5,30546.99\nR001,2004-03-30,1.7,17.9,35603.93\nR001,2004-03-31,1.2,5.1,27173.19\nR001,2004-04-01,2.0,0.0,42080.0\nR001,2004-04-02,0.1,0.8,45246.31\nR001,2004-04-03,-7.3,5.4,34288.93\nR001,2004-04-04,-7.9,0.3,43450.6\nR001,2004-04-05,-4.0,0.0,44142.91\nR001,2004-04-06,1.6,0.0,44343.08\nR001,2004-04-07,6.6,0.3,44214.75\nR001,2004-04-08,8.9,0.0,46179.01\nR001,2004-04-09,11.3,0.0,46250.27\nR001,2004-04-10,11.6,0.0,44212.54\nR001,2004-04-11,13.5,0.0,39295.43\nR001,2004-04-12,15.4,0.0,46464.5\nR001,2004-04-13,13.3,0.5,19064.77\nR001,2004-04-14,12.6,4.1,5411.0\nR001,2004-04-15,6.2,30.9,10638.07\nR001,2004-04-16,5.8,0.0,46500.13\nR001,2004-04-17,3.7,11.7,2375.7\nR001,2004-04-18,5.1,0.2,45889.15\nR001,2004-04-19,6.9,0.0,46800.0\nR001,2004-04-20,8.6,0.0,46800.0\nR001,2004-04-21,10.3,1.1,44755.67\nR001,2004-04-22,9.6,12.7,23079.08\nR001,2004-04-23,8.5,13.1,12825.73\nR001,2004-04-24,7.0,8.1,41520.43\nR001,2004-04-25,6.5,3.7,32898.82\nR001,2004-04-26,8.5,0.6,44985.19\nR001,2004-04-27,12.4,0.0,47981.8\nR001,2004-04-28,13.0,4.5,44566.57\nR001,2004-04-29,11.1,6.9,45175.21\nR001,2004-04-30,9.0,20.4,0.0\nR001,2004-05-01,9.1,5.3,20113.1\nR001,2004-05-02,11.0,0.0,46800.0\nR001,2004-05-03,12.6,0.0,47112.6\nR001,2004-05-04,12.6,0.2,47388.07\nR001,2004-05-05,12.6,0.2,48366.16\nR001,2004-05-06,14.1,0.0,37706.55\nR001,2004-05-07,14.8,2.0,43610.32\nR001,2004-05-08,15.2,0.8,41313.5\nR001,2004-05-09,14.5,0.8,47243.89\nR001,2004-05-10,14.3,2.2,48554.16\nR001,2004-05-11,11.7,11.7,24448.79\nR001,2004-05-12,9.0,46.4,7279.67\nR001,2004-05-13,10.2,17.5,41938.73\nR001,2004-05-14,10.9,0.2,47985.36\nR001,2004-05-15,11.5,3.5,40725.54\nR001,2004-05-16,10.2,12.8,12991.15\nR001,2004-05-17,8.2,11.7,3208.08\nR001,2004-05-18,10.1,0.0,43625.6\nR001,2004-05-19,6.2,9.0,26944.0\nR001,2004-05-20,6.3,0.7,30110.99\nR001,2004-05-21,10.4,0.0,46036.57\nR001,2004-05-22,12.8,0.1,49128.88\nR001,2004-05-23,15.3,0.7,50605.03\nR001,2004-05-24,15.9,0.5,50936.26\nR001,2004-05-25,15.9,2.0,50273.42\nR001,2004-05-26,11.6,7.5,0.0\nR001,2004-05-27,12.9,7.9,8025.22\nR001,2004-05-28,12.9,11.9,2882.76\nR001,2004-05-29,13.9,4.7,34411.2\nR001,2004-05-30,13.7,9.4,16445.53\nR001,2004-05-31,16.4,2.8,48533.99\nR001,2004-06-01,13.7,19.4,33529.5\nR001,2004-06-02,12.2,10.5,26199.83\nR001,2004-06-03,13.2,8.3,32166.9\nR001,2004-06-04,12.9,1.0,44710.64\nR001,2004-06-05,15.3,0.0,51728.46\nR001,2004-06-06,18.3,0.3,43425.9\nR001,2004-06-07,16.7,11.4,40491.97\nR001,2004-06-08,15.3,9.5,45661.56\nR001,2004-06-09,13.2,11.3,9113.65\nR001,2004-06-10,11.2,9.4,34705.64\nR001,2004-06-11,8.9,0.7,50686.95\nR001,2004-06-12,11.9,0.0,53130.1\nR001,2004-06-13,16.1,0.0,53118.15\nR001,2004-06-14,18.1,0.0,53473.98\nR001,2004-06-15,17.1,0.8,52360.77\nR001,2004-06-16,17.9,0.5,51221.24\nR001,2004-06-17,17.0,4.0,46745.19\nR001,2004-06-18,17.4,0.1,50678.36\nR001,2004-06-19,19.1,0.6,53425.69\nR001,2004-06-20,18.8,11.3,45788.89\nR001,2004-06-21,18.6,0.9,49315.17\nR001,2004-06-22,18.4,6.8,51258.54\nR001,2004-06-23,17.3,9.2,38490.03\nR001,2004-06-24,17.6,17.2,40296.87\nR001,2004-06-25,17.7,12.3,39823.76\nR001,2004-06-26,18.2,8.7,45612.08\nR001,2004-06-27,17.9,5.2,47640.53\nR001,2004-06-28,16.6,6.2,42351.46\nR001,2004-06-29,14.4,2.5,52755.14\nR001,2004-06-30,15.3,2.2,52283.94\nR001,2004-07-01,13.7,5.9,46120.33\nR001,2004-07-02,13.3,0.0,51583.69\nR001,2004-07-03,15.5,0.0,53476.23\nR001,2004-07-04,17.6,0.8,50835.83\nR001,2004-07-05,18.6,2.3,42012.98\nR001,2004-07-06,19.5,0.0,52716.79\nR001,2004-07-07,20.4,0.2,53034.1\nR001,2004-07-08,20.4,0.4,49869.06\nR001,2004-07-09,18.6,2.5,51590.42\nR001,2004-07-10,17.9,0.0,51317.28\nR001,2004-07-11,17.1,1.0,52701.15\nR001,2004-07-12,18.2,1.9,50710.28\nR001,2004-07-13,19.1,4.5,53662.15\nR001,2004-07-14,20.1,0.8,52732.83\nR001,2004-07-15,17.7,7.0,39792.92\nR001,2004-07-16,17.8,19.4,20092.16\nR001,2004-07-17,16.6,19.3,29511.95\nR001,2004-07-18,17.4,18.4,27984.73\nR001,2004-07-19,17.8,2.3,44893.8\nR001,2004-07-20,18.3,0.0,49968.8\nR001,2004-07-21,18.5,0.3,51725.78\nR001,2004-07-22,16.9,3.6,47899.67\nR001,2004-07-23,14.9,15.4,40994.51\nR001,2004-07-24,14.9,2.9,45245.74\nR001,2004-07-25,16.7,0.0,50137.98\nR001,2004-07-26,17.8,0.0,49137.26\nR001,2004-07-27,19.3,0.4,50752.5\nR001,2004-07-28,20.6,0.0,51065.9\nR001,2004-07-29,21.5,0.0,50335.95\nR001,2004-07-30,21.9,1.1,50383.08\nR001,2004-07-31,22.1,0.3,49399.21\nR001,2004-08-01,22.7,0.2,48765.25\nR001,2004-08-02,22.6,0.9,49166.47\nR001,2004-08-03,19.3,15.0,37771.28\nR001,2004-08-04,17.8,6.3,35760.92\nR001,2004-08-05,18.3,1.6,46883.79\nR001,2004-08-06,19.9,0.8,48614.56\nR001,2004-08-07,20.2,2.3,47960.71\nR001,2004-08-08,19.3,3.7,46971.43\nR001,2004-08-09,19.3,3.5,47339.1\nR001,2004-08-10,20.5,0.0,46800.0\nR001,2004-08-11,18.1,16.9,20034.92\nR001,2004-08-12,16.1,5.9,19459.52\nR001,2004-08-13,17.2,4.8,40807.81\nR001,2004-08-14,19.1,0.0,46800.0\nR001,2004-08-15,20.4,0.0,47182.76\nR001,2004-08-16,21.3,1.3,46880.8\nR001,2004-08-17,23.0,0.3,46912.3\nR001,2004-08-18,21.4,9.5,41609.56\nR001,2004-08-19,18.6,29.1,21991.01\nR001,2004-08-20,18.6,9.1,37850.12\nR001,2004-08-21,18.4,3.6,36837.97\nR001,2004-08-22,18.7,15.3,40325.62\nR001,2004-08-23,19.4,1.1,46419.96\nR001,2004-08-24,18.5,3.3,45487.95\nR001,2004-08-25,17.6,1.2,40199.99\nR001,2004-08-26,18.1,0.0,44997.03\nR001,2004-08-27,19.1,0.2,46705.57\nR001,2004-08-28,21.2,0.0,46603.44\nR001,2004-08-29,20.4,6.6,45099.53\nR001,2004-08-30,18.6,9.2,41580.25\nR001,2004-08-31,18.7,3.2,43907.6\nR001,2004-09-01,18.6,2.6,43592.38\nR001,2004-09-02,18.9,3.2,41975.09\nR001,2004-09-03,17.3,1.9,36175.41\nR001,2004-09-04,16.6,0.4,42869.96\nR001,2004-09-05,14.6,3.0,42163.65\nR001,2004-09-06,12.4,3.0,35899.92\nR001,2004-09-07,11.5,3.0,41487.72\nR001,2004-09-08,12.4,0.0,42089.32\nR001,2004-09-09,14.3,0.9,40288.38\nR001,2004-09-10,9.6,16.1,23738.42\nR001,2004-09-11,9.2,0.8,40202.23\nR001,2004-09-12,10.6,0.0,42838.87\nR001,2004-09-13,11.5,0.0,43154.64\nR001,2004-09-14,13.5,0.0,43087.46\nR001,2004-09-15,14.5,0.0,43020.19\nR001,2004-09-16,15.8,0.0,42952.8\nR001,2004-09-17,15.0,0.5,38495.64\nR001,2004-09-18,14.0,0.4,39900.86\nR001,2004-09-19,15.7,0.0,40889.58\nR001,2004-09-20,17.2,0.0,42681.81\nR001,2004-09-21,16.6,0.0,42231.59\nR001,2004-09-22,16.9,0.0,42545.41\nR001,2004-09-23,16.7,2.9,41665.01\nR001,2004-09-24,17.4,0.1,39600.0\nR001,2004-09-25,20.3,0.0,38371.71\nR001,2004-09-26,20.8,0.0,40497.11\nR001,2004-09-27,18.9,0.0,39600.0\nR001,2004-09-28,18.5,0.0,41207.71\nR001,2004-09-29,17.5,0.0,39568.66\nR001,2004-09-30,18.8,0.0,40454.97\nR001,2004-10-01,16.3,1.4,39286.78\nR001,2004-10-02,13.5,0.0,38016.09\nR001,2004-10-03,10.4,1.2,34472.93\nR001,2004-10-04,9.1,0.0,35615.26\nR001,2004-10-05,9.4,0.0,36818.06\nR001,2004-10-06,9.9,0.0,39078.57\nR001,2004-10-07,13.2,0.0,38562.65\nR001,2004-10-08,15.4,0.0,37687.54\nR001,2004-10-09,13.3,10.0,18981.68\nR001,2004-10-10,12.2,2.9,30840.27\nR001,2004-10-11,13.1,0.0,36176.16\nR001,2004-10-12,10.8,22.2,844.48\nR001,2004-10-13,3.9,0.0,34709.62\nR001,2004-10-14,6.0,0.0,36338.06\nR001,2004-10-15,8.2,1.1,32036.47\nR001,2004-10-16,11.0,0.6,19784.65\nR001,2004-10-17,14.0,0.0,34186.62\nR001,2004-10-18,15.5,0.0,36000.0\nR001,2004-10-19,14.8,0.7,31903.68\nR001,2004-10-20,12.5,6.7,8378.47\nR001,2004-10-21,7.9,24.8,32174.23\nR001,2004-10-22,4.5,1.4,36000.0\nR001,2004-10-23,4.3,0.0,36000.0\nR001,2004-10-24,7.8,0.0,34146.07\nR001,2004-10-25,10.3,0.0,36000.0\nR001,2004-10-26,11.1,0.0,36000.0\nR001,2004-10-27,13.9,0.0,36000.0\nR001,2004-10-28,14.6,0.9,35231.3\nR001,2004-10-29,12.7,1.6,36000.0\nR001,2004-10-30,12.0,0.2,33287.45\nR001,2004-10-31,13.0,0.0,36000.0\nR001,2004-11-01,11.3,0.1,30937.67\nR001,2004-11-02,12.0,3.4,31474.01\nR001,2004-11-03,8.2,15.6,8167.77\nR001,2004-11-04,5.6,11.3,30329.43\nR001,2004-11-05,3.3,0.0,35634.23\nR001,2004-11-06,5.6,0.0,35478.27\nR001,2004-11-07,6.9,0.0,35324.3\nR001,2004-11-08,8.8,0.0,35172.51\nR001,2004-11-09,8.9,0.0,32464.88\nR001,2004-11-10,10.1,0.0,33421.33\nR001,2004-11-11,9.5,0.0,34732.18\nR001,2004-11-12,9.4,0.0,34591.09\nR001,2004-11-13,9.0,0.0,31381.0\nR001,2004-11-14,9.0,0.0,34082.0\nR001,2004-11-15,11.0,0.0,30505.91\nR001,2004-11-16,9.2,14.7,0.0\nR001,2004-11-17,5.9,17.1,666.37\nR001,2004-11-18,1.4,0.0,32235.32\nR001,2004-11-19,4.1,0.0,32594.73\nR001,2004-11-20,6.7,0.5,31216.58\nR001,2004-11-21,1.8,56.9,0.0\nR001,2004-11-22,-2.7,26.1,0.0\nR001,2004-11-23,-3.8,4.3,21231.31\nR001,2004-11-24,-3.0,4.3,19733.32\nR001,2004-11-25,0.9,15.9,0.0\nR001,2004-11-26,0.4,17.2,0.0\nR001,2004-11-27,-2.5,2.8,21982.34\nR001,2004-11-28,0.8,35.2,0.0\nR001,2004-11-29,1.4,17.6,3792.67\nR001,2004-11-30,2.1,21.1,506.92\nR001,2004-12-01,-0.2,13.0,28083.21\nR001,2004-12-02,-1.1,0.0,30347.4\nR001,2004-12-03,2.0,15.1,3618.27\nR001,2004-12-04,5.0,25.8,10791.14\nR001,2004-12-05,0.6,1.4,28375.84\nR001,2004-12-06,-0.7,0.0,22811.08\nR001,2004-12-07,0.2,2.3,27656.51\nR001,2004-12-08,-1.2,0.2,27735.73\nR001,2004-12-09,-1.8,12.6,1367.71\nR001,2004-12-10,-2.0,6.6,25439.11\nR001,2004-12-11,-3.2,9.9,17389.47\nR001,2004-12-12,-6.3,1.5,27803.22\nR001,2004-12-13,-2.4,15.7,1212.5\nR001,2004-12-14,-6.5,1.8,27745.56\nR001,2004-12-15,-10.0,0.0,30413.47\nR001,2004-12-16,-10.1,0.6,29238.54\nR001,2004-12-17,-6.5,0.1,30011.1\nR001,2004-12-18,-5.9,0.0,30783.64\nR001,2004-12-19,-4.7,0.0,30774.29\nR001,2004-12-20,-4.0,0.0,30763.48\nR001,2004-12-21,0.1,4.0,0.0\nR001,2004-12-22,1.5,0.9,12956.1\nR001,2004-12-23,1.4,6.2,17.06\nR001,2004-12-24,0.6,4.2,23738.73\nR001,2004-12-25,1.2,1.4,26802.14\nR001,2004-12-26,-0.8,0.0,31329.46\nR001,2004-12-27,2.5,0.0,31035.37\nR001,2004-12-28,4.1,0.0,30620.26\nR001,2004-12-29,5.5,0.0,22522.46\nR001,2004-12-30,7.6,0.0,29005.31\nR001,2004-12-31,4.5,2.8,1409.55\nR001,2005-01-01,3.6,0.3,30900.18\nR001,2005-01-02,2.8,1.9,2761.21\nR001,2005-01-03,0.7,0.7,29185.21\nR001,2005-01-04,-0.4,1.4,4531.69\nR001,2005-01-05,-1.5,8.6,16622.51\nR001,2005-01-06,-2.9,2.3,23550.69\nR001,2005-01-07,-3.3,0.0,31979.14\nR001,2005-01-08,-3.1,0.1,30658.35\nR001,2005-01-09,0.9,0.0,28508.49\nR001,2005-01-10,-0.9,4.2,15589.92\nR001,2005-01-11,-3.0,4.8,10763.38\nR001,2005-01-12,1.5,4.7,26413.01\nR001,2005-01-13,-0.0,0.0,32377.44\nR001,2005-01-14,0.4,0.0,32400.0\nR001,2005-01-15,1.2,0.0,28800.0\nR001,2005-01-16,-0.2,0.0,32400.0\nR001,2005-01-17,-0.4,0.0,32400.0\nR001,2005-01-18,-0.9,0.0,32400.0\nR001,2005-01-19,-1.8,0.0,32400.0\nR001,2005-01-20,-2.4,0.0,32400.0\nR001,2005-01-21,-0.7,0.0,28800.0\nR001,2005-01-22,3.3,0.0,32400.0\nR001,2005-01-23,2.0,2.4,24795.66\nR001,2005-01-24,-1.2,13.7,0.0\nR001,2005-01-25,-4.0,0.0,30714.78\nR001,2005-01-26,1.3,3.4,6725.67\nR001,2005-01-27,2.6,0.0,32400.0\nR001,2005-01-28,5.2,0.3,26635.84\nR001,2005-01-29,4.1,4.0,5490.85\nR001,2005-01-30,6.1,6.5,5074.85\nR001,2005-01-31,1.2,20.8,0.0\nR001,2005-02-01,-1.6,0.0,28643.65\nR001,2005-02-02,-2.2,0.0,30516.95\nR001,2005-02-03,-2.3,0.0,32623.54\nR001,2005-02-04,-3.3,0.0,33644.58\nR001,2005-02-05,-4.6,0.0,33229.68\nR001,2005-02-06,-5.5,0.0,33529.68\nR001,2005-02-07,-8.4,2.1,33730.08\nR001,2005-02-08,-7.6,4.0,29651.19\nR001,2005-02-09,-8.0,0.4,34579.65\nR001,2005-02-10,-6.1,3.0,27407.61\nR001,2005-02-11,-6.6,1.3,34050.87\nR001,2005-02-12,-7.3,0.0,34297.34\nR001,2005-02-13,-6.3,0.0,36222.72\nR001,2005-02-14,-1.9,0.0,30909.13\nR001,2005-02-15,3.5,0.0,36964.64\nR001,2005-02-16,5.7,0.0,35601.12\nR001,2005-02-17,8.1,0.0,35681.1\nR001,2005-02-18,7.5,7.6,21600.0\nR001,2005-02-19,3.7,24.6,0.0\nR001,2005-02-20,1.9,0.3,33247.1\nR001,2005-02-21,0.2,10.8,23547.39\nR001,2005-02-22,-0.2,0.0,38487.19\nR001,2005-02-23,0.5,0.0,38661.09\nR001,2005-02-24,1.4,0.0,36763.46\nR001,2005-02-25,2.4,3.2,29349.58\nR001,2005-02-26,4.5,3.3,29981.3\nR001,2005-02-27,5.1,7.8,6349.27\nR001,2005-02-28,2.4,23.5,5919.81\nR001,2005-03-01,-0.6,17.3,0.0\nR001,2005-03-02,1.9,26.5,0.0\nR001,2005-03-03,5.6,25.3,0.0\nR001,2005-03-04,1.7,16.8,13618.55\nR001,2005-03-05,2.3,0.0,36000.0\nR001,2005-03-06,5.2,2.5,13296.22\nR001,2005-03-07,4.5,8.9,34714.31\nR001,2005-03-08,4.1,10.4,8660.73\nR001,2005-03-09,0.6,23.6,22794.01\nR001,2005-03-10,-1.9,6.8,36570.8\nR001,2005-03-11,-2.6,4.7,31757.4\nR001,2005-03-12,-2.9,0.0,39391.25\nR001,2005-03-13,1.1,0.0,39628.2\nR001,2005-03-14,2.9,3.8,36758.08\nR001,2005-03-15,2.6,6.3,35114.01\nR001,2005-03-16,0.3,1.9,34366.05\nR001,2005-03-17,0.3,5.0,31232.03\nR001,2005-03-18,3.2,10.5,6757.26\nR001,2005-03-19,3.0,49.3,0.0\nR001,2005-03-20,-5.1,7.7,39579.12\nR001,2005-03-21,-6.6,2.2,39949.24\nR001,2005-03-22,-5.4,0.0,39999.56\nR001,2005-03-23,-5.1,1.6,36450.34\nR001,2005-03-24,-5.4,5.8,30688.67\nR001,2005-03-25,-5.4,0.0,41809.28\nR001,2005-03-26,-1.8,0.0,34645.79\nR001,2005-03-27,3.3,0.0,41728.29\nR001,2005-03-28,5.2,7.6,29566.66\nR001,2005-03-29,4.3,37.7,0.0\nR001,2005-03-30,4.2,4.7,22862.11\nR001,2005-03-31,2.7,0.2,31968.67\nR001,2005-04-01,1.9,7.1,6215.57\nR001,2005-04-02,2.0,1.7,32929.0\nR001,2005-04-03,1.8,7.5,0.0\nR001,2005-04-04,2.6,3.0,38432.79\nR001,2005-04-05,2.9,0.0,43745.0\nR001,2005-04-06,4.3,0.0,41637.86\nR001,2005-04-07,4.8,0.0,44959.15\nR001,2005-04-08,7.2,0.0,46161.83\nR001,2005-04-09,8.5,0.0,46233.38\nR001,2005-04-10,10.0,0.0,46304.92\nR001,2005-04-11,9.3,0.0,46376.45\nR001,2005-04-12,10.2,0.0,46447.94\nR001,2005-04-13,12.4,0.0,46519.39\nR001,2005-04-14,12.4,0.0,38301.38\nR001,2005-04-15,9.4,10.2,9787.07\nR001,2005-04-16,8.4,5.1,38103.6\nR001,2005-04-17,8.4,10.5,29045.47\nR001,2005-04-18,11.9,1.5,46800.0\nR001,2005-04-19,9.9,28.2,19614.83\nR001,2005-04-20,8.3,4.4,37943.08\nR001,2005-04-21,11.7,0.0,46800.0\nR001,2005-04-22,14.6,0.7,34411.29\nR001,2005-04-23,13.2,7.9,32664.37\nR001,2005-04-24,8.9,35.5,0.0\nR001,2005-04-25,8.0,0.0,2352.53\nR001,2005-04-26,9.6,0.0,41113.58\nR001,2005-04-27,10.9,0.0,46800.0\nR001,2005-04-28,8.4,4.5,24311.64\nR001,2005-04-29,7.0,7.1,39108.8\nR001,2005-04-30,9.4,3.0,43552.54\nR001,2005-05-01,8.6,6.0,42324.64\nR001,2005-05-02,7.6,1.3,33625.67\nR001,2005-05-03,7.1,6.3,25016.09\nR001,2005-05-04,7.1,10.4,0.0\nR001,2005-05-05,8.2,7.8,33329.34\nR001,2005-05-06,10.7,0.0,49356.95\nR001,2005-05-07,15.4,0.3,41010.75\nR001,2005-05-08,15.5,4.3,34189.28\nR001,2005-05-09,11.1,14.1,5782.31\nR001,2005-05-10,12.1,0.0,48559.57\nR001,2005-05-11,14.1,0.2,49904.69\nR001,2005-05-12,15.5,0.9,50266.26\nR001,2005-05-13,14.6,19.7,36224.52\nR001,2005-05-14,10.6,27.4,319.79\nR001,2005-05-15,11.1,0.5,45457.61\nR001,2005-05-16,12.0,1.1,45713.94\nR001,2005-05-17,13.6,0.1,45798.74\nR001,2005-05-18,15.1,0.6,48071.09\nR001,2005-05-19,17.1,0.9,50089.65\nR001,2005-05-20,17.2,3.9,50400.0\nR001,2005-05-21,18.2,2.4,46800.0\nR001,2005-05-22,20.7,1.6,47754.1\nR001,2005-05-23,19.9,5.7,49376.48\nR001,2005-05-24,18.1,8.6,46238.3\nR001,2005-05-25,15.3,25.8,19629.76\nR001,2005-05-26,15.0,18.5,36187.74\nR001,2005-05-27,14.3,8.6,29032.23\nR001,2005-05-28,15.9,7.9,42506.75\nR001,2005-05-29,17.9,2.4,50821.2\nR001,2005-05-30,18.2,6.8,48227.75\nR001,2005-05-31,17.5,12.8,40263.72\nR001,2005-06-01,17.1,6.4,41065.79\nR001,2005-06-02,15.4,7.0,27498.01\nR001,2005-06-03,13.8,6.3,45672.76\nR001,2005-06-04,12.7,1.2,45964.06\nR001,2005-06-05,13.0,1.8,50144.39\nR001,2005-06-06,12.8,8.7,7721.09\nR001,2005-06-07,15.5,0.6,45997.55\nR001,2005-06-08,17.7,0.0,50962.19\nR001,2005-06-09,19.9,0.4,51254.28\nR001,2005-06-10,19.9,0.7,52790.74\nR001,2005-06-11,20.0,13.6,50379.38\nR001,2005-06-12,16.7,12.8,43883.39\nR001,2005-06-13,14.8,0.0,49118.08\nR001,2005-06-14,16.1,0.8,50555.89\nR001,2005-06-15,16.3,11.2,50228.48\nR001,2005-06-16,14.8,18.1,20166.05\nR001,2005-06-17,14.4,3.6,45968.56\nR001,2005-06-18,16.2,0.0,53067.89\nR001,2005-06-19,16.4,3.2,52324.34\nR001,2005-06-20,13.5,8.9,44973.83\nR001,2005-06-21,12.1,4.8,39702.74\nR001,2005-06-22,11.1,5.9,42656.64\nR001,2005-06-23,11.7,7.2,45835.64\nR001,2005-06-24,14.5,0.0,53398.43\nR001,2005-06-25,15.6,1.9,48996.94\nR001,2005-06-26,16.2,7.9,48154.91\nR001,2005-06-27,16.3,3.0,51338.83\nR001,2005-06-28,17.6,0.6,47614.34\nR001,2005-06-29,17.6,1.9,51892.52\nR001,2005-06-30,18.1,1.1,51311.01\nR001,2005-07-01,18.8,0.0,50594.72\nR001,2005-07-02,20.4,0.9,53746.65\nR001,2005-07-03,20.9,1.9,53921.41\nR001,2005-07-04,22.7,0.3,53384.07\nR001,2005-07-05,19.8,21.0,38734.02\nR001,2005-07-06,16.8,6.0,46670.57\nR001,2005-07-07,17.2,0.0,51942.24\nR001,2005-07-08,19.1,0.0,53761.68\nR001,2005-07-09,20.2,0.2,53043.31\nR001,2005-07-10,20.4,0.6,51420.55\nR001,2005-07-11,19.5,5.2,49702.75\nR001,2005-07-12,19.5,0.7,52051.15\nR001,2005-07-13,20.6,0.1,48964.97\nR001,2005-07-14,21.3,1.2,51757.97\nR001,2005-07-15,20.4,3.2,50807.4\nR001,2005-07-16,19.5,17.0,42314.96\nR001,2005-07-17,17.5,10.5,43274.39\nR001,2005-07-18,18.4,0.0,51717.48\nR001,2005-07-19,18.4,0.0,49849.8\nR001,2005-07-20,19.3,0.1,51326.53\nR001,2005-07-21,20.0,1.0,50659.55\nR001,2005-07-22,20.2,0.5,52346.7\nR001,2005-07-23,21.4,0.6,52307.59\nR001,2005-07-24,22.5,0.6,51917.11\nR001,2005-07-25,20.5,6.4,48026.35\nR001,2005-07-26,20.3,8.3,45214.93\nR001,2005-07-27,21.2,0.0,47432.1\nR001,2005-07-28,22.5,0.0,48295.62\nR001,2005-07-29,22.3,0.2,46895.67\nR001,2005-07-30,22.2,0.2,46704.54\nR001,2005-07-31,22.4,0.4,49037.61\nR001,2005-08-01,21.7,3.4,42462.47\nR001,2005-08-02,19.2,9.2,39976.61\nR001,2005-08-03,18.2,9.5,40780.01\nR001,2005-08-04,18.8,14.2,46753.35\nR001,2005-08-05,20.0,9.5,47557.81\nR001,2005-08-06,21.9,0.0,47002.0\nR001,2005-08-07,24.4,0.1,48768.48\nR001,2005-08-08,21.9,3.0,46375.31\nR001,2005-08-09,21.2,0.0,45111.38\nR001,2005-08-10,21.9,9.1,42294.99\nR001,2005-08-11,20.1,4.9,37036.6\nR001,2005-08-12,19.8,0.3,45214.6\nR001,2005-08-13,19.3,0.8,46800.0\nR001,2005-08-14,19.8,0.7,46811.35\nR001,2005-08-15,21.2,0.0,47176.22\nR001,2005-08-16,23.0,0.0,47206.83\nR001,2005-08-17,23.8,0.0,46905.34\nR001,2005-08-18,24.6,2.2,46936.63\nR001,2005-08-19,23.4,0.3,46800.0\nR001,2005-08-20,20.3,3.4,46396.67\nR001,2005-08-21,19.1,2.7,13955.22\nR001,2005-08-22,17.3,7.7,35047.91\nR001,2005-08-23,19.1,0.1,44731.54\nR001,2005-08-24,20.9,0.1,46800.0\nR001,2005-08-25,21.1,3.2,46800.0\nR001,2005-08-26,19.8,1.8,45317.37\nR001,2005-08-27,20.2,1.2,43928.95\nR001,2005-08-28,19.5,1.5,44995.03\nR001,2005-08-29,18.3,3.6,43875.34\nR001,2005-08-30,16.8,2.5,43913.83\nR001,2005-08-31,17.1,0.6,45830.21\nR001,2005-09-01,16.2,1.7,43627.26\nR001,2005-09-02,15.4,0.7,42378.5\nR001,2005-09-03,14.1,2.9,42977.71\nR001,2005-09-04,14.5,2.9,36393.57\nR001,2005-09-05,13.1,2.2,42110.27\nR001,2005-09-06,13.1,0.2,43200.0\nR001,2005-09-07,14.0,0.4,43295.51\nR001,2005-09-08,14.9,0.6,42044.08\nR001,2005-09-09,16.4,4.3,43200.0\nR001,2005-09-10,15.9,17.1,36770.92\nR001,2005-09-11,15.1,5.4,37476.52\nR001,2005-09-12,14.2,3.6,33017.74\nR001,2005-09-13,15.0,1.4,37577.02\nR001,2005-09-14,15.5,0.7,38749.41\nR001,2005-09-15,15.9,0.0,38078.22\nR001,2005-09-16,17.5,0.0,42969.08\nR001,2005-09-17,18.1,1.4,31887.35\nR001,2005-09-18,18.4,1.6,41484.15\nR001,2005-09-19,17.3,1.8,39515.08\nR001,2005-09-20,17.0,1.3,41552.13\nR001,2005-09-21,16.6,1.9,42223.26\nR001,2005-09-22,17.4,0.0,41939.78\nR001,2005-09-23,17.8,0.2,41974.98\nR001,2005-09-24,16.7,2.0,41373.35\nR001,2005-09-25,14.9,9.6,1642.01\nR001,2005-09-26,14.0,8.1,23538.93\nR001,2005-09-27,14.2,6.3,35414.94\nR001,2005-09-28,13.7,7.5,37302.21\nR001,2005-09-29,14.0,3.0,35346.51\nR001,2005-09-30,13.5,2.4,13139.8\nR001,2005-10-01,13.5,10.8,27786.54\nR001,2005-10-02,14.5,0.0,39698.57\nR001,2005-10-03,15.7,1.6,19411.79\nR001,2005-10-04,12.9,13.3,11040.52\nR001,2005-10-05,10.3,2.9,31397.47\nR001,2005-10-06,8.2,6.8,36293.82\nR001,2005-10-07,6.2,1.0,29433.9\nR001,2005-10-08,7.5,0.3,31224.9\nR001,2005-10-09,7.1,0.0,36381.66\nR001,2005-10-10,8.1,1.7,35034.75\nR001,2005-10-11,10.0,0.0,36166.83\nR001,2005-10-12,13.7,1.1,36546.6\nR001,2005-10-13,13.2,9.4,27721.98\nR001,2005-10-14,13.5,8.4,21725.54\nR001,2005-10-15,12.5,5.7,16438.36\nR001,2005-10-16,10.7,23.7,11626.81\nR001,2005-10-17,9.6,64.5,4784.69\nR001,2005-10-18,6.6,12.2,0.0\nR001,2005-10-19,5.5,19.1,0.0\nR001,2005-10-20,5.3,0.5,26730.26\nR001,2005-10-21,3.7,4.8,36000.0\nR001,2005-10-22,3.9,0.0,36000.0\nR001,2005-10-23,6.4,0.0,36000.0\nR001,2005-10-24,8.3,0.0,36000.0\nR001,2005-10-25,9.1,3.7,26223.64\nR001,2005-10-26,6.9,0.0,36000.0\nR001,2005-10-27,8.8,7.2,31931.64\nR001,2005-10-28,5.6,13.0,31610.68\nR001,2005-10-29,3.1,3.1,30774.17\nR001,2005-10-30,3.4,0.7,0.0\nR001,2005-10-31,7.3,3.6,32582.26\nR001,2005-11-01,5.0,16.1,15610.48\nR001,2005-11-02,6.2,9.0,26109.85\nR001,2005-11-03,4.4,19.0,3242.09\nR001,2005-11-04,6.2,0.0,33169.51\nR001,2005-11-05,7.2,0.0,31425.39\nR001,2005-11-06,6.7,1.8,22073.51\nR001,2005-11-07,7.5,4.3,15721.13\nR001,2005-11-08,6.6,0.2,33469.72\nR001,2005-11-09,6.0,0.0,35070.49\nR001,2005-11-10,5.3,0.2,32358.96\nR001,2005-11-11,3.9,2.8,28895.27\nR001,2005-11-12,3.1,0.0,34636.03\nR001,2005-11-13,2.8,0.0,34493.85\nR001,2005-11-14,3.5,0.0,34353.25\nR001,2005-11-15,3.7,0.0,34214.45\nR001,2005-11-16,3.1,7.2,17437.8\nR001,2005-11-17,5.5,0.0,33338.36\nR001,2005-11-18,7.5,0.0,29586.81\nR001,2005-11-19,7.9,0.0,15839.8\nR001,2005-11-20,8.0,0.0,24921.45\nR001,2005-11-21,6.3,10.3,4532.84\nR001,2005-11-22,3.0,24.9,10107.41\nR001,2005-11-23,-2.1,0.0,31058.15\nR001,2005-11-24,-1.0,0.0,29377.74\nR001,2005-11-25,4.3,0.0,26767.79\nR001,2005-11-26,7.9,7.1,0.0\nR001,2005-11-27,6.4,2.0,14528.77\nR001,2005-11-28,10.2,1.2,30647.3\nR001,2005-11-29,11.5,0.6,22505.58\nR001,2005-11-30,6.3,6.4,29929.86\nR001,2005-12-01,4.6,0.0,25200.0\nR001,2005-12-02,7.6,11.2,0.0\nR001,2005-12-03,8.0,21.8,0.0\nR001,2005-12-04,6.8,0.0,29986.7\nR001,2005-12-05,6.1,9.2,22838.19\nR001,2005-12-06,5.0,0.0,30390.73\nR001,2005-12-07,8.1,0.0,14791.65\nR001,2005-12-08,10.3,0.0,24387.69\nR001,2005-12-09,5.8,5.5,10800.0\nR001,2005-12-10,3.0,0.4,25737.99\nR001,2005-12-11,4.9,6.8,0.0\nR001,2005-12-12,2.2,0.2,22264.63\nR001,2005-12-13,1.6,0.0,29644.49\nR001,2005-12-14,3.2,0.0,29642.03\nR001,2005-12-15,4.8,0.0,28859.39\nR001,2005-12-16,3.7,11.8,0.0\nR001,2005-12-17,2.0,18.0,15820.81\nR001,2005-12-18,5.8,2.2,30399.23\nR001,2005-12-19,5.5,9.2,24435.02\nR001,2005-12-20,-0.7,22.1,14806.02\nR001,2005-12-21,-5.9,0.0,29974.88\nR001,2005-12-22,-2.3,0.4,28689.81\nR001,2005-12-23,-1.4,7.0,0.0\nR001,2005-12-24,-2.1,7.7,838.3\nR001,2005-12-25,-2.8,0.0,20643.15\nR001,2005-12-26,-3.5,0.0,30668.63\nR001,2005-12-27,-1.1,0.0,31367.43\nR001,2005-12-28,0.2,0.0,18594.34\nR001,2005-12-29,2.6,0.0,31460.13\nR001,2005-12-30,3.6,0.0,31509.35\nR001,2005-12-31,3.2,1.1,23197.22\nR001,2006-01-01,-0.5,7.3,31612.91\nR001,2006-01-02,-1.8,0.0,29705.43\nR001,2006-01-03,1.9,0.0,30816.33\nR001,2006-01-04,2.7,0.0,31779.23\nR001,2006-01-05,3.3,0.0,21274.61\nR001,2006-01-06,3.5,10.4,6234.72\nR001,2006-01-07,2.5,6.8,401.43\nR001,2006-01-08,0.5,4.2,27067.06\nR001,2006-01-09,-0.8,6.4,19924.02\nR001,2006-01-10,-2.5,3.4,22175.39\nR001,2006-01-11,-5.9,0.0,32216.95\nR001,2006-01-12,-3.2,0.0,32286.84\nR001,2006-01-13,-2.2,0.0,32358.31\nR001,2006-01-14,-1.8,1.8,29629.01\nR001,2006-01-15,-2.8,0.2,32400.0\nR001,2006-01-16,-3.1,1.2,32400.0\nR001,2006-01-17,-5.0,4.5,25056.94\nR001,2006-01-18,-6.5,2.5,32344.48\nR001,2006-01-19,-3.5,1.0,26787.13\nR001,2006-01-20,-6.0,27.5,1977.46\nR001,2006-01-21,-12.9,1.7,32400.0\nR001,2006-01-22,-7.0,10.6,1081.84\nR001,2006-01-23,-4.2,7.9,20049.44\nR001,2006-01-24,-8.5,0.0,29278.9\nR001,2006-01-25,-7.5,0.0,25532.37\nR001,2006-01-26,-3.5,6.7,0.0\nR001,2006-01-27,-4.8,5.6,26500.97\nR001,2006-01-28,-5.7,4.6,30184.82\nR001,2006-01-29,-4.7,10.6,20467.74\nR001,2006-01-30,-4.9,12.5,4552.53\nR001,2006-01-31,-4.0,36.1,615.91\nR001,2006-02-01,-3.2,24.1,897.31\nR001,2006-02-02,-1.1,3.4,25761.57\nR001,2006-02-03,0.4,0.0,28077.58\nR001,2006-02-04,-2.4,0.0,32938.43\nR001,2006-02-05,-2.8,0.0,30631.53\nR001,2006-02-06,1.0,0.9,12250.6\nR001,2006-02-07,1.3,25.0,0.0\nR001,2006-02-08,1.3,3.6,8301.35\nR001,2006-02-09,0.3,8.3,0.0\nR001,2006-02-10,-0.9,2.1,34504.67\nR001,2006-02-11,1.7,41.7,7059.31\nR001,2006-02-12,1.3,36.1,0.0\nR001,2006-02-13,-1.2,33.6,0.0\nR001,2006-02-14,-2.4,9.7,539.56\nR001,2006-02-15,-2.6,6.9,0.0\nR001,2006-02-16,-5.2,4.0,29472.78\nR001,2006-02-17,-7.7,0.0,36010.53\nR001,2006-02-18,-0.9,5.4,30556.18\nR001,2006-02-19,0.4,9.1,29211.92\nR001,2006-02-20,1.4,0.0,32687.04\nR001,2006-02-21,3.5,0.0,38266.2\nR001,2006-02-22,2.6,0.0,38442.02\nR001,2006-02-23,4.2,0.0,36693.01\nR001,2006-02-24,4.5,0.0,38796.7\nR001,2006-02-25,4.4,2.6,36947.04\nR001,2006-02-26,4.4,11.8,26357.15\nR001,2006-02-27,3.2,20.9,14516.48\nR001,2006-02-28,3.3,2.3,34289.3\nR001,2006-03-01,3.3,7.8,17662.71\nR001,2006-03-02,2.2,2.1,37706.2\nR001,2006-03-03,2.0,23.1,15229.96\nR001,2006-03-04,-3.1,8.3,35584.22\nR001,2006-03-05,1.9,0.0,39600.0\nR001,2006-03-06,7.0,0.0,39600.0\nR001,2006-03-07,9.1,0.0,37421.92\nR001,2006-03-08,7.7,14.8,15309.44\nR001,2006-03-09,3.2,14.3,5071.29\nR001,2006-03-10,-1.7,2.8,28250.76\nR001,2006-03-11,1.4,0.0,39600.0\nR001,2006-03-12,5.8,0.0,39217.01\nR001,2006-03-13,6.9,0.0,34348.92\nR001,2006-03-14,8.3,0.0,39600.0\nR001,2006-03-15,5.0,3.9,10186.45\nR001,2006-03-16,3.8,3.3,17505.64\nR001,2006-03-17,3.1,1.2,34478.21\nR001,2006-03-18,2.7,2.4,38146.36\nR001,2006-03-19,2.2,8.8,19053.98\nR001,2006-03-20,0.1,11.6,39600.0\nR001,2006-03-21,2.1,0.0,41238.53\nR001,2006-03-22,4.5,0.0,41147.36\nR001,2006-03-23,5.4,0.0,40366.64\nR001,2006-03-24,8.9,0.4,39645.41\nR001,2006-03-25,4.2,8.6,568.79\nR001,2006-03-26,3.2,7.5,7093.92\nR001,2006-03-27,3.4,0.0,18966.19\nR001,2006-03-28,4.8,0.0,40356.39\nR001,2006-03-29,4.6,0.0,42432.49\nR001,2006-03-30,6.5,0.4,37815.08\nR001,2006-03-31,5.8,6.6,9085.14\nR001,2006-04-01,6.0,0.0,43928.68\nR001,2006-04-02,4.5,20.6,8786.49\nR001,2006-04-03,4.2,15.7,0.0\nR001,2006-04-04,4.0,2.9,3790.1\nR001,2006-04-05,4.0,0.0,45931.15\nR001,2006-04-06,4.7,0.0,44678.83\nR001,2006-04-07,6.7,0.5,37238.56\nR001,2006-04-08,11.1,0.5,22595.11\nR001,2006-04-09,12.0,0.9,46186.78\nR001,2006-04-10,11.6,1.0,39600.0\nR001,2006-04-11,11.1,3.6,37199.74\nR001,2006-04-12,12.6,2.2,46097.39\nR001,2006-04-13,13.6,1.5,42136.76\nR001,2006-04-14,14.0,0.3,40622.16\nR001,2006-04-15,10.3,27.3,0.0\nR001,2006-04-16,7.9,3.5,43114.65\nR001,2006-04-17,7.7,0.0,46785.32\nR001,2006-04-18,9.1,1.5,46800.0\nR001,2006-04-19,9.2,0.0,45650.97\nR001,2006-04-20,12.6,0.0,46800.0\nR001,2006-04-21,10.9,5.6,42059.57\nR001,2006-04-22,9.4,7.3,33142.12\nR001,2006-04-23,8.8,3.0,43309.35\nR001,2006-04-24,7.8,5.9,34805.7\nR001,2006-04-25,4.0,7.6,25622.54\nR001,2006-04-26,1.9,9.5,35952.9\nR001,2006-04-27,1.5,5.2,36082.08\nR001,2006-04-28,3.2,2.0,46800.0\nR001,2006-04-29,6.1,0.0,48256.42\nR001,2006-04-30,8.0,1.4,41845.33\nR001,2006-05-01,8.6,0.0,38232.51\nR001,2006-05-02,8.2,5.2,17617.47\nR001,2006-05-03,7.5,8.2,1717.94\nR001,2006-05-04,7.0,2.0,11545.59\nR001,2006-05-05,10.2,0.9,31975.18\nR001,2006-05-06,12.9,6.0,37573.7\nR001,2006-05-07,11.0,17.0,32100.87\nR001,2006-05-08,10.7,12.9,7433.61\nR001,2006-05-09,9.5,5.5,34009.32\nR001,2006-05-10,10.1,0.0,45442.7\nR001,2006-05-11,12.0,0.0,50175.06\nR001,2006-05-12,13.6,3.6,45925.48\nR001,2006-05-13,11.2,7.6,34605.71\nR001,2006-05-14,9.3,8.2,1165.36\nR001,2006-05-15,9.8,3.8,37480.28\nR001,2006-05-16,12.5,0.3,48273.58\nR001,2006-05-17,13.1,8.9,36245.63\nR001,2006-05-18,10.9,13.8,14400.0\nR001,2006-05-19,12.6,2.8,35875.08\nR001,2006-05-20,14.4,2.4,47782.82\nR001,2006-05-21,15.3,2.8,49306.22\nR001,2006-05-22,16.1,0.7,47706.25\nR001,2006-05-23,16.0,0.6,48854.3\nR001,2006-05-24,16.6,0.2,45971.67\nR001,2006-05-25,18.6,0.0,50471.65\nR001,2006-05-26,19.9,0.0,50539.06\nR001,2006-05-27,14.3,28.1,11253.61\nR001,2006-05-28,11.5,37.7,3869.85\nR001,2006-05-29,13.1,1.4,41573.84\nR001,2006-05-30,17.8,0.0,51889.69\nR001,2006-05-31,22.7,0.4,51869.45\nR001,2006-06-01,22.9,0.6,51849.59\nR001,2006-06-02,21.8,1.1,51104.48\nR001,2006-06-03,23.1,0.9,52174.26\nR001,2006-06-04,23.0,2.9,52519.94\nR001,2006-06-05,22.5,0.6,51774.17\nR001,2006-06-06,17.9,11.7,32558.45\nR001,2006-06-07,17.4,6.9,46259.27\nR001,2006-06-08,16.1,10.1,34017.75\nR001,2006-06-09,17.3,4.5,49026.01\nR001,2006-06-10,15.5,12.3,40096.07\nR001,2006-06-11,14.8,0.9,49328.64\nR001,2006-06-12,14.2,8.5,40977.37\nR001,2006-06-13,15.1,0.7,50623.56\nR001,2006-06-14,16.6,8.2,46305.65\nR001,2006-06-15,16.9,5.9,46225.11\nR001,2006-06-16,17.0,7.1,45057.51\nR001,2006-06-17,15.8,0.8,51915.38\nR001,2006-06-18,15.2,0.0,53881.98\nR001,2006-06-19,16.0,0.1,51225.02\nR001,2006-06-20,18.3,0.5,51464.21\nR001,2006-06-21,20.0,1.1,51887.71\nR001,2006-06-22,20.2,3.7,53042.57\nR001,2006-06-23,19.2,7.3,48350.82\nR001,2006-06-24,18.7,8.1,48566.36\nR001,2006-06-25,18.6,10.7,49687.25\nR001,2006-06-26,18.4,10.6,49969.0\nR001,2006-06-27,19.5,2.2,51582.52\nR001,2006-06-28,21.2,0.9,48682.81\nR001,2006-06-29,21.3,8.7,33202.57\nR001,2006-06-30,21.1,1.2,45593.92\nR001,2006-07-01,20.3,4.8,44990.09\nR001,2006-07-02,18.6,4.5,39807.28\nR001,2006-07-03,17.2,0.2,46418.24\nR001,2006-07-04,16.6,0.1,53914.75\nR001,2006-07-05,17.2,0.7,53384.66\nR001,2006-07-06,16.6,6.7,30518.17\nR001,2006-07-07,16.5,19.1,26271.28\nR001,2006-07-08,14.7,34.3,10863.65\nR001,2006-07-09,16.9,2.3,52680.19\nR001,2006-07-10,18.3,2.1,51603.65\nR001,2006-07-11,18.3,5.7,49395.54\nR001,2006-07-12,19.6,3.2,47869.8\nR001,2006-07-13,21.4,0.6,50387.87\nR001,2006-07-14,20.4,14.3,51381.42\nR001,2006-07-15,17.7,7.9,35384.8\nR001,2006-07-16,18.0,1.3,46957.67\nR001,2006-07-17,19.2,0.7,50985.26\nR001,2006-07-18,18.7,1.2,52068.92\nR001,2006-07-19,18.2,0.3,48130.55\nR001,2006-07-20,19.3,0.0,50966.26\nR001,2006-07-21,20.0,0.0,50617.2\nR001,2006-07-22,20.8,0.0,49985.78\nR001,2006-07-23,20.2,4.9,40445.56\nR001,2006-07-24,18.7,31.9,20841.22\nR001,2006-07-25,19.4,1.9,46298.08\nR001,2006-07-26,19.5,0.7,45634.29\nR001,2006-07-27,20.1,0.9,48812.63\nR001,2006-07-28,18.7,0.3,47399.51\nR001,2006-07-29,17.4,1.3,40753.36\nR001,2006-07-30,17.2,1.7,42681.09\nR001,2006-07-31,18.2,0.0,47070.37\nR001,2006-08-01,19.5,0.7,49511.32\nR001,2006-08-02,19.9,0.9,47469.93\nR001,2006-08-03,20.6,0.5,47478.82\nR001,2006-08-04,22.5,0.0,48910.82\nR001,2006-08-05,24.0,0.2,48971.79\nR001,2006-08-06,24.3,0.0,48692.99\nR001,2006-08-07,24.4,0.4,48076.73\nR001,2006-08-08,24.2,0.2,48315.38\nR001,2006-08-09,21.3,4.5,46988.92\nR001,2006-08-10,21.3,0.0,48031.57\nR001,2006-08-11,23.9,0.3,47722.57\nR001,2006-08-12,25.2,0.3,47415.16\nR001,2006-08-13,24.8,0.0,47109.29\nR001,2006-08-14,25.7,0.4,47139.38\nR001,2006-08-15,24.8,1.4,46800.0\nR001,2006-08-16,24.9,1.8,46866.52\nR001,2006-08-17,25.2,0.3,46897.53\nR001,2006-08-18,24.1,1.1,46928.65\nR001,2006-08-19,22.2,6.2,41629.42\nR001,2006-08-20,21.1,13.4,14009.63\nR001,2006-08-21,19.7,12.3,36569.81\nR001,2006-08-22,19.8,5.3,45378.13\nR001,2006-08-23,19.9,0.3,46800.0\nR001,2006-08-24,19.8,0.3,43659.49\nR001,2006-08-25,20.8,0.0,46800.0\nR001,2006-08-26,22.1,0.0,46800.0\nR001,2006-08-27,24.0,0.0,46746.81\nR001,2006-08-28,23.9,0.5,46644.01\nR001,2006-08-29,22.3,1.9,46540.78\nR001,2006-08-30,21.7,0.0,46437.24\nR001,2006-08-31,23.6,0.0,46333.47\nR001,2006-09-01,22.5,0.8,43265.94\nR001,2006-09-02,20.7,13.1,34024.15\nR001,2006-09-03,16.7,28.2,26619.02\nR001,2006-09-04,16.1,0.0,42792.68\nR001,2006-09-05,15.4,11.1,39039.76\nR001,2006-09-06,13.4,22.5,17820.82\nR001,2006-09-07,12.5,2.4,38328.14\nR001,2006-09-08,14.4,0.0,43200.0\nR001,2006-09-09,15.9,0.0,43385.26\nR001,2006-09-10,16.0,0.3,40401.35\nR001,2006-09-11,16.0,0.0,31656.51\nR001,2006-09-12,15.7,0.4,41586.02\nR001,2006-09-13,14.0,2.1,39665.79\nR001,2006-09-14,13.4,2.5,41026.06\nR001,2006-09-15,12.9,0.8,41037.6\nR001,2006-09-16,11.9,0.2,41171.71\nR001,2006-09-17,12.2,0.0,42916.52\nR001,2006-09-18,14.3,0.0,42848.88\nR001,2006-09-19,17.3,0.0,42146.99\nR001,2006-09-20,18.2,0.5,42499.36\nR001,2006-09-21,15.5,3.3,38042.61\nR001,2006-09-22,14.5,7.4,2772.48\nR001,2006-09-23,14.7,5.1,38298.14\nR001,2006-09-24,15.5,7.1,30840.93\nR001,2006-09-25,15.6,1.5,41400.68\nR001,2006-09-26,15.2,5.6,28505.65\nR001,2006-09-27,14.4,1.9,21359.61\nR001,2006-09-28,15.2,0.1,40870.92\nR001,2006-09-29,16.9,0.0,39934.43\nR001,2006-09-30,16.3,6.5,29970.15\nR001,2006-10-01,13.6,10.2,9141.72\nR001,2006-10-02,12.3,0.0,38552.99\nR001,2006-10-03,12.3,0.0,38139.64\nR001,2006-10-04,13.4,0.0,38670.43\nR001,2006-10-05,14.7,0.0,39175.52\nR001,2006-10-06,16.8,0.0,38750.55\nR001,2006-10-07,16.8,0.1,36583.37\nR001,2006-10-08,15.4,0.0,37286.95\nR001,2006-10-09,14.7,0.0,34224.47\nR001,2006-10-10,12.8,2.2,27961.54\nR001,2006-10-11,12.5,0.2,28375.0\nR001,2006-10-12,13.5,0.0,32515.59\nR001,2006-10-13,12.3,2.4,29677.43\nR001,2006-10-14,11.5,0.1,34514.74\nR001,2006-10-15,9.8,5.0,13449.99\nR001,2006-10-16,8.6,10.5,19869.36\nR001,2006-10-17,7.4,9.1,19346.46\nR001,2006-10-18,5.2,26.7,0.0\nR001,2006-10-19,5.2,0.0,36000.0\nR001,2006-10-20,9.1,0.0,36000.0\nR001,2006-10-21,10.4,0.2,33766.94\nR001,2006-10-22,9.4,0.2,35497.3\nR001,2006-10-23,9.9,0.0,32615.01\nR001,2006-10-24,10.5,0.0,34318.75\nR001,2006-10-25,11.2,0.0,36000.0\nR001,2006-10-26,10.5,6.0,20149.83\nR001,2006-10-27,11.2,0.0,36000.0\nR001,2006-10-28,10.4,0.0,36000.0\nR001,2006-10-29,10.3,0.4,32095.33\nR001,2006-10-30,9.5,11.5,10102.57\nR001,2006-10-31,12.0,0.0,32996.47\nR001,2006-11-01,13.8,0.0,36000.0\nR001,2006-11-02,11.3,17.2,8696.66\nR001,2006-11-03,10.6,0.4,33807.96\nR001,2006-11-04,9.1,44.0,0.0\nR001,2006-11-05,4.1,31.2,0.0\nR001,2006-11-06,1.1,1.9,33271.08\nR001,2006-11-07,0.2,34.7,2438.5\nR001,2006-11-08,0.0,5.7,29760.37\nR001,2006-11-09,3.6,0.8,26579.0\nR001,2006-11-10,3.4,0.0,34959.78\nR001,2006-11-11,2.4,18.5,0.0\nR001,2006-11-12,-0.3,0.5,19369.03\nR001,2006-11-13,2.7,3.0,23127.42\nR001,2006-11-14,1.8,34.2,23343.92\nR001,2006-11-15,1.4,21.7,8934.25\nR001,2006-11-16,3.2,18.8,2509.92\nR001,2006-11-17,3.2,16.4,0.0\nR001,2006-11-18,-1.9,0.0,31040.14\nR001,2006-11-19,-4.0,0.0,32578.79\nR001,2006-11-20,-0.9,0.0,31559.96\nR001,2006-11-21,1.1,0.0,29977.34\nR001,2006-11-22,2.7,0.0,28800.0\nR001,2006-11-23,2.7,0.0,32214.67\nR001,2006-11-24,2.4,0.0,31104.52\nR001,2006-11-25,2.1,0.0,31544.24\nR001,2006-11-26,2.5,0.0,30863.66\nR001,2006-11-27,5.3,0.0,23572.05\nR001,2006-11-28,3.8,0.0,26882.23\nR001,2006-11-29,2.4,0.0,28138.19\nR001,2006-11-30,0.9,0.3,29214.02\nR001,2006-12-01,1.4,10.0,18000.0\nR001,2006-12-02,-0.3,1.6,29559.05\nR001,2006-12-03,0.9,0.0,29970.54\nR001,2006-12-04,2.5,0.0,30365.74\nR001,2006-12-05,2.0,0.0,30378.21\nR001,2006-12-06,2.5,0.0,29621.66\nR001,2006-12-07,1.6,0.0,30398.88\nR001,2006-12-08,1.4,0.0,28865.89\nR001,2006-12-09,0.1,0.0,30413.69\nR001,2006-12-10,2.6,0.0,30418.85\nR001,2006-12-11,1.6,0.0,30422.45\nR001,2006-12-12,0.8,0.0,30036.71\nR001,2006-12-13,0.6,0.6,28871.73\nR001,2006-12-14,0.7,6.9,3603.47\nR001,2006-12-15,-0.0,1.6,30031.21\nR001,2006-12-16,1.0,0.1,29636.34\nR001,2006-12-17,3.9,0.0,29487.83\nR001,2006-12-18,5.5,0.0,23607.6\nR001,2006-12-19,5.5,0.0,1510.76\nR001,2006-12-20,5.1,5.5,4236.27\nR001,2006-12-21,1.9,19.5,0.0\nR001,2006-12-22,-2.6,10.2,29960.91\nR001,2006-12-23,-3.0,3.9,24752.35\nR001,2006-12-24,-2.2,4.8,30318.85\nR001,2006-12-25,-1.2,33.7,0.0\nR001,2006-12-26,-5.3,10.1,23826.28\nR001,2006-12-27,-13.2,0.0,31360.67\nR001,2006-12-28,-8.8,0.3,28800.0\nR001,2006-12-29,-2.0,5.7,2203.94\nR001,2006-12-30,-2.3,10.4,20585.15\nR001,2006-12-31,-6.2,0.0,31544.18\nR001,2007-01-01,-3.8,1.7,27902.29\nR001,2007-01-02,-1.0,0.3,19645.38\nR001,2007-01-03,1.9,0.0,16954.84\nR001,2007-01-04,2.7,0.0,31656.96\nR001,2007-01-05,0.3,0.0,27231.53\nR001,2007-01-06,-0.4,3.9,927.5\nR001,2007-01-07,-1.0,0.8,30182.61\nR001,2007-01-08,-0.6,3.6,12421.81\nR001,2007-01-09,0.3,2.5,17475.61\nR001,2007-01-10,0.3,0.0,28391.45\nR001,2007-01-11,-0.3,0.0,32204.68\nR001,2007-01-12,0.6,0.0,32275.44\nR001,2007-01-13,1.4,3.9,28405.54\nR001,2007-01-14,-0.6,30.6,4370.78\nR001,2007-01-15,0.2,15.7,0.0\nR001,2007-01-16,-2.7,10.8,13552.12\nR001,2007-01-17,-1.9,0.6,30347.87\nR001,2007-01-18,-0.6,0.0,28932.65\nR001,2007-01-19,2.1,0.0,32400.0\nR001,2007-01-20,1.1,34.9,0.0\nR001,2007-01-21,-1.6,2.5,32400.0\nR001,2007-01-22,0.7,1.4,18000.0\nR001,2007-01-23,2.5,0.0,32400.0\nR001,2007-01-24,4.5,0.0,29530.79\nR001,2007-01-25,6.6,0.0,32400.0\nR001,2007-01-26,6.0,0.0,32130.43\nR001,2007-01-27,5.5,1.5,2619.6\nR001,2007-01-28,1.7,37.6,0.0\nR001,2007-01-29,-7.2,1.4,26464.87\nR001,2007-01-30,-5.2,22.9,12135.33\nR001,2007-01-31,-7.2,6.5,18947.5\nR001,2007-02-01,-6.9,1.8,26941.81\nR001,2007-02-02,-2.7,6.7,6980.61\nR001,2007-02-03,-1.7,1.9,30420.19\nR001,2007-02-04,-1.4,1.0,29403.47\nR001,2007-02-05,-2.5,7.6,26115.86\nR001,2007-02-06,-1.7,2.7,14776.68\nR001,2007-02-07,0.0,0.3,31421.37\nR001,2007-02-08,1.1,0.0,26549.85\nR001,2007-02-09,2.6,0.0,35307.92\nR001,2007-02-10,1.7,0.0,35759.91\nR001,2007-02-11,2.4,0.0,35796.76\nR001,2007-02-12,2.8,0.0,32300.58\nR001,2007-02-13,3.1,0.0,35528.33\nR001,2007-02-14,2.9,0.0,36919.41\nR001,2007-02-15,3.2,0.0,37191.23\nR001,2007-02-16,1.9,0.0,37361.43\nR001,2007-02-17,1.1,2.9,21594.64\nR001,2007-02-18,-1.4,0.1,35360.91\nR001,2007-02-19,-0.9,0.0,37880.21\nR001,2007-02-20,0.1,6.7,588.06\nR001,2007-02-21,0.4,8.4,0.0\nR001,2007-02-22,-0.8,2.2,36387.43\nR001,2007-02-23,-1.7,4.9,3995.4\nR001,2007-02-24,-5.8,11.6,31988.97\nR001,2007-02-25,-9.7,0.0,38940.42\nR001,2007-02-26,-7.9,0.0,37844.18\nR001,2007-02-27,-5.5,0.0,39296.11\nR001,2007-02-28,0.0,0.0,32010.06\nR001,2007-03-01,2.7,9.1,5659.69\nR001,2007-03-02,3.0,2.7,23283.0\nR001,2007-03-03,3.6,3.2,20430.23\nR001,2007-03-04,4.4,6.9,32592.47\nR001,2007-03-05,3.6,6.1,7178.59\nR001,2007-03-06,4.1,6.8,20264.47\nR001,2007-03-07,2.8,2.1,14712.52\nR001,2007-03-08,3.1,0.0,39600.0\nR001,2007-03-09,2.9,0.0,39600.0\nR001,2007-03-10,2.9,0.0,39492.37\nR001,2007-03-11,3.1,0.0,39600.0\nR001,2007-03-12,2.1,6.2,27897.24\nR001,2007-03-13,1.2,8.0,337.82\nR001,2007-03-14,3.7,0.2,36000.0\nR001,2007-03-15,3.3,0.0,37306.08\nR001,2007-03-16,1.9,2.9,35646.75\nR001,2007-03-17,0.5,2.9,36471.2\nR001,2007-03-18,0.1,23.7,0.0\nR001,2007-03-19,-0.9,0.5,39600.0\nR001,2007-03-20,1.7,0.0,41350.83\nR001,2007-03-21,4.2,0.0,41895.77\nR001,2007-03-22,5.3,0.0,42123.3\nR001,2007-03-23,4.3,0.0,42228.55\nR001,2007-03-24,5.6,11.6,13574.11\nR001,2007-03-25,3.9,9.2,33633.04\nR001,2007-03-26,1.0,14.8,695.93\nR001,2007-03-27,1.4,6.3,25712.1\nR001,2007-03-28,1.3,1.0,38600.81\nR001,2007-03-29,1.8,0.6,38501.37\nR001,2007-03-30,2.5,2.2,36883.19\nR001,2007-03-31,2.1,0.0,35110.64\nR001,2007-04-01,4.8,0.9,38156.81\nR001,2007-04-02,4.9,6.8,0.0\nR001,2007-04-03,3.4,7.5,9067.54\nR001,2007-04-04,2.6,4.8,24448.42\nR001,2007-04-05,4.3,0.0,40238.91\nR001,2007-04-06,3.4,20.5,97.64\nR001,2007-04-07,1.7,0.0,45935.01\nR001,2007-04-08,4.5,0.0,46040.52\nR001,2007-04-09,3.7,0.0,45552.22\nR001,2007-04-10,4.5,0.0,45199.98\nR001,2007-04-11,5.5,1.7,28066.11\nR001,2007-04-12,5.3,0.4,43406.38\nR001,2007-04-13,3.4,13.1,22925.16\nR001,2007-04-14,1.4,1.8,46554.68\nR001,2007-04-15,2.0,0.5,35340.87\nR001,2007-04-16,2.4,4.9,26514.15\nR001,2007-04-17,2.0,0.8,45010.87\nR001,2007-04-18,3.7,0.0,46171.1\nR001,2007-04-19,4.3,3.2,22732.96\nR001,2007-04-20,3.0,8.1,28268.7\nR001,2007-04-21,4.7,0.0,45115.46\nR001,2007-04-22,7.1,1.3,36000.0\nR001,2007-04-23,4.5,8.3,20505.08\nR001,2007-04-24,4.8,0.0,43438.26\nR001,2007-04-25,5.3,3.0,39697.93\nR001,2007-04-26,6.8,1.3,44323.57\nR001,2007-04-27,8.7,1.3,45450.77\nR001,2007-04-28,8.6,2.5,45742.58\nR001,2007-04-29,9.3,0.0,43329.38\nR001,2007-04-30,7.4,12.0,3860.69\nR001,2007-05-01,7.6,5.1,37336.96\nR001,2007-05-02,7.6,17.4,344.77\nR001,2007-05-03,7.1,16.4,43902.05\nR001,2007-05-04,6.5,0.0,49434.49\nR001,2007-05-05,9.2,0.0,49705.53\nR001,2007-05-06,12.2,0.0,49712.57\nR001,2007-05-07,14.5,0.0,49683.94\nR001,2007-05-08,17.2,0.2,44475.23\nR001,2007-05-09,13.0,15.1,42159.89\nR001,2007-05-10,11.8,2.5,46177.43\nR001,2007-05-11,11.5,0.0,42273.69\nR001,2007-05-12,14.0,0.0,50228.65\nR001,2007-05-13,14.0,1.1,40385.24\nR001,2007-05-14,13.1,0.1,45562.23\nR001,2007-05-15,12.3,0.0,50400.0\nR001,2007-05-16,15.3,0.0,50462.63\nR001,2007-05-17,18.4,0.0,50433.24\nR001,2007-05-18,19.8,0.0,50404.3\nR001,2007-05-19,19.5,0.0,50400.0\nR001,2007-05-20,19.0,0.2,50400.0\nR001,2007-05-21,19.2,2.5,50400.0\nR001,2007-05-22,19.9,1.5,50649.08\nR001,2007-05-23,20.8,0.8,50400.0\nR001,2007-05-24,21.5,0.4,48726.8\nR001,2007-05-25,21.2,1.3,50400.0\nR001,2007-05-26,22.4,0.7,50400.0\nR001,2007-05-27,21.9,3.2,32380.0\nR001,2007-05-28,21.0,4.9,43200.0\nR001,2007-05-29,20.7,2.2,51553.79\nR001,2007-05-30,19.4,5.1,43850.28\nR001,2007-05-31,19.2,8.8,47944.83\nR001,2007-06-01,19.0,10.9,48522.11\nR001,2007-06-02,17.7,2.9,49403.88\nR001,2007-06-03,16.9,10.7,26685.26\nR001,2007-06-04,15.6,14.6,21955.92\nR001,2007-06-05,15.4,22.3,3542.59\nR001,2007-06-06,16.8,0.0,49156.04\nR001,2007-06-07,17.5,0.5,51012.15\nR001,2007-06-08,15.8,9.2,13828.75\nR001,2007-06-09,15.4,19.3,21917.74\nR001,2007-06-10,15.3,12.9,48354.92\nR001,2007-06-11,15.9,3.3,50158.57\nR001,2007-06-12,16.3,3.6,50929.21\nR001,2007-06-13,16.0,1.9,51661.47\nR001,2007-06-14,18.4,0.1,53114.85\nR001,2007-06-15,20.1,1.0,53104.01\nR001,2007-06-16,21.3,0.8,53461.12\nR001,2007-06-17,19.9,4.5,52716.0\nR001,2007-06-18,17.4,14.0,46776.39\nR001,2007-06-19,17.2,8.0,35723.99\nR001,2007-06-20,15.9,7.5,34726.29\nR001,2007-06-21,15.0,5.9,47105.65\nR001,2007-06-22,15.2,0.9,51619.58\nR001,2007-06-23,17.9,0.2,53673.69\nR001,2007-06-24,18.9,0.4,51927.68\nR001,2007-06-25,16.2,0.4,53965.94\nR001,2007-06-26,15.4,0.5,53388.33\nR001,2007-06-27,18.6,0.0,53971.55\nR001,2007-06-28,21.0,0.4,53970.73\nR001,2007-06-29,20.7,1.0,53967.42\nR001,2007-06-30,16.8,8.7,47482.98\nR001,2007-07-01,16.9,3.5,53197.19\nR001,2007-07-02,18.1,1.0,53380.09\nR001,2007-07-03,18.4,0.4,52850.09\nR001,2007-07-04,20.0,0.1,53147.46\nR001,2007-07-05,21.4,0.0,53898.17\nR001,2007-07-06,22.7,0.0,53879.07\nR001,2007-07-07,16.5,25.8,19175.75\nR001,2007-07-08,15.7,0.8,53760.92\nR001,2007-07-09,15.5,0.2,53041.53\nR001,2007-07-10,16.7,0.0,53191.58\nR001,2007-07-11,20.2,0.0,53752.02\nR001,2007-07-12,22.9,1.0,53720.37\nR001,2007-07-13,22.7,1.4,53073.74\nR001,2007-07-14,18.2,14.0,29549.05\nR001,2007-07-15,17.7,4.3,45150.53\nR001,2007-07-16,17.1,0.4,51173.78\nR001,2007-07-17,18.5,0.0,52404.94\nR001,2007-07-18,19.5,0.0,53487.74\nR001,2007-07-19,20.0,0.0,53441.92\nR001,2007-07-20,22.0,0.0,53376.17\nR001,2007-07-21,22.4,0.0,53271.69\nR001,2007-07-22,23.5,0.2,52713.97\nR001,2007-07-23,23.0,0.0,52676.09\nR001,2007-07-24,22.6,0.0,51934.13\nR001,2007-07-25,24.3,0.0,52248.61\nR001,2007-07-26,24.5,0.1,51510.4\nR001,2007-07-27,22.1,0.1,50894.65\nR001,2007-07-28,20.8,0.0,49895.39\nR001,2007-07-29,21.3,0.5,49945.9\nR001,2007-07-30,22.9,0.0,51020.37\nR001,2007-07-31,24.5,0.0,50639.57\nR001,2007-08-01,24.5,0.3,48034.89\nR001,2007-08-02,21.2,11.8,46916.44\nR001,2007-08-03,19.2,15.5,32745.75\nR001,2007-08-04,19.9,2.8,46800.0\nR001,2007-08-05,21.6,1.5,46844.69\nR001,2007-08-06,24.4,0.0,48679.13\nR001,2007-08-07,25.5,2.0,48401.44\nR001,2007-08-08,23.9,1.8,47970.49\nR001,2007-08-09,21.7,7.7,46045.66\nR001,2007-08-10,20.8,1.0,47349.19\nR001,2007-08-11,22.2,2.8,47377.98\nR001,2007-08-12,23.7,0.4,47406.97\nR001,2007-08-13,24.5,0.0,47771.6\nR001,2007-08-14,21.2,6.8,39331.58\nR001,2007-08-15,20.7,15.1,44847.39\nR001,2007-08-16,20.0,4.2,46800.0\nR001,2007-08-17,19.9,4.2,31634.64\nR001,2007-08-18,19.6,3.2,46302.96\nR001,2007-08-19,19.9,5.4,46476.32\nR001,2007-08-20,20.5,4.1,46800.0\nR001,2007-08-21,20.9,4.2,46800.0\nR001,2007-08-22,19.7,11.7,46211.64\nR001,2007-08-23,18.8,6.0,36490.89\nR001,2007-08-24,20.3,3.0,39215.78\nR001,2007-08-25,20.8,8.7,39598.75\nR001,2007-08-26,20.1,2.1,46800.0\nR001,2007-08-27,19.7,4.0,38085.68\nR001,2007-08-28,18.5,6.0,37966.74\nR001,2007-08-29,17.9,2.5,30090.61\nR001,2007-08-30,18.5,0.9,40869.38\nR001,2007-08-31,18.9,0.0,43569.34\nR001,2007-09-01,20.0,0.0,46262.86\nR001,2007-09-02,21.2,0.1,45576.95\nR001,2007-09-03,21.6,0.6,45304.38\nR001,2007-09-04,23.0,0.0,45033.2\nR001,2007-09-05,22.9,0.4,44443.1\nR001,2007-09-06,22.3,1.2,44174.69\nR001,2007-09-07,21.8,1.5,43587.76\nR001,2007-09-08,21.2,3.8,43640.99\nR001,2007-09-09,18.1,17.7,35890.18\nR001,2007-09-10,15.4,20.4,34541.76\nR001,2007-09-11,16.1,0.0,42839.52\nR001,2007-09-12,17.1,0.0,43200.0\nR001,2007-09-13,16.8,12.5,43200.0\nR001,2007-09-14,12.2,14.8,34468.97\nR001,2007-09-15,13.1,0.0,41050.78\nR001,2007-09-16,12.3,6.5,30999.58\nR001,2007-09-17,8.8,4.0,20747.72\nR001,2007-09-18,12.3,0.0,38164.2\nR001,2007-09-19,15.1,0.0,42456.92\nR001,2007-09-20,16.5,0.1,42730.1\nR001,2007-09-21,16.3,0.0,42525.19\nR001,2007-09-22,16.7,0.0,42559.93\nR001,2007-09-23,17.1,1.2,39648.43\nR001,2007-09-24,16.1,6.8,19115.66\nR001,2007-09-25,14.8,20.5,12792.14\nR001,2007-09-26,14.9,0.1,38195.81\nR001,2007-09-27,15.2,0.0,41464.06\nR001,2007-09-28,16.1,0.0,41181.41\nR001,2007-09-29,15.8,0.0,41218.04\nR001,2007-09-30,15.6,0.0,40830.02\nR001,2007-10-01,15.4,1.4,40117.07\nR001,2007-10-02,15.0,0.2,40044.05\nR001,2007-10-03,15.2,0.3,39971.97\nR001,2007-10-04,16.4,0.0,38858.38\nR001,2007-10-05,16.9,0.1,38072.04\nR001,2007-10-06,15.7,1.5,37483.49\nR001,2007-10-07,14.1,0.6,36883.1\nR001,2007-10-08,14.2,1.1,26634.66\nR001,2007-10-09,13.7,0.4,35404.1\nR001,2007-10-10,13.7,0.7,36083.49\nR001,2007-10-11,13.9,1.2,36461.7\nR001,2007-10-12,13.5,0.0,36840.89\nR001,2007-10-13,15.0,0.7,36243.68\nR001,2007-10-14,13.5,13.4,20632.01\nR001,2007-10-15,9.2,32.7,168.14\nR001,2007-10-16,7.5,0.2,33719.86\nR001,2007-10-17,7.6,0.0,33603.95\nR001,2007-10-18,8.5,0.0,36000.0\nR001,2007-10-19,10.9,0.0,36000.0\nR001,2007-10-20,12.7,0.0,36000.0\nR001,2007-10-21,13.6,0.0,36000.0\nR001,2007-10-22,13.3,0.0,36000.0\nR001,2007-10-23,14.0,0.0,36000.0\nR001,2007-10-24,15.1,0.0,36000.0\nR001,2007-10-25,15.2,0.1,32038.09\nR001,2007-10-26,11.5,7.6,25488.0\nR001,2007-10-27,9.7,0.6,26569.07\nR001,2007-10-28,8.4,4.1,23703.92\nR001,2007-10-29,9.2,0.0,35938.9\nR001,2007-10-30,8.7,4.6,21600.0\nR001,2007-10-31,9.6,2.3,18394.66\nR001,2007-11-01,9.3,0.0,33555.41\nR001,2007-11-02,7.6,5.9,22946.35\nR001,2007-11-03,2.6,30.0,14400.0\nR001,2007-11-04,0.4,0.0,27409.32\nR001,2007-11-05,5.5,10.6,19236.39\nR001,2007-11-06,5.8,18.2,2697.54\nR001,2007-11-07,4.0,0.1,30195.98\nR001,2007-11-08,2.7,8.1,1380.68\nR001,2007-11-09,2.0,12.6,17955.02\nR001,2007-11-10,1.9,0.0,29661.06\nR001,2007-11-11,1.4,26.1,0.0\nR001,2007-11-12,-0.1,7.8,26160.21\nR001,2007-11-13,3.4,12.5,30686.67\nR001,2007-11-14,3.1,15.7,0.0\nR001,2007-11-15,5.1,0.0,34268.18\nR001,2007-11-16,8.2,0.0,32670.8\nR001,2007-11-17,8.3,0.0,33122.12\nR001,2007-11-18,6.8,2.7,28794.31\nR001,2007-11-19,7.0,0.0,28406.85\nR001,2007-11-20,7.3,0.0,27800.42\nR001,2007-11-21,5.6,0.0,31611.56\nR001,2007-11-22,0.8,1.9,24438.67\nR001,2007-11-23,-0.8,0.0,31018.59\nR001,2007-11-24,0.4,0.0,31830.18\nR001,2007-11-25,1.0,0.0,31852.98\nR001,2007-11-26,2.9,0.0,31223.22\nR001,2007-11-27,3.5,26.3,17703.93\nR001,2007-11-28,-5.1,9.0,2632.38\nR001,2007-11-29,-7.7,0.7,29900.86\nR001,2007-11-30,-1.8,0.0,27145.39\nR001,2007-12-01,2.8,14.8,10905.61\nR001,2007-12-02,-1.2,20.7,0.0\nR001,2007-12-03,-4.2,0.0,27915.13\nR001,2007-12-04,-0.9,0.0,29982.82\nR001,2007-12-05,-0.4,0.0,30377.8\nR001,2007-12-06,0.5,0.0,28800.0\nR001,2007-12-07,2.5,8.9,0.0\nR001,2007-12-08,1.3,2.3,24721.94\nR001,2007-12-09,-0.2,0.0,29256.58\nR001,2007-12-10,1.6,2.2,28245.11\nR001,2007-12-11,2.8,11.3,12847.53\nR001,2007-12-12,2.5,12.6,2070.54\nR001,2007-12-13,3.1,4.4,13161.66\nR001,2007-12-14,3.6,15.7,0.0\nR001,2007-12-15,1.2,3.3,25438.47\nR001,2007-12-16,-4.0,0.0,30024.28\nR001,2007-12-17,-4.1,6.5,20628.14\nR001,2007-12-18,-6.1,6.4,24665.62\nR001,2007-12-19,-8.2,2.4,12844.08\nR001,2007-12-20,-4.2,1.9,29207.46\nR001,2007-12-21,-2.7,8.3,8134.77\nR001,2007-12-22,-4.1,0.2,29415.54\nR001,2007-12-23,-2.2,2.4,29556.91\nR001,2007-12-24,-2.6,0.0,29932.05\nR001,2007-12-25,-4.0,0.0,31268.43\nR001,2007-12-26,-4.8,0.0,31305.58\nR001,2007-12-27,-4.6,0.0,29876.1\nR001,2007-12-28,-1.9,1.2,17640.48\nR001,2007-12-29,-2.0,3.9,14227.83\nR001,2007-12-30,-2.5,0.2,30029.92\nR001,2007-12-31,-3.4,1.0,31344.41\nR001,2008-01-01,-5.0,0.0,31575.52\nR001,2008-01-02,-4.9,0.0,31629.28\nR001,2008-01-03,-4.4,0.0,30863.88\nR001,2008-01-04,-2.4,4.2,394.34\nR001,2008-01-05,-2.8,10.8,2514.92\nR001,2008-01-06,-7.4,2.7,31864.93\nR001,2008-01-07,-9.0,0.7,31928.29\nR001,2008-01-08,-9.7,0.0,31993.14\nR001,2008-01-09,-7.7,0.0,32059.34\nR001,2008-01-10,-6.2,0.0,32126.76\nR001,2008-01-11,-7.0,0.4,32195.23\nR001,2008-01-12,-7.8,0.0,32264.64\nR001,2008-01-13,-5.9,0.0,32334.83\nR001,2008-01-14,-5.0,0.0,32400.0\nR001,2008-01-15,-3.8,0.0,32400.0\nR001,2008-01-16,-3.7,0.0,32400.0\nR001,2008-01-17,-2.9,0.0,32400.0\nR001,2008-01-18,-2.4,0.0,32400.0\nR001,2008-01-19,-2.6,0.0,32400.0\nR001,2008-01-20,-3.4,0.0,32400.0\nR001,2008-01-21,-1.5,0.9,29074.0\nR001,2008-01-22,0.7,7.8,29772.64\nR001,2008-01-23,3.4,0.0,32400.0\nR001,2008-01-24,0.4,11.9,0.0\nR001,2008-01-25,-5.2,0.2,32127.48\nR001,2008-01-26,-4.1,1.5,24230.69\nR001,2008-01-27,-3.1,6.8,29774.99\nR001,2008-01-28,-3.0,25.9,0.0\nR001,2008-01-29,-3.8,2.8,1208.34\nR001,2008-01-30,-4.2,3.0,11216.41\nR001,2008-01-31,-5.8,5.2,32400.0\nR001,2008-02-01,-6.2,0.0,32114.39\nR001,2008-02-02,-4.4,0.0,32696.2\nR001,2008-02-03,-3.6,0.0,33083.25\nR001,2008-02-04,-2.2,0.0,33002.0\nR001,2008-02-05,-1.5,0.0,32919.72\nR001,2008-02-06,1.0,0.0,33184.67\nR001,2008-02-07,0.7,0.0,32822.88\nR001,2008-02-08,1.1,0.0,29051.14\nR001,2008-02-09,0.5,0.0,33765.86\nR001,2008-02-10,0.5,0.0,35408.64\nR001,2008-02-11,0.0,0.0,34805.34\nR001,2008-02-12,-2.8,0.0,36164.83\nR001,2008-02-13,-2.7,3.9,32285.96\nR001,2008-02-14,-5.2,8.0,30282.31\nR001,2008-02-15,-6.2,11.0,26740.8\nR001,2008-02-16,-5.8,14.2,509.74\nR001,2008-02-17,-5.2,10.4,0.0\nR001,2008-02-18,-2.0,12.7,0.0\nR001,2008-02-19,-4.4,12.9,0.0\nR001,2008-02-20,-6.1,20.9,24594.44\nR001,2008-02-21,-4.4,2.1,31400.7\nR001,2008-02-22,0.5,1.0,30309.78\nR001,2008-02-23,2.8,0.5,33805.72\nR001,2008-02-24,0.5,17.1,1.55\nR001,2008-02-25,-1.9,0.0,36561.14\nR001,2008-02-26,-0.1,0.0,39068.54\nR001,2008-02-27,2.9,0.0,39243.44\nR001,2008-02-28,4.8,0.0,39389.83\nR001,2008-02-29,3.8,9.9,3597.81\nR001,2008-03-01,1.9,0.0,33954.1\nR001,2008-03-02,4.8,7.0,39600.0\nR001,2008-03-03,1.6,19.5,20732.21\nR001,2008-03-04,1.3,0.6,38678.8\nR001,2008-03-05,7.1,0.0,38291.04\nR001,2008-03-06,7.4,0.2,34953.74\nR001,2008-03-07,4.4,0.9,28417.72\nR001,2008-03-08,7.0,0.0,39293.43\nR001,2008-03-09,10.9,0.0,20817.37\nR001,2008-03-10,9.7,1.7,30411.91\nR001,2008-03-11,3.3,16.1,23853.41\nR001,2008-03-12,3.0,0.0,39600.0\nR001,2008-03-13,5.5,0.0,28513.17\nR001,2008-03-14,6.6,0.9,39600.0\nR001,2008-03-15,2.6,10.7,19222.98\nR001,2008-03-16,-0.4,15.5,19427.65\nR001,2008-03-17,3.3,0.0,39600.0\nR001,2008-03-18,8.7,0.6,39979.45\nR001,2008-03-19,5.8,17.8,5716.15\nR001,2008-03-20,4.1,15.4,3589.09\nR001,2008-03-21,5.3,26.1,6249.35\nR001,2008-03-22,5.3,6.1,32763.33\nR001,2008-03-23,10.0,0.0,39708.47\nR001,2008-03-24,16.5,0.6,34461.58\nR001,2008-03-25,14.9,9.2,42142.28\nR001,2008-03-26,5.6,13.9,41060.56\nR001,2008-03-27,4.0,1.9,37138.63\nR001,2008-03-28,7.7,0.0,42472.22\nR001,2008-03-29,8.8,2.7,35780.59\nR001,2008-03-30,9.8,2.0,18000.0\nR001,2008-03-31,8.4,6.6,15363.09\nR001,2008-04-01,6.9,10.1,28316.49\nR001,2008-04-02,3.7,20.6,0.0\nR001,2008-04-03,4.7,0.7,42227.16\nR001,2008-04-04,5.6,0.0,43649.52\nR001,2008-04-05,5.7,3.1,42654.56\nR001,2008-04-06,7.6,0.1,33879.53\nR001,2008-04-07,10.1,5.3,38875.65\nR001,2008-04-08,8.3,11.4,37052.37\nR001,2008-04-09,7.5,1.1,37102.51\nR001,2008-04-10,9.3,1.3,44094.73\nR001,2008-04-11,10.3,0.0,41891.89\nR001,2008-04-12,10.6,1.3,37940.76\nR001,2008-04-13,12.9,0.3,40525.31\nR001,2008-04-14,15.3,0.2,42296.87\nR001,2008-04-15,17.7,0.0,43200.0\nR001,2008-04-16,15.7,3.4,40697.41\nR001,2008-04-17,6.8,24.2,9473.06\nR001,2008-04-18,8.0,0.0,46344.45\nR001,2008-04-19,10.0,0.0,33010.36\nR001,2008-04-20,10.1,0.8,46800.0\nR001,2008-04-21,11.4,1.6,46800.0\nR001,2008-04-22,12.5,0.0,46866.53\nR001,2008-04-23,17.5,2.1,19406.32\nR001,2008-04-24,14.1,3.8,24674.07\nR001,2008-04-25,11.2,16.7,13380.53\nR001,2008-04-26,13.5,1.5,37057.7\nR001,2008-04-27,16.0,5.1,10547.17\nR001,2008-04-28,13.8,1.9,45943.71\nR001,2008-04-29,12.3,14.3,24755.11\nR001,2008-04-30,10.2,16.3,23270.8\nR001,2008-05-01,9.8,4.1,32917.4\nR001,2008-05-02,9.4,6.2,39285.59\nR001,2008-05-03,9.5,7.4,30195.26\nR001,2008-05-04,9.5,7.9,38499.75\nR001,2008-05-05,8.0,13.0,12315.7\nR001,2008-05-06,8.4,8.6,18261.39\nR001,2008-05-07,6.9,9.7,31045.67\nR001,2008-05-08,6.2,3.6,47929.12\nR001,2008-05-09,6.4,5.3,29943.42\nR001,2008-05-10,5.3,13.2,0.0\nR001,2008-05-11,6.3,6.5,39927.71\nR001,2008-05-12,6.2,3.8,45785.46\nR001,2008-05-13,5.3,9.5,34469.66\nR001,2008-05-14,8.5,2.1,46641.92\nR001,2008-05-15,10.9,0.0,50400.0\nR001,2008-05-16,11.2,3.8,43705.09\nR001,2008-05-17,11.5,3.3,46800.0\nR001,2008-05-18,13.2,2.5,48656.77\nR001,2008-05-19,13.5,9.2,23260.5\nR001,2008-05-20,14.5,1.9,46477.55\nR001,2008-05-21,16.0,1.0,48410.32\nR001,2008-05-22,17.4,0.1,50629.2\nR001,2008-05-23,19.3,0.1,50959.35\nR001,2008-05-24,18.6,2.0,50897.92\nR001,2008-05-25,15.1,24.7,5819.59\nR001,2008-05-26,13.4,31.2,21346.08\nR001,2008-05-27,12.3,1.4,47812.78\nR001,2008-05-28,11.2,5.6,18452.63\nR001,2008-05-29,9.2,29.3,18377.18\nR001,2008-05-30,8.7,5.2,39031.58\nR001,2008-05-31,10.0,0.0,50769.3\nR001,2008-06-01,12.5,0.0,51836.01\nR001,2008-06-02,11.1,1.2,36691.08\nR001,2008-06-03,9.6,3.9,44502.23\nR001,2008-06-04,11.8,0.0,50502.49\nR001,2008-06-05,17.0,0.0,52859.24\nR001,2008-06-06,16.0,5.8,32770.51\nR001,2008-06-07,15.9,2.5,49975.12\nR001,2008-06-08,14.6,8.9,35494.85\nR001,2008-06-09,14.3,10.4,38797.22\nR001,2008-06-10,15.9,3.2,45725.47\nR001,2008-06-11,14.7,14.6,1122.81\nR001,2008-06-12,14.0,21.6,8909.4\nR001,2008-06-13,15.2,5.2,40135.58\nR001,2008-06-14,16.6,1.4,53105.3\nR001,2008-06-15,17.1,5.3,53094.39\nR001,2008-06-16,16.8,1.2,48113.87\nR001,2008-06-17,16.3,1.4,42399.98\nR001,2008-06-18,15.8,6.4,43059.35\nR001,2008-06-19,17.4,0.7,50363.42\nR001,2008-06-20,18.0,2.3,50736.56\nR001,2008-06-21,16.7,11.0,42488.88\nR001,2008-06-22,17.4,0.4,49976.92\nR001,2008-06-23,16.0,1.9,51804.73\nR001,2008-06-24,14.6,0.0,50217.16\nR001,2008-06-25,17.0,0.5,51002.18\nR001,2008-06-26,18.1,0.9,51265.87\nR001,2008-06-27,19.6,0.5,50629.19\nR001,2008-06-28,20.4,0.8,52283.97\nR001,2008-06-29,18.6,15.1,48741.54\nR001,2008-06-30,17.7,2.4,51135.01\nR001,2008-07-01,15.7,1.5,49888.64\nR001,2008-07-02,14.7,3.4,51687.83\nR001,2008-07-03,14.3,8.7,44151.76\nR001,2008-07-04,15.0,1.8,51421.21\nR001,2008-07-05,17.3,0.2,51685.63\nR001,2008-07-06,17.6,0.1,49359.17\nR001,2008-07-07,15.5,6.9,44559.18\nR001,2008-07-08,16.3,1.8,49047.35\nR001,2008-07-09,17.4,0.3,52627.92\nR001,2008-07-10,17.3,0.5,52613.82\nR001,2008-07-11,17.9,3.2,48631.3\nR001,2008-07-12,18.5,0.0,51137.49\nR001,2008-07-13,20.8,0.0,52721.01\nR001,2008-07-14,20.2,9.6,48628.87\nR001,2008-07-15,18.6,15.9,14719.37\nR001,2008-07-16,19.8,1.2,52401.29\nR001,2008-07-17,18.5,12.9,47816.74\nR001,2008-07-18,17.5,4.7,45853.26\nR001,2008-07-19,19.6,0.0,51738.97\nR001,2008-07-20,20.8,1.4,50275.0\nR001,2008-07-21,22.4,0.0,47844.05\nR001,2008-07-22,23.5,0.8,49652.65\nR001,2008-07-23,23.4,1.0,51593.0\nR001,2008-07-24,24.1,0.5,51555.9\nR001,2008-07-25,25.0,0.0,51518.91\nR001,2008-07-26,22.9,5.7,49140.14\nR001,2008-07-27,20.7,20.0,44126.25\nR001,2008-07-28,19.4,0.0,49589.9\nR001,2008-07-29,19.0,0.5,48253.74\nR001,2008-07-30,19.0,4.6,46732.33\nR001,2008-07-31,19.8,1.1,45479.46\nR001,2008-08-01,19.0,2.2,19985.39\nR001,2008-08-02,20.7,0.3,42459.65\nR001,2008-08-03,21.2,0.7,40893.89\nR001,2008-08-04,20.4,13.1,41451.32\nR001,2008-08-05,20.0,6.0,35187.99\nR001,2008-08-06,19.7,3.2,46800.0\nR001,2008-08-07,19.6,0.0,46943.65\nR001,2008-08-08,18.9,0.6,46973.16\nR001,2008-08-09,20.2,0.0,47002.84\nR001,2008-08-10,20.7,1.9,47707.1\nR001,2008-08-11,20.3,1.8,47062.79\nR001,2008-08-12,20.6,0.6,47092.99\nR001,2008-08-13,19.7,0.0,47123.32\nR001,2008-08-14,19.1,13.6,46508.57\nR001,2008-08-15,20.3,2.0,46851.01\nR001,2008-08-16,20.7,1.0,46800.0\nR001,2008-08-17,21.9,3.5,46800.0\nR001,2008-08-18,22.6,0.0,46945.31\nR001,2008-08-19,22.4,1.3,46800.0\nR001,2008-08-20,20.7,7.8,46800.0\nR001,2008-08-21,18.9,10.3,46800.0\nR001,2008-08-22,20.9,0.7,46800.0\nR001,2008-08-23,21.5,0.6,46800.0\nR001,2008-08-24,23.6,0.0,46800.0\nR001,2008-08-25,23.0,3.1,46800.0\nR001,2008-08-26,21.9,2.0,46575.6\nR001,2008-08-27,20.7,0.0,46701.89\nR001,2008-08-28,19.2,0.0,46599.26\nR001,2008-08-29,20.7,0.0,46495.56\nR001,2008-08-30,21.3,0.0,46115.88\nR001,2008-08-31,17.7,12.0,37134.92\nR001,2008-09-01,15.2,0.0,45567.43\nR001,2008-09-02,15.9,0.0,45616.24\nR001,2008-09-03,18.1,0.4,44703.55\nR001,2008-09-04,19.3,0.7,44113.7\nR001,2008-09-05,17.7,7.5,40460.03\nR001,2008-09-06,17.1,6.6,36250.86\nR001,2008-09-07,16.9,4.5,35928.57\nR001,2008-09-08,16.8,4.1,39193.51\nR001,2008-09-09,15.7,11.3,33660.23\nR001,2008-09-10,14.7,5.8,24061.24\nR001,2008-09-11,13.3,2.8,37918.06\nR001,2008-09-12,14.3,0.0,42846.09\nR001,2008-09-13,16.7,0.0,43152.05\nR001,2008-09-14,16.0,3.9,36300.45\nR001,2008-09-15,15.3,0.0,39428.13\nR001,2008-09-16,16.6,0.0,42950.11\nR001,2008-09-17,19.7,0.2,42882.54\nR001,2008-09-18,18.6,11.3,40939.04\nR001,2008-09-19,15.3,4.2,31258.34\nR001,2008-09-20,14.7,3.4,30474.07\nR001,2008-09-21,16.5,0.5,35141.59\nR001,2008-09-22,16.7,11.9,24513.49\nR001,2008-09-23,14.2,13.0,24857.98\nR001,2008-09-24,11.7,11.0,39088.59\nR001,2008-09-25,11.9,0.0,38697.7\nR001,2008-09-26,12.0,0.0,36461.89\nR001,2008-09-27,13.5,0.0,38965.65\nR001,2008-09-28,12.9,0.6,34761.22\nR001,2008-09-29,12.5,0.3,35014.66\nR001,2008-09-30,11.4,7.0,5067.05\nR001,2008-10-01,11.5,4.1,29933.7\nR001,2008-10-02,11.3,0.0,39195.88\nR001,2008-10-03,12.8,0.0,39924.61\nR001,2008-10-04,13.9,0.0,38926.46\nR001,2008-10-05,15.5,0.0,38110.32\nR001,2008-10-06,15.1,2.6,35239.15\nR001,2008-10-07,12.1,5.1,20480.33\nR001,2008-10-08,10.3,7.5,30840.86\nR001,2008-10-09,9.7,3.1,32824.6\nR001,2008-10-10,9.7,0.0,36124.52\nR001,2008-10-11,9.7,0.0,33764.68\nR001,2008-10-12,9.6,0.0,36558.99\nR001,2008-10-13,10.3,0.0,35633.19\nR001,2008-10-14,13.0,0.0,36341.04\nR001,2008-10-15,15.6,0.8,34268.36\nR001,2008-10-16,13.5,23.6,0.0\nR001,2008-10-17,11.7,4.3,32285.17\nR001,2008-10-18,10.5,0.0,36000.0\nR001,2008-10-19,9.7,9.5,34711.96\nR001,2008-10-20,8.7,0.0,36000.0\nR001,2008-10-21,8.3,3.1,212.58\nR001,2008-10-22,8.3,0.1,34156.74\nR001,2008-10-23,9.4,0.0,36000.0\nR001,2008-10-24,10.9,0.0,36000.0\nR001,2008-10-25,10.3,5.3,18784.83\nR001,2008-10-26,10.7,3.6,23406.87\nR001,2008-10-27,10.6,0.0,36000.0\nR001,2008-10-28,10.5,2.5,34207.73\nR001,2008-10-29,10.0,0.0,33964.07\nR001,2008-10-30,10.2,0.0,36000.0\nR001,2008-10-31,10.7,0.0,36000.0\nR001,2008-11-01,9.3,0.0,35992.79\nR001,2008-11-02,9.8,0.0,35912.63\nR001,2008-11-03,9.9,0.0,35832.38\nR001,2008-11-04,8.6,0.0,30954.89\nR001,2008-11-05,9.1,0.0,35242.59\nR001,2008-11-06,7.9,0.3,29168.36\nR001,2008-11-07,6.6,4.3,29626.73\nR001,2008-11-08,5.9,0.0,33460.04\nR001,2008-11-09,5.7,0.0,35019.93\nR001,2008-11-10,6.6,0.0,34874.01\nR001,2008-11-11,6.6,0.0,34730.99\nR001,2008-11-12,5.7,0.0,34590.9\nR001,2008-11-13,5.2,0.0,31438.4\nR001,2008-11-14,5.8,0.0,30010.18\nR001,2008-11-15,6.0,0.0,28731.06\nR001,2008-11-16,5.8,0.4,21375.69\nR001,2008-11-17,6.4,0.0,31353.92\nR001,2008-11-18,6.1,2.9,28581.47\nR001,2008-11-19,6.0,6.2,9818.78\nR001,2008-11-20,6.3,29.5,0.0\nR001,2008-11-21,6.4,10.8,19066.3\nR001,2008-11-22,8.1,0.0,31376.26\nR001,2008-11-23,7.0,30.4,0.0\nR001,2008-11-24,5.6,37.5,53.95\nR001,2008-11-25,7.4,1.8,17116.71\nR001,2008-11-26,7.6,0.0,30606.72\nR001,2008-11-27,6.0,4.1,28756.12\nR001,2008-11-28,5.5,5.2,19202.88\nR001,2008-11-29,5.0,0.0,30296.42\nR001,2008-11-30,5.0,0.0,28800.0\nR001,2008-12-01,5.3,1.4,26331.98\nR001,2008-12-02,3.8,0.0,30729.0\nR001,2008-12-03,4.0,0.0,30743.93\nR001,2008-12-04,6.9,0.0,30374.75\nR001,2008-12-05,9.7,0.0,27072.01\nR001,2008-12-06,11.7,0.0,25259.07\nR001,2008-12-07,9.7,0.0,28864.2\nR001,2008-12-08,10.0,0.0,24301.14\nR001,2008-12-09,4.6,28.5,0.0\nR001,2008-12-10,1.1,14.3,6632.47\nR001,2008-12-11,-3.7,0.0,30420.71\nR001,2008-12-12,-2.0,0.0,30033.34\nR001,2008-12-13,-0.4,0.0,30031.92\nR001,2008-12-14,-0.7,0.0,30029.11\nR001,2008-12-15,-1.7,0.0,30414.35\nR001,2008-12-16,-0.8,0.0,29629.35\nR001,2008-12-17,-0.9,0.0,29231.72\nR001,2008-12-18,0.1,0.0,30003.76\nR001,2008-12-19,1.4,0.0,28731.84\nR001,2008-12-20,1.7,0.1,26955.16\nR001,2008-12-21,2.6,3.2,4189.47\nR001,2008-12-22,3.1,2.3,24001.35\nR001,2008-12-23,2.8,7.5,0.0\nR001,2008-12-24,3.1,0.7,13852.91\nR001,2008-12-25,-0.8,15.3,1622.28\nR001,2008-12-26,-5.6,0.0,31328.93\nR001,2008-12-27,-3.5,4.3,3183.96\nR001,2008-12-28,-3.8,2.4,15712.79\nR001,2008-12-29,-6.4,3.1,26607.83\nR001,2008-12-30,-8.9,2.4,24744.07\nR001,2008-12-31,-8.7,0.0,31567.96\nR001,2009-01-01,-8.4,0.0,31622.04\nR001,2009-01-02,-4.6,3.7,25003.57\nR001,2009-01-03,-2.5,30.9,0.0\nR001,2009-01-04,-2.4,4.7,23662.24\nR001,2009-01-05,1.2,31.7,0.0\nR001,2009-01-06,-0.4,3.9,6248.31\nR001,2009-01-07,-0.8,9.5,343.86\nR001,2009-01-08,-2.4,0.9,31575.39\nR001,2009-01-09,-7.4,15.5,26242.23\nR001,2009-01-10,-7.2,1.6,21180.14\nR001,2009-01-11,-4.8,1.4,26513.16\nR001,2009-01-12,-5.7,0.0,32037.41\nR001,2009-01-13,-4.0,0.0,32377.24\nR001,2009-01-14,-2.3,0.0,32400.0\nR001,2009-01-15,1.8,0.1,15269.15\nR001,2009-01-16,4.8,0.0,31837.76\nR001,2009-01-17,4.4,24.3,0.0\nR001,2009-01-18,-1.3,10.6,12564.03\nR001,2009-01-19,-1.4,0.4,21959.37\nR001,2009-01-20,-0.2,5.5,215.46\nR001,2009-01-21,-1.3,0.0,32400.0\nR001,2009-01-22,-0.9,0.0,32400.0\nR001,2009-01-23,0.8,0.0,32400.0\nR001,2009-01-24,6.5,0.0,17152.6\nR001,2009-01-25,4.7,0.4,32400.0\nR001,2009-01-26,4.3,12.2,1915.94\nR001,2009-01-27,4.3,4.6,25767.47\nR001,2009-01-28,3.0,1.2,32400.0\nR001,2009-01-29,4.0,0.0,28993.45\nR001,2009-01-30,4.6,4.8,21925.11\nR001,2009-01-31,3.3,1.9,20492.93\nR001,2009-02-01,1.8,12.7,14874.63\nR001,2009-02-02,-0.3,2.6,27589.42\nR001,2009-02-03,-1.5,0.4,32622.27\nR001,2009-02-04,-1.3,0.0,31809.77\nR001,2009-02-05,5.4,0.0,32533.61\nR001,2009-02-06,5.3,4.0,23729.64\nR001,2009-02-07,4.7,3.2,28800.0\nR001,2009-02-08,5.2,0.0,30680.64\nR001,2009-02-09,5.4,8.6,1815.3\nR001,2009-02-10,5.3,16.3,17088.35\nR001,2009-02-11,2.9,3.2,27661.61\nR001,2009-02-12,4.0,0.4,32450.35\nR001,2009-02-13,7.1,2.0,36223.18\nR001,2009-02-14,8.1,0.0,36256.81\nR001,2009-02-15,7.9,0.0,37294.31\nR001,2009-02-16,3.0,16.1,18674.21\nR001,2009-02-17,-1.4,19.6,10428.25\nR001,2009-02-18,-5.3,0.0,34396.42\nR001,2009-02-19,-4.8,0.0,37083.66\nR001,2009-02-20,-3.0,0.0,35092.47\nR001,2009-02-21,-0.3,0.4,24684.1\nR001,2009-02-22,-1.5,2.2,26912.88\nR001,2009-02-23,-1.3,1.5,36196.6\nR001,2009-02-24,0.9,0.0,23649.62\nR001,2009-02-25,1.7,9.3,1128.37\nR001,2009-02-26,1.1,0.1,38254.63\nR001,2009-02-27,1.8,0.0,39367.64\nR001,2009-02-28,2.6,9.9,0.0\nR001,2009-03-01,-1.2,10.1,28164.87\nR001,2009-03-02,-4.6,13.9,31500.68\nR001,2009-03-03,-7.0,4.4,27534.63\nR001,2009-03-04,-3.9,7.0,1987.48\nR001,2009-03-05,-3.2,0.0,39600.0\nR001,2009-03-06,1.5,2.0,21377.48\nR001,2009-03-07,6.7,0.0,32734.48\nR001,2009-03-08,9.8,5.0,17807.91\nR001,2009-03-09,6.3,16.8,0.0\nR001,2009-03-10,6.0,2.9,34112.42\nR001,2009-03-11,3.9,18.1,8612.69\nR001,2009-03-12,2.9,2.9,37951.31\nR001,2009-03-13,7.0,5.3,21512.75\nR001,2009-03-14,4.4,21.5,1482.94\nR001,2009-03-15,-0.7,6.8,35538.02\nR001,2009-03-16,-1.9,4.0,37390.5\nR001,2009-03-17,0.1,2.6,39600.0\nR001,2009-03-18,0.5,0.1,32612.88\nR001,2009-03-19,2.9,6.1,27995.13\nR001,2009-03-20,0.0,19.3,25035.84\nR001,2009-03-21,-2.9,1.9,39944.94\nR001,2009-03-22,-1.6,0.1,41044.44\nR001,2009-03-23,2.9,6.7,13936.97\nR001,2009-03-24,2.2,9.1,1729.32\nR001,2009-03-25,3.8,0.0,42122.51\nR001,2009-03-26,6.5,0.4,41397.44\nR001,2009-03-27,6.4,4.1,40259.19\nR001,2009-03-28,5.4,3.5,23446.82\nR001,2009-03-29,5.5,0.6,37035.43\nR001,2009-03-30,8.3,0.4,44052.25\nR001,2009-03-31,10.0,0.1,39303.9\nR001,2009-04-01,9.0,1.2,40551.48\nR001,2009-04-02,7.6,7.5,29467.22\nR001,2009-04-03,4.7,0.2,44851.1\nR001,2009-04-04,5.1,0.0,45687.64\nR001,2009-04-05,6.7,0.0,45341.82\nR001,2009-04-06,8.1,0.0,45629.7\nR001,2009-04-07,8.1,2.9,22100.18\nR001,2009-04-08,7.6,11.0,29777.01\nR001,2009-04-09,4.4,3.1,45158.06\nR001,2009-04-10,2.3,3.3,42708.88\nR001,2009-04-11,1.5,0.1,46380.28\nR001,2009-04-12,2.9,0.4,46451.62\nR001,2009-04-13,4.7,0.0,46522.91\nR001,2009-04-14,8.7,0.1,39600.0\nR001,2009-04-15,7.4,4.5,4982.13\nR001,2009-04-16,5.8,10.9,154.77\nR001,2009-04-17,5.9,5.2,31420.87\nR001,2009-04-18,6.3,3.0,25963.57\nR001,2009-04-19,7.2,0.4,44853.42\nR001,2009-04-20,6.8,1.3,46800.0\nR001,2009-04-21,6.0,0.0,46800.0\nR001,2009-04-22,3.6,0.7,44975.32\nR001,2009-04-23,4.7,0.0,46699.19\nR001,2009-04-24,5.8,4.3,38844.88\nR001,2009-04-25,7.3,0.9,44400.86\nR001,2009-04-26,7.5,0.6,44302.98\nR001,2009-04-27,9.3,0.4,47659.82\nR001,2009-04-28,10.4,6.0,39793.56\nR001,2009-04-29,10.1,5.8,40699.09\nR001,2009-04-30,9.8,5.5,37003.14\nR001,2009-05-01,9.8,9.8,45933.12\nR001,2009-05-02,11.1,0.9,44162.83\nR001,2009-05-03,9.1,11.4,25831.98\nR001,2009-05-04,10.8,1.2,44675.71\nR001,2009-05-05,10.6,10.7,18771.44\nR001,2009-05-06,9.4,9.0,11963.36\nR001,2009-05-07,9.5,3.7,38208.07\nR001,2009-05-08,8.2,4.6,46830.43\nR001,2009-05-09,6.0,12.9,30080.03\nR001,2009-05-10,6.4,8.2,30283.09\nR001,2009-05-11,6.8,9.9,26029.72\nR001,2009-05-12,8.1,1.9,46580.1\nR001,2009-05-13,10.4,0.9,49514.31\nR001,2009-05-14,11.1,12.2,17720.1\nR001,2009-05-15,11.7,1.9,42212.05\nR001,2009-05-16,14.3,1.0,49220.76\nR001,2009-05-17,15.1,2.4,49101.15\nR001,2009-05-18,16.3,0.1,50400.0\nR001,2009-05-19,14.2,0.3,50300.65\nR001,2009-05-20,15.5,0.0,47922.58\nR001,2009-05-21,17.3,0.2,50164.31\nR001,2009-05-22,16.6,1.4,39214.83\nR001,2009-05-23,16.1,4.3,46487.49\nR001,2009-05-24,12.2,16.0,34549.56\nR001,2009-05-25,8.7,2.1,48968.14\nR001,2009-05-26,9.1,3.2,46008.98\nR001,2009-05-27,8.4,8.6,33637.64\nR001,2009-05-28,9.2,7.9,36247.88\nR001,2009-05-29,9.2,5.0,43185.19\nR001,2009-05-30,12.2,0.0,51159.42\nR001,2009-05-31,13.7,0.3,49157.83\nR001,2009-06-01,15.3,0.0,49819.39\nR001,2009-06-02,17.0,2.4,51824.94\nR001,2009-06-03,19.3,0.0,48053.34\nR001,2009-06-04,21.2,0.2,48998.89\nR001,2009-06-05,16.1,6.1,36517.66\nR001,2009-06-06,14.7,8.5,48637.28\nR001,2009-06-07,16.2,0.1,50996.24\nR001,2009-06-08,18.1,0.7,48677.34\nR001,2009-06-09,18.6,3.0,47646.29\nR001,2009-06-10,17.6,3.3,45039.43\nR001,2009-06-11,17.5,7.6,41590.33\nR001,2009-06-12,19.6,2.0,53129.27\nR001,2009-06-13,19.7,4.7,51648.84\nR001,2009-06-14,17.0,21.8,31166.71\nR001,2009-06-15,15.2,38.0,32984.79\nR001,2009-06-16,15.2,3.0,48675.19\nR001,2009-06-17,16.2,4.9,49852.71\nR001,2009-06-18,15.6,20.9,22275.39\nR001,2009-06-19,16.0,1.2,43553.48\nR001,2009-06-20,18.4,0.0,50777.74\nR001,2009-06-21,20.3,3.3,51207.62\nR001,2009-06-22,18.9,9.8,52483.07\nR001,2009-06-23,20.3,1.3,53951.83\nR001,2009-06-24,21.8,0.0,53958.07\nR001,2009-06-25,22.3,0.5,53962.35\nR001,2009-06-26,21.6,2.3,52655.61\nR001,2009-06-27,19.1,3.2,51924.88\nR001,2009-06-28,19.7,5.4,48743.19\nR001,2009-06-29,20.1,7.2,53382.11\nR001,2009-06-30,19.8,5.9,52262.48\nR001,2009-07-01,18.8,4.8,52283.57\nR001,2009-07-02,18.9,0.8,53015.37\nR001,2009-07-03,18.2,6.7,50928.85\nR001,2009-07-04,17.0,8.7,47062.73\nR001,2009-07-05,17.1,5.0,51653.4\nR001,2009-07-06,16.8,0.6,50640.05\nR001,2009-07-07,19.0,0.3,52612.56\nR001,2009-07-08,20.6,1.5,53398.98\nR001,2009-07-09,19.9,6.5,53798.92\nR001,2009-07-10,19.3,6.7,46111.96\nR001,2009-07-11,19.5,3.1,51831.36\nR001,2009-07-12,21.7,0.4,53647.34\nR001,2009-07-13,22.7,5.1,52719.31\nR001,2009-07-14,19.9,30.6,34911.66\nR001,2009-07-15,19.6,12.1,34765.64\nR001,2009-07-16,20.2,8.7,37034.21\nR001,2009-07-17,21.0,7.4,51391.61\nR001,2009-07-18,20.0,17.6,35200.6\nR001,2009-07-19,20.5,8.6,48711.69\nR001,2009-07-20,19.8,14.7,42513.84\nR001,2009-07-21,20.1,5.6,44075.08\nR001,2009-07-22,20.1,1.8,46636.1\nR001,2009-07-23,20.9,0.3,46020.3\nR001,2009-07-24,20.5,1.0,51559.24\nR001,2009-07-25,21.1,0.0,51872.75\nR001,2009-07-26,20.9,1.2,50525.48\nR001,2009-07-27,19.0,12.8,44763.44\nR001,2009-07-28,15.8,11.7,27955.98\nR001,2009-07-29,13.4,9.7,33926.97\nR001,2009-07-30,16.6,1.5,40147.79\nR001,2009-07-31,17.5,1.7,43971.37\nR001,2009-08-01,17.1,10.8,35524.47\nR001,2009-08-02,16.6,12.2,33997.97\nR001,2009-08-03,18.0,8.6,45739.89\nR001,2009-08-04,19.5,0.7,46482.45\nR001,2009-08-05,20.5,0.5,47901.87\nR001,2009-08-06,20.0,1.6,47928.9\nR001,2009-08-07,19.6,1.3,40505.17\nR001,2009-08-08,16.7,18.0,41951.63\nR001,2009-08-09,14.8,4.0,43985.25\nR001,2009-08-10,13.4,3.8,45107.88\nR001,2009-08-11,12.5,1.1,46398.19\nR001,2009-08-12,12.1,2.2,47087.5\nR001,2009-08-13,14.5,0.0,47117.89\nR001,2009-08-14,16.6,0.0,46960.42\nR001,2009-08-15,18.3,0.2,44864.44\nR001,2009-08-16,19.5,0.1,46291.15\nR001,2009-08-17,20.1,0.1,45870.62\nR001,2009-08-18,19.7,9.0,42433.55\nR001,2009-08-19,19.0,6.0,44014.67\nR001,2009-08-20,17.9,4.7,44671.34\nR001,2009-08-21,14.2,0.1,45486.77\nR001,2009-08-22,11.7,0.0,46800.0\nR001,2009-08-23,11.4,4.3,30341.27\nR001,2009-08-24,14.2,0.0,46482.8\nR001,2009-08-25,16.9,0.8,46800.0\nR001,2009-08-26,16.2,1.0,44930.78\nR001,2009-08-27,15.6,0.0,45932.31\nR001,2009-08-28,15.2,0.0,45726.4\nR001,2009-08-29,16.5,0.0,46515.38\nR001,2009-08-30,17.3,0.1,46410.57\nR001,2009-08-31,17.4,0.4,44866.37\nR001,2009-09-01,17.4,3.8,45558.14\nR001,2009-09-02,16.4,13.0,25895.12\nR001,2009-09-03,15.7,6.1,40794.39\nR001,2009-09-04,16.0,3.7,41059.95\nR001,2009-09-05,15.8,4.4,42118.0\nR001,2009-09-06,15.7,0.0,42028.19\nR001,2009-09-07,17.1,0.5,43615.62\nR001,2009-09-08,18.1,0.7,43667.17\nR001,2009-09-09,18.9,0.0,41188.24\nR001,2009-09-10,17.5,6.4,29812.32\nR001,2009-09-11,16.0,3.8,37685.6\nR001,2009-09-12,16.1,0.0,41240.63\nR001,2009-09-13,15.8,20.8,14082.14\nR001,2009-09-14,16.1,7.4,5433.56\nR001,2009-09-15,15.5,3.4,35148.33\nR001,2009-09-16,13.9,4.6,33499.34\nR001,2009-09-17,13.3,7.9,29296.56\nR001,2009-09-18,13.0,0.7,39586.95\nR001,2009-09-19,13.4,1.8,36328.83\nR001,2009-09-20,12.5,8.9,2974.18\nR001,2009-09-21,13.0,6.0,28130.76\nR001,2009-09-22,12.8,17.5,12752.15\nR001,2009-09-23,13.5,3.0,34499.11\nR001,2009-09-24,12.6,0.0,41694.38\nR001,2009-09-25,13.1,0.2,39648.82\nR001,2009-09-26,12.3,1.2,38502.78\nR001,2009-09-27,8.2,5.1,35638.52\nR001,2009-09-28,7.1,0.6,14608.17\nR001,2009-09-29,8.4,0.3,0.0\nR001,2009-09-30,11.0,0.0,38772.64\nR001,2009-10-01,9.0,3.7,30669.46\nR001,2009-10-02,11.6,0.0,40018.54\nR001,2009-10-03,15.9,0.0,38596.06\nR001,2009-10-04,17.6,0.0,39555.2\nR001,2009-10-05,16.5,0.0,38678.44\nR001,2009-10-06,14.0,1.5,34628.79\nR001,2009-10-07,13.1,2.6,14588.05\nR001,2009-10-08,13.1,0.9,31832.72\nR001,2009-10-09,13.4,1.1,36383.97\nR001,2009-10-10,12.9,2.9,36439.18\nR001,2009-10-11,13.3,0.0,36819.19\nR001,2009-10-12,15.3,0.0,36874.2\nR001,2009-10-13,16.7,0.0,36276.53\nR001,2009-10-14,17.4,0.0,36329.56\nR001,2009-10-15,16.0,0.0,36054.82\nR001,2009-10-16,15.6,0.0,36106.04\nR001,2009-10-17,16.7,0.0,36000.0\nR001,2009-10-18,16.2,0.0,36000.0\nR001,2009-10-19,16.6,0.0,36000.0\nR001,2009-10-20,16.4,0.0,36000.0\nR001,2009-10-21,16.3,0.0,36000.0\nR001,2009-10-22,13.9,1.0,25982.9\nR001,2009-10-23,12.1,0.4,34246.93\nR001,2009-10-24,13.3,0.0,36000.0\nR001,2009-10-25,13.0,0.0,36000.0\nR001,2009-10-26,11.6,0.0,36000.0\nR001,2009-10-27,11.8,0.0,36000.0\nR001,2009-10-28,11.3,0.0,36000.0\nR001,2009-10-29,8.2,12.4,5785.43\nR001,2009-10-30,7.4,0.0,15274.39\nR001,2009-10-31,7.8,0.2,29306.89\nR001,2009-11-01,6.5,10.8,15651.54\nR001,2009-11-02,5.9,21.6,0.0\nR001,2009-11-03,3.4,17.9,2273.34\nR001,2009-11-04,3.5,3.5,4356.04\nR001,2009-11-05,7.7,21.5,5056.51\nR001,2009-11-06,7.6,0.0,35514.65\nR001,2009-11-07,9.7,0.0,29581.32\nR001,2009-11-08,11.7,0.0,34887.59\nR001,2009-11-09,11.2,0.0,33704.93\nR001,2009-11-10,10.6,0.0,34921.99\nR001,2009-11-11,10.5,0.0,34125.35\nR001,2009-11-12,10.8,0.0,28865.35\nR001,2009-11-13,9.2,1.0,28720.52\nR001,2009-11-14,7.4,19.0,0.0\nR001,2009-11-15,5.6,11.4,8059.52\nR001,2009-11-16,4.8,4.2,30828.9\nR001,2009-11-17,4.7,0.0,14143.51\nR001,2009-11-18,3.9,0.0,32153.27\nR001,2009-11-19,4.4,0.9,16861.2\nR001,2009-11-20,2.9,23.5,18873.9\nR001,2009-11-21,0.9,0.0,30091.81\nR001,2009-11-22,0.9,10.9,0.0\nR001,2009-11-23,2.6,4.8,26921.99\nR001,2009-11-24,4.1,0.0,29635.58\nR001,2009-11-25,4.7,0.2,17238.25\nR001,2009-11-26,1.7,9.7,30976.74\nR001,2009-11-27,-0.0,0.0,30947.03\nR001,2009-11-28,2.6,0.0,29516.06\nR001,2009-11-29,3.5,0.0,25880.01\nR001,2009-11-30,3.6,0.0,12688.0\nR001,2009-12-01,3.0,0.0,30325.68\nR001,2009-12-02,2.8,0.0,30722.13\nR001,2009-12-03,4.5,0.0,30737.4\nR001,2009-12-04,4.9,0.0,21600.0\nR001,2009-12-05,6.3,0.0,26953.61\nR001,2009-12-06,4.8,3.2,13108.31\nR001,2009-12-07,5.1,2.8,0.0\nR001,2009-12-08,3.7,3.3,3109.3\nR001,2009-12-09,3.1,8.7,4743.32\nR001,2009-12-10,1.0,0.4,28870.37\nR001,2009-12-11,1.0,0.0,25113.59\nR001,2009-12-12,6.2,1.7,17550.86\nR001,2009-12-13,5.8,15.4,1049.88\nR001,2009-12-14,0.2,24.0,178.11\nR001,2009-12-15,-3.2,0.0,28860.08\nR001,2009-12-16,2.3,0.0,11559.27\nR001,2009-12-17,5.4,0.0,22525.52\nR001,2009-12-18,3.8,3.0,146.89\nR001,2009-12-19,1.7,11.5,4736.03\nR001,2009-12-20,3.6,0.9,28814.88\nR001,2009-12-21,7.2,16.3,0.0\nR001,2009-12-22,0.9,5.5,30742.02\nR001,2009-12-23,2.9,0.1,23362.56\nR001,2009-12-24,2.2,0.9,842.76\nR001,2009-12-25,5.9,0.0,29606.58\nR001,2009-12-26,4.6,9.1,0.0\nR001,2009-12-27,6.1,3.9,29080.62\nR001,2009-12-28,7.3,15.2,2780.39\nR001,2009-12-29,4.5,37.4,0.0\nR001,2009-12-30,-0.5,3.6,31509.84\nR001,2009-12-31,-0.6,0.0,31560.43\nR001,2010-01-01,4.7,10.6,7200.0\nR001,2010-01-02,6.4,18.7,3311.02\nR001,2010-01-03,2.3,29.5,2031.09\nR001,2010-01-04,-0.7,23.1,0.0\nR001,2010-01-05,-5.5,0.1,31834.61\nR001,2010-01-06,-3.5,0.0,28800.0\nR001,2010-01-07,4.4,0.0,31872.43\nR001,2010-01-08,4.2,0.2,22198.55\nR001,2010-01-09,3.1,0.0,26637.56\nR001,2010-01-10,6.1,0.0,30595.35\nR001,2010-01-11,6.4,3.9,21794.6\nR001,2010-01-12,1.7,4.4,29703.56\nR001,2010-01-13,2.5,0.0,22181.71\nR001,2010-01-14,2.8,6.6,878.19\nR001,2010-01-15,3.5,14.0,0.0\nR001,2010-01-16,2.8,12.3,17920.08\nR001,2010-01-17,1.8,0.0,25344.39\nR001,2010-01-18,3.9,0.8,31042.64\nR001,2010-01-19,4.9,2.8,1550.21\nR001,2010-01-20,6.3,1.0,22061.22\nR001,2010-01-21,4.0,14.3,0.0\nR001,2010-01-22,-1.5,8.3,26921.1\nR001,2010-01-23,-1.7,12.1,0.0\nR001,2010-01-24,0.1,12.2,85.45\nR001,2010-01-25,-2.7,9.2,2146.05\nR001,2010-01-26,-6.6,4.9,29864.23\nR001,2010-01-27,-11.2,0.0,32400.0\nR001,2010-01-28,-6.6,3.6,2409.35\nR001,2010-01-29,-0.7,3.1,10361.07\nR001,2010-01-30,2.5,12.6,13543.46\nR001,2010-01-31,4.3,0.0,32414.91\nR001,2010-02-01,3.1,9.2,11410.59\nR001,2010-02-02,2.3,0.0,32778.41\nR001,2010-02-03,2.9,0.3,32400.0\nR001,2010-02-04,-0.1,8.7,5325.67\nR001,2010-02-05,-1.7,2.1,20871.56\nR001,2010-02-06,-2.1,4.8,18105.29\nR001,2010-02-07,-3.7,0.0,32249.96\nR001,2010-02-08,-0.9,0.0,22525.04\nR001,2010-02-09,1.5,0.7,14034.48\nR001,2010-02-10,2.3,1.0,14784.36\nR001,2010-02-11,2.5,0.7,32960.36\nR001,2010-02-12,5.7,4.0,25917.22\nR001,2010-02-13,6.6,0.3,33881.58\nR001,2010-02-14,5.6,5.9,20304.92\nR001,2010-02-15,9.8,7.1,16087.93\nR001,2010-02-16,3.5,17.6,33595.78\nR001,2010-02-17,2.5,0.8,34230.45\nR001,2010-02-18,4.7,7.5,989.84\nR001,2010-02-19,5.9,12.8,5170.02\nR001,2010-02-20,8.0,1.1,35232.81\nR001,2010-02-21,8.3,18.2,0.0\nR001,2010-02-22,2.9,25.7,13572.76\nR001,2010-02-23,-0.8,0.0,32429.42\nR001,2010-02-24,1.6,0.0,38804.16\nR001,2010-02-25,2.9,0.0,38982.76\nR001,2010-02-26,4.6,0.0,39161.85\nR001,2010-02-27,5.7,0.1,38899.35\nR001,2010-02-28,5.7,2.6,36575.27\nR001,2010-03-01,5.9,0.0,37934.94\nR001,2010-03-02,5.7,0.0,22279.43\nR001,2010-03-03,4.8,9.9,2405.96\nR001,2010-03-04,1.3,9.3,35777.13\nR001,2010-03-05,4.4,4.8,28372.34\nR001,2010-03-06,4.3,19.8,9700.14\nR001,2010-03-07,-0.4,20.1,31159.67\nR001,2010-03-08,-2.0,2.5,21696.47\nR001,2010-03-09,8.4,8.7,0.0\nR001,2010-03-10,3.4,36.7,2883.69\nR001,2010-03-11,4.0,0.5,32995.67\nR001,2010-03-12,6.9,5.1,16241.75\nR001,2010-03-13,3.5,18.6,3666.97\nR001,2010-03-14,1.2,5.9,22286.38\nR001,2010-03-15,1.9,8.3,2816.96\nR001,2010-03-16,-1.5,13.1,24139.66\nR001,2010-03-17,-3.3,4.8,33316.01\nR001,2010-03-18,-3.3,3.6,39600.0\nR001,2010-03-19,-2.3,14.4,14203.57\nR001,2010-03-20,-1.2,0.0,40373.31\nR001,2010-03-21,1.9,0.0,41235.17\nR001,2010-03-22,6.9,1.0,28367.24\nR001,2010-03-23,7.5,0.2,36688.21\nR001,2010-03-24,9.0,0.0,42644.69\nR001,2010-03-25,9.6,0.0,41846.43\nR001,2010-03-26,8.2,0.0,41102.39\nR001,2010-03-27,8.8,0.0,42540.03\nR001,2010-03-28,9.8,0.0,43153.39\nR001,2010-03-29,10.1,3.2,42431.02\nR001,2010-03-30,8.3,6.9,37025.22\nR001,2010-03-31,5.8,4.2,6240.37\nR001,2010-04-01,7.0,0.7,40526.2\nR001,2010-04-02,7.8,1.1,44565.36\nR001,2010-04-03,6.8,0.8,18014.88\nR001,2010-04-04,5.3,9.1,14123.88\nR001,2010-04-05,6.7,0.2,45349.77\nR001,2010-04-06,9.0,0.0,42081.31\nR001,2010-04-07,11.5,2.0,38885.53\nR001,2010-04-08,7.1,12.3,24304.69\nR001,2010-04-09,5.8,3.8,31223.02\nR001,2010-04-10,4.8,6.1,27385.34\nR001,2010-04-11,6.1,0.7,45159.91\nR001,2010-04-12,6.5,5.8,18049.5\nR001,2010-04-13,4.8,12.0,0.0\nR001,2010-04-14,4.5,1.0,46575.63\nR001,2010-04-15,5.9,1.6,19562.35\nR001,2010-04-16,11.1,0.0,46717.06\nR001,2010-04-17,14.2,0.3,46787.89\nR001,2010-04-18,10.8,5.3,23584.49\nR001,2010-04-19,10.4,0.0,45117.46\nR001,2010-04-20,13.1,0.0,46800.0\nR001,2010-04-21,12.3,17.7,12835.86\nR001,2010-04-22,8.7,8.8,28360.32\nR001,2010-04-23,9.3,0.6,45182.0\nR001,2010-04-24,9.5,0.3,42656.12\nR001,2010-04-25,10.1,0.7,45928.23\nR001,2010-04-26,9.8,5.5,35093.41\nR001,2010-04-27,8.7,0.0,39978.67\nR001,2010-04-28,10.3,0.0,47626.95\nR001,2010-04-29,10.7,0.2,48254.56\nR001,2010-04-30,10.0,1.0,43091.04\nR001,2010-05-01,10.2,5.1,46017.47\nR001,2010-05-02,10.3,0.1,47135.21\nR001,2010-05-03,9.0,4.5,16518.02\nR001,2010-05-04,9.1,3.4,28007.98\nR001,2010-05-05,10.3,4.6,44530.39\nR001,2010-05-06,11.9,3.7,48346.29\nR001,2010-05-07,13.1,4.4,47933.07\nR001,2010-05-08,13.8,0.6,49647.15\nR001,2010-05-09,13.6,0.5,46454.32\nR001,2010-05-10,12.0,8.9,20960.72\nR001,2010-05-11,11.7,7.2,44514.91\nR001,2010-05-12,12.6,0.4,47826.44\nR001,2010-05-13,14.6,0.0,48487.76\nR001,2010-05-14,17.1,0.0,49694.4\nR001,2010-05-15,18.5,0.0,50400.0\nR001,2010-05-16,19.0,0.4,43200.0\nR001,2010-05-17,18.1,5.6,40413.15\nR001,2010-05-18,16.1,0.9,44906.51\nR001,2010-05-19,13.5,10.2,41040.75\nR001,2010-05-20,11.6,0.0,50400.0\nR001,2010-05-21,11.4,1.0,43131.85\nR001,2010-05-22,12.1,1.8,41313.51\nR001,2010-05-23,12.8,5.4,46039.45\nR001,2010-05-24,11.0,8.4,37683.74\nR001,2010-05-25,11.9,1.6,44078.98\nR001,2010-05-26,13.9,0.0,51256.0\nR001,2010-05-27,16.9,0.0,51233.27\nR001,2010-05-28,16.8,0.0,51210.76\nR001,2010-05-29,17.1,0.5,51188.54\nR001,2010-05-30,16.9,6.4,51166.62\nR001,2010-05-31,17.6,1.6,50783.39\nR001,2010-06-01,19.5,1.5,51848.4\nR001,2010-06-02,22.1,0.6,45032.35\nR001,2010-06-03,21.9,0.5,47701.99\nR001,2010-06-04,20.8,0.1,50699.5\nR001,2010-06-05,21.4,1.0,49862.22\nR001,2010-06-06,21.1,3.3,49890.65\nR001,2010-06-07,19.6,20.1,37207.01\nR001,2010-06-08,16.2,43.9,6909.91\nR001,2010-06-09,17.0,17.7,30354.0\nR001,2010-06-10,18.4,1.6,52424.08\nR001,2010-06-11,20.0,8.9,51677.47\nR001,2010-06-12,20.5,4.8,53131.54\nR001,2010-06-13,22.2,0.4,53769.51\nR001,2010-06-14,23.6,0.6,52742.16\nR001,2010-06-15,23.0,0.8,49192.15\nR001,2010-06-16,20.3,1.5,38995.23\nR001,2010-06-17,19.6,6.0,47467.19\nR001,2010-06-18,18.8,9.0,40850.31\nR001,2010-06-19,19.1,2.8,45713.2\nR001,2010-06-20,20.0,5.5,52107.87\nR001,2010-06-21,19.0,21.4,38916.56\nR001,2010-06-22,18.2,7.7,52710.12\nR001,2010-06-23,20.3,0.2,53442.74\nR001,2010-06-24,17.0,21.5,18719.68\nR001,2010-06-25,16.0,4.5,31616.2\nR001,2010-06-26,18.4,3.5,48541.59\nR001,2010-06-27,16.6,14.2,39088.68\nR001,2010-06-28,18.1,6.1,51732.31\nR001,2010-06-29,17.4,18.4,46540.14\nR001,2010-06-30,17.8,7.0,45046.77\nR001,2010-07-01,18.7,3.9,53340.55\nR001,2010-07-02,18.8,5.3,53013.76\nR001,2010-07-03,20.6,2.3,51083.93\nR001,2010-07-04,19.9,6.3,45054.5\nR001,2010-07-05,19.9,6.9,42236.41\nR001,2010-07-06,21.1,0.5,48701.0\nR001,2010-07-07,21.1,0.5,51949.92\nR001,2010-07-08,23.0,0.2,53399.31\nR001,2010-07-09,22.7,3.5,52189.77\nR001,2010-07-10,23.4,1.8,48615.21\nR001,2010-07-11,23.6,1.0,52697.9\nR001,2010-07-12,23.2,2.0,50620.85\nR001,2010-07-13,21.9,3.9,52201.58\nR001,2010-07-14,20.1,8.9,45967.76\nR001,2010-07-15,19.5,10.9,41542.83\nR001,2010-07-16,20.1,0.0,53468.91\nR001,2010-07-17,20.7,0.2,53124.28\nR001,2010-07-18,18.7,6.0,13199.71\nR001,2010-07-19,19.5,10.0,45333.35\nR001,2010-07-20,21.5,0.8,52068.78\nR001,2010-07-21,21.1,4.7,51676.67\nR001,2010-07-22,20.3,9.7,50339.9\nR001,2010-07-23,19.5,15.4,46662.18\nR001,2010-07-24,18.7,26.4,29442.97\nR001,2010-07-25,20.0,6.9,45822.83\nR001,2010-07-26,22.2,0.7,51846.7\nR001,2010-07-27,24.5,0.2,51461.52\nR001,2010-07-28,24.7,0.7,51077.94\nR001,2010-07-29,23.2,1.2,49263.94\nR001,2010-07-30,24.1,0.2,49664.67\nR001,2010-07-31,24.6,0.0,50629.86\nR001,2010-08-01,24.9,0.0,50116.92\nR001,2010-08-02,23.8,0.5,48795.58\nR001,2010-08-03,22.5,0.9,48511.67\nR001,2010-08-04,22.4,0.0,49257.15\nR001,2010-08-05,23.4,0.5,48975.32\nR001,2010-08-06,24.0,0.8,48696.49\nR001,2010-08-07,23.8,0.9,48420.02\nR001,2010-08-08,23.4,0.0,48485.01\nR001,2010-08-09,23.7,0.0,48006.0\nR001,2010-08-10,24.1,0.9,47696.13\nR001,2010-08-11,23.5,4.8,47051.17\nR001,2010-08-12,22.1,7.1,47081.16\nR001,2010-08-13,22.0,2.2,47111.15\nR001,2010-08-14,22.2,2.0,47141.09\nR001,2010-08-15,20.6,10.1,46837.37\nR001,2010-08-16,20.2,12.4,45258.39\nR001,2010-08-17,18.5,23.7,38313.9\nR001,2010-08-18,18.3,7.2,38687.94\nR001,2010-08-19,18.6,3.7,46800.0\nR001,2010-08-20,18.5,0.5,45567.81\nR001,2010-08-21,17.4,0.0,46398.65\nR001,2010-08-22,17.4,0.6,43336.29\nR001,2010-08-23,18.3,0.0,46800.0\nR001,2010-08-24,20.0,0.4,46800.0\nR001,2010-08-25,20.8,0.8,46800.0\nR001,2010-08-26,19.5,3.3,46800.0\nR001,2010-08-27,19.3,1.7,44990.5\nR001,2010-08-28,19.7,0.0,45316.57\nR001,2010-08-29,22.3,0.0,46536.94\nR001,2010-08-30,23.1,0.1,46433.77\nR001,2010-08-31,23.4,0.0,46330.44\nR001,2010-09-01,25.5,0.7,46227.05\nR001,2010-09-02,19.2,22.1,30815.71\nR001,2010-09-03,18.0,0.2,43752.28\nR001,2010-09-04,20.2,0.0,43601.43\nR001,2010-09-05,21.2,1.1,43816.2\nR001,2010-09-06,20.6,4.3,39659.6\nR001,2010-09-07,20.2,9.8,42201.06\nR001,2010-09-08,18.0,4.6,39363.98\nR001,2010-09-09,17.0,0.0,43200.0\nR001,2010-09-10,16.8,0.1,39501.7\nR001,2010-09-11,17.4,0.3,43200.0\nR001,2010-09-12,20.2,0.0,43200.0\nR001,2010-09-13,18.4,19.1,20399.55\nR001,2010-09-14,16.2,15.8,0.0\nR001,2010-09-15,15.5,6.4,20946.81\nR001,2010-09-16,16.6,0.0,37912.16\nR001,2010-09-17,17.3,0.0,42913.8\nR001,2010-09-18,17.9,0.2,41478.06\nR001,2010-09-19,17.2,2.1,36005.08\nR001,2010-09-20,17.6,0.0,40272.79\nR001,2010-09-21,19.3,0.0,42216.96\nR001,2010-09-22,17.7,5.8,40963.36\nR001,2010-09-23,15.6,4.8,27004.28\nR001,2010-09-24,16.1,0.0,40411.05\nR001,2010-09-25,18.6,0.0,41720.95\nR001,2010-09-26,19.2,1.2,40162.89\nR001,2010-09-27,18.6,0.4,40517.46\nR001,2010-09-28,18.7,0.5,40234.31\nR001,2010-09-29,16.7,9.7,38411.92\nR001,2010-09-30,15.9,14.1,29312.06\nR001,2010-10-01,11.1,2.6,36924.38\nR001,2010-10-02,10.7,0.9,32601.76\nR001,2010-10-03,10.5,24.8,7.11\nR001,2010-10-04,9.0,1.7,36871.57\nR001,2010-10-05,7.8,1.7,30527.45\nR001,2010-10-06,7.0,1.2,34410.38\nR001,2010-10-07,8.0,0.6,33122.56\nR001,2010-10-08,11.4,4.7,21142.44\nR001,2010-10-09,11.9,11.9,26531.66\nR001,2010-10-10,9.4,20.4,23282.55\nR001,2010-10-11,6.6,14.4,25942.18\nR001,2010-10-12,6.6,11.4,29727.52\nR001,2010-10-13,7.3,2.2,28800.0\nR001,2010-10-14,8.4,0.2,32808.79\nR001,2010-10-15,10.0,23.5,6682.91\nR001,2010-10-16,11.8,14.3,19428.63\nR001,2010-10-17,11.5,4.3,17423.82\nR001,2010-10-18,12.6,0.3,34550.73\nR001,2010-10-19,13.7,0.0,34321.69\nR001,2010-10-20,16.7,5.5,28679.92\nR001,2010-10-21,15.9,3.8,35584.28\nR001,2010-10-22,11.6,2.3,14751.74\nR001,2010-10-23,10.5,1.2,15257.66\nR001,2010-10-24,10.9,0.8,21343.93\nR001,2010-10-25,10.2,0.0,36000.0\nR001,2010-10-26,11.8,0.0,36000.0\nR001,2010-10-27,11.2,0.4,30234.01\nR001,2010-10-28,9.8,17.8,2024.11\nR001,2010-10-29,10.5,8.8,3321.99\nR001,2010-10-30,7.9,18.5,14801.79\nR001,2010-10-31,2.3,0.1,36000.0\nR001,2010-11-01,3.3,0.0,36000.0\nR001,2010-11-02,4.5,0.0,35953.15\nR001,2010-11-03,7.6,0.0,35874.7\nR001,2010-11-04,9.0,0.0,35795.67\nR001,2010-11-05,8.9,0.0,35716.13\nR001,2010-11-06,9.5,0.0,35562.33\nR001,2010-11-07,10.3,0.0,35408.51\nR001,2010-11-08,11.2,0.0,35255.8\nR001,2010-11-09,11.9,0.0,35104.39\nR001,2010-11-10,11.4,0.0,34954.45\nR001,2010-11-11,13.1,0.0,34806.17\nR001,2010-11-12,12.9,0.0,34114.72\nR001,2010-11-13,13.1,0.0,34460.22\nR001,2010-11-14,12.1,0.0,34373.14\nR001,2010-11-15,11.6,0.0,34078.08\nR001,2010-11-16,11.3,0.0,34069.31\nR001,2010-11-17,10.6,0.0,32055.31\nR001,2010-11-18,11.2,0.0,32964.17\nR001,2010-11-19,11.7,0.0,29645.82\nR001,2010-11-20,8.5,0.0,32587.44\nR001,2010-11-21,6.7,0.0,30155.66\nR001,2010-11-22,6.3,4.3,14669.92\nR001,2010-11-23,6.0,0.0,30296.31\nR001,2010-11-24,8.7,8.6,27452.31\nR001,2010-11-25,8.4,28.4,19977.18\nR001,2010-11-26,8.0,21.4,428.23\nR001,2010-11-27,9.2,11.3,5380.64\nR001,2010-11-28,8.7,27.2,0.0\nR001,2010-11-29,9.0,0.4,25500.76\nR001,2010-11-30,10.5,3.0,28800.0\nR001,2010-12-01,9.7,6.0,9515.67\nR001,2010-12-02,10.2,13.4,17500.4\nR001,2010-12-03,10.5,0.9,1534.96\nR001,2010-12-04,11.4,0.0,29219.25\nR001,2010-12-05,14.7,0.0,28846.24\nR001,2010-12-06,8.4,0.0,28181.17\nR001,2010-12-07,5.7,0.0,30014.45\nR001,2010-12-08,6.2,0.0,26791.95\nR001,2010-12-09,7.1,0.0,30027.82\nR001,2010-12-10,7.4,0.0,25062.11\nR001,2010-12-11,8.2,0.2,26821.44\nR001,2010-12-12,6.0,8.9,20687.36\nR001,2010-12-13,2.1,9.6,23644.49\nR001,2010-12-14,1.2,5.9,10171.33\nR001,2010-12-15,3.2,7.6,0.0\nR001,2010-12-16,3.3,3.6,0.0\nR001,2010-12-17,4.9,3.6,25436.76\nR001,2010-12-18,2.4,19.8,0.0\nR001,2010-12-19,0.6,0.2,30389.59\nR001,2010-12-20,3.5,5.5,30378.18\nR001,2010-12-21,5.3,0.0,30365.25\nR001,2010-12-22,6.9,0.0,30742.06\nR001,2010-12-23,5.6,0.0,31117.7\nR001,2010-12-24,4.3,0.0,31243.39\nR001,2010-12-25,3.8,0.0,31280.78\nR001,2010-12-26,7.2,0.0,20720.79\nR001,2010-12-27,10.2,0.0,19113.5\nR001,2010-12-28,9.8,0.0,16232.34\nR001,2010-12-29,8.2,0.0,31385.93\nR001,2010-12-30,5.0,0.8,20084.71\nR001,2010-12-31,2.7,8.8,0.0\nR001,2011-01-01,0.4,0.3,30026.91\nR001,2011-01-02,0.2,0.0,31643.8\nR001,2011-01-03,1.3,0.0,26729.53\nR001,2011-01-04,2.7,2.3,29488.01\nR001,2011-01-05,1.9,7.1,16865.76\nR001,2011-01-06,1.4,4.1,19912.07\nR001,2011-01-07,1.5,0.0,17777.75\nR001,2011-01-08,1.8,8.3,1202.23\nR001,2011-01-09,0.8,2.2,22744.04\nR001,2011-01-10,-0.8,0.3,32137.64\nR001,2011-01-11,-0.8,0.0,32207.13\nR001,2011-01-12,-0.5,0.0,32277.88\nR001,2011-01-13,-0.3,0.0,32349.72\nR001,2011-01-14,0.4,0.0,28800.0\nR001,2011-01-15,0.8,0.3,25483.85\nR001,2011-01-16,0.7,0.9,26914.0\nR001,2011-01-17,0.9,8.1,18346.94\nR001,2011-01-18,-0.7,8.0,0.0\nR001,2011-01-19,-1.0,4.0,24152.56\nR001,2011-01-20,-3.8,0.0,32400.0\nR001,2011-01-21,-3.1,0.0,26247.39\nR001,2011-01-22,-2.2,0.0,32400.0\nR001,2011-01-23,1.8,0.0,30657.92\nR001,2011-01-24,2.9,26.5,13205.87\nR001,2011-01-25,4.4,9.9,24720.44\nR001,2011-01-26,3.0,18.9,1302.15\nR001,2011-01-27,0.1,10.4,1177.83\nR001,2011-01-28,-2.6,2.3,32400.0\nR001,2011-01-29,-5.6,0.0,32400.0\nR001,2011-01-30,-4.2,0.0,32400.0\nR001,2011-01-31,-2.0,8.0,283.25\nR001,2011-02-01,-3.3,4.1,29697.12\nR001,2011-02-02,-5.8,4.0,16135.27\nR001,2011-02-03,-4.6,0.9,32641.35\nR001,2011-02-04,-3.0,6.5,21595.91\nR001,2011-02-05,-2.6,3.8,18557.87\nR001,2011-02-06,-0.4,34.5,0.0\nR001,2011-02-07,-1.7,4.0,32462.99\nR001,2011-02-08,-0.3,6.1,1168.15\nR001,2011-02-09,1.0,6.9,24296.65\nR001,2011-02-10,-4.4,2.1,33360.85\nR001,2011-02-11,-7.7,0.1,34676.66\nR001,2011-02-12,-3.2,6.6,1234.41\nR001,2011-02-13,-4.7,25.5,10937.52\nR001,2011-02-14,-7.7,7.3,22169.69\nR001,2011-02-15,-7.2,5.6,35004.67\nR001,2011-02-16,-10.8,0.0,37318.22\nR001,2011-02-17,-9.6,0.0,37432.56\nR001,2011-02-18,-7.0,0.0,36128.56\nR001,2011-02-19,-3.3,0.0,30598.87\nR001,2011-02-20,0.5,2.8,0.0\nR001,2011-02-21,1.4,9.8,175.97\nR001,2011-02-22,0.5,0.0,38415.89\nR001,2011-02-23,2.7,0.1,25250.88\nR001,2011-02-24,4.4,0.0,38194.38\nR001,2011-02-25,4.7,0.4,37225.32\nR001,2011-02-26,3.6,0.5,23129.3\nR001,2011-02-27,0.9,7.5,0.0\nR001,2011-02-28,2.1,0.1,39427.06\nR001,2011-03-01,1.6,2.2,26746.12\nR001,2011-03-02,0.1,9.7,27948.21\nR001,2011-03-03,-2.4,3.2,39600.0\nR001,2011-03-04,-3.3,1.6,39600.0\nR001,2011-03-05,-3.0,0.0,36685.4\nR001,2011-03-06,1.1,1.1,39600.0\nR001,2011-03-07,-0.3,27.1,0.0\nR001,2011-03-08,-0.4,2.8,22396.32\nR001,2011-03-09,3.0,0.0,39600.0\nR001,2011-03-10,2.8,4.8,19650.97\nR001,2011-03-11,1.7,1.6,31253.9\nR001,2011-03-12,-0.5,3.6,33647.09\nR001,2011-03-13,0.4,2.3,36000.0\nR001,2011-03-14,2.9,0.0,39600.0\nR001,2011-03-15,5.5,0.0,39638.9\nR001,2011-03-16,7.8,0.0,39600.0\nR001,2011-03-17,6.1,0.0,39750.66\nR001,2011-03-18,5.4,4.1,16327.13\nR001,2011-03-19,5.0,11.6,6392.65\nR001,2011-03-20,5.7,0.0,39600.0\nR001,2011-03-21,4.8,7.3,6634.02\nR001,2011-03-22,2.9,9.9,3077.79\nR001,2011-03-23,2.4,3.9,32776.51\nR001,2011-03-24,0.7,5.4,22425.64\nR001,2011-03-25,1.1,12.0,21036.67\nR001,2011-03-26,4.0,0.0,43079.56\nR001,2011-03-27,5.8,0.6,42051.63\nR001,2011-03-28,6.3,0.0,42221.57\nR001,2011-03-29,5.9,0.0,43617.76\nR001,2011-03-30,5.8,0.5,32446.42\nR001,2011-03-31,7.3,2.0,39665.21\nR001,2011-04-01,4.6,1.0,42507.39\nR001,2011-04-02,6.7,0.0,25548.09\nR001,2011-04-03,7.6,3.4,14638.5\nR001,2011-04-04,6.6,0.4,23022.46\nR001,2011-04-05,4.5,12.8,0.0\nR001,2011-04-06,3.7,5.7,16838.43\nR001,2011-04-07,2.4,11.7,26077.92\nR001,2011-04-08,4.1,15.9,4638.36\nR001,2011-04-09,2.1,28.0,33388.46\nR001,2011-04-10,0.6,9.6,40034.56\nR001,2011-04-11,-0.7,11.7,34715.02\nR001,2011-04-12,-0.9,0.0,30086.54\nR001,2011-04-13,0.1,0.3,46060.24\nR001,2011-04-14,5.1,0.0,46558.57\nR001,2011-04-15,5.9,15.7,13769.47\nR001,2011-04-16,3.6,5.2,46701.28\nR001,2011-04-17,3.8,3.8,29317.4\nR001,2011-04-18,6.1,0.0,39600.0\nR001,2011-04-19,11.7,1.8,41952.71\nR001,2011-04-20,8.8,9.1,44033.21\nR001,2011-04-21,3.4,20.5,12468.65\nR001,2011-04-22,5.5,0.0,45017.64\nR001,2011-04-23,5.8,1.2,16331.64\nR001,2011-04-24,6.0,3.0,1834.48\nR001,2011-04-25,8.0,1.5,45545.38\nR001,2011-04-26,8.8,3.5,36088.54\nR001,2011-04-27,10.2,0.6,39842.9\nR001,2011-04-28,12.4,1.0,43050.96\nR001,2011-04-29,10.8,5.3,7283.46\nR001,2011-04-30,10.0,5.9,22185.99\nR001,2011-05-01,10.8,0.2,44270.6\nR001,2011-05-02,12.3,0.0,46811.12\nR001,2011-05-03,12.4,1.6,46976.73\nR001,2011-05-04,12.3,2.5,38070.95\nR001,2011-05-05,10.7,15.0,1647.26\nR001,2011-05-06,9.6,7.8,19271.41\nR001,2011-05-07,8.3,6.1,0.0\nR001,2011-05-08,7.3,6.1,11454.15\nR001,2011-05-09,8.6,0.3,44177.46\nR001,2011-05-10,7.8,1.9,47041.77\nR001,2011-05-11,10.3,0.6,46547.2\nR001,2011-05-12,9.8,12.5,20113.93\nR001,2011-05-13,9.1,14.8,2270.51\nR001,2011-05-14,10.3,13.5,12898.18\nR001,2011-05-15,9.9,3.7,23806.61\nR001,2011-05-16,11.2,0.3,47470.67\nR001,2011-05-17,13.2,0.0,50322.87\nR001,2011-05-18,15.0,0.0,49034.73\nR001,2011-05-19,15.4,4.8,37323.81\nR001,2011-05-20,12.7,17.9,29787.33\nR001,2011-05-21,13.1,8.7,46422.88\nR001,2011-05-22,11.8,12.8,1632.1\nR001,2011-05-23,11.8,30.3,3758.79\nR001,2011-05-24,14.3,2.5,34462.41\nR001,2011-05-25,15.0,7.4,43689.05\nR001,2011-05-26,13.9,13.4,39662.07\nR001,2011-05-27,15.1,0.4,45953.48\nR001,2011-05-28,17.1,1.1,47183.8\nR001,2011-05-29,17.4,5.4,28057.87\nR001,2011-05-30,18.0,2.6,50488.09\nR001,2011-05-31,19.8,3.8,51509.16\nR001,2011-06-01,20.1,6.7,51126.16\nR001,2011-06-02,17.7,16.1,19607.76\nR001,2011-06-03,17.0,19.9,10874.45\nR001,2011-06-04,15.6,13.3,17140.79\nR001,2011-06-05,15.1,8.0,45884.88\nR001,2011-06-06,16.3,1.4,49126.49\nR001,2011-06-07,16.6,0.9,50415.84\nR001,2011-06-08,17.7,0.9,50994.81\nR001,2011-06-09,17.1,0.0,51439.02\nR001,2011-06-10,17.8,0.0,53527.81\nR001,2011-06-11,15.5,4.3,36257.31\nR001,2011-06-12,15.5,5.2,35121.07\nR001,2011-06-13,15.8,3.9,42292.52\nR001,2011-06-14,16.5,4.9,46568.8\nR001,2011-06-15,16.0,4.4,42692.5\nR001,2011-06-16,16.9,0.0,51050.45\nR001,2011-06-17,17.9,0.9,52714.78\nR001,2011-06-18,18.1,2.2,52705.25\nR001,2011-06-19,17.1,13.2,51225.36\nR001,2011-06-20,16.7,10.6,41144.85\nR001,2011-06-21,16.6,3.6,48459.53\nR001,2011-06-22,15.7,0.2,52673.45\nR001,2011-06-23,14.5,4.0,51475.2\nR001,2011-06-24,15.6,0.1,53954.99\nR001,2011-06-25,17.8,0.0,53966.85\nR001,2011-06-26,18.4,2.3,51919.19\nR001,2011-06-27,13.9,21.3,5959.43\nR001,2011-06-28,14.2,23.6,31018.17\nR001,2011-06-29,16.8,2.3,44850.74\nR001,2011-06-30,15.1,22.6,18006.19\nR001,2011-07-01,16.4,0.0,53951.77\nR001,2011-07-02,17.4,3.7,47055.46\nR001,2011-07-03,18.3,0.5,49308.49\nR001,2011-07-04,19.4,0.9,52892.78\nR001,2011-07-05,18.5,3.1,47535.91\nR001,2011-07-06,20.4,0.0,53511.93\nR001,2011-07-07,22.1,3.4,52304.01\nR001,2011-07-08,20.9,3.7,38791.47\nR001,2011-07-09,21.5,1.3,41991.59\nR001,2011-07-10,21.0,2.6,49279.21\nR001,2011-07-11,19.9,0.3,49822.55\nR001,2011-07-12,20.4,1.5,52704.39\nR001,2011-07-13,19.8,9.3,47375.47\nR001,2011-07-14,21.5,4.1,52655.7\nR001,2011-07-15,21.6,8.3,51609.5\nR001,2011-07-16,20.3,4.6,39451.75\nR001,2011-07-17,20.0,15.7,36101.71\nR001,2011-07-18,20.8,8.0,52476.86\nR001,2011-07-19,20.3,11.1,45832.42\nR001,2011-07-20,19.8,6.3,24830.86\nR001,2011-07-21,20.5,12.6,52043.24\nR001,2011-07-22,21.3,2.4,52006.78\nR001,2011-07-23,19.1,5.1,47402.35\nR001,2011-07-24,19.2,2.6,48226.23\nR001,2011-07-25,21.6,0.9,51545.06\nR001,2011-07-26,24.9,0.3,51859.13\nR001,2011-07-27,25.6,0.1,51472.59\nR001,2011-07-28,24.4,1.2,50246.3\nR001,2011-07-29,25.6,0.0,49602.77\nR001,2011-07-30,25.9,0.3,49308.17\nR001,2011-07-31,23.7,4.1,43937.93\nR001,2011-08-01,23.6,0.0,45487.23\nR001,2011-08-02,23.1,1.7,45805.2\nR001,2011-08-03,20.0,29.0,23979.2\nR001,2011-08-04,19.6,1.3,47531.91\nR001,2011-08-05,18.7,0.0,47595.81\nR001,2011-08-06,17.5,0.0,48259.03\nR001,2011-08-07,18.3,0.8,45545.5\nR001,2011-08-08,19.0,1.1,47293.6\nR001,2011-08-09,19.7,0.0,46983.61\nR001,2011-08-10,20.5,0.0,48026.21\nR001,2011-08-11,21.4,0.0,47716.57\nR001,2011-08-12,20.8,3.5,45646.48\nR001,2011-08-13,17.9,16.6,7799.68\nR001,2011-08-14,17.8,8.2,45308.76\nR001,2011-08-15,18.9,3.7,45838.18\nR001,2011-08-16,19.4,0.9,45758.69\nR001,2011-08-17,18.5,0.0,45503.34\nR001,2011-08-18,17.9,0.9,43915.59\nR001,2011-08-19,18.5,1.2,45562.6\nR001,2011-08-20,18.4,0.0,45896.35\nR001,2011-08-21,17.9,0.0,46800.0\nR001,2011-08-22,16.5,2.2,45886.31\nR001,2011-08-23,16.9,0.1,43280.84\nR001,2011-08-24,16.8,1.1,46250.37\nR001,2011-08-25,16.3,1.7,46747.21\nR001,2011-08-26,15.6,2.5,43795.17\nR001,2011-08-27,15.7,0.0,46769.13\nR001,2011-08-28,16.6,0.0,46669.07\nR001,2011-08-29,18.7,0.0,46568.61\nR001,2011-08-30,21.0,0.0,46466.81\nR001,2011-08-31,20.1,2.9,45805.72\nR001,2011-09-01,18.5,6.3,42625.22\nR001,2011-09-02,17.8,0.3,43532.52\nR001,2011-09-03,16.1,3.7,33609.27\nR001,2011-09-04,15.0,6.6,38173.39\nR001,2011-09-05,13.0,3.4,42107.38\nR001,2011-09-06,13.1,0.1,42898.88\nR001,2011-09-07,12.9,0.0,42953.14\nR001,2011-09-08,14.4,0.0,43962.91\nR001,2011-09-09,14.7,3.3,36596.99\nR001,2011-09-10,13.2,6.8,41475.87\nR001,2011-09-11,12.2,0.0,40174.14\nR001,2011-09-12,13.0,4.7,34014.68\nR001,2011-09-13,13.8,0.0,43200.0\nR001,2011-09-14,16.3,0.0,43133.33\nR001,2011-09-15,17.5,2.2,42146.99\nR001,2011-09-16,17.0,2.0,31330.97\nR001,2011-09-17,15.6,3.6,36218.84\nR001,2011-09-18,14.9,0.0,40514.42\nR001,2011-09-19,15.5,0.0,41184.64\nR001,2011-09-20,16.6,0.0,42728.41\nR001,2011-09-21,16.3,0.4,38537.09\nR001,2011-09-22,16.2,0.0,39174.49\nR001,2011-09-23,16.8,1.1,38167.47\nR001,2011-09-24,15.2,10.6,18682.18\nR001,2011-09-25,14.0,1.7,37772.01\nR001,2011-09-26,11.9,0.2,40472.66\nR001,2011-09-27,11.4,0.0,40189.43\nR001,2011-09-28,11.9,0.0,40914.95\nR001,2011-09-29,11.2,1.1,38106.64\nR001,2011-09-30,8.1,26.7,25972.38\nR001,2011-10-01,6.6,0.3,39380.03\nR001,2011-10-02,7.6,29.0,1235.67\nR001,2011-10-03,4.6,2.9,38447.85\nR001,2011-10-04,6.0,0.0,36000.0\nR001,2011-10-05,8.5,4.4,33009.79\nR001,2011-10-06,10.8,0.2,36584.45\nR001,2011-10-07,12.3,0.0,38186.29\nR001,2011-10-08,13.5,0.0,37948.48\nR001,2011-10-09,15.8,0.0,32742.29\nR001,2011-10-10,16.8,0.6,840.35\nR001,2011-10-11,19.8,0.0,37112.53\nR001,2011-10-12,18.2,8.9,32545.76\nR001,2011-10-13,12.0,8.5,8125.71\nR001,2011-10-14,10.6,9.1,30969.88\nR001,2011-10-15,8.5,23.6,14750.43\nR001,2011-10-16,9.4,7.4,6311.76\nR001,2011-10-17,11.0,2.6,19365.04\nR001,2011-10-18,11.1,6.9,32417.12\nR001,2011-10-19,5.9,21.7,19261.45\nR001,2011-10-20,5.3,0.0,27731.68\nR001,2011-10-21,7.1,0.0,36000.0\nR001,2011-10-22,8.0,0.0,35068.98\nR001,2011-10-23,8.2,0.1,31117.04\nR001,2011-10-24,8.9,0.0,34284.66\nR001,2011-10-25,7.7,0.0,28226.63\nR001,2011-10-26,6.8,4.3,36000.0\nR001,2011-10-27,5.2,2.7,36000.0\nR001,2011-10-28,6.4,0.3,36000.0\nR001,2011-10-29,5.6,0.0,36000.0\nR001,2011-10-30,4.8,0.5,36000.0\nR001,2011-10-31,4.3,0.0,36000.0\nR001,2011-11-01,4.6,0.0,36000.0\nR001,2011-11-02,4.6,0.1,18593.02\nR001,2011-11-03,5.4,0.0,34495.19\nR001,2011-11-04,4.7,0.0,35813.37\nR001,2011-11-05,5.8,0.0,34522.36\nR001,2011-11-06,3.7,0.0,32816.31\nR001,2011-11-07,-0.5,0.0,35438.3\nR001,2011-11-08,0.1,0.0,33846.79\nR001,2011-11-09,1.7,12.7,3101.62\nR001,2011-11-10,-1.1,13.6,15434.3\nR001,2011-11-11,-2.1,16.4,904.83\nR001,2011-11-12,-5.3,4.4,31286.21\nR001,2011-11-13,-6.9,0.0,33739.74\nR001,2011-11-14,-5.3,0.0,34404.3\nR001,2011-11-15,-3.5,0.0,34267.1\nR001,2011-11-16,-0.7,0.3,31215.91\nR001,2011-11-17,0.4,1.8,23862.09\nR001,2011-11-18,-0.7,0.0,32235.63\nR001,2011-11-19,-1.2,0.0,32228.14\nR001,2011-11-20,-1.9,0.0,32222.59\nR001,2011-11-21,-1.5,0.0,30875.98\nR001,2011-11-22,-1.2,0.0,30204.61\nR001,2011-11-23,-0.5,1.2,15430.38\nR001,2011-11-24,-5.8,0.0,31296.12\nR001,2011-11-25,-3.6,0.1,30037.25\nR001,2011-11-26,-2.6,4.0,9008.4\nR001,2011-11-27,-1.3,3.9,17919.05\nR001,2011-11-28,0.5,2.2,28290.34\nR001,2011-11-29,-2.7,13.6,25546.45\nR001,2011-11-30,-3.9,15.7,16440.27\nR001,2011-12-01,-6.0,0.0,25607.79\nR001,2011-12-02,-1.9,5.2,0.0\nR001,2011-12-03,-1.3,1.1,29588.56\nR001,2011-12-04,-0.9,0.0,30365.31\nR001,2011-12-05,1.2,0.0,30378.1\nR001,2011-12-06,3.9,0.0,29238.48\nR001,2011-12-07,6.8,0.0,13390.81\nR001,2011-12-08,5.1,1.0,19633.58\nR001,2011-12-09,2.8,17.3,0.0\nR001,2011-12-10,-1.4,1.6,30031.49\nR001,2011-12-11,-1.6,0.0,21790.34\nR001,2011-12-12,2.0,7.0,8903.93\nR001,2011-12-13,1.2,0.0,29162.53\nR001,2011-12-14,2.6,0.0,20882.5\nR001,2011-12-15,4.2,0.0,27308.29\nR001,2011-12-16,2.4,0.0,30412.7\nR001,2011-12-17,3.6,0.0,23799.93\nR001,2011-12-18,5.2,0.0,5946.31\nR001,2011-12-19,7.2,0.0,24059.68\nR001,2011-12-20,7.0,0.0,16134.02\nR001,2011-12-21,7.6,8.1,12743.83\nR001,2011-12-22,4.1,5.9,24229.1\nR001,2011-12-23,2.7,0.2,17975.25\nR001,2011-12-24,3.5,5.3,6637.65\nR001,2011-12-25,1.9,3.4,22645.32\nR001,2011-12-26,0.3,0.4,25887.25\nR001,2011-12-27,-0.9,1.3,26880.71\nR001,2011-12-28,-0.2,2.0,26151.74\nR001,2011-12-29,-0.8,0.1,31429.17\nR001,2011-12-30,-0.9,0.0,31476.05\nR001,2011-12-31,1.6,0.0,31525.45\nR001,2012-01-01,1.9,0.8,28625.86\nR001,2012-01-02,-2.8,12.5,29964.64\nR001,2012-01-03,-4.2,2.6,14878.38\nR001,2012-01-04,-0.0,1.0,18943.92\nR001,2012-01-05,-0.7,0.0,30406.62\nR001,2012-01-06,1.0,0.0,22930.44\nR001,2012-01-07,-0.2,0.0,29310.15\nR001,2012-01-08,2.9,0.0,2319.22\nR001,2012-01-09,3.2,5.1,25200.0\nR001,2012-01-10,-0.1,15.5,22277.8\nR001,2012-01-11,1.5,3.9,187.35\nR001,2012-01-12,1.6,11.4,593.35\nR001,2012-01-13,-1.0,2.0,21312.53\nR001,2012-01-14,-1.7,6.5,18347.02\nR001,2012-01-15,-1.6,24.0,0.0\nR001,2012-01-16,-2.2,23.2,0.0\nR001,2012-01-17,-3.6,10.2,0.0\nR001,2012-01-18,-6.0,3.1,32037.62\nR001,2012-01-19,-8.1,1.0,21247.2\nR001,2012-01-20,-9.7,0.0,32400.0\nR001,2012-01-21,-9.5,0.0,32400.0\nR001,2012-01-22,-1.8,2.0,25200.0\nR001,2012-01-23,-1.1,13.3,3602.67\nR001,2012-01-24,-2.6,5.6,12210.84\nR001,2012-01-25,-3.4,0.0,22433.55\nR001,2012-01-26,-0.3,0.8,0.0\nR001,2012-01-27,-1.8,0.0,14857.72\nR001,2012-01-28,-6.0,0.0,32400.0\nR001,2012-01-29,-8.5,1.8,26895.91\nR001,2012-01-30,-9.0,0.0,32475.36\nR001,2012-01-31,-6.6,9.3,0.0\nR001,2012-02-01,-7.7,12.2,11648.37\nR001,2012-02-02,-10.0,0.4,32695.86\nR001,2012-02-03,-7.0,0.0,31901.26\nR001,2012-02-04,-5.6,0.0,33349.54\nR001,2012-02-05,-1.3,0.0,33962.45\nR001,2012-02-06,-1.1,0.0,33877.33\nR001,2012-02-07,-3.6,0.0,33021.26\nR001,2012-02-08,-9.0,0.0,34737.79\nR001,2012-02-09,-6.6,1.5,25501.41\nR001,2012-02-10,-6.8,1.4,29126.78\nR001,2012-02-11,-9.1,0.5,31599.83\nR001,2012-02-12,-5.9,0.0,32597.41\nR001,2012-02-13,-0.6,0.0,35857.34\nR001,2012-02-14,1.0,0.5,23543.5\nR001,2012-02-15,2.7,1.5,13672.38\nR001,2012-02-16,-0.6,6.0,20713.69\nR001,2012-02-17,-1.1,0.5,23884.23\nR001,2012-02-18,-1.8,7.9,3013.73\nR001,2012-02-19,-3.3,8.5,301.39\nR001,2012-02-20,-4.3,0.9,35751.04\nR001,2012-02-21,-2.8,0.0,35841.41\nR001,2012-02-22,-3.2,0.1,32400.0\nR001,2012-02-23,-3.4,2.3,36969.18\nR001,2012-02-24,-2.0,1.0,30548.43\nR001,2012-02-25,-1.8,13.6,7250.28\nR001,2012-02-26,-1.6,12.6,16824.61\nR001,2012-02-27,1.3,1.0,24100.03\nR001,2012-02-28,3.7,7.4,34381.29\nR001,2012-02-29,1.8,9.6,919.48\nR001,2012-03-01,0.6,8.0,3213.51\nR001,2012-03-02,-2.3,9.0,3661.63\nR001,2012-03-03,-4.7,11.3,25200.0\nR001,2012-03-04,-3.6,0.9,39600.0\nR001,2012-03-05,-4.1,7.0,32917.29\nR001,2012-03-06,-4.8,11.5,19715.3\nR001,2012-03-07,-6.4,5.1,32805.81\nR001,2012-03-08,-7.0,0.0,39600.0\nR001,2012-03-09,-6.5,0.1,39600.0\nR001,2012-03-10,-7.9,0.7,32400.0\nR001,2012-03-11,-6.2,0.0,39600.0\nR001,2012-03-12,-2.2,5.4,8761.16\nR001,2012-03-13,1.7,2.5,15221.13\nR001,2012-03-14,1.4,2.1,30526.92\nR001,2012-03-15,-3.0,9.4,30303.45\nR001,2012-03-16,-6.3,1.6,37809.67\nR001,2012-03-17,-5.8,0.0,40031.43\nR001,2012-03-18,-0.4,0.0,39877.6\nR001,2012-03-19,4.5,0.0,41058.19\nR001,2012-03-20,6.3,0.0,41140.77\nR001,2012-03-21,4.5,0.0,39449.27\nR001,2012-03-22,0.6,8.9,10350.67\nR001,2012-03-23,0.5,0.7,39740.7\nR001,2012-03-24,2.3,5.4,14904.2\nR001,2012-03-25,3.6,1.1,40515.98\nR001,2012-03-26,3.7,5.4,38661.25\nR001,2012-03-27,-3.6,4.4,43215.26\nR001,2012-03-28,-3.0,0.0,38198.28\nR001,2012-03-29,-0.9,2.2,17465.77\nR001,2012-03-30,3.8,0.0,42697.81\nR001,2012-03-31,2.0,17.5,10699.09\nR001,2012-04-01,3.9,0.0,42439.93\nR001,2012-04-02,2.5,35.1,459.3\nR001,2012-04-03,3.2,0.0,45753.15\nR001,2012-04-04,7.3,0.0,42761.78\nR001,2012-04-05,9.2,0.0,45653.61\nR001,2012-04-06,11.6,0.0,38160.83\nR001,2012-04-07,10.0,3.5,34002.86\nR001,2012-04-08,11.4,0.6,43612.85\nR001,2012-04-09,11.2,1.0,39130.64\nR001,2012-04-10,8.1,5.0,36604.43\nR001,2012-04-11,9.4,4.3,42082.89\nR001,2012-04-12,5.5,17.1,14423.92\nR001,2012-04-13,7.3,0.6,46541.15\nR001,2012-04-14,9.3,0.0,46612.64\nR001,2012-04-15,12.0,0.0,39321.87\nR001,2012-04-16,12.7,2.9,44992.18\nR001,2012-04-17,10.1,5.5,10591.4\nR001,2012-04-18,12.5,0.0,46800.0\nR001,2012-04-19,12.2,5.7,13969.39\nR001,2012-04-20,9.5,0.0,45259.35\nR001,2012-04-21,10.3,0.2,46800.0\nR001,2012-04-22,10.2,5.1,46461.47\nR001,2012-04-23,9.0,1.7,46800.0\nR001,2012-04-24,10.9,0.0,47438.4\nR001,2012-04-25,13.0,0.0,47728.26\nR001,2012-04-26,14.0,0.5,47687.99\nR001,2012-04-27,13.5,3.7,47648.15\nR001,2012-04-28,13.5,1.0,47942.15\nR001,2012-04-29,13.5,1.3,48238.08\nR001,2012-04-30,14.6,0.0,48201.06\nR001,2012-05-01,13.9,0.0,47828.99\nR001,2012-05-02,15.1,0.0,48801.49\nR001,2012-05-03,15.7,0.5,46768.09\nR001,2012-05-04,15.4,2.9,42433.56\nR001,2012-05-05,14.6,0.2,47686.3\nR001,2012-05-06,13.5,3.6,49010.08\nR001,2012-05-07,13.5,8.2,31426.23\nR001,2012-05-08,13.4,9.8,18597.35\nR001,2012-05-09,13.8,8.8,40676.86\nR001,2012-05-10,15.6,3.3,50147.97\nR001,2012-05-11,16.3,2.0,47447.51\nR001,2012-05-12,14.9,7.5,15881.45\nR001,2012-05-13,14.7,8.9,43436.58\nR001,2012-05-14,14.2,5.6,40601.64\nR001,2012-05-15,13.7,5.9,28525.78\nR001,2012-05-16,15.8,0.8,49501.77\nR001,2012-05-17,16.7,3.0,44457.91\nR001,2012-05-18,15.8,6.4,48655.18\nR001,2012-05-19,18.0,0.8,50400.0\nR001,2012-05-20,17.1,1.7,34037.06\nR001,2012-05-21,15.6,11.5,36094.81\nR001,2012-05-22,14.8,4.0,29043.17\nR001,2012-05-23,14.5,13.1,32627.39\nR001,2012-05-24,13.4,6.9,30870.76\nR001,2012-05-25,11.8,1.0,46649.3\nR001,2012-05-26,11.6,6.3,29998.25\nR001,2012-05-27,13.3,4.9,30544.97\nR001,2012-05-28,15.9,2.9,48065.1\nR001,2012-05-29,15.3,6.2,41620.35\nR001,2012-05-30,14.7,6.5,39624.68\nR001,2012-05-31,15.6,8.4,46777.21\nR001,2012-06-01,13.8,8.9,41834.54\nR001,2012-06-02,14.0,0.1,50597.45\nR001,2012-06-03,16.3,0.0,52527.14\nR001,2012-06-04,18.8,0.0,49434.23\nR001,2012-06-05,17.8,16.1,44922.16\nR001,2012-06-06,19.0,0.0,52844.42\nR001,2012-06-07,17.1,2.7,45795.22\nR001,2012-06-08,14.5,16.5,17943.48\nR001,2012-06-09,16.1,1.0,49323.69\nR001,2012-06-10,18.8,0.0,53003.78\nR001,2012-06-11,21.8,0.0,53140.49\nR001,2012-06-12,21.8,1.8,51984.93\nR001,2012-06-13,22.8,0.6,53115.91\nR001,2012-06-14,24.6,0.6,53806.61\nR001,2012-06-15,22.1,4.3,45840.57\nR001,2012-06-16,18.9,7.8,46468.73\nR001,2012-06-17,15.6,1.7,46737.26\nR001,2012-06-18,18.0,0.1,48729.13\nR001,2012-06-19,19.4,0.0,53791.91\nR001,2012-06-20,19.7,0.0,53928.35\nR001,2012-06-21,19.7,0.2,53940.73\nR001,2012-06-22,20.0,0.6,53771.8\nR001,2012-06-23,20.3,1.2,53957.97\nR001,2012-06-24,20.8,0.1,53962.87\nR001,2012-06-25,21.0,3.3,52656.77\nR001,2012-06-26,19.6,7.0,52631.13\nR001,2012-06-27,18.6,11.3,40724.76\nR001,2012-06-28,15.7,9.6,5588.72\nR001,2012-06-29,15.0,3.7,50564.04\nR001,2012-06-30,13.1,15.2,35852.3\nR001,2012-07-01,12.4,11.8,29048.48\nR001,2012-07-02,14.2,1.1,37163.61\nR001,2012-07-03,16.2,5.1,43404.0\nR001,2012-07-04,15.2,9.4,35514.63\nR001,2012-07-05,15.3,16.0,40198.01\nR001,2012-07-06,16.7,2.9,46935.09\nR001,2012-07-07,18.4,3.3,42482.15\nR001,2012-07-08,19.6,1.6,47537.3\nR001,2012-07-09,19.6,0.7,51804.52\nR001,2012-07-10,17.1,10.1,50593.2\nR001,2012-07-11,17.2,1.7,45526.37\nR001,2012-07-12,18.3,8.8,48356.15\nR001,2012-07-13,20.2,0.3,50431.01\nR001,2012-07-14,20.1,1.8,50987.04\nR001,2012-07-15,21.2,0.1,53292.18\nR001,2012-07-16,21.7,0.0,52045.19\nR001,2012-07-17,21.8,0.0,52417.62\nR001,2012-07-18,20.4,0.2,50490.33\nR001,2012-07-19,19.7,0.2,52450.01\nR001,2012-07-20,19.7,0.0,50986.77\nR001,2012-07-21,22.0,0.0,53080.11\nR001,2012-07-22,23.1,0.0,52334.09\nR001,2012-07-23,24.3,1.3,52295.26\nR001,2012-07-24,23.9,0.0,50449.0\nR001,2012-07-25,24.4,0.0,51867.23\nR001,2012-07-26,25.0,0.2,51829.47\nR001,2012-07-27,24.4,0.1,51443.12\nR001,2012-07-28,23.6,0.8,49862.18\nR001,2012-07-29,21.6,4.6,46914.88\nR001,2012-07-30,19.1,8.2,46788.78\nR001,2012-07-31,20.3,1.8,49057.99\nR001,2012-08-01,19.9,14.0,33280.92\nR001,2012-08-02,20.6,6.8,47452.78\nR001,2012-08-03,21.4,4.6,47856.61\nR001,2012-08-04,22.6,3.4,48601.96\nR001,2012-08-05,21.2,1.4,47227.46\nR001,2012-08-06,21.1,7.4,29894.18\nR001,2012-08-07,20.1,7.4,44070.31\nR001,2012-08-08,20.5,7.5,43041.74\nR001,2012-08-09,19.7,7.1,47087.94\nR001,2012-08-10,19.0,8.1,39525.37\nR001,2012-08-11,18.8,8.9,45629.82\nR001,2012-08-12,21.3,1.5,47094.86\nR001,2012-08-13,20.8,4.9,47125.3\nR001,2012-08-14,20.2,9.1,46311.5\nR001,2012-08-15,19.8,7.7,38742.48\nR001,2012-08-16,18.4,25.0,23091.41\nR001,2012-08-17,17.8,0.2,45391.8\nR001,2012-08-18,17.5,0.4,46800.0\nR001,2012-08-19,16.5,8.8,39358.35\nR001,2012-08-20,16.0,8.4,44794.16\nR001,2012-08-21,15.7,0.0,36804.8\nR001,2012-08-22,17.0,6.2,36000.0\nR001,2012-08-23,16.8,0.0,46800.0\nR001,2012-08-24,18.2,0.0,46800.0\nR001,2012-08-25,20.5,0.0,46800.0\nR001,2012-08-26,21.7,0.0,46800.0\nR001,2012-08-27,22.5,0.0,46698.6\nR001,2012-08-28,21.1,2.9,45056.38\nR001,2012-08-29,16.1,13.4,30478.85\nR001,2012-08-30,14.8,0.0,46386.58\nR001,2012-08-31,16.0,0.0,41265.89\nR001,2012-09-01,17.3,0.9,32891.7\nR001,2012-09-02,17.9,0.0,45297.35\nR001,2012-09-03,18.3,0.0,45347.01\nR001,2012-09-04,18.3,0.0,45076.76\nR001,2012-09-05,17.8,0.0,44167.53\nR001,2012-09-06,17.4,0.0,44219.39\nR001,2012-09-07,17.2,0.0,40855.09\nR001,2012-09-08,16.0,0.0,42379.49\nR001,2012-09-09,15.4,0.0,43200.0\nR001,2012-09-10,14.5,0.0,41797.81\nR001,2012-09-11,14.2,0.0,40853.5\nR001,2012-09-12,16.1,0.0,43259.42\nR001,2012-09-13,18.8,0.0,43149.72\nR001,2012-09-14,18.1,2.4,41680.69\nR001,2012-09-15,17.1,0.2,43015.23\nR001,2012-09-16,17.9,0.0,42947.8\nR001,2012-09-17,18.1,0.0,41142.85\nR001,2012-09-18,18.6,0.0,42812.54\nR001,2012-09-19,18.5,0.0,42744.71\nR001,2012-09-20,18.4,1.6,39279.78\nR001,2012-09-21,17.0,3.7,40404.1\nR001,2012-09-22,15.1,3.4,23706.03\nR001,2012-09-23,13.4,18.4,34176.42\nR001,2012-09-24,14.4,0.0,39791.5\nR001,2012-09-25,17.0,0.0,41739.18\nR001,2012-09-26,16.9,0.0,41456.52\nR001,2012-09-27,16.6,0.0,40854.72\nR001,2012-09-28,16.9,0.0,41210.33\nR001,2012-09-29,17.4,0.0,41162.65\nR001,2012-09-30,16.9,0.0,40772.33\nR001,2012-10-01,17.3,0.0,40382.78\nR001,2012-10-02,17.8,0.0,39993.75\nR001,2012-10-03,18.6,0.0,35737.96\nR001,2012-10-04,17.1,2.7,25734.07\nR001,2012-10-05,14.7,11.4,0.0\nR001,2012-10-06,14.6,0.2,33904.9\nR001,2012-10-07,14.9,0.0,37902.98\nR001,2012-10-08,15.3,2.7,34522.49\nR001,2012-10-09,11.2,12.2,24769.02\nR001,2012-10-10,9.3,0.0,36775.99\nR001,2012-10-11,10.6,6.5,16240.69\nR001,2012-10-12,11.4,25.9,25760.95\nR001,2012-10-13,10.6,0.0,36000.0\nR001,2012-10-14,12.7,0.0,36342.83\nR001,2012-10-15,14.5,0.0,36396.03\nR001,2012-10-16,14.5,0.0,36120.87\nR001,2012-10-17,14.5,0.0,36000.0\nR001,2012-10-18,13.6,0.6,36000.0\nR001,2012-10-19,11.5,2.2,32497.92\nR001,2012-10-20,12.0,0.0,36000.0\nR001,2012-10-21,12.9,0.1,36000.0\nR001,2012-10-22,13.2,0.0,36000.0\nR001,2012-10-23,12.0,0.0,36000.0\nR001,2012-10-24,11.4,0.1,31102.23\nR001,2012-10-25,10.6,0.5,36000.0\nR001,2012-10-26,10.6,0.4,34686.15\nR001,2012-10-27,9.2,0.0,36000.0\nR001,2012-10-28,11.3,0.0,36000.0\nR001,2012-10-29,12.6,1.1,31442.68\nR001,2012-10-30,15.4,0.0,30856.23\nR001,2012-10-31,11.9,3.8,33561.65\nR001,2012-11-01,9.7,3.5,8721.11\nR001,2012-11-02,9.6,0.0,30953.06\nR001,2012-11-03,12.7,3.7,32415.95\nR001,2012-11-04,10.4,2.7,34236.24\nR001,2012-11-05,9.9,0.0,35623.3\nR001,2012-11-06,11.7,0.2,35468.39\nR001,2012-11-07,10.6,44.1,1080.72\nR001,2012-11-08,8.1,3.8,13784.4\nR001,2012-11-09,7.3,9.9,0.0\nR001,2012-11-10,9.1,7.8,0.0\nR001,2012-11-11,9.1,7.2,19603.0\nR001,2012-11-12,9.8,0.0,32309.26\nR001,2012-11-13,8.5,0.0,34091.78\nR001,2012-11-14,7.6,0.3,28189.53\nR001,2012-11-15,6.9,1.2,32647.14\nR001,2012-11-16,7.1,0.0,31242.26\nR001,2012-11-17,6.4,0.0,32970.41\nR001,2012-11-18,5.7,0.0,33329.0\nR001,2012-11-19,6.5,0.0,32959.95\nR001,2012-11-20,5.7,0.0,32590.76\nR001,2012-11-21,5.3,0.0,32588.89\nR001,2012-11-22,5.8,0.0,31378.75\nR001,2012-11-23,6.2,0.0,31447.46\nR001,2012-11-24,5.2,0.3,20547.66\nR001,2012-11-25,4.8,0.9,15421.28\nR001,2012-11-26,4.2,0.2,30231.95\nR001,2012-11-27,3.3,0.0,29878.52\nR001,2012-11-28,6.9,0.0,30653.88\nR001,2012-11-29,7.5,0.0,21824.84\nR001,2012-11-30,8.2,0.0,30315.09\nR001,2012-12-01,10.5,0.0,28800.0\nR001,2012-12-02,11.4,0.0,27801.81\nR001,2012-12-03,10.4,6.2,29216.21\nR001,2012-12-04,7.2,19.3,25200.0\nR001,2012-12-05,6.2,5.2,1687.51\nR001,2012-12-06,4.3,3.2,26162.35\nR001,2012-12-07,4.0,2.6,25200.0\nR001,2012-12-08,1.7,7.2,0.0\nR001,2012-12-09,-0.4,0.0,30027.62\nR001,2012-12-10,2.9,0.0,30804.21\nR001,2012-12-11,2.8,2.3,28800.0\nR001,2012-12-12,4.6,4.7,27090.14\nR001,2012-12-13,3.7,6.6,0.0\nR001,2012-12-14,-0.7,5.6,27044.9\nR001,2012-12-15,-4.0,0.4,28855.33\nR001,2012-12-16,-2.9,0.2,30018.16\nR001,2012-12-17,-4.7,0.0,30791.66\nR001,2012-12-18,-3.2,0.9,2975.37\nR001,2012-12-19,0.4,0.3,14784.43\nR001,2012-12-20,2.7,1.1,10309.13\nR001,2012-12-21,2.8,2.5,19867.31\nR001,2012-12-22,2.0,9.2,14458.83\nR001,2012-12-23,1.2,15.6,8784.22\nR001,2012-12-24,-2.2,6.9,20583.32\nR001,2012-12-25,-5.4,0.8,15363.85\nR001,2012-12-26,-4.1,0.0,31333.23\nR001,2012-12-27,3.5,0.0,29076.46\nR001,2012-12-28,7.2,0.0,3533.86\nR001,2012-12-29,4.1,5.1,24879.95\nR001,2012-12-30,-1.8,5.8,30752.16\nR001,2012-12-31,-6.1,0.0,31572.73\nR001,2013-01-01,-4.9,0.0,31626.64\nR001,2013-01-02,-3.1,0.0,31682.26\nR001,2013-01-03,-1.8,0.0,31739.44\nR001,2013-01-04,-0.2,9.4,2451.05\nR001,2013-01-05,0.4,13.2,772.95\nR001,2013-01-06,-1.5,2.0,24164.46\nR001,2013-01-07,-2.7,6.8,0.0\nR001,2013-01-08,-2.3,12.7,10426.63\nR001,2013-01-09,-6.2,0.0,30233.84\nR001,2013-01-10,-8.2,0.0,28898.24\nR001,2013-01-11,-5.1,0.0,27973.69\nR001,2013-01-12,-0.4,2.5,78.07\nR001,2013-01-13,1.4,35.6,1185.57\nR001,2013-01-14,-5.3,0.2,32400.0\nR001,2013-01-15,-7.7,0.0,32400.0\nR001,2013-01-16,-3.3,0.0,23285.15\nR001,2013-01-17,1.5,1.3,19511.81\nR001,2013-01-18,3.4,1.8,31052.7\nR001,2013-01-19,4.7,0.6,30964.0\nR001,2013-01-20,1.8,24.7,0.0\nR001,2013-01-21,3.1,13.0,6150.16\nR001,2013-01-22,6.2,0.0,30007.47\nR001,2013-01-23,5.1,10.5,16261.34\nR001,2013-01-24,3.0,12.9,30247.46\nR001,2013-01-25,2.0,3.8,20602.07\nR001,2013-01-26,4.0,8.6,27503.55\nR001,2013-01-27,4.6,0.1,25200.0\nR001,2013-01-28,3.1,2.3,23973.15\nR001,2013-01-29,2.0,0.0,24514.28\nR001,2013-01-30,1.2,7.4,18000.0\nR001,2013-01-31,0.8,3.6,31334.34\nR001,2013-02-01,0.2,8.3,3604.79\nR001,2013-02-02,-0.9,6.5,25661.03\nR001,2013-02-03,1.0,0.0,32621.53\nR001,2013-02-04,5.5,0.0,31706.21\nR001,2013-02-05,3.7,7.3,11301.9\nR001,2013-02-06,4.0,0.0,32834.07\nR001,2013-02-07,6.1,0.0,30519.25\nR001,2013-02-08,5.5,0.0,34664.52\nR001,2013-02-09,6.2,0.0,35257.65\nR001,2013-02-10,5.4,0.0,34364.43\nR001,2013-02-11,4.3,0.0,34656.2\nR001,2013-02-12,3.0,4.1,0.0\nR001,2013-02-13,2.0,0.0,35206.17\nR001,2013-02-14,2.3,0.0,32940.72\nR001,2013-02-15,5.0,0.7,34679.18\nR001,2013-02-16,2.5,10.5,16684.46\nR001,2013-02-17,-0.8,8.0,31337.67\nR001,2013-02-18,-1.8,1.3,34892.73\nR001,2013-02-19,-1.4,1.0,33524.42\nR001,2013-02-20,-1.0,0.2,24728.29\nR001,2013-02-21,1.3,5.3,24735.44\nR001,2013-02-22,1.5,0.3,26256.54\nR001,2013-02-23,4.2,7.0,527.88\nR001,2013-02-24,4.0,5.9,0.0\nR001,2013-02-25,4.1,0.0,37241.01\nR001,2013-02-26,4.8,0.0,33286.81\nR001,2013-02-27,5.0,0.1,39372.86\nR001,2013-02-28,4.0,2.9,26871.88\nR001,2013-03-01,3.6,5.9,30644.77\nR001,2013-03-02,2.2,2.7,30938.51\nR001,2013-03-03,-0.4,11.9,27642.67\nR001,2013-03-04,-0.7,0.7,13427.21\nR001,2013-03-05,-3.7,6.2,36878.42\nR001,2013-03-06,-4.7,0.0,39600.0\nR001,2013-03-07,1.5,0.0,39600.0\nR001,2013-03-08,7.3,0.0,39600.0\nR001,2013-03-09,4.5,2.8,3961.38\nR001,2013-03-10,5.1,10.2,25879.88\nR001,2013-03-11,6.2,15.1,23730.09\nR001,2013-03-12,7.0,6.9,18255.71\nR001,2013-03-13,6.1,31.7,15701.23\nR001,2013-03-14,7.3,0.0,39310.76\nR001,2013-03-15,13.8,0.0,37920.2\nR001,2013-03-16,12.0,23.9,3164.92\nR001,2013-03-17,4.9,40.8,0.0\nR001,2013-03-18,-0.4,9.5,26118.35\nR001,2013-03-19,2.4,0.0,32015.39\nR001,2013-03-20,6.5,2.2,34956.11\nR001,2013-03-21,6.9,1.1,40259.82\nR001,2013-03-22,11.5,0.0,26735.04\nR001,2013-03-23,5.6,21.0,19721.47\nR001,2013-03-24,-2.3,16.9,23351.36\nR001,2013-03-25,-0.2,5.9,6429.26\nR001,2013-03-26,4.0,5.8,18587.65\nR001,2013-03-27,5.6,12.2,16683.64\nR001,2013-03-28,5.1,14.3,18518.02\nR001,2013-03-29,1.8,2.0,40065.52\nR001,2013-03-30,4.1,0.0,44691.93\nR001,2013-03-31,11.3,0.0,44763.32\nR001,2013-04-01,14.0,0.0,45153.54\nR001,2013-04-02,12.2,6.3,31706.52\nR001,2013-04-03,11.8,1.2,44612.39\nR001,2013-04-04,14.5,0.0,45691.5\nR001,2013-04-05,13.0,1.7,43743.03\nR001,2013-04-06,13.0,0.0,44349.4\nR001,2013-04-07,13.5,0.0,40633.84\nR001,2013-04-08,15.6,8.0,39163.82\nR001,2013-04-09,8.8,8.2,22813.19\nR001,2013-04-10,6.5,4.4,33240.43\nR001,2013-04-11,6.1,7.3,18104.98\nR001,2013-04-12,6.6,1.4,37375.9\nR001,2013-04-13,8.0,0.9,46382.3\nR001,2013-04-14,9.3,0.3,46594.76\nR001,2013-04-15,10.9,0.8,46665.91\nR001,2013-04-16,10.7,5.9,36647.91\nR001,2013-04-17,6.8,15.7,13590.73\nR001,2013-04-18,6.8,5.6,15637.0\nR001,2013-04-19,8.1,2.0,41298.63\nR001,2013-04-20,8.6,7.8,41465.1\nR001,2013-04-21,7.3,5.9,2786.32\nR001,2013-04-22,6.5,9.8,29821.13\nR001,2013-04-23,3.8,13.8,41939.31\nR001,2013-04-24,4.0,2.7,45897.06\nR001,2013-04-25,7.4,0.6,36801.04\nR001,2013-04-26,8.4,2.9,39425.49\nR001,2013-04-27,11.3,0.0,46800.0\nR001,2013-04-28,15.5,0.0,47951.45\nR001,2013-04-29,17.2,0.0,48580.35\nR001,2013-04-30,16.3,0.0,48208.64\nR001,2013-05-01,17.1,0.0,48842.56\nR001,2013-05-02,17.7,0.0,48807.67\nR001,2013-05-03,17.0,0.0,49110.55\nR001,2013-05-04,17.3,1.6,49077.99\nR001,2013-05-05,15.0,2.2,46613.93\nR001,2013-05-06,14.7,0.3,46941.38\nR001,2013-05-07,13.9,0.7,44966.19\nR001,2013-05-08,14.4,0.8,46800.0\nR001,2013-05-09,14.7,0.1,49269.76\nR001,2013-05-10,14.5,3.2,49570.72\nR001,2013-05-11,15.9,5.5,47831.02\nR001,2013-05-12,13.5,20.0,17338.02\nR001,2013-05-13,13.3,5.8,46229.64\nR001,2013-05-14,14.3,7.5,45266.47\nR001,2013-05-15,14.0,5.4,29754.04\nR001,2013-05-16,13.8,4.4,32011.21\nR001,2013-05-17,14.6,0.3,48151.89\nR001,2013-05-18,13.9,3.0,48411.57\nR001,2013-05-19,13.7,5.3,42901.06\nR001,2013-05-20,15.4,3.1,50543.43\nR001,2013-05-21,16.3,0.9,47767.46\nR001,2013-05-22,16.5,0.4,49106.63\nR001,2013-05-23,17.7,0.0,50963.92\nR001,2013-05-24,20.5,0.3,48928.63\nR001,2013-05-25,17.0,7.3,44450.62\nR001,2013-05-26,15.0,1.8,34487.08\nR001,2013-05-27,15.6,1.4,47514.9\nR001,2013-05-28,14.5,4.3,50400.0\nR001,2013-05-29,14.3,5.8,47271.3\nR001,2013-05-30,17.0,1.0,51522.26\nR001,2013-05-31,18.7,0.0,51864.04\nR001,2013-06-01,16.4,1.4,50400.0\nR001,2013-06-02,16.1,0.0,51462.89\nR001,2013-06-03,14.1,11.7,41481.44\nR001,2013-06-04,12.8,6.5,44961.79\nR001,2013-06-05,14.9,0.4,52499.99\nR001,2013-06-06,15.3,5.5,50659.29\nR001,2013-06-07,12.9,14.4,0.0\nR001,2013-06-08,13.7,13.4,36103.79\nR001,2013-06-09,14.7,4.9,50017.69\nR001,2013-06-10,15.5,6.6,45942.14\nR001,2013-06-11,14.7,10.6,30624.83\nR001,2013-06-12,16.7,2.7,50778.32\nR001,2013-06-13,19.2,1.4,53484.39\nR001,2013-06-14,21.3,0.1,52738.67\nR001,2013-06-15,20.4,6.3,52727.9\nR001,2013-06-16,17.9,9.1,38156.47\nR001,2013-06-17,17.7,16.3,48801.63\nR001,2013-06-18,17.6,4.7,35346.99\nR001,2013-06-19,17.9,0.6,51589.54\nR001,2013-06-20,15.8,0.3,51881.04\nR001,2013-06-21,15.1,0.6,49430.6\nR001,2013-06-22,14.2,3.6,45497.86\nR001,2013-06-23,16.1,0.7,46398.35\nR001,2013-06-24,17.3,0.1,51384.76\nR001,2013-06-25,18.4,1.1,51924.81\nR001,2013-06-26,19.5,0.1,53963.47\nR001,2013-06-27,20.2,1.2,53963.83\nR001,2013-06-28,21.2,0.2,53962.1\nR001,2013-06-29,20.2,1.3,53958.26\nR001,2013-06-30,20.7,0.1,53952.26\nR001,2013-07-01,19.8,15.0,49023.23\nR001,2013-07-02,16.7,3.4,47641.85\nR001,2013-07-03,16.3,15.1,32624.36\nR001,2013-07-04,16.5,16.5,39994.83\nR001,2013-07-05,18.3,0.4,49695.92\nR001,2013-07-06,18.6,0.6,53390.01\nR001,2013-07-07,20.0,0.7,53757.42\nR001,2013-07-08,20.4,1.4,53762.34\nR001,2013-07-09,21.0,0.0,53796.27\nR001,2013-07-10,19.5,1.0,52690.78\nR001,2013-07-11,19.0,1.2,52026.8\nR001,2013-07-12,19.1,3.7,51402.72\nR001,2013-07-13,19.8,1.1,52361.21\nR001,2013-07-14,19.0,2.2,53449.9\nR001,2013-07-15,19.6,1.5,51027.24\nR001,2013-07-16,19.2,4.3,48852.23\nR001,2013-07-17,19.7,3.0,48667.62\nR001,2013-07-18,19.2,4.4,46888.44\nR001,2013-07-19,19.3,1.1,44443.66\nR001,2013-07-20,19.1,0.7,51355.1\nR001,2013-07-21,18.8,1.7,50964.31\nR001,2013-07-22,17.9,0.2,47585.06\nR001,2013-07-23,17.2,0.0,49690.12\nR001,2013-07-24,17.3,0.0,50439.49\nR001,2013-07-25,16.9,0.1,50119.94\nR001,2013-07-26,16.9,0.7,50178.94\nR001,2013-07-27,16.4,1.8,48830.41\nR001,2013-07-28,15.8,11.5,45041.3\nR001,2013-07-29,14.3,5.8,44355.37\nR001,2013-07-30,14.2,5.2,48116.24\nR001,2013-07-31,15.3,0.0,48069.48\nR001,2013-08-01,16.8,0.0,45481.62\nR001,2013-08-02,17.4,1.6,48509.5\nR001,2013-08-03,16.7,0.6,44505.28\nR001,2013-08-04,14.8,8.5,43226.95\nR001,2013-08-05,15.4,8.0,43200.0\nR001,2013-08-06,17.0,1.1,42313.79\nR001,2013-08-07,17.0,3.2,40754.58\nR001,2013-08-08,17.8,1.4,46591.03\nR001,2013-08-09,18.0,0.0,47673.75\nR001,2013-08-10,18.5,0.0,47365.04\nR001,2013-08-11,19.5,0.0,47731.09\nR001,2013-08-12,20.3,0.3,47759.91\nR001,2013-08-13,18.4,8.8,37665.95\nR001,2013-08-14,19.2,2.6,46800.0\nR001,2013-08-15,20.8,0.8,46800.0\nR001,2013-08-16,20.6,2.7,46877.37\nR001,2013-08-17,20.4,2.8,46800.0\nR001,2013-08-18,18.1,7.3,35952.55\nR001,2013-08-19,18.6,0.0,46800.0\nR001,2013-08-20,18.3,0.3,46800.0\nR001,2013-08-21,18.4,0.1,46800.0\nR001,2013-08-22,18.6,0.0,46800.0\nR001,2013-08-23,19.6,0.1,46800.0\nR001,2013-08-24,17.5,6.7,46315.95\nR001,2013-08-25,17.3,5.5,43683.53\nR001,2013-08-26,18.2,0.2,46503.31\nR001,2013-08-27,19.2,0.5,46296.0\nR001,2013-08-28,19.4,0.4,46614.63\nR001,2013-08-29,21.0,0.0,46510.48\nR001,2013-08-30,21.1,0.2,44175.53\nR001,2013-08-31,17.6,16.9,31018.32\nR001,2013-09-01,15.7,24.7,21123.88\nR001,2013-09-02,15.7,0.0,44644.97\nR001,2013-09-03,15.7,0.0,40598.21\nR001,2013-09-04,13.0,14.3,31286.24\nR001,2013-09-05,10.3,7.1,30765.25\nR001,2013-09-06,10.5,7.3,35220.33\nR001,2013-09-07,11.2,2.1,41668.1\nR001,2013-09-08,12.5,0.1,43200.0\nR001,2013-09-09,11.8,5.4,39976.22\nR001,2013-09-10,11.8,0.0,42179.72\nR001,2013-09-11,13.5,0.0,43200.0\nR001,2013-09-12,13.7,0.0,41555.99\nR001,2013-09-13,15.7,0.0,41640.58\nR001,2013-09-14,17.9,1.5,41361.84\nR001,2013-09-15,14.3,21.9,15181.83\nR001,2013-09-16,12.5,8.7,30268.84\nR001,2013-09-17,13.1,2.7,0.0\nR001,2013-09-18,15.2,1.2,34554.41\nR001,2013-09-19,15.9,2.0,36298.91\nR001,2013-09-20,15.9,2.8,36996.63\nR001,2013-09-21,14.1,3.9,23015.52\nR001,2013-09-22,12.1,5.1,37172.56\nR001,2013-09-23,9.7,1.5,36276.02\nR001,2013-09-24,8.5,75.3,225.71\nR001,2013-09-25,6.9,29.8,19427.64\nR001,2013-09-26,7.7,0.0,41128.46\nR001,2013-09-27,12.2,0.0,41483.54\nR001,2013-09-28,9.0,6.3,6764.57\nR001,2013-09-29,6.0,10.5,24884.15\nR001,2013-09-30,7.6,2.1,24849.54\nR001,2013-10-01,12.5,2.1,32420.01\nR001,2013-10-02,13.9,2.6,29305.66\nR001,2013-10-03,8.7,40.3,0.0\nR001,2013-10-04,4.8,3.0,25755.51\nR001,2013-10-05,2.9,4.5,30658.15\nR001,2013-10-06,3.0,2.9,36542.63\nR001,2013-10-07,4.6,0.1,36922.18\nR001,2013-10-08,4.8,0.0,37351.8\nR001,2013-10-09,7.9,0.0,37770.23\nR001,2013-10-10,11.3,0.0,37540.16\nR001,2013-10-11,12.5,0.0,35850.52\nR001,2013-10-12,11.0,0.0,36550.43\nR001,2013-10-13,9.3,0.0,31714.23\nR001,2013-10-14,9.2,0.0,34313.76\nR001,2013-10-15,8.8,0.0,36055.68\nR001,2013-10-16,10.8,0.0,34557.04\nR001,2013-10-17,14.5,0.0,35869.36\nR001,2013-10-18,17.5,0.0,36000.0\nR001,2013-10-19,10.2,33.3,16420.1\nR001,2013-10-20,6.4,2.9,33920.41\nR001,2013-10-21,6.3,0.0,36000.0\nR001,2013-10-22,8.8,0.9,36000.0\nR001,2013-10-23,7.7,1.5,34389.33\nR001,2013-10-24,7.3,0.0,36000.0\nR001,2013-10-25,8.8,0.0,36000.0\nR001,2013-10-26,8.6,0.0,36000.0\nR001,2013-10-27,7.4,0.0,36000.0\nR001,2013-10-28,7.3,0.8,29823.27\nR001,2013-10-29,9.0,0.0,36000.0\nR001,2013-10-30,10.5,0.0,36000.0\nR001,2013-10-31,11.1,0.0,36000.0\nR001,2013-11-01,9.5,0.0,36000.0\nR001,2013-11-02,9.9,0.0,35928.1\nR001,2013-11-03,9.5,0.0,35849.14\nR001,2013-11-04,10.8,0.0,35770.35\nR001,2013-11-05,11.9,0.0,35301.76\nR001,2013-11-06,12.9,0.0,31050.91\nR001,2013-11-07,11.7,0.0,27451.32\nR001,2013-11-08,7.0,14.7,27109.63\nR001,2013-11-09,4.7,0.0,35066.93\nR001,2013-11-10,8.0,0.0,34919.55\nR001,2013-11-11,10.1,0.0,34773.45\nR001,2013-11-12,9.2,0.0,34628.79\nR001,2013-11-13,8.0,0.0,33172.66\nR001,2013-11-14,6.1,4.1,26840.66\nR001,2013-11-15,5.2,0.2,31709.29\nR001,2013-11-16,3.0,0.0,33706.76\nR001,2013-11-17,2.7,0.1,24934.42\nR001,2013-11-18,3.2,0.0,32600.53\nR001,2013-11-19,4.7,0.0,32959.33\nR001,2013-11-20,7.2,0.0,32955.29\nR001,2013-11-21,7.3,0.0,21596.49\nR001,2013-11-22,8.2,0.0,30987.93\nR001,2013-11-23,7.1,0.0,29944.13\nR001,2013-11-24,7.3,0.0,31128.37\nR001,2013-11-25,9.0,0.0,30202.84\nR001,2013-11-26,9.3,4.5,4160.26\nR001,2013-11-27,6.9,39.5,0.0\nR001,2013-11-28,4.1,2.2,26458.43\nR001,2013-11-29,1.5,19.9,0.0\nR001,2013-11-30,-1.5,6.4,27584.08\nR001,2013-12-01,-2.2,0.0,24317.2\nR001,2013-12-02,3.0,14.3,0.0\nR001,2013-12-03,-5.4,19.5,27530.63\nR001,2013-12-04,-9.6,0.0,29986.53\nR001,2013-12-05,-5.2,0.0,30380.54\nR001,2013-12-06,-1.3,1.3,26921.13\nR001,2013-12-07,-0.3,4.3,29245.56\nR001,2013-12-08,-2.9,14.9,24641.05\nR001,2013-12-09,-5.6,4.6,29640.82\nR001,2013-12-10,-4.8,29.5,909.0\nR001,2013-12-11,-12.6,0.0,30033.22\nR001,2013-12-12,-13.2,0.0,30810.07\nR001,2013-12-13,-9.4,0.0,30033.73\nR001,2013-12-14,-5.1,6.9,20824.67\nR001,2013-12-15,-3.9,6.9,28373.14\nR001,2013-12-16,-5.4,0.0,30023.47\nR001,2013-12-17,-6.0,0.0,30406.91\nR001,2013-12-18,-4.6,0.0,30789.12\nR001,2013-12-19,-2.2,0.0,30388.83\nR001,2013-12-20,-0.8,0.0,26684.47\nR001,2013-12-21,0.7,0.0,30363.95\nR001,2013-12-22,-0.0,0.0,30349.06\nR001,2013-12-23,-1.0,0.0,29941.33\nR001,2013-12-24,-0.9,0.0,31249.38\nR001,2013-12-25,-1.9,0.0,31288.23\nR001,2013-12-26,-0.5,0.0,31329.26\nR001,2013-12-27,2.3,0.0,15665.4\nR001,2013-12-28,1.0,0.0,31417.44\nR001,2013-12-29,1.0,0.0,31464.36\nR001,2013-12-30,1.3,0.0,31513.01\nR001,2013-12-31,1.3,0.0,31563.28\nR001,2014-01-01,0.3,0.0,31615.06\nR001,2014-01-02,-0.1,0.0,31668.21\nR001,2014-01-03,-1.1,0.0,31722.64\nR001,2014-01-04,-1.3,0.0,31778.21\nR001,2014-01-05,-0.9,0.0,31835.07\nR001,2014-01-06,-0.4,0.0,27924.45\nR001,2014-01-07,1.8,2.1,26871.97\nR001,2014-01-08,-0.2,0.8,32019.15\nR001,2014-01-09,-0.2,0.0,32084.67\nR001,2014-01-10,1.8,0.0,32152.03\nR001,2014-01-11,1.5,4.8,3588.61\nR001,2014-01-12,-0.1,0.4,21973.33\nR001,2014-01-13,-1.2,28.7,6717.06\nR001,2014-01-14,-4.2,6.1,28800.0\nR001,2014-01-15,-3.9,0.0,32400.0\nR001,2014-01-16,1.2,0.0,10729.07\nR001,2014-01-17,4.3,6.4,10916.13\nR001,2014-01-18,4.6,5.1,405.73\nR001,2014-01-19,4.2,19.6,0.0\nR001,2014-01-20,3.3,11.9,0.0\nR001,2014-01-21,3.8,8.8,25391.32\nR001,2014-01-22,3.7,10.0,21600.0\nR001,2014-01-23,1.5,24.5,3887.95\nR001,2014-01-24,-0.4,20.9,0.0\nR001,2014-01-25,-0.3,1.1,31480.76\nR001,2014-01-26,3.1,2.9,20583.3\nR001,2014-01-27,5.4,4.4,20359.8\nR001,2014-01-28,4.9,0.0,24855.17\nR001,2014-01-29,3.9,5.5,29128.84\nR001,2014-01-30,2.8,13.5,22829.45\nR001,2014-01-31,0.4,8.1,5341.33\nR001,2014-02-01,-1.0,1.5,28800.0\nR001,2014-02-02,-3.8,0.0,32774.65\nR001,2014-02-03,-4.8,0.0,33041.92\nR001,2014-02-04,-4.3,0.0,32958.15\nR001,2014-02-05,-3.0,0.0,33567.62\nR001,2014-02-06,-2.9,0.0,33826.93\nR001,2014-02-07,-1.0,0.0,34428.65\nR001,2014-02-08,0.2,0.0,34681.66\nR001,2014-02-09,2.0,0.0,23164.86\nR001,2014-02-10,4.8,0.0,32132.06\nR001,2014-02-11,4.2,0.7,29449.54\nR001,2014-02-12,4.3,0.0,34313.12\nR001,2014-02-13,6.5,0.0,36259.0\nR001,2014-02-14,7.9,0.0,29784.7\nR001,2014-02-15,6.5,0.0,36283.39\nR001,2014-02-16,4.1,0.0,35109.25\nR001,2014-02-17,3.3,0.1,33621.68\nR001,2014-02-18,3.7,20.0,12322.39\nR001,2014-02-19,2.0,1.1,35512.57\nR001,2014-02-20,4.9,0.0,31833.45\nR001,2014-02-21,4.6,10.4,24785.15\nR001,2014-02-22,4.1,0.0,33680.99\nR001,2014-02-23,5.3,0.0,38633.83\nR001,2014-02-24,6.2,0.0,35768.52\nR001,2014-02-25,7.5,0.0,34788.85\nR001,2014-02-26,5.0,5.9,4040.83\nR001,2014-02-27,2.9,5.7,27683.62\nR001,2014-02-28,2.4,3.3,36545.17\nR001,2014-03-01,3.4,0.0,39557.33\nR001,2014-03-02,5.6,0.2,34089.92\nR001,2014-03-03,7.7,4.8,39514.14\nR001,2014-03-04,8.5,0.2,21843.77\nR001,2014-03-05,6.6,3.5,34250.64\nR001,2014-03-06,5.6,0.0,36696.5\nR001,2014-03-07,6.2,3.3,9020.3\nR001,2014-03-08,5.8,0.7,23435.78\nR001,2014-03-09,9.0,0.7,39600.0\nR001,2014-03-10,8.2,0.0,39600.0\nR001,2014-03-11,5.4,13.7,2288.93\nR001,2014-03-12,1.3,14.2,0.0\nR001,2014-03-13,1.6,1.7,34969.49\nR001,2014-03-14,2.9,0.5,36445.18\nR001,2014-03-15,4.2,0.0,39600.0\nR001,2014-03-16,4.9,7.6,39600.0\nR001,2014-03-17,-1.5,28.0,18135.19\nR001,2014-03-18,0.2,12.9,4149.84\nR001,2014-03-19,3.0,0.0,39600.0\nR001,2014-03-20,4.6,6.8,15819.66\nR001,2014-03-21,2.8,2.6,30394.6\nR001,2014-03-22,4.6,0.0,41776.7\nR001,2014-03-23,8.5,0.0,42575.58\nR001,2014-03-24,10.6,0.0,42647.11\nR001,2014-03-25,11.4,0.0,43035.8\nR001,2014-03-26,12.0,0.6,41958.18\nR001,2014-03-27,6.8,5.9,667.56\nR001,2014-03-28,5.3,3.0,41562.11\nR001,2014-03-29,0.7,25.9,63.88\nR001,2014-03-30,-5.9,0.6,43393.02\nR001,2014-03-31,-3.8,0.0,43784.46\nR001,2014-04-01,2.9,0.0,45299.6\nR001,2014-04-02,1.4,4.0,36924.13\nR001,2014-04-03,-0.4,2.7,40963.46\nR001,2014-04-04,2.5,0.0,43759.97\nR001,2014-04-05,5.9,5.8,31038.03\nR001,2014-04-06,6.9,0.0,45637.26\nR001,2014-04-07,10.7,0.8,40668.97\nR001,2014-04-08,9.1,9.2,16191.18\nR001,2014-04-09,6.3,3.7,26454.6\nR001,2014-04-10,8.4,0.0,46155.21\nR001,2014-04-11,9.2,0.0,46363.37\nR001,2014-04-12,10.7,2.7,46434.45\nR001,2014-04-13,8.4,5.9,3810.06\nR001,2014-04-14,9.1,0.4,39742.3\nR001,2014-04-15,9.6,4.0,28210.89\nR001,2014-04-16,11.2,2.9,46717.94\nR001,2014-04-17,13.0,0.4,46789.01\nR001,2014-04-18,15.2,0.2,46800.0\nR001,2014-04-19,16.3,1.3,43200.0\nR001,2014-04-20,12.2,9.9,39824.49\nR001,2014-04-21,10.9,2.4,42836.76\nR001,2014-04-22,9.0,9.5,0.0\nR001,2014-04-23,8.6,6.9,666.09\nR001,2014-04-24,9.7,0.7,45281.12\nR001,2014-04-25,12.2,0.5,38017.66\nR001,2014-04-26,10.6,10.6,16933.55\nR001,2014-04-27,11.5,6.6,30693.01\nR001,2014-04-28,14.6,1.0,43953.1\nR001,2014-04-29,11.9,8.8,37547.97\nR001,2014-04-30,11.9,3.0,39919.15\nR001,2014-05-01,11.6,9.0,32400.0\nR001,2014-05-02,10.3,2.0,42234.15\nR001,2014-05-03,11.2,2.5,48444.35\nR001,2014-05-04,12.4,1.5,44951.98\nR001,2014-05-05,14.5,6.3,13655.91\nR001,2014-05-06,12.6,14.5,24504.09\nR001,2014-05-07,12.1,6.4,11447.94\nR001,2014-05-08,14.5,0.6,46800.0\nR001,2014-05-09,16.7,3.5,32266.64\nR001,2014-05-10,16.8,14.7,39145.9\nR001,2014-05-11,17.1,1.5,48414.12\nR001,2014-05-12,15.5,7.4,46606.92\nR001,2014-05-13,13.8,12.6,17803.71\nR001,2014-05-14,14.3,9.1,21801.17\nR001,2014-05-15,17.2,0.8,47697.6\nR001,2014-05-16,19.9,0.9,50400.0\nR001,2014-05-17,20.6,0.8,50400.0\nR001,2014-05-18,17.0,13.0,35907.92\nR001,2014-05-19,13.8,37.8,2198.6\nR001,2014-05-20,13.8,9.5,14872.67\nR001,2014-05-21,13.8,10.1,18712.04\nR001,2014-05-22,14.4,7.2,43477.81\nR001,2014-05-23,15.3,8.3,45662.46\nR001,2014-05-24,16.7,5.9,50400.0\nR001,2014-05-25,15.0,16.5,11444.6\nR001,2014-05-26,14.0,3.8,15778.78\nR001,2014-05-27,14.4,7.7,27478.93\nR001,2014-05-28,16.0,6.6,46304.62\nR001,2014-05-29,16.6,2.6,49903.02\nR001,2014-05-30,18.9,0.0,50400.0\nR001,2014-05-31,21.5,2.3,50400.0\nR001,2014-06-01,18.9,9.7,47540.05\nR001,2014-06-02,17.4,15.2,29580.25\nR001,2014-06-03,14.4,22.2,21272.09\nR001,2014-06-04,16.1,3.5,46541.59\nR001,2014-06-05,19.2,0.9,50679.62\nR001,2014-06-06,16.7,11.1,12329.91\nR001,2014-06-07,15.8,14.8,30241.29\nR001,2014-06-08,16.2,8.2,42937.36\nR001,2014-06-09,16.8,3.0,51263.55\nR001,2014-06-10,17.1,5.0,52424.31\nR001,2014-06-11,15.1,3.2,48077.89\nR001,2014-06-12,13.1,9.0,42082.1\nR001,2014-06-13,12.5,0.6,51358.98\nR001,2014-06-14,14.6,0.0,49329.45\nR001,2014-06-15,16.1,0.2,52732.8\nR001,2014-06-16,18.4,1.3,52722.88\nR001,2014-06-17,19.1,2.5,53816.44\nR001,2014-06-18,18.6,0.0,53881.05\nR001,2014-06-19,17.7,1.0,45488.08\nR001,2014-06-20,20.6,0.7,53914.38\nR001,2014-06-21,16.9,16.5,45252.22\nR001,2014-06-22,16.0,0.0,49843.34\nR001,2014-06-23,15.9,0.0,52747.99\nR001,2014-06-24,16.7,0.5,52222.7\nR001,2014-06-25,17.3,1.4,49702.35\nR001,2014-06-26,18.3,1.0,47878.96\nR001,2014-06-27,18.6,0.0,53968.55\nR001,2014-06-28,19.1,1.8,53658.05\nR001,2014-06-29,18.2,2.0,52724.83\nR001,2014-06-30,18.7,1.6,52267.24\nR001,2014-07-01,20.5,1.7,53012.83\nR001,2014-07-02,20.2,4.0,51917.78\nR001,2014-07-03,20.1,2.8,49179.8\nR001,2014-07-04,19.6,2.2,51195.98\nR001,2014-07-05,19.2,9.6,40261.63\nR001,2014-07-06,17.9,34.1,3119.69\nR001,2014-07-07,18.6,7.4,44337.14\nR001,2014-07-08,17.2,0.6,49060.38\nR001,2014-07-09,17.0,2.3,46745.22\nR001,2014-07-10,18.5,0.5,50495.4\nR001,2014-07-11,21.5,0.0,53741.62\nR001,2014-07-12,24.8,0.2,53708.84\nR001,2014-07-13,22.5,0.9,52718.16\nR001,2014-07-14,19.6,11.0,49881.33\nR001,2014-07-15,20.3,4.6,50735.76\nR001,2014-07-16,22.8,0.0,52370.05\nR001,2014-07-17,22.6,4.5,52767.79\nR001,2014-07-18,20.3,9.5,43977.31\nR001,2014-07-19,20.0,3.9,51255.29\nR001,2014-07-20,19.1,3.8,49159.86\nR001,2014-07-21,18.4,11.0,37699.87\nR001,2014-07-22,18.8,2.1,48976.94\nR001,2014-07-23,20.4,0.1,51790.95\nR001,2014-07-24,22.9,0.0,51918.53\nR001,2014-07-25,20.7,2.3,50468.26\nR001,2014-07-26,21.4,1.0,49465.74\nR001,2014-07-27,20.8,4.5,37081.68\nR001,2014-07-28,18.9,9.8,43179.39\nR001,2014-07-29,18.3,4.4,48226.2\nR001,2014-07-30,18.7,3.3,48627.58\nR001,2014-07-31,19.4,2.1,48338.16\nR001,2014-08-01,20.1,1.2,48740.8\nR001,2014-08-02,22.1,0.0,50174.4\nR001,2014-08-03,22.2,0.4,48171.42\nR001,2014-08-04,20.9,0.6,46168.02\nR001,2014-08-05,19.4,5.8,41400.92\nR001,2014-08-06,17.5,16.6,2202.46\nR001,2014-08-07,17.9,2.9,46932.43\nR001,2014-08-08,19.4,0.0,46962.26\nR001,2014-08-09,19.7,0.2,47668.64\nR001,2014-08-10,19.2,3.5,47359.53\nR001,2014-08-11,19.6,1.4,46504.5\nR001,2014-08-12,19.9,0.7,46800.0\nR001,2014-08-13,20.0,1.2,47111.71\nR001,2014-08-14,22.2,0.0,46800.0\nR001,2014-08-15,24.4,0.0,47171.75\nR001,2014-08-16,23.7,1.2,46869.0\nR001,2014-08-17,23.5,0.0,46800.0\nR001,2014-08-18,23.0,4.7,46931.4\nR001,2014-08-19,21.4,4.9,40353.68\nR001,2014-08-20,20.7,4.0,39742.09\nR001,2014-08-21,20.5,1.5,43869.64\nR001,2014-08-22,21.2,0.2,46800.0\nR001,2014-08-23,21.1,0.2,46800.0\nR001,2014-08-24,20.8,1.0,46800.0\nR001,2014-08-25,21.5,1.4,46215.64\nR001,2014-08-26,21.0,6.2,46285.55\nR001,2014-08-27,22.3,0.0,46529.92\nR001,2014-08-28,23.1,0.1,46636.99\nR001,2014-08-29,20.8,3.9,40419.12\nR001,2014-08-30,19.3,0.0,43299.1\nR001,2014-08-31,18.9,0.3,45177.31\nR001,2014-09-01,19.7,1.3,44902.72\nR001,2014-09-02,20.7,0.0,45914.0\nR001,2014-09-03,22.4,0.3,45319.68\nR001,2014-09-04,20.7,3.8,44203.33\nR001,2014-09-05,19.5,1.1,43817.35\nR001,2014-09-06,20.6,0.2,44508.83\nR001,2014-09-07,20.7,2.2,43601.99\nR001,2014-09-08,19.8,4.9,43335.86\nR001,2014-09-09,17.1,18.0,33381.19\nR001,2014-09-10,17.9,0.6,41567.21\nR001,2014-09-11,16.1,3.4,16568.72\nR001,2014-09-12,17.3,0.7,42457.23\nR001,2014-09-13,16.2,0.0,43181.06\nR001,2014-09-14,16.0,1.0,41984.14\nR001,2014-09-15,16.2,4.1,36256.17\nR001,2014-09-16,14.3,4.7,37178.79\nR001,2014-09-17,15.5,0.6,41020.37\nR001,2014-09-18,14.0,8.2,17113.46\nR001,2014-09-19,14.7,7.4,2704.31\nR001,2014-09-20,13.4,6.9,15499.85\nR001,2014-09-21,12.7,6.4,23252.83\nR001,2014-09-22,13.1,0.0,40023.9\nR001,2014-09-23,14.9,1.0,41969.22\nR001,2014-09-24,11.0,33.2,24269.55\nR001,2014-09-25,7.3,10.5,25238.85\nR001,2014-09-26,9.0,0.0,40163.64\nR001,2014-09-27,12.1,0.0,39880.21\nR001,2014-09-28,15.5,7.8,16062.02\nR001,2014-09-29,10.7,20.2,16700.23\nR001,2014-09-30,10.7,4.3,39468.02\nR001,2014-10-01,10.9,2.0,29692.75\nR001,2014-10-02,10.3,0.3,27813.75\nR001,2014-10-03,9.9,0.0,33134.03\nR001,2014-10-04,11.0,0.3,33783.56\nR001,2014-10-05,10.6,8.3,23938.58\nR001,2014-10-06,10.8,0.0,28441.69\nR001,2014-10-07,10.3,0.0,37880.81\nR001,2014-10-08,10.7,0.0,28418.15\nR001,2014-10-09,10.8,0.0,37670.11\nR001,2014-10-10,11.2,0.0,37840.94\nR001,2014-10-11,13.0,0.0,37283.72\nR001,2014-10-12,13.9,0.0,36859.15\nR001,2014-10-13,13.8,0.2,36260.6\nR001,2014-10-14,13.8,0.0,36313.04\nR001,2014-10-15,13.5,8.6,30157.29\nR001,2014-10-16,13.1,9.8,24674.13\nR001,2014-10-17,10.8,18.2,1383.68\nR001,2014-10-18,9.1,28.8,0.0\nR001,2014-10-19,4.9,7.6,34870.97\nR001,2014-10-20,4.1,0.1,34810.89\nR001,2014-10-21,6.7,0.0,36000.0\nR001,2014-10-22,11.1,1.7,33173.42\nR001,2014-10-23,13.0,0.0,36000.0\nR001,2014-10-24,13.3,0.0,36000.0\nR001,2014-10-25,9.0,17.4,2879.5\nR001,2014-10-26,7.4,0.0,36000.0\nR001,2014-10-27,9.2,0.1,36000.0\nR001,2014-10-28,9.5,0.0,36000.0\nR001,2014-10-29,9.6,0.0,36000.0\nR001,2014-10-30,10.2,0.0,36000.0\nR001,2014-10-31,9.2,1.5,21470.63\nR001,2014-11-01,8.4,4.9,14548.75\nR001,2014-11-02,1.8,4.7,25900.19\nR001,2014-11-03,-0.8,0.0,35873.04\nR001,2014-11-04,0.9,1.1,33857.68\nR001,2014-11-05,4.2,0.0,35713.81\nR001,2014-11-06,7.6,0.0,35558.69\nR001,2014-11-07,8.1,0.0,35404.4\nR001,2014-11-08,7.9,0.0,35251.33\nR001,2014-11-09,8.5,0.0,35099.68\nR001,2014-11-10,10.1,0.0,34031.81\nR001,2014-11-11,8.5,0.0,25201.94\nR001,2014-11-12,7.4,0.0,30604.63\nR001,2014-11-13,6.8,0.0,34510.82\nR001,2014-11-14,7.7,0.0,33525.81\nR001,2014-11-15,9.2,0.3,26268.81\nR001,2014-11-16,8.5,0.6,30094.7\nR001,2014-11-17,7.0,0.5,30102.36\nR001,2014-11-18,7.2,0.7,28009.43\nR001,2014-11-19,7.2,0.3,31487.26\nR001,2014-11-20,6.8,0.0,31560.35\nR001,2014-11-21,7.4,0.0,16187.86\nR001,2014-11-22,5.9,0.0,30228.53\nR001,2014-11-23,1.9,0.0,31780.02\nR001,2014-11-24,2.2,0.0,30358.58\nR001,2014-11-25,1.8,0.0,30800.33\nR001,2014-11-26,2.4,0.0,28769.57\nR001,2014-11-27,3.8,0.7,9661.39\nR001,2014-11-28,3.2,5.7,16169.06\nR001,2014-11-29,1.0,0.7,23166.84\nR001,2014-11-30,1.9,25.7,0.0\nR001,2014-12-01,-3.1,0.8,28783.18\nR001,2014-12-02,-5.7,0.0,26092.97\nR001,2014-12-03,-5.1,0.0,30352.84\nR001,2014-12-04,-3.8,0.0,28800.0\nR001,2014-12-05,2.9,2.5,6958.96\nR001,2014-12-06,0.7,0.0,30390.42\nR001,2014-12-07,4.0,0.0,29630.85\nR001,2014-12-08,7.6,0.0,22767.48\nR001,2014-12-09,8.1,0.0,30028.69\nR001,2014-12-10,9.3,0.6,17838.48\nR001,2014-12-11,6.4,8.1,11252.96\nR001,2014-12-12,7.3,0.1,20116.29\nR001,2014-12-13,6.5,3.9,15480.52\nR001,2014-12-14,3.4,0.0,23336.75\nR001,2014-12-15,3.6,0.0,29640.61\nR001,2014-12-16,2.9,0.0,29245.02\nR001,2014-12-17,4.3,0.0,28847.14\nR001,2014-12-18,8.1,0.0,25105.85\nR001,2014-12-19,4.6,3.7,23359.57\nR001,2014-12-20,3.1,0.0,27224.02\nR001,2014-12-21,3.8,0.3,23075.66\nR001,2014-12-22,-0.1,19.8,5152.22\nR001,2014-12-23,-3.6,6.2,1363.42\nR001,2014-12-24,2.6,32.2,0.0\nR001,2014-12-25,4.4,1.5,31282.22\nR001,2014-12-26,3.3,0.0,30670.27\nR001,2014-12-27,2.9,0.0,28940.7\nR001,2014-12-28,2.0,20.8,17004.82\nR001,2014-12-29,2.3,6.9,5290.09\nR001,2014-12-30,3.2,37.2,0.0\nR001,2014-12-31,2.2,29.1,0.0\nR001,2015-01-01,1.9,9.0,3967.65\nR001,2015-01-02,3.0,3.0,6533.18\nR001,2015-01-03,0.6,7.8,20785.96\nR001,2015-01-04,-2.1,11.8,22440.76\nR001,2015-01-05,-1.8,24.1,0.0\nR001,2015-01-06,-0.9,16.8,0.0\nR001,2015-01-07,-0.9,25.0,0.0\nR001,2015-01-08,-5.5,0.5,23433.1\nR001,2015-01-09,-8.6,0.0,27241.51\nR001,2015-01-10,-5.3,0.4,27302.41\nR001,2015-01-11,-2.9,5.2,4131.0\nR001,2015-01-12,2.7,24.0,0.0\nR001,2015-01-13,1.8,17.3,22202.13\nR001,2015-01-14,2.5,0.0,32400.0\nR001,2015-01-15,2.8,0.0,32400.0\nR001,2015-01-16,0.9,0.0,29781.39\nR001,2015-01-17,-0.6,0.0,32400.0\nR001,2015-01-18,-0.8,0.0,32400.0\nR001,2015-01-19,0.9,0.0,30109.61\nR001,2015-01-20,1.9,2.1,32400.0\nR001,2015-01-21,4.2,0.0,30058.74\nR001,2015-01-22,3.6,0.0,28541.86\nR001,2015-01-23,2.0,0.0,32400.0\nR001,2015-01-24,2.9,0.0,32400.0\nR001,2015-01-25,6.3,0.1,32400.0\nR001,2015-01-26,3.5,6.6,28577.8\nR001,2015-01-27,1.7,0.0,32400.0\nR001,2015-01-28,2.7,0.0,25513.12\nR001,2015-01-29,1.8,6.4,22945.22\nR001,2015-01-30,3.6,0.0,28956.37\nR001,2015-01-31,4.2,0.0,27798.05\nR001,2015-02-01,6.3,0.2,13938.4\nR001,2015-02-02,8.4,0.0,32685.24\nR001,2015-02-03,6.8,6.5,0.0\nR001,2015-02-04,2.0,2.3,32595.83\nR001,2015-02-05,-0.4,4.5,22011.24\nR001,2015-02-06,3.6,0.0,32096.94\nR001,2015-02-07,3.6,3.6,12588.47\nR001,2015-02-08,0.1,5.6,33125.88\nR001,2015-02-09,0.5,7.8,8946.71\nR001,2015-02-10,0.6,0.4,31599.29\nR001,2015-02-11,3.9,0.6,30380.67\nR001,2015-02-12,2.5,6.1,20776.09\nR001,2015-02-13,2.4,2.3,19720.95\nR001,2015-02-14,1.2,1.2,29094.42\nR001,2015-02-15,-0.2,0.1,34734.79\nR001,2015-02-16,-1.2,8.6,30017.2\nR001,2015-02-17,-2.5,10.9,16588.24\nR001,2015-02-18,-2.6,0.0,36702.43\nR001,2015-02-19,-1.1,0.3,37895.06\nR001,2015-02-20,-1.7,3.3,24334.97\nR001,2015-02-21,-2.4,0.0,35787.22\nR001,2015-02-22,0.5,0.0,38422.28\nR001,2015-02-23,2.8,0.0,38598.83\nR001,2015-02-24,3.9,0.0,35438.2\nR001,2015-02-25,5.3,0.0,38952.27\nR001,2015-02-26,4.6,0.0,39128.89\nR001,2015-02-27,5.9,0.0,38558.46\nR001,2015-02-28,6.1,0.3,25395.32\nR001,2015-03-01,4.1,8.8,1361.31\nR001,2015-03-02,3.0,2.0,38402.86\nR001,2015-03-03,3.4,0.0,39600.0\nR001,2015-03-04,0.5,6.9,14546.1\nR001,2015-03-05,-1.1,3.3,38041.91\nR001,2015-03-06,-0.1,0.0,36038.06\nR001,2015-03-07,4.2,5.6,20340.29\nR001,2015-03-08,3.5,0.0,37130.33\nR001,2015-03-09,4.1,0.0,39600.0\nR001,2015-03-10,7.5,0.0,39600.0\nR001,2015-03-11,11.2,0.0,39600.0\nR001,2015-03-12,10.4,3.9,13864.47\nR001,2015-03-13,7.7,7.9,31922.36\nR001,2015-03-14,8.1,0.0,36190.23\nR001,2015-03-15,7.3,7.2,36612.95\nR001,2015-03-16,5.8,9.3,13431.79\nR001,2015-03-17,2.6,9.7,30075.13\nR001,2015-03-18,3.7,0.6,36837.16\nR001,2015-03-19,5.5,0.3,36758.26\nR001,2015-03-20,4.9,7.5,14617.47\nR001,2015-03-21,3.7,5.5,19706.15\nR001,2015-03-22,5.5,1.5,37662.22\nR001,2015-03-23,2.8,0.0,33830.28\nR001,2015-03-24,2.1,0.9,39831.08\nR001,2015-03-25,5.6,0.0,40435.51\nR001,2015-03-26,9.4,0.0,43082.73\nR001,2015-03-27,11.7,0.0,40170.11\nR001,2015-03-28,12.5,2.2,5286.39\nR001,2015-03-29,11.3,10.5,37218.55\nR001,2015-03-30,8.4,6.2,10958.79\nR001,2015-03-31,8.0,0.3,37911.5\nR001,2015-04-01,6.9,4.5,21550.01\nR001,2015-04-02,6.4,18.1,29948.4\nR001,2015-04-03,1.7,4.3,43642.1\nR001,2015-04-04,0.4,12.9,4831.81\nR001,2015-04-05,0.8,0.1,41507.5\nR001,2015-04-06,6.5,0.0,42929.53\nR001,2015-04-07,9.1,8.5,30606.42\nR001,2015-04-08,4.0,17.0,19038.05\nR001,2015-04-09,4.5,23.0,2034.79\nR001,2015-04-10,3.0,28.0,4337.27\nR001,2015-04-11,4.2,0.0,38024.08\nR001,2015-04-12,4.5,3.3,2568.5\nR001,2015-04-13,5.5,0.6,38140.13\nR001,2015-04-14,5.8,2.5,46558.38\nR001,2015-04-15,1.8,18.6,12948.0\nR001,2015-04-16,4.2,0.2,43057.8\nR001,2015-04-17,9.4,0.0,46772.87\nR001,2015-04-18,11.0,3.5,35476.72\nR001,2015-04-19,8.1,31.6,17773.14\nR001,2015-04-20,1.3,4.4,46800.0\nR001,2015-04-21,2.1,0.7,38358.97\nR001,2015-04-22,2.3,5.8,44520.96\nR001,2015-04-23,4.1,0.0,46528.38\nR001,2015-04-24,4.6,0.3,47139.28\nR001,2015-04-25,7.8,0.0,46800.0\nR001,2015-04-26,11.1,0.0,44845.8\nR001,2015-04-27,11.0,3.3,39310.22\nR001,2015-04-28,11.0,2.4,47301.65\nR001,2015-04-29,13.5,0.6,43581.03\nR001,2015-04-30,12.1,4.0,44937.44\nR001,2015-05-01,11.0,3.6,44938.66\nR001,2015-05-02,10.3,0.3,46662.91\nR001,2015-05-03,12.0,0.0,43993.18\nR001,2015-05-04,10.0,6.7,8134.7\nR001,2015-05-05,8.8,8.6,43200.0\nR001,2015-05-06,7.5,2.4,36633.57\nR001,2015-05-07,8.0,3.9,20582.74\nR001,2015-05-08,9.6,2.2,38338.51\nR001,2015-05-09,9.8,2.9,37889.91\nR001,2015-05-10,9.3,0.7,45159.03\nR001,2015-05-11,9.9,1.3,45307.99\nR001,2015-05-12,10.4,2.9,42390.02\nR001,2015-05-13,10.1,6.2,40222.06\nR001,2015-05-14,9.8,7.7,43076.26\nR001,2015-05-15,11.4,0.5,46482.93\nR001,2015-05-16,11.4,10.3,17311.29\nR001,2015-05-17,10.8,8.6,36684.57\nR001,2015-05-18,11.1,0.0,50404.12\nR001,2015-05-19,13.0,0.0,50400.0\nR001,2015-05-20,15.6,0.0,50702.33\nR001,2015-05-21,18.5,0.0,50675.6\nR001,2015-05-22,18.6,5.4,50400.0\nR001,2015-05-23,16.6,7.1,38690.24\nR001,2015-05-24,14.8,17.6,37039.26\nR001,2015-05-25,16.4,3.3,46327.65\nR001,2015-05-26,17.3,3.7,49895.05\nR001,2015-05-27,17.6,1.0,50879.21\nR001,2015-05-28,16.0,7.0,48566.1\nR001,2015-05-29,14.9,3.7,48744.03\nR001,2015-05-30,15.3,9.9,43109.76\nR001,2015-05-31,13.6,15.7,32896.25\nR001,2015-06-01,15.0,5.5,51186.67\nR001,2015-06-02,17.0,0.0,50032.81\nR001,2015-06-03,17.5,4.6,51085.56\nR001,2015-06-04,17.3,10.9,48343.83\nR001,2015-06-05,18.3,2.5,47863.45\nR001,2015-06-06,19.1,3.9,39755.81\nR001,2015-06-07,18.5,0.0,43219.04\nR001,2015-06-08,19.4,0.0,52823.91\nR001,2015-06-09,18.4,8.0,48111.18\nR001,2015-06-10,18.7,2.8,52796.99\nR001,2015-06-11,17.0,6.6,52051.07\nR001,2015-06-12,15.8,10.5,48535.04\nR001,2015-06-13,16.7,3.2,49666.67\nR001,2015-06-14,17.2,7.0,50701.2\nR001,2015-06-15,15.9,19.2,41654.55\nR001,2015-06-16,15.6,16.8,14203.42\nR001,2015-06-17,16.2,9.3,38590.34\nR001,2015-06-18,17.8,0.1,52358.85\nR001,2015-06-19,18.6,4.6,53432.03\nR001,2015-06-20,18.4,9.8,46334.68\nR001,2015-06-21,18.1,8.2,49819.04\nR001,2015-06-22,18.9,7.4,51679.46\nR001,2015-06-23,19.5,11.2,51033.55\nR001,2015-06-24,20.0,16.2,47243.9\nR001,2015-06-25,18.1,16.7,32236.9\nR001,2015-06-26,17.9,27.2,17784.17\nR001,2015-06-27,17.3,4.2,43926.9\nR001,2015-06-28,17.0,8.2,48897.78\nR001,2015-06-29,17.3,16.3,46901.7\nR001,2015-06-30,17.6,3.0,49504.36\nR001,2015-07-01,18.3,0.0,50100.59\nR001,2015-07-02,19.5,1.0,49231.16\nR001,2015-07-03,18.0,10.8,31303.43\nR001,2015-07-04,20.1,1.5,45766.85\nR001,2015-07-05,21.1,1.9,41643.18\nR001,2015-07-06,19.4,11.6,7765.58\nR001,2015-07-07,19.8,1.7,50763.9\nR001,2015-07-08,20.3,0.8,53290.03\nR001,2015-07-09,22.4,1.3,53042.12\nR001,2015-07-10,20.5,25.4,26996.67\nR001,2015-07-11,18.6,5.1,47106.52\nR001,2015-07-12,17.5,0.0,51893.93\nR001,2015-07-13,16.8,0.5,50335.52\nR001,2015-07-14,18.1,0.1,52725.35\nR001,2015-07-15,17.3,5.8,39865.55\nR001,2015-07-16,18.0,0.0,50825.81\nR001,2015-07-17,18.0,0.0,49954.78\nR001,2015-07-18,16.5,0.0,52777.34\nR001,2015-07-19,17.5,0.0,53438.46\nR001,2015-07-20,18.2,0.0,52248.6\nR001,2015-07-21,20.1,0.0,52751.65\nR001,2015-07-22,19.4,1.9,49975.1\nR001,2015-07-23,19.2,0.0,47444.26\nR001,2015-07-24,22.1,0.0,52284.56\nR001,2015-07-25,24.2,0.7,51544.5\nR001,2015-07-26,23.9,0.7,48437.48\nR001,2015-07-27,21.3,8.0,49152.62\nR001,2015-07-28,20.9,3.5,47811.6\nR001,2015-07-29,22.0,0.0,50298.28\nR001,2015-07-30,23.2,0.0,48964.61\nR001,2015-07-31,26.0,0.0,50636.12\nR001,2015-08-01,25.1,1.7,50257.22\nR001,2015-08-02,23.8,3.5,48098.32\nR001,2015-08-03,23.1,7.8,44087.27\nR001,2015-08-04,23.2,3.7,48219.1\nR001,2015-08-05,22.5,2.7,48622.23\nR001,2015-08-06,21.8,0.2,46978.73\nR001,2015-08-07,21.7,0.0,48625.24\nR001,2015-08-08,21.5,0.0,47789.66\nR001,2015-08-09,21.1,0.0,47321.93\nR001,2015-08-10,22.6,0.0,48026.07\nR001,2015-08-11,23.9,0.0,47716.55\nR001,2015-08-12,23.7,1.5,47408.77\nR001,2015-08-13,21.9,3.8,46509.23\nR001,2015-08-14,23.1,1.2,46560.35\nR001,2015-08-15,23.1,0.8,46829.43\nR001,2015-08-16,21.0,5.8,46728.67\nR001,2015-08-17,21.4,0.0,46800.0\nR001,2015-08-18,20.4,0.0,46923.5\nR001,2015-08-19,21.4,0.1,46800.0\nR001,2015-08-20,21.6,0.6,46800.0\nR001,2015-08-21,21.7,0.0,46800.0\nR001,2015-08-22,21.4,0.8,46800.0\nR001,2015-08-23,19.9,4.6,45177.75\nR001,2015-08-24,17.3,13.3,19635.87\nR001,2015-08-25,16.6,2.4,35377.37\nR001,2015-08-26,16.4,10.9,38203.35\nR001,2015-08-27,17.3,0.0,46768.15\nR001,2015-08-28,17.3,0.6,42749.5\nR001,2015-08-29,17.7,0.1,46567.29\nR001,2015-08-30,18.2,0.0,46464.85\nR001,2015-08-31,17.7,0.8,45484.69\nR001,2015-09-01,18.1,0.0,46256.47\nR001,2015-09-02,19.9,0.0,46150.75\nR001,2015-09-03,20.8,0.0,45628.99\nR001,2015-09-04,21.5,0.0,45357.79\nR001,2015-09-05,21.8,0.4,44447.34\nR001,2015-09-06,21.6,1.2,43859.23\nR001,2015-09-07,21.9,0.3,43911.88\nR001,2015-09-08,19.6,1.5,38198.87\nR001,2015-09-09,18.6,5.3,33290.08\nR001,2015-09-10,19.7,0.4,38209.37\nR001,2015-09-11,20.7,1.3,42852.46\nR001,2015-09-12,19.6,0.3,41516.12\nR001,2015-09-13,19.2,4.1,29098.71\nR001,2015-09-14,17.6,6.3,40751.81\nR001,2015-09-15,17.9,1.2,39600.0\nR001,2015-09-16,17.4,0.3,38785.59\nR001,2015-09-17,17.0,0.0,42073.97\nR001,2015-09-18,18.2,0.0,42861.06\nR001,2015-09-19,19.8,0.0,42793.46\nR001,2015-09-20,21.7,0.0,42725.7\nR001,2015-09-21,22.4,0.3,42209.11\nR001,2015-09-22,21.6,0.1,40970.4\nR001,2015-09-23,20.9,0.5,40368.34\nR001,2015-09-24,20.2,2.1,40757.9\nR001,2015-09-25,19.6,0.7,41713.0\nR001,2015-09-26,18.6,0.2,41430.19\nR001,2015-09-27,17.7,1.8,40605.24\nR001,2015-09-28,17.1,2.5,39907.1\nR001,2015-09-29,16.9,0.0,40897.32\nR001,2015-09-30,17.1,1.3,36262.93\nR001,2015-10-01,15.2,7.5,11918.83\nR001,2015-10-02,15.9,0.6,28661.26\nR001,2015-10-03,15.2,1.2,34523.94\nR001,2015-10-04,13.3,3.5,38069.12\nR001,2015-10-05,13.1,1.8,37025.32\nR001,2015-10-06,13.1,0.8,38453.3\nR001,2015-10-07,12.6,0.2,29518.59\nR001,2015-10-08,10.0,0.0,36238.86\nR001,2015-10-09,8.0,0.7,36355.58\nR001,2015-10-10,6.9,0.1,36232.57\nR001,2015-10-11,8.3,1.8,34228.1\nR001,2015-10-12,10.7,55.4,180.32\nR001,2015-10-13,12.2,11.5,25155.25\nR001,2015-10-14,11.4,10.8,5609.34\nR001,2015-10-15,10.6,3.6,30243.97\nR001,2015-10-16,9.0,0.3,35264.07\nR001,2015-10-17,8.7,0.0,36132.49\nR001,2015-10-18,10.5,0.0,30316.02\nR001,2015-10-19,11.9,0.0,36000.0\nR001,2015-10-20,11.8,0.0,36000.0\nR001,2015-10-21,12.1,0.0,36000.0\nR001,2015-10-22,13.6,1.1,26682.09\nR001,2015-10-23,14.8,0.0,36000.0\nR001,2015-10-24,14.1,20.3,1460.34\nR001,2015-10-25,10.2,22.7,3243.27\nR001,2015-10-26,6.9,0.0,36000.0\nR001,2015-10-27,6.7,0.0,35002.58\nR001,2015-10-28,5.9,0.0,36000.0\nR001,2015-10-29,4.5,0.0,36000.0\nR001,2015-10-30,5.6,0.0,36000.0\nR001,2015-10-31,5.9,0.0,31002.44\nR001,2015-11-01,5.6,0.9,36000.0\nR001,2015-11-02,6.5,2.2,31756.17\nR001,2015-11-03,7.4,0.0,29512.15\nR001,2015-11-04,7.4,0.0,35810.01\nR001,2015-11-05,8.4,0.0,35729.51\nR001,2015-11-06,8.0,0.0,35588.61\nR001,2015-11-07,8.1,0.0,35432.82\nR001,2015-11-08,6.3,0.5,31342.39\nR001,2015-11-09,5.5,9.6,10850.17\nR001,2015-11-10,4.5,1.8,32858.71\nR001,2015-11-11,3.2,46.2,178.48\nR001,2015-11-12,2.6,4.4,32686.29\nR001,2015-11-13,2.1,39.0,1001.62\nR001,2015-11-14,0.6,0.0,34087.5\nR001,2015-11-15,1.5,7.0,101.28\nR001,2015-11-16,0.3,2.7,13866.09\nR001,2015-11-17,0.9,14.9,22161.37\nR001,2015-11-18,-1.3,0.0,30345.86\nR001,2015-11-19,1.2,6.3,18153.22\nR001,2015-11-20,4.2,1.1,23923.24\nR001,2015-11-21,7.2,0.0,23942.9\nR001,2015-11-22,11.3,0.0,30950.19\nR001,2015-11-23,9.7,0.0,32219.36\nR001,2015-11-24,7.4,0.0,26512.3\nR001,2015-11-25,9.5,0.0,4016.45\nR001,2015-11-26,11.1,0.0,29837.18\nR001,2015-11-27,7.3,13.8,0.0\nR001,2015-11-28,8.1,0.0,29505.23\nR001,2015-11-29,6.5,12.4,1091.92\nR001,2015-11-30,5.3,9.2,17923.51\nR001,2015-12-01,3.3,17.7,25713.57\nR001,2015-12-02,0.2,44.4,0.0\nR001,2015-12-03,-4.6,0.0,30168.9\nR001,2015-12-04,-6.4,0.0,28725.49\nR001,2015-12-05,-4.9,0.0,28800.0\nR001,2015-12-06,-0.4,1.7,12310.05\nR001,2015-12-07,2.6,0.6,24398.52\nR001,2015-12-08,2.2,0.5,29226.59\nR001,2015-12-09,-2.6,0.0,30026.98\nR001,2015-12-10,0.1,0.0,29644.57\nR001,2015-12-11,1.1,0.0,28800.0\nR001,2015-12-12,0.9,0.0,30034.1\nR001,2015-12-13,0.4,1.7,16067.82\nR001,2015-12-14,1.2,0.4,28247.11\nR001,2015-12-15,0.0,7.1,30236.63\nR001,2015-12-16,-2.0,0.0,26741.39\nR001,2015-12-17,-2.4,0.0,30405.88\nR001,2015-12-18,-2.4,0.0,28800.0\nR001,2015-12-19,-0.9,0.0,30779.85\nR001,2015-12-20,0.6,4.9,18000.0\nR001,2015-12-21,3.1,5.5,17599.96\nR001,2015-12-22,4.0,0.0,29460.6\nR001,2015-12-23,3.1,7.2,23678.47\nR001,2015-12-24,2.9,4.3,22558.38\nR001,2015-12-25,1.8,1.8,30697.75\nR001,2015-12-26,0.7,0.0,26691.44\nR001,2015-12-27,3.1,8.1,10787.53\nR001,2015-12-28,3.0,6.5,9018.72\nR001,2015-12-29,4.3,3.6,27838.08\nR001,2015-12-30,-0.7,22.0,0.0\nR001,2015-12-31,-7.5,6.3,28848.76\nR001,2016-01-01,-9.8,1.2,30284.1\nR001,2016-01-02,-13.8,0.0,31634.91\nR001,2016-01-03,-11.2,6.0,112.44\nR001,2016-01-04,-9.5,5.5,0.0\nR001,2016-01-05,-0.3,23.2,0.0\nR001,2016-01-06,1.6,14.1,16988.66\nR001,2016-01-07,4.7,10.0,4693.58\nR001,2016-01-08,4.6,19.2,905.05\nR001,2016-01-09,0.2,17.4,5164.56\nR001,2016-01-10,0.4,2.6,28642.0\nR001,2016-01-11,3.6,8.4,0.0\nR001,2016-01-12,5.0,1.0,25933.62\nR001,2016-01-13,4.7,22.1,0.0\nR001,2016-01-14,1.4,8.7,22237.42\nR001,2016-01-15,-2.3,0.3,32400.0\nR001,2016-01-16,-1.2,0.0,32400.0\nR001,2016-01-17,5.5,0.0,21200.78\nR001,2016-01-18,6.4,3.5,23491.42\nR001,2016-01-19,1.1,23.7,118.83\nR001,2016-01-20,-4.0,5.9,31267.08\nR001,2016-01-21,-4.0,4.2,7561.56\nR001,2016-01-22,-0.8,23.0,0.0\nR001,2016-01-23,-2.4,10.2,0.0\nR001,2016-01-24,-5.1,5.7,0.0\nR001,2016-01-25,-7.7,1.4,20919.95\nR001,2016-01-26,-9.6,0.0,32400.0\nR001,2016-01-27,-10.4,0.0,32400.0\nR001,2016-01-28,-4.7,7.2,1265.9\nR001,2016-01-29,-0.2,18.5,0.0\nR001,2016-01-30,0.0,0.4,21806.52\nR001,2016-01-31,0.7,0.0,32430.48\nR001,2016-02-01,1.9,8.7,3849.27\nR001,2016-02-02,-0.7,20.7,32053.11\nR001,2016-02-03,1.3,0.0,33001.01\nR001,2016-02-04,7.1,0.0,30203.12\nR001,2016-02-05,8.6,0.0,30474.82\nR001,2016-02-06,5.1,4.1,17179.86\nR001,2016-02-07,1.4,1.9,27664.74\nR001,2016-02-08,0.9,0.8,32776.96\nR001,2016-02-09,0.8,0.0,34987.44\nR001,2016-02-10,-0.0,0.0,35749.22\nR001,2016-02-11,1.0,0.3,34455.1\nR001,2016-02-12,4.3,5.4,18622.69\nR001,2016-02-13,7.2,0.0,34608.18\nR001,2016-02-14,6.6,1.6,16466.5\nR001,2016-02-15,8.1,0.5,33555.15\nR001,2016-02-16,9.3,3.4,27871.15\nR001,2016-02-17,8.7,3.5,34740.83\nR001,2016-02-18,3.7,0.2,36024.36\nR001,2016-02-19,1.1,0.0,37860.33\nR001,2016-02-20,-0.6,0.5,33867.53\nR001,2016-02-21,0.8,1.2,21465.74\nR001,2016-02-22,2.4,6.6,31167.48\nR001,2016-02-23,6.2,0.0,38552.64\nR001,2016-02-24,7.1,0.0,38725.32\nR001,2016-02-25,5.7,14.0,0.0\nR001,2016-02-26,6.1,0.3,37578.58\nR001,2016-02-27,8.0,0.0,39250.93\nR001,2016-02-28,6.3,9.4,28092.31\nR001,2016-02-29,6.2,0.0,39496.9\nR001,2016-03-01,9.5,0.0,39598.65\nR001,2016-03-02,13.6,0.0,21600.0\nR001,2016-03-03,13.3,0.0,30847.08\nR001,2016-03-04,10.4,5.9,39600.0\nR001,2016-03-05,7.9,6.2,27821.5\nR001,2016-03-06,6.7,2.7,28454.25\nR001,2016-03-07,6.7,0.0,39600.0\nR001,2016-03-08,8.1,0.0,39600.0\nR001,2016-03-09,8.5,0.0,39600.0\nR001,2016-03-10,8.4,0.0,39600.0\nR001,2016-03-11,8.1,4.4,32286.44\nR001,2016-03-12,5.1,2.5,32400.0\nR001,2016-03-13,5.5,0.4,39600.0\nR001,2016-03-14,4.8,2.8,1631.71\nR001,2016-03-15,1.1,2.9,38147.14\nR001,2016-03-16,-0.4,4.3,37981.78\nR001,2016-03-17,-2.1,5.0,30488.2\nR001,2016-03-18,-0.1,3.5,34250.63\nR001,2016-03-19,1.7,5.1,24901.52\nR001,2016-03-20,-2.2,8.4,28718.41\nR001,2016-03-21,-1.3,0.0,40237.26\nR001,2016-03-22,3.1,0.0,40146.1\nR001,2016-03-23,5.2,11.8,20960.39\nR001,2016-03-24,10.3,3.8,26910.08\nR001,2016-03-25,9.4,10.2,31924.47\nR001,2016-03-26,6.4,10.1,28348.77\nR001,2016-03-27,3.9,10.1,12895.06\nR001,2016-03-28,3.9,3.6,12712.32\nR001,2016-03-29,3.1,8.6,2843.85\nR001,2016-03-30,3.3,3.2,31683.1\nR001,2016-03-31,3.9,0.0,43797.81\nR001,2016-04-01,6.7,0.0,44091.03\nR001,2016-04-02,8.1,1.3,31189.22\nR001,2016-04-03,1.1,2.2,44686.04\nR001,2016-04-04,2.7,0.0,39600.0\nR001,2016-04-05,6.1,0.0,45971.42\nR001,2016-04-06,8.5,0.0,46042.74\nR001,2016-04-07,11.0,0.0,45572.98\nR001,2016-04-08,12.3,0.5,46185.54\nR001,2016-04-09,13.9,0.0,46257.01\nR001,2016-04-10,12.6,2.7,23027.07\nR001,2016-04-11,12.8,6.2,36786.66\nR001,2016-04-12,10.4,6.5,8491.7\nR001,2016-04-13,10.4,2.4,40206.44\nR001,2016-04-14,10.1,0.9,41516.1\nR001,2016-04-15,11.2,1.4,45404.93\nR001,2016-04-16,8.1,18.9,31500.56\nR001,2016-04-17,9.0,0.0,46800.0\nR001,2016-04-18,11.3,0.3,43200.0\nR001,2016-04-19,14.4,0.5,46465.57\nR001,2016-04-20,16.5,0.1,40304.83\nR001,2016-04-21,6.7,27.7,0.0\nR001,2016-04-22,3.5,1.0,46556.7\nR001,2016-04-23,10.0,0.0,43918.02\nR001,2016-04-24,14.1,0.0,32400.0\nR001,2016-04-25,13.8,5.1,40570.6\nR001,2016-04-26,13.2,3.2,45605.92\nR001,2016-04-27,10.1,17.4,17965.23\nR001,2016-04-28,10.6,0.3,47274.74\nR001,2016-04-29,9.8,1.3,44735.78\nR001,2016-04-30,11.0,0.4,39769.4\nR001,2016-05-01,10.0,2.4,35494.06\nR001,2016-05-02,10.7,0.0,46158.74\nR001,2016-05-03,12.4,0.0,49104.13\nR001,2016-05-04,10.2,10.4,8193.92\nR001,2016-05-05,10.8,8.3,37165.2\nR001,2016-05-06,9.5,9.4,13685.11\nR001,2016-05-07,9.7,6.5,21490.28\nR001,2016-05-08,10.6,1.2,46903.83\nR001,2016-05-09,12.0,1.0,47554.35\nR001,2016-05-10,13.5,0.8,45645.69\nR001,2016-05-11,10.2,24.2,27735.82\nR001,2016-05-12,11.8,0.1,48841.76\nR001,2016-05-13,13.0,2.6,32902.92\nR001,2016-05-14,16.1,2.3,48494.74\nR001,2016-05-15,15.6,0.8,48285.33\nR001,2016-05-16,10.6,21.0,5223.83\nR001,2016-05-17,13.3,0.0,48454.08\nR001,2016-05-18,13.8,3.8,46800.0\nR001,2016-05-19,11.6,9.5,31005.81\nR001,2016-05-20,11.8,0.0,49355.91\nR001,2016-05-21,13.3,0.0,50400.0\nR001,2016-05-22,15.7,0.0,50287.83\nR001,2016-05-23,15.0,12.9,34020.26\nR001,2016-05-24,12.3,28.7,356.5\nR001,2016-05-25,14.5,1.8,48142.58\nR001,2016-05-26,13.4,5.4,22257.88\nR001,2016-05-27,15.1,4.6,47261.67\nR001,2016-05-28,14.3,13.1,46637.01\nR001,2016-05-29,14.4,10.5,29987.88\nR001,2016-05-30,13.7,2.4,34593.79\nR001,2016-05-31,15.6,2.8,49016.26\nR001,2016-06-01,14.9,2.1,48336.74\nR001,2016-06-02,15.2,9.3,44362.26\nR001,2016-06-03,15.1,1.4,38178.63\nR001,2016-06-04,14.0,9.6,16264.05\nR001,2016-06-05,14.5,4.6,43660.65\nR001,2016-06-06,15.2,0.3,51217.81\nR001,2016-06-07,14.9,0.0,51446.92\nR001,2016-06-08,13.5,2.0,50086.82\nR001,2016-06-09,13.0,4.6,48536.85\nR001,2016-06-10,12.6,11.8,34311.04\nR001,2016-06-11,13.6,4.8,47553.09\nR001,2016-06-12,15.6,0.0,49813.08\nR001,2016-06-13,17.5,0.0,52928.16\nR001,2016-06-14,21.1,0.0,45892.23\nR001,2016-06-15,19.9,7.0,40986.33\nR001,2016-06-16,16.7,11.9,28318.14\nR001,2016-06-17,17.3,4.2,37366.78\nR001,2016-06-18,18.3,5.2,48645.06\nR001,2016-06-19,19.9,7.0,47035.74\nR001,2016-06-20,19.9,4.0,50002.63\nR001,2016-06-21,21.4,1.5,53778.55\nR001,2016-06-22,22.2,0.0,52654.94\nR001,2016-06-23,21.6,1.0,52665.29\nR001,2016-06-24,20.3,2.6,40040.38\nR001,2016-06-25,21.1,0.5,53025.02\nR001,2016-06-26,22.0,0.0,53964.53\nR001,2016-06-27,22.7,0.2,53753.1\nR001,2016-06-28,20.9,3.9,39306.98\nR001,2016-06-29,20.6,6.2,50683.82\nR001,2016-06-30,19.9,10.1,50089.78\nR001,2016-07-01,18.7,14.0,37725.72\nR001,2016-07-02,19.3,12.2,39812.68\nR001,2016-07-03,19.9,4.9,49054.32\nR001,2016-07-04,19.2,4.2,45331.68\nR001,2016-07-05,19.6,1.8,51366.19\nR001,2016-07-06,19.6,4.1,52895.93\nR001,2016-07-07,20.3,1.8,51716.81\nR001,2016-07-08,17.5,11.4,34868.3\nR001,2016-07-09,16.4,0.2,52813.82\nR001,2016-07-10,15.8,5.1,48486.31\nR001,2016-07-11,15.3,15.0,44408.51\nR001,2016-07-12,17.8,0.2,53694.91\nR001,2016-07-13,20.6,0.0,53659.27\nR001,2016-07-14,22.7,0.0,53451.31\nR001,2016-07-15,21.7,4.0,50860.78\nR001,2016-07-16,22.3,0.4,51972.02\nR001,2016-07-17,24.5,0.0,53488.05\nR001,2016-07-18,24.2,3.6,51367.06\nR001,2016-07-19,19.8,9.2,45353.03\nR001,2016-07-20,20.0,0.0,49925.47\nR001,2016-07-21,18.5,12.9,18515.36\nR001,2016-07-22,18.7,0.0,49304.36\nR001,2016-07-23,17.8,0.0,51811.76\nR001,2016-07-24,16.1,2.3,47994.4\nR001,2016-07-25,17.0,2.3,46474.43\nR001,2016-07-26,18.5,0.2,47299.64\nR001,2016-07-27,19.3,0.7,48846.26\nR001,2016-07-28,19.2,2.5,49593.3\nR001,2016-07-29,19.3,0.7,49992.08\nR001,2016-07-30,19.7,2.8,50042.35\nR001,2016-07-31,19.6,1.3,50095.34\nR001,2016-08-01,20.3,0.0,50492.73\nR001,2016-08-02,21.4,0.0,50200.85\nR001,2016-08-03,20.7,1.7,48542.87\nR001,2016-08-04,20.0,3.4,48260.66\nR001,2016-08-05,20.4,1.3,47299.7\nR001,2016-08-06,21.4,5.7,47705.8\nR001,2016-08-07,20.7,7.9,47963.56\nR001,2016-08-08,19.6,6.2,46783.36\nR001,2016-08-09,21.1,0.3,46800.0\nR001,2016-08-10,22.6,0.7,47708.86\nR001,2016-08-11,21.7,4.9,46800.0\nR001,2016-08-12,20.9,1.6,45388.09\nR001,2016-08-13,21.7,2.1,46800.0\nR001,2016-08-14,19.0,33.6,14606.66\nR001,2016-08-15,18.2,0.0,46800.0\nR001,2016-08-16,19.0,2.8,45998.09\nR001,2016-08-17,20.1,13.6,43710.46\nR001,2016-08-18,19.5,14.7,39787.66\nR001,2016-08-19,21.0,0.2,46800.0\nR001,2016-08-20,23.5,0.1,46800.0\nR001,2016-08-21,24.3,0.1,46800.0\nR001,2016-08-22,24.4,1.1,38405.04\nR001,2016-08-23,22.9,4.8,46800.0\nR001,2016-08-24,22.1,2.5,46378.31\nR001,2016-08-25,21.6,2.5,46037.96\nR001,2016-08-26,19.9,11.6,45938.17\nR001,2016-08-27,20.3,1.7,46694.51\nR001,2016-08-28,20.1,1.3,46591.2\nR001,2016-08-29,20.6,1.6,46396.14\nR001,2016-08-30,20.3,4.8,43864.46\nR001,2016-08-31,18.4,6.8,41002.14\nR001,2016-09-01,18.0,0.0,41928.24\nR001,2016-09-02,17.1,1.3,44978.41\nR001,2016-09-03,17.1,0.9,42650.44\nR001,2016-09-04,16.4,3.0,38593.68\nR001,2016-09-05,17.0,0.9,40400.16\nR001,2016-09-06,16.6,6.5,38439.98\nR001,2016-09-07,16.9,0.0,43953.31\nR001,2016-09-08,17.3,0.0,43367.24\nR001,2016-09-09,17.1,0.0,42930.42\nR001,2016-09-10,17.2,0.1,42808.91\nR001,2016-09-11,16.3,0.0,42984.32\nR001,2016-09-12,16.6,0.0,43200.0\nR001,2016-09-13,17.1,0.1,43148.06\nR001,2016-09-14,16.3,0.6,35580.29\nR001,2016-09-15,14.2,3.0,27687.43\nR001,2016-09-16,13.3,0.0,41428.7\nR001,2016-09-17,13.1,0.2,37644.08\nR001,2016-09-18,13.7,0.0,40540.77\nR001,2016-09-19,14.9,0.0,42166.08\nR001,2016-09-20,17.8,1.9,41526.56\nR001,2016-09-21,13.2,27.5,9355.55\nR001,2016-09-22,10.8,23.0,1784.36\nR001,2016-09-23,10.2,24.6,8686.87\nR001,2016-09-24,8.7,2.6,35545.34\nR001,2016-09-25,7.1,1.2,39525.21\nR001,2016-09-26,8.7,0.1,40819.32\nR001,2016-09-27,8.4,5.4,27141.89\nR001,2016-09-28,7.0,15.5,5062.44\nR001,2016-09-29,6.5,1.6,26524.88\nR001,2016-09-30,9.2,22.5,31190.22\nR001,2016-10-01,11.0,0.0,39742.74\nR001,2016-10-02,14.4,0.0,40313.51\nR001,2016-10-03,15.1,0.0,39602.86\nR001,2016-10-04,14.3,0.0,39532.98\nR001,2016-10-05,15.6,0.0,39025.65\nR001,2016-10-06,16.0,0.0,37507.29\nR001,2016-10-07,14.9,0.3,37601.43\nR001,2016-10-08,16.1,0.0,38021.15\nR001,2016-10-09,13.6,8.4,24752.91\nR001,2016-10-10,12.1,0.6,36778.36\nR001,2016-10-11,11.5,0.0,36000.0\nR001,2016-10-12,10.9,5.0,6582.02\nR001,2016-10-13,13.1,10.1,28627.44\nR001,2016-10-14,8.2,17.6,31636.49\nR001,2016-10-15,7.0,0.5,36070.77\nR001,2016-10-16,6.2,1.4,23496.73\nR001,2016-10-17,4.6,1.6,35309.71\nR001,2016-10-18,5.7,0.0,36000.0\nR001,2016-10-19,7.1,0.0,36000.0\nR001,2016-10-20,7.9,0.2,33907.76\nR001,2016-10-21,8.5,0.0,36000.0\nR001,2016-10-22,9.8,0.0,35384.51\nR001,2016-10-23,9.3,0.4,23745.28\nR001,2016-10-24,9.2,5.0,35419.94\nR001,2016-10-25,7.2,8.3,19560.52\nR001,2016-10-26,4.8,1.7,31651.39\nR001,2016-10-27,5.4,0.0,32051.55\nR001,2016-10-28,6.0,2.3,17320.52\nR001,2016-10-29,5.9,0.4,36000.0\nR001,2016-10-30,2.5,23.2,574.76\nR001,2016-10-31,-0.7,6.9,28744.64\nR001,2016-11-01,0.6,3.5,30336.57\nR001,2016-11-02,-1.1,1.9,35185.01\nR001,2016-11-03,3.8,0.0,35827.22\nR001,2016-11-04,5.5,10.6,24947.32\nR001,2016-11-05,4.9,1.5,27187.71\nR001,2016-11-06,4.2,0.0,35465.27\nR001,2016-11-07,8.4,0.2,30078.03\nR001,2016-11-08,12.8,0.1,35163.99\nR001,2016-11-09,13.7,0.0,32066.17\nR001,2016-11-10,13.8,1.1,25419.01\nR001,2016-11-11,9.9,9.3,27442.86\nR001,2016-11-12,7.5,1.8,33823.25\nR001,2016-11-13,8.0,1.7,34449.76\nR001,2016-11-14,6.1,18.0,0.0\nR001,2016-11-15,4.1,2.2,13680.06\nR001,2016-11-16,2.8,5.3,3793.8\nR001,2016-11-17,1.5,10.0,25987.24\nR001,2016-11-18,-0.5,0.0,31073.33\nR001,2016-11-19,1.9,0.0,32960.23\nR001,2016-11-20,4.5,0.0,32957.41\nR001,2016-11-21,6.1,0.0,32588.77\nR001,2016-11-22,5.6,0.0,32588.37\nR001,2016-11-23,5.4,0.0,32220.43\nR001,2016-11-24,3.7,0.0,31852.39\nR001,2016-11-25,3.5,0.0,31855.35\nR001,2016-11-26,1.9,0.0,30096.76\nR001,2016-11-27,3.2,0.0,31008.58\nR001,2016-11-28,5.6,0.0,28257.8\nR001,2016-11-29,6.1,8.7,18173.8\nR001,2016-11-30,3.2,29.2,0.0\nR001,2016-12-01,-0.5,8.2,7137.49\nR001,2016-12-02,-1.8,9.5,0.0\nR001,2016-12-03,-2.1,30.5,0.0\nR001,2016-12-04,-3.4,23.1,755.91\nR001,2016-12-05,-6.8,9.0,952.35\nR001,2016-12-06,-8.8,0.0,30394.09\nR001,2016-12-07,-5.2,13.0,0.0\nR001,2016-12-08,-10.3,0.0,27969.79\nR001,2016-12-09,-2.9,8.1,7326.5\nR001,2016-12-10,0.7,20.8,0.0\nR001,2016-12-11,1.4,30.2,0.0\nR001,2016-12-12,4.1,9.0,13971.46\nR001,2016-12-13,1.7,34.4,0.0\nR001,2016-12-14,-5.3,0.1,26998.06\nR001,2016-12-15,-5.2,1.6,28611.03\nR001,2016-12-16,-5.3,7.0,16008.18\nR001,2016-12-17,-5.6,5.8,22636.71\nR001,2016-12-18,-7.3,0.0,29222.84\nR001,2016-12-19,-6.2,4.7,26513.37\nR001,2016-12-20,-3.0,20.4,0.0\nR001,2016-12-21,-1.8,2.7,25791.33\nR001,2016-12-22,-3.1,0.0,31129.41\nR001,2016-12-23,-3.6,0.0,31217.64\nR001,2016-12-24,-1.0,0.0,31095.35\nR001,2016-12-25,-1.0,0.8,29901.66\nR001,2016-12-26,-1.1,3.5,23985.28\nR001,2016-12-27,-0.9,0.9,27389.61\nR001,2016-12-28,-0.3,7.9,1228.3\nR001,2016-12-29,1.5,3.9,1170.87\nR001,2016-12-30,3.1,0.7,27922.6\nR001,2016-12-31,4.1,0.0,30144.83\nR001,2017-01-01,1.1,5.0,9163.41\nR001,2017-01-02,-0.6,3.7,26824.84\nR001,2017-01-03,-2.3,0.0,31741.01\nR001,2017-01-04,-2.4,14.4,0.0\nR001,2017-01-05,-0.2,1.9,29300.59\nR001,2017-01-06,4.7,2.9,20389.33\nR001,2017-01-07,5.4,0.0,31981.29\nR001,2017-01-08,2.0,3.1,5630.56\nR001,2017-01-09,2.4,1.1,21896.36\nR001,2017-01-10,0.8,3.0,30666.0\nR001,2017-01-11,-0.1,1.1,32240.22\nR001,2017-01-12,1.4,0.1,29192.19\nR001,2017-01-13,-2.1,5.4,25035.7\nR001,2017-01-14,-2.1,0.0,32400.0\nR001,2017-01-15,2.4,0.0,30091.42\nR001,2017-01-16,1.3,0.4,24437.44\nR001,2017-01-17,2.2,6.3,10988.49\nR001,2017-01-18,2.7,0.0,25059.17\nR001,2017-01-19,1.3,7.9,0.0\nR001,2017-01-20,-1.0,8.6,32279.6\nR001,2017-01-21,-2.2,1.3,28457.73\nR001,2017-01-22,-2.7,6.9,0.0\nR001,2017-01-23,-2.7,0.0,32260.48\nR001,2017-01-24,-0.3,0.0,32400.0\nR001,2017-01-25,-0.5,4.9,26335.89\nR001,2017-01-26,-4.0,10.6,5831.8\nR001,2017-01-27,-8.0,0.0,32400.0\nR001,2017-01-28,-9.0,0.0,32528.37\nR001,2017-01-29,-9.2,6.9,5836.14\nR001,2017-01-30,-12.2,0.5,32400.0\nR001,2017-01-31,-11.4,0.0,32753.16\nR001,2017-02-01,-7.6,0.0,32526.59\nR001,2017-02-02,-5.9,0.7,31877.16\nR001,2017-02-03,-2.6,0.0,32126.66\nR001,2017-02-04,0.6,4.7,0.0\nR001,2017-02-05,3.2,12.7,23044.26\nR001,2017-02-06,4.0,11.4,7441.88\nR001,2017-02-07,5.1,1.9,24784.26\nR001,2017-02-08,4.1,0.0,29409.73\nR001,2017-02-09,3.4,0.8,33537.78\nR001,2017-02-10,1.2,7.5,1627.61\nR001,2017-02-11,-1.1,5.4,27176.02\nR001,2017-02-12,-2.4,6.6,20638.65\nR001,2017-02-13,-3.5,1.2,30731.21\nR001,2017-02-14,-4.5,5.0,33148.75\nR001,2017-02-15,-5.1,20.6,9064.6\nR001,2017-02-16,-7.5,0.0,33909.95\nR001,2017-02-17,-8.5,0.0,37362.07\nR001,2017-02-18,-2.9,0.0,27219.79\nR001,2017-02-19,1.0,0.0,37979.71\nR001,2017-02-20,2.1,0.0,37913.27\nR001,2017-02-21,1.6,2.4,7845.92\nR001,2017-02-22,1.6,16.0,24674.6\nR001,2017-02-23,1.4,9.9,17828.3\nR001,2017-02-24,5.2,9.9,12046.38\nR001,2017-02-25,6.8,0.8,39031.31\nR001,2017-02-26,1.7,19.5,36604.04\nR001,2017-02-27,2.3,0.0,30062.65\nR001,2017-02-28,5.1,0.0,39478.88\nR001,2017-03-01,7.7,0.0,39581.15\nR001,2017-03-02,8.6,0.0,26223.14\nR001,2017-03-03,8.7,0.0,39600.0\nR001,2017-03-04,4.0,5.3,2894.57\nR001,2017-03-05,4.1,0.0,37141.99\nR001,2017-03-06,6.5,0.0,39600.0\nR001,2017-03-07,6.5,0.0,39600.0\nR001,2017-03-08,6.4,0.0,39600.0\nR001,2017-03-09,6.3,0.0,39600.0\nR001,2017-03-10,7.5,0.0,39600.0\nR001,2017-03-11,7.9,0.0,39600.0\nR001,2017-03-12,6.8,3.2,33944.13\nR001,2017-03-13,6.6,0.5,28230.87\nR001,2017-03-14,5.0,6.5,763.85\nR001,2017-03-15,2.6,7.8,1018.92\nR001,2017-03-16,1.9,0.9,37802.42\nR001,2017-03-17,3.2,1.2,36531.03\nR001,2017-03-18,2.4,7.1,6473.93\nR001,2017-03-19,3.5,0.0,19024.09\nR001,2017-03-20,2.0,8.0,21088.67\nR001,2017-03-21,-0.5,3.4,39715.17\nR001,2017-03-22,0.5,1.2,16768.19\nR001,2017-03-23,3.4,0.0,33586.76\nR001,2017-03-24,4.0,0.0,39279.13\nR001,2017-03-25,4.2,3.3,38821.67\nR001,2017-03-26,3.5,3.4,4080.16\nR001,2017-03-27,2.5,13.3,352.3\nR001,2017-03-28,0.9,0.3,42806.37\nR001,2017-03-29,3.5,0.4,38189.8\nR001,2017-03-30,6.2,0.0,44690.18\nR001,2017-03-31,5.0,0.0,42364.51\nR001,2017-04-01,5.0,0.0,43559.02\nR001,2017-04-02,4.9,0.0,42679.13\nR001,2017-04-03,6.0,0.0,45704.28\nR001,2017-04-04,9.0,0.0,45882.34\nR001,2017-04-05,10.7,0.0,44701.09\nR001,2017-04-06,11.9,0.0,42093.37\nR001,2017-04-07,9.5,3.6,27217.41\nR001,2017-04-08,8.3,5.2,37843.96\nR001,2017-04-09,3.1,12.5,22185.87\nR001,2017-04-10,3.8,1.6,40842.18\nR001,2017-04-11,4.2,3.8,44829.04\nR001,2017-04-12,8.7,0.0,40300.7\nR001,2017-04-13,12.8,2.7,32080.39\nR001,2017-04-14,8.4,17.4,2756.9\nR001,2017-04-15,6.5,0.8,38547.81\nR001,2017-04-16,8.4,0.0,46501.09\nR001,2017-04-17,12.7,0.4,46800.0\nR001,2017-04-18,14.6,0.9,46800.0\nR001,2017-04-19,5.3,6.9,44912.78\nR001,2017-04-20,9.8,0.0,46969.29\nR001,2017-04-21,8.0,23.8,8331.25\nR001,2017-04-22,1.9,14.5,33353.54\nR001,2017-04-23,9.4,22.6,26893.61\nR001,2017-04-24,1.6,15.1,14196.17\nR001,2017-04-25,0.9,0.4,46800.0\nR001,2017-04-26,5.7,0.0,47698.27\nR001,2017-04-27,11.8,0.0,47989.93\nR001,2017-04-28,13.4,0.0,48283.54\nR001,2017-04-29,14.0,0.8,48245.24\nR001,2017-04-30,13.9,0.5,47537.99\nR001,2017-05-01,15.0,0.4,48505.99\nR001,2017-05-02,16.0,0.9,48806.71\nR001,2017-05-03,15.3,0.1,25185.87\nR001,2017-05-04,13.0,2.4,28246.58\nR001,2017-05-05,13.1,0.0,48707.07\nR001,2017-05-06,13.3,1.5,49354.24\nR001,2017-05-07,11.5,14.8,24792.32\nR001,2017-05-08,11.6,2.4,45889.95\nR001,2017-05-09,11.2,11.0,32999.15\nR001,2017-05-10,11.6,25.9,2609.85\nR001,2017-05-11,10.2,11.3,37675.57\nR001,2017-05-12,10.7,0.0,48161.78\nR001,2017-05-13,13.4,0.0,40436.38\nR001,2017-05-14,13.4,8.4,39813.57\nR001,2017-05-15,11.5,7.8,21334.91\nR001,2017-05-16,11.2,7.7,16477.56\nR001,2017-05-17,11.4,5.4,28986.65\nR001,2017-05-18,12.0,2.5,39046.52\nR001,2017-05-19,11.0,8.4,1503.76\nR001,2017-05-20,10.2,24.3,847.06\nR001,2017-05-21,9.5,13.2,28689.57\nR001,2017-05-22,8.8,11.7,23234.37\nR001,2017-05-23,9.1,8.2,24268.01\nR001,2017-05-24,7.8,16.1,1121.98\nR001,2017-05-25,9.6,0.4,39831.71\nR001,2017-05-26,13.5,0.0,50529.86\nR001,2017-05-27,14.1,2.2,44897.06\nR001,2017-05-28,14.1,5.9,50047.92\nR001,2017-05-29,15.0,2.0,49022.37\nR001,2017-05-30,17.0,2.7,50437.61\nR001,2017-05-31,14.4,9.1,19057.62\nR001,2017-06-01,13.8,6.9,33161.04\nR001,2017-06-02,14.0,4.5,46799.91\nR001,2017-06-03,14.0,1.7,44078.89\nR001,2017-06-04,15.9,0.0,51201.34\nR001,2017-06-05,19.2,0.0,52862.94\nR001,2017-06-06,20.9,2.1,51387.52\nR001,2017-06-07,17.0,29.4,13431.87\nR001,2017-06-08,18.5,2.6,44957.66\nR001,2017-06-09,20.1,0.7,51703.72\nR001,2017-06-10,17.8,6.8,33618.46\nR001,2017-06-11,16.3,13.1,42793.49\nR001,2017-06-12,15.4,0.9,48328.22\nR001,2017-06-13,16.2,0.0,52015.38\nR001,2017-06-14,17.1,0.6,50901.77\nR001,2017-06-15,14.2,4.7,47688.09\nR001,2017-06-16,11.9,4.9,47991.81\nR001,2017-06-17,13.2,0.0,53444.47\nR001,2017-06-18,15.9,1.3,48991.14\nR001,2017-06-19,15.2,9.6,19152.16\nR001,2017-06-20,14.2,29.1,8348.71\nR001,2017-06-21,15.8,16.2,44652.96\nR001,2017-06-22,16.0,6.6,45424.14\nR001,2017-06-23,16.0,4.3,52159.85\nR001,2017-06-24,17.7,0.0,53956.67\nR001,2017-06-25,17.6,0.1,53026.23\nR001,2017-06-26,20.1,0.1,53963.86\nR001,2017-06-27,21.4,2.1,53254.82\nR001,2017-06-28,21.9,1.8,53416.24\nR001,2017-06-29,20.2,1.9,53250.19\nR001,2017-06-30,19.9,2.0,48251.18\nR001,2017-07-01,21.5,0.1,52605.33\nR001,2017-07-02,24.6,0.0,53934.51\nR001,2017-07-03,26.0,0.0,53921.8\nR001,2017-07-04,25.1,3.7,50832.74\nR001,2017-07-05,20.3,8.1,41374.11\nR001,2017-07-06,19.4,2.2,49334.03\nR001,2017-07-07,17.4,13.9,47481.18\nR001,2017-07-08,15.8,1.2,49373.31\nR001,2017-07-09,17.2,0.0,51295.52\nR001,2017-07-10,18.2,0.0,53052.19\nR001,2017-07-11,18.4,0.2,50210.16\nR001,2017-07-12,19.6,0.4,52461.79\nR001,2017-07-13,19.0,7.3,52001.91\nR001,2017-07-14,18.0,6.0,49863.21\nR001,2017-07-15,19.2,5.8,51433.6\nR001,2017-07-16,21.0,1.1,53472.85\nR001,2017-07-17,22.5,1.5,53485.51\nR001,2017-07-18,23.1,0.8,53143.04\nR001,2017-07-19,22.7,1.3,50934.75\nR001,2017-07-20,19.4,18.4,30753.46\nR001,2017-07-21,20.9,1.4,46893.26\nR001,2017-07-22,20.7,5.4,48704.89\nR001,2017-07-23,20.1,9.7,32093.47\nR001,2017-07-24,20.8,0.5,48335.26\nR001,2017-07-25,20.0,6.1,45034.56\nR001,2017-07-26,20.8,0.9,48783.28\nR001,2017-07-27,23.0,0.0,51796.26\nR001,2017-07-28,25.4,0.2,51064.19\nR001,2017-07-29,19.5,28.4,33198.86\nR001,2017-07-30,18.9,0.0,48643.25\nR001,2017-07-31,18.8,0.0,50079.39\nR001,2017-08-01,21.3,0.0,48755.14\nR001,2017-08-02,23.0,0.0,50187.35\nR001,2017-08-03,23.8,0.1,49556.92\nR001,2017-08-04,24.3,0.5,49273.54\nR001,2017-08-05,24.4,0.0,48309.56\nR001,2017-08-06,24.4,0.4,48373.38\nR001,2017-08-07,24.8,2.6,48098.4\nR001,2017-08-08,24.6,1.2,47985.47\nR001,2017-08-09,23.9,0.6,47337.0\nR001,2017-08-10,24.3,0.0,47703.6\nR001,2017-08-11,24.2,0.4,47732.24\nR001,2017-08-12,25.0,1.1,47425.27\nR001,2017-08-13,24.4,3.7,47119.87\nR001,2017-08-14,22.9,2.3,45817.85\nR001,2017-08-15,20.1,13.8,38095.96\nR001,2017-08-16,19.8,1.1,46800.0\nR001,2017-08-17,20.0,10.4,46908.97\nR001,2017-08-18,19.0,7.3,46455.92\nR001,2017-08-19,19.4,3.5,45515.91\nR001,2017-08-20,19.6,3.7,46800.0\nR001,2017-08-21,22.0,0.0,46800.0\nR001,2017-08-22,22.3,3.5,38734.22\nR001,2017-08-23,18.3,12.4,40158.09\nR001,2017-08-24,18.3,0.3,44606.89\nR001,2017-08-25,19.7,0.1,45407.85\nR001,2017-08-26,19.7,0.1,43637.9\nR001,2017-08-27,20.3,0.1,46714.15\nR001,2017-08-28,20.4,1.0,46610.72\nR001,2017-08-29,18.4,10.3,33385.08\nR001,2017-08-30,17.8,13.3,39505.83\nR001,2017-08-31,15.0,33.8,13766.23\nR001,2017-09-01,16.0,1.1,42065.73\nR001,2017-09-02,17.3,0.0,40685.37\nR001,2017-09-03,18.3,0.0,45336.76\nR001,2017-09-04,20.9,0.0,45065.16\nR001,2017-09-05,20.6,0.0,44474.29\nR001,2017-09-06,18.8,13.7,40373.72\nR001,2017-09-07,15.0,23.0,27241.17\nR001,2017-09-08,15.1,0.0,43668.98\nR001,2017-09-09,18.0,0.0,43721.79\nR001,2017-09-10,19.2,0.0,43456.43\nR001,2017-09-11,18.9,0.0,43200.0\nR001,2017-09-12,20.6,0.0,43246.49\nR001,2017-09-13,21.0,0.0,43163.55\nR001,2017-09-14,20.3,0.0,43096.21\nR001,2017-09-15,20.6,0.0,43028.8\nR001,2017-09-16,20.5,0.0,42961.28\nR001,2017-09-17,20.3,0.0,42893.66\nR001,2017-09-18,20.2,0.0,42825.95\nR001,2017-09-19,19.6,0.0,42758.12\nR001,2017-09-20,20.5,0.0,42690.17\nR001,2017-09-21,20.9,0.0,42622.1\nR001,2017-09-22,20.6,0.6,42262.27\nR001,2017-09-23,18.0,6.1,31100.1\nR001,2017-09-24,16.6,3.3,31407.8\nR001,2017-09-25,16.2,0.5,38579.2\nR001,2017-09-26,16.6,0.0,40811.18\nR001,2017-09-27,14.9,0.3,37647.72\nR001,2017-09-28,11.8,1.0,39925.51\nR001,2017-09-29,12.1,0.3,37672.58\nR001,2017-09-30,12.8,0.7,37568.8\nR001,2017-10-01,12.3,0.7,34254.56\nR001,2017-10-02,11.2,0.1,35599.76\nR001,2017-10-03,8.5,6.4,31406.36\nR001,2017-10-04,8.1,0.0,39003.93\nR001,2017-10-05,9.9,0.0,39007.41\nR001,2017-10-06,11.8,0.0,38780.57\nR001,2017-10-07,13.5,0.0,36936.19\nR001,2017-10-08,15.3,0.0,38648.86\nR001,2017-10-09,13.4,2.0,35864.15\nR001,2017-10-10,11.0,7.2,33146.71\nR001,2017-10-11,9.7,1.4,36497.64\nR001,2017-10-12,8.8,0.0,36551.7\nR001,2017-10-13,8.8,0.0,36279.05\nR001,2017-10-14,7.6,7.1,32400.0\nR001,2017-10-15,7.1,10.8,28828.56\nR001,2017-10-16,6.0,22.0,30438.68\nR001,2017-10-17,5.8,0.5,27688.51\nR001,2017-10-18,9.1,0.0,30632.74\nR001,2017-10-19,9.6,0.0,36000.0\nR001,2017-10-20,10.5,0.0,36000.0\nR001,2017-10-21,10.5,0.0,36000.0\nR001,2017-10-22,9.3,9.7,22170.77\nR001,2017-10-23,8.6,1.0,33373.82\nR001,2017-10-24,11.8,0.0,32400.0\nR001,2017-10-25,14.6,0.0,36000.0\nR001,2017-10-26,11.9,8.1,23073.76\nR001,2017-10-27,8.7,1.2,34252.32\nR001,2017-10-28,10.4,0.0,36000.0\nR001,2017-10-29,9.8,20.6,313.04\nR001,2017-10-30,6.8,51.3,0.0\nR001,2017-10-31,0.9,18.5,0.0\nR001,2017-11-01,-1.0,15.9,17843.29\nR001,2017-11-02,-1.5,0.0,35926.65\nR001,2017-11-03,0.1,0.0,35847.91\nR001,2017-11-04,1.8,0.0,32769.74\nR001,2017-11-05,3.8,0.0,33039.97\nR001,2017-11-06,8.0,1.3,30008.06\nR001,2017-11-07,7.6,2.0,33139.85\nR001,2017-11-08,7.9,0.7,33470.05\nR001,2017-11-09,6.0,0.0,34773.93\nR001,2017-11-10,6.5,1.0,31817.7\nR001,2017-11-11,6.3,1.8,33410.87\nR001,2017-11-12,7.4,3.0,25738.86\nR001,2017-11-13,9.2,1.5,32300.62\nR001,2017-11-14,10.2,0.0,29268.44\nR001,2017-11-15,9.8,0.0,34199.63\nR001,2017-11-16,7.5,0.0,31990.43\nR001,2017-11-17,6.7,0.0,32973.27\nR001,2017-11-18,7.9,0.0,32154.11\nR001,2017-11-19,10.2,0.0,28800.0\nR001,2017-11-20,10.0,1.4,30805.91\nR001,2017-11-21,7.9,17.4,14972.31\nR001,2017-11-22,0.6,3.8,30995.63\nR001,2017-11-23,-3.2,13.6,26900.33\nR001,2017-11-24,-6.6,5.7,14106.24\nR001,2017-11-25,-6.5,0.0,31233.82\nR001,2017-11-26,-3.9,0.0,30976.27\nR001,2017-11-27,-2.8,0.0,31000.61\nR001,2017-11-28,0.2,0.0,29497.21\nR001,2017-11-29,3.6,6.2,2320.25\nR001,2017-11-30,3.2,0.5,30307.91\nR001,2017-12-01,4.5,0.0,29945.29\nR001,2017-12-02,6.9,0.0,29960.29\nR001,2017-12-03,8.3,0.0,28800.0\nR001,2017-12-04,10.3,0.0,28860.09\nR001,2017-12-05,5.6,29.9,0.0\nR001,2017-12-06,0.2,5.8,11090.14\nR001,2017-12-07,-2.0,17.4,0.0\nR001,2017-12-08,0.9,16.3,0.0\nR001,2017-12-09,2.3,0.4,30413.26\nR001,2017-12-10,3.0,0.0,30417.86\nR001,2017-12-11,1.2,8.8,30033.53\nR001,2017-12-12,2.6,0.0,28800.0\nR001,2017-12-13,1.3,0.0,30422.59\nR001,2017-12-14,2.9,0.0,27873.81\nR001,2017-12-15,4.8,0.0,28421.91\nR001,2017-12-16,4.8,0.0,29050.61\nR001,2017-12-17,7.5,0.0,28800.0\nR001,2017-12-18,8.2,0.4,19492.86\nR001,2017-12-19,5.8,6.4,11814.51\nR001,2017-12-20,3.4,10.2,0.0\nR001,2017-12-21,2.8,16.2,14772.61\nR001,2017-12-22,1.8,20.6,20598.22\nR001,2017-12-23,1.6,33.2,0.0\nR001,2017-12-24,-0.2,4.3,27104.84\nR001,2017-12-25,1.1,36.8,0.0\nR001,2017-12-26,3.6,8.5,10510.57\nR001,2017-12-27,5.9,0.0,28800.0\nR001,2017-12-28,7.6,0.0,31401.32\nR001,2017-12-29,6.7,0.0,30985.4\nR001,2017-12-30,6.4,0.8,29005.77\nR001,2017-12-31,5.4,4.7,28800.0\nR001,2018-01-01,3.4,11.9,0.0\nR001,2018-01-02,1.9,0.0,29638.72\nR001,2018-01-03,2.1,0.0,28896.69\nR001,2018-01-04,2.7,0.0,16896.04\nR001,2018-01-05,3.7,0.1,30002.03\nR001,2018-01-06,3.4,0.0,30913.84\nR001,2018-01-07,1.7,0.0,31956.7\nR001,2018-01-08,3.0,0.2,29123.41\nR001,2018-01-09,1.2,15.9,8159.52\nR001,2018-01-10,-5.6,0.0,32154.9\nR001,2018-01-11,-1.2,0.5,28800.0\nR001,2018-01-12,2.3,4.9,14312.71\nR001,2018-01-13,0.9,11.6,630.58\nR001,2018-01-14,1.4,0.0,28800.0\nR001,2018-01-15,0.1,6.3,14429.53\nR001,2018-01-16,-1.3,0.9,31840.49\nR001,2018-01-17,0.6,1.6,6478.83\nR001,2018-01-18,2.3,0.3,26602.66\nR001,2018-01-19,2.3,14.3,0.0\nR001,2018-01-20,1.9,1.3,30410.04\nR001,2018-01-21,3.5,1.6,10165.34\nR001,2018-01-22,3.0,0.0,21515.17\nR001,2018-01-23,3.8,2.1,11076.28\nR001,2018-01-24,-0.2,9.9,14132.27\nR001,2018-01-25,-1.9,0.0,22136.33\nR001,2018-01-26,0.4,0.0,28628.88\nR001,2018-01-27,-1.1,0.5,32400.0\nR001,2018-01-28,-0.7,0.0,32400.0\nR001,2018-01-29,-0.8,6.8,2324.24\nR001,2018-01-30,0.4,8.6,2082.65\nR001,2018-01-31,-0.2,19.1,26378.87\nR001,2018-02-01,-3.6,0.4,32719.43\nR001,2018-02-02,0.5,0.6,32496.62\nR001,2018-02-03,4.1,5.8,3808.62\nR001,2018-02-04,4.9,1.9,2875.81\nR001,2018-02-05,4.2,12.9,3227.74\nR001,2018-02-06,-0.1,0.0,32495.23\nR001,2018-02-07,0.9,0.0,34080.84\nR001,2018-02-08,4.2,0.1,32950.98\nR001,2018-02-09,4.1,6.2,15782.66\nR001,2018-02-10,2.5,0.0,34621.18\nR001,2018-02-11,5.7,0.1,30041.16\nR001,2018-02-12,3.5,21.8,11084.38\nR001,2018-02-13,1.0,7.0,25695.29\nR001,2018-02-14,0.9,0.6,35235.17\nR001,2018-02-15,3.0,0.7,31618.63\nR001,2018-02-16,5.8,0.0,32400.0\nR001,2018-02-17,5.4,0.2,35527.06\nR001,2018-02-18,3.7,0.0,33747.48\nR001,2018-02-19,1.7,0.0,34093.94\nR001,2018-02-20,2.3,0.3,24042.19\nR001,2018-02-21,4.3,0.4,28195.87\nR001,2018-02-22,5.2,0.0,37821.43\nR001,2018-02-23,4.6,6.6,34260.89\nR001,2018-02-24,3.1,3.6,33638.45\nR001,2018-02-25,4.1,0.3,29939.6\nR001,2018-02-26,2.1,4.4,28544.31\nR001,2018-02-27,4.3,1.9,26335.89\nR001,2018-02-28,6.3,0.0,38166.96\nR001,2018-03-01,0.8,20.9,10749.18\nR001,2018-03-02,-2.7,0.1,36000.0\nR001,2018-03-03,4.7,5.0,12033.12\nR001,2018-03-04,5.3,19.6,30083.43\nR001,2018-03-05,2.1,14.8,23822.26\nR001,2018-03-06,1.9,0.0,39600.0\nR001,2018-03-07,8.5,3.7,24325.5\nR001,2018-03-08,8.2,7.8,17634.84\nR001,2018-03-09,7.0,2.6,33124.52\nR001,2018-03-10,1.8,9.3,27788.57\nR001,2018-03-11,2.4,7.8,30533.28\nR001,2018-03-12,2.8,0.4,39600.0\nR001,2018-03-13,6.9,0.0,39600.0\nR001,2018-03-14,7.1,8.0,22614.85\nR001,2018-03-15,5.6,5.9,33844.36\nR001,2018-03-16,5.4,5.6,37520.74\nR001,2018-03-17,6.5,0.0,36685.79\nR001,2018-03-18,12.1,0.2,28363.54\nR001,2018-03-19,12.2,1.2,36859.15\nR001,2018-03-20,11.1,4.4,36000.0\nR001,2018-03-21,6.8,6.2,39600.0\nR001,2018-03-22,4.4,17.5,1162.7\nR001,2018-03-23,7.6,5.6,10800.0\nR001,2018-03-24,8.5,22.1,0.0\nR001,2018-03-25,4.3,6.2,26395.58\nR001,2018-03-26,7.0,1.7,36846.82\nR001,2018-03-27,8.3,1.5,32074.6\nR001,2018-03-28,7.8,7.8,34485.5\nR001,2018-03-29,9.1,10.9,9499.17\nR001,2018-03-30,7.1,12.5,1740.96\nR001,2018-03-31,6.5,0.9,40659.53\nR001,2018-04-01,9.8,0.0,44178.23\nR001,2018-04-02,11.4,10.8,37798.66\nR001,2018-04-03,4.4,6.0,43908.28\nR001,2018-04-04,4.7,0.0,45678.94\nR001,2018-04-05,7.5,0.0,45937.6\nR001,2018-04-06,10.2,0.0,46009.06\nR001,2018-04-07,12.1,0.0,46080.44\nR001,2018-04-08,12.3,0.0,46151.76\nR001,2018-04-09,14.5,0.0,45863.25\nR001,2018-04-10,15.8,0.0,46294.13\nR001,2018-04-11,14.8,0.0,46365.17\nR001,2018-04-12,13.8,1.1,45772.87\nR001,2018-04-13,11.9,2.0,38830.61\nR001,2018-04-14,9.2,0.0,44614.61\nR001,2018-04-15,7.8,0.0,46648.61\nR001,2018-04-16,8.1,0.0,46719.67\nR001,2018-04-17,9.5,0.0,46790.83\nR001,2018-04-18,11.3,0.0,46800.0\nR001,2018-04-19,8.1,7.8,34572.68\nR001,2018-04-20,5.6,13.9,23123.27\nR001,2018-04-21,7.4,1.7,43200.0\nR001,2018-04-22,7.2,1.9,44177.13\nR001,2018-04-23,7.7,0.2,46800.0\nR001,2018-04-24,8.3,0.0,46800.0\nR001,2018-04-25,11.5,0.0,47085.65\nR001,2018-04-26,14.3,0.0,47374.12\nR001,2018-04-27,16.1,0.3,47664.62\nR001,2018-04-28,15.3,0.4,45600.43\nR001,2018-04-29,14.2,0.0,46800.0\nR001,2018-04-30,15.0,0.0,48548.96\nR001,2018-05-01,16.5,1.3,48512.94\nR001,2018-05-02,16.2,4.2,48477.83\nR001,2018-05-03,17.0,5.5,48780.45\nR001,2018-05-04,17.3,6.3,47735.33\nR001,2018-05-05,16.2,5.1,25596.14\nR001,2018-05-06,15.5,9.3,47668.03\nR001,2018-05-07,14.9,13.3,34740.88\nR001,2018-05-08,12.8,15.3,10542.39\nR001,2018-05-09,12.1,0.7,9288.41\nR001,2018-05-10,12.3,8.8,10442.52\nR001,2018-05-11,11.2,9.3,20843.5\nR001,2018-05-12,10.5,4.0,38466.74\nR001,2018-05-13,9.9,9.3,3231.74\nR001,2018-05-14,12.3,2.4,26076.57\nR001,2018-05-15,13.9,4.7,42863.09\nR001,2018-05-16,15.6,2.8,46775.59\nR001,2018-05-17,16.4,3.7,49176.14\nR001,2018-05-18,17.2,0.3,47533.5\nR001,2018-05-19,18.1,1.1,49204.85\nR001,2018-05-20,17.5,2.4,50269.84\nR001,2018-05-21,16.2,9.8,41511.6\nR001,2018-05-22,16.5,3.9,48085.25\nR001,2018-05-23,17.5,11.1,47458.22\nR001,2018-05-24,16.3,16.1,42335.94\nR001,2018-05-25,17.3,11.0,38436.38\nR001,2018-05-26,16.9,6.5,36056.65\nR001,2018-05-27,19.0,2.3,45783.78\nR001,2018-05-28,20.0,0.8,49637.74\nR001,2018-05-29,17.0,0.0,17822.03\nR001,2018-05-30,15.1,10.4,7535.16\nR001,2018-05-31,15.9,13.1,37730.4\nR001,2018-06-01,15.6,4.9,45395.24\nR001,2018-06-02,15.6,1.0,44544.59\nR001,2018-06-03,15.9,1.2,47163.89\nR001,2018-06-04,16.2,2.0,50273.32\nR001,2018-06-05,16.2,5.5,48410.35\nR001,2018-06-06,17.1,0.6,52483.41\nR001,2018-06-07,17.8,0.0,51371.94\nR001,2018-06-08,17.0,2.7,51355.54\nR001,2018-06-09,16.7,0.0,50420.48\nR001,2018-06-10,16.7,0.4,52790.21\nR001,2018-06-11,16.2,2.0,51677.75\nR001,2018-06-12,16.9,0.8,53499.17\nR001,2018-06-13,18.3,0.1,53121.28\nR001,2018-06-14,20.9,0.0,53793.15\nR001,2018-06-15,21.0,5.4,52732.84\nR001,2018-06-16,17.7,12.9,7966.67\nR001,2018-06-17,16.3,13.7,50699.54\nR001,2018-06-18,16.4,8.1,43527.13\nR001,2018-06-19,17.0,8.2,37829.42\nR001,2018-06-20,18.8,0.2,53569.73\nR001,2018-06-21,20.1,0.0,53928.94\nR001,2018-06-22,20.3,4.8,49887.95\nR001,2018-06-23,17.9,17.0,20774.31\nR001,2018-06-24,19.0,4.2,49875.71\nR001,2018-06-25,20.2,1.8,53964.27\nR001,2018-06-26,21.0,8.3,53755.15\nR001,2018-06-27,21.5,5.0,53968.9\nR001,2018-06-28,23.7,0.2,52931.3\nR001,2018-06-29,24.2,2.2,53964.27\nR001,2018-06-30,22.4,3.0,50816.99\nR001,2018-07-01,22.4,3.2,50860.93\nR001,2018-07-02,22.0,1.1,51629.19\nR001,2018-07-03,21.8,2.8,47919.64\nR001,2018-07-04,20.2,6.4,1238.6\nR001,2018-07-05,19.9,8.4,12032.23\nR001,2018-07-06,17.8,16.9,3155.03\nR001,2018-07-07,19.2,0.8,47428.49\nR001,2018-07-08,20.0,1.0,53762.72\nR001,2018-07-09,21.4,0.7,53799.44\nR001,2018-07-10,24.3,0.0,53770.95\nR001,2018-07-11,23.9,2.1,53059.4\nR001,2018-07-12,23.2,1.1,53428.42\nR001,2018-07-13,22.3,2.7,47261.06\nR001,2018-07-14,20.4,1.9,47939.72\nR001,2018-07-15,18.7,11.6,34615.97\nR001,2018-07-16,17.0,3.0,49558.82\nR001,2018-07-17,17.2,1.8,45453.98\nR001,2018-07-18,19.3,0.0,53469.88\nR001,2018-07-19,18.8,2.6,50548.94\nR001,2018-07-20,18.4,2.3,49306.86\nR001,2018-07-21,19.0,0.0,49116.36\nR001,2018-07-22,20.0,0.1,50693.26\nR001,2018-07-23,18.9,7.3,48849.71\nR001,2018-07-24,20.2,2.3,50076.13\nR001,2018-07-25,24.5,0.2,51881.92\nR001,2018-07-26,24.0,3.2,49814.92\nR001,2018-07-27,23.5,0.8,51460.24\nR001,2018-07-28,22.7,5.6,50727.92\nR001,2018-07-29,22.2,7.3,47479.61\nR001,2018-07-30,22.5,2.5,48283.83\nR001,2018-07-31,23.7,0.2,50068.1\nR001,2018-08-01,22.1,8.0,48480.63\nR001,2018-08-02,20.5,5.1,45792.89\nR001,2018-08-03,19.1,10.6,43589.49\nR001,2018-08-04,19.5,5.1,39969.25\nR001,2018-08-05,19.7,2.4,46295.55\nR001,2018-08-06,20.6,0.6,47913.86\nR001,2018-08-07,21.9,0.1,48292.77\nR001,2018-08-08,20.6,0.8,46718.52\nR001,2018-08-09,19.2,6.4,26264.75\nR001,2018-08-10,17.3,7.9,33205.08\nR001,2018-08-11,17.3,0.2,45905.3\nR001,2018-08-12,16.9,2.1,45413.87\nR001,2018-08-13,17.4,0.2,46800.0\nR001,2018-08-14,18.7,0.0,47476.47\nR001,2018-08-15,20.1,0.0,47172.18\nR001,2018-08-16,21.0,0.0,47202.46\nR001,2018-08-17,21.4,1.0,46900.63\nR001,2018-08-18,20.2,0.9,46800.0\nR001,2018-08-19,19.9,0.0,46800.0\nR001,2018-08-20,19.9,0.0,46800.0\nR001,2018-08-21,20.1,1.7,46800.0\nR001,2018-08-22,19.9,0.4,46800.0\nR001,2018-08-23,20.5,0.0,46800.0\nR001,2018-08-24,20.8,1.3,46800.0\nR001,2018-08-25,21.0,1.1,46800.0\nR001,2018-08-26,20.5,1.3,46482.6\nR001,2018-08-27,20.7,2.9,46735.84\nR001,2018-08-28,20.2,1.3,46634.04\nR001,2018-08-29,21.4,0.1,46532.0\nR001,2018-08-30,22.1,2.8,46429.83\nR001,2018-08-31,19.7,6.6,31283.33\nR001,2018-09-01,18.4,8.8,38470.68\nR001,2018-09-02,18.0,4.4,44630.1\nR001,2018-09-03,19.1,2.7,44466.57\nR001,2018-09-04,18.9,1.1,44408.6\nR001,2018-09-05,19.5,0.4,44459.39\nR001,2018-09-06,21.3,0.6,44168.53\nR001,2018-09-07,17.2,10.5,26779.83\nR001,2018-09-08,15.6,2.2,24484.93\nR001,2018-09-09,16.7,0.0,40886.04\nR001,2018-09-10,15.4,3.4,16695.27\nR001,2018-09-11,15.4,5.4,24467.58\nR001,2018-09-12,15.9,0.2,41874.53\nR001,2018-09-13,19.7,0.0,43180.16\nR001,2018-09-14,19.9,14.1,39512.8\nR001,2018-09-15,16.8,6.6,38709.98\nR001,2018-09-16,14.9,9.3,16612.53\nR001,2018-09-17,15.7,2.2,39600.0\nR001,2018-09-18,14.6,0.0,40843.21\nR001,2018-09-19,15.0,0.0,40876.99\nR001,2018-09-20,16.0,0.0,42707.17\nR001,2018-09-21,17.1,0.0,42639.27\nR001,2018-09-22,16.9,2.7,33734.15\nR001,2018-09-23,16.4,1.8,37306.82\nR001,2018-09-24,16.9,0.1,39600.0\nR001,2018-09-25,17.4,0.6,41722.64\nR001,2018-09-26,12.2,7.3,24430.69\nR001,2018-09-27,12.6,0.0,37166.16\nR001,2018-09-28,13.1,0.0,29451.85\nR001,2018-09-29,14.6,0.1,37551.01\nR001,2018-09-30,13.4,2.1,38829.03\nR001,2018-10-01,17.0,0.0,33464.94\nR001,2018-10-02,17.6,4.3,38700.21\nR001,2018-10-03,15.4,0.0,39117.2\nR001,2018-10-04,11.6,10.9,20754.12\nR001,2018-10-05,6.9,1.3,38735.72\nR001,2018-10-06,6.8,0.0,39404.29\nR001,2018-10-07,11.9,0.0,39027.63\nR001,2018-10-08,15.0,0.0,38302.3\nR001,2018-10-09,14.7,0.1,30606.58\nR001,2018-10-10,15.0,3.1,32762.52\nR001,2018-10-11,14.6,1.6,36805.94\nR001,2018-10-12,13.6,0.0,36859.58\nR001,2018-10-13,14.6,0.0,36913.33\nR001,2018-10-14,13.7,0.0,36640.63\nR001,2018-10-15,13.9,0.0,36366.87\nR001,2018-10-16,15.6,0.0,36091.92\nR001,2018-10-17,15.4,0.0,36143.84\nR001,2018-10-18,15.0,0.0,36000.0\nR001,2018-10-19,14.5,0.0,36000.0\nR001,2018-10-20,12.9,0.3,36000.0\nR001,2018-10-21,12.6,2.8,33775.87\nR001,2018-10-22,12.1,4.4,2130.21\nR001,2018-10-23,10.8,1.9,24600.32\nR001,2018-10-24,11.0,17.1,199.79\nR001,2018-10-25,8.8,50.0,0.0\nR001,2018-10-26,4.7,3.9,29937.57\nR001,2018-10-27,4.1,0.0,36000.0\nR001,2018-10-28,7.9,0.0,36000.0\nR001,2018-10-29,10.7,0.0,36000.0\nR001,2018-10-30,11.6,0.0,36000.0\nR001,2018-10-31,11.6,0.0,36000.0\nR001,2018-11-01,11.4,0.0,36000.0\nR001,2018-11-02,9.9,0.0,35951.96\nR001,2018-11-03,9.3,0.0,35872.6\nR001,2018-11-04,9.8,0.0,35792.79\nR001,2018-11-05,8.9,0.0,35710.22\nR001,2018-11-06,8.4,0.2,35554.3\nR001,2018-11-07,8.1,0.4,32793.18\nR001,2018-11-08,7.4,0.0,35246.12\nR001,2018-11-09,7.0,0.0,35094.21\nR001,2018-11-10,7.3,0.0,34944.0\nR001,2018-11-11,6.6,0.0,34795.68\nR001,2018-11-12,6.2,3.7,27050.61\nR001,2018-11-13,3.7,7.8,18376.64\nR001,2018-11-14,3.4,3.3,6830.44\nR001,2018-11-15,2.7,8.8,1818.44\nR001,2018-11-16,2.2,0.2,22820.29\nR001,2018-11-17,3.4,15.0,1790.66\nR001,2018-11-18,2.2,0.1,31869.01\nR001,2018-11-19,2.1,0.0,29656.54\nR001,2018-11-20,6.8,4.0,0.0\nR001,2018-11-21,3.3,0.0,30530.29\nR001,2018-11-22,4.6,0.4,19397.41\nR001,2018-11-23,5.2,2.3,30141.73\nR001,2018-11-24,3.4,1.2,30914.08\nR001,2018-11-25,3.0,0.0,30802.17\nR001,2018-11-26,3.2,0.0,29742.12\nR001,2018-11-27,6.0,27.2,17432.97\nR001,2018-11-28,8.1,34.3,13310.1\nR001,2018-11-29,6.2,36.9,0.0\nR001,2018-11-30,8.3,2.9,18514.09\nR001,2018-12-01,7.0,15.5,0.0\nR001,2018-12-02,5.8,5.2,23853.8\nR001,2018-12-03,4.4,0.0,29590.64\nR001,2018-12-04,5.0,0.0,25948.46\nR001,2018-12-05,7.9,2.3,16674.38\nR001,2018-12-06,6.1,2.0,22022.51\nR001,2018-12-07,4.0,12.4,127.14\nR001,2018-12-08,0.1,5.8,28290.34\nR001,2018-12-09,-0.1,0.0,30414.92\nR001,2018-12-10,4.7,0.2,26382.93\nR001,2018-12-11,5.2,11.8,86.13\nR001,2018-12-12,2.5,9.5,1105.42\nR001,2018-12-13,0.5,4.8,19463.64\nR001,2018-12-14,-1.0,2.6,29255.54\nR001,2018-12-15,1.0,0.6,26378.03\nR001,2018-12-16,3.8,0.5,29244.07\nR001,2018-12-17,5.2,0.0,26026.08\nR001,2018-12-18,4.8,0.0,30007.34\nR001,2018-12-19,4.2,0.3,19393.11\nR001,2018-12-20,6.0,0.2,26768.93\nR001,2018-12-21,4.2,3.1,23342.41\nR001,2018-12-22,3.0,5.7,11880.32\nR001,2018-12-23,3.2,9.7,8000.41\nR001,2018-12-24,0.2,9.3,27247.2\nR001,2018-12-25,0.9,17.9,0.0\nR001,2018-12-26,-1.2,26.1,0.0\nR001,2018-12-27,-2.7,13.8,19082.12\nR001,2018-12-28,-4.3,0.1,29453.98\nR001,2018-12-29,-2.3,0.0,27108.47\nR001,2018-12-30,-1.2,1.3,16715.44\nR001,2018-12-31,-2.8,0.1,26065.59\nR001,2019-01-01,-3.0,0.4,31592.16\nR001,2019-01-02,-4.3,4.0,18000.0\nR001,2019-01-03,2.7,0.5,4752.19\nR001,2019-01-04,2.6,19.6,6003.79\nR001,2019-01-05,-0.9,4.1,24618.49\nR001,2019-01-06,1.7,10.4,0.0\nR001,2019-01-07,-0.3,14.9,17229.2\nR001,2019-01-08,-3.3,0.5,20912.39\nR001,2019-01-09,-2.5,14.2,4461.76\nR001,2019-01-10,-2.3,0.0,32145.04\nR001,2019-01-11,2.4,0.0,29105.89\nR001,2019-01-12,1.9,14.1,0.0\nR001,2019-01-13,1.1,18.4,0.0\nR001,2019-01-14,1.5,4.0,13539.52\nR001,2019-01-15,3.4,5.1,20357.11\nR001,2019-01-16,2.4,36.6,0.0\nR001,2019-01-17,-2.4,4.3,29551.21\nR001,2019-01-18,-4.0,0.0,32400.0\nR001,2019-01-19,-1.3,0.0,24609.87\nR001,2019-01-20,0.2,1.7,22233.48\nR001,2019-01-21,0.0,0.0,32400.0\nR001,2019-01-22,-0.3,8.7,22204.27\nR001,2019-01-23,0.6,0.0,30651.24\nR001,2019-01-24,4.0,0.0,29341.51\nR001,2019-01-25,4.6,0.0,2015.19\nR001,2019-01-26,4.0,0.0,2804.35\nR001,2019-01-27,4.0,11.2,4491.17\nR001,2019-01-28,3.2,3.2,25904.36\nR001,2019-01-29,2.2,0.0,32400.0\nR001,2019-01-30,3.8,0.0,31730.12\nR001,2019-01-31,4.9,0.0,28861.09\nR001,2019-02-01,3.5,3.1,14546.34\nR001,2019-02-02,2.4,0.5,32029.18\nR001,2019-02-03,2.3,0.0,32987.29\nR001,2019-02-04,3.9,0.0,32624.1\nR001,2019-02-05,4.4,0.0,33591.65\nR001,2019-02-06,6.0,0.0,32700.96\nR001,2019-02-07,6.3,1.5,0.0\nR001,2019-02-08,4.1,0.7,24824.44\nR001,2019-02-09,3.4,3.6,4360.55\nR001,2019-02-10,3.0,8.4,1945.24\nR001,2019-02-11,2.3,0.1,33654.9\nR001,2019-02-12,2.5,0.0,34813.2\nR001,2019-02-13,5.1,0.0,31818.4\nR001,2019-02-14,3.1,0.0,25955.26\nR001,2019-02-15,1.6,0.0,12170.84\nR001,2019-02-16,1.0,2.7,24168.71\nR001,2019-02-17,0.7,1.9,33763.29\nR001,2019-02-18,1.3,6.3,11432.57\nR001,2019-02-19,-1.0,0.0,37902.07\nR001,2019-02-20,-0.5,0.0,38077.1\nR001,2019-02-21,2.0,0.0,36995.31\nR001,2019-02-22,-0.1,2.7,35044.33\nR001,2019-02-23,2.7,1.4,21029.07\nR001,2019-02-24,2.1,0.4,38780.69\nR001,2019-02-25,0.3,0.6,27373.66\nR001,2019-02-26,-0.9,7.9,3525.75\nR001,2019-02-27,-1.4,14.8,119.78\nR001,2019-02-28,-2.7,0.9,39427.92\nR001,2019-03-01,-2.0,19.5,3540.66\nR001,2019-03-02,-1.7,7.0,8143.77\nR001,2019-03-03,-3.0,2.7,39600.0\nR001,2019-03-04,-4.1,0.0,39600.0\nR001,2019-03-05,3.0,0.0,36678.7\nR001,2019-03-06,4.4,8.5,14111.96\nR001,2019-03-07,2.3,11.7,28091.62\nR001,2019-03-08,1.3,0.0,37136.86\nR001,2019-03-09,5.9,0.0,35534.61\nR001,2019-03-10,7.1,7.4,37270.57\nR001,2019-03-11,5.0,2.8,23901.59\nR001,2019-03-12,6.7,0.9,30092.64\nR001,2019-03-13,5.5,31.1,0.0\nR001,2019-03-14,5.6,2.5,31676.47\nR001,2019-03-15,6.5,2.1,29020.02\nR001,2019-03-16,4.7,15.1,5384.83\nR001,2019-03-17,1.7,12.6,9258.89\nR001,2019-03-18,0.2,0.5,39307.92\nR001,2019-03-19,2.4,0.0,40482.17\nR001,2019-03-20,7.0,0.0,41027.97\nR001,2019-03-21,5.2,0.2,39144.05\nR001,2019-03-22,2.8,2.6,36352.08\nR001,2019-03-23,-1.0,19.1,474.61\nR001,2019-03-24,-2.9,1.2,40349.25\nR001,2019-03-25,-0.2,0.0,40540.1\nR001,2019-03-26,3.2,0.0,43082.97\nR001,2019-03-27,4.6,0.1,42521.05\nR001,2019-03-28,0.7,9.2,35089.76\nR001,2019-03-29,-1.1,4.2,38635.22\nR001,2019-03-30,-0.5,0.0,44585.03\nR001,2019-03-31,6.6,0.0,44727.3\nR001,2019-04-01,5.8,3.2,33309.17\nR001,2019-04-02,3.0,11.5,6967.71\nR001,2019-04-03,2.8,5.3,35507.85\nR001,2019-04-04,3.7,2.7,40876.66\nR001,2019-04-05,6.0,0.0,44715.84\nR001,2019-04-06,9.1,0.0,45993.69\nR001,2019-04-07,10.9,0.0,46064.88\nR001,2019-04-08,11.8,0.0,45900.63\nR001,2019-04-09,12.9,1.9,46206.93\nR001,2019-04-10,10.7,5.6,36029.49\nR001,2019-04-11,8.9,4.3,41692.14\nR001,2019-04-12,8.2,9.5,29108.4\nR001,2019-04-13,9.3,4.1,34596.22\nR001,2019-04-14,11.8,0.0,34785.85\nR001,2019-04-15,10.1,10.6,23730.56\nR001,2019-04-16,8.3,7.8,30245.74\nR001,2019-04-17,8.7,4.1,36261.02\nR001,2019-04-18,6.1,10.2,0.0\nR001,2019-04-19,5.0,11.6,21498.19\nR001,2019-04-20,4.6,5.7,23157.98\nR001,2019-04-21,4.6,3.9,36116.63\nR001,2019-04-22,5.6,2.4,25938.23\nR001,2019-04-23,6.8,3.9,32280.14\nR001,2019-04-24,8.1,0.1,45828.97\nR001,2019-04-25,9.1,0.3,47094.45\nR001,2019-04-26,10.7,0.7,47383.18\nR001,2019-04-27,11.6,0.6,47674.11\nR001,2019-04-28,12.6,0.1,47634.28\nR001,2019-04-29,12.3,0.3,47261.74\nR001,2019-04-30,14.3,0.7,46800.0\nR001,2019-05-01,16.1,0.0,47519.5\nR001,2019-05-02,14.3,6.9,36010.52\nR001,2019-05-03,9.1,15.0,22734.95\nR001,2019-05-04,9.7,1.9,47072.03\nR001,2019-05-05,11.9,0.0,48095.24\nR001,2019-05-06,13.4,1.4,41252.27\nR001,2019-05-07,16.4,0.9,49342.52\nR001,2019-05-08,13.6,14.3,44118.42\nR001,2019-05-09,11.8,11.7,31564.84\nR001,2019-05-10,12.2,3.6,43134.6\nR001,2019-05-11,12.9,0.9,45862.65\nR001,2019-05-12,11.6,13.0,34734.97\nR001,2019-05-13,12.4,2.3,45427.03\nR001,2019-05-14,13.8,0.0,50371.78\nR001,2019-05-15,16.5,0.0,50400.0\nR001,2019-05-16,19.7,0.0,50460.32\nR001,2019-05-17,20.8,0.2,50431.38\nR001,2019-05-18,21.5,0.7,50400.0\nR001,2019-05-19,17.7,12.7,44049.92\nR001,2019-05-20,14.9,8.2,17328.2\nR001,2019-05-21,16.1,2.3,47883.83\nR001,2019-05-22,14.2,7.1,34414.14\nR001,2019-05-23,13.9,1.7,50620.67\nR001,2019-05-24,16.0,0.0,48926.88\nR001,2019-05-25,15.1,13.5,34403.33\nR001,2019-05-26,13.6,1.8,44613.35\nR001,2019-05-27,14.3,2.2,41731.09\nR001,2019-05-28,15.8,0.2,49229.73\nR001,2019-05-29,18.7,0.0,51910.21\nR001,2019-05-30,20.2,0.0,51889.04\nR001,2019-05-31,20.3,0.4,51506.82\nR001,2019-06-01,20.3,2.0,51848.33\nR001,2019-06-02,21.7,1.9,50400.0\nR001,2019-06-03,21.1,5.2,43744.73\nR001,2019-06-04,20.2,2.9,33651.69\nR001,2019-06-05,19.8,7.2,45325.53\nR001,2019-06-06,20.0,1.6,51392.8\nR001,2019-06-07,19.3,4.4,48792.06\nR001,2019-06-08,19.1,1.8,51726.28\nR001,2019-06-09,19.6,0.3,52229.18\nR001,2019-06-10,21.5,1.1,52795.53\nR001,2019-06-11,21.6,5.1,52782.44\nR001,2019-06-12,20.0,12.6,52402.99\nR001,2019-06-13,17.8,5.7,31375.54\nR001,2019-06-14,19.0,4.0,38769.9\nR001,2019-06-15,20.4,3.2,48941.51\nR001,2019-06-16,20.7,4.0,53826.06\nR001,2019-06-17,20.2,4.1,53448.57\nR001,2019-06-18,20.0,1.6,53439.17\nR001,2019-06-19,20.6,0.4,52327.14\nR001,2019-06-20,20.7,0.6,48753.2\nR001,2019-06-21,19.9,1.5,33045.9\nR001,2019-06-22,19.0,7.6,49447.94\nR001,2019-06-23,19.4,9.0,53919.49\nR001,2019-06-24,19.6,9.0,49342.03\nR001,2019-06-25,21.7,0.7,49103.73\nR001,2019-06-26,21.5,2.2,53020.73\nR001,2019-06-27,21.3,2.3,52284.18\nR001,2019-06-28,19.7,3.9,35176.02\nR001,2019-06-29,16.9,6.6,36856.74\nR001,2019-06-30,12.4,8.8,48698.58\nR001,2019-07-01,13.5,0.4,53949.56\nR001,2019-07-02,19.0,0.0,53939.05\nR001,2019-07-03,21.1,0.0,53926.44\nR001,2019-07-04,20.0,2.9,53401.92\nR001,2019-07-05,18.7,6.0,48547.68\nR001,2019-07-06,18.4,3.0,53131.78\nR001,2019-07-07,19.0,0.2,51746.15\nR001,2019-07-08,19.4,3.3,52672.95\nR001,2019-07-09,18.7,0.8,49789.84\nR001,2019-07-10,18.8,0.0,51470.15\nR001,2019-07-11,17.7,5.8,47943.83\nR001,2019-07-12,16.4,41.1,27733.82\nR001,2019-07-13,15.1,1.9,50747.35\nR001,2019-07-14,16.3,0.0,42496.74\nR001,2019-07-15,15.5,12.2,36845.66\nR001,2019-07-16,16.1,4.8,48715.37\nR001,2019-07-17,17.9,0.0,52762.7\nR001,2019-07-18,18.1,19.7,45982.11\nR001,2019-07-19,18.3,13.5,47901.4\nR001,2019-07-20,20.9,0.7,47679.1\nR001,2019-07-21,20.3,5.6,35612.72\nR001,2019-07-22,19.8,1.2,33560.61\nR001,2019-07-23,19.5,7.1,26174.93\nR001,2019-07-24,19.7,7.0,25353.94\nR001,2019-07-25,18.4,16.5,33943.85\nR001,2019-07-26,17.2,0.4,48625.27\nR001,2019-07-27,17.5,0.0,50769.58\nR001,2019-07-28,18.5,1.1,47012.49\nR001,2019-07-29,19.5,0.9,45641.16\nR001,2019-07-30,21.6,0.9,51013.0\nR001,2019-07-31,20.8,0.8,49711.52\nR001,2019-08-01,19.9,0.2,45398.23\nR001,2019-08-02,19.4,1.5,46581.96\nR001,2019-08-03,18.7,0.0,47816.78\nR001,2019-08-04,18.3,6.1,42776.32\nR001,2019-08-05,16.3,1.3,47256.36\nR001,2019-08-06,16.0,0.0,48288.95\nR001,2019-08-07,16.9,0.0,47946.45\nR001,2019-08-08,18.6,0.0,48469.28\nR001,2019-08-09,20.5,0.0,48338.69\nR001,2019-08-10,19.8,0.2,48027.46\nR001,2019-08-11,19.6,0.0,47044.13\nR001,2019-08-12,19.0,0.0,47746.35\nR001,2019-08-13,20.2,0.0,47774.93\nR001,2019-08-14,22.8,0.0,47469.02\nR001,2019-08-15,24.7,0.0,47498.64\nR001,2019-08-16,23.1,5.6,42335.04\nR001,2019-08-17,19.2,33.2,17190.73\nR001,2019-08-18,20.2,3.5,35984.3\nR001,2019-08-19,21.7,0.7,43418.77\nR001,2019-08-20,23.5,0.3,46800.0\nR001,2019-08-21,24.4,2.7,46800.0\nR001,2019-08-22,22.1,2.8,43570.46\nR001,2019-08-23,21.8,5.2,36484.92\nR001,2019-08-24,20.4,7.2,16194.44\nR001,2019-08-25,20.2,11.1,36948.86\nR001,2019-08-26,20.1,0.0,45873.69\nR001,2019-08-27,19.9,0.8,46767.59\nR001,2019-08-28,19.1,0.0,46667.11\nR001,2019-08-29,19.5,0.1,46565.28\nR001,2019-08-30,19.7,2.9,46462.22\nR001,2019-08-31,17.8,0.1,44653.15\nR001,2019-09-01,17.8,0.9,46177.2\nR001,2019-09-02,17.7,2.3,44939.53\nR001,2019-09-03,17.4,0.0,45631.01\nR001,2019-09-04,17.5,1.0,45039.33\nR001,2019-09-05,17.9,2.2,43920.13\nR001,2019-09-06,16.9,9.0,41105.35\nR001,2019-09-07,16.1,3.3,38186.39\nR001,2019-09-08,15.6,0.0,40626.38\nR001,2019-09-09,15.9,0.1,43044.97\nR001,2019-09-10,15.6,3.0,38270.77\nR001,2019-09-11,15.4,1.0,39610.26\nR001,2019-09-12,15.8,0.0,43200.0\nR001,2019-09-13,15.7,0.0,42900.62\nR001,2019-09-14,16.0,0.0,43129.64\nR001,2019-09-15,15.3,0.2,42319.19\nR001,2019-09-16,15.3,0.0,42995.15\nR001,2019-09-17,14.7,4.8,32372.85\nR001,2019-09-18,16.0,0.0,42860.27\nR001,2019-09-19,15.7,0.0,42792.6\nR001,2019-09-20,14.9,4.4,39400.71\nR001,2019-09-21,8.8,10.8,36107.92\nR001,2019-09-22,7.2,0.0,42244.43\nR001,2019-09-23,10.3,0.0,42279.56\nR001,2019-09-24,11.3,0.0,33285.69\nR001,2019-09-25,13.9,2.1,6878.75\nR001,2019-09-26,13.4,11.4,13212.51\nR001,2019-09-27,14.6,3.7,34082.95\nR001,2019-09-28,13.5,27.9,25412.85\nR001,2019-09-29,12.7,0.0,38053.49\nR001,2019-09-30,13.0,0.0,40181.59\nR001,2019-10-01,14.2,0.2,40508.17\nR001,2019-10-02,15.6,0.0,40357.18\nR001,2019-10-03,17.4,0.9,37160.61\nR001,2019-10-04,17.7,0.3,39576.32\nR001,2019-10-05,18.5,0.0,38961.46\nR001,2019-10-06,13.7,22.3,22266.52\nR001,2019-10-07,10.0,2.6,38510.29\nR001,2019-10-08,12.5,6.7,32760.63\nR001,2019-10-09,11.9,9.6,31429.23\nR001,2019-10-10,13.2,0.0,37490.88\nR001,2019-10-11,14.7,0.7,34102.36\nR001,2019-10-12,11.4,5.5,33797.72\nR001,2019-10-13,11.6,0.0,36575.11\nR001,2019-10-14,14.4,0.0,36629.46\nR001,2019-10-15,15.3,0.0,36356.7\nR001,2019-10-16,15.0,0.0,36082.54\nR001,2019-10-17,15.3,0.0,36134.92\nR001,2019-10-18,14.8,0.7,36000.0\nR001,2019-10-19,14.4,1.3,36000.0\nR001,2019-10-20,13.1,4.6,36000.0\nR001,2019-10-21,12.6,0.0,36000.0\nR001,2019-10-22,12.7,0.0,36000.0\nR001,2019-10-23,11.9,0.0,32400.0\nR001,2019-10-24,11.8,0.0,36000.0\nR001,2019-10-25,11.5,0.0,36000.0\nR001,2019-10-26,10.9,0.4,36000.0\nR001,2019-10-27,10.3,0.0,36000.0\nR001,2019-10-28,10.3,0.0,36000.0\nR001,2019-10-29,10.4,0.0,36000.0\nR001,2019-10-30,8.8,13.9,10846.7\nR001,2019-10-31,6.3,5.1,1163.09\nR001,2019-11-01,5.4,5.7,2741.68\nR001,2019-11-02,4.0,0.3,34447.06\nR001,2019-11-03,4.4,0.0,35889.76\nR001,2019-11-04,7.1,0.0,35809.42\nR001,2019-11-05,10.2,0.4,31430.46\nR001,2019-11-06,11.8,0.8,35583.13\nR001,2019-11-07,11.9,0.0,35427.5\nR001,2019-11-08,12.5,0.0,35273.76\nR001,2019-11-09,12.0,0.0,35122.11\nR001,2019-11-10,11.3,0.0,34972.73\nR001,2019-11-11,12.1,0.0,34825.82\nR001,2019-11-12,10.5,0.0,34681.59\nR001,2019-11-13,9.6,0.0,33736.8\nR001,2019-11-14,11.2,0.0,34401.99\nR001,2019-11-15,12.4,0.0,34266.86\nR001,2019-11-16,10.0,0.0,34133.56\nR001,2019-11-17,8.3,0.0,34001.75\nR001,2019-11-18,7.8,0.0,33691.4\nR001,2019-11-19,6.7,0.0,33322.98\nR001,2019-11-20,5.2,1.0,31534.38\nR001,2019-11-21,4.6,11.2,8056.68\nR001,2019-11-22,3.9,0.0,32217.56\nR001,2019-11-23,4.9,0.0,32217.61\nR001,2019-11-24,4.9,0.0,32206.13\nR001,2019-11-25,5.9,0.0,30935.78\nR001,2019-11-26,6.9,0.0,29067.16\nR001,2019-11-27,8.4,4.3,0.0\nR001,2019-11-28,5.1,12.3,13793.3\nR001,2019-11-29,6.0,1.4,22786.67\nR001,2019-11-30,5.9,20.8,2579.22\nR001,2019-12-01,3.3,17.3,0.0\nR001,2019-12-02,0.3,0.7,28962.09\nR001,2019-12-03,0.1,3.3,12032.94\nR001,2019-12-04,-2.1,16.1,20696.86\nR001,2019-12-05,-7.3,0.0,30379.26\nR001,2019-12-06,-4.3,0.0,30390.04\nR001,2019-12-07,-1.8,0.0,30399.26\nR001,2019-12-08,-0.4,0.0,29636.8\nR001,2019-12-09,1.8,0.0,29255.78\nR001,2019-12-10,0.5,0.0,30417.4\nR001,2019-12-11,-0.7,0.0,30420.33\nR001,2019-12-12,0.7,0.0,30421.78\nR001,2019-12-13,2.9,0.5,24136.66\nR001,2019-12-14,6.5,0.2,27499.41\nR001,2019-12-15,7.4,8.1,23279.51\nR001,2019-12-16,5.4,3.2,27398.88\nR001,2019-12-17,3.3,0.0,25620.48\nR001,2019-12-18,4.3,0.0,30789.17\nR001,2019-12-19,6.1,0.0,30780.57\nR001,2019-12-20,4.8,0.0,30770.54\nR001,2019-12-21,5.5,0.0,29313.18\nR001,2019-12-22,7.0,0.0,28456.18\nR001,2019-12-23,9.4,0.0,29557.86\nR001,2019-12-24,9.5,0.0,22623.81\nR001,2019-12-25,8.0,1.9,0.0\nR001,2019-12-26,6.0,0.3,26297.47\nR001,2019-12-27,4.1,11.5,420.55\nR001,2019-12-28,1.3,5.8,28800.0\nR001,2019-12-29,0.9,0.0,29991.16\nR001,2019-12-30,2.7,0.4,26596.44\nR001,2019-12-31,1.1,0.0,30169.45\nR001,2020-01-01,1.6,7.3,25700.36\nR001,2020-01-02,-0.6,1.7,30338.81\nR001,2020-01-03,-2.9,0.0,16973.42\nR001,2020-01-04,-1.5,0.0,31750.72\nR001,2020-01-05,0.2,0.0,30015.0\nR001,2020-01-06,0.2,0.0,26611.42\nR001,2020-01-07,2.6,5.6,0.0\nR001,2020-01-08,2.7,0.2,28300.61\nR001,2020-01-09,1.4,8.8,12770.99\nR001,2020-01-10,-0.6,2.1,31009.04\nR001,2020-01-11,0.1,2.1,27568.28\nR001,2020-01-12,0.3,11.7,206.16\nR001,2020-01-13,-3.0,0.0,32333.98\nR001,2020-01-14,0.8,0.0,32400.0\nR001,2020-01-15,0.1,0.0,32400.0\nR001,2020-01-16,-0.4,0.0,32400.0\nR001,2020-01-17,-0.3,0.0,32400.0\nR001,2020-01-18,-1.2,0.3,32400.0\nR001,2020-01-19,-1.1,0.0,32400.0\nR001,2020-01-20,-0.9,0.0,32400.0\nR001,2020-01-21,-1.1,0.0,32400.0\nR001,2020-01-22,0.1,5.2,15680.94\nR001,2020-01-23,-1.8,32.2,0.0\nR001,2020-01-24,-7.9,0.0,32400.0\nR001,2020-01-25,-2.8,0.0,23199.46\nR001,2020-01-26,-0.2,0.0,31634.03\nR001,2020-01-27,1.8,0.0,32400.0\nR001,2020-01-28,4.9,0.0,25030.1\nR001,2020-01-29,4.3,8.0,3259.76\nR001,2020-01-30,3.6,11.2,131.37\nR001,2020-01-31,1.0,27.6,0.0\nR001,2020-02-01,-1.6,0.4,32400.0\nR001,2020-02-02,1.3,6.7,5661.47\nR001,2020-02-03,4.0,34.9,0.0\nR001,2020-02-04,0.9,35.0,25387.03\nR001,2020-02-05,1.7,7.7,0.0\nR001,2020-02-06,6.7,18.1,0.0\nR001,2020-02-07,0.4,28.9,4069.63\nR001,2020-02-08,-6.3,6.1,7567.37\nR001,2020-02-09,-13.7,0.0,30601.99\nR001,2020-02-10,-12.2,0.0,35408.44\nR001,2020-02-11,-7.8,0.0,33869.92\nR001,2020-02-12,-3.2,10.6,6396.59\nR001,2020-02-13,-3.3,7.3,22072.3\nR001,2020-02-14,-3.5,1.6,34909.18\nR001,2020-02-15,-1.1,0.0,26439.29\nR001,2020-02-16,1.3,0.0,35624.23\nR001,2020-02-17,0.3,0.0,37520.59\nR001,2020-02-18,2.5,0.0,37692.63\nR001,2020-02-19,5.4,0.0,37864.93\nR001,2020-02-20,3.2,0.0,32313.5\nR001,2020-02-21,3.6,0.0,34651.42\nR001,2020-02-22,3.8,0.3,30218.83\nR001,2020-02-23,2.3,4.9,24577.09\nR001,2020-02-24,3.0,16.6,10333.71\nR001,2020-02-25,-1.1,13.0,38901.93\nR001,2020-02-26,0.9,0.0,39078.02\nR001,2020-02-27,8.3,0.0,39255.32\nR001,2020-02-28,3.7,20.6,12612.9\nR001,2020-02-29,2.7,6.5,22460.07\nR001,2020-03-01,2.3,12.1,19564.23\nR001,2020-03-02,3.0,0.0,39600.0\nR001,2020-03-03,6.1,0.0,39600.0\nR001,2020-03-04,9.1,0.0,39600.0\nR001,2020-03-05,10.0,0.0,39600.0\nR001,2020-03-06,10.2,0.0,28800.0\nR001,2020-03-07,10.1,0.0,37331.4\nR001,2020-03-08,8.5,0.0,39600.0\nR001,2020-03-09,9.8,0.0,39600.0\nR001,2020-03-10,11.3,0.0,38871.19\nR001,2020-03-11,10.0,0.4,39600.0\nR001,2020-03-12,6.4,1.0,39600.0\nR001,2020-03-13,8.6,0.0,39606.96\nR001,2020-03-14,8.8,5.8,24683.88\nR001,2020-03-15,4.7,18.6,1362.59\nR001,2020-03-16,-1.1,12.0,13813.81\nR001,2020-03-17,-2.8,4.4,14775.91\nR001,2020-03-18,0.7,0.5,27944.02\nR001,2020-03-19,3.0,2.0,17222.14\nR001,2020-03-20,4.6,0.0,33072.17\nR001,2020-03-21,4.8,0.7,29865.2\nR001,2020-03-22,6.4,0.0,39587.93\nR001,2020-03-23,6.9,0.0,35989.58\nR001,2020-03-24,6.4,2.1,40914.01\nR001,2020-03-25,6.9,0.9,40228.06\nR001,2020-03-26,7.5,0.0,29705.37\nR001,2020-03-27,9.7,0.1,42233.41\nR001,2020-03-28,10.6,3.9,40358.99\nR001,2020-03-29,8.9,7.2,23839.7\nR001,2020-03-30,6.3,13.9,0.0\nR001,2020-03-31,6.8,3.6,35674.17\nR001,2020-04-01,6.1,2.6,35988.17\nR001,2020-04-02,5.4,2.8,44584.52\nR001,2020-04-03,6.2,0.1,39397.55\nR001,2020-04-04,7.9,2.0,36920.02\nR001,2020-04-05,7.5,0.0,35593.85\nR001,2020-04-06,6.8,0.0,46047.11\nR001,2020-04-07,5.0,0.0,46118.57\nR001,2020-04-08,6.6,0.0,46190.05\nR001,2020-04-09,7.4,0.0,46261.55\nR001,2020-04-10,7.4,0.0,46333.04\nR001,2020-04-11,5.4,3.3,44004.18\nR001,2020-04-12,4.1,2.9,41402.39\nR001,2020-04-13,4.5,0.0,46547.41\nR001,2020-04-14,8.3,0.0,46618.78\nR001,2020-04-15,9.4,7.5,27810.06\nR001,2020-04-16,5.0,12.5,34120.34\nR001,2020-04-17,7.6,0.0,46800.0\nR001,2020-04-18,10.1,0.0,43545.56\nR001,2020-04-19,9.9,0.0,46800.0\nR001,2020-04-20,10.4,6.0,38624.54\nR001,2020-04-21,6.2,5.7,42281.87\nR001,2020-04-22,4.4,0.0,26793.31\nR001,2020-04-23,4.5,0.6,46820.23\nR001,2020-04-24,5.1,0.5,47438.36\nR001,2020-04-25,7.7,0.1,45795.67\nR001,2020-04-26,7.6,0.3,47023.24\nR001,2020-04-27,5.9,0.7,39963.31\nR001,2020-04-28,6.2,0.2,46805.43\nR001,2020-04-29,8.7,0.0,48236.67\nR001,2020-04-30,12.1,0.0,46800.0\nR001,2020-05-01,12.3,2.4,41590.97\nR001,2020-05-02,10.9,11.1,788.29\nR001,2020-05-03,10.9,5.8,42664.44\nR001,2020-05-04,10.8,9.0,33516.42\nR001,2020-05-05,12.3,4.1,43941.15\nR001,2020-05-06,10.8,8.8,13327.74\nR001,2020-05-07,12.0,0.8,43728.51\nR001,2020-05-08,8.3,8.0,26561.86\nR001,2020-05-09,6.1,4.2,39390.67\nR001,2020-05-10,6.1,1.4,49578.5\nR001,2020-05-11,8.9,0.0,49210.17\nR001,2020-05-12,12.9,0.1,44991.77\nR001,2020-05-13,15.3,2.0,50355.79\nR001,2020-05-14,13.4,0.0,50340.87\nR001,2020-05-15,16.3,1.0,41966.16\nR001,2020-05-16,17.3,0.6,47649.28\nR001,2020-05-17,13.7,0.0,47933.87\nR001,2020-05-18,14.4,1.2,49264.08\nR001,2020-05-19,13.0,0.0,50400.0\nR001,2020-05-20,15.0,0.0,49634.03\nR001,2020-05-21,20.1,0.4,50400.0\nR001,2020-05-22,15.5,24.8,34022.19\nR001,2020-05-23,8.1,47.0,27314.53\nR001,2020-05-24,7.7,0.0,41586.15\nR001,2020-05-25,9.1,6.7,21963.66\nR001,2020-05-26,13.0,0.7,45795.81\nR001,2020-05-27,17.8,0.4,51575.98\nR001,2020-05-28,20.4,0.9,51193.78\nR001,2020-05-29,16.2,14.1,41938.96\nR001,2020-05-30,13.2,3.4,48273.44\nR001,2020-05-31,14.4,0.1,52216.06\nR001,2020-06-01,14.9,4.3,44608.09\nR001,2020-06-02,15.7,7.3,48801.87\nR001,2020-06-03,13.8,14.9,43556.16\nR001,2020-06-04,14.1,5.5,51454.61\nR001,2020-06-05,15.4,0.1,50527.98\nR001,2020-06-06,17.7,0.0,52477.78\nR001,2020-06-07,20.9,0.0,53558.35\nR001,2020-06-08,22.3,0.0,53178.6\nR001,2020-06-09,22.5,0.0,53164.63\nR001,2020-06-10,22.1,3.1,53517.48\nR001,2020-06-11,22.4,2.8,51671.33\nR001,2020-06-12,18.8,17.8,34422.55\nR001,2020-06-13,17.4,20.0,42955.85\nR001,2020-06-14,17.3,15.8,50126.24\nR001,2020-06-15,19.0,4.6,45352.88\nR001,2020-06-16,20.6,1.4,52609.55\nR001,2020-06-17,20.7,3.2,51148.73\nR001,2020-06-18,19.5,2.1,53899.63\nR001,2020-06-19,17.5,11.3,26551.39\nR001,2020-06-20,16.6,18.9,30309.31\nR001,2020-06-21,16.6,11.0,35027.53\nR001,2020-06-22,18.4,2.5,51107.97\nR001,2020-06-23,21.1,0.1,53957.1\nR001,2020-06-24,19.9,4.9,51190.73\nR001,2020-06-25,18.7,1.8,50357.19\nR001,2020-06-26,19.7,0.0,53818.43\nR001,2020-06-27,19.2,1.2,53965.46\nR001,2020-06-28,18.6,0.4,53962.5\nR001,2020-06-29,19.0,0.5,53957.49\nR001,2020-06-30,19.4,0.6,53380.81\nR001,2020-07-01,19.3,5.5,53746.14\nR001,2020-07-02,20.7,1.7,53929.85\nR001,2020-07-03,21.6,1.1,53916.37\nR001,2020-07-04,21.9,3.6,53748.8\nR001,2020-07-05,21.7,4.0,53882.88\nR001,2020-07-06,21.9,3.4,53027.77\nR001,2020-07-07,23.0,1.4,53840.57\nR001,2020-07-08,20.6,2.7,51472.09\nR001,2020-07-09,19.7,1.6,47477.03\nR001,2020-07-10,20.2,0.0,48480.67\nR001,2020-07-11,20.9,0.0,51852.02\nR001,2020-07-12,20.4,8.0,46960.85\nR001,2020-07-13,17.9,17.1,12145.88\nR001,2020-07-14,18.8,0.3,49751.31\nR001,2020-07-15,19.2,0.1,51464.35\nR001,2020-07-16,20.5,0.0,53474.96\nR001,2020-07-17,19.7,7.7,49852.5\nR001,2020-07-18,21.1,0.4,48600.78\nR001,2020-07-19,23.2,0.3,52090.62\nR001,2020-07-20,25.3,0.3,51698.12\nR001,2020-07-21,19.9,14.7,35413.65\nR001,2020-07-22,19.9,1.0,48082.65\nR001,2020-07-23,19.6,0.4,51233.37\nR001,2020-07-24,19.6,1.9,49189.96\nR001,2020-07-25,21.4,1.0,51196.56\nR001,2020-07-26,22.5,5.7,50774.56\nR001,2020-07-27,22.3,1.7,47955.83\nR001,2020-07-28,20.2,35.8,40951.59\nR001,2020-07-29,19.0,6.1,46748.24\nR001,2020-07-30,19.6,0.8,47231.3\nR001,2020-07-31,20.9,1.2,49059.87\nR001,2020-08-01,18.8,5.4,47117.74\nR001,2020-08-02,17.6,6.0,45270.16\nR001,2020-08-03,17.4,0.2,48200.95\nR001,2020-08-04,18.2,0.0,49630.45\nR001,2020-08-05,19.1,0.0,49348.73\nR001,2020-08-06,19.8,0.0,48728.81\nR001,2020-08-07,21.7,0.0,48453.29\nR001,2020-08-08,22.8,0.0,48331.52\nR001,2020-08-09,22.2,0.0,48020.39\nR001,2020-08-10,20.9,1.5,47711.08\nR001,2020-08-11,19.1,6.2,44878.96\nR001,2020-08-12,19.1,1.4,45334.99\nR001,2020-08-13,17.3,2.5,45455.27\nR001,2020-08-14,17.3,0.0,46178.41\nR001,2020-08-15,16.6,0.0,46800.0\nR001,2020-08-16,16.2,0.0,46216.3\nR001,2020-08-17,17.0,0.0,46918.44\nR001,2020-08-18,18.3,0.0,46949.8\nR001,2020-08-19,19.5,0.0,46800.0\nR001,2020-08-20,20.6,0.3,46800.0\nR001,2020-08-21,17.6,9.2,37828.33\nR001,2020-08-22,17.0,2.0,42850.56\nR001,2020-08-23,16.9,0.0,46800.0\nR001,2020-08-24,17.1,0.0,46800.0\nR001,2020-08-25,19.0,0.0,46800.0\nR001,2020-08-26,19.6,0.0,46794.61\nR001,2020-08-27,18.6,0.2,46691.92\nR001,2020-08-28,17.6,0.5,43213.93\nR001,2020-08-29,18.9,0.0,46483.85\nR001,2020-08-30,21.2,0.0,46378.68\nR001,2020-08-31,22.1,0.0,46272.89\nR001,2020-09-01,22.7,0.0,46166.59\nR001,2020-09-02,23.2,0.2,45300.92\nR001,2020-09-03,24.0,0.2,45029.8\nR001,2020-09-04,24.7,0.0,42096.48\nR001,2020-09-05,22.6,2.2,40988.84\nR001,2020-09-06,17.7,15.7,31676.65\nR001,2020-09-07,18.7,0.3,39389.42\nR001,2020-09-08,18.4,3.6,39992.93\nR001,2020-09-09,18.6,1.4,40734.92\nR001,2020-09-10,19.0,0.0,41044.55\nR001,2020-09-11,20.4,0.0,43207.18\nR001,2020-09-12,21.1,0.0,43200.0\nR001,2020-09-13,21.8,0.0,43145.92\nR001,2020-09-14,21.6,0.0,43078.67\nR001,2020-09-15,19.6,0.0,40123.18\nR001,2020-09-16,18.8,0.0,42943.81\nR001,2020-09-17,19.7,0.0,42876.2\nR001,2020-09-18,18.9,1.1,40787.72\nR001,2020-09-19,15.7,0.3,39277.78\nR001,2020-09-20,13.9,0.0,40291.82\nR001,2020-09-21,15.2,0.0,42604.44\nR001,2020-09-22,17.1,0.0,42271.44\nR001,2020-09-23,17.6,0.3,41988.36\nR001,2020-09-24,17.5,0.0,41386.8\nR001,2020-09-25,17.8,0.0,41422.58\nR001,2020-09-26,17.4,0.4,40501.85\nR001,2020-09-27,19.0,0.0,41175.93\nR001,2020-09-28,19.0,1.4,40573.77\nR001,2020-09-29,18.7,0.4,40068.02\nR001,2020-09-30,18.1,0.8,33911.59\nR001,2020-10-01,16.0,2.3,30348.29\nR001,2020-10-02,13.7,11.0,106.81\nR001,2020-10-03,13.8,4.4,30470.3\nR001,2020-10-04,13.8,1.9,38057.29\nR001,2020-10-05,13.8,0.4,37469.43\nR001,2020-10-06,13.4,0.0,37526.47\nR001,2020-10-07,14.3,0.0,38575.77\nR001,2020-10-08,15.7,0.0,38347.71\nR001,2020-10-09,18.0,0.0,37796.01\nR001,2020-10-10,18.3,0.0,37105.36\nR001,2020-10-11,16.4,0.0,36510.75\nR001,2020-10-12,16.3,0.0,36891.2\nR001,2020-10-13,18.0,0.0,36619.76\nR001,2020-10-14,18.3,0.0,36346.85\nR001,2020-10-15,16.8,0.0,36072.34\nR001,2020-10-16,15.4,0.0,36124.08\nR001,2020-10-17,15.7,0.0,36175.14\nR001,2020-10-18,17.2,0.0,36000.0\nR001,2020-10-19,13.9,14.0,30588.39\nR001,2020-10-20,9.3,0.0,36000.0\nR001,2020-10-21,9.3,0.0,36000.0\nR001,2020-10-22,10.3,0.0,30949.0\nR001,2020-10-23,12.6,0.0,36000.0\nR001,2020-10-24,13.6,0.0,36000.0\nR001,2020-10-25,14.1,0.0,36000.0\nR001,2020-10-26,14.4,0.0,36000.0\nR001,2020-10-27,14.1,1.3,35580.05\nR001,2020-10-28,16.5,0.0,36000.0\nR001,2020-10-29,16.2,0.0,36000.0\nR001,2020-10-30,15.0,0.9,34768.15\nR001,2020-10-31,11.9,10.9,35859.54\nR001,2020-11-01,11.3,14.3,27830.58\nR001,2020-11-02,8.4,10.1,35186.53\nR001,2020-11-03,7.7,0.0,35746.68\nR001,2020-11-04,10.5,0.0,35746.06\nR001,2020-11-05,11.1,0.0,34940.31\nR001,2020-11-06,9.3,2.2,34505.61\nR001,2020-11-07,6.9,0.0,30585.81\nR001,2020-11-08,6.4,0.0,35162.31\nR001,2020-11-09,8.8,0.0,35016.68\nR001,2020-11-10,7.7,0.0,34872.35\nR001,2020-11-11,6.9,0.7,33400.54\nR001,2020-11-12,6.4,0.8,32308.18\nR001,2020-11-13,5.2,0.0,34446.2\nR001,2020-11-14,6.4,0.0,34080.78\nR001,2020-11-15,5.5,0.0,31193.95\nR001,2020-11-16,4.3,5.7,7742.75\nR001,2020-11-17,3.3,5.0,30043.31\nR001,2020-11-18,0.9,2.0,27913.92\nR001,2020-11-19,0.2,4.9,26111.48\nR001,2020-11-20,-0.0,7.7,24736.99\nR001,2020-11-21,2.0,4.0,29785.44\nR001,2020-11-22,0.8,18.2,13267.74\nR001,2020-11-23,-3.5,0.0,31453.14\nR001,2020-11-24,1.3,0.2,16755.93\nR001,2020-11-25,1.0,1.3,30959.46\nR001,2020-11-26,-0.5,5.3,12234.55\nR001,2020-11-27,-0.5,0.0,30633.66\nR001,2020-11-28,3.7,0.0,31404.94\nR001,2020-11-29,3.7,0.0,31087.88\nR001,2020-11-30,3.1,0.0,30763.03\nR001,2020-12-01,3.4,0.0,30054.65\nR001,2020-12-02,4.1,0.0,27632.18\nR001,2020-12-03,4.9,0.0,29598.2\nR001,2020-12-04,3.9,0.0,29608.96\nR001,2020-12-05,3.9,0.0,30385.07\nR001,2020-12-06,2.0,0.5,29626.34\nR001,2020-12-07,1.0,0.0,30402.65\nR001,2020-12-08,2.1,0.0,30409.26\nR001,2020-12-09,3.1,3.8,11509.37\nR001,2020-12-10,2.8,0.0,30418.18\nR001,2020-12-11,3.7,0.0,26569.49\nR001,2020-12-12,6.2,0.0,24610.52\nR001,2020-12-13,7.8,0.0,27463.76\nR001,2020-12-14,7.6,0.0,29232.56\nR001,2020-12-15,7.6,0.4,28800.0\nR001,2020-12-16,8.1,1.7,22611.47\nR001,2020-12-17,3.6,3.0,24401.8\nR001,2020-12-18,2.4,0.1,29614.39\nR001,2020-12-19,2.0,0.0,27888.93\nR001,2020-12-20,1.1,4.5,3535.39\nR001,2020-12-21,0.5,13.8,23923.3\nR001,2020-12-22,-1.4,7.6,15722.29\nR001,2020-12-23,-3.4,1.6,22461.37\nR001,2020-12-24,-5.4,0.0,31254.78\nR001,2020-12-25,-3.1,1.2,25200.0\nR001,2020-12-26,4.7,1.7,27193.93\nR001,2020-12-27,6.6,0.0,31030.9\nR001,2020-12-28,5.7,0.0,29833.42\nR001,2020-12-29,7.2,0.0,31473.64\nR001,2020-12-30,5.5,0.0,31523.69\nR001,2020-12-31,5.4,0.0,31575.55\nR001,2021-01-01,6.4,0.0,31629.1\nR001,2021-01-02,5.6,0.0,31684.2\nR001,2021-01-03,2.9,0.0,31740.74\nR001,2021-01-04,2.3,0.0,31798.59\nR001,2021-01-05,4.8,0.0,28838.95\nR001,2021-01-06,4.5,0.8,17827.16\nR001,2021-01-07,3.8,0.0,29921.2\nR001,2021-01-08,7.4,0.0,14012.11\nR001,2021-01-09,8.6,0.0,21440.24\nR001,2021-01-10,6.9,20.5,2760.62\nR001,2021-01-11,4.4,20.8,0.0\nR001,2021-01-12,6.3,11.8,1121.75\nR001,2021-01-13,9.5,11.3,18291.37\nR001,2021-01-14,3.4,40.9,0.0\nR001,2021-01-15,0.3,27.5,10823.24\nR001,2021-01-16,-2.8,2.0,30759.97\nR001,2021-01-17,0.9,33.2,0.0\nR001,2021-01-18,-2.3,11.3,22609.88\nR001,2021-01-19,-6.6,4.4,27160.91\nR001,2021-01-20,-7.7,2.4,31736.36\nR001,2021-01-21,-7.2,0.2,32400.0\nR001,2021-01-22,-1.3,0.0,32400.0\nR001,2021-01-23,0.2,0.0,32400.0\nR001,2021-01-24,1.8,0.0,30823.41\nR001,2021-01-25,6.5,0.0,32400.0\nR001,2021-01-26,4.9,0.3,28800.0\nR001,2021-01-27,7.0,0.0,25200.0\nR001,2021-01-28,4.6,10.7,4473.43\nR001,2021-01-29,-1.0,4.9,3781.67\nR001,2021-01-30,-1.0,0.0,32443.68\nR001,2021-01-31,2.8,2.5,16154.14\nR001,2021-02-01,5.0,0.0,31609.87\nR001,2021-02-02,8.2,0.0,32374.79\nR001,2021-02-03,8.3,0.0,32400.0\nR001,2021-02-04,8.3,0.0,32453.03\nR001,2021-02-05,2.7,24.4,14198.09\nR001,2021-02-06,-0.3,9.8,1947.31\nR001,2021-02-07,3.2,20.8,0.0\nR001,2021-02-08,6.9,1.4,24478.15\nR001,2021-02-09,6.8,7.9,17938.79\nR001,2021-02-10,6.7,0.0,35155.49\nR001,2021-02-11,7.6,0.0,33969.73\nR001,2021-02-12,5.6,4.4,34831.57\nR001,2021-02-13,-4.1,21.2,2432.66\nR001,2021-02-14,-2.5,10.9,2993.45\nR001,2021-02-15,-0.3,32.3,0.0\nR001,2021-02-16,-2.3,22.4,0.0\nR001,2021-02-17,-3.3,8.4,0.0\nR001,2021-02-18,-4.2,3.6,32400.0\nR001,2021-02-19,-4.1,5.5,13654.99\nR001,2021-02-20,-4.6,0.0,37447.76\nR001,2021-02-21,-3.5,0.0,38329.64\nR001,2021-02-22,-0.7,6.5,23368.64\nR001,2021-02-23,0.2,3.9,2056.17\nR001,2021-02-24,-0.9,0.0,38859.35\nR001,2021-02-25,-1.0,0.1,39037.83\nR001,2021-02-26,1.7,0.0,21613.36\nR001,2021-02-27,3.4,0.0,37930.34\nR001,2021-02-28,0.7,11.6,670.7\nR001,2021-03-01,-0.9,0.1,37920.38\nR001,2021-03-02,-0.9,0.0,39600.0\nR001,2021-03-03,-2.2,15.1,12541.09\nR001,2021-03-04,-2.7,0.0,39600.0\nR001,2021-03-05,1.0,0.0,39600.0\nR001,2021-03-06,4.7,0.6,39600.0\nR001,2021-03-07,-0.2,9.7,36010.82\nR001,2021-03-08,-2.4,0.0,39600.0\nR001,2021-03-09,-3.1,3.5,39123.83\nR001,2021-03-10,-1.7,0.2,39600.0\nR001,2021-03-11,-0.0,9.8,996.01\nR001,2021-03-12,-2.9,11.2,16423.03\nR001,2021-03-13,-0.5,0.0,35632.85\nR001,2021-03-14,2.9,0.0,39600.0\nR001,2021-03-15,5.0,0.0,32633.64\nR001,2021-03-16,6.8,5.4,35059.97\nR001,2021-03-17,5.3,4.6,24237.94\nR001,2021-03-18,2.3,14.0,21772.7\nR001,2021-03-19,0.9,17.1,22165.67\nR001,2021-03-20,-0.0,5.5,20300.19\nR001,2021-03-21,-0.4,0.8,5023.83\nR001,2021-03-22,2.5,2.5,34844.93\nR001,2021-03-23,4.5,23.4,13792.01\nR001,2021-03-24,2.4,23.4,6006.81\nR001,2021-03-25,0.3,5.3,18642.63\nR001,2021-03-26,-0.7,3.6,37327.79\nR001,2021-03-27,-0.3,0.0,41619.3\nR001,2021-03-28,1.0,0.8,40456.9\nR001,2021-03-29,1.6,0.0,41126.43\nR001,2021-03-30,2.6,0.1,41995.29\nR001,2021-03-31,4.5,0.0,44124.62\nR001,2021-04-01,5.6,1.7,36340.17\nR001,2021-04-02,4.7,5.1,3087.87\nR001,2021-04-03,5.2,0.9,33152.18\nR001,2021-04-04,7.9,2.1,41138.98\nR001,2021-04-05,7.6,35.6,11382.22\nR001,2021-04-06,6.1,1.3,44026.15\nR001,2021-04-07,13.9,2.1,35503.56\nR001,2021-04-08,14.0,1.2,46172.81\nR001,2021-04-09,7.1,9.9,3447.25\nR001,2021-04-10,5.3,4.3,19801.72\nR001,2021-04-11,7.4,4.8,14404.07\nR001,2021-04-12,6.6,7.2,17433.06\nR001,2021-04-13,6.9,6.0,29452.9\nR001,2021-04-14,7.3,2.2,45279.39\nR001,2021-04-15,9.7,0.0,46647.18\nR001,2021-04-16,10.5,0.2,46742.55\nR001,2021-04-17,8.4,9.4,0.0\nR001,2021-04-18,11.7,9.8,42417.15\nR001,2021-04-19,11.8,14.2,4435.59\nR001,2021-04-20,12.8,16.2,34909.18\nR001,2021-04-21,9.6,9.8,35818.4\nR001,2021-04-22,9.5,2.0,43938.74\nR001,2021-04-23,10.0,0.0,45521.6\nR001,2021-04-24,9.1,3.2,21762.07\nR001,2021-04-25,10.6,1.9,30198.27\nR001,2021-04-26,9.9,25.2,4514.31\nR001,2021-04-27,8.3,2.6,12128.9\nR001,2021-04-28,12.2,0.3,40415.51\nR001,2021-04-29,13.0,2.3,38959.08\nR001,2021-04-30,14.1,0.7,47871.27\nR001,2021-05-01,12.9,1.6,46826.92\nR001,2021-05-02,14.4,0.1,48469.11\nR001,2021-05-03,17.2,0.0,46371.53\nR001,2021-05-04,12.8,9.7,15305.72\nR001,2021-05-05,11.1,0.0,44686.78\nR001,2021-05-06,12.8,0.0,47327.17\nR001,2021-05-07,13.1,0.0,48984.58\nR001,2021-05-08,15.0,0.1,48955.96\nR001,2021-05-09,8.0,33.3,2350.99\nR001,2021-05-10,3.5,4.0,47873.98\nR001,2021-05-11,4.0,2.7,27227.97\nR001,2021-05-12,10.1,0.0,50268.61\nR001,2021-05-13,16.6,0.0,50335.65\nR001,2021-05-14,20.4,0.3,50400.0\nR001,2021-05-15,19.6,1.0,50400.0\nR001,2021-05-16,21.0,0.9,50400.0\nR001,2021-05-17,16.3,9.7,43770.07\nR001,2021-05-18,15.6,0.1,49902.72\nR001,2021-05-19,12.7,6.1,39780.68\nR001,2021-05-20,17.2,0.0,50684.69\nR001,2021-05-21,21.9,0.0,50400.0\nR001,2021-05-22,18.6,9.5,33000.65\nR001,2021-05-23,15.7,20.2,2480.7\nR001,2021-05-24,15.6,5.8,11288.22\nR001,2021-05-25,13.5,7.9,39344.04\nR001,2021-05-26,13.7,0.4,49644.98\nR001,2021-05-27,15.1,0.9,45696.26\nR001,2021-05-28,17.3,1.3,51202.25\nR001,2021-05-29,16.5,9.4,48745.07\nR001,2021-05-30,13.4,6.5,37793.37\nR001,2021-05-31,16.9,0.0,51500.02\nR001,2021-06-01,17.9,12.8,43262.05\nR001,2021-06-02,13.7,24.7,22885.61\nR001,2021-06-03,11.1,4.9,47540.33\nR001,2021-06-04,10.0,9.9,12356.77\nR001,2021-06-05,12.4,7.4,35900.74\nR001,2021-06-06,11.4,3.4,49779.63\nR001,2021-06-07,11.3,2.3,48549.19\nR001,2021-06-08,13.8,0.0,52814.32\nR001,2021-06-09,13.5,0.8,45104.58\nR001,2021-06-10,14.9,0.4,52053.15\nR001,2021-06-11,15.7,0.0,52039.28\nR001,2021-06-12,16.3,2.8,52232.68\nR001,2021-06-13,16.0,6.4,49094.31\nR001,2021-06-14,18.3,0.1,52736.88\nR001,2021-06-15,19.0,4.5,50416.63\nR001,2021-06-16,19.6,1.0,53452.26\nR001,2021-06-17,17.1,13.1,39352.62\nR001,2021-06-18,18.7,0.0,51778.87\nR001,2021-06-19,22.0,0.1,53906.88\nR001,2021-06-20,22.5,0.9,52020.22\nR001,2021-06-21,23.4,2.3,53933.6\nR001,2021-06-22,22.1,8.3,52953.2\nR001,2021-06-23,22.2,2.6,52267.66\nR001,2021-06-24,21.6,8.8,47602.28\nR001,2021-06-25,21.4,3.8,51199.3\nR001,2021-06-26,20.1,8.5,49966.8\nR001,2021-06-27,20.5,9.1,45297.46\nR001,2021-06-28,20.3,6.3,50521.43\nR001,2021-06-29,19.2,9.8,49217.23\nR001,2021-06-30,20.0,0.0,51320.33\nR001,2021-07-01,21.1,0.0,53947.11\nR001,2021-07-02,22.4,0.7,53936.47\nR001,2021-07-03,23.0,0.1,53923.49\nR001,2021-07-04,19.8,9.9,39133.19\nR001,2021-07-05,16.0,30.3,9932.89\nR001,2021-07-06,18.5,1.9,45313.04\nR001,2021-07-07,18.7,1.5,50067.02\nR001,2021-07-08,19.7,1.1,51503.86\nR001,2021-07-09,20.6,0.3,52419.12\nR001,2021-07-10,19.6,0.0,53766.1\nR001,2021-07-11,20.3,0.3,53734.56\nR001,2021-07-12,21.0,2.9,53700.89\nR001,2021-07-13,20.1,8.7,52539.43\nR001,2021-07-14,20.0,2.5,51155.55\nR001,2021-07-15,20.7,0.0,53587.42\nR001,2021-07-16,21.5,0.0,53545.56\nR001,2021-07-17,22.6,0.0,53501.76\nR001,2021-07-18,23.3,0.0,53143.06\nR001,2021-07-19,24.2,0.0,53408.4\nR001,2021-07-20,24.4,0.0,53305.95\nR001,2021-07-21,23.6,2.5,52372.74\nR001,2021-07-22,20.2,4.6,45403.91\nR001,2021-07-23,17.2,15.9,41797.85\nR001,2021-07-24,19.1,5.8,46108.74\nR001,2021-07-25,19.8,3.0,49783.75\nR001,2021-07-26,21.1,3.1,48432.42\nR001,2021-07-27,19.1,18.7,15180.13\nR001,2021-07-28,18.6,0.0,47359.5\nR001,2021-07-29,19.3,1.7,48647.2\nR001,2021-07-30,19.2,0.8,46071.64\nR001,2021-07-31,20.3,3.3,49734.84\nR001,2021-08-01,21.4,0.2,49100.02\nR001,2021-08-02,21.6,2.3,49157.51\nR001,2021-08-03,21.8,0.5,47158.98\nR001,2021-08-04,22.4,1.0,48933.56\nR001,2021-08-05,22.1,0.8,46946.96\nR001,2021-08-06,23.5,0.3,48612.87\nR001,2021-08-07,23.4,2.2,41749.34\nR001,2021-08-08,21.8,3.9,41905.9\nR001,2021-08-09,20.7,13.8,42325.87\nR001,2021-08-10,20.2,11.0,33994.1\nR001,2021-08-11,20.2,9.6,34180.7\nR001,2021-08-12,19.7,15.0,39401.18\nR001,2021-08-13,19.1,7.6,24318.17\nR001,2021-08-14,18.2,0.3,42684.1\nR001,2021-08-15,18.4,1.4,46800.0\nR001,2021-08-16,17.8,9.4,42051.08\nR001,2021-08-17,19.4,2.3,46800.0\nR001,2021-08-18,21.2,1.0,46941.44\nR001,2021-08-19,19.7,10.6,44928.72\nR001,2021-08-20,18.5,5.6,43200.0\nR001,2021-08-21,19.3,1.3,43343.83\nR001,2021-08-22,19.6,3.2,45566.24\nR001,2021-08-23,19.4,3.3,46762.7\nR001,2021-08-24,19.7,1.6,45211.25\nR001,2021-08-25,20.8,0.0,46800.0\nR001,2021-08-26,20.8,1.4,46800.0\nR001,2021-08-27,21.7,0.2,46710.85\nR001,2021-08-28,22.9,0.1,46607.51\nR001,2021-08-29,23.3,0.0,46503.69\nR001,2021-08-30,25.0,0.0,46399.5\nR001,2021-08-31,21.4,10.3,44871.08\nR001,2021-09-01,20.0,4.2,40716.23\nR001,2021-09-02,15.7,22.2,34712.13\nR001,2021-09-03,12.0,9.3,34134.54\nR001,2021-09-04,12.3,0.0,44745.11\nR001,2021-09-05,13.6,0.0,40843.43\nR001,2021-09-06,14.8,1.5,38743.9\nR001,2021-09-07,15.0,0.3,42587.4\nR001,2021-09-08,14.8,0.3,40542.35\nR001,2021-09-09,15.6,0.2,43404.46\nR001,2021-09-10,17.5,0.0,43458.25\nR001,2021-09-11,15.3,24.2,12566.58\nR001,2021-09-12,15.4,3.3,38097.65\nR001,2021-09-13,15.7,0.3,41323.87\nR001,2021-09-14,16.3,0.0,42630.3\nR001,2021-09-15,17.3,0.0,41388.92\nR001,2021-09-16,15.1,43.9,3298.83\nR001,2021-09-17,14.8,28.4,1881.25\nR001,2021-09-18,15.2,0.3,34661.44\nR001,2021-09-19,14.4,3.0,38191.95\nR001,2021-09-20,15.3,0.0,39499.37\nR001,2021-09-21,15.1,2.9,35713.82\nR001,2021-09-22,11.7,6.1,17702.48\nR001,2021-09-23,10.1,5.4,10404.4\nR001,2021-09-24,10.3,1.4,34909.71\nR001,2021-09-25,6.6,36.4,13456.48\nR001,2021-09-26,8.2,0.4,40492.8\nR001,2021-09-27,10.0,3.0,7332.7\nR001,2021-09-28,10.6,1.2,37007.0\nR001,2021-09-29,11.2,0.4,36473.35\nR001,2021-09-30,11.3,8.4,33308.85\nR001,2021-10-01,10.2,17.7,6855.4\nR001,2021-10-02,11.0,0.3,38573.24\nR001,2021-10-03,10.7,3.3,2886.58\nR001,2021-10-04,10.4,8.7,24336.6\nR001,2021-10-05,9.5,7.1,27142.98\nR001,2021-10-06,8.6,4.0,24447.36\nR001,2021-10-07,9.8,1.4,35729.39\nR001,2021-10-08,9.8,0.4,35998.67\nR001,2021-10-09,11.7,0.0,34818.32\nR001,2021-10-10,12.0,0.0,37094.5\nR001,2021-10-11,12.6,0.0,36824.54\nR001,2021-10-12,13.9,0.0,36878.76\nR001,2021-10-13,14.1,3.3,36280.12\nR001,2021-10-14,13.1,0.1,36200.8\nR001,2021-10-15,13.8,0.0,36056.33\nR001,2021-10-16,14.4,0.4,36107.17\nR001,2021-10-17,13.0,2.3,35873.54\nR001,2021-10-18,11.1,1.9,33319.81\nR001,2021-10-19,7.6,3.4,26723.82\nR001,2021-10-20,5.0,0.0,36000.0\nR001,2021-10-21,7.5,0.0,36000.0\nR001,2021-10-22,11.3,0.0,36000.0\nR001,2021-10-23,10.8,0.0,36000.0\nR001,2021-10-24,9.2,6.3,32788.86\nR001,2021-10-25,2.6,13.3,35824.85\nR001,2021-10-26,1.1,0.0,36000.0\nR001,2021-10-27,5.5,0.0,36000.0\nR001,2021-10-28,5.7,0.0,36000.0\nR001,2021-10-29,5.5,0.0,34253.83\nR001,2021-10-30,5.6,0.0,36000.0\nR001,2021-10-31,9.4,0.0,36000.0\nR001,2021-11-01,12.4,0.1,36000.0\nR001,2021-11-02,12.7,2.0,20961.84\nR001,2021-11-03,11.5,0.3,5656.91\nR001,2021-11-04,10.6,0.0,34416.8\nR001,2021-11-05,11.1,0.0,35663.97\nR001,2021-11-06,11.4,0.0,35511.32\nR001,2021-11-07,9.0,0.0,33917.11\nR001,2021-11-08,11.3,0.0,27898.06\nR001,2021-11-09,12.0,0.0,33158.85\nR001,2021-11-10,4.4,21.5,21648.0\nR001,2021-11-11,0.6,0.0,32334.73\nR001,2021-11-12,3.4,0.0,34617.83\nR001,2021-11-13,6.5,0.0,34474.28\nR001,2021-11-14,7.4,0.0,32065.59\nR001,2021-11-15,7.0,0.0,30622.73\nR001,2021-11-16,4.4,0.0,33703.43\nR001,2021-11-17,5.4,0.0,33332.42\nR001,2021-11-18,5.2,0.0,32596.67\nR001,2021-11-19,3.8,4.0,29145.56\nR001,2021-11-20,3.7,10.5,25674.57\nR001,2021-11-21,4.6,11.6,20353.57\nR001,2021-11-22,5.0,0.0,32212.43\nR001,2021-11-23,6.1,0.3,30693.61\nR001,2021-11-24,2.9,16.5,19940.78\nR001,2021-11-25,-1.0,0.0,31194.0\nR001,2021-11-26,1.6,0.0,31261.04\nR001,2021-11-27,4.4,5.4,2214.35\nR001,2021-11-28,8.1,11.4,7035.53\nR001,2021-11-29,10.0,1.6,25925.03\nR001,2021-11-30,10.5,34.3,3353.68\nR001,2021-12-01,1.6,15.9,22421.67\nR001,2021-12-02,-0.7,8.8,29241.48\nR001,2021-12-03,1.1,0.0,30738.75\nR001,2021-12-04,4.7,0.0,29987.94\nR001,2021-12-05,6.1,2.3,12647.05\nR001,2021-12-06,7.5,0.0,27126.53\nR001,2021-12-07,8.1,0.0,30401.58\nR001,2021-12-08,8.2,0.3,26923.17\nR001,2021-12-09,6.4,3.2,2366.23\nR001,2021-12-10,4.3,0.0,30419.74\nR001,2021-12-11,5.3,0.0,30035.43\nR001,2021-12-12,6.5,0.0,16840.54\nR001,2021-12-13,7.8,0.0,30424.4\nR001,2021-12-14,7.9,0.0,10384.96\nR001,2021-12-15,8.1,0.0,30419.45\nR001,2021-12-16,7.1,0.0,29634.78\nR001,2021-12-17,3.2,12.4,1199.75\nR001,2021-12-18,1.8,14.0,16431.3\nR001,2021-12-19,-1.4,0.0,30779.13\nR001,2021-12-20,1.1,19.2,226.51\nR001,2021-12-21,-2.1,13.4,5771.96\nR001,2021-12-22,-8.6,4.7,22331.53\nR001,2021-12-23,-10.0,1.7,29157.76\nR001,2021-12-24,-9.9,0.0,28800.0\nR001,2021-12-25,-2.2,6.7,0.0\nR001,2021-12-26,2.8,0.7,26665.02\nR001,2021-12-27,4.6,4.8,754.78\nR001,2021-12-28,4.9,0.3,28996.53\nR001,2021-12-29,2.7,0.0,31464.19\nR001,2021-12-30,1.1,0.0,28883.3\nR001,2021-12-31,0.0,3.7,25200.0\nR001,2022-01-01,2.4,11.9,5003.13\nR001,2022-01-02,0.1,23.8,17887.24\nR001,2022-01-03,-3.2,1.6,2033.9\nR001,2022-01-04,0.9,27.0,7657.93\nR001,2022-01-05,2.2,1.7,11065.54\nR001,2022-01-06,7.0,0.0,31892.58\nR001,2022-01-07,6.9,0.0,26122.77\nR001,2022-01-08,4.1,0.0,32019.91\nR001,2022-01-09,3.3,0.0,32086.38\nR001,2022-01-10,3.9,0.8,23034.47\nR001,2022-01-11,5.9,3.7,32224.22\nR001,2022-01-12,3.9,26.9,26.32\nR001,2022-01-13,-1.7,14.3,19887.02\nR001,2022-01-14,-5.2,5.5,26674.2\nR001,2022-01-15,-4.1,25.6,19941.76\nR001,2022-01-16,-5.9,0.3,32400.0\nR001,2022-01-17,-7.1,0.0,32400.0\nR001,2022-01-18,-2.9,20.8,0.0\nR001,2022-01-19,-5.7,2.7,25200.0\nR001,2022-01-20,-6.9,3.9,27884.08\nR001,2022-01-21,-2.5,3.3,24966.86\nR001,2022-01-22,0.5,21.6,0.0\nR001,2022-01-23,1.8,12.5,8977.94\nR001,2022-01-24,0.3,10.6,14506.49\nR001,2022-01-25,-3.5,3.2,30726.2\nR001,2022-01-26,-3.3,5.6,400.09\nR001,2022-01-27,-4.1,2.9,32400.0\nR001,2022-01-28,-4.3,1.8,32400.0\nR001,2022-01-29,-3.2,3.8,23149.85\nR001,2022-01-30,-2.1,12.0,0.0\nR001,2022-01-31,-1.0,12.6,0.0\nR001,2022-02-01,-0.4,1.8,32400.0\nR001,2022-02-02,3.9,0.0,31315.89\nR001,2022-02-03,5.0,0.0,32400.0\nR001,2022-02-04,1.9,12.9,11917.76\nR001,2022-02-05,-5.1,5.7,33233.86\nR001,2022-02-06,-4.4,13.5,1174.74\nR001,2022-02-07,-3.3,0.0,25755.94\nR001,2022-02-08,1.3,4.8,5706.28\nR001,2022-02-09,2.6,37.9,0.0\nR001,2022-02-10,0.7,3.1,34373.52\nR001,2022-02-11,-0.1,0.0,34745.99\nR001,2022-02-12,2.0,0.0,36183.16\nR001,2022-02-13,0.6,2.1,33887.56\nR001,2022-02-14,0.9,0.0,33320.67\nR001,2022-02-15,1.0,0.0,36621.52\nR001,2022-02-16,-0.6,0.0,37406.06\nR001,2022-02-17,0.6,0.0,37593.88\nR001,2022-02-18,3.9,0.0,34348.36\nR001,2022-02-19,2.3,5.3,5944.77\nR001,2022-02-20,3.0,0.1,38105.09\nR001,2022-02-21,2.9,6.6,12147.34\nR001,2022-02-22,4.7,0.0,38468.87\nR001,2022-02-23,8.8,0.3,33387.49\nR001,2022-02-24,7.7,0.0,28989.34\nR001,2022-02-25,6.3,4.8,342.06\nR001,2022-02-26,4.3,6.2,25621.65\nR001,2022-02-27,2.8,2.1,37279.64\nR001,2022-02-28,3.1,1.4,16724.85\nR001,2022-03-01,3.3,14.9,8749.66\nR001,2022-03-02,3.6,2.7,30482.87\nR001,2022-03-03,4.0,14.2,17627.53\nR001,2022-03-04,-0.1,9.3,27205.67\nR001,2022-03-05,-0.7,14.7,20711.09\nR001,2022-03-06,-2.1,2.1,39600.0\nR001,2022-03-07,-0.9,0.0,39600.0\nR001,2022-03-08,-0.6,7.9,9225.65\nR001,2022-03-09,-2.0,5.4,2045.31\nR001,2022-03-10,-2.6,0.8,32400.0\nR001,2022-03-11,-2.3,4.5,9735.03\nR001,2022-03-12,-3.5,4.4,39600.0\nR001,2022-03-13,-4.0,1.1,38460.04\nR001,2022-03-14,-3.8,0.0,39600.0\nR001,2022-03-15,-3.8,0.2,39600.0\nR001,2022-03-16,-4.0,7.3,29065.94\nR001,2022-03-17,-6.0,3.5,34770.39\nR001,2022-03-18,-6.0,2.2,38852.52\nR001,2022-03-19,-3.1,0.6,30965.91\nR001,2022-03-20,-3.2,0.0,40047.0\nR001,2022-03-21,-1.8,0.0,41545.03\nR001,2022-03-22,-0.3,0.0,41772.53\nR001,2022-03-23,1.2,0.0,42258.98\nR001,2022-03-24,1.1,5.6,4394.05\nR001,2022-03-25,0.4,3.4,34075.74\nR001,2022-03-26,0.5,4.0,34152.6\nR001,2022-03-27,0.2,13.7,24541.95\nR001,2022-03-28,-1.8,3.0,43147.78\nR001,2022-03-29,4.4,0.0,43641.44\nR001,2022-03-30,7.3,1.0,15268.39\nR001,2022-03-31,7.8,0.0,44425.79\nR001,2022-04-01,11.7,0.6,34576.2\nR001,2022-04-02,14.1,0.3,38313.95\nR001,2022-04-03,10.3,5.1,40535.49\nR001,2022-04-04,4.1,9.3,29841.65\nR001,2022-04-05,3.0,8.0,11419.28\nR001,2022-04-06,5.6,1.7,36399.49\nR001,2022-04-07,6.2,0.0,46083.95\nR001,2022-04-08,6.3,8.8,14931.38\nR001,2022-04-09,7.6,3.0,44072.67\nR001,2022-04-10,10.2,0.0,38061.51\nR001,2022-04-11,12.3,0.3,46369.01\nR001,2022-04-12,4.4,33.3,0.0\nR001,2022-04-13,1.0,9.3,17283.83\nR001,2022-04-14,0.7,9.9,9847.76\nR001,2022-04-15,3.9,3.3,19735.97\nR001,2022-04-16,5.5,0.0,46724.35\nR001,2022-04-17,9.2,0.0,46795.71\nR001,2022-04-18,15.7,1.2,38703.98\nR001,2022-04-19,13.2,7.5,28428.85\nR001,2022-04-20,7.2,21.7,28800.0\nR001,2022-04-21,6.8,3.3,46800.0\nR001,2022-04-22,7.9,0.3,46800.0\nR001,2022-04-23,10.7,0.0,47172.42\nR001,2022-04-24,14.7,0.4,47458.43\nR001,2022-04-25,15.8,0.0,47415.24\nR001,2022-04-26,17.0,0.0,47704.49\nR001,2022-04-27,15.2,0.9,47663.34\nR001,2022-04-28,14.4,1.3,44830.24\nR001,2022-04-29,12.8,2.0,44454.74\nR001,2022-04-30,12.2,1.3,46689.63\nR001,2022-05-01,11.2,3.8,15363.2\nR001,2022-05-02,8.3,13.1,0.0\nR001,2022-05-03,9.7,0.4,44834.3\nR001,2022-05-04,10.5,1.4,37743.62\nR001,2022-05-05,7.8,20.9,0.0\nR001,2022-05-06,8.7,1.9,20582.83\nR001,2022-05-07,7.4,6.9,0.0\nR001,2022-05-08,8.5,5.8,12993.38\nR001,2022-05-09,9.2,4.1,40700.04\nR001,2022-05-10,7.5,4.6,34074.17\nR001,2022-05-11,6.4,3.6,41895.54\nR001,2022-05-12,6.9,0.2,48512.21\nR001,2022-05-13,10.7,0.0,50319.83\nR001,2022-05-14,14.4,0.0,50386.69\nR001,2022-05-15,14.6,0.0,50481.26\nR001,2022-05-16,11.0,1.1,47537.0\nR001,2022-05-17,10.0,0.5,50400.0\nR001,2022-05-18,12.0,0.0,49897.25\nR001,2022-05-19,8.1,20.1,29146.59\nR001,2022-05-20,5.8,6.8,32020.04\nR001,2022-05-21,9.8,0.0,50131.64\nR001,2022-05-22,14.1,0.0,50993.88\nR001,2022-05-23,10.4,1.7,33720.3\nR001,2022-05-24,8.0,3.9,41764.23\nR001,2022-05-25,9.2,4.9,43279.79\nR001,2022-05-26,11.0,0.4,49052.55\nR001,2022-05-27,13.0,0.0,47170.14\nR001,2022-05-28,15.8,0.0,51567.6\nR001,2022-05-29,18.1,0.1,51545.58\nR001,2022-05-30,19.5,0.5,51885.08\nR001,2022-05-31,20.6,1.8,50989.42\nR001,2022-06-01,19.5,2.0,51119.7\nR001,2022-06-02,19.9,0.4,52187.47\nR001,2022-06-03,20.1,0.6,50400.0\nR001,2022-06-04,22.3,0.9,51786.9\nR001,2022-06-05,20.7,4.0,45183.74\nR001,2022-06-06,18.6,3.4,48896.3\nR001,2022-06-07,17.8,1.5,49809.47\nR001,2022-06-08,18.0,5.5,50794.36\nR001,2022-06-09,16.2,12.4,49268.82\nR001,2022-06-10,16.8,6.3,44879.23\nR001,2022-06-11,17.9,1.6,53142.69\nR001,2022-06-12,18.9,1.8,52030.54\nR001,2022-06-13,17.1,3.4,49188.73\nR001,2022-06-14,16.2,11.6,32612.49\nR001,2022-06-15,16.1,10.8,47914.55\nR001,2022-06-16,16.6,5.0,49315.49\nR001,2022-06-17,17.1,1.1,53199.48\nR001,2022-06-18,17.7,0.2,53069.71\nR001,2022-06-19,18.3,1.3,53061.11\nR001,2022-06-20,19.6,0.5,53919.8\nR001,2022-06-21,19.3,1.7,52310.24\nR001,2022-06-22,16.8,13.7,51017.51\nR001,2022-06-23,13.0,8.9,42707.03\nR001,2022-06-24,12.4,4.5,13042.77\nR001,2022-06-25,15.2,4.8,45920.06\nR001,2022-06-26,14.8,25.2,0.0\nR001,2022-06-27,16.9,14.5,42182.44\nR001,2022-06-28,18.3,4.0,49778.21\nR001,2022-06-29,19.1,2.4,47435.51\nR001,2022-06-30,19.9,2.5,52441.15\nR001,2022-07-01,19.0,2.2,52477.92\nR001,2022-07-02,17.9,0.0,53597.33\nR001,2022-07-03,16.0,0.2,53927.94\nR001,2022-07-04,16.7,0.4,53912.18\nR001,2022-07-05,18.5,0.0,53894.25\nR001,2022-07-06,20.3,0.1,53874.17\nR001,2022-07-07,20.8,0.9,52304.58\nR001,2022-07-08,19.5,3.2,52313.47\nR001,2022-07-09,19.9,4.0,53801.15\nR001,2022-07-10,21.0,4.4,52688.83\nR001,2022-07-11,18.7,18.1,30508.51\nR001,2022-07-12,18.6,4.9,38008.86\nR001,2022-07-13,19.2,6.7,34837.51\nR001,2022-07-14,18.8,1.3,45965.85\nR001,2022-07-15,19.8,0.0,52639.84\nR001,2022-07-16,20.1,4.9,41671.31\nR001,2022-07-17,20.6,2.5,44261.15\nR001,2022-07-18,19.3,6.3,27246.58\nR001,2022-07-19,18.3,11.5,10373.08\nR001,2022-07-20,18.7,10.4,25331.69\nR001,2022-07-21,17.6,12.2,42886.95\nR001,2022-07-22,15.3,2.3,51752.99\nR001,2022-07-23,15.0,6.9,46159.78\nR001,2022-07-24,14.9,13.0,48193.26\nR001,2022-07-25,16.3,1.7,47603.89\nR001,2022-07-26,17.6,5.5,45998.25\nR001,2022-07-27,17.7,1.5,47257.05\nR001,2022-07-28,18.6,0.5,48558.73\nR001,2022-07-29,20.0,0.2,50659.42\nR001,2022-07-30,21.5,0.6,49671.04\nR001,2022-07-31,21.9,0.6,50415.69\nR001,2022-08-01,21.5,3.7,49779.09\nR001,2022-08-02,20.3,3.9,46791.71\nR001,2022-08-03,20.7,4.8,46556.31\nR001,2022-08-04,20.8,1.3,48921.99\nR001,2022-08-05,20.0,1.3,30835.31\nR001,2022-08-06,19.5,1.0,46894.21\nR001,2022-08-07,18.4,1.4,47954.97\nR001,2022-08-08,19.7,0.6,47304.3\nR001,2022-08-09,20.2,0.0,48347.84\nR001,2022-08-10,21.1,1.3,48036.97\nR001,2022-08-11,20.6,10.7,45959.63\nR001,2022-08-12,20.0,2.7,46800.0\nR001,2022-08-13,21.3,1.1,46800.0\nR001,2022-08-14,21.2,5.1,47116.38\nR001,2022-08-15,22.5,2.0,46800.0\nR001,2022-08-16,22.3,5.2,44427.03\nR001,2022-08-17,21.8,2.6,46412.81\nR001,2022-08-18,21.8,9.1,43286.41\nR001,2022-08-19,20.6,8.0,46069.97\nR001,2022-08-20,20.7,3.6,44069.67\nR001,2022-08-21,22.2,1.5,46718.75\nR001,2022-08-22,23.5,0.3,46800.0\nR001,2022-08-23,23.2,2.0,46800.0\nR001,2022-08-24,23.5,0.2,46800.0\nR001,2022-08-25,24.8,0.4,46800.0\nR001,2022-08-26,24.1,0.2,46800.0\nR001,2022-08-27,23.5,0.8,46734.77\nR001,2022-08-28,22.5,0.9,46633.4\nR001,2022-08-29,20.3,4.5,45407.94\nR001,2022-08-30,21.1,0.2,46430.2\nR001,2022-08-31,22.5,0.1,46328.24\nR001,2022-09-01,23.6,0.1,45224.8\nR001,2022-09-02,23.5,0.7,43988.04\nR001,2022-09-03,24.2,0.0,45642.3\nR001,2022-09-04,19.9,5.1,38568.52\nR001,2022-09-05,16.3,3.2,38241.63\nR001,2022-09-06,14.0,1.0,41104.15\nR001,2022-09-07,11.8,0.7,41849.37\nR001,2022-09-08,12.4,0.2,43659.03\nR001,2022-09-09,12.7,0.0,43712.68\nR001,2022-09-10,14.7,0.0,43447.88\nR001,2022-09-11,17.2,0.0,43200.0\nR001,2022-09-12,18.5,0.5,43200.0\nR001,2022-09-13,15.2,12.5,26503.09\nR001,2022-09-14,12.8,0.8,38081.1\nR001,2022-09-15,13.7,0.4,29112.28\nR001,2022-09-16,17.2,2.2,42583.69\nR001,2022-09-17,20.9,0.0,42084.37\nR001,2022-09-18,23.6,0.1,42839.8\nR001,2022-09-19,18.0,18.8,35229.83\nR001,2022-09-20,16.9,13.2,31111.74\nR001,2022-09-21,15.1,39.5,1681.95\nR001,2022-09-22,14.1,61.4,1002.53\nR001,2022-09-23,11.7,36.7,21600.0\nR001,2022-09-24,8.7,11.3,432.67\nR001,2022-09-25,8.8,3.5,39811.92\nR001,2022-09-26,11.3,0.0,42079.0\nR001,2022-09-27,15.6,0.0,34240.42\nR001,2022-09-28,17.6,0.4,33515.76\nR001,2022-09-29,15.5,7.6,36771.53\nR001,2022-09-30,15.8,0.0,40806.29\nR001,2022-10-01,16.2,0.0,40413.23\nR001,2022-10-02,16.9,0.5,33197.76\nR001,2022-10-03,13.9,20.0,36915.43\nR001,2022-10-04,13.4,13.0,0.0\nR001,2022-10-05,12.8,38.1,4013.59\nR001,2022-10-06,13.2,1.3,32095.29\nR001,2022-10-07,13.9,0.0,37108.9\nR001,2022-10-08,17.2,0.2,30045.05\nR001,2022-10-09,14.5,7.3,3295.26\nR001,2022-10-10,12.2,10.7,22348.49\nR001,2022-10-11,8.7,1.7,35235.13\nR001,2022-10-12,8.1,0.0,36535.17\nR001,2022-10-13,8.9,0.0,36000.0\nR001,2022-10-14,8.9,0.2,36000.0\nR001,2022-10-15,10.2,0.2,35038.46\nR001,2022-10-16,11.4,0.3,31940.16\nR001,2022-10-17,11.6,3.9,1268.56\nR001,2022-10-18,11.3,0.0,36000.0\nR001,2022-10-19,13.1,0.0,36000.0\nR001,2022-10-20,9.9,23.5,19840.45\nR001,2022-10-21,5.0,0.3,36000.0\nR001,2022-10-22,5.3,0.0,36000.0\nR001,2022-10-23,7.9,1.1,31717.76\nR001,2022-10-24,8.0,9.5,24006.04\nR001,2022-10-25,6.9,0.0,34487.92\nR001,2022-10-26,9.5,0.1,27031.01\nR001,2022-10-27,8.3,9.8,31761.39\nR001,2022-10-28,6.5,1.5,35695.13\nR001,2022-10-29,6.4,0.0,36000.0\nR001,2022-10-30,7.1,0.0,34679.21\nR001,2022-10-31,6.8,23.2,0.0\nR001,2022-11-01,4.6,0.2,33682.36\nR001,2022-11-02,3.7,2.9,28991.47\nR001,2022-11-03,6.7,9.4,24749.21\nR001,2022-11-04,6.7,8.9,29559.57\nR001,2022-11-05,5.6,0.0,34547.5\nR001,2022-11-06,3.8,0.0,35546.75\nR001,2022-11-07,5.8,0.0,35391.78\nR001,2022-11-08,6.1,0.0,35238.26\nR001,2022-11-09,6.1,1.5,19545.0\nR001,2022-11-10,6.2,0.3,34143.08\nR001,2022-11-11,5.5,0.3,34125.47\nR001,2022-11-12,4.9,0.0,34642.34\nR001,2022-11-13,5.8,0.0,34498.87\nR001,2022-11-14,5.0,1.2,33976.24\nR001,2022-11-15,3.6,0.0,33711.17\nR001,2022-11-16,6.5,0.8,30524.45\nR001,2022-11-17,6.9,1.6,31876.8\nR001,2022-11-18,8.2,6.6,18201.24\nR001,2022-11-19,9.8,3.6,20630.74\nR001,2022-11-20,11.7,3.9,29567.52\nR001,2022-11-21,12.3,0.0,9756.55\nR001,2022-11-22,12.3,3.4,16426.7\nR001,2022-11-23,8.8,1.4,30224.59\nR001,2022-11-24,10.2,0.0,30169.74\nR001,2022-11-25,11.0,0.3,26033.92\nR001,2022-11-26,8.4,7.7,12022.78\nR001,2022-11-27,7.4,10.7,17689.27\nR001,2022-11-28,5.2,0.0,30265.06\nR001,2022-11-29,6.0,0.0,27623.52\nR001,2022-11-30,7.5,0.0,30737.1\nR001,2022-12-01,7.9,0.0,30412.36\nR001,2022-12-02,7.1,3.3,2571.99\nR001,2022-12-03,6.6,3.5,4099.91\nR001,2022-12-04,5.6,2.5,21651.62\nR001,2022-12-05,5.1,2.2,26258.85\nR001,2022-12-06,3.2,0.1,29241.35\nR001,2022-12-07,2.6,0.0,29633.18\nR001,2022-12-08,4.0,0.0,21174.52\nR001,2022-12-09,5.7,0.0,26937.54\nR001,2022-12-10,6.7,0.0,30034.45\nR001,2022-12-11,6.8,0.0,30036.69\nR001,2022-12-12,7.9,2.7,12611.85\nR001,2022-12-13,7.3,1.5,25659.69\nR001,2022-12-14,5.6,23.4,0.0\nR001,2022-12-15,7.4,2.4,26318.17\nR001,2022-12-16,7.6,4.2,21771.35\nR001,2022-12-17,7.3,0.0,30406.42\nR001,2022-12-18,8.5,0.5,28836.74\nR001,2022-12-19,5.1,18.0,302.75\nR001,2022-12-20,-0.2,0.1,30376.85\nR001,2022-12-21,0.4,0.0,25757.57\nR001,2022-12-22,2.7,0.0,31132.41\nR001,2022-12-23,3.0,0.0,31117.14\nR001,2022-12-24,1.4,0.0,31244.4\nR001,2022-12-25,1.1,0.0,30690.98\nR001,2022-12-26,2.1,0.4,21204.77\nR001,2022-12-27,2.5,1.0,19493.08\nR001,2022-12-28,1.5,16.3,21084.06\nR001,2022-12-29,-3.8,0.0,31444.45\nR001,2022-12-30,-0.1,0.0,31490.63\nR001,2022-12-31,1.2,0.0,30945.07\nR001,2023-01-01,1.9,0.0,30916.53\nR001,2023-01-02,4.2,0.0,31644.71\nR001,2023-01-03,6.7,0.0,31700.78\nR001,2023-01-04,4.7,1.1,31758.98\nR001,2023-01-05,1.5,0.8,31819.15\nR001,2023-01-06,1.6,28.5,0.0\nR001,2023-01-07,-2.7,8.5,11073.83\nR001,2023-01-08,-7.0,0.2,29515.95\nR001,2023-01-09,-8.6,0.1,32076.86\nR001,2023-01-10,-5.8,0.0,32144.83\nR001,2023-01-11,-1.3,0.0,32213.94\nR001,2023-01-12,-0.6,0.0,32284.04\nR001,2023-01-13,-0.6,0.0,32355.0\nR001,2023-01-14,-2.5,0.0,32400.0\nR001,2023-01-15,-0.7,0.0,32400.0\nR001,2023-01-16,1.1,0.0,32400.0\nR001,2023-01-17,3.0,0.0,32354.67\nR001,2023-01-18,5.2,0.0,32400.0\nR001,2023-01-19,6.4,0.0,32400.0\nR001,2023-01-20,7.2,0.0,32400.0\nR001,2023-01-21,8.1,0.0,32400.0\nR001,2023-01-22,5.4,0.0,30730.23\nR001,2023-01-23,3.6,0.0,32400.0\nR001,2023-01-24,1.8,0.0,32400.0\nR001,2023-01-25,0.3,0.0,32400.0\nR001,2023-01-26,0.0,0.0,32400.0\nR001,2023-01-27,1.3,0.0,32400.0\nR001,2023-01-28,3.9,1.9,3866.23\nR001,2023-01-29,3.5,3.7,32213.55\nR001,2023-01-30,1.6,5.0,29829.12\nR001,2023-01-31,1.0,0.3,31846.57\nR001,2023-02-01,1.0,4.1,21182.22\nR001,2023-02-02,-0.8,7.1,7597.27\nR001,2023-02-03,-1.7,11.2,28406.11\nR001,2023-02-04,-2.1,7.6,3785.18\nR001,2023-02-05,-2.8,0.7,22481.32\nR001,2023-02-06,-3.3,2.6,15979.44\nR001,2023-02-07,-2.7,10.4,1599.47\nR001,2023-02-08,-3.9,3.9,18189.61\nR001,2023-02-09,-4.3,2.0,33584.8\nR001,2023-02-10,-2.9,2.1,26589.68\nR001,2023-02-11,-2.3,2.2,33406.24\nR001,2023-02-12,-2.9,3.1,31804.8\nR001,2023-02-13,-5.4,14.1,34843.65\nR001,2023-02-14,-4.4,5.5,24629.76\nR001,2023-02-15,-4.3,14.7,12048.6\nR001,2023-02-16,-7.0,0.3,35629.77\nR001,2023-02-17,-5.2,6.0,0.0\nR001,2023-02-18,0.7,12.6,636.67\nR001,2023-02-19,2.0,33.7,0.0\nR001,2023-02-20,1.9,36.4,0.0\nR001,2023-02-21,-2.0,20.9,9222.01\nR001,2023-02-22,0.6,21.4,23345.85\nR001,2023-02-23,-3.1,1.4,36511.05\nR001,2023-02-24,-3.9,0.0,38783.98\nR001,2023-02-25,3.9,0.0,36895.71\nR001,2023-02-26,7.3,0.0,38571.62\nR001,2023-02-27,9.0,0.0,36251.2\nR001,2023-02-28,9.4,0.3,34572.45\nR001,2023-03-01,5.8,11.0,0.0\nR001,2023-03-02,3.1,0.0,38070.88\nR001,2023-03-03,2.7,3.0,28800.0\nR001,2023-03-04,3.7,15.7,18869.94\nR001,2023-03-05,4.4,1.4,18787.17\nR001,2023-03-06,3.2,32.1,4047.46\nR001,2023-03-07,2.3,0.0,39600.0\nR001,2023-03-08,5.8,0.0,39600.0\nR001,2023-03-09,7.5,0.8,39600.0\nR001,2023-03-10,5.9,7.5,15913.26\nR001,2023-03-11,7.2,3.6,23504.18\nR001,2023-03-12,9.8,8.6,33752.39\nR001,2023-03-13,4.7,26.6,612.86\nR001,2023-03-14,6.3,2.2,12365.11\nR001,2023-03-15,9.4,0.2,39600.0\nR001,2023-03-16,8.9,0.8,34743.56\nR001,2023-03-17,8.4,4.6,31237.36\nR001,2023-03-18,8.0,6.0,10380.62\nR001,2023-03-19,8.3,0.0,39600.0\nR001,2023-03-20,7.0,5.5,19850.44\nR001,2023-03-21,6.4,3.3,8053.24\nR001,2023-03-22,8.2,0.0,38894.45\nR001,2023-03-23,8.4,2.4,24986.17\nR001,2023-03-24,7.4,5.4,35154.49\nR001,2023-03-25,8.1,0.6,33637.87\nR001,2023-03-26,9.3,13.2,35637.51\nR001,2023-03-27,6.1,4.3,25917.92\nR001,2023-03-28,9.1,4.7,42403.55\nR001,2023-03-29,4.4,32.5,220.96\nR001,2023-03-30,-0.9,8.5,39164.32\nR001,2023-03-31,-1.8,0.0,43858.85\nR001,2023-04-01,2.6,0.0,45127.33\nR001,2023-04-02,7.4,0.0,42944.53\nR001,2023-04-03,8.9,0.6,22773.54\nR001,2023-04-04,11.8,0.0,42317.98\nR001,2023-04-05,11.4,3.4,29225.3\nR001,2023-04-06,6.7,7.9,7918.06\nR001,2023-04-07,7.6,4.4,29667.48\nR001,2023-04-08,6.0,8.2,4732.39\nR001,2023-04-09,8.8,0.1,43622.8\nR001,2023-04-10,9.2,2.4,27235.47\nR001,2023-04-11,10.7,12.9,11590.59\nR001,2023-04-12,10.1,11.4,15716.63\nR001,2023-04-13,7.3,20.9,28820.1\nR001,2023-04-14,6.0,13.2,42964.21\nR001,2023-04-15,8.8,0.0,46635.81\nR001,2023-04-16,12.4,0.3,46707.41\nR001,2023-04-17,13.9,2.5,44817.96\nR001,2023-04-18,14.6,1.5,40305.14\nR001,2023-04-19,15.1,1.7,46800.0\nR001,2023-04-20,10.6,19.7,5526.16\nR001,2023-04-21,7.7,16.3,28059.83\nR001,2023-04-22,5.9,17.6,28337.67\nR001,2023-04-23,8.4,0.0,44107.92\nR001,2023-04-24,8.3,12.9,5694.97\nR001,2023-04-25,8.6,4.7,43385.72\nR001,2023-04-26,11.7,0.3,47050.14\nR001,2023-04-27,14.2,2.1,42600.92\nR001,2023-04-28,11.8,11.3,33910.6\nR001,2023-04-29,9.4,13.5,3181.06\nR001,2023-04-30,8.4,9.2,30043.6\nR001,2023-05-01,7.2,3.2,42496.62\nR001,2023-05-02,6.4,7.3,38861.95\nR001,2023-05-03,8.6,1.2,46786.4\nR001,2023-05-04,11.2,0.0,48757.53\nR001,2023-05-05,12.9,0.0,47372.32\nR001,2023-05-06,14.1,0.1,33901.77\nR001,2023-05-07,11.7,11.4,16528.77\nR001,2023-05-08,9.6,9.5,7924.95\nR001,2023-05-09,9.5,3.5,23226.64\nR001,2023-05-10,8.4,11.6,722.9\nR001,2023-05-11,8.2,8.0,30045.29\nR001,2023-05-12,11.4,0.4,40881.17\nR001,2023-05-13,11.7,17.6,1368.11\nR001,2023-05-14,9.5,15.6,10531.02\nR001,2023-05-15,10.6,0.7,48383.89\nR001,2023-05-16,12.4,0.3,50400.0\nR001,2023-05-17,13.2,0.0,50431.02\nR001,2023-05-18,14.0,0.0,50400.0\nR001,2023-05-19,14.9,2.3,48292.39\nR001,2023-05-20,15.0,5.3,44155.95\nR001,2023-05-21,13.7,15.0,28206.03\nR001,2023-05-22,14.9,9.8,47256.82\nR001,2023-05-23,17.0,4.7,50400.0\nR001,2023-05-24,17.6,4.3,50593.18\nR001,2023-05-25,17.2,3.3,48034.32\nR001,2023-05-26,15.5,7.7,50315.92\nR001,2023-05-27,15.9,2.9,49349.71\nR001,2023-05-28,17.3,3.1,44947.7\nR001,2023-05-29,17.2,5.4,34600.05\nR001,2023-05-30,16.1,16.4,28810.35\nR001,2023-05-31,15.5,4.4,48570.64\nR001,2023-06-01,15.1,5.4,45087.41\nR001,2023-06-02,15.9,0.5,51828.12\nR001,2023-06-03,18.4,1.2,51082.81\nR001,2023-06-04,18.0,8.6,39917.93\nR001,2023-06-05,16.7,5.5,41158.73\nR001,2023-06-06,15.9,4.0,30611.3\nR001,2023-06-07,16.8,3.2,46834.46\nR001,2023-06-08,17.8,0.1,51474.59\nR001,2023-06-09,19.3,1.9,39493.43\nR001,2023-06-10,17.2,19.2,23298.26\nR001,2023-06-11,18.4,3.8,47246.72\nR001,2023-06-12,18.0,9.4,37794.64\nR001,2023-06-13,17.8,19.9,43449.76\nR001,2023-06-14,17.5,19.4,29833.73\nR001,2023-06-15,17.2,13.6,42236.06\nR001,2023-06-16,17.3,3.6,46843.71\nR001,2023-06-17,18.1,0.4,50555.93\nR001,2023-06-18,18.3,2.9,53069.97\nR001,2023-06-19,17.0,14.5,47121.59\nR001,2023-06-20,17.0,9.5,38456.79\nR001,2023-06-21,17.8,0.9,49724.68\nR001,2023-06-22,17.6,0.7,46759.62\nR001,2023-06-23,18.0,1.0,53400.23\nR001,2023-06-24,18.6,5.0,51925.03\nR001,2023-06-25,20.2,0.2,53969.76\nR001,2023-06-26,20.7,2.9,53972.21\nR001,2023-06-27,18.7,5.0,46579.64\nR001,2023-06-28,18.9,0.4,53749.01\nR001,2023-06-29,18.9,2.7,53100.44\nR001,2023-06-30,15.6,1.2,52165.43\nR001,2023-07-01,15.9,1.2,53951.18\nR001,2023-07-02,17.6,0.3,50383.87\nR001,2023-07-03,19.9,0.1,53928.15\nR001,2023-07-04,21.5,0.6,53913.48\nR001,2023-07-05,22.0,0.5,53896.71\nR001,2023-07-06,23.1,0.0,53877.81\nR001,2023-07-07,22.5,14.4,47180.99\nR001,2023-07-08,20.0,12.8,35444.05\nR001,2023-07-09,17.4,17.9,3093.8\nR001,2023-07-10,16.8,33.6,8132.72\nR001,2023-07-11,14.5,3.4,48962.62\nR001,2023-07-12,15.5,1.0,50272.58\nR001,2023-07-13,18.7,0.0,53686.19\nR001,2023-07-14,19.8,0.0,52877.69\nR001,2023-07-15,19.6,1.1,49458.51\nR001,2023-07-16,17.5,0.2,53230.54\nR001,2023-07-17,17.0,0.0,51001.09\nR001,2023-07-18,17.5,0.4,52420.71\nR001,2023-07-19,17.8,0.7,52436.97\nR001,2023-07-20,19.1,0.5,50959.96\nR001,2023-07-21,20.2,2.0,51331.65\nR001,2023-07-22,21.4,1.1,48920.52\nR001,2023-07-23,21.2,1.0,50019.21\nR001,2023-07-24,20.3,4.7,46745.0\nR001,2023-07-25,19.7,0.6,49761.45\nR001,2023-07-26,20.9,0.1,51502.37\nR001,2023-07-27,24.2,0.2,51815.18\nR001,2023-07-28,18.1,19.7,29066.59\nR001,2023-07-29,17.0,0.2,48219.53\nR001,2023-07-30,18.8,0.4,47389.43\nR001,2023-07-31,20.4,0.7,50058.78\nR001,2023-08-01,20.8,1.4,48348.67\nR001,2023-08-02,21.3,1.1,46821.87\nR001,2023-08-03,20.4,6.9,47134.48\nR001,2023-08-04,19.9,0.2,47879.28\nR001,2023-08-05,22.0,0.0,48282.71\nR001,2023-08-06,25.3,0.0,48684.65\nR001,2023-08-07,25.2,0.2,48406.69\nR001,2023-08-08,24.2,0.6,47295.47\nR001,2023-08-09,23.7,1.2,47324.17\nR001,2023-08-10,24.5,1.4,47690.74\nR001,2023-08-11,24.8,0.8,47382.19\nR001,2023-08-12,23.5,3.5,42967.48\nR001,2023-08-13,24.1,1.6,46180.28\nR001,2023-08-14,24.0,0.5,46800.0\nR001,2023-08-15,24.4,0.5,46832.6\nR001,2023-08-16,24.5,0.0,47197.06\nR001,2023-08-17,24.2,0.0,47227.74\nR001,2023-08-18,22.9,0.4,46926.88\nR001,2023-08-19,21.8,1.2,46800.0\nR001,2023-08-20,22.2,0.0,46800.0\nR001,2023-08-21,22.0,0.3,46800.0\nR001,2023-08-22,21.5,4.5,46800.0\nR001,2023-08-23,20.7,0.0,46800.0\nR001,2023-08-24,20.4,0.0,46800.0\nR001,2023-08-25,18.6,0.3,46800.0\nR001,2023-08-26,18.8,0.0,46800.0\nR001,2023-08-27,19.6,0.0,46767.13\nR001,2023-08-28,20.8,0.0,46665.97\nR001,2023-08-29,20.9,0.0,46563.54\nR001,2023-08-30,21.1,0.5,46407.65\nR001,2023-08-31,23.1,0.0,46355.35\nR001,2023-09-01,23.4,0.0,45213.47\nR001,2023-09-02,21.6,0.4,44619.74\nR001,2023-09-03,18.7,9.7,6657.78\nR001,2023-09-04,19.0,14.0,27258.42\nR001,2023-09-05,20.3,1.4,43810.87\nR001,2023-09-06,20.9,1.2,43223.45\nR001,2023-09-07,19.3,5.6,34049.55\nR001,2023-09-08,18.1,0.3,40902.15\nR001,2023-09-09,15.0,6.7,35486.52\nR001,2023-09-10,12.2,0.7,42934.92\nR001,2023-09-11,12.4,1.6,41776.86\nR001,2023-09-12,13.4,1.4,41502.59\nR001,2023-09-13,13.0,1.7,40778.99\nR001,2023-09-14,13.0,0.4,43074.79\nR001,2023-09-15,14.7,0.0,43058.67\nR001,2023-09-16,14.8,2.0,40996.1\nR001,2023-09-17,14.8,0.0,38179.04\nR001,2023-09-18,14.0,0.0,42856.64\nR001,2023-09-19,14.9,0.0,42789.0\nR001,2023-09-20,16.3,0.0,42721.23\nR001,2023-09-21,17.2,0.0,42529.71\nR001,2023-09-22,18.1,0.3,42564.5\nR001,2023-09-23,18.1,0.2,41007.49\nR001,2023-09-24,18.2,0.4,41679.67\nR001,2023-09-25,16.8,1.2,41396.68\nR001,2023-09-26,16.1,0.3,39332.36\nR001,2023-09-27,17.2,0.0,41149.74\nR001,2023-09-28,17.9,1.9,40650.0\nR001,2023-09-29,17.8,0.3,39361.78\nR001,2023-09-30,17.5,0.3,40498.85\nR001,2023-10-01,15.3,4.6,36473.49\nR001,2023-10-02,14.3,8.5,4745.87\nR001,2023-10-03,13.3,2.3,37400.94\nR001,2023-10-04,12.0,0.8,35940.04\nR001,2023-10-05,12.3,0.1,38944.11\nR001,2023-10-06,11.2,0.0,37163.13\nR001,2023-10-07,11.2,0.2,37229.05\nR001,2023-10-08,12.8,0.0,37954.49\nR001,2023-10-09,7.3,26.2,29468.08\nR001,2023-10-10,4.4,4.8,9633.14\nR001,2023-10-11,5.6,4.7,32922.16\nR001,2023-10-12,5.3,0.0,37173.7\nR001,2023-10-13,8.1,0.0,36903.41\nR001,2023-10-14,10.5,0.0,36631.93\nR001,2023-10-15,12.9,0.0,36359.13\nR001,2023-10-16,13.1,2.2,36000.0\nR001,2023-10-17,7.9,10.2,35963.93\nR001,2023-10-18,6.6,4.3,11043.25\nR001,2023-10-19,8.0,2.8,28800.0\nR001,2023-10-20,10.4,0.0,36000.0\nR001,2023-10-21,14.5,0.0,36000.0\nR001,2023-10-22,15.2,0.0,36000.0\nR001,2023-10-23,15.8,0.0,33457.27\nR001,2023-10-24,14.5,1.8,31561.64\nR001,2023-10-25,11.8,0.5,28800.0\nR001,2023-10-26,13.9,6.0,10058.27\nR001,2023-10-27,14.4,5.1,14400.6\nR001,2023-10-28,15.8,0.7,28409.9\nR001,2023-10-29,14.3,2.4,31581.71\nR001,2023-10-30,12.7,0.0,36000.0\nR001,2023-10-31,14.0,0.0,36000.0\nR001,2023-11-01,15.9,0.0,32400.0\nR001,2023-11-02,14.4,2.7,28321.13\nR001,2023-11-03,12.4,12.6,15553.88\nR001,2023-11-04,12.3,0.0,35803.93\nR001,2023-11-05,14.8,0.2,35723.61\nR001,2023-11-06,12.2,17.2,15022.61\nR001,2023-11-07,9.5,0.4,34914.67\nR001,2023-11-08,9.1,0.0,33830.36\nR001,2023-11-09,9.5,18.6,11240.38\nR001,2023-11-10,4.3,0.0,34966.13\nR001,2023-11-11,7.7,0.0,34820.06\nR001,2023-11-12,13.0,5.1,33764.15\nR001,2023-11-13,10.1,21.3,2482.24\nR001,2023-11-14,8.9,31.3,13788.26\nR001,2023-11-15,7.4,0.3,19432.37\nR001,2023-11-16,7.7,55.7,0.0\nR001,2023-11-17,3.3,1.6,32608.67\nR001,2023-11-18,5.8,0.0,16772.04\nR001,2023-11-19,8.8,31.5,0.0\nR001,2023-11-20,3.9,15.7,1828.56\nR001,2023-11-21,1.1,4.2,22507.73\nR001,2023-11-22,3.3,6.2,24201.32\nR001,2023-11-23,3.5,32.6,15547.48\nR001,2023-11-24,4.3,18.3,0.0\nR001,2023-11-25,8.4,23.5,11086.13\nR001,2023-11-26,10.4,25.1,27874.36\nR001,2023-11-27,-0.8,21.7,11641.47\nR001,2023-11-28,-2.8,7.2,3884.14\nR001,2023-11-29,1.7,3.7,16294.84\nR001,2023-11-30,5.5,20.7,0.0\nR001,2023-12-01,7.5,0.2,24625.98\nR001,2023-12-02,10.6,0.1,26794.11\nR001,2023-12-03,10.4,0.0,27018.45\nR001,2023-12-04,10.5,0.8,29985.78\nR001,2023-12-05,7.2,5.2,15561.41\nR001,2023-12-06,7.3,0.4,19531.55\nR001,2023-12-07,6.4,0.0,29246.63\nR001,2023-12-08,9.0,0.7,25895.46\nR001,2023-12-09,8.9,3.7,0.0\nR001,2023-12-10,6.4,13.7,3592.13\nR001,2023-12-11,6.5,0.8,20537.3\nR001,2023-12-12,6.1,11.8,3026.47\nR001,2023-12-13,3.7,7.7,2128.61\nR001,2023-12-14,3.2,0.0,26567.88\nR001,2023-12-15,7.9,0.0,11009.03\nR001,2023-12-16,8.1,3.0,21925.29\nR001,2023-12-17,1.7,24.8,17889.59\nR001,2023-12-18,-0.3,7.3,5896.79\nR001,2023-12-19,2.2,4.3,13718.96\nR001,2023-12-20,2.7,0.0,30771.12\nR001,2023-12-21,5.5,0.5,20368.91\nR001,2023-12-22,6.6,5.7,4589.43\nR001,2023-12-23,2.0,36.7,367.28\nR001,2023-12-24,-3.2,27.7,4022.06\nR001,2023-12-25,-1.2,22.2,1789.54\nR001,2023-12-26,4.8,14.7,6181.2\nR001,2023-12-27,5.7,6.4,16967.63\nR001,2023-12-28,-1.0,2.0,31386.22\nR001,2023-12-29,-1.5,0.0,31432.22\nR001,2023-12-30,4.4,0.0,31480.58\nR001,2023-12-31,3.6,0.3,26393.9\nR001,2024-01-01,2.8,0.0,31583.92\nR001,2024-01-02,4.9,1.0,31279.63\nR001,2024-01-03,2.4,13.6,20738.57\nR001,2024-01-04,3.1,29.5,0.0\nR001,2024-01-05,2.8,19.9,0.0\nR001,2024-01-06,2.0,1.6,31874.55\nR001,2024-01-07,6.4,0.0,28677.0\nR001,2024-01-08,4.9,22.1,1565.73\nR001,2024-01-09,2.4,12.4,2504.43\nR001,2024-01-10,0.4,0.3,25783.99\nR001,2024-01-11,0.0,3.2,30134.49\nR001,2024-01-12,-1.9,27.1,635.64\nR001,2024-01-13,-8.6,0.9,27295.67\nR001,2024-01-14,-7.6,0.0,32400.0\nR001,2024-01-15,-3.1,0.0,31846.88\nR001,2024-01-16,2.2,17.5,15366.56\nR001,2024-01-17,1.4,7.3,25842.4\nR001,2024-01-18,1.6,0.0,32400.0\nR001,2024-01-19,5.2,5.3,17962.93\nR001,2024-01-20,5.2,12.5,24001.67\nR001,2024-01-21,2.9,39.1,0.0\nR001,2024-01-22,-0.4,13.4,20449.57\nR001,2024-01-23,-2.8,0.6,32400.0\nR001,2024-01-24,-4.0,0.0,32400.0\nR001,2024-01-25,-3.7,0.2,29024.27\nR001,2024-01-26,-0.8,0.7,17230.35\nR001,2024-01-27,-1.4,0.8,26799.41\nR001,2024-01-28,-3.2,2.6,32400.0\nR001,2024-01-29,-3.6,0.1,32400.0\nR001,2024-01-30,-4.1,0.0,32472.53\nR001,2024-01-31,-2.4,0.0,31413.18\nR001,2024-02-01,0.7,0.0,32400.0\nR001,2024-02-02,0.4,4.1,16415.48\nR001,2024-02-03,-0.6,13.4,21753.81\nR001,2024-02-04,-0.2,17.0,5782.66\nR001,2024-02-05,-0.3,68.6,0.0\nR001,2024-02-06,1.4,27.0,5200.43\nR001,2024-02-07,4.7,3.7,28720.7\nR001,2024-02-08,5.3,4.8,22084.37\nR001,2024-02-09,6.5,7.0,28263.02\nR001,2024-02-10,7.1,0.0,32828.95\nR001,2024-02-11,8.7,0.0,19482.15\nR001,2024-02-12,9.8,0.0,32400.0\nR001,2024-02-13,9.1,0.0,25693.12\nR001,2024-02-14,8.8,0.2,29243.03\nR001,2024-02-15,5.5,4.1,12644.43\nR001,2024-02-16,5.5,3.2,28713.09\nR001,2024-02-17,4.5,5.2,28939.5\nR001,2024-02-18,3.6,5.5,30706.69\nR001,2024-02-19,1.5,2.9,29738.32\nR001,2024-02-20,0.8,1.8,34996.61\nR001,2024-02-21,0.5,0.0,37118.39\nR001,2024-02-22,1.2,2.2,23517.45\nR001,2024-02-23,2.5,3.1,36176.1\nR001,2024-02-24,2.4,0.0,38730.03\nR001,2024-02-25,2.5,0.0,38905.6\nR001,2024-02-26,3.1,0.0,39082.43\nR001,2024-02-27,3.2,0.0,39260.39\nR001,2024-02-28,2.1,0.0,39403.81\nR001,2024-02-29,1.9,0.0,39505.24\nR001,2024-03-01,2.1,0.0,39600.0\nR001,2024-03-02,4.6,0.0,37332.31\nR001,2024-03-03,5.4,0.0,27873.11\nR001,2024-03-04,4.1,0.9,3545.25\nR001,2024-03-05,5.5,0.1,37365.82\nR001,2024-03-06,5.2,6.5,6831.47\nR001,2024-03-07,3.6,15.7,2901.54\nR001,2024-03-08,1.9,8.5,24801.72\nR001,2024-03-09,1.7,1.4,39600.0\nR001,2024-03-10,1.4,0.0,39600.0\nR001,2024-03-11,1.6,0.2,38076.95\nR001,2024-03-12,3.6,0.0,39658.81\nR001,2024-03-13,9.2,0.7,21627.82\nR001,2024-03-14,8.2,2.5,19479.49\nR001,2024-03-15,5.4,3.5,24151.37\nR001,2024-03-16,5.4,4.3,24431.17\nR001,2024-03-17,4.2,3.5,38170.75\nR001,2024-03-18,5.5,0.1,39600.0\nR001,2024-03-19,6.2,0.0,36794.23\nR001,2024-03-20,4.9,0.4,38300.72\nR001,2024-03-21,5.0,0.7,36000.0\nR001,2024-03-22,4.0,10.1,0.0\nR001,2024-03-23,3.3,2.7,37370.31\nR001,2024-03-24,3.3,5.0,27192.2\nR001,2024-03-25,3.8,2.0,42442.37\nR001,2024-03-26,1.9,23.6,12842.58\nR001,2024-03-27,2.2,11.9,33656.14\nR001,2024-03-28,4.2,0.0,44571.53\nR001,2024-03-29,7.7,2.2,27061.87\nR001,2024-03-30,10.5,1.5,41649.43\nR001,2024-03-31,11.9,0.0,45068.05\nR001,2024-04-01,13.4,0.0,44863.72\nR001,2024-04-02,14.8,0.0,45573.14\nR001,2024-04-03,12.8,4.3,44048.5\nR001,2024-04-04,8.8,0.2,43407.31\nR001,2024-04-05,8.0,0.4,23679.71\nR001,2024-04-06,7.5,0.7,44658.62\nR001,2024-04-07,9.0,0.0,46118.32\nR001,2024-04-08,11.3,0.0,46189.93\nR001,2024-04-09,12.3,0.5,46261.54\nR001,2024-04-10,11.8,0.2,36368.07\nR001,2024-04-11,11.3,2.1,22302.74\nR001,2024-04-12,13.5,0.7,44788.06\nR001,2024-04-13,11.5,4.0,15602.83\nR001,2024-04-14,9.9,2.7,40114.53\nR001,2024-04-15,10.4,0.2,45788.7\nR001,2024-04-16,14.5,0.0,46761.88\nR001,2024-04-17,17.4,0.0,43200.0\nR001,2024-04-18,17.3,0.1,46800.0\nR001,2024-04-19,13.4,5.1,42302.1\nR001,2024-04-20,15.3,1.8,46800.0\nR001,2024-04-21,12.3,7.0,36657.61\nR001,2024-04-22,11.8,1.2,47192.46\nR001,2024-04-23,13.6,0.0,46204.23\nR001,2024-04-24,15.1,0.0,38630.69\nR001,2024-04-25,20.2,0.2,47063.81\nR001,2024-04-26,21.1,1.6,42371.51\nR001,2024-04-27,18.5,2.8,36434.52\nR001,2024-04-28,19.3,1.9,47272.17\nR001,2024-04-29,18.5,1.0,44690.85\nR001,2024-04-30,16.2,0.4,37647.62\nR001,2024-05-01,15.4,1.4,45231.36\nR001,2024-05-02,14.4,7.3,27118.77\nR001,2024-05-03,12.2,6.6,10172.78\nR001,2024-05-04,10.1,15.1,31621.56\nR001,2024-05-05,7.2,34.8,16138.21\nR001,2024-05-06,7.8,3.8,41028.56\nR001,2024-05-07,8.7,3.0,25637.93\nR001,2024-05-08,10.6,2.1,44309.39\nR001,2024-05-09,11.6,3.5,34620.4\nR001,2024-05-10,12.0,3.2,32942.45\nR001,2024-05-11,12.6,10.2,24554.14\nR001,2024-05-12,10.3,29.0,954.71\nR001,2024-05-13,9.1,3.9,39148.47\nR001,2024-05-14,7.6,11.8,23752.47\nR001,2024-05-15,6.8,3.5,47343.26\nR001,2024-05-16,7.3,3.2,50400.0\nR001,2024-05-17,7.5,3.9,44410.79\nR001,2024-05-18,9.0,0.8,47487.66\nR001,2024-05-19,9.8,6.4,46800.0\nR001,2024-05-20,10.8,3.4,48147.19\nR001,2024-05-21,13.2,0.8,50073.42\nR001,2024-05-22,15.2,1.0,50624.08\nR001,2024-05-23,16.1,2.7,50597.56\nR001,2024-05-24,15.4,2.8,50571.53\nR001,2024-05-25,12.8,8.1,43019.26\nR001,2024-05-26,12.8,11.9,20932.1\nR001,2024-05-27,14.5,3.2,47752.6\nR001,2024-05-28,15.7,8.5,39364.34\nR001,2024-05-29,15.7,10.4,29463.98\nR001,2024-05-30,18.0,2.3,51512.14\nR001,2024-05-31,17.5,8.8,42964.58\nR001,2024-06-01,16.8,2.4,30825.24\nR001,2024-06-02,18.1,0.1,51817.44\nR001,2024-06-03,19.0,0.7,51435.72\nR001,2024-06-04,20.7,0.4,51781.64\nR001,2024-06-05,22.6,0.0,52820.64\nR001,2024-06-06,21.3,0.3,50652.38\nR001,2024-06-07,19.1,2.8,44674.5\nR001,2024-06-08,18.6,0.8,46186.6\nR001,2024-06-09,16.6,6.1,35109.32\nR001,2024-06-10,16.9,4.8,34893.1\nR001,2024-06-11,18.0,7.8,49280.5\nR001,2024-06-12,20.0,0.8,50773.48\nR001,2024-06-13,20.1,0.6,53786.47\nR001,2024-06-14,20.8,0.4,53813.67\nR001,2024-06-15,20.4,3.5,51224.52\nR001,2024-06-16,18.3,10.6,47662.94\nR001,2024-06-17,19.7,1.0,52011.02\nR001,2024-06-18,20.9,0.9,50598.82\nR001,2024-06-19,21.2,1.1,53055.19\nR001,2024-06-20,20.2,3.6,51814.29\nR001,2024-06-21,19.7,0.2,53168.3\nR001,2024-06-22,17.6,0.2,53950.15\nR001,2024-06-23,19.2,0.0,53957.35\nR001,2024-06-24,21.9,0.0,53962.6\nR001,2024-06-25,20.3,2.0,51922.0\nR001,2024-06-26,16.5,2.5,51618.16\nR001,2024-06-27,15.2,0.6,50493.09\nR001,2024-06-28,16.9,0.1,53963.69\nR001,2024-06-29,18.1,0.0,53958.83\nR001,2024-06-30,19.1,0.0,53951.87\nR001,2024-07-01,18.3,1.7,35169.24\nR001,2024-07-02,17.0,14.8,30769.21\nR001,2024-07-03,18.4,10.4,51075.81\nR001,2024-07-04,21.5,0.6,53902.49\nR001,2024-07-05,23.6,1.8,53386.33\nR001,2024-07-06,22.9,0.5,53740.62\nR001,2024-07-07,22.2,1.1,37579.02\nR001,2024-07-08,22.7,0.6,45831.04\nR001,2024-07-09,22.4,6.2,41801.03\nR001,2024-07-10,21.6,6.3,44888.71\nR001,2024-07-11,21.1,2.5,51200.79\nR001,2024-07-12,22.5,4.3,53695.58\nR001,2024-07-13,22.3,0.0,53659.32\nR001,2024-07-14,22.6,0.0,53620.85\nR001,2024-07-15,22.6,0.0,48719.96\nR001,2024-07-16,23.3,0.2,53117.7\nR001,2024-07-17,24.4,0.7,49691.65\nR001,2024-07-18,23.4,0.2,53146.2\nR001,2024-07-19,22.5,0.0,51022.38\nR001,2024-07-20,21.7,0.3,48869.02\nR001,2024-07-21,21.4,1.7,52010.81\nR001,2024-07-22,20.9,3.0,50914.29\nR001,2024-07-23,20.2,2.2,45922.87\nR001,2024-07-24,19.7,2.7,48698.87\nR001,2024-07-25,20.0,1.5,47774.85\nR001,2024-07-26,19.8,3.5,33633.07\nR001,2024-07-27,19.1,10.3,34715.93\nR001,2024-07-28,20.0,0.0,48902.24\nR001,2024-07-29,20.4,0.0,51016.85\nR001,2024-07-30,19.9,1.9,49943.79\nR001,2024-07-31,18.7,0.2,48026.02\nR001,2024-08-01,18.8,0.8,46057.49\nR001,2024-08-02,19.2,0.6,47169.44\nR001,2024-08-03,21.1,0.4,49231.08\nR001,2024-08-04,21.7,0.2,48606.11\nR001,2024-08-05,20.6,3.8,47912.1\nR001,2024-08-06,19.4,2.6,45436.69\nR001,2024-08-07,19.2,1.1,46948.1\nR001,2024-08-08,19.4,0.8,47993.84\nR001,2024-08-09,20.4,1.0,45841.48\nR001,2024-08-10,20.8,0.8,45618.47\nR001,2024-08-11,20.9,0.9,40364.52\nR001,2024-08-12,19.7,2.4,44953.38\nR001,2024-08-13,18.4,0.0,47129.18\nR001,2024-08-14,17.9,4.9,44868.91\nR001,2024-08-15,17.6,0.0,46800.0\nR001,2024-08-16,18.3,0.0,46888.37\nR001,2024-08-17,18.6,0.0,46919.68\nR001,2024-08-18,20.6,0.0,46950.99\nR001,2024-08-19,22.6,0.0,46982.25\nR001,2024-08-20,22.7,0.6,46800.0\nR001,2024-08-21,23.3,0.3,46800.0\nR001,2024-08-22,23.4,1.0,43200.0\nR001,2024-08-23,22.5,3.0,46800.0\nR001,2024-08-24,20.5,7.7,22045.37\nR001,2024-08-25,20.9,19.3,38377.47\nR001,2024-08-26,17.8,17.0,3354.29\nR001,2024-08-27,20.4,1.0,46308.93\nR001,2024-08-28,21.6,0.0,46584.02\nR001,2024-08-29,21.4,1.1,46479.51\nR001,2024-08-30,20.2,5.0,35262.46\nR001,2024-08-31,19.1,4.8,44982.5\nR001,2024-09-01,20.8,1.3,44932.28\nR001,2024-09-02,21.3,4.9,43200.0\nR001,2024-09-03,20.6,5.1,44390.19\nR001,2024-09-04,19.6,3.4,43568.97\nR001,2024-09-05,19.2,2.8,42247.3\nR001,2024-09-06,18.0,3.2,35462.76\nR001,2024-09-07,18.3,1.5,41289.15\nR001,2024-09-08,19.0,3.9,41967.04\nR001,2024-09-09,18.9,6.3,38297.96\nR001,2024-09-10,20.9,2.7,40983.24\nR001,2024-09-11,22.5,0.0,40561.09\nR001,2024-09-12,22.0,3.9,43200.0\nR001,2024-09-13,19.2,3.2,41702.78\nR001,2024-09-14,19.0,0.7,42958.18\nR001,2024-09-15,19.1,0.0,42990.42\nR001,2024-09-16,17.5,10.4,32142.36\nR001,2024-09-17,16.2,2.5,36042.26\nR001,2024-09-18,15.1,4.6,37086.52\nR001,2024-09-19,14.9,6.8,34104.36\nR001,2024-09-20,14.1,9.3,10951.83\nR001,2024-09-21,14.5,2.9,21940.42\nR001,2024-09-22,15.1,0.7,31802.05\nR001,2024-09-23,13.9,15.7,14413.15\nR001,2024-09-24,14.4,7.0,33200.08\nR001,2024-09-25,14.6,15.7,25903.12\nR001,2024-09-26,14.6,1.1,36082.45\nR001,2024-09-27,15.7,0.1,40219.61\nR001,2024-09-28,17.2,0.0,40894.1\nR001,2024-09-29,17.8,0.0,40842.42\nR001,2024-09-30,17.3,0.2,40452.28\nR001,2024-10-01,13.6,21.2,15555.79\nR001,2024-10-02,12.1,19.0,1870.11\nR001,2024-10-03,12.9,1.9,29681.13\nR001,2024-10-04,12.6,0.0,36998.72\nR001,2024-10-05,13.9,0.0,39355.57\nR001,2024-10-06,17.0,0.0,39066.29\nR001,2024-10-07,18.3,0.0,37285.68\nR001,2024-10-08,13.9,5.8,29222.16\nR001,2024-10-09,13.7,0.0,37475.6\nR001,2024-10-10,14.9,0.6,32400.0\nR001,2024-10-11,14.3,0.6,36017.61\nR001,2024-10-12,14.1,0.0,36242.31\nR001,2024-10-13,12.7,6.1,7224.36\nR001,2024-10-14,11.8,0.6,30089.72\nR001,2024-10-15,11.1,0.0,33139.58\nR001,2024-10-16,10.3,1.5,32497.18\nR001,2024-10-17,8.8,7.8,19507.23\nR001,2024-10-18,6.1,6.8,11297.61\nR001,2024-10-19,4.6,7.7,21789.29\nR001,2024-10-20,3.9,14.4,0.0\nR001,2024-10-21,2.8,2.0,24941.27\nR001,2024-10-22,4.3,0.0,36000.0\nR001,2024-10-23,7.2,2.3,36000.0\nR001,2024-10-24,8.1,2.9,33906.65\nR001,2024-10-25,8.0,4.0,19601.06\nR001,2024-10-26,5.1,10.4,34973.52\nR001,2024-10-27,4.5,0.0,36000.0\nR001,2024-10-28,5.6,0.0,36000.0\nR001,2024-10-29,6.9,0.0,36000.0\nR001,2024-10-30,8.1,0.0,34776.45\nR001,2024-10-31,6.5,0.0,36000.0\nR001,2024-11-01,7.8,1.2,28367.1\nR001,2024-11-02,9.2,0.0,35901.77\nR001,2024-11-03,4.4,7.1,21756.96\nR001,2024-11-04,0.3,14.1,28542.86\nR001,2024-11-05,-3.0,13.7,35609.47\nR001,2024-11-06,-4.2,0.0,28968.09\nR001,2024-11-07,1.7,0.0,34542.77\nR001,2024-11-08,2.3,0.0,35159.57\nR001,2024-11-09,2.4,0.0,35013.43\nR001,2024-11-10,3.7,0.0,34868.24\nR001,2024-11-11,4.7,0.0,25792.11\nR001,2024-11-12,5.8,0.0,34581.41\nR001,2024-11-13,4.4,0.0,34440.1\nR001,2024-11-14,4.8,0.0,27659.25\nR001,2024-11-15,6.5,8.0,18591.35\nR001,2024-11-16,4.0,21.7,18000.05\nR001,2024-11-17,1.9,0.0,32968.72\nR001,2024-11-18,5.2,0.9,28553.15\nR001,2024-11-19,6.8,3.5,21584.03\nR001,2024-11-20,6.7,0.0,31854.2\nR001,2024-11-21,8.7,0.0,32050.63\nR001,2024-11-22,8.4,9.2,4984.03\nR001,2024-11-23,11.0,0.0,31084.07\nR001,2024-11-24,6.3,20.8,0.0\nR001,2024-11-25,2.9,3.0,14680.16\nR001,2024-11-26,2.9,9.8,23043.23\nR001,2024-11-27,2.5,6.9,0.0\nR001,2024-11-28,1.3,1.4,26218.71\nR001,2024-11-29,2.2,0.0,30298.63\nR001,2024-11-30,2.7,0.0,30384.62\nR001,2024-12-01,2.7,0.8,17960.2\nR001,2024-12-02,2.6,0.2,30348.52\nR001,2024-12-03,1.9,0.0,30744.13\nR001,2024-12-04,1.8,0.0,30374.76\nR001,2024-12-05,3.1,0.9,7514.0\nR001,2024-12-06,2.7,2.7,13434.28\nR001,2024-12-07,3.0,0.0,30403.45\nR001,2024-12-08,5.4,0.0,12904.0\nR001,2024-12-09,7.3,0.0,28870.26\nR001,2024-12-10,7.5,0.7,29258.42\nR001,2024-12-11,7.9,12.9,13914.56\nR001,2024-12-12,2.4,28.2,6256.76\nR001,2024-12-13,-3.7,14.7,5585.04\nR001,2024-12-14,-11.9,0.0,30419.79\nR001,2024-12-15,-4.7,1.4,25259.36\nR001,2024-12-16,0.0,18.3,9030.17\nR001,2024-12-17,0.5,60.3,0.0\nR001,2024-12-18,0.8,36.1,23872.74\nR001,2024-12-19,-2.2,0.0,30777.7\nR001,2024-12-20,0.3,0.0,30766.25\nR001,2024-12-21,1.8,0.0,28800.0\nR001,2024-12-22,3.0,2.7,14116.82\nR001,2024-12-23,3.9,7.7,22113.07\nR001,2024-12-24,4.3,1.2,31094.29\nR001,2024-12-25,4.1,0.0,27083.03\nR001,2024-12-26,4.7,0.0,31337.33\nR001,2024-12-27,3.3,0.0,26635.05\nR001,2024-12-28,2.6,1.8,19288.85\nR001,2024-12-29,2.2,4.4,5843.3\nR001,2024-12-30,0.2,0.7,29071.36\nR001,2024-12-31,-0.2,0.0,30924.16\nR001,2025-01-01,1.8,0.7,21178.16\nR001,2025-01-02,3.0,0.0,31685.61\nR001,2025-01-03,4.7,0.0,31741.79\nR001,2025-01-04,2.7,6.1,24391.01\nR001,2025-01-05,0.7,8.9,14964.44\nR001,2025-01-06,-1.3,0.0,31917.21\nR001,2025-01-07,2.6,0.0,31977.83\nR001,2025-01-08,5.5,0.0,32040.56\nR001,2025-01-09,6.4,0.0,32105.42\nR001,2025-01-10,4.8,0.0,32172.25\nR001,2025-01-11,6.5,0.0,32240.91\nR001,2025-01-12,5.2,0.0,15436.65\nR001,2025-01-13,5.9,18.2,14488.57\nR001,2025-01-14,3.1,18.6,0.0\nR001,2025-01-15,2.7,0.3,32400.0\nR001,2025-01-16,1.4,1.9,31778.1\nR001,2025-01-17,0.4,3.9,20761.06\nR001,2025-01-18,-0.2,8.2,25831.3\nR001,2025-01-19,-0.8,11.3,14998.31\nR001,2025-01-20,-3.1,0.0,32400.0\nR001,2025-01-21,-0.6,0.0,32400.0\nR001,2025-01-22,1.8,0.0,32400.0\nR001,2025-01-23,3.0,0.0,1271.05\nR001,2025-01-24,2.3,1.9,22277.39\nR001,2025-01-25,2.0,1.0,32400.0\nR001,2025-01-26,-0.1,0.0,32400.0\nR001,2025-01-27,0.3,0.0,32400.0\nR001,2025-01-28,1.8,0.0,32529.05\nR001,2025-01-29,3.7,0.3,32400.0\nR001,2025-01-30,4.8,0.3,32443.78\nR001,2025-01-31,4.2,0.0,32752.47\nR001,2025-02-01,3.5,0.0,29605.06\nR001,2025-02-02,2.0,0.0,32663.62\nR001,2025-02-03,2.5,0.4,27904.05\nR001,2025-02-04,0.1,21.5,4573.71\nR001,2025-02-05,-4.4,0.5,28774.57\nR001,2025-02-06,-1.3,0.4,28802.18\nR001,2025-02-07,-1.5,0.4,29902.32\nR001,2025-02-08,-2.0,0.1,33966.42\nR001,2025-02-09,-3.1,1.4,31591.94\nR001,2025-02-10,-1.9,0.8,33440.97\nR001,2025-02-11,-2.7,0.0,36080.81\nR001,2025-02-12,-1.9,0.0,36530.28\nR001,2025-02-13,-1.2,0.1,33732.9\nR001,2025-02-14,-0.4,7.4,1720.53\nR001,2025-02-15,1.4,1.4,31048.93\nR001,2025-02-16,1.8,19.0,0.0\nR001,2025-02-17,0.7,24.1,0.0\nR001,2025-02-18,0.1,23.8,0.0\nR001,2025-02-19,-1.7,4.2,29497.21\nR001,2025-02-20,-5.2,1.8,35458.43\nR001,2025-02-21,-7.1,2.3,36430.45\nR001,2025-02-22,-8.8,0.0,36832.09\nR001,2025-02-23,-8.8,0.0,38687.76\nR001,2025-02-24,-7.2,7.0,15810.11\nR001,2025-02-25,-6.6,8.9,17496.25\nR001,2025-02-26,-7.3,0.6,27746.28\nR001,2025-02-27,-5.3,0.0,39386.88\nR001,2025-02-28,-3.8,0.0,39488.33\nR001,2025-03-01,-3.8,0.0,39590.31\nR001,2025-03-02,-1.8,0.0,36000.0\nR001,2025-03-03,-2.1,7.6,0.0\nR001,2025-03-04,-1.6,1.0,39600.0\nR001,2025-03-05,0.5,0.0,31542.17\nR001,2025-03-06,3.2,0.0,30749.1\nR001,2025-03-07,7.1,0.1,39600.0\nR001,2025-03-08,7.2,0.0,39600.0\nR001,2025-03-09,9.0,0.0,39600.0\nR001,2025-03-10,8.0,0.0,39600.0\nR001,2025-03-11,9.0,0.0,39600.0\nR001,2025-03-12,10.5,0.0,39600.0\nR001,2025-03-13,13.6,0.0,39621.6\nR001,2025-03-14,13.0,0.0,20882.56\nR001,2025-03-15,11.4,0.0,39600.0\nR001,2025-03-16,14.7,0.0,39776.0\nR001,2025-03-17,12.7,1.2,36697.67\nR001,2025-03-18,6.3,23.7,3061.34\nR001,2025-03-19,0.9,9.4,9188.19\nR001,2025-03-20,0.1,0.5,33310.42\nR001,2025-03-21,1.6,1.6,12671.76\nR001,2025-03-22,2.1,4.1,22363.5\nR001,2025-03-23,3.1,5.1,31172.25\nR001,2025-03-24,5.4,0.0,41566.46\nR001,2025-03-25,8.6,1.1,36630.8\nR001,2025-03-26,10.1,6.3,20686.54\nR001,2025-03-27,11.2,0.6,37537.22\nR001,2025-03-28,12.1,0.0,43277.16\nR001,2025-03-29,15.2,0.3,43671.35\nR001,2025-03-30,14.2,3.7,27887.43\nR001,2025-03-31,9.6,25.6,14474.28\nR001,2025-04-01,11.3,0.1,44520.06\nR001,2025-04-02,10.9,3.8,21510.23\nR001,2025-04-03,8.7,8.4,815.95\nR001,2025-04-04,8.8,2.4,35311.41\nR001,2025-04-05,7.3,5.2,26156.8\nR001,2025-04-06,8.2,7.5,36225.31\nR001,2025-04-07,10.6,2.4,42536.58\nR001,2025-04-08,5.7,18.3,10424.26\nR001,2025-04-09,2.3,28.1,0.0\nR001,2025-04-10,-0.3,7.0,22472.29\nR001,2025-04-11,0.8,0.3,34032.42\nR001,2025-04-12,0.2,14.6,24485.64\nR001,2025-04-13,1.7,2.0,34318.21\nR001,2025-04-14,6.2,1.3,44964.98\nR001,2025-04-15,7.3,0.7,13209.43\nR001,2025-04-16,8.8,0.0,42156.16\nR001,2025-04-17,9.7,0.0,46800.0\nR001,2025-04-18,11.3,1.1,46800.0\nR001,2025-04-19,13.1,2.0,46800.0\nR001,2025-04-20,10.9,6.0,31075.7\nR001,2025-04-21,10.7,9.0,38150.35\nR001,2025-04-22,12.4,0.2,46876.18\nR001,2025-04-23,12.0,5.4,44325.83\nR001,2025-04-24,11.5,3.3,46782.69\nR001,2025-04-25,10.2,11.5,22778.65\nR001,2025-04-26,11.1,10.8,39171.8\nR001,2025-04-27,8.2,8.0,5703.73\nR001,2025-04-28,7.0,3.0,34620.76\nR001,2025-04-29,7.9,0.0,47907.39\nR001,2025-04-30,9.2,0.0,48538.61\nR001,2025-05-01,12.2,1.5,36131.64\nR001,2025-05-02,12.7,8.7,28897.94\nR001,2025-05-03,9.9,16.4,7502.03\nR001,2025-05-04,11.3,3.5,46212.05\nR001,2025-05-05,12.4,5.4,46800.0\nR001,2025-05-06,13.4,3.5,39821.76\nR001,2025-05-07,14.8,0.0,46943.14\nR001,2025-05-08,13.4,9.3,43488.09\nR001,2025-05-09,14.2,2.6,49142.76\nR001,2025-05-10,15.1,5.1,45300.38\nR001,2025-05-11,11.6,12.5,0.0\nR001,2025-05-12,12.7,6.5,33755.45\nR001,2025-05-13,10.4,15.3,559.84\nR001,2025-05-14,9.5,4.9,46270.23\nR001,2025-05-15,7.9,5.9,15807.49\nR001,2025-05-16,12.1,0.1,50134.26\nR001,2025-05-17,18.5,0.4,50415.63\nR001,2025-05-18,15.3,8.6,40865.22\nR001,2025-05-19,11.0,10.9,45391.03\nR001,2025-05-20,11.6,0.8,47915.61\nR001,2025-05-21,10.4,0.0,48458.96\nR001,2025-05-22,13.5,0.0,50985.72\nR001,2025-05-23,16.8,0.3,50603.67\nR001,2025-05-24,17.8,0.5,50935.54\nR001,2025-05-25,18.3,0.8,50911.34\nR001,2025-05-26,19.8,0.3,50400.0\nR001,2025-05-27,18.9,5.2,44557.94\nR001,2025-05-28,16.4,12.5,38783.06\nR001,2025-05-29,15.1,15.4,27072.74\nR001,2025-05-30,14.4,21.8,34525.6\nR001,2025-05-31,14.3,9.3,39482.36\nR001,2025-06-01,16.3,4.6,46800.0\nR001,2025-06-02,14.9,15.6,0.0\nR001,2025-06-03,15.3,0.5,47839.82\nR001,2025-06-04,15.9,0.0,50749.03\nR001,2025-06-05,15.6,2.2,48350.81\nR001,2025-06-06,16.3,0.0,53209.15\nR001,2025-06-07,16.7,0.0,53559.3\nR001,2025-06-08,19.1,0.0,53545.03\nR001,2025-06-09,19.7,0.2,53165.28\nR001,2025-06-10,19.5,0.2,52419.04\nR001,2025-06-11,19.3,1.3,52405.62\nR001,2025-06-12,17.5,5.2,50925.24\nR001,2025-06-13,18.5,0.2,50759.49\nR001,2025-06-14,15.3,3.9,39660.1\nR001,2025-06-15,12.5,2.1,48077.37\nR001,2025-06-16,11.6,6.3,42114.82\nR001,2025-06-17,12.2,5.6,37768.87\nR001,2025-06-18,13.6,1.3,51004.04\nR001,2025-06-19,15.5,0.4,53906.49\nR001,2025-06-20,16.4,0.0,53921.06\nR001,2025-06-21,13.8,10.9,33782.33\nR001,2025-06-22,13.8,0.9,51936.58\nR001,2025-06-23,12.6,2.4,48697.86\nR001,2025-06-24,14.8,2.4,50911.24\nR001,2025-06-25,16.9,0.1,53965.17\nR001,2025-06-26,18.4,0.2,53967.9\nR001,2025-06-27,18.9,0.4,53144.24\nR001,2025-06-28,18.3,0.0,53966.92\nR001,2025-06-29,15.5,8.7,30687.12\nR001,2025-06-30,15.4,0.2,48849.32\nR001,2025-07-01,14.8,3.0,49892.86\nR001,2025-07-02,14.9,0.5,53057.68\nR001,2025-07-03,14.8,0.2,51264.87\nR001,2025-07-04,15.7,1.4,50070.15\nR001,2025-07-05,17.4,1.1,51668.49\nR001,2025-07-06,19.3,0.4,53870.56\nR001,2025-07-07,20.3,0.8,53757.05\nR001,2025-07-08,21.9,0.6,53822.71\nR001,2025-07-09,22.8,0.6,53795.46\nR001,2025-07-10,22.4,4.7,53766.02\nR001,2025-07-11,21.9,0.7,53421.14\nR001,2025-07-12,23.6,0.1,52172.62\nR001,2025-07-13,24.1,1.0,53665.0\nR001,2025-07-14,23.0,2.4,52014.72\nR001,2025-07-15,21.2,1.4,52175.91\nR001,2025-07-16,20.2,10.6,44621.98\nR001,2025-07-17,21.0,3.8,50745.85\nR001,2025-07-18,22.8,0.0,53456.15\nR001,2025-07-19,22.4,1.5,52801.26\nR001,2025-07-20,19.7,14.7,39066.14\nR001,2025-07-21,20.6,6.0,48902.93\nR001,2025-07-22,21.7,1.3,49299.28\nR001,2025-07-23,22.0,0.1,50045.93\nR001,2025-07-24,22.6,0.0,49458.97\nR001,2025-07-25,21.9,0.0,51185.9\nR001,2025-07-26,21.7,0.1,51828.89\nR001,2025-07-27,22.9,0.0,51793.38\nR001,2025-07-28,24.5,0.0,51409.68\nR001,2025-07-29,26.7,0.0,51375.13\nR001,2025-07-30,27.0,0.1,50994.05\nR001,2025-07-31,24.3,3.8,47440.95\nR001,2025-08-01,23.4,2.6,37369.73\nR001,2025-08-02,24.1,17.0,44842.93\nR001,2025-08-03,22.1,10.7,46076.98\nR001,2025-08-04,23.2,3.6,43624.1\nR001,2025-08-05,23.3,4.1,44420.03\nR001,2025-08-06,24.6,0.6,46904.46\nR001,2025-08-07,24.2,3.6,48300.83\nR001,2025-08-08,21.6,9.6,40432.87\nR001,2025-08-09,21.3,4.7,39831.31\nR001,2025-08-10,20.3,0.2,47032.53\nR001,2025-08-11,19.7,0.3,47735.54\nR001,2025-08-12,19.5,0.0,47092.8\nR001,2025-08-13,18.4,0.0,47122.96\nR001,2025-08-14,17.6,0.1,47487.26\nR001,2025-08-15,18.0,0.3,47183.18\nR001,2025-08-16,18.0,0.6,46880.61\nR001,2025-08-17,19.4,0.0,47243.63\nR001,2025-08-18,21.1,0.0,46942.76\nR001,2025-08-19,21.3,0.5,46800.0\nR001,2025-08-20,20.2,9.3,46534.77\nR001,2025-08-21,18.8,13.2,45497.69\nR001,2025-08-22,19.8,0.4,46800.0\nR001,2025-08-23,21.4,0.0,46800.0\nR001,2025-08-24,20.7,0.5,46800.0\nR001,2025-08-25,19.1,2.4,43333.65\nR001,2025-08-26,17.4,2.7,22522.58\nR001,2025-08-27,19.2,0.0,45330.04\nR001,2025-08-28,20.9,0.3,46603.99\nR001,2025-08-29,22.2,0.0,46500.48\nR001,2025-08-30,23.6,0.0,46396.67\nR001,2025-08-31,23.5,0.9,46292.65\nR001,2025-09-01,23.8,0.1,46188.51\nR001,2025-09-02,21.1,3.8,41107.58\nR001,2025-09-03,19.0,5.1,37928.32\nR001,2025-09-04,19.1,0.6,44085.03\nR001,2025-09-05,19.8,0.1,39461.42\nR001,2025-09-06,20.0,0.0,44206.01\nR001,2025-09-07,17.0,11.0,34170.62\nR001,2025-09-08,16.3,6.1,39554.37\nR001,2025-09-09,16.3,17.2,18311.49\nR001,2025-09-10,15.6,7.7,30364.57\nR001,2025-09-11,15.0,5.3,36116.78\nR001,2025-09-12,15.8,6.3,40279.79\nR001,2025-09-13,14.3,0.0,41007.74\nR001,2025-09-14,14.5,0.7,43089.62\nR001,2025-09-15,15.6,1.4,43022.25\nR001,2025-09-16,15.0,3.9,36963.71\nR001,2025-09-17,15.8,0.5,40821.07\nR001,2025-09-18,13.8,14.7,36299.86\nR001,2025-09-19,12.2,3.7,12564.46\nR001,2025-09-20,13.3,2.0,40604.36\nR001,2025-09-21,12.6,0.1,40002.02\nR001,2025-09-22,13.1,0.0,42265.32\nR001,2025-09-23,15.6,0.0,42300.54\nR001,2025-09-24,16.2,0.0,42336.14\nR001,2025-09-25,15.4,0.3,38232.16\nR001,2025-09-26,12.1,8.7,20569.92\nR001,2025-09-27,11.2,2.1,29971.66\nR001,2025-09-28,11.5,1.1,31674.85\nR001,2025-09-29,12.1,10.5,0.0\nR001,2025-09-30,11.8,15.0,24603.18\nR001,2025-10-01,11.5,0.4,36648.87\nR001,2025-10-02,12.5,0.8,38727.62\nR001,2025-10-03,14.2,0.0,39103.15\nR001,2025-10-04,18.1,0.0,39541.58\nR001,2025-10-05,15.9,4.4,37402.4\nR001,2025-10-06,14.4,0.0,37173.57\nR001,2025-10-07,16.1,0.0,38561.09\nR001,2025-10-08,18.3,0.1,37686.33\nR001,2025-10-09,13.2,25.1,7851.33\nR001,2025-10-10,9.3,28.1,11283.04\nR001,2025-10-11,8.4,9.6,31678.09\nR001,2025-10-12,7.7,9.1,4581.72\nR001,2025-10-13,7.3,1.4,36280.89\nR001,2025-10-14,8.3,2.8,31120.0\nR001,2025-10-15,8.3,1.0,36057.11\nR001,2025-10-16,7.9,1.3,36108.28\nR001,2025-10-17,7.9,0.0,36159.37\nR001,2025-10-18,9.5,0.0,33646.39\nR001,2025-10-19,10.0,9.3,27772.85\nR001,2025-10-20,7.5,7.1,16695.66\nR001,2025-10-21,5.1,1.3,36000.0\nR001,2025-10-22,7.5,0.0,34322.08\nR001,2025-10-23,10.7,1.5,22671.19\nR001,2025-10-24,11.5,4.0,21611.25\nR001,2025-10-25,12.2,4.3,1346.74\nR001,2025-10-26,9.3,26.8,16492.81\nR001,2025-10-27,8.8,0.9,26725.55\nR001,2025-10-28,12.3,4.4,30379.25\nR001,2025-10-29,7.9,23.2,18418.18\nR001,2025-10-30,5.1,1.5,29952.2\nR001,2025-10-31,6.9,0.0,36000.0\nR001,2025-11-01,8.5,0.0,36000.0\nR001,2025-11-02,5.2,0.0,35922.85\nR001,2025-11-03,9.6,0.0,35844.67\nR001,2025-11-04,11.5,0.0,35766.11\nR001,2025-11-05,10.9,0.0,34247.04\nR001,2025-11-06,9.8,0.0,35505.85\nR001,2025-11-07,11.1,0.0,32430.99\nR001,2025-11-08,13.3,0.3,35201.76\nR001,2025-11-09,12.1,0.3,34514.72\nR001,2025-11-10,11.6,0.0,32016.4\nR001,2025-11-11,11.7,0.0,27270.67\nR001,2025-11-12,10.8,4.1,22868.9\nR001,2025-11-13,8.2,22.2,1801.62\nR001,2025-11-14,7.0,20.6,1511.56\nR001,2025-11-15,8.8,0.5,32989.83\nR001,2025-11-16,8.9,0.0,32361.68\nR001,2025-11-17,8.5,0.0,32605.78\nR001,2025-11-18,9.0,0.0,33326.54\nR001,2025-11-19,12.2,0.0,30771.57\nR001,2025-11-20,12.2,0.0,29631.95\nR001,2025-11-21,10.7,0.0,32401.0\nR001,2025-11-22,11.9,0.0,31367.17\nR001,2025-11-23,12.6,0.0,31063.93\nR001,2025-11-24,11.4,0.0,29378.28\nR001,2025-11-25,9.2,0.0,29081.2\nR001,2025-11-26,8.5,0.0,26743.08\nR001,2025-11-27,8.4,0.0,30626.1\nR001,2025-11-28,7.9,0.0,30648.27\nR001,2025-11-29,8.2,0.0,30669.15\nR001,2025-11-30,9.1,0.6,28860.49\nR001,2025-12-01,7.0,6.5,3239.37\nR001,2025-12-02,5.2,2.1,7031.82\nR001,2025-12-03,3.6,0.0,29472.95\nR001,2025-12-04,3.4,0.0,28841.16\nR001,2025-12-05,4.1,0.0,30383.01\nR001,2025-12-06,4.1,0.0,28857.18\nR001,2025-12-07,5.7,3.9,0.0\nR001,2025-12-08,7.1,11.2,26410.28\nR001,2025-12-09,5.8,0.2,26059.71\nR001,2025-12-10,3.6,1.8,28536.46\nR001,2025-12-11,3.7,5.6,14558.37\nR001,2025-12-12,2.7,6.0,18319.67\nR001,2025-12-13,0.8,12.6,10648.35\nR001,2025-12-14,-4.3,0.0,26572.85\nR001,2025-12-15,0.1,2.2,1607.5\nR001,2025-12-16,-0.5,5.8,22824.27\nR001,2025-12-17,-1.5,0.0,30016.75\nR001,2025-12-18,2.5,0.0,30007.63\nR001,2025-12-19,3.0,0.0,29996.95\nR001,2025-12-20,2.4,0.0,30766.5\nR001,2025-12-21,3.6,0.0,29579.93\nR001,2025-12-22,4.2,2.2,10324.76\nR001,2025-12-23,3.9,1.9,28768.63\nR001,2025-12-24,3.1,6.9,5442.85\nR001,2025-12-25,0.8,0.0,30003.52\nR001,2025-12-26,-0.7,3.5,30272.87\nR001,2025-12-27,-3.7,13.8,14790.57\nR001,2025-12-28,-5.1,18.4,12431.41\nR001,2025-12-29,-3.8,13.8,10163.35\nR001,2025-12-30,-3.6,22.6,8419.04\nR001,2025-12-31,-1.9,36.5,0.0\nR002,1996-01-01,4.9,0.9,27974.96\nR002,1996-01-02,5.1,0.0,31626.87\nR002,1996-01-03,5.9,0.2,1888.06\nR002,1996-01-04,6.2,0.0,31737.93\nR002,1996-01-05,6.7,0.2,16920.48\nR002,1996-01-06,5.6,0.0,31857.7\nR002,1996-01-07,5.6,3.9,15718.35\nR002,1996-01-08,4.6,2.0,23787.38\nR002,1996-01-09,2.4,0.0,32051.21\nR002,1996-01-10,0.5,0.0,32118.84\nR002,1996-01-11,-1.0,0.0,32187.78\nR002,1996-01-12,-0.0,0.0,32257.91\nR002,1996-01-13,-0.7,0.0,32329.08\nR002,1996-01-14,-1.0,0.0,32400.0\nR002,1996-01-15,0.3,0.0,32400.0\nR002,1996-01-16,-1.1,0.0,32400.0\nR002,1996-01-17,-0.9,0.0,32400.0\nR002,1996-01-18,-1.7,0.0,32400.0\nR002,1996-01-19,-1.8,2.5,30456.52\nR002,1996-01-20,-0.1,6.2,16980.8\nR002,1996-01-21,-0.1,5.7,6137.44\nR002,1996-01-22,-2.6,0.3,32400.0\nR002,1996-01-23,-3.0,0.5,26761.33\nR002,1996-01-24,-3.2,0.3,31481.38\nR002,1996-01-25,0.2,0.3,28800.0\nR002,1996-01-26,2.7,0.0,32400.0\nR002,1996-01-27,4.1,0.0,30351.34\nR002,1996-01-28,5.4,2.0,3395.71\nR002,1996-01-29,5.1,5.3,15213.75\nR002,1996-01-30,3.9,8.1,3191.67\nR002,1996-01-31,3.2,11.8,1.56\nR002,1996-02-01,-3.6,10.5,30631.22\nR002,1996-02-02,-7.3,0.0,32700.41\nR002,1996-02-03,-3.5,0.0,33089.41\nR002,1996-02-04,-0.6,0.0,33704.15\nR002,1996-02-05,0.0,0.0,33619.37\nR002,1996-02-06,2.0,6.4,2540.12\nR002,1996-02-07,5.1,28.4,15962.83\nR002,1996-02-08,6.8,3.2,19393.09\nR002,1996-02-09,8.8,5.8,19952.24\nR002,1996-02-10,5.9,5.1,5618.7\nR002,1996-02-11,4.4,14.2,0.0\nR002,1996-02-12,2.6,3.4,21352.66\nR002,1996-02-13,3.2,0.0,26548.84\nR002,1996-02-14,5.8,0.0,27074.19\nR002,1996-02-15,5.9,3.3,27978.59\nR002,1996-02-16,4.4,5.6,8988.85\nR002,1996-02-17,4.9,2.9,17089.18\nR002,1996-02-18,7.0,0.4,37654.32\nR002,1996-02-19,7.5,0.3,37828.42\nR002,1996-02-20,6.1,0.0,38003.31\nR002,1996-02-21,6.6,0.0,38178.82\nR002,1996-02-22,7.8,0.0,38354.79\nR002,1996-02-23,9.2,0.0,35630.05\nR002,1996-02-24,10.1,0.0,27277.01\nR002,1996-02-25,6.2,6.3,17519.13\nR002,1996-02-26,4.6,3.1,21434.34\nR002,1996-02-27,4.4,0.5,37270.78\nR002,1996-02-28,3.9,1.7,27036.78\nR002,1996-02-29,3.1,6.0,27736.69\nR002,1996-03-01,2.9,1.7,35982.65\nR002,1996-03-02,3.4,12.1,0.0\nR002,1996-03-03,4.0,0.9,39600.0\nR002,1996-03-04,3.5,13.2,13736.15\nR002,1996-03-05,3.5,2.9,36075.25\nR002,1996-03-06,5.6,14.8,0.0\nR002,1996-03-07,6.6,1.1,35028.6\nR002,1996-03-08,4.7,4.1,19192.79\nR002,1996-03-09,3.3,4.7,16926.55\nR002,1996-03-10,3.5,0.7,30223.94\nR002,1996-03-11,5.6,0.1,37596.1\nR002,1996-03-12,4.5,9.4,8170.99\nR002,1996-03-13,4.3,5.3,37825.09\nR002,1996-03-14,4.8,0.0,39363.12\nR002,1996-03-15,7.1,0.0,39839.5\nR002,1996-03-16,8.0,0.0,40089.51\nR002,1996-03-17,6.6,0.0,40035.22\nR002,1996-03-18,5.9,1.1,37029.06\nR002,1996-03-19,5.6,2.9,18256.69\nR002,1996-03-20,4.7,0.3,35612.17\nR002,1996-03-21,6.2,0.2,39183.51\nR002,1996-03-22,7.3,2.2,29650.3\nR002,1996-03-23,7.1,0.8,15297.18\nR002,1996-03-24,6.8,3.2,5603.46\nR002,1996-03-25,4.9,7.1,857.46\nR002,1996-03-26,4.5,11.0,19972.43\nR002,1996-03-27,4.1,6.4,21506.13\nR002,1996-03-28,6.4,0.0,39400.97\nR002,1996-03-29,7.5,9.9,35200.8\nR002,1996-03-30,7.9,11.0,13851.49\nR002,1996-03-31,7.2,1.7,40663.09\nR002,1996-04-01,9.8,0.7,43408.06\nR002,1996-04-02,9.5,4.8,8387.42\nR002,1996-04-03,9.4,0.0,35648.67\nR002,1996-04-04,10.6,0.1,45688.81\nR002,1996-04-05,10.3,0.0,45962.75\nR002,1996-04-06,10.4,0.4,44024.23\nR002,1996-04-07,9.5,3.4,44632.1\nR002,1996-04-08,9.2,11.9,9112.6\nR002,1996-04-09,7.7,4.1,41147.91\nR002,1996-04-10,6.7,3.2,39856.54\nR002,1996-04-11,5.9,0.6,46389.12\nR002,1996-04-12,7.2,1.3,30569.82\nR002,1996-04-13,7.1,7.1,0.0\nR002,1996-04-14,6.3,2.6,36552.6\nR002,1996-04-15,7.8,0.0,45816.35\nR002,1996-04-16,10.4,0.1,35200.78\nR002,1996-04-17,9.2,17.3,0.0\nR002,1996-04-18,7.2,31.3,17852.29\nR002,1996-04-19,4.5,7.3,9809.37\nR002,1996-04-20,5.3,2.5,44583.79\nR002,1996-04-21,7.9,1.2,20258.72\nR002,1996-04-22,7.3,11.5,0.0\nR002,1996-04-23,8.5,1.7,18746.26\nR002,1996-04-24,9.7,1.1,43618.01\nR002,1996-04-25,10.2,0.9,36598.84\nR002,1996-04-26,11.6,0.1,46800.0\nR002,1996-04-27,13.7,0.0,47650.5\nR002,1996-04-28,13.4,0.0,47944.42\nR002,1996-04-29,14.9,0.0,48574.8\nR002,1996-04-30,15.3,5.3,39408.16\nR002,1996-05-01,15.8,4.4,46800.0\nR002,1996-05-02,16.9,0.6,48805.49\nR002,1996-05-03,17.7,2.2,48435.1\nR002,1996-05-04,17.0,3.6,46414.3\nR002,1996-05-05,16.5,2.1,45132.83\nR002,1996-05-06,16.3,4.1,48164.17\nR002,1996-05-07,16.9,2.8,47474.35\nR002,1996-05-08,18.1,0.6,49637.68\nR002,1996-05-09,19.9,0.3,49952.37\nR002,1996-05-10,19.7,2.0,47529.71\nR002,1996-05-11,19.6,5.1,46058.39\nR002,1996-05-12,19.9,4.4,49190.36\nR002,1996-05-13,17.7,7.5,41981.7\nR002,1996-05-14,18.3,3.1,49226.4\nR002,1996-05-15,17.8,2.9,45823.98\nR002,1996-05-16,15.2,6.3,40558.97\nR002,1996-05-17,16.1,0.0,46589.41\nR002,1996-05-18,17.0,0.0,50400.0\nR002,1996-05-19,17.9,1.3,48892.66\nR002,1996-05-20,17.5,0.5,44619.46\nR002,1996-05-21,20.3,0.1,47329.03\nR002,1996-05-22,19.1,0.0,48577.64\nR002,1996-05-23,16.6,2.0,38731.65\nR002,1996-05-24,14.5,4.7,44506.15\nR002,1996-05-25,12.9,5.9,40098.25\nR002,1996-05-26,15.2,0.0,49126.45\nR002,1996-05-27,18.5,2.3,47688.33\nR002,1996-05-28,16.4,22.8,19122.03\nR002,1996-05-29,18.2,0.5,48769.81\nR002,1996-05-30,18.3,3.1,41885.38\nR002,1996-05-31,17.8,16.0,10237.6\nR002,1996-06-01,14.8,25.3,6154.43\nR002,1996-06-02,11.9,6.6,45995.95\nR002,1996-06-03,12.9,3.1,49937.74\nR002,1996-06-04,13.5,8.2,26583.11\nR002,1996-06-05,13.3,3.2,24963.01\nR002,1996-06-06,13.9,11.7,1747.63\nR002,1996-06-07,14.4,5.3,45028.63\nR002,1996-06-08,14.4,3.9,48952.27\nR002,1996-06-09,14.4,17.6,29660.02\nR002,1996-06-10,15.9,0.0,52177.96\nR002,1996-06-11,17.6,0.0,52777.32\nR002,1996-06-12,20.4,0.8,52397.99\nR002,1996-06-13,21.5,1.3,49299.26\nR002,1996-06-14,21.2,1.6,49276.49\nR002,1996-06-15,21.1,0.0,53097.85\nR002,1996-06-16,19.9,0.8,51447.81\nR002,1996-06-17,19.0,11.4,4651.51\nR002,1996-06-18,17.5,33.0,12617.7\nR002,1996-06-19,16.8,3.3,44142.67\nR002,1996-06-20,18.2,1.8,50598.99\nR002,1996-06-21,17.0,17.8,0.0\nR002,1996-06-22,18.5,3.5,35661.1\nR002,1996-06-23,20.4,0.4,51360.52\nR002,1996-06-24,21.9,3.2,52185.79\nR002,1996-06-25,22.9,2.8,51861.97\nR002,1996-06-26,23.8,0.3,52630.07\nR002,1996-06-27,25.7,0.0,53967.67\nR002,1996-06-28,24.8,0.2,53964.35\nR002,1996-06-29,18.6,31.0,10593.67\nR002,1996-06-30,18.1,0.1,53381.9\nR002,1996-07-01,20.3,0.3,53016.49\nR002,1996-07-02,21.8,0.1,52287.31\nR002,1996-07-03,24.0,0.0,53893.57\nR002,1996-07-04,26.2,0.2,53900.03\nR002,1996-07-05,25.4,3.1,53255.7\nR002,1996-07-06,24.4,1.4,53393.35\nR002,1996-07-07,23.0,2.6,51618.54\nR002,1996-07-08,23.8,2.2,52678.21\nR002,1996-07-09,24.6,3.8,53047.01\nR002,1996-07-10,26.5,0.9,53759.92\nR002,1996-07-11,26.6,0.7,53729.22\nR002,1996-07-12,25.8,2.8,52351.41\nR002,1996-07-13,25.7,4.4,51240.73\nR002,1996-07-14,25.5,2.1,44606.57\nR002,1996-07-15,25.6,0.2,53585.55\nR002,1996-07-16,24.6,2.3,51930.31\nR002,1996-07-17,23.5,3.3,51303.11\nR002,1996-07-18,21.3,12.4,20748.65\nR002,1996-07-19,22.0,8.4,46863.09\nR002,1996-07-20,21.1,19.0,41894.54\nR002,1996-07-21,22.5,5.7,44557.86\nR002,1996-07-22,22.3,4.5,43537.41\nR002,1996-07-23,21.5,11.0,26032.62\nR002,1996-07-24,19.3,42.2,837.6\nR002,1996-07-25,20.3,0.4,49085.02\nR002,1996-07-26,20.3,2.0,49389.74\nR002,1996-07-27,21.3,5.3,42042.38\nR002,1996-07-28,20.1,0.0,49235.59\nR002,1996-07-29,20.8,0.0,48572.81\nR002,1996-07-30,22.1,0.3,48303.61\nR002,1996-07-31,24.9,2.6,49153.63\nR002,1996-08-01,25.0,3.5,48143.71\nR002,1996-08-02,24.7,1.6,46800.0\nR002,1996-08-03,25.3,6.0,47852.53\nR002,1996-08-04,24.6,8.6,38240.06\nR002,1996-08-05,23.7,15.6,43325.87\nR002,1996-08-06,23.7,6.7,41647.56\nR002,1996-08-07,21.4,6.7,39108.77\nR002,1996-08-08,20.6,7.2,45570.17\nR002,1996-08-09,19.1,0.2,45115.09\nR002,1996-08-10,20.0,4.3,45949.74\nR002,1996-08-11,20.2,8.7,43792.26\nR002,1996-08-12,20.7,0.6,46800.0\nR002,1996-08-13,22.1,0.5,46800.0\nR002,1996-08-14,22.9,0.6,45028.06\nR002,1996-08-15,23.8,0.5,46846.75\nR002,1996-08-16,22.6,2.0,43000.28\nR002,1996-08-17,22.6,4.5,41310.19\nR002,1996-08-18,22.5,1.3,44168.45\nR002,1996-08-19,22.0,10.8,43548.6\nR002,1996-08-20,22.6,1.9,46070.05\nR002,1996-08-21,20.3,26.2,25121.78\nR002,1996-08-22,19.7,5.4,44428.54\nR002,1996-08-23,20.3,0.6,45656.05\nR002,1996-08-24,22.3,0.1,46800.0\nR002,1996-08-25,21.1,4.4,43218.71\nR002,1996-08-26,19.5,1.9,44350.81\nR002,1996-08-27,18.8,1.7,44358.52\nR002,1996-08-28,20.1,3.8,43231.02\nR002,1996-08-29,21.2,2.2,44129.1\nR002,1996-08-30,20.3,7.1,42631.33\nR002,1996-08-31,21.5,1.1,41468.42\nR002,1996-09-01,21.9,3.1,43000.35\nR002,1996-09-02,23.2,0.0,44056.51\nR002,1996-09-03,22.0,9.3,34909.24\nR002,1996-09-04,19.8,4.9,38607.77\nR002,1996-09-05,20.0,1.1,40729.11\nR002,1996-09-06,19.3,14.5,29978.03\nR002,1996-09-07,18.2,10.3,30407.26\nR002,1996-09-08,17.6,17.6,34791.79\nR002,1996-09-09,14.2,19.8,23854.27\nR002,1996-09-10,14.0,14.0,16615.91\nR002,1996-09-11,14.3,4.6,35898.08\nR002,1996-09-12,14.9,0.0,41884.07\nR002,1996-09-13,17.4,0.0,43159.04\nR002,1996-09-14,21.4,0.1,37834.12\nR002,1996-09-15,17.9,3.4,33282.89\nR002,1996-09-16,13.8,0.3,40468.25\nR002,1996-09-17,14.0,0.1,41456.51\nR002,1996-09-18,13.5,0.5,40216.94\nR002,1996-09-19,11.4,3.6,39396.35\nR002,1996-09-20,12.1,0.0,39639.49\nR002,1996-09-21,13.0,4.6,21226.27\nR002,1996-09-22,13.0,0.5,38140.27\nR002,1996-09-23,13.7,0.2,38353.91\nR002,1996-09-24,15.9,3.6,21939.96\nR002,1996-09-25,18.4,8.0,35859.85\nR002,1996-09-26,17.1,1.3,39856.86\nR002,1996-09-27,18.1,0.0,40797.84\nR002,1996-09-28,20.0,5.5,38619.02\nR002,1996-09-29,14.7,27.6,6190.75\nR002,1996-09-30,14.7,2.4,31463.02\nR002,1996-10-01,14.9,0.0,39746.5\nR002,1996-10-02,19.0,0.0,39995.12\nR002,1996-10-03,16.5,7.4,4075.48\nR002,1996-10-04,15.0,2.6,37784.29\nR002,1996-10-05,12.1,4.6,38430.58\nR002,1996-10-06,12.8,0.0,38786.88\nR002,1996-10-07,14.2,0.0,36618.44\nR002,1996-10-08,13.9,0.1,36981.67\nR002,1996-10-09,14.3,0.0,36478.42\nR002,1996-10-10,15.7,0.1,36767.54\nR002,1996-10-11,15.9,3.8,28200.02\nR002,1996-10-12,14.9,6.0,381.26\nR002,1996-10-13,14.0,1.0,29909.82\nR002,1996-10-14,13.7,4.0,24559.25\nR002,1996-10-15,13.9,6.0,24308.4\nR002,1996-10-16,14.0,2.0,28492.66\nR002,1996-10-17,13.1,0.0,36000.0\nR002,1996-10-18,14.3,0.0,36000.0\nR002,1996-10-19,16.0,0.0,33419.15\nR002,1996-10-20,16.9,0.2,15146.47\nR002,1996-10-21,16.3,0.3,36000.0\nR002,1996-10-22,16.7,7.8,15234.54\nR002,1996-10-23,12.0,42.3,655.65\nR002,1996-10-24,9.8,3.7,29472.05\nR002,1996-10-25,11.0,6.3,11855.98\nR002,1996-10-26,10.2,16.7,18340.19\nR002,1996-10-27,8.5,7.3,26774.54\nR002,1996-10-28,6.2,8.4,32400.0\nR002,1996-10-29,5.4,0.0,36000.0\nR002,1996-10-30,7.3,0.1,36000.0\nR002,1996-10-31,8.8,0.0,29891.49\nR002,1996-11-01,7.4,9.1,19090.1\nR002,1996-11-02,5.1,0.0,35920.86\nR002,1996-11-03,6.9,0.6,27013.13\nR002,1996-11-04,8.5,2.2,17804.35\nR002,1996-11-05,8.4,0.0,35648.87\nR002,1996-11-06,9.1,0.0,35492.41\nR002,1996-11-07,10.1,0.0,35337.7\nR002,1996-11-08,10.8,0.0,35184.91\nR002,1996-11-09,10.3,0.0,32622.28\nR002,1996-11-10,6.9,8.6,28408.0\nR002,1996-11-11,3.9,0.0,34740.11\nR002,1996-11-12,8.6,0.0,31229.02\nR002,1996-11-13,8.9,0.0,34456.85\nR002,1996-11-14,9.8,0.0,34319.84\nR002,1996-11-15,8.6,0.0,34186.0\nR002,1996-11-16,7.8,0.0,34054.05\nR002,1996-11-17,7.4,0.0,33923.67\nR002,1996-11-18,8.2,0.0,33693.06\nR002,1996-11-19,7.7,0.0,33325.1\nR002,1996-11-20,10.9,0.0,32590.76\nR002,1996-11-21,12.2,0.2,31202.83\nR002,1996-11-22,10.3,2.0,27683.92\nR002,1996-11-23,9.4,0.0,30085.61\nR002,1996-11-24,10.2,0.0,30549.56\nR002,1996-11-25,10.4,0.0,30828.13\nR002,1996-11-26,11.0,0.0,28800.0\nR002,1996-11-27,12.5,0.0,29873.39\nR002,1996-11-28,10.9,0.0,28800.0\nR002,1996-11-29,9.6,1.8,25733.15\nR002,1996-11-30,7.6,0.0,26926.52\nR002,1996-12-01,9.9,0.0,28116.11\nR002,1996-12-02,11.4,0.7,29964.47\nR002,1996-12-03,10.7,0.0,29978.42\nR002,1996-12-04,12.7,0.0,26138.04\nR002,1996-12-05,11.1,0.9,28800.0\nR002,1996-12-06,10.2,4.0,13756.65\nR002,1996-12-07,9.0,0.9,22430.25\nR002,1996-12-08,7.8,3.7,5595.64\nR002,1996-12-09,8.1,0.2,27601.43\nR002,1996-12-10,7.5,0.2,24463.08\nR002,1996-12-11,7.4,1.1,17255.22\nR002,1996-12-12,6.8,0.0,27956.17\nR002,1996-12-13,6.6,1.6,5078.86\nR002,1996-12-14,6.4,1.4,24344.49\nR002,1996-12-15,6.4,0.3,12156.93\nR002,1996-12-16,6.4,32.4,0.0\nR002,1996-12-17,-1.5,4.7,30400.76\nR002,1996-12-18,1.1,0.0,26096.48\nR002,1996-12-19,5.8,0.6,27973.04\nR002,1996-12-20,6.8,19.5,5678.86\nR002,1996-12-21,7.6,5.2,15786.91\nR002,1996-12-22,5.6,35.8,0.0\nR002,1996-12-23,2.2,10.3,0.0\nR002,1996-12-24,5.4,57.5,0.0\nR002,1996-12-25,6.0,52.7,0.0\nR002,1996-12-26,6.5,26.3,0.0\nR002,1996-12-27,6.2,42.6,0.0\nR002,1996-12-28,6.5,33.3,4351.96\nR002,1996-12-29,5.0,23.5,0.0\nR002,1996-12-30,-0.2,8.2,25562.53\nR002,1996-12-31,6.0,3.7,4003.25\nR002,1997-01-01,5.9,12.8,5866.58\nR002,1997-01-02,6.0,18.9,5427.23\nR002,1997-01-03,3.9,8.0,1155.42\nR002,1997-01-04,0.4,2.6,25200.0\nR002,1997-01-05,6.2,3.0,0.0\nR002,1997-01-06,5.1,34.9,16242.17\nR002,1997-01-07,2.5,3.6,31976.54\nR002,1997-01-08,3.2,0.3,26672.51\nR002,1997-01-09,7.1,3.0,20553.04\nR002,1997-01-10,4.4,5.0,18419.92\nR002,1997-01-11,4.2,0.0,22761.71\nR002,1997-01-12,5.9,0.0,28800.0\nR002,1997-01-13,5.2,0.1,32380.25\nR002,1997-01-14,3.7,0.0,32400.0\nR002,1997-01-15,3.3,0.1,26980.72\nR002,1997-01-16,0.7,0.8,32400.0\nR002,1997-01-17,1.7,3.7,16165.29\nR002,1997-01-18,1.1,0.0,32400.0\nR002,1997-01-19,2.9,0.0,32400.0\nR002,1997-01-20,4.0,0.0,32400.0\nR002,1997-01-21,2.1,9.2,0.0\nR002,1997-01-22,1.8,0.9,31844.15\nR002,1997-01-23,0.9,0.6,32400.0\nR002,1997-01-24,2.1,0.0,28800.0\nR002,1997-01-25,0.8,12.0,5967.47\nR002,1997-01-26,-6.2,0.0,31141.96\nR002,1997-01-27,-8.8,0.2,32400.0\nR002,1997-01-28,-7.7,0.2,32400.0\nR002,1997-01-29,-3.5,22.4,0.0\nR002,1997-01-30,-7.7,0.8,32400.0\nR002,1997-01-31,-6.8,3.1,28800.0\nR002,1997-02-01,-1.8,30.0,0.0\nR002,1997-02-02,-3.0,11.1,19098.53\nR002,1997-02-03,-4.7,3.4,21051.5\nR002,1997-02-04,-4.3,0.0,25169.5\nR002,1997-02-05,-5.2,0.0,33562.48\nR002,1997-02-06,-4.4,0.0,32691.85\nR002,1997-02-07,-0.5,7.8,0.0\nR002,1997-02-08,-2.4,0.0,34669.39\nR002,1997-02-09,0.1,0.2,33733.58\nR002,1997-02-10,0.3,3.3,27435.06\nR002,1997-02-11,1.1,1.2,19090.44\nR002,1997-02-12,3.2,0.3,34304.86\nR002,1997-02-13,4.2,0.0,31612.35\nR002,1997-02-14,6.2,0.3,29188.37\nR002,1997-02-15,5.2,11.9,20765.45\nR002,1997-02-16,5.8,0.0,37331.35\nR002,1997-02-17,7.0,28.5,4715.23\nR002,1997-02-18,-0.6,1.1,35407.8\nR002,1997-02-19,-2.6,0.0,37965.81\nR002,1997-02-20,-2.6,0.0,38138.71\nR002,1997-02-21,2.4,0.0,31939.19\nR002,1997-02-22,2.7,7.2,7437.98\nR002,1997-02-23,1.9,2.4,32395.65\nR002,1997-02-24,2.3,0.0,35768.18\nR002,1997-02-25,1.8,0.0,39005.17\nR002,1997-02-26,1.9,0.0,39181.56\nR002,1997-02-27,3.4,0.0,39354.99\nR002,1997-02-28,5.1,9.5,0.0\nR002,1997-03-01,2.2,8.1,21956.73\nR002,1997-03-02,-0.4,0.0,39600.0\nR002,1997-03-03,0.9,0.0,39600.0\nR002,1997-03-04,1.0,2.9,7200.0\nR002,1997-03-05,0.1,13.3,29269.72\nR002,1997-03-06,-2.0,3.7,39600.0\nR002,1997-03-07,-3.1,0.0,39360.58\nR002,1997-03-08,-3.0,0.0,39600.0\nR002,1997-03-09,-2.3,0.0,38960.7\nR002,1997-03-10,0.7,0.0,39600.0\nR002,1997-03-11,1.6,0.0,39600.0\nR002,1997-03-12,3.6,0.0,39600.0\nR002,1997-03-13,4.8,0.0,39600.0\nR002,1997-03-14,5.3,0.0,39600.0\nR002,1997-03-15,5.6,0.0,39841.96\nR002,1997-03-16,4.5,0.0,39788.79\nR002,1997-03-17,1.9,10.2,16555.52\nR002,1997-03-18,-1.3,5.2,18557.16\nR002,1997-03-19,-3.5,0.0,35571.63\nR002,1997-03-20,-0.5,0.0,36065.61\nR002,1997-03-21,6.9,0.0,38875.92\nR002,1997-03-22,6.6,1.5,11441.47\nR002,1997-03-23,2.7,10.6,8719.68\nR002,1997-03-24,-0.6,2.4,39999.45\nR002,1997-03-25,-0.1,0.0,41497.83\nR002,1997-03-26,-1.0,3.3,29942.97\nR002,1997-03-27,-0.8,0.0,35752.09\nR002,1997-03-28,-0.8,0.0,39836.64\nR002,1997-03-29,2.4,0.0,43972.15\nR002,1997-03-30,9.2,0.0,43091.29\nR002,1997-03-31,12.4,0.0,44438.33\nR002,1997-04-01,11.1,0.0,45149.03\nR002,1997-04-02,11.1,0.0,44514.84\nR002,1997-04-03,11.2,0.0,40973.21\nR002,1997-04-04,9.6,11.8,40456.4\nR002,1997-04-05,8.1,0.6,35781.76\nR002,1997-04-06,8.3,0.1,44352.97\nR002,1997-04-07,6.7,9.2,22856.7\nR002,1997-04-08,5.8,0.2,42606.58\nR002,1997-04-09,4.7,13.4,1472.23\nR002,1997-04-10,1.9,10.9,16932.8\nR002,1997-04-11,0.0,3.2,46372.24\nR002,1997-04-12,4.7,0.2,43512.59\nR002,1997-04-13,8.3,0.1,46515.12\nR002,1997-04-14,9.0,2.1,23063.32\nR002,1997-04-15,6.9,13.9,21908.31\nR002,1997-04-16,6.7,0.3,45637.05\nR002,1997-04-17,10.0,0.0,46800.0\nR002,1997-04-18,8.9,10.8,40116.12\nR002,1997-04-19,4.1,4.1,44917.74\nR002,1997-04-20,7.4,0.9,30898.99\nR002,1997-04-21,10.2,3.4,40127.13\nR002,1997-04-22,13.8,0.0,38068.05\nR002,1997-04-23,17.8,0.6,33699.01\nR002,1997-04-24,14.3,7.2,22240.3\nR002,1997-04-25,11.6,13.7,31123.12\nR002,1997-04-26,10.7,12.4,34016.56\nR002,1997-04-27,10.4,1.6,41026.4\nR002,1997-04-28,7.8,14.5,2951.24\nR002,1997-04-29,7.1,10.3,1750.94\nR002,1997-04-30,8.5,1.6,38461.99\nR002,1997-05-01,9.1,0.0,47507.08\nR002,1997-05-02,8.7,0.4,46488.67\nR002,1997-05-03,8.9,5.8,15713.54\nR002,1997-05-04,8.8,6.4,34830.61\nR002,1997-05-05,9.4,0.5,45046.58\nR002,1997-05-06,12.5,0.0,47324.99\nR002,1997-05-07,15.3,0.0,49897.99\nR002,1997-05-08,17.5,0.0,47257.19\nR002,1997-05-09,18.1,0.0,50049.12\nR002,1997-05-10,19.3,0.0,50122.14\nR002,1997-05-11,20.3,0.0,50193.31\nR002,1997-05-12,19.9,0.5,48163.46\nR002,1997-05-13,18.3,0.6,50204.64\nR002,1997-05-14,17.4,0.7,48801.99\nR002,1997-05-15,16.7,0.1,48353.15\nR002,1997-05-16,17.6,0.0,50400.0\nR002,1997-05-17,18.5,0.0,50400.0\nR002,1997-05-18,16.3,9.6,15912.51\nR002,1997-05-19,15.6,0.7,43554.77\nR002,1997-05-20,17.3,0.6,49392.42\nR002,1997-05-21,18.5,2.6,50153.72\nR002,1997-05-22,18.0,3.9,45424.63\nR002,1997-05-23,19.4,0.9,46335.23\nR002,1997-05-24,19.9,2.1,50400.0\nR002,1997-05-25,17.3,11.5,42524.46\nR002,1997-05-26,15.1,10.8,16436.05\nR002,1997-05-27,16.0,2.3,46117.49\nR002,1997-05-28,13.4,25.7,36882.46\nR002,1997-05-29,16.2,0.0,51543.75\nR002,1997-05-30,19.8,2.4,50439.07\nR002,1997-05-31,16.3,21.0,27439.35\nR002,1997-06-01,16.2,9.7,38264.99\nR002,1997-06-02,13.1,12.2,28941.72\nR002,1997-06-03,14.2,0.0,50400.0\nR002,1997-06-04,16.7,0.0,51063.46\nR002,1997-06-05,17.5,6.9,42169.16\nR002,1997-06-06,16.7,7.8,19072.63\nR002,1997-06-07,18.9,1.2,44817.32\nR002,1997-06-08,20.6,3.1,38321.41\nR002,1997-06-09,18.2,19.3,4599.96\nR002,1997-06-10,20.2,2.7,50223.86\nR002,1997-06-11,21.7,6.1,49556.76\nR002,1997-06-12,20.3,20.0,29032.38\nR002,1997-06-13,18.7,16.7,44558.99\nR002,1997-06-14,19.7,5.4,49736.01\nR002,1997-06-15,18.9,8.5,42355.81\nR002,1997-06-16,19.3,3.6,47403.54\nR002,1997-06-17,19.5,1.2,45018.09\nR002,1997-06-18,20.8,0.1,49680.6\nR002,1997-06-19,20.5,0.6,51048.2\nR002,1997-06-20,20.1,0.2,50358.78\nR002,1997-06-21,22.6,0.8,53414.44\nR002,1997-06-22,20.7,1.6,51324.22\nR002,1997-06-23,19.3,0.2,50347.63\nR002,1997-06-24,21.1,1.0,51367.85\nR002,1997-06-25,22.2,0.0,53762.13\nR002,1997-06-26,24.2,0.7,53391.28\nR002,1997-06-27,23.1,7.9,45010.69\nR002,1997-06-28,23.4,3.3,50868.79\nR002,1997-06-29,21.2,6.2,27012.25\nR002,1997-06-30,21.6,11.4,47053.7\nR002,1997-07-01,21.9,14.1,21781.76\nR002,1997-07-02,21.9,14.3,38581.77\nR002,1997-07-03,22.6,4.1,46770.64\nR002,1997-07-04,20.0,5.2,38606.85\nR002,1997-07-05,19.3,8.3,33936.31\nR002,1997-07-06,19.8,5.6,44880.79\nR002,1997-07-07,22.6,0.0,49294.9\nR002,1997-07-08,24.3,0.4,52006.14\nR002,1997-07-09,24.7,0.8,53767.74\nR002,1997-07-10,21.7,8.8,49361.54\nR002,1997-07-11,20.5,0.4,51229.76\nR002,1997-07-12,20.6,1.8,48112.15\nR002,1997-07-13,20.3,0.0,53078.15\nR002,1997-07-14,20.2,2.7,50761.44\nR002,1997-07-15,18.5,6.9,44095.96\nR002,1997-07-16,17.8,4.8,46033.48\nR002,1997-07-17,17.5,8.9,32697.19\nR002,1997-07-18,17.7,1.2,49541.06\nR002,1997-07-19,19.1,0.0,52800.81\nR002,1997-07-20,22.7,0.0,52783.73\nR002,1997-07-21,22.1,2.0,51682.46\nR002,1997-07-22,20.9,0.4,49639.79\nR002,1997-07-23,19.8,0.1,50385.8\nR002,1997-07-24,18.9,3.8,47183.33\nR002,1997-07-25,19.0,2.0,47674.06\nR002,1997-07-26,21.5,0.0,50796.12\nR002,1997-07-27,24.3,0.2,51110.2\nR002,1997-07-28,24.6,1.6,49570.93\nR002,1997-07-29,21.2,60.3,25371.41\nR002,1997-07-30,21.4,1.6,46675.08\nR002,1997-07-31,22.6,0.5,45824.29\nR002,1997-08-01,23.6,6.5,42853.05\nR002,1997-08-02,22.8,12.6,35259.7\nR002,1997-08-03,24.0,0.0,47489.07\nR002,1997-08-04,22.2,0.3,48213.39\nR002,1997-08-05,19.7,4.4,40136.5\nR002,1997-08-06,21.1,8.3,32551.77\nR002,1997-08-07,22.1,3.0,46406.94\nR002,1997-08-08,23.2,1.5,46800.0\nR002,1997-08-09,22.7,5.7,18454.36\nR002,1997-08-10,22.6,19.1,33107.6\nR002,1997-08-11,24.2,9.9,40469.71\nR002,1997-08-12,21.7,16.2,41111.97\nR002,1997-08-13,21.3,3.7,46800.0\nR002,1997-08-14,20.2,0.4,41570.64\nR002,1997-08-15,20.3,6.1,41475.61\nR002,1997-08-16,20.9,0.0,46180.28\nR002,1997-08-17,20.9,1.2,46800.0\nR002,1997-08-18,21.0,0.9,44789.44\nR002,1997-08-19,20.2,2.0,40141.28\nR002,1997-08-20,19.7,10.2,33116.11\nR002,1997-08-21,20.2,8.5,41975.5\nR002,1997-08-22,19.3,9.8,38245.01\nR002,1997-08-23,21.1,8.2,44023.01\nR002,1997-08-24,22.2,1.8,40927.55\nR002,1997-08-25,25.1,1.3,46008.55\nR002,1997-08-26,22.1,16.2,19581.97\nR002,1997-08-27,21.0,18.1,38373.26\nR002,1997-08-28,21.5,2.8,44013.74\nR002,1997-08-29,21.8,4.4,43959.16\nR002,1997-08-30,22.5,0.0,44868.11\nR002,1997-08-31,22.4,0.9,45504.65\nR002,1997-09-01,18.5,35.0,11080.68\nR002,1997-09-02,15.4,4.1,26451.77\nR002,1997-09-03,14.6,6.5,25362.47\nR002,1997-09-04,12.3,8.0,23601.63\nR002,1997-09-05,12.4,5.8,27582.91\nR002,1997-09-06,13.7,0.0,43560.88\nR002,1997-09-07,14.8,0.4,43932.62\nR002,1997-09-08,17.1,0.0,43346.21\nR002,1997-09-09,17.1,0.0,43200.0\nR002,1997-09-10,17.5,0.0,42119.61\nR002,1997-09-11,13.7,17.1,26282.12\nR002,1997-09-12,12.1,0.5,41504.08\nR002,1997-09-13,12.9,0.0,41636.03\nR002,1997-09-14,16.8,0.0,43108.37\nR002,1997-09-15,19.4,0.0,43041.03\nR002,1997-09-16,18.4,1.0,42973.6\nR002,1997-09-17,15.9,5.0,30320.42\nR002,1997-09-18,16.4,0.0,42838.37\nR002,1997-09-19,19.0,0.0,39924.36\nR002,1997-09-20,20.0,1.5,42186.44\nR002,1997-09-21,15.6,29.9,597.58\nR002,1997-09-22,13.0,0.5,37014.56\nR002,1997-09-23,13.3,0.0,40061.85\nR002,1997-09-24,14.2,0.0,41371.07\nR002,1997-09-25,13.0,3.7,27980.64\nR002,1997-09-26,10.4,1.8,40167.23\nR002,1997-09-27,10.9,0.0,39600.0\nR002,1997-09-28,11.8,3.1,1378.84\nR002,1997-09-29,11.0,6.0,28349.49\nR002,1997-09-30,10.8,3.7,37569.61\nR002,1997-10-01,10.7,2.7,20393.86\nR002,1997-10-02,11.1,14.9,0.0\nR002,1997-10-03,13.3,0.0,36454.85\nR002,1997-10-04,13.6,28.1,15462.05\nR002,1997-10-05,11.4,0.0,31833.41\nR002,1997-10-06,10.9,8.1,22270.41\nR002,1997-10-07,12.2,1.3,37882.47\nR002,1997-10-08,13.3,0.0,38630.01\nR002,1997-10-09,15.5,0.0,37754.83\nR002,1997-10-10,17.4,0.0,37525.12\nR002,1997-10-11,19.2,0.0,37137.23\nR002,1997-10-12,20.2,0.0,36541.95\nR002,1997-10-13,18.8,1.4,36000.0\nR002,1997-10-14,17.8,3.7,12586.25\nR002,1997-10-15,20.1,0.0,36050.15\nR002,1997-10-16,22.6,0.0,36102.24\nR002,1997-10-17,18.6,7.5,23582.97\nR002,1997-10-18,16.1,10.7,16582.82\nR002,1997-10-19,15.1,23.0,0.0\nR002,1997-10-20,15.4,0.0,33026.5\nR002,1997-10-21,14.5,8.0,10263.56\nR002,1997-10-22,11.5,12.3,70.6\nR002,1997-10-23,11.1,0.1,33505.77\nR002,1997-10-24,13.2,0.0,36000.0\nR002,1997-10-25,14.0,14.4,11799.3\nR002,1997-10-26,11.6,70.0,0.0\nR002,1997-10-27,9.3,14.8,2728.9\nR002,1997-10-28,8.9,6.7,17962.39\nR002,1997-10-29,8.7,7.8,20053.38\nR002,1997-10-30,5.7,8.5,29949.24\nR002,1997-10-31,2.5,0.0,36000.0\nR002,1997-11-01,4.7,0.0,36000.0\nR002,1997-11-02,6.8,0.0,35937.36\nR002,1997-11-03,7.6,0.0,35857.42\nR002,1997-11-04,8.4,13.6,7961.33\nR002,1997-11-05,4.1,4.7,33195.07\nR002,1997-11-06,2.9,0.0,35524.97\nR002,1997-11-07,7.2,0.0,35372.37\nR002,1997-11-08,8.8,0.0,35222.31\nR002,1997-11-09,10.1,0.0,35074.97\nR002,1997-11-10,9.9,0.0,34930.36\nR002,1997-11-11,9.2,0.0,34787.12\nR002,1997-11-12,8.8,0.0,30854.01\nR002,1997-11-13,9.7,0.0,34504.07\nR002,1997-11-14,10.0,0.0,34364.59\nR002,1997-11-15,11.5,0.0,29316.0\nR002,1997-11-16,12.4,0.3,21712.43\nR002,1997-11-17,12.0,0.0,32061.19\nR002,1997-11-18,9.3,7.1,2847.38\nR002,1997-11-19,11.5,1.0,25285.65\nR002,1997-11-20,11.0,6.8,15767.24\nR002,1997-11-21,11.1,0.0,28322.46\nR002,1997-11-22,11.4,0.0,29003.45\nR002,1997-11-23,10.3,0.5,8199.27\nR002,1997-11-24,8.5,0.0,30545.68\nR002,1997-11-25,8.4,0.0,31561.94\nR002,1997-11-26,8.1,0.0,31256.79\nR002,1997-11-27,7.2,0.0,30246.83\nR002,1997-11-28,7.3,3.8,18914.25\nR002,1997-11-29,7.5,2.6,21600.0\nR002,1997-11-30,6.9,0.0,19633.64\nR002,1997-12-01,8.1,0.0,30325.17\nR002,1997-12-02,8.8,1.7,28818.74\nR002,1997-12-03,9.4,2.8,11941.39\nR002,1997-12-04,9.3,2.9,28800.0\nR002,1997-12-05,7.6,30.8,22187.72\nR002,1997-12-06,5.7,7.9,12722.98\nR002,1997-12-07,6.6,2.8,3037.52\nR002,1997-12-08,8.1,17.5,8888.31\nR002,1997-12-09,6.3,20.8,0.0\nR002,1997-12-10,1.3,0.9,28436.08\nR002,1997-12-11,0.2,0.0,28800.0\nR002,1997-12-12,3.4,0.0,24192.44\nR002,1997-12-13,6.1,0.1,11983.98\nR002,1997-12-14,7.4,8.3,4860.41\nR002,1997-12-15,6.7,16.3,0.0\nR002,1997-12-16,4.6,23.3,0.0\nR002,1997-12-17,2.0,11.1,0.0\nR002,1997-12-18,-2.2,3.5,17133.82\nR002,1997-12-19,-2.8,0.1,23350.16\nR002,1997-12-20,-0.7,0.0,30769.55\nR002,1997-12-21,3.5,5.4,10191.98\nR002,1997-12-22,6.5,3.4,18998.38\nR002,1997-12-23,7.2,0.0,22844.31\nR002,1997-12-24,5.4,2.3,7469.97\nR002,1997-12-25,3.7,3.2,10799.04\nR002,1997-12-26,2.8,1.5,25765.6\nR002,1997-12-27,3.3,0.0,28800.0\nR002,1997-12-28,4.6,0.0,31407.54\nR002,1997-12-29,5.8,0.0,29818.13\nR002,1997-12-30,7.0,0.0,24088.4\nR002,1997-12-31,3.7,6.8,3151.15\nR002,1998-01-01,0.7,0.1,31608.83\nR002,1998-01-02,-1.0,0.0,31663.59\nR002,1998-01-03,-2.6,0.0,31719.86\nR002,1998-01-04,-1.5,0.0,31777.53\nR002,1998-01-05,2.4,0.0,29126.67\nR002,1998-01-06,2.8,6.5,20077.72\nR002,1998-01-07,2.3,5.5,20420.87\nR002,1998-01-08,1.1,4.1,30256.7\nR002,1998-01-09,1.6,11.4,0.0\nR002,1998-01-10,1.2,34.1,639.95\nR002,1998-01-11,-2.8,0.0,32214.38\nR002,1998-01-12,-3.3,5.9,0.0\nR002,1998-01-13,-0.2,10.6,0.0\nR002,1998-01-14,3.7,3.5,15530.02\nR002,1998-01-15,3.3,0.0,32400.0\nR002,1998-01-16,3.2,0.0,23181.74\nR002,1998-01-17,3.0,0.0,26972.93\nR002,1998-01-18,2.5,0.0,29242.54\nR002,1998-01-19,2.4,2.0,20406.15\nR002,1998-01-20,2.3,0.0,32335.2\nR002,1998-01-21,3.1,0.0,32400.0\nR002,1998-01-22,1.9,0.0,32400.0\nR002,1998-01-23,1.6,0.0,32400.0\nR002,1998-01-24,3.7,0.0,32400.0\nR002,1998-01-25,4.2,0.0,29884.99\nR002,1998-01-26,3.4,5.2,0.0\nR002,1998-01-27,2.9,2.7,18371.63\nR002,1998-01-28,2.7,0.0,29730.09\nR002,1998-01-29,3.9,12.2,0.0\nR002,1998-01-30,0.2,12.4,17421.69\nR002,1998-01-31,-4.9,0.0,32417.59\nR002,1998-02-01,-1.3,7.2,32400.0\nR002,1998-02-02,0.2,10.3,442.83\nR002,1998-02-03,3.0,12.3,266.0\nR002,1998-02-04,5.3,16.2,4421.12\nR002,1998-02-05,7.4,19.8,0.0\nR002,1998-02-06,8.4,24.4,0.0\nR002,1998-02-07,3.9,22.1,15204.86\nR002,1998-02-08,-4.2,1.1,32413.95\nR002,1998-02-09,-4.8,0.0,34947.56\nR002,1998-02-10,-1.6,0.0,10347.31\nR002,1998-02-11,-2.1,0.0,35460.13\nR002,1998-02-12,0.1,2.7,32441.82\nR002,1998-02-13,2.2,22.7,0.0\nR002,1998-02-14,-4.3,3.3,35569.38\nR002,1998-02-15,-6.5,4.7,0.0\nR002,1998-02-16,-4.3,1.0,36312.59\nR002,1998-02-17,1.0,22.4,15223.52\nR002,1998-02-18,-1.4,21.3,31032.84\nR002,1998-02-19,-6.0,0.0,37745.08\nR002,1998-02-20,1.0,0.3,34909.51\nR002,1998-02-21,3.2,0.0,38130.83\nR002,1998-02-22,5.2,0.0,38430.72\nR002,1998-02-23,8.2,0.0,38606.58\nR002,1998-02-24,7.8,0.0,38783.58\nR002,1998-02-25,6.0,0.2,38961.56\nR002,1998-02-26,0.4,13.3,36266.41\nR002,1998-02-27,-2.7,0.0,37609.27\nR002,1998-02-28,2.3,0.0,39437.86\nR002,1998-03-01,4.2,0.0,39539.85\nR002,1998-03-02,3.9,0.0,39600.0\nR002,1998-03-03,2.7,6.5,39600.0\nR002,1998-03-04,2.8,1.8,28800.0\nR002,1998-03-05,5.6,6.5,5526.37\nR002,1998-03-06,7.8,0.0,39600.0\nR002,1998-03-07,4.4,13.8,22947.43\nR002,1998-03-08,4.4,0.0,39600.0\nR002,1998-03-09,9.0,0.0,39455.99\nR002,1998-03-10,9.9,0.0,34512.94\nR002,1998-03-11,11.3,0.0,34397.77\nR002,1998-03-12,8.7,7.5,18936.03\nR002,1998-03-13,2.5,5.2,36541.05\nR002,1998-03-14,2.7,3.5,12885.04\nR002,1998-03-15,4.1,0.0,39600.0\nR002,1998-03-16,4.9,3.9,13940.39\nR002,1998-03-17,5.2,7.2,1236.52\nR002,1998-03-18,4.3,0.4,33783.65\nR002,1998-03-19,1.6,14.3,23920.92\nR002,1998-03-20,1.3,15.2,21850.68\nR002,1998-03-21,3.0,1.3,36456.38\nR002,1998-03-22,2.8,11.4,10997.46\nR002,1998-03-23,2.8,10.1,5448.98\nR002,1998-03-24,0.9,3.9,26615.57\nR002,1998-03-25,-0.4,0.0,40881.3\nR002,1998-03-26,1.0,0.0,36904.43\nR002,1998-03-27,6.9,0.0,42306.8\nR002,1998-03-28,8.8,1.7,30086.6\nR002,1998-03-29,6.8,9.6,0.0\nR002,1998-03-30,5.2,5.5,163.77\nR002,1998-03-31,5.0,2.6,42731.64\nR002,1998-04-01,6.1,0.1,42497.34\nR002,1998-04-02,8.9,0.0,45195.24\nR002,1998-04-03,9.1,0.0,44311.41\nR002,1998-04-04,10.0,0.2,44703.87\nR002,1998-04-05,10.7,0.0,45926.98\nR002,1998-04-06,12.4,0.0,45998.49\nR002,1998-04-07,13.3,0.0,46069.98\nR002,1998-04-08,14.7,0.0,41790.09\nR002,1998-04-09,14.9,0.0,46212.86\nR002,1998-04-10,16.1,0.0,44229.58\nR002,1998-04-11,16.8,1.8,46355.56\nR002,1998-04-12,19.0,0.0,46426.81\nR002,1998-04-13,19.3,0.0,46498.0\nR002,1998-04-14,15.1,2.8,39600.0\nR002,1998-04-15,12.5,7.8,38286.19\nR002,1998-04-16,10.4,2.1,41090.54\nR002,1998-04-17,10.0,3.1,14485.56\nR002,1998-04-18,13.3,3.7,37598.1\nR002,1998-04-19,14.3,4.2,36081.01\nR002,1998-04-20,11.2,18.7,6835.13\nR002,1998-04-21,9.5,3.5,4076.47\nR002,1998-04-22,8.7,7.9,34030.03\nR002,1998-04-23,8.7,2.2,34170.72\nR002,1998-04-24,9.1,4.6,37197.53\nR002,1998-04-25,10.8,0.0,45528.87\nR002,1998-04-26,14.0,0.1,47382.89\nR002,1998-04-27,14.5,2.4,25161.93\nR002,1998-04-28,11.9,14.2,17401.65\nR002,1998-04-29,11.3,6.0,13557.15\nR002,1998-04-30,11.7,3.4,29896.04\nR002,1998-05-01,12.4,3.7,42941.61\nR002,1998-05-02,13.1,7.5,45629.36\nR002,1998-05-03,14.9,0.0,48785.81\nR002,1998-05-04,19.0,0.2,36570.35\nR002,1998-05-05,21.0,0.5,39089.85\nR002,1998-05-06,21.5,3.8,33452.43\nR002,1998-05-07,17.3,4.7,43370.82\nR002,1998-05-08,18.7,0.6,42847.01\nR002,1998-05-09,19.3,1.0,41914.6\nR002,1998-05-10,21.0,3.7,14560.35\nR002,1998-05-11,22.1,6.2,34413.77\nR002,1998-05-12,18.3,26.8,12330.12\nR002,1998-05-13,17.5,9.6,35868.65\nR002,1998-05-14,15.3,11.1,22255.42\nR002,1998-05-15,14.4,12.3,2327.61\nR002,1998-05-16,15.3,2.3,42129.72\nR002,1998-05-17,18.5,0.0,37481.14\nR002,1998-05-18,20.1,4.6,41217.61\nR002,1998-05-19,19.1,10.0,36499.82\nR002,1998-05-20,15.7,18.2,32397.01\nR002,1998-05-21,14.0,10.7,4921.67\nR002,1998-05-22,13.4,10.5,32883.32\nR002,1998-05-23,15.4,1.7,46783.31\nR002,1998-05-24,13.6,4.8,36368.08\nR002,1998-05-25,15.6,0.1,49153.61\nR002,1998-05-26,19.2,0.0,50539.78\nR002,1998-05-27,23.1,0.3,50874.78\nR002,1998-05-28,20.4,12.2,38992.2\nR002,1998-05-29,18.8,20.1,723.37\nR002,1998-05-30,18.9,16.9,14827.16\nR002,1998-05-31,17.5,15.1,26702.44\nR002,1998-06-01,17.6,15.2,44457.24\nR002,1998-06-02,17.4,13.2,29989.97\nR002,1998-06-03,18.0,13.4,33160.06\nR002,1998-06-04,18.5,12.1,42941.16\nR002,1998-06-05,18.9,9.7,42591.89\nR002,1998-06-06,21.1,4.7,47682.12\nR002,1998-06-07,22.2,13.0,37917.13\nR002,1998-06-08,23.1,1.7,52091.21\nR002,1998-06-09,22.9,2.5,52075.74\nR002,1998-06-10,23.4,5.7,51328.46\nR002,1998-06-11,23.4,8.3,46685.62\nR002,1998-06-12,24.6,2.5,50733.44\nR002,1998-06-13,25.3,3.4,50919.36\nR002,1998-06-14,25.5,2.0,53477.46\nR002,1998-06-15,24.6,2.0,50788.24\nR002,1998-06-16,24.5,0.9,44455.1\nR002,1998-06-17,25.2,1.4,48139.24\nR002,1998-06-18,24.9,6.1,48799.46\nR002,1998-06-19,24.8,2.0,53432.4\nR002,1998-06-20,25.3,0.5,51234.97\nR002,1998-06-21,26.1,1.3,53418.07\nR002,1998-06-22,21.7,24.8,13880.9\nR002,1998-06-23,20.0,3.2,43280.49\nR002,1998-06-24,20.2,0.7,50844.8\nR002,1998-06-25,21.4,0.0,53959.55\nR002,1998-06-26,21.3,2.1,52362.44\nR002,1998-06-27,21.0,6.5,51175.84\nR002,1998-06-28,20.7,3.4,45315.19\nR002,1998-06-29,21.9,0.1,51275.31\nR002,1998-06-30,21.9,2.3,50326.88\nR002,1998-07-01,22.4,2.1,48151.23\nR002,1998-07-02,22.4,4.2,43322.33\nR002,1998-07-03,23.2,0.0,49912.84\nR002,1998-07-04,20.4,5.3,36577.68\nR002,1998-07-05,19.3,4.9,41120.09\nR002,1998-07-06,18.5,21.4,48552.06\nR002,1998-07-07,17.7,0.9,50059.41\nR002,1998-07-08,19.6,0.1,52747.4\nR002,1998-07-09,23.5,0.0,53806.34\nR002,1998-07-10,21.3,15.0,44652.13\nR002,1998-07-11,19.2,0.1,52476.71\nR002,1998-07-12,20.2,0.0,51495.35\nR002,1998-07-13,20.8,0.0,51944.61\nR002,1998-07-14,22.1,0.0,53642.87\nR002,1998-07-15,25.2,0.0,53603.85\nR002,1998-07-16,26.6,0.0,53468.36\nR002,1998-07-17,26.1,5.5,51338.87\nR002,1998-07-18,22.6,6.2,42293.31\nR002,1998-07-19,22.5,0.0,48419.78\nR002,1998-07-20,24.4,0.0,52380.6\nR002,1998-07-21,25.7,1.3,51684.16\nR002,1998-07-22,24.4,1.0,50688.02\nR002,1998-07-23,24.7,1.0,49675.52\nR002,1998-07-24,25.8,0.0,52275.36\nR002,1998-07-25,26.2,0.0,50112.31\nR002,1998-07-26,25.8,0.5,50859.75\nR002,1998-07-27,25.5,1.2,51463.54\nR002,1998-07-28,25.2,3.4,50034.3\nR002,1998-07-29,25.5,1.9,49954.19\nR002,1998-07-30,25.7,4.8,50005.61\nR002,1998-07-31,24.3,5.6,45386.31\nR002,1998-08-01,25.0,0.5,48040.9\nR002,1998-08-02,26.1,0.0,49132.18\nR002,1998-08-03,26.1,0.0,48846.99\nR002,1998-08-04,25.8,0.6,47879.1\nR002,1998-08-05,25.4,1.5,47550.44\nR002,1998-08-06,25.8,0.3,47665.9\nR002,1998-08-07,23.9,4.2,43252.33\nR002,1998-08-08,23.7,0.2,44503.06\nR002,1998-08-09,23.4,0.0,46986.3\nR002,1998-08-10,23.2,1.3,47492.57\nR002,1998-08-11,21.8,0.8,46800.0\nR002,1998-08-12,19.3,0.0,47412.85\nR002,1998-08-13,19.0,0.0,46800.0\nR002,1998-08-14,20.5,0.0,47471.87\nR002,1998-08-15,21.5,2.7,47167.64\nR002,1998-08-16,19.8,0.0,45607.54\nR002,1998-08-17,20.0,0.0,43247.52\nR002,1998-08-18,20.5,0.6,46800.0\nR002,1998-08-19,22.3,0.1,46800.0\nR002,1998-08-20,22.3,1.2,45967.48\nR002,1998-08-21,22.4,0.7,46800.0\nR002,1998-08-22,23.4,0.8,46800.0\nR002,1998-08-23,23.9,0.0,46800.0\nR002,1998-08-24,22.9,0.0,45137.51\nR002,1998-08-25,24.7,0.0,46800.0\nR002,1998-08-26,24.6,0.0,46800.0\nR002,1998-08-27,23.2,0.0,46126.97\nR002,1998-08-28,24.9,0.0,46651.37\nR002,1998-08-29,26.6,0.2,46547.69\nR002,1998-08-30,25.9,0.1,46443.55\nR002,1998-08-31,22.0,5.1,37846.79\nR002,1998-09-01,20.9,3.9,37074.28\nR002,1998-09-02,16.5,3.8,36972.42\nR002,1998-09-03,14.5,1.1,43712.99\nR002,1998-09-04,13.8,0.0,44083.85\nR002,1998-09-05,13.1,1.2,42131.58\nR002,1998-09-06,13.5,1.6,42733.62\nR002,1998-09-07,15.0,0.0,43915.49\nR002,1998-09-08,17.2,4.0,37407.64\nR002,1998-09-09,18.0,1.4,39414.69\nR002,1998-09-10,17.9,0.0,40683.42\nR002,1998-09-11,19.9,0.0,43200.0\nR002,1998-09-12,21.1,0.0,43200.0\nR002,1998-09-13,22.5,0.5,42736.02\nR002,1998-09-14,24.0,0.0,43123.72\nR002,1998-09-15,25.5,0.0,43056.31\nR002,1998-09-16,23.7,2.6,42988.8\nR002,1998-09-17,20.3,4.1,36271.39\nR002,1998-09-18,20.3,3.1,39195.07\nR002,1998-09-19,22.5,0.0,42781.28\nR002,1998-09-20,23.5,0.2,42717.77\nR002,1998-09-21,20.5,7.5,38540.29\nR002,1998-09-22,21.0,0.5,39208.33\nR002,1998-09-23,18.3,40.2,30024.32\nR002,1998-09-24,15.4,12.4,159.78\nR002,1998-09-25,17.0,1.2,33875.76\nR002,1998-09-26,15.6,0.0,38877.43\nR002,1998-09-27,16.2,0.0,39875.59\nR002,1998-09-28,17.7,0.0,41187.8\nR002,1998-09-29,21.6,0.0,41217.54\nR002,1998-09-30,22.8,0.0,39483.86\nR002,1998-10-01,16.6,7.1,6008.03\nR002,1998-10-02,14.7,0.0,38655.59\nR002,1998-10-03,15.8,0.0,39648.29\nR002,1998-10-04,18.4,0.0,37054.1\nR002,1998-10-05,18.1,1.1,37433.72\nR002,1998-10-06,16.7,0.0,38040.36\nR002,1998-10-07,12.7,1.8,29422.24\nR002,1998-10-08,11.8,0.0,37929.41\nR002,1998-10-09,13.3,0.0,37736.0\nR002,1998-10-10,14.8,0.0,37505.87\nR002,1998-10-11,19.1,0.0,36476.18\nR002,1998-10-12,21.1,0.0,36855.75\nR002,1998-10-13,23.9,0.0,36258.0\nR002,1998-10-14,23.6,0.0,32958.51\nR002,1998-10-15,20.6,0.0,36362.31\nR002,1998-10-16,18.5,0.0,36000.0\nR002,1998-10-17,15.5,3.1,24303.61\nR002,1998-10-18,14.5,3.0,28282.76\nR002,1998-10-19,13.6,6.5,24707.9\nR002,1998-10-20,14.4,0.0,36000.0\nR002,1998-10-21,15.3,2.1,36000.0\nR002,1998-10-22,9.3,1.4,36000.0\nR002,1998-10-23,8.0,0.0,36000.0\nR002,1998-10-24,10.5,0.0,36000.0\nR002,1998-10-25,15.7,0.0,32757.95\nR002,1998-10-26,17.8,0.1,34334.42\nR002,1998-10-27,17.6,0.0,13169.2\nR002,1998-10-28,14.1,24.1,1348.08\nR002,1998-10-29,13.1,21.0,9389.45\nR002,1998-10-30,13.8,0.0,33408.45\nR002,1998-10-31,13.0,0.0,34586.74\nR002,1998-11-01,10.3,10.1,24849.39\nR002,1998-11-02,10.2,3.2,19327.7\nR002,1998-11-03,10.9,8.3,21051.75\nR002,1998-11-04,11.6,0.0,35800.27\nR002,1998-11-05,15.1,0.0,29692.95\nR002,1998-11-06,13.0,7.3,22891.36\nR002,1998-11-07,11.6,4.3,8332.58\nR002,1998-11-08,8.4,15.3,0.0\nR002,1998-11-09,1.9,0.0,25680.04\nR002,1998-11-10,0.3,16.7,18046.09\nR002,1998-11-11,-3.1,0.0,34825.49\nR002,1998-11-12,5.5,0.0,31968.11\nR002,1998-11-13,7.9,0.0,32062.83\nR002,1998-11-14,9.8,17.2,0.0\nR002,1998-11-15,10.1,13.6,8843.11\nR002,1998-11-16,9.1,0.4,25067.37\nR002,1998-11-17,7.3,18.0,20878.03\nR002,1998-11-18,7.6,17.7,9652.49\nR002,1998-11-19,4.3,3.7,22322.4\nR002,1998-11-20,4.2,0.4,28928.64\nR002,1998-11-21,8.5,0.0,31999.71\nR002,1998-11-22,9.9,0.0,29849.94\nR002,1998-11-23,8.4,0.2,29919.71\nR002,1998-11-24,9.2,0.0,31475.65\nR002,1998-11-25,8.9,2.9,23522.09\nR002,1998-11-26,8.9,0.0,28800.0\nR002,1998-11-27,8.5,0.1,25878.94\nR002,1998-11-28,9.2,0.1,12972.03\nR002,1998-11-29,8.0,3.9,13809.95\nR002,1998-11-30,7.6,1.4,1898.74\nR002,1998-12-01,5.6,7.8,11783.12\nR002,1998-12-02,5.0,3.2,16788.44\nR002,1998-12-03,2.9,0.0,29968.77\nR002,1998-12-04,3.2,2.7,28488.25\nR002,1998-12-05,6.5,2.6,20509.58\nR002,1998-12-06,8.7,11.1,12153.95\nR002,1998-12-07,9.1,14.6,0.0\nR002,1998-12-08,8.8,2.1,26985.89\nR002,1998-12-09,8.7,3.6,3600.0\nR002,1998-12-10,8.1,56.6,0.0\nR002,1998-12-11,1.5,11.4,5379.72\nR002,1998-12-12,4.2,4.9,1300.31\nR002,1998-12-13,4.2,7.4,1568.88\nR002,1998-12-14,2.4,0.0,29644.74\nR002,1998-12-15,2.8,2.8,14305.43\nR002,1998-12-16,4.9,1.5,26036.32\nR002,1998-12-17,3.2,2.8,936.03\nR002,1998-12-18,5.1,0.0,25873.53\nR002,1998-12-19,4.1,0.0,30782.75\nR002,1998-12-20,4.5,4.3,14400.0\nR002,1998-12-21,6.3,5.8,15815.15\nR002,1998-12-22,6.4,0.0,29180.35\nR002,1998-12-23,5.2,0.9,23557.12\nR002,1998-12-24,4.5,6.3,4352.73\nR002,1998-12-25,5.1,4.2,20180.12\nR002,1998-12-26,4.5,0.2,27539.97\nR002,1998-12-27,2.7,0.0,31042.71\nR002,1998-12-28,1.3,0.0,31401.56\nR002,1998-12-29,4.5,0.0,31447.34\nR002,1998-12-30,5.5,0.3,10926.67\nR002,1998-12-31,3.4,3.9,22072.4\nR002,1999-01-01,-1.9,0.0,31594.47\nR002,1999-01-02,0.6,0.0,31646.39\nR002,1999-01-03,1.7,0.0,31699.57\nR002,1999-01-04,2.0,0.0,28800.0\nR002,1999-01-05,4.8,0.0,31811.09\nR002,1999-01-06,2.6,0.0,31870.41\nR002,1999-01-07,4.0,0.0,30200.09\nR002,1999-01-08,5.5,0.0,29137.34\nR002,1999-01-09,6.5,0.0,30807.02\nR002,1999-01-10,6.0,10.6,7301.47\nR002,1999-01-11,3.3,12.1,32197.52\nR002,1999-01-12,-0.4,0.0,21874.32\nR002,1999-01-13,4.5,0.0,20752.19\nR002,1999-01-14,7.1,0.0,29692.56\nR002,1999-01-15,7.0,0.0,25766.88\nR002,1999-01-16,5.7,1.5,32400.0\nR002,1999-01-17,2.9,18.0,20594.31\nR002,1999-01-18,2.3,0.0,32400.0\nR002,1999-01-19,3.3,0.0,32400.0\nR002,1999-01-20,3.1,0.0,32400.0\nR002,1999-01-21,2.8,0.0,32400.0\nR002,1999-01-22,2.5,0.0,32400.0\nR002,1999-01-23,2.4,0.0,32400.0\nR002,1999-01-24,2.1,3.8,19057.77\nR002,1999-01-25,0.6,0.0,32113.47\nR002,1999-01-26,2.0,0.0,32400.0\nR002,1999-01-27,2.2,0.0,32400.0\nR002,1999-01-28,4.2,0.0,32400.0\nR002,1999-01-29,6.2,0.0,29897.77\nR002,1999-01-30,8.0,0.0,27591.76\nR002,1999-01-31,8.0,0.0,22921.59\nR002,1999-02-01,5.7,3.5,21554.98\nR002,1999-02-02,5.0,26.3,0.0\nR002,1999-02-03,-1.1,18.6,321.29\nR002,1999-02-04,-3.6,0.0,27087.99\nR002,1999-02-05,0.2,1.3,32037.72\nR002,1999-02-06,2.7,0.0,33552.76\nR002,1999-02-07,7.0,5.3,24658.58\nR002,1999-02-08,4.3,22.5,11168.17\nR002,1999-02-09,2.8,3.9,33269.13\nR002,1999-02-10,3.8,4.7,34588.92\nR002,1999-02-11,7.0,0.0,35110.56\nR002,1999-02-12,7.5,0.0,36170.51\nR002,1999-02-13,9.4,0.0,36544.9\nR002,1999-02-14,7.1,0.0,35561.03\nR002,1999-02-15,6.4,0.8,29975.7\nR002,1999-02-16,3.8,13.2,18521.7\nR002,1999-02-17,3.4,0.0,37008.63\nR002,1999-02-18,5.0,9.6,12923.25\nR002,1999-02-19,6.4,4.8,33475.24\nR002,1999-02-20,1.7,22.0,5624.07\nR002,1999-02-21,-1.3,5.2,23891.75\nR002,1999-02-22,2.5,14.4,13434.99\nR002,1999-02-23,7.8,1.9,3787.46\nR002,1999-02-24,9.2,1.1,28530.82\nR002,1999-02-25,3.0,23.3,18884.89\nR002,1999-02-26,1.9,0.0,39104.72\nR002,1999-02-27,2.8,1.0,26550.5\nR002,1999-02-28,3.3,4.8,29632.18\nR002,1999-03-01,6.3,0.1,39519.35\nR002,1999-03-02,8.6,0.0,39600.0\nR002,1999-03-03,9.3,0.0,39600.0\nR002,1999-03-04,7.0,0.0,39600.0\nR002,1999-03-05,9.2,0.0,37624.02\nR002,1999-03-06,11.6,0.0,39600.0\nR002,1999-03-07,9.5,0.0,39600.0\nR002,1999-03-08,9.9,0.0,39600.0\nR002,1999-03-09,9.3,0.1,36639.47\nR002,1999-03-10,8.6,2.8,7493.06\nR002,1999-03-11,6.9,1.7,1337.54\nR002,1999-03-12,7.4,0.0,33785.7\nR002,1999-03-13,6.4,1.9,39600.0\nR002,1999-03-14,5.7,5.3,27948.82\nR002,1999-03-15,4.3,22.6,0.0\nR002,1999-03-16,2.5,9.9,0.0\nR002,1999-03-17,2.3,2.9,32742.44\nR002,1999-03-18,3.9,0.0,37704.75\nR002,1999-03-19,7.3,5.3,520.27\nR002,1999-03-20,8.4,0.1,37926.93\nR002,1999-03-21,6.6,10.7,0.0\nR002,1999-03-22,4.6,3.9,28484.75\nR002,1999-03-23,4.7,6.5,12873.2\nR002,1999-03-24,5.3,0.0,40359.67\nR002,1999-03-25,6.1,0.6,29943.44\nR002,1999-03-26,6.2,4.6,5195.76\nR002,1999-03-27,6.1,0.6,24136.37\nR002,1999-03-28,8.5,0.0,41276.54\nR002,1999-03-29,9.6,0.0,42655.18\nR002,1999-03-30,9.9,0.2,41694.99\nR002,1999-03-31,11.6,0.0,43437.92\nR002,1999-04-01,10.9,7.4,28374.16\nR002,1999-04-02,9.4,1.1,34926.39\nR002,1999-04-03,7.7,3.1,37738.76\nR002,1999-04-04,8.0,2.7,44176.72\nR002,1999-04-05,5.6,8.9,30982.98\nR002,1999-04-06,5.0,0.8,45968.62\nR002,1999-04-07,5.8,0.0,43615.78\nR002,1999-04-08,8.0,0.0,44944.32\nR002,1999-04-09,9.2,0.0,43949.47\nR002,1999-04-10,9.9,0.0,43301.49\nR002,1999-04-11,10.7,0.0,46336.99\nR002,1999-04-12,11.8,2.3,29451.68\nR002,1999-04-13,8.7,16.9,4143.05\nR002,1999-04-14,9.4,0.4,46549.51\nR002,1999-04-15,9.7,0.7,33432.64\nR002,1999-04-16,10.9,5.5,30921.19\nR002,1999-04-17,12.0,0.1,46103.12\nR002,1999-04-18,15.2,0.0,44528.18\nR002,1999-04-19,15.9,1.8,46800.0\nR002,1999-04-20,12.3,0.6,35029.06\nR002,1999-04-21,12.9,1.1,44846.91\nR002,1999-04-22,11.3,9.6,8762.32\nR002,1999-04-23,10.6,8.2,10866.5\nR002,1999-04-24,12.1,0.0,42997.46\nR002,1999-04-25,14.1,0.3,46800.0\nR002,1999-04-26,16.3,0.0,47390.53\nR002,1999-04-27,17.6,0.0,46997.26\nR002,1999-04-28,15.6,4.0,34722.11\nR002,1999-04-29,11.8,20.0,1763.57\nR002,1999-04-30,12.0,9.1,29379.06\nR002,1999-05-01,15.2,0.6,44397.21\nR002,1999-05-02,15.7,3.3,46719.89\nR002,1999-05-03,14.5,6.7,43240.56\nR002,1999-05-04,11.6,2.0,45161.12\nR002,1999-05-05,9.3,10.5,32400.0\nR002,1999-05-06,9.2,18.8,998.48\nR002,1999-05-07,6.9,8.2,23799.18\nR002,1999-05-08,7.4,0.9,47275.33\nR002,1999-05-09,9.9,0.0,50007.31\nR002,1999-05-10,12.8,0.0,50080.48\nR002,1999-05-11,15.1,0.0,47865.02\nR002,1999-05-12,15.7,0.0,50222.75\nR002,1999-05-13,14.1,5.4,18871.4\nR002,1999-05-14,13.5,4.6,37898.0\nR002,1999-05-15,15.3,1.2,44128.51\nR002,1999-05-16,18.0,2.1,49623.03\nR002,1999-05-17,15.9,25.2,27821.57\nR002,1999-05-18,9.9,11.5,44187.73\nR002,1999-05-19,8.2,16.6,24445.16\nR002,1999-05-20,8.2,6.8,28829.1\nR002,1999-05-21,11.4,6.7,46112.03\nR002,1999-05-22,14.0,0.0,48641.9\nR002,1999-05-23,17.4,0.0,50400.0\nR002,1999-05-24,19.5,0.6,50400.0\nR002,1999-05-25,17.5,7.2,34390.82\nR002,1999-05-26,16.2,16.6,26401.8\nR002,1999-05-27,15.5,2.7,41601.54\nR002,1999-05-28,16.8,0.0,50511.38\nR002,1999-05-29,17.2,0.1,48705.4\nR002,1999-05-30,17.1,13.4,10929.19\nR002,1999-05-31,17.0,4.5,32703.9\nR002,1999-06-01,17.2,4.0,45645.94\nR002,1999-06-02,17.4,3.5,44336.62\nR002,1999-06-03,17.7,3.3,47260.9\nR002,1999-06-04,17.7,10.8,44817.03\nR002,1999-06-05,17.3,17.6,40975.53\nR002,1999-06-06,17.0,9.1,49729.13\nR002,1999-06-07,19.5,1.7,48727.98\nR002,1999-06-08,20.0,8.7,38281.17\nR002,1999-06-09,21.4,3.8,47784.35\nR002,1999-06-10,22.3,8.5,50599.19\nR002,1999-06-11,23.2,2.1,49201.34\nR002,1999-06-12,21.2,8.8,27667.59\nR002,1999-06-13,19.8,9.5,19344.72\nR002,1999-06-14,21.0,4.0,43079.05\nR002,1999-06-15,21.3,1.4,53472.44\nR002,1999-06-16,21.2,0.7,51014.08\nR002,1999-06-17,21.0,4.2,45103.52\nR002,1999-06-18,20.7,6.4,50574.7\nR002,1999-06-19,22.2,0.8,52921.75\nR002,1999-06-20,23.5,0.9,53426.83\nR002,1999-06-21,20.9,7.3,50844.96\nR002,1999-06-22,20.9,4.4,48718.27\nR002,1999-06-23,22.8,0.9,50819.71\nR002,1999-06-24,18.5,23.4,35673.77\nR002,1999-06-25,17.7,16.6,33650.47\nR002,1999-06-26,18.1,11.0,39459.12\nR002,1999-06-27,18.0,3.2,28231.12\nR002,1999-06-28,18.8,3.9,47949.81\nR002,1999-06-29,19.2,2.9,34102.92\nR002,1999-06-30,20.1,2.2,47170.76\nR002,1999-07-01,20.6,1.8,50216.01\nR002,1999-07-02,21.8,2.3,53942.74\nR002,1999-07-03,21.3,1.7,53930.21\nR002,1999-07-04,21.8,5.7,53383.47\nR002,1999-07-05,22.1,1.5,49313.56\nR002,1999-07-06,23.5,1.5,49870.67\nR002,1999-07-07,24.3,0.0,52667.19\nR002,1999-07-08,24.5,0.2,51947.56\nR002,1999-07-09,24.8,0.9,50868.72\nR002,1999-07-10,25.3,0.3,53734.99\nR002,1999-07-11,24.9,1.0,53417.73\nR002,1999-07-12,24.2,0.0,50373.97\nR002,1999-07-13,22.0,1.3,52180.83\nR002,1999-07-14,21.9,0.0,53650.67\nR002,1999-07-15,23.2,0.5,53612.85\nR002,1999-07-16,24.1,0.3,52749.34\nR002,1999-07-17,23.5,3.1,52269.78\nR002,1999-07-18,23.3,6.5,51091.8\nR002,1999-07-19,22.4,10.7,41671.06\nR002,1999-07-20,21.8,1.6,42669.99\nR002,1999-07-21,21.2,0.0,49224.96\nR002,1999-07-22,20.7,0.0,49618.2\nR002,1999-07-23,21.9,0.0,47799.58\nR002,1999-07-24,23.1,0.5,49353.75\nR002,1999-07-25,24.6,0.0,51899.83\nR002,1999-07-26,25.3,0.0,51863.55\nR002,1999-07-27,26.5,0.1,51477.64\nR002,1999-07-28,27.3,0.1,51093.32\nR002,1999-07-29,23.5,8.9,22696.5\nR002,1999-07-30,23.6,4.9,49300.0\nR002,1999-07-31,24.7,3.1,49699.95\nR002,1999-08-01,23.7,4.2,48965.55\nR002,1999-08-02,23.9,5.8,44418.74\nR002,1999-08-03,25.8,3.1,46800.0\nR002,1999-08-04,25.8,5.8,43094.15\nR002,1999-08-05,24.2,17.1,31134.73\nR002,1999-08-06,22.4,6.1,44954.73\nR002,1999-08-07,21.8,0.0,47717.66\nR002,1999-08-08,22.7,0.0,47968.3\nR002,1999-08-09,24.3,0.8,47657.11\nR002,1999-08-10,24.8,1.5,47347.52\nR002,1999-08-11,24.8,2.0,46800.0\nR002,1999-08-12,22.7,14.3,39144.9\nR002,1999-08-13,23.7,2.2,46764.53\nR002,1999-08-14,25.0,0.0,46800.0\nR002,1999-08-15,24.3,0.6,47158.92\nR002,1999-08-16,24.2,0.3,46800.0\nR002,1999-08-17,24.8,0.0,46312.33\nR002,1999-08-18,25.9,0.9,46734.39\nR002,1999-08-19,26.7,0.0,45974.76\nR002,1999-08-20,26.6,0.1,46800.0\nR002,1999-08-21,26.1,0.3,46800.0\nR002,1999-08-22,25.9,0.8,46800.0\nR002,1999-08-23,24.8,2.7,46800.0\nR002,1999-08-24,22.3,10.5,37522.87\nR002,1999-08-25,22.0,5.6,29696.68\nR002,1999-08-26,19.8,12.1,38153.53\nR002,1999-08-27,20.1,1.1,46775.85\nR002,1999-08-28,17.4,21.6,12677.58\nR002,1999-08-29,17.4,8.1,34601.15\nR002,1999-08-30,18.7,2.6,38569.64\nR002,1999-08-31,19.7,0.7,44515.03\nR002,1999-09-01,20.6,0.0,43032.46\nR002,1999-09-02,21.5,0.4,44289.37\nR002,1999-09-03,21.5,0.3,44017.73\nR002,1999-09-04,21.1,1.4,42819.17\nR002,1999-09-05,22.2,0.3,44439.4\nR002,1999-09-06,22.0,6.1,33519.55\nR002,1999-09-07,20.0,28.3,13719.02\nR002,1999-09-08,19.5,18.7,14472.84\nR002,1999-09-09,19.3,7.8,37071.18\nR002,1999-09-10,19.3,0.4,40658.78\nR002,1999-09-11,18.8,3.5,39141.55\nR002,1999-09-12,17.6,0.7,39942.17\nR002,1999-09-13,15.7,0.0,42876.25\nR002,1999-09-14,15.8,1.3,36773.17\nR002,1999-09-15,15.8,0.0,41048.09\nR002,1999-09-16,14.0,0.0,41080.99\nR002,1999-09-17,15.5,0.0,42938.32\nR002,1999-09-18,16.2,0.0,42739.07\nR002,1999-09-19,17.1,0.0,39337.35\nR002,1999-09-20,17.5,0.1,39942.1\nR002,1999-09-21,16.0,4.0,32316.04\nR002,1999-09-22,16.0,0.0,41283.92\nR002,1999-09-23,16.2,0.0,33926.37\nR002,1999-09-24,16.3,0.0,40398.36\nR002,1999-09-25,15.6,0.2,39688.41\nR002,1999-09-26,16.1,0.7,38493.01\nR002,1999-09-27,16.2,0.0,40185.64\nR002,1999-09-28,19.0,0.0,41178.77\nR002,1999-09-29,20.6,0.0,40896.07\nR002,1999-09-30,22.4,0.0,40837.92\nR002,1999-10-01,23.0,0.0,40764.48\nR002,1999-10-02,22.5,0.0,40371.63\nR002,1999-10-03,21.3,0.0,39978.96\nR002,1999-10-04,20.7,0.0,38208.58\nR002,1999-10-05,19.5,1.3,33719.11\nR002,1999-10-06,20.2,0.2,38399.2\nR002,1999-10-07,19.9,0.0,38170.62\nR002,1999-10-08,19.1,0.0,38264.7\nR002,1999-10-09,17.9,0.0,36028.73\nR002,1999-10-10,15.1,26.4,0.0\nR002,1999-10-11,11.8,19.6,22728.76\nR002,1999-10-12,8.3,3.8,33228.18\nR002,1999-10-13,11.2,3.1,22125.03\nR002,1999-10-14,11.3,4.4,31809.06\nR002,1999-10-15,10.4,25.9,0.0\nR002,1999-10-16,9.7,19.2,28692.35\nR002,1999-10-17,10.4,8.3,25076.53\nR002,1999-10-18,8.5,9.7,33886.66\nR002,1999-10-19,6.8,0.0,34631.86\nR002,1999-10-20,9.8,3.0,22240.97\nR002,1999-10-21,11.8,8.1,17812.67\nR002,1999-10-22,7.2,7.0,32400.0\nR002,1999-10-23,5.2,0.0,36000.0\nR002,1999-10-24,6.9,0.0,36000.0\nR002,1999-10-25,8.7,0.0,36000.0\nR002,1999-10-26,8.8,0.0,34324.89\nR002,1999-10-27,10.2,0.0,34464.48\nR002,1999-10-28,10.5,26.6,19378.07\nR002,1999-10-29,6.7,7.3,34506.09\nR002,1999-10-30,5.7,4.0,4542.18\nR002,1999-10-31,4.2,2.5,35101.13\nR002,1999-11-01,6.3,0.8,25859.63\nR002,1999-11-02,6.1,0.0,35982.01\nR002,1999-11-03,8.6,0.0,35902.65\nR002,1999-11-04,10.8,0.0,35822.82\nR002,1999-11-05,8.2,0.0,35742.62\nR002,1999-11-06,7.4,0.0,35614.35\nR002,1999-11-07,8.7,0.0,35207.61\nR002,1999-11-08,7.9,0.2,33399.61\nR002,1999-11-09,3.1,4.8,22727.45\nR002,1999-11-10,2.5,0.0,35001.16\nR002,1999-11-11,3.5,0.0,34852.05\nR002,1999-11-12,4.3,2.1,29669.46\nR002,1999-11-13,5.4,0.0,18323.96\nR002,1999-11-14,7.6,0.0,34091.81\nR002,1999-11-15,9.9,1.2,32219.8\nR002,1999-11-16,6.6,39.5,16.29\nR002,1999-11-17,1.5,5.6,29500.1\nR002,1999-11-18,8.4,14.8,11012.52\nR002,1999-11-19,8.8,25.0,14516.85\nR002,1999-11-20,6.1,0.0,29917.68\nR002,1999-11-21,6.4,10.9,0.0\nR002,1999-11-22,8.4,0.0,26792.89\nR002,1999-11-23,8.4,0.4,27728.69\nR002,1999-11-24,8.1,20.9,514.74\nR002,1999-11-25,3.1,29.5,18390.74\nR002,1999-11-26,-1.5,11.9,13961.85\nR002,1999-11-27,-3.7,8.2,14400.0\nR002,1999-11-28,-1.1,5.0,26836.41\nR002,1999-11-29,1.0,5.9,521.65\nR002,1999-11-30,2.0,0.4,29538.74\nR002,1999-12-01,2.0,0.0,29555.52\nR002,1999-12-02,5.1,0.0,30711.67\nR002,1999-12-03,5.0,1.2,28800.0\nR002,1999-12-04,2.1,12.4,30361.89\nR002,1999-12-05,2.5,0.0,30757.81\nR002,1999-12-06,6.6,0.0,30386.61\nR002,1999-12-07,6.6,4.1,27408.91\nR002,1999-12-08,5.6,1.2,22813.49\nR002,1999-12-09,5.6,3.7,11203.0\nR002,1999-12-10,4.2,0.0,30417.66\nR002,1999-12-11,6.4,0.0,22171.64\nR002,1999-12-12,5.6,0.0,30423.2\nR002,1999-12-13,4.6,0.0,29259.26\nR002,1999-12-14,5.1,0.0,30422.05\nR002,1999-12-15,3.7,8.7,0.0\nR002,1999-12-16,3.6,1.0,19500.9\nR002,1999-12-17,9.7,0.1,28848.75\nR002,1999-12-18,8.5,0.0,29619.99\nR002,1999-12-19,6.2,0.3,21474.2\nR002,1999-12-20,6.7,0.0,30380.33\nR002,1999-12-21,10.1,0.0,27743.59\nR002,1999-12-22,6.7,0.0,30354.47\nR002,1999-12-23,5.7,0.0,29948.17\nR002,1999-12-24,6.6,0.0,27114.49\nR002,1999-12-25,5.5,0.0,16696.6\nR002,1999-12-26,4.0,8.2,0.0\nR002,1999-12-27,4.9,8.6,0.0\nR002,1999-12-28,6.7,31.1,10594.36\nR002,1999-12-29,10.6,22.1,0.0\nR002,1999-12-30,7.9,19.5,0.0\nR002,1999-12-31,7.4,0.0,6398.06\nR002,2000-01-01,9.0,0.0,11855.68\nR002,2000-01-02,5.1,1.2,8165.53\nR002,2000-01-03,4.8,20.0,0.0\nR002,2000-01-04,3.2,6.1,11305.47\nR002,2000-01-05,2.7,1.4,1700.6\nR002,2000-01-06,3.3,0.0,19771.76\nR002,2000-01-07,1.9,0.0,30556.74\nR002,2000-01-08,1.0,0.0,31986.23\nR002,2000-01-09,1.4,0.0,32052.5\nR002,2000-01-10,0.9,0.0,32120.15\nR002,2000-01-11,0.6,3.3,26324.2\nR002,2000-01-12,1.4,8.6,16983.75\nR002,2000-01-13,-0.3,0.5,32329.98\nR002,2000-01-14,-1.1,4.2,32076.27\nR002,2000-01-15,-1.1,2.9,32400.0\nR002,2000-01-16,-1.0,4.7,31911.78\nR002,2000-01-17,-1.3,0.6,29223.18\nR002,2000-01-18,-0.0,15.6,0.0\nR002,2000-01-19,-0.3,25.1,3600.0\nR002,2000-01-20,-2.2,11.3,18000.0\nR002,2000-01-21,-5.7,1.5,31451.62\nR002,2000-01-22,-4.7,23.7,8302.45\nR002,2000-01-23,-7.6,0.4,29181.68\nR002,2000-01-24,-2.8,0.0,32400.0\nR002,2000-01-25,2.2,34.8,0.0\nR002,2000-01-26,-3.3,4.1,30167.32\nR002,2000-01-27,-7.1,0.0,32400.0\nR002,2000-01-28,-3.9,1.7,32400.0\nR002,2000-01-29,-1.9,0.5,32523.64\nR002,2000-01-30,0.5,0.0,32480.01\nR002,2000-01-31,2.0,18.5,13258.56\nR002,2000-02-01,1.1,13.2,8500.5\nR002,2000-02-02,-0.5,0.0,32699.52\nR002,2000-02-03,3.3,0.0,32400.0\nR002,2000-02-04,2.9,25.5,0.0\nR002,2000-02-05,-2.2,0.1,33609.81\nR002,2000-02-06,-2.6,4.1,25200.0\nR002,2000-02-07,-2.6,0.3,33448.63\nR002,2000-02-08,1.5,0.0,33363.38\nR002,2000-02-09,3.7,0.0,34112.46\nR002,2000-02-10,5.9,0.0,35406.71\nR002,2000-02-11,4.6,0.6,35496.51\nR002,2000-02-12,3.6,2.1,28454.48\nR002,2000-02-13,2.8,0.0,34600.72\nR002,2000-02-14,2.1,0.2,35207.97\nR002,2000-02-15,2.2,0.0,32182.32\nR002,2000-02-16,4.3,0.6,26253.15\nR002,2000-02-17,5.2,0.0,29683.55\nR002,2000-02-18,5.6,0.0,36694.01\nR002,2000-02-19,7.3,0.0,35386.68\nR002,2000-02-20,2.9,6.7,27625.0\nR002,2000-02-21,5.3,0.5,24361.19\nR002,2000-02-22,4.9,14.4,7440.72\nR002,2000-02-23,1.2,18.8,1058.88\nR002,2000-02-24,-1.0,0.2,36861.61\nR002,2000-02-25,-1.8,3.2,22938.85\nR002,2000-02-26,-0.7,0.9,20764.58\nR002,2000-02-27,-1.4,16.9,2488.69\nR002,2000-02-28,-2.4,0.4,39387.63\nR002,2000-02-29,-1.8,0.0,39485.65\nR002,2000-03-01,-0.2,0.0,39585.17\nR002,2000-03-02,0.9,0.0,39600.0\nR002,2000-03-03,2.4,0.0,39600.0\nR002,2000-03-04,7.4,0.0,33826.1\nR002,2000-03-05,7.1,0.0,36000.0\nR002,2000-03-06,4.8,19.6,0.0\nR002,2000-03-07,0.6,6.1,20527.58\nR002,2000-03-08,-0.0,0.0,39600.0\nR002,2000-03-09,3.6,4.1,25200.0\nR002,2000-03-10,4.6,29.4,0.0\nR002,2000-03-11,1.0,25.9,21663.46\nR002,2000-03-12,-0.5,3.9,30623.51\nR002,2000-03-13,-0.3,19.3,22054.49\nR002,2000-03-14,-3.8,0.0,39600.0\nR002,2000-03-15,-1.1,0.0,39837.67\nR002,2000-03-16,4.9,0.0,39600.0\nR002,2000-03-17,6.6,3.9,22857.39\nR002,2000-03-18,8.3,3.2,25005.49\nR002,2000-03-19,3.3,12.5,37369.14\nR002,2000-03-20,3.9,21.7,0.0\nR002,2000-03-21,2.6,4.6,32400.0\nR002,2000-03-22,4.0,0.2,39840.64\nR002,2000-03-23,3.7,0.0,39379.63\nR002,2000-03-24,4.9,0.0,40295.18\nR002,2000-03-25,2.3,6.4,24348.83\nR002,2000-03-26,3.5,0.0,41635.36\nR002,2000-03-27,7.7,0.4,40342.99\nR002,2000-03-28,9.3,0.5,29669.48\nR002,2000-03-29,9.3,0.4,36436.97\nR002,2000-03-30,8.2,0.0,40979.47\nR002,2000-03-31,7.4,0.0,45092.77\nR002,2000-04-01,10.5,0.0,43574.09\nR002,2000-04-02,13.9,0.0,42665.85\nR002,2000-04-03,14.0,0.2,44678.76\nR002,2000-04-04,12.7,0.0,45892.42\nR002,2000-04-05,14.0,0.1,45963.69\nR002,2000-04-06,12.3,3.0,35144.88\nR002,2000-04-07,9.7,17.5,27902.66\nR002,2000-04-08,8.8,3.3,42441.27\nR002,2000-04-09,6.6,14.6,3093.71\nR002,2000-04-10,4.0,6.9,14797.32\nR002,2000-04-11,5.2,0.4,46224.32\nR002,2000-04-12,7.7,0.8,31178.85\nR002,2000-04-13,12.0,0.0,46532.9\nR002,2000-04-14,12.1,3.8,29482.06\nR002,2000-04-15,10.8,2.4,0.0\nR002,2000-04-16,11.1,0.0,41877.64\nR002,2000-04-17,11.4,0.0,46800.0\nR002,2000-04-18,14.0,0.0,46800.0\nR002,2000-04-19,15.3,0.1,46800.0\nR002,2000-04-20,12.6,4.4,43702.87\nR002,2000-04-21,12.5,1.7,44217.09\nR002,2000-04-22,12.4,0.5,46800.0\nR002,2000-04-23,12.6,1.4,43107.19\nR002,2000-04-24,13.8,0.3,46517.46\nR002,2000-04-25,14.7,1.3,47067.76\nR002,2000-04-26,15.8,2.0,47357.48\nR002,2000-04-27,16.8,0.9,47316.69\nR002,2000-04-28,17.9,0.7,41924.94\nR002,2000-04-29,17.9,0.4,45565.09\nR002,2000-04-30,15.6,2.2,36641.47\nR002,2000-05-01,12.7,10.6,41490.65\nR002,2000-05-02,12.0,5.3,34665.37\nR002,2000-05-03,9.0,12.6,10438.73\nR002,2000-05-04,7.7,0.0,44864.3\nR002,2000-05-05,11.5,0.0,47179.98\nR002,2000-05-06,13.3,15.3,23964.47\nR002,2000-05-07,11.7,4.4,36303.43\nR002,2000-05-08,8.2,18.6,10079.47\nR002,2000-05-09,7.1,5.6,42096.29\nR002,2000-05-10,9.2,0.1,38761.8\nR002,2000-05-11,12.8,0.1,49214.31\nR002,2000-05-12,13.5,2.3,47273.75\nR002,2000-05-13,13.6,10.7,27279.41\nR002,2000-05-14,13.2,0.0,43190.62\nR002,2000-05-15,10.9,0.0,50400.0\nR002,2000-05-16,11.7,0.2,48542.71\nR002,2000-05-17,11.7,0.3,48999.64\nR002,2000-05-18,13.8,1.1,47744.13\nR002,2000-05-19,14.8,1.2,45136.7\nR002,2000-05-20,15.3,0.0,49068.71\nR002,2000-05-21,18.1,0.0,50659.41\nR002,2000-05-22,19.6,0.2,50632.72\nR002,2000-05-23,20.0,0.0,50962.95\nR002,2000-05-24,20.5,0.3,50937.54\nR002,2000-05-25,20.1,9.3,49579.66\nR002,2000-05-26,19.6,1.8,47355.56\nR002,2000-05-27,18.1,3.8,11194.44\nR002,2000-05-28,18.1,9.4,44911.42\nR002,2000-05-29,18.7,7.5,49764.4\nR002,2000-05-30,19.3,2.9,50794.0\nR002,2000-05-31,18.6,16.0,45691.64\nR002,2000-06-01,18.4,6.8,40848.3\nR002,2000-06-02,20.5,3.6,50729.85\nR002,2000-06-03,21.2,7.0,39796.25\nR002,2000-06-04,19.1,19.7,34383.49\nR002,2000-06-05,18.8,13.0,36522.68\nR002,2000-06-06,20.8,0.2,48436.97\nR002,2000-06-07,21.8,9.7,49502.6\nR002,2000-06-08,19.6,20.4,40044.3\nR002,2000-06-09,19.4,11.2,41344.11\nR002,2000-06-10,17.3,0.0,51217.64\nR002,2000-06-11,16.4,0.0,52776.45\nR002,2000-06-12,16.7,0.0,53497.75\nR002,2000-06-13,18.3,0.0,52751.89\nR002,2000-06-14,17.9,3.4,37931.88\nR002,2000-06-15,18.8,0.0,53464.23\nR002,2000-06-16,21.1,0.0,53821.65\nR002,2000-06-17,22.5,0.0,53444.39\nR002,2000-06-18,19.5,34.5,37326.77\nR002,2000-06-19,13.4,10.5,49030.84\nR002,2000-06-20,12.4,5.8,37368.36\nR002,2000-06-21,13.3,0.0,51311.83\nR002,2000-06-22,15.2,0.0,32499.94\nR002,2000-06-23,16.7,0.6,53767.43\nR002,2000-06-24,17.2,0.1,50914.42\nR002,2000-06-25,16.8,0.0,52289.27\nR002,2000-06-26,19.6,0.0,52621.7\nR002,2000-06-27,21.1,0.0,52678.25\nR002,2000-06-28,21.6,0.3,53963.25\nR002,2000-06-29,21.5,1.2,48352.14\nR002,2000-06-30,23.3,1.7,49782.28\nR002,2000-07-01,22.9,4.1,49319.48\nR002,2000-07-02,21.3,5.6,26162.2\nR002,2000-07-03,21.5,1.3,47939.56\nR002,2000-07-04,22.5,0.0,52394.68\nR002,2000-07-05,23.0,0.0,51934.18\nR002,2000-07-06,24.2,0.5,53861.38\nR002,2000-07-07,23.0,2.5,53839.2\nR002,2000-07-08,19.7,0.0,52617.68\nR002,2000-07-09,21.7,0.0,53173.74\nR002,2000-07-10,24.8,1.5,53759.98\nR002,2000-07-11,22.9,0.0,51980.46\nR002,2000-07-12,22.8,0.0,52446.38\nR002,2000-07-13,24.4,0.0,51615.19\nR002,2000-07-14,25.9,0.1,53450.28\nR002,2000-07-15,25.3,0.4,44015.39\nR002,2000-07-16,26.2,0.5,52758.4\nR002,2000-07-17,26.1,0.7,53129.18\nR002,2000-07-18,22.9,11.0,7835.91\nR002,2000-07-19,22.4,6.4,26920.72\nR002,2000-07-20,24.9,2.0,45330.3\nR002,2000-07-21,25.7,2.1,43869.79\nR002,2000-07-22,27.2,1.0,48945.98\nR002,2000-07-23,26.6,2.2,51099.08\nR002,2000-07-24,22.3,15.7,2417.93\nR002,2000-07-25,23.1,0.0,45482.01\nR002,2000-07-26,24.3,1.0,49833.26\nR002,2000-07-27,26.2,0.2,50928.66\nR002,2000-07-28,25.7,1.1,50975.77\nR002,2000-07-29,26.0,0.0,49637.6\nR002,2000-07-30,29.7,0.0,50996.64\nR002,2000-07-31,30.0,0.0,50270.6\nR002,2000-08-01,26.1,4.5,45992.07\nR002,2000-08-02,23.6,21.8,9688.26\nR002,2000-08-03,19.4,11.7,19427.31\nR002,2000-08-04,22.8,0.0,46229.94\nR002,2000-08-05,23.2,0.0,48316.83\nR002,2000-08-06,24.3,0.3,47933.81\nR002,2000-08-07,24.6,1.3,47620.98\nR002,2000-08-08,25.8,0.3,47648.7\nR002,2000-08-09,25.9,0.0,48014.71\nR002,2000-08-10,24.4,1.2,47030.38\nR002,2000-08-11,25.0,1.4,45082.17\nR002,2000-08-12,26.0,1.1,47090.41\nR002,2000-08-13,23.9,4.6,46800.0\nR002,2000-08-14,22.8,5.2,29195.62\nR002,2000-08-15,22.4,0.6,46800.0\nR002,2000-08-16,22.0,0.5,46879.42\nR002,2000-08-17,23.3,1.1,46800.0\nR002,2000-08-18,22.7,3.3,45504.01\nR002,2000-08-19,23.1,1.0,45785.19\nR002,2000-08-20,24.3,0.4,46800.0\nR002,2000-08-21,23.7,0.0,46800.0\nR002,2000-08-22,24.1,0.1,46800.0\nR002,2000-08-23,23.9,2.3,46800.0\nR002,2000-08-24,23.5,1.1,46800.0\nR002,2000-08-25,21.0,2.5,39452.54\nR002,2000-08-26,21.1,0.9,44707.05\nR002,2000-08-27,18.7,17.6,21367.87\nR002,2000-08-28,16.9,3.1,35308.12\nR002,2000-08-29,16.1,6.2,16684.55\nR002,2000-08-30,17.0,1.2,33998.58\nR002,2000-08-31,16.9,1.7,34199.31\nR002,2000-09-01,16.3,2.4,40022.02\nR002,2000-09-02,18.2,0.0,45612.08\nR002,2000-09-03,19.7,0.8,42307.13\nR002,2000-09-04,20.1,0.0,43788.97\nR002,2000-09-05,20.9,0.0,43031.03\nR002,2000-09-06,22.3,0.2,43893.64\nR002,2000-09-07,20.3,6.0,30590.27\nR002,2000-09-08,18.5,3.0,40546.52\nR002,2000-09-09,18.0,0.0,43414.59\nR002,2000-09-10,18.2,0.0,41317.2\nR002,2000-09-11,18.4,0.0,43203.4\nR002,2000-09-12,19.2,0.4,43200.0\nR002,2000-09-13,18.8,5.9,35737.62\nR002,2000-09-14,18.9,0.1,38215.99\nR002,2000-09-15,17.7,7.9,39728.41\nR002,2000-09-16,17.3,2.3,37139.35\nR002,2000-09-17,17.5,0.0,42730.34\nR002,2000-09-18,18.1,0.0,41172.43\nR002,2000-09-19,19.3,0.0,41842.88\nR002,2000-09-20,20.0,1.2,41558.84\nR002,2000-09-21,19.1,0.6,41275.06\nR002,2000-09-22,19.0,1.1,36873.35\nR002,2000-09-23,18.0,0.0,41320.36\nR002,2000-09-24,20.2,0.0,41380.52\nR002,2000-09-25,19.9,0.2,40800.26\nR002,2000-09-26,15.9,28.7,18864.06\nR002,2000-09-27,13.3,0.6,40148.42\nR002,2000-09-28,13.8,0.5,37669.27\nR002,2000-09-29,14.7,0.0,32158.34\nR002,2000-09-30,15.7,2.8,24899.93\nR002,2000-10-01,15.3,5.7,35190.49\nR002,2000-10-02,15.1,0.0,39199.93\nR002,2000-10-03,16.2,0.0,39604.38\nR002,2000-10-04,16.0,0.0,39535.86\nR002,2000-10-05,16.4,0.0,38430.57\nR002,2000-10-06,17.4,0.0,38142.39\nR002,2000-10-07,17.6,0.0,31585.21\nR002,2000-10-08,16.6,1.9,25682.93\nR002,2000-10-09,15.2,4.4,19878.66\nR002,2000-10-10,15.7,0.1,35942.64\nR002,2000-10-11,16.4,5.4,30167.17\nR002,2000-10-12,14.5,4.1,25865.51\nR002,2000-10-13,13.4,0.7,33910.76\nR002,2000-10-14,13.9,0.0,36335.64\nR002,2000-10-15,13.5,0.0,36061.82\nR002,2000-10-16,13.3,0.0,33778.63\nR002,2000-10-17,11.2,0.0,36000.0\nR002,2000-10-18,8.5,7.7,17217.28\nR002,2000-10-19,6.9,2.7,29852.26\nR002,2000-10-20,7.2,0.0,36000.0\nR002,2000-10-21,7.7,0.3,148.04\nR002,2000-10-22,10.0,0.3,36000.0\nR002,2000-10-23,10.6,1.1,12398.29\nR002,2000-10-24,11.2,1.3,31892.27\nR002,2000-10-25,10.8,0.2,35665.65\nR002,2000-10-26,10.9,0.1,33583.95\nR002,2000-10-27,11.1,0.0,36000.0\nR002,2000-10-28,12.0,0.0,36000.0\nR002,2000-10-29,10.6,4.0,10336.01\nR002,2000-10-30,9.8,0.0,30065.66\nR002,2000-10-31,9.6,0.0,36000.0\nR002,2000-11-01,11.0,0.0,35999.46\nR002,2000-11-02,10.8,2.8,35919.29\nR002,2000-11-03,10.4,0.0,35838.93\nR002,2000-11-04,10.7,0.0,32603.42\nR002,2000-11-05,10.5,0.0,35643.12\nR002,2000-11-06,10.5,0.0,35486.82\nR002,2000-11-07,11.8,0.0,33337.1\nR002,2000-11-08,11.8,0.0,34443.84\nR002,2000-11-09,10.4,0.0,35029.89\nR002,2000-11-10,11.6,0.0,31122.74\nR002,2000-11-11,12.1,0.1,26702.54\nR002,2000-11-12,8.7,16.0,23289.58\nR002,2000-11-13,4.4,0.0,33011.58\nR002,2000-11-14,5.0,0.0,34319.85\nR002,2000-11-15,7.5,0.0,33709.99\nR002,2000-11-16,8.1,0.0,34052.84\nR002,2000-11-17,8.5,0.0,33696.22\nR002,2000-11-18,8.0,0.0,33692.14\nR002,2000-11-19,8.3,0.0,33324.32\nR002,2000-11-20,8.9,0.0,31950.81\nR002,2000-11-21,8.2,0.0,32030.09\nR002,2000-11-22,6.9,0.0,27739.76\nR002,2000-11-23,6.4,10.9,0.0\nR002,2000-11-24,4.9,5.7,14975.64\nR002,2000-11-25,5.1,0.0,25002.56\nR002,2000-11-26,5.2,0.1,30227.93\nR002,2000-11-27,5.4,0.0,27195.41\nR002,2000-11-28,7.3,0.0,28800.0\nR002,2000-11-29,7.7,0.0,22457.87\nR002,2000-11-30,8.5,0.0,29933.47\nR002,2000-12-01,7.3,0.0,29190.08\nR002,2000-12-02,7.7,0.0,20576.3\nR002,2000-12-03,7.5,1.5,29491.14\nR002,2000-12-04,7.1,0.0,29991.3\nR002,2000-12-05,6.9,0.0,30385.19\nR002,2000-12-06,5.5,0.0,29626.83\nR002,2000-12-07,3.5,0.6,28864.04\nR002,2000-12-08,3.4,1.6,29638.87\nR002,2000-12-09,3.5,0.0,30414.86\nR002,2000-12-10,4.2,0.0,30418.42\nR002,2000-12-11,5.1,0.0,25056.86\nR002,2000-12-12,4.0,0.0,27944.32\nR002,2000-12-13,3.8,1.4,27356.55\nR002,2000-12-14,3.6,0.6,29250.64\nR002,2000-12-15,4.9,2.0,14878.39\nR002,2000-12-16,5.9,0.0,10353.76\nR002,2000-12-17,6.2,20.8,5271.48\nR002,2000-12-18,2.5,0.5,19740.11\nR002,2000-12-19,2.3,0.0,21629.88\nR002,2000-12-20,0.4,0.0,12542.22\nR002,2000-12-21,-2.3,0.5,26824.81\nR002,2000-12-22,-5.2,3.7,29413.96\nR002,2000-12-23,-6.7,0.0,31218.38\nR002,2000-12-24,-3.8,0.0,18884.82\nR002,2000-12-25,1.8,0.0,25381.73\nR002,2000-12-26,2.5,0.0,23117.12\nR002,2000-12-27,5.3,0.2,29259.32\nR002,2000-12-28,8.1,0.0,21644.4\nR002,2000-12-29,8.7,0.0,27343.76\nR002,2000-12-30,10.9,0.0,31511.25\nR002,2000-12-31,7.4,4.0,10877.91\nR002,2001-01-01,7.3,1.4,30700.08\nR002,2001-01-02,5.3,0.0,14617.65\nR002,2001-01-03,4.4,0.0,31730.28\nR002,2001-01-04,4.1,0.0,31789.57\nR002,2001-01-05,5.8,0.0,28996.8\nR002,2001-01-06,5.7,0.0,31912.68\nR002,2001-01-07,5.2,0.0,23913.48\nR002,2001-01-08,5.9,0.0,32040.86\nR002,2001-01-09,8.7,0.0,32106.52\nR002,2001-01-10,8.2,4.3,18503.49\nR002,2001-01-11,3.5,2.3,28640.3\nR002,2001-01-12,3.4,13.3,3671.68\nR002,2001-01-13,1.3,19.8,10877.01\nR002,2001-01-14,-2.2,1.6,12169.7\nR002,2001-01-15,-7.5,0.0,32400.0\nR002,2001-01-16,-4.5,0.0,32400.0\nR002,2001-01-17,-3.0,0.0,32400.0\nR002,2001-01-18,-1.8,0.0,32400.0\nR002,2001-01-19,-2.0,1.1,32400.0\nR002,2001-01-20,0.8,0.4,30700.18\nR002,2001-01-21,0.3,0.0,30131.13\nR002,2001-01-22,-1.6,2.2,30085.53\nR002,2001-01-23,-3.0,0.0,29697.99\nR002,2001-01-24,-5.0,0.0,32400.0\nR002,2001-01-25,-4.8,0.0,32400.0\nR002,2001-01-26,-3.6,0.0,32400.0\nR002,2001-01-27,-0.3,0.0,32400.0\nR002,2001-01-28,4.1,0.0,30939.11\nR002,2001-01-29,6.3,0.0,23843.43\nR002,2001-01-30,8.1,0.0,32445.6\nR002,2001-01-31,8.2,0.0,26448.13\nR002,2001-02-01,9.4,0.0,29450.76\nR002,2001-02-02,5.3,5.5,22954.63\nR002,2001-02-03,4.3,0.9,28800.0\nR002,2001-02-04,2.7,7.6,15828.27\nR002,2001-02-05,0.7,0.2,27033.44\nR002,2001-02-06,3.9,9.7,0.0\nR002,2001-02-07,3.6,6.2,24630.88\nR002,2001-02-08,1.9,0.0,34333.05\nR002,2001-02-09,4.2,0.0,35394.76\nR002,2001-02-10,6.2,0.0,35774.74\nR002,2001-02-11,6.1,0.0,36152.2\nR002,2001-02-12,4.2,0.0,36344.76\nR002,2001-02-13,3.5,0.0,36221.11\nR002,2001-02-14,4.2,0.3,32916.34\nR002,2001-02-15,2.1,0.6,26277.44\nR002,2001-02-16,1.6,0.1,34594.74\nR002,2001-02-17,0.7,0.0,30258.86\nR002,2001-02-18,0.7,15.6,0.0\nR002,2001-02-19,1.8,4.4,25181.22\nR002,2001-02-20,0.2,20.3,0.0\nR002,2001-02-21,-4.8,1.7,36470.2\nR002,2001-02-22,-2.9,22.4,1187.83\nR002,2001-02-23,-0.3,30.7,0.0\nR002,2001-02-24,1.8,13.0,19200.35\nR002,2001-02-25,5.6,42.7,0.0\nR002,2001-02-26,6.8,18.4,8568.22\nR002,2001-02-27,9.3,2.7,36625.42\nR002,2001-02-28,6.4,11.2,19272.36\nR002,2001-03-01,7.6,1.0,20882.38\nR002,2001-03-02,8.1,13.5,9916.8\nR002,2001-03-03,4.5,5.8,24767.38\nR002,2001-03-04,9.4,5.1,22143.73\nR002,2001-03-05,11.6,0.0,39373.48\nR002,2001-03-06,10.5,4.4,33468.11\nR002,2001-03-07,6.8,34.8,0.0\nR002,2001-03-08,6.1,1.3,32890.99\nR002,2001-03-09,4.4,0.0,39600.0\nR002,2001-03-10,5.7,0.0,39600.0\nR002,2001-03-11,9.1,0.0,39600.0\nR002,2001-03-12,10.3,0.0,38357.24\nR002,2001-03-13,10.2,0.2,34703.45\nR002,2001-03-14,8.6,6.0,32400.0\nR002,2001-03-15,9.0,6.2,39600.0\nR002,2001-03-16,5.9,11.2,34594.07\nR002,2001-03-17,5.0,3.6,30657.04\nR002,2001-03-18,9.0,0.0,40545.7\nR002,2001-03-19,11.3,2.9,39600.0\nR002,2001-03-20,10.5,6.4,11228.16\nR002,2001-03-21,8.3,17.2,29015.37\nR002,2001-03-22,8.1,0.0,36879.12\nR002,2001-03-23,10.4,4.6,18362.01\nR002,2001-03-24,6.9,24.1,0.0\nR002,2001-03-25,5.5,0.0,38849.71\nR002,2001-03-26,10.0,1.7,17958.44\nR002,2001-03-27,11.9,3.1,42335.65\nR002,2001-03-28,12.3,1.4,41544.35\nR002,2001-03-29,6.8,10.0,29900.83\nR002,2001-03-30,6.5,0.0,36841.85\nR002,2001-03-31,8.5,0.0,39998.57\nR002,2001-04-01,13.2,0.6,30815.34\nR002,2001-04-02,13.4,17.1,19281.22\nR002,2001-04-03,10.5,2.8,26286.72\nR002,2001-04-04,12.6,0.2,41579.88\nR002,2001-04-05,14.0,4.5,30893.21\nR002,2001-04-06,9.6,19.3,6491.07\nR002,2001-04-07,12.1,0.0,45280.9\nR002,2001-04-08,12.6,0.9,43110.39\nR002,2001-04-09,12.6,0.6,45538.79\nR002,2001-04-10,13.2,0.1,46301.66\nR002,2001-04-11,10.8,17.5,21778.06\nR002,2001-04-12,8.6,8.8,40346.21\nR002,2001-04-13,9.9,0.2,46515.86\nR002,2001-04-14,13.7,0.0,38773.79\nR002,2001-04-15,11.8,1.7,15418.07\nR002,2001-04-16,11.7,1.6,46300.68\nR002,2001-04-17,8.7,9.1,43415.58\nR002,2001-04-18,11.7,0.3,39600.0\nR002,2001-04-19,11.6,19.9,16897.41\nR002,2001-04-20,11.5,0.0,46017.15\nR002,2001-04-21,12.5,0.0,46926.04\nR002,2001-04-22,16.0,0.0,45464.84\nR002,2001-04-23,17.8,2.1,22894.17\nR002,2001-04-24,13.2,23.6,119.87\nR002,2001-04-25,11.4,17.2,2715.61\nR002,2001-04-26,9.7,4.2,39989.27\nR002,2001-04-27,9.7,4.8,36792.97\nR002,2001-04-28,11.8,0.0,46800.0\nR002,2001-04-29,13.1,0.0,43725.17\nR002,2001-04-30,14.0,2.6,48212.84\nR002,2001-05-01,13.3,2.1,46834.4\nR002,2001-05-02,12.9,2.4,15068.05\nR002,2001-05-03,13.7,4.2,23216.98\nR002,2001-05-04,13.6,11.4,15939.89\nR002,2001-05-05,13.5,5.5,29247.79\nR002,2001-05-06,14.2,4.6,45654.23\nR002,2001-05-07,15.6,0.5,48308.47\nR002,2001-05-08,15.1,3.0,47254.08\nR002,2001-05-09,13.3,2.5,34682.86\nR002,2001-05-10,13.4,4.9,31099.47\nR002,2001-05-11,11.9,9.5,7912.32\nR002,2001-05-12,13.2,15.9,42701.5\nR002,2001-05-13,14.6,5.0,43424.08\nR002,2001-05-14,15.3,16.8,25307.55\nR002,2001-05-15,12.8,20.5,82.87\nR002,2001-05-16,12.6,8.5,32229.02\nR002,2001-05-17,12.5,4.9,40775.35\nR002,2001-05-18,14.4,1.1,47593.62\nR002,2001-05-19,14.3,7.8,33244.65\nR002,2001-05-20,15.7,3.6,36692.51\nR002,2001-05-21,13.6,15.2,27904.16\nR002,2001-05-22,14.3,0.0,46055.22\nR002,2001-05-23,14.0,6.1,11378.34\nR002,2001-05-24,10.7,23.0,38885.93\nR002,2001-05-25,12.0,0.0,48164.77\nR002,2001-05-26,16.6,0.0,50400.0\nR002,2001-05-27,16.8,2.1,42407.42\nR002,2001-05-28,15.3,4.2,40603.27\nR002,2001-05-29,13.1,8.3,37224.97\nR002,2001-05-30,12.8,25.9,27242.53\nR002,2001-05-31,11.8,12.5,32191.66\nR002,2001-06-01,14.2,2.1,30000.0\nR002,2001-06-02,21.9,1.0,51100.69\nR002,2001-06-03,18.5,7.8,29475.87\nR002,2001-06-04,21.0,0.0,52518.6\nR002,2001-06-05,18.5,8.4,39064.93\nR002,2001-06-06,13.2,8.7,36626.68\nR002,2001-06-07,13.4,24.2,33061.54\nR002,2001-06-08,14.5,0.0,53186.07\nR002,2001-06-09,15.3,0.0,51342.08\nR002,2001-06-10,17.0,0.2,48618.19\nR002,2001-06-11,19.5,0.0,51391.17\nR002,2001-06-12,22.6,0.0,53132.08\nR002,2001-06-13,23.9,0.7,52385.73\nR002,2001-06-14,22.5,0.2,48371.9\nR002,2001-06-15,25.2,0.1,53464.67\nR002,2001-06-16,20.5,14.0,12202.99\nR002,2001-06-17,18.7,29.8,6709.13\nR002,2001-06-18,19.2,13.4,24818.15\nR002,2001-06-19,20.3,4.8,46555.55\nR002,2001-06-20,22.4,0.0,52386.87\nR002,2001-06-21,26.3,1.6,53046.5\nR002,2001-06-22,21.9,14.0,50382.76\nR002,2001-06-23,20.5,0.2,50351.23\nR002,2001-06-24,20.4,0.0,53957.19\nR002,2001-06-25,20.7,0.0,53961.2\nR002,2001-06-26,18.3,0.0,49395.17\nR002,2001-06-27,17.5,2.5,50950.73\nR002,2001-06-28,17.2,0.4,50361.25\nR002,2001-06-29,15.7,3.2,51952.92\nR002,2001-06-30,18.0,0.2,53164.17\nR002,2001-07-01,20.1,0.4,51127.5\nR002,2001-07-02,21.0,4.6,52921.1\nR002,2001-07-03,21.4,0.6,51455.77\nR002,2001-07-04,22.1,2.4,51615.21\nR002,2001-07-05,22.3,1.6,53888.31\nR002,2001-07-06,22.4,0.8,53869.21\nR002,2001-07-07,22.3,1.3,53757.81\nR002,2001-07-08,20.6,25.7,19719.06\nR002,2001-07-09,20.1,17.1,28970.16\nR002,2001-07-10,20.4,24.1,34370.33\nR002,2001-07-11,21.4,14.1,44499.83\nR002,2001-07-12,21.8,6.4,48598.08\nR002,2001-07-13,23.4,2.0,51005.42\nR002,2001-07-14,23.5,1.6,53089.37\nR002,2001-07-15,24.6,0.4,52125.91\nR002,2001-07-16,25.0,0.0,51274.46\nR002,2001-07-17,24.5,0.2,53395.57\nR002,2001-07-18,24.0,0.8,53141.4\nR002,2001-07-19,23.1,1.8,51529.85\nR002,2001-07-20,23.1,8.0,48293.01\nR002,2001-07-21,24.6,0.0,52387.47\nR002,2001-07-22,25.7,0.5,52348.21\nR002,2001-07-23,26.9,0.4,52308.97\nR002,2001-07-24,26.9,0.1,51918.3\nR002,2001-07-25,26.9,1.0,51529.38\nR002,2001-07-26,27.3,0.0,51221.92\nR002,2001-07-27,27.3,0.2,51455.57\nR002,2001-07-28,26.9,2.8,47113.54\nR002,2001-07-29,26.8,4.6,49623.75\nR002,2001-07-30,26.6,5.2,47598.68\nR002,2001-07-31,27.1,0.5,48845.25\nR002,2001-08-01,26.4,0.1,47368.84\nR002,2001-08-02,25.9,1.9,47554.01\nR002,2001-08-03,25.3,0.3,31877.61\nR002,2001-08-04,25.1,0.0,45097.03\nR002,2001-08-05,25.2,0.0,46851.47\nR002,2001-08-06,26.3,0.0,48362.27\nR002,2001-08-07,25.1,0.5,47953.24\nR002,2001-08-08,25.6,0.1,47980.6\nR002,2001-08-09,25.8,0.5,47669.95\nR002,2001-08-10,26.0,0.0,47361.02\nR002,2001-08-11,26.1,0.0,47390.49\nR002,2001-08-12,27.7,0.0,47420.03\nR002,2001-08-13,27.8,0.2,47449.69\nR002,2001-08-14,26.6,7.9,46800.0\nR002,2001-08-15,23.7,1.2,43958.69\nR002,2001-08-16,23.4,0.7,45301.61\nR002,2001-08-17,24.7,0.6,46800.0\nR002,2001-08-18,24.3,2.6,46800.0\nR002,2001-08-19,22.3,7.8,46173.91\nR002,2001-08-20,22.2,13.6,37687.99\nR002,2001-08-21,23.2,0.6,46190.92\nR002,2001-08-22,24.5,0.8,46800.0\nR002,2001-08-23,23.2,3.6,46800.0\nR002,2001-08-24,21.8,2.2,46250.14\nR002,2001-08-25,21.4,1.4,42801.89\nR002,2001-08-26,19.6,7.4,30266.19\nR002,2001-08-27,19.2,0.0,46732.15\nR002,2001-08-28,19.6,0.0,44673.09\nR002,2001-08-29,19.3,0.0,45724.89\nR002,2001-08-30,18.5,7.9,32439.53\nR002,2001-08-31,14.5,3.8,45146.78\nR002,2001-09-01,16.0,0.0,46197.33\nR002,2001-09-02,17.4,0.0,45602.83\nR002,2001-09-03,21.4,0.0,45651.74\nR002,2001-09-04,18.0,10.7,27271.01\nR002,2001-09-05,17.8,0.0,39680.25\nR002,2001-09-06,19.8,0.0,43881.72\nR002,2001-09-07,22.8,0.4,43933.45\nR002,2001-09-08,19.2,14.3,16007.66\nR002,2001-09-09,17.7,2.0,20544.05\nR002,2001-09-10,19.1,0.0,42745.52\nR002,2001-09-11,21.3,0.0,43200.0\nR002,2001-09-12,22.7,0.0,43200.0\nR002,2001-09-13,21.4,4.2,43173.35\nR002,2001-09-14,19.1,14.0,38453.68\nR002,2001-09-15,19.8,0.0,40746.58\nR002,2001-09-16,19.3,4.5,39226.32\nR002,2001-09-17,19.2,2.0,40789.56\nR002,2001-09-18,20.6,0.0,42835.76\nR002,2001-09-19,20.3,0.0,42767.88\nR002,2001-09-20,19.3,0.2,38276.34\nR002,2001-09-21,18.3,0.0,35517.91\nR002,2001-09-22,17.8,0.0,41939.01\nR002,2001-09-23,19.0,0.1,40700.33\nR002,2001-09-24,18.6,0.0,35206.6\nR002,2001-09-25,19.9,0.0,41727.07\nR002,2001-09-26,18.4,3.5,28377.95\nR002,2001-09-27,17.6,0.1,25382.58\nR002,2001-09-28,17.5,7.0,28856.77\nR002,2001-09-29,14.3,3.1,36041.37\nR002,2001-09-30,12.6,0.0,39843.19\nR002,2001-10-01,13.8,0.0,39203.17\nR002,2001-10-02,15.5,0.0,39236.09\nR002,2001-10-03,16.4,0.0,39310.52\nR002,2001-10-04,18.8,0.0,39560.5\nR002,2001-10-05,20.3,0.0,39314.16\nR002,2001-10-06,18.2,1.0,35353.51\nR002,2001-10-07,15.6,8.4,17737.35\nR002,2001-10-08,16.2,0.0,37618.18\nR002,2001-10-09,17.5,0.0,37434.78\nR002,2001-10-10,17.6,0.0,37204.61\nR002,2001-10-11,16.8,0.0,36814.11\nR002,2001-10-12,14.0,13.4,19409.84\nR002,2001-10-13,11.2,0.0,36000.0\nR002,2001-10-14,11.1,0.0,32114.94\nR002,2001-10-15,12.2,2.1,32134.21\nR002,2001-10-16,11.2,0.0,34652.35\nR002,2001-10-17,11.7,0.4,34459.19\nR002,2001-10-18,11.3,3.0,28698.73\nR002,2001-10-19,9.5,1.6,31457.57\nR002,2001-10-20,10.1,8.2,23617.7\nR002,2001-10-21,10.6,2.2,36000.0\nR002,2001-10-22,9.3,1.1,35116.76\nR002,2001-10-23,9.6,1.4,21008.07\nR002,2001-10-24,9.6,42.3,1784.28\nR002,2001-10-25,1.4,0.2,36000.0\nR002,2001-10-26,1.5,0.0,36000.0\nR002,2001-10-27,4.3,0.0,36000.0\nR002,2001-10-28,6.5,0.0,36000.0\nR002,2001-10-29,6.9,0.6,31571.17\nR002,2001-10-30,8.7,4.8,30423.71\nR002,2001-10-31,7.4,19.6,293.83\nR002,2001-11-01,11.0,2.4,25722.01\nR002,2001-11-02,11.4,0.0,35934.56\nR002,2001-11-03,10.4,0.7,33331.2\nR002,2001-11-04,11.1,0.0,35775.37\nR002,2001-11-05,11.6,0.0,35675.13\nR002,2001-11-06,12.2,3.1,20943.64\nR002,2001-11-07,10.9,3.4,25670.65\nR002,2001-11-08,10.6,0.0,32879.96\nR002,2001-11-09,13.6,0.0,30721.67\nR002,2001-11-10,14.3,0.5,23324.2\nR002,2001-11-11,11.8,7.1,15166.83\nR002,2001-11-12,6.7,2.4,31149.11\nR002,2001-11-13,9.1,3.3,14578.47\nR002,2001-11-14,13.4,5.6,6470.04\nR002,2001-11-15,11.6,24.1,0.0\nR002,2001-11-16,4.0,10.6,22425.7\nR002,2001-11-17,-1.2,22.0,31884.65\nR002,2001-11-18,-6.4,0.6,32142.78\nR002,2001-11-19,0.8,3.9,22917.62\nR002,2001-11-20,4.4,3.9,18341.0\nR002,2001-11-21,2.7,37.0,0.0\nR002,2001-11-22,-1.6,3.0,14152.01\nR002,2001-11-23,2.5,0.0,31851.41\nR002,2001-11-24,3.2,0.0,22227.88\nR002,2001-11-25,5.5,2.3,9209.87\nR002,2001-11-26,5.0,18.9,0.0\nR002,2001-11-27,3.6,2.5,28800.0\nR002,2001-11-28,2.8,4.4,28957.74\nR002,2001-11-29,2.2,0.0,30667.46\nR002,2001-11-30,4.2,0.0,24971.22\nR002,2001-12-01,6.6,0.0,30325.38\nR002,2001-12-02,6.7,0.0,30341.2\nR002,2001-12-03,6.5,0.0,30355.56\nR002,2001-12-04,6.3,0.0,29221.7\nR002,2001-12-05,4.4,4.8,1610.03\nR002,2001-12-06,1.0,5.9,20910.2\nR002,2001-12-07,0.8,0.0,29629.61\nR002,2001-12-08,4.6,2.3,6444.11\nR002,2001-12-09,6.1,0.3,28800.0\nR002,2001-12-10,5.1,12.8,4173.54\nR002,2001-12-11,1.5,4.8,0.0\nR002,2001-12-12,2.5,2.5,18610.4\nR002,2001-12-13,3.4,10.9,3969.61\nR002,2001-12-14,4.1,12.9,12212.95\nR002,2001-12-15,0.3,10.5,24035.5\nR002,2001-12-16,-1.4,0.3,25536.26\nR002,2001-12-17,3.2,29.8,0.0\nR002,2001-12-18,5.1,35.6,0.0\nR002,2001-12-19,3.8,30.2,0.0\nR002,2001-12-20,-1.6,1.7,28542.31\nR002,2001-12-21,-3.4,0.5,29192.9\nR002,2001-12-22,1.0,1.8,25200.0\nR002,2001-12-23,3.2,7.2,2339.2\nR002,2001-12-24,5.7,1.7,29476.79\nR002,2001-12-25,6.8,32.9,0.0\nR002,2001-12-26,5.5,47.0,0.0\nR002,2001-12-27,4.4,0.7,31364.49\nR002,2001-12-28,7.5,21.3,3422.02\nR002,2001-12-29,6.3,23.8,2650.55\nR002,2001-12-30,3.5,0.0,31321.85\nR002,2001-12-31,5.7,16.7,0.0\nR002,2002-01-01,6.9,27.0,0.0\nR002,2002-01-02,2.4,39.5,0.0\nR002,2002-01-03,1.1,21.9,0.0\nR002,2002-01-04,-2.5,1.4,25200.0\nR002,2002-01-05,-1.3,5.2,0.0\nR002,2002-01-06,1.6,0.9,22484.0\nR002,2002-01-07,2.1,5.5,22380.75\nR002,2002-01-08,-2.5,1.9,21076.2\nR002,2002-01-09,-6.9,0.0,32080.72\nR002,2002-01-10,-4.4,3.2,23607.8\nR002,2002-01-11,-1.6,6.3,20395.5\nR002,2002-01-12,-0.9,1.8,30327.36\nR002,2002-01-13,0.4,0.2,32354.56\nR002,2002-01-14,1.7,0.0,32400.0\nR002,2002-01-15,1.6,0.0,32400.0\nR002,2002-01-16,1.1,0.0,32400.0\nR002,2002-01-17,1.3,0.0,32400.0\nR002,2002-01-18,-0.3,0.0,28236.59\nR002,2002-01-19,2.7,3.4,10563.32\nR002,2002-01-20,2.8,3.2,7670.38\nR002,2002-01-21,2.1,16.5,1632.86\nR002,2002-01-22,1.3,2.7,29859.93\nR002,2002-01-23,-0.2,0.0,32400.0\nR002,2002-01-24,1.9,0.0,32400.0\nR002,2002-01-25,5.1,0.0,32400.0\nR002,2002-01-26,4.8,0.1,26236.04\nR002,2002-01-27,5.5,0.0,32400.0\nR002,2002-01-28,4.3,0.0,32400.0\nR002,2002-01-29,2.9,5.5,32400.0\nR002,2002-01-30,3.8,16.3,0.0\nR002,2002-01-31,2.3,17.3,28006.33\nR002,2002-02-01,4.4,0.7,17111.35\nR002,2002-02-02,2.6,0.2,29939.39\nR002,2002-02-03,1.8,0.0,32400.0\nR002,2002-02-04,4.9,0.0,32940.01\nR002,2002-02-05,8.1,0.0,33927.11\nR002,2002-02-06,9.8,0.0,24432.87\nR002,2002-02-07,9.6,0.0,34440.65\nR002,2002-02-08,6.8,0.0,35038.01\nR002,2002-02-09,6.5,0.0,28198.35\nR002,2002-02-10,4.4,3.7,33607.33\nR002,2002-02-11,6.2,0.0,19500.3\nR002,2002-02-12,5.9,2.7,970.06\nR002,2002-02-13,4.4,12.9,4421.33\nR002,2002-02-14,4.1,21.1,17657.12\nR002,2002-02-15,0.5,15.2,10622.02\nR002,2002-02-16,-2.4,0.0,36651.11\nR002,2002-02-17,-1.5,0.6,28807.01\nR002,2002-02-18,1.8,0.0,37047.77\nR002,2002-02-19,5.3,0.0,37913.72\nR002,2002-02-20,8.8,0.0,36577.54\nR002,2002-02-21,7.4,2.8,16923.53\nR002,2002-02-22,9.9,0.0,38435.14\nR002,2002-02-23,10.2,0.0,38611.49\nR002,2002-02-24,8.6,0.0,30331.01\nR002,2002-02-25,8.9,0.4,34420.23\nR002,2002-02-26,4.6,14.1,2903.08\nR002,2002-02-27,1.1,0.0,39325.64\nR002,2002-02-28,4.4,0.0,38809.29\nR002,2002-03-01,7.6,2.7,7816.81\nR002,2002-03-02,8.7,5.2,24974.25\nR002,2002-03-03,9.7,0.0,36000.0\nR002,2002-03-04,6.8,17.1,0.0\nR002,2002-03-05,1.9,3.7,37949.26\nR002,2002-03-06,6.6,0.0,39600.0\nR002,2002-03-07,11.4,0.0,36402.14\nR002,2002-03-08,10.1,11.9,21600.0\nR002,2002-03-09,1.8,5.5,39600.0\nR002,2002-03-10,6.2,0.2,39600.0\nR002,2002-03-11,9.1,6.3,4133.14\nR002,2002-03-12,10.8,0.5,36019.97\nR002,2002-03-13,9.9,1.4,30594.26\nR002,2002-03-14,8.7,1.5,39158.73\nR002,2002-03-15,7.8,2.5,8974.47\nR002,2002-03-16,6.3,1.5,37592.74\nR002,2002-03-17,4.6,8.4,33860.92\nR002,2002-03-18,6.5,0.0,39690.27\nR002,2002-03-19,7.1,0.0,32369.08\nR002,2002-03-20,6.6,0.0,21199.51\nR002,2002-03-21,6.7,2.8,4431.72\nR002,2002-03-22,5.7,7.7,29867.93\nR002,2002-03-23,7.2,0.4,37544.24\nR002,2002-03-24,8.3,4.9,12130.41\nR002,2002-03-25,8.7,14.7,22662.96\nR002,2002-03-26,7.8,7.7,26166.93\nR002,2002-03-27,5.9,4.8,28980.64\nR002,2002-03-28,4.5,3.7,40100.36\nR002,2002-03-29,4.5,1.3,39759.42\nR002,2002-03-30,5.2,0.0,40239.15\nR002,2002-03-31,6.5,0.0,43218.07\nR002,2002-04-01,8.4,0.0,42840.73\nR002,2002-04-02,10.3,0.0,44879.93\nR002,2002-04-03,9.7,0.0,40163.71\nR002,2002-04-04,6.0,0.2,38477.96\nR002,2002-04-05,7.0,0.9,38620.42\nR002,2002-04-06,9.5,13.6,27156.11\nR002,2002-04-07,7.0,9.8,394.28\nR002,2002-04-08,2.9,8.9,26060.23\nR002,2002-04-09,1.4,10.1,36054.43\nR002,2002-04-10,3.1,8.7,42685.12\nR002,2002-04-11,4.6,0.0,46356.45\nR002,2002-04-12,6.2,0.0,45776.42\nR002,2002-04-13,8.0,0.0,41918.55\nR002,2002-04-14,9.4,0.0,21767.85\nR002,2002-04-15,10.1,2.9,22127.9\nR002,2002-04-16,11.7,0.0,45567.42\nR002,2002-04-17,10.2,7.3,28782.3\nR002,2002-04-18,10.9,1.2,19680.25\nR002,2002-04-19,10.9,1.0,45363.66\nR002,2002-04-20,10.1,4.1,0.0\nR002,2002-04-21,8.4,11.7,1414.82\nR002,2002-04-22,7.9,8.1,14079.39\nR002,2002-04-23,7.6,7.5,4164.65\nR002,2002-04-24,8.4,1.5,41087.78\nR002,2002-04-25,8.9,1.1,39455.27\nR002,2002-04-26,8.5,0.0,41111.12\nR002,2002-04-27,7.3,0.8,45521.86\nR002,2002-04-28,8.2,0.5,46800.0\nR002,2002-04-29,10.2,0.2,47923.96\nR002,2002-04-30,11.5,2.8,42751.57\nR002,2002-05-01,12.7,0.7,47176.74\nR002,2002-05-02,12.9,1.7,46850.56\nR002,2002-05-03,12.6,4.3,44332.87\nR002,2002-05-04,11.3,1.5,46150.39\nR002,2002-05-05,10.4,6.0,42184.26\nR002,2002-05-06,10.4,12.0,11485.96\nR002,2002-05-07,10.9,6.1,40115.9\nR002,2002-05-08,11.2,9.8,14492.32\nR002,2002-05-09,11.8,6.3,41040.26\nR002,2002-05-10,12.4,6.4,49594.89\nR002,2002-05-11,12.3,2.7,44180.67\nR002,2002-05-12,13.6,0.6,46093.69\nR002,2002-05-13,15.2,0.1,45307.12\nR002,2002-05-14,15.8,0.0,50377.75\nR002,2002-05-15,14.9,0.0,50400.0\nR002,2002-05-16,14.9,3.4,47616.7\nR002,2002-05-17,15.2,3.8,47543.72\nR002,2002-05-18,15.8,0.3,47782.49\nR002,2002-05-19,15.6,0.9,47410.03\nR002,2002-05-20,14.1,1.6,45757.27\nR002,2002-05-21,14.3,0.0,39794.6\nR002,2002-05-22,15.2,0.0,48372.85\nR002,2002-05-23,16.1,3.6,47419.76\nR002,2002-05-24,15.4,7.0,30491.16\nR002,2002-05-25,17.0,1.0,48280.48\nR002,2002-05-26,18.5,0.1,50540.6\nR002,2002-05-27,20.8,0.0,51234.97\nR002,2002-05-28,19.2,5.5,50300.56\nR002,2002-05-29,17.3,4.6,46871.96\nR002,2002-05-30,20.2,0.4,51169.43\nR002,2002-05-31,18.0,10.7,24970.0\nR002,2002-06-01,15.8,6.8,43718.64\nR002,2002-06-02,15.3,4.1,46264.69\nR002,2002-06-03,16.0,4.1,46842.42\nR002,2002-06-04,18.5,0.4,49695.38\nR002,2002-06-05,16.8,14.8,37366.94\nR002,2002-06-06,14.0,5.2,33184.54\nR002,2002-06-07,14.7,2.9,43599.19\nR002,2002-06-08,17.8,0.0,50829.43\nR002,2002-06-09,20.8,0.9,51673.85\nR002,2002-06-10,21.5,0.0,53525.24\nR002,2002-06-11,24.5,0.5,50580.02\nR002,2002-06-12,18.9,24.4,8284.28\nR002,2002-06-13,18.3,13.4,28491.77\nR002,2002-06-14,18.7,7.8,48189.86\nR002,2002-06-15,19.2,0.9,43127.04\nR002,2002-06-16,20.4,0.6,42090.39\nR002,2002-06-17,20.6,0.2,49011.85\nR002,2002-06-18,15.8,19.9,10618.76\nR002,2002-06-19,15.6,6.2,28363.45\nR002,2002-06-20,15.8,37.2,0.0\nR002,2002-06-21,21.5,4.9,48180.84\nR002,2002-06-22,21.6,0.0,51480.52\nR002,2002-06-23,22.8,0.0,48216.31\nR002,2002-06-24,23.8,0.0,48938.3\nR002,2002-06-25,25.6,0.0,53960.26\nR002,2002-06-26,23.4,7.6,46439.65\nR002,2002-06-27,20.9,23.3,25145.69\nR002,2002-06-28,20.5,11.0,34088.52\nR002,2002-06-29,19.9,5.4,41283.99\nR002,2002-06-30,20.4,7.5,38056.55\nR002,2002-07-01,20.9,0.8,47650.07\nR002,2002-07-02,21.7,1.9,51603.31\nR002,2002-07-03,21.8,6.9,50789.06\nR002,2002-07-04,21.4,5.4,43090.05\nR002,2002-07-05,20.7,12.5,51188.33\nR002,2002-07-06,21.7,1.3,50549.01\nR002,2002-07-07,23.5,0.6,51136.49\nR002,2002-07-08,24.9,0.5,51585.81\nR002,2002-07-09,25.2,6.0,53804.52\nR002,2002-07-10,25.6,1.1,53411.92\nR002,2002-07-11,24.7,6.1,53745.42\nR002,2002-07-12,24.1,7.8,51283.06\nR002,2002-07-13,24.0,5.7,49769.37\nR002,2002-07-14,23.6,3.2,53640.84\nR002,2002-07-15,22.7,5.9,44445.26\nR002,2002-07-16,22.6,4.0,49180.07\nR002,2002-07-17,23.0,1.7,51306.26\nR002,2002-07-18,23.3,0.7,52133.19\nR002,2002-07-19,25.1,0.1,50880.78\nR002,2002-07-20,27.1,0.5,53141.01\nR002,2002-07-21,26.7,0.0,52746.18\nR002,2002-07-22,26.3,0.3,52353.06\nR002,2002-07-23,21.4,25.5,1877.97\nR002,2002-07-24,20.7,4.8,7497.74\nR002,2002-07-25,20.0,10.1,22742.1\nR002,2002-07-26,21.4,2.0,47109.07\nR002,2002-07-27,22.5,0.2,50555.23\nR002,2002-07-28,24.3,1.1,50733.17\nR002,2002-07-29,24.6,2.6,50649.74\nR002,2002-07-30,23.9,1.8,49660.05\nR002,2002-07-31,24.9,0.3,50634.48\nR002,2002-08-01,24.8,2.1,49766.26\nR002,2002-08-02,24.7,0.7,49477.71\nR002,2002-08-03,22.8,2.0,48505.56\nR002,2002-08-04,21.8,8.0,39802.77\nR002,2002-08-05,21.7,0.7,46844.77\nR002,2002-08-06,20.9,11.7,5979.82\nR002,2002-08-07,20.9,0.9,41355.48\nR002,2002-08-08,21.9,0.0,47296.64\nR002,2002-08-09,21.2,7.6,38909.78\nR002,2002-08-10,20.6,6.1,36494.97\nR002,2002-08-11,21.5,4.9,45631.44\nR002,2002-08-12,23.0,0.1,45354.16\nR002,2002-08-13,25.6,0.0,47443.58\nR002,2002-08-14,24.4,1.7,47138.56\nR002,2002-08-15,21.8,9.7,25605.83\nR002,2002-08-16,21.4,2.1,46743.28\nR002,2002-08-17,20.9,3.2,44099.37\nR002,2002-08-18,20.4,4.4,44760.53\nR002,2002-08-19,20.3,3.1,46503.12\nR002,2002-08-20,21.3,1.3,44938.74\nR002,2002-08-21,20.9,12.9,37493.77\nR002,2002-08-22,19.4,9.7,43815.04\nR002,2002-08-23,20.2,6.3,38218.38\nR002,2002-08-24,19.9,0.0,46262.67\nR002,2002-08-25,20.0,0.1,46725.93\nR002,2002-08-26,19.6,1.3,44234.59\nR002,2002-08-27,19.0,0.2,45634.29\nR002,2002-08-28,18.6,1.5,44095.88\nR002,2002-08-29,18.6,0.1,45513.18\nR002,2002-08-30,19.8,0.6,46440.62\nR002,2002-08-31,20.8,0.3,46140.16\nR002,2002-09-01,22.4,0.9,45865.05\nR002,2002-09-02,21.7,9.0,44306.06\nR002,2002-09-03,19.5,9.4,44355.33\nR002,2002-09-04,19.6,5.7,42670.21\nR002,2002-09-05,20.9,2.1,43813.95\nR002,2002-09-06,21.5,4.8,40440.16\nR002,2002-09-07,19.8,12.7,25670.67\nR002,2002-09-08,19.7,1.5,36525.61\nR002,2002-09-09,18.5,6.2,19825.84\nR002,2002-09-10,18.3,2.6,39007.85\nR002,2002-09-11,17.1,0.0,43200.0\nR002,2002-09-12,17.1,3.8,25516.76\nR002,2002-09-13,17.9,0.0,39600.0\nR002,2002-09-14,18.3,0.0,41024.48\nR002,2002-09-15,18.0,2.7,40273.05\nR002,2002-09-16,20.6,0.0,41408.49\nR002,2002-09-17,18.0,31.2,22339.61\nR002,2002-09-18,14.7,1.1,34528.09\nR002,2002-09-19,14.2,1.6,40235.83\nR002,2002-09-20,14.6,0.0,39423.93\nR002,2002-09-21,16.7,0.0,41577.38\nR002,2002-09-22,18.7,0.0,42567.2\nR002,2002-09-23,21.9,0.0,42511.84\nR002,2002-09-24,23.3,0.0,41682.38\nR002,2002-09-25,23.2,0.0,41718.01\nR002,2002-09-26,22.0,0.0,40797.53\nR002,2002-09-27,23.3,0.0,41152.36\nR002,2002-09-28,21.3,2.1,40231.22\nR002,2002-09-29,19.6,0.4,36754.06\nR002,2002-09-30,19.0,4.2,30312.51\nR002,2002-10-01,17.4,2.8,38589.43\nR002,2002-10-02,16.7,12.8,20770.13\nR002,2002-10-03,14.1,0.3,31041.51\nR002,2002-10-04,12.4,0.0,38582.22\nR002,2002-10-05,12.4,0.0,39294.64\nR002,2002-10-06,14.6,0.0,38785.1\nR002,2002-10-07,17.5,0.0,35017.66\nR002,2002-10-08,17.5,3.5,25805.34\nR002,2002-10-09,14.8,2.1,32470.08\nR002,2002-10-10,14.5,0.8,36747.82\nR002,2002-10-11,16.0,0.0,35443.31\nR002,2002-10-12,18.2,0.6,36281.75\nR002,2002-10-13,19.2,0.0,34869.99\nR002,2002-10-14,19.9,0.0,32380.77\nR002,2002-10-15,18.0,3.5,20367.71\nR002,2002-10-16,15.2,13.8,1527.28\nR002,2002-10-17,15.8,0.0,34829.07\nR002,2002-10-18,17.2,0.0,36000.0\nR002,2002-10-19,19.2,0.0,36000.0\nR002,2002-10-20,17.3,0.5,36000.0\nR002,2002-10-21,9.5,23.0,11730.04\nR002,2002-10-22,6.4,0.0,36000.0\nR002,2002-10-23,13.5,1.9,19489.68\nR002,2002-10-24,15.7,0.3,34335.63\nR002,2002-10-25,16.4,0.0,28699.18\nR002,2002-10-26,13.3,2.7,28488.24\nR002,2002-10-27,12.0,1.8,16173.07\nR002,2002-10-28,9.7,2.9,35786.76\nR002,2002-10-29,6.6,30.2,513.34\nR002,2002-10-30,3.0,4.9,35335.32\nR002,2002-10-31,4.2,1.0,32133.06\nR002,2002-11-01,6.0,3.1,35070.76\nR002,2002-11-02,5.7,0.0,32367.17\nR002,2002-11-03,7.6,0.0,35877.14\nR002,2002-11-04,8.7,22.0,1749.23\nR002,2002-11-05,11.6,0.0,35720.29\nR002,2002-11-06,12.7,0.0,19793.74\nR002,2002-11-07,13.2,0.0,27835.35\nR002,2002-11-08,13.0,0.0,35266.79\nR002,2002-11-09,14.5,0.0,35116.26\nR002,2002-11-10,12.4,15.4,1985.45\nR002,2002-11-11,10.8,23.7,10637.96\nR002,2002-11-12,6.2,2.7,32683.39\nR002,2002-11-13,5.2,9.2,29227.93\nR002,2002-11-14,5.4,0.0,29259.74\nR002,2002-11-15,5.9,0.0,31870.4\nR002,2002-11-16,8.8,0.0,34071.59\nR002,2002-11-17,11.6,0.0,33701.32\nR002,2002-11-18,12.0,0.0,33330.78\nR002,2002-11-19,11.9,0.0,33325.3\nR002,2002-11-20,12.2,0.0,32955.5\nR002,2002-11-21,11.4,0.0,30890.1\nR002,2002-11-22,9.4,0.0,29650.95\nR002,2002-11-23,11.9,0.0,29769.44\nR002,2002-11-24,11.3,0.0,28287.98\nR002,2002-11-25,9.7,0.0,29071.79\nR002,2002-11-26,8.3,3.4,14419.14\nR002,2002-11-27,7.6,0.0,26906.96\nR002,2002-11-28,6.3,0.1,30261.88\nR002,2002-11-29,8.2,0.0,29536.53\nR002,2002-11-30,7.1,0.0,30725.88\nR002,2002-12-01,5.5,0.0,29560.15\nR002,2002-12-02,6.7,0.0,30335.67\nR002,2002-12-03,8.4,0.0,29206.95\nR002,2002-12-04,9.6,0.0,26376.52\nR002,2002-12-05,9.1,0.0,29611.05\nR002,2002-12-06,9.5,0.0,7637.66\nR002,2002-12-07,7.9,14.9,3557.91\nR002,2002-12-08,2.6,15.5,264.49\nR002,2002-12-09,-0.8,3.4,11704.4\nR002,2002-12-10,-1.4,0.2,26565.84\nR002,2002-12-11,-1.6,3.2,26470.5\nR002,2002-12-12,-4.8,1.5,30423.24\nR002,2002-12-13,-5.3,0.0,27424.15\nR002,2002-12-14,-4.3,0.0,29645.24\nR002,2002-12-15,-5.3,0.0,28863.11\nR002,2002-12-16,-5.6,0.0,29636.15\nR002,2002-12-17,-6.7,0.0,30799.05\nR002,2002-12-18,-7.0,0.0,28800.0\nR002,2002-12-19,-4.7,0.0,28513.27\nR002,2002-12-20,-5.9,0.0,4430.8\nR002,2002-12-21,-2.7,1.9,28624.44\nR002,2002-12-22,-1.0,8.6,22260.14\nR002,2002-12-23,-1.4,16.2,28315.56\nR002,2002-12-24,-2.2,22.4,0.0\nR002,2002-12-25,-9.8,0.9,29517.99\nR002,2002-12-26,-9.5,0.9,26892.32\nR002,2002-12-27,-5.8,0.6,31359.64\nR002,2002-12-28,-1.2,0.0,24791.9\nR002,2002-12-29,3.3,0.4,29291.7\nR002,2002-12-30,7.2,8.9,20560.81\nR002,2002-12-31,7.7,12.1,962.15\nR002,2003-01-01,7.6,10.0,0.0\nR002,2003-01-02,6.9,14.5,3161.06\nR002,2003-01-03,6.7,3.1,15694.1\nR002,2003-01-04,6.0,17.2,0.0\nR002,2003-01-05,4.4,0.3,1880.1\nR002,2003-01-06,6.0,21.1,9080.89\nR002,2003-01-07,3.1,14.8,31109.19\nR002,2003-01-08,7.2,4.0,19110.13\nR002,2003-01-09,4.1,10.3,24976.56\nR002,2003-01-10,3.4,0.0,32131.49\nR002,2003-01-11,6.8,3.9,13666.74\nR002,2003-01-12,8.0,0.0,0.0\nR002,2003-01-13,3.1,13.7,0.0\nR002,2003-01-14,-0.4,3.4,30702.07\nR002,2003-01-15,-1.1,4.5,22307.12\nR002,2003-01-16,0.5,2.5,30578.07\nR002,2003-01-17,0.8,1.4,23362.88\nR002,2003-01-18,2.0,0.3,28247.68\nR002,2003-01-19,-0.4,0.0,32400.0\nR002,2003-01-20,0.5,0.0,32400.0\nR002,2003-01-21,-0.8,0.0,32400.0\nR002,2003-01-22,-0.8,0.0,32400.0\nR002,2003-01-23,0.6,0.5,21252.4\nR002,2003-01-24,3.1,0.0,30336.0\nR002,2003-01-25,4.5,0.0,23169.23\nR002,2003-01-26,2.7,1.1,29200.84\nR002,2003-01-27,1.7,1.8,32400.0\nR002,2003-01-28,0.2,0.1,32051.39\nR002,2003-01-29,2.1,1.5,32400.0\nR002,2003-01-30,4.9,11.1,2302.36\nR002,2003-01-31,8.1,26.5,2321.6\nR002,2003-02-01,6.1,21.8,7795.27\nR002,2003-02-02,6.0,4.3,24238.98\nR002,2003-02-03,5.3,14.0,317.24\nR002,2003-02-04,2.6,1.2,30947.35\nR002,2003-02-05,4.1,0.2,33597.93\nR002,2003-02-06,6.4,0.0,29348.15\nR002,2003-02-07,6.5,0.0,29602.39\nR002,2003-02-08,6.4,1.0,23730.61\nR002,2003-02-09,3.1,12.8,0.0\nR002,2003-02-10,1.0,9.6,27763.33\nR002,2003-02-11,-0.0,1.7,30400.17\nR002,2003-02-12,-1.3,0.0,35492.93\nR002,2003-02-13,-0.4,0.0,36853.08\nR002,2003-02-14,-0.5,1.1,34500.37\nR002,2003-02-15,0.7,3.8,24488.04\nR002,2003-02-16,-1.0,3.6,30454.22\nR002,2003-02-17,-2.7,1.6,35665.67\nR002,2003-02-18,-3.7,0.0,37699.58\nR002,2003-02-19,-4.4,0.0,36034.04\nR002,2003-02-20,-2.3,3.5,22400.14\nR002,2003-02-21,-3.3,4.2,27807.33\nR002,2003-02-22,-3.2,0.9,29704.07\nR002,2003-02-23,-3.8,0.0,38577.75\nR002,2003-02-24,-0.5,0.0,29438.53\nR002,2003-02-25,1.4,1.3,35365.42\nR002,2003-02-26,1.8,6.2,21097.68\nR002,2003-02-27,1.2,1.8,28842.89\nR002,2003-02-28,0.8,1.7,37250.08\nR002,2003-03-01,0.6,0.0,39165.36\nR002,2003-03-02,1.6,0.0,39600.0\nR002,2003-03-03,4.3,0.2,6253.43\nR002,2003-03-04,5.5,0.0,19508.01\nR002,2003-03-05,6.0,0.0,39600.0\nR002,2003-03-06,7.0,0.0,38516.45\nR002,2003-03-07,6.6,8.9,5735.2\nR002,2003-03-08,5.2,1.2,39014.17\nR002,2003-03-09,6.0,0.0,39494.49\nR002,2003-03-10,4.7,5.0,3996.41\nR002,2003-03-11,4.8,3.9,24586.08\nR002,2003-03-12,4.5,6.0,27158.65\nR002,2003-03-13,4.6,0.3,24179.98\nR002,2003-03-14,5.0,0.9,21040.54\nR002,2003-03-15,4.7,8.2,34754.83\nR002,2003-03-16,3.3,0.1,39600.0\nR002,2003-03-17,2.1,2.3,38810.13\nR002,2003-03-18,2.7,0.0,40018.61\nR002,2003-03-19,3.1,12.0,423.73\nR002,2003-03-20,0.7,7.1,39600.0\nR002,2003-03-21,0.4,10.3,18841.64\nR002,2003-03-22,-2.1,17.7,0.0\nR002,2003-03-23,-3.4,0.0,41018.85\nR002,2003-03-24,0.5,0.0,40956.72\nR002,2003-03-25,4.2,0.0,41230.88\nR002,2003-03-26,3.8,0.0,41177.92\nR002,2003-03-27,1.8,8.9,39874.12\nR002,2003-03-28,-0.7,3.5,39044.45\nR002,2003-03-29,-0.4,0.0,41821.43\nR002,2003-03-30,2.1,0.0,44639.73\nR002,2003-03-31,5.7,0.0,45032.6\nR002,2003-04-01,7.2,0.1,42816.02\nR002,2003-04-02,3.0,11.7,34764.03\nR002,2003-04-03,6.5,1.7,14393.03\nR002,2003-04-04,9.2,4.3,13888.64\nR002,2003-04-05,9.8,0.7,43979.88\nR002,2003-04-06,7.6,25.9,15859.23\nR002,2003-04-07,8.5,1.3,34174.85\nR002,2003-04-08,12.9,0.0,46127.11\nR002,2003-04-09,6.3,11.9,5445.87\nR002,2003-04-10,6.3,0.0,43593.56\nR002,2003-04-11,7.3,0.0,46077.07\nR002,2003-04-12,7.9,0.0,46411.45\nR002,2003-04-13,9.9,0.0,46482.28\nR002,2003-04-14,11.1,0.0,46553.25\nR002,2003-04-15,10.6,0.0,46624.34\nR002,2003-04-16,10.4,0.0,46695.54\nR002,2003-04-17,9.6,1.4,33146.56\nR002,2003-04-18,9.6,0.3,44994.12\nR002,2003-04-19,9.1,2.5,16980.2\nR002,2003-04-20,7.5,10.4,2136.77\nR002,2003-04-21,8.0,5.0,1899.57\nR002,2003-04-22,7.5,0.6,18375.2\nR002,2003-04-23,8.5,2.2,37855.67\nR002,2003-04-24,8.3,4.5,33128.7\nR002,2003-04-25,9.5,9.0,4629.55\nR002,2003-04-26,8.2,13.2,16239.84\nR002,2003-04-27,6.9,0.1,39993.68\nR002,2003-04-28,7.5,0.0,47305.29\nR002,2003-04-29,9.2,0.0,47199.71\nR002,2003-04-30,11.0,1.0,45506.99\nR002,2003-05-01,12.3,0.0,48190.97\nR002,2003-05-02,14.9,0.0,48826.67\nR002,2003-05-03,16.3,0.0,48120.16\nR002,2003-05-04,18.6,0.0,48423.21\nR002,2003-05-05,17.0,0.0,46800.0\nR002,2003-05-06,10.4,2.6,42007.56\nR002,2003-05-07,10.1,0.0,47647.33\nR002,2003-05-08,11.9,0.0,49656.74\nR002,2003-05-09,13.3,1.3,47923.98\nR002,2003-05-10,14.3,0.0,48919.23\nR002,2003-05-11,17.8,0.0,50155.26\nR002,2003-05-12,18.9,0.0,50225.92\nR002,2003-05-13,19.1,0.5,50295.08\nR002,2003-05-14,19.4,0.3,50362.69\nR002,2003-05-15,19.4,0.5,47853.83\nR002,2003-05-16,18.0,4.4,49425.72\nR002,2003-05-17,16.5,5.9,44943.64\nR002,2003-05-18,16.7,0.8,46051.57\nR002,2003-05-19,16.8,8.8,42515.43\nR002,2003-05-20,17.0,3.6,41587.37\nR002,2003-05-21,19.2,0.5,47596.54\nR002,2003-05-22,21.7,0.5,50400.0\nR002,2003-05-23,21.4,0.5,50400.0\nR002,2003-05-24,21.6,0.7,50955.77\nR002,2003-05-25,23.1,1.0,50931.1\nR002,2003-05-26,21.1,2.6,49084.25\nR002,2003-05-27,18.9,7.8,24821.01\nR002,2003-05-28,17.8,9.3,21582.67\nR002,2003-05-29,18.0,4.7,48267.37\nR002,2003-05-30,17.1,12.7,40210.11\nR002,2003-05-31,15.5,4.1,29666.52\nR002,2003-06-01,15.1,12.0,44233.92\nR002,2003-06-02,17.2,0.0,50277.81\nR002,2003-06-03,16.5,0.5,49740.17\nR002,2003-06-04,12.9,4.7,40769.58\nR002,2003-06-05,12.2,0.0,51777.58\nR002,2003-06-06,14.4,0.0,51286.23\nR002,2003-06-07,17.1,0.9,51743.23\nR002,2003-06-08,18.8,0.2,45728.82\nR002,2003-06-09,19.8,0.0,49662.26\nR002,2003-06-10,19.9,0.0,52795.85\nR002,2003-06-11,21.2,0.0,53149.58\nR002,2003-06-12,20.3,0.6,50570.44\nR002,2003-06-13,18.6,3.8,41659.01\nR002,2003-06-14,19.0,0.0,51205.36\nR002,2003-06-15,19.3,0.0,47823.52\nR002,2003-06-16,19.5,0.0,50851.67\nR002,2003-06-17,20.6,2.4,52716.74\nR002,2003-06-18,17.6,0.0,52372.31\nR002,2003-06-19,17.0,0.0,53893.85\nR002,2003-06-20,18.2,0.0,52690.17\nR002,2003-06-21,19.5,0.8,51579.23\nR002,2003-06-22,18.3,1.4,50474.4\nR002,2003-06-23,16.3,0.0,51495.01\nR002,2003-06-24,15.3,1.3,50040.63\nR002,2003-06-25,17.9,0.0,53964.29\nR002,2003-06-26,20.5,0.0,53968.47\nR002,2003-06-27,18.8,1.1,53018.49\nR002,2003-06-28,18.4,0.0,52452.94\nR002,2003-06-29,18.0,0.0,50903.67\nR002,2003-06-30,17.2,0.9,48345.73\nR002,2003-07-01,17.8,1.3,50739.57\nR002,2003-07-02,19.4,0.3,50420.65\nR002,2003-07-03,23.2,0.0,53930.32\nR002,2003-07-04,26.4,0.0,53915.47\nR002,2003-07-05,27.3,0.0,53898.46\nR002,2003-07-06,26.2,0.7,53879.29\nR002,2003-07-07,21.2,9.1,47179.6\nR002,2003-07-08,19.7,0.3,50728.62\nR002,2003-07-09,19.2,0.3,52570.23\nR002,2003-07-10,18.7,2.2,48778.81\nR002,2003-07-11,20.6,0.0,52334.54\nR002,2003-07-12,21.9,0.2,53065.1\nR002,2003-07-13,21.0,0.0,53686.17\nR002,2003-07-14,21.1,0.0,52026.89\nR002,2003-07-15,23.1,0.6,52378.5\nR002,2003-07-16,21.4,6.4,25555.41\nR002,2003-07-17,20.5,0.5,47059.02\nR002,2003-07-18,19.1,4.8,48741.81\nR002,2003-07-19,19.4,5.6,45567.26\nR002,2003-07-20,20.5,2.1,48827.46\nR002,2003-07-21,21.8,2.1,49575.99\nR002,2003-07-22,22.1,4.5,49970.81\nR002,2003-07-23,21.0,9.4,34304.71\nR002,2003-07-24,19.7,18.6,42655.61\nR002,2003-07-25,19.2,6.8,37636.36\nR002,2003-07-26,19.3,13.4,38743.08\nR002,2003-07-27,20.9,7.4,47749.94\nR002,2003-07-28,20.9,17.2,30117.07\nR002,2003-07-29,19.6,48.5,16726.61\nR002,2003-07-30,19.8,18.7,29021.36\nR002,2003-07-31,20.4,9.4,45076.52\nR002,2003-08-01,21.4,5.0,48031.42\nR002,2003-08-02,23.1,0.0,49811.54\nR002,2003-08-03,23.1,0.2,48863.67\nR002,2003-08-04,23.1,1.9,46833.02\nR002,2003-08-05,21.8,6.5,43055.02\nR002,2003-08-06,20.5,0.7,46800.0\nR002,2003-08-07,21.6,0.4,47261.72\nR002,2003-08-08,21.8,6.6,43522.15\nR002,2003-08-09,21.9,14.0,27637.11\nR002,2003-08-10,22.4,3.2,43335.03\nR002,2003-08-11,22.6,2.7,33509.47\nR002,2003-08-12,19.6,7.6,46800.0\nR002,2003-08-13,17.4,0.0,46012.15\nR002,2003-08-14,18.5,0.0,45928.64\nR002,2003-08-15,20.2,0.0,46825.91\nR002,2003-08-16,21.6,0.1,46616.7\nR002,2003-08-17,22.7,0.9,46479.35\nR002,2003-08-18,22.2,5.0,43070.75\nR002,2003-08-19,22.5,0.1,43660.17\nR002,2003-08-20,22.2,0.2,46233.07\nR002,2003-08-21,22.1,1.7,45794.63\nR002,2003-08-22,21.3,3.5,46800.0\nR002,2003-08-23,19.8,5.0,44873.33\nR002,2003-08-24,20.1,1.5,43591.7\nR002,2003-08-25,20.9,0.0,45769.22\nR002,2003-08-26,21.7,0.4,46800.0\nR002,2003-08-27,21.7,0.0,45895.96\nR002,2003-08-28,22.6,0.0,46673.02\nR002,2003-08-29,22.6,1.5,46359.26\nR002,2003-08-30,22.6,0.3,44790.48\nR002,2003-08-31,23.4,0.0,45803.56\nR002,2003-09-01,24.8,0.2,46265.71\nR002,2003-09-02,24.6,0.2,45254.05\nR002,2003-09-03,22.8,3.2,39981.75\nR002,2003-09-04,20.8,23.5,28872.85\nR002,2003-09-05,17.5,74.9,1326.87\nR002,2003-09-06,14.5,65.9,0.0\nR002,2003-09-07,13.0,0.2,43265.69\nR002,2003-09-08,12.9,0.0,41194.07\nR002,2003-09-09,14.8,0.0,41688.38\nR002,2003-09-10,16.2,0.0,43427.05\nR002,2003-09-11,16.8,0.0,43200.0\nR002,2003-09-12,17.5,9.9,16648.62\nR002,2003-09-13,17.7,8.5,12421.65\nR002,2003-09-14,18.0,0.4,41972.17\nR002,2003-09-15,20.2,0.0,43071.42\nR002,2003-09-16,18.2,5.0,35898.06\nR002,2003-09-17,16.9,11.1,2947.24\nR002,2003-09-18,14.7,6.4,36715.38\nR002,2003-09-19,13.6,9.7,32830.31\nR002,2003-09-20,14.5,0.3,41215.98\nR002,2003-09-21,15.0,0.0,40931.87\nR002,2003-09-22,15.9,0.0,41284.71\nR002,2003-09-23,17.1,0.0,41319.56\nR002,2003-09-24,18.2,0.3,41991.81\nR002,2003-09-25,18.0,1.1,40590.22\nR002,2003-09-26,17.1,0.0,41426.15\nR002,2003-09-27,17.7,0.0,39147.96\nR002,2003-09-28,16.9,7.3,31103.97\nR002,2003-09-29,15.6,4.9,34071.42\nR002,2003-09-30,15.0,0.1,38416.2\nR002,2003-10-01,15.7,4.7,38570.17\nR002,2003-10-02,15.8,0.5,38343.4\nR002,2003-10-03,14.9,3.5,36033.82\nR002,2003-10-04,15.4,0.0,38656.65\nR002,2003-10-05,17.1,0.0,38630.25\nR002,2003-10-06,20.6,0.0,38450.1\nR002,2003-10-07,20.4,0.0,37850.15\nR002,2003-10-08,20.0,0.0,36000.0\nR002,2003-10-09,22.3,0.7,37325.34\nR002,2003-10-10,16.9,18.9,24992.54\nR002,2003-10-11,15.7,51.8,7200.0\nR002,2003-10-12,13.2,6.8,24064.3\nR002,2003-10-13,14.0,0.0,33721.92\nR002,2003-10-14,14.7,2.0,31389.84\nR002,2003-10-15,15.3,0.0,36021.03\nR002,2003-10-16,14.7,5.9,2806.37\nR002,2003-10-17,16.2,6.7,23669.79\nR002,2003-10-18,14.6,21.7,7855.48\nR002,2003-10-19,10.0,6.3,29765.82\nR002,2003-10-20,9.9,5.9,18611.02\nR002,2003-10-21,15.1,0.4,32312.85\nR002,2003-10-22,16.0,8.6,20780.92\nR002,2003-10-23,19.5,0.1,33947.22\nR002,2003-10-24,16.8,22.2,3600.0\nR002,2003-10-25,14.7,46.7,0.0\nR002,2003-10-26,10.2,0.5,34597.19\nR002,2003-10-27,7.0,2.5,29128.25\nR002,2003-10-28,6.1,17.0,837.29\nR002,2003-10-29,5.4,0.8,27859.24\nR002,2003-10-30,6.7,15.1,0.0\nR002,2003-10-31,8.2,0.5,26236.49\nR002,2003-11-01,10.6,10.7,1185.92\nR002,2003-11-02,9.5,0.0,35978.51\nR002,2003-11-03,10.9,0.0,35898.91\nR002,2003-11-04,14.3,0.0,32153.67\nR002,2003-11-05,15.2,0.0,35738.67\nR002,2003-11-06,13.2,0.0,29446.85\nR002,2003-11-07,13.9,0.0,32450.68\nR002,2003-11-08,10.9,1.2,33712.52\nR002,2003-11-09,7.9,14.0,4665.76\nR002,2003-11-10,6.4,0.4,13142.6\nR002,2003-11-11,9.9,4.2,30952.85\nR002,2003-11-12,7.7,25.9,3659.2\nR002,2003-11-13,4.1,0.0,32665.54\nR002,2003-11-14,1.4,0.0,32648.62\nR002,2003-11-15,2.0,0.0,33716.95\nR002,2003-11-16,1.7,0.0,34068.86\nR002,2003-11-17,5.2,11.4,6713.48\nR002,2003-11-18,6.4,6.3,11530.64\nR002,2003-11-19,3.4,9.7,28569.83\nR002,2003-11-20,3.4,37.5,0.0\nR002,2003-11-21,3.8,8.8,22805.55\nR002,2003-11-22,2.7,0.0,32215.98\nR002,2003-11-23,5.9,0.0,32215.93\nR002,2003-11-24,6.1,0.0,31827.13\nR002,2003-11-25,5.4,0.0,31851.3\nR002,2003-11-26,7.9,0.0,21600.0\nR002,2003-11-27,8.3,0.0,14541.95\nR002,2003-11-28,7.2,0.0,29879.54\nR002,2003-11-29,6.8,0.0,30658.55\nR002,2003-11-30,5.0,0.0,29959.47\nR002,2003-12-01,3.9,0.1,30316.01\nR002,2003-12-02,4.2,0.0,30333.35\nR002,2003-12-03,3.7,0.0,30349.29\nR002,2003-12-04,3.2,0.0,30363.78\nR002,2003-12-05,2.5,0.4,25228.27\nR002,2003-12-06,5.6,3.6,20047.8\nR002,2003-12-07,6.0,0.1,28860.92\nR002,2003-12-08,5.1,18.6,0.0\nR002,2003-12-09,1.2,3.5,20721.94\nR002,2003-12-10,-2.3,0.0,30804.98\nR002,2003-12-11,-3.6,0.0,30422.0\nR002,2003-12-12,0.1,0.0,29261.09\nR002,2003-12-13,2.6,0.0,28169.72\nR002,2003-12-14,4.2,3.5,12509.92\nR002,2003-12-15,6.3,0.0,29251.91\nR002,2003-12-16,7.9,2.2,20295.57\nR002,2003-12-17,8.3,7.2,18000.0\nR002,2003-12-18,6.0,26.1,0.0\nR002,2003-12-19,1.2,0.3,18272.26\nR002,2003-12-20,0.8,2.7,1208.38\nR002,2003-12-21,2.2,0.0,25493.32\nR002,2003-12-22,1.8,0.0,30745.76\nR002,2003-12-23,0.4,0.0,25858.22\nR002,2003-12-24,0.0,0.0,31233.12\nR002,2003-12-25,1.3,0.0,31268.75\nR002,2003-12-26,4.7,0.0,19106.86\nR002,2003-12-27,5.0,1.4,5751.77\nR002,2003-12-28,3.5,0.0,30245.82\nR002,2003-12-29,1.5,0.0,31429.13\nR002,2003-12-30,0.5,0.0,31474.58\nR002,2003-12-31,0.2,0.0,30563.99\nR002,2004-01-01,1.3,0.0,29755.52\nR002,2004-01-02,1.9,0.0,28946.84\nR002,2004-01-03,3.0,0.0,31681.96\nR002,2004-01-04,4.6,1.6,18437.66\nR002,2004-01-05,6.3,7.0,0.0\nR002,2004-01-06,4.6,8.1,27163.83\nR002,2004-01-07,2.3,4.1,4349.72\nR002,2004-01-08,0.1,11.8,12673.33\nR002,2004-01-09,-3.4,1.1,29491.63\nR002,2004-01-10,-3.6,0.0,32121.96\nR002,2004-01-11,-5.2,0.0,32190.57\nR002,2004-01-12,-0.6,0.0,28800.0\nR002,2004-01-13,3.9,0.0,32330.72\nR002,2004-01-14,2.8,0.0,32400.0\nR002,2004-01-15,3.8,5.2,6151.0\nR002,2004-01-16,8.1,13.2,4100.34\nR002,2004-01-17,3.8,23.0,5280.36\nR002,2004-01-18,0.5,0.0,32400.0\nR002,2004-01-19,1.4,0.0,32400.0\nR002,2004-01-20,4.1,0.0,27919.95\nR002,2004-01-21,5.5,0.0,17027.6\nR002,2004-01-22,6.9,0.0,18938.05\nR002,2004-01-23,7.8,6.6,29453.75\nR002,2004-01-24,4.7,14.2,21600.0\nR002,2004-01-25,2.8,5.4,31862.26\nR002,2004-01-26,4.0,0.0,32400.0\nR002,2004-01-27,3.6,0.0,24501.43\nR002,2004-01-28,3.5,0.0,32400.0\nR002,2004-01-29,6.6,0.0,26094.71\nR002,2004-01-30,6.8,0.0,32400.0\nR002,2004-01-31,5.9,2.0,18378.37\nR002,2004-02-01,4.3,13.3,1122.17\nR002,2004-02-02,2.9,0.9,10226.03\nR002,2004-02-03,1.3,1.9,15360.21\nR002,2004-02-04,0.7,12.1,17938.0\nR002,2004-02-05,-0.0,0.6,26939.87\nR002,2004-02-06,2.2,15.2,1563.89\nR002,2004-02-07,4.4,14.1,11121.27\nR002,2004-02-08,7.9,5.6,28875.83\nR002,2004-02-09,6.1,8.3,24414.3\nR002,2004-02-10,2.7,31.6,0.0\nR002,2004-02-11,-2.4,4.2,34760.7\nR002,2004-02-12,-2.6,0.0,29048.15\nR002,2004-02-13,3.0,1.9,19596.0\nR002,2004-02-14,2.3,16.9,3800.88\nR002,2004-02-15,-1.5,6.6,19210.41\nR002,2004-02-16,-3.9,2.3,33655.85\nR002,2004-02-17,-1.5,7.6,26016.52\nR002,2004-02-18,-0.5,3.1,35351.41\nR002,2004-02-19,-0.9,0.0,36389.46\nR002,2004-02-20,-0.0,12.4,955.51\nR002,2004-02-21,-1.5,16.7,2175.94\nR002,2004-02-22,-1.5,0.5,2900.3\nR002,2004-02-23,0.8,0.0,34804.37\nR002,2004-02-24,2.7,0.1,38715.35\nR002,2004-02-25,6.0,7.1,17080.41\nR002,2004-02-26,6.8,4.3,32270.75\nR002,2004-02-27,9.0,0.0,32348.99\nR002,2004-02-28,11.5,2.2,31589.06\nR002,2004-02-29,11.6,0.3,32312.43\nR002,2004-03-01,9.1,10.0,5057.54\nR002,2004-03-02,7.7,0.2,39600.0\nR002,2004-03-03,7.6,2.8,18889.5\nR002,2004-03-04,8.1,13.2,0.0\nR002,2004-03-05,5.4,59.5,0.0\nR002,2004-03-06,-0.1,9.4,27937.01\nR002,2004-03-07,-4.0,0.0,38813.61\nR002,2004-03-08,-1.1,0.0,39600.0\nR002,2004-03-09,4.1,0.0,39600.0\nR002,2004-03-10,6.0,0.0,31621.59\nR002,2004-03-11,6.2,6.9,870.91\nR002,2004-03-12,4.7,4.3,29333.06\nR002,2004-03-13,3.8,4.6,23965.03\nR002,2004-03-14,2.4,1.2,37878.95\nR002,2004-03-15,2.4,0.0,39600.0\nR002,2004-03-16,1.7,4.1,34366.15\nR002,2004-03-17,2.2,2.0,39600.0\nR002,2004-03-18,2.9,0.0,39600.0\nR002,2004-03-19,3.2,0.5,34204.36\nR002,2004-03-20,5.8,0.3,21742.2\nR002,2004-03-21,8.6,0.6,40136.67\nR002,2004-03-22,11.3,0.0,39764.69\nR002,2004-03-23,12.7,0.0,42594.08\nR002,2004-03-24,13.0,0.0,40534.09\nR002,2004-03-25,13.6,0.0,43058.52\nR002,2004-03-26,8.9,6.8,40500.67\nR002,2004-03-27,10.1,0.7,2538.81\nR002,2004-03-28,12.7,0.8,22545.2\nR002,2004-03-29,10.3,13.5,30546.99\nR002,2004-03-30,4.6,17.9,35603.93\nR002,2004-03-31,4.0,5.1,27173.19\nR002,2004-04-01,4.8,0.0,42080.0\nR002,2004-04-02,2.9,0.8,45246.31\nR002,2004-04-03,-4.5,5.4,34288.93\nR002,2004-04-04,-5.1,0.3,43450.6\nR002,2004-04-05,-1.1,0.0,44142.91\nR002,2004-04-06,4.4,0.0,44343.08\nR002,2004-04-07,9.5,0.3,44214.75\nR002,2004-04-08,11.7,0.0,46179.01\nR002,2004-04-09,14.2,0.0,46250.27\nR002,2004-04-10,14.4,0.0,44212.54\nR002,2004-04-11,16.3,0.0,39295.43\nR002,2004-04-12,18.3,0.0,46464.5\nR002,2004-04-13,16.2,0.5,19064.77\nR002,2004-04-14,15.5,4.1,5411.0\nR002,2004-04-15,9.0,30.9,10638.07\nR002,2004-04-16,8.6,0.0,46500.13\nR002,2004-04-17,6.5,11.7,2375.7\nR002,2004-04-18,8.0,0.2,45889.15\nR002,2004-04-19,9.7,0.0,46800.0\nR002,2004-04-20,11.5,0.0,46800.0\nR002,2004-04-21,13.1,1.1,44755.67\nR002,2004-04-22,12.4,12.7,23079.08\nR002,2004-04-23,11.3,13.1,12825.73\nR002,2004-04-24,9.8,8.1,41520.43\nR002,2004-04-25,9.4,3.7,32898.82\nR002,2004-04-26,11.3,0.6,44985.19\nR002,2004-04-27,15.2,0.0,47981.8\nR002,2004-04-28,15.9,4.5,44566.57\nR002,2004-04-29,14.0,6.9,45175.21\nR002,2004-04-30,11.8,20.4,0.0\nR002,2004-05-01,12.0,5.3,20113.1\nR002,2004-05-02,13.8,0.0,46800.0\nR002,2004-05-03,15.5,0.0,47112.6\nR002,2004-05-04,15.4,0.2,47388.07\nR002,2004-05-05,15.4,0.2,48366.16\nR002,2004-05-06,17.0,0.0,37706.55\nR002,2004-05-07,17.7,2.0,43610.32\nR002,2004-05-08,18.1,0.8,41313.5\nR002,2004-05-09,17.3,0.8,47243.89\nR002,2004-05-10,17.1,2.2,48554.16\nR002,2004-05-11,14.5,11.7,24448.79\nR002,2004-05-12,11.9,46.4,7279.67\nR002,2004-05-13,13.0,17.5,41938.73\nR002,2004-05-14,13.8,0.2,47985.36\nR002,2004-05-15,14.4,3.5,40725.54\nR002,2004-05-16,13.1,12.8,12991.15\nR002,2004-05-17,11.0,11.7,3208.08\nR002,2004-05-18,12.9,0.0,43625.6\nR002,2004-05-19,9.0,9.0,26944.0\nR002,2004-05-20,9.2,0.7,30110.99\nR002,2004-05-21,13.3,0.0,46036.57\nR002,2004-05-22,15.6,0.1,49128.88\nR002,2004-05-23,18.1,0.7,50605.03\nR002,2004-05-24,18.8,0.5,50936.26\nR002,2004-05-25,18.7,2.0,50273.42\nR002,2004-05-26,14.4,7.5,0.0\nR002,2004-05-27,15.7,7.9,8025.22\nR002,2004-05-28,15.7,11.9,2882.76\nR002,2004-05-29,16.8,4.7,34411.2\nR002,2004-05-30,16.5,9.4,16445.53\nR002,2004-05-31,19.2,2.8,48533.99\nR002,2004-06-01,16.5,19.4,33529.5\nR002,2004-06-02,15.1,10.5,26199.83\nR002,2004-06-03,16.0,8.3,32166.9\nR002,2004-06-04,15.8,1.0,44710.64\nR002,2004-06-05,18.2,0.0,51728.46\nR002,2004-06-06,21.2,0.3,43425.9\nR002,2004-06-07,19.5,11.4,40491.97\nR002,2004-06-08,18.1,9.5,45661.56\nR002,2004-06-09,16.0,11.3,9113.65\nR002,2004-06-10,14.1,9.4,34705.64\nR002,2004-06-11,11.7,0.7,50686.95\nR002,2004-06-12,14.8,0.0,53130.1\nR002,2004-06-13,18.9,0.0,53118.15\nR002,2004-06-14,20.9,0.0,53473.98\nR002,2004-06-15,19.9,0.8,52360.77\nR002,2004-06-16,20.8,0.5,51221.24\nR002,2004-06-17,19.9,4.0,46745.19\nR002,2004-06-18,20.3,0.1,50678.36\nR002,2004-06-19,21.9,0.6,53425.69\nR002,2004-06-20,21.6,11.3,45788.89\nR002,2004-06-21,21.5,0.9,49315.17\nR002,2004-06-22,21.2,6.8,51258.54\nR002,2004-06-23,20.1,9.2,38490.03\nR002,2004-06-24,20.5,17.2,40296.87\nR002,2004-06-25,20.5,12.3,39823.76\nR002,2004-06-26,21.1,8.7,45612.08\nR002,2004-06-27,20.7,5.2,47640.53\nR002,2004-06-28,19.5,6.2,42351.46\nR002,2004-06-29,17.3,2.5,52755.14\nR002,2004-06-30,18.2,2.2,52283.94\nR002,2004-07-01,16.5,5.9,46120.33\nR002,2004-07-02,16.2,0.0,51583.69\nR002,2004-07-03,18.4,0.0,53476.23\nR002,2004-07-04,20.4,0.8,50835.83\nR002,2004-07-05,21.5,2.3,42012.98\nR002,2004-07-06,22.3,0.0,52716.79\nR002,2004-07-07,23.3,0.2,53034.1\nR002,2004-07-08,23.3,0.4,49869.06\nR002,2004-07-09,21.4,2.5,51590.42\nR002,2004-07-10,20.8,0.0,51317.28\nR002,2004-07-11,20.0,1.0,52701.15\nR002,2004-07-12,21.0,1.9,50710.28\nR002,2004-07-13,21.9,4.5,53662.15\nR002,2004-07-14,22.9,0.8,52732.83\nR002,2004-07-15,20.6,7.0,39792.92\nR002,2004-07-16,20.7,19.4,20092.16\nR002,2004-07-17,19.5,19.3,29511.95\nR002,2004-07-18,20.2,18.4,27984.73\nR002,2004-07-19,20.6,2.3,44893.8\nR002,2004-07-20,21.2,0.0,49968.8\nR002,2004-07-21,21.3,0.3,51725.78\nR002,2004-07-22,19.7,3.6,47899.67\nR002,2004-07-23,17.8,15.4,40994.51\nR002,2004-07-24,17.7,2.9,45245.74\nR002,2004-07-25,19.5,0.0,50137.98\nR002,2004-07-26,20.6,0.0,49137.26\nR002,2004-07-27,22.2,0.4,50752.5\nR002,2004-07-28,23.5,0.0,51065.9\nR002,2004-07-29,24.3,0.0,50335.95\nR002,2004-07-30,24.7,1.1,50383.08\nR002,2004-07-31,25.0,0.3,49399.21\nR002,2004-08-01,25.5,0.2,48765.25\nR002,2004-08-02,25.5,0.9,49166.47\nR002,2004-08-03,22.1,15.0,37771.28\nR002,2004-08-04,20.6,6.3,35760.92\nR002,2004-08-05,21.1,1.6,46883.79\nR002,2004-08-06,22.7,0.8,48614.56\nR002,2004-08-07,23.0,2.3,47960.71\nR002,2004-08-08,22.2,3.7,46971.43\nR002,2004-08-09,22.2,3.5,47339.1\nR002,2004-08-10,23.4,0.0,46800.0\nR002,2004-08-11,20.9,16.9,20034.92\nR002,2004-08-12,19.0,5.9,19459.52\nR002,2004-08-13,20.1,4.8,40807.81\nR002,2004-08-14,22.0,0.0,46800.0\nR002,2004-08-15,23.3,0.0,47182.76\nR002,2004-08-16,24.2,1.3,46880.8\nR002,2004-08-17,25.8,0.3,46912.3\nR002,2004-08-18,24.3,9.5,41609.56\nR002,2004-08-19,21.5,29.1,21991.01\nR002,2004-08-20,21.4,9.1,37850.12\nR002,2004-08-21,21.3,3.6,36837.97\nR002,2004-08-22,21.5,15.3,40325.62\nR002,2004-08-23,22.3,1.1,46419.96\nR002,2004-08-24,21.4,3.3,45487.95\nR002,2004-08-25,20.4,1.2,40199.99\nR002,2004-08-26,21.0,0.0,44997.03\nR002,2004-08-27,21.9,0.2,46705.57\nR002,2004-08-28,24.0,0.0,46603.44\nR002,2004-08-29,23.2,6.6,45099.53\nR002,2004-08-30,21.4,9.2,41580.25\nR002,2004-08-31,21.5,3.2,43907.6\nR002,2004-09-01,21.4,2.6,43592.38\nR002,2004-09-02,21.7,3.2,41975.09\nR002,2004-09-03,20.2,1.9,36175.41\nR002,2004-09-04,19.5,0.4,42869.96\nR002,2004-09-05,17.4,3.0,42163.65\nR002,2004-09-06,15.3,3.0,35899.92\nR002,2004-09-07,14.3,3.0,41487.72\nR002,2004-09-08,15.2,0.0,42089.32\nR002,2004-09-09,17.2,0.9,40288.38\nR002,2004-09-10,12.5,16.1,23738.42\nR002,2004-09-11,12.1,0.8,40202.23\nR002,2004-09-12,13.5,0.0,42838.87\nR002,2004-09-13,14.3,0.0,43154.64\nR002,2004-09-14,16.4,0.0,43087.46\nR002,2004-09-15,17.4,0.0,43020.19\nR002,2004-09-16,18.6,0.0,42952.8\nR002,2004-09-17,17.9,0.5,38495.64\nR002,2004-09-18,16.8,0.4,39900.86\nR002,2004-09-19,18.5,0.0,40889.58\nR002,2004-09-20,20.1,0.0,42681.81\nR002,2004-09-21,19.4,0.0,42231.59\nR002,2004-09-22,19.8,0.0,42545.41\nR002,2004-09-23,19.6,2.9,41665.01\nR002,2004-09-24,20.3,0.1,39600.0\nR002,2004-09-25,23.2,0.0,38371.71\nR002,2004-09-26,23.6,0.0,40497.11\nR002,2004-09-27,21.8,0.0,39600.0\nR002,2004-09-28,21.3,0.0,41207.71\nR002,2004-09-29,20.4,0.0,39568.66\nR002,2004-09-30,21.7,0.0,40454.97\nR002,2004-10-01,19.1,1.4,39286.78\nR002,2004-10-02,16.3,0.0,38016.09\nR002,2004-10-03,13.2,1.2,34472.93\nR002,2004-10-04,11.9,0.0,35615.26\nR002,2004-10-05,12.2,0.0,36818.06\nR002,2004-10-06,12.7,0.0,39078.57\nR002,2004-10-07,16.0,0.0,38562.65\nR002,2004-10-08,18.3,0.0,37687.54\nR002,2004-10-09,16.1,10.0,18981.68\nR002,2004-10-10,15.0,2.9,30840.27\nR002,2004-10-11,15.9,0.0,36176.16\nR002,2004-10-12,13.7,22.2,844.48\nR002,2004-10-13,6.8,0.0,34709.62\nR002,2004-10-14,8.8,0.0,36338.06\nR002,2004-10-15,11.0,1.1,32036.47\nR002,2004-10-16,13.9,0.6,19784.65\nR002,2004-10-17,16.8,0.0,34186.62\nR002,2004-10-18,18.3,0.0,36000.0\nR002,2004-10-19,17.6,0.7,31903.68\nR002,2004-10-20,15.3,6.7,8378.47\nR002,2004-10-21,10.7,24.8,32174.23\nR002,2004-10-22,7.4,1.4,36000.0\nR002,2004-10-23,7.1,0.0,36000.0\nR002,2004-10-24,10.6,0.0,34146.07\nR002,2004-10-25,13.1,0.0,36000.0\nR002,2004-10-26,13.9,0.0,36000.0\nR002,2004-10-27,16.8,0.0,36000.0\nR002,2004-10-28,17.5,0.9,35231.3\nR002,2004-10-29,15.5,1.6,36000.0\nR002,2004-10-30,14.8,0.2,33287.45\nR002,2004-10-31,15.9,0.0,36000.0\nR002,2004-11-01,14.1,0.1,30937.67\nR002,2004-11-02,14.9,3.4,31474.01\nR002,2004-11-03,11.1,15.6,8167.77\nR002,2004-11-04,8.4,11.3,30329.43\nR002,2004-11-05,6.1,0.0,35634.23\nR002,2004-11-06,8.4,0.0,35478.27\nR002,2004-11-07,9.8,0.0,35324.3\nR002,2004-11-08,11.6,0.0,35172.51\nR002,2004-11-09,11.7,0.0,32464.88\nR002,2004-11-10,13.0,0.0,33421.33\nR002,2004-11-11,12.3,0.0,34732.18\nR002,2004-11-12,12.3,0.0,34591.09\nR002,2004-11-13,11.9,0.0,31381.0\nR002,2004-11-14,11.9,0.0,34082.0\nR002,2004-11-15,13.8,0.0,30505.91\nR002,2004-11-16,12.0,14.7,0.0\nR002,2004-11-17,8.8,17.1,666.37\nR002,2004-11-18,4.3,0.0,32235.32\nR002,2004-11-19,6.9,0.0,32594.73\nR002,2004-11-20,9.6,0.5,31216.58\nR002,2004-11-21,4.6,56.9,0.0\nR002,2004-11-22,0.1,26.1,0.0\nR002,2004-11-23,-0.9,4.3,21231.31\nR002,2004-11-24,-0.2,4.3,19733.32\nR002,2004-11-25,3.8,15.9,0.0\nR002,2004-11-26,3.2,17.2,0.0\nR002,2004-11-27,0.4,2.8,21982.34\nR002,2004-11-28,3.7,35.2,0.0\nR002,2004-11-29,4.3,17.6,3792.67\nR002,2004-11-30,4.9,21.1,506.92\nR002,2004-12-01,2.7,13.0,28083.21\nR002,2004-12-02,1.8,0.0,30347.4\nR002,2004-12-03,4.9,15.1,3618.27\nR002,2004-12-04,7.8,25.8,10791.14\nR002,2004-12-05,3.4,1.4,28375.84\nR002,2004-12-06,2.1,0.0,22811.08\nR002,2004-12-07,3.1,2.3,27656.51\nR002,2004-12-08,1.7,0.2,27735.73\nR002,2004-12-09,1.1,12.6,1367.71\nR002,2004-12-10,0.9,6.6,25439.11\nR002,2004-12-11,-0.4,9.9,17389.47\nR002,2004-12-12,-3.5,1.5,27803.22\nR002,2004-12-13,0.4,15.7,1212.5\nR002,2004-12-14,-3.6,1.8,27745.56\nR002,2004-12-15,-7.2,0.0,30413.47\nR002,2004-12-16,-7.3,0.6,29238.54\nR002,2004-12-17,-3.7,0.1,30011.1\nR002,2004-12-18,-3.0,0.0,30783.64\nR002,2004-12-19,-1.9,0.0,30774.29\nR002,2004-12-20,-1.2,0.0,30763.48\nR002,2004-12-21,3.0,4.0,0.0\nR002,2004-12-22,4.4,0.9,12956.1\nR002,2004-12-23,4.2,6.2,17.06\nR002,2004-12-24,3.5,4.2,23738.73\nR002,2004-12-25,4.0,1.4,26802.14\nR002,2004-12-26,2.1,0.0,31329.46\nR002,2004-12-27,5.3,0.0,31035.37\nR002,2004-12-28,7.0,0.0,30620.26\nR002,2004-12-29,8.4,0.0,22522.46\nR002,2004-12-30,10.5,0.0,29005.31\nR002,2004-12-31,7.4,2.8,1409.55\nR002,2005-01-01,6.4,0.3,30900.18\nR002,2005-01-02,5.7,1.9,2761.21\nR002,2005-01-03,3.6,0.7,29185.21\nR002,2005-01-04,2.5,1.4,4531.69\nR002,2005-01-05,1.3,8.6,16622.51\nR002,2005-01-06,-0.1,2.3,23550.69\nR002,2005-01-07,-0.4,0.0,31979.14\nR002,2005-01-08,-0.2,0.1,30658.35\nR002,2005-01-09,3.7,0.0,28508.49\nR002,2005-01-10,1.9,4.2,15589.92\nR002,2005-01-11,-0.2,4.8,10763.38\nR002,2005-01-12,4.3,4.7,26413.01\nR002,2005-01-13,2.8,0.0,32377.44\nR002,2005-01-14,3.2,0.0,32400.0\nR002,2005-01-15,4.0,0.0,28800.0\nR002,2005-01-16,2.7,0.0,32400.0\nR002,2005-01-17,2.4,0.0,32400.0\nR002,2005-01-18,1.9,0.0,32400.0\nR002,2005-01-19,1.0,0.0,32400.0\nR002,2005-01-20,0.4,0.0,32400.0\nR002,2005-01-21,2.2,0.0,28800.0\nR002,2005-01-22,6.2,0.0,32400.0\nR002,2005-01-23,4.9,2.4,24795.66\nR002,2005-01-24,1.6,13.7,0.0\nR002,2005-01-25,-1.1,0.0,30714.78\nR002,2005-01-26,4.1,3.4,6725.67\nR002,2005-01-27,5.5,0.0,32400.0\nR002,2005-01-28,8.0,0.3,26635.84\nR002,2005-01-29,6.9,4.0,5490.85\nR002,2005-01-30,9.0,6.5,5074.85\nR002,2005-01-31,4.1,20.8,0.0\nR002,2005-02-01,1.2,0.0,28643.65\nR002,2005-02-02,0.6,0.0,30516.95\nR002,2005-02-03,0.5,0.0,32623.54\nR002,2005-02-04,-0.5,0.0,33644.58\nR002,2005-02-05,-1.8,0.0,33229.68\nR002,2005-02-06,-2.7,0.0,33529.68\nR002,2005-02-07,-5.6,2.1,33730.08\nR002,2005-02-08,-4.7,4.0,29651.19\nR002,2005-02-09,-5.2,0.4,34579.65\nR002,2005-02-10,-3.2,3.0,27407.61\nR002,2005-02-11,-3.8,1.3,34050.87\nR002,2005-02-12,-4.4,0.0,34297.34\nR002,2005-02-13,-3.5,0.0,36222.72\nR002,2005-02-14,0.9,0.0,30909.13\nR002,2005-02-15,6.3,0.0,36964.64\nR002,2005-02-16,8.6,0.0,35601.12\nR002,2005-02-17,10.9,0.0,35681.1\nR002,2005-02-18,10.4,7.6,21600.0\nR002,2005-02-19,6.5,24.6,0.0\nR002,2005-02-20,4.8,0.3,33247.1\nR002,2005-02-21,3.1,10.8,23547.39\nR002,2005-02-22,2.7,0.0,38487.19\nR002,2005-02-23,3.3,0.0,38661.09\nR002,2005-02-24,4.3,0.0,36763.46\nR002,2005-02-25,5.2,3.2,29349.58\nR002,2005-02-26,7.4,3.3,29981.3\nR002,2005-02-27,7.9,7.8,6349.27\nR002,2005-02-28,5.2,23.5,5919.81\nR002,2005-03-01,2.2,17.3,0.0\nR002,2005-03-02,4.8,26.5,0.0\nR002,2005-03-03,8.4,25.3,0.0\nR002,2005-03-04,4.5,16.8,13618.55\nR002,2005-03-05,5.2,0.0,36000.0\nR002,2005-03-06,8.0,2.5,13296.22\nR002,2005-03-07,7.4,8.9,34714.31\nR002,2005-03-08,6.9,10.4,8660.73\nR002,2005-03-09,3.4,23.6,22794.01\nR002,2005-03-10,0.9,6.8,36570.8\nR002,2005-03-11,0.2,4.7,31757.4\nR002,2005-03-12,-0.1,0.0,39391.25\nR002,2005-03-13,3.9,0.0,39628.2\nR002,2005-03-14,5.8,3.8,36758.08\nR002,2005-03-15,5.4,6.3,35114.01\nR002,2005-03-16,3.1,1.9,34366.05\nR002,2005-03-17,3.1,5.0,31232.03\nR002,2005-03-18,6.1,10.5,6757.26\nR002,2005-03-19,5.8,49.3,0.0\nR002,2005-03-20,-2.2,7.7,39579.12\nR002,2005-03-21,-3.7,2.2,39949.24\nR002,2005-03-22,-2.5,0.0,39999.56\nR002,2005-03-23,-2.2,1.6,36450.34\nR002,2005-03-24,-2.5,5.8,30688.67\nR002,2005-03-25,-2.6,0.0,41809.28\nR002,2005-03-26,1.0,0.0,34645.79\nR002,2005-03-27,6.2,0.0,41728.29\nR002,2005-03-28,8.0,7.6,29566.66\nR002,2005-03-29,7.2,37.7,0.0\nR002,2005-03-30,7.1,4.7,22862.11\nR002,2005-03-31,5.6,0.2,31968.67\nR002,2005-04-01,4.7,7.1,6215.57\nR002,2005-04-02,4.9,1.7,32929.0\nR002,2005-04-03,4.7,7.5,0.0\nR002,2005-04-04,5.4,3.0,38432.79\nR002,2005-04-05,5.7,0.0,43745.0\nR002,2005-04-06,7.2,0.0,41637.86\nR002,2005-04-07,7.6,0.0,44959.15\nR002,2005-04-08,10.0,0.0,46161.83\nR002,2005-04-09,11.4,0.0,46233.38\nR002,2005-04-10,12.8,0.0,46304.92\nR002,2005-04-11,12.1,0.0,46376.45\nR002,2005-04-12,13.1,0.0,46447.94\nR002,2005-04-13,15.2,0.0,46519.39\nR002,2005-04-14,15.2,0.0,38301.38\nR002,2005-04-15,12.2,10.2,9787.07\nR002,2005-04-16,11.3,5.1,38103.6\nR002,2005-04-17,11.2,10.5,29045.47\nR002,2005-04-18,14.7,1.5,46800.0\nR002,2005-04-19,12.8,28.2,19614.83\nR002,2005-04-20,11.1,4.4,37943.08\nR002,2005-04-21,14.5,0.0,46800.0\nR002,2005-04-22,17.4,0.7,34411.29\nR002,2005-04-23,16.1,7.9,32664.37\nR002,2005-04-24,11.7,35.5,0.0\nR002,2005-04-25,10.8,0.0,2352.53\nR002,2005-04-26,12.4,0.0,41113.58\nR002,2005-04-27,13.7,0.0,46800.0\nR002,2005-04-28,11.2,4.5,24311.64\nR002,2005-04-29,9.8,7.1,39108.8\nR002,2005-04-30,12.3,3.0,43552.54\nR002,2005-05-01,11.4,6.0,42324.64\nR002,2005-05-02,10.4,1.3,33625.67\nR002,2005-05-03,9.9,6.3,25016.09\nR002,2005-05-04,10.0,10.4,0.0\nR002,2005-05-05,11.0,7.8,33329.34\nR002,2005-05-06,13.6,0.0,49356.95\nR002,2005-05-07,18.2,0.3,41010.75\nR002,2005-05-08,18.3,4.3,34189.28\nR002,2005-05-09,14.0,14.1,5782.31\nR002,2005-05-10,15.0,0.0,48559.57\nR002,2005-05-11,16.9,0.2,49904.69\nR002,2005-05-12,18.3,0.9,50266.26\nR002,2005-05-13,17.4,19.7,36224.52\nR002,2005-05-14,13.4,27.4,319.79\nR002,2005-05-15,13.9,0.5,45457.61\nR002,2005-05-16,14.9,1.1,45713.94\nR002,2005-05-17,16.5,0.1,45798.74\nR002,2005-05-18,17.9,0.6,48071.09\nR002,2005-05-19,19.9,0.9,50089.65\nR002,2005-05-20,20.0,3.9,50400.0\nR002,2005-05-21,21.1,2.4,46800.0\nR002,2005-05-22,23.5,1.6,47754.1\nR002,2005-05-23,22.7,5.7,49376.48\nR002,2005-05-24,20.9,8.6,46238.3\nR002,2005-05-25,18.1,25.8,19629.76\nR002,2005-05-26,17.8,18.5,36187.74\nR002,2005-05-27,17.1,8.6,29032.23\nR002,2005-05-28,18.7,7.9,42506.75\nR002,2005-05-29,20.8,2.4,50821.2\nR002,2005-05-30,21.1,6.8,48227.75\nR002,2005-05-31,20.3,12.8,40263.72\nR002,2005-06-01,19.9,6.4,41065.79\nR002,2005-06-02,18.3,7.0,27498.01\nR002,2005-06-03,16.7,6.3,45672.76\nR002,2005-06-04,15.6,1.2,45964.06\nR002,2005-06-05,15.9,1.8,50144.39\nR002,2005-06-06,15.7,8.7,7721.09\nR002,2005-06-07,18.3,0.6,45997.55\nR002,2005-06-08,20.6,0.0,50962.19\nR002,2005-06-09,22.7,0.4,51254.28\nR002,2005-06-10,22.7,0.7,52790.74\nR002,2005-06-11,22.8,13.6,50379.38\nR002,2005-06-12,19.6,12.8,43883.39\nR002,2005-06-13,17.6,0.0,49118.08\nR002,2005-06-14,19.0,0.8,50555.89\nR002,2005-06-15,19.1,11.2,50228.48\nR002,2005-06-16,17.6,18.1,20166.05\nR002,2005-06-17,17.2,3.6,45968.56\nR002,2005-06-18,19.0,0.0,53067.89\nR002,2005-06-19,19.3,3.2,52324.34\nR002,2005-06-20,16.3,8.9,44973.83\nR002,2005-06-21,14.9,4.8,39702.74\nR002,2005-06-22,14.0,5.9,42656.64\nR002,2005-06-23,14.5,7.2,45835.64\nR002,2005-06-24,17.3,0.0,53398.43\nR002,2005-06-25,18.5,1.9,48996.94\nR002,2005-06-26,19.1,7.9,48154.91\nR002,2005-06-27,19.2,3.0,51338.83\nR002,2005-06-28,20.5,0.6,47614.34\nR002,2005-06-29,20.5,1.9,51892.52\nR002,2005-06-30,20.9,1.1,51311.01\nR002,2005-07-01,21.6,0.0,50594.72\nR002,2005-07-02,23.3,0.9,53746.65\nR002,2005-07-03,23.7,1.9,53921.41\nR002,2005-07-04,25.6,0.3,53384.07\nR002,2005-07-05,22.6,21.0,38734.02\nR002,2005-07-06,19.7,6.0,46670.57\nR002,2005-07-07,20.1,0.0,51942.24\nR002,2005-07-08,21.9,0.0,53761.68\nR002,2005-07-09,23.0,0.2,53043.31\nR002,2005-07-10,23.2,0.6,51420.55\nR002,2005-07-11,22.3,5.2,49702.75\nR002,2005-07-12,22.4,0.7,52051.15\nR002,2005-07-13,23.4,0.1,48964.97\nR002,2005-07-14,24.1,1.2,51757.97\nR002,2005-07-15,23.2,3.2,50807.4\nR002,2005-07-16,22.3,17.0,42314.96\nR002,2005-07-17,20.4,10.5,43274.39\nR002,2005-07-18,21.3,0.0,51717.48\nR002,2005-07-19,21.2,0.0,49849.8\nR002,2005-07-20,22.2,0.1,51326.53\nR002,2005-07-21,22.9,1.0,50659.55\nR002,2005-07-22,23.1,0.5,52346.7\nR002,2005-07-23,24.3,0.6,52307.59\nR002,2005-07-24,25.3,0.6,51917.11\nR002,2005-07-25,23.3,6.4,48026.35\nR002,2005-07-26,23.2,8.3,45214.93\nR002,2005-07-27,24.0,0.0,47432.1\nR002,2005-07-28,25.3,0.0,48295.62\nR002,2005-07-29,25.1,0.2,46895.67\nR002,2005-07-30,25.1,0.2,46704.54\nR002,2005-07-31,25.2,0.4,49037.61\nR002,2005-08-01,24.5,3.4,42462.47\nR002,2005-08-02,22.1,9.2,39976.61\nR002,2005-08-03,21.1,9.5,40780.01\nR002,2005-08-04,21.6,14.2,46753.35\nR002,2005-08-05,22.9,9.5,47557.81\nR002,2005-08-06,24.7,0.0,47002.0\nR002,2005-08-07,27.2,0.1,48768.48\nR002,2005-08-08,24.7,3.0,46375.31\nR002,2005-08-09,24.0,0.0,45111.38\nR002,2005-08-10,24.7,9.1,42294.99\nR002,2005-08-11,23.0,4.9,37036.6\nR002,2005-08-12,22.7,0.3,45214.6\nR002,2005-08-13,22.1,0.8,46800.0\nR002,2005-08-14,22.7,0.7,46811.35\nR002,2005-08-15,24.1,0.0,47176.22\nR002,2005-08-16,25.9,0.0,47206.83\nR002,2005-08-17,26.6,0.0,46905.34\nR002,2005-08-18,27.4,2.2,46936.63\nR002,2005-08-19,26.2,0.3,46800.0\nR002,2005-08-20,23.2,3.4,46396.67\nR002,2005-08-21,21.9,2.7,13955.22\nR002,2005-08-22,20.1,7.7,35047.91\nR002,2005-08-23,22.0,0.1,44731.54\nR002,2005-08-24,23.7,0.1,46800.0\nR002,2005-08-25,23.9,3.2,46800.0\nR002,2005-08-26,22.7,1.8,45317.37\nR002,2005-08-27,23.0,1.2,43928.95\nR002,2005-08-28,22.3,1.5,44995.03\nR002,2005-08-29,21.1,3.6,43875.34\nR002,2005-08-30,19.7,2.5,43913.83\nR002,2005-08-31,20.0,0.6,45830.21\nR002,2005-09-01,19.0,1.7,43627.26\nR002,2005-09-02,18.2,0.7,42378.5\nR002,2005-09-03,17.0,2.9,42977.71\nR002,2005-09-04,17.3,2.9,36393.57\nR002,2005-09-05,15.9,2.2,42110.27\nR002,2005-09-06,15.9,0.2,43200.0\nR002,2005-09-07,16.9,0.4,43295.51\nR002,2005-09-08,17.7,0.6,42044.08\nR002,2005-09-09,19.2,4.3,43200.0\nR002,2005-09-10,18.8,17.1,36770.92\nR002,2005-09-11,17.9,5.4,37476.52\nR002,2005-09-12,17.1,3.6,33017.74\nR002,2005-09-13,17.9,1.4,37577.02\nR002,2005-09-14,18.3,0.7,38749.41\nR002,2005-09-15,18.7,0.0,38078.22\nR002,2005-09-16,20.4,0.0,42969.08\nR002,2005-09-17,20.9,1.4,31887.35\nR002,2005-09-18,21.3,1.6,41484.15\nR002,2005-09-19,20.2,1.8,39515.08\nR002,2005-09-20,19.8,1.3,41552.13\nR002,2005-09-21,19.5,1.9,42223.26\nR002,2005-09-22,20.3,0.0,41939.78\nR002,2005-09-23,20.7,0.2,41974.98\nR002,2005-09-24,19.5,2.0,41373.35\nR002,2005-09-25,17.8,9.6,1642.01\nR002,2005-09-26,16.8,8.1,23538.93\nR002,2005-09-27,17.1,6.3,35414.94\nR002,2005-09-28,16.5,7.5,37302.21\nR002,2005-09-29,16.8,3.0,35346.51\nR002,2005-09-30,16.3,2.4,13139.8\nR002,2005-10-01,16.3,10.8,27786.54\nR002,2005-10-02,17.3,0.0,39698.57\nR002,2005-10-03,18.5,1.6,19411.79\nR002,2005-10-04,15.8,13.3,11040.52\nR002,2005-10-05,13.2,2.9,31397.47\nR002,2005-10-06,11.0,6.8,36293.82\nR002,2005-10-07,9.0,1.0,29433.9\nR002,2005-10-08,10.3,0.3,31224.9\nR002,2005-10-09,10.0,0.0,36381.66\nR002,2005-10-10,11.0,1.7,35034.75\nR002,2005-10-11,12.8,0.0,36166.83\nR002,2005-10-12,16.5,1.1,36546.6\nR002,2005-10-13,16.1,9.4,27721.98\nR002,2005-10-14,16.4,8.4,21725.54\nR002,2005-10-15,15.4,5.7,16438.36\nR002,2005-10-16,13.5,23.7,11626.81\nR002,2005-10-17,12.4,64.5,4784.69\nR002,2005-10-18,9.4,12.2,0.0\nR002,2005-10-19,8.4,19.1,0.0\nR002,2005-10-20,8.1,0.5,26730.26\nR002,2005-10-21,6.5,4.8,36000.0\nR002,2005-10-22,6.8,0.0,36000.0\nR002,2005-10-23,9.3,0.0,36000.0\nR002,2005-10-24,11.1,0.0,36000.0\nR002,2005-10-25,11.9,3.7,26223.64\nR002,2005-10-26,9.7,0.0,36000.0\nR002,2005-10-27,11.7,7.2,31931.64\nR002,2005-10-28,8.4,13.0,31610.68\nR002,2005-10-29,6.0,3.1,30774.17\nR002,2005-10-30,6.3,0.7,0.0\nR002,2005-10-31,10.1,3.6,32582.26\nR002,2005-11-01,7.8,16.1,15610.48\nR002,2005-11-02,9.0,9.0,26109.85\nR002,2005-11-03,7.3,19.0,3242.09\nR002,2005-11-04,9.1,0.0,33169.51\nR002,2005-11-05,10.1,0.0,31425.39\nR002,2005-11-06,9.5,1.8,22073.51\nR002,2005-11-07,10.4,4.3,15721.13\nR002,2005-11-08,9.5,0.2,33469.72\nR002,2005-11-09,8.8,0.0,35070.49\nR002,2005-11-10,8.2,0.2,32358.96\nR002,2005-11-11,6.8,2.8,28895.27\nR002,2005-11-12,6.0,0.0,34636.03\nR002,2005-11-13,5.7,0.0,34493.85\nR002,2005-11-14,6.4,0.0,34353.25\nR002,2005-11-15,6.6,0.0,34214.45\nR002,2005-11-16,5.9,7.2,17437.8\nR002,2005-11-17,8.3,0.0,33338.36\nR002,2005-11-18,10.4,0.0,29586.81\nR002,2005-11-19,10.7,0.0,15839.8\nR002,2005-11-20,10.8,0.0,24921.45\nR002,2005-11-21,9.1,10.3,4532.84\nR002,2005-11-22,5.8,24.9,10107.41\nR002,2005-11-23,0.7,0.0,31058.15\nR002,2005-11-24,1.8,0.0,29377.74\nR002,2005-11-25,7.1,0.0,26767.79\nR002,2005-11-26,10.8,7.1,0.0\nR002,2005-11-27,9.3,2.0,14528.77\nR002,2005-11-28,13.0,1.2,30647.3\nR002,2005-11-29,14.3,0.6,22505.58\nR002,2005-11-30,9.1,6.4,29929.86\nR002,2005-12-01,7.5,0.0,25200.0\nR002,2005-12-02,10.5,11.2,0.0\nR002,2005-12-03,10.8,21.8,0.0\nR002,2005-12-04,9.6,0.0,29986.7\nR002,2005-12-05,8.9,9.2,22838.19\nR002,2005-12-06,7.8,0.0,30390.73\nR002,2005-12-07,10.9,0.0,14791.65\nR002,2005-12-08,13.2,0.0,24387.69\nR002,2005-12-09,8.6,5.5,10800.0\nR002,2005-12-10,5.9,0.4,25737.99\nR002,2005-12-11,7.7,6.8,0.0\nR002,2005-12-12,5.0,0.2,22264.63\nR002,2005-12-13,4.4,0.0,29644.49\nR002,2005-12-14,6.0,0.0,29642.03\nR002,2005-12-15,7.6,0.0,28859.39\nR002,2005-12-16,6.6,11.8,0.0\nR002,2005-12-17,4.9,18.0,15820.81\nR002,2005-12-18,8.6,2.2,30399.23\nR002,2005-12-19,8.3,9.2,24435.02\nR002,2005-12-20,2.1,22.1,14806.02\nR002,2005-12-21,-3.1,0.0,29974.88\nR002,2005-12-22,0.5,0.4,28689.81\nR002,2005-12-23,1.4,7.0,0.0\nR002,2005-12-24,0.7,7.7,838.3\nR002,2005-12-25,0.1,0.0,20643.15\nR002,2005-12-26,-0.7,0.0,30668.63\nR002,2005-12-27,1.8,0.0,31367.43\nR002,2005-12-28,3.0,0.0,18594.34\nR002,2005-12-29,5.5,0.0,31460.13\nR002,2005-12-30,6.4,0.0,31509.35\nR002,2005-12-31,6.0,1.1,23197.22\nR002,2006-01-01,2.3,7.3,31612.91\nR002,2006-01-02,1.1,0.0,29705.43\nR002,2006-01-03,4.7,0.0,30816.33\nR002,2006-01-04,5.6,0.0,31779.23\nR002,2006-01-05,6.1,0.0,21274.61\nR002,2006-01-06,6.4,10.4,6234.72\nR002,2006-01-07,5.3,6.8,401.43\nR002,2006-01-08,3.4,4.2,27067.06\nR002,2006-01-09,2.0,6.4,19924.02\nR002,2006-01-10,0.4,3.4,22175.39\nR002,2006-01-11,-3.0,0.0,32216.95\nR002,2006-01-12,-0.4,0.0,32286.84\nR002,2006-01-13,0.6,0.0,32358.31\nR002,2006-01-14,1.0,1.8,29629.01\nR002,2006-01-15,0.1,0.2,32400.0\nR002,2006-01-16,-0.3,1.2,32400.0\nR002,2006-01-17,-2.1,4.5,25056.94\nR002,2006-01-18,-3.6,2.5,32344.48\nR002,2006-01-19,-0.7,1.0,26787.13\nR002,2006-01-20,-3.1,27.5,1977.46\nR002,2006-01-21,-10.1,1.7,32400.0\nR002,2006-01-22,-4.1,10.6,1081.84\nR002,2006-01-23,-1.4,7.9,20049.44\nR002,2006-01-24,-5.7,0.0,29278.9\nR002,2006-01-25,-4.7,0.0,25532.37\nR002,2006-01-26,-0.7,6.7,0.0\nR002,2006-01-27,-2.0,5.6,26500.97\nR002,2006-01-28,-2.8,4.6,30184.82\nR002,2006-01-29,-1.9,10.6,20467.74\nR002,2006-01-30,-2.1,12.5,4552.53\nR002,2006-01-31,-1.2,36.1,615.91\nR002,2006-02-01,-0.4,24.1,897.31\nR002,2006-02-02,1.7,3.4,25761.57\nR002,2006-02-03,3.3,0.0,28077.58\nR002,2006-02-04,0.5,0.0,32938.43\nR002,2006-02-05,0.1,0.0,30631.53\nR002,2006-02-06,3.8,0.9,12250.6\nR002,2006-02-07,4.1,25.0,0.0\nR002,2006-02-08,4.1,3.6,8301.35\nR002,2006-02-09,3.2,8.3,0.0\nR002,2006-02-10,2.0,2.1,34504.67\nR002,2006-02-11,4.6,41.7,7059.31\nR002,2006-02-12,4.1,36.1,0.0\nR002,2006-02-13,1.7,33.6,0.0\nR002,2006-02-14,0.5,9.7,539.56\nR002,2006-02-15,0.3,6.9,0.0\nR002,2006-02-16,-2.3,4.0,29472.78\nR002,2006-02-17,-4.8,0.0,36010.53\nR002,2006-02-18,1.9,5.4,30556.18\nR002,2006-02-19,3.3,9.1,29211.92\nR002,2006-02-20,4.2,0.0,32687.04\nR002,2006-02-21,6.3,0.0,38266.2\nR002,2006-02-22,5.5,0.0,38442.02\nR002,2006-02-23,7.1,0.0,36693.01\nR002,2006-02-24,7.4,0.0,38796.7\nR002,2006-02-25,7.3,2.6,36947.04\nR002,2006-02-26,7.3,11.8,26357.15\nR002,2006-02-27,6.1,20.9,14516.48\nR002,2006-02-28,6.1,2.3,34289.3\nR002,2006-03-01,6.1,7.8,17662.71\nR002,2006-03-02,5.0,2.1,37706.2\nR002,2006-03-03,4.8,23.1,15229.96\nR002,2006-03-04,-0.3,8.3,35584.22\nR002,2006-03-05,4.7,0.0,39600.0\nR002,2006-03-06,9.8,0.0,39600.0\nR002,2006-03-07,12.0,0.0,37421.92\nR002,2006-03-08,10.5,14.8,15309.44\nR002,2006-03-09,6.0,14.3,5071.29\nR002,2006-03-10,1.1,2.8,28250.76\nR002,2006-03-11,4.2,0.0,39600.0\nR002,2006-03-12,8.6,0.0,39217.01\nR002,2006-03-13,9.7,0.0,34348.92\nR002,2006-03-14,11.2,0.0,39600.0\nR002,2006-03-15,7.9,3.9,10186.45\nR002,2006-03-16,6.7,3.3,17505.64\nR002,2006-03-17,6.0,1.2,34478.21\nR002,2006-03-18,5.5,2.4,38146.36\nR002,2006-03-19,5.1,8.8,19053.98\nR002,2006-03-20,2.9,11.6,39600.0\nR002,2006-03-21,4.9,0.0,41238.53\nR002,2006-03-22,7.4,0.0,41147.36\nR002,2006-03-23,8.2,0.0,40366.64\nR002,2006-03-24,11.8,0.4,39645.41\nR002,2006-03-25,7.0,8.6,568.79\nR002,2006-03-26,6.0,7.5,7093.92\nR002,2006-03-27,6.3,0.0,18966.19\nR002,2006-03-28,7.7,0.0,40356.39\nR002,2006-03-29,7.5,0.0,42432.49\nR002,2006-03-30,9.3,0.4,37815.08\nR002,2006-03-31,8.6,6.6,9085.14\nR002,2006-04-01,8.9,0.0,43928.68\nR002,2006-04-02,7.4,20.6,8786.49\nR002,2006-04-03,7.0,15.7,0.0\nR002,2006-04-04,6.8,2.9,3790.1\nR002,2006-04-05,6.9,0.0,45931.15\nR002,2006-04-06,7.5,0.0,44678.83\nR002,2006-04-07,9.5,0.5,37238.56\nR002,2006-04-08,14.0,0.5,22595.11\nR002,2006-04-09,14.8,0.9,46186.78\nR002,2006-04-10,14.4,1.0,39600.0\nR002,2006-04-11,13.9,3.6,37199.74\nR002,2006-04-12,15.5,2.2,46097.39\nR002,2006-04-13,16.4,1.5,42136.76\nR002,2006-04-14,16.8,0.3,40622.16\nR002,2006-04-15,13.1,27.3,0.0\nR002,2006-04-16,10.8,3.5,43114.65\nR002,2006-04-17,10.5,0.0,46785.32\nR002,2006-04-18,12.0,1.5,46800.0\nR002,2006-04-19,12.1,0.0,45650.97\nR002,2006-04-20,15.5,0.0,46800.0\nR002,2006-04-21,13.8,5.6,42059.57\nR002,2006-04-22,12.3,7.3,33142.12\nR002,2006-04-23,11.7,3.0,43309.35\nR002,2006-04-24,10.6,5.9,34805.7\nR002,2006-04-25,6.8,7.6,25622.54\nR002,2006-04-26,4.7,9.5,35952.9\nR002,2006-04-27,4.3,5.2,36082.08\nR002,2006-04-28,6.1,2.0,46800.0\nR002,2006-04-29,9.0,0.0,48256.42\nR002,2006-04-30,10.8,1.4,41845.33\nR002,2006-05-01,11.4,0.0,38232.51\nR002,2006-05-02,11.1,5.2,17617.47\nR002,2006-05-03,10.4,8.2,1717.94\nR002,2006-05-04,9.9,2.0,11545.59\nR002,2006-05-05,13.0,0.9,31975.18\nR002,2006-05-06,15.8,6.0,37573.7\nR002,2006-05-07,13.9,17.0,32100.87\nR002,2006-05-08,13.5,12.9,7433.61\nR002,2006-05-09,12.4,5.5,34009.32\nR002,2006-05-10,12.9,0.0,45442.7\nR002,2006-05-11,14.8,0.0,50175.06\nR002,2006-05-12,16.4,3.6,45925.48\nR002,2006-05-13,14.0,7.6,34605.71\nR002,2006-05-14,12.1,8.2,1165.36\nR002,2006-05-15,12.6,3.8,37480.28\nR002,2006-05-16,15.4,0.3,48273.58\nR002,2006-05-17,15.9,8.9,36245.63\nR002,2006-05-18,13.7,13.8,14400.0\nR002,2006-05-19,15.5,2.8,35875.08\nR002,2006-05-20,17.2,2.4,47782.82\nR002,2006-05-21,18.2,2.8,49306.22\nR002,2006-05-22,18.9,0.7,47706.25\nR002,2006-05-23,18.8,0.6,48854.3\nR002,2006-05-24,19.5,0.2,45971.67\nR002,2006-05-25,21.4,0.0,50471.65\nR002,2006-05-26,22.8,0.0,50539.06\nR002,2006-05-27,17.1,28.1,11253.61\nR002,2006-05-28,14.3,37.7,3869.85\nR002,2006-05-29,15.9,1.4,41573.84\nR002,2006-05-30,20.7,0.0,51889.69\nR002,2006-05-31,25.5,0.4,51869.45\nR002,2006-06-01,25.7,0.6,51849.59\nR002,2006-06-02,24.6,1.1,51104.48\nR002,2006-06-03,25.9,0.9,52174.26\nR002,2006-06-04,25.8,2.9,52519.94\nR002,2006-06-05,25.4,0.6,51774.17\nR002,2006-06-06,20.8,11.7,32558.45\nR002,2006-06-07,20.2,6.9,46259.27\nR002,2006-06-08,19.0,10.1,34017.75\nR002,2006-06-09,20.2,4.5,49026.01\nR002,2006-06-10,18.3,12.3,40096.07\nR002,2006-06-11,17.6,0.9,49328.64\nR002,2006-06-12,17.0,8.5,40977.37\nR002,2006-06-13,17.9,0.7,50623.56\nR002,2006-06-14,19.4,8.2,46305.65\nR002,2006-06-15,19.8,5.9,46225.11\nR002,2006-06-16,19.8,7.1,45057.51\nR002,2006-06-17,18.7,0.8,51915.38\nR002,2006-06-18,18.1,0.0,53881.98\nR002,2006-06-19,18.8,0.1,51225.02\nR002,2006-06-20,21.2,0.5,51464.21\nR002,2006-06-21,22.8,1.1,51887.71\nR002,2006-06-22,23.0,3.7,53042.57\nR002,2006-06-23,22.0,7.3,48350.82\nR002,2006-06-24,21.5,8.1,48566.36\nR002,2006-06-25,21.4,10.7,49687.25\nR002,2006-06-26,21.2,10.6,49969.0\nR002,2006-06-27,22.4,2.2,51582.52\nR002,2006-06-28,24.1,0.9,48682.81\nR002,2006-06-29,24.1,8.7,33202.57\nR002,2006-06-30,23.9,1.2,45593.92\nR002,2006-07-01,23.2,4.8,44990.09\nR002,2006-07-02,21.4,4.5,39807.28\nR002,2006-07-03,20.0,0.2,46418.24\nR002,2006-07-04,19.5,0.1,53914.75\nR002,2006-07-05,20.0,0.7,53384.66\nR002,2006-07-06,19.4,6.7,30518.17\nR002,2006-07-07,19.4,19.1,26271.28\nR002,2006-07-08,17.6,34.3,10863.65\nR002,2006-07-09,19.8,2.3,52680.19\nR002,2006-07-10,21.1,2.1,51603.65\nR002,2006-07-11,21.1,5.7,49395.54\nR002,2006-07-12,22.4,3.2,47869.8\nR002,2006-07-13,24.3,0.6,50387.87\nR002,2006-07-14,23.2,14.3,51381.42\nR002,2006-07-15,20.5,7.9,35384.8\nR002,2006-07-16,20.8,1.3,46957.67\nR002,2006-07-17,22.0,0.7,50985.26\nR002,2006-07-18,21.5,1.2,52068.92\nR002,2006-07-19,21.0,0.3,48130.55\nR002,2006-07-20,22.1,0.0,50966.26\nR002,2006-07-21,22.8,0.0,50617.2\nR002,2006-07-22,23.7,0.0,49985.78\nR002,2006-07-23,23.0,4.9,40445.56\nR002,2006-07-24,21.5,31.9,20841.22\nR002,2006-07-25,22.3,1.9,46298.08\nR002,2006-07-26,22.3,0.7,45634.29\nR002,2006-07-27,23.0,0.9,48812.63\nR002,2006-07-28,21.5,0.3,47399.51\nR002,2006-07-29,20.2,1.3,40753.36\nR002,2006-07-30,20.0,1.7,42681.09\nR002,2006-07-31,21.1,0.0,47070.37\nR002,2006-08-01,22.3,0.7,49511.32\nR002,2006-08-02,22.7,0.9,47469.93\nR002,2006-08-03,23.4,0.5,47478.82\nR002,2006-08-04,25.4,0.0,48910.82\nR002,2006-08-05,26.8,0.2,48971.79\nR002,2006-08-06,27.2,0.0,48692.99\nR002,2006-08-07,27.2,0.4,48076.73\nR002,2006-08-08,27.0,0.2,48315.38\nR002,2006-08-09,24.1,4.5,46988.92\nR002,2006-08-10,24.2,0.0,48031.57\nR002,2006-08-11,26.7,0.3,47722.57\nR002,2006-08-12,28.0,0.3,47415.16\nR002,2006-08-13,27.7,0.0,47109.29\nR002,2006-08-14,28.5,0.4,47139.38\nR002,2006-08-15,27.7,1.4,46800.0\nR002,2006-08-16,27.8,1.8,46866.52\nR002,2006-08-17,28.0,0.3,46897.53\nR002,2006-08-18,27.0,1.1,46928.65\nR002,2006-08-19,25.0,6.2,41629.42\nR002,2006-08-20,24.0,13.4,14009.63\nR002,2006-08-21,22.5,12.3,36569.81\nR002,2006-08-22,22.6,5.3,45378.13\nR002,2006-08-23,22.8,0.3,46800.0\nR002,2006-08-24,22.6,0.3,43659.49\nR002,2006-08-25,23.6,0.0,46800.0\nR002,2006-08-26,24.9,0.0,46800.0\nR002,2006-08-27,26.8,0.0,46746.81\nR002,2006-08-28,26.7,0.5,46644.01\nR002,2006-08-29,25.1,1.9,46540.78\nR002,2006-08-30,24.5,0.0,46437.24\nR002,2006-08-31,26.4,0.0,46333.47\nR002,2006-09-01,25.3,0.8,43265.94\nR002,2006-09-02,23.5,13.1,34024.15\nR002,2006-09-03,19.6,28.2,26619.02\nR002,2006-09-04,19.0,0.0,42792.68\nR002,2006-09-05,18.2,11.1,39039.76\nR002,2006-09-06,16.3,22.5,17820.82\nR002,2006-09-07,15.3,2.4,38328.14\nR002,2006-09-08,17.2,0.0,43200.0\nR002,2006-09-09,18.7,0.0,43385.26\nR002,2006-09-10,18.9,0.3,40401.35\nR002,2006-09-11,18.9,0.0,31656.51\nR002,2006-09-12,18.5,0.4,41586.02\nR002,2006-09-13,16.8,2.1,39665.79\nR002,2006-09-14,16.3,2.5,41026.06\nR002,2006-09-15,15.7,0.8,41037.6\nR002,2006-09-16,14.8,0.2,41171.71\nR002,2006-09-17,15.0,0.0,42916.52\nR002,2006-09-18,17.2,0.0,42848.88\nR002,2006-09-19,20.1,0.0,42146.99\nR002,2006-09-20,21.0,0.5,42499.36\nR002,2006-09-21,18.3,3.3,38042.61\nR002,2006-09-22,17.3,7.4,2772.48\nR002,2006-09-23,17.5,5.1,38298.14\nR002,2006-09-24,18.3,7.1,30840.93\nR002,2006-09-25,18.4,1.5,41400.68\nR002,2006-09-26,18.0,5.6,28505.65\nR002,2006-09-27,17.3,1.9,21359.61\nR002,2006-09-28,18.1,0.1,40870.92\nR002,2006-09-29,19.8,0.0,39934.43\nR002,2006-09-30,19.2,6.5,29970.15\nR002,2006-10-01,16.5,10.2,9141.72\nR002,2006-10-02,15.1,0.0,38552.99\nR002,2006-10-03,15.2,0.0,38139.64\nR002,2006-10-04,16.2,0.0,38670.43\nR002,2006-10-05,17.5,0.0,39175.52\nR002,2006-10-06,19.6,0.0,38750.55\nR002,2006-10-07,19.6,0.1,36583.37\nR002,2006-10-08,18.3,0.0,37286.95\nR002,2006-10-09,17.6,0.0,34224.47\nR002,2006-10-10,15.6,2.2,27961.54\nR002,2006-10-11,15.3,0.2,28375.0\nR002,2006-10-12,16.3,0.0,32515.59\nR002,2006-10-13,15.1,2.4,29677.43\nR002,2006-10-14,14.3,0.1,34514.74\nR002,2006-10-15,12.7,5.0,13449.99\nR002,2006-10-16,11.4,10.5,19869.36\nR002,2006-10-17,10.3,9.1,19346.46\nR002,2006-10-18,8.0,26.7,0.0\nR002,2006-10-19,8.0,0.0,36000.0\nR002,2006-10-20,12.0,0.0,36000.0\nR002,2006-10-21,13.2,0.2,33766.94\nR002,2006-10-22,12.3,0.2,35497.3\nR002,2006-10-23,12.8,0.0,32615.01\nR002,2006-10-24,13.4,0.0,34318.75\nR002,2006-10-25,14.1,0.0,36000.0\nR002,2006-10-26,13.3,6.0,20149.83\nR002,2006-10-27,14.0,0.0,36000.0\nR002,2006-10-28,13.2,0.0,36000.0\nR002,2006-10-29,13.1,0.4,32095.33\nR002,2006-10-30,12.3,11.5,10102.57\nR002,2006-10-31,14.9,0.0,32996.47\nR002,2006-11-01,16.7,0.0,36000.0\nR002,2006-11-02,14.1,17.2,8696.66\nR002,2006-11-03,13.4,0.4,33807.96\nR002,2006-11-04,11.9,44.0,0.0\nR002,2006-11-05,6.9,31.2,0.0\nR002,2006-11-06,4.0,1.9,33271.08\nR002,2006-11-07,3.0,34.7,2438.5\nR002,2006-11-08,2.8,5.7,29760.37\nR002,2006-11-09,6.5,0.8,26579.0\nR002,2006-11-10,6.3,0.0,34959.78\nR002,2006-11-11,5.3,18.5,0.0\nR002,2006-11-12,2.5,0.5,19369.03\nR002,2006-11-13,5.6,3.0,23127.42\nR002,2006-11-14,4.6,34.2,23343.92\nR002,2006-11-15,4.2,21.7,8934.25\nR002,2006-11-16,6.0,18.8,2509.92\nR002,2006-11-17,6.0,16.4,0.0\nR002,2006-11-18,1.0,0.0,31040.14\nR002,2006-11-19,-1.2,0.0,32578.79\nR002,2006-11-20,1.9,0.0,31559.96\nR002,2006-11-21,3.9,0.0,29977.34\nR002,2006-11-22,5.6,0.0,28800.0\nR002,2006-11-23,5.5,0.0,32214.67\nR002,2006-11-24,5.2,0.0,31104.52\nR002,2006-11-25,4.9,0.0,31544.24\nR002,2006-11-26,5.4,0.0,30863.66\nR002,2006-11-27,8.1,0.0,23572.05\nR002,2006-11-28,6.6,0.0,26882.23\nR002,2006-11-29,5.3,0.0,28138.19\nR002,2006-11-30,3.7,0.3,29214.02\nR002,2006-12-01,4.3,10.0,18000.0\nR002,2006-12-02,2.5,1.6,29559.05\nR002,2006-12-03,3.7,0.0,29970.54\nR002,2006-12-04,5.3,0.0,30365.74\nR002,2006-12-05,4.9,0.0,30378.21\nR002,2006-12-06,5.4,0.0,29621.66\nR002,2006-12-07,4.4,0.0,30398.88\nR002,2006-12-08,4.3,0.0,28865.89\nR002,2006-12-09,3.0,0.0,30413.69\nR002,2006-12-10,5.4,0.0,30418.85\nR002,2006-12-11,4.5,0.0,30422.45\nR002,2006-12-12,3.7,0.0,30036.71\nR002,2006-12-13,3.5,0.6,28871.73\nR002,2006-12-14,3.6,6.9,3603.47\nR002,2006-12-15,2.8,1.6,30031.21\nR002,2006-12-16,3.9,0.1,29636.34\nR002,2006-12-17,6.8,0.0,29487.83\nR002,2006-12-18,8.3,0.0,23607.6\nR002,2006-12-19,8.4,0.0,1510.76\nR002,2006-12-20,8.0,5.5,4236.27\nR002,2006-12-21,4.8,19.5,0.0\nR002,2006-12-22,0.3,10.2,29960.91\nR002,2006-12-23,-0.2,3.9,24752.35\nR002,2006-12-24,0.7,4.8,30318.85\nR002,2006-12-25,1.6,33.7,0.0\nR002,2006-12-26,-2.5,10.1,23826.28\nR002,2006-12-27,-10.4,0.0,31360.67\nR002,2006-12-28,-6.0,0.3,28800.0\nR002,2006-12-29,0.8,5.7,2203.94\nR002,2006-12-30,0.6,10.4,20585.15\nR002,2006-12-31,-3.3,0.0,31544.18\nR002,2007-01-01,-1.0,1.7,27902.29\nR002,2007-01-02,1.9,0.3,19645.38\nR002,2007-01-03,4.7,0.0,16954.84\nR002,2007-01-04,5.6,0.0,31656.96\nR002,2007-01-05,3.1,0.0,27231.53\nR002,2007-01-06,2.5,3.9,927.5\nR002,2007-01-07,1.8,0.8,30182.61\nR002,2007-01-08,2.2,3.6,12421.81\nR002,2007-01-09,3.1,2.5,17475.61\nR002,2007-01-10,3.1,0.0,28391.45\nR002,2007-01-11,2.6,0.0,32204.68\nR002,2007-01-12,3.5,0.0,32275.44\nR002,2007-01-13,4.3,3.9,28405.54\nR002,2007-01-14,2.2,30.6,4370.78\nR002,2007-01-15,3.0,15.7,0.0\nR002,2007-01-16,0.1,10.8,13552.12\nR002,2007-01-17,0.9,0.6,30347.87\nR002,2007-01-18,2.2,0.0,28932.65\nR002,2007-01-19,4.9,0.0,32400.0\nR002,2007-01-20,3.9,34.9,0.0\nR002,2007-01-21,1.2,2.5,32400.0\nR002,2007-01-22,3.5,1.4,18000.0\nR002,2007-01-23,5.4,0.0,32400.0\nR002,2007-01-24,7.4,0.0,29530.79\nR002,2007-01-25,9.4,0.0,32400.0\nR002,2007-01-26,8.8,0.0,32130.43\nR002,2007-01-27,8.3,1.5,2619.6\nR002,2007-01-28,4.6,37.6,0.0\nR002,2007-01-29,-4.4,1.4,26464.87\nR002,2007-01-30,-2.3,22.9,12135.33\nR002,2007-01-31,-4.3,6.5,18947.5\nR002,2007-02-01,-4.0,1.8,26941.81\nR002,2007-02-02,0.1,6.7,6980.61\nR002,2007-02-03,1.1,1.9,30420.19\nR002,2007-02-04,1.4,1.0,29403.47\nR002,2007-02-05,0.4,7.6,26115.86\nR002,2007-02-06,1.1,2.7,14776.68\nR002,2007-02-07,2.9,0.3,31421.37\nR002,2007-02-08,4.0,0.0,26549.85\nR002,2007-02-09,5.5,0.0,35307.92\nR002,2007-02-10,4.5,0.0,35759.91\nR002,2007-02-11,5.3,0.0,35796.76\nR002,2007-02-12,5.7,0.0,32300.58\nR002,2007-02-13,5.9,0.0,35528.33\nR002,2007-02-14,5.7,0.0,36919.41\nR002,2007-02-15,6.0,0.0,37191.23\nR002,2007-02-16,4.7,0.0,37361.43\nR002,2007-02-17,4.0,2.9,21594.64\nR002,2007-02-18,1.4,0.1,35360.91\nR002,2007-02-19,2.0,0.0,37880.21\nR002,2007-02-20,3.0,6.7,588.06\nR002,2007-02-21,3.3,8.4,0.0\nR002,2007-02-22,2.0,2.2,36387.43\nR002,2007-02-23,1.2,4.9,3995.4\nR002,2007-02-24,-3.0,11.6,31988.97\nR002,2007-02-25,-6.8,0.0,38940.42\nR002,2007-02-26,-5.1,0.0,37844.18\nR002,2007-02-27,-2.7,0.0,39296.11\nR002,2007-02-28,2.8,0.0,32010.06\nR002,2007-03-01,5.5,9.1,5659.69\nR002,2007-03-02,5.8,2.7,23283.0\nR002,2007-03-03,6.5,3.2,20430.23\nR002,2007-03-04,7.2,6.9,32592.47\nR002,2007-03-05,6.5,6.1,7178.59\nR002,2007-03-06,6.9,6.8,20264.47\nR002,2007-03-07,5.6,2.1,14712.52\nR002,2007-03-08,5.9,0.0,39600.0\nR002,2007-03-09,5.7,0.0,39600.0\nR002,2007-03-10,5.7,0.0,39492.37\nR002,2007-03-11,6.0,0.0,39600.0\nR002,2007-03-12,4.9,6.2,27897.24\nR002,2007-03-13,4.1,8.0,337.82\nR002,2007-03-14,6.5,0.2,36000.0\nR002,2007-03-15,6.1,0.0,37306.08\nR002,2007-03-16,4.7,2.9,35646.75\nR002,2007-03-17,3.3,2.9,36471.2\nR002,2007-03-18,2.9,23.7,0.0\nR002,2007-03-19,2.0,0.5,39600.0\nR002,2007-03-20,4.5,0.0,41350.83\nR002,2007-03-21,7.1,0.0,41895.77\nR002,2007-03-22,8.1,0.0,42123.3\nR002,2007-03-23,7.2,0.0,42228.55\nR002,2007-03-24,8.4,11.6,13574.11\nR002,2007-03-25,6.7,9.2,33633.04\nR002,2007-03-26,3.8,14.8,695.93\nR002,2007-03-27,4.3,6.3,25712.1\nR002,2007-03-28,4.2,1.0,38600.81\nR002,2007-03-29,4.6,0.6,38501.37\nR002,2007-03-30,5.3,2.2,36883.19\nR002,2007-03-31,5.0,0.0,35110.64\nR002,2007-04-01,7.6,0.9,38156.81\nR002,2007-04-02,7.7,6.8,0.0\nR002,2007-04-03,6.3,7.5,9067.54\nR002,2007-04-04,5.4,4.8,24448.42\nR002,2007-04-05,7.1,0.0,40238.91\nR002,2007-04-06,6.2,20.5,97.64\nR002,2007-04-07,4.5,0.0,45935.01\nR002,2007-04-08,7.3,0.0,46040.52\nR002,2007-04-09,6.5,0.0,45552.22\nR002,2007-04-10,7.4,0.0,45199.98\nR002,2007-04-11,8.4,1.7,28066.11\nR002,2007-04-12,8.1,0.4,43406.38\nR002,2007-04-13,6.2,13.1,22925.16\nR002,2007-04-14,4.2,1.8,46554.68\nR002,2007-04-15,4.8,0.5,35340.87\nR002,2007-04-16,5.2,4.9,26514.15\nR002,2007-04-17,4.8,0.8,45010.87\nR002,2007-04-18,6.6,0.0,46171.1\nR002,2007-04-19,7.1,3.2,22732.96\nR002,2007-04-20,5.8,8.1,28268.7\nR002,2007-04-21,7.5,0.0,45115.46\nR002,2007-04-22,9.9,1.3,36000.0\nR002,2007-04-23,7.4,8.3,20505.08\nR002,2007-04-24,7.6,0.0,43438.26\nR002,2007-04-25,8.1,3.0,39697.93\nR002,2007-04-26,9.7,1.3,44323.57\nR002,2007-04-27,11.5,1.3,45450.77\nR002,2007-04-28,11.4,2.5,45742.58\nR002,2007-04-29,12.1,0.0,43329.38\nR002,2007-04-30,10.2,12.0,3860.69\nR002,2007-05-01,10.4,5.1,37336.96\nR002,2007-05-02,10.5,17.4,344.77\nR002,2007-05-03,10.0,16.4,43902.05\nR002,2007-05-04,9.3,0.0,49434.49\nR002,2007-05-05,12.1,0.0,49705.53\nR002,2007-05-06,15.0,0.0,49712.57\nR002,2007-05-07,17.4,0.0,49683.94\nR002,2007-05-08,20.0,0.2,44475.23\nR002,2007-05-09,15.8,15.1,42159.89\nR002,2007-05-10,14.7,2.5,46177.43\nR002,2007-05-11,14.3,0.0,42273.69\nR002,2007-05-12,16.9,0.0,50228.65\nR002,2007-05-13,16.8,1.1,40385.24\nR002,2007-05-14,16.0,0.1,45562.23\nR002,2007-05-15,15.1,0.0,50400.0\nR002,2007-05-16,18.1,0.0,50462.63\nR002,2007-05-17,21.2,0.0,50433.24\nR002,2007-05-18,22.6,0.0,50404.3\nR002,2007-05-19,22.3,0.0,50400.0\nR002,2007-05-20,21.8,0.2,50400.0\nR002,2007-05-21,22.1,2.5,50400.0\nR002,2007-05-22,22.7,1.5,50649.08\nR002,2007-05-23,23.6,0.8,50400.0\nR002,2007-05-24,24.3,0.4,48726.8\nR002,2007-05-25,24.0,1.3,50400.0\nR002,2007-05-26,25.2,0.7,50400.0\nR002,2007-05-27,24.7,3.2,32380.0\nR002,2007-05-28,23.9,4.9,43200.0\nR002,2007-05-29,23.6,2.2,51553.79\nR002,2007-05-30,22.3,5.1,43850.28\nR002,2007-05-31,22.0,8.8,47944.83\nR002,2007-06-01,21.9,10.9,48522.11\nR002,2007-06-02,20.5,2.9,49403.88\nR002,2007-06-03,19.7,10.7,26685.26\nR002,2007-06-04,18.5,14.6,21955.92\nR002,2007-06-05,18.2,22.3,3542.59\nR002,2007-06-06,19.7,0.0,49156.04\nR002,2007-06-07,20.3,0.5,51012.15\nR002,2007-06-08,18.7,9.2,13828.75\nR002,2007-06-09,18.2,19.3,21917.74\nR002,2007-06-10,18.1,12.9,48354.92\nR002,2007-06-11,18.7,3.3,50158.57\nR002,2007-06-12,19.1,3.6,50929.21\nR002,2007-06-13,18.9,1.9,51661.47\nR002,2007-06-14,21.3,0.1,53114.85\nR002,2007-06-15,23.0,1.0,53104.01\nR002,2007-06-16,24.1,0.8,53461.12\nR002,2007-06-17,22.8,4.5,52716.0\nR002,2007-06-18,20.3,14.0,46776.39\nR002,2007-06-19,20.0,8.0,35723.99\nR002,2007-06-20,18.7,7.5,34726.29\nR002,2007-06-21,17.8,5.9,47105.65\nR002,2007-06-22,18.1,0.9,51619.58\nR002,2007-06-23,20.7,0.2,53673.69\nR002,2007-06-24,21.8,0.4,51927.68\nR002,2007-06-25,19.1,0.4,53965.94\nR002,2007-06-26,18.3,0.5,53388.33\nR002,2007-06-27,21.5,0.0,53971.55\nR002,2007-06-28,23.8,0.4,53970.73\nR002,2007-06-29,23.6,1.0,53967.42\nR002,2007-06-30,19.6,8.7,47482.98\nR002,2007-07-01,19.8,3.5,53197.19\nR002,2007-07-02,21.0,1.0,53380.09\nR002,2007-07-03,21.3,0.4,52850.09\nR002,2007-07-04,22.8,0.1,53147.46\nR002,2007-07-05,24.3,0.0,53898.17\nR002,2007-07-06,25.5,0.0,53879.07\nR002,2007-07-07,19.3,25.8,19175.75\nR002,2007-07-08,18.6,0.8,53760.92\nR002,2007-07-09,18.4,0.2,53041.53\nR002,2007-07-10,19.5,0.0,53191.58\nR002,2007-07-11,23.1,0.0,53752.02\nR002,2007-07-12,25.7,1.0,53720.37\nR002,2007-07-13,25.6,1.4,53073.74\nR002,2007-07-14,21.0,14.0,29549.05\nR002,2007-07-15,20.5,4.3,45150.53\nR002,2007-07-16,20.0,0.4,51173.78\nR002,2007-07-17,21.4,0.0,52404.94\nR002,2007-07-18,22.3,0.0,53487.74\nR002,2007-07-19,22.9,0.0,53441.92\nR002,2007-07-20,24.8,0.0,53376.17\nR002,2007-07-21,25.3,0.0,53271.69\nR002,2007-07-22,26.4,0.2,52713.97\nR002,2007-07-23,25.8,0.0,52676.09\nR002,2007-07-24,25.5,0.0,51934.13\nR002,2007-07-25,27.1,0.0,52248.61\nR002,2007-07-26,27.4,0.1,51510.4\nR002,2007-07-27,25.0,0.1,50894.65\nR002,2007-07-28,23.6,0.0,49895.39\nR002,2007-07-29,24.1,0.5,49945.9\nR002,2007-07-30,25.7,0.0,51020.37\nR002,2007-07-31,27.3,0.0,50639.57\nR002,2007-08-01,27.3,0.3,48034.89\nR002,2007-08-02,24.0,11.8,46916.44\nR002,2007-08-03,22.1,15.5,32745.75\nR002,2007-08-04,22.7,2.8,46800.0\nR002,2007-08-05,24.5,1.5,46844.69\nR002,2007-08-06,27.2,0.0,48679.13\nR002,2007-08-07,28.4,2.0,48401.44\nR002,2007-08-08,26.8,1.8,47970.49\nR002,2007-08-09,24.5,7.7,46045.66\nR002,2007-08-10,23.7,1.0,47349.19\nR002,2007-08-11,25.0,2.8,47377.98\nR002,2007-08-12,26.5,0.4,47406.97\nR002,2007-08-13,27.4,0.0,47771.6\nR002,2007-08-14,24.0,6.8,39331.58\nR002,2007-08-15,23.5,15.1,44847.39\nR002,2007-08-16,22.9,4.2,46800.0\nR002,2007-08-17,22.7,4.2,31634.64\nR002,2007-08-18,22.5,3.2,46302.96\nR002,2007-08-19,22.8,5.4,46476.32\nR002,2007-08-20,23.3,4.1,46800.0\nR002,2007-08-21,23.8,4.2,46800.0\nR002,2007-08-22,22.6,11.7,46211.64\nR002,2007-08-23,21.7,6.0,36490.89\nR002,2007-08-24,23.2,3.0,39215.78\nR002,2007-08-25,23.6,8.7,39598.75\nR002,2007-08-26,23.0,2.1,46800.0\nR002,2007-08-27,22.5,4.0,38085.68\nR002,2007-08-28,21.4,6.0,37966.74\nR002,2007-08-29,20.7,2.5,30090.61\nR002,2007-08-30,21.3,0.9,40869.38\nR002,2007-08-31,21.8,0.0,43569.34\nR002,2007-09-01,22.8,0.0,46262.86\nR002,2007-09-02,24.0,0.1,45576.95\nR002,2007-09-03,24.5,0.6,45304.38\nR002,2007-09-04,25.8,0.0,45033.2\nR002,2007-09-05,25.8,0.4,44443.1\nR002,2007-09-06,25.2,1.2,44174.69\nR002,2007-09-07,24.7,1.5,43587.76\nR002,2007-09-08,24.0,3.8,43640.99\nR002,2007-09-09,20.9,17.7,35890.18\nR002,2007-09-10,18.3,20.4,34541.76\nR002,2007-09-11,18.9,0.0,42839.52\nR002,2007-09-12,19.9,0.0,43200.0\nR002,2007-09-13,19.7,12.5,43200.0\nR002,2007-09-14,15.1,14.8,34468.97\nR002,2007-09-15,15.9,0.0,41050.78\nR002,2007-09-16,15.1,6.5,30999.58\nR002,2007-09-17,11.6,4.0,20747.72\nR002,2007-09-18,15.2,0.0,38164.2\nR002,2007-09-19,17.9,0.0,42456.92\nR002,2007-09-20,19.3,0.1,42730.1\nR002,2007-09-21,19.1,0.0,42525.19\nR002,2007-09-22,19.5,0.0,42559.93\nR002,2007-09-23,20.0,1.2,39648.43\nR002,2007-09-24,19.0,6.8,19115.66\nR002,2007-09-25,17.6,20.5,12792.14\nR002,2007-09-26,17.7,0.1,38195.81\nR002,2007-09-27,18.1,0.0,41464.06\nR002,2007-09-28,19.0,0.0,41181.41\nR002,2007-09-29,18.6,0.0,41218.04\nR002,2007-09-30,18.5,0.0,40830.02\nR002,2007-10-01,18.3,1.4,40117.07\nR002,2007-10-02,17.9,0.2,40044.05\nR002,2007-10-03,18.0,0.3,39971.97\nR002,2007-10-04,19.2,0.0,38858.38\nR002,2007-10-05,19.7,0.1,38072.04\nR002,2007-10-06,18.5,1.5,37483.49\nR002,2007-10-07,17.0,0.6,36883.1\nR002,2007-10-08,17.0,1.1,26634.66\nR002,2007-10-09,16.6,0.4,35404.1\nR002,2007-10-10,16.6,0.7,36083.49\nR002,2007-10-11,16.8,1.2,36461.7\nR002,2007-10-12,16.4,0.0,36840.89\nR002,2007-10-13,17.8,0.7,36243.68\nR002,2007-10-14,16.3,13.4,20632.01\nR002,2007-10-15,12.0,32.7,168.14\nR002,2007-10-16,10.3,0.2,33719.86\nR002,2007-10-17,10.5,0.0,33603.95\nR002,2007-10-18,11.4,0.0,36000.0\nR002,2007-10-19,13.7,0.0,36000.0\nR002,2007-10-20,15.6,0.0,36000.0\nR002,2007-10-21,16.5,0.0,36000.0\nR002,2007-10-22,16.2,0.0,36000.0\nR002,2007-10-23,16.8,0.0,36000.0\nR002,2007-10-24,18.0,0.0,36000.0\nR002,2007-10-25,18.1,0.1,32038.09\nR002,2007-10-26,14.3,7.6,25488.0\nR002,2007-10-27,12.5,0.6,26569.07\nR002,2007-10-28,11.2,4.1,23703.92\nR002,2007-10-29,12.0,0.0,35938.9\nR002,2007-10-30,11.5,4.6,21600.0\nR002,2007-10-31,12.5,2.3,18394.66\nR002,2007-11-01,12.1,0.0,33555.41\nR002,2007-11-02,10.4,5.9,22946.35\nR002,2007-11-03,5.5,30.0,14400.0\nR002,2007-11-04,3.3,0.0,27409.32\nR002,2007-11-05,8.3,10.6,19236.39\nR002,2007-11-06,8.6,18.2,2697.54\nR002,2007-11-07,6.9,0.1,30195.98\nR002,2007-11-08,5.6,8.1,1380.68\nR002,2007-11-09,4.8,12.6,17955.02\nR002,2007-11-10,4.8,0.0,29661.06\nR002,2007-11-11,4.3,26.1,0.0\nR002,2007-11-12,2.7,7.8,26160.21\nR002,2007-11-13,6.2,12.5,30686.67\nR002,2007-11-14,5.9,15.7,0.0\nR002,2007-11-15,8.0,0.0,34268.18\nR002,2007-11-16,11.0,0.0,32670.8\nR002,2007-11-17,11.1,0.0,33122.12\nR002,2007-11-18,9.6,2.7,28794.31\nR002,2007-11-19,9.8,0.0,28406.85\nR002,2007-11-20,10.2,0.0,27800.42\nR002,2007-11-21,8.4,0.0,31611.56\nR002,2007-11-22,3.7,1.9,24438.67\nR002,2007-11-23,2.1,0.0,31018.59\nR002,2007-11-24,3.2,0.0,31830.18\nR002,2007-11-25,3.8,0.0,31852.98\nR002,2007-11-26,5.7,0.0,31223.22\nR002,2007-11-27,6.3,26.3,17703.93\nR002,2007-11-28,-2.2,9.0,2632.38\nR002,2007-11-29,-4.8,0.7,29900.86\nR002,2007-11-30,1.0,0.0,27145.39\nR002,2007-12-01,5.7,14.8,10905.61\nR002,2007-12-02,1.6,20.7,0.0\nR002,2007-12-03,-1.4,0.0,27915.13\nR002,2007-12-04,1.9,0.0,29982.82\nR002,2007-12-05,2.4,0.0,30377.8\nR002,2007-12-06,3.4,0.0,28800.0\nR002,2007-12-07,5.3,8.9,0.0\nR002,2007-12-08,4.2,2.3,24721.94\nR002,2007-12-09,2.6,0.0,29256.58\nR002,2007-12-10,4.4,2.2,28245.11\nR002,2007-12-11,5.7,11.3,12847.53\nR002,2007-12-12,5.4,12.6,2070.54\nR002,2007-12-13,5.9,4.4,13161.66\nR002,2007-12-14,6.4,15.7,0.0\nR002,2007-12-15,4.1,3.3,25438.47\nR002,2007-12-16,-1.1,0.0,30024.28\nR002,2007-12-17,-1.2,6.5,20628.14\nR002,2007-12-18,-3.3,6.4,24665.62\nR002,2007-12-19,-5.3,2.4,12844.08\nR002,2007-12-20,-1.4,1.9,29207.46\nR002,2007-12-21,0.2,8.3,8134.77\nR002,2007-12-22,-1.2,0.2,29415.54\nR002,2007-12-23,0.7,2.4,29556.91\nR002,2007-12-24,0.2,0.0,29932.05\nR002,2007-12-25,-1.1,0.0,31268.43\nR002,2007-12-26,-2.0,0.0,31305.58\nR002,2007-12-27,-1.7,0.0,29876.1\nR002,2007-12-28,1.0,1.2,17640.48\nR002,2007-12-29,0.8,3.9,14227.83\nR002,2007-12-30,0.3,0.2,30029.92\nR002,2007-12-31,-0.6,1.0,31344.41\nR002,2008-01-01,-2.2,0.0,31575.52\nR002,2008-01-02,-2.0,0.0,31629.28\nR002,2008-01-03,-1.6,0.0,30863.88\nR002,2008-01-04,0.4,4.2,394.34\nR002,2008-01-05,-0.0,10.8,2514.92\nR002,2008-01-06,-4.5,2.7,31864.93\nR002,2008-01-07,-6.2,0.7,31928.29\nR002,2008-01-08,-6.9,0.0,31993.14\nR002,2008-01-09,-4.9,0.0,32059.34\nR002,2008-01-10,-3.4,0.0,32126.76\nR002,2008-01-11,-4.2,0.4,32195.23\nR002,2008-01-12,-4.9,0.0,32264.64\nR002,2008-01-13,-3.1,0.0,32334.83\nR002,2008-01-14,-2.2,0.0,32400.0\nR002,2008-01-15,-1.0,0.0,32400.0\nR002,2008-01-16,-0.9,0.0,32400.0\nR002,2008-01-17,-0.1,0.0,32400.0\nR002,2008-01-18,0.5,0.0,32400.0\nR002,2008-01-19,0.3,0.0,32400.0\nR002,2008-01-20,-0.6,0.0,32400.0\nR002,2008-01-21,1.4,0.9,29074.0\nR002,2008-01-22,3.6,7.8,29772.64\nR002,2008-01-23,6.2,0.0,32400.0\nR002,2008-01-24,3.3,11.9,0.0\nR002,2008-01-25,-2.3,0.2,32127.48\nR002,2008-01-26,-1.3,1.5,24230.69\nR002,2008-01-27,-0.2,6.8,29774.99\nR002,2008-01-28,-0.2,25.9,0.0\nR002,2008-01-29,-0.9,2.8,1208.34\nR002,2008-01-30,-1.3,3.0,11216.41\nR002,2008-01-31,-3.0,5.2,32400.0\nR002,2008-02-01,-3.3,0.0,32114.39\nR002,2008-02-02,-1.6,0.0,32696.2\nR002,2008-02-03,-0.8,0.0,33083.25\nR002,2008-02-04,0.7,0.0,33002.0\nR002,2008-02-05,1.3,0.0,32919.72\nR002,2008-02-06,3.8,0.0,33184.67\nR002,2008-02-07,3.6,0.0,32822.88\nR002,2008-02-08,3.9,0.0,29051.14\nR002,2008-02-09,3.3,0.0,33765.86\nR002,2008-02-10,3.3,0.0,35408.64\nR002,2008-02-11,2.9,0.0,34805.34\nR002,2008-02-12,0.1,0.0,36164.83\nR002,2008-02-13,0.2,3.9,32285.96\nR002,2008-02-14,-2.4,8.0,30282.31\nR002,2008-02-15,-3.4,11.0,26740.8\nR002,2008-02-16,-3.0,14.2,509.74\nR002,2008-02-17,-2.4,10.4,0.0\nR002,2008-02-18,0.8,12.7,0.0\nR002,2008-02-19,-1.5,12.9,0.0\nR002,2008-02-20,-3.3,20.9,24594.44\nR002,2008-02-21,-1.5,2.1,31400.7\nR002,2008-02-22,3.4,1.0,30309.78\nR002,2008-02-23,5.6,0.5,33805.72\nR002,2008-02-24,3.4,17.1,1.55\nR002,2008-02-25,0.9,0.0,36561.14\nR002,2008-02-26,2.7,0.0,39068.54\nR002,2008-02-27,5.7,0.0,39243.44\nR002,2008-02-28,7.7,0.0,39389.83\nR002,2008-02-29,6.7,9.9,3597.81\nR002,2008-03-01,4.7,0.0,33954.1\nR002,2008-03-02,7.6,7.0,39600.0\nR002,2008-03-03,4.5,19.5,20732.21\nR002,2008-03-04,4.1,0.6,38678.8\nR002,2008-03-05,9.9,0.0,38291.04\nR002,2008-03-06,10.2,0.2,34953.74\nR002,2008-03-07,7.2,0.9,28417.72\nR002,2008-03-08,9.8,0.0,39293.43\nR002,2008-03-09,13.8,0.0,20817.37\nR002,2008-03-10,12.5,1.7,30411.91\nR002,2008-03-11,6.1,16.1,23853.41\nR002,2008-03-12,5.8,0.0,39600.0\nR002,2008-03-13,8.4,0.0,28513.17\nR002,2008-03-14,9.4,0.9,39600.0\nR002,2008-03-15,5.5,10.7,19222.98\nR002,2008-03-16,2.4,15.5,19427.65\nR002,2008-03-17,6.2,0.0,39600.0\nR002,2008-03-18,11.6,0.6,39979.45\nR002,2008-03-19,8.7,17.8,5716.15\nR002,2008-03-20,6.9,15.4,3589.09\nR002,2008-03-21,8.2,26.1,6249.35\nR002,2008-03-22,8.1,6.1,32763.33\nR002,2008-03-23,12.8,0.0,39708.47\nR002,2008-03-24,19.3,0.6,34461.58\nR002,2008-03-25,17.7,9.2,42142.28\nR002,2008-03-26,8.5,13.9,41060.56\nR002,2008-03-27,6.8,1.9,37138.63\nR002,2008-03-28,10.6,0.0,42472.22\nR002,2008-03-29,11.6,2.7,35780.59\nR002,2008-03-30,12.6,2.0,18000.0\nR002,2008-03-31,11.3,6.6,15363.09\nR002,2008-04-01,9.8,10.1,28316.49\nR002,2008-04-02,6.6,20.6,0.0\nR002,2008-04-03,7.5,0.7,42227.16\nR002,2008-04-04,8.4,0.0,43649.52\nR002,2008-04-05,8.5,3.1,42654.56\nR002,2008-04-06,10.4,0.1,33879.53\nR002,2008-04-07,12.9,5.3,38875.65\nR002,2008-04-08,11.1,11.4,37052.37\nR002,2008-04-09,10.3,1.1,37102.51\nR002,2008-04-10,12.1,1.3,44094.73\nR002,2008-04-11,13.2,0.0,41891.89\nR002,2008-04-12,13.4,1.3,37940.76\nR002,2008-04-13,15.8,0.3,40525.31\nR002,2008-04-14,18.2,0.2,42296.87\nR002,2008-04-15,20.5,0.0,43200.0\nR002,2008-04-16,18.5,3.4,40697.41\nR002,2008-04-17,9.7,24.2,9473.06\nR002,2008-04-18,10.8,0.0,46344.45\nR002,2008-04-19,12.8,0.0,33010.36\nR002,2008-04-20,12.9,0.8,46800.0\nR002,2008-04-21,14.3,1.6,46800.0\nR002,2008-04-22,15.3,0.0,46866.53\nR002,2008-04-23,20.3,2.1,19406.32\nR002,2008-04-24,17.0,3.8,24674.07\nR002,2008-04-25,14.0,16.7,13380.53\nR002,2008-04-26,16.3,1.5,37057.7\nR002,2008-04-27,18.9,5.1,10547.17\nR002,2008-04-28,16.6,1.9,45943.71\nR002,2008-04-29,15.2,14.3,24755.11\nR002,2008-04-30,13.0,16.3,23270.8\nR002,2008-05-01,12.7,4.1,32917.4\nR002,2008-05-02,12.3,6.2,39285.59\nR002,2008-05-03,12.3,7.4,30195.26\nR002,2008-05-04,12.3,7.9,38499.75\nR002,2008-05-05,10.8,13.0,12315.7\nR002,2008-05-06,11.3,8.6,18261.39\nR002,2008-05-07,9.7,9.7,31045.67\nR002,2008-05-08,9.1,3.6,47929.12\nR002,2008-05-09,9.2,5.3,29943.42\nR002,2008-05-10,8.2,13.2,0.0\nR002,2008-05-11,9.1,6.5,39927.71\nR002,2008-05-12,9.0,3.8,45785.46\nR002,2008-05-13,8.2,9.5,34469.66\nR002,2008-05-14,11.3,2.1,46641.92\nR002,2008-05-15,13.7,0.0,50400.0\nR002,2008-05-16,14.1,3.8,43705.09\nR002,2008-05-17,14.4,3.3,46800.0\nR002,2008-05-18,16.0,2.5,48656.77\nR002,2008-05-19,16.4,9.2,23260.5\nR002,2008-05-20,17.4,1.9,46477.55\nR002,2008-05-21,18.8,1.0,48410.32\nR002,2008-05-22,20.3,0.1,50629.2\nR002,2008-05-23,22.1,0.1,50959.35\nR002,2008-05-24,21.5,2.0,50897.92\nR002,2008-05-25,17.9,24.7,5819.59\nR002,2008-05-26,16.2,31.2,21346.08\nR002,2008-05-27,15.2,1.4,47812.78\nR002,2008-05-28,14.0,5.6,18452.63\nR002,2008-05-29,12.1,29.3,18377.18\nR002,2008-05-30,11.5,5.2,39031.58\nR002,2008-05-31,12.9,0.0,50769.3\nR002,2008-06-01,15.4,0.0,51836.01\nR002,2008-06-02,13.9,1.2,36691.08\nR002,2008-06-03,12.4,3.9,44502.23\nR002,2008-06-04,14.6,0.0,50502.49\nR002,2008-06-05,19.9,0.0,52859.24\nR002,2008-06-06,18.9,5.8,32770.51\nR002,2008-06-07,18.7,2.5,49975.12\nR002,2008-06-08,17.4,8.9,35494.85\nR002,2008-06-09,17.2,10.4,38797.22\nR002,2008-06-10,18.7,3.2,45725.47\nR002,2008-06-11,17.5,14.6,1122.81\nR002,2008-06-12,16.8,21.6,8909.4\nR002,2008-06-13,18.1,5.2,40135.58\nR002,2008-06-14,19.4,1.4,53105.3\nR002,2008-06-15,20.0,5.3,53094.39\nR002,2008-06-16,19.6,1.2,48113.87\nR002,2008-06-17,19.2,1.4,42399.98\nR002,2008-06-18,18.6,6.4,43059.35\nR002,2008-06-19,20.2,0.7,50363.42\nR002,2008-06-20,20.9,2.3,50736.56\nR002,2008-06-21,19.5,11.0,42488.88\nR002,2008-06-22,20.3,0.4,49976.92\nR002,2008-06-23,18.9,1.9,51804.73\nR002,2008-06-24,17.4,0.0,50217.16\nR002,2008-06-25,19.8,0.5,51002.18\nR002,2008-06-26,20.9,0.9,51265.87\nR002,2008-06-27,22.5,0.5,50629.19\nR002,2008-06-28,23.3,0.8,52283.97\nR002,2008-06-29,21.4,15.1,48741.54\nR002,2008-06-30,20.5,2.4,51135.01\nR002,2008-07-01,18.5,1.5,49888.64\nR002,2008-07-02,17.5,3.4,51687.83\nR002,2008-07-03,17.1,8.7,44151.76\nR002,2008-07-04,17.8,1.8,51421.21\nR002,2008-07-05,20.1,0.2,51685.63\nR002,2008-07-06,20.4,0.1,49359.17\nR002,2008-07-07,18.3,6.9,44559.18\nR002,2008-07-08,19.1,1.8,49047.35\nR002,2008-07-09,20.2,0.3,52627.92\nR002,2008-07-10,20.2,0.5,52613.82\nR002,2008-07-11,20.7,3.2,48631.3\nR002,2008-07-12,21.4,0.0,51137.49\nR002,2008-07-13,23.6,0.0,52721.01\nR002,2008-07-14,23.1,9.6,48628.87\nR002,2008-07-15,21.4,15.9,14719.37\nR002,2008-07-16,22.7,1.2,52401.29\nR002,2008-07-17,21.3,12.9,47816.74\nR002,2008-07-18,20.4,4.7,45853.26\nR002,2008-07-19,22.4,0.0,51738.97\nR002,2008-07-20,23.6,1.4,50275.0\nR002,2008-07-21,25.3,0.0,47844.05\nR002,2008-07-22,26.3,0.8,49652.65\nR002,2008-07-23,26.3,1.0,51593.0\nR002,2008-07-24,27.0,0.5,51555.9\nR002,2008-07-25,27.9,0.0,51518.91\nR002,2008-07-26,25.7,5.7,49140.14\nR002,2008-07-27,23.6,20.0,44126.25\nR002,2008-07-28,22.2,0.0,49589.9\nR002,2008-07-29,21.8,0.5,48253.74\nR002,2008-07-30,21.9,4.6,46732.33\nR002,2008-07-31,22.6,1.1,45479.46\nR002,2008-08-01,21.8,2.2,19985.39\nR002,2008-08-02,23.6,0.3,42459.65\nR002,2008-08-03,24.0,0.7,40893.89\nR002,2008-08-04,23.2,13.1,41451.32\nR002,2008-08-05,22.8,6.0,35187.99\nR002,2008-08-06,22.6,3.2,46800.0\nR002,2008-08-07,22.4,0.0,46943.65\nR002,2008-08-08,21.7,0.6,46973.16\nR002,2008-08-09,23.0,0.0,47002.84\nR002,2008-08-10,23.5,1.9,47707.1\nR002,2008-08-11,23.1,1.8,47062.79\nR002,2008-08-12,23.4,0.6,47092.99\nR002,2008-08-13,22.6,0.0,47123.32\nR002,2008-08-14,21.9,13.6,46508.57\nR002,2008-08-15,23.2,2.0,46851.01\nR002,2008-08-16,23.5,1.0,46800.0\nR002,2008-08-17,24.7,3.5,46800.0\nR002,2008-08-18,25.5,0.0,46945.31\nR002,2008-08-19,25.3,1.3,46800.0\nR002,2008-08-20,23.5,7.8,46800.0\nR002,2008-08-21,21.8,10.3,46800.0\nR002,2008-08-22,23.7,0.7,46800.0\nR002,2008-08-23,24.3,0.6,46800.0\nR002,2008-08-24,26.5,0.0,46800.0\nR002,2008-08-25,25.8,3.1,46800.0\nR002,2008-08-26,24.7,2.0,46575.6\nR002,2008-08-27,23.5,0.0,46701.89\nR002,2008-08-28,22.0,0.0,46599.26\nR002,2008-08-29,23.6,0.0,46495.56\nR002,2008-08-30,24.2,0.0,46115.88\nR002,2008-08-31,20.5,12.0,37134.92\nR002,2008-09-01,18.0,0.0,45567.43\nR002,2008-09-02,18.7,0.0,45616.24\nR002,2008-09-03,20.9,0.4,44703.55\nR002,2008-09-04,22.1,0.7,44113.7\nR002,2008-09-05,20.6,7.5,40460.03\nR002,2008-09-06,20.0,6.6,36250.86\nR002,2008-09-07,19.8,4.5,35928.57\nR002,2008-09-08,19.7,4.1,39193.51\nR002,2008-09-09,18.5,11.3,33660.23\nR002,2008-09-10,17.6,5.8,24061.24\nR002,2008-09-11,16.2,2.8,37918.06\nR002,2008-09-12,17.2,0.0,42846.09\nR002,2008-09-13,19.6,0.0,43152.05\nR002,2008-09-14,18.9,3.9,36300.45\nR002,2008-09-15,18.1,0.0,39428.13\nR002,2008-09-16,19.4,0.0,42950.11\nR002,2008-09-17,22.5,0.2,42882.54\nR002,2008-09-18,21.5,11.3,40939.04\nR002,2008-09-19,18.1,4.2,31258.34\nR002,2008-09-20,17.5,3.4,30474.07\nR002,2008-09-21,19.4,0.5,35141.59\nR002,2008-09-22,19.5,11.9,24513.49\nR002,2008-09-23,17.1,13.0,24857.98\nR002,2008-09-24,14.6,11.0,39088.59\nR002,2008-09-25,14.7,0.0,38697.7\nR002,2008-09-26,14.8,0.0,36461.89\nR002,2008-09-27,16.4,0.0,38965.65\nR002,2008-09-28,15.8,0.6,34761.22\nR002,2008-09-29,15.4,0.3,35014.66\nR002,2008-09-30,14.2,7.0,5067.05\nR002,2008-10-01,14.3,4.1,29933.7\nR002,2008-10-02,14.1,0.0,39195.88\nR002,2008-10-03,15.6,0.0,39924.61\nR002,2008-10-04,16.7,0.0,38926.46\nR002,2008-10-05,18.4,0.0,38110.32\nR002,2008-10-06,17.9,2.6,35239.15\nR002,2008-10-07,14.9,5.1,20480.33\nR002,2008-10-08,13.2,7.5,30840.86\nR002,2008-10-09,12.6,3.1,32824.6\nR002,2008-10-10,12.6,0.0,36124.52\nR002,2008-10-11,12.6,0.0,33764.68\nR002,2008-10-12,12.4,0.0,36558.99\nR002,2008-10-13,13.2,0.0,35633.19\nR002,2008-10-14,15.9,0.0,36341.04\nR002,2008-10-15,18.4,0.8,34268.36\nR002,2008-10-16,16.3,23.6,0.0\nR002,2008-10-17,14.5,4.3,32285.17\nR002,2008-10-18,13.3,0.0,36000.0\nR002,2008-10-19,12.5,9.5,34711.96\nR002,2008-10-20,11.6,0.0,36000.0\nR002,2008-10-21,11.2,3.1,212.58\nR002,2008-10-22,11.2,0.1,34156.74\nR002,2008-10-23,12.3,0.0,36000.0\nR002,2008-10-24,13.8,0.0,36000.0\nR002,2008-10-25,13.1,5.3,18784.83\nR002,2008-10-26,13.5,3.6,23406.87\nR002,2008-10-27,13.5,0.0,36000.0\nR002,2008-10-28,13.4,2.5,34207.73\nR002,2008-10-29,12.8,0.0,33964.07\nR002,2008-10-30,13.1,0.0,36000.0\nR002,2008-10-31,13.5,0.0,36000.0\nR002,2008-11-01,12.2,0.0,35992.79\nR002,2008-11-02,12.7,0.0,35912.63\nR002,2008-11-03,12.7,0.0,35832.38\nR002,2008-11-04,11.5,0.0,30954.89\nR002,2008-11-05,12.0,0.0,35242.59\nR002,2008-11-06,10.8,0.3,29168.36\nR002,2008-11-07,9.4,4.3,29626.73\nR002,2008-11-08,8.7,0.0,33460.04\nR002,2008-11-09,8.5,0.0,35019.93\nR002,2008-11-10,9.5,0.0,34874.01\nR002,2008-11-11,9.4,0.0,34730.99\nR002,2008-11-12,8.6,0.0,34590.9\nR002,2008-11-13,8.1,0.0,31438.4\nR002,2008-11-14,8.7,0.0,30010.18\nR002,2008-11-15,8.8,0.0,28731.06\nR002,2008-11-16,8.7,0.4,21375.69\nR002,2008-11-17,9.2,0.0,31353.92\nR002,2008-11-18,8.9,2.9,28581.47\nR002,2008-11-19,8.8,6.2,9818.78\nR002,2008-11-20,9.1,29.5,0.0\nR002,2008-11-21,9.3,10.8,19066.3\nR002,2008-11-22,10.9,0.0,31376.26\nR002,2008-11-23,9.9,30.4,0.0\nR002,2008-11-24,8.5,37.5,53.95\nR002,2008-11-25,10.3,1.8,17116.71\nR002,2008-11-26,10.4,0.0,30606.72\nR002,2008-11-27,8.8,4.1,28756.12\nR002,2008-11-28,8.3,5.2,19202.88\nR002,2008-11-29,7.8,0.0,30296.42\nR002,2008-11-30,7.8,0.0,28800.0\nR002,2008-12-01,8.1,1.4,26331.98\nR002,2008-12-02,6.6,0.0,30729.0\nR002,2008-12-03,6.9,0.0,30743.93\nR002,2008-12-04,9.7,0.0,30374.75\nR002,2008-12-05,12.6,0.0,27072.01\nR002,2008-12-06,14.6,0.0,25259.07\nR002,2008-12-07,12.5,0.0,28864.2\nR002,2008-12-08,12.9,0.0,24301.14\nR002,2008-12-09,7.4,28.5,0.0\nR002,2008-12-10,3.9,14.3,6632.47\nR002,2008-12-11,-0.8,0.0,30420.71\nR002,2008-12-12,0.8,0.0,30033.34\nR002,2008-12-13,2.5,0.0,30031.92\nR002,2008-12-14,2.1,0.0,30029.11\nR002,2008-12-15,1.2,0.0,30414.35\nR002,2008-12-16,2.1,0.0,29629.35\nR002,2008-12-17,2.0,0.0,29231.72\nR002,2008-12-18,3.0,0.0,30003.76\nR002,2008-12-19,4.2,0.0,28731.84\nR002,2008-12-20,4.5,0.1,26955.16\nR002,2008-12-21,5.4,3.2,4189.47\nR002,2008-12-22,5.9,2.3,24001.35\nR002,2008-12-23,5.7,7.5,0.0\nR002,2008-12-24,5.9,0.7,13852.91\nR002,2008-12-25,2.0,15.3,1622.28\nR002,2008-12-26,-2.8,0.0,31328.93\nR002,2008-12-27,-0.6,4.3,3183.96\nR002,2008-12-28,-1.0,2.4,15712.79\nR002,2008-12-29,-3.6,3.1,26607.83\nR002,2008-12-30,-6.0,2.4,24744.07\nR002,2008-12-31,-5.9,0.0,31567.96\nR002,2009-01-01,-5.6,0.0,31622.04\nR002,2009-01-02,-1.7,3.7,25003.57\nR002,2009-01-03,0.3,30.9,0.0\nR002,2009-01-04,0.4,4.7,23662.24\nR002,2009-01-05,4.1,31.7,0.0\nR002,2009-01-06,2.4,3.9,6248.31\nR002,2009-01-07,2.0,9.5,343.86\nR002,2009-01-08,0.4,0.9,31575.39\nR002,2009-01-09,-4.6,15.5,26242.23\nR002,2009-01-10,-4.3,1.6,21180.14\nR002,2009-01-11,-2.0,1.4,26513.16\nR002,2009-01-12,-2.9,0.0,32037.41\nR002,2009-01-13,-1.2,0.0,32377.24\nR002,2009-01-14,0.5,0.0,32400.0\nR002,2009-01-15,4.7,0.1,15269.15\nR002,2009-01-16,7.7,0.0,31837.76\nR002,2009-01-17,7.3,24.3,0.0\nR002,2009-01-18,1.6,10.6,12564.03\nR002,2009-01-19,1.5,0.4,21959.37\nR002,2009-01-20,2.7,5.5,215.46\nR002,2009-01-21,1.6,0.0,32400.0\nR002,2009-01-22,2.0,0.0,32400.0\nR002,2009-01-23,3.7,0.0,32400.0\nR002,2009-01-24,9.3,0.0,17152.6\nR002,2009-01-25,7.5,0.4,32400.0\nR002,2009-01-26,7.1,12.2,1915.94\nR002,2009-01-27,7.1,4.6,25767.47\nR002,2009-01-28,5.9,1.2,32400.0\nR002,2009-01-29,6.8,0.0,28993.45\nR002,2009-01-30,7.4,4.8,21925.11\nR002,2009-01-31,6.1,1.9,20492.93\nR002,2009-02-01,4.6,12.7,14874.63\nR002,2009-02-02,2.5,2.6,27589.42\nR002,2009-02-03,1.3,0.4,32622.27\nR002,2009-02-04,1.5,0.0,31809.77\nR002,2009-02-05,8.3,0.0,32533.61\nR002,2009-02-06,8.2,4.0,23729.64\nR002,2009-02-07,7.5,3.2,28800.0\nR002,2009-02-08,8.0,0.0,30680.64\nR002,2009-02-09,8.3,8.6,1815.3\nR002,2009-02-10,8.2,16.3,17088.35\nR002,2009-02-11,5.7,3.2,27661.61\nR002,2009-02-12,6.9,0.4,32450.35\nR002,2009-02-13,9.9,2.0,36223.18\nR002,2009-02-14,10.9,0.0,36256.81\nR002,2009-02-15,10.8,0.0,37294.31\nR002,2009-02-16,5.8,16.1,18674.21\nR002,2009-02-17,1.5,19.6,10428.25\nR002,2009-02-18,-2.4,0.0,34396.42\nR002,2009-02-19,-2.0,0.0,37083.66\nR002,2009-02-20,-0.1,0.0,35092.47\nR002,2009-02-21,2.5,0.4,24684.1\nR002,2009-02-22,1.4,2.2,26912.88\nR002,2009-02-23,1.5,1.5,36196.6\nR002,2009-02-24,3.7,0.0,23649.62\nR002,2009-02-25,4.5,9.3,1128.37\nR002,2009-02-26,4.0,0.1,38254.63\nR002,2009-02-27,4.7,0.0,39367.64\nR002,2009-02-28,5.5,9.9,0.0\nR002,2009-03-01,1.6,10.1,28164.87\nR002,2009-03-02,-1.8,13.9,31500.68\nR002,2009-03-03,-4.2,4.4,27534.63\nR002,2009-03-04,-1.1,7.0,1987.48\nR002,2009-03-05,-0.4,0.0,39600.0\nR002,2009-03-06,4.3,2.0,21377.48\nR002,2009-03-07,9.6,0.0,32734.48\nR002,2009-03-08,12.6,5.0,17807.91\nR002,2009-03-09,9.1,16.8,0.0\nR002,2009-03-10,8.9,2.9,34112.42\nR002,2009-03-11,6.7,18.1,8612.69\nR002,2009-03-12,5.8,2.9,37951.31\nR002,2009-03-13,9.8,5.3,21512.75\nR002,2009-03-14,7.2,21.5,1482.94\nR002,2009-03-15,2.1,6.8,35538.02\nR002,2009-03-16,0.9,4.0,37390.5\nR002,2009-03-17,2.9,2.6,39600.0\nR002,2009-03-18,3.3,0.1,32612.88\nR002,2009-03-19,5.7,6.1,27995.13\nR002,2009-03-20,2.9,19.3,25035.84\nR002,2009-03-21,-0.0,1.9,39944.94\nR002,2009-03-22,1.3,0.1,41044.44\nR002,2009-03-23,5.8,6.7,13936.97\nR002,2009-03-24,5.1,9.1,1729.32\nR002,2009-03-25,6.6,0.0,42122.51\nR002,2009-03-26,9.3,0.4,41397.44\nR002,2009-03-27,9.3,4.1,40259.19\nR002,2009-03-28,8.2,3.5,23446.82\nR002,2009-03-29,8.4,0.6,37035.43\nR002,2009-03-30,11.1,0.4,44052.25\nR002,2009-03-31,12.9,0.1,39303.9\nR002,2009-04-01,11.9,1.2,40551.48\nR002,2009-04-02,10.5,7.5,29467.22\nR002,2009-04-03,7.6,0.2,44851.1\nR002,2009-04-04,7.9,0.0,45687.64\nR002,2009-04-05,9.6,0.0,45341.82\nR002,2009-04-06,10.9,0.0,45629.7\nR002,2009-04-07,11.0,2.9,22100.18\nR002,2009-04-08,10.4,11.0,29777.01\nR002,2009-04-09,7.2,3.1,45158.06\nR002,2009-04-10,5.1,3.3,42708.88\nR002,2009-04-11,4.3,0.1,46380.28\nR002,2009-04-12,5.8,0.4,46451.62\nR002,2009-04-13,7.5,0.0,46522.91\nR002,2009-04-14,11.5,0.1,39600.0\nR002,2009-04-15,10.2,4.5,4982.13\nR002,2009-04-16,8.6,10.9,154.77\nR002,2009-04-17,8.7,5.2,31420.87\nR002,2009-04-18,9.1,3.0,25963.57\nR002,2009-04-19,10.0,0.4,44853.42\nR002,2009-04-20,9.7,1.3,46800.0\nR002,2009-04-21,8.9,0.0,46800.0\nR002,2009-04-22,6.5,0.7,44975.32\nR002,2009-04-23,7.6,0.0,46699.19\nR002,2009-04-24,8.6,4.3,38844.88\nR002,2009-04-25,10.2,0.9,44400.86\nR002,2009-04-26,10.4,0.6,44302.98\nR002,2009-04-27,12.2,0.4,47659.82\nR002,2009-04-28,13.2,6.0,39793.56\nR002,2009-04-29,12.9,5.8,40699.09\nR002,2009-04-30,12.6,5.5,37003.14\nR002,2009-05-01,12.6,9.8,45933.12\nR002,2009-05-02,13.9,0.9,44162.83\nR002,2009-05-03,11.9,11.4,25831.98\nR002,2009-05-04,13.6,1.2,44675.71\nR002,2009-05-05,13.4,10.7,18771.44\nR002,2009-05-06,12.2,9.0,11963.36\nR002,2009-05-07,12.3,3.7,38208.07\nR002,2009-05-08,11.1,4.6,46830.43\nR002,2009-05-09,8.8,12.9,30080.03\nR002,2009-05-10,9.3,8.2,30283.09\nR002,2009-05-11,9.6,9.9,26029.72\nR002,2009-05-12,10.9,1.9,46580.1\nR002,2009-05-13,13.2,0.9,49514.31\nR002,2009-05-14,14.0,12.2,17720.1\nR002,2009-05-15,14.5,1.9,42212.05\nR002,2009-05-16,17.1,1.0,49220.76\nR002,2009-05-17,18.0,2.4,49101.15\nR002,2009-05-18,19.1,0.1,50400.0\nR002,2009-05-19,17.0,0.3,50300.65\nR002,2009-05-20,18.3,0.0,47922.58\nR002,2009-05-21,20.1,0.2,50164.31\nR002,2009-05-22,19.4,1.4,39214.83\nR002,2009-05-23,18.9,4.3,46487.49\nR002,2009-05-24,15.1,16.0,34549.56\nR002,2009-05-25,11.6,2.1,48968.14\nR002,2009-05-26,11.9,3.2,46008.98\nR002,2009-05-27,11.2,8.6,33637.64\nR002,2009-05-28,12.0,7.9,36247.88\nR002,2009-05-29,12.1,5.0,43185.19\nR002,2009-05-30,15.1,0.0,51159.42\nR002,2009-05-31,16.5,0.3,49157.83\nR002,2009-06-01,18.1,0.0,49819.39\nR002,2009-06-02,19.8,2.4,51824.94\nR002,2009-06-03,22.2,0.0,48053.34\nR002,2009-06-04,24.1,0.2,48998.89\nR002,2009-06-05,19.0,6.1,36517.66\nR002,2009-06-06,17.6,8.5,48637.28\nR002,2009-06-07,19.0,0.1,50996.24\nR002,2009-06-08,21.0,0.7,48677.34\nR002,2009-06-09,21.4,3.0,47646.29\nR002,2009-06-10,20.4,3.3,45039.43\nR002,2009-06-11,20.4,7.6,41590.33\nR002,2009-06-12,22.4,2.0,53129.27\nR002,2009-06-13,22.6,4.7,51648.84\nR002,2009-06-14,19.8,21.8,31166.71\nR002,2009-06-15,18.1,38.0,32984.79\nR002,2009-06-16,18.1,3.0,48675.19\nR002,2009-06-17,19.1,4.9,49852.71\nR002,2009-06-18,18.5,20.9,22275.39\nR002,2009-06-19,18.9,1.2,43553.48\nR002,2009-06-20,21.2,0.0,50777.74\nR002,2009-06-21,23.2,3.3,51207.62\nR002,2009-06-22,21.7,9.8,52483.07\nR002,2009-06-23,23.1,1.3,53951.83\nR002,2009-06-24,24.6,0.0,53958.07\nR002,2009-06-25,25.2,0.5,53962.35\nR002,2009-06-26,24.4,2.3,52655.61\nR002,2009-06-27,21.9,3.2,51924.88\nR002,2009-06-28,22.6,5.4,48743.19\nR002,2009-06-29,22.9,7.2,53382.11\nR002,2009-06-30,22.6,5.9,52262.48\nR002,2009-07-01,21.6,4.8,52283.57\nR002,2009-07-02,21.8,0.8,53015.37\nR002,2009-07-03,21.0,6.7,50928.85\nR002,2009-07-04,19.8,8.7,47062.73\nR002,2009-07-05,20.0,5.0,51653.4\nR002,2009-07-06,19.6,0.6,50640.05\nR002,2009-07-07,21.9,0.3,52612.56\nR002,2009-07-08,23.4,1.5,53398.98\nR002,2009-07-09,22.8,6.5,53798.92\nR002,2009-07-10,22.1,6.7,46111.96\nR002,2009-07-11,22.3,3.1,51831.36\nR002,2009-07-12,24.6,0.4,53647.34\nR002,2009-07-13,25.5,5.1,52719.31\nR002,2009-07-14,22.7,30.6,34911.66\nR002,2009-07-15,22.4,12.1,34765.64\nR002,2009-07-16,23.0,8.7,37034.21\nR002,2009-07-17,23.8,7.4,51391.61\nR002,2009-07-18,22.9,17.6,35200.6\nR002,2009-07-19,23.3,8.6,48711.69\nR002,2009-07-20,22.6,14.7,42513.84\nR002,2009-07-21,22.9,5.6,44075.08\nR002,2009-07-22,22.9,1.8,46636.1\nR002,2009-07-23,23.7,0.3,46020.3\nR002,2009-07-24,23.3,1.0,51559.24\nR002,2009-07-25,23.9,0.0,51872.75\nR002,2009-07-26,23.7,1.2,50525.48\nR002,2009-07-27,21.8,12.8,44763.44\nR002,2009-07-28,18.7,11.7,27955.98\nR002,2009-07-29,16.3,9.7,33926.97\nR002,2009-07-30,19.4,1.5,40147.79\nR002,2009-07-31,20.3,1.7,43971.37\nR002,2009-08-01,19.9,10.8,35524.47\nR002,2009-08-02,19.5,12.2,33997.97\nR002,2009-08-03,20.8,8.6,45739.89\nR002,2009-08-04,22.3,0.7,46482.45\nR002,2009-08-05,23.4,0.5,47901.87\nR002,2009-08-06,22.9,1.6,47928.9\nR002,2009-08-07,22.4,1.3,40505.17\nR002,2009-08-08,19.5,18.0,41951.63\nR002,2009-08-09,17.7,4.0,43985.25\nR002,2009-08-10,16.2,3.8,45107.88\nR002,2009-08-11,15.3,1.1,46398.19\nR002,2009-08-12,14.9,2.2,47087.5\nR002,2009-08-13,17.4,0.0,47117.89\nR002,2009-08-14,19.4,0.0,46960.42\nR002,2009-08-15,21.2,0.2,44864.44\nR002,2009-08-16,22.3,0.1,46291.15\nR002,2009-08-17,22.9,0.1,45870.62\nR002,2009-08-18,22.5,9.0,42433.55\nR002,2009-08-19,21.8,6.0,44014.67\nR002,2009-08-20,20.7,4.7,44671.34\nR002,2009-08-21,17.0,0.1,45486.77\nR002,2009-08-22,14.6,0.0,46800.0\nR002,2009-08-23,14.2,4.3,30341.27\nR002,2009-08-24,17.0,0.0,46482.8\nR002,2009-08-25,19.7,0.8,46800.0\nR002,2009-08-26,19.0,1.0,44930.78\nR002,2009-08-27,18.5,0.0,45932.31\nR002,2009-08-28,18.0,0.0,45726.4\nR002,2009-08-29,19.4,0.0,46515.38\nR002,2009-08-30,20.2,0.1,46410.57\nR002,2009-08-31,20.2,0.4,44866.37\nR002,2009-09-01,20.2,3.8,45558.14\nR002,2009-09-02,19.3,13.0,25895.12\nR002,2009-09-03,18.5,6.1,40794.39\nR002,2009-09-04,18.8,3.7,41059.95\nR002,2009-09-05,18.7,4.4,42118.0\nR002,2009-09-06,18.6,0.0,42028.19\nR002,2009-09-07,20.0,0.5,43615.62\nR002,2009-09-08,20.9,0.7,43667.17\nR002,2009-09-09,21.7,0.0,41188.24\nR002,2009-09-10,20.3,6.4,29812.32\nR002,2009-09-11,18.9,3.8,37685.6\nR002,2009-09-12,19.0,0.0,41240.63\nR002,2009-09-13,18.7,20.8,14082.14\nR002,2009-09-14,18.9,7.4,5433.56\nR002,2009-09-15,18.3,3.4,35148.33\nR002,2009-09-16,16.7,4.6,33499.34\nR002,2009-09-17,16.1,7.9,29296.56\nR002,2009-09-18,15.8,0.7,39586.95\nR002,2009-09-19,16.3,1.8,36328.83\nR002,2009-09-20,15.4,8.9,2974.18\nR002,2009-09-21,15.8,6.0,28130.76\nR002,2009-09-22,15.6,17.5,12752.15\nR002,2009-09-23,16.3,3.0,34499.11\nR002,2009-09-24,15.4,0.0,41694.38\nR002,2009-09-25,16.0,0.2,39648.82\nR002,2009-09-26,15.1,1.2,38502.78\nR002,2009-09-27,11.0,5.1,35638.52\nR002,2009-09-28,9.9,0.6,14608.17\nR002,2009-09-29,11.3,0.3,0.0\nR002,2009-09-30,13.9,0.0,38772.64\nR002,2009-10-01,11.8,3.7,30669.46\nR002,2009-10-02,14.4,0.0,40018.54\nR002,2009-10-03,18.8,0.0,38596.06\nR002,2009-10-04,20.5,0.0,39555.2\nR002,2009-10-05,19.3,0.0,38678.44\nR002,2009-10-06,16.8,1.5,34628.79\nR002,2009-10-07,16.0,2.6,14588.05\nR002,2009-10-08,16.0,0.9,31832.72\nR002,2009-10-09,16.3,1.1,36383.97\nR002,2009-10-10,15.7,2.9,36439.18\nR002,2009-10-11,16.2,0.0,36819.19\nR002,2009-10-12,18.2,0.0,36874.2\nR002,2009-10-13,19.5,0.0,36276.53\nR002,2009-10-14,20.3,0.0,36329.56\nR002,2009-10-15,18.8,0.0,36054.82\nR002,2009-10-16,18.5,0.0,36106.04\nR002,2009-10-17,19.5,0.0,36000.0\nR002,2009-10-18,19.0,0.0,36000.0\nR002,2009-10-19,19.4,0.0,36000.0\nR002,2009-10-20,19.2,0.0,36000.0\nR002,2009-10-21,19.1,0.0,36000.0\nR002,2009-10-22,16.7,1.0,25982.9\nR002,2009-10-23,14.9,0.4,34246.93\nR002,2009-10-24,16.1,0.0,36000.0\nR002,2009-10-25,15.9,0.0,36000.0\nR002,2009-10-26,14.4,0.0,36000.0\nR002,2009-10-27,14.6,0.0,36000.0\nR002,2009-10-28,14.1,0.0,36000.0\nR002,2009-10-29,11.1,12.4,5785.43\nR002,2009-10-30,10.2,0.0,15274.39\nR002,2009-10-31,10.6,0.2,29306.89\nR002,2009-11-01,9.4,10.8,15651.54\nR002,2009-11-02,8.8,21.6,0.0\nR002,2009-11-03,6.2,17.9,2273.34\nR002,2009-11-04,6.4,3.5,4356.04\nR002,2009-11-05,10.5,21.5,5056.51\nR002,2009-11-06,10.4,0.0,35514.65\nR002,2009-11-07,12.5,0.0,29581.32\nR002,2009-11-08,14.6,0.0,34887.59\nR002,2009-11-09,14.1,0.0,33704.93\nR002,2009-11-10,13.4,0.0,34921.99\nR002,2009-11-11,13.4,0.0,34125.35\nR002,2009-11-12,13.7,0.0,28865.35\nR002,2009-11-13,12.0,1.0,28720.52\nR002,2009-11-14,10.2,19.0,0.0\nR002,2009-11-15,8.5,11.4,8059.52\nR002,2009-11-16,7.6,4.2,30828.9\nR002,2009-11-17,7.6,0.0,14143.51\nR002,2009-11-18,6.7,0.0,32153.27\nR002,2009-11-19,7.3,0.9,16861.2\nR002,2009-11-20,5.7,23.5,18873.9\nR002,2009-11-21,3.8,0.0,30091.81\nR002,2009-11-22,3.7,10.9,0.0\nR002,2009-11-23,5.4,4.8,26921.99\nR002,2009-11-24,6.9,0.0,29635.58\nR002,2009-11-25,7.5,0.2,17238.25\nR002,2009-11-26,4.5,9.7,30976.74\nR002,2009-11-27,2.8,0.0,30947.03\nR002,2009-11-28,5.5,0.0,29516.06\nR002,2009-11-29,6.3,0.0,25880.01\nR002,2009-11-30,6.5,0.0,12688.0\nR002,2009-12-01,5.8,0.0,30325.68\nR002,2009-12-02,5.7,0.0,30722.13\nR002,2009-12-03,7.3,0.0,30737.4\nR002,2009-12-04,7.7,0.0,21600.0\nR002,2009-12-05,9.2,0.0,26953.61\nR002,2009-12-06,7.7,3.2,13108.31\nR002,2009-12-07,7.9,2.8,0.0\nR002,2009-12-08,6.5,3.3,3109.3\nR002,2009-12-09,5.9,8.7,4743.32\nR002,2009-12-10,3.9,0.4,28870.37\nR002,2009-12-11,3.9,0.0,25113.59\nR002,2009-12-12,9.0,1.7,17550.86\nR002,2009-12-13,8.7,15.4,1049.88\nR002,2009-12-14,3.1,24.0,178.11\nR002,2009-12-15,-0.4,0.0,28860.08\nR002,2009-12-16,5.2,0.0,11559.27\nR002,2009-12-17,8.2,0.0,22525.52\nR002,2009-12-18,6.7,3.0,146.89\nR002,2009-12-19,4.5,11.5,4736.03\nR002,2009-12-20,6.4,0.9,28814.88\nR002,2009-12-21,10.0,16.3,0.0\nR002,2009-12-22,3.8,5.5,30742.02\nR002,2009-12-23,5.8,0.1,23362.56\nR002,2009-12-24,5.0,0.9,842.76\nR002,2009-12-25,8.8,0.0,29606.58\nR002,2009-12-26,7.5,9.1,0.0\nR002,2009-12-27,9.0,3.9,29080.62\nR002,2009-12-28,10.1,15.2,2780.39\nR002,2009-12-29,7.4,37.4,0.0\nR002,2009-12-30,2.4,3.6,31509.84\nR002,2009-12-31,2.3,0.0,31560.43\nR002,2010-01-01,7.5,10.6,7200.0\nR002,2010-01-02,9.3,18.7,3311.02\nR002,2010-01-03,5.2,29.5,2031.09\nR002,2010-01-04,2.2,23.1,0.0\nR002,2010-01-05,-2.7,0.1,31834.61\nR002,2010-01-06,-0.7,0.0,28800.0\nR002,2010-01-07,7.2,0.0,31872.43\nR002,2010-01-08,7.1,0.2,22198.55\nR002,2010-01-09,6.0,0.0,26637.56\nR002,2010-01-10,9.0,0.0,30595.35\nR002,2010-01-11,9.3,3.9,21794.6\nR002,2010-01-12,4.5,4.4,29703.56\nR002,2010-01-13,5.3,0.0,22181.71\nR002,2010-01-14,5.6,6.6,878.19\nR002,2010-01-15,6.3,14.0,0.0\nR002,2010-01-16,5.6,12.3,17920.08\nR002,2010-01-17,4.6,0.0,25344.39\nR002,2010-01-18,6.8,0.8,31042.64\nR002,2010-01-19,7.7,2.8,1550.21\nR002,2010-01-20,9.2,1.0,22061.22\nR002,2010-01-21,6.8,14.3,0.0\nR002,2010-01-22,1.3,8.3,26921.1\nR002,2010-01-23,1.1,12.1,0.0\nR002,2010-01-24,3.0,12.2,85.45\nR002,2010-01-25,0.1,9.2,2146.05\nR002,2010-01-26,-3.7,4.9,29864.23\nR002,2010-01-27,-8.4,0.0,32400.0\nR002,2010-01-28,-3.8,3.6,2409.35\nR002,2010-01-29,2.2,3.1,10361.07\nR002,2010-01-30,5.3,12.6,13543.46\nR002,2010-01-31,7.1,0.0,32414.91\nR002,2010-02-01,6.0,9.2,11410.59\nR002,2010-02-02,5.1,0.0,32778.41\nR002,2010-02-03,5.8,0.3,32400.0\nR002,2010-02-04,2.8,8.7,5325.67\nR002,2010-02-05,1.2,2.1,20871.56\nR002,2010-02-06,0.8,4.8,18105.29\nR002,2010-02-07,-0.9,0.0,32249.96\nR002,2010-02-08,1.9,0.0,22525.04\nR002,2010-02-09,4.3,0.7,14034.48\nR002,2010-02-10,5.1,1.0,14784.36\nR002,2010-02-11,5.3,0.7,32960.36\nR002,2010-02-12,8.5,4.0,25917.22\nR002,2010-02-13,9.5,0.3,33881.58\nR002,2010-02-14,8.5,5.9,20304.92\nR002,2010-02-15,12.7,7.1,16087.93\nR002,2010-02-16,6.3,17.6,33595.78\nR002,2010-02-17,5.3,0.8,34230.45\nR002,2010-02-18,7.6,7.5,989.84\nR002,2010-02-19,8.8,12.8,5170.02\nR002,2010-02-20,10.9,1.1,35232.81\nR002,2010-02-21,11.1,18.2,0.0\nR002,2010-02-22,5.7,25.7,13572.76\nR002,2010-02-23,2.1,0.0,32429.42\nR002,2010-02-24,4.5,0.0,38804.16\nR002,2010-02-25,5.8,0.0,38982.76\nR002,2010-02-26,7.4,0.0,39161.85\nR002,2010-02-27,8.5,0.1,38899.35\nR002,2010-02-28,8.5,2.6,36575.27\nR002,2010-03-01,8.7,0.0,37934.94\nR002,2010-03-02,8.5,0.0,22279.43\nR002,2010-03-03,7.7,9.9,2405.96\nR002,2010-03-04,4.1,9.3,35777.13\nR002,2010-03-05,7.2,4.8,28372.34\nR002,2010-03-06,7.1,19.8,9700.14\nR002,2010-03-07,2.4,20.1,31159.67\nR002,2010-03-08,0.8,2.5,21696.47\nR002,2010-03-09,11.2,8.7,0.0\nR002,2010-03-10,6.3,36.7,2883.69\nR002,2010-03-11,6.9,0.5,32995.67\nR002,2010-03-12,9.8,5.1,16241.75\nR002,2010-03-13,6.4,18.6,3666.97\nR002,2010-03-14,4.0,5.9,22286.38\nR002,2010-03-15,4.8,8.3,2816.96\nR002,2010-03-16,1.3,13.1,24139.66\nR002,2010-03-17,-0.5,4.8,33316.01\nR002,2010-03-18,-0.5,3.6,39600.0\nR002,2010-03-19,0.6,14.4,14203.57\nR002,2010-03-20,1.7,0.0,40373.31\nR002,2010-03-21,4.8,0.0,41235.17\nR002,2010-03-22,9.8,1.0,28367.24\nR002,2010-03-23,10.3,0.2,36688.21\nR002,2010-03-24,11.8,0.0,42644.69\nR002,2010-03-25,12.4,0.0,41846.43\nR002,2010-03-26,11.1,0.0,41102.39\nR002,2010-03-27,11.7,0.0,42540.03\nR002,2010-03-28,12.6,0.0,43153.39\nR002,2010-03-29,13.0,3.2,42431.02\nR002,2010-03-30,11.1,6.9,37025.22\nR002,2010-03-31,8.7,4.2,6240.37\nR002,2010-04-01,9.9,0.7,40526.2\nR002,2010-04-02,10.6,1.1,44565.36\nR002,2010-04-03,9.6,0.8,18014.88\nR002,2010-04-04,8.1,9.1,14123.88\nR002,2010-04-05,9.6,0.2,45349.77\nR002,2010-04-06,11.9,0.0,42081.31\nR002,2010-04-07,14.3,2.0,38885.53\nR002,2010-04-08,9.9,12.3,24304.69\nR002,2010-04-09,8.6,3.8,31223.02\nR002,2010-04-10,7.7,6.1,27385.34\nR002,2010-04-11,8.9,0.7,45159.91\nR002,2010-04-12,9.4,5.8,18049.5\nR002,2010-04-13,7.7,12.0,0.0\nR002,2010-04-14,7.3,1.0,46575.63\nR002,2010-04-15,8.8,1.6,19562.35\nR002,2010-04-16,14.0,0.0,46717.06\nR002,2010-04-17,17.1,0.3,46787.89\nR002,2010-04-18,13.7,5.3,23584.49\nR002,2010-04-19,13.2,0.0,45117.46\nR002,2010-04-20,15.9,0.0,46800.0\nR002,2010-04-21,15.1,17.7,12835.86\nR002,2010-04-22,11.6,8.8,28360.32\nR002,2010-04-23,12.1,0.6,45182.0\nR002,2010-04-24,12.4,0.3,42656.12\nR002,2010-04-25,12.9,0.7,45928.23\nR002,2010-04-26,12.7,5.5,35093.41\nR002,2010-04-27,11.5,0.0,39978.67\nR002,2010-04-28,13.1,0.0,47626.95\nR002,2010-04-29,13.5,0.2,48254.56\nR002,2010-04-30,12.9,1.0,43091.04\nR002,2010-05-01,13.1,5.1,46017.47\nR002,2010-05-02,13.1,0.1,47135.21\nR002,2010-05-03,11.8,4.5,16518.02\nR002,2010-05-04,12.0,3.4,28007.98\nR002,2010-05-05,13.2,4.6,44530.39\nR002,2010-05-06,14.8,3.7,48346.29\nR002,2010-05-07,16.0,4.4,47933.07\nR002,2010-05-08,16.6,0.6,49647.15\nR002,2010-05-09,16.5,0.5,46454.32\nR002,2010-05-10,14.8,8.9,20960.72\nR002,2010-05-11,14.6,7.2,44514.91\nR002,2010-05-12,15.4,0.4,47826.44\nR002,2010-05-13,17.5,0.0,48487.76\nR002,2010-05-14,19.9,0.0,49694.4\nR002,2010-05-15,21.4,0.0,50400.0\nR002,2010-05-16,21.8,0.4,43200.0\nR002,2010-05-17,20.9,5.6,40413.15\nR002,2010-05-18,18.9,0.9,44906.51\nR002,2010-05-19,16.3,10.2,41040.75\nR002,2010-05-20,14.4,0.0,50400.0\nR002,2010-05-21,14.2,1.0,43131.85\nR002,2010-05-22,14.9,1.8,41313.51\nR002,2010-05-23,15.6,5.4,46039.45\nR002,2010-05-24,13.8,8.4,37683.74\nR002,2010-05-25,14.7,1.6,44078.98\nR002,2010-05-26,16.8,0.0,51256.0\nR002,2010-05-27,19.7,0.0,51233.27\nR002,2010-05-28,19.6,0.0,51210.76\nR002,2010-05-29,19.9,0.5,51188.54\nR002,2010-05-30,19.7,6.4,51166.62\nR002,2010-05-31,20.4,1.6,50783.39\nR002,2010-06-01,22.3,1.5,51848.4\nR002,2010-06-02,25.0,0.6,45032.35\nR002,2010-06-03,24.7,0.5,47701.99\nR002,2010-06-04,23.7,0.1,50699.5\nR002,2010-06-05,24.3,1.0,49862.22\nR002,2010-06-06,23.9,3.3,49890.65\nR002,2010-06-07,22.5,20.1,37207.01\nR002,2010-06-08,19.1,43.9,6909.91\nR002,2010-06-09,19.8,17.7,30354.0\nR002,2010-06-10,21.3,1.6,52424.08\nR002,2010-06-11,22.8,8.9,51677.47\nR002,2010-06-12,23.4,4.8,53131.54\nR002,2010-06-13,25.0,0.4,53769.51\nR002,2010-06-14,26.4,0.6,52742.16\nR002,2010-06-15,25.9,0.8,49192.15\nR002,2010-06-16,23.2,1.5,38995.23\nR002,2010-06-17,22.5,6.0,47467.19\nR002,2010-06-18,21.6,9.0,40850.31\nR002,2010-06-19,21.9,2.8,45713.2\nR002,2010-06-20,22.9,5.5,52107.87\nR002,2010-06-21,21.9,21.4,38916.56\nR002,2010-06-22,21.0,7.7,52710.12\nR002,2010-06-23,23.1,0.2,53442.74\nR002,2010-06-24,19.8,21.5,18719.68\nR002,2010-06-25,18.9,4.5,31616.2\nR002,2010-06-26,21.3,3.5,48541.59\nR002,2010-06-27,19.5,14.2,39088.68\nR002,2010-06-28,21.0,6.1,51732.31\nR002,2010-06-29,20.2,18.4,46540.14\nR002,2010-06-30,20.7,7.0,45046.77\nR002,2010-07-01,21.5,3.9,53340.55\nR002,2010-07-02,21.6,5.3,53013.76\nR002,2010-07-03,23.4,2.3,51083.93\nR002,2010-07-04,22.8,6.3,45054.5\nR002,2010-07-05,22.7,6.9,42236.41\nR002,2010-07-06,24.0,0.5,48701.0\nR002,2010-07-07,23.9,0.5,51949.92\nR002,2010-07-08,25.9,0.2,53399.31\nR002,2010-07-09,25.5,3.5,52189.77\nR002,2010-07-10,26.2,1.8,48615.21\nR002,2010-07-11,26.4,1.0,52697.9\nR002,2010-07-12,26.0,2.0,50620.85\nR002,2010-07-13,24.7,3.9,52201.58\nR002,2010-07-14,23.0,8.9,45967.76\nR002,2010-07-15,22.3,10.9,41542.83\nR002,2010-07-16,22.9,0.0,53468.91\nR002,2010-07-17,23.6,0.2,53124.28\nR002,2010-07-18,21.6,6.0,13199.71\nR002,2010-07-19,22.4,10.0,45333.35\nR002,2010-07-20,24.4,0.8,52068.78\nR002,2010-07-21,23.9,4.7,51676.67\nR002,2010-07-22,23.2,9.7,50339.9\nR002,2010-07-23,22.3,15.4,46662.18\nR002,2010-07-24,21.5,26.4,29442.97\nR002,2010-07-25,22.8,6.9,45822.83\nR002,2010-07-26,25.1,0.7,51846.7\nR002,2010-07-27,27.3,0.2,51461.52\nR002,2010-07-28,27.6,0.7,51077.94\nR002,2010-07-29,26.0,1.2,49263.94\nR002,2010-07-30,27.0,0.2,49664.67\nR002,2010-07-31,27.4,0.0,50629.86\nR002,2010-08-01,27.7,0.0,50116.92\nR002,2010-08-02,26.6,0.5,48795.58\nR002,2010-08-03,25.3,0.9,48511.67\nR002,2010-08-04,25.2,0.0,49257.15\nR002,2010-08-05,26.3,0.5,48975.32\nR002,2010-08-06,26.8,0.8,48696.49\nR002,2010-08-07,26.6,0.9,48420.02\nR002,2010-08-08,26.2,0.0,48485.01\nR002,2010-08-09,26.5,0.0,48006.0\nR002,2010-08-10,27.0,0.9,47696.13\nR002,2010-08-11,26.3,4.8,47051.17\nR002,2010-08-12,24.9,7.1,47081.16\nR002,2010-08-13,24.8,2.2,47111.15\nR002,2010-08-14,25.1,2.0,47141.09\nR002,2010-08-15,23.5,10.1,46837.37\nR002,2010-08-16,23.1,12.4,45258.39\nR002,2010-08-17,21.3,23.7,38313.9\nR002,2010-08-18,21.1,7.2,38687.94\nR002,2010-08-19,21.4,3.7,46800.0\nR002,2010-08-20,21.4,0.5,45567.81\nR002,2010-08-21,20.3,0.0,46398.65\nR002,2010-08-22,20.3,0.6,43336.29\nR002,2010-08-23,21.2,0.0,46800.0\nR002,2010-08-24,22.8,0.4,46800.0\nR002,2010-08-25,23.6,0.8,46800.0\nR002,2010-08-26,22.3,3.3,46800.0\nR002,2010-08-27,22.2,1.7,44990.5\nR002,2010-08-28,22.6,0.0,45316.57\nR002,2010-08-29,25.2,0.0,46536.94\nR002,2010-08-30,26.0,0.1,46433.77\nR002,2010-08-31,26.3,0.0,46330.44\nR002,2010-09-01,28.4,0.7,46227.05\nR002,2010-09-02,22.1,22.1,30815.71\nR002,2010-09-03,20.8,0.2,43752.28\nR002,2010-09-04,23.0,0.0,43601.43\nR002,2010-09-05,24.1,1.1,43816.2\nR002,2010-09-06,23.5,4.3,39659.6\nR002,2010-09-07,23.1,9.8,42201.06\nR002,2010-09-08,20.8,4.6,39363.98\nR002,2010-09-09,19.9,0.0,43200.0\nR002,2010-09-10,19.6,0.1,39501.7\nR002,2010-09-11,20.2,0.3,43200.0\nR002,2010-09-12,23.0,0.0,43200.0\nR002,2010-09-13,21.3,19.1,20399.55\nR002,2010-09-14,19.1,15.8,0.0\nR002,2010-09-15,18.3,6.4,20946.81\nR002,2010-09-16,19.4,0.0,37912.16\nR002,2010-09-17,20.1,0.0,42913.8\nR002,2010-09-18,20.7,0.2,41478.06\nR002,2010-09-19,20.0,2.1,36005.08\nR002,2010-09-20,20.5,0.0,40272.79\nR002,2010-09-21,22.2,0.0,42216.96\nR002,2010-09-22,20.5,5.8,40963.36\nR002,2010-09-23,18.5,4.8,27004.28\nR002,2010-09-24,19.0,0.0,40411.05\nR002,2010-09-25,21.4,0.0,41720.95\nR002,2010-09-26,22.0,1.2,40162.89\nR002,2010-09-27,21.5,0.4,40517.46\nR002,2010-09-28,21.6,0.5,40234.31\nR002,2010-09-29,19.6,9.7,38411.92\nR002,2010-09-30,18.8,14.1,29312.06\nR002,2010-10-01,13.9,2.6,36924.38\nR002,2010-10-02,13.6,0.9,32601.76\nR002,2010-10-03,13.3,24.8,7.11\nR002,2010-10-04,11.9,1.7,36871.57\nR002,2010-10-05,10.6,1.7,30527.45\nR002,2010-10-06,9.8,1.2,34410.38\nR002,2010-10-07,10.9,0.6,33122.56\nR002,2010-10-08,14.2,4.7,21142.44\nR002,2010-10-09,14.7,11.9,26531.66\nR002,2010-10-10,12.3,20.4,23282.55\nR002,2010-10-11,9.4,14.4,25942.18\nR002,2010-10-12,9.4,11.4,29727.52\nR002,2010-10-13,10.2,2.2,28800.0\nR002,2010-10-14,11.3,0.2,32808.79\nR002,2010-10-15,12.8,23.5,6682.91\nR002,2010-10-16,14.6,14.3,19428.63\nR002,2010-10-17,14.4,4.3,17423.82\nR002,2010-10-18,15.4,0.3,34550.73\nR002,2010-10-19,16.5,0.0,34321.69\nR002,2010-10-20,19.5,5.5,28679.92\nR002,2010-10-21,18.8,3.8,35584.28\nR002,2010-10-22,14.4,2.3,14751.74\nR002,2010-10-23,13.4,1.2,15257.66\nR002,2010-10-24,13.8,0.8,21343.93\nR002,2010-10-25,13.0,0.0,36000.0\nR002,2010-10-26,14.7,0.0,36000.0\nR002,2010-10-27,14.0,0.4,30234.01\nR002,2010-10-28,12.6,17.8,2024.11\nR002,2010-10-29,13.4,8.8,3321.99\nR002,2010-10-30,10.7,18.5,14801.79\nR002,2010-10-31,5.2,0.1,36000.0\nR002,2010-11-01,6.1,0.0,36000.0\nR002,2010-11-02,7.3,0.0,35953.15\nR002,2010-11-03,10.4,0.0,35874.7\nR002,2010-11-04,11.9,0.0,35795.67\nR002,2010-11-05,11.7,0.0,35716.13\nR002,2010-11-06,12.3,0.0,35562.33\nR002,2010-11-07,13.2,0.0,35408.51\nR002,2010-11-08,14.1,0.0,35255.8\nR002,2010-11-09,14.7,0.0,35104.39\nR002,2010-11-10,14.3,0.0,34954.45\nR002,2010-11-11,15.9,0.0,34806.17\nR002,2010-11-12,15.8,0.0,34114.72\nR002,2010-11-13,16.0,0.0,34460.22\nR002,2010-11-14,15.0,0.0,34373.14\nR002,2010-11-15,14.4,0.0,34078.08\nR002,2010-11-16,14.1,0.0,34069.31\nR002,2010-11-17,13.5,0.0,32055.31\nR002,2010-11-18,14.0,0.0,32964.17\nR002,2010-11-19,14.6,0.0,29645.82\nR002,2010-11-20,11.3,0.0,32587.44\nR002,2010-11-21,9.5,0.0,30155.66\nR002,2010-11-22,9.2,4.3,14669.92\nR002,2010-11-23,8.8,0.0,30296.31\nR002,2010-11-24,11.5,8.6,27452.31\nR002,2010-11-25,11.3,28.4,19977.18\nR002,2010-11-26,10.8,21.4,428.23\nR002,2010-11-27,12.0,11.3,5380.64\nR002,2010-11-28,11.5,27.2,0.0\nR002,2010-11-29,11.8,0.4,25500.76\nR002,2010-11-30,13.4,3.0,28800.0\nR002,2010-12-01,12.5,6.0,9515.67\nR002,2010-12-02,13.0,13.4,17500.4\nR002,2010-12-03,13.3,0.9,1534.96\nR002,2010-12-04,14.2,0.0,29219.25\nR002,2010-12-05,17.6,0.0,28846.24\nR002,2010-12-06,11.2,0.0,28181.17\nR002,2010-12-07,8.6,0.0,30014.45\nR002,2010-12-08,9.0,0.0,26791.95\nR002,2010-12-09,9.9,0.0,30027.82\nR002,2010-12-10,10.3,0.0,25062.11\nR002,2010-12-11,11.0,0.2,26821.44\nR002,2010-12-12,8.8,8.9,20687.36\nR002,2010-12-13,4.9,9.6,23644.49\nR002,2010-12-14,4.1,5.9,10171.33\nR002,2010-12-15,6.1,7.6,0.0\nR002,2010-12-16,6.2,3.6,0.0\nR002,2010-12-17,7.7,3.6,25436.76\nR002,2010-12-18,5.3,19.8,0.0\nR002,2010-12-19,3.4,0.2,30389.59\nR002,2010-12-20,6.4,5.5,30378.18\nR002,2010-12-21,8.1,0.0,30365.25\nR002,2010-12-22,9.7,0.0,30742.06\nR002,2010-12-23,8.5,0.0,31117.7\nR002,2010-12-24,7.2,0.0,31243.39\nR002,2010-12-25,6.6,0.0,31280.78\nR002,2010-12-26,10.1,0.0,20720.79\nR002,2010-12-27,13.0,0.0,19113.5\nR002,2010-12-28,12.7,0.0,16232.34\nR002,2010-12-29,11.1,0.0,31385.93\nR002,2010-12-30,7.8,0.8,20084.71\nR002,2010-12-31,5.6,8.8,0.0\nR002,2011-01-01,3.3,0.3,30026.91\nR002,2011-01-02,3.0,0.0,31643.8\nR002,2011-01-03,4.2,0.0,26729.53\nR002,2011-01-04,5.6,2.3,29488.01\nR002,2011-01-05,4.7,7.1,16865.76\nR002,2011-01-06,4.2,4.1,19912.07\nR002,2011-01-07,4.4,0.0,17777.75\nR002,2011-01-08,4.6,8.3,1202.23\nR002,2011-01-09,3.6,2.2,22744.04\nR002,2011-01-10,2.0,0.3,32137.64\nR002,2011-01-11,2.0,0.0,32207.13\nR002,2011-01-12,2.3,0.0,32277.88\nR002,2011-01-13,2.6,0.0,32349.72\nR002,2011-01-14,3.2,0.0,28800.0\nR002,2011-01-15,3.7,0.3,25483.85\nR002,2011-01-16,3.6,0.9,26914.0\nR002,2011-01-17,3.8,8.1,18346.94\nR002,2011-01-18,2.2,8.0,0.0\nR002,2011-01-19,1.8,4.0,24152.56\nR002,2011-01-20,-0.9,0.0,32400.0\nR002,2011-01-21,-0.3,0.0,26247.39\nR002,2011-01-22,0.6,0.0,32400.0\nR002,2011-01-23,4.6,0.0,30657.92\nR002,2011-01-24,5.8,26.5,13205.87\nR002,2011-01-25,7.2,9.9,24720.44\nR002,2011-01-26,5.8,18.9,1302.15\nR002,2011-01-27,2.9,10.4,1177.83\nR002,2011-01-28,0.3,2.3,32400.0\nR002,2011-01-29,-2.8,0.0,32400.0\nR002,2011-01-30,-1.4,0.0,32400.0\nR002,2011-01-31,0.8,8.0,283.25\nR002,2011-02-01,-0.4,4.1,29697.12\nR002,2011-02-02,-3.0,4.0,16135.27\nR002,2011-02-03,-1.8,0.9,32641.35\nR002,2011-02-04,-0.2,6.5,21595.91\nR002,2011-02-05,0.3,3.8,18557.87\nR002,2011-02-06,2.5,34.5,0.0\nR002,2011-02-07,1.2,4.0,32462.99\nR002,2011-02-08,2.5,6.1,1168.15\nR002,2011-02-09,3.8,6.9,24296.65\nR002,2011-02-10,-1.6,2.1,33360.85\nR002,2011-02-11,-4.8,0.1,34676.66\nR002,2011-02-12,-0.3,6.6,1234.41\nR002,2011-02-13,-1.8,25.5,10937.52\nR002,2011-02-14,-4.8,7.3,22169.69\nR002,2011-02-15,-4.4,5.6,35004.67\nR002,2011-02-16,-7.9,0.0,37318.22\nR002,2011-02-17,-6.7,0.0,37432.56\nR002,2011-02-18,-4.1,0.0,36128.56\nR002,2011-02-19,-0.4,0.0,30598.87\nR002,2011-02-20,3.3,2.8,0.0\nR002,2011-02-21,4.2,9.8,175.97\nR002,2011-02-22,3.3,0.0,38415.89\nR002,2011-02-23,5.6,0.1,25250.88\nR002,2011-02-24,7.2,0.0,38194.38\nR002,2011-02-25,7.5,0.4,37225.32\nR002,2011-02-26,6.5,0.5,23129.3\nR002,2011-02-27,3.7,7.5,0.0\nR002,2011-02-28,4.9,0.1,39427.06\nR002,2011-03-01,4.5,2.2,26746.12\nR002,2011-03-02,2.9,9.7,27948.21\nR002,2011-03-03,0.4,3.2,39600.0\nR002,2011-03-04,-0.4,1.6,39600.0\nR002,2011-03-05,-0.2,0.0,36685.4\nR002,2011-03-06,4.0,1.1,39600.0\nR002,2011-03-07,2.5,27.1,0.0\nR002,2011-03-08,2.4,2.8,22396.32\nR002,2011-03-09,5.9,0.0,39600.0\nR002,2011-03-10,5.7,4.8,19650.97\nR002,2011-03-11,4.5,1.6,31253.9\nR002,2011-03-12,2.3,3.6,33647.09\nR002,2011-03-13,3.2,2.3,36000.0\nR002,2011-03-14,5.7,0.0,39600.0\nR002,2011-03-15,8.3,0.0,39638.9\nR002,2011-03-16,10.7,0.0,39600.0\nR002,2011-03-17,9.0,0.0,39750.66\nR002,2011-03-18,8.2,4.1,16327.13\nR002,2011-03-19,7.9,11.6,6392.65\nR002,2011-03-20,8.5,0.0,39600.0\nR002,2011-03-21,7.7,7.3,6634.02\nR002,2011-03-22,5.7,9.9,3077.79\nR002,2011-03-23,5.2,3.9,32776.51\nR002,2011-03-24,3.5,5.4,22425.64\nR002,2011-03-25,4.0,12.0,21036.67\nR002,2011-03-26,6.8,0.0,43079.56\nR002,2011-03-27,8.6,0.6,42051.63\nR002,2011-03-28,9.2,0.0,42221.57\nR002,2011-03-29,8.8,0.0,43617.76\nR002,2011-03-30,8.7,0.5,32446.42\nR002,2011-03-31,10.1,2.0,39665.21\nR002,2011-04-01,7.5,1.0,42507.39\nR002,2011-04-02,9.6,0.0,25548.09\nR002,2011-04-03,10.5,3.4,14638.5\nR002,2011-04-04,9.4,0.4,23022.46\nR002,2011-04-05,7.3,12.8,0.0\nR002,2011-04-06,6.6,5.7,16838.43\nR002,2011-04-07,5.2,11.7,26077.92\nR002,2011-04-08,6.9,15.9,4638.36\nR002,2011-04-09,5.0,28.0,33388.46\nR002,2011-04-10,3.5,9.6,40034.56\nR002,2011-04-11,2.1,11.7,34715.02\nR002,2011-04-12,2.0,0.0,30086.54\nR002,2011-04-13,2.9,0.3,46060.24\nR002,2011-04-14,8.0,0.0,46558.57\nR002,2011-04-15,8.8,15.7,13769.47\nR002,2011-04-16,6.4,5.2,46701.28\nR002,2011-04-17,6.6,3.8,29317.4\nR002,2011-04-18,9.0,0.0,39600.0\nR002,2011-04-19,14.5,1.8,41952.71\nR002,2011-04-20,11.7,9.1,44033.21\nR002,2011-04-21,6.2,20.5,12468.65\nR002,2011-04-22,8.4,0.0,45017.64\nR002,2011-04-23,8.7,1.2,16331.64\nR002,2011-04-24,8.8,3.0,1834.48\nR002,2011-04-25,10.8,1.5,45545.38\nR002,2011-04-26,11.7,3.5,36088.54\nR002,2011-04-27,13.0,0.6,39842.9\nR002,2011-04-28,15.2,1.0,43050.96\nR002,2011-04-29,13.7,5.3,7283.46\nR002,2011-04-30,12.8,5.9,22185.99\nR002,2011-05-01,13.6,0.2,44270.6\nR002,2011-05-02,15.1,0.0,46811.12\nR002,2011-05-03,15.2,1.6,46976.73\nR002,2011-05-04,15.1,2.5,38070.95\nR002,2011-05-05,13.6,15.0,1647.26\nR002,2011-05-06,12.4,7.8,19271.41\nR002,2011-05-07,11.1,6.1,0.0\nR002,2011-05-08,10.2,6.1,11454.15\nR002,2011-05-09,11.5,0.3,44177.46\nR002,2011-05-10,10.7,1.9,47041.77\nR002,2011-05-11,13.2,0.6,46547.2\nR002,2011-05-12,12.7,12.5,20113.93\nR002,2011-05-13,11.9,14.8,2270.51\nR002,2011-05-14,13.1,13.5,12898.18\nR002,2011-05-15,12.7,3.7,23806.61\nR002,2011-05-16,14.1,0.3,47470.67\nR002,2011-05-17,16.1,0.0,50322.87\nR002,2011-05-18,17.8,0.0,49034.73\nR002,2011-05-19,18.2,4.8,37323.81\nR002,2011-05-20,15.5,17.9,29787.33\nR002,2011-05-21,16.0,8.7,46422.88\nR002,2011-05-22,14.6,12.8,1632.1\nR002,2011-05-23,14.6,30.3,3758.79\nR002,2011-05-24,17.1,2.5,34462.41\nR002,2011-05-25,17.8,7.4,43689.05\nR002,2011-05-26,16.7,13.4,39662.07\nR002,2011-05-27,17.9,0.4,45953.48\nR002,2011-05-28,20.0,1.1,47183.8\nR002,2011-05-29,20.2,5.4,28057.87\nR002,2011-05-30,20.9,2.6,50488.09\nR002,2011-05-31,22.6,3.8,51509.16\nR002,2011-06-01,23.0,6.7,51126.16\nR002,2011-06-02,20.5,16.1,19607.76\nR002,2011-06-03,19.9,19.9,10874.45\nR002,2011-06-04,18.4,13.3,17140.79\nR002,2011-06-05,17.9,8.0,45884.88\nR002,2011-06-06,19.2,1.4,49126.49\nR002,2011-06-07,19.4,0.9,50415.84\nR002,2011-06-08,20.6,0.9,50994.81\nR002,2011-06-09,19.9,0.0,51439.02\nR002,2011-06-10,20.6,0.0,53527.81\nR002,2011-06-11,18.4,4.3,36257.31\nR002,2011-06-12,18.3,5.2,35121.07\nR002,2011-06-13,18.6,3.9,42292.52\nR002,2011-06-14,19.4,4.9,46568.8\nR002,2011-06-15,18.8,4.4,42692.5\nR002,2011-06-16,19.8,0.0,51050.45\nR002,2011-06-17,20.8,0.9,52714.78\nR002,2011-06-18,21.0,2.2,52705.25\nR002,2011-06-19,20.0,13.2,51225.36\nR002,2011-06-20,19.5,10.6,41144.85\nR002,2011-06-21,19.5,3.6,48459.53\nR002,2011-06-22,18.5,0.2,52673.45\nR002,2011-06-23,17.3,4.0,51475.2\nR002,2011-06-24,18.5,0.1,53954.99\nR002,2011-06-25,20.7,0.0,53966.85\nR002,2011-06-26,21.2,2.3,51919.19\nR002,2011-06-27,16.7,21.3,5959.43\nR002,2011-06-28,17.1,23.6,31018.17\nR002,2011-06-29,19.7,2.3,44850.74\nR002,2011-06-30,17.9,22.6,18006.19\nR002,2011-07-01,19.2,0.0,53951.77\nR002,2011-07-02,20.3,3.7,47055.46\nR002,2011-07-03,21.1,0.5,49308.49\nR002,2011-07-04,22.3,0.9,52892.78\nR002,2011-07-05,21.3,3.1,47535.91\nR002,2011-07-06,23.2,0.0,53511.93\nR002,2011-07-07,24.9,3.4,52304.01\nR002,2011-07-08,23.8,3.7,38791.47\nR002,2011-07-09,24.4,1.3,41991.59\nR002,2011-07-10,23.8,2.6,49279.21\nR002,2011-07-11,22.8,0.3,49822.55\nR002,2011-07-12,23.3,1.5,52704.39\nR002,2011-07-13,22.6,9.3,47375.47\nR002,2011-07-14,24.4,4.1,52655.7\nR002,2011-07-15,24.5,8.3,51609.5\nR002,2011-07-16,23.2,4.6,39451.75\nR002,2011-07-17,22.9,15.7,36101.71\nR002,2011-07-18,23.6,8.0,52476.86\nR002,2011-07-19,23.2,11.1,45832.42\nR002,2011-07-20,22.7,6.3,24830.86\nR002,2011-07-21,23.3,12.6,52043.24\nR002,2011-07-22,24.1,2.4,52006.78\nR002,2011-07-23,21.9,5.1,47402.35\nR002,2011-07-24,22.1,2.6,48226.23\nR002,2011-07-25,24.5,0.9,51545.06\nR002,2011-07-26,27.7,0.3,51859.13\nR002,2011-07-27,28.5,0.1,51472.59\nR002,2011-07-28,27.3,1.2,50246.3\nR002,2011-07-29,28.5,0.0,49602.77\nR002,2011-07-30,28.8,0.3,49308.17\nR002,2011-07-31,26.5,4.1,43937.93\nR002,2011-08-01,26.4,0.0,45487.23\nR002,2011-08-02,26.0,1.7,45805.2\nR002,2011-08-03,22.9,29.0,23979.2\nR002,2011-08-04,22.4,1.3,47531.91\nR002,2011-08-05,21.6,0.0,47595.81\nR002,2011-08-06,20.3,0.0,48259.03\nR002,2011-08-07,21.1,0.8,45545.5\nR002,2011-08-08,21.8,1.1,47293.6\nR002,2011-08-09,22.5,0.0,46983.61\nR002,2011-08-10,23.3,0.0,48026.21\nR002,2011-08-11,24.3,0.0,47716.57\nR002,2011-08-12,23.6,3.5,45646.48\nR002,2011-08-13,20.7,16.6,7799.68\nR002,2011-08-14,20.6,8.2,45308.76\nR002,2011-08-15,21.7,3.7,45838.18\nR002,2011-08-16,22.2,0.9,45758.69\nR002,2011-08-17,21.4,0.0,45503.34\nR002,2011-08-18,20.8,0.9,43915.59\nR002,2011-08-19,21.4,1.2,45562.6\nR002,2011-08-20,21.3,0.0,45896.35\nR002,2011-08-21,20.7,0.0,46800.0\nR002,2011-08-22,19.3,2.2,45886.31\nR002,2011-08-23,19.7,0.1,43280.84\nR002,2011-08-24,19.6,1.1,46250.37\nR002,2011-08-25,19.1,1.7,46747.21\nR002,2011-08-26,18.5,2.5,43795.17\nR002,2011-08-27,18.6,0.0,46769.13\nR002,2011-08-28,19.5,0.0,46669.07\nR002,2011-08-29,21.5,0.0,46568.61\nR002,2011-08-30,23.8,0.0,46466.81\nR002,2011-08-31,22.9,2.9,45805.72\nR002,2011-09-01,21.4,6.3,42625.22\nR002,2011-09-02,20.6,0.3,43532.52\nR002,2011-09-03,18.9,3.7,33609.27\nR002,2011-09-04,17.9,6.6,38173.39\nR002,2011-09-05,15.8,3.4,42107.38\nR002,2011-09-06,15.9,0.1,42898.88\nR002,2011-09-07,15.8,0.0,42953.14\nR002,2011-09-08,17.2,0.0,43962.91\nR002,2011-09-09,17.6,3.3,36596.99\nR002,2011-09-10,16.0,6.8,41475.87\nR002,2011-09-11,15.0,0.0,40174.14\nR002,2011-09-12,15.9,4.7,34014.68\nR002,2011-09-13,16.6,0.0,43200.0\nR002,2011-09-14,19.2,0.0,43133.33\nR002,2011-09-15,20.3,2.2,42146.99\nR002,2011-09-16,19.8,2.0,31330.97\nR002,2011-09-17,18.4,3.6,36218.84\nR002,2011-09-18,17.8,0.0,40514.42\nR002,2011-09-19,18.3,0.0,41184.64\nR002,2011-09-20,19.4,0.0,42728.41\nR002,2011-09-21,19.1,0.4,38537.09\nR002,2011-09-22,19.1,0.0,39174.49\nR002,2011-09-23,19.6,1.1,38167.47\nR002,2011-09-24,18.1,10.6,18682.18\nR002,2011-09-25,16.8,1.7,37772.01\nR002,2011-09-26,14.8,0.2,40472.66\nR002,2011-09-27,14.2,0.0,40189.43\nR002,2011-09-28,14.8,0.0,40914.95\nR002,2011-09-29,14.1,1.1,38106.64\nR002,2011-09-30,10.9,26.7,25972.38\nR002,2011-10-01,9.5,0.3,39380.03\nR002,2011-10-02,10.4,29.0,1235.67\nR002,2011-10-03,7.4,2.9,38447.85\nR002,2011-10-04,8.8,0.0,36000.0\nR002,2011-10-05,11.3,4.4,33009.79\nR002,2011-10-06,13.6,0.2,36584.45\nR002,2011-10-07,15.1,0.0,38186.29\nR002,2011-10-08,16.4,0.0,37948.48\nR002,2011-10-09,18.6,0.0,32742.29\nR002,2011-10-10,19.6,0.6,840.35\nR002,2011-10-11,22.6,0.0,37112.53\nR002,2011-10-12,21.1,8.9,32545.76\nR002,2011-10-13,14.8,8.5,8125.71\nR002,2011-10-14,13.4,9.1,30969.88\nR002,2011-10-15,11.4,23.6,14750.43\nR002,2011-10-16,12.3,7.4,6311.76\nR002,2011-10-17,13.8,2.6,19365.04\nR002,2011-10-18,14.0,6.9,32417.12\nR002,2011-10-19,8.8,21.7,19261.45\nR002,2011-10-20,8.2,0.0,27731.68\nR002,2011-10-21,9.9,0.0,36000.0\nR002,2011-10-22,10.8,0.0,35068.98\nR002,2011-10-23,11.1,0.1,31117.04\nR002,2011-10-24,11.7,0.0,34284.66\nR002,2011-10-25,10.6,0.0,28226.63\nR002,2011-10-26,9.6,4.3,36000.0\nR002,2011-10-27,8.1,2.7,36000.0\nR002,2011-10-28,9.2,0.3,36000.0\nR002,2011-10-29,8.4,0.0,36000.0\nR002,2011-10-30,7.6,0.5,36000.0\nR002,2011-10-31,7.2,0.0,36000.0\nR002,2011-11-01,7.5,0.0,36000.0\nR002,2011-11-02,7.4,0.1,18593.02\nR002,2011-11-03,8.2,0.0,34495.19\nR002,2011-11-04,7.6,0.0,35813.37\nR002,2011-11-05,8.6,0.0,34522.36\nR002,2011-11-06,6.6,0.0,32816.31\nR002,2011-11-07,2.3,0.0,35438.3\nR002,2011-11-08,3.0,0.0,33846.79\nR002,2011-11-09,4.5,12.7,3101.62\nR002,2011-11-10,1.7,13.6,15434.3\nR002,2011-11-11,0.8,16.4,904.83\nR002,2011-11-12,-2.4,4.4,31286.21\nR002,2011-11-13,-4.1,0.0,33739.74\nR002,2011-11-14,-2.4,0.0,34404.3\nR002,2011-11-15,-0.7,0.0,34267.1\nR002,2011-11-16,2.1,0.3,31215.91\nR002,2011-11-17,3.2,1.8,23862.09\nR002,2011-11-18,2.1,0.0,32235.63\nR002,2011-11-19,1.6,0.0,32228.14\nR002,2011-11-20,1.0,0.0,32222.59\nR002,2011-11-21,1.4,0.0,30875.98\nR002,2011-11-22,1.7,0.0,30204.61\nR002,2011-11-23,2.3,1.2,15430.38\nR002,2011-11-24,-3.0,0.0,31296.12\nR002,2011-11-25,-0.8,0.1,30037.25\nR002,2011-11-26,0.3,4.0,9008.4\nR002,2011-11-27,1.5,3.9,17919.05\nR002,2011-11-28,3.3,2.2,28290.34\nR002,2011-11-29,0.1,13.6,25546.45\nR002,2011-11-30,-1.1,15.7,16440.27\nR002,2011-12-01,-3.2,0.0,25607.79\nR002,2011-12-02,1.0,5.2,0.0\nR002,2011-12-03,1.5,1.1,29588.56\nR002,2011-12-04,2.0,0.0,30365.31\nR002,2011-12-05,4.0,0.0,30378.1\nR002,2011-12-06,6.8,0.0,29238.48\nR002,2011-12-07,9.7,0.0,13390.81\nR002,2011-12-08,7.9,1.0,19633.58\nR002,2011-12-09,5.7,17.3,0.0\nR002,2011-12-10,1.4,1.6,30031.49\nR002,2011-12-11,1.3,0.0,21790.34\nR002,2011-12-12,4.9,7.0,8903.93\nR002,2011-12-13,4.1,0.0,29162.53\nR002,2011-12-14,5.5,0.0,20882.5\nR002,2011-12-15,7.1,0.0,27308.29\nR002,2011-12-16,5.3,0.0,30412.7\nR002,2011-12-17,6.4,0.0,23799.93\nR002,2011-12-18,8.0,0.0,5946.31\nR002,2011-12-19,10.1,0.0,24059.68\nR002,2011-12-20,9.9,0.0,16134.02\nR002,2011-12-21,10.4,8.1,12743.83\nR002,2011-12-22,6.9,5.9,24229.1\nR002,2011-12-23,5.6,0.2,17975.25\nR002,2011-12-24,6.4,5.3,6637.65\nR002,2011-12-25,4.8,3.4,22645.32\nR002,2011-12-26,3.2,0.4,25887.25\nR002,2011-12-27,2.0,1.3,26880.71\nR002,2011-12-28,2.7,2.0,26151.74\nR002,2011-12-29,2.1,0.1,31429.17\nR002,2011-12-30,1.9,0.0,31476.05\nR002,2011-12-31,4.5,0.0,31525.45\nR002,2012-01-01,4.7,0.8,28625.86\nR002,2012-01-02,-0.0,12.5,29964.64\nR002,2012-01-03,-1.3,2.6,14878.38\nR002,2012-01-04,2.8,1.0,18943.92\nR002,2012-01-05,2.1,0.0,30406.62\nR002,2012-01-06,3.8,0.0,22930.44\nR002,2012-01-07,2.6,0.0,29310.15\nR002,2012-01-08,5.7,0.0,2319.22\nR002,2012-01-09,6.1,5.1,25200.0\nR002,2012-01-10,2.7,15.5,22277.8\nR002,2012-01-11,4.4,3.9,187.35\nR002,2012-01-12,4.4,11.4,593.35\nR002,2012-01-13,1.8,2.0,21312.53\nR002,2012-01-14,1.2,6.5,18347.02\nR002,2012-01-15,1.3,24.0,0.0\nR002,2012-01-16,0.6,23.2,0.0\nR002,2012-01-17,-0.7,10.2,0.0\nR002,2012-01-18,-3.2,3.1,32037.62\nR002,2012-01-19,-5.2,1.0,21247.2\nR002,2012-01-20,-6.8,0.0,32400.0\nR002,2012-01-21,-6.7,0.0,32400.0\nR002,2012-01-22,1.1,2.0,25200.0\nR002,2012-01-23,1.7,13.3,3602.67\nR002,2012-01-24,0.2,5.6,12210.84\nR002,2012-01-25,-0.6,0.0,22433.55\nR002,2012-01-26,2.5,0.8,0.0\nR002,2012-01-27,1.0,0.0,14857.72\nR002,2012-01-28,-3.2,0.0,32400.0\nR002,2012-01-29,-5.7,1.8,26895.91\nR002,2012-01-30,-6.2,0.0,32475.36\nR002,2012-01-31,-3.7,9.3,0.0\nR002,2012-02-01,-4.9,12.2,11648.37\nR002,2012-02-02,-7.2,0.4,32695.86\nR002,2012-02-03,-4.2,0.0,31901.26\nR002,2012-02-04,-2.8,0.0,33349.54\nR002,2012-02-05,1.5,0.0,33962.45\nR002,2012-02-06,1.8,0.0,33877.33\nR002,2012-02-07,-0.8,0.0,33021.26\nR002,2012-02-08,-6.1,0.0,34737.79\nR002,2012-02-09,-3.8,1.5,25501.41\nR002,2012-02-10,-4.0,1.4,29126.78\nR002,2012-02-11,-6.3,0.5,31599.83\nR002,2012-02-12,-3.0,0.0,32597.41\nR002,2012-02-13,2.2,0.0,35857.34\nR002,2012-02-14,3.8,0.5,23543.5\nR002,2012-02-15,5.6,1.5,13672.38\nR002,2012-02-16,2.2,6.0,20713.69\nR002,2012-02-17,1.7,0.5,23884.23\nR002,2012-02-18,1.0,7.9,3013.73\nR002,2012-02-19,-0.5,8.5,301.39\nR002,2012-02-20,-1.4,0.9,35751.04\nR002,2012-02-21,0.0,0.0,35841.41\nR002,2012-02-22,-0.3,0.1,32400.0\nR002,2012-02-23,-0.6,2.3,36969.18\nR002,2012-02-24,0.9,1.0,30548.43\nR002,2012-02-25,1.1,13.6,7250.28\nR002,2012-02-26,1.2,12.6,16824.61\nR002,2012-02-27,4.1,1.0,24100.03\nR002,2012-02-28,6.6,7.4,34381.29\nR002,2012-02-29,4.7,9.6,919.48\nR002,2012-03-01,3.5,8.0,3213.51\nR002,2012-03-02,0.6,9.0,3661.63\nR002,2012-03-03,-1.8,11.3,25200.0\nR002,2012-03-04,-0.8,0.9,39600.0\nR002,2012-03-05,-1.2,7.0,32917.29\nR002,2012-03-06,-2.0,11.5,19715.3\nR002,2012-03-07,-3.5,5.1,32805.81\nR002,2012-03-08,-4.2,0.0,39600.0\nR002,2012-03-09,-3.7,0.1,39600.0\nR002,2012-03-10,-5.0,0.7,32400.0\nR002,2012-03-11,-3.4,0.0,39600.0\nR002,2012-03-12,0.7,5.4,8761.16\nR002,2012-03-13,4.5,2.5,15221.13\nR002,2012-03-14,4.3,2.1,30526.92\nR002,2012-03-15,-0.2,9.4,30303.45\nR002,2012-03-16,-3.5,1.6,37809.67\nR002,2012-03-17,-2.9,0.0,40031.43\nR002,2012-03-18,2.4,0.0,39877.6\nR002,2012-03-19,7.4,0.0,41058.19\nR002,2012-03-20,9.1,0.0,41140.77\nR002,2012-03-21,7.3,0.0,39449.27\nR002,2012-03-22,3.4,8.9,10350.67\nR002,2012-03-23,3.4,0.7,39740.7\nR002,2012-03-24,5.1,5.4,14904.2\nR002,2012-03-25,6.5,1.1,40515.98\nR002,2012-03-26,6.5,5.4,38661.25\nR002,2012-03-27,-0.8,4.4,43215.26\nR002,2012-03-28,-0.1,0.0,38198.28\nR002,2012-03-29,1.9,2.2,17465.77\nR002,2012-03-30,6.6,0.0,42697.81\nR002,2012-03-31,4.8,17.5,10699.09\nR002,2012-04-01,6.7,0.0,42439.93\nR002,2012-04-02,5.3,35.1,459.3\nR002,2012-04-03,6.0,0.0,45753.15\nR002,2012-04-04,10.2,0.0,42761.78\nR002,2012-04-05,12.0,0.0,45653.61\nR002,2012-04-06,14.4,0.0,38160.83\nR002,2012-04-07,12.8,3.5,34002.86\nR002,2012-04-08,14.3,0.6,43612.85\nR002,2012-04-09,14.0,1.0,39130.64\nR002,2012-04-10,11.0,5.0,36604.43\nR002,2012-04-11,12.3,4.3,42082.89\nR002,2012-04-12,8.3,17.1,14423.92\nR002,2012-04-13,10.1,0.6,46541.15\nR002,2012-04-14,12.2,0.0,46612.64\nR002,2012-04-15,14.8,0.0,39321.87\nR002,2012-04-16,15.5,2.9,44992.18\nR002,2012-04-17,13.0,5.5,10591.4\nR002,2012-04-18,15.3,0.0,46800.0\nR002,2012-04-19,15.0,5.7,13969.39\nR002,2012-04-20,12.3,0.0,45259.35\nR002,2012-04-21,13.1,0.2,46800.0\nR002,2012-04-22,13.0,5.1,46461.47\nR002,2012-04-23,11.9,1.7,46800.0\nR002,2012-04-24,13.7,0.0,47438.4\nR002,2012-04-25,15.8,0.0,47728.26\nR002,2012-04-26,16.8,0.5,47687.99\nR002,2012-04-27,16.4,3.7,47648.15\nR002,2012-04-28,16.4,1.0,47942.15\nR002,2012-04-29,16.3,1.3,48238.08\nR002,2012-04-30,17.4,0.0,48201.06\nR002,2012-05-01,16.8,0.0,47828.99\nR002,2012-05-02,17.9,0.0,48801.49\nR002,2012-05-03,18.5,0.5,46768.09\nR002,2012-05-04,18.3,2.9,42433.56\nR002,2012-05-05,17.4,0.2,47686.3\nR002,2012-05-06,16.3,3.6,49010.08\nR002,2012-05-07,16.3,8.2,31426.23\nR002,2012-05-08,16.2,9.8,18597.35\nR002,2012-05-09,16.7,8.8,40676.86\nR002,2012-05-10,18.4,3.3,50147.97\nR002,2012-05-11,19.1,2.0,47447.51\nR002,2012-05-12,17.7,7.5,15881.45\nR002,2012-05-13,17.5,8.9,43436.58\nR002,2012-05-14,17.0,5.6,40601.64\nR002,2012-05-15,16.5,5.9,28525.78\nR002,2012-05-16,18.6,0.8,49501.77\nR002,2012-05-17,19.5,3.0,44457.91\nR002,2012-05-18,18.6,6.4,48655.18\nR002,2012-05-19,20.9,0.8,50400.0\nR002,2012-05-20,19.9,1.7,34037.06\nR002,2012-05-21,18.4,11.5,36094.81\nR002,2012-05-22,17.6,4.0,29043.17\nR002,2012-05-23,17.3,13.1,32627.39\nR002,2012-05-24,16.3,6.9,30870.76\nR002,2012-05-25,14.6,1.0,46649.3\nR002,2012-05-26,14.4,6.3,29998.25\nR002,2012-05-27,16.1,4.9,30544.97\nR002,2012-05-28,18.7,2.9,48065.1\nR002,2012-05-29,18.1,6.2,41620.35\nR002,2012-05-30,17.6,6.5,39624.68\nR002,2012-05-31,18.5,8.4,46777.21\nR002,2012-06-01,16.7,8.9,41834.54\nR002,2012-06-02,16.9,0.1,50597.45\nR002,2012-06-03,19.1,0.0,52527.14\nR002,2012-06-04,21.7,0.0,49434.23\nR002,2012-06-05,20.7,16.1,44922.16\nR002,2012-06-06,21.9,0.0,52844.42\nR002,2012-06-07,20.0,2.7,45795.22\nR002,2012-06-08,17.3,16.5,17943.48\nR002,2012-06-09,18.9,1.0,49323.69\nR002,2012-06-10,21.7,0.0,53003.78\nR002,2012-06-11,24.6,0.0,53140.49\nR002,2012-06-12,24.7,1.8,51984.93\nR002,2012-06-13,25.6,0.6,53115.91\nR002,2012-06-14,27.5,0.6,53806.61\nR002,2012-06-15,25.0,4.3,45840.57\nR002,2012-06-16,21.8,7.8,46468.73\nR002,2012-06-17,18.5,1.7,46737.26\nR002,2012-06-18,20.9,0.1,48729.13\nR002,2012-06-19,22.2,0.0,53791.91\nR002,2012-06-20,22.5,0.0,53928.35\nR002,2012-06-21,22.5,0.2,53940.73\nR002,2012-06-22,22.9,0.6,53771.8\nR002,2012-06-23,23.2,1.2,53957.97\nR002,2012-06-24,23.7,0.1,53962.87\nR002,2012-06-25,23.9,3.3,52656.77\nR002,2012-06-26,22.4,7.0,52631.13\nR002,2012-06-27,21.4,11.3,40724.76\nR002,2012-06-28,18.5,9.6,5588.72\nR002,2012-06-29,17.9,3.7,50564.04\nR002,2012-06-30,16.0,15.2,35852.3\nR002,2012-07-01,15.3,11.8,29048.48\nR002,2012-07-02,17.1,1.1,37163.61\nR002,2012-07-03,19.0,5.1,43404.0\nR002,2012-07-04,18.0,9.4,35514.63\nR002,2012-07-05,18.1,16.0,40198.01\nR002,2012-07-06,19.5,2.9,46935.09\nR002,2012-07-07,21.2,3.3,42482.15\nR002,2012-07-08,22.4,1.6,47537.3\nR002,2012-07-09,22.5,0.7,51804.52\nR002,2012-07-10,19.9,10.1,50593.2\nR002,2012-07-11,20.1,1.7,45526.37\nR002,2012-07-12,21.1,8.8,48356.15\nR002,2012-07-13,23.1,0.3,50431.01\nR002,2012-07-14,22.9,1.8,50987.04\nR002,2012-07-15,24.1,0.1,53292.18\nR002,2012-07-16,24.6,0.0,52045.19\nR002,2012-07-17,24.7,0.0,52417.62\nR002,2012-07-18,23.3,0.2,50490.33\nR002,2012-07-19,22.5,0.2,52450.01\nR002,2012-07-20,22.5,0.0,50986.77\nR002,2012-07-21,24.9,0.0,53080.11\nR002,2012-07-22,25.9,0.0,52334.09\nR002,2012-07-23,27.1,1.3,52295.26\nR002,2012-07-24,26.8,0.0,50449.0\nR002,2012-07-25,27.3,0.0,51867.23\nR002,2012-07-26,27.9,0.2,51829.47\nR002,2012-07-27,27.3,0.1,51443.12\nR002,2012-07-28,26.4,0.8,49862.18\nR002,2012-07-29,24.5,4.6,46914.88\nR002,2012-07-30,21.9,8.2,46788.78\nR002,2012-07-31,23.2,1.8,49057.99\nR002,2012-08-01,22.7,14.0,33280.92\nR002,2012-08-02,23.4,6.8,47452.78\nR002,2012-08-03,24.3,4.6,47856.61\nR002,2012-08-04,25.4,3.4,48601.96\nR002,2012-08-05,24.0,1.4,47227.46\nR002,2012-08-06,23.9,7.4,29894.18\nR002,2012-08-07,22.9,7.4,44070.31\nR002,2012-08-08,23.3,7.5,43041.74\nR002,2012-08-09,22.6,7.1,47087.94\nR002,2012-08-10,21.8,8.1,39525.37\nR002,2012-08-11,21.6,8.9,45629.82\nR002,2012-08-12,24.2,1.5,47094.86\nR002,2012-08-13,23.6,4.9,47125.3\nR002,2012-08-14,23.0,9.1,46311.5\nR002,2012-08-15,22.7,7.7,38742.48\nR002,2012-08-16,21.3,25.0,23091.41\nR002,2012-08-17,20.6,0.2,45391.8\nR002,2012-08-18,20.4,0.4,46800.0\nR002,2012-08-19,19.3,8.8,39358.35\nR002,2012-08-20,18.9,8.4,44794.16\nR002,2012-08-21,18.5,0.0,36804.8\nR002,2012-08-22,19.8,6.2,36000.0\nR002,2012-08-23,19.7,0.0,46800.0\nR002,2012-08-24,21.0,0.0,46800.0\nR002,2012-08-25,23.3,0.0,46800.0\nR002,2012-08-26,24.6,0.0,46800.0\nR002,2012-08-27,25.4,0.0,46698.6\nR002,2012-08-28,23.9,2.9,45056.38\nR002,2012-08-29,19.0,13.4,30478.85\nR002,2012-08-30,17.6,0.0,46386.58\nR002,2012-08-31,18.9,0.0,41265.89\nR002,2012-09-01,20.1,0.9,32891.7\nR002,2012-09-02,20.8,0.0,45297.35\nR002,2012-09-03,21.1,0.0,45347.01\nR002,2012-09-04,21.2,0.0,45076.76\nR002,2012-09-05,20.6,0.0,44167.53\nR002,2012-09-06,20.3,0.0,44219.39\nR002,2012-09-07,20.0,0.0,40855.09\nR002,2012-09-08,18.8,0.0,42379.49\nR002,2012-09-09,18.3,0.0,43200.0\nR002,2012-09-10,17.4,0.0,41797.81\nR002,2012-09-11,17.0,0.0,40853.5\nR002,2012-09-12,19.0,0.0,43259.42\nR002,2012-09-13,21.6,0.0,43149.72\nR002,2012-09-14,20.9,2.4,41680.69\nR002,2012-09-15,19.9,0.2,43015.23\nR002,2012-09-16,20.8,0.0,42947.8\nR002,2012-09-17,20.9,0.0,41142.85\nR002,2012-09-18,21.5,0.0,42812.54\nR002,2012-09-19,21.3,0.0,42744.71\nR002,2012-09-20,21.2,1.6,39279.78\nR002,2012-09-21,19.9,3.7,40404.1\nR002,2012-09-22,18.0,3.4,23706.03\nR002,2012-09-23,16.3,18.4,34176.42\nR002,2012-09-24,17.2,0.0,39791.5\nR002,2012-09-25,19.9,0.0,41739.18\nR002,2012-09-26,19.7,0.0,41456.52\nR002,2012-09-27,19.5,0.0,40854.72\nR002,2012-09-28,19.7,0.0,41210.33\nR002,2012-09-29,20.2,0.0,41162.65\nR002,2012-09-30,19.7,0.0,40772.33\nR002,2012-10-01,20.2,0.0,40382.78\nR002,2012-10-02,20.7,0.0,39993.75\nR002,2012-10-03,21.4,0.0,35737.96\nR002,2012-10-04,19.9,2.7,25734.07\nR002,2012-10-05,17.5,11.4,0.0\nR002,2012-10-06,17.5,0.2,33904.9\nR002,2012-10-07,17.7,0.0,37902.98\nR002,2012-10-08,18.1,2.7,34522.49\nR002,2012-10-09,14.1,12.2,24769.02\nR002,2012-10-10,12.2,0.0,36775.99\nR002,2012-10-11,13.5,6.5,16240.69\nR002,2012-10-12,14.3,25.9,25760.95\nR002,2012-10-13,13.4,0.0,36000.0\nR002,2012-10-14,15.5,0.0,36342.83\nR002,2012-10-15,17.3,0.0,36396.03\nR002,2012-10-16,17.4,0.0,36120.87\nR002,2012-10-17,17.4,0.0,36000.0\nR002,2012-10-18,16.4,0.6,36000.0\nR002,2012-10-19,14.3,2.2,32497.92\nR002,2012-10-20,14.9,0.0,36000.0\nR002,2012-10-21,15.7,0.1,36000.0\nR002,2012-10-22,16.0,0.0,36000.0\nR002,2012-10-23,14.8,0.0,36000.0\nR002,2012-10-24,14.2,0.1,31102.23\nR002,2012-10-25,13.4,0.5,36000.0\nR002,2012-10-26,13.5,0.4,34686.15\nR002,2012-10-27,12.1,0.0,36000.0\nR002,2012-10-28,14.2,0.0,36000.0\nR002,2012-10-29,15.5,1.1,31442.68\nR002,2012-10-30,18.3,0.0,30856.23\nR002,2012-10-31,14.7,3.8,33561.65\nR002,2012-11-01,12.6,3.5,8721.11\nR002,2012-11-02,12.4,0.0,30953.06\nR002,2012-11-03,15.6,3.7,32415.95\nR002,2012-11-04,13.2,2.7,34236.24\nR002,2012-11-05,12.7,0.0,35623.3\nR002,2012-11-06,14.6,0.2,35468.39\nR002,2012-11-07,13.5,44.1,1080.72\nR002,2012-11-08,10.9,3.8,13784.4\nR002,2012-11-09,10.2,9.9,0.0\nR002,2012-11-10,12.0,7.8,0.0\nR002,2012-11-11,11.9,7.2,19603.0\nR002,2012-11-12,12.6,0.0,32309.26\nR002,2012-11-13,11.4,0.0,34091.78\nR002,2012-11-14,10.4,0.3,28189.53\nR002,2012-11-15,9.7,1.2,32647.14\nR002,2012-11-16,9.9,0.0,31242.26\nR002,2012-11-17,9.3,0.0,32970.41\nR002,2012-11-18,8.5,0.0,33329.0\nR002,2012-11-19,9.4,0.0,32959.95\nR002,2012-11-20,8.6,0.0,32590.76\nR002,2012-11-21,8.1,0.0,32588.89\nR002,2012-11-22,8.6,0.0,31378.75\nR002,2012-11-23,9.0,0.0,31447.46\nR002,2012-11-24,8.0,0.3,20547.66\nR002,2012-11-25,7.7,0.9,15421.28\nR002,2012-11-26,7.0,0.2,30231.95\nR002,2012-11-27,6.1,0.0,29878.52\nR002,2012-11-28,9.8,0.0,30653.88\nR002,2012-11-29,10.4,0.0,21824.84\nR002,2012-11-30,11.0,0.0,30315.09\nR002,2012-12-01,13.3,0.0,28800.0\nR002,2012-12-02,14.3,0.0,27801.81\nR002,2012-12-03,13.2,6.2,29216.21\nR002,2012-12-04,10.1,19.3,25200.0\nR002,2012-12-05,9.0,5.2,1687.51\nR002,2012-12-06,7.1,3.2,26162.35\nR002,2012-12-07,6.8,2.6,25200.0\nR002,2012-12-08,4.6,7.2,0.0\nR002,2012-12-09,2.5,0.0,30027.62\nR002,2012-12-10,5.8,0.0,30804.21\nR002,2012-12-11,5.7,2.3,28800.0\nR002,2012-12-12,7.4,4.7,27090.14\nR002,2012-12-13,6.5,6.6,0.0\nR002,2012-12-14,2.1,5.6,27044.9\nR002,2012-12-15,-1.2,0.4,28855.33\nR002,2012-12-16,-0.1,0.2,30018.16\nR002,2012-12-17,-1.8,0.0,30791.66\nR002,2012-12-18,-0.3,0.9,2975.37\nR002,2012-12-19,3.3,0.3,14784.43\nR002,2012-12-20,5.6,1.1,10309.13\nR002,2012-12-21,5.7,2.5,19867.31\nR002,2012-12-22,4.8,9.2,14458.83\nR002,2012-12-23,4.0,15.6,8784.22\nR002,2012-12-24,0.7,6.9,20583.32\nR002,2012-12-25,-2.6,0.8,15363.85\nR002,2012-12-26,-1.3,0.0,31333.23\nR002,2012-12-27,6.4,0.0,29076.46\nR002,2012-12-28,10.0,0.0,3533.86\nR002,2012-12-29,6.9,5.1,24879.95\nR002,2012-12-30,1.0,5.8,30752.16\nR002,2012-12-31,-3.2,0.0,31572.73\nR002,2013-01-01,-2.0,0.0,31626.64\nR002,2013-01-02,-0.3,0.0,31682.26\nR002,2013-01-03,1.1,0.0,31739.44\nR002,2013-01-04,2.7,9.4,2451.05\nR002,2013-01-05,3.3,13.2,772.95\nR002,2013-01-06,1.3,2.0,24164.46\nR002,2013-01-07,0.1,6.8,0.0\nR002,2013-01-08,0.5,12.7,10426.63\nR002,2013-01-09,-3.4,0.0,30233.84\nR002,2013-01-10,-5.3,0.0,28898.24\nR002,2013-01-11,-2.3,0.0,27973.69\nR002,2013-01-12,2.5,2.5,78.07\nR002,2013-01-13,4.2,35.6,1185.57\nR002,2013-01-14,-2.4,0.2,32400.0\nR002,2013-01-15,-4.9,0.0,32400.0\nR002,2013-01-16,-0.5,0.0,23285.15\nR002,2013-01-17,4.4,1.3,19511.81\nR002,2013-01-18,6.2,1.8,31052.7\nR002,2013-01-19,7.5,0.6,30964.0\nR002,2013-01-20,4.7,24.7,0.0\nR002,2013-01-21,5.9,13.0,6150.16\nR002,2013-01-22,9.0,0.0,30007.47\nR002,2013-01-23,8.0,10.5,16261.34\nR002,2013-01-24,5.9,12.9,30247.46\nR002,2013-01-25,4.8,3.8,20602.07\nR002,2013-01-26,6.8,8.6,27503.55\nR002,2013-01-27,7.4,0.1,25200.0\nR002,2013-01-28,5.9,2.3,23973.15\nR002,2013-01-29,4.8,0.0,24514.28\nR002,2013-01-30,4.1,7.4,18000.0\nR002,2013-01-31,3.6,3.6,31334.34\nR002,2013-02-01,3.0,8.3,3604.79\nR002,2013-02-02,1.9,6.5,25661.03\nR002,2013-02-03,3.9,0.0,32621.53\nR002,2013-02-04,8.4,0.0,31706.21\nR002,2013-02-05,6.5,7.3,11301.9\nR002,2013-02-06,6.8,0.0,32834.07\nR002,2013-02-07,9.0,0.0,30519.25\nR002,2013-02-08,8.3,0.0,34664.52\nR002,2013-02-09,9.0,0.0,35257.65\nR002,2013-02-10,8.3,0.0,34364.43\nR002,2013-02-11,7.1,0.0,34656.2\nR002,2013-02-12,5.9,4.1,0.0\nR002,2013-02-13,4.9,0.0,35206.17\nR002,2013-02-14,5.2,0.0,32940.72\nR002,2013-02-15,7.8,0.7,34679.18\nR002,2013-02-16,5.4,10.5,16684.46\nR002,2013-02-17,2.1,8.0,31337.67\nR002,2013-02-18,1.1,1.3,34892.73\nR002,2013-02-19,1.4,1.0,33524.42\nR002,2013-02-20,1.9,0.2,24728.29\nR002,2013-02-21,4.2,5.3,24735.44\nR002,2013-02-22,4.3,0.3,26256.54\nR002,2013-02-23,7.0,7.0,527.88\nR002,2013-02-24,6.9,5.9,0.0\nR002,2013-02-25,7.0,0.0,37241.01\nR002,2013-02-26,7.7,0.0,33286.81\nR002,2013-02-27,7.8,0.1,39372.86\nR002,2013-02-28,6.8,2.9,26871.88\nR002,2013-03-01,6.5,5.9,30644.77\nR002,2013-03-02,5.1,2.7,30938.51\nR002,2013-03-03,2.4,11.9,27642.67\nR002,2013-03-04,2.1,0.7,13427.21\nR002,2013-03-05,-0.8,6.2,36878.42\nR002,2013-03-06,-1.8,0.0,39600.0\nR002,2013-03-07,4.4,0.0,39600.0\nR002,2013-03-08,10.1,0.0,39600.0\nR002,2013-03-09,7.3,2.8,3961.38\nR002,2013-03-10,7.9,10.2,25879.88\nR002,2013-03-11,9.1,15.1,23730.09\nR002,2013-03-12,9.8,6.9,18255.71\nR002,2013-03-13,8.9,31.7,15701.23\nR002,2013-03-14,10.1,0.0,39310.76\nR002,2013-03-15,16.7,0.0,37920.2\nR002,2013-03-16,14.8,23.9,3164.92\nR002,2013-03-17,7.8,40.8,0.0\nR002,2013-03-18,2.4,9.5,26118.35\nR002,2013-03-19,5.2,0.0,32015.39\nR002,2013-03-20,9.4,2.2,34956.11\nR002,2013-03-21,9.8,1.1,40259.82\nR002,2013-03-22,14.4,0.0,26735.04\nR002,2013-03-23,8.5,21.0,19721.47\nR002,2013-03-24,0.6,16.9,23351.36\nR002,2013-03-25,2.6,5.9,6429.26\nR002,2013-03-26,6.8,5.8,18587.65\nR002,2013-03-27,8.5,12.2,16683.64\nR002,2013-03-28,8.0,14.3,18518.02\nR002,2013-03-29,4.6,2.0,40065.52\nR002,2013-03-30,6.9,0.0,44691.93\nR002,2013-03-31,14.1,0.0,44763.32\nR002,2013-04-01,16.9,0.0,45153.54\nR002,2013-04-02,15.1,6.3,31706.52\nR002,2013-04-03,14.6,1.2,44612.39\nR002,2013-04-04,17.4,0.0,45691.5\nR002,2013-04-05,15.8,1.7,43743.03\nR002,2013-04-06,15.9,0.0,44349.4\nR002,2013-04-07,16.3,0.0,40633.84\nR002,2013-04-08,18.4,8.0,39163.82\nR002,2013-04-09,11.6,8.2,22813.19\nR002,2013-04-10,9.3,4.4,33240.43\nR002,2013-04-11,8.9,7.3,18104.98\nR002,2013-04-12,9.4,1.4,37375.9\nR002,2013-04-13,10.8,0.9,46382.3\nR002,2013-04-14,12.1,0.3,46594.76\nR002,2013-04-15,13.8,0.8,46665.91\nR002,2013-04-16,13.6,5.9,36647.91\nR002,2013-04-17,9.6,15.7,13590.73\nR002,2013-04-18,9.6,5.6,15637.0\nR002,2013-04-19,11.0,2.0,41298.63\nR002,2013-04-20,11.4,7.8,41465.1\nR002,2013-04-21,10.1,5.9,2786.32\nR002,2013-04-22,9.3,9.8,29821.13\nR002,2013-04-23,6.6,13.8,41939.31\nR002,2013-04-24,6.8,2.7,45897.06\nR002,2013-04-25,10.2,0.6,36801.04\nR002,2013-04-26,11.3,2.9,39425.49\nR002,2013-04-27,14.2,0.0,46800.0\nR002,2013-04-28,18.4,0.0,47951.45\nR002,2013-04-29,20.0,0.0,48580.35\nR002,2013-04-30,19.1,0.0,48208.64\nR002,2013-05-01,20.0,0.0,48842.56\nR002,2013-05-02,20.5,0.0,48807.67\nR002,2013-05-03,19.8,0.0,49110.55\nR002,2013-05-04,20.1,1.6,49077.99\nR002,2013-05-05,17.9,2.2,46613.93\nR002,2013-05-06,17.5,0.3,46941.38\nR002,2013-05-07,16.8,0.7,44966.19\nR002,2013-05-08,17.2,0.8,46800.0\nR002,2013-05-09,17.5,0.1,49269.76\nR002,2013-05-10,17.3,3.2,49570.72\nR002,2013-05-11,18.7,5.5,47831.02\nR002,2013-05-12,16.3,20.0,17338.02\nR002,2013-05-13,16.1,5.8,46229.64\nR002,2013-05-14,17.2,7.5,45266.47\nR002,2013-05-15,16.9,5.4,29754.04\nR002,2013-05-16,16.6,4.4,32011.21\nR002,2013-05-17,17.4,0.3,48151.89\nR002,2013-05-18,16.7,3.0,48411.57\nR002,2013-05-19,16.6,5.3,42901.06\nR002,2013-05-20,18.3,3.1,50543.43\nR002,2013-05-21,19.1,0.9,47767.46\nR002,2013-05-22,19.3,0.4,49106.63\nR002,2013-05-23,20.5,0.0,50963.92\nR002,2013-05-24,23.4,0.3,48928.63\nR002,2013-05-25,19.8,7.3,44450.62\nR002,2013-05-26,17.8,1.8,34487.08\nR002,2013-05-27,18.5,1.4,47514.9\nR002,2013-05-28,17.3,4.3,50400.0\nR002,2013-05-29,17.1,5.8,47271.3\nR002,2013-05-30,19.9,1.0,51522.26\nR002,2013-05-31,21.6,0.0,51864.04\nR002,2013-06-01,19.3,1.4,50400.0\nR002,2013-06-02,18.9,0.0,51462.89\nR002,2013-06-03,17.0,11.7,41481.44\nR002,2013-06-04,15.7,6.5,44961.79\nR002,2013-06-05,17.8,0.4,52499.99\nR002,2013-06-06,18.1,5.5,50659.29\nR002,2013-06-07,15.7,14.4,0.0\nR002,2013-06-08,16.6,13.4,36103.79\nR002,2013-06-09,17.5,4.9,50017.69\nR002,2013-06-10,18.3,6.6,45942.14\nR002,2013-06-11,17.6,10.6,30624.83\nR002,2013-06-12,19.5,2.7,50778.32\nR002,2013-06-13,22.0,1.4,53484.39\nR002,2013-06-14,24.1,0.1,52738.67\nR002,2013-06-15,23.2,6.3,52727.9\nR002,2013-06-16,20.8,9.1,38156.47\nR002,2013-06-17,20.6,16.3,48801.63\nR002,2013-06-18,20.4,4.7,35346.99\nR002,2013-06-19,20.8,0.6,51589.54\nR002,2013-06-20,18.6,0.3,51881.04\nR002,2013-06-21,18.0,0.6,49430.6\nR002,2013-06-22,17.1,3.6,45497.86\nR002,2013-06-23,18.9,0.7,46398.35\nR002,2013-06-24,20.1,0.1,51384.76\nR002,2013-06-25,21.2,1.1,51924.81\nR002,2013-06-26,22.4,0.1,53963.47\nR002,2013-06-27,23.0,1.2,53963.83\nR002,2013-06-28,24.1,0.2,53962.1\nR002,2013-06-29,23.0,1.3,53958.26\nR002,2013-06-30,23.6,0.1,53952.26\nR002,2013-07-01,22.7,15.0,49023.23\nR002,2013-07-02,19.6,3.4,47641.85\nR002,2013-07-03,19.1,15.1,32624.36\nR002,2013-07-04,19.3,16.5,39994.83\nR002,2013-07-05,21.1,0.4,49695.92\nR002,2013-07-06,21.4,0.6,53390.01\nR002,2013-07-07,22.8,0.7,53757.42\nR002,2013-07-08,23.3,1.4,53762.34\nR002,2013-07-09,23.8,0.0,53796.27\nR002,2013-07-10,22.4,1.0,52690.78\nR002,2013-07-11,21.9,1.2,52026.8\nR002,2013-07-12,21.9,3.7,51402.72\nR002,2013-07-13,22.6,1.1,52361.21\nR002,2013-07-14,21.8,2.2,53449.9\nR002,2013-07-15,22.5,1.5,51027.24\nR002,2013-07-16,22.0,4.3,48852.23\nR002,2013-07-17,22.5,3.0,48667.62\nR002,2013-07-18,22.1,4.4,46888.44\nR002,2013-07-19,22.2,1.1,44443.66\nR002,2013-07-20,22.0,0.7,51355.1\nR002,2013-07-21,21.7,1.7,50964.31\nR002,2013-07-22,20.7,0.2,47585.06\nR002,2013-07-23,20.0,0.0,49690.12\nR002,2013-07-24,20.1,0.0,50439.49\nR002,2013-07-25,19.7,0.1,50119.94\nR002,2013-07-26,19.7,0.7,50178.94\nR002,2013-07-27,19.3,1.8,48830.41\nR002,2013-07-28,18.6,11.5,45041.3\nR002,2013-07-29,17.2,5.8,44355.37\nR002,2013-07-30,17.0,5.2,48116.24\nR002,2013-07-31,18.1,0.0,48069.48\nR002,2013-08-01,19.7,0.0,45481.62\nR002,2013-08-02,20.3,1.6,48509.5\nR002,2013-08-03,19.6,0.6,44505.28\nR002,2013-08-04,17.6,8.5,43226.95\nR002,2013-08-05,18.3,8.0,43200.0\nR002,2013-08-06,19.9,1.1,42313.79\nR002,2013-08-07,19.8,3.2,40754.58\nR002,2013-08-08,20.7,1.4,46591.03\nR002,2013-08-09,20.9,0.0,47673.75\nR002,2013-08-10,21.3,0.0,47365.04\nR002,2013-08-11,22.3,0.0,47731.09\nR002,2013-08-12,23.1,0.3,47759.91\nR002,2013-08-13,21.2,8.8,37665.95\nR002,2013-08-14,22.1,2.6,46800.0\nR002,2013-08-15,23.6,0.8,46800.0\nR002,2013-08-16,23.4,2.7,46877.37\nR002,2013-08-17,23.2,2.8,46800.0\nR002,2013-08-18,20.9,7.3,35952.55\nR002,2013-08-19,21.5,0.0,46800.0\nR002,2013-08-20,21.2,0.3,46800.0\nR002,2013-08-21,21.3,0.1,46800.0\nR002,2013-08-22,21.4,0.0,46800.0\nR002,2013-08-23,22.5,0.1,46800.0\nR002,2013-08-24,20.4,6.7,46315.95\nR002,2013-08-25,20.2,5.5,43683.53\nR002,2013-08-26,21.0,0.2,46503.31\nR002,2013-08-27,22.0,0.5,46296.0\nR002,2013-08-28,22.3,0.4,46614.63\nR002,2013-08-29,23.9,0.0,46510.48\nR002,2013-08-30,23.9,0.2,44175.53\nR002,2013-08-31,20.5,16.9,31018.32\nR002,2013-09-01,18.5,24.7,21123.88\nR002,2013-09-02,18.6,0.0,44644.97\nR002,2013-09-03,18.5,0.0,40598.21\nR002,2013-09-04,15.9,14.3,31286.24\nR002,2013-09-05,13.2,7.1,30765.25\nR002,2013-09-06,13.4,7.3,35220.33\nR002,2013-09-07,14.1,2.1,41668.1\nR002,2013-09-08,15.4,0.1,43200.0\nR002,2013-09-09,14.7,5.4,39976.22\nR002,2013-09-10,14.6,0.0,42179.72\nR002,2013-09-11,16.3,0.0,43200.0\nR002,2013-09-12,16.6,0.0,41555.99\nR002,2013-09-13,18.5,0.0,41640.58\nR002,2013-09-14,20.7,1.5,41361.84\nR002,2013-09-15,17.2,21.9,15181.83\nR002,2013-09-16,15.4,8.7,30268.84\nR002,2013-09-17,15.9,2.7,0.0\nR002,2013-09-18,18.1,1.2,34554.41\nR002,2013-09-19,18.7,2.0,36298.91\nR002,2013-09-20,18.7,2.8,36996.63\nR002,2013-09-21,16.9,3.9,23015.52\nR002,2013-09-22,14.9,5.1,37172.56\nR002,2013-09-23,12.6,1.5,36276.02\nR002,2013-09-24,11.4,75.3,225.71\nR002,2013-09-25,9.8,29.8,19427.64\nR002,2013-09-26,10.6,0.0,41128.46\nR002,2013-09-27,15.0,0.0,41483.54\nR002,2013-09-28,11.9,6.3,6764.57\nR002,2013-09-29,8.9,10.5,24884.15\nR002,2013-09-30,10.5,2.1,24849.54\nR002,2013-10-01,15.4,2.1,32420.01\nR002,2013-10-02,16.8,2.6,29305.66\nR002,2013-10-03,11.6,40.3,0.0\nR002,2013-10-04,7.6,3.0,25755.51\nR002,2013-10-05,5.8,4.5,30658.15\nR002,2013-10-06,5.9,2.9,36542.63\nR002,2013-10-07,7.4,0.1,36922.18\nR002,2013-10-08,7.7,0.0,37351.8\nR002,2013-10-09,10.7,0.0,37770.23\nR002,2013-10-10,14.1,0.0,37540.16\nR002,2013-10-11,15.4,0.0,35850.52\nR002,2013-10-12,13.9,0.0,36550.43\nR002,2013-10-13,12.1,0.0,31714.23\nR002,2013-10-14,12.0,0.0,34313.76\nR002,2013-10-15,11.7,0.0,36055.68\nR002,2013-10-16,13.7,0.0,34557.04\nR002,2013-10-17,17.4,0.0,35869.36\nR002,2013-10-18,20.3,0.0,36000.0\nR002,2013-10-19,13.0,33.3,16420.1\nR002,2013-10-20,9.3,2.9,33920.41\nR002,2013-10-21,9.1,0.0,36000.0\nR002,2013-10-22,11.7,0.9,36000.0\nR002,2013-10-23,10.5,1.5,34389.33\nR002,2013-10-24,10.2,0.0,36000.0\nR002,2013-10-25,11.7,0.0,36000.0\nR002,2013-10-26,11.5,0.0,36000.0\nR002,2013-10-27,10.2,0.0,36000.0\nR002,2013-10-28,10.1,0.8,29823.27\nR002,2013-10-29,11.8,0.0,36000.0\nR002,2013-10-30,13.3,0.0,36000.0\nR002,2013-10-31,14.0,0.0,36000.0\nR002,2013-11-01,12.4,0.0,36000.0\nR002,2013-11-02,12.8,0.0,35928.1\nR002,2013-11-03,12.4,0.0,35849.14\nR002,2013-11-04,13.6,0.0,35770.35\nR002,2013-11-05,14.7,0.0,35301.76\nR002,2013-11-06,15.7,0.0,31050.91\nR002,2013-11-07,14.6,0.0,27451.32\nR002,2013-11-08,9.9,14.7,27109.63\nR002,2013-11-09,7.6,0.0,35066.93\nR002,2013-11-10,10.9,0.0,34919.55\nR002,2013-11-11,12.9,0.0,34773.45\nR002,2013-11-12,12.1,0.0,34628.79\nR002,2013-11-13,10.8,0.0,33172.66\nR002,2013-11-14,9.0,4.1,26840.66\nR002,2013-11-15,8.1,0.2,31709.29\nR002,2013-11-16,5.8,0.0,33706.76\nR002,2013-11-17,5.6,0.1,24934.42\nR002,2013-11-18,6.0,0.0,32600.53\nR002,2013-11-19,7.5,0.0,32959.33\nR002,2013-11-20,10.0,0.0,32955.29\nR002,2013-11-21,10.2,0.0,21596.49\nR002,2013-11-22,11.0,0.0,30987.93\nR002,2013-11-23,10.0,0.0,29944.13\nR002,2013-11-24,10.1,0.0,31128.37\nR002,2013-11-25,11.8,0.0,30202.84\nR002,2013-11-26,12.1,4.5,4160.26\nR002,2013-11-27,9.7,39.5,0.0\nR002,2013-11-28,6.9,2.2,26458.43\nR002,2013-11-29,4.4,19.9,0.0\nR002,2013-11-30,1.3,6.4,27584.08\nR002,2013-12-01,0.6,0.0,24317.2\nR002,2013-12-02,5.8,14.3,0.0\nR002,2013-12-03,-2.5,19.5,27530.63\nR002,2013-12-04,-6.7,0.0,29986.53\nR002,2013-12-05,-2.3,0.0,30380.54\nR002,2013-12-06,1.6,1.3,26921.13\nR002,2013-12-07,2.6,4.3,29245.56\nR002,2013-12-08,-0.1,14.9,24641.05\nR002,2013-12-09,-2.8,4.6,29640.82\nR002,2013-12-10,-2.0,29.5,909.0\nR002,2013-12-11,-9.8,0.0,30033.22\nR002,2013-12-12,-10.4,0.0,30810.07\nR002,2013-12-13,-6.5,0.0,30033.73\nR002,2013-12-14,-2.2,6.9,20824.67\nR002,2013-12-15,-1.0,6.9,28373.14\nR002,2013-12-16,-2.6,0.0,30023.47\nR002,2013-12-17,-3.2,0.0,30406.91\nR002,2013-12-18,-1.8,0.0,30789.12\nR002,2013-12-19,0.6,0.0,30388.83\nR002,2013-12-20,2.0,0.0,26684.47\nR002,2013-12-21,3.5,0.0,30363.95\nR002,2013-12-22,2.8,0.0,30349.06\nR002,2013-12-23,1.9,0.0,29941.33\nR002,2013-12-24,2.0,0.0,31249.38\nR002,2013-12-25,0.9,0.0,31288.23\nR002,2013-12-26,2.3,0.0,31329.26\nR002,2013-12-27,5.2,0.0,15665.4\nR002,2013-12-28,3.9,0.0,31417.44\nR002,2013-12-29,3.8,0.0,31464.36\nR002,2013-12-30,4.2,0.0,31513.01\nR002,2013-12-31,4.2,0.0,31563.28\nR002,2014-01-01,3.2,0.0,31615.06\nR002,2014-01-02,2.7,0.0,31668.21\nR002,2014-01-03,1.8,0.0,31722.64\nR002,2014-01-04,1.6,0.0,31778.21\nR002,2014-01-05,2.0,0.0,31835.07\nR002,2014-01-06,2.4,0.0,27924.45\nR002,2014-01-07,4.6,2.1,26871.97\nR002,2014-01-08,2.7,0.8,32019.15\nR002,2014-01-09,2.6,0.0,32084.67\nR002,2014-01-10,4.6,0.0,32152.03\nR002,2014-01-11,4.4,4.8,3588.61\nR002,2014-01-12,2.7,0.4,21973.33\nR002,2014-01-13,1.6,28.7,6717.06\nR002,2014-01-14,-1.4,6.1,28800.0\nR002,2014-01-15,-1.0,0.0,32400.0\nR002,2014-01-16,4.0,0.0,10729.07\nR002,2014-01-17,7.1,6.4,10916.13\nR002,2014-01-18,7.5,5.1,405.73\nR002,2014-01-19,7.0,19.6,0.0\nR002,2014-01-20,6.1,11.9,0.0\nR002,2014-01-21,6.6,8.8,25391.32\nR002,2014-01-22,6.5,10.0,21600.0\nR002,2014-01-23,4.3,24.5,3887.95\nR002,2014-01-24,2.4,20.9,0.0\nR002,2014-01-25,2.5,1.1,31480.76\nR002,2014-01-26,5.9,2.9,20583.3\nR002,2014-01-27,8.2,4.4,20359.8\nR002,2014-01-28,7.8,0.0,24855.17\nR002,2014-01-29,6.7,5.5,29128.84\nR002,2014-01-30,5.7,13.5,22829.45\nR002,2014-01-31,3.2,8.1,5341.33\nR002,2014-02-01,1.8,1.5,28800.0\nR002,2014-02-02,-0.9,0.0,32774.65\nR002,2014-02-03,-2.0,0.0,33041.92\nR002,2014-02-04,-1.5,0.0,32958.15\nR002,2014-02-05,-0.2,0.0,33567.62\nR002,2014-02-06,-0.0,0.0,33826.93\nR002,2014-02-07,1.9,0.0,34428.65\nR002,2014-02-08,3.0,0.0,34681.66\nR002,2014-02-09,4.8,0.0,23164.86\nR002,2014-02-10,7.7,0.0,32132.06\nR002,2014-02-11,7.0,0.7,29449.54\nR002,2014-02-12,7.2,0.0,34313.12\nR002,2014-02-13,9.3,0.0,36259.0\nR002,2014-02-14,10.8,0.0,29784.7\nR002,2014-02-15,9.3,0.0,36283.39\nR002,2014-02-16,7.0,0.0,35109.25\nR002,2014-02-17,6.1,0.1,33621.68\nR002,2014-02-18,6.5,20.0,12322.39\nR002,2014-02-19,4.8,1.1,35512.57\nR002,2014-02-20,7.7,0.0,31833.45\nR002,2014-02-21,7.5,10.4,24785.15\nR002,2014-02-22,6.9,0.0,33680.99\nR002,2014-02-23,8.2,0.0,38633.83\nR002,2014-02-24,9.0,0.0,35768.52\nR002,2014-02-25,10.3,0.0,34788.85\nR002,2014-02-26,7.8,5.9,4040.83\nR002,2014-02-27,5.8,5.7,27683.62\nR002,2014-02-28,5.3,3.3,36545.17\nR002,2014-03-01,6.3,0.0,39557.33\nR002,2014-03-02,8.4,0.2,34089.92\nR002,2014-03-03,10.6,4.8,39514.14\nR002,2014-03-04,11.3,0.2,21843.77\nR002,2014-03-05,9.4,3.5,34250.64\nR002,2014-03-06,8.5,0.0,36696.5\nR002,2014-03-07,9.0,3.3,9020.3\nR002,2014-03-08,8.6,0.7,23435.78\nR002,2014-03-09,11.9,0.7,39600.0\nR002,2014-03-10,11.1,0.0,39600.0\nR002,2014-03-11,8.3,13.7,2288.93\nR002,2014-03-12,4.2,14.2,0.0\nR002,2014-03-13,4.4,1.7,34969.49\nR002,2014-03-14,5.8,0.5,36445.18\nR002,2014-03-15,7.1,0.0,39600.0\nR002,2014-03-16,7.7,7.6,39600.0\nR002,2014-03-17,1.3,28.0,18135.19\nR002,2014-03-18,3.1,12.9,4149.84\nR002,2014-03-19,5.9,0.0,39600.0\nR002,2014-03-20,7.5,6.8,15819.66\nR002,2014-03-21,5.6,2.6,30394.6\nR002,2014-03-22,7.4,0.0,41776.7\nR002,2014-03-23,11.4,0.0,42575.58\nR002,2014-03-24,13.4,0.0,42647.11\nR002,2014-03-25,14.3,0.0,43035.8\nR002,2014-03-26,14.9,0.6,41958.18\nR002,2014-03-27,9.6,5.9,667.56\nR002,2014-03-28,8.1,3.0,41562.11\nR002,2014-03-29,3.6,25.9,63.88\nR002,2014-03-30,-3.1,0.6,43393.02\nR002,2014-03-31,-0.9,0.0,43784.46\nR002,2014-04-01,5.7,0.0,45299.6\nR002,2014-04-02,4.3,4.0,36924.13\nR002,2014-04-03,2.5,2.7,40963.46\nR002,2014-04-04,5.4,0.0,43759.97\nR002,2014-04-05,8.7,5.8,31038.03\nR002,2014-04-06,9.8,0.0,45637.26\nR002,2014-04-07,13.6,0.8,40668.97\nR002,2014-04-08,12.0,9.2,16191.18\nR002,2014-04-09,9.1,3.7,26454.6\nR002,2014-04-10,11.2,0.0,46155.21\nR002,2014-04-11,12.0,0.0,46363.37\nR002,2014-04-12,13.6,2.7,46434.45\nR002,2014-04-13,11.3,5.9,3810.06\nR002,2014-04-14,11.9,0.4,39742.3\nR002,2014-04-15,12.4,4.0,28210.89\nR002,2014-04-16,14.1,2.9,46717.94\nR002,2014-04-17,15.8,0.4,46789.01\nR002,2014-04-18,18.0,0.2,46800.0\nR002,2014-04-19,19.2,1.3,43200.0\nR002,2014-04-20,15.0,9.9,39824.49\nR002,2014-04-21,13.7,2.4,42836.76\nR002,2014-04-22,11.8,9.5,0.0\nR002,2014-04-23,11.4,6.9,666.09\nR002,2014-04-24,12.5,0.7,45281.12\nR002,2014-04-25,15.0,0.5,38017.66\nR002,2014-04-26,13.4,10.6,16933.55\nR002,2014-04-27,14.3,6.6,30693.01\nR002,2014-04-28,17.5,1.0,43953.1\nR002,2014-04-29,14.7,8.8,37547.97\nR002,2014-04-30,14.7,3.0,39919.15\nR002,2014-05-01,14.4,9.0,32400.0\nR002,2014-05-02,13.2,2.0,42234.15\nR002,2014-05-03,14.0,2.5,48444.35\nR002,2014-05-04,15.3,1.5,44951.98\nR002,2014-05-05,17.4,6.3,13655.91\nR002,2014-05-06,15.5,14.5,24504.09\nR002,2014-05-07,14.9,6.4,11447.94\nR002,2014-05-08,17.4,0.6,46800.0\nR002,2014-05-09,19.6,3.5,32266.64\nR002,2014-05-10,19.6,14.7,39145.9\nR002,2014-05-11,19.9,1.5,48414.12\nR002,2014-05-12,18.3,7.4,46606.92\nR002,2014-05-13,16.7,12.6,17803.71\nR002,2014-05-14,17.1,9.1,21801.17\nR002,2014-05-15,20.0,0.8,47697.6\nR002,2014-05-16,22.7,0.9,50400.0\nR002,2014-05-17,23.4,0.8,50400.0\nR002,2014-05-18,19.9,13.0,35907.92\nR002,2014-05-19,16.6,37.8,2198.6\nR002,2014-05-20,16.6,9.5,14872.67\nR002,2014-05-21,16.6,10.1,18712.04\nR002,2014-05-22,17.3,7.2,43477.81\nR002,2014-05-23,18.2,8.3,45662.46\nR002,2014-05-24,19.5,5.9,50400.0\nR002,2014-05-25,17.9,16.5,11444.6\nR002,2014-05-26,16.9,3.8,15778.78\nR002,2014-05-27,17.2,7.7,27478.93\nR002,2014-05-28,18.8,6.6,46304.62\nR002,2014-05-29,19.4,2.6,49903.02\nR002,2014-05-30,21.8,0.0,50400.0\nR002,2014-05-31,24.3,2.3,50400.0\nR002,2014-06-01,21.8,9.7,47540.05\nR002,2014-06-02,20.2,15.2,29580.25\nR002,2014-06-03,17.2,22.2,21272.09\nR002,2014-06-04,19.0,3.5,46541.59\nR002,2014-06-05,22.0,0.9,50679.62\nR002,2014-06-06,19.6,11.1,12329.91\nR002,2014-06-07,18.6,14.8,30241.29\nR002,2014-06-08,19.0,8.2,42937.36\nR002,2014-06-09,19.6,3.0,51263.55\nR002,2014-06-10,19.9,5.0,52424.31\nR002,2014-06-11,17.9,3.2,48077.89\nR002,2014-06-12,15.9,9.0,42082.1\nR002,2014-06-13,15.4,0.6,51358.98\nR002,2014-06-14,17.5,0.0,49329.45\nR002,2014-06-15,18.9,0.2,52732.8\nR002,2014-06-16,21.3,1.3,52722.88\nR002,2014-06-17,21.9,2.5,53816.44\nR002,2014-06-18,21.5,0.0,53881.05\nR002,2014-06-19,20.5,1.0,45488.08\nR002,2014-06-20,23.4,0.7,53914.38\nR002,2014-06-21,19.8,16.5,45252.22\nR002,2014-06-22,18.9,0.0,49843.34\nR002,2014-06-23,18.8,0.0,52747.99\nR002,2014-06-24,19.5,0.5,52222.7\nR002,2014-06-25,20.1,1.4,49702.35\nR002,2014-06-26,21.1,1.0,47878.96\nR002,2014-06-27,21.4,0.0,53968.55\nR002,2014-06-28,21.9,1.8,53658.05\nR002,2014-06-29,21.0,2.0,52724.83\nR002,2014-06-30,21.6,1.6,52267.24\nR002,2014-07-01,23.3,1.7,53012.83\nR002,2014-07-02,23.0,4.0,51917.78\nR002,2014-07-03,22.9,2.8,49179.8\nR002,2014-07-04,22.4,2.2,51195.98\nR002,2014-07-05,22.1,9.6,40261.63\nR002,2014-07-06,20.7,34.1,3119.69\nR002,2014-07-07,21.4,7.4,44337.14\nR002,2014-07-08,20.0,0.6,49060.38\nR002,2014-07-09,19.9,2.3,46745.22\nR002,2014-07-10,21.4,0.5,50495.4\nR002,2014-07-11,24.3,0.0,53741.62\nR002,2014-07-12,27.6,0.2,53708.84\nR002,2014-07-13,25.3,0.9,52718.16\nR002,2014-07-14,22.4,11.0,49881.33\nR002,2014-07-15,23.2,4.6,50735.76\nR002,2014-07-16,25.6,0.0,52370.05\nR002,2014-07-17,25.5,4.5,52767.79\nR002,2014-07-18,23.1,9.5,43977.31\nR002,2014-07-19,22.9,3.9,51255.29\nR002,2014-07-20,21.9,3.8,49159.86\nR002,2014-07-21,21.2,11.0,37699.87\nR002,2014-07-22,21.7,2.1,48976.94\nR002,2014-07-23,23.3,0.1,51790.95\nR002,2014-07-24,25.7,0.0,51918.53\nR002,2014-07-25,23.6,2.3,50468.26\nR002,2014-07-26,24.2,1.0,49465.74\nR002,2014-07-27,23.6,4.5,37081.68\nR002,2014-07-28,21.7,9.8,43179.39\nR002,2014-07-29,21.1,4.4,48226.2\nR002,2014-07-30,21.5,3.3,48627.58\nR002,2014-07-31,22.2,2.1,48338.16\nR002,2014-08-01,22.9,1.2,48740.8\nR002,2014-08-02,24.9,0.0,50174.4\nR002,2014-08-03,25.0,0.4,48171.42\nR002,2014-08-04,23.7,0.6,46168.02\nR002,2014-08-05,22.2,5.8,41400.92\nR002,2014-08-06,20.3,16.6,2202.46\nR002,2014-08-07,20.8,2.9,46932.43\nR002,2014-08-08,22.2,0.0,46962.26\nR002,2014-08-09,22.5,0.2,47668.64\nR002,2014-08-10,22.0,3.5,47359.53\nR002,2014-08-11,22.5,1.4,46504.5\nR002,2014-08-12,22.7,0.7,46800.0\nR002,2014-08-13,22.8,1.2,47111.71\nR002,2014-08-14,25.0,0.0,46800.0\nR002,2014-08-15,27.2,0.0,47171.75\nR002,2014-08-16,26.5,1.2,46869.0\nR002,2014-08-17,26.3,0.0,46800.0\nR002,2014-08-18,25.8,4.7,46931.4\nR002,2014-08-19,24.2,4.9,40353.68\nR002,2014-08-20,23.5,4.0,39742.09\nR002,2014-08-21,23.4,1.5,43869.64\nR002,2014-08-22,24.1,0.2,46800.0\nR002,2014-08-23,23.9,0.2,46800.0\nR002,2014-08-24,23.6,1.0,46800.0\nR002,2014-08-25,24.3,1.4,46215.64\nR002,2014-08-26,23.8,6.2,46285.55\nR002,2014-08-27,25.1,0.0,46529.92\nR002,2014-08-28,25.9,0.1,46636.99\nR002,2014-08-29,23.6,3.9,40419.12\nR002,2014-08-30,22.2,0.0,43299.1\nR002,2014-08-31,21.7,0.3,45177.31\nR002,2014-09-01,22.5,1.3,44902.72\nR002,2014-09-02,23.6,0.0,45914.0\nR002,2014-09-03,25.3,0.3,45319.68\nR002,2014-09-04,23.6,3.8,44203.33\nR002,2014-09-05,22.4,1.1,43817.35\nR002,2014-09-06,23.4,0.2,44508.83\nR002,2014-09-07,23.5,2.2,43601.99\nR002,2014-09-08,22.7,4.9,43335.86\nR002,2014-09-09,19.9,18.0,33381.19\nR002,2014-09-10,20.7,0.6,41567.21\nR002,2014-09-11,18.9,3.4,16568.72\nR002,2014-09-12,20.1,0.7,42457.23\nR002,2014-09-13,19.1,0.0,43181.06\nR002,2014-09-14,18.9,1.0,41984.14\nR002,2014-09-15,19.1,4.1,36256.17\nR002,2014-09-16,17.2,4.7,37178.79\nR002,2014-09-17,18.3,0.6,41020.37\nR002,2014-09-18,16.8,8.2,17113.46\nR002,2014-09-19,17.6,7.4,2704.31\nR002,2014-09-20,16.3,6.9,15499.85\nR002,2014-09-21,15.6,6.4,23252.83\nR002,2014-09-22,15.9,0.0,40023.9\nR002,2014-09-23,17.8,1.0,41969.22\nR002,2014-09-24,13.9,33.2,24269.55\nR002,2014-09-25,10.2,10.5,25238.85\nR002,2014-09-26,11.8,0.0,40163.64\nR002,2014-09-27,14.9,0.0,39880.21\nR002,2014-09-28,18.3,7.8,16062.02\nR002,2014-09-29,13.6,20.2,16700.23\nR002,2014-09-30,13.6,4.3,39468.02\nR002,2014-10-01,13.7,2.0,29692.75\nR002,2014-10-02,13.2,0.3,27813.75\nR002,2014-10-03,12.7,0.0,33134.03\nR002,2014-10-04,13.8,0.3,33783.56\nR002,2014-10-05,13.4,8.3,23938.58\nR002,2014-10-06,13.7,0.0,28441.69\nR002,2014-10-07,13.1,0.0,37880.81\nR002,2014-10-08,13.5,0.0,28418.15\nR002,2014-10-09,13.6,0.0,37670.11\nR002,2014-10-10,14.0,0.0,37840.94\nR002,2014-10-11,15.8,0.0,37283.72\nR002,2014-10-12,16.7,0.0,36859.15\nR002,2014-10-13,16.6,0.2,36260.6\nR002,2014-10-14,16.7,0.0,36313.04\nR002,2014-10-15,16.4,8.6,30157.29\nR002,2014-10-16,15.9,9.8,24674.13\nR002,2014-10-17,13.7,18.2,1383.68\nR002,2014-10-18,12.0,28.8,0.0\nR002,2014-10-19,7.8,7.6,34870.97\nR002,2014-10-20,7.0,0.1,34810.89\nR002,2014-10-21,9.6,0.0,36000.0\nR002,2014-10-22,14.0,1.7,33173.42\nR002,2014-10-23,15.9,0.0,36000.0\nR002,2014-10-24,16.1,0.0,36000.0\nR002,2014-10-25,11.9,17.4,2879.5\nR002,2014-10-26,10.3,0.0,36000.0\nR002,2014-10-27,12.1,0.1,36000.0\nR002,2014-10-28,12.4,0.0,36000.0\nR002,2014-10-29,12.4,0.0,36000.0\nR002,2014-10-30,13.0,0.0,36000.0\nR002,2014-10-31,12.1,1.5,21470.63\nR002,2014-11-01,11.3,4.9,14548.75\nR002,2014-11-02,4.6,4.7,25900.19\nR002,2014-11-03,2.0,0.0,35873.04\nR002,2014-11-04,3.7,1.1,33857.68\nR002,2014-11-05,7.1,0.0,35713.81\nR002,2014-11-06,10.4,0.0,35558.69\nR002,2014-11-07,11.0,0.0,35404.4\nR002,2014-11-08,10.8,0.0,35251.33\nR002,2014-11-09,11.4,0.0,35099.68\nR002,2014-11-10,12.9,0.0,34031.81\nR002,2014-11-11,11.3,0.0,25201.94\nR002,2014-11-12,10.2,0.0,30604.63\nR002,2014-11-13,9.6,0.0,34510.82\nR002,2014-11-14,10.5,0.0,33525.81\nR002,2014-11-15,12.1,0.3,26268.81\nR002,2014-11-16,11.3,0.6,30094.7\nR002,2014-11-17,9.8,0.5,30102.36\nR002,2014-11-18,10.0,0.7,28009.43\nR002,2014-11-19,10.0,0.3,31487.26\nR002,2014-11-20,9.6,0.0,31560.35\nR002,2014-11-21,10.2,0.0,16187.86\nR002,2014-11-22,8.7,0.0,30228.53\nR002,2014-11-23,4.7,0.0,31780.02\nR002,2014-11-24,5.1,0.0,30358.58\nR002,2014-11-25,4.6,0.0,30800.33\nR002,2014-11-26,5.2,0.0,28769.57\nR002,2014-11-27,6.7,0.7,9661.39\nR002,2014-11-28,6.1,5.7,16169.06\nR002,2014-11-29,3.8,0.7,23166.84\nR002,2014-11-30,4.7,25.7,0.0\nR002,2014-12-01,-0.2,0.8,28783.18\nR002,2014-12-02,-2.8,0.0,26092.97\nR002,2014-12-03,-2.3,0.0,30352.84\nR002,2014-12-04,-1.0,0.0,28800.0\nR002,2014-12-05,5.8,2.5,6958.96\nR002,2014-12-06,3.6,0.0,30390.42\nR002,2014-12-07,6.8,0.0,29630.85\nR002,2014-12-08,10.5,0.0,22767.48\nR002,2014-12-09,10.9,0.0,30028.69\nR002,2014-12-10,12.1,0.6,17838.48\nR002,2014-12-11,9.2,8.1,11252.96\nR002,2014-12-12,10.2,0.1,20116.29\nR002,2014-12-13,9.3,3.9,15480.52\nR002,2014-12-14,6.3,0.0,23336.75\nR002,2014-12-15,6.5,0.0,29640.61\nR002,2014-12-16,5.8,0.0,29245.02\nR002,2014-12-17,7.2,0.0,28847.14\nR002,2014-12-18,10.9,0.0,25105.85\nR002,2014-12-19,7.4,3.7,23359.57\nR002,2014-12-20,5.9,0.0,27224.02\nR002,2014-12-21,6.6,0.3,23075.66\nR002,2014-12-22,2.8,19.8,5152.22\nR002,2014-12-23,-0.8,6.2,1363.42\nR002,2014-12-24,5.4,32.2,0.0\nR002,2014-12-25,7.2,1.5,31282.22\nR002,2014-12-26,6.2,0.0,30670.27\nR002,2014-12-27,5.8,0.0,28940.7\nR002,2014-12-28,4.9,20.8,17004.82\nR002,2014-12-29,5.1,6.9,5290.09\nR002,2014-12-30,6.0,37.2,0.0\nR002,2014-12-31,5.1,29.1,0.0\nR002,2015-01-01,4.7,9.0,3967.65\nR002,2015-01-02,5.8,3.0,6533.18\nR002,2015-01-03,3.5,7.8,20785.96\nR002,2015-01-04,0.7,11.8,22440.76\nR002,2015-01-05,1.0,24.1,0.0\nR002,2015-01-06,2.0,16.8,0.0\nR002,2015-01-07,1.9,25.0,0.0\nR002,2015-01-08,-2.6,0.5,23433.1\nR002,2015-01-09,-5.8,0.0,27241.51\nR002,2015-01-10,-2.5,0.4,27302.41\nR002,2015-01-11,-0.1,5.2,4131.0\nR002,2015-01-12,5.5,24.0,0.0\nR002,2015-01-13,4.7,17.3,22202.13\nR002,2015-01-14,5.4,0.0,32400.0\nR002,2015-01-15,5.7,0.0,32400.0\nR002,2015-01-16,3.8,0.0,29781.39\nR002,2015-01-17,2.2,0.0,32400.0\nR002,2015-01-18,2.1,0.0,32400.0\nR002,2015-01-19,3.8,0.0,30109.61\nR002,2015-01-20,4.8,2.1,32400.0\nR002,2015-01-21,7.1,0.0,30058.74\nR002,2015-01-22,6.5,0.0,28541.86\nR002,2015-01-23,4.8,0.0,32400.0\nR002,2015-01-24,5.7,0.0,32400.0\nR002,2015-01-25,9.2,0.1,32400.0\nR002,2015-01-26,6.3,6.6,28577.8\nR002,2015-01-27,4.6,0.0,32400.0\nR002,2015-01-28,5.5,0.0,25513.12\nR002,2015-01-29,4.7,6.4,22945.22\nR002,2015-01-30,6.4,0.0,28956.37\nR002,2015-01-31,7.0,0.0,27798.05\nR002,2015-02-01,9.1,0.2,13938.4\nR002,2015-02-02,11.2,0.0,32685.24\nR002,2015-02-03,9.7,6.5,0.0\nR002,2015-02-04,4.8,2.3,32595.83\nR002,2015-02-05,2.4,4.5,22011.24\nR002,2015-02-06,6.5,0.0,32096.94\nR002,2015-02-07,6.5,3.6,12588.47\nR002,2015-02-08,3.0,5.6,33125.88\nR002,2015-02-09,3.3,7.8,8946.71\nR002,2015-02-10,3.5,0.4,31599.29\nR002,2015-02-11,6.7,0.6,30380.67\nR002,2015-02-12,5.4,6.1,20776.09\nR002,2015-02-13,5.2,2.3,19720.95\nR002,2015-02-14,4.0,1.2,29094.42\nR002,2015-02-15,2.7,0.1,34734.79\nR002,2015-02-16,1.7,8.6,30017.2\nR002,2015-02-17,0.3,10.9,16588.24\nR002,2015-02-18,0.3,0.0,36702.43\nR002,2015-02-19,1.7,0.3,37895.06\nR002,2015-02-20,1.1,3.3,24334.97\nR002,2015-02-21,0.5,0.0,35787.22\nR002,2015-02-22,3.3,0.0,38422.28\nR002,2015-02-23,5.6,0.0,38598.83\nR002,2015-02-24,6.8,0.0,35438.2\nR002,2015-02-25,8.2,0.0,38952.27\nR002,2015-02-26,7.4,0.0,39128.89\nR002,2015-02-27,8.7,0.0,38558.46\nR002,2015-02-28,8.9,0.3,25395.32\nR002,2015-03-01,7.0,8.8,1361.31\nR002,2015-03-02,5.8,2.0,38402.86\nR002,2015-03-03,6.2,0.0,39600.0\nR002,2015-03-04,3.3,6.9,14546.1\nR002,2015-03-05,1.7,3.3,38041.91\nR002,2015-03-06,2.7,0.0,36038.06\nR002,2015-03-07,7.1,5.6,20340.29\nR002,2015-03-08,6.3,0.0,37130.33\nR002,2015-03-09,7.0,0.0,39600.0\nR002,2015-03-10,10.4,0.0,39600.0\nR002,2015-03-11,14.0,0.0,39600.0\nR002,2015-03-12,13.2,3.9,13864.47\nR002,2015-03-13,10.5,7.9,31922.36\nR002,2015-03-14,11.0,0.0,36190.23\nR002,2015-03-15,10.1,7.2,36612.95\nR002,2015-03-16,8.6,9.3,13431.79\nR002,2015-03-17,5.5,9.7,30075.13\nR002,2015-03-18,6.5,0.6,36837.16\nR002,2015-03-19,8.3,0.3,36758.26\nR002,2015-03-20,7.7,7.5,14617.47\nR002,2015-03-21,6.6,5.5,19706.15\nR002,2015-03-22,8.3,1.5,37662.22\nR002,2015-03-23,5.6,0.0,33830.28\nR002,2015-03-24,4.9,0.9,39831.08\nR002,2015-03-25,8.5,0.0,40435.51\nR002,2015-03-26,12.2,0.0,43082.73\nR002,2015-03-27,14.5,0.0,40170.11\nR002,2015-03-28,15.4,2.2,5286.39\nR002,2015-03-29,14.2,10.5,37218.55\nR002,2015-03-30,11.2,6.2,10958.79\nR002,2015-03-31,10.8,0.3,37911.5\nR002,2015-04-01,9.8,4.5,21550.01\nR002,2015-04-02,9.2,18.1,29948.4\nR002,2015-04-03,4.6,4.3,43642.1\nR002,2015-04-04,3.3,12.9,4831.81\nR002,2015-04-05,3.6,0.1,41507.5\nR002,2015-04-06,9.4,0.0,42929.53\nR002,2015-04-07,12.0,8.5,30606.42\nR002,2015-04-08,6.8,17.0,19038.05\nR002,2015-04-09,7.4,23.0,2034.79\nR002,2015-04-10,5.9,28.0,4337.27\nR002,2015-04-11,7.1,0.0,38024.08\nR002,2015-04-12,7.3,3.3,2568.5\nR002,2015-04-13,8.4,0.6,38140.13\nR002,2015-04-14,8.6,2.5,46558.38\nR002,2015-04-15,4.6,18.6,12948.0\nR002,2015-04-16,7.1,0.2,43057.8\nR002,2015-04-17,12.3,0.0,46772.87\nR002,2015-04-18,13.8,3.5,35476.72\nR002,2015-04-19,11.0,31.6,17773.14\nR002,2015-04-20,4.1,4.4,46800.0\nR002,2015-04-21,4.9,0.7,38358.97\nR002,2015-04-22,5.2,5.8,44520.96\nR002,2015-04-23,6.9,0.0,46528.38\nR002,2015-04-24,7.4,0.3,47139.28\nR002,2015-04-25,10.6,0.0,46800.0\nR002,2015-04-26,13.9,0.0,44845.8\nR002,2015-04-27,13.8,3.3,39310.22\nR002,2015-04-28,13.9,2.4,47301.65\nR002,2015-04-29,16.4,0.6,43581.03\nR002,2015-04-30,14.9,4.0,44937.44\nR002,2015-05-01,13.9,3.6,44938.66\nR002,2015-05-02,13.2,0.3,46662.91\nR002,2015-05-03,14.8,0.0,43993.18\nR002,2015-05-04,12.8,6.7,8134.7\nR002,2015-05-05,11.6,8.6,43200.0\nR002,2015-05-06,10.3,2.4,36633.57\nR002,2015-05-07,10.9,3.9,20582.74\nR002,2015-05-08,12.5,2.2,38338.51\nR002,2015-05-09,12.7,2.9,37889.91\nR002,2015-05-10,12.1,0.7,45159.03\nR002,2015-05-11,12.8,1.3,45307.99\nR002,2015-05-12,13.2,2.9,42390.02\nR002,2015-05-13,12.9,6.2,40222.06\nR002,2015-05-14,12.7,7.7,43076.26\nR002,2015-05-15,14.3,0.5,46482.93\nR002,2015-05-16,14.2,10.3,17311.29\nR002,2015-05-17,13.7,8.6,36684.57\nR002,2015-05-18,13.9,0.0,50404.12\nR002,2015-05-19,15.8,0.0,50400.0\nR002,2015-05-20,18.5,0.0,50702.33\nR002,2015-05-21,21.3,0.0,50675.6\nR002,2015-05-22,21.5,5.4,50400.0\nR002,2015-05-23,19.5,7.1,38690.24\nR002,2015-05-24,17.7,17.6,37039.26\nR002,2015-05-25,19.2,3.3,46327.65\nR002,2015-05-26,20.2,3.7,49895.05\nR002,2015-05-27,20.4,1.0,50879.21\nR002,2015-05-28,18.9,7.0,48566.1\nR002,2015-05-29,17.7,3.7,48744.03\nR002,2015-05-30,18.2,9.9,43109.76\nR002,2015-05-31,16.5,15.7,32896.25\nR002,2015-06-01,17.8,5.5,51186.67\nR002,2015-06-02,19.9,0.0,50032.81\nR002,2015-06-03,20.4,4.6,51085.56\nR002,2015-06-04,20.1,10.9,48343.83\nR002,2015-06-05,21.1,2.5,47863.45\nR002,2015-06-06,22.0,3.9,39755.81\nR002,2015-06-07,21.3,0.0,43219.04\nR002,2015-06-08,22.3,0.0,52823.91\nR002,2015-06-09,21.2,8.0,48111.18\nR002,2015-06-10,21.5,2.8,52796.99\nR002,2015-06-11,19.8,6.6,52051.07\nR002,2015-06-12,18.6,10.5,48535.04\nR002,2015-06-13,19.5,3.2,49666.67\nR002,2015-06-14,20.0,7.0,50701.2\nR002,2015-06-15,18.7,19.2,41654.55\nR002,2015-06-16,18.5,16.8,14203.42\nR002,2015-06-17,19.0,9.3,38590.34\nR002,2015-06-18,20.6,0.1,52358.85\nR002,2015-06-19,21.5,4.6,53432.03\nR002,2015-06-20,21.3,9.8,46334.68\nR002,2015-06-21,20.9,8.2,49819.04\nR002,2015-06-22,21.8,7.4,51679.46\nR002,2015-06-23,22.3,11.2,51033.55\nR002,2015-06-24,22.8,16.2,47243.9\nR002,2015-06-25,21.0,16.7,32236.9\nR002,2015-06-26,20.8,27.2,17784.17\nR002,2015-06-27,20.1,4.2,43926.9\nR002,2015-06-28,19.8,8.2,48897.78\nR002,2015-06-29,20.1,16.3,46901.7\nR002,2015-06-30,20.4,3.0,49504.36\nR002,2015-07-01,21.1,0.0,50100.59\nR002,2015-07-02,22.3,1.0,49231.16\nR002,2015-07-03,20.9,10.8,31303.43\nR002,2015-07-04,23.0,1.5,45766.85\nR002,2015-07-05,24.0,1.9,41643.18\nR002,2015-07-06,22.2,11.6,7765.58\nR002,2015-07-07,22.6,1.7,50763.9\nR002,2015-07-08,23.1,0.8,53290.03\nR002,2015-07-09,25.2,1.3,53042.12\nR002,2015-07-10,23.4,25.4,26996.67\nR002,2015-07-11,21.5,5.1,47106.52\nR002,2015-07-12,20.4,0.0,51893.93\nR002,2015-07-13,19.7,0.5,50335.52\nR002,2015-07-14,20.9,0.1,52725.35\nR002,2015-07-15,20.1,5.8,39865.55\nR002,2015-07-16,20.8,0.0,50825.81\nR002,2015-07-17,20.8,0.0,49954.78\nR002,2015-07-18,19.3,0.0,52777.34\nR002,2015-07-19,20.4,0.0,53438.46\nR002,2015-07-20,21.1,0.0,52248.6\nR002,2015-07-21,23.0,0.0,52751.65\nR002,2015-07-22,22.2,1.9,49975.1\nR002,2015-07-23,22.0,0.0,47444.26\nR002,2015-07-24,25.0,0.0,52284.56\nR002,2015-07-25,27.0,0.7,51544.5\nR002,2015-07-26,26.7,0.7,48437.48\nR002,2015-07-27,24.1,8.0,49152.62\nR002,2015-07-28,23.7,3.5,47811.6\nR002,2015-07-29,24.8,0.0,50298.28\nR002,2015-07-30,26.0,0.0,48964.61\nR002,2015-07-31,28.8,0.0,50636.12\nR002,2015-08-01,28.0,1.7,50257.22\nR002,2015-08-02,26.6,3.5,48098.32\nR002,2015-08-03,26.0,7.8,44087.27\nR002,2015-08-04,26.0,3.7,48219.1\nR002,2015-08-05,25.4,2.7,48622.23\nR002,2015-08-06,24.7,0.2,46978.73\nR002,2015-08-07,24.6,0.0,48625.24\nR002,2015-08-08,24.4,0.0,47789.66\nR002,2015-08-09,23.9,0.0,47321.93\nR002,2015-08-10,25.5,0.0,48026.07\nR002,2015-08-11,26.8,0.0,47716.55\nR002,2015-08-12,26.6,1.5,47408.77\nR002,2015-08-13,24.8,3.8,46509.23\nR002,2015-08-14,25.9,1.2,46560.35\nR002,2015-08-15,26.0,0.8,46829.43\nR002,2015-08-16,23.9,5.8,46728.67\nR002,2015-08-17,24.2,0.0,46800.0\nR002,2015-08-18,23.2,0.0,46923.5\nR002,2015-08-19,24.2,0.1,46800.0\nR002,2015-08-20,24.5,0.6,46800.0\nR002,2015-08-21,24.6,0.0,46800.0\nR002,2015-08-22,24.2,0.8,46800.0\nR002,2015-08-23,22.8,4.6,45177.75\nR002,2015-08-24,20.1,13.3,19635.87\nR002,2015-08-25,19.5,2.4,35377.37\nR002,2015-08-26,19.2,10.9,38203.35\nR002,2015-08-27,20.1,0.0,46768.15\nR002,2015-08-28,20.2,0.6,42749.5\nR002,2015-08-29,20.6,0.1,46567.29\nR002,2015-08-30,21.0,0.0,46464.85\nR002,2015-08-31,20.6,0.8,45484.69\nR002,2015-09-01,20.9,0.0,46256.47\nR002,2015-09-02,22.7,0.0,46150.75\nR002,2015-09-03,23.7,0.0,45628.99\nR002,2015-09-04,24.4,0.0,45357.79\nR002,2015-09-05,24.7,0.4,44447.34\nR002,2015-09-06,24.4,1.2,43859.23\nR002,2015-09-07,24.7,0.3,43911.88\nR002,2015-09-08,22.5,1.5,38198.87\nR002,2015-09-09,21.4,5.3,33290.08\nR002,2015-09-10,22.5,0.4,38209.37\nR002,2015-09-11,23.6,1.3,42852.46\nR002,2015-09-12,22.5,0.3,41516.12\nR002,2015-09-13,22.0,4.1,29098.71\nR002,2015-09-14,20.5,6.3,40751.81\nR002,2015-09-15,20.7,1.2,39600.0\nR002,2015-09-16,20.3,0.3,38785.59\nR002,2015-09-17,19.9,0.0,42073.97\nR002,2015-09-18,21.1,0.0,42861.06\nR002,2015-09-19,22.6,0.0,42793.46\nR002,2015-09-20,24.5,0.0,42725.7\nR002,2015-09-21,25.2,0.3,42209.11\nR002,2015-09-22,24.5,0.1,40970.4\nR002,2015-09-23,23.8,0.5,40368.34\nR002,2015-09-24,23.0,2.1,40757.9\nR002,2015-09-25,22.4,0.7,41713.0\nR002,2015-09-26,21.5,0.2,41430.19\nR002,2015-09-27,20.5,1.8,40605.24\nR002,2015-09-28,20.0,2.5,39907.1\nR002,2015-09-29,19.7,0.0,40897.32\nR002,2015-09-30,20.0,1.3,36262.93\nR002,2015-10-01,18.0,7.5,11918.83\nR002,2015-10-02,18.8,0.6,28661.26\nR002,2015-10-03,18.0,1.2,34523.94\nR002,2015-10-04,16.2,3.5,38069.12\nR002,2015-10-05,16.0,1.8,37025.32\nR002,2015-10-06,15.9,0.8,38453.3\nR002,2015-10-07,15.4,0.2,29518.59\nR002,2015-10-08,12.8,0.0,36238.86\nR002,2015-10-09,10.8,0.7,36355.58\nR002,2015-10-10,9.7,0.1,36232.57\nR002,2015-10-11,11.2,1.8,34228.1\nR002,2015-10-12,13.5,55.4,180.32\nR002,2015-10-13,15.1,11.5,25155.25\nR002,2015-10-14,14.3,10.8,5609.34\nR002,2015-10-15,13.4,3.6,30243.97\nR002,2015-10-16,11.8,0.3,35264.07\nR002,2015-10-17,11.5,0.0,36132.49\nR002,2015-10-18,13.4,0.0,30316.02\nR002,2015-10-19,14.8,0.0,36000.0\nR002,2015-10-20,14.7,0.0,36000.0\nR002,2015-10-21,15.0,0.0,36000.0\nR002,2015-10-22,16.5,1.1,26682.09\nR002,2015-10-23,17.7,0.0,36000.0\nR002,2015-10-24,16.9,20.3,1460.34\nR002,2015-10-25,13.1,22.7,3243.27\nR002,2015-10-26,9.7,0.0,36000.0\nR002,2015-10-27,9.6,0.0,35002.58\nR002,2015-10-28,8.7,0.0,36000.0\nR002,2015-10-29,7.3,0.0,36000.0\nR002,2015-10-30,8.5,0.0,36000.0\nR002,2015-10-31,8.7,0.0,31002.44\nR002,2015-11-01,8.4,0.9,36000.0\nR002,2015-11-02,9.4,2.2,31756.17\nR002,2015-11-03,10.3,0.0,29512.15\nR002,2015-11-04,10.2,0.0,35810.01\nR002,2015-11-05,11.3,0.0,35729.51\nR002,2015-11-06,10.8,0.0,35588.61\nR002,2015-11-07,11.0,0.0,35432.82\nR002,2015-11-08,9.1,0.5,31342.39\nR002,2015-11-09,8.4,9.6,10850.17\nR002,2015-11-10,7.4,1.8,32858.71\nR002,2015-11-11,6.1,46.2,178.48\nR002,2015-11-12,5.5,4.4,32686.29\nR002,2015-11-13,4.9,39.0,1001.62\nR002,2015-11-14,3.4,0.0,34087.5\nR002,2015-11-15,4.4,7.0,101.28\nR002,2015-11-16,3.1,2.7,13866.09\nR002,2015-11-17,3.8,14.9,22161.37\nR002,2015-11-18,1.6,0.0,30345.86\nR002,2015-11-19,4.0,6.3,18153.22\nR002,2015-11-20,7.1,1.1,23923.24\nR002,2015-11-21,10.1,0.0,23942.9\nR002,2015-11-22,14.1,0.0,30950.19\nR002,2015-11-23,12.5,0.0,32219.36\nR002,2015-11-24,10.2,0.0,26512.3\nR002,2015-11-25,12.3,0.0,4016.45\nR002,2015-11-26,13.9,0.0,29837.18\nR002,2015-11-27,10.2,13.8,0.0\nR002,2015-11-28,10.9,0.0,29505.23\nR002,2015-11-29,9.3,12.4,1091.92\nR002,2015-11-30,8.1,9.2,17923.51\nR002,2015-12-01,6.2,17.7,25713.57\nR002,2015-12-02,3.1,44.4,0.0\nR002,2015-12-03,-1.8,0.0,30168.9\nR002,2015-12-04,-3.6,0.0,28725.49\nR002,2015-12-05,-2.0,0.0,28800.0\nR002,2015-12-06,2.5,1.7,12310.05\nR002,2015-12-07,5.5,0.6,24398.52\nR002,2015-12-08,5.1,0.5,29226.59\nR002,2015-12-09,0.2,0.0,30026.98\nR002,2015-12-10,2.9,0.0,29644.57\nR002,2015-12-11,3.9,0.0,28800.0\nR002,2015-12-12,3.8,0.0,30034.1\nR002,2015-12-13,3.2,1.7,16067.82\nR002,2015-12-14,4.0,0.4,28247.11\nR002,2015-12-15,2.9,7.1,30236.63\nR002,2015-12-16,0.9,0.0,26741.39\nR002,2015-12-17,0.5,0.0,30405.88\nR002,2015-12-18,0.4,0.0,28800.0\nR002,2015-12-19,1.9,0.0,30779.85\nR002,2015-12-20,3.5,4.9,18000.0\nR002,2015-12-21,5.9,5.5,17599.96\nR002,2015-12-22,6.8,0.0,29460.6\nR002,2015-12-23,5.9,7.2,23678.47\nR002,2015-12-24,5.7,4.3,22558.38\nR002,2015-12-25,4.7,1.8,30697.75\nR002,2015-12-26,3.5,0.0,26691.44\nR002,2015-12-27,6.0,8.1,10787.53\nR002,2015-12-28,5.8,6.5,9018.72\nR002,2015-12-29,7.2,3.6,27838.08\nR002,2015-12-30,2.1,22.0,0.0\nR002,2015-12-31,-4.6,6.3,28848.76\nR002,2016-01-01,-6.9,1.2,30284.1\nR002,2016-01-02,-11.0,0.0,31634.91\nR002,2016-01-03,-8.3,6.0,112.44\nR002,2016-01-04,-6.6,5.5,0.0\nR002,2016-01-05,2.5,23.2,0.0\nR002,2016-01-06,4.5,14.1,16988.66\nR002,2016-01-07,7.6,10.0,4693.58\nR002,2016-01-08,7.4,19.2,905.05\nR002,2016-01-09,3.0,17.4,5164.56\nR002,2016-01-10,3.3,2.6,28642.0\nR002,2016-01-11,6.4,8.4,0.0\nR002,2016-01-12,7.9,1.0,25933.62\nR002,2016-01-13,7.6,22.1,0.0\nR002,2016-01-14,4.3,8.7,22237.42\nR002,2016-01-15,0.6,0.3,32400.0\nR002,2016-01-16,1.6,0.0,32400.0\nR002,2016-01-17,8.3,0.0,21200.78\nR002,2016-01-18,9.2,3.5,23491.42\nR002,2016-01-19,4.0,23.7,118.83\nR002,2016-01-20,-1.2,5.9,31267.08\nR002,2016-01-21,-1.1,4.2,7561.56\nR002,2016-01-22,2.1,23.0,0.0\nR002,2016-01-23,0.5,10.2,0.0\nR002,2016-01-24,-2.3,5.7,0.0\nR002,2016-01-25,-4.9,1.4,20919.95\nR002,2016-01-26,-6.8,0.0,32400.0\nR002,2016-01-27,-7.5,0.0,32400.0\nR002,2016-01-28,-1.9,7.2,1265.9\nR002,2016-01-29,2.6,18.5,0.0\nR002,2016-01-30,2.8,0.4,21806.52\nR002,2016-01-31,3.5,0.0,32430.48\nR002,2016-02-01,4.7,8.7,3849.27\nR002,2016-02-02,2.2,20.7,32053.11\nR002,2016-02-03,4.1,0.0,33001.01\nR002,2016-02-04,9.9,0.0,30203.12\nR002,2016-02-05,11.4,0.0,30474.82\nR002,2016-02-06,7.9,4.1,17179.86\nR002,2016-02-07,4.2,1.9,27664.74\nR002,2016-02-08,3.7,0.8,32776.96\nR002,2016-02-09,3.6,0.0,34987.44\nR002,2016-02-10,2.8,0.0,35749.22\nR002,2016-02-11,3.8,0.3,34455.1\nR002,2016-02-12,7.1,5.4,18622.69\nR002,2016-02-13,10.1,0.0,34608.18\nR002,2016-02-14,9.4,1.6,16466.5\nR002,2016-02-15,11.0,0.5,33555.15\nR002,2016-02-16,12.1,3.4,27871.15\nR002,2016-02-17,11.5,3.5,34740.83\nR002,2016-02-18,6.6,0.2,36024.36\nR002,2016-02-19,3.9,0.0,37860.33\nR002,2016-02-20,2.2,0.5,33867.53\nR002,2016-02-21,3.6,1.2,21465.74\nR002,2016-02-22,5.2,6.6,31167.48\nR002,2016-02-23,9.0,0.0,38552.64\nR002,2016-02-24,10.0,0.0,38725.32\nR002,2016-02-25,8.5,14.0,0.0\nR002,2016-02-26,9.0,0.3,37578.58\nR002,2016-02-27,10.8,0.0,39250.93\nR002,2016-02-28,9.1,9.4,28092.31\nR002,2016-02-29,9.0,0.0,39496.9\nR002,2016-03-01,12.3,0.0,39598.65\nR002,2016-03-02,16.4,0.0,21600.0\nR002,2016-03-03,16.1,0.0,30847.08\nR002,2016-03-04,13.2,5.9,39600.0\nR002,2016-03-05,10.7,6.2,27821.5\nR002,2016-03-06,9.5,2.7,28454.25\nR002,2016-03-07,9.5,0.0,39600.0\nR002,2016-03-08,10.9,0.0,39600.0\nR002,2016-03-09,11.4,0.0,39600.0\nR002,2016-03-10,11.3,0.0,39600.0\nR002,2016-03-11,10.9,4.4,32286.44\nR002,2016-03-12,8.0,2.5,32400.0\nR002,2016-03-13,8.4,0.4,39600.0\nR002,2016-03-14,7.7,2.8,1631.71\nR002,2016-03-15,4.0,2.9,38147.14\nR002,2016-03-16,2.5,4.3,37981.78\nR002,2016-03-17,0.8,5.0,30488.2\nR002,2016-03-18,2.8,3.5,34250.63\nR002,2016-03-19,4.6,5.1,24901.52\nR002,2016-03-20,0.6,8.4,28718.41\nR002,2016-03-21,1.5,0.0,40237.26\nR002,2016-03-22,5.9,0.0,40146.1\nR002,2016-03-23,8.0,11.8,20960.39\nR002,2016-03-24,13.1,3.8,26910.08\nR002,2016-03-25,12.3,10.2,31924.47\nR002,2016-03-26,9.2,10.1,28348.77\nR002,2016-03-27,6.7,10.1,12895.06\nR002,2016-03-28,6.7,3.6,12712.32\nR002,2016-03-29,5.9,8.6,2843.85\nR002,2016-03-30,6.2,3.2,31683.1\nR002,2016-03-31,6.8,0.0,43797.81\nR002,2016-04-01,9.5,0.0,44091.03\nR002,2016-04-02,10.9,1.3,31189.22\nR002,2016-04-03,3.9,2.2,44686.04\nR002,2016-04-04,5.5,0.0,39600.0\nR002,2016-04-05,8.9,0.0,45971.42\nR002,2016-04-06,11.3,0.0,46042.74\nR002,2016-04-07,13.9,0.0,45572.98\nR002,2016-04-08,15.1,0.5,46185.54\nR002,2016-04-09,16.7,0.0,46257.01\nR002,2016-04-10,15.5,2.7,23027.07\nR002,2016-04-11,15.6,6.2,36786.66\nR002,2016-04-12,13.2,6.5,8491.7\nR002,2016-04-13,13.3,2.4,40206.44\nR002,2016-04-14,12.9,0.9,41516.1\nR002,2016-04-15,14.0,1.4,45404.93\nR002,2016-04-16,11.0,18.9,31500.56\nR002,2016-04-17,11.9,0.0,46800.0\nR002,2016-04-18,14.1,0.3,43200.0\nR002,2016-04-19,17.2,0.5,46465.57\nR002,2016-04-20,19.3,0.1,40304.83\nR002,2016-04-21,9.6,27.7,0.0\nR002,2016-04-22,6.3,1.0,46556.7\nR002,2016-04-23,12.8,0.0,43918.02\nR002,2016-04-24,16.9,0.0,32400.0\nR002,2016-04-25,16.7,5.1,40570.6\nR002,2016-04-26,16.1,3.2,45605.92\nR002,2016-04-27,12.9,17.4,17965.23\nR002,2016-04-28,13.4,0.3,47274.74\nR002,2016-04-29,12.7,1.3,44735.78\nR002,2016-04-30,13.8,0.4,39769.4\nR002,2016-05-01,12.8,2.4,35494.06\nR002,2016-05-02,13.5,0.0,46158.74\nR002,2016-05-03,15.3,0.0,49104.13\nR002,2016-05-04,13.1,10.4,8193.92\nR002,2016-05-05,13.7,8.3,37165.2\nR002,2016-05-06,12.3,9.4,13685.11\nR002,2016-05-07,12.5,6.5,21490.28\nR002,2016-05-08,13.5,1.2,46903.83\nR002,2016-05-09,14.8,1.0,47554.35\nR002,2016-05-10,16.3,0.8,45645.69\nR002,2016-05-11,13.1,24.2,27735.82\nR002,2016-05-12,14.6,0.1,48841.76\nR002,2016-05-13,15.9,2.6,32902.92\nR002,2016-05-14,19.0,2.3,48494.74\nR002,2016-05-15,18.4,0.8,48285.33\nR002,2016-05-16,13.4,21.0,5223.83\nR002,2016-05-17,16.2,0.0,48454.08\nR002,2016-05-18,16.7,3.8,46800.0\nR002,2016-05-19,14.5,9.5,31005.81\nR002,2016-05-20,14.7,0.0,49355.91\nR002,2016-05-21,16.2,0.0,50400.0\nR002,2016-05-22,18.6,0.0,50287.83\nR002,2016-05-23,17.8,12.9,34020.26\nR002,2016-05-24,15.1,28.7,356.5\nR002,2016-05-25,17.3,1.8,48142.58\nR002,2016-05-26,16.2,5.4,22257.88\nR002,2016-05-27,18.0,4.6,47261.67\nR002,2016-05-28,17.2,13.1,46637.01\nR002,2016-05-29,17.2,10.5,29987.88\nR002,2016-05-30,16.6,2.4,34593.79\nR002,2016-05-31,18.5,2.8,49016.26\nR002,2016-06-01,17.8,2.1,48336.74\nR002,2016-06-02,18.0,9.3,44362.26\nR002,2016-06-03,17.9,1.4,38178.63\nR002,2016-06-04,16.8,9.6,16264.05\nR002,2016-06-05,17.3,4.6,43660.65\nR002,2016-06-06,18.1,0.3,51217.81\nR002,2016-06-07,17.7,0.0,51446.92\nR002,2016-06-08,16.3,2.0,50086.82\nR002,2016-06-09,15.8,4.6,48536.85\nR002,2016-06-10,15.4,11.8,34311.04\nR002,2016-06-11,16.5,4.8,47553.09\nR002,2016-06-12,18.5,0.0,49813.08\nR002,2016-06-13,20.4,0.0,52928.16\nR002,2016-06-14,23.9,0.0,45892.23\nR002,2016-06-15,22.7,7.0,40986.33\nR002,2016-06-16,19.5,11.9,28318.14\nR002,2016-06-17,20.2,4.2,37366.78\nR002,2016-06-18,21.2,5.2,48645.06\nR002,2016-06-19,22.8,7.0,47035.74\nR002,2016-06-20,22.7,4.0,50002.63\nR002,2016-06-21,24.2,1.5,53778.55\nR002,2016-06-22,25.0,0.0,52654.94\nR002,2016-06-23,24.4,1.0,52665.29\nR002,2016-06-24,23.2,2.6,40040.38\nR002,2016-06-25,24.0,0.5,53025.02\nR002,2016-06-26,24.9,0.0,53964.53\nR002,2016-06-27,25.5,0.2,53753.1\nR002,2016-06-28,23.8,3.9,39306.98\nR002,2016-06-29,23.5,6.2,50683.82\nR002,2016-06-30,22.8,10.1,50089.78\nR002,2016-07-01,21.5,14.0,37725.72\nR002,2016-07-02,22.1,12.2,39812.68\nR002,2016-07-03,22.7,4.9,49054.32\nR002,2016-07-04,22.1,4.2,45331.68\nR002,2016-07-05,22.4,1.8,51366.19\nR002,2016-07-06,22.4,4.1,52895.93\nR002,2016-07-07,23.1,1.8,51716.81\nR002,2016-07-08,20.3,11.4,34868.3\nR002,2016-07-09,19.3,0.2,52813.82\nR002,2016-07-10,18.7,5.1,48486.31\nR002,2016-07-11,18.1,15.0,44408.51\nR002,2016-07-12,20.7,0.2,53694.91\nR002,2016-07-13,23.5,0.0,53659.27\nR002,2016-07-14,25.5,0.0,53451.31\nR002,2016-07-15,24.5,4.0,50860.78\nR002,2016-07-16,25.2,0.4,51972.02\nR002,2016-07-17,27.4,0.0,53488.05\nR002,2016-07-18,27.0,3.6,51367.06\nR002,2016-07-19,22.6,9.2,45353.03\nR002,2016-07-20,22.9,0.0,49925.47\nR002,2016-07-21,21.4,12.9,18515.36\nR002,2016-07-22,21.5,0.0,49304.36\nR002,2016-07-23,20.6,0.0,51811.76\nR002,2016-07-24,19.0,2.3,47994.4\nR002,2016-07-25,19.8,2.3,46474.43\nR002,2016-07-26,21.4,0.2,47299.64\nR002,2016-07-27,22.1,0.7,48846.26\nR002,2016-07-28,22.0,2.5,49593.3\nR002,2016-07-29,22.1,0.7,49992.08\nR002,2016-07-30,22.6,2.8,50042.35\nR002,2016-07-31,22.5,1.3,50095.34\nR002,2016-08-01,23.2,0.0,50492.73\nR002,2016-08-02,24.2,0.0,50200.85\nR002,2016-08-03,23.6,1.7,48542.87\nR002,2016-08-04,22.8,3.4,48260.66\nR002,2016-08-05,23.2,1.3,47299.7\nR002,2016-08-06,24.3,5.7,47705.8\nR002,2016-08-07,23.6,7.9,47963.56\nR002,2016-08-08,22.5,6.2,46783.36\nR002,2016-08-09,23.9,0.3,46800.0\nR002,2016-08-10,25.5,0.7,47708.86\nR002,2016-08-11,24.5,4.9,46800.0\nR002,2016-08-12,23.8,1.6,45388.09\nR002,2016-08-13,24.5,2.1,46800.0\nR002,2016-08-14,21.8,33.6,14606.66\nR002,2016-08-15,21.0,0.0,46800.0\nR002,2016-08-16,21.9,2.8,45998.09\nR002,2016-08-17,23.0,13.6,43710.46\nR002,2016-08-18,22.4,14.7,39787.66\nR002,2016-08-19,23.9,0.2,46800.0\nR002,2016-08-20,26.3,0.1,46800.0\nR002,2016-08-21,27.2,0.1,46800.0\nR002,2016-08-22,27.3,1.1,38405.04\nR002,2016-08-23,25.7,4.8,46800.0\nR002,2016-08-24,25.0,2.5,46378.31\nR002,2016-08-25,24.5,2.5,46037.96\nR002,2016-08-26,22.8,11.6,45938.17\nR002,2016-08-27,23.1,1.7,46694.51\nR002,2016-08-28,22.9,1.3,46591.2\nR002,2016-08-29,23.5,1.6,46396.14\nR002,2016-08-30,23.1,4.8,43864.46\nR002,2016-08-31,21.3,6.8,41002.14\nR002,2016-09-01,20.9,0.0,41928.24\nR002,2016-09-02,20.0,1.3,44978.41\nR002,2016-09-03,19.9,0.9,42650.44\nR002,2016-09-04,19.2,3.0,38593.68\nR002,2016-09-05,19.9,0.9,40400.16\nR002,2016-09-06,19.4,6.5,38439.98\nR002,2016-09-07,19.8,0.0,43953.31\nR002,2016-09-08,20.2,0.0,43367.24\nR002,2016-09-09,19.9,0.0,42930.42\nR002,2016-09-10,20.0,0.1,42808.91\nR002,2016-09-11,19.2,0.0,42984.32\nR002,2016-09-12,19.4,0.0,43200.0\nR002,2016-09-13,19.9,0.1,43148.06\nR002,2016-09-14,19.2,0.6,35580.29\nR002,2016-09-15,17.0,3.0,27687.43\nR002,2016-09-16,16.1,0.0,41428.7\nR002,2016-09-17,16.0,0.2,37644.08\nR002,2016-09-18,16.5,0.0,40540.77\nR002,2016-09-19,17.7,0.0,42166.08\nR002,2016-09-20,20.6,1.9,41526.56\nR002,2016-09-21,16.0,27.5,9355.55\nR002,2016-09-22,13.6,23.0,1784.36\nR002,2016-09-23,13.1,24.6,8686.87\nR002,2016-09-24,11.5,2.6,35545.34\nR002,2016-09-25,9.9,1.2,39525.21\nR002,2016-09-26,11.5,0.1,40819.32\nR002,2016-09-27,11.2,5.4,27141.89\nR002,2016-09-28,9.8,15.5,5062.44\nR002,2016-09-29,9.3,1.6,26524.88\nR002,2016-09-30,12.0,22.5,31190.22\nR002,2016-10-01,13.9,0.0,39742.74\nR002,2016-10-02,17.3,0.0,40313.51\nR002,2016-10-03,18.0,0.0,39602.86\nR002,2016-10-04,17.1,0.0,39532.98\nR002,2016-10-05,18.4,0.0,39025.65\nR002,2016-10-06,18.8,0.0,37507.29\nR002,2016-10-07,17.8,0.3,37601.43\nR002,2016-10-08,19.0,0.0,38021.15\nR002,2016-10-09,16.4,8.4,24752.91\nR002,2016-10-10,14.9,0.6,36778.36\nR002,2016-10-11,14.3,0.0,36000.0\nR002,2016-10-12,13.8,5.0,6582.02\nR002,2016-10-13,15.9,10.1,28627.44\nR002,2016-10-14,11.1,17.6,31636.49\nR002,2016-10-15,9.8,0.5,36070.77\nR002,2016-10-16,9.0,1.4,23496.73\nR002,2016-10-17,7.4,1.6,35309.71\nR002,2016-10-18,8.5,0.0,36000.0\nR002,2016-10-19,9.9,0.0,36000.0\nR002,2016-10-20,10.7,0.2,33907.76\nR002,2016-10-21,11.4,0.0,36000.0\nR002,2016-10-22,12.6,0.0,35384.51\nR002,2016-10-23,12.1,0.4,23745.28\nR002,2016-10-24,12.0,5.0,35419.94\nR002,2016-10-25,10.0,8.3,19560.52\nR002,2016-10-26,7.6,1.7,31651.39\nR002,2016-10-27,8.3,0.0,32051.55\nR002,2016-10-28,8.8,2.3,17320.52\nR002,2016-10-29,8.7,0.4,36000.0\nR002,2016-10-30,5.4,23.2,574.76\nR002,2016-10-31,2.1,6.9,28744.64\nR002,2016-11-01,3.4,3.5,30336.57\nR002,2016-11-02,1.8,1.9,35185.01\nR002,2016-11-03,6.7,0.0,35827.22\nR002,2016-11-04,8.3,10.6,24947.32\nR002,2016-11-05,7.7,1.5,27187.71\nR002,2016-11-06,7.0,0.0,35465.27\nR002,2016-11-07,11.2,0.2,30078.03\nR002,2016-11-08,15.7,0.1,35163.99\nR002,2016-11-09,16.5,0.0,32066.17\nR002,2016-11-10,16.6,1.1,25419.01\nR002,2016-11-11,12.7,9.3,27442.86\nR002,2016-11-12,10.3,1.8,33823.25\nR002,2016-11-13,10.9,1.7,34449.76\nR002,2016-11-14,8.9,18.0,0.0\nR002,2016-11-15,7.0,2.2,13680.06\nR002,2016-11-16,5.6,5.3,3793.8\nR002,2016-11-17,4.3,10.0,25987.24\nR002,2016-11-18,2.3,0.0,31073.33\nR002,2016-11-19,4.8,0.0,32960.23\nR002,2016-11-20,7.3,0.0,32957.41\nR002,2016-11-21,9.0,0.0,32588.77\nR002,2016-11-22,8.5,0.0,32588.37\nR002,2016-11-23,8.3,0.0,32220.43\nR002,2016-11-24,6.6,0.0,31852.39\nR002,2016-11-25,6.4,0.0,31855.35\nR002,2016-11-26,4.7,0.0,30096.76\nR002,2016-11-27,6.1,0.0,31008.58\nR002,2016-11-28,8.4,0.0,28257.8\nR002,2016-11-29,8.9,8.7,18173.8\nR002,2016-11-30,6.0,29.2,0.0\nR002,2016-12-01,2.4,8.2,7137.49\nR002,2016-12-02,1.1,9.5,0.0\nR002,2016-12-03,0.8,30.5,0.0\nR002,2016-12-04,-0.6,23.1,755.91\nR002,2016-12-05,-4.0,9.0,952.35\nR002,2016-12-06,-6.0,0.0,30394.09\nR002,2016-12-07,-2.4,13.0,0.0\nR002,2016-12-08,-7.4,0.0,27969.79\nR002,2016-12-09,-0.0,8.1,7326.5\nR002,2016-12-10,3.6,20.8,0.0\nR002,2016-12-11,4.3,30.2,0.0\nR002,2016-12-12,7.0,9.0,13971.46\nR002,2016-12-13,4.5,34.4,0.0\nR002,2016-12-14,-2.5,0.1,26998.06\nR002,2016-12-15,-2.4,1.6,28611.03\nR002,2016-12-16,-2.5,7.0,16008.18\nR002,2016-12-17,-2.7,5.8,22636.71\nR002,2016-12-18,-4.5,0.0,29222.84\nR002,2016-12-19,-3.4,4.7,26513.37\nR002,2016-12-20,-0.2,20.4,0.0\nR002,2016-12-21,1.0,2.7,25791.33\nR002,2016-12-22,-0.3,0.0,31129.41\nR002,2016-12-23,-0.8,0.0,31217.64\nR002,2016-12-24,1.8,0.0,31095.35\nR002,2016-12-25,1.8,0.8,29901.66\nR002,2016-12-26,1.8,3.5,23985.28\nR002,2016-12-27,2.0,0.9,27389.61\nR002,2016-12-28,2.5,7.9,1228.3\nR002,2016-12-29,4.4,3.9,1170.87\nR002,2016-12-30,5.9,0.7,27922.6\nR002,2016-12-31,6.9,0.0,30144.83\nR002,2017-01-01,3.9,5.0,9163.41\nR002,2017-01-02,2.2,3.7,26824.84\nR002,2017-01-03,0.6,0.0,31741.01\nR002,2017-01-04,0.5,14.4,0.0\nR002,2017-01-05,2.6,1.9,29300.59\nR002,2017-01-06,7.5,2.9,20389.33\nR002,2017-01-07,8.3,0.0,31981.29\nR002,2017-01-08,4.8,3.1,5630.56\nR002,2017-01-09,5.2,1.1,21896.36\nR002,2017-01-10,3.6,3.0,30666.0\nR002,2017-01-11,2.7,1.1,32240.22\nR002,2017-01-12,4.2,0.1,29192.19\nR002,2017-01-13,0.7,5.4,25035.7\nR002,2017-01-14,0.7,0.0,32400.0\nR002,2017-01-15,5.2,0.0,30091.42\nR002,2017-01-16,4.2,0.4,24437.44\nR002,2017-01-17,5.0,6.3,10988.49\nR002,2017-01-18,5.5,0.0,25059.17\nR002,2017-01-19,4.2,7.9,0.0\nR002,2017-01-20,1.9,8.6,32279.6\nR002,2017-01-21,0.7,1.3,28457.73\nR002,2017-01-22,0.2,6.9,0.0\nR002,2017-01-23,0.2,0.0,32260.48\nR002,2017-01-24,2.6,0.0,32400.0\nR002,2017-01-25,2.3,4.9,26335.89\nR002,2017-01-26,-1.2,10.6,5831.8\nR002,2017-01-27,-5.2,0.0,32400.0\nR002,2017-01-28,-6.1,0.0,32528.37\nR002,2017-01-29,-6.3,6.9,5836.14\nR002,2017-01-30,-9.3,0.5,32400.0\nR002,2017-01-31,-8.6,0.0,32753.16\nR002,2017-02-01,-4.7,0.0,32526.59\nR002,2017-02-02,-3.0,0.7,31877.16\nR002,2017-02-03,0.2,0.0,32126.66\nR002,2017-02-04,3.5,4.7,0.0\nR002,2017-02-05,6.1,12.7,23044.26\nR002,2017-02-06,6.8,11.4,7441.88\nR002,2017-02-07,7.9,1.9,24784.26\nR002,2017-02-08,6.9,0.0,29409.73\nR002,2017-02-09,6.2,0.8,33537.78\nR002,2017-02-10,4.1,7.5,1627.61\nR002,2017-02-11,1.8,5.4,27176.02\nR002,2017-02-12,0.5,6.6,20638.65\nR002,2017-02-13,-0.7,1.2,30731.21\nR002,2017-02-14,-1.7,5.0,33148.75\nR002,2017-02-15,-2.3,20.6,9064.6\nR002,2017-02-16,-4.7,0.0,33909.95\nR002,2017-02-17,-5.7,0.0,37362.07\nR002,2017-02-18,-0.1,0.0,27219.79\nR002,2017-02-19,3.8,0.0,37979.71\nR002,2017-02-20,4.9,0.0,37913.27\nR002,2017-02-21,4.4,2.4,7845.92\nR002,2017-02-22,4.4,16.0,24674.6\nR002,2017-02-23,4.2,9.9,17828.3\nR002,2017-02-24,8.1,9.9,12046.38\nR002,2017-02-25,9.7,0.8,39031.31\nR002,2017-02-26,4.5,19.5,36604.04\nR002,2017-02-27,5.1,0.0,30062.65\nR002,2017-02-28,7.9,0.0,39478.88\nR002,2017-03-01,10.6,0.0,39581.15\nR002,2017-03-02,11.4,0.0,26223.14\nR002,2017-03-03,11.5,0.0,39600.0\nR002,2017-03-04,6.9,5.3,2894.57\nR002,2017-03-05,6.9,0.0,37141.99\nR002,2017-03-06,9.3,0.0,39600.0\nR002,2017-03-07,9.4,0.0,39600.0\nR002,2017-03-08,9.2,0.0,39600.0\nR002,2017-03-09,9.2,0.0,39600.0\nR002,2017-03-10,10.3,0.0,39600.0\nR002,2017-03-11,10.8,0.0,39600.0\nR002,2017-03-12,9.6,3.2,33944.13\nR002,2017-03-13,9.4,0.5,28230.87\nR002,2017-03-14,7.9,6.5,763.85\nR002,2017-03-15,5.5,7.8,1018.92\nR002,2017-03-16,4.8,0.9,37802.42\nR002,2017-03-17,6.1,1.2,36531.03\nR002,2017-03-18,5.3,7.1,6473.93\nR002,2017-03-19,6.4,0.0,19024.09\nR002,2017-03-20,4.8,8.0,21088.67\nR002,2017-03-21,2.3,3.4,39715.17\nR002,2017-03-22,3.3,1.2,16768.19\nR002,2017-03-23,6.3,0.0,33586.76\nR002,2017-03-24,6.9,0.0,39279.13\nR002,2017-03-25,7.0,3.3,38821.67\nR002,2017-03-26,6.3,3.4,4080.16\nR002,2017-03-27,5.3,13.3,352.3\nR002,2017-03-28,3.7,0.3,42806.37\nR002,2017-03-29,6.4,0.4,38189.8\nR002,2017-03-30,9.1,0.0,44690.18\nR002,2017-03-31,7.9,0.0,42364.51\nR002,2017-04-01,7.8,0.0,43559.02\nR002,2017-04-02,7.7,0.0,42679.13\nR002,2017-04-03,8.8,0.0,45704.28\nR002,2017-04-04,11.9,0.0,45882.34\nR002,2017-04-05,13.6,0.0,44701.09\nR002,2017-04-06,14.7,0.0,42093.37\nR002,2017-04-07,12.3,3.6,27217.41\nR002,2017-04-08,11.1,5.2,37843.96\nR002,2017-04-09,5.9,12.5,22185.87\nR002,2017-04-10,6.6,1.6,40842.18\nR002,2017-04-11,7.1,3.8,44829.04\nR002,2017-04-12,11.6,0.0,40300.7\nR002,2017-04-13,15.6,2.7,32080.39\nR002,2017-04-14,11.2,17.4,2756.9\nR002,2017-04-15,9.4,0.8,38547.81\nR002,2017-04-16,11.2,0.0,46501.09\nR002,2017-04-17,15.5,0.4,46800.0\nR002,2017-04-18,17.5,0.9,46800.0\nR002,2017-04-19,8.2,6.9,44912.78\nR002,2017-04-20,12.6,0.0,46969.29\nR002,2017-04-21,10.9,23.8,8331.25\nR002,2017-04-22,4.7,14.5,33353.54\nR002,2017-04-23,12.2,22.6,26893.61\nR002,2017-04-24,4.4,15.1,14196.17\nR002,2017-04-25,3.8,0.4,46800.0\nR002,2017-04-26,8.5,0.0,47698.27\nR002,2017-04-27,14.7,0.0,47989.93\nR002,2017-04-28,16.2,0.0,48283.54\nR002,2017-04-29,16.8,0.8,48245.24\nR002,2017-04-30,16.7,0.5,47537.99\nR002,2017-05-01,17.8,0.4,48505.99\nR002,2017-05-02,18.8,0.9,48806.71\nR002,2017-05-03,18.2,0.1,25185.87\nR002,2017-05-04,15.9,2.4,28246.58\nR002,2017-05-05,15.9,0.0,48707.07\nR002,2017-05-06,16.1,1.5,49354.24\nR002,2017-05-07,14.4,14.8,24792.32\nR002,2017-05-08,14.5,2.4,45889.95\nR002,2017-05-09,14.1,11.0,32999.15\nR002,2017-05-10,14.5,25.9,2609.85\nR002,2017-05-11,13.0,11.3,37675.57\nR002,2017-05-12,13.6,0.0,48161.78\nR002,2017-05-13,16.3,0.0,40436.38\nR002,2017-05-14,16.2,8.4,39813.57\nR002,2017-05-15,14.3,7.8,21334.91\nR002,2017-05-16,14.0,7.7,16477.56\nR002,2017-05-17,14.2,5.4,28986.65\nR002,2017-05-18,14.9,2.5,39046.52\nR002,2017-05-19,13.9,8.4,1503.76\nR002,2017-05-20,13.1,24.3,847.06\nR002,2017-05-21,12.3,13.2,28689.57\nR002,2017-05-22,11.7,11.7,23234.37\nR002,2017-05-23,12.0,8.2,24268.01\nR002,2017-05-24,10.7,16.1,1121.98\nR002,2017-05-25,12.5,0.4,39831.71\nR002,2017-05-26,16.3,0.0,50529.86\nR002,2017-05-27,17.0,2.2,44897.06\nR002,2017-05-28,16.9,5.9,50047.92\nR002,2017-05-29,17.9,2.0,49022.37\nR002,2017-05-30,19.8,2.7,50437.61\nR002,2017-05-31,17.2,9.1,19057.62\nR002,2017-06-01,16.7,6.9,33161.04\nR002,2017-06-02,16.8,4.5,46799.91\nR002,2017-06-03,16.8,1.7,44078.89\nR002,2017-06-04,18.8,0.0,51201.34\nR002,2017-06-05,22.0,0.0,52862.94\nR002,2017-06-06,23.7,2.1,51387.52\nR002,2017-06-07,19.9,29.4,13431.87\nR002,2017-06-08,21.4,2.6,44957.66\nR002,2017-06-09,22.9,0.7,51703.72\nR002,2017-06-10,20.6,6.8,33618.46\nR002,2017-06-11,19.2,13.1,42793.49\nR002,2017-06-12,18.3,0.9,48328.22\nR002,2017-06-13,19.0,0.0,52015.38\nR002,2017-06-14,19.9,0.6,50901.77\nR002,2017-06-15,17.1,4.7,47688.09\nR002,2017-06-16,14.7,4.9,47991.81\nR002,2017-06-17,16.0,0.0,53444.47\nR002,2017-06-18,18.7,1.3,48991.14\nR002,2017-06-19,18.1,9.6,19152.16\nR002,2017-06-20,17.1,29.1,8348.71\nR002,2017-06-21,18.6,16.2,44652.96\nR002,2017-06-22,18.8,6.6,45424.14\nR002,2017-06-23,18.8,4.3,52159.85\nR002,2017-06-24,20.5,0.0,53956.67\nR002,2017-06-25,20.4,0.1,53026.23\nR002,2017-06-26,22.9,0.1,53963.86\nR002,2017-06-27,24.3,2.1,53254.82\nR002,2017-06-28,24.7,1.8,53416.24\nR002,2017-06-29,23.0,1.9,53250.19\nR002,2017-06-30,22.7,2.0,48251.18\nR002,2017-07-01,24.4,0.1,52605.33\nR002,2017-07-02,27.4,0.0,53934.51\nR002,2017-07-03,28.9,0.0,53921.8\nR002,2017-07-04,28.0,3.7,50832.74\nR002,2017-07-05,23.2,8.1,41374.11\nR002,2017-07-06,22.3,2.2,49334.03\nR002,2017-07-07,20.3,13.9,47481.18\nR002,2017-07-08,18.7,1.2,49373.31\nR002,2017-07-09,20.0,0.0,51295.52\nR002,2017-07-10,21.0,0.0,53052.19\nR002,2017-07-11,21.3,0.2,50210.16\nR002,2017-07-12,22.4,0.4,52461.79\nR002,2017-07-13,21.9,7.3,52001.91\nR002,2017-07-14,20.9,6.0,49863.21\nR002,2017-07-15,22.1,5.8,51433.6\nR002,2017-07-16,23.8,1.1,53472.85\nR002,2017-07-17,25.3,1.5,53485.51\nR002,2017-07-18,26.0,0.8,53143.04\nR002,2017-07-19,25.5,1.3,50934.75\nR002,2017-07-20,22.3,18.4,30753.46\nR002,2017-07-21,23.8,1.4,46893.26\nR002,2017-07-22,23.6,5.4,48704.89\nR002,2017-07-23,22.9,9.7,32093.47\nR002,2017-07-24,23.6,0.5,48335.26\nR002,2017-07-25,22.8,6.1,45034.56\nR002,2017-07-26,23.7,0.9,48783.28\nR002,2017-07-27,25.9,0.0,51796.26\nR002,2017-07-28,28.2,0.2,51064.19\nR002,2017-07-29,22.4,28.4,33198.86\nR002,2017-07-30,21.7,0.0,48643.25\nR002,2017-07-31,21.6,0.0,50079.39\nR002,2017-08-01,24.1,0.0,48755.14\nR002,2017-08-02,25.9,0.0,50187.35\nR002,2017-08-03,26.7,0.1,49556.92\nR002,2017-08-04,27.1,0.5,49273.54\nR002,2017-08-05,27.3,0.0,48309.56\nR002,2017-08-06,27.3,0.4,48373.38\nR002,2017-08-07,27.6,2.6,48098.4\nR002,2017-08-08,27.4,1.2,47985.47\nR002,2017-08-09,26.8,0.6,47337.0\nR002,2017-08-10,27.2,0.0,47703.6\nR002,2017-08-11,27.1,0.4,47732.24\nR002,2017-08-12,27.9,1.1,47425.27\nR002,2017-08-13,27.3,3.7,47119.87\nR002,2017-08-14,25.7,2.3,45817.85\nR002,2017-08-15,22.9,13.8,38095.96\nR002,2017-08-16,22.6,1.1,46800.0\nR002,2017-08-17,22.8,10.4,46908.97\nR002,2017-08-18,21.8,7.3,46455.92\nR002,2017-08-19,22.2,3.5,45515.91\nR002,2017-08-20,22.4,3.7,46800.0\nR002,2017-08-21,24.8,0.0,46800.0\nR002,2017-08-22,25.1,3.5,38734.22\nR002,2017-08-23,21.1,12.4,40158.09\nR002,2017-08-24,21.1,0.3,44606.89\nR002,2017-08-25,22.5,0.1,45407.85\nR002,2017-08-26,22.5,0.1,43637.9\nR002,2017-08-27,23.2,0.1,46714.15\nR002,2017-08-28,23.2,1.0,46610.72\nR002,2017-08-29,21.3,10.3,33385.08\nR002,2017-08-30,20.7,13.3,39505.83\nR002,2017-08-31,17.9,33.8,13766.23\nR002,2017-09-01,18.8,1.1,42065.73\nR002,2017-09-02,20.1,0.0,40685.37\nR002,2017-09-03,21.1,0.0,45336.76\nR002,2017-09-04,23.7,0.0,45065.16\nR002,2017-09-05,23.5,0.0,44474.29\nR002,2017-09-06,21.7,13.7,40373.72\nR002,2017-09-07,17.9,23.0,27241.17\nR002,2017-09-08,17.9,0.0,43668.98\nR002,2017-09-09,20.8,0.0,43721.79\nR002,2017-09-10,22.1,0.0,43456.43\nR002,2017-09-11,21.8,0.0,43200.0\nR002,2017-09-12,23.4,0.0,43246.49\nR002,2017-09-13,23.9,0.0,43163.55\nR002,2017-09-14,23.2,0.0,43096.21\nR002,2017-09-15,23.4,0.0,43028.8\nR002,2017-09-16,23.4,0.0,42961.28\nR002,2017-09-17,23.2,0.0,42893.66\nR002,2017-09-18,23.0,0.0,42825.95\nR002,2017-09-19,22.5,0.0,42758.12\nR002,2017-09-20,23.4,0.0,42690.17\nR002,2017-09-21,23.7,0.0,42622.1\nR002,2017-09-22,23.4,0.6,42262.27\nR002,2017-09-23,20.9,6.1,31100.1\nR002,2017-09-24,19.5,3.3,31407.8\nR002,2017-09-25,19.0,0.5,38579.2\nR002,2017-09-26,19.5,0.0,40811.18\nR002,2017-09-27,17.7,0.3,37647.72\nR002,2017-09-28,14.6,1.0,39925.51\nR002,2017-09-29,14.9,0.3,37672.58\nR002,2017-09-30,15.6,0.7,37568.8\nR002,2017-10-01,15.1,0.7,34254.56\nR002,2017-10-02,14.1,0.1,35599.76\nR002,2017-10-03,11.4,6.4,31406.36\nR002,2017-10-04,10.9,0.0,39003.93\nR002,2017-10-05,12.7,0.0,39007.41\nR002,2017-10-06,14.6,0.0,38780.57\nR002,2017-10-07,16.4,0.0,36936.19\nR002,2017-10-08,18.2,0.0,38648.86\nR002,2017-10-09,16.3,2.0,35864.15\nR002,2017-10-10,13.8,7.2,33146.71\nR002,2017-10-11,12.6,1.4,36497.64\nR002,2017-10-12,11.6,0.0,36551.7\nR002,2017-10-13,11.6,0.0,36279.05\nR002,2017-10-14,10.5,7.1,32400.0\nR002,2017-10-15,9.9,10.8,28828.56\nR002,2017-10-16,8.9,22.0,30438.68\nR002,2017-10-17,8.7,0.5,27688.51\nR002,2017-10-18,11.9,0.0,30632.74\nR002,2017-10-19,12.4,0.0,36000.0\nR002,2017-10-20,13.4,0.0,36000.0\nR002,2017-10-21,13.4,0.0,36000.0\nR002,2017-10-22,12.1,9.7,22170.77\nR002,2017-10-23,11.4,1.0,33373.82\nR002,2017-10-24,14.7,0.0,32400.0\nR002,2017-10-25,17.4,0.0,36000.0\nR002,2017-10-26,14.8,8.1,23073.76\nR002,2017-10-27,11.6,1.2,34252.32\nR002,2017-10-28,13.2,0.0,36000.0\nR002,2017-10-29,12.7,20.6,313.04\nR002,2017-10-30,9.6,51.3,0.0\nR002,2017-10-31,3.8,18.5,0.0\nR002,2017-11-01,1.9,15.9,17843.29\nR002,2017-11-02,1.3,0.0,35926.65\nR002,2017-11-03,2.9,0.0,35847.91\nR002,2017-11-04,4.7,0.0,32769.74\nR002,2017-11-05,6.6,0.0,33039.97\nR002,2017-11-06,10.9,1.3,30008.06\nR002,2017-11-07,10.5,2.0,33139.85\nR002,2017-11-08,10.7,0.7,33470.05\nR002,2017-11-09,8.8,0.0,34773.93\nR002,2017-11-10,9.4,1.0,31817.7\nR002,2017-11-11,9.2,1.8,33410.87\nR002,2017-11-12,10.2,3.0,25738.86\nR002,2017-11-13,12.0,1.5,32300.62\nR002,2017-11-14,13.1,0.0,29268.44\nR002,2017-11-15,12.7,0.0,34199.63\nR002,2017-11-16,10.3,0.0,31990.43\nR002,2017-11-17,9.5,0.0,32973.27\nR002,2017-11-18,10.8,0.0,32154.11\nR002,2017-11-19,13.0,0.0,28800.0\nR002,2017-11-20,12.8,1.4,30805.91\nR002,2017-11-21,10.7,17.4,14972.31\nR002,2017-11-22,3.5,3.8,30995.63\nR002,2017-11-23,-0.3,13.6,26900.33\nR002,2017-11-24,-3.7,5.7,14106.24\nR002,2017-11-25,-3.7,0.0,31233.82\nR002,2017-11-26,-1.1,0.0,30976.27\nR002,2017-11-27,0.1,0.0,31000.61\nR002,2017-11-28,3.1,0.0,29497.21\nR002,2017-11-29,6.5,6.2,2320.25\nR002,2017-11-30,6.1,0.5,30307.91\nR002,2017-12-01,7.3,0.0,29945.29\nR002,2017-12-02,9.7,0.0,29960.29\nR002,2017-12-03,11.1,0.0,28800.0\nR002,2017-12-04,13.1,0.0,28860.09\nR002,2017-12-05,8.4,29.9,0.0\nR002,2017-12-06,3.0,5.8,11090.14\nR002,2017-12-07,0.9,17.4,0.0\nR002,2017-12-08,3.7,16.3,0.0\nR002,2017-12-09,5.2,0.4,30413.26\nR002,2017-12-10,5.8,0.0,30417.86\nR002,2017-12-11,4.1,8.8,30033.53\nR002,2017-12-12,5.5,0.0,28800.0\nR002,2017-12-13,4.1,0.0,30422.59\nR002,2017-12-14,5.7,0.0,27873.81\nR002,2017-12-15,7.6,0.0,28421.91\nR002,2017-12-16,7.6,0.0,29050.61\nR002,2017-12-17,10.3,0.0,28800.0\nR002,2017-12-18,11.0,0.4,19492.86\nR002,2017-12-19,8.7,6.4,11814.51\nR002,2017-12-20,6.2,10.2,0.0\nR002,2017-12-21,5.6,16.2,14772.61\nR002,2017-12-22,4.6,20.6,20598.22\nR002,2017-12-23,4.4,33.2,0.0\nR002,2017-12-24,2.6,4.3,27104.84\nR002,2017-12-25,4.0,36.8,0.0\nR002,2017-12-26,6.5,8.5,10510.57\nR002,2017-12-27,8.7,0.0,28800.0\nR002,2017-12-28,10.4,0.0,31401.32\nR002,2017-12-29,9.6,0.0,30985.4\nR002,2017-12-30,9.3,0.8,29005.77\nR002,2017-12-31,8.3,4.7,28800.0\nR002,2018-01-01,6.3,11.9,0.0\nR002,2018-01-02,4.8,0.0,29638.72\nR002,2018-01-03,4.9,0.0,28896.69\nR002,2018-01-04,5.6,0.0,16896.04\nR002,2018-01-05,6.6,0.1,30002.03\nR002,2018-01-06,6.2,0.0,30913.84\nR002,2018-01-07,4.5,0.0,31956.7\nR002,2018-01-08,5.8,0.2,29123.41\nR002,2018-01-09,4.0,15.9,8159.52\nR002,2018-01-10,-2.7,0.0,32154.9\nR002,2018-01-11,1.6,0.5,28800.0\nR002,2018-01-12,5.2,4.9,14312.71\nR002,2018-01-13,3.8,11.6,630.58\nR002,2018-01-14,4.2,0.0,28800.0\nR002,2018-01-15,3.0,6.3,14429.53\nR002,2018-01-16,1.6,0.9,31840.49\nR002,2018-01-17,3.4,1.6,6478.83\nR002,2018-01-18,5.1,0.3,26602.66\nR002,2018-01-19,5.2,14.3,0.0\nR002,2018-01-20,4.8,1.3,30410.04\nR002,2018-01-21,6.4,1.6,10165.34\nR002,2018-01-22,5.9,0.0,21515.17\nR002,2018-01-23,6.7,2.1,11076.28\nR002,2018-01-24,2.6,9.9,14132.27\nR002,2018-01-25,1.0,0.0,22136.33\nR002,2018-01-26,3.2,0.0,28628.88\nR002,2018-01-27,1.8,0.5,32400.0\nR002,2018-01-28,2.1,0.0,32400.0\nR002,2018-01-29,2.0,6.8,2324.24\nR002,2018-01-30,3.2,8.6,2082.65\nR002,2018-01-31,2.6,19.1,26378.87\nR002,2018-02-01,-0.7,0.4,32719.43\nR002,2018-02-02,3.3,0.6,32496.62\nR002,2018-02-03,6.9,5.8,3808.62\nR002,2018-02-04,7.7,1.9,2875.81\nR002,2018-02-05,7.0,12.9,3227.74\nR002,2018-02-06,2.7,0.0,32495.23\nR002,2018-02-07,3.8,0.0,34080.84\nR002,2018-02-08,7.1,0.1,32950.98\nR002,2018-02-09,7.0,6.2,15782.66\nR002,2018-02-10,5.4,0.0,34621.18\nR002,2018-02-11,8.6,0.1,30041.16\nR002,2018-02-12,6.4,21.8,11084.38\nR002,2018-02-13,3.9,7.0,25695.29\nR002,2018-02-14,3.7,0.6,35235.17\nR002,2018-02-15,5.8,0.7,31618.63\nR002,2018-02-16,8.6,0.0,32400.0\nR002,2018-02-17,8.2,0.2,35527.06\nR002,2018-02-18,6.6,0.0,33747.48\nR002,2018-02-19,4.5,0.0,34093.94\nR002,2018-02-20,5.1,0.3,24042.19\nR002,2018-02-21,7.1,0.4,28195.87\nR002,2018-02-22,8.0,0.0,37821.43\nR002,2018-02-23,7.4,6.6,34260.89\nR002,2018-02-24,6.0,3.6,33638.45\nR002,2018-02-25,6.9,0.3,29939.6\nR002,2018-02-26,5.0,4.4,28544.31\nR002,2018-02-27,7.1,1.9,26335.89\nR002,2018-02-28,9.1,0.0,38166.96\nR002,2018-03-01,3.6,20.9,10749.18\nR002,2018-03-02,0.2,0.1,36000.0\nR002,2018-03-03,7.6,5.0,12033.12\nR002,2018-03-04,8.1,19.6,30083.43\nR002,2018-03-05,4.9,14.8,23822.26\nR002,2018-03-06,4.7,0.0,39600.0\nR002,2018-03-07,11.3,3.7,24325.5\nR002,2018-03-08,11.0,7.8,17634.84\nR002,2018-03-09,9.8,2.6,33124.52\nR002,2018-03-10,4.7,9.3,27788.57\nR002,2018-03-11,5.2,7.8,30533.28\nR002,2018-03-12,5.7,0.4,39600.0\nR002,2018-03-13,9.8,0.0,39600.0\nR002,2018-03-14,10.0,8.0,22614.85\nR002,2018-03-15,8.4,5.9,33844.36\nR002,2018-03-16,8.2,5.6,37520.74\nR002,2018-03-17,9.3,0.0,36685.79\nR002,2018-03-18,15.0,0.2,28363.54\nR002,2018-03-19,15.1,1.2,36859.15\nR002,2018-03-20,13.9,4.4,36000.0\nR002,2018-03-21,9.7,6.2,39600.0\nR002,2018-03-22,7.2,17.5,1162.7\nR002,2018-03-23,10.5,5.6,10800.0\nR002,2018-03-24,11.3,22.1,0.0\nR002,2018-03-25,7.1,6.2,26395.58\nR002,2018-03-26,9.8,1.7,36846.82\nR002,2018-03-27,11.1,1.5,32074.6\nR002,2018-03-28,10.6,7.8,34485.5\nR002,2018-03-29,12.0,10.9,9499.17\nR002,2018-03-30,9.9,12.5,1740.96\nR002,2018-03-31,9.4,0.9,40659.53\nR002,2018-04-01,12.7,0.0,44178.23\nR002,2018-04-02,14.2,10.8,37798.66\nR002,2018-04-03,7.3,6.0,43908.28\nR002,2018-04-04,7.6,0.0,45678.94\nR002,2018-04-05,10.4,0.0,45937.6\nR002,2018-04-06,13.1,0.0,46009.06\nR002,2018-04-07,15.0,0.0,46080.44\nR002,2018-04-08,15.2,0.0,46151.76\nR002,2018-04-09,17.3,0.0,45863.25\nR002,2018-04-10,18.7,0.0,46294.13\nR002,2018-04-11,17.7,0.0,46365.17\nR002,2018-04-12,16.7,1.1,45772.87\nR002,2018-04-13,14.7,2.0,38830.61\nR002,2018-04-14,12.1,0.0,44614.61\nR002,2018-04-15,10.7,0.0,46648.61\nR002,2018-04-16,11.0,0.0,46719.67\nR002,2018-04-17,12.3,0.0,46790.83\nR002,2018-04-18,14.1,0.0,46800.0\nR002,2018-04-19,11.0,7.8,34572.68\nR002,2018-04-20,8.5,13.9,23123.27\nR002,2018-04-21,10.2,1.7,43200.0\nR002,2018-04-22,10.0,1.9,44177.13\nR002,2018-04-23,10.6,0.2,46800.0\nR002,2018-04-24,11.2,0.0,46800.0\nR002,2018-04-25,14.3,0.0,47085.65\nR002,2018-04-26,17.1,0.0,47374.12\nR002,2018-04-27,18.9,0.3,47664.62\nR002,2018-04-28,18.2,0.4,45600.43\nR002,2018-04-29,17.1,0.0,46800.0\nR002,2018-04-30,17.8,0.0,48548.96\nR002,2018-05-01,19.4,1.3,48512.94\nR002,2018-05-02,19.0,4.2,48477.83\nR002,2018-05-03,19.8,5.5,48780.45\nR002,2018-05-04,20.2,6.3,47735.33\nR002,2018-05-05,19.0,5.1,25596.14\nR002,2018-05-06,18.3,9.3,47668.03\nR002,2018-05-07,17.8,13.3,34740.88\nR002,2018-05-08,15.6,15.3,10542.39\nR002,2018-05-09,14.9,0.7,9288.41\nR002,2018-05-10,15.1,8.8,10442.52\nR002,2018-05-11,14.1,9.3,20843.5\nR002,2018-05-12,13.3,4.0,38466.74\nR002,2018-05-13,12.8,9.3,3231.74\nR002,2018-05-14,15.1,2.4,26076.57\nR002,2018-05-15,16.7,4.7,42863.09\nR002,2018-05-16,18.4,2.8,46775.59\nR002,2018-05-17,19.3,3.7,49176.14\nR002,2018-05-18,20.1,0.3,47533.5\nR002,2018-05-19,20.9,1.1,49204.85\nR002,2018-05-20,20.3,2.4,50269.84\nR002,2018-05-21,19.0,9.8,41511.6\nR002,2018-05-22,19.4,3.9,48085.25\nR002,2018-05-23,20.4,11.1,47458.22\nR002,2018-05-24,19.1,16.1,42335.94\nR002,2018-05-25,20.1,11.0,38436.38\nR002,2018-05-26,19.7,6.5,36056.65\nR002,2018-05-27,21.8,2.3,45783.78\nR002,2018-05-28,22.9,0.8,49637.74\nR002,2018-05-29,19.8,0.0,17822.03\nR002,2018-05-30,17.9,10.4,7535.16\nR002,2018-05-31,18.8,13.1,37730.4\nR002,2018-06-01,18.5,4.9,45395.24\nR002,2018-06-02,18.4,1.0,44544.59\nR002,2018-06-03,18.7,1.2,47163.89\nR002,2018-06-04,19.0,2.0,50273.32\nR002,2018-06-05,19.0,5.5,48410.35\nR002,2018-06-06,20.0,0.6,52483.41\nR002,2018-06-07,20.6,0.0,51371.94\nR002,2018-06-08,19.9,2.7,51355.54\nR002,2018-06-09,19.6,0.0,50420.48\nR002,2018-06-10,19.6,0.4,52790.21\nR002,2018-06-11,19.0,2.0,51677.75\nR002,2018-06-12,19.8,0.8,53499.17\nR002,2018-06-13,21.1,0.1,53121.28\nR002,2018-06-14,23.8,0.0,53793.15\nR002,2018-06-15,23.9,5.4,52732.84\nR002,2018-06-16,20.6,12.9,7966.67\nR002,2018-06-17,19.1,13.7,50699.54\nR002,2018-06-18,19.3,8.1,43527.13\nR002,2018-06-19,19.9,8.2,37829.42\nR002,2018-06-20,21.6,0.2,53569.73\nR002,2018-06-21,22.9,0.0,53928.94\nR002,2018-06-22,23.2,4.8,49887.95\nR002,2018-06-23,20.7,17.0,20774.31\nR002,2018-06-24,21.8,4.2,49875.71\nR002,2018-06-25,23.0,1.8,53964.27\nR002,2018-06-26,23.8,8.3,53755.15\nR002,2018-06-27,24.4,5.0,53968.9\nR002,2018-06-28,26.6,0.2,52931.3\nR002,2018-06-29,27.1,2.2,53964.27\nR002,2018-06-30,25.2,3.0,50816.99\nR002,2018-07-01,25.3,3.2,50860.93\nR002,2018-07-02,24.8,1.1,51629.19\nR002,2018-07-03,24.6,2.8,47919.64\nR002,2018-07-04,23.1,6.4,1238.6\nR002,2018-07-05,22.8,8.4,12032.23\nR002,2018-07-06,20.7,16.9,3155.03\nR002,2018-07-07,22.1,0.8,47428.49\nR002,2018-07-08,22.9,1.0,53762.72\nR002,2018-07-09,24.2,0.7,53799.44\nR002,2018-07-10,27.2,0.0,53770.95\nR002,2018-07-11,26.7,2.1,53059.4\nR002,2018-07-12,26.0,1.1,53428.42\nR002,2018-07-13,25.1,2.7,47261.06\nR002,2018-07-14,23.2,1.9,47939.72\nR002,2018-07-15,21.5,11.6,34615.97\nR002,2018-07-16,19.8,3.0,49558.82\nR002,2018-07-17,20.0,1.8,45453.98\nR002,2018-07-18,22.1,0.0,53469.88\nR002,2018-07-19,21.6,2.6,50548.94\nR002,2018-07-20,21.2,2.3,49306.86\nR002,2018-07-21,21.9,0.0,49116.36\nR002,2018-07-22,22.9,0.1,50693.26\nR002,2018-07-23,21.8,7.3,48849.71\nR002,2018-07-24,23.1,2.3,50076.13\nR002,2018-07-25,27.3,0.2,51881.92\nR002,2018-07-26,26.8,3.2,49814.92\nR002,2018-07-27,26.3,0.8,51460.24\nR002,2018-07-28,25.6,5.6,50727.92\nR002,2018-07-29,25.1,7.3,47479.61\nR002,2018-07-30,25.4,2.5,48283.83\nR002,2018-07-31,26.6,0.2,50068.1\nR002,2018-08-01,25.0,8.0,48480.63\nR002,2018-08-02,23.3,5.1,45792.89\nR002,2018-08-03,21.9,10.6,43589.49\nR002,2018-08-04,22.4,5.1,39969.25\nR002,2018-08-05,22.5,2.4,46295.55\nR002,2018-08-06,23.4,0.6,47913.86\nR002,2018-08-07,24.8,0.1,48292.77\nR002,2018-08-08,23.5,0.8,46718.52\nR002,2018-08-09,22.0,6.4,26264.75\nR002,2018-08-10,20.2,7.9,33205.08\nR002,2018-08-11,20.2,0.2,45905.3\nR002,2018-08-12,19.7,2.1,45413.87\nR002,2018-08-13,20.3,0.2,46800.0\nR002,2018-08-14,21.6,0.0,47476.47\nR002,2018-08-15,23.0,0.0,47172.18\nR002,2018-08-16,23.9,0.0,47202.46\nR002,2018-08-17,24.2,1.0,46900.63\nR002,2018-08-18,23.0,0.9,46800.0\nR002,2018-08-19,22.8,0.0,46800.0\nR002,2018-08-20,22.7,0.0,46800.0\nR002,2018-08-21,23.0,1.7,46800.0\nR002,2018-08-22,22.7,0.4,46800.0\nR002,2018-08-23,23.4,0.0,46800.0\nR002,2018-08-24,23.6,1.3,46800.0\nR002,2018-08-25,23.9,1.1,46800.0\nR002,2018-08-26,23.4,1.3,46482.6\nR002,2018-08-27,23.5,2.9,46735.84\nR002,2018-08-28,23.0,1.3,46634.04\nR002,2018-08-29,24.2,0.1,46532.0\nR002,2018-08-30,24.9,2.8,46429.83\nR002,2018-08-31,22.6,6.6,31283.33\nR002,2018-09-01,21.2,8.8,38470.68\nR002,2018-09-02,20.8,4.4,44630.1\nR002,2018-09-03,21.9,2.7,44466.57\nR002,2018-09-04,21.7,1.1,44408.6\nR002,2018-09-05,22.3,0.4,44459.39\nR002,2018-09-06,24.1,0.6,44168.53\nR002,2018-09-07,20.0,10.5,26779.83\nR002,2018-09-08,18.5,2.2,24484.93\nR002,2018-09-09,19.5,0.0,40886.04\nR002,2018-09-10,18.3,3.4,16695.27\nR002,2018-09-11,18.2,5.4,24467.58\nR002,2018-09-12,18.7,0.2,41874.53\nR002,2018-09-13,22.6,0.0,43180.16\nR002,2018-09-14,22.8,14.1,39512.8\nR002,2018-09-15,19.7,6.6,38709.98\nR002,2018-09-16,17.8,9.3,16612.53\nR002,2018-09-17,18.6,2.2,39600.0\nR002,2018-09-18,17.5,0.0,40843.21\nR002,2018-09-19,17.8,0.0,40876.99\nR002,2018-09-20,18.8,0.0,42707.17\nR002,2018-09-21,19.9,0.0,42639.27\nR002,2018-09-22,19.7,2.7,33734.15\nR002,2018-09-23,19.2,1.8,37306.82\nR002,2018-09-24,19.8,0.1,39600.0\nR002,2018-09-25,20.3,0.6,41722.64\nR002,2018-09-26,15.0,7.3,24430.69\nR002,2018-09-27,15.4,0.0,37166.16\nR002,2018-09-28,16.0,0.0,29451.85\nR002,2018-09-29,17.4,0.1,37551.01\nR002,2018-09-30,16.2,2.1,38829.03\nR002,2018-10-01,19.9,0.0,33464.94\nR002,2018-10-02,20.4,4.3,38700.21\nR002,2018-10-03,18.2,0.0,39117.2\nR002,2018-10-04,14.5,10.9,20754.12\nR002,2018-10-05,9.8,1.3,38735.72\nR002,2018-10-06,9.6,0.0,39404.29\nR002,2018-10-07,14.7,0.0,39027.63\nR002,2018-10-08,17.9,0.0,38302.3\nR002,2018-10-09,17.5,0.1,30606.58\nR002,2018-10-10,17.9,3.1,32762.52\nR002,2018-10-11,17.4,1.6,36805.94\nR002,2018-10-12,16.5,0.0,36859.58\nR002,2018-10-13,17.4,0.0,36913.33\nR002,2018-10-14,16.6,0.0,36640.63\nR002,2018-10-15,16.8,0.0,36366.87\nR002,2018-10-16,18.5,0.0,36091.92\nR002,2018-10-17,18.3,0.0,36143.84\nR002,2018-10-18,17.8,0.0,36000.0\nR002,2018-10-19,17.3,0.0,36000.0\nR002,2018-10-20,15.7,0.3,36000.0\nR002,2018-10-21,15.5,2.8,33775.87\nR002,2018-10-22,15.0,4.4,2130.21\nR002,2018-10-23,13.6,1.9,24600.32\nR002,2018-10-24,13.8,17.1,199.79\nR002,2018-10-25,11.6,50.0,0.0\nR002,2018-10-26,7.5,3.9,29937.57\nR002,2018-10-27,6.9,0.0,36000.0\nR002,2018-10-28,10.7,0.0,36000.0\nR002,2018-10-29,13.6,0.0,36000.0\nR002,2018-10-30,14.4,0.0,36000.0\nR002,2018-10-31,14.5,0.0,36000.0\nR002,2018-11-01,14.2,0.0,36000.0\nR002,2018-11-02,12.7,0.0,35951.96\nR002,2018-11-03,12.2,0.0,35872.6\nR002,2018-11-04,12.6,0.0,35792.79\nR002,2018-11-05,11.8,0.0,35710.22\nR002,2018-11-06,11.2,0.2,35554.3\nR002,2018-11-07,11.0,0.4,32793.18\nR002,2018-11-08,10.3,0.0,35246.12\nR002,2018-11-09,9.8,0.0,35094.21\nR002,2018-11-10,10.1,0.0,34944.0\nR002,2018-11-11,9.4,0.0,34795.68\nR002,2018-11-12,9.0,3.7,27050.61\nR002,2018-11-13,6.5,7.8,18376.64\nR002,2018-11-14,6.2,3.3,6830.44\nR002,2018-11-15,5.5,8.8,1818.44\nR002,2018-11-16,5.0,0.2,22820.29\nR002,2018-11-17,6.2,15.0,1790.66\nR002,2018-11-18,5.0,0.1,31869.01\nR002,2018-11-19,4.9,0.0,29656.54\nR002,2018-11-20,9.6,4.0,0.0\nR002,2018-11-21,6.2,0.0,30530.29\nR002,2018-11-22,7.4,0.4,19397.41\nR002,2018-11-23,8.0,2.3,30141.73\nR002,2018-11-24,6.2,1.2,30914.08\nR002,2018-11-25,5.8,0.0,30802.17\nR002,2018-11-26,6.0,0.0,29742.12\nR002,2018-11-27,8.8,27.2,17432.97\nR002,2018-11-28,11.0,34.3,13310.1\nR002,2018-11-29,9.0,36.9,0.0\nR002,2018-11-30,11.1,2.9,18514.09\nR002,2018-12-01,9.8,15.5,0.0\nR002,2018-12-02,8.6,5.2,23853.8\nR002,2018-12-03,7.2,0.0,29590.64\nR002,2018-12-04,7.8,0.0,25948.46\nR002,2018-12-05,10.8,2.3,16674.38\nR002,2018-12-06,8.9,2.0,22022.51\nR002,2018-12-07,6.8,12.4,127.14\nR002,2018-12-08,3.0,5.8,28290.34\nR002,2018-12-09,2.7,0.0,30414.92\nR002,2018-12-10,7.5,0.2,26382.93\nR002,2018-12-11,8.1,11.8,86.13\nR002,2018-12-12,5.4,9.5,1105.42\nR002,2018-12-13,3.4,4.8,19463.64\nR002,2018-12-14,1.8,2.6,29255.54\nR002,2018-12-15,3.8,0.6,26378.03\nR002,2018-12-16,6.6,0.5,29244.07\nR002,2018-12-17,8.0,0.0,26026.08\nR002,2018-12-18,7.6,0.0,30007.34\nR002,2018-12-19,7.0,0.3,19393.11\nR002,2018-12-20,8.8,0.2,26768.93\nR002,2018-12-21,7.0,3.1,23342.41\nR002,2018-12-22,5.8,5.7,11880.32\nR002,2018-12-23,6.1,9.7,8000.41\nR002,2018-12-24,3.1,9.3,27247.2\nR002,2018-12-25,3.7,17.9,0.0\nR002,2018-12-26,1.7,26.1,0.0\nR002,2018-12-27,0.2,13.8,19082.12\nR002,2018-12-28,-1.4,0.1,29453.98\nR002,2018-12-29,0.5,0.0,27108.47\nR002,2018-12-30,1.7,1.3,16715.44\nR002,2018-12-31,0.0,0.1,26065.59\nR002,2019-01-01,-0.2,0.4,31592.16\nR002,2019-01-02,-1.5,4.0,18000.0\nR002,2019-01-03,5.5,0.5,4752.19\nR002,2019-01-04,5.4,19.6,6003.79\nR002,2019-01-05,1.9,4.1,24618.49\nR002,2019-01-06,4.5,10.4,0.0\nR002,2019-01-07,2.6,14.9,17229.2\nR002,2019-01-08,-0.4,0.5,20912.39\nR002,2019-01-09,0.3,14.2,4461.76\nR002,2019-01-10,0.6,0.0,32145.04\nR002,2019-01-11,5.2,0.0,29105.89\nR002,2019-01-12,4.7,14.1,0.0\nR002,2019-01-13,3.9,18.4,0.0\nR002,2019-01-14,4.4,4.0,13539.52\nR002,2019-01-15,6.3,5.1,20357.11\nR002,2019-01-16,5.3,36.6,0.0\nR002,2019-01-17,0.4,4.3,29551.21\nR002,2019-01-18,-1.1,0.0,32400.0\nR002,2019-01-19,1.5,0.0,24609.87\nR002,2019-01-20,3.0,1.7,22233.48\nR002,2019-01-21,2.9,0.0,32400.0\nR002,2019-01-22,2.5,8.7,22204.27\nR002,2019-01-23,3.4,0.0,30651.24\nR002,2019-01-24,6.9,0.0,29341.51\nR002,2019-01-25,7.5,0.0,2015.19\nR002,2019-01-26,6.8,0.0,2804.35\nR002,2019-01-27,6.8,11.2,4491.17\nR002,2019-01-28,6.1,3.2,25904.36\nR002,2019-01-29,5.1,0.0,32400.0\nR002,2019-01-30,6.6,0.0,31730.12\nR002,2019-01-31,7.7,0.0,28861.09\nR002,2019-02-01,6.3,3.1,14546.34\nR002,2019-02-02,5.2,0.5,32029.18\nR002,2019-02-03,5.1,0.0,32987.29\nR002,2019-02-04,6.7,0.0,32624.1\nR002,2019-02-05,7.2,0.0,33591.65\nR002,2019-02-06,8.8,0.0,32700.96\nR002,2019-02-07,9.1,1.5,0.0\nR002,2019-02-08,6.9,0.7,24824.44\nR002,2019-02-09,6.3,3.6,4360.55\nR002,2019-02-10,5.8,8.4,1945.24\nR002,2019-02-11,5.2,0.1,33654.9\nR002,2019-02-12,5.3,0.0,34813.2\nR002,2019-02-13,7.9,0.0,31818.4\nR002,2019-02-14,5.9,0.0,25955.26\nR002,2019-02-15,4.4,0.0,12170.84\nR002,2019-02-16,3.8,2.7,24168.71\nR002,2019-02-17,3.6,1.9,33763.29\nR002,2019-02-18,4.1,6.3,11432.57\nR002,2019-02-19,1.8,0.0,37902.07\nR002,2019-02-20,2.4,0.0,38077.1\nR002,2019-02-21,4.8,0.0,36995.31\nR002,2019-02-22,2.7,2.7,35044.33\nR002,2019-02-23,5.6,1.4,21029.07\nR002,2019-02-24,5.0,0.4,38780.69\nR002,2019-02-25,3.1,0.6,27373.66\nR002,2019-02-26,2.0,7.9,3525.75\nR002,2019-02-27,1.4,14.8,119.78\nR002,2019-02-28,0.1,0.9,39427.92\nR002,2019-03-01,0.9,19.5,3540.66\nR002,2019-03-02,1.1,7.0,8143.77\nR002,2019-03-03,-0.2,2.7,39600.0\nR002,2019-03-04,-1.2,0.0,39600.0\nR002,2019-03-05,5.8,0.0,36678.7\nR002,2019-03-06,7.3,8.5,14111.96\nR002,2019-03-07,5.2,11.7,28091.62\nR002,2019-03-08,4.1,0.0,37136.86\nR002,2019-03-09,8.7,0.0,35534.61\nR002,2019-03-10,9.9,7.4,37270.57\nR002,2019-03-11,7.8,2.8,23901.59\nR002,2019-03-12,9.5,0.9,30092.64\nR002,2019-03-13,8.3,31.1,0.0\nR002,2019-03-14,8.5,2.5,31676.47\nR002,2019-03-15,9.4,2.1,29020.02\nR002,2019-03-16,7.6,15.1,5384.83\nR002,2019-03-17,4.5,12.6,9258.89\nR002,2019-03-18,3.0,0.5,39307.92\nR002,2019-03-19,5.3,0.0,40482.17\nR002,2019-03-20,9.9,0.0,41027.97\nR002,2019-03-21,8.0,0.2,39144.05\nR002,2019-03-22,5.7,2.6,36352.08\nR002,2019-03-23,1.8,19.1,474.61\nR002,2019-03-24,-0.0,1.2,40349.25\nR002,2019-03-25,2.6,0.0,40540.1\nR002,2019-03-26,6.0,0.0,43082.97\nR002,2019-03-27,7.4,0.1,42521.05\nR002,2019-03-28,3.5,9.2,35089.76\nR002,2019-03-29,1.7,4.2,38635.22\nR002,2019-03-30,2.3,0.0,44585.03\nR002,2019-03-31,9.4,0.0,44727.3\nR002,2019-04-01,8.6,3.2,33309.17\nR002,2019-04-02,5.9,11.5,6967.71\nR002,2019-04-03,5.6,5.3,35507.85\nR002,2019-04-04,6.5,2.7,40876.66\nR002,2019-04-05,8.8,0.0,44715.84\nR002,2019-04-06,11.9,0.0,45993.69\nR002,2019-04-07,13.7,0.0,46064.88\nR002,2019-04-08,14.7,0.0,45900.63\nR002,2019-04-09,15.7,1.9,46206.93\nR002,2019-04-10,13.6,5.6,36029.49\nR002,2019-04-11,11.7,4.3,41692.14\nR002,2019-04-12,11.0,9.5,29108.4\nR002,2019-04-13,12.2,4.1,34596.22\nR002,2019-04-14,14.6,0.0,34785.85\nR002,2019-04-15,12.9,10.6,23730.56\nR002,2019-04-16,11.2,7.8,30245.74\nR002,2019-04-17,11.5,4.1,36261.02\nR002,2019-04-18,9.0,10.2,0.0\nR002,2019-04-19,7.9,11.6,21498.19\nR002,2019-04-20,7.5,5.7,23157.98\nR002,2019-04-21,7.4,3.9,36116.63\nR002,2019-04-22,8.5,2.4,25938.23\nR002,2019-04-23,9.7,3.9,32280.14\nR002,2019-04-24,10.9,0.1,45828.97\nR002,2019-04-25,12.0,0.3,47094.45\nR002,2019-04-26,13.5,0.7,47383.18\nR002,2019-04-27,14.4,0.6,47674.11\nR002,2019-04-28,15.4,0.1,47634.28\nR002,2019-04-29,15.1,0.3,47261.74\nR002,2019-04-30,17.2,0.7,46800.0\nR002,2019-05-01,19.0,0.0,47519.5\nR002,2019-05-02,17.2,6.9,36010.52\nR002,2019-05-03,11.9,15.0,22734.95\nR002,2019-05-04,12.6,1.9,47072.03\nR002,2019-05-05,14.7,0.0,48095.24\nR002,2019-05-06,16.3,1.4,41252.27\nR002,2019-05-07,19.3,0.9,49342.52\nR002,2019-05-08,16.5,14.3,44118.42\nR002,2019-05-09,14.6,11.7,31564.84\nR002,2019-05-10,15.0,3.6,43134.6\nR002,2019-05-11,15.7,0.9,45862.65\nR002,2019-05-12,14.5,13.0,34734.97\nR002,2019-05-13,15.2,2.3,45427.03\nR002,2019-05-14,16.6,0.0,50371.78\nR002,2019-05-15,19.4,0.0,50400.0\nR002,2019-05-16,22.6,0.0,50460.32\nR002,2019-05-17,23.7,0.2,50431.38\nR002,2019-05-18,24.4,0.7,50400.0\nR002,2019-05-19,20.5,12.7,44049.92\nR002,2019-05-20,17.8,8.2,17328.2\nR002,2019-05-21,18.9,2.3,47883.83\nR002,2019-05-22,17.1,7.1,34414.14\nR002,2019-05-23,16.7,1.7,50620.67\nR002,2019-05-24,18.8,0.0,48926.88\nR002,2019-05-25,17.9,13.5,34403.33\nR002,2019-05-26,16.4,1.8,44613.35\nR002,2019-05-27,17.2,2.2,41731.09\nR002,2019-05-28,18.6,0.2,49229.73\nR002,2019-05-29,21.5,0.0,51910.21\nR002,2019-05-30,23.1,0.0,51889.04\nR002,2019-05-31,23.2,0.4,51506.82\nR002,2019-06-01,23.1,2.0,51848.33\nR002,2019-06-02,24.6,1.9,50400.0\nR002,2019-06-03,23.9,5.2,43744.73\nR002,2019-06-04,23.1,2.9,33651.69\nR002,2019-06-05,22.6,7.2,45325.53\nR002,2019-06-06,22.8,1.6,51392.8\nR002,2019-06-07,22.2,4.4,48792.06\nR002,2019-06-08,22.0,1.8,51726.28\nR002,2019-06-09,22.4,0.3,52229.18\nR002,2019-06-10,24.3,1.1,52795.53\nR002,2019-06-11,24.5,5.1,52782.44\nR002,2019-06-12,22.8,12.6,52402.99\nR002,2019-06-13,20.7,5.7,31375.54\nR002,2019-06-14,21.8,4.0,38769.9\nR002,2019-06-15,23.3,3.2,48941.51\nR002,2019-06-16,23.5,4.0,53826.06\nR002,2019-06-17,23.1,4.1,53448.57\nR002,2019-06-18,22.9,1.6,53439.17\nR002,2019-06-19,23.4,0.4,52327.14\nR002,2019-06-20,23.6,0.6,48753.2\nR002,2019-06-21,22.8,1.5,33045.9\nR002,2019-06-22,21.8,7.6,49447.94\nR002,2019-06-23,22.2,9.0,53919.49\nR002,2019-06-24,22.5,9.0,49342.03\nR002,2019-06-25,24.6,0.7,49103.73\nR002,2019-06-26,24.3,2.2,53020.73\nR002,2019-06-27,24.2,2.3,52284.18\nR002,2019-06-28,22.5,3.9,35176.02\nR002,2019-06-29,19.8,6.6,36856.74\nR002,2019-06-30,15.2,8.8,48698.58\nR002,2019-07-01,16.3,0.4,53949.56\nR002,2019-07-02,21.8,0.0,53939.05\nR002,2019-07-03,23.9,0.0,53926.44\nR002,2019-07-04,22.9,2.9,53401.92\nR002,2019-07-05,21.6,6.0,48547.68\nR002,2019-07-06,21.3,3.0,53131.78\nR002,2019-07-07,21.8,0.2,51746.15\nR002,2019-07-08,22.3,3.3,52672.95\nR002,2019-07-09,21.5,0.8,49789.84\nR002,2019-07-10,21.6,0.0,51470.15\nR002,2019-07-11,20.5,5.8,47943.83\nR002,2019-07-12,19.3,41.1,27733.82\nR002,2019-07-13,18.0,1.9,50747.35\nR002,2019-07-14,19.1,0.0,42496.74\nR002,2019-07-15,18.3,12.2,36845.66\nR002,2019-07-16,19.0,4.8,48715.37\nR002,2019-07-17,20.8,0.0,52762.7\nR002,2019-07-18,20.9,19.7,45982.11\nR002,2019-07-19,21.2,13.5,47901.4\nR002,2019-07-20,23.7,0.7,47679.1\nR002,2019-07-21,23.1,5.6,35612.72\nR002,2019-07-22,22.7,1.2,33560.61\nR002,2019-07-23,22.3,7.1,26174.93\nR002,2019-07-24,22.5,7.0,25353.94\nR002,2019-07-25,21.2,16.5,33943.85\nR002,2019-07-26,20.1,0.4,48625.27\nR002,2019-07-27,20.3,0.0,50769.58\nR002,2019-07-28,21.4,1.1,47012.49\nR002,2019-07-29,22.4,0.9,45641.16\nR002,2019-07-30,24.5,0.9,51013.0\nR002,2019-07-31,23.6,0.8,49711.52\nR002,2019-08-01,22.8,0.2,45398.23\nR002,2019-08-02,22.2,1.5,46581.96\nR002,2019-08-03,21.5,0.0,47816.78\nR002,2019-08-04,21.2,6.1,42776.32\nR002,2019-08-05,19.2,1.3,47256.36\nR002,2019-08-06,18.8,0.0,48288.95\nR002,2019-08-07,19.8,0.0,47946.45\nR002,2019-08-08,21.4,0.0,48469.28\nR002,2019-08-09,23.4,0.0,48338.69\nR002,2019-08-10,22.7,0.2,48027.46\nR002,2019-08-11,22.4,0.0,47044.13\nR002,2019-08-12,21.8,0.0,47746.35\nR002,2019-08-13,23.1,0.0,47774.93\nR002,2019-08-14,25.6,0.0,47469.02\nR002,2019-08-15,27.6,0.0,47498.64\nR002,2019-08-16,25.9,5.6,42335.04\nR002,2019-08-17,22.1,33.2,17190.73\nR002,2019-08-18,23.1,3.5,35984.3\nR002,2019-08-19,24.5,0.7,43418.77\nR002,2019-08-20,26.3,0.3,46800.0\nR002,2019-08-21,27.3,2.7,46800.0\nR002,2019-08-22,25.0,2.8,43570.46\nR002,2019-08-23,24.7,5.2,36484.92\nR002,2019-08-24,23.2,7.2,16194.44\nR002,2019-08-25,23.1,11.1,36948.86\nR002,2019-08-26,23.0,0.0,45873.69\nR002,2019-08-27,22.7,0.8,46767.59\nR002,2019-08-28,21.9,0.0,46667.11\nR002,2019-08-29,22.4,0.1,46565.28\nR002,2019-08-30,22.6,2.9,46462.22\nR002,2019-08-31,20.7,0.1,44653.15\nR002,2019-09-01,20.6,0.9,46177.2\nR002,2019-09-02,20.5,2.3,44939.53\nR002,2019-09-03,20.3,0.0,45631.01\nR002,2019-09-04,20.3,1.0,45039.33\nR002,2019-09-05,20.8,2.2,43920.13\nR002,2019-09-06,19.7,9.0,41105.35\nR002,2019-09-07,19.0,3.3,38186.39\nR002,2019-09-08,18.5,0.0,40626.38\nR002,2019-09-09,18.8,0.1,43044.97\nR002,2019-09-10,18.5,3.0,38270.77\nR002,2019-09-11,18.2,1.0,39610.26\nR002,2019-09-12,18.6,0.0,43200.0\nR002,2019-09-13,18.5,0.0,42900.62\nR002,2019-09-14,18.9,0.0,43129.64\nR002,2019-09-15,18.1,0.2,42319.19\nR002,2019-09-16,18.2,0.0,42995.15\nR002,2019-09-17,17.5,4.8,32372.85\nR002,2019-09-18,18.8,0.0,42860.27\nR002,2019-09-19,18.5,0.0,42792.6\nR002,2019-09-20,17.7,4.4,39400.71\nR002,2019-09-21,11.7,10.8,36107.92\nR002,2019-09-22,10.0,0.0,42244.43\nR002,2019-09-23,13.2,0.0,42279.56\nR002,2019-09-24,14.1,0.0,33285.69\nR002,2019-09-25,16.8,2.1,6878.75\nR002,2019-09-26,16.2,11.4,13212.51\nR002,2019-09-27,17.4,3.7,34082.95\nR002,2019-09-28,16.4,27.9,25412.85\nR002,2019-09-29,15.5,0.0,38053.49\nR002,2019-09-30,15.9,0.0,40181.59\nR002,2019-10-01,17.0,0.2,40508.17\nR002,2019-10-02,18.4,0.0,40357.18\nR002,2019-10-03,20.2,0.9,37160.61\nR002,2019-10-04,20.6,0.3,39576.32\nR002,2019-10-05,21.4,0.0,38961.46\nR002,2019-10-06,16.6,22.3,22266.52\nR002,2019-10-07,12.8,2.6,38510.29\nR002,2019-10-08,15.3,6.7,32760.63\nR002,2019-10-09,14.7,9.6,31429.23\nR002,2019-10-10,16.0,0.0,37490.88\nR002,2019-10-11,17.5,0.7,34102.36\nR002,2019-10-12,14.2,5.5,33797.72\nR002,2019-10-13,14.5,0.0,36575.11\nR002,2019-10-14,17.3,0.0,36629.46\nR002,2019-10-15,18.1,0.0,36356.7\nR002,2019-10-16,17.8,0.0,36082.54\nR002,2019-10-17,18.2,0.0,36134.92\nR002,2019-10-18,17.7,0.7,36000.0\nR002,2019-10-19,17.3,1.3,36000.0\nR002,2019-10-20,16.0,4.6,36000.0\nR002,2019-10-21,15.5,0.0,36000.0\nR002,2019-10-22,15.6,0.0,36000.0\nR002,2019-10-23,14.8,0.0,32400.0\nR002,2019-10-24,14.7,0.0,36000.0\nR002,2019-10-25,14.4,0.0,36000.0\nR002,2019-10-26,13.7,0.4,36000.0\nR002,2019-10-27,13.2,0.0,36000.0\nR002,2019-10-28,13.1,0.0,36000.0\nR002,2019-10-29,13.3,0.0,36000.0\nR002,2019-10-30,11.7,13.9,10846.7\nR002,2019-10-31,9.1,5.1,1163.09\nR002,2019-11-01,8.3,5.7,2741.68\nR002,2019-11-02,6.9,0.3,34447.06\nR002,2019-11-03,7.3,0.0,35889.76\nR002,2019-11-04,10.0,0.0,35809.42\nR002,2019-11-05,13.1,0.4,31430.46\nR002,2019-11-06,14.7,0.8,35583.13\nR002,2019-11-07,14.8,0.0,35427.5\nR002,2019-11-08,15.3,0.0,35273.76\nR002,2019-11-09,14.9,0.0,35122.11\nR002,2019-11-10,14.2,0.0,34972.73\nR002,2019-11-11,14.9,0.0,34825.82\nR002,2019-11-12,13.4,0.0,34681.59\nR002,2019-11-13,12.5,0.0,33736.8\nR002,2019-11-14,14.1,0.0,34401.99\nR002,2019-11-15,15.3,0.0,34266.86\nR002,2019-11-16,12.9,0.0,34133.56\nR002,2019-11-17,11.1,0.0,34001.75\nR002,2019-11-18,10.6,0.0,33691.4\nR002,2019-11-19,9.6,0.0,33322.98\nR002,2019-11-20,8.1,1.0,31534.38\nR002,2019-11-21,7.4,11.2,8056.68\nR002,2019-11-22,6.7,0.0,32217.56\nR002,2019-11-23,7.8,0.0,32217.61\nR002,2019-11-24,7.8,0.0,32206.13\nR002,2019-11-25,8.8,0.0,30935.78\nR002,2019-11-26,9.8,0.0,29067.16\nR002,2019-11-27,11.3,4.3,0.0\nR002,2019-11-28,7.9,12.3,13793.3\nR002,2019-11-29,8.8,1.4,22786.67\nR002,2019-11-30,8.7,20.8,2579.22\nR002,2019-12-01,6.1,17.3,0.0\nR002,2019-12-02,3.2,0.7,28962.09\nR002,2019-12-03,2.9,3.3,12032.94\nR002,2019-12-04,0.8,16.1,20696.86\nR002,2019-12-05,-4.4,0.0,30379.26\nR002,2019-12-06,-1.5,0.0,30390.04\nR002,2019-12-07,1.0,0.0,30399.26\nR002,2019-12-08,2.5,0.0,29636.8\nR002,2019-12-09,4.7,0.0,29255.78\nR002,2019-12-10,3.4,0.0,30417.4\nR002,2019-12-11,2.2,0.0,30420.33\nR002,2019-12-12,3.5,0.0,30421.78\nR002,2019-12-13,5.7,0.5,24136.66\nR002,2019-12-14,9.3,0.2,27499.41\nR002,2019-12-15,10.2,8.1,23279.51\nR002,2019-12-16,8.2,3.2,27398.88\nR002,2019-12-17,6.1,0.0,25620.48\nR002,2019-12-18,7.1,0.0,30789.17\nR002,2019-12-19,8.9,0.0,30780.57\nR002,2019-12-20,7.7,0.0,30770.54\nR002,2019-12-21,8.3,0.0,29313.18\nR002,2019-12-22,9.8,0.0,28456.18\nR002,2019-12-23,12.2,0.0,29557.86\nR002,2019-12-24,12.3,0.0,22623.81\nR002,2019-12-25,10.8,1.9,0.0\nR002,2019-12-26,8.9,0.3,26297.47\nR002,2019-12-27,6.9,11.5,420.55\nR002,2019-12-28,4.1,5.8,28800.0\nR002,2019-12-29,3.8,0.0,29991.16\nR002,2019-12-30,5.6,0.4,26596.44\nR002,2019-12-31,4.0,0.0,30169.45\nR002,2020-01-01,4.4,7.3,25700.36\nR002,2020-01-02,2.3,1.7,30338.81\nR002,2020-01-03,-0.1,0.0,16973.42\nR002,2020-01-04,1.4,0.0,31750.72\nR002,2020-01-05,3.0,0.0,30015.0\nR002,2020-01-06,3.0,0.0,26611.42\nR002,2020-01-07,5.4,5.6,0.0\nR002,2020-01-08,5.5,0.2,28300.61\nR002,2020-01-09,4.2,8.8,12770.99\nR002,2020-01-10,2.2,2.1,31009.04\nR002,2020-01-11,2.9,2.1,27568.28\nR002,2020-01-12,3.1,11.7,206.16\nR002,2020-01-13,-0.2,0.0,32333.98\nR002,2020-01-14,3.6,0.0,32400.0\nR002,2020-01-15,2.9,0.0,32400.0\nR002,2020-01-16,2.5,0.0,32400.0\nR002,2020-01-17,2.6,0.0,32400.0\nR002,2020-01-18,1.6,0.3,32400.0\nR002,2020-01-19,1.7,0.0,32400.0\nR002,2020-01-20,1.9,0.0,32400.0\nR002,2020-01-21,1.7,0.0,32400.0\nR002,2020-01-22,2.9,5.2,15680.94\nR002,2020-01-23,1.0,32.2,0.0\nR002,2020-01-24,-5.1,0.0,32400.0\nR002,2020-01-25,0.0,0.0,23199.46\nR002,2020-01-26,2.7,0.0,31634.03\nR002,2020-01-27,4.6,0.0,32400.0\nR002,2020-01-28,7.8,0.0,25030.1\nR002,2020-01-29,7.2,8.0,3259.76\nR002,2020-01-30,6.4,11.2,131.37\nR002,2020-01-31,3.9,27.6,0.0\nR002,2020-02-01,1.2,0.4,32400.0\nR002,2020-02-02,4.2,6.7,5661.47\nR002,2020-02-03,6.9,34.9,0.0\nR002,2020-02-04,3.8,35.0,25387.03\nR002,2020-02-05,4.5,7.7,0.0\nR002,2020-02-06,9.5,18.1,0.0\nR002,2020-02-07,3.2,28.9,4069.63\nR002,2020-02-08,-3.5,6.1,7567.37\nR002,2020-02-09,-10.8,0.0,30601.99\nR002,2020-02-10,-9.4,0.0,35408.44\nR002,2020-02-11,-4.9,0.0,33869.92\nR002,2020-02-12,-0.4,10.6,6396.59\nR002,2020-02-13,-0.5,7.3,22072.3\nR002,2020-02-14,-0.7,1.6,34909.18\nR002,2020-02-15,1.7,0.0,26439.29\nR002,2020-02-16,4.1,0.0,35624.23\nR002,2020-02-17,3.1,0.0,37520.59\nR002,2020-02-18,5.4,0.0,37692.63\nR002,2020-02-19,8.2,0.0,37864.93\nR002,2020-02-20,6.1,0.0,32313.5\nR002,2020-02-21,6.4,0.0,34651.42\nR002,2020-02-22,6.7,0.3,30218.83\nR002,2020-02-23,5.2,4.9,24577.09\nR002,2020-02-24,5.8,16.6,10333.71\nR002,2020-02-25,1.7,13.0,38901.93\nR002,2020-02-26,3.7,0.0,39078.02\nR002,2020-02-27,11.1,0.0,39255.32\nR002,2020-02-28,6.5,20.6,12612.9\nR002,2020-02-29,5.6,6.5,22460.07\nR002,2020-03-01,5.2,12.1,19564.23\nR002,2020-03-02,5.8,0.0,39600.0\nR002,2020-03-03,8.9,0.0,39600.0\nR002,2020-03-04,12.0,0.0,39600.0\nR002,2020-03-05,12.8,0.0,39600.0\nR002,2020-03-06,13.1,0.0,28800.0\nR002,2020-03-07,13.0,0.0,37331.4\nR002,2020-03-08,11.4,0.0,39600.0\nR002,2020-03-09,12.7,0.0,39600.0\nR002,2020-03-10,14.1,0.0,38871.19\nR002,2020-03-11,12.8,0.4,39600.0\nR002,2020-03-12,9.2,1.0,39600.0\nR002,2020-03-13,11.4,0.0,39606.96\nR002,2020-03-14,11.6,5.8,24683.88\nR002,2020-03-15,7.6,18.6,1362.59\nR002,2020-03-16,1.7,12.0,13813.81\nR002,2020-03-17,0.0,4.4,14775.91\nR002,2020-03-18,3.5,0.5,27944.02\nR002,2020-03-19,5.9,2.0,17222.14\nR002,2020-03-20,7.4,0.0,33072.17\nR002,2020-03-21,7.6,0.7,29865.2\nR002,2020-03-22,9.2,0.0,39587.93\nR002,2020-03-23,9.7,0.0,35989.58\nR002,2020-03-24,9.2,2.1,40914.01\nR002,2020-03-25,9.7,0.9,40228.06\nR002,2020-03-26,10.4,0.0,29705.37\nR002,2020-03-27,12.5,0.1,42233.41\nR002,2020-03-28,13.4,3.9,40358.99\nR002,2020-03-29,11.8,7.2,23839.7\nR002,2020-03-30,9.2,13.9,0.0\nR002,2020-03-31,9.7,3.6,35674.17\nR002,2020-04-01,9.0,2.6,35988.17\nR002,2020-04-02,8.3,2.8,44584.52\nR002,2020-04-03,9.0,0.1,39397.55\nR002,2020-04-04,10.8,2.0,36920.02\nR002,2020-04-05,10.4,0.0,35593.85\nR002,2020-04-06,9.7,0.0,46047.11\nR002,2020-04-07,7.9,0.0,46118.57\nR002,2020-04-08,9.4,0.0,46190.05\nR002,2020-04-09,10.3,0.0,46261.55\nR002,2020-04-10,10.2,0.0,46333.04\nR002,2020-04-11,8.3,3.3,44004.18\nR002,2020-04-12,6.9,2.9,41402.39\nR002,2020-04-13,7.3,0.0,46547.41\nR002,2020-04-14,11.2,0.0,46618.78\nR002,2020-04-15,12.3,7.5,27810.06\nR002,2020-04-16,7.9,12.5,34120.34\nR002,2020-04-17,10.4,0.0,46800.0\nR002,2020-04-18,13.0,0.0,43545.56\nR002,2020-04-19,12.7,0.0,46800.0\nR002,2020-04-20,13.2,6.0,38624.54\nR002,2020-04-21,9.0,5.7,42281.87\nR002,2020-04-22,7.2,0.0,26793.31\nR002,2020-04-23,7.3,0.6,46820.23\nR002,2020-04-24,7.9,0.5,47438.36\nR002,2020-04-25,10.5,0.1,45795.67\nR002,2020-04-26,10.5,0.3,47023.24\nR002,2020-04-27,8.8,0.7,39963.31\nR002,2020-04-28,9.1,0.2,46805.43\nR002,2020-04-29,11.6,0.0,48236.67\nR002,2020-04-30,15.0,0.0,46800.0\nR002,2020-05-01,15.1,2.4,41590.97\nR002,2020-05-02,13.8,11.1,788.29\nR002,2020-05-03,13.7,5.8,42664.44\nR002,2020-05-04,13.7,9.0,33516.42\nR002,2020-05-05,15.2,4.1,43941.15\nR002,2020-05-06,13.6,8.8,13327.74\nR002,2020-05-07,14.9,0.8,43728.51\nR002,2020-05-08,11.1,8.0,26561.86\nR002,2020-05-09,8.9,4.2,39390.67\nR002,2020-05-10,8.9,1.4,49578.5\nR002,2020-05-11,11.8,0.0,49210.17\nR002,2020-05-12,15.8,0.1,44991.77\nR002,2020-05-13,18.1,2.0,50355.79\nR002,2020-05-14,16.3,0.0,50340.87\nR002,2020-05-15,19.2,1.0,41966.16\nR002,2020-05-16,20.1,0.6,47649.28\nR002,2020-05-17,16.6,0.0,47933.87\nR002,2020-05-18,17.2,1.2,49264.08\nR002,2020-05-19,15.8,0.0,50400.0\nR002,2020-05-20,17.9,0.0,49634.03\nR002,2020-05-21,22.9,0.4,50400.0\nR002,2020-05-22,18.3,24.8,34022.19\nR002,2020-05-23,10.9,47.0,27314.53\nR002,2020-05-24,10.5,0.0,41586.15\nR002,2020-05-25,11.9,6.7,21963.66\nR002,2020-05-26,15.8,0.7,45795.81\nR002,2020-05-27,20.6,0.4,51575.98\nR002,2020-05-28,23.3,0.9,51193.78\nR002,2020-05-29,19.1,14.1,41938.96\nR002,2020-05-30,16.1,3.4,48273.44\nR002,2020-05-31,17.3,0.1,52216.06\nR002,2020-06-01,17.7,4.3,44608.09\nR002,2020-06-02,18.5,7.3,48801.87\nR002,2020-06-03,16.7,14.9,43556.16\nR002,2020-06-04,17.0,5.5,51454.61\nR002,2020-06-05,18.2,0.1,50527.98\nR002,2020-06-06,20.5,0.0,52477.78\nR002,2020-06-07,23.7,0.0,53558.35\nR002,2020-06-08,25.2,0.0,53178.6\nR002,2020-06-09,25.4,0.0,53164.63\nR002,2020-06-10,24.9,3.1,53517.48\nR002,2020-06-11,25.3,2.8,51671.33\nR002,2020-06-12,21.7,17.8,34422.55\nR002,2020-06-13,20.2,20.0,42955.85\nR002,2020-06-14,20.1,15.8,50126.24\nR002,2020-06-15,21.8,4.6,45352.88\nR002,2020-06-16,23.5,1.4,52609.55\nR002,2020-06-17,23.5,3.2,51148.73\nR002,2020-06-18,22.4,2.1,53899.63\nR002,2020-06-19,20.3,11.3,26551.39\nR002,2020-06-20,19.5,18.9,30309.31\nR002,2020-06-21,19.5,11.0,35027.53\nR002,2020-06-22,21.3,2.5,51107.97\nR002,2020-06-23,23.9,0.1,53957.1\nR002,2020-06-24,22.7,4.9,51190.73\nR002,2020-06-25,21.5,1.8,50357.19\nR002,2020-06-26,22.6,0.0,53818.43\nR002,2020-06-27,22.0,1.2,53965.46\nR002,2020-06-28,21.4,0.4,53962.5\nR002,2020-06-29,21.8,0.5,53957.49\nR002,2020-06-30,22.2,0.6,53380.81\nR002,2020-07-01,22.1,5.5,53746.14\nR002,2020-07-02,23.5,1.7,53929.85\nR002,2020-07-03,24.4,1.1,53916.37\nR002,2020-07-04,24.8,3.6,53748.8\nR002,2020-07-05,24.5,4.0,53882.88\nR002,2020-07-06,24.7,3.4,53027.77\nR002,2020-07-07,25.8,1.4,53840.57\nR002,2020-07-08,23.4,2.7,51472.09\nR002,2020-07-09,22.5,1.6,47477.03\nR002,2020-07-10,23.0,0.0,48480.67\nR002,2020-07-11,23.7,0.0,51852.02\nR002,2020-07-12,23.2,8.0,46960.85\nR002,2020-07-13,20.7,17.1,12145.88\nR002,2020-07-14,21.7,0.3,49751.31\nR002,2020-07-15,22.1,0.1,51464.35\nR002,2020-07-16,23.4,0.0,53474.96\nR002,2020-07-17,22.5,7.7,49852.5\nR002,2020-07-18,23.9,0.4,48600.78\nR002,2020-07-19,26.1,0.3,52090.62\nR002,2020-07-20,28.1,0.3,51698.12\nR002,2020-07-21,22.8,14.7,35413.65\nR002,2020-07-22,22.8,1.0,48082.65\nR002,2020-07-23,22.5,0.4,51233.37\nR002,2020-07-24,22.5,1.9,49189.96\nR002,2020-07-25,24.3,1.0,51196.56\nR002,2020-07-26,25.3,5.7,50774.56\nR002,2020-07-27,25.2,1.7,47955.83\nR002,2020-07-28,23.1,35.8,40951.59\nR002,2020-07-29,21.9,6.1,46748.24\nR002,2020-07-30,22.4,0.8,47231.3\nR002,2020-07-31,23.8,1.2,49059.87\nR002,2020-08-01,21.7,5.4,47117.74\nR002,2020-08-02,20.4,6.0,45270.16\nR002,2020-08-03,20.3,0.2,48200.95\nR002,2020-08-04,21.0,0.0,49630.45\nR002,2020-08-05,21.9,0.0,49348.73\nR002,2020-08-06,22.7,0.0,48728.81\nR002,2020-08-07,24.5,0.0,48453.29\nR002,2020-08-08,25.7,0.0,48331.52\nR002,2020-08-09,25.1,0.0,48020.39\nR002,2020-08-10,23.8,1.5,47711.08\nR002,2020-08-11,22.0,6.2,44878.96\nR002,2020-08-12,21.9,1.4,45334.99\nR002,2020-08-13,20.2,2.5,45455.27\nR002,2020-08-14,20.1,0.0,46178.41\nR002,2020-08-15,19.4,0.0,46800.0\nR002,2020-08-16,19.0,0.0,46216.3\nR002,2020-08-17,19.8,0.0,46918.44\nR002,2020-08-18,21.1,0.0,46949.8\nR002,2020-08-19,22.4,0.0,46800.0\nR002,2020-08-20,23.4,0.3,46800.0\nR002,2020-08-21,20.4,9.2,37828.33\nR002,2020-08-22,19.8,2.0,42850.56\nR002,2020-08-23,19.8,0.0,46800.0\nR002,2020-08-24,19.9,0.0,46800.0\nR002,2020-08-25,21.8,0.0,46800.0\nR002,2020-08-26,22.4,0.0,46794.61\nR002,2020-08-27,21.5,0.2,46691.92\nR002,2020-08-28,20.5,0.5,43213.93\nR002,2020-08-29,21.8,0.0,46483.85\nR002,2020-08-30,24.0,0.0,46378.68\nR002,2020-08-31,24.9,0.0,46272.89\nR002,2020-09-01,25.5,0.0,46166.59\nR002,2020-09-02,26.1,0.2,45300.92\nR002,2020-09-03,26.9,0.2,45029.8\nR002,2020-09-04,27.6,0.0,42096.48\nR002,2020-09-05,25.5,2.2,40988.84\nR002,2020-09-06,20.5,15.7,31676.65\nR002,2020-09-07,21.5,0.3,39389.42\nR002,2020-09-08,21.2,3.6,39992.93\nR002,2020-09-09,21.5,1.4,40734.92\nR002,2020-09-10,21.8,0.0,41044.55\nR002,2020-09-11,23.2,0.0,43207.18\nR002,2020-09-12,23.9,0.0,43200.0\nR002,2020-09-13,24.6,0.0,43145.92\nR002,2020-09-14,24.4,0.0,43078.67\nR002,2020-09-15,22.5,0.0,40123.18\nR002,2020-09-16,21.7,0.0,42943.81\nR002,2020-09-17,22.6,0.0,42876.2\nR002,2020-09-18,21.7,1.1,40787.72\nR002,2020-09-19,18.6,0.3,39277.78\nR002,2020-09-20,16.7,0.0,40291.82\nR002,2020-09-21,18.1,0.0,42604.44\nR002,2020-09-22,20.0,0.0,42271.44\nR002,2020-09-23,20.4,0.3,41988.36\nR002,2020-09-24,20.4,0.0,41386.8\nR002,2020-09-25,20.6,0.0,41422.58\nR002,2020-09-26,20.2,0.4,40501.85\nR002,2020-09-27,21.8,0.0,41175.93\nR002,2020-09-28,21.9,1.4,40573.77\nR002,2020-09-29,21.5,0.4,40068.02\nR002,2020-09-30,20.9,0.8,33911.59\nR002,2020-10-01,18.8,2.3,30348.29\nR002,2020-10-02,16.6,11.0,106.81\nR002,2020-10-03,16.7,4.4,30470.3\nR002,2020-10-04,16.7,1.9,38057.29\nR002,2020-10-05,16.6,0.4,37469.43\nR002,2020-10-06,16.2,0.0,37526.47\nR002,2020-10-07,17.2,0.0,38575.77\nR002,2020-10-08,18.5,0.0,38347.71\nR002,2020-10-09,20.9,0.0,37796.01\nR002,2020-10-10,21.1,0.0,37105.36\nR002,2020-10-11,19.3,0.0,36510.75\nR002,2020-10-12,19.2,0.0,36891.2\nR002,2020-10-13,20.8,0.0,36619.76\nR002,2020-10-14,21.1,0.0,36346.85\nR002,2020-10-15,19.6,0.0,36072.34\nR002,2020-10-16,18.2,0.0,36124.08\nR002,2020-10-17,18.6,0.0,36175.14\nR002,2020-10-18,20.0,0.0,36000.0\nR002,2020-10-19,16.7,14.0,30588.39\nR002,2020-10-20,12.1,0.0,36000.0\nR002,2020-10-21,12.1,0.0,36000.0\nR002,2020-10-22,13.1,0.0,30949.0\nR002,2020-10-23,15.4,0.0,36000.0\nR002,2020-10-24,16.5,0.0,36000.0\nR002,2020-10-25,17.0,0.0,36000.0\nR002,2020-10-26,17.2,0.0,36000.0\nR002,2020-10-27,16.9,1.3,35580.05\nR002,2020-10-28,19.4,0.0,36000.0\nR002,2020-10-29,19.1,0.0,36000.0\nR002,2020-10-30,17.9,0.9,34768.15\nR002,2020-10-31,14.8,10.9,35859.54\nR002,2020-11-01,14.1,14.3,27830.58\nR002,2020-11-02,11.2,10.1,35186.53\nR002,2020-11-03,10.5,0.0,35746.68\nR002,2020-11-04,13.3,0.0,35746.06\nR002,2020-11-05,14.0,0.0,34940.31\nR002,2020-11-06,12.1,2.2,34505.61\nR002,2020-11-07,9.7,0.0,30585.81\nR002,2020-11-08,9.2,0.0,35162.31\nR002,2020-11-09,11.7,0.0,35016.68\nR002,2020-11-10,10.6,0.0,34872.35\nR002,2020-11-11,9.7,0.7,33400.54\nR002,2020-11-12,9.3,0.8,32308.18\nR002,2020-11-13,8.0,0.0,34446.2\nR002,2020-11-14,9.3,0.0,34080.78\nR002,2020-11-15,8.3,0.0,31193.95\nR002,2020-11-16,7.2,5.7,7742.75\nR002,2020-11-17,6.1,5.0,30043.31\nR002,2020-11-18,3.7,2.0,27913.92\nR002,2020-11-19,3.0,4.9,26111.48\nR002,2020-11-20,2.8,7.7,24736.99\nR002,2020-11-21,4.8,4.0,29785.44\nR002,2020-11-22,3.6,18.2,13267.74\nR002,2020-11-23,-0.6,0.0,31453.14\nR002,2020-11-24,4.1,0.2,16755.93\nR002,2020-11-25,3.8,1.3,30959.46\nR002,2020-11-26,2.3,5.3,12234.55\nR002,2020-11-27,2.3,0.0,30633.66\nR002,2020-11-28,6.6,0.0,31404.94\nR002,2020-11-29,6.5,0.0,31087.88\nR002,2020-11-30,6.0,0.0,30763.03\nR002,2020-12-01,6.2,0.0,30054.65\nR002,2020-12-02,6.9,0.0,27632.18\nR002,2020-12-03,7.7,0.0,29598.2\nR002,2020-12-04,6.8,0.0,29608.96\nR002,2020-12-05,6.7,0.0,30385.07\nR002,2020-12-06,4.8,0.5,29626.34\nR002,2020-12-07,3.8,0.0,30402.65\nR002,2020-12-08,4.9,0.0,30409.26\nR002,2020-12-09,5.9,3.8,11509.37\nR002,2020-12-10,5.6,0.0,30418.18\nR002,2020-12-11,6.5,0.0,26569.49\nR002,2020-12-12,9.1,0.0,24610.52\nR002,2020-12-13,10.7,0.0,27463.76\nR002,2020-12-14,10.5,0.0,29232.56\nR002,2020-12-15,10.4,0.4,28800.0\nR002,2020-12-16,10.9,1.7,22611.47\nR002,2020-12-17,6.5,3.0,24401.8\nR002,2020-12-18,5.2,0.1,29614.39\nR002,2020-12-19,4.8,0.0,27888.93\nR002,2020-12-20,3.9,4.5,3535.39\nR002,2020-12-21,3.4,13.8,23923.3\nR002,2020-12-22,1.4,7.6,15722.29\nR002,2020-12-23,-0.6,1.6,22461.37\nR002,2020-12-24,-2.6,0.0,31254.78\nR002,2020-12-25,-0.3,1.2,25200.0\nR002,2020-12-26,7.5,1.7,27193.93\nR002,2020-12-27,9.5,0.0,31030.9\nR002,2020-12-28,8.5,0.0,29833.42\nR002,2020-12-29,10.1,0.0,31473.64\nR002,2020-12-30,8.4,0.0,31523.69\nR002,2020-12-31,8.2,0.0,31575.55\nR002,2021-01-01,9.2,0.0,31629.1\nR002,2021-01-02,8.4,0.0,31684.2\nR002,2021-01-03,5.7,0.0,31740.74\nR002,2021-01-04,5.1,0.0,31798.59\nR002,2021-01-05,7.6,0.0,28838.95\nR002,2021-01-06,7.3,0.8,17827.16\nR002,2021-01-07,6.7,0.0,29921.2\nR002,2021-01-08,10.3,0.0,14012.11\nR002,2021-01-09,11.5,0.0,21440.24\nR002,2021-01-10,9.7,20.5,2760.62\nR002,2021-01-11,7.2,20.8,0.0\nR002,2021-01-12,9.2,11.8,1121.75\nR002,2021-01-13,12.4,11.3,18291.37\nR002,2021-01-14,6.2,40.9,0.0\nR002,2021-01-15,3.1,27.5,10823.24\nR002,2021-01-16,0.0,2.0,30759.97\nR002,2021-01-17,3.8,33.2,0.0\nR002,2021-01-18,0.5,11.3,22609.88\nR002,2021-01-19,-3.7,4.4,27160.91\nR002,2021-01-20,-4.9,2.4,31736.36\nR002,2021-01-21,-4.4,0.2,32400.0\nR002,2021-01-22,1.6,0.0,32400.0\nR002,2021-01-23,3.1,0.0,32400.0\nR002,2021-01-24,4.6,0.0,30823.41\nR002,2021-01-25,9.3,0.0,32400.0\nR002,2021-01-26,7.7,0.3,28800.0\nR002,2021-01-27,9.8,0.0,25200.0\nR002,2021-01-28,7.4,10.7,4473.43\nR002,2021-01-29,1.8,4.9,3781.67\nR002,2021-01-30,1.9,0.0,32443.68\nR002,2021-01-31,5.7,2.5,16154.14\nR002,2021-02-01,7.8,0.0,31609.87\nR002,2021-02-02,11.1,0.0,32374.79\nR002,2021-02-03,11.2,0.0,32400.0\nR002,2021-02-04,11.2,0.0,32453.03\nR002,2021-02-05,5.6,24.4,14198.09\nR002,2021-02-06,2.6,9.8,1947.31\nR002,2021-02-07,6.0,20.8,0.0\nR002,2021-02-08,9.7,1.4,24478.15\nR002,2021-02-09,9.7,7.9,17938.79\nR002,2021-02-10,9.6,0.0,35155.49\nR002,2021-02-11,10.5,0.0,33969.73\nR002,2021-02-12,8.5,4.4,34831.57\nR002,2021-02-13,-1.3,21.2,2432.66\nR002,2021-02-14,0.4,10.9,2993.45\nR002,2021-02-15,2.6,32.3,0.0\nR002,2021-02-16,0.5,22.4,0.0\nR002,2021-02-17,-0.4,8.4,0.0\nR002,2021-02-18,-1.3,3.6,32400.0\nR002,2021-02-19,-1.2,5.5,13654.99\nR002,2021-02-20,-1.8,0.0,37447.76\nR002,2021-02-21,-0.6,0.0,38329.64\nR002,2021-02-22,2.1,6.5,23368.64\nR002,2021-02-23,3.1,3.9,2056.17\nR002,2021-02-24,1.9,0.0,38859.35\nR002,2021-02-25,1.8,0.1,39037.83\nR002,2021-02-26,4.6,0.0,21613.36\nR002,2021-02-27,6.2,0.0,37930.34\nR002,2021-02-28,3.5,11.6,670.7\nR002,2021-03-01,2.0,0.1,37920.38\nR002,2021-03-02,1.9,0.0,39600.0\nR002,2021-03-03,0.7,15.1,12541.09\nR002,2021-03-04,0.1,0.0,39600.0\nR002,2021-03-05,3.8,0.0,39600.0\nR002,2021-03-06,7.5,0.6,39600.0\nR002,2021-03-07,2.6,9.7,36010.82\nR002,2021-03-08,0.4,0.0,39600.0\nR002,2021-03-09,-0.2,3.5,39123.83\nR002,2021-03-10,1.2,0.2,39600.0\nR002,2021-03-11,2.8,9.8,996.01\nR002,2021-03-12,-0.1,11.2,16423.03\nR002,2021-03-13,2.3,0.0,35632.85\nR002,2021-03-14,5.8,0.0,39600.0\nR002,2021-03-15,7.8,0.0,32633.64\nR002,2021-03-16,9.6,5.4,35059.97\nR002,2021-03-17,8.1,4.6,24237.94\nR002,2021-03-18,5.1,14.0,21772.7\nR002,2021-03-19,3.8,17.1,22165.67\nR002,2021-03-20,2.8,5.5,20300.19\nR002,2021-03-21,2.4,0.8,5023.83\nR002,2021-03-22,5.3,2.5,34844.93\nR002,2021-03-23,7.3,23.4,13792.01\nR002,2021-03-24,5.2,23.4,6006.81\nR002,2021-03-25,3.1,5.3,18642.63\nR002,2021-03-26,2.1,3.6,37327.79\nR002,2021-03-27,2.5,0.0,41619.3\nR002,2021-03-28,3.8,0.8,40456.9\nR002,2021-03-29,4.5,0.0,41126.43\nR002,2021-03-30,5.4,0.1,41995.29\nR002,2021-03-31,7.3,0.0,44124.62\nR002,2021-04-01,8.4,1.7,36340.17\nR002,2021-04-02,7.6,5.1,3087.87\nR002,2021-04-03,8.0,0.9,33152.18\nR002,2021-04-04,10.8,2.1,41138.98\nR002,2021-04-05,10.5,35.6,11382.22\nR002,2021-04-06,8.9,1.3,44026.15\nR002,2021-04-07,16.7,2.1,35503.56\nR002,2021-04-08,16.9,1.2,46172.81\nR002,2021-04-09,9.9,9.9,3447.25\nR002,2021-04-10,8.1,4.3,19801.72\nR002,2021-04-11,10.3,4.8,14404.07\nR002,2021-04-12,9.4,7.2,17433.06\nR002,2021-04-13,9.8,6.0,29452.9\nR002,2021-04-14,10.1,2.2,45279.39\nR002,2021-04-15,12.5,0.0,46647.18\nR002,2021-04-16,13.4,0.2,46742.55\nR002,2021-04-17,11.2,9.4,0.0\nR002,2021-04-18,14.5,9.8,42417.15\nR002,2021-04-19,14.6,14.2,4435.59\nR002,2021-04-20,15.7,16.2,34909.18\nR002,2021-04-21,12.5,9.8,35818.4\nR002,2021-04-22,12.3,2.0,43938.74\nR002,2021-04-23,12.8,0.0,45521.6\nR002,2021-04-24,12.0,3.2,21762.07\nR002,2021-04-25,13.4,1.9,30198.27\nR002,2021-04-26,12.8,25.2,4514.31\nR002,2021-04-27,11.2,2.6,12128.9\nR002,2021-04-28,15.1,0.3,40415.51\nR002,2021-04-29,15.8,2.3,38959.08\nR002,2021-04-30,16.9,0.7,47871.27\nR002,2021-05-01,15.8,1.6,46826.92\nR002,2021-05-02,17.2,0.1,48469.11\nR002,2021-05-03,20.1,0.0,46371.53\nR002,2021-05-04,15.6,9.7,15305.72\nR002,2021-05-05,13.9,0.0,44686.78\nR002,2021-05-06,15.7,0.0,47327.17\nR002,2021-05-07,15.9,0.0,48984.58\nR002,2021-05-08,17.9,0.1,48955.96\nR002,2021-05-09,10.8,33.3,2350.99\nR002,2021-05-10,6.3,4.0,47873.98\nR002,2021-05-11,6.9,2.7,27227.97\nR002,2021-05-12,13.0,0.0,50268.61\nR002,2021-05-13,19.5,0.0,50335.65\nR002,2021-05-14,23.3,0.3,50400.0\nR002,2021-05-15,22.5,1.0,50400.0\nR002,2021-05-16,23.9,0.9,50400.0\nR002,2021-05-17,19.2,9.7,43770.07\nR002,2021-05-18,18.4,0.1,49902.72\nR002,2021-05-19,15.6,6.1,39780.68\nR002,2021-05-20,20.0,0.0,50684.69\nR002,2021-05-21,24.8,0.0,50400.0\nR002,2021-05-22,21.4,9.5,33000.65\nR002,2021-05-23,18.6,20.2,2480.7\nR002,2021-05-24,18.4,5.8,11288.22\nR002,2021-05-25,16.3,7.9,39344.04\nR002,2021-05-26,16.6,0.4,49644.98\nR002,2021-05-27,18.0,0.9,45696.26\nR002,2021-05-28,20.2,1.3,51202.25\nR002,2021-05-29,19.3,9.4,48745.07\nR002,2021-05-30,16.2,6.5,37793.37\nR002,2021-05-31,19.7,0.0,51500.02\nR002,2021-06-01,20.7,12.8,43262.05\nR002,2021-06-02,16.5,24.7,22885.61\nR002,2021-06-03,13.9,4.9,47540.33\nR002,2021-06-04,12.9,9.9,12356.77\nR002,2021-06-05,15.2,7.4,35900.74\nR002,2021-06-06,14.2,3.4,49779.63\nR002,2021-06-07,14.2,2.3,48549.19\nR002,2021-06-08,16.6,0.0,52814.32\nR002,2021-06-09,16.3,0.8,45104.58\nR002,2021-06-10,17.8,0.4,52053.15\nR002,2021-06-11,18.6,0.0,52039.28\nR002,2021-06-12,19.1,2.8,52232.68\nR002,2021-06-13,18.8,6.4,49094.31\nR002,2021-06-14,21.1,0.1,52736.88\nR002,2021-06-15,21.8,4.5,50416.63\nR002,2021-06-16,22.4,1.0,53452.26\nR002,2021-06-17,19.9,13.1,39352.62\nR002,2021-06-18,21.5,0.0,51778.87\nR002,2021-06-19,24.9,0.1,53906.88\nR002,2021-06-20,25.4,0.9,52020.22\nR002,2021-06-21,26.3,2.3,53933.6\nR002,2021-06-22,24.9,8.3,52953.2\nR002,2021-06-23,25.0,2.6,52267.66\nR002,2021-06-24,24.5,8.8,47602.28\nR002,2021-06-25,24.2,3.8,51199.3\nR002,2021-06-26,22.9,8.5,49966.8\nR002,2021-06-27,23.3,9.1,45297.46\nR002,2021-06-28,23.2,6.3,50521.43\nR002,2021-06-29,22.0,9.8,49217.23\nR002,2021-06-30,22.8,0.0,51320.33\nR002,2021-07-01,24.0,0.0,53947.11\nR002,2021-07-02,25.2,0.7,53936.47\nR002,2021-07-03,25.9,0.1,53923.49\nR002,2021-07-04,22.7,9.9,39133.19\nR002,2021-07-05,18.9,30.3,9932.89\nR002,2021-07-06,21.3,1.9,45313.04\nR002,2021-07-07,21.5,1.5,50067.02\nR002,2021-07-08,22.6,1.1,51503.86\nR002,2021-07-09,23.4,0.3,52419.12\nR002,2021-07-10,22.4,0.0,53766.1\nR002,2021-07-11,23.2,0.3,53734.56\nR002,2021-07-12,23.9,2.9,53700.89\nR002,2021-07-13,22.9,8.7,52539.43\nR002,2021-07-14,22.8,2.5,51155.55\nR002,2021-07-15,23.5,0.0,53587.42\nR002,2021-07-16,24.3,0.0,53545.56\nR002,2021-07-17,25.4,0.0,53501.76\nR002,2021-07-18,26.2,0.0,53143.06\nR002,2021-07-19,27.1,0.0,53408.4\nR002,2021-07-20,27.2,0.0,53305.95\nR002,2021-07-21,26.4,2.5,52372.74\nR002,2021-07-22,23.0,4.6,45403.91\nR002,2021-07-23,20.0,15.9,41797.85\nR002,2021-07-24,22.0,5.8,46108.74\nR002,2021-07-25,22.6,3.0,49783.75\nR002,2021-07-26,23.9,3.1,48432.42\nR002,2021-07-27,21.9,18.7,15180.13\nR002,2021-07-28,21.4,0.0,47359.5\nR002,2021-07-29,22.1,1.7,48647.2\nR002,2021-07-30,22.1,0.8,46071.64\nR002,2021-07-31,23.1,3.3,49734.84\nR002,2021-08-01,24.2,0.2,49100.02\nR002,2021-08-02,24.5,2.3,49157.51\nR002,2021-08-03,24.6,0.5,47158.98\nR002,2021-08-04,25.3,1.0,48933.56\nR002,2021-08-05,25.0,0.8,46946.96\nR002,2021-08-06,26.3,0.3,48612.87\nR002,2021-08-07,26.3,2.2,41749.34\nR002,2021-08-08,24.7,3.9,41905.9\nR002,2021-08-09,23.6,13.8,42325.87\nR002,2021-08-10,23.0,11.0,33994.1\nR002,2021-08-11,23.0,9.6,34180.7\nR002,2021-08-12,22.5,15.0,39401.18\nR002,2021-08-13,21.9,7.6,24318.17\nR002,2021-08-14,21.0,0.3,42684.1\nR002,2021-08-15,21.2,1.4,46800.0\nR002,2021-08-16,20.7,9.4,42051.08\nR002,2021-08-17,22.3,2.3,46800.0\nR002,2021-08-18,24.0,1.0,46941.44\nR002,2021-08-19,22.6,10.6,44928.72\nR002,2021-08-20,21.3,5.6,43200.0\nR002,2021-08-21,22.1,1.3,43343.83\nR002,2021-08-22,22.4,3.2,45566.24\nR002,2021-08-23,22.2,3.3,46762.7\nR002,2021-08-24,22.5,1.6,45211.25\nR002,2021-08-25,23.6,0.0,46800.0\nR002,2021-08-26,23.6,1.4,46800.0\nR002,2021-08-27,24.5,0.2,46710.85\nR002,2021-08-28,25.7,0.1,46607.51\nR002,2021-08-29,26.2,0.0,46503.69\nR002,2021-08-30,27.9,0.0,46399.5\nR002,2021-08-31,24.2,10.3,44871.08\nR002,2021-09-01,22.8,4.2,40716.23\nR002,2021-09-02,18.5,22.2,34712.13\nR002,2021-09-03,14.8,9.3,34134.54\nR002,2021-09-04,15.1,0.0,44745.11\nR002,2021-09-05,16.4,0.0,40843.43\nR002,2021-09-06,17.7,1.5,38743.9\nR002,2021-09-07,17.9,0.3,42587.4\nR002,2021-09-08,17.6,0.3,40542.35\nR002,2021-09-09,18.4,0.2,43404.46\nR002,2021-09-10,20.3,0.0,43458.25\nR002,2021-09-11,18.2,24.2,12566.58\nR002,2021-09-12,18.3,3.3,38097.65\nR002,2021-09-13,18.5,0.3,41323.87\nR002,2021-09-14,19.1,0.0,42630.3\nR002,2021-09-15,20.2,0.0,41388.92\nR002,2021-09-16,17.9,43.9,3298.83\nR002,2021-09-17,17.6,28.4,1881.25\nR002,2021-09-18,18.0,0.3,34661.44\nR002,2021-09-19,17.2,3.0,38191.95\nR002,2021-09-20,18.2,0.0,39499.37\nR002,2021-09-21,17.9,2.9,35713.82\nR002,2021-09-22,14.5,6.1,17702.48\nR002,2021-09-23,12.9,5.4,10404.4\nR002,2021-09-24,13.1,1.4,34909.71\nR002,2021-09-25,9.5,36.4,13456.48\nR002,2021-09-26,11.0,0.4,40492.8\nR002,2021-09-27,12.9,3.0,7332.7\nR002,2021-09-28,13.5,1.2,37007.0\nR002,2021-09-29,14.0,0.4,36473.35\nR002,2021-09-30,14.2,8.4,33308.85\nR002,2021-10-01,13.1,17.7,6855.4\nR002,2021-10-02,13.8,0.3,38573.24\nR002,2021-10-03,13.5,3.3,2886.58\nR002,2021-10-04,13.3,8.7,24336.6\nR002,2021-10-05,12.3,7.1,27142.98\nR002,2021-10-06,11.4,4.0,24447.36\nR002,2021-10-07,12.6,1.4,35729.39\nR002,2021-10-08,12.6,0.4,35998.67\nR002,2021-10-09,14.5,0.0,34818.32\nR002,2021-10-10,14.9,0.0,37094.5\nR002,2021-10-11,15.4,0.0,36824.54\nR002,2021-10-12,16.7,0.0,36878.76\nR002,2021-10-13,16.9,3.3,36280.12\nR002,2021-10-14,15.9,0.1,36200.8\nR002,2021-10-15,16.6,0.0,36056.33\nR002,2021-10-16,17.3,0.4,36107.17\nR002,2021-10-17,15.8,2.3,35873.54\nR002,2021-10-18,13.9,1.9,33319.81\nR002,2021-10-19,10.5,3.4,26723.82\nR002,2021-10-20,7.9,0.0,36000.0\nR002,2021-10-21,10.4,0.0,36000.0\nR002,2021-10-22,14.1,0.0,36000.0\nR002,2021-10-23,13.6,0.0,36000.0\nR002,2021-10-24,12.0,6.3,32788.86\nR002,2021-10-25,5.4,13.3,35824.85\nR002,2021-10-26,4.0,0.0,36000.0\nR002,2021-10-27,8.4,0.0,36000.0\nR002,2021-10-28,8.5,0.0,36000.0\nR002,2021-10-29,8.3,0.0,34253.83\nR002,2021-10-30,8.4,0.0,36000.0\nR002,2021-10-31,12.3,0.0,36000.0\nR002,2021-11-01,15.2,0.1,36000.0\nR002,2021-11-02,15.5,2.0,20961.84\nR002,2021-11-03,14.3,0.3,5656.91\nR002,2021-11-04,13.5,0.0,34416.8\nR002,2021-11-05,14.0,0.0,35663.97\nR002,2021-11-06,14.3,0.0,35511.32\nR002,2021-11-07,11.8,0.0,33917.11\nR002,2021-11-08,14.2,0.0,27898.06\nR002,2021-11-09,14.8,0.0,33158.85\nR002,2021-11-10,7.2,21.5,21648.0\nR002,2021-11-11,3.5,0.0,32334.73\nR002,2021-11-12,6.3,0.0,34617.83\nR002,2021-11-13,9.3,0.0,34474.28\nR002,2021-11-14,10.2,0.0,32065.59\nR002,2021-11-15,9.9,0.0,30622.73\nR002,2021-11-16,7.2,0.0,33703.43\nR002,2021-11-17,8.3,0.0,33332.42\nR002,2021-11-18,8.0,0.0,32596.67\nR002,2021-11-19,6.7,4.0,29145.56\nR002,2021-11-20,6.5,10.5,25674.57\nR002,2021-11-21,7.5,11.6,20353.57\nR002,2021-11-22,7.9,0.0,32212.43\nR002,2021-11-23,9.0,0.3,30693.61\nR002,2021-11-24,5.7,16.5,19940.78\nR002,2021-11-25,1.9,0.0,31194.0\nR002,2021-11-26,4.5,0.0,31261.04\nR002,2021-11-27,7.3,5.4,2214.35\nR002,2021-11-28,10.9,11.4,7035.53\nR002,2021-11-29,12.9,1.6,25925.03\nR002,2021-11-30,13.4,34.3,3353.68\nR002,2021-12-01,4.4,15.9,22421.67\nR002,2021-12-02,2.2,8.8,29241.48\nR002,2021-12-03,3.9,0.0,30738.75\nR002,2021-12-04,7.5,0.0,29987.94\nR002,2021-12-05,9.0,2.3,12647.05\nR002,2021-12-06,10.3,0.0,27126.53\nR002,2021-12-07,10.9,0.0,30401.58\nR002,2021-12-08,11.0,0.3,26923.17\nR002,2021-12-09,9.3,3.2,2366.23\nR002,2021-12-10,7.1,0.0,30419.74\nR002,2021-12-11,8.1,0.0,30035.43\nR002,2021-12-12,9.3,0.0,16840.54\nR002,2021-12-13,10.6,0.0,30424.4\nR002,2021-12-14,10.7,0.0,10384.96\nR002,2021-12-15,10.9,0.0,30419.45\nR002,2021-12-16,10.0,0.0,29634.78\nR002,2021-12-17,6.1,12.4,1199.75\nR002,2021-12-18,4.6,14.0,16431.3\nR002,2021-12-19,1.4,0.0,30779.13\nR002,2021-12-20,4.0,19.2,226.51\nR002,2021-12-21,0.8,13.4,5771.96\nR002,2021-12-22,-5.8,4.7,22331.53\nR002,2021-12-23,-7.1,1.7,29157.76\nR002,2021-12-24,-7.0,0.0,28800.0\nR002,2021-12-25,0.6,6.7,0.0\nR002,2021-12-26,5.6,0.7,26665.02\nR002,2021-12-27,7.4,4.8,754.78\nR002,2021-12-28,7.7,0.3,28996.53\nR002,2021-12-29,5.5,0.0,31464.19\nR002,2021-12-30,4.0,0.0,28883.3\nR002,2021-12-31,2.9,3.7,25200.0\nR002,2022-01-01,5.3,11.9,5003.13\nR002,2022-01-02,3.0,23.8,17887.24\nR002,2022-01-03,-0.3,1.6,2033.9\nR002,2022-01-04,3.7,27.0,7657.93\nR002,2022-01-05,5.0,1.7,11065.54\nR002,2022-01-06,9.8,0.0,31892.58\nR002,2022-01-07,9.8,0.0,26122.77\nR002,2022-01-08,6.9,0.0,32019.91\nR002,2022-01-09,6.1,0.0,32086.38\nR002,2022-01-10,6.8,0.8,23034.47\nR002,2022-01-11,8.7,3.7,32224.22\nR002,2022-01-12,6.7,26.9,26.32\nR002,2022-01-13,1.1,14.3,19887.02\nR002,2022-01-14,-2.3,5.5,26674.2\nR002,2022-01-15,-1.3,25.6,19941.76\nR002,2022-01-16,-3.1,0.3,32400.0\nR002,2022-01-17,-4.3,0.0,32400.0\nR002,2022-01-18,-0.0,20.8,0.0\nR002,2022-01-19,-2.8,2.7,25200.0\nR002,2022-01-20,-4.1,3.9,27884.08\nR002,2022-01-21,0.4,3.3,24966.86\nR002,2022-01-22,3.3,21.6,0.0\nR002,2022-01-23,4.7,12.5,8977.94\nR002,2022-01-24,3.1,10.6,14506.49\nR002,2022-01-25,-0.6,3.2,30726.2\nR002,2022-01-26,-0.5,5.6,400.09\nR002,2022-01-27,-1.3,2.9,32400.0\nR002,2022-01-28,-1.5,1.8,32400.0\nR002,2022-01-29,-0.3,3.8,23149.85\nR002,2022-01-30,0.7,12.0,0.0\nR002,2022-01-31,1.8,12.6,0.0\nR002,2022-02-01,2.4,1.8,32400.0\nR002,2022-02-02,6.7,0.0,31315.89\nR002,2022-02-03,7.8,0.0,32400.0\nR002,2022-02-04,4.8,12.9,11917.76\nR002,2022-02-05,-2.2,5.7,33233.86\nR002,2022-02-06,-1.5,13.5,1174.74\nR002,2022-02-07,-0.5,0.0,25755.94\nR002,2022-02-08,4.2,4.8,5706.28\nR002,2022-02-09,5.4,37.9,0.0\nR002,2022-02-10,3.6,3.1,34373.52\nR002,2022-02-11,2.8,0.0,34745.99\nR002,2022-02-12,4.8,0.0,36183.16\nR002,2022-02-13,3.5,2.1,33887.56\nR002,2022-02-14,3.7,0.0,33320.67\nR002,2022-02-15,3.8,0.0,36621.52\nR002,2022-02-16,2.3,0.0,37406.06\nR002,2022-02-17,3.4,0.0,37593.88\nR002,2022-02-18,6.8,0.0,34348.36\nR002,2022-02-19,5.1,5.3,5944.77\nR002,2022-02-20,5.8,0.1,38105.09\nR002,2022-02-21,5.8,6.6,12147.34\nR002,2022-02-22,7.6,0.0,38468.87\nR002,2022-02-23,11.6,0.3,33387.49\nR002,2022-02-24,10.5,0.0,28989.34\nR002,2022-02-25,9.1,4.8,342.06\nR002,2022-02-26,7.1,6.2,25621.65\nR002,2022-02-27,5.6,2.1,37279.64\nR002,2022-02-28,6.0,1.4,16724.85\nR002,2022-03-01,6.2,14.9,8749.66\nR002,2022-03-02,6.5,2.7,30482.87\nR002,2022-03-03,6.9,14.2,17627.53\nR002,2022-03-04,2.8,9.3,27205.67\nR002,2022-03-05,2.1,14.7,20711.09\nR002,2022-03-06,0.7,2.1,39600.0\nR002,2022-03-07,1.9,0.0,39600.0\nR002,2022-03-08,2.3,7.9,9225.65\nR002,2022-03-09,0.8,5.4,2045.31\nR002,2022-03-10,0.2,0.8,32400.0\nR002,2022-03-11,0.5,4.5,9735.03\nR002,2022-03-12,-0.6,4.4,39600.0\nR002,2022-03-13,-1.2,1.1,38460.04\nR002,2022-03-14,-0.9,0.0,39600.0\nR002,2022-03-15,-1.0,0.2,39600.0\nR002,2022-03-16,-1.2,7.3,29065.94\nR002,2022-03-17,-3.1,3.5,34770.39\nR002,2022-03-18,-3.2,2.2,38852.52\nR002,2022-03-19,-0.2,0.6,30965.91\nR002,2022-03-20,-0.4,0.0,40047.0\nR002,2022-03-21,1.1,0.0,41545.03\nR002,2022-03-22,2.5,0.0,41772.53\nR002,2022-03-23,4.1,0.0,42258.98\nR002,2022-03-24,3.9,5.6,4394.05\nR002,2022-03-25,3.2,3.4,34075.74\nR002,2022-03-26,3.4,4.0,34152.6\nR002,2022-03-27,3.0,13.7,24541.95\nR002,2022-03-28,1.1,3.0,43147.78\nR002,2022-03-29,7.2,0.0,43641.44\nR002,2022-03-30,10.2,1.0,15268.39\nR002,2022-03-31,10.6,0.0,44425.79\nR002,2022-04-01,14.6,0.6,34576.2\nR002,2022-04-02,16.9,0.3,38313.95\nR002,2022-04-03,13.2,5.1,40535.49\nR002,2022-04-04,6.9,9.3,29841.65\nR002,2022-04-05,5.9,8.0,11419.28\nR002,2022-04-06,8.4,1.7,36399.49\nR002,2022-04-07,9.0,0.0,46083.95\nR002,2022-04-08,9.1,8.8,14931.38\nR002,2022-04-09,10.4,3.0,44072.67\nR002,2022-04-10,13.0,0.0,38061.51\nR002,2022-04-11,15.2,0.3,46369.01\nR002,2022-04-12,7.3,33.3,0.0\nR002,2022-04-13,3.8,9.3,17283.83\nR002,2022-04-14,3.6,9.9,9847.76\nR002,2022-04-15,6.7,3.3,19735.97\nR002,2022-04-16,8.4,0.0,46724.35\nR002,2022-04-17,12.1,0.0,46795.71\nR002,2022-04-18,18.5,1.2,38703.98\nR002,2022-04-19,16.1,7.5,28428.85\nR002,2022-04-20,10.0,21.7,28800.0\nR002,2022-04-21,9.7,3.3,46800.0\nR002,2022-04-22,10.8,0.3,46800.0\nR002,2022-04-23,13.5,0.0,47172.42\nR002,2022-04-24,17.5,0.4,47458.43\nR002,2022-04-25,18.7,0.0,47415.24\nR002,2022-04-26,19.8,0.0,47704.49\nR002,2022-04-27,18.1,0.9,47663.34\nR002,2022-04-28,17.2,1.3,44830.24\nR002,2022-04-29,15.6,2.0,44454.74\nR002,2022-04-30,15.0,1.3,46689.63\nR002,2022-05-01,14.1,3.8,15363.2\nR002,2022-05-02,11.2,13.1,0.0\nR002,2022-05-03,12.6,0.4,44834.3\nR002,2022-05-04,13.3,1.4,37743.62\nR002,2022-05-05,10.7,20.9,0.0\nR002,2022-05-06,11.6,1.9,20582.83\nR002,2022-05-07,10.2,6.9,0.0\nR002,2022-05-08,11.3,5.8,12993.38\nR002,2022-05-09,12.0,4.1,40700.04\nR002,2022-05-10,10.3,4.6,34074.17\nR002,2022-05-11,9.3,3.6,41895.54\nR002,2022-05-12,9.7,0.2,48512.21\nR002,2022-05-13,13.6,0.0,50319.83\nR002,2022-05-14,17.2,0.0,50386.69\nR002,2022-05-15,17.5,0.0,50481.26\nR002,2022-05-16,13.8,1.1,47537.0\nR002,2022-05-17,12.8,0.5,50400.0\nR002,2022-05-18,14.8,0.0,49897.25\nR002,2022-05-19,10.9,20.1,29146.59\nR002,2022-05-20,8.6,6.8,32020.04\nR002,2022-05-21,12.6,0.0,50131.64\nR002,2022-05-22,17.0,0.0,50993.88\nR002,2022-05-23,13.2,1.7,33720.3\nR002,2022-05-24,10.8,3.9,41764.23\nR002,2022-05-25,12.1,4.9,43279.79\nR002,2022-05-26,13.8,0.4,49052.55\nR002,2022-05-27,15.9,0.0,47170.14\nR002,2022-05-28,18.6,0.0,51567.6\nR002,2022-05-29,21.0,0.1,51545.58\nR002,2022-05-30,22.4,0.5,51885.08\nR002,2022-05-31,23.4,1.8,50989.42\nR002,2022-06-01,22.3,2.0,51119.7\nR002,2022-06-02,22.7,0.4,52187.47\nR002,2022-06-03,22.9,0.6,50400.0\nR002,2022-06-04,25.1,0.9,51786.9\nR002,2022-06-05,23.5,4.0,45183.74\nR002,2022-06-06,21.4,3.4,48896.3\nR002,2022-06-07,20.6,1.5,49809.47\nR002,2022-06-08,20.9,5.5,50794.36\nR002,2022-06-09,19.0,12.4,49268.82\nR002,2022-06-10,19.7,6.3,44879.23\nR002,2022-06-11,20.8,1.6,53142.69\nR002,2022-06-12,21.8,1.8,52030.54\nR002,2022-06-13,20.0,3.4,49188.73\nR002,2022-06-14,19.1,11.6,32612.49\nR002,2022-06-15,18.9,10.8,47914.55\nR002,2022-06-16,19.4,5.0,49315.49\nR002,2022-06-17,19.9,1.1,53199.48\nR002,2022-06-18,20.5,0.2,53069.71\nR002,2022-06-19,21.2,1.3,53061.11\nR002,2022-06-20,22.4,0.5,53919.8\nR002,2022-06-21,22.1,1.7,52310.24\nR002,2022-06-22,19.7,13.7,51017.51\nR002,2022-06-23,15.8,8.9,42707.03\nR002,2022-06-24,15.3,4.5,13042.77\nR002,2022-06-25,18.0,4.8,45920.06\nR002,2022-06-26,17.7,25.2,0.0\nR002,2022-06-27,19.8,14.5,42182.44\nR002,2022-06-28,21.2,4.0,49778.21\nR002,2022-06-29,21.9,2.4,47435.51\nR002,2022-06-30,22.7,2.5,52441.15\nR002,2022-07-01,21.9,2.2,52477.92\nR002,2022-07-02,20.7,0.0,53597.33\nR002,2022-07-03,18.8,0.2,53927.94\nR002,2022-07-04,19.5,0.4,53912.18\nR002,2022-07-05,21.4,0.0,53894.25\nR002,2022-07-06,23.1,0.1,53874.17\nR002,2022-07-07,23.7,0.9,52304.58\nR002,2022-07-08,22.4,3.2,52313.47\nR002,2022-07-09,22.7,4.0,53801.15\nR002,2022-07-10,23.9,4.4,52688.83\nR002,2022-07-11,21.5,18.1,30508.51\nR002,2022-07-12,21.5,4.9,38008.86\nR002,2022-07-13,22.1,6.7,34837.51\nR002,2022-07-14,21.6,1.3,45965.85\nR002,2022-07-15,22.7,0.0,52639.84\nR002,2022-07-16,22.9,4.9,41671.31\nR002,2022-07-17,23.5,2.5,44261.15\nR002,2022-07-18,22.1,6.3,27246.58\nR002,2022-07-19,21.2,11.5,10373.08\nR002,2022-07-20,21.5,10.4,25331.69\nR002,2022-07-21,20.4,12.2,42886.95\nR002,2022-07-22,18.1,2.3,51752.99\nR002,2022-07-23,17.8,6.9,46159.78\nR002,2022-07-24,17.8,13.0,48193.26\nR002,2022-07-25,19.2,1.7,47603.89\nR002,2022-07-26,20.4,5.5,45998.25\nR002,2022-07-27,20.5,1.5,47257.05\nR002,2022-07-28,21.5,0.5,48558.73\nR002,2022-07-29,22.9,0.2,50659.42\nR002,2022-07-30,24.4,0.6,49671.04\nR002,2022-07-31,24.8,0.6,50415.69\nR002,2022-08-01,24.3,3.7,49779.09\nR002,2022-08-02,23.2,3.9,46791.71\nR002,2022-08-03,23.6,4.8,46556.31\nR002,2022-08-04,23.7,1.3,48921.99\nR002,2022-08-05,22.9,1.3,30835.31\nR002,2022-08-06,22.3,1.0,46894.21\nR002,2022-08-07,21.3,1.4,47954.97\nR002,2022-08-08,22.5,0.6,47304.3\nR002,2022-08-09,23.0,0.0,48347.84\nR002,2022-08-10,24.0,1.3,48036.97\nR002,2022-08-11,23.4,10.7,45959.63\nR002,2022-08-12,22.8,2.7,46800.0\nR002,2022-08-13,24.2,1.1,46800.0\nR002,2022-08-14,24.1,5.1,47116.38\nR002,2022-08-15,25.4,2.0,46800.0\nR002,2022-08-16,25.1,5.2,44427.03\nR002,2022-08-17,24.6,2.6,46412.81\nR002,2022-08-18,24.6,9.1,43286.41\nR002,2022-08-19,23.5,8.0,46069.97\nR002,2022-08-20,23.5,3.6,44069.67\nR002,2022-08-21,25.1,1.5,46718.75\nR002,2022-08-22,26.4,0.3,46800.0\nR002,2022-08-23,26.0,2.0,46800.0\nR002,2022-08-24,26.4,0.2,46800.0\nR002,2022-08-25,27.6,0.4,46800.0\nR002,2022-08-26,27.0,0.2,46800.0\nR002,2022-08-27,26.3,0.8,46734.77\nR002,2022-08-28,25.3,0.9,46633.4\nR002,2022-08-29,23.1,4.5,45407.94\nR002,2022-08-30,23.9,0.2,46430.2\nR002,2022-08-31,25.3,0.1,46328.24\nR002,2022-09-01,26.4,0.1,45224.8\nR002,2022-09-02,26.4,0.7,43988.04\nR002,2022-09-03,27.1,0.0,45642.3\nR002,2022-09-04,22.7,5.1,38568.52\nR002,2022-09-05,19.1,3.2,38241.63\nR002,2022-09-06,16.8,1.0,41104.15\nR002,2022-09-07,14.6,0.7,41849.37\nR002,2022-09-08,15.2,0.2,43659.03\nR002,2022-09-09,15.6,0.0,43712.68\nR002,2022-09-10,17.5,0.0,43447.88\nR002,2022-09-11,20.1,0.0,43200.0\nR002,2022-09-12,21.4,0.5,43200.0\nR002,2022-09-13,18.1,12.5,26503.09\nR002,2022-09-14,15.6,0.8,38081.1\nR002,2022-09-15,16.5,0.4,29112.28\nR002,2022-09-16,20.0,2.2,42583.69\nR002,2022-09-17,23.7,0.0,42084.37\nR002,2022-09-18,26.4,0.1,42839.8\nR002,2022-09-19,20.9,18.8,35229.83\nR002,2022-09-20,19.8,13.2,31111.74\nR002,2022-09-21,17.9,39.5,1681.95\nR002,2022-09-22,17.0,61.4,1002.53\nR002,2022-09-23,14.6,36.7,21600.0\nR002,2022-09-24,11.6,11.3,432.67\nR002,2022-09-25,11.6,3.5,39811.92\nR002,2022-09-26,14.2,0.0,42079.0\nR002,2022-09-27,18.5,0.0,34240.42\nR002,2022-09-28,20.5,0.4,33515.76\nR002,2022-09-29,18.3,7.6,36771.53\nR002,2022-09-30,18.7,0.0,40806.29\nR002,2022-10-01,19.0,0.0,40413.23\nR002,2022-10-02,19.7,0.5,33197.76\nR002,2022-10-03,16.8,20.0,36915.43\nR002,2022-10-04,16.2,13.0,0.0\nR002,2022-10-05,15.7,38.1,4013.59\nR002,2022-10-06,16.1,1.3,32095.29\nR002,2022-10-07,16.8,0.0,37108.9\nR002,2022-10-08,20.1,0.2,30045.05\nR002,2022-10-09,17.3,7.3,3295.26\nR002,2022-10-10,15.0,10.7,22348.49\nR002,2022-10-11,11.6,1.7,35235.13\nR002,2022-10-12,10.9,0.0,36535.17\nR002,2022-10-13,11.8,0.0,36000.0\nR002,2022-10-14,11.7,0.2,36000.0\nR002,2022-10-15,13.1,0.2,35038.46\nR002,2022-10-16,14.2,0.3,31940.16\nR002,2022-10-17,14.4,3.9,1268.56\nR002,2022-10-18,14.1,0.0,36000.0\nR002,2022-10-19,15.9,0.0,36000.0\nR002,2022-10-20,12.8,23.5,19840.45\nR002,2022-10-21,7.9,0.3,36000.0\nR002,2022-10-22,8.1,0.0,36000.0\nR002,2022-10-23,10.7,1.1,31717.76\nR002,2022-10-24,10.9,9.5,24006.04\nR002,2022-10-25,9.8,0.0,34487.92\nR002,2022-10-26,12.4,0.1,27031.01\nR002,2022-10-27,11.1,9.8,31761.39\nR002,2022-10-28,9.4,1.5,35695.13\nR002,2022-10-29,9.3,0.0,36000.0\nR002,2022-10-30,9.9,0.0,34679.21\nR002,2022-10-31,9.6,23.2,0.0\nR002,2022-11-01,7.5,0.2,33682.36\nR002,2022-11-02,6.6,2.9,28991.47\nR002,2022-11-03,9.6,9.4,24749.21\nR002,2022-11-04,9.6,8.9,29559.57\nR002,2022-11-05,8.4,0.0,34547.5\nR002,2022-11-06,6.6,0.0,35546.75\nR002,2022-11-07,8.6,0.0,35391.78\nR002,2022-11-08,8.9,0.0,35238.26\nR002,2022-11-09,9.0,1.5,19545.0\nR002,2022-11-10,9.0,0.3,34143.08\nR002,2022-11-11,8.4,0.3,34125.47\nR002,2022-11-12,7.7,0.0,34642.34\nR002,2022-11-13,8.6,0.0,34498.87\nR002,2022-11-14,7.8,1.2,33976.24\nR002,2022-11-15,6.4,0.0,33711.17\nR002,2022-11-16,9.4,0.8,30524.45\nR002,2022-11-17,9.7,1.6,31876.8\nR002,2022-11-18,11.1,6.6,18201.24\nR002,2022-11-19,12.6,3.6,20630.74\nR002,2022-11-20,14.6,3.9,29567.52\nR002,2022-11-21,15.1,0.0,9756.55\nR002,2022-11-22,15.1,3.4,16426.7\nR002,2022-11-23,11.7,1.4,30224.59\nR002,2022-11-24,13.0,0.0,30169.74\nR002,2022-11-25,13.8,0.3,26033.92\nR002,2022-11-26,11.3,7.7,12022.78\nR002,2022-11-27,10.2,10.7,17689.27\nR002,2022-11-28,8.1,0.0,30265.06\nR002,2022-11-29,8.9,0.0,27623.52\nR002,2022-11-30,10.3,0.0,30737.1\nR002,2022-12-01,10.7,0.0,30412.36\nR002,2022-12-02,9.9,3.3,2571.99\nR002,2022-12-03,9.4,3.5,4099.91\nR002,2022-12-04,8.4,2.5,21651.62\nR002,2022-12-05,8.0,2.2,26258.85\nR002,2022-12-06,6.0,0.1,29241.35\nR002,2022-12-07,5.4,0.0,29633.18\nR002,2022-12-08,6.8,0.0,21174.52\nR002,2022-12-09,8.6,0.0,26937.54\nR002,2022-12-10,9.5,0.0,30034.45\nR002,2022-12-11,9.6,0.0,30036.69\nR002,2022-12-12,10.7,2.7,12611.85\nR002,2022-12-13,10.2,1.5,25659.69\nR002,2022-12-14,8.4,23.4,0.0\nR002,2022-12-15,10.3,2.4,26318.17\nR002,2022-12-16,10.4,4.2,21771.35\nR002,2022-12-17,10.1,0.0,30406.42\nR002,2022-12-18,11.4,0.5,28836.74\nR002,2022-12-19,7.9,18.0,302.75\nR002,2022-12-20,2.6,0.1,30376.85\nR002,2022-12-21,3.3,0.0,25757.57\nR002,2022-12-22,5.6,0.0,31132.41\nR002,2022-12-23,5.8,0.0,31117.14\nR002,2022-12-24,4.2,0.0,31244.4\nR002,2022-12-25,4.0,0.0,30690.98\nR002,2022-12-26,4.9,0.4,21204.77\nR002,2022-12-27,5.4,1.0,19493.08\nR002,2022-12-28,4.3,16.3,21084.06\nR002,2022-12-29,-1.0,0.0,31444.45\nR002,2022-12-30,2.7,0.0,31490.63\nR002,2022-12-31,4.1,0.0,30945.07\nR002,2023-01-01,4.7,0.0,30916.53\nR002,2023-01-02,7.1,0.0,31644.71\nR002,2023-01-03,9.5,0.0,31700.78\nR002,2023-01-04,7.6,1.1,31758.98\nR002,2023-01-05,4.3,0.8,31819.15\nR002,2023-01-06,4.5,28.5,0.0\nR002,2023-01-07,0.2,8.5,11073.83\nR002,2023-01-08,-4.1,0.2,29515.95\nR002,2023-01-09,-5.8,0.1,32076.86\nR002,2023-01-10,-2.9,0.0,32144.83\nR002,2023-01-11,1.6,0.0,32213.94\nR002,2023-01-12,2.3,0.0,32284.04\nR002,2023-01-13,2.2,0.0,32355.0\nR002,2023-01-14,0.3,0.0,32400.0\nR002,2023-01-15,2.2,0.0,32400.0\nR002,2023-01-16,3.9,0.0,32400.0\nR002,2023-01-17,5.8,0.0,32354.67\nR002,2023-01-18,8.0,0.0,32400.0\nR002,2023-01-19,9.2,0.0,32400.0\nR002,2023-01-20,10.0,0.0,32400.0\nR002,2023-01-21,10.9,0.0,32400.0\nR002,2023-01-22,8.3,0.0,30730.23\nR002,2023-01-23,6.5,0.0,32400.0\nR002,2023-01-24,4.7,0.0,32400.0\nR002,2023-01-25,3.2,0.0,32400.0\nR002,2023-01-26,2.8,0.0,32400.0\nR002,2023-01-27,4.2,0.0,32400.0\nR002,2023-01-28,6.8,1.9,3866.23\nR002,2023-01-29,6.4,3.7,32213.55\nR002,2023-01-30,4.4,5.0,29829.12\nR002,2023-01-31,3.8,0.3,31846.57\nR002,2023-02-01,3.8,4.1,21182.22\nR002,2023-02-02,2.1,7.1,7597.27\nR002,2023-02-03,1.2,11.2,28406.11\nR002,2023-02-04,0.7,7.6,3785.18\nR002,2023-02-05,0.0,0.7,22481.32\nR002,2023-02-06,-0.4,2.6,15979.44\nR002,2023-02-07,0.1,10.4,1599.47\nR002,2023-02-08,-1.0,3.9,18189.61\nR002,2023-02-09,-1.4,2.0,33584.8\nR002,2023-02-10,-0.0,2.1,26589.68\nR002,2023-02-11,0.5,2.2,33406.24\nR002,2023-02-12,-0.1,3.1,31804.8\nR002,2023-02-13,-2.6,14.1,34843.65\nR002,2023-02-14,-1.6,5.5,24629.76\nR002,2023-02-15,-1.4,14.7,12048.6\nR002,2023-02-16,-4.1,0.3,35629.77\nR002,2023-02-17,-2.3,6.0,0.0\nR002,2023-02-18,3.6,12.6,636.67\nR002,2023-02-19,4.9,33.7,0.0\nR002,2023-02-20,4.8,36.4,0.0\nR002,2023-02-21,0.9,20.9,9222.01\nR002,2023-02-22,3.4,21.4,23345.85\nR002,2023-02-23,-0.3,1.4,36511.05\nR002,2023-02-24,-1.1,0.0,38783.98\nR002,2023-02-25,6.7,0.0,36895.71\nR002,2023-02-26,10.1,0.0,38571.62\nR002,2023-02-27,11.9,0.0,36251.2\nR002,2023-02-28,12.2,0.3,34572.45\nR002,2023-03-01,8.6,11.0,0.0\nR002,2023-03-02,5.9,0.0,38070.88\nR002,2023-03-03,5.6,3.0,28800.0\nR002,2023-03-04,6.5,15.7,18869.94\nR002,2023-03-05,7.3,1.4,18787.17\nR002,2023-03-06,6.1,32.1,4047.46\nR002,2023-03-07,5.2,0.0,39600.0\nR002,2023-03-08,8.7,0.0,39600.0\nR002,2023-03-09,10.3,0.8,39600.0\nR002,2023-03-10,8.7,7.5,15913.26\nR002,2023-03-11,10.1,3.6,23504.18\nR002,2023-03-12,12.7,8.6,33752.39\nR002,2023-03-13,7.6,26.6,612.86\nR002,2023-03-14,9.1,2.2,12365.11\nR002,2023-03-15,12.3,0.2,39600.0\nR002,2023-03-16,11.7,0.8,34743.56\nR002,2023-03-17,11.3,4.6,31237.36\nR002,2023-03-18,10.8,6.0,10380.62\nR002,2023-03-19,11.1,0.0,39600.0\nR002,2023-03-20,9.9,5.5,19850.44\nR002,2023-03-21,9.3,3.3,8053.24\nR002,2023-03-22,11.0,0.0,38894.45\nR002,2023-03-23,11.2,2.4,24986.17\nR002,2023-03-24,10.3,5.4,35154.49\nR002,2023-03-25,10.9,0.6,33637.87\nR002,2023-03-26,12.1,13.2,35637.51\nR002,2023-03-27,8.9,4.3,25917.92\nR002,2023-03-28,11.9,4.7,42403.55\nR002,2023-03-29,7.2,32.5,220.96\nR002,2023-03-30,1.9,8.5,39164.32\nR002,2023-03-31,1.0,0.0,43858.85\nR002,2023-04-01,5.5,0.0,45127.33\nR002,2023-04-02,10.3,0.0,42944.53\nR002,2023-04-03,11.7,0.6,22773.54\nR002,2023-04-04,14.7,0.0,42317.98\nR002,2023-04-05,14.2,3.4,29225.3\nR002,2023-04-06,9.6,7.9,7918.06\nR002,2023-04-07,10.5,4.4,29667.48\nR002,2023-04-08,8.9,8.2,4732.39\nR002,2023-04-09,11.7,0.1,43622.8\nR002,2023-04-10,12.1,2.4,27235.47\nR002,2023-04-11,13.5,12.9,11590.59\nR002,2023-04-12,12.9,11.4,15716.63\nR002,2023-04-13,10.2,20.9,28820.1\nR002,2023-04-14,8.8,13.2,42964.21\nR002,2023-04-15,11.6,0.0,46635.81\nR002,2023-04-16,15.3,0.3,46707.41\nR002,2023-04-17,16.8,2.5,44817.96\nR002,2023-04-18,17.4,1.5,40305.14\nR002,2023-04-19,18.0,1.7,46800.0\nR002,2023-04-20,13.4,19.7,5526.16\nR002,2023-04-21,10.5,16.3,28059.83\nR002,2023-04-22,8.8,17.6,28337.67\nR002,2023-04-23,11.2,0.0,44107.92\nR002,2023-04-24,11.2,12.9,5694.97\nR002,2023-04-25,11.5,4.7,43385.72\nR002,2023-04-26,14.6,0.3,47050.14\nR002,2023-04-27,17.1,2.1,42600.92\nR002,2023-04-28,14.6,11.3,33910.6\nR002,2023-04-29,12.2,13.5,3181.06\nR002,2023-04-30,11.3,9.2,30043.6\nR002,2023-05-01,10.0,3.2,42496.62\nR002,2023-05-02,9.3,7.3,38861.95\nR002,2023-05-03,11.5,1.2,46786.4\nR002,2023-05-04,14.0,0.0,48757.53\nR002,2023-05-05,15.7,0.0,47372.32\nR002,2023-05-06,17.0,0.1,33901.77\nR002,2023-05-07,14.5,11.4,16528.77\nR002,2023-05-08,12.4,9.5,7924.95\nR002,2023-05-09,12.4,3.5,23226.64\nR002,2023-05-10,11.3,11.6,722.9\nR002,2023-05-11,11.0,8.0,30045.29\nR002,2023-05-12,14.2,0.4,40881.17\nR002,2023-05-13,14.5,17.6,1368.11\nR002,2023-05-14,12.4,15.6,10531.02\nR002,2023-05-15,13.4,0.7,48383.89\nR002,2023-05-16,15.3,0.3,50400.0\nR002,2023-05-17,16.0,0.0,50431.02\nR002,2023-05-18,16.8,0.0,50400.0\nR002,2023-05-19,17.7,2.3,48292.39\nR002,2023-05-20,17.9,5.3,44155.95\nR002,2023-05-21,16.6,15.0,28206.03\nR002,2023-05-22,17.7,9.8,47256.82\nR002,2023-05-23,19.9,4.7,50400.0\nR002,2023-05-24,20.5,4.3,50593.18\nR002,2023-05-25,20.1,3.3,48034.32\nR002,2023-05-26,18.4,7.7,50315.92\nR002,2023-05-27,18.8,2.9,49349.71\nR002,2023-05-28,20.1,3.1,44947.7\nR002,2023-05-29,20.1,5.4,34600.05\nR002,2023-05-30,18.9,16.4,28810.35\nR002,2023-05-31,18.3,4.4,48570.64\nR002,2023-06-01,17.9,5.4,45087.41\nR002,2023-06-02,18.8,0.5,51828.12\nR002,2023-06-03,21.2,1.2,51082.81\nR002,2023-06-04,20.8,8.6,39917.93\nR002,2023-06-05,19.5,5.5,41158.73\nR002,2023-06-06,18.8,4.0,30611.3\nR002,2023-06-07,19.6,3.2,46834.46\nR002,2023-06-08,20.7,0.1,51474.59\nR002,2023-06-09,22.2,1.9,39493.43\nR002,2023-06-10,20.1,19.2,23298.26\nR002,2023-06-11,21.2,3.8,47246.72\nR002,2023-06-12,20.8,9.4,37794.64\nR002,2023-06-13,20.6,19.9,43449.76\nR002,2023-06-14,20.3,19.4,29833.73\nR002,2023-06-15,20.0,13.6,42236.06\nR002,2023-06-16,20.1,3.6,46843.71\nR002,2023-06-17,21.0,0.4,50555.93\nR002,2023-06-18,21.1,2.9,53069.97\nR002,2023-06-19,19.8,14.5,47121.59\nR002,2023-06-20,19.8,9.5,38456.79\nR002,2023-06-21,20.6,0.9,49724.68\nR002,2023-06-22,20.5,0.7,46759.62\nR002,2023-06-23,20.9,1.0,53400.23\nR002,2023-06-24,21.4,5.0,51925.03\nR002,2023-06-25,23.0,0.2,53969.76\nR002,2023-06-26,23.5,2.9,53972.21\nR002,2023-06-27,21.5,5.0,46579.64\nR002,2023-06-28,21.8,0.4,53749.01\nR002,2023-06-29,21.8,2.7,53100.44\nR002,2023-06-30,18.4,1.2,52165.43\nR002,2023-07-01,18.8,1.2,53951.18\nR002,2023-07-02,20.5,0.3,50383.87\nR002,2023-07-03,22.8,0.1,53928.15\nR002,2023-07-04,24.3,0.6,53913.48\nR002,2023-07-05,24.8,0.5,53896.71\nR002,2023-07-06,26.0,0.0,53877.81\nR002,2023-07-07,25.3,14.4,47180.99\nR002,2023-07-08,22.8,12.8,35444.05\nR002,2023-07-09,20.2,17.9,3093.8\nR002,2023-07-10,19.7,33.6,8132.72\nR002,2023-07-11,17.4,3.4,48962.62\nR002,2023-07-12,18.3,1.0,50272.58\nR002,2023-07-13,21.6,0.0,53686.19\nR002,2023-07-14,22.7,0.0,52877.69\nR002,2023-07-15,22.4,1.1,49458.51\nR002,2023-07-16,20.4,0.2,53230.54\nR002,2023-07-17,19.8,0.0,51001.09\nR002,2023-07-18,20.3,0.4,52420.71\nR002,2023-07-19,20.6,0.7,52436.97\nR002,2023-07-20,22.0,0.5,50959.96\nR002,2023-07-21,23.0,2.0,51331.65\nR002,2023-07-22,24.2,1.1,48920.52\nR002,2023-07-23,24.0,1.0,50019.21\nR002,2023-07-24,23.1,4.7,46745.0\nR002,2023-07-25,22.6,0.6,49761.45\nR002,2023-07-26,23.7,0.1,51502.37\nR002,2023-07-27,27.1,0.2,51815.18\nR002,2023-07-28,21.0,19.7,29066.59\nR002,2023-07-29,19.8,0.2,48219.53\nR002,2023-07-30,21.6,0.4,47389.43\nR002,2023-07-31,23.2,0.7,50058.78\nR002,2023-08-01,23.7,1.4,48348.67\nR002,2023-08-02,24.1,1.1,46821.87\nR002,2023-08-03,23.2,6.9,47134.48\nR002,2023-08-04,22.8,0.2,47879.28\nR002,2023-08-05,24.9,0.0,48282.71\nR002,2023-08-06,28.1,0.0,48684.65\nR002,2023-08-07,28.0,0.2,48406.69\nR002,2023-08-08,27.0,0.6,47295.47\nR002,2023-08-09,26.5,1.2,47324.17\nR002,2023-08-10,27.3,1.4,47690.74\nR002,2023-08-11,27.7,0.8,47382.19\nR002,2023-08-12,26.3,3.5,42967.48\nR002,2023-08-13,26.9,1.6,46180.28\nR002,2023-08-14,26.8,0.5,46800.0\nR002,2023-08-15,27.2,0.5,46832.6\nR002,2023-08-16,27.3,0.0,47197.06\nR002,2023-08-17,27.0,0.0,47227.74\nR002,2023-08-18,25.7,0.4,46926.88\nR002,2023-08-19,24.6,1.2,46800.0\nR002,2023-08-20,25.0,0.0,46800.0\nR002,2023-08-21,24.8,0.3,46800.0\nR002,2023-08-22,24.4,4.5,46800.0\nR002,2023-08-23,23.6,0.0,46800.0\nR002,2023-08-24,23.2,0.0,46800.0\nR002,2023-08-25,21.5,0.3,46800.0\nR002,2023-08-26,21.6,0.0,46800.0\nR002,2023-08-27,22.5,0.0,46767.13\nR002,2023-08-28,23.6,0.0,46665.97\nR002,2023-08-29,23.7,0.0,46563.54\nR002,2023-08-30,24.0,0.5,46407.65\nR002,2023-08-31,25.9,0.0,46355.35\nR002,2023-09-01,26.2,0.0,45213.47\nR002,2023-09-02,24.5,0.4,44619.74\nR002,2023-09-03,21.5,9.7,6657.78\nR002,2023-09-04,21.8,14.0,27258.42\nR002,2023-09-05,23.1,1.4,43810.87\nR002,2023-09-06,23.8,1.2,43223.45\nR002,2023-09-07,22.2,5.6,34049.55\nR002,2023-09-08,20.9,0.3,40902.15\nR002,2023-09-09,17.8,6.7,35486.52\nR002,2023-09-10,15.0,0.7,42934.92\nR002,2023-09-11,15.2,1.6,41776.86\nR002,2023-09-12,16.3,1.4,41502.59\nR002,2023-09-13,15.9,1.7,40778.99\nR002,2023-09-14,15.8,0.4,43074.79\nR002,2023-09-15,17.6,0.0,43058.67\nR002,2023-09-16,17.6,2.0,40996.1\nR002,2023-09-17,17.6,0.0,38179.04\nR002,2023-09-18,16.9,0.0,42856.64\nR002,2023-09-19,17.7,0.0,42789.0\nR002,2023-09-20,19.2,0.0,42721.23\nR002,2023-09-21,20.0,0.0,42529.71\nR002,2023-09-22,21.0,0.3,42564.5\nR002,2023-09-23,20.9,0.2,41007.49\nR002,2023-09-24,21.0,0.4,41679.67\nR002,2023-09-25,19.7,1.2,41396.68\nR002,2023-09-26,18.9,0.3,39332.36\nR002,2023-09-27,20.0,0.0,41149.74\nR002,2023-09-28,20.8,1.9,40650.0\nR002,2023-09-29,20.6,0.3,39361.78\nR002,2023-09-30,20.3,0.3,40498.85\nR002,2023-10-01,18.2,4.6,36473.49\nR002,2023-10-02,17.2,8.5,4745.87\nR002,2023-10-03,16.1,2.3,37400.94\nR002,2023-10-04,14.9,0.8,35940.04\nR002,2023-10-05,15.1,0.1,38944.11\nR002,2023-10-06,14.1,0.0,37163.13\nR002,2023-10-07,14.1,0.2,37229.05\nR002,2023-10-08,15.6,0.0,37954.49\nR002,2023-10-09,10.2,26.2,29468.08\nR002,2023-10-10,7.2,4.8,9633.14\nR002,2023-10-11,8.5,4.7,32922.16\nR002,2023-10-12,8.1,0.0,37173.7\nR002,2023-10-13,11.0,0.0,36903.41\nR002,2023-10-14,13.4,0.0,36631.93\nR002,2023-10-15,15.7,0.0,36359.13\nR002,2023-10-16,15.9,2.2,36000.0\nR002,2023-10-17,10.7,10.2,35963.93\nR002,2023-10-18,9.4,4.3,11043.25\nR002,2023-10-19,10.9,2.8,28800.0\nR002,2023-10-20,13.2,0.0,36000.0\nR002,2023-10-21,17.4,0.0,36000.0\nR002,2023-10-22,18.0,0.0,36000.0\nR002,2023-10-23,18.7,0.0,33457.27\nR002,2023-10-24,17.3,1.8,31561.64\nR002,2023-10-25,14.6,0.5,28800.0\nR002,2023-10-26,16.8,6.0,10058.27\nR002,2023-10-27,17.2,5.1,14400.6\nR002,2023-10-28,18.7,0.7,28409.9\nR002,2023-10-29,17.1,2.4,31581.71\nR002,2023-10-30,15.5,0.0,36000.0\nR002,2023-10-31,16.9,0.0,36000.0\nR002,2023-11-01,18.8,0.0,32400.0\nR002,2023-11-02,17.2,2.7,28321.13\nR002,2023-11-03,15.2,12.6,15553.88\nR002,2023-11-04,15.2,0.0,35803.93\nR002,2023-11-05,17.6,0.2,35723.61\nR002,2023-11-06,15.0,17.2,15022.61\nR002,2023-11-07,12.3,0.4,34914.67\nR002,2023-11-08,11.9,0.0,33830.36\nR002,2023-11-09,12.3,18.6,11240.38\nR002,2023-11-10,7.2,0.0,34966.13\nR002,2023-11-11,10.6,0.0,34820.06\nR002,2023-11-12,15.9,5.1,33764.15\nR002,2023-11-13,12.9,21.3,2482.24\nR002,2023-11-14,11.8,31.3,13788.26\nR002,2023-11-15,10.2,0.3,19432.37\nR002,2023-11-16,10.6,55.7,0.0\nR002,2023-11-17,6.1,1.6,32608.67\nR002,2023-11-18,8.7,0.0,16772.04\nR002,2023-11-19,11.6,31.5,0.0\nR002,2023-11-20,6.8,15.7,1828.56\nR002,2023-11-21,4.0,4.2,22507.73\nR002,2023-11-22,6.1,6.2,24201.32\nR002,2023-11-23,6.4,32.6,15547.48\nR002,2023-11-24,7.1,18.3,0.0\nR002,2023-11-25,11.2,23.5,11086.13\nR002,2023-11-26,13.2,25.1,27874.36\nR002,2023-11-27,2.0,21.7,11641.47\nR002,2023-11-28,0.1,7.2,3884.14\nR002,2023-11-29,4.5,3.7,16294.84\nR002,2023-11-30,8.3,20.7,0.0\nR002,2023-12-01,10.3,0.2,24625.98\nR002,2023-12-02,13.4,0.1,26794.11\nR002,2023-12-03,13.3,0.0,27018.45\nR002,2023-12-04,13.3,0.8,29985.78\nR002,2023-12-05,10.1,5.2,15561.41\nR002,2023-12-06,10.2,0.4,19531.55\nR002,2023-12-07,9.3,0.0,29246.63\nR002,2023-12-08,11.8,0.7,25895.46\nR002,2023-12-09,11.8,3.7,0.0\nR002,2023-12-10,9.2,13.7,3592.13\nR002,2023-12-11,9.4,0.8,20537.3\nR002,2023-12-12,8.9,11.8,3026.47\nR002,2023-12-13,6.6,7.7,2128.61\nR002,2023-12-14,6.0,0.0,26567.88\nR002,2023-12-15,10.8,0.0,11009.03\nR002,2023-12-16,10.9,3.0,21925.29\nR002,2023-12-17,4.5,24.8,17889.59\nR002,2023-12-18,2.5,7.3,5896.79\nR002,2023-12-19,5.1,4.3,13718.96\nR002,2023-12-20,5.5,0.0,30771.12\nR002,2023-12-21,8.3,0.5,20368.91\nR002,2023-12-22,9.4,5.7,4589.43\nR002,2023-12-23,4.8,36.7,367.28\nR002,2023-12-24,-0.4,27.7,4022.06\nR002,2023-12-25,1.6,22.2,1789.54\nR002,2023-12-26,7.7,14.7,6181.2\nR002,2023-12-27,8.6,6.4,16967.63\nR002,2023-12-28,1.9,2.0,31386.22\nR002,2023-12-29,1.3,0.0,31432.22\nR002,2023-12-30,7.3,0.0,31480.58\nR002,2023-12-31,6.4,0.3,26393.9\nR002,2024-01-01,5.6,0.0,31583.92\nR002,2024-01-02,7.7,1.0,31279.63\nR002,2024-01-03,5.3,13.6,20738.57\nR002,2024-01-04,5.9,29.5,0.0\nR002,2024-01-05,5.6,19.9,0.0\nR002,2024-01-06,4.8,1.6,31874.55\nR002,2024-01-07,9.3,0.0,28677.0\nR002,2024-01-08,7.7,22.1,1565.73\nR002,2024-01-09,5.2,12.4,2504.43\nR002,2024-01-10,3.3,0.3,25783.99\nR002,2024-01-11,2.9,3.2,30134.49\nR002,2024-01-12,1.0,27.1,635.64\nR002,2024-01-13,-5.7,0.9,27295.67\nR002,2024-01-14,-4.7,0.0,32400.0\nR002,2024-01-15,-0.2,0.0,31846.88\nR002,2024-01-16,5.0,17.5,15366.56\nR002,2024-01-17,4.2,7.3,25842.4\nR002,2024-01-18,4.4,0.0,32400.0\nR002,2024-01-19,8.0,5.3,17962.93\nR002,2024-01-20,8.0,12.5,24001.67\nR002,2024-01-21,5.7,39.1,0.0\nR002,2024-01-22,2.4,13.4,20449.57\nR002,2024-01-23,0.0,0.6,32400.0\nR002,2024-01-24,-1.2,0.0,32400.0\nR002,2024-01-25,-0.9,0.2,29024.27\nR002,2024-01-26,2.0,0.7,17230.35\nR002,2024-01-27,1.4,0.8,26799.41\nR002,2024-01-28,-0.3,2.6,32400.0\nR002,2024-01-29,-0.8,0.1,32400.0\nR002,2024-01-30,-1.2,0.0,32472.53\nR002,2024-01-31,0.4,0.0,31413.18\nR002,2024-02-01,3.5,0.0,32400.0\nR002,2024-02-02,3.3,4.1,16415.48\nR002,2024-02-03,2.2,13.4,21753.81\nR002,2024-02-04,2.7,17.0,5782.66\nR002,2024-02-05,2.6,68.6,0.0\nR002,2024-02-06,4.2,27.0,5200.43\nR002,2024-02-07,7.5,3.7,28720.7\nR002,2024-02-08,8.2,4.8,22084.37\nR002,2024-02-09,9.3,7.0,28263.02\nR002,2024-02-10,9.9,0.0,32828.95\nR002,2024-02-11,11.5,0.0,19482.15\nR002,2024-02-12,12.6,0.0,32400.0\nR002,2024-02-13,12.0,0.0,25693.12\nR002,2024-02-14,11.7,0.2,29243.03\nR002,2024-02-15,8.3,4.1,12644.43\nR002,2024-02-16,8.3,3.2,28713.09\nR002,2024-02-17,7.3,5.2,28939.5\nR002,2024-02-18,6.4,5.5,30706.69\nR002,2024-02-19,4.3,2.9,29738.32\nR002,2024-02-20,3.6,1.8,34996.61\nR002,2024-02-21,3.3,0.0,37118.39\nR002,2024-02-22,4.0,2.2,23517.45\nR002,2024-02-23,5.3,3.1,36176.1\nR002,2024-02-24,5.3,0.0,38730.03\nR002,2024-02-25,5.4,0.0,38905.6\nR002,2024-02-26,5.9,0.0,39082.43\nR002,2024-02-27,6.1,0.0,39260.39\nR002,2024-02-28,5.0,0.0,39403.81\nR002,2024-02-29,4.8,0.0,39505.24\nR002,2024-03-01,4.9,0.0,39600.0\nR002,2024-03-02,7.4,0.0,37332.31\nR002,2024-03-03,8.3,0.0,27873.11\nR002,2024-03-04,6.9,0.9,3545.25\nR002,2024-03-05,8.4,0.1,37365.82\nR002,2024-03-06,8.0,6.5,6831.47\nR002,2024-03-07,6.4,15.7,2901.54\nR002,2024-03-08,4.7,8.5,24801.72\nR002,2024-03-09,4.5,1.4,39600.0\nR002,2024-03-10,4.3,0.0,39600.0\nR002,2024-03-11,4.5,0.2,38076.95\nR002,2024-03-12,6.5,0.0,39658.81\nR002,2024-03-13,12.0,0.7,21627.82\nR002,2024-03-14,11.0,2.5,19479.49\nR002,2024-03-15,8.3,3.5,24151.37\nR002,2024-03-16,8.2,4.3,24431.17\nR002,2024-03-17,7.0,3.5,38170.75\nR002,2024-03-18,8.3,0.1,39600.0\nR002,2024-03-19,9.0,0.0,36794.23\nR002,2024-03-20,7.8,0.4,38300.72\nR002,2024-03-21,7.8,0.7,36000.0\nR002,2024-03-22,6.8,10.1,0.0\nR002,2024-03-23,6.1,2.7,37370.31\nR002,2024-03-24,6.1,5.0,27192.2\nR002,2024-03-25,6.7,2.0,42442.37\nR002,2024-03-26,4.8,23.6,12842.58\nR002,2024-03-27,5.0,11.9,33656.14\nR002,2024-03-28,7.1,0.0,44571.53\nR002,2024-03-29,10.5,2.2,27061.87\nR002,2024-03-30,13.3,1.5,41649.43\nR002,2024-03-31,14.8,0.0,45068.05\nR002,2024-04-01,16.2,0.0,44863.72\nR002,2024-04-02,17.7,0.0,45573.14\nR002,2024-04-03,15.6,4.3,44048.5\nR002,2024-04-04,11.6,0.2,43407.31\nR002,2024-04-05,10.8,0.4,23679.71\nR002,2024-04-06,10.3,0.7,44658.62\nR002,2024-04-07,11.8,0.0,46118.32\nR002,2024-04-08,14.2,0.0,46189.93\nR002,2024-04-09,15.2,0.5,46261.54\nR002,2024-04-10,14.6,0.2,36368.07\nR002,2024-04-11,14.1,2.1,22302.74\nR002,2024-04-12,16.3,0.7,44788.06\nR002,2024-04-13,14.3,4.0,15602.83\nR002,2024-04-14,12.7,2.7,40114.53\nR002,2024-04-15,13.3,0.2,45788.7\nR002,2024-04-16,17.3,0.0,46761.88\nR002,2024-04-17,20.2,0.0,43200.0\nR002,2024-04-18,20.1,0.1,46800.0\nR002,2024-04-19,16.2,5.1,42302.1\nR002,2024-04-20,18.1,1.8,46800.0\nR002,2024-04-21,15.2,7.0,36657.61\nR002,2024-04-22,14.6,1.2,47192.46\nR002,2024-04-23,16.4,0.0,46204.23\nR002,2024-04-24,17.9,0.0,38630.69\nR002,2024-04-25,23.1,0.2,47063.81\nR002,2024-04-26,24.0,1.6,42371.51\nR002,2024-04-27,21.4,2.8,36434.52\nR002,2024-04-28,22.1,1.9,47272.17\nR002,2024-04-29,21.3,1.0,44690.85\nR002,2024-04-30,19.1,0.4,37647.62\nR002,2024-05-01,18.2,1.4,45231.36\nR002,2024-05-02,17.2,7.3,27118.77\nR002,2024-05-03,15.0,6.6,10172.78\nR002,2024-05-04,13.0,15.1,31621.56\nR002,2024-05-05,10.0,34.8,16138.21\nR002,2024-05-06,10.7,3.8,41028.56\nR002,2024-05-07,11.5,3.0,25637.93\nR002,2024-05-08,13.5,2.1,44309.39\nR002,2024-05-09,14.5,3.5,34620.4\nR002,2024-05-10,14.8,3.2,32942.45\nR002,2024-05-11,15.5,10.2,24554.14\nR002,2024-05-12,13.2,29.0,954.71\nR002,2024-05-13,11.9,3.9,39148.47\nR002,2024-05-14,10.5,11.8,23752.47\nR002,2024-05-15,9.6,3.5,47343.26\nR002,2024-05-16,10.1,3.2,50400.0\nR002,2024-05-17,10.4,3.9,44410.79\nR002,2024-05-18,11.8,0.8,47487.66\nR002,2024-05-19,12.6,6.4,46800.0\nR002,2024-05-20,13.6,3.4,48147.19\nR002,2024-05-21,16.1,0.8,50073.42\nR002,2024-05-22,18.0,1.0,50624.08\nR002,2024-05-23,18.9,2.7,50597.56\nR002,2024-05-24,18.2,2.8,50571.53\nR002,2024-05-25,15.6,8.1,43019.26\nR002,2024-05-26,15.7,11.9,20932.1\nR002,2024-05-27,17.3,3.2,47752.6\nR002,2024-05-28,18.5,8.5,39364.34\nR002,2024-05-29,18.6,10.4,29463.98\nR002,2024-05-30,20.8,2.3,51512.14\nR002,2024-05-31,20.3,8.8,42964.58\nR002,2024-06-01,19.6,2.4,30825.24\nR002,2024-06-02,20.9,0.1,51817.44\nR002,2024-06-03,21.9,0.7,51435.72\nR002,2024-06-04,23.5,0.4,51781.64\nR002,2024-06-05,25.5,0.0,52820.64\nR002,2024-06-06,24.1,0.3,50652.38\nR002,2024-06-07,21.9,2.8,44674.5\nR002,2024-06-08,21.5,0.8,46186.6\nR002,2024-06-09,19.4,6.1,35109.32\nR002,2024-06-10,19.7,4.8,34893.1\nR002,2024-06-11,20.8,7.8,49280.5\nR002,2024-06-12,22.8,0.8,50773.48\nR002,2024-06-13,22.9,0.6,53786.47\nR002,2024-06-14,23.6,0.4,53813.67\nR002,2024-06-15,23.3,3.5,51224.52\nR002,2024-06-16,21.1,10.6,47662.94\nR002,2024-06-17,22.6,1.0,52011.02\nR002,2024-06-18,23.8,0.9,50598.82\nR002,2024-06-19,24.0,1.1,53055.19\nR002,2024-06-20,23.1,3.6,51814.29\nR002,2024-06-21,22.5,0.2,53168.3\nR002,2024-06-22,20.4,0.2,53950.15\nR002,2024-06-23,22.0,0.0,53957.35\nR002,2024-06-24,24.7,0.0,53962.6\nR002,2024-06-25,23.1,2.0,51922.0\nR002,2024-06-26,19.4,2.5,51618.16\nR002,2024-06-27,18.1,0.6,50493.09\nR002,2024-06-28,19.8,0.1,53963.69\nR002,2024-06-29,21.0,0.0,53958.83\nR002,2024-06-30,21.9,0.0,53951.87\nR002,2024-07-01,21.2,1.7,35169.24\nR002,2024-07-02,19.8,14.8,30769.21\nR002,2024-07-03,21.2,10.4,51075.81\nR002,2024-07-04,24.4,0.6,53902.49\nR002,2024-07-05,26.5,1.8,53386.33\nR002,2024-07-06,25.7,0.5,53740.62\nR002,2024-07-07,25.1,1.1,37579.02\nR002,2024-07-08,25.5,0.6,45831.04\nR002,2024-07-09,25.3,6.2,41801.03\nR002,2024-07-10,24.4,6.3,44888.71\nR002,2024-07-11,24.0,2.5,51200.79\nR002,2024-07-12,25.3,4.3,53695.58\nR002,2024-07-13,25.1,0.0,53659.32\nR002,2024-07-14,25.4,0.0,53620.85\nR002,2024-07-15,25.4,0.0,48719.96\nR002,2024-07-16,26.1,0.2,53117.7\nR002,2024-07-17,27.2,0.7,49691.65\nR002,2024-07-18,26.2,0.2,53146.2\nR002,2024-07-19,25.3,0.0,51022.38\nR002,2024-07-20,24.5,0.3,48869.02\nR002,2024-07-21,24.3,1.7,52010.81\nR002,2024-07-22,23.8,3.0,50914.29\nR002,2024-07-23,23.1,2.2,45922.87\nR002,2024-07-24,22.6,2.7,48698.87\nR002,2024-07-25,22.9,1.5,47774.85\nR002,2024-07-26,22.6,3.5,33633.07\nR002,2024-07-27,22.0,10.3,34715.93\nR002,2024-07-28,22.9,0.0,48902.24\nR002,2024-07-29,23.2,0.0,51016.85\nR002,2024-07-30,22.7,1.9,49943.79\nR002,2024-07-31,21.5,0.2,48026.02\nR002,2024-08-01,21.6,0.8,46057.49\nR002,2024-08-02,22.0,0.6,47169.44\nR002,2024-08-03,23.9,0.4,49231.08\nR002,2024-08-04,24.5,0.2,48606.11\nR002,2024-08-05,23.5,3.8,47912.1\nR002,2024-08-06,22.2,2.6,45436.69\nR002,2024-08-07,22.1,1.1,46948.1\nR002,2024-08-08,22.3,0.8,47993.84\nR002,2024-08-09,23.2,1.0,45841.48\nR002,2024-08-10,23.7,0.8,45618.47\nR002,2024-08-11,23.8,0.9,40364.52\nR002,2024-08-12,22.6,2.4,44953.38\nR002,2024-08-13,21.3,0.0,47129.18\nR002,2024-08-14,20.7,4.9,44868.91\nR002,2024-08-15,20.5,0.0,46800.0\nR002,2024-08-16,21.1,0.0,46888.37\nR002,2024-08-17,21.4,0.0,46919.68\nR002,2024-08-18,23.4,0.0,46950.99\nR002,2024-08-19,25.5,0.0,46982.25\nR002,2024-08-20,25.5,0.6,46800.0\nR002,2024-08-21,26.1,0.3,46800.0\nR002,2024-08-22,26.3,1.0,43200.0\nR002,2024-08-23,25.4,3.0,46800.0\nR002,2024-08-24,23.4,7.7,22045.37\nR002,2024-08-25,23.7,19.3,38377.47\nR002,2024-08-26,20.7,17.0,3354.29\nR002,2024-08-27,23.2,1.0,46308.93\nR002,2024-08-28,24.5,0.0,46584.02\nR002,2024-08-29,24.2,1.1,46479.51\nR002,2024-08-30,23.0,5.0,35262.46\nR002,2024-08-31,21.9,4.8,44982.5\nR002,2024-09-01,23.6,1.3,44932.28\nR002,2024-09-02,24.2,4.9,43200.0\nR002,2024-09-03,23.5,5.1,44390.19\nR002,2024-09-04,22.5,3.4,43568.97\nR002,2024-09-05,22.0,2.8,42247.3\nR002,2024-09-06,20.8,3.2,35462.76\nR002,2024-09-07,21.1,1.5,41289.15\nR002,2024-09-08,21.8,3.9,41967.04\nR002,2024-09-09,21.8,6.3,38297.96\nR002,2024-09-10,23.7,2.7,40983.24\nR002,2024-09-11,25.4,0.0,40561.09\nR002,2024-09-12,24.8,3.9,43200.0\nR002,2024-09-13,22.1,3.2,41702.78\nR002,2024-09-14,21.8,0.7,42958.18\nR002,2024-09-15,21.9,0.0,42990.42\nR002,2024-09-16,20.3,10.4,32142.36\nR002,2024-09-17,19.1,2.5,36042.26\nR002,2024-09-18,18.0,4.6,37086.52\nR002,2024-09-19,17.7,6.8,34104.36\nR002,2024-09-20,16.9,9.3,10951.83\nR002,2024-09-21,17.3,2.9,21940.42\nR002,2024-09-22,17.9,0.7,31802.05\nR002,2024-09-23,16.8,15.7,14413.15\nR002,2024-09-24,17.3,7.0,33200.08\nR002,2024-09-25,17.4,15.7,25903.12\nR002,2024-09-26,17.5,1.1,36082.45\nR002,2024-09-27,18.5,0.1,40219.61\nR002,2024-09-28,20.0,0.0,40894.1\nR002,2024-09-29,20.7,0.0,40842.42\nR002,2024-09-30,20.2,0.2,40452.28\nR002,2024-10-01,16.4,21.2,15555.79\nR002,2024-10-02,14.9,19.0,1870.11\nR002,2024-10-03,15.7,1.9,29681.13\nR002,2024-10-04,15.5,0.0,36998.72\nR002,2024-10-05,16.8,0.0,39355.57\nR002,2024-10-06,19.8,0.0,39066.29\nR002,2024-10-07,21.2,0.0,37285.68\nR002,2024-10-08,16.8,5.8,29222.16\nR002,2024-10-09,16.5,0.0,37475.6\nR002,2024-10-10,17.8,0.6,32400.0\nR002,2024-10-11,17.2,0.6,36017.61\nR002,2024-10-12,17.0,0.0,36242.31\nR002,2024-10-13,15.6,6.1,7224.36\nR002,2024-10-14,14.6,0.6,30089.72\nR002,2024-10-15,14.0,0.0,33139.58\nR002,2024-10-16,13.1,1.5,32497.18\nR002,2024-10-17,11.6,7.8,19507.23\nR002,2024-10-18,9.0,6.8,11297.61\nR002,2024-10-19,7.5,7.7,21789.29\nR002,2024-10-20,6.7,14.4,0.0\nR002,2024-10-21,5.6,2.0,24941.27\nR002,2024-10-22,7.1,0.0,36000.0\nR002,2024-10-23,10.0,2.3,36000.0\nR002,2024-10-24,11.0,2.9,33906.65\nR002,2024-10-25,10.8,4.0,19601.06\nR002,2024-10-26,8.0,10.4,34973.52\nR002,2024-10-27,7.4,0.0,36000.0\nR002,2024-10-28,8.5,0.0,36000.0\nR002,2024-10-29,9.8,0.0,36000.0\nR002,2024-10-30,10.9,0.0,34776.45\nR002,2024-10-31,9.3,0.0,36000.0\nR002,2024-11-01,10.6,1.2,28367.1\nR002,2024-11-02,12.0,0.0,35901.77\nR002,2024-11-03,7.2,7.1,21756.96\nR002,2024-11-04,3.2,14.1,28542.86\nR002,2024-11-05,-0.2,13.7,35609.47\nR002,2024-11-06,-1.3,0.0,28968.09\nR002,2024-11-07,4.6,0.0,34542.77\nR002,2024-11-08,5.2,0.0,35159.57\nR002,2024-11-09,5.3,0.0,35013.43\nR002,2024-11-10,6.5,0.0,34868.24\nR002,2024-11-11,7.6,0.0,25792.11\nR002,2024-11-12,8.6,0.0,34581.41\nR002,2024-11-13,7.3,0.0,34440.1\nR002,2024-11-14,7.7,0.0,27659.25\nR002,2024-11-15,9.3,8.0,18591.35\nR002,2024-11-16,6.9,21.7,18000.05\nR002,2024-11-17,4.7,0.0,32968.72\nR002,2024-11-18,8.1,0.9,28553.15\nR002,2024-11-19,9.6,3.5,21584.03\nR002,2024-11-20,9.5,0.0,31854.2\nR002,2024-11-21,11.5,0.0,32050.63\nR002,2024-11-22,11.2,9.2,4984.03\nR002,2024-11-23,13.8,0.0,31084.07\nR002,2024-11-24,9.2,20.8,0.0\nR002,2024-11-25,5.8,3.0,14680.16\nR002,2024-11-26,5.7,9.8,23043.23\nR002,2024-11-27,5.3,6.9,0.0\nR002,2024-11-28,4.2,1.4,26218.71\nR002,2024-11-29,5.0,0.0,30298.63\nR002,2024-11-30,5.5,0.0,30384.62\nR002,2024-12-01,5.5,0.8,17960.2\nR002,2024-12-02,5.4,0.2,30348.52\nR002,2024-12-03,4.8,0.0,30744.13\nR002,2024-12-04,4.7,0.0,30374.76\nR002,2024-12-05,6.0,0.9,7514.0\nR002,2024-12-06,5.5,2.7,13434.28\nR002,2024-12-07,5.8,0.0,30403.45\nR002,2024-12-08,8.3,0.0,12904.0\nR002,2024-12-09,10.2,0.0,28870.26\nR002,2024-12-10,10.3,0.7,29258.42\nR002,2024-12-11,10.8,12.9,13914.56\nR002,2024-12-12,5.2,28.2,6256.76\nR002,2024-12-13,-0.9,14.7,5585.04\nR002,2024-12-14,-9.0,0.0,30419.79\nR002,2024-12-15,-1.8,1.4,25259.36\nR002,2024-12-16,2.8,18.3,9030.17\nR002,2024-12-17,3.3,60.3,0.0\nR002,2024-12-18,3.6,36.1,23872.74\nR002,2024-12-19,0.7,0.0,30777.7\nR002,2024-12-20,3.1,0.0,30766.25\nR002,2024-12-21,4.7,0.0,28800.0\nR002,2024-12-22,5.8,2.7,14116.82\nR002,2024-12-23,6.8,7.7,22113.07\nR002,2024-12-24,7.1,1.2,31094.29\nR002,2024-12-25,7.0,0.0,27083.03\nR002,2024-12-26,7.6,0.0,31337.33\nR002,2024-12-27,6.2,0.0,26635.05\nR002,2024-12-28,5.4,1.8,19288.85\nR002,2024-12-29,5.1,4.4,5843.3\nR002,2024-12-30,3.1,0.7,29071.36\nR002,2024-12-31,2.7,0.0,30924.16\nR002,2025-01-01,4.7,0.7,21178.16\nR002,2025-01-02,5.8,0.0,31685.61\nR002,2025-01-03,7.6,0.0,31741.79\nR002,2025-01-04,5.6,6.1,24391.01\nR002,2025-01-05,3.5,8.9,14964.44\nR002,2025-01-06,1.5,0.0,31917.21\nR002,2025-01-07,5.4,0.0,31977.83\nR002,2025-01-08,8.4,0.0,32040.56\nR002,2025-01-09,9.2,0.0,32105.42\nR002,2025-01-10,7.7,0.0,32172.25\nR002,2025-01-11,9.3,0.0,32240.91\nR002,2025-01-12,8.0,0.0,15436.65\nR002,2025-01-13,8.7,18.2,14488.57\nR002,2025-01-14,6.0,18.6,0.0\nR002,2025-01-15,5.6,0.3,32400.0\nR002,2025-01-16,4.3,1.9,31778.1\nR002,2025-01-17,3.3,3.9,20761.06\nR002,2025-01-18,2.7,8.2,25831.3\nR002,2025-01-19,2.1,11.3,14998.31\nR002,2025-01-20,-0.3,0.0,32400.0\nR002,2025-01-21,2.3,0.0,32400.0\nR002,2025-01-22,4.6,0.0,32400.0\nR002,2025-01-23,5.9,0.0,1271.05\nR002,2025-01-24,5.2,1.9,22277.39\nR002,2025-01-25,4.9,1.0,32400.0\nR002,2025-01-26,2.7,0.0,32400.0\nR002,2025-01-27,3.1,0.0,32400.0\nR002,2025-01-28,4.7,0.0,32529.05\nR002,2025-01-29,6.5,0.3,32400.0\nR002,2025-01-30,7.6,0.3,32443.78\nR002,2025-01-31,7.1,0.0,32752.47\nR002,2025-02-01,6.3,0.0,29605.06\nR002,2025-02-02,4.9,0.0,32663.62\nR002,2025-02-03,5.4,0.4,27904.05\nR002,2025-02-04,3.0,21.5,4573.71\nR002,2025-02-05,-1.6,0.5,28774.57\nR002,2025-02-06,1.5,0.4,28802.18\nR002,2025-02-07,1.4,0.4,29902.32\nR002,2025-02-08,0.9,0.1,33966.42\nR002,2025-02-09,-0.3,1.4,31591.94\nR002,2025-02-10,0.9,0.8,33440.97\nR002,2025-02-11,0.1,0.0,36080.81\nR002,2025-02-12,0.9,0.0,36530.28\nR002,2025-02-13,1.7,0.1,33732.9\nR002,2025-02-14,2.5,7.4,1720.53\nR002,2025-02-15,4.2,1.4,31048.93\nR002,2025-02-16,4.6,19.0,0.0\nR002,2025-02-17,3.5,24.1,0.0\nR002,2025-02-18,2.9,23.8,0.0\nR002,2025-02-19,1.1,4.2,29497.21\nR002,2025-02-20,-2.3,1.8,35458.43\nR002,2025-02-21,-4.3,2.3,36430.45\nR002,2025-02-22,-5.9,0.0,36832.09\nR002,2025-02-23,-6.0,0.0,38687.76\nR002,2025-02-24,-4.4,7.0,15810.11\nR002,2025-02-25,-3.8,8.9,17496.25\nR002,2025-02-26,-4.4,0.6,27746.28\nR002,2025-02-27,-2.4,0.0,39386.88\nR002,2025-02-28,-0.9,0.0,39488.33\nR002,2025-03-01,-1.0,0.0,39590.31\nR002,2025-03-02,1.0,0.0,36000.0\nR002,2025-03-03,0.7,7.6,0.0\nR002,2025-03-04,1.3,1.0,39600.0\nR002,2025-03-05,3.3,0.0,31542.17\nR002,2025-03-06,6.0,0.0,30749.1\nR002,2025-03-07,9.9,0.1,39600.0\nR002,2025-03-08,10.1,0.0,39600.0\nR002,2025-03-09,11.9,0.0,39600.0\nR002,2025-03-10,10.8,0.0,39600.0\nR002,2025-03-11,11.8,0.0,39600.0\nR002,2025-03-12,13.4,0.0,39600.0\nR002,2025-03-13,16.5,0.0,39621.6\nR002,2025-03-14,15.8,0.0,20882.56\nR002,2025-03-15,14.3,0.0,39600.0\nR002,2025-03-16,17.5,0.0,39776.0\nR002,2025-03-17,15.5,1.2,36697.67\nR002,2025-03-18,9.1,23.7,3061.34\nR002,2025-03-19,3.7,9.4,9188.19\nR002,2025-03-20,3.0,0.5,33310.42\nR002,2025-03-21,4.4,1.6,12671.76\nR002,2025-03-22,5.0,4.1,22363.5\nR002,2025-03-23,5.9,5.1,31172.25\nR002,2025-03-24,8.2,0.0,41566.46\nR002,2025-03-25,11.4,1.1,36630.8\nR002,2025-03-26,12.9,6.3,20686.54\nR002,2025-03-27,14.0,0.6,37537.22\nR002,2025-03-28,15.0,0.0,43277.16\nR002,2025-03-29,18.0,0.3,43671.35\nR002,2025-03-30,17.1,3.7,27887.43\nR002,2025-03-31,12.4,25.6,14474.28\nR002,2025-04-01,14.1,0.1,44520.06\nR002,2025-04-02,13.7,3.8,21510.23\nR002,2025-04-03,11.6,8.4,815.95\nR002,2025-04-04,11.7,2.4,35311.41\nR002,2025-04-05,10.1,5.2,26156.8\nR002,2025-04-06,11.1,7.5,36225.31\nR002,2025-04-07,13.4,2.4,42536.58\nR002,2025-04-08,8.6,18.3,10424.26\nR002,2025-04-09,5.1,28.1,0.0\nR002,2025-04-10,2.6,7.0,22472.29\nR002,2025-04-11,3.7,0.3,34032.42\nR002,2025-04-12,3.0,14.6,24485.64\nR002,2025-04-13,4.5,2.0,34318.21\nR002,2025-04-14,9.1,1.3,44964.98\nR002,2025-04-15,10.1,0.7,13209.43\nR002,2025-04-16,11.6,0.0,42156.16\nR002,2025-04-17,12.6,0.0,46800.0\nR002,2025-04-18,14.1,1.1,46800.0\nR002,2025-04-19,15.9,2.0,46800.0\nR002,2025-04-20,13.8,6.0,31075.7\nR002,2025-04-21,13.5,9.0,38150.35\nR002,2025-04-22,15.3,0.2,46876.18\nR002,2025-04-23,14.8,5.4,44325.83\nR002,2025-04-24,14.3,3.3,46782.69\nR002,2025-04-25,13.1,11.5,22778.65\nR002,2025-04-26,14.0,10.8,39171.8\nR002,2025-04-27,11.0,8.0,5703.73\nR002,2025-04-28,9.8,3.0,34620.76\nR002,2025-04-29,10.8,0.0,47907.39\nR002,2025-04-30,12.0,0.0,48538.61\nR002,2025-05-01,15.0,1.5,36131.64\nR002,2025-05-02,15.6,8.7,28897.94\nR002,2025-05-03,12.7,16.4,7502.03\nR002,2025-05-04,14.1,3.5,46212.05\nR002,2025-05-05,15.2,5.4,46800.0\nR002,2025-05-06,16.3,3.5,39821.76\nR002,2025-05-07,17.6,0.0,46943.14\nR002,2025-05-08,16.3,9.3,43488.09\nR002,2025-05-09,17.1,2.6,49142.76\nR002,2025-05-10,17.9,5.1,45300.38\nR002,2025-05-11,14.5,12.5,0.0\nR002,2025-05-12,15.6,6.5,33755.45\nR002,2025-05-13,13.2,15.3,559.84\nR002,2025-05-14,12.3,4.9,46270.23\nR002,2025-05-15,10.7,5.9,15807.49\nR002,2025-05-16,14.9,0.1,50134.26\nR002,2025-05-17,21.3,0.4,50415.63\nR002,2025-05-18,18.2,8.6,40865.22\nR002,2025-05-19,13.8,10.9,45391.03\nR002,2025-05-20,14.4,0.8,47915.61\nR002,2025-05-21,13.3,0.0,48458.96\nR002,2025-05-22,16.3,0.0,50985.72\nR002,2025-05-23,19.6,0.3,50603.67\nR002,2025-05-24,20.7,0.5,50935.54\nR002,2025-05-25,21.1,0.8,50911.34\nR002,2025-05-26,22.6,0.3,50400.0\nR002,2025-05-27,21.7,5.2,44557.94\nR002,2025-05-28,19.2,12.5,38783.06\nR002,2025-05-29,18.0,15.4,27072.74\nR002,2025-05-30,17.3,21.8,34525.6\nR002,2025-05-31,17.2,9.3,39482.36\nR002,2025-06-01,19.1,4.6,46800.0\nR002,2025-06-02,17.7,15.6,0.0\nR002,2025-06-03,18.2,0.5,47839.82\nR002,2025-06-04,18.7,0.0,50749.03\nR002,2025-06-05,18.5,2.2,48350.81\nR002,2025-06-06,19.1,0.0,53209.15\nR002,2025-06-07,19.5,0.0,53559.3\nR002,2025-06-08,21.9,0.0,53545.03\nR002,2025-06-09,22.6,0.2,53165.28\nR002,2025-06-10,22.4,0.2,52419.04\nR002,2025-06-11,22.1,1.3,52405.62\nR002,2025-06-12,20.3,5.2,50925.24\nR002,2025-06-13,21.3,0.2,50759.49\nR002,2025-06-14,18.1,3.9,39660.1\nR002,2025-06-15,15.3,2.1,48077.37\nR002,2025-06-16,14.5,6.3,42114.82\nR002,2025-06-17,15.0,5.6,37768.87\nR002,2025-06-18,16.4,1.3,51004.04\nR002,2025-06-19,18.4,0.4,53906.49\nR002,2025-06-20,19.3,0.0,53921.06\nR002,2025-06-21,16.6,10.9,33782.33\nR002,2025-06-22,16.7,0.9,51936.58\nR002,2025-06-23,15.5,2.4,48697.86\nR002,2025-06-24,17.6,2.4,50911.24\nR002,2025-06-25,19.7,0.1,53965.17\nR002,2025-06-26,21.2,0.2,53967.9\nR002,2025-06-27,21.7,0.4,53144.24\nR002,2025-06-28,21.1,0.0,53966.92\nR002,2025-06-29,18.4,8.7,30687.12\nR002,2025-06-30,18.2,0.2,48849.32\nR002,2025-07-01,17.6,3.0,49892.86\nR002,2025-07-02,17.8,0.5,53057.68\nR002,2025-07-03,17.6,0.2,51264.87\nR002,2025-07-04,18.6,1.4,50070.15\nR002,2025-07-05,20.3,1.1,51668.49\nR002,2025-07-06,22.2,0.4,53870.56\nR002,2025-07-07,23.2,0.8,53757.05\nR002,2025-07-08,24.8,0.6,53822.71\nR002,2025-07-09,25.7,0.6,53795.46\nR002,2025-07-10,25.3,4.7,53766.02\nR002,2025-07-11,24.8,0.7,53421.14\nR002,2025-07-12,26.4,0.1,52172.62\nR002,2025-07-13,27.0,1.0,53665.0\nR002,2025-07-14,25.9,2.4,52014.72\nR002,2025-07-15,24.1,1.4,52175.91\nR002,2025-07-16,23.0,10.6,44621.98\nR002,2025-07-17,23.8,3.8,50745.85\nR002,2025-07-18,25.6,0.0,53456.15\nR002,2025-07-19,25.3,1.5,52801.26\nR002,2025-07-20,22.5,14.7,39066.14\nR002,2025-07-21,23.5,6.0,48902.93\nR002,2025-07-22,24.5,1.3,49299.28\nR002,2025-07-23,24.9,0.1,50045.93\nR002,2025-07-24,25.4,0.0,49458.97\nR002,2025-07-25,24.8,0.0,51185.9\nR002,2025-07-26,24.5,0.1,51828.89\nR002,2025-07-27,25.8,0.0,51793.38\nR002,2025-07-28,27.4,0.0,51409.68\nR002,2025-07-29,29.6,0.0,51375.13\nR002,2025-07-30,29.8,0.1,50994.05\nR002,2025-07-31,27.2,3.8,47440.95\nR002,2025-08-01,26.3,2.6,37369.73\nR002,2025-08-02,26.9,17.0,44842.93\nR002,2025-08-03,24.9,10.7,46076.98\nR002,2025-08-04,26.0,3.6,43624.1\nR002,2025-08-05,26.1,4.1,44420.03\nR002,2025-08-06,27.5,0.6,46904.46\nR002,2025-08-07,27.1,3.6,48300.83\nR002,2025-08-08,24.5,9.6,40432.87\nR002,2025-08-09,24.1,4.7,39831.31\nR002,2025-08-10,23.1,0.2,47032.53\nR002,2025-08-11,22.5,0.3,47735.54\nR002,2025-08-12,22.4,0.0,47092.8\nR002,2025-08-13,21.2,0.0,47122.96\nR002,2025-08-14,20.4,0.1,47487.26\nR002,2025-08-15,20.8,0.3,47183.18\nR002,2025-08-16,20.8,0.6,46880.61\nR002,2025-08-17,22.2,0.0,47243.63\nR002,2025-08-18,23.9,0.0,46942.76\nR002,2025-08-19,24.1,0.5,46800.0\nR002,2025-08-20,23.0,9.3,46534.77\nR002,2025-08-21,21.6,13.2,45497.69\nR002,2025-08-22,22.6,0.4,46800.0\nR002,2025-08-23,24.2,0.0,46800.0\nR002,2025-08-24,23.6,0.5,46800.0\nR002,2025-08-25,21.9,2.4,43333.65\nR002,2025-08-26,20.2,2.7,22522.58\nR002,2025-08-27,22.0,0.0,45330.04\nR002,2025-08-28,23.7,0.3,46603.99\nR002,2025-08-29,25.0,0.0,46500.48\nR002,2025-08-30,26.4,0.0,46396.67\nR002,2025-08-31,26.3,0.9,46292.65\nR002,2025-09-01,26.7,0.1,46188.51\nR002,2025-09-02,23.9,3.8,41107.58\nR002,2025-09-03,21.8,5.1,37928.32\nR002,2025-09-04,22.0,0.6,44085.03\nR002,2025-09-05,22.6,0.1,39461.42\nR002,2025-09-06,22.9,0.0,44206.01\nR002,2025-09-07,19.9,11.0,34170.62\nR002,2025-09-08,19.1,6.1,39554.37\nR002,2025-09-09,19.1,17.2,18311.49\nR002,2025-09-10,18.4,7.7,30364.57\nR002,2025-09-11,17.8,5.3,36116.78\nR002,2025-09-12,18.6,6.3,40279.79\nR002,2025-09-13,17.2,0.0,41007.74\nR002,2025-09-14,17.4,0.7,43089.62\nR002,2025-09-15,18.4,1.4,43022.25\nR002,2025-09-16,17.9,3.9,36963.71\nR002,2025-09-17,18.7,0.5,40821.07\nR002,2025-09-18,16.7,14.7,36299.86\nR002,2025-09-19,15.0,3.7,12564.46\nR002,2025-09-20,16.1,2.0,40604.36\nR002,2025-09-21,15.4,0.1,40002.02\nR002,2025-09-22,16.0,0.0,42265.32\nR002,2025-09-23,18.5,0.0,42300.54\nR002,2025-09-24,19.1,0.0,42336.14\nR002,2025-09-25,18.3,0.3,38232.16\nR002,2025-09-26,15.0,8.7,20569.92\nR002,2025-09-27,14.1,2.1,29971.66\nR002,2025-09-28,14.3,1.1,31674.85\nR002,2025-09-29,15.0,10.5,0.0\nR002,2025-09-30,14.7,15.0,24603.18\nR002,2025-10-01,14.3,0.4,36648.87\nR002,2025-10-02,15.4,0.8,38727.62\nR002,2025-10-03,17.0,0.0,39103.15\nR002,2025-10-04,21.0,0.0,39541.58\nR002,2025-10-05,18.7,4.4,37402.4\nR002,2025-10-06,17.2,0.0,37173.57\nR002,2025-10-07,19.0,0.0,38561.09\nR002,2025-10-08,21.2,0.1,37686.33\nR002,2025-10-09,16.0,25.1,7851.33\nR002,2025-10-10,12.1,28.1,11283.04\nR002,2025-10-11,11.3,9.6,31678.09\nR002,2025-10-12,10.6,9.1,4581.72\nR002,2025-10-13,10.1,1.4,36280.89\nR002,2025-10-14,11.1,2.8,31120.0\nR002,2025-10-15,11.1,1.0,36057.11\nR002,2025-10-16,10.7,1.3,36108.28\nR002,2025-10-17,10.7,0.0,36159.37\nR002,2025-10-18,12.4,0.0,33646.39\nR002,2025-10-19,12.9,9.3,27772.85\nR002,2025-10-20,10.4,7.1,16695.66\nR002,2025-10-21,7.9,1.3,36000.0\nR002,2025-10-22,10.4,0.0,34322.08\nR002,2025-10-23,13.5,1.5,22671.19\nR002,2025-10-24,14.3,4.0,21611.25\nR002,2025-10-25,15.1,4.3,1346.74\nR002,2025-10-26,12.1,26.8,16492.81\nR002,2025-10-27,11.7,0.9,26725.55\nR002,2025-10-28,15.2,4.4,30379.25\nR002,2025-10-29,10.8,23.2,18418.18\nR002,2025-10-30,8.0,1.5,29952.2\nR002,2025-10-31,9.7,0.0,36000.0\nR002,2025-11-01,11.3,0.0,36000.0\nR002,2025-11-02,8.0,0.0,35922.85\nR002,2025-11-03,12.5,0.0,35844.67\nR002,2025-11-04,14.4,0.0,35766.11\nR002,2025-11-05,13.7,0.0,34247.04\nR002,2025-11-06,12.7,0.0,35505.85\nR002,2025-11-07,13.9,0.0,32430.99\nR002,2025-11-08,16.2,0.3,35201.76\nR002,2025-11-09,14.9,0.3,34514.72\nR002,2025-11-10,14.5,0.0,32016.4\nR002,2025-11-11,14.5,0.0,27270.67\nR002,2025-11-12,13.7,4.1,22868.9\nR002,2025-11-13,11.0,22.2,1801.62\nR002,2025-11-14,9.8,20.6,1511.56\nR002,2025-11-15,11.6,0.5,32989.83\nR002,2025-11-16,11.7,0.0,32361.68\nR002,2025-11-17,11.3,0.0,32605.78\nR002,2025-11-18,11.8,0.0,33326.54\nR002,2025-11-19,15.0,0.0,30771.57\nR002,2025-11-20,15.0,0.0,29631.95\nR002,2025-11-21,13.6,0.0,32401.0\nR002,2025-11-22,14.7,0.0,31367.17\nR002,2025-11-23,15.5,0.0,31063.93\nR002,2025-11-24,14.3,0.0,29378.28\nR002,2025-11-25,12.0,0.0,29081.2\nR002,2025-11-26,11.3,0.0,26743.08\nR002,2025-11-27,11.2,0.0,30626.1\nR002,2025-11-28,10.7,0.0,30648.27\nR002,2025-11-29,11.1,0.0,30669.15\nR002,2025-11-30,11.9,0.6,28860.49\nR002,2025-12-01,9.8,6.5,3239.37\nR002,2025-12-02,8.0,2.1,7031.82\nR002,2025-12-03,6.4,0.0,29472.95\nR002,2025-12-04,6.3,0.0,28841.16\nR002,2025-12-05,7.0,0.0,30383.01\nR002,2025-12-06,6.9,0.0,28857.18\nR002,2025-12-07,8.6,3.9,0.0\nR002,2025-12-08,9.9,11.2,26410.28\nR002,2025-12-09,8.7,0.2,26059.71\nR002,2025-12-10,6.5,1.8,28536.46\nR002,2025-12-11,6.5,5.6,14558.37\nR002,2025-12-12,5.5,6.0,18319.67\nR002,2025-12-13,3.6,12.6,10648.35\nR002,2025-12-14,-1.4,0.0,26572.85\nR002,2025-12-15,2.9,2.2,1607.5\nR002,2025-12-16,2.4,5.8,22824.27\nR002,2025-12-17,1.3,0.0,30016.75\nR002,2025-12-18,5.3,0.0,30007.63\nR002,2025-12-19,5.9,0.0,29996.95\nR002,2025-12-20,5.2,0.0,30766.5\nR002,2025-12-21,6.5,0.0,29579.93\nR002,2025-12-22,7.0,2.2,10324.76\nR002,2025-12-23,6.7,1.9,28768.63\nR002,2025-12-24,5.9,6.9,5442.85\nR002,2025-12-25,3.7,0.0,30003.52\nR002,2025-12-26,2.1,3.5,30272.87\nR002,2025-12-27,-0.9,13.8,14790.57\nR002,2025-12-28,-2.3,18.4,12431.41\nR002,2025-12-29,-0.9,13.8,10163.35\nR002,2025-12-30,-0.8,22.6,8419.04\nR002,2025-12-31,0.9,36.5,0.0\nR003,1996-01-01,4.7,0.9,27974.96\nR003,1996-01-02,4.9,0.0,31626.87\nR003,1996-01-03,5.7,0.2,1888.06\nR003,1996-01-04,6.0,0.0,31737.93\nR003,1996-01-05,6.5,0.2,16920.48\nR003,1996-01-06,5.4,0.0,31857.7\nR003,1996-01-07,5.3,3.9,15718.35\nR003,1996-01-08,4.4,2.0,23787.38\nR003,1996-01-09,2.2,0.0,32051.21\nR003,1996-01-10,0.2,0.0,32118.84\nR003,1996-01-11,-1.2,0.0,32187.78\nR003,1996-01-12,-0.2,0.0,32257.91\nR003,1996-01-13,-0.9,0.0,32329.08\nR003,1996-01-14,-1.2,0.0,32400.0\nR003,1996-01-15,0.0,0.0,32400.0\nR003,1996-01-16,-1.3,0.0,32400.0\nR003,1996-01-17,-1.2,0.0,32400.0\nR003,1996-01-18,-2.0,0.0,32400.0\nR003,1996-01-19,-2.0,2.5,30456.52\nR003,1996-01-20,-0.3,6.2,16980.8\nR003,1996-01-21,-0.3,5.7,6137.44\nR003,1996-01-22,-2.8,0.3,32400.0\nR003,1996-01-23,-3.2,0.5,26761.33\nR003,1996-01-24,-3.5,0.3,31481.38\nR003,1996-01-25,-0.1,0.3,28800.0\nR003,1996-01-26,2.5,0.0,32400.0\nR003,1996-01-27,3.9,0.0,30351.34\nR003,1996-01-28,5.2,2.0,3395.71\nR003,1996-01-29,4.9,5.3,15213.75\nR003,1996-01-30,3.7,8.1,3191.67\nR003,1996-01-31,3.0,11.8,1.56\nR003,1996-02-01,-3.9,10.5,30631.22\nR003,1996-02-02,-7.5,0.0,32700.41\nR003,1996-02-03,-3.7,0.0,33089.41\nR003,1996-02-04,-0.9,0.0,33704.15\nR003,1996-02-05,-0.2,0.0,33619.37\nR003,1996-02-06,1.8,6.4,2540.12\nR003,1996-02-07,4.9,28.4,15962.83\nR003,1996-02-08,6.6,3.2,19393.09\nR003,1996-02-09,8.6,5.8,19952.24\nR003,1996-02-10,5.7,5.1,5618.7\nR003,1996-02-11,4.2,14.2,0.0\nR003,1996-02-12,2.4,3.4,21352.66\nR003,1996-02-13,3.0,0.0,26548.84\nR003,1996-02-14,5.6,0.0,27074.19\nR003,1996-02-15,5.7,3.3,27978.59\nR003,1996-02-16,4.2,5.6,8988.85\nR003,1996-02-17,4.7,2.9,17089.18\nR003,1996-02-18,6.8,0.4,37654.32\nR003,1996-02-19,7.3,0.3,37828.42\nR003,1996-02-20,5.9,0.0,38003.31\nR003,1996-02-21,6.4,0.0,38178.82\nR003,1996-02-22,7.6,0.0,38354.79\nR003,1996-02-23,9.0,0.0,35630.05\nR003,1996-02-24,9.9,0.0,27277.01\nR003,1996-02-25,6.0,6.3,17519.13\nR003,1996-02-26,4.4,3.1,21434.34\nR003,1996-02-27,4.2,0.5,37270.78\nR003,1996-02-28,3.7,1.7,27036.78\nR003,1996-02-29,2.9,6.0,27736.69\nR003,1996-03-01,2.6,1.7,35982.65\nR003,1996-03-02,3.2,12.1,0.0\nR003,1996-03-03,3.8,0.9,39600.0\nR003,1996-03-04,3.3,13.2,13736.15\nR003,1996-03-05,3.3,2.9,36075.25\nR003,1996-03-06,5.4,14.8,0.0\nR003,1996-03-07,6.4,1.1,35028.6\nR003,1996-03-08,4.5,4.1,19192.79\nR003,1996-03-09,3.1,4.7,16926.55\nR003,1996-03-10,3.3,0.7,30223.94\nR003,1996-03-11,5.4,0.1,37596.1\nR003,1996-03-12,4.3,9.4,8170.99\nR003,1996-03-13,4.1,5.3,37825.09\nR003,1996-03-14,4.6,0.0,39363.12\nR003,1996-03-15,6.9,0.0,39839.5\nR003,1996-03-16,7.8,0.0,40089.51\nR003,1996-03-17,6.4,0.0,40035.22\nR003,1996-03-18,5.7,1.1,37029.06\nR003,1996-03-19,5.4,2.9,18256.69\nR003,1996-03-20,4.5,0.3,35612.17\nR003,1996-03-21,6.0,0.2,39183.51\nR003,1996-03-22,7.1,2.2,29650.3\nR003,1996-03-23,6.8,0.8,15297.18\nR003,1996-03-24,6.5,3.2,5603.46\nR003,1996-03-25,4.7,7.1,857.46\nR003,1996-03-26,4.2,11.0,19972.43\nR003,1996-03-27,3.9,6.4,21506.13\nR003,1996-03-28,6.2,0.0,39400.97\nR003,1996-03-29,7.3,9.9,35200.8\nR003,1996-03-30,7.6,11.0,13851.49\nR003,1996-03-31,7.0,1.7,40663.09\nR003,1996-04-01,9.6,0.7,43408.06\nR003,1996-04-02,9.3,4.8,8387.42\nR003,1996-04-03,9.2,0.0,35648.67\nR003,1996-04-04,10.4,0.1,45688.81\nR003,1996-04-05,10.1,0.0,45962.75\nR003,1996-04-06,10.2,0.4,44024.23\nR003,1996-04-07,9.3,3.4,44632.1\nR003,1996-04-08,9.0,11.9,9112.6\nR003,1996-04-09,7.5,4.1,41147.91\nR003,1996-04-10,6.5,3.2,39856.54\nR003,1996-04-11,5.7,0.6,46389.12\nR003,1996-04-12,7.0,1.3,30569.82\nR003,1996-04-13,6.9,7.1,0.0\nR003,1996-04-14,6.1,2.6,36552.6\nR003,1996-04-15,7.6,0.0,45816.35\nR003,1996-04-16,10.2,0.1,35200.78\nR003,1996-04-17,9.0,17.3,0.0\nR003,1996-04-18,7.0,31.3,17852.29\nR003,1996-04-19,4.3,7.3,9809.37\nR003,1996-04-20,5.1,2.5,44583.79\nR003,1996-04-21,7.7,1.2,20258.72\nR003,1996-04-22,7.1,11.5,0.0\nR003,1996-04-23,8.3,1.7,18746.26\nR003,1996-04-24,9.4,1.1,43618.01\nR003,1996-04-25,10.0,0.9,36598.84\nR003,1996-04-26,11.4,0.1,46800.0\nR003,1996-04-27,13.4,0.0,47650.5\nR003,1996-04-28,13.2,0.0,47944.42\nR003,1996-04-29,14.7,0.0,48574.8\nR003,1996-04-30,15.1,5.3,39408.16\nR003,1996-05-01,15.6,4.4,46800.0\nR003,1996-05-02,16.7,0.6,48805.49\nR003,1996-05-03,17.4,2.2,48435.1\nR003,1996-05-04,16.8,3.6,46414.3\nR003,1996-05-05,16.3,2.1,45132.83\nR003,1996-05-06,16.1,4.1,48164.17\nR003,1996-05-07,16.7,2.8,47474.35\nR003,1996-05-08,17.8,0.6,49637.68\nR003,1996-05-09,19.6,0.3,49952.37\nR003,1996-05-10,19.5,2.0,47529.71\nR003,1996-05-11,19.4,5.1,46058.39\nR003,1996-05-12,19.7,4.4,49190.36\nR003,1996-05-13,17.5,7.5,41981.7\nR003,1996-05-14,18.1,3.1,49226.4\nR003,1996-05-15,17.6,2.9,45823.98\nR003,1996-05-16,15.0,6.3,40558.97\nR003,1996-05-17,15.9,0.0,46589.41\nR003,1996-05-18,16.8,0.0,50400.0\nR003,1996-05-19,17.6,1.3,48892.66\nR003,1996-05-20,17.3,0.5,44619.46\nR003,1996-05-21,20.1,0.1,47329.03\nR003,1996-05-22,18.9,0.0,48577.64\nR003,1996-05-23,16.4,2.0,38731.65\nR003,1996-05-24,14.2,4.7,44506.15\nR003,1996-05-25,12.7,5.9,40098.25\nR003,1996-05-26,15.0,0.0,49126.45\nR003,1996-05-27,18.3,2.3,47688.33\nR003,1996-05-28,16.2,22.8,19122.03\nR003,1996-05-29,18.0,0.5,48769.81\nR003,1996-05-30,18.1,3.1,41885.38\nR003,1996-05-31,17.6,16.0,10237.6\nR003,1996-06-01,14.5,25.3,6154.43\nR003,1996-06-02,11.6,6.6,45995.95\nR003,1996-06-03,12.6,3.1,49937.74\nR003,1996-06-04,13.3,8.2,26583.11\nR003,1996-06-05,13.1,3.2,24963.01\nR003,1996-06-06,13.7,11.7,1747.63\nR003,1996-06-07,14.2,5.3,45028.63\nR003,1996-06-08,14.2,3.9,48952.27\nR003,1996-06-09,14.2,17.6,29660.02\nR003,1996-06-10,15.7,0.0,52177.96\nR003,1996-06-11,17.4,0.0,52777.32\nR003,1996-06-12,20.2,0.8,52397.99\nR003,1996-06-13,21.2,1.3,49299.26\nR003,1996-06-14,20.9,1.6,49276.49\nR003,1996-06-15,20.9,0.0,53097.85\nR003,1996-06-16,19.7,0.8,51447.81\nR003,1996-06-17,18.8,11.4,4651.51\nR003,1996-06-18,17.3,33.0,12617.7\nR003,1996-06-19,16.6,3.3,44142.67\nR003,1996-06-20,18.0,1.8,50598.99\nR003,1996-06-21,16.8,17.8,0.0\nR003,1996-06-22,18.3,3.5,35661.1\nR003,1996-06-23,20.2,0.4,51360.52\nR003,1996-06-24,21.7,3.2,52185.79\nR003,1996-06-25,22.7,2.8,51861.97\nR003,1996-06-26,23.6,0.3,52630.07\nR003,1996-06-27,25.5,0.0,53967.67\nR003,1996-06-28,24.6,0.2,53964.35\nR003,1996-06-29,18.4,31.0,10593.67\nR003,1996-06-30,17.8,0.1,53381.9\nR003,1996-07-01,20.1,0.3,53016.49\nR003,1996-07-02,21.6,0.1,52287.31\nR003,1996-07-03,23.8,0.0,53893.57\nR003,1996-07-04,25.9,0.2,53900.03\nR003,1996-07-05,25.2,3.1,53255.7\nR003,1996-07-06,24.2,1.4,53393.35\nR003,1996-07-07,22.8,2.6,51618.54\nR003,1996-07-08,23.6,2.2,52678.21\nR003,1996-07-09,24.4,3.8,53047.01\nR003,1996-07-10,26.3,0.9,53759.92\nR003,1996-07-11,26.4,0.7,53729.22\nR003,1996-07-12,25.5,2.8,52351.41\nR003,1996-07-13,25.4,4.4,51240.73\nR003,1996-07-14,25.3,2.1,44606.57\nR003,1996-07-15,25.3,0.2,53585.55\nR003,1996-07-16,24.4,2.3,51930.31\nR003,1996-07-17,23.3,3.3,51303.11\nR003,1996-07-18,21.1,12.4,20748.65\nR003,1996-07-19,21.8,8.4,46863.09\nR003,1996-07-20,20.9,19.0,41894.54\nR003,1996-07-21,22.3,5.7,44557.86\nR003,1996-07-22,22.1,4.5,43537.41\nR003,1996-07-23,21.3,11.0,26032.62\nR003,1996-07-24,19.1,42.2,837.6\nR003,1996-07-25,20.0,0.4,49085.02\nR003,1996-07-26,20.0,2.0,49389.74\nR003,1996-07-27,21.1,5.3,42042.38\nR003,1996-07-28,19.8,0.0,49235.59\nR003,1996-07-29,20.6,0.0,48572.81\nR003,1996-07-30,21.9,0.3,48303.61\nR003,1996-07-31,24.7,2.6,49153.63\nR003,1996-08-01,24.8,3.5,48143.71\nR003,1996-08-02,24.5,1.6,46800.0\nR003,1996-08-03,25.0,6.0,47852.53\nR003,1996-08-04,24.3,8.6,38240.06\nR003,1996-08-05,23.5,15.6,43325.87\nR003,1996-08-06,23.5,6.7,41647.56\nR003,1996-08-07,21.2,6.7,39108.77\nR003,1996-08-08,20.4,7.2,45570.17\nR003,1996-08-09,18.9,0.2,45115.09\nR003,1996-08-10,19.8,4.3,45949.74\nR003,1996-08-11,20.0,8.7,43792.26\nR003,1996-08-12,20.5,0.6,46800.0\nR003,1996-08-13,21.9,0.5,46800.0\nR003,1996-08-14,22.7,0.6,45028.06\nR003,1996-08-15,23.5,0.5,46846.75\nR003,1996-08-16,22.4,2.0,43000.28\nR003,1996-08-17,22.4,4.5,41310.19\nR003,1996-08-18,22.2,1.3,44168.45\nR003,1996-08-19,21.8,10.8,43548.6\nR003,1996-08-20,22.4,1.9,46070.05\nR003,1996-08-21,20.1,26.2,25121.78\nR003,1996-08-22,19.5,5.4,44428.54\nR003,1996-08-23,20.1,0.6,45656.05\nR003,1996-08-24,22.1,0.1,46800.0\nR003,1996-08-25,20.9,4.4,43218.71\nR003,1996-08-26,19.3,1.9,44350.81\nR003,1996-08-27,18.6,1.7,44358.52\nR003,1996-08-28,19.8,3.8,43231.02\nR003,1996-08-29,21.0,2.2,44129.1\nR003,1996-08-30,20.1,7.1,42631.33\nR003,1996-08-31,21.3,1.1,41468.42\nR003,1996-09-01,21.7,3.1,43000.35\nR003,1996-09-02,23.0,0.0,44056.51\nR003,1996-09-03,21.8,9.3,34909.24\nR003,1996-09-04,19.5,4.9,38607.77\nR003,1996-09-05,19.8,1.1,40729.11\nR003,1996-09-06,19.1,14.5,29978.03\nR003,1996-09-07,18.0,10.3,30407.26\nR003,1996-09-08,17.4,17.6,34791.79\nR003,1996-09-09,14.0,19.8,23854.27\nR003,1996-09-10,13.8,14.0,16615.91\nR003,1996-09-11,14.1,4.6,35898.08\nR003,1996-09-12,14.7,0.0,41884.07\nR003,1996-09-13,17.2,0.0,43159.04\nR003,1996-09-14,21.2,0.1,37834.12\nR003,1996-09-15,17.7,3.4,33282.89\nR003,1996-09-16,13.6,0.3,40468.25\nR003,1996-09-17,13.7,0.1,41456.51\nR003,1996-09-18,13.2,0.5,40216.94\nR003,1996-09-19,11.2,3.6,39396.35\nR003,1996-09-20,11.8,0.0,39639.49\nR003,1996-09-21,12.8,4.6,21226.27\nR003,1996-09-22,12.8,0.5,38140.27\nR003,1996-09-23,13.5,0.2,38353.91\nR003,1996-09-24,15.6,3.6,21939.96\nR003,1996-09-25,18.2,8.0,35859.85\nR003,1996-09-26,16.9,1.3,39856.86\nR003,1996-09-27,17.9,0.0,40797.84\nR003,1996-09-28,19.8,5.5,38619.02\nR003,1996-09-29,14.5,27.6,6190.75\nR003,1996-09-30,14.5,2.4,31463.02\nR003,1996-10-01,14.7,0.0,39746.5\nR003,1996-10-02,18.8,0.0,39995.12\nR003,1996-10-03,16.3,7.4,4075.48\nR003,1996-10-04,14.8,2.6,37784.29\nR003,1996-10-05,11.9,4.6,38430.58\nR003,1996-10-06,12.6,0.0,38786.88\nR003,1996-10-07,14.0,0.0,36618.44\nR003,1996-10-08,13.6,0.1,36981.67\nR003,1996-10-09,14.1,0.0,36478.42\nR003,1996-10-10,15.5,0.1,36767.54\nR003,1996-10-11,15.7,3.8,28200.02\nR003,1996-10-12,14.7,6.0,381.26\nR003,1996-10-13,13.8,1.0,29909.82\nR003,1996-10-14,13.4,4.0,24559.25\nR003,1996-10-15,13.7,6.0,24308.4\nR003,1996-10-16,13.8,2.0,28492.66\nR003,1996-10-17,12.9,0.0,36000.0\nR003,1996-10-18,14.1,0.0,36000.0\nR003,1996-10-19,15.8,0.0,33419.15\nR003,1996-10-20,16.7,0.2,15146.47\nR003,1996-10-21,16.0,0.3,36000.0\nR003,1996-10-22,16.4,7.8,15234.54\nR003,1996-10-23,11.8,42.3,655.65\nR003,1996-10-24,9.6,3.7,29472.05\nR003,1996-10-25,10.8,6.3,11855.98\nR003,1996-10-26,10.0,16.7,18340.19\nR003,1996-10-27,8.3,7.3,26774.54\nR003,1996-10-28,6.0,8.4,32400.0\nR003,1996-10-29,5.2,0.0,36000.0\nR003,1996-10-30,7.0,0.1,36000.0\nR003,1996-10-31,8.6,0.0,29891.49\nR003,1996-11-01,7.2,9.1,19090.1\nR003,1996-11-02,4.9,0.0,35920.86\nR003,1996-11-03,6.7,0.6,27013.13\nR003,1996-11-04,8.3,2.2,17804.35\nR003,1996-11-05,8.2,0.0,35648.87\nR003,1996-11-06,8.9,0.0,35492.41\nR003,1996-11-07,9.8,0.0,35337.7\nR003,1996-11-08,10.5,0.0,35184.91\nR003,1996-11-09,10.1,0.0,32622.28\nR003,1996-11-10,6.6,8.6,28408.0\nR003,1996-11-11,3.6,0.0,34740.11\nR003,1996-11-12,8.4,0.0,31229.02\nR003,1996-11-13,8.7,0.0,34456.85\nR003,1996-11-14,9.6,0.0,34319.84\nR003,1996-11-15,8.3,0.0,34186.0\nR003,1996-11-16,7.6,0.0,34054.05\nR003,1996-11-17,7.2,0.0,33923.67\nR003,1996-11-18,8.0,0.0,33693.06\nR003,1996-11-19,7.5,0.0,33325.1\nR003,1996-11-20,10.6,0.0,32590.76\nR003,1996-11-21,11.9,0.2,31202.83\nR003,1996-11-22,10.1,2.0,27683.92\nR003,1996-11-23,9.2,0.0,30085.61\nR003,1996-11-24,10.0,0.0,30549.56\nR003,1996-11-25,10.2,0.0,30828.13\nR003,1996-11-26,10.8,0.0,28800.0\nR003,1996-11-27,12.3,0.0,29873.39\nR003,1996-11-28,10.7,0.0,28800.0\nR003,1996-11-29,9.4,1.8,25733.15\nR003,1996-11-30,7.4,0.0,26926.52\nR003,1996-12-01,9.7,0.0,28116.11\nR003,1996-12-02,11.2,0.7,29964.47\nR003,1996-12-03,10.5,0.0,29978.42\nR003,1996-12-04,12.4,0.0,26138.04\nR003,1996-12-05,10.9,0.9,28800.0\nR003,1996-12-06,10.0,4.0,13756.65\nR003,1996-12-07,8.8,0.9,22430.25\nR003,1996-12-08,7.6,3.7,5595.64\nR003,1996-12-09,7.9,0.2,27601.43\nR003,1996-12-10,7.2,0.2,24463.08\nR003,1996-12-11,7.2,1.1,17255.22\nR003,1996-12-12,6.6,0.0,27956.17\nR003,1996-12-13,6.4,1.6,5078.86\nR003,1996-12-14,6.1,1.4,24344.49\nR003,1996-12-15,6.2,0.3,12156.93\nR003,1996-12-16,6.2,32.4,0.0\nR003,1996-12-17,-1.7,4.7,30400.76\nR003,1996-12-18,0.9,0.0,26096.48\nR003,1996-12-19,5.6,0.6,27973.04\nR003,1996-12-20,6.5,19.5,5678.86\nR003,1996-12-21,7.4,5.2,15786.91\nR003,1996-12-22,5.4,35.8,0.0\nR003,1996-12-23,2.0,10.3,0.0\nR003,1996-12-24,5.1,57.5,0.0\nR003,1996-12-25,5.8,52.7,0.0\nR003,1996-12-26,6.3,26.3,0.0\nR003,1996-12-27,5.9,42.6,0.0\nR003,1996-12-28,6.3,33.3,4351.96\nR003,1996-12-29,4.7,23.5,0.0\nR003,1996-12-30,-0.4,8.2,25562.53\nR003,1996-12-31,5.8,3.7,4003.25\nR003,1997-01-01,5.6,12.8,5866.58\nR003,1997-01-02,5.8,18.9,5427.23\nR003,1997-01-03,3.6,8.0,1155.42\nR003,1997-01-04,0.2,2.6,25200.0\nR003,1997-01-05,6.0,3.0,0.0\nR003,1997-01-06,4.9,34.9,16242.17\nR003,1997-01-07,2.3,3.6,31976.54\nR003,1997-01-08,3.0,0.3,26672.51\nR003,1997-01-09,6.8,3.0,20553.04\nR003,1997-01-10,4.2,5.0,18419.92\nR003,1997-01-11,4.0,0.0,22761.71\nR003,1997-01-12,5.6,0.0,28800.0\nR003,1997-01-13,5.0,0.1,32380.25\nR003,1997-01-14,3.5,0.0,32400.0\nR003,1997-01-15,3.1,0.1,26980.72\nR003,1997-01-16,0.5,0.8,32400.0\nR003,1997-01-17,1.5,3.7,16165.29\nR003,1997-01-18,0.9,0.0,32400.0\nR003,1997-01-19,2.7,0.0,32400.0\nR003,1997-01-20,3.8,0.0,32400.0\nR003,1997-01-21,1.9,9.2,0.0\nR003,1997-01-22,1.5,0.9,31844.15\nR003,1997-01-23,0.7,0.6,32400.0\nR003,1997-01-24,1.9,0.0,28800.0\nR003,1997-01-25,0.6,12.0,5967.47\nR003,1997-01-26,-6.5,0.0,31141.96\nR003,1997-01-27,-9.0,0.2,32400.0\nR003,1997-01-28,-7.9,0.2,32400.0\nR003,1997-01-29,-3.7,22.4,0.0\nR003,1997-01-30,-7.9,0.8,32400.0\nR003,1997-01-31,-7.0,3.1,28800.0\nR003,1997-02-01,-2.0,30.0,0.0\nR003,1997-02-02,-3.2,11.1,19098.53\nR003,1997-02-03,-4.9,3.4,21051.5\nR003,1997-02-04,-4.5,0.0,25169.5\nR003,1997-02-05,-5.4,0.0,33562.48\nR003,1997-02-06,-4.7,0.0,32691.85\nR003,1997-02-07,-0.7,7.8,0.0\nR003,1997-02-08,-2.6,0.0,34669.39\nR003,1997-02-09,-0.2,0.2,33733.58\nR003,1997-02-10,0.1,3.3,27435.06\nR003,1997-02-11,0.9,1.2,19090.44\nR003,1997-02-12,3.0,0.3,34304.86\nR003,1997-02-13,4.0,0.0,31612.35\nR003,1997-02-14,6.0,0.3,29188.37\nR003,1997-02-15,5.0,11.9,20765.45\nR003,1997-02-16,5.6,0.0,37331.35\nR003,1997-02-17,6.8,28.5,4715.23\nR003,1997-02-18,-0.8,1.1,35407.8\nR003,1997-02-19,-2.8,0.0,37965.81\nR003,1997-02-20,-2.8,0.0,38138.71\nR003,1997-02-21,2.2,0.0,31939.19\nR003,1997-02-22,2.5,7.2,7437.98\nR003,1997-02-23,1.7,2.4,32395.65\nR003,1997-02-24,2.1,0.0,35768.18\nR003,1997-02-25,1.6,0.0,39005.17\nR003,1997-02-26,1.7,0.0,39181.56\nR003,1997-02-27,3.1,0.0,39354.99\nR003,1997-02-28,4.9,9.5,0.0\nR003,1997-03-01,2.0,8.1,21956.73\nR003,1997-03-02,-0.7,0.0,39600.0\nR003,1997-03-03,0.7,0.0,39600.0\nR003,1997-03-04,0.8,2.9,7200.0\nR003,1997-03-05,-0.1,13.3,29269.72\nR003,1997-03-06,-2.2,3.7,39600.0\nR003,1997-03-07,-3.4,0.0,39360.58\nR003,1997-03-08,-3.2,0.0,39600.0\nR003,1997-03-09,-2.5,0.0,38960.7\nR003,1997-03-10,0.4,0.0,39600.0\nR003,1997-03-11,1.4,0.0,39600.0\nR003,1997-03-12,3.4,0.0,39600.0\nR003,1997-03-13,4.6,0.0,39600.0\nR003,1997-03-14,5.1,0.0,39600.0\nR003,1997-03-15,5.3,0.0,39841.96\nR003,1997-03-16,4.3,0.0,39788.79\nR003,1997-03-17,1.7,10.2,16555.52\nR003,1997-03-18,-1.6,5.2,18557.16\nR003,1997-03-19,-3.7,0.0,35571.63\nR003,1997-03-20,-0.7,0.0,36065.61\nR003,1997-03-21,6.7,0.0,38875.92\nR003,1997-03-22,6.4,1.5,11441.47\nR003,1997-03-23,2.5,10.6,8719.68\nR003,1997-03-24,-0.8,2.4,39999.45\nR003,1997-03-25,-0.3,0.0,41497.83\nR003,1997-03-26,-1.2,3.3,29942.97\nR003,1997-03-27,-1.0,0.0,35752.09\nR003,1997-03-28,-1.0,0.0,39836.64\nR003,1997-03-29,2.2,0.0,43972.15\nR003,1997-03-30,9.0,0.0,43091.29\nR003,1997-03-31,12.2,0.0,44438.33\nR003,1997-04-01,10.9,0.0,45149.03\nR003,1997-04-02,10.8,0.0,44514.84\nR003,1997-04-03,11.0,0.0,40973.21\nR003,1997-04-04,9.4,11.8,40456.4\nR003,1997-04-05,7.9,0.6,35781.76\nR003,1997-04-06,8.1,0.1,44352.97\nR003,1997-04-07,6.5,9.2,22856.7\nR003,1997-04-08,5.6,0.2,42606.58\nR003,1997-04-09,4.5,13.4,1472.23\nR003,1997-04-10,1.7,10.9,16932.8\nR003,1997-04-11,-0.2,3.2,46372.24\nR003,1997-04-12,4.5,0.2,43512.59\nR003,1997-04-13,8.1,0.1,46515.12\nR003,1997-04-14,8.8,2.1,23063.32\nR003,1997-04-15,6.7,13.9,21908.31\nR003,1997-04-16,6.5,0.3,45637.05\nR003,1997-04-17,9.8,0.0,46800.0\nR003,1997-04-18,8.7,10.8,40116.12\nR003,1997-04-19,3.9,4.1,44917.74\nR003,1997-04-20,7.2,0.9,30898.99\nR003,1997-04-21,10.0,3.4,40127.13\nR003,1997-04-22,13.5,0.0,38068.05\nR003,1997-04-23,17.6,0.6,33699.01\nR003,1997-04-24,14.1,7.2,22240.3\nR003,1997-04-25,11.4,13.7,31123.12\nR003,1997-04-26,10.5,12.4,34016.56\nR003,1997-04-27,10.1,1.6,41026.4\nR003,1997-04-28,7.6,14.5,2951.24\nR003,1997-04-29,6.8,10.3,1750.94\nR003,1997-04-30,8.3,1.6,38461.99\nR003,1997-05-01,8.9,0.0,47507.08\nR003,1997-05-02,8.5,0.4,46488.67\nR003,1997-05-03,8.7,5.8,15713.54\nR003,1997-05-04,8.5,6.4,34830.61\nR003,1997-05-05,9.2,0.5,45046.58\nR003,1997-05-06,12.3,0.0,47324.99\nR003,1997-05-07,15.1,0.0,49897.99\nR003,1997-05-08,17.3,0.0,47257.19\nR003,1997-05-09,17.9,0.0,50049.12\nR003,1997-05-10,19.1,0.0,50122.14\nR003,1997-05-11,20.1,0.0,50193.31\nR003,1997-05-12,19.7,0.5,48163.46\nR003,1997-05-13,18.1,0.6,50204.64\nR003,1997-05-14,17.2,0.7,48801.99\nR003,1997-05-15,16.5,0.1,48353.15\nR003,1997-05-16,17.4,0.0,50400.0\nR003,1997-05-17,18.3,0.0,50400.0\nR003,1997-05-18,16.1,9.6,15912.51\nR003,1997-05-19,15.4,0.7,43554.77\nR003,1997-05-20,17.1,0.6,49392.42\nR003,1997-05-21,18.3,2.6,50153.72\nR003,1997-05-22,17.8,3.9,45424.63\nR003,1997-05-23,19.2,0.9,46335.23\nR003,1997-05-24,19.7,2.1,50400.0\nR003,1997-05-25,17.1,11.5,42524.46\nR003,1997-05-26,14.9,10.8,16436.05\nR003,1997-05-27,15.8,2.3,46117.49\nR003,1997-05-28,13.2,25.7,36882.46\nR003,1997-05-29,16.0,0.0,51543.75\nR003,1997-05-30,19.6,2.4,50439.07\nR003,1997-05-31,16.1,21.0,27439.35\nR003,1997-06-01,16.0,9.7,38264.99\nR003,1997-06-02,12.9,12.2,28941.72\nR003,1997-06-03,14.0,0.0,50400.0\nR003,1997-06-04,16.5,0.0,51063.46\nR003,1997-06-05,17.3,6.9,42169.16\nR003,1997-06-06,16.5,7.8,19072.63\nR003,1997-06-07,18.7,1.2,44817.32\nR003,1997-06-08,20.4,3.1,38321.41\nR003,1997-06-09,18.0,19.3,4599.96\nR003,1997-06-10,19.9,2.7,50223.86\nR003,1997-06-11,21.5,6.1,49556.76\nR003,1997-06-12,20.1,20.0,29032.38\nR003,1997-06-13,18.5,16.7,44558.99\nR003,1997-06-14,19.5,5.4,49736.01\nR003,1997-06-15,18.6,8.5,42355.81\nR003,1997-06-16,19.1,3.6,47403.54\nR003,1997-06-17,19.3,1.2,45018.09\nR003,1997-06-18,20.6,0.1,49680.6\nR003,1997-06-19,20.3,0.6,51048.2\nR003,1997-06-20,19.9,0.2,50358.78\nR003,1997-06-21,22.4,0.8,53414.44\nR003,1997-06-22,20.5,1.6,51324.22\nR003,1997-06-23,19.1,0.2,50347.63\nR003,1997-06-24,20.9,1.0,51367.85\nR003,1997-06-25,22.0,0.0,53762.13\nR003,1997-06-26,24.0,0.7,53391.28\nR003,1997-06-27,22.9,7.9,45010.69\nR003,1997-06-28,23.2,3.3,50868.79\nR003,1997-06-29,20.9,6.2,27012.25\nR003,1997-06-30,21.4,11.4,47053.7\nR003,1997-07-01,21.7,14.1,21781.76\nR003,1997-07-02,21.7,14.3,38581.77\nR003,1997-07-03,22.4,4.1,46770.64\nR003,1997-07-04,19.8,5.2,38606.85\nR003,1997-07-05,19.1,8.3,33936.31\nR003,1997-07-06,19.6,5.6,44880.79\nR003,1997-07-07,22.4,0.0,49294.9\nR003,1997-07-08,24.1,0.4,52006.14\nR003,1997-07-09,24.5,0.8,53767.74\nR003,1997-07-10,21.5,8.8,49361.54\nR003,1997-07-11,20.3,0.4,51229.76\nR003,1997-07-12,20.4,1.8,48112.15\nR003,1997-07-13,20.1,0.0,53078.15\nR003,1997-07-14,20.0,2.7,50761.44\nR003,1997-07-15,18.3,6.9,44095.96\nR003,1997-07-16,17.6,4.8,46033.48\nR003,1997-07-17,17.3,8.9,32697.19\nR003,1997-07-18,17.5,1.2,49541.06\nR003,1997-07-19,18.9,0.0,52800.81\nR003,1997-07-20,22.5,0.0,52783.73\nR003,1997-07-21,21.8,2.0,51682.46\nR003,1997-07-22,20.6,0.4,49639.79\nR003,1997-07-23,19.6,0.1,50385.8\nR003,1997-07-24,18.7,3.8,47183.33\nR003,1997-07-25,18.8,2.0,47674.06\nR003,1997-07-26,21.3,0.0,50796.12\nR003,1997-07-27,24.1,0.2,51110.2\nR003,1997-07-28,24.4,1.6,49570.93\nR003,1997-07-29,21.0,60.3,25371.41\nR003,1997-07-30,21.2,1.6,46675.08\nR003,1997-07-31,22.4,0.5,45824.29\nR003,1997-08-01,23.4,6.5,42853.05\nR003,1997-08-02,22.6,12.6,35259.7\nR003,1997-08-03,23.8,0.0,47489.07\nR003,1997-08-04,22.0,0.3,48213.39\nR003,1997-08-05,19.5,4.4,40136.5\nR003,1997-08-06,20.9,8.3,32551.77\nR003,1997-08-07,21.9,3.0,46406.94\nR003,1997-08-08,22.9,1.5,46800.0\nR003,1997-08-09,22.5,5.7,18454.36\nR003,1997-08-10,22.4,19.1,33107.6\nR003,1997-08-11,24.0,9.9,40469.71\nR003,1997-08-12,21.5,16.2,41111.97\nR003,1997-08-13,21.1,3.7,46800.0\nR003,1997-08-14,19.9,0.4,41570.64\nR003,1997-08-15,20.1,6.1,41475.61\nR003,1997-08-16,20.7,0.0,46180.28\nR003,1997-08-17,20.7,1.2,46800.0\nR003,1997-08-18,20.8,0.9,44789.44\nR003,1997-08-19,20.0,2.0,40141.28\nR003,1997-08-20,19.4,10.2,33116.11\nR003,1997-08-21,20.0,8.5,41975.5\nR003,1997-08-22,19.1,9.8,38245.01\nR003,1997-08-23,20.9,8.2,44023.01\nR003,1997-08-24,22.0,1.8,40927.55\nR003,1997-08-25,24.9,1.3,46008.55\nR003,1997-08-26,21.9,16.2,19581.97\nR003,1997-08-27,20.8,18.1,38373.26\nR003,1997-08-28,21.2,2.8,44013.74\nR003,1997-08-29,21.5,4.4,43959.16\nR003,1997-08-30,22.3,0.0,44868.11\nR003,1997-08-31,22.2,0.9,45504.65\nR003,1997-09-01,18.2,35.0,11080.68\nR003,1997-09-02,15.2,4.1,26451.77\nR003,1997-09-03,14.4,6.5,25362.47\nR003,1997-09-04,12.1,8.0,23601.63\nR003,1997-09-05,12.2,5.8,27582.91\nR003,1997-09-06,13.5,0.0,43560.88\nR003,1997-09-07,14.6,0.4,43932.62\nR003,1997-09-08,16.9,0.0,43346.21\nR003,1997-09-09,16.9,0.0,43200.0\nR003,1997-09-10,17.3,0.0,42119.61\nR003,1997-09-11,13.5,17.1,26282.12\nR003,1997-09-12,11.9,0.5,41504.08\nR003,1997-09-13,12.6,0.0,41636.03\nR003,1997-09-14,16.5,0.0,43108.37\nR003,1997-09-15,19.2,0.0,43041.03\nR003,1997-09-16,18.2,1.0,42973.6\nR003,1997-09-17,15.7,5.0,30320.42\nR003,1997-09-18,16.1,0.0,42838.37\nR003,1997-09-19,18.8,0.0,39924.36\nR003,1997-09-20,19.8,1.5,42186.44\nR003,1997-09-21,15.3,29.9,597.58\nR003,1997-09-22,12.8,0.5,37014.56\nR003,1997-09-23,13.1,0.0,40061.85\nR003,1997-09-24,14.0,0.0,41371.07\nR003,1997-09-25,12.8,3.7,27980.64\nR003,1997-09-26,10.1,1.8,40167.23\nR003,1997-09-27,10.7,0.0,39600.0\nR003,1997-09-28,11.6,3.1,1378.84\nR003,1997-09-29,10.7,6.0,28349.49\nR003,1997-09-30,10.6,3.7,37569.61\nR003,1997-10-01,10.5,2.7,20393.86\nR003,1997-10-02,10.9,14.9,0.0\nR003,1997-10-03,13.1,0.0,36454.85\nR003,1997-10-04,13.3,28.1,15462.05\nR003,1997-10-05,11.2,0.0,31833.41\nR003,1997-10-06,10.7,8.1,22270.41\nR003,1997-10-07,12.0,1.3,37882.47\nR003,1997-10-08,13.1,0.0,38630.01\nR003,1997-10-09,15.3,0.0,37754.83\nR003,1997-10-10,17.2,0.0,37525.12\nR003,1997-10-11,18.9,0.0,37137.23\nR003,1997-10-12,20.0,0.0,36541.95\nR003,1997-10-13,18.6,1.4,36000.0\nR003,1997-10-14,17.6,3.7,12586.25\nR003,1997-10-15,19.9,0.0,36050.15\nR003,1997-10-16,22.4,0.0,36102.24\nR003,1997-10-17,18.4,7.5,23582.97\nR003,1997-10-18,15.9,10.7,16582.82\nR003,1997-10-19,14.9,23.0,0.0\nR003,1997-10-20,15.2,0.0,33026.5\nR003,1997-10-21,14.3,8.0,10263.56\nR003,1997-10-22,11.3,12.3,70.6\nR003,1997-10-23,10.9,0.1,33505.77\nR003,1997-10-24,13.0,0.0,36000.0\nR003,1997-10-25,13.8,14.4,11799.3\nR003,1997-10-26,11.4,70.0,0.0\nR003,1997-10-27,9.1,14.8,2728.9\nR003,1997-10-28,8.6,6.7,17962.39\nR003,1997-10-29,8.5,7.8,20053.38\nR003,1997-10-30,5.5,8.5,29949.24\nR003,1997-10-31,2.3,0.0,36000.0\nR003,1997-11-01,4.5,0.0,36000.0\nR003,1997-11-02,6.6,0.0,35937.36\nR003,1997-11-03,7.4,0.0,35857.42\nR003,1997-11-04,8.2,13.6,7961.33\nR003,1997-11-05,3.9,4.7,33195.07\nR003,1997-11-06,2.7,0.0,35524.97\nR003,1997-11-07,7.0,0.0,35372.37\nR003,1997-11-08,8.6,0.0,35222.31\nR003,1997-11-09,9.9,0.0,35074.97\nR003,1997-11-10,9.7,0.0,34930.36\nR003,1997-11-11,9.0,0.0,34787.12\nR003,1997-11-12,8.6,0.0,30854.01\nR003,1997-11-13,9.4,0.0,34504.07\nR003,1997-11-14,9.8,0.0,34364.59\nR003,1997-11-15,11.3,0.0,29316.0\nR003,1997-11-16,12.2,0.3,21712.43\nR003,1997-11-17,11.8,0.0,32061.19\nR003,1997-11-18,9.1,7.1,2847.38\nR003,1997-11-19,11.3,1.0,25285.65\nR003,1997-11-20,10.8,6.8,15767.24\nR003,1997-11-21,10.9,0.0,28322.46\nR003,1997-11-22,11.2,0.0,29003.45\nR003,1997-11-23,10.1,0.5,8199.27\nR003,1997-11-24,8.2,0.0,30545.68\nR003,1997-11-25,8.2,0.0,31561.94\nR003,1997-11-26,7.9,0.0,31256.79\nR003,1997-11-27,7.0,0.0,30246.83\nR003,1997-11-28,7.0,3.8,18914.25\nR003,1997-11-29,7.3,2.6,21600.0\nR003,1997-11-30,6.7,0.0,19633.64\nR003,1997-12-01,7.8,0.0,30325.17\nR003,1997-12-02,8.6,1.7,28818.74\nR003,1997-12-03,9.2,2.8,11941.39\nR003,1997-12-04,9.1,2.9,28800.0\nR003,1997-12-05,7.4,30.8,22187.72\nR003,1997-12-06,5.5,7.9,12722.98\nR003,1997-12-07,6.4,2.8,3037.52\nR003,1997-12-08,7.9,17.5,8888.31\nR003,1997-12-09,6.1,20.8,0.0\nR003,1997-12-10,1.1,0.9,28436.08\nR003,1997-12-11,-0.1,0.0,28800.0\nR003,1997-12-12,3.1,0.0,24192.44\nR003,1997-12-13,5.9,0.1,11983.98\nR003,1997-12-14,7.2,8.3,4860.41\nR003,1997-12-15,6.5,16.3,0.0\nR003,1997-12-16,4.4,23.3,0.0\nR003,1997-12-17,1.8,11.1,0.0\nR003,1997-12-18,-2.5,3.5,17133.82\nR003,1997-12-19,-3.1,0.1,23350.16\nR003,1997-12-20,-0.9,0.0,30769.55\nR003,1997-12-21,3.3,5.4,10191.98\nR003,1997-12-22,6.3,3.4,18998.38\nR003,1997-12-23,7.0,0.0,22844.31\nR003,1997-12-24,5.2,2.3,7469.97\nR003,1997-12-25,3.5,3.2,10799.04\nR003,1997-12-26,2.5,1.5,25765.6\nR003,1997-12-27,3.1,0.0,28800.0\nR003,1997-12-28,4.4,0.0,31407.54\nR003,1997-12-29,5.5,0.0,29818.13\nR003,1997-12-30,6.8,0.0,24088.4\nR003,1997-12-31,3.5,6.8,3151.15\nR003,1998-01-01,0.5,0.1,31608.83\nR003,1998-01-02,-1.2,0.0,31663.59\nR003,1998-01-03,-2.8,0.0,31719.86\nR003,1998-01-04,-1.7,0.0,31777.53\nR003,1998-01-05,2.2,0.0,29126.67\nR003,1998-01-06,2.6,6.5,20077.72\nR003,1998-01-07,2.1,5.5,20420.87\nR003,1998-01-08,0.9,4.1,30256.7\nR003,1998-01-09,1.4,11.4,0.0\nR003,1998-01-10,1.0,34.1,639.95\nR003,1998-01-11,-3.1,0.0,32214.38\nR003,1998-01-12,-3.5,5.9,0.0\nR003,1998-01-13,-0.4,10.6,0.0\nR003,1998-01-14,3.4,3.5,15530.02\nR003,1998-01-15,3.1,0.0,32400.0\nR003,1998-01-16,3.0,0.0,23181.74\nR003,1998-01-17,2.8,0.0,26972.93\nR003,1998-01-18,2.3,0.0,29242.54\nR003,1998-01-19,2.2,2.0,20406.15\nR003,1998-01-20,2.1,0.0,32335.2\nR003,1998-01-21,2.9,0.0,32400.0\nR003,1998-01-22,1.7,0.0,32400.0\nR003,1998-01-23,1.4,0.0,32400.0\nR003,1998-01-24,3.5,0.0,32400.0\nR003,1998-01-25,4.0,0.0,29884.99\nR003,1998-01-26,3.2,5.2,0.0\nR003,1998-01-27,2.7,2.7,18371.63\nR003,1998-01-28,2.5,0.0,29730.09\nR003,1998-01-29,3.7,12.2,0.0\nR003,1998-01-30,0.0,12.4,17421.69\nR003,1998-01-31,-5.2,0.0,32417.59\nR003,1998-02-01,-1.6,7.2,32400.0\nR003,1998-02-02,0.0,10.3,442.83\nR003,1998-02-03,2.7,12.3,266.0\nR003,1998-02-04,5.1,16.2,4421.12\nR003,1998-02-05,7.2,19.8,0.0\nR003,1998-02-06,8.1,24.4,0.0\nR003,1998-02-07,3.7,22.1,15204.86\nR003,1998-02-08,-4.4,1.1,32413.95\nR003,1998-02-09,-5.0,0.0,34947.56\nR003,1998-02-10,-1.9,0.0,10347.31\nR003,1998-02-11,-2.3,0.0,35460.13\nR003,1998-02-12,-0.1,2.7,32441.82\nR003,1998-02-13,2.0,22.7,0.0\nR003,1998-02-14,-4.5,3.3,35569.38\nR003,1998-02-15,-6.7,4.7,0.0\nR003,1998-02-16,-4.5,1.0,36312.59\nR003,1998-02-17,0.8,22.4,15223.52\nR003,1998-02-18,-1.7,21.3,31032.84\nR003,1998-02-19,-6.2,0.0,37745.08\nR003,1998-02-20,0.8,0.3,34909.51\nR003,1998-02-21,3.0,0.0,38130.83\nR003,1998-02-22,5.0,0.0,38430.72\nR003,1998-02-23,7.9,0.0,38606.58\nR003,1998-02-24,7.6,0.0,38783.58\nR003,1998-02-25,5.8,0.2,38961.56\nR003,1998-02-26,0.2,13.3,36266.41\nR003,1998-02-27,-2.9,0.0,37609.27\nR003,1998-02-28,2.1,0.0,39437.86\nR003,1998-03-01,4.0,0.0,39539.85\nR003,1998-03-02,3.7,0.0,39600.0\nR003,1998-03-03,2.5,6.5,39600.0\nR003,1998-03-04,2.6,1.8,28800.0\nR003,1998-03-05,5.3,6.5,5526.37\nR003,1998-03-06,7.6,0.0,39600.0\nR003,1998-03-07,4.1,13.8,22947.43\nR003,1998-03-08,4.2,0.0,39600.0\nR003,1998-03-09,8.8,0.0,39455.99\nR003,1998-03-10,9.6,0.0,34512.94\nR003,1998-03-11,11.1,0.0,34397.77\nR003,1998-03-12,8.5,7.5,18936.03\nR003,1998-03-13,2.3,5.2,36541.05\nR003,1998-03-14,2.5,3.5,12885.04\nR003,1998-03-15,3.9,0.0,39600.0\nR003,1998-03-16,4.7,3.9,13940.39\nR003,1998-03-17,5.0,7.2,1236.52\nR003,1998-03-18,4.1,0.4,33783.65\nR003,1998-03-19,1.4,14.3,23920.92\nR003,1998-03-20,1.1,15.2,21850.68\nR003,1998-03-21,2.7,1.3,36456.38\nR003,1998-03-22,2.6,11.4,10997.46\nR003,1998-03-23,2.5,10.1,5448.98\nR003,1998-03-24,0.6,3.9,26615.57\nR003,1998-03-25,-0.6,0.0,40881.3\nR003,1998-03-26,0.8,0.0,36904.43\nR003,1998-03-27,6.7,0.0,42306.8\nR003,1998-03-28,8.6,1.7,30086.6\nR003,1998-03-29,6.6,9.6,0.0\nR003,1998-03-30,5.0,5.5,163.77\nR003,1998-03-31,4.8,2.6,42731.64\nR003,1998-04-01,5.9,0.1,42497.34\nR003,1998-04-02,8.7,0.0,45195.24\nR003,1998-04-03,8.9,0.0,44311.41\nR003,1998-04-04,9.8,0.2,44703.87\nR003,1998-04-05,10.5,0.0,45926.98\nR003,1998-04-06,12.2,0.0,45998.49\nR003,1998-04-07,13.1,0.0,46069.98\nR003,1998-04-08,14.5,0.0,41790.09\nR003,1998-04-09,14.7,0.0,46212.86\nR003,1998-04-10,15.9,0.0,44229.58\nR003,1998-04-11,16.6,1.8,46355.56\nR003,1998-04-12,18.8,0.0,46426.81\nR003,1998-04-13,19.1,0.0,46498.0\nR003,1998-04-14,14.9,2.8,39600.0\nR003,1998-04-15,12.3,7.8,38286.19\nR003,1998-04-16,10.1,2.1,41090.54\nR003,1998-04-17,9.8,3.1,14485.56\nR003,1998-04-18,13.1,3.7,37598.1\nR003,1998-04-19,14.1,4.2,36081.01\nR003,1998-04-20,11.0,18.7,6835.13\nR003,1998-04-21,9.2,3.5,4076.47\nR003,1998-04-22,8.5,7.9,34030.03\nR003,1998-04-23,8.5,2.2,34170.72\nR003,1998-04-24,8.9,4.6,37197.53\nR003,1998-04-25,10.6,0.0,45528.87\nR003,1998-04-26,13.7,0.1,47382.89\nR003,1998-04-27,14.3,2.4,25161.93\nR003,1998-04-28,11.7,14.2,17401.65\nR003,1998-04-29,11.1,6.0,13557.15\nR003,1998-04-30,11.5,3.4,29896.04\nR003,1998-05-01,12.2,3.7,42941.61\nR003,1998-05-02,12.9,7.5,45629.36\nR003,1998-05-03,14.7,0.0,48785.81\nR003,1998-05-04,18.8,0.2,36570.35\nR003,1998-05-05,20.8,0.5,39089.85\nR003,1998-05-06,21.3,3.8,33452.43\nR003,1998-05-07,17.1,4.7,43370.82\nR003,1998-05-08,18.5,0.6,42847.01\nR003,1998-05-09,19.0,1.0,41914.6\nR003,1998-05-10,20.8,3.7,14560.35\nR003,1998-05-11,21.9,6.2,34413.77\nR003,1998-05-12,18.1,26.8,12330.12\nR003,1998-05-13,17.3,9.6,35868.65\nR003,1998-05-14,15.1,11.1,22255.42\nR003,1998-05-15,14.2,12.3,2327.61\nR003,1998-05-16,15.1,2.3,42129.72\nR003,1998-05-17,18.3,0.0,37481.14\nR003,1998-05-18,19.9,4.6,41217.61\nR003,1998-05-19,18.8,10.0,36499.82\nR003,1998-05-20,15.5,18.2,32397.01\nR003,1998-05-21,13.8,10.7,4921.67\nR003,1998-05-22,13.2,10.5,32883.32\nR003,1998-05-23,15.2,1.7,46783.31\nR003,1998-05-24,13.4,4.8,36368.08\nR003,1998-05-25,15.4,0.1,49153.61\nR003,1998-05-26,19.0,0.0,50539.78\nR003,1998-05-27,22.9,0.3,50874.78\nR003,1998-05-28,20.2,12.2,38992.2\nR003,1998-05-29,18.6,20.1,723.37\nR003,1998-05-30,18.7,16.9,14827.16\nR003,1998-05-31,17.3,15.1,26702.44\nR003,1998-06-01,17.4,15.2,44457.24\nR003,1998-06-02,17.2,13.2,29989.97\nR003,1998-06-03,17.8,13.4,33160.06\nR003,1998-06-04,18.3,12.1,42941.16\nR003,1998-06-05,18.7,9.7,42591.89\nR003,1998-06-06,20.9,4.7,47682.12\nR003,1998-06-07,22.0,13.0,37917.13\nR003,1998-06-08,22.9,1.7,52091.21\nR003,1998-06-09,22.7,2.5,52075.74\nR003,1998-06-10,23.2,5.7,51328.46\nR003,1998-06-11,23.2,8.3,46685.62\nR003,1998-06-12,24.3,2.5,50733.44\nR003,1998-06-13,25.1,3.4,50919.36\nR003,1998-06-14,25.3,2.0,53477.46\nR003,1998-06-15,24.4,2.0,50788.24\nR003,1998-06-16,24.3,0.9,44455.1\nR003,1998-06-17,24.9,1.4,48139.24\nR003,1998-06-18,24.7,6.1,48799.46\nR003,1998-06-19,24.6,2.0,53432.4\nR003,1998-06-20,25.0,0.5,51234.97\nR003,1998-06-21,25.9,1.3,53418.07\nR003,1998-06-22,21.5,24.8,13880.9\nR003,1998-06-23,19.8,3.2,43280.49\nR003,1998-06-24,20.0,0.7,50844.8\nR003,1998-06-25,21.2,0.0,53959.55\nR003,1998-06-26,21.1,2.1,52362.44\nR003,1998-06-27,20.8,6.5,51175.84\nR003,1998-06-28,20.5,3.4,45315.19\nR003,1998-06-29,21.6,0.1,51275.31\nR003,1998-06-30,21.7,2.3,50326.88\nR003,1998-07-01,22.2,2.1,48151.23\nR003,1998-07-02,22.2,4.2,43322.33\nR003,1998-07-03,23.0,0.0,49912.84\nR003,1998-07-04,20.2,5.3,36577.68\nR003,1998-07-05,19.1,4.9,41120.09\nR003,1998-07-06,18.3,21.4,48552.06\nR003,1998-07-07,17.5,0.9,50059.41\nR003,1998-07-08,19.4,0.1,52747.4\nR003,1998-07-09,23.3,0.0,53806.34\nR003,1998-07-10,21.1,15.0,44652.13\nR003,1998-07-11,19.0,0.1,52476.71\nR003,1998-07-12,20.0,0.0,51495.35\nR003,1998-07-13,20.6,0.0,51944.61\nR003,1998-07-14,21.9,0.0,53642.87\nR003,1998-07-15,25.0,0.0,53603.85\nR003,1998-07-16,26.4,0.0,53468.36\nR003,1998-07-17,25.9,5.5,51338.87\nR003,1998-07-18,22.4,6.2,42293.31\nR003,1998-07-19,22.3,0.0,48419.78\nR003,1998-07-20,24.2,0.0,52380.6\nR003,1998-07-21,25.5,1.3,51684.16\nR003,1998-07-22,24.2,1.0,50688.02\nR003,1998-07-23,24.5,1.0,49675.52\nR003,1998-07-24,25.6,0.0,52275.36\nR003,1998-07-25,26.0,0.0,50112.31\nR003,1998-07-26,25.5,0.5,50859.75\nR003,1998-07-27,25.2,1.2,51463.54\nR003,1998-07-28,25.0,3.4,50034.3\nR003,1998-07-29,25.3,1.9,49954.19\nR003,1998-07-30,25.5,4.8,50005.61\nR003,1998-07-31,24.0,5.6,45386.31\nR003,1998-08-01,24.8,0.5,48040.9\nR003,1998-08-02,25.9,0.0,49132.18\nR003,1998-08-03,25.9,0.0,48846.99\nR003,1998-08-04,25.6,0.6,47879.1\nR003,1998-08-05,25.2,1.5,47550.44\nR003,1998-08-06,25.5,0.3,47665.9\nR003,1998-08-07,23.6,4.2,43252.33\nR003,1998-08-08,23.5,0.2,44503.06\nR003,1998-08-09,23.1,0.0,46986.3\nR003,1998-08-10,23.0,1.3,47492.57\nR003,1998-08-11,21.6,0.8,46800.0\nR003,1998-08-12,19.1,0.0,47412.85\nR003,1998-08-13,18.8,0.0,46800.0\nR003,1998-08-14,20.3,0.0,47471.87\nR003,1998-08-15,21.3,2.7,47167.64\nR003,1998-08-16,19.6,0.0,45607.54\nR003,1998-08-17,19.8,0.0,43247.52\nR003,1998-08-18,20.3,0.6,46800.0\nR003,1998-08-19,22.1,0.1,46800.0\nR003,1998-08-20,22.1,1.2,45967.48\nR003,1998-08-21,22.2,0.7,46800.0\nR003,1998-08-22,23.2,0.8,46800.0\nR003,1998-08-23,23.6,0.0,46800.0\nR003,1998-08-24,22.7,0.0,45137.51\nR003,1998-08-25,24.4,0.0,46800.0\nR003,1998-08-26,24.4,0.0,46800.0\nR003,1998-08-27,23.0,0.0,46126.97\nR003,1998-08-28,24.7,0.0,46651.37\nR003,1998-08-29,26.4,0.2,46547.69\nR003,1998-08-30,25.7,0.1,46443.55\nR003,1998-08-31,21.8,5.1,37846.79\nR003,1998-09-01,20.7,3.9,37074.28\nR003,1998-09-02,16.3,3.8,36972.42\nR003,1998-09-03,14.3,1.1,43712.99\nR003,1998-09-04,13.6,0.0,44083.85\nR003,1998-09-05,12.8,1.2,42131.58\nR003,1998-09-06,13.3,1.6,42733.62\nR003,1998-09-07,14.8,0.0,43915.49\nR003,1998-09-08,17.0,4.0,37407.64\nR003,1998-09-09,17.8,1.4,39414.69\nR003,1998-09-10,17.7,0.0,40683.42\nR003,1998-09-11,19.7,0.0,43200.0\nR003,1998-09-12,20.9,0.0,43200.0\nR003,1998-09-13,22.2,0.5,42736.02\nR003,1998-09-14,23.8,0.0,43123.72\nR003,1998-09-15,25.3,0.0,43056.31\nR003,1998-09-16,23.5,2.6,42988.8\nR003,1998-09-17,20.1,4.1,36271.39\nR003,1998-09-18,20.1,3.1,39195.07\nR003,1998-09-19,22.3,0.0,42781.28\nR003,1998-09-20,23.3,0.2,42717.77\nR003,1998-09-21,20.3,7.5,38540.29\nR003,1998-09-22,20.8,0.5,39208.33\nR003,1998-09-23,18.1,40.2,30024.32\nR003,1998-09-24,15.2,12.4,159.78\nR003,1998-09-25,16.8,1.2,33875.76\nR003,1998-09-26,15.4,0.0,38877.43\nR003,1998-09-27,16.0,0.0,39875.59\nR003,1998-09-28,17.5,0.0,41187.8\nR003,1998-09-29,21.4,0.0,41217.54\nR003,1998-09-30,22.6,0.0,39483.86\nR003,1998-10-01,16.4,7.1,6008.03\nR003,1998-10-02,14.5,0.0,38655.59\nR003,1998-10-03,15.6,0.0,39648.29\nR003,1998-10-04,18.2,0.0,37054.1\nR003,1998-10-05,17.9,1.1,37433.72\nR003,1998-10-06,16.5,0.0,38040.36\nR003,1998-10-07,12.5,1.8,29422.24\nR003,1998-10-08,11.6,0.0,37929.41\nR003,1998-10-09,13.1,0.0,37736.0\nR003,1998-10-10,14.6,0.0,37505.87\nR003,1998-10-11,18.9,0.0,36476.18\nR003,1998-10-12,20.9,0.0,36855.75\nR003,1998-10-13,23.7,0.0,36258.0\nR003,1998-10-14,23.4,0.0,32958.51\nR003,1998-10-15,20.4,0.0,36362.31\nR003,1998-10-16,18.2,0.0,36000.0\nR003,1998-10-17,15.3,3.1,24303.61\nR003,1998-10-18,14.3,3.0,28282.76\nR003,1998-10-19,13.4,6.5,24707.9\nR003,1998-10-20,14.2,0.0,36000.0\nR003,1998-10-21,15.1,2.1,36000.0\nR003,1998-10-22,9.1,1.4,36000.0\nR003,1998-10-23,7.8,0.0,36000.0\nR003,1998-10-24,10.3,0.0,36000.0\nR003,1998-10-25,15.5,0.0,32757.95\nR003,1998-10-26,17.6,0.1,34334.42\nR003,1998-10-27,17.4,0.0,13169.2\nR003,1998-10-28,13.9,24.1,1348.08\nR003,1998-10-29,12.9,21.0,9389.45\nR003,1998-10-30,13.5,0.0,33408.45\nR003,1998-10-31,12.8,0.0,34586.74\nR003,1998-11-01,10.1,10.1,24849.39\nR003,1998-11-02,10.0,3.2,19327.7\nR003,1998-11-03,10.7,8.3,21051.75\nR003,1998-11-04,11.4,0.0,35800.27\nR003,1998-11-05,14.9,0.0,29692.95\nR003,1998-11-06,12.8,7.3,22891.36\nR003,1998-11-07,11.3,4.3,8332.58\nR003,1998-11-08,8.2,15.3,0.0\nR003,1998-11-09,1.7,0.0,25680.04\nR003,1998-11-10,0.1,16.7,18046.09\nR003,1998-11-11,-3.4,0.0,34825.49\nR003,1998-11-12,5.3,0.0,31968.11\nR003,1998-11-13,7.6,0.0,32062.83\nR003,1998-11-14,9.6,17.2,0.0\nR003,1998-11-15,9.9,13.6,8843.11\nR003,1998-11-16,8.8,0.4,25067.37\nR003,1998-11-17,7.1,18.0,20878.03\nR003,1998-11-18,7.4,17.7,9652.49\nR003,1998-11-19,4.1,3.7,22322.4\nR003,1998-11-20,4.0,0.4,28928.64\nR003,1998-11-21,8.3,0.0,31999.71\nR003,1998-11-22,9.7,0.0,29849.94\nR003,1998-11-23,8.2,0.2,29919.71\nR003,1998-11-24,9.0,0.0,31475.65\nR003,1998-11-25,8.7,2.9,23522.09\nR003,1998-11-26,8.7,0.0,28800.0\nR003,1998-11-27,8.2,0.1,25878.94\nR003,1998-11-28,9.0,0.1,12972.03\nR003,1998-11-29,7.8,3.9,13809.95\nR003,1998-11-30,7.4,1.4,1898.74\nR003,1998-12-01,5.4,7.8,11783.12\nR003,1998-12-02,4.7,3.2,16788.44\nR003,1998-12-03,2.7,0.0,29968.77\nR003,1998-12-04,3.0,2.7,28488.25\nR003,1998-12-05,6.3,2.6,20509.58\nR003,1998-12-06,8.5,11.1,12153.95\nR003,1998-12-07,8.9,14.6,0.0\nR003,1998-12-08,8.5,2.1,26985.89\nR003,1998-12-09,8.5,3.6,3600.0\nR003,1998-12-10,7.8,56.6,0.0\nR003,1998-12-11,1.3,11.4,5379.72\nR003,1998-12-12,4.0,4.9,1300.31\nR003,1998-12-13,4.0,7.4,1568.88\nR003,1998-12-14,2.1,0.0,29644.74\nR003,1998-12-15,2.6,2.8,14305.43\nR003,1998-12-16,4.7,1.5,26036.32\nR003,1998-12-17,3.0,2.8,936.03\nR003,1998-12-18,4.9,0.0,25873.53\nR003,1998-12-19,3.9,0.0,30782.75\nR003,1998-12-20,4.3,4.3,14400.0\nR003,1998-12-21,6.1,5.8,15815.15\nR003,1998-12-22,6.1,0.0,29180.35\nR003,1998-12-23,5.0,0.9,23557.12\nR003,1998-12-24,4.3,6.3,4352.73\nR003,1998-12-25,4.9,4.2,20180.12\nR003,1998-12-26,4.3,0.2,27539.97\nR003,1998-12-27,2.5,0.0,31042.71\nR003,1998-12-28,1.1,0.0,31401.56\nR003,1998-12-29,4.3,0.0,31447.34\nR003,1998-12-30,5.3,0.3,10926.67\nR003,1998-12-31,3.2,3.9,22072.4\nR003,1999-01-01,-2.2,0.0,31594.47\nR003,1999-01-02,0.4,0.0,31646.39\nR003,1999-01-03,1.5,0.0,31699.57\nR003,1999-01-04,1.7,0.0,28800.0\nR003,1999-01-05,4.6,0.0,31811.09\nR003,1999-01-06,2.4,0.0,31870.41\nR003,1999-01-07,3.8,0.0,30200.09\nR003,1999-01-08,5.3,0.0,29137.34\nR003,1999-01-09,6.3,0.0,30807.02\nR003,1999-01-10,5.8,10.6,7301.47\nR003,1999-01-11,3.1,12.1,32197.52\nR003,1999-01-12,-0.6,0.0,21874.32\nR003,1999-01-13,4.3,0.0,20752.19\nR003,1999-01-14,6.9,0.0,29692.56\nR003,1999-01-15,6.8,0.0,25766.88\nR003,1999-01-16,5.5,1.5,32400.0\nR003,1999-01-17,2.7,18.0,20594.31\nR003,1999-01-18,2.1,0.0,32400.0\nR003,1999-01-19,3.1,0.0,32400.0\nR003,1999-01-20,2.9,0.0,32400.0\nR003,1999-01-21,2.6,0.0,32400.0\nR003,1999-01-22,2.3,0.0,32400.0\nR003,1999-01-23,2.2,0.0,32400.0\nR003,1999-01-24,1.8,3.8,19057.77\nR003,1999-01-25,0.4,0.0,32113.47\nR003,1999-01-26,1.8,0.0,32400.0\nR003,1999-01-27,2.0,0.0,32400.0\nR003,1999-01-28,4.0,0.0,32400.0\nR003,1999-01-29,6.0,0.0,29897.77\nR003,1999-01-30,7.8,0.0,27591.76\nR003,1999-01-31,7.8,0.0,22921.59\nR003,1999-02-01,5.5,3.5,21554.98\nR003,1999-02-02,4.8,26.3,0.0\nR003,1999-02-03,-1.3,18.6,321.29\nR003,1999-02-04,-3.8,0.0,27087.99\nR003,1999-02-05,0.0,1.3,32037.72\nR003,1999-02-06,2.5,0.0,33552.76\nR003,1999-02-07,6.8,5.3,24658.58\nR003,1999-02-08,4.1,22.5,11168.17\nR003,1999-02-09,2.6,3.9,33269.13\nR003,1999-02-10,3.6,4.7,34588.92\nR003,1999-02-11,6.7,0.0,35110.56\nR003,1999-02-12,7.3,0.0,36170.51\nR003,1999-02-13,9.2,0.0,36544.9\nR003,1999-02-14,6.8,0.0,35561.03\nR003,1999-02-15,6.2,0.8,29975.7\nR003,1999-02-16,3.6,13.2,18521.7\nR003,1999-02-17,3.2,0.0,37008.63\nR003,1999-02-18,4.8,9.6,12923.25\nR003,1999-02-19,6.2,4.8,33475.24\nR003,1999-02-20,1.5,22.0,5624.07\nR003,1999-02-21,-1.5,5.2,23891.75\nR003,1999-02-22,2.3,14.4,13434.99\nR003,1999-02-23,7.6,1.9,3787.46\nR003,1999-02-24,9.0,1.1,28530.82\nR003,1999-02-25,2.8,23.3,18884.89\nR003,1999-02-26,1.7,0.0,39104.72\nR003,1999-02-27,2.6,1.0,26550.5\nR003,1999-02-28,3.1,4.8,29632.18\nR003,1999-03-01,6.0,0.1,39519.35\nR003,1999-03-02,8.3,0.0,39600.0\nR003,1999-03-03,9.1,0.0,39600.0\nR003,1999-03-04,6.8,0.0,39600.0\nR003,1999-03-05,8.9,0.0,37624.02\nR003,1999-03-06,11.3,0.0,39600.0\nR003,1999-03-07,9.3,0.0,39600.0\nR003,1999-03-08,9.6,0.0,39600.0\nR003,1999-03-09,9.1,0.1,36639.47\nR003,1999-03-10,8.4,2.8,7493.06\nR003,1999-03-11,6.7,1.7,1337.54\nR003,1999-03-12,7.1,0.0,33785.7\nR003,1999-03-13,6.2,1.9,39600.0\nR003,1999-03-14,5.4,5.3,27948.82\nR003,1999-03-15,4.1,22.6,0.0\nR003,1999-03-16,2.3,9.9,0.0\nR003,1999-03-17,2.1,2.9,32742.44\nR003,1999-03-18,3.7,0.0,37704.75\nR003,1999-03-19,7.1,5.3,520.27\nR003,1999-03-20,8.2,0.1,37926.93\nR003,1999-03-21,6.4,10.7,0.0\nR003,1999-03-22,4.4,3.9,28484.75\nR003,1999-03-23,4.5,6.5,12873.2\nR003,1999-03-24,5.1,0.0,40359.67\nR003,1999-03-25,5.9,0.6,29943.44\nR003,1999-03-26,6.0,4.6,5195.76\nR003,1999-03-27,5.9,0.6,24136.37\nR003,1999-03-28,8.3,0.0,41276.54\nR003,1999-03-29,9.4,0.0,42655.18\nR003,1999-03-30,9.7,0.2,41694.99\nR003,1999-03-31,11.4,0.0,43437.92\nR003,1999-04-01,10.7,7.4,28374.16\nR003,1999-04-02,9.2,1.1,34926.39\nR003,1999-04-03,7.5,3.1,37738.76\nR003,1999-04-04,7.7,2.7,44176.72\nR003,1999-04-05,5.4,8.9,30982.98\nR003,1999-04-06,4.8,0.8,45968.62\nR003,1999-04-07,5.6,0.0,43615.78\nR003,1999-04-08,7.7,0.0,44944.32\nR003,1999-04-09,9.0,0.0,43949.47\nR003,1999-04-10,9.7,0.0,43301.49\nR003,1999-04-11,10.5,0.0,46336.99\nR003,1999-04-12,11.6,2.3,29451.68\nR003,1999-04-13,8.5,16.9,4143.05\nR003,1999-04-14,9.2,0.4,46549.51\nR003,1999-04-15,9.5,0.7,33432.64\nR003,1999-04-16,10.7,5.5,30921.19\nR003,1999-04-17,11.8,0.1,46103.12\nR003,1999-04-18,15.0,0.0,44528.18\nR003,1999-04-19,15.7,1.8,46800.0\nR003,1999-04-20,12.1,0.6,35029.06\nR003,1999-04-21,12.7,1.1,44846.91\nR003,1999-04-22,11.0,9.6,8762.32\nR003,1999-04-23,10.3,8.2,10866.5\nR003,1999-04-24,11.9,0.0,42997.46\nR003,1999-04-25,13.9,0.3,46800.0\nR003,1999-04-26,16.1,0.0,47390.53\nR003,1999-04-27,17.4,0.0,46997.26\nR003,1999-04-28,15.3,4.0,34722.11\nR003,1999-04-29,11.6,20.0,1763.57\nR003,1999-04-30,11.8,9.1,29379.06\nR003,1999-05-01,15.0,0.6,44397.21\nR003,1999-05-02,15.5,3.3,46719.89\nR003,1999-05-03,14.3,6.7,43240.56\nR003,1999-05-04,11.4,2.0,45161.12\nR003,1999-05-05,9.1,10.5,32400.0\nR003,1999-05-06,9.0,18.8,998.48\nR003,1999-05-07,6.7,8.2,23799.18\nR003,1999-05-08,7.1,0.9,47275.33\nR003,1999-05-09,9.7,0.0,50007.31\nR003,1999-05-10,12.6,0.0,50080.48\nR003,1999-05-11,14.9,0.0,47865.02\nR003,1999-05-12,15.5,0.0,50222.75\nR003,1999-05-13,13.9,5.4,18871.4\nR003,1999-05-14,13.3,4.6,37898.0\nR003,1999-05-15,15.1,1.2,44128.51\nR003,1999-05-16,17.8,2.1,49623.03\nR003,1999-05-17,15.7,25.2,27821.57\nR003,1999-05-18,9.7,11.5,44187.73\nR003,1999-05-19,8.0,16.6,24445.16\nR003,1999-05-20,8.0,6.8,28829.1\nR003,1999-05-21,11.1,6.7,46112.03\nR003,1999-05-22,13.8,0.0,48641.9\nR003,1999-05-23,17.2,0.0,50400.0\nR003,1999-05-24,19.3,0.6,50400.0\nR003,1999-05-25,17.3,7.2,34390.82\nR003,1999-05-26,16.0,16.6,26401.8\nR003,1999-05-27,15.2,2.7,41601.54\nR003,1999-05-28,16.6,0.0,50511.38\nR003,1999-05-29,17.0,0.1,48705.4\nR003,1999-05-30,16.9,13.4,10929.19\nR003,1999-05-31,16.8,4.5,32703.9\nR003,1999-06-01,17.0,4.0,45645.94\nR003,1999-06-02,17.2,3.5,44336.62\nR003,1999-06-03,17.5,3.3,47260.9\nR003,1999-06-04,17.5,10.8,44817.03\nR003,1999-06-05,17.1,17.6,40975.53\nR003,1999-06-06,16.8,9.1,49729.13\nR003,1999-06-07,19.3,1.7,48727.98\nR003,1999-06-08,19.8,8.7,38281.17\nR003,1999-06-09,21.2,3.8,47784.35\nR003,1999-06-10,22.1,8.5,50599.19\nR003,1999-06-11,23.0,2.1,49201.34\nR003,1999-06-12,20.9,8.8,27667.59\nR003,1999-06-13,19.6,9.5,19344.72\nR003,1999-06-14,20.8,4.0,43079.05\nR003,1999-06-15,21.1,1.4,53472.44\nR003,1999-06-16,21.0,0.7,51014.08\nR003,1999-06-17,20.8,4.2,45103.52\nR003,1999-06-18,20.5,6.4,50574.7\nR003,1999-06-19,22.0,0.8,52921.75\nR003,1999-06-20,23.3,0.9,53426.83\nR003,1999-06-21,20.7,7.3,50844.96\nR003,1999-06-22,20.7,4.4,48718.27\nR003,1999-06-23,22.6,0.9,50819.71\nR003,1999-06-24,18.3,23.4,35673.77\nR003,1999-06-25,17.5,16.6,33650.47\nR003,1999-06-26,17.9,11.0,39459.12\nR003,1999-06-27,17.8,3.2,28231.12\nR003,1999-06-28,18.6,3.9,47949.81\nR003,1999-06-29,19.0,2.9,34102.92\nR003,1999-06-30,19.8,2.2,47170.76\nR003,1999-07-01,20.4,1.8,50216.01\nR003,1999-07-02,21.5,2.3,53942.74\nR003,1999-07-03,21.1,1.7,53930.21\nR003,1999-07-04,21.6,5.7,53383.47\nR003,1999-07-05,21.9,1.5,49313.56\nR003,1999-07-06,23.3,1.5,49870.67\nR003,1999-07-07,24.1,0.0,52667.19\nR003,1999-07-08,24.3,0.2,51947.56\nR003,1999-07-09,24.6,0.9,50868.72\nR003,1999-07-10,25.1,0.3,53734.99\nR003,1999-07-11,24.7,1.0,53417.73\nR003,1999-07-12,24.0,0.0,50373.97\nR003,1999-07-13,21.8,1.3,52180.83\nR003,1999-07-14,21.7,0.0,53650.67\nR003,1999-07-15,23.0,0.5,53612.85\nR003,1999-07-16,23.8,0.3,52749.34\nR003,1999-07-17,23.3,3.1,52269.78\nR003,1999-07-18,23.1,6.5,51091.8\nR003,1999-07-19,22.2,10.7,41671.06\nR003,1999-07-20,21.6,1.6,42669.99\nR003,1999-07-21,21.0,0.0,49224.96\nR003,1999-07-22,20.5,0.0,49618.2\nR003,1999-07-23,21.7,0.0,47799.58\nR003,1999-07-24,22.9,0.5,49353.75\nR003,1999-07-25,24.4,0.0,51899.83\nR003,1999-07-26,25.1,0.0,51863.55\nR003,1999-07-27,26.3,0.1,51477.64\nR003,1999-07-28,27.1,0.1,51093.32\nR003,1999-07-29,23.3,8.9,22696.5\nR003,1999-07-30,23.4,4.9,49300.0\nR003,1999-07-31,24.5,3.1,49699.95\nR003,1999-08-01,23.5,4.2,48965.55\nR003,1999-08-02,23.7,5.8,44418.74\nR003,1999-08-03,25.6,3.1,46800.0\nR003,1999-08-04,25.6,5.8,43094.15\nR003,1999-08-05,24.0,17.1,31134.73\nR003,1999-08-06,22.1,6.1,44954.73\nR003,1999-08-07,21.6,0.0,47717.66\nR003,1999-08-08,22.5,0.0,47968.3\nR003,1999-08-09,24.1,0.8,47657.11\nR003,1999-08-10,24.6,1.5,47347.52\nR003,1999-08-11,24.6,2.0,46800.0\nR003,1999-08-12,22.5,14.3,39144.9\nR003,1999-08-13,23.5,2.2,46764.53\nR003,1999-08-14,24.8,0.0,46800.0\nR003,1999-08-15,24.1,0.6,47158.92\nR003,1999-08-16,24.0,0.3,46800.0\nR003,1999-08-17,24.5,0.0,46312.33\nR003,1999-08-18,25.7,0.9,46734.39\nR003,1999-08-19,26.5,0.0,45974.76\nR003,1999-08-20,26.4,0.1,46800.0\nR003,1999-08-21,25.9,0.3,46800.0\nR003,1999-08-22,25.7,0.8,46800.0\nR003,1999-08-23,24.6,2.7,46800.0\nR003,1999-08-24,22.1,10.5,37522.87\nR003,1999-08-25,21.8,5.6,29696.68\nR003,1999-08-26,19.5,12.1,38153.53\nR003,1999-08-27,19.9,1.1,46775.85\nR003,1999-08-28,17.2,21.6,12677.58\nR003,1999-08-29,17.2,8.1,34601.15\nR003,1999-08-30,18.5,2.6,38569.64\nR003,1999-08-31,19.5,0.7,44515.03\nR003,1999-09-01,20.4,0.0,43032.46\nR003,1999-09-02,21.3,0.4,44289.37\nR003,1999-09-03,21.3,0.3,44017.73\nR003,1999-09-04,20.9,1.4,42819.17\nR003,1999-09-05,22.0,0.3,44439.4\nR003,1999-09-06,21.8,6.1,33519.55\nR003,1999-09-07,19.8,28.3,13719.02\nR003,1999-09-08,19.3,18.7,14472.84\nR003,1999-09-09,19.1,7.8,37071.18\nR003,1999-09-10,19.1,0.4,40658.78\nR003,1999-09-11,18.6,3.5,39141.55\nR003,1999-09-12,17.4,0.7,39942.17\nR003,1999-09-13,15.5,0.0,42876.25\nR003,1999-09-14,15.6,1.3,36773.17\nR003,1999-09-15,15.6,0.0,41048.09\nR003,1999-09-16,13.7,0.0,41080.99\nR003,1999-09-17,15.2,0.0,42938.32\nR003,1999-09-18,15.9,0.0,42739.07\nR003,1999-09-19,16.9,0.0,39337.35\nR003,1999-09-20,17.3,0.1,39942.1\nR003,1999-09-21,15.8,4.0,32316.04\nR003,1999-09-22,15.8,0.0,41283.92\nR003,1999-09-23,16.0,0.0,33926.37\nR003,1999-09-24,16.1,0.0,40398.36\nR003,1999-09-25,15.4,0.2,39688.41\nR003,1999-09-26,15.9,0.7,38493.01\nR003,1999-09-27,15.9,0.0,40185.64\nR003,1999-09-28,18.8,0.0,41178.77\nR003,1999-09-29,20.3,0.0,40896.07\nR003,1999-09-30,22.2,0.0,40837.92\nR003,1999-10-01,22.8,0.0,40764.48\nR003,1999-10-02,22.3,0.0,40371.63\nR003,1999-10-03,21.1,0.0,39978.96\nR003,1999-10-04,20.5,0.0,38208.58\nR003,1999-10-05,19.3,1.3,33719.11\nR003,1999-10-06,20.0,0.2,38399.2\nR003,1999-10-07,19.7,0.0,38170.62\nR003,1999-10-08,18.9,0.0,38264.7\nR003,1999-10-09,17.7,0.0,36028.73\nR003,1999-10-10,14.8,26.4,0.0\nR003,1999-10-11,11.6,19.6,22728.76\nR003,1999-10-12,8.1,3.8,33228.18\nR003,1999-10-13,10.9,3.1,22125.03\nR003,1999-10-14,11.1,4.4,31809.06\nR003,1999-10-15,10.2,25.9,0.0\nR003,1999-10-16,9.5,19.2,28692.35\nR003,1999-10-17,10.1,8.3,25076.53\nR003,1999-10-18,8.3,9.7,33886.66\nR003,1999-10-19,6.6,0.0,34631.86\nR003,1999-10-20,9.6,3.0,22240.97\nR003,1999-10-21,11.6,8.1,17812.67\nR003,1999-10-22,7.0,7.0,32400.0\nR003,1999-10-23,5.0,0.0,36000.0\nR003,1999-10-24,6.7,0.0,36000.0\nR003,1999-10-25,8.4,0.0,36000.0\nR003,1999-10-26,8.5,0.0,34324.89\nR003,1999-10-27,10.0,0.0,34464.48\nR003,1999-10-28,10.3,26.6,19378.07\nR003,1999-10-29,6.4,7.3,34506.09\nR003,1999-10-30,5.5,4.0,4542.18\nR003,1999-10-31,4.0,2.5,35101.13\nR003,1999-11-01,6.1,0.8,25859.63\nR003,1999-11-02,5.8,0.0,35982.01\nR003,1999-11-03,8.4,0.0,35902.65\nR003,1999-11-04,10.6,0.0,35822.82\nR003,1999-11-05,8.0,0.0,35742.62\nR003,1999-11-06,7.2,0.0,35614.35\nR003,1999-11-07,8.4,0.0,35207.61\nR003,1999-11-08,7.7,0.2,33399.61\nR003,1999-11-09,2.9,4.8,22727.45\nR003,1999-11-10,2.2,0.0,35001.16\nR003,1999-11-11,3.3,0.0,34852.05\nR003,1999-11-12,4.1,2.1,29669.46\nR003,1999-11-13,5.1,0.0,18323.96\nR003,1999-11-14,7.4,0.0,34091.81\nR003,1999-11-15,9.7,1.2,32219.8\nR003,1999-11-16,6.4,39.5,16.29\nR003,1999-11-17,1.3,5.6,29500.1\nR003,1999-11-18,8.2,14.8,11012.52\nR003,1999-11-19,8.6,25.0,14516.85\nR003,1999-11-20,5.9,0.0,29917.68\nR003,1999-11-21,6.2,10.9,0.0\nR003,1999-11-22,8.2,0.0,26792.89\nR003,1999-11-23,8.2,0.4,27728.69\nR003,1999-11-24,7.9,20.9,514.74\nR003,1999-11-25,2.9,29.5,18390.74\nR003,1999-11-26,-1.7,11.9,13961.85\nR003,1999-11-27,-3.9,8.2,14400.0\nR003,1999-11-28,-1.3,5.0,26836.41\nR003,1999-11-29,0.8,5.9,521.65\nR003,1999-11-30,1.8,0.4,29538.74\nR003,1999-12-01,1.8,0.0,29555.52\nR003,1999-12-02,4.9,0.0,30711.67\nR003,1999-12-03,4.7,1.2,28800.0\nR003,1999-12-04,1.9,12.4,30361.89\nR003,1999-12-05,2.3,0.0,30757.81\nR003,1999-12-06,6.4,0.0,30386.61\nR003,1999-12-07,6.4,4.1,27408.91\nR003,1999-12-08,5.4,1.2,22813.49\nR003,1999-12-09,5.4,3.7,11203.0\nR003,1999-12-10,4.0,0.0,30417.66\nR003,1999-12-11,6.2,0.0,22171.64\nR003,1999-12-12,5.4,0.0,30423.2\nR003,1999-12-13,4.4,0.0,29259.26\nR003,1999-12-14,4.9,0.0,30422.05\nR003,1999-12-15,3.5,8.7,0.0\nR003,1999-12-16,3.3,1.0,19500.9\nR003,1999-12-17,9.5,0.1,28848.75\nR003,1999-12-18,8.3,0.0,29619.99\nR003,1999-12-19,6.0,0.3,21474.2\nR003,1999-12-20,6.4,0.0,30380.33\nR003,1999-12-21,9.9,0.0,27743.59\nR003,1999-12-22,6.5,0.0,30354.47\nR003,1999-12-23,5.4,0.0,29948.17\nR003,1999-12-24,6.4,0.0,27114.49\nR003,1999-12-25,5.3,0.0,16696.6\nR003,1999-12-26,3.8,8.2,0.0\nR003,1999-12-27,4.6,8.6,0.0\nR003,1999-12-28,6.4,31.1,10594.36\nR003,1999-12-29,10.4,22.1,0.0\nR003,1999-12-30,7.7,19.5,0.0\nR003,1999-12-31,7.2,0.0,6398.06\nR003,2000-01-01,8.8,0.0,11855.68\nR003,2000-01-02,4.9,1.2,8165.53\nR003,2000-01-03,4.6,20.0,0.0\nR003,2000-01-04,2.9,6.1,11305.47\nR003,2000-01-05,2.5,1.4,1700.6\nR003,2000-01-06,3.1,0.0,19771.76\nR003,2000-01-07,1.7,0.0,30556.74\nR003,2000-01-08,0.8,0.0,31986.23\nR003,2000-01-09,1.2,0.0,32052.5\nR003,2000-01-10,0.7,0.0,32120.15\nR003,2000-01-11,0.4,3.3,26324.2\nR003,2000-01-12,1.1,8.6,16983.75\nR003,2000-01-13,-0.6,0.5,32329.98\nR003,2000-01-14,-1.3,4.2,32076.27\nR003,2000-01-15,-1.3,2.9,32400.0\nR003,2000-01-16,-1.2,4.7,31911.78\nR003,2000-01-17,-1.5,0.6,29223.18\nR003,2000-01-18,-0.2,15.6,0.0\nR003,2000-01-19,-0.5,25.1,3600.0\nR003,2000-01-20,-2.5,11.3,18000.0\nR003,2000-01-21,-5.9,1.5,31451.62\nR003,2000-01-22,-4.9,23.7,8302.45\nR003,2000-01-23,-7.9,0.4,29181.68\nR003,2000-01-24,-3.0,0.0,32400.0\nR003,2000-01-25,2.0,34.8,0.0\nR003,2000-01-26,-3.5,4.1,30167.32\nR003,2000-01-27,-7.3,0.0,32400.0\nR003,2000-01-28,-4.1,1.7,32400.0\nR003,2000-01-29,-2.1,0.5,32523.64\nR003,2000-01-30,0.2,0.0,32480.01\nR003,2000-01-31,1.8,18.5,13258.56\nR003,2000-02-01,0.9,13.2,8500.5\nR003,2000-02-02,-0.7,0.0,32699.52\nR003,2000-02-03,3.1,0.0,32400.0\nR003,2000-02-04,2.7,25.5,0.0\nR003,2000-02-05,-2.4,0.1,33609.81\nR003,2000-02-06,-2.8,4.1,25200.0\nR003,2000-02-07,-2.8,0.3,33448.63\nR003,2000-02-08,1.3,0.0,33363.38\nR003,2000-02-09,3.4,0.0,34112.46\nR003,2000-02-10,5.7,0.0,35406.71\nR003,2000-02-11,4.3,0.6,35496.51\nR003,2000-02-12,3.4,2.1,28454.48\nR003,2000-02-13,2.6,0.0,34600.72\nR003,2000-02-14,1.9,0.2,35207.97\nR003,2000-02-15,2.0,0.0,32182.32\nR003,2000-02-16,4.1,0.6,26253.15\nR003,2000-02-17,5.0,0.0,29683.55\nR003,2000-02-18,5.4,0.0,36694.01\nR003,2000-02-19,7.1,0.0,35386.68\nR003,2000-02-20,2.7,6.7,27625.0\nR003,2000-02-21,5.1,0.5,24361.19\nR003,2000-02-22,4.7,14.4,7440.72\nR003,2000-02-23,0.9,18.8,1058.88\nR003,2000-02-24,-1.2,0.2,36861.61\nR003,2000-02-25,-2.0,3.2,22938.85\nR003,2000-02-26,-1.0,0.9,20764.58\nR003,2000-02-27,-1.7,16.9,2488.69\nR003,2000-02-28,-2.7,0.4,39387.63\nR003,2000-02-29,-2.0,0.0,39485.65\nR003,2000-03-01,-0.4,0.0,39585.17\nR003,2000-03-02,0.7,0.0,39600.0\nR003,2000-03-03,2.2,0.0,39600.0\nR003,2000-03-04,7.2,0.0,33826.1\nR003,2000-03-05,6.9,0.0,36000.0\nR003,2000-03-06,4.6,19.6,0.0\nR003,2000-03-07,0.4,6.1,20527.58\nR003,2000-03-08,-0.3,0.0,39600.0\nR003,2000-03-09,3.4,4.1,25200.0\nR003,2000-03-10,4.4,29.4,0.0\nR003,2000-03-11,0.8,25.9,21663.46\nR003,2000-03-12,-0.7,3.9,30623.51\nR003,2000-03-13,-0.5,19.3,22054.49\nR003,2000-03-14,-4.0,0.0,39600.0\nR003,2000-03-15,-1.3,0.0,39837.67\nR003,2000-03-16,4.6,0.0,39600.0\nR003,2000-03-17,6.4,3.9,22857.39\nR003,2000-03-18,8.1,3.2,25005.49\nR003,2000-03-19,3.1,12.5,37369.14\nR003,2000-03-20,3.7,21.7,0.0\nR003,2000-03-21,2.4,4.6,32400.0\nR003,2000-03-22,3.8,0.2,39840.64\nR003,2000-03-23,3.5,0.0,39379.63\nR003,2000-03-24,4.7,0.0,40295.18\nR003,2000-03-25,2.1,6.4,24348.83\nR003,2000-03-26,3.3,0.0,41635.36\nR003,2000-03-27,7.5,0.4,40342.99\nR003,2000-03-28,9.1,0.5,29669.48\nR003,2000-03-29,9.0,0.4,36436.97\nR003,2000-03-30,8.0,0.0,40979.47\nR003,2000-03-31,7.2,0.0,45092.77\nR003,2000-04-01,10.2,0.0,43574.09\nR003,2000-04-02,13.7,0.0,42665.85\nR003,2000-04-03,13.8,0.2,44678.76\nR003,2000-04-04,12.5,0.0,45892.42\nR003,2000-04-05,13.8,0.1,45963.69\nR003,2000-04-06,12.0,3.0,35144.88\nR003,2000-04-07,9.5,17.5,27902.66\nR003,2000-04-08,8.5,3.3,42441.27\nR003,2000-04-09,6.4,14.6,3093.71\nR003,2000-04-10,3.8,6.9,14797.32\nR003,2000-04-11,5.0,0.4,46224.32\nR003,2000-04-12,7.5,0.8,31178.85\nR003,2000-04-13,11.8,0.0,46532.9\nR003,2000-04-14,11.9,3.8,29482.06\nR003,2000-04-15,10.6,2.4,0.0\nR003,2000-04-16,10.9,0.0,41877.64\nR003,2000-04-17,11.2,0.0,46800.0\nR003,2000-04-18,13.8,0.0,46800.0\nR003,2000-04-19,15.1,0.1,46800.0\nR003,2000-04-20,12.4,4.4,43702.87\nR003,2000-04-21,12.3,1.7,44217.09\nR003,2000-04-22,12.2,0.5,46800.0\nR003,2000-04-23,12.4,1.4,43107.19\nR003,2000-04-24,13.6,0.3,46517.46\nR003,2000-04-25,14.5,1.3,47067.76\nR003,2000-04-26,15.6,2.0,47357.48\nR003,2000-04-27,16.6,0.9,47316.69\nR003,2000-04-28,17.7,0.7,41924.94\nR003,2000-04-29,17.7,0.4,45565.09\nR003,2000-04-30,15.4,2.2,36641.47\nR003,2000-05-01,12.5,10.6,41490.65\nR003,2000-05-02,11.7,5.3,34665.37\nR003,2000-05-03,8.8,12.6,10438.73\nR003,2000-05-04,7.4,0.0,44864.3\nR003,2000-05-05,11.3,0.0,47179.98\nR003,2000-05-06,13.1,15.3,23964.47\nR003,2000-05-07,11.5,4.4,36303.43\nR003,2000-05-08,8.0,18.6,10079.47\nR003,2000-05-09,6.9,5.6,42096.29\nR003,2000-05-10,9.0,0.1,38761.8\nR003,2000-05-11,12.6,0.1,49214.31\nR003,2000-05-12,13.3,2.3,47273.75\nR003,2000-05-13,13.3,10.7,27279.41\nR003,2000-05-14,13.0,0.0,43190.62\nR003,2000-05-15,10.7,0.0,50400.0\nR003,2000-05-16,11.5,0.2,48542.71\nR003,2000-05-17,11.5,0.3,48999.64\nR003,2000-05-18,13.6,1.1,47744.13\nR003,2000-05-19,14.6,1.2,45136.7\nR003,2000-05-20,15.1,0.0,49068.71\nR003,2000-05-21,17.9,0.0,50659.41\nR003,2000-05-22,19.4,0.2,50632.72\nR003,2000-05-23,19.8,0.0,50962.95\nR003,2000-05-24,20.3,0.3,50937.54\nR003,2000-05-25,19.9,9.3,49579.66\nR003,2000-05-26,19.3,1.8,47355.56\nR003,2000-05-27,17.9,3.8,11194.44\nR003,2000-05-28,17.9,9.4,44911.42\nR003,2000-05-29,18.5,7.5,49764.4\nR003,2000-05-30,19.1,2.9,50794.0\nR003,2000-05-31,18.4,16.0,45691.64\nR003,2000-06-01,18.2,6.8,40848.3\nR003,2000-06-02,20.2,3.6,50729.85\nR003,2000-06-03,21.0,7.0,39796.25\nR003,2000-06-04,18.8,19.7,34383.49\nR003,2000-06-05,18.6,13.0,36522.68\nR003,2000-06-06,20.6,0.2,48436.97\nR003,2000-06-07,21.6,9.7,49502.6\nR003,2000-06-08,19.4,20.4,40044.3\nR003,2000-06-09,19.1,11.2,41344.11\nR003,2000-06-10,17.1,0.0,51217.64\nR003,2000-06-11,16.2,0.0,52776.45\nR003,2000-06-12,16.5,0.0,53497.75\nR003,2000-06-13,18.1,0.0,52751.89\nR003,2000-06-14,17.7,3.4,37931.88\nR003,2000-06-15,18.6,0.0,53464.23\nR003,2000-06-16,20.8,0.0,53821.65\nR003,2000-06-17,22.3,0.0,53444.39\nR003,2000-06-18,19.3,34.5,37326.77\nR003,2000-06-19,13.2,10.5,49030.84\nR003,2000-06-20,12.2,5.8,37368.36\nR003,2000-06-21,13.1,0.0,51311.83\nR003,2000-06-22,14.9,0.0,32499.94\nR003,2000-06-23,16.5,0.6,53767.43\nR003,2000-06-24,17.0,0.1,50914.42\nR003,2000-06-25,16.6,0.0,52289.27\nR003,2000-06-26,19.4,0.0,52621.7\nR003,2000-06-27,20.8,0.0,52678.25\nR003,2000-06-28,21.4,0.3,53963.25\nR003,2000-06-29,21.2,1.2,48352.14\nR003,2000-06-30,23.0,1.7,49782.28\nR003,2000-07-01,22.6,4.1,49319.48\nR003,2000-07-02,21.1,5.6,26162.2\nR003,2000-07-03,21.3,1.3,47939.56\nR003,2000-07-04,22.3,0.0,52394.68\nR003,2000-07-05,22.8,0.0,51934.18\nR003,2000-07-06,24.0,0.5,53861.38\nR003,2000-07-07,22.8,2.5,53839.2\nR003,2000-07-08,19.5,0.0,52617.68\nR003,2000-07-09,21.5,0.0,53173.74\nR003,2000-07-10,24.6,1.5,53759.98\nR003,2000-07-11,22.7,0.0,51980.46\nR003,2000-07-12,22.6,0.0,52446.38\nR003,2000-07-13,24.2,0.0,51615.19\nR003,2000-07-14,25.7,0.1,53450.28\nR003,2000-07-15,25.0,0.4,44015.39\nR003,2000-07-16,26.0,0.5,52758.4\nR003,2000-07-17,25.9,0.7,53129.18\nR003,2000-07-18,22.7,11.0,7835.91\nR003,2000-07-19,22.1,6.4,26920.72\nR003,2000-07-20,24.7,2.0,45330.3\nR003,2000-07-21,25.5,2.1,43869.79\nR003,2000-07-22,27.0,1.0,48945.98\nR003,2000-07-23,26.4,2.2,51099.08\nR003,2000-07-24,22.1,15.7,2417.93\nR003,2000-07-25,22.9,0.0,45482.01\nR003,2000-07-26,24.1,1.0,49833.26\nR003,2000-07-27,26.0,0.2,50928.66\nR003,2000-07-28,25.4,1.1,50975.77\nR003,2000-07-29,25.8,0.0,49637.6\nR003,2000-07-30,29.5,0.0,50996.64\nR003,2000-07-31,29.8,0.0,50270.6\nR003,2000-08-01,25.9,4.5,45992.07\nR003,2000-08-02,23.4,21.8,9688.26\nR003,2000-08-03,19.2,11.7,19427.31\nR003,2000-08-04,22.6,0.0,46229.94\nR003,2000-08-05,22.9,0.0,48316.83\nR003,2000-08-06,24.1,0.3,47933.81\nR003,2000-08-07,24.4,1.3,47620.98\nR003,2000-08-08,25.6,0.3,47648.7\nR003,2000-08-09,25.7,0.0,48014.71\nR003,2000-08-10,24.2,1.2,47030.38\nR003,2000-08-11,24.8,1.4,45082.17\nR003,2000-08-12,25.7,1.1,47090.41\nR003,2000-08-13,23.6,4.6,46800.0\nR003,2000-08-14,22.6,5.2,29195.62\nR003,2000-08-15,22.1,0.6,46800.0\nR003,2000-08-16,21.8,0.5,46879.42\nR003,2000-08-17,23.1,1.1,46800.0\nR003,2000-08-18,22.5,3.3,45504.01\nR003,2000-08-19,22.8,1.0,45785.19\nR003,2000-08-20,24.1,0.4,46800.0\nR003,2000-08-21,23.5,0.0,46800.0\nR003,2000-08-22,23.9,0.1,46800.0\nR003,2000-08-23,23.7,2.3,46800.0\nR003,2000-08-24,23.3,1.1,46800.0\nR003,2000-08-25,20.8,2.5,39452.54\nR003,2000-08-26,20.8,0.9,44707.05\nR003,2000-08-27,18.5,17.6,21367.87\nR003,2000-08-28,16.7,3.1,35308.12\nR003,2000-08-29,15.9,6.2,16684.55\nR003,2000-08-30,16.7,1.2,33998.58\nR003,2000-08-31,16.7,1.7,34199.31\nR003,2000-09-01,16.0,2.4,40022.02\nR003,2000-09-02,18.0,0.0,45612.08\nR003,2000-09-03,19.4,0.8,42307.13\nR003,2000-09-04,19.8,0.0,43788.97\nR003,2000-09-05,20.7,0.0,43031.03\nR003,2000-09-06,22.1,0.2,43893.64\nR003,2000-09-07,20.1,6.0,30590.27\nR003,2000-09-08,18.3,3.0,40546.52\nR003,2000-09-09,17.8,0.0,43414.59\nR003,2000-09-10,17.9,0.0,41317.2\nR003,2000-09-11,18.2,0.0,43203.4\nR003,2000-09-12,19.0,0.4,43200.0\nR003,2000-09-13,18.6,5.9,35737.62\nR003,2000-09-14,18.7,0.1,38215.99\nR003,2000-09-15,17.5,7.9,39728.41\nR003,2000-09-16,17.1,2.3,37139.35\nR003,2000-09-17,17.2,0.0,42730.34\nR003,2000-09-18,17.8,0.0,41172.43\nR003,2000-09-19,19.0,0.0,41842.88\nR003,2000-09-20,19.8,1.2,41558.84\nR003,2000-09-21,18.9,0.6,41275.06\nR003,2000-09-22,18.7,1.1,36873.35\nR003,2000-09-23,17.8,0.0,41320.36\nR003,2000-09-24,20.0,0.0,41380.52\nR003,2000-09-25,19.7,0.2,40800.26\nR003,2000-09-26,15.7,28.7,18864.06\nR003,2000-09-27,13.1,0.6,40148.42\nR003,2000-09-28,13.5,0.5,37669.27\nR003,2000-09-29,14.5,0.0,32158.34\nR003,2000-09-30,15.5,2.8,24899.93\nR003,2000-10-01,15.1,5.7,35190.49\nR003,2000-10-02,14.8,0.0,39199.93\nR003,2000-10-03,16.0,0.0,39604.38\nR003,2000-10-04,15.8,0.0,39535.86\nR003,2000-10-05,16.2,0.0,38430.57\nR003,2000-10-06,17.2,0.0,38142.39\nR003,2000-10-07,17.4,0.0,31585.21\nR003,2000-10-08,16.4,1.9,25682.93\nR003,2000-10-09,15.0,4.4,19878.66\nR003,2000-10-10,15.5,0.1,35942.64\nR003,2000-10-11,16.2,5.4,30167.17\nR003,2000-10-12,14.3,4.1,25865.51\nR003,2000-10-13,13.2,0.7,33910.76\nR003,2000-10-14,13.7,0.0,36335.64\nR003,2000-10-15,13.3,0.0,36061.82\nR003,2000-10-16,13.1,0.0,33778.63\nR003,2000-10-17,11.0,0.0,36000.0\nR003,2000-10-18,8.3,7.7,17217.28\nR003,2000-10-19,6.7,2.7,29852.26\nR003,2000-10-20,7.0,0.0,36000.0\nR003,2000-10-21,7.5,0.3,148.04\nR003,2000-10-22,9.8,0.3,36000.0\nR003,2000-10-23,10.4,1.1,12398.29\nR003,2000-10-24,11.0,1.3,31892.27\nR003,2000-10-25,10.5,0.2,35665.65\nR003,2000-10-26,10.7,0.1,33583.95\nR003,2000-10-27,10.9,0.0,36000.0\nR003,2000-10-28,11.8,0.0,36000.0\nR003,2000-10-29,10.3,4.0,10336.01\nR003,2000-10-30,9.5,0.0,30065.66\nR003,2000-10-31,9.4,0.0,36000.0\nR003,2000-11-01,10.8,0.0,35999.46\nR003,2000-11-02,10.6,2.8,35919.29\nR003,2000-11-03,10.1,0.0,35838.93\nR003,2000-11-04,10.4,0.0,32603.42\nR003,2000-11-05,10.2,0.0,35643.12\nR003,2000-11-06,10.3,0.0,35486.82\nR003,2000-11-07,11.6,0.0,33337.1\nR003,2000-11-08,11.6,0.0,34443.84\nR003,2000-11-09,10.2,0.0,35029.89\nR003,2000-11-10,11.4,0.0,31122.74\nR003,2000-11-11,11.9,0.1,26702.54\nR003,2000-11-12,8.5,16.0,23289.58\nR003,2000-11-13,4.2,0.0,33011.58\nR003,2000-11-14,4.8,0.0,34319.85\nR003,2000-11-15,7.3,0.0,33709.99\nR003,2000-11-16,7.9,0.0,34052.84\nR003,2000-11-17,8.3,0.0,33696.22\nR003,2000-11-18,7.8,0.0,33692.14\nR003,2000-11-19,8.1,0.0,33324.32\nR003,2000-11-20,8.7,0.0,31950.81\nR003,2000-11-21,8.0,0.0,32030.09\nR003,2000-11-22,6.7,0.0,27739.76\nR003,2000-11-23,6.2,10.9,0.0\nR003,2000-11-24,4.7,5.7,14975.64\nR003,2000-11-25,4.8,0.0,25002.56\nR003,2000-11-26,5.0,0.1,30227.93\nR003,2000-11-27,5.2,0.0,27195.41\nR003,2000-11-28,7.1,0.0,28800.0\nR003,2000-11-29,7.5,0.0,22457.87\nR003,2000-11-30,8.3,0.0,29933.47\nR003,2000-12-01,7.0,0.0,29190.08\nR003,2000-12-02,7.5,0.0,20576.3\nR003,2000-12-03,7.3,1.5,29491.14\nR003,2000-12-04,6.9,0.0,29991.3\nR003,2000-12-05,6.7,0.0,30385.19\nR003,2000-12-06,5.3,0.0,29626.83\nR003,2000-12-07,3.3,0.6,28864.04\nR003,2000-12-08,3.2,1.6,29638.87\nR003,2000-12-09,3.3,0.0,30414.86\nR003,2000-12-10,4.0,0.0,30418.42\nR003,2000-12-11,4.9,0.0,25056.86\nR003,2000-12-12,3.8,0.0,27944.32\nR003,2000-12-13,3.6,1.4,27356.55\nR003,2000-12-14,3.4,0.6,29250.64\nR003,2000-12-15,4.7,2.0,14878.39\nR003,2000-12-16,5.7,0.0,10353.76\nR003,2000-12-17,6.0,20.8,5271.48\nR003,2000-12-18,2.2,0.5,19740.11\nR003,2000-12-19,2.1,0.0,21629.88\nR003,2000-12-20,0.1,0.0,12542.22\nR003,2000-12-21,-2.5,0.5,26824.81\nR003,2000-12-22,-5.5,3.7,29413.96\nR003,2000-12-23,-6.9,0.0,31218.38\nR003,2000-12-24,-4.0,0.0,18884.82\nR003,2000-12-25,1.6,0.0,25381.73\nR003,2000-12-26,2.3,0.0,23117.12\nR003,2000-12-27,5.0,0.2,29259.32\nR003,2000-12-28,7.8,0.0,21644.4\nR003,2000-12-29,8.5,0.0,27343.76\nR003,2000-12-30,10.7,0.0,31511.25\nR003,2000-12-31,7.2,4.0,10877.91\nR003,2001-01-01,7.0,1.4,30700.08\nR003,2001-01-02,5.1,0.0,14617.65\nR003,2001-01-03,4.2,0.0,31730.28\nR003,2001-01-04,3.9,0.0,31789.57\nR003,2001-01-05,5.6,0.0,28996.8\nR003,2001-01-06,5.5,0.0,31912.68\nR003,2001-01-07,5.0,0.0,23913.48\nR003,2001-01-08,5.7,0.0,32040.86\nR003,2001-01-09,8.4,0.0,32106.52\nR003,2001-01-10,8.0,4.3,18503.49\nR003,2001-01-11,3.3,2.3,28640.3\nR003,2001-01-12,3.2,13.3,3671.68\nR003,2001-01-13,1.1,19.8,10877.01\nR003,2001-01-14,-2.4,1.6,12169.7\nR003,2001-01-15,-7.8,0.0,32400.0\nR003,2001-01-16,-4.7,0.0,32400.0\nR003,2001-01-17,-3.2,0.0,32400.0\nR003,2001-01-18,-2.1,0.0,32400.0\nR003,2001-01-19,-2.2,1.1,32400.0\nR003,2001-01-20,0.6,0.4,30700.18\nR003,2001-01-21,0.1,0.0,30131.13\nR003,2001-01-22,-1.8,2.2,30085.53\nR003,2001-01-23,-3.3,0.0,29697.99\nR003,2001-01-24,-5.2,0.0,32400.0\nR003,2001-01-25,-5.0,0.0,32400.0\nR003,2001-01-26,-3.8,0.0,32400.0\nR003,2001-01-27,-0.5,0.0,32400.0\nR003,2001-01-28,3.8,0.0,30939.11\nR003,2001-01-29,6.1,0.0,23843.43\nR003,2001-01-30,7.9,0.0,32445.6\nR003,2001-01-31,8.0,0.0,26448.13\nR003,2001-02-01,9.1,0.0,29450.76\nR003,2001-02-02,5.1,5.5,22954.63\nR003,2001-02-03,4.1,0.9,28800.0\nR003,2001-02-04,2.4,7.6,15828.27\nR003,2001-02-05,0.5,0.2,27033.44\nR003,2001-02-06,3.7,9.7,0.0\nR003,2001-02-07,3.4,6.2,24630.88\nR003,2001-02-08,1.7,0.0,34333.05\nR003,2001-02-09,4.0,0.0,35394.76\nR003,2001-02-10,6.0,0.0,35774.74\nR003,2001-02-11,5.9,0.0,36152.2\nR003,2001-02-12,4.0,0.0,36344.76\nR003,2001-02-13,3.3,0.0,36221.11\nR003,2001-02-14,4.0,0.3,32916.34\nR003,2001-02-15,1.9,0.6,26277.44\nR003,2001-02-16,1.4,0.1,34594.74\nR003,2001-02-17,0.5,0.0,30258.86\nR003,2001-02-18,0.5,15.6,0.0\nR003,2001-02-19,1.6,4.4,25181.22\nR003,2001-02-20,0.0,20.3,0.0\nR003,2001-02-21,-5.0,1.7,36470.2\nR003,2001-02-22,-3.1,22.4,1187.83\nR003,2001-02-23,-0.5,30.7,0.0\nR003,2001-02-24,1.6,13.0,19200.35\nR003,2001-02-25,5.4,42.7,0.0\nR003,2001-02-26,6.6,18.4,8568.22\nR003,2001-02-27,9.1,2.7,36625.42\nR003,2001-02-28,6.2,11.2,19272.36\nR003,2001-03-01,7.4,1.0,20882.38\nR003,2001-03-02,7.9,13.5,9916.8\nR003,2001-03-03,4.3,5.8,24767.38\nR003,2001-03-04,9.1,5.1,22143.73\nR003,2001-03-05,11.4,0.0,39373.48\nR003,2001-03-06,10.3,4.4,33468.11\nR003,2001-03-07,6.5,34.8,0.0\nR003,2001-03-08,5.9,1.3,32890.99\nR003,2001-03-09,4.2,0.0,39600.0\nR003,2001-03-10,5.5,0.0,39600.0\nR003,2001-03-11,8.9,0.0,39600.0\nR003,2001-03-12,10.0,0.0,38357.24\nR003,2001-03-13,9.9,0.2,34703.45\nR003,2001-03-14,8.4,6.0,32400.0\nR003,2001-03-15,8.7,6.2,39600.0\nR003,2001-03-16,5.7,11.2,34594.07\nR003,2001-03-17,4.8,3.6,30657.04\nR003,2001-03-18,8.8,0.0,40545.7\nR003,2001-03-19,11.1,2.9,39600.0\nR003,2001-03-20,10.2,6.4,11228.16\nR003,2001-03-21,8.1,17.2,29015.37\nR003,2001-03-22,7.9,0.0,36879.12\nR003,2001-03-23,10.2,4.6,18362.01\nR003,2001-03-24,6.6,24.1,0.0\nR003,2001-03-25,5.3,0.0,38849.71\nR003,2001-03-26,9.8,1.7,17958.44\nR003,2001-03-27,11.7,3.1,42335.65\nR003,2001-03-28,12.1,1.4,41544.35\nR003,2001-03-29,6.6,10.0,29900.83\nR003,2001-03-30,6.3,0.0,36841.85\nR003,2001-03-31,8.3,0.0,39998.57\nR003,2001-04-01,13.0,0.6,30815.34\nR003,2001-04-02,13.2,17.1,19281.22\nR003,2001-04-03,10.3,2.8,26286.72\nR003,2001-04-04,12.4,0.2,41579.88\nR003,2001-04-05,13.8,4.5,30893.21\nR003,2001-04-06,9.4,19.3,6491.07\nR003,2001-04-07,11.9,0.0,45280.9\nR003,2001-04-08,12.4,0.9,43110.39\nR003,2001-04-09,12.4,0.6,45538.79\nR003,2001-04-10,13.0,0.1,46301.66\nR003,2001-04-11,10.6,17.5,21778.06\nR003,2001-04-12,8.4,8.8,40346.21\nR003,2001-04-13,9.7,0.2,46515.86\nR003,2001-04-14,13.5,0.0,38773.79\nR003,2001-04-15,11.6,1.7,15418.07\nR003,2001-04-16,11.5,1.6,46300.68\nR003,2001-04-17,8.5,9.1,43415.58\nR003,2001-04-18,11.5,0.3,39600.0\nR003,2001-04-19,11.4,19.9,16897.41\nR003,2001-04-20,11.3,0.0,46017.15\nR003,2001-04-21,12.3,0.0,46926.04\nR003,2001-04-22,15.8,0.0,45464.84\nR003,2001-04-23,17.6,2.1,22894.17\nR003,2001-04-24,12.9,23.6,119.87\nR003,2001-04-25,11.2,17.2,2715.61\nR003,2001-04-26,9.5,4.2,39989.27\nR003,2001-04-27,9.5,4.8,36792.97\nR003,2001-04-28,11.6,0.0,46800.0\nR003,2001-04-29,12.9,0.0,43725.17\nR003,2001-04-30,13.7,2.6,48212.84\nR003,2001-05-01,13.1,2.1,46834.4\nR003,2001-05-02,12.7,2.4,15068.05\nR003,2001-05-03,13.4,4.2,23216.98\nR003,2001-05-04,13.4,11.4,15939.89\nR003,2001-05-05,13.3,5.5,29247.79\nR003,2001-05-06,14.0,4.6,45654.23\nR003,2001-05-07,15.4,0.5,48308.47\nR003,2001-05-08,14.9,3.0,47254.08\nR003,2001-05-09,13.1,2.5,34682.86\nR003,2001-05-10,13.2,4.9,31099.47\nR003,2001-05-11,11.6,9.5,7912.32\nR003,2001-05-12,13.0,15.9,42701.5\nR003,2001-05-13,14.4,5.0,43424.08\nR003,2001-05-14,15.1,16.8,25307.55\nR003,2001-05-15,12.6,20.5,82.87\nR003,2001-05-16,12.3,8.5,32229.02\nR003,2001-05-17,12.2,4.9,40775.35\nR003,2001-05-18,14.1,1.1,47593.62\nR003,2001-05-19,14.1,7.8,33244.65\nR003,2001-05-20,15.5,3.6,36692.51\nR003,2001-05-21,13.4,15.2,27904.16\nR003,2001-05-22,14.1,0.0,46055.22\nR003,2001-05-23,13.8,6.1,11378.34\nR003,2001-05-24,10.5,23.0,38885.93\nR003,2001-05-25,11.8,0.0,48164.77\nR003,2001-05-26,16.3,0.0,50400.0\nR003,2001-05-27,16.6,2.1,42407.42\nR003,2001-05-28,15.1,4.2,40603.27\nR003,2001-05-29,12.9,8.3,37224.97\nR003,2001-05-30,12.6,25.9,27242.53\nR003,2001-05-31,11.5,12.5,32191.66\nR003,2001-06-01,14.0,2.1,30000.0\nR003,2001-06-02,21.7,1.0,51100.69\nR003,2001-06-03,18.3,7.8,29475.87\nR003,2001-06-04,20.8,0.0,52518.6\nR003,2001-06-05,18.3,8.4,39064.93\nR003,2001-06-06,13.0,8.7,36626.68\nR003,2001-06-07,13.2,24.2,33061.54\nR003,2001-06-08,14.3,0.0,53186.07\nR003,2001-06-09,15.1,0.0,51342.08\nR003,2001-06-10,16.8,0.2,48618.19\nR003,2001-06-11,19.3,0.0,51391.17\nR003,2001-06-12,22.3,0.0,53132.08\nR003,2001-06-13,23.7,0.7,52385.73\nR003,2001-06-14,22.3,0.2,48371.9\nR003,2001-06-15,24.9,0.1,53464.67\nR003,2001-06-16,20.3,14.0,12202.99\nR003,2001-06-17,18.5,29.8,6709.13\nR003,2001-06-18,19.0,13.4,24818.15\nR003,2001-06-19,20.1,4.8,46555.55\nR003,2001-06-20,22.2,0.0,52386.87\nR003,2001-06-21,26.1,1.6,53046.5\nR003,2001-06-22,21.7,14.0,50382.76\nR003,2001-06-23,20.3,0.2,50351.23\nR003,2001-06-24,20.1,0.0,53957.19\nR003,2001-06-25,20.5,0.0,53961.2\nR003,2001-06-26,18.1,0.0,49395.17\nR003,2001-06-27,17.3,2.5,50950.73\nR003,2001-06-28,17.0,0.4,50361.25\nR003,2001-06-29,15.5,3.2,51952.92\nR003,2001-06-30,17.7,0.2,53164.17\nR003,2001-07-01,19.9,0.4,51127.5\nR003,2001-07-02,20.7,4.6,52921.1\nR003,2001-07-03,21.2,0.6,51455.77\nR003,2001-07-04,21.9,2.4,51615.21\nR003,2001-07-05,22.1,1.6,53888.31\nR003,2001-07-06,22.1,0.8,53869.21\nR003,2001-07-07,22.0,1.3,53757.81\nR003,2001-07-08,20.4,25.7,19719.06\nR003,2001-07-09,19.9,17.1,28970.16\nR003,2001-07-10,20.2,24.1,34370.33\nR003,2001-07-11,21.2,14.1,44499.83\nR003,2001-07-12,21.6,6.4,48598.08\nR003,2001-07-13,23.2,2.0,51005.42\nR003,2001-07-14,23.3,1.6,53089.37\nR003,2001-07-15,24.4,0.4,52125.91\nR003,2001-07-16,24.8,0.0,51274.46\nR003,2001-07-17,24.3,0.2,53395.57\nR003,2001-07-18,23.7,0.8,53141.4\nR003,2001-07-19,22.9,1.8,51529.85\nR003,2001-07-20,22.9,8.0,48293.01\nR003,2001-07-21,24.4,0.0,52387.47\nR003,2001-07-22,25.4,0.5,52348.21\nR003,2001-07-23,26.7,0.4,52308.97\nR003,2001-07-24,26.7,0.1,51918.3\nR003,2001-07-25,26.7,1.0,51529.38\nR003,2001-07-26,27.1,0.0,51221.92\nR003,2001-07-27,27.1,0.2,51455.57\nR003,2001-07-28,26.7,2.8,47113.54\nR003,2001-07-29,26.6,4.6,49623.75\nR003,2001-07-30,26.4,5.2,47598.68\nR003,2001-07-31,26.9,0.5,48845.25\nR003,2001-08-01,26.2,0.1,47368.84\nR003,2001-08-02,25.7,1.9,47554.01\nR003,2001-08-03,25.1,0.3,31877.61\nR003,2001-08-04,24.9,0.0,45097.03\nR003,2001-08-05,25.0,0.0,46851.47\nR003,2001-08-06,26.1,0.0,48362.27\nR003,2001-08-07,24.9,0.5,47953.24\nR003,2001-08-08,25.4,0.1,47980.6\nR003,2001-08-09,25.5,0.5,47669.95\nR003,2001-08-10,25.8,0.0,47361.02\nR003,2001-08-11,25.9,0.0,47390.49\nR003,2001-08-12,27.5,0.0,47420.03\nR003,2001-08-13,27.6,0.2,47449.69\nR003,2001-08-14,26.3,7.9,46800.0\nR003,2001-08-15,23.5,1.2,43958.69\nR003,2001-08-16,23.2,0.7,45301.61\nR003,2001-08-17,24.5,0.6,46800.0\nR003,2001-08-18,24.1,2.6,46800.0\nR003,2001-08-19,22.1,7.8,46173.91\nR003,2001-08-20,22.0,13.6,37687.99\nR003,2001-08-21,23.0,0.6,46190.92\nR003,2001-08-22,24.3,0.8,46800.0\nR003,2001-08-23,23.0,3.6,46800.0\nR003,2001-08-24,21.6,2.2,46250.14\nR003,2001-08-25,21.2,1.4,42801.89\nR003,2001-08-26,19.4,7.4,30266.19\nR003,2001-08-27,19.0,0.0,46732.15\nR003,2001-08-28,19.4,0.0,44673.09\nR003,2001-08-29,19.1,0.0,45724.89\nR003,2001-08-30,18.3,7.9,32439.53\nR003,2001-08-31,14.2,3.8,45146.78\nR003,2001-09-01,15.8,0.0,46197.33\nR003,2001-09-02,17.2,0.0,45602.83\nR003,2001-09-03,21.2,0.0,45651.74\nR003,2001-09-04,17.8,10.7,27271.01\nR003,2001-09-05,17.6,0.0,39680.25\nR003,2001-09-06,19.6,0.0,43881.72\nR003,2001-09-07,22.6,0.4,43933.45\nR003,2001-09-08,19.0,14.3,16007.66\nR003,2001-09-09,17.5,2.0,20544.05\nR003,2001-09-10,18.9,0.0,42745.52\nR003,2001-09-11,21.1,0.0,43200.0\nR003,2001-09-12,22.5,0.0,43200.0\nR003,2001-09-13,21.2,4.2,43173.35\nR003,2001-09-14,18.9,14.0,38453.68\nR003,2001-09-15,19.6,0.0,40746.58\nR003,2001-09-16,19.1,4.5,39226.32\nR003,2001-09-17,18.9,2.0,40789.56\nR003,2001-09-18,20.4,0.0,42835.76\nR003,2001-09-19,20.1,0.0,42767.88\nR003,2001-09-20,19.1,0.2,38276.34\nR003,2001-09-21,18.1,0.0,35517.91\nR003,2001-09-22,17.5,0.0,41939.01\nR003,2001-09-23,18.8,0.1,40700.33\nR003,2001-09-24,18.4,0.0,35206.6\nR003,2001-09-25,19.7,0.0,41727.07\nR003,2001-09-26,18.2,3.5,28377.95\nR003,2001-09-27,17.3,0.1,25382.58\nR003,2001-09-28,17.3,7.0,28856.77\nR003,2001-09-29,14.1,3.1,36041.37\nR003,2001-09-30,12.3,0.0,39843.19\nR003,2001-10-01,13.6,0.0,39203.17\nR003,2001-10-02,15.3,0.0,39236.09\nR003,2001-10-03,16.2,0.0,39310.52\nR003,2001-10-04,18.6,0.0,39560.5\nR003,2001-10-05,20.1,0.0,39314.16\nR003,2001-10-06,18.0,1.0,35353.51\nR003,2001-10-07,15.3,8.4,17737.35\nR003,2001-10-08,16.0,0.0,37618.18\nR003,2001-10-09,17.3,0.0,37434.78\nR003,2001-10-10,17.3,0.0,37204.61\nR003,2001-10-11,16.6,0.0,36814.11\nR003,2001-10-12,13.7,13.4,19409.84\nR003,2001-10-13,11.0,0.0,36000.0\nR003,2001-10-14,10.9,0.0,32114.94\nR003,2001-10-15,12.0,2.1,32134.21\nR003,2001-10-16,11.0,0.0,34652.35\nR003,2001-10-17,11.5,0.4,34459.19\nR003,2001-10-18,11.1,3.0,28698.73\nR003,2001-10-19,9.3,1.6,31457.57\nR003,2001-10-20,9.9,8.2,23617.7\nR003,2001-10-21,10.4,2.2,36000.0\nR003,2001-10-22,9.1,1.1,35116.76\nR003,2001-10-23,9.4,1.4,21008.07\nR003,2001-10-24,9.4,42.3,1784.28\nR003,2001-10-25,1.2,0.2,36000.0\nR003,2001-10-26,1.3,0.0,36000.0\nR003,2001-10-27,4.1,0.0,36000.0\nR003,2001-10-28,6.3,0.0,36000.0\nR003,2001-10-29,6.7,0.6,31571.17\nR003,2001-10-30,8.5,4.8,30423.71\nR003,2001-10-31,7.2,19.6,293.83\nR003,2001-11-01,10.8,2.4,25722.01\nR003,2001-11-02,11.1,0.0,35934.56\nR003,2001-11-03,10.2,0.7,33331.2\nR003,2001-11-04,10.9,0.0,35775.37\nR003,2001-11-05,11.4,0.0,35675.13\nR003,2001-11-06,12.0,3.1,20943.64\nR003,2001-11-07,10.7,3.4,25670.65\nR003,2001-11-08,10.4,0.0,32879.96\nR003,2001-11-09,13.4,0.0,30721.67\nR003,2001-11-10,14.1,0.5,23324.2\nR003,2001-11-11,11.6,7.1,15166.83\nR003,2001-11-12,6.4,2.4,31149.11\nR003,2001-11-13,8.9,3.3,14578.47\nR003,2001-11-14,13.2,5.6,6470.04\nR003,2001-11-15,11.3,24.1,0.0\nR003,2001-11-16,3.8,10.6,22425.7\nR003,2001-11-17,-1.4,22.0,31884.65\nR003,2001-11-18,-6.6,0.6,32142.78\nR003,2001-11-19,0.6,3.9,22917.62\nR003,2001-11-20,4.2,3.9,18341.0\nR003,2001-11-21,2.5,37.0,0.0\nR003,2001-11-22,-1.8,3.0,14152.01\nR003,2001-11-23,2.2,0.0,31851.41\nR003,2001-11-24,3.0,0.0,22227.88\nR003,2001-11-25,5.2,2.3,9209.87\nR003,2001-11-26,4.8,18.9,0.0\nR003,2001-11-27,3.3,2.5,28800.0\nR003,2001-11-28,2.6,4.4,28957.74\nR003,2001-11-29,2.0,0.0,30667.46\nR003,2001-11-30,4.0,0.0,24971.22\nR003,2001-12-01,6.4,0.0,30325.38\nR003,2001-12-02,6.5,0.0,30341.2\nR003,2001-12-03,6.3,0.0,30355.56\nR003,2001-12-04,6.0,0.0,29221.7\nR003,2001-12-05,4.2,4.8,1610.03\nR003,2001-12-06,0.8,5.9,20910.2\nR003,2001-12-07,0.6,0.0,29629.61\nR003,2001-12-08,4.4,2.3,6444.11\nR003,2001-12-09,5.9,0.3,28800.0\nR003,2001-12-10,4.9,12.8,4173.54\nR003,2001-12-11,1.3,4.8,0.0\nR003,2001-12-12,2.3,2.5,18610.4\nR003,2001-12-13,3.1,10.9,3969.61\nR003,2001-12-14,3.9,12.9,12212.95\nR003,2001-12-15,0.1,10.5,24035.5\nR003,2001-12-16,-1.6,0.3,25536.26\nR003,2001-12-17,2.9,29.8,0.0\nR003,2001-12-18,4.9,35.6,0.0\nR003,2001-12-19,3.6,30.2,0.0\nR003,2001-12-20,-1.8,1.7,28542.31\nR003,2001-12-21,-3.6,0.5,29192.9\nR003,2001-12-22,0.8,1.8,25200.0\nR003,2001-12-23,3.0,7.2,2339.2\nR003,2001-12-24,5.5,1.7,29476.79\nR003,2001-12-25,6.6,32.9,0.0\nR003,2001-12-26,5.3,47.0,0.0\nR003,2001-12-27,4.1,0.7,31364.49\nR003,2001-12-28,7.3,21.3,3422.02\nR003,2001-12-29,6.1,23.8,2650.55\nR003,2001-12-30,3.3,0.0,31321.85\nR003,2001-12-31,5.5,16.7,0.0\nR003,2002-01-01,6.7,27.0,0.0\nR003,2002-01-02,2.2,39.5,0.0\nR003,2002-01-03,0.9,21.9,0.0\nR003,2002-01-04,-2.7,1.4,25200.0\nR003,2002-01-05,-1.5,5.2,0.0\nR003,2002-01-06,1.3,0.9,22484.0\nR003,2002-01-07,1.9,5.5,22380.75\nR003,2002-01-08,-2.7,1.9,21076.2\nR003,2002-01-09,-7.1,0.0,32080.72\nR003,2002-01-10,-4.6,3.2,23607.8\nR003,2002-01-11,-1.8,6.3,20395.5\nR003,2002-01-12,-1.2,1.8,30327.36\nR003,2002-01-13,0.2,0.2,32354.56\nR003,2002-01-14,1.5,0.0,32400.0\nR003,2002-01-15,1.4,0.0,32400.0\nR003,2002-01-16,0.9,0.0,32400.0\nR003,2002-01-17,1.1,0.0,32400.0\nR003,2002-01-18,-0.5,0.0,28236.59\nR003,2002-01-19,2.5,3.4,10563.32\nR003,2002-01-20,2.6,3.2,7670.38\nR003,2002-01-21,1.9,16.5,1632.86\nR003,2002-01-22,1.1,2.7,29859.93\nR003,2002-01-23,-0.5,0.0,32400.0\nR003,2002-01-24,1.7,0.0,32400.0\nR003,2002-01-25,4.9,0.0,32400.0\nR003,2002-01-26,4.6,0.1,26236.04\nR003,2002-01-27,5.3,0.0,32400.0\nR003,2002-01-28,4.1,0.0,32400.0\nR003,2002-01-29,2.7,5.5,32400.0\nR003,2002-01-30,3.6,16.3,0.0\nR003,2002-01-31,2.1,17.3,28006.33\nR003,2002-02-01,4.2,0.7,17111.35\nR003,2002-02-02,2.4,0.2,29939.39\nR003,2002-02-03,1.6,0.0,32400.0\nR003,2002-02-04,4.7,0.0,32940.01\nR003,2002-02-05,7.9,0.0,33927.11\nR003,2002-02-06,9.6,0.0,24432.87\nR003,2002-02-07,9.4,0.0,34440.65\nR003,2002-02-08,6.6,0.0,35038.01\nR003,2002-02-09,6.3,0.0,28198.35\nR003,2002-02-10,4.2,3.7,33607.33\nR003,2002-02-11,6.0,0.0,19500.3\nR003,2002-02-12,5.7,2.7,970.06\nR003,2002-02-13,4.2,12.9,4421.33\nR003,2002-02-14,3.8,21.1,17657.12\nR003,2002-02-15,0.3,15.2,10622.02\nR003,2002-02-16,-2.6,0.0,36651.11\nR003,2002-02-17,-1.7,0.6,28807.01\nR003,2002-02-18,1.6,0.0,37047.77\nR003,2002-02-19,5.1,0.0,37913.72\nR003,2002-02-20,8.6,0.0,36577.54\nR003,2002-02-21,7.2,2.8,16923.53\nR003,2002-02-22,9.6,0.0,38435.14\nR003,2002-02-23,10.0,0.0,38611.49\nR003,2002-02-24,8.4,0.0,30331.01\nR003,2002-02-25,8.7,0.4,34420.23\nR003,2002-02-26,4.4,14.1,2903.08\nR003,2002-02-27,0.9,0.0,39325.64\nR003,2002-02-28,4.2,0.0,38809.29\nR003,2002-03-01,7.3,2.7,7816.81\nR003,2002-03-02,8.5,5.2,24974.25\nR003,2002-03-03,9.5,0.0,36000.0\nR003,2002-03-04,6.6,17.1,0.0\nR003,2002-03-05,1.7,3.7,37949.26\nR003,2002-03-06,6.4,0.0,39600.0\nR003,2002-03-07,11.2,0.0,36402.14\nR003,2002-03-08,9.9,11.9,21600.0\nR003,2002-03-09,1.6,5.5,39600.0\nR003,2002-03-10,6.0,0.2,39600.0\nR003,2002-03-11,8.9,6.3,4133.14\nR003,2002-03-12,10.6,0.5,36019.97\nR003,2002-03-13,9.7,1.4,30594.26\nR003,2002-03-14,8.5,1.5,39158.73\nR003,2002-03-15,7.6,2.5,8974.47\nR003,2002-03-16,6.1,1.5,37592.74\nR003,2002-03-17,4.4,8.4,33860.92\nR003,2002-03-18,6.3,0.0,39690.27\nR003,2002-03-19,6.9,0.0,32369.08\nR003,2002-03-20,6.4,0.0,21199.51\nR003,2002-03-21,6.5,2.8,4431.72\nR003,2002-03-22,5.5,7.7,29867.93\nR003,2002-03-23,7.0,0.4,37544.24\nR003,2002-03-24,8.1,4.9,12130.41\nR003,2002-03-25,8.5,14.7,22662.96\nR003,2002-03-26,7.6,7.7,26166.93\nR003,2002-03-27,5.6,4.8,28980.64\nR003,2002-03-28,4.3,3.7,40100.36\nR003,2002-03-29,4.3,1.3,39759.42\nR003,2002-03-30,5.0,0.0,40239.15\nR003,2002-03-31,6.3,0.0,43218.07\nR003,2002-04-01,8.2,0.0,42840.73\nR003,2002-04-02,10.1,0.0,44879.93\nR003,2002-04-03,9.5,0.0,40163.71\nR003,2002-04-04,5.8,0.2,38477.96\nR003,2002-04-05,6.8,0.9,38620.42\nR003,2002-04-06,9.3,13.6,27156.11\nR003,2002-04-07,6.7,9.8,394.28\nR003,2002-04-08,2.7,8.9,26060.23\nR003,2002-04-09,1.1,10.1,36054.43\nR003,2002-04-10,2.9,8.7,42685.12\nR003,2002-04-11,4.4,0.0,46356.45\nR003,2002-04-12,6.0,0.0,45776.42\nR003,2002-04-13,7.8,0.0,41918.55\nR003,2002-04-14,9.2,0.0,21767.85\nR003,2002-04-15,9.9,2.9,22127.9\nR003,2002-04-16,11.5,0.0,45567.42\nR003,2002-04-17,10.0,7.3,28782.3\nR003,2002-04-18,10.7,1.2,19680.25\nR003,2002-04-19,10.7,1.0,45363.66\nR003,2002-04-20,9.8,4.1,0.0\nR003,2002-04-21,8.2,11.7,1414.82\nR003,2002-04-22,7.7,8.1,14079.39\nR003,2002-04-23,7.4,7.5,4164.65\nR003,2002-04-24,8.2,1.5,41087.78\nR003,2002-04-25,8.6,1.1,39455.27\nR003,2002-04-26,8.3,0.0,41111.12\nR003,2002-04-27,7.1,0.8,45521.86\nR003,2002-04-28,8.0,0.5,46800.0\nR003,2002-04-29,10.0,0.2,47923.96\nR003,2002-04-30,11.3,2.8,42751.57\nR003,2002-05-01,12.5,0.7,47176.74\nR003,2002-05-02,12.7,1.7,46850.56\nR003,2002-05-03,12.4,4.3,44332.87\nR003,2002-05-04,11.1,1.5,46150.39\nR003,2002-05-05,10.2,6.0,42184.26\nR003,2002-05-06,10.2,12.0,11485.96\nR003,2002-05-07,10.7,6.1,40115.9\nR003,2002-05-08,11.0,9.8,14492.32\nR003,2002-05-09,11.6,6.3,41040.26\nR003,2002-05-10,12.2,6.4,49594.89\nR003,2002-05-11,12.1,2.7,44180.67\nR003,2002-05-12,13.4,0.6,46093.69\nR003,2002-05-13,15.0,0.1,45307.12\nR003,2002-05-14,15.6,0.0,50377.75\nR003,2002-05-15,14.7,0.0,50400.0\nR003,2002-05-16,14.7,3.4,47616.7\nR003,2002-05-17,15.0,3.8,47543.72\nR003,2002-05-18,15.6,0.3,47782.49\nR003,2002-05-19,15.4,0.9,47410.03\nR003,2002-05-20,13.9,1.6,45757.27\nR003,2002-05-21,14.1,0.0,39794.6\nR003,2002-05-22,15.0,0.0,48372.85\nR003,2002-05-23,15.9,3.6,47419.76\nR003,2002-05-24,15.2,7.0,30491.16\nR003,2002-05-25,16.8,1.0,48280.48\nR003,2002-05-26,18.3,0.1,50540.6\nR003,2002-05-27,20.6,0.0,51234.97\nR003,2002-05-28,19.0,5.5,50300.56\nR003,2002-05-29,17.1,4.6,46871.96\nR003,2002-05-30,20.0,0.4,51169.43\nR003,2002-05-31,17.8,10.7,24970.0\nR003,2002-06-01,15.6,6.8,43718.64\nR003,2002-06-02,15.1,4.1,46264.69\nR003,2002-06-03,15.8,4.1,46842.42\nR003,2002-06-04,18.3,0.4,49695.38\nR003,2002-06-05,16.6,14.8,37366.94\nR003,2002-06-06,13.7,5.2,33184.54\nR003,2002-06-07,14.5,2.9,43599.19\nR003,2002-06-08,17.6,0.0,50829.43\nR003,2002-06-09,20.6,0.9,51673.85\nR003,2002-06-10,21.2,0.0,53525.24\nR003,2002-06-11,24.3,0.5,50580.02\nR003,2002-06-12,18.7,24.4,8284.28\nR003,2002-06-13,18.1,13.4,28491.77\nR003,2002-06-14,18.4,7.8,48189.86\nR003,2002-06-15,19.0,0.9,43127.04\nR003,2002-06-16,20.2,0.6,42090.39\nR003,2002-06-17,20.4,0.2,49011.85\nR003,2002-06-18,15.6,19.9,10618.76\nR003,2002-06-19,15.4,6.2,28363.45\nR003,2002-06-20,15.6,37.2,0.0\nR003,2002-06-21,21.2,4.9,48180.84\nR003,2002-06-22,21.3,0.0,51480.52\nR003,2002-06-23,22.6,0.0,48216.31\nR003,2002-06-24,23.6,0.0,48938.3\nR003,2002-06-25,25.4,0.0,53960.26\nR003,2002-06-26,23.2,7.6,46439.65\nR003,2002-06-27,20.7,23.3,25145.69\nR003,2002-06-28,20.3,11.0,34088.52\nR003,2002-06-29,19.7,5.4,41283.99\nR003,2002-06-30,20.1,7.5,38056.55\nR003,2002-07-01,20.7,0.8,47650.07\nR003,2002-07-02,21.5,1.9,51603.31\nR003,2002-07-03,21.6,6.9,50789.06\nR003,2002-07-04,21.2,5.4,43090.05\nR003,2002-07-05,20.5,12.5,51188.33\nR003,2002-07-06,21.5,1.3,50549.01\nR003,2002-07-07,23.3,0.6,51136.49\nR003,2002-07-08,24.7,0.5,51585.81\nR003,2002-07-09,25.0,6.0,53804.52\nR003,2002-07-10,25.4,1.1,53411.92\nR003,2002-07-11,24.5,6.1,53745.42\nR003,2002-07-12,23.9,7.8,51283.06\nR003,2002-07-13,23.8,5.7,49769.37\nR003,2002-07-14,23.3,3.2,53640.84\nR003,2002-07-15,22.5,5.9,44445.26\nR003,2002-07-16,22.3,4.0,49180.07\nR003,2002-07-17,22.8,1.7,51306.26\nR003,2002-07-18,23.1,0.7,52133.19\nR003,2002-07-19,24.9,0.1,50880.78\nR003,2002-07-20,26.9,0.5,53141.01\nR003,2002-07-21,26.5,0.0,52746.18\nR003,2002-07-22,26.0,0.3,52353.06\nR003,2002-07-23,21.2,25.5,1877.97\nR003,2002-07-24,20.5,4.8,7497.74\nR003,2002-07-25,19.8,10.1,22742.1\nR003,2002-07-26,21.2,2.0,47109.07\nR003,2002-07-27,22.3,0.2,50555.23\nR003,2002-07-28,24.1,1.1,50733.17\nR003,2002-07-29,24.3,2.6,50649.74\nR003,2002-07-30,23.7,1.8,49660.05\nR003,2002-07-31,24.6,0.3,50634.48\nR003,2002-08-01,24.6,2.1,49766.26\nR003,2002-08-02,24.5,0.7,49477.71\nR003,2002-08-03,22.5,2.0,48505.56\nR003,2002-08-04,21.6,8.0,39802.77\nR003,2002-08-05,21.5,0.7,46844.77\nR003,2002-08-06,20.7,11.7,5979.82\nR003,2002-08-07,20.7,0.9,41355.48\nR003,2002-08-08,21.7,0.0,47296.64\nR003,2002-08-09,21.0,7.6,38909.78\nR003,2002-08-10,20.4,6.1,36494.97\nR003,2002-08-11,21.2,4.9,45631.44\nR003,2002-08-12,22.8,0.1,45354.16\nR003,2002-08-13,25.4,0.0,47443.58\nR003,2002-08-14,24.2,1.7,47138.56\nR003,2002-08-15,21.6,9.7,25605.83\nR003,2002-08-16,21.2,2.1,46743.28\nR003,2002-08-17,20.7,3.2,44099.37\nR003,2002-08-18,20.2,4.4,44760.53\nR003,2002-08-19,20.1,3.1,46503.12\nR003,2002-08-20,21.1,1.3,44938.74\nR003,2002-08-21,20.7,12.9,37493.77\nR003,2002-08-22,19.2,9.7,43815.04\nR003,2002-08-23,20.0,6.3,38218.38\nR003,2002-08-24,19.7,0.0,46262.67\nR003,2002-08-25,19.8,0.1,46725.93\nR003,2002-08-26,19.4,1.3,44234.59\nR003,2002-08-27,18.8,0.2,45634.29\nR003,2002-08-28,18.4,1.5,44095.88\nR003,2002-08-29,18.4,0.1,45513.18\nR003,2002-08-30,19.6,0.6,46440.62\nR003,2002-08-31,20.6,0.3,46140.16\nR003,2002-09-01,22.2,0.9,45865.05\nR003,2002-09-02,21.5,9.0,44306.06\nR003,2002-09-03,19.3,9.4,44355.33\nR003,2002-09-04,19.4,5.7,42670.21\nR003,2002-09-05,20.7,2.1,43813.95\nR003,2002-09-06,21.3,4.8,40440.16\nR003,2002-09-07,19.6,12.7,25670.67\nR003,2002-09-08,19.5,1.5,36525.61\nR003,2002-09-09,18.3,6.2,19825.84\nR003,2002-09-10,18.1,2.6,39007.85\nR003,2002-09-11,16.9,0.0,43200.0\nR003,2002-09-12,16.9,3.8,25516.76\nR003,2002-09-13,17.7,0.0,39600.0\nR003,2002-09-14,18.1,0.0,41024.48\nR003,2002-09-15,17.8,2.7,40273.05\nR003,2002-09-16,20.3,0.0,41408.49\nR003,2002-09-17,17.8,31.2,22339.61\nR003,2002-09-18,14.5,1.1,34528.09\nR003,2002-09-19,14.0,1.6,40235.83\nR003,2002-09-20,14.4,0.0,39423.93\nR003,2002-09-21,16.4,0.0,41577.38\nR003,2002-09-22,18.5,0.0,42567.2\nR003,2002-09-23,21.6,0.0,42511.84\nR003,2002-09-24,23.1,0.0,41682.38\nR003,2002-09-25,23.0,0.0,41718.01\nR003,2002-09-26,21.7,0.0,40797.53\nR003,2002-09-27,23.1,0.0,41152.36\nR003,2002-09-28,21.1,2.1,40231.22\nR003,2002-09-29,19.4,0.4,36754.06\nR003,2002-09-30,18.8,4.2,30312.51\nR003,2002-10-01,17.2,2.8,38589.43\nR003,2002-10-02,16.5,12.8,20770.13\nR003,2002-10-03,13.9,0.3,31041.51\nR003,2002-10-04,12.1,0.0,38582.22\nR003,2002-10-05,12.2,0.0,39294.64\nR003,2002-10-06,14.4,0.0,38785.1\nR003,2002-10-07,17.3,0.0,35017.66\nR003,2002-10-08,17.3,3.5,25805.34\nR003,2002-10-09,14.6,2.1,32470.08\nR003,2002-10-10,14.3,0.8,36747.82\nR003,2002-10-11,15.8,0.0,35443.31\nR003,2002-10-12,18.0,0.6,36281.75\nR003,2002-10-13,19.0,0.0,34869.99\nR003,2002-10-14,19.7,0.0,32380.77\nR003,2002-10-15,17.8,3.5,20367.71\nR003,2002-10-16,15.0,13.8,1527.28\nR003,2002-10-17,15.6,0.0,34829.07\nR003,2002-10-18,17.0,0.0,36000.0\nR003,2002-10-19,19.0,0.0,36000.0\nR003,2002-10-20,17.1,0.5,36000.0\nR003,2002-10-21,9.3,23.0,11730.04\nR003,2002-10-22,6.2,0.0,36000.0\nR003,2002-10-23,13.3,1.9,19489.68\nR003,2002-10-24,15.5,0.3,34335.63\nR003,2002-10-25,16.2,0.0,28699.18\nR003,2002-10-26,13.1,2.7,28488.24\nR003,2002-10-27,11.8,1.8,16173.07\nR003,2002-10-28,9.5,2.9,35786.76\nR003,2002-10-29,6.4,30.2,513.34\nR003,2002-10-30,2.8,4.9,35335.32\nR003,2002-10-31,4.0,1.0,32133.06\nR003,2002-11-01,5.8,3.1,35070.76\nR003,2002-11-02,5.5,0.0,32367.17\nR003,2002-11-03,7.4,0.0,35877.14\nR003,2002-11-04,8.5,22.0,1749.23\nR003,2002-11-05,11.4,0.0,35720.29\nR003,2002-11-06,12.5,0.0,19793.74\nR003,2002-11-07,13.0,0.0,27835.35\nR003,2002-11-08,12.7,0.0,35266.79\nR003,2002-11-09,14.3,0.0,35116.26\nR003,2002-11-10,12.2,15.4,1985.45\nR003,2002-11-11,10.6,23.7,10637.96\nR003,2002-11-12,6.0,2.7,32683.39\nR003,2002-11-13,5.0,9.2,29227.93\nR003,2002-11-14,5.2,0.0,29259.74\nR003,2002-11-15,5.7,0.0,31870.4\nR003,2002-11-16,8.6,0.0,34071.59\nR003,2002-11-17,11.4,0.0,33701.32\nR003,2002-11-18,11.8,0.0,33330.78\nR003,2002-11-19,11.6,0.0,33325.3\nR003,2002-11-20,12.0,0.0,32955.5\nR003,2002-11-21,11.2,0.0,30890.1\nR003,2002-11-22,9.2,0.0,29650.95\nR003,2002-11-23,11.6,0.0,29769.44\nR003,2002-11-24,11.1,0.0,28287.98\nR003,2002-11-25,9.5,0.0,29071.79\nR003,2002-11-26,8.0,3.4,14419.14\nR003,2002-11-27,7.3,0.0,26906.96\nR003,2002-11-28,6.1,0.1,30261.88\nR003,2002-11-29,8.0,0.0,29536.53\nR003,2002-11-30,6.9,0.0,30725.88\nR003,2002-12-01,5.2,0.0,29560.15\nR003,2002-12-02,6.5,0.0,30335.67\nR003,2002-12-03,8.1,0.0,29206.95\nR003,2002-12-04,9.4,0.0,26376.52\nR003,2002-12-05,8.9,0.0,29611.05\nR003,2002-12-06,9.3,0.0,7637.66\nR003,2002-12-07,7.7,14.9,3557.91\nR003,2002-12-08,2.4,15.5,264.49\nR003,2002-12-09,-1.0,3.4,11704.4\nR003,2002-12-10,-1.6,0.2,26565.84\nR003,2002-12-11,-1.8,3.2,26470.5\nR003,2002-12-12,-5.0,1.5,30423.24\nR003,2002-12-13,-5.5,0.0,27424.15\nR003,2002-12-14,-4.6,0.0,29645.24\nR003,2002-12-15,-5.5,0.0,28863.11\nR003,2002-12-16,-5.8,0.0,29636.15\nR003,2002-12-17,-6.9,0.0,30799.05\nR003,2002-12-18,-7.2,0.0,28800.0\nR003,2002-12-19,-4.9,0.0,28513.27\nR003,2002-12-20,-6.1,0.0,4430.8\nR003,2002-12-21,-2.9,1.9,28624.44\nR003,2002-12-22,-1.2,8.6,22260.14\nR003,2002-12-23,-1.6,16.2,28315.56\nR003,2002-12-24,-2.4,22.4,0.0\nR003,2002-12-25,-10.0,0.9,29517.99\nR003,2002-12-26,-9.7,0.9,26892.32\nR003,2002-12-27,-6.0,0.6,31359.64\nR003,2002-12-28,-1.4,0.0,24791.9\nR003,2002-12-29,3.1,0.4,29291.7\nR003,2002-12-30,7.0,8.9,20560.81\nR003,2002-12-31,7.5,12.1,962.15\nR003,2003-01-01,7.4,10.0,0.0\nR003,2003-01-02,6.7,14.5,3161.06\nR003,2003-01-03,6.5,3.1,15694.1\nR003,2003-01-04,5.8,17.2,0.0\nR003,2003-01-05,4.2,0.3,1880.1\nR003,2003-01-06,5.8,21.1,9080.89\nR003,2003-01-07,2.9,14.8,31109.19\nR003,2003-01-08,7.0,4.0,19110.13\nR003,2003-01-09,3.9,10.3,24976.56\nR003,2003-01-10,3.2,0.0,32131.49\nR003,2003-01-11,6.6,3.9,13666.74\nR003,2003-01-12,7.8,0.0,0.0\nR003,2003-01-13,2.9,13.7,0.0\nR003,2003-01-14,-0.6,3.4,30702.07\nR003,2003-01-15,-1.3,4.5,22307.12\nR003,2003-01-16,0.3,2.5,30578.07\nR003,2003-01-17,0.6,1.4,23362.88\nR003,2003-01-18,1.7,0.3,28247.68\nR003,2003-01-19,-0.7,0.0,32400.0\nR003,2003-01-20,0.3,0.0,32400.0\nR003,2003-01-21,-1.0,0.0,32400.0\nR003,2003-01-22,-1.0,0.0,32400.0\nR003,2003-01-23,0.4,0.5,21252.4\nR003,2003-01-24,2.9,0.0,30336.0\nR003,2003-01-25,4.3,0.0,23169.23\nR003,2003-01-26,2.4,1.1,29200.84\nR003,2003-01-27,1.5,1.8,32400.0\nR003,2003-01-28,-0.1,0.1,32051.39\nR003,2003-01-29,1.8,1.5,32400.0\nR003,2003-01-30,4.7,11.1,2302.36\nR003,2003-01-31,7.9,26.5,2321.6\nR003,2003-02-01,5.9,21.8,7795.27\nR003,2003-02-02,5.8,4.3,24238.98\nR003,2003-02-03,5.1,14.0,317.24\nR003,2003-02-04,2.4,1.2,30947.35\nR003,2003-02-05,3.9,0.2,33597.93\nR003,2003-02-06,6.2,0.0,29348.15\nR003,2003-02-07,6.3,0.0,29602.39\nR003,2003-02-08,6.2,1.0,23730.61\nR003,2003-02-09,2.9,12.8,0.0\nR003,2003-02-10,0.8,9.6,27763.33\nR003,2003-02-11,-0.2,1.7,30400.17\nR003,2003-02-12,-1.5,0.0,35492.93\nR003,2003-02-13,-0.6,0.0,36853.08\nR003,2003-02-14,-0.7,1.1,34500.37\nR003,2003-02-15,0.5,3.8,24488.04\nR003,2003-02-16,-1.2,3.6,30454.22\nR003,2003-02-17,-2.9,1.6,35665.67\nR003,2003-02-18,-3.9,0.0,37699.58\nR003,2003-02-19,-4.6,0.0,36034.04\nR003,2003-02-20,-2.5,3.5,22400.14\nR003,2003-02-21,-3.5,4.2,27807.33\nR003,2003-02-22,-3.5,0.9,29704.07\nR003,2003-02-23,-4.0,0.0,38577.75\nR003,2003-02-24,-0.7,0.0,29438.53\nR003,2003-02-25,1.1,1.3,35365.42\nR003,2003-02-26,1.6,6.2,21097.68\nR003,2003-02-27,1.0,1.8,28842.89\nR003,2003-02-28,0.6,1.7,37250.08\nR003,2003-03-01,0.4,0.0,39165.36\nR003,2003-03-02,1.4,0.0,39600.0\nR003,2003-03-03,4.1,0.2,6253.43\nR003,2003-03-04,5.3,0.0,19508.01\nR003,2003-03-05,5.7,0.0,39600.0\nR003,2003-03-06,6.8,0.0,38516.45\nR003,2003-03-07,6.3,8.9,5735.2\nR003,2003-03-08,5.0,1.2,39014.17\nR003,2003-03-09,5.8,0.0,39494.49\nR003,2003-03-10,4.5,5.0,3996.41\nR003,2003-03-11,4.6,3.9,24586.08\nR003,2003-03-12,4.3,6.0,27158.65\nR003,2003-03-13,4.4,0.3,24179.98\nR003,2003-03-14,4.8,0.9,21040.54\nR003,2003-03-15,4.4,8.2,34754.83\nR003,2003-03-16,3.1,0.1,39600.0\nR003,2003-03-17,1.9,2.3,38810.13\nR003,2003-03-18,2.5,0.0,40018.61\nR003,2003-03-19,2.9,12.0,423.73\nR003,2003-03-20,0.5,7.1,39600.0\nR003,2003-03-21,0.1,10.3,18841.64\nR003,2003-03-22,-2.3,17.7,0.0\nR003,2003-03-23,-3.6,0.0,41018.85\nR003,2003-03-24,0.3,0.0,40956.72\nR003,2003-03-25,4.0,0.0,41230.88\nR003,2003-03-26,3.6,0.0,41177.92\nR003,2003-03-27,1.6,8.9,39874.12\nR003,2003-03-28,-0.9,3.5,39044.45\nR003,2003-03-29,-0.6,0.0,41821.43\nR003,2003-03-30,1.9,0.0,44639.73\nR003,2003-03-31,5.5,0.0,45032.6\nR003,2003-04-01,7.0,0.1,42816.02\nR003,2003-04-02,2.8,11.7,34764.03\nR003,2003-04-03,6.2,1.7,14393.03\nR003,2003-04-04,8.9,4.3,13888.64\nR003,2003-04-05,9.6,0.7,43979.88\nR003,2003-04-06,7.4,25.9,15859.23\nR003,2003-04-07,8.3,1.3,34174.85\nR003,2003-04-08,12.7,0.0,46127.11\nR003,2003-04-09,6.1,11.9,5445.87\nR003,2003-04-10,6.0,0.0,43593.56\nR003,2003-04-11,7.1,0.0,46077.07\nR003,2003-04-12,7.7,0.0,46411.45\nR003,2003-04-13,9.7,0.0,46482.28\nR003,2003-04-14,10.9,0.0,46553.25\nR003,2003-04-15,10.4,0.0,46624.34\nR003,2003-04-16,10.1,0.0,46695.54\nR003,2003-04-17,9.4,1.4,33146.56\nR003,2003-04-18,9.4,0.3,44994.12\nR003,2003-04-19,8.9,2.5,16980.2\nR003,2003-04-20,7.3,10.4,2136.77\nR003,2003-04-21,7.7,5.0,1899.57\nR003,2003-04-22,7.3,0.6,18375.2\nR003,2003-04-23,8.2,2.2,37855.67\nR003,2003-04-24,8.1,4.5,33128.7\nR003,2003-04-25,9.3,9.0,4629.55\nR003,2003-04-26,8.0,13.2,16239.84\nR003,2003-04-27,6.7,0.1,39993.68\nR003,2003-04-28,7.3,0.0,47305.29\nR003,2003-04-29,9.0,0.0,47199.71\nR003,2003-04-30,10.8,1.0,45506.99\nR003,2003-05-01,12.1,0.0,48190.97\nR003,2003-05-02,14.7,0.0,48826.67\nR003,2003-05-03,16.1,0.0,48120.16\nR003,2003-05-04,18.3,0.0,48423.21\nR003,2003-05-05,16.8,0.0,46800.0\nR003,2003-05-06,10.2,2.6,42007.56\nR003,2003-05-07,9.9,0.0,47647.33\nR003,2003-05-08,11.7,0.0,49656.74\nR003,2003-05-09,13.0,1.3,47923.98\nR003,2003-05-10,14.1,0.0,48919.23\nR003,2003-05-11,17.6,0.0,50155.26\nR003,2003-05-12,18.7,0.0,50225.92\nR003,2003-05-13,18.9,0.5,50295.08\nR003,2003-05-14,19.2,0.3,50362.69\nR003,2003-05-15,19.2,0.5,47853.83\nR003,2003-05-16,17.7,4.4,49425.72\nR003,2003-05-17,16.3,5.9,44943.64\nR003,2003-05-18,16.5,0.8,46051.57\nR003,2003-05-19,16.6,8.8,42515.43\nR003,2003-05-20,16.8,3.6,41587.37\nR003,2003-05-21,19.0,0.5,47596.54\nR003,2003-05-22,21.5,0.5,50400.0\nR003,2003-05-23,21.2,0.5,50400.0\nR003,2003-05-24,21.4,0.7,50955.77\nR003,2003-05-25,22.9,1.0,50931.1\nR003,2003-05-26,20.9,2.6,49084.25\nR003,2003-05-27,18.7,7.8,24821.01\nR003,2003-05-28,17.6,9.3,21582.67\nR003,2003-05-29,17.8,4.7,48267.37\nR003,2003-05-30,16.9,12.7,40210.11\nR003,2003-05-31,15.3,4.1,29666.52\nR003,2003-06-01,14.9,12.0,44233.92\nR003,2003-06-02,17.0,0.0,50277.81\nR003,2003-06-03,16.2,0.5,49740.17\nR003,2003-06-04,12.7,4.7,40769.58\nR003,2003-06-05,12.0,0.0,51777.58\nR003,2003-06-06,14.2,0.0,51286.23\nR003,2003-06-07,16.9,0.9,51743.23\nR003,2003-06-08,18.6,0.2,45728.82\nR003,2003-06-09,19.5,0.0,49662.26\nR003,2003-06-10,19.7,0.0,52795.85\nR003,2003-06-11,21.0,0.0,53149.58\nR003,2003-06-12,20.1,0.6,50570.44\nR003,2003-06-13,18.4,3.8,41659.01\nR003,2003-06-14,18.8,0.0,51205.36\nR003,2003-06-15,19.1,0.0,47823.52\nR003,2003-06-16,19.3,0.0,50851.67\nR003,2003-06-17,20.4,2.4,52716.74\nR003,2003-06-18,17.4,0.0,52372.31\nR003,2003-06-19,16.8,0.0,53893.85\nR003,2003-06-20,18.0,0.0,52690.17\nR003,2003-06-21,19.3,0.8,51579.23\nR003,2003-06-22,18.1,1.4,50474.4\nR003,2003-06-23,16.1,0.0,51495.01\nR003,2003-06-24,15.1,1.3,50040.63\nR003,2003-06-25,17.7,0.0,53964.29\nR003,2003-06-26,20.3,0.0,53968.47\nR003,2003-06-27,18.5,1.1,53018.49\nR003,2003-06-28,18.2,0.0,52452.94\nR003,2003-06-29,17.8,0.0,50903.67\nR003,2003-06-30,17.0,0.9,48345.73\nR003,2003-07-01,17.5,1.3,50739.57\nR003,2003-07-02,19.2,0.3,50420.65\nR003,2003-07-03,22.9,0.0,53930.32\nR003,2003-07-04,26.1,0.0,53915.47\nR003,2003-07-05,27.1,0.0,53898.46\nR003,2003-07-06,26.0,0.7,53879.29\nR003,2003-07-07,21.0,9.1,47179.6\nR003,2003-07-08,19.5,0.3,50728.62\nR003,2003-07-09,19.0,0.3,52570.23\nR003,2003-07-10,18.5,2.2,48778.81\nR003,2003-07-11,20.3,0.0,52334.54\nR003,2003-07-12,21.7,0.2,53065.1\nR003,2003-07-13,20.8,0.0,53686.17\nR003,2003-07-14,20.9,0.0,52026.89\nR003,2003-07-15,22.9,0.6,52378.5\nR003,2003-07-16,21.2,6.4,25555.41\nR003,2003-07-17,20.3,0.5,47059.02\nR003,2003-07-18,18.8,4.8,48741.81\nR003,2003-07-19,19.1,5.6,45567.26\nR003,2003-07-20,20.3,2.1,48827.46\nR003,2003-07-21,21.6,2.1,49575.99\nR003,2003-07-22,21.9,4.5,49970.81\nR003,2003-07-23,20.8,9.4,34304.71\nR003,2003-07-24,19.5,18.6,42655.61\nR003,2003-07-25,19.0,6.8,37636.36\nR003,2003-07-26,19.1,13.4,38743.08\nR003,2003-07-27,20.7,7.4,47749.94\nR003,2003-07-28,20.7,17.2,30117.07\nR003,2003-07-29,19.4,48.5,16726.61\nR003,2003-07-30,19.6,18.7,29021.36\nR003,2003-07-31,20.2,9.4,45076.52\nR003,2003-08-01,21.2,5.0,48031.42\nR003,2003-08-02,22.9,0.0,49811.54\nR003,2003-08-03,22.9,0.2,48863.67\nR003,2003-08-04,22.8,1.9,46833.02\nR003,2003-08-05,21.6,6.5,43055.02\nR003,2003-08-06,20.3,0.7,46800.0\nR003,2003-08-07,21.4,0.4,47261.72\nR003,2003-08-08,21.6,6.6,43522.15\nR003,2003-08-09,21.7,14.0,27637.11\nR003,2003-08-10,22.2,3.2,43335.03\nR003,2003-08-11,22.4,2.7,33509.47\nR003,2003-08-12,19.4,7.6,46800.0\nR003,2003-08-13,17.2,0.0,46012.15\nR003,2003-08-14,18.3,0.0,45928.64\nR003,2003-08-15,20.0,0.0,46825.91\nR003,2003-08-16,21.4,0.1,46616.7\nR003,2003-08-17,22.5,0.9,46479.35\nR003,2003-08-18,21.9,5.0,43070.75\nR003,2003-08-19,22.3,0.1,43660.17\nR003,2003-08-20,22.0,0.2,46233.07\nR003,2003-08-21,21.8,1.7,45794.63\nR003,2003-08-22,21.1,3.5,46800.0\nR003,2003-08-23,19.6,5.0,44873.33\nR003,2003-08-24,19.9,1.5,43591.7\nR003,2003-08-25,20.7,0.0,45769.22\nR003,2003-08-26,21.5,0.4,46800.0\nR003,2003-08-27,21.5,0.0,45895.96\nR003,2003-08-28,22.4,0.0,46673.02\nR003,2003-08-29,22.4,1.5,46359.26\nR003,2003-08-30,22.4,0.3,44790.48\nR003,2003-08-31,23.2,0.0,45803.56\nR003,2003-09-01,24.6,0.2,46265.71\nR003,2003-09-02,24.4,0.2,45254.05\nR003,2003-09-03,22.5,3.2,39981.75\nR003,2003-09-04,20.6,23.5,28872.85\nR003,2003-09-05,17.3,74.9,1326.87\nR003,2003-09-06,14.3,65.9,0.0\nR003,2003-09-07,12.8,0.2,43265.69\nR003,2003-09-08,12.7,0.0,41194.07\nR003,2003-09-09,14.6,0.0,41688.38\nR003,2003-09-10,16.0,0.0,43427.05\nR003,2003-09-11,16.6,0.0,43200.0\nR003,2003-09-12,17.3,9.9,16648.62\nR003,2003-09-13,17.5,8.5,12421.65\nR003,2003-09-14,17.8,0.4,41972.17\nR003,2003-09-15,20.0,0.0,43071.42\nR003,2003-09-16,18.0,5.0,35898.06\nR003,2003-09-17,16.7,11.1,2947.24\nR003,2003-09-18,14.4,6.4,36715.38\nR003,2003-09-19,13.4,9.7,32830.31\nR003,2003-09-20,14.3,0.3,41215.98\nR003,2003-09-21,14.8,0.0,40931.87\nR003,2003-09-22,15.7,0.0,41284.71\nR003,2003-09-23,16.9,0.0,41319.56\nR003,2003-09-24,18.0,0.3,41991.81\nR003,2003-09-25,17.8,1.1,40590.22\nR003,2003-09-26,16.8,0.0,41426.15\nR003,2003-09-27,17.5,0.0,39147.96\nR003,2003-09-28,16.7,7.3,31103.97\nR003,2003-09-29,15.4,4.9,34071.42\nR003,2003-09-30,14.8,0.1,38416.2\nR003,2003-10-01,15.4,4.7,38570.17\nR003,2003-10-02,15.6,0.5,38343.4\nR003,2003-10-03,14.7,3.5,36033.82\nR003,2003-10-04,15.2,0.0,38656.65\nR003,2003-10-05,16.8,0.0,38630.25\nR003,2003-10-06,20.4,0.0,38450.1\nR003,2003-10-07,20.1,0.0,37850.15\nR003,2003-10-08,19.7,0.0,36000.0\nR003,2003-10-09,22.1,0.7,37325.34\nR003,2003-10-10,16.7,18.9,24992.54\nR003,2003-10-11,15.5,51.8,7200.0\nR003,2003-10-12,13.0,6.8,24064.3\nR003,2003-10-13,13.7,0.0,33721.92\nR003,2003-10-14,14.5,2.0,31389.84\nR003,2003-10-15,15.1,0.0,36021.03\nR003,2003-10-16,14.5,5.9,2806.37\nR003,2003-10-17,16.0,6.7,23669.79\nR003,2003-10-18,14.4,21.7,7855.48\nR003,2003-10-19,9.8,6.3,29765.82\nR003,2003-10-20,9.7,5.9,18611.02\nR003,2003-10-21,14.8,0.4,32312.85\nR003,2003-10-22,15.8,8.6,20780.92\nR003,2003-10-23,19.3,0.1,33947.22\nR003,2003-10-24,16.6,22.2,3600.0\nR003,2003-10-25,14.5,46.7,0.0\nR003,2003-10-26,10.0,0.5,34597.19\nR003,2003-10-27,6.7,2.5,29128.25\nR003,2003-10-28,5.9,17.0,837.29\nR003,2003-10-29,5.2,0.8,27859.24\nR003,2003-10-30,6.5,15.1,0.0\nR003,2003-10-31,8.0,0.5,26236.49\nR003,2003-11-01,10.4,10.7,1185.92\nR003,2003-11-02,9.3,0.0,35978.51\nR003,2003-11-03,10.7,0.0,35898.91\nR003,2003-11-04,14.1,0.0,32153.67\nR003,2003-11-05,15.0,0.0,35738.67\nR003,2003-11-06,13.0,0.0,29446.85\nR003,2003-11-07,13.6,0.0,32450.68\nR003,2003-11-08,10.7,1.2,33712.52\nR003,2003-11-09,7.6,14.0,4665.76\nR003,2003-11-10,6.2,0.4,13142.6\nR003,2003-11-11,9.6,4.2,30952.85\nR003,2003-11-12,7.4,25.9,3659.2\nR003,2003-11-13,3.9,0.0,32665.54\nR003,2003-11-14,1.2,0.0,32648.62\nR003,2003-11-15,1.7,0.0,33716.95\nR003,2003-11-16,1.5,0.0,34068.86\nR003,2003-11-17,5.0,11.4,6713.48\nR003,2003-11-18,6.2,6.3,11530.64\nR003,2003-11-19,3.2,9.7,28569.83\nR003,2003-11-20,3.2,37.5,0.0\nR003,2003-11-21,3.6,8.8,22805.55\nR003,2003-11-22,2.4,0.0,32215.98\nR003,2003-11-23,5.7,0.0,32215.93\nR003,2003-11-24,5.9,0.0,31827.13\nR003,2003-11-25,5.2,0.0,31851.3\nR003,2003-11-26,7.7,0.0,21600.0\nR003,2003-11-27,8.1,0.0,14541.95\nR003,2003-11-28,7.0,0.0,29879.54\nR003,2003-11-29,6.6,0.0,30658.55\nR003,2003-11-30,4.8,0.0,29959.47\nR003,2003-12-01,3.7,0.1,30316.01\nR003,2003-12-02,4.0,0.0,30333.35\nR003,2003-12-03,3.5,0.0,30349.29\nR003,2003-12-04,3.0,0.0,30363.78\nR003,2003-12-05,2.3,0.4,25228.27\nR003,2003-12-06,5.4,3.6,20047.8\nR003,2003-12-07,5.8,0.1,28860.92\nR003,2003-12-08,4.9,18.6,0.0\nR003,2003-12-09,1.0,3.5,20721.94\nR003,2003-12-10,-2.5,0.0,30804.98\nR003,2003-12-11,-3.8,0.0,30422.0\nR003,2003-12-12,-0.2,0.0,29261.09\nR003,2003-12-13,2.4,0.0,28169.72\nR003,2003-12-14,4.0,3.5,12509.92\nR003,2003-12-15,6.1,0.0,29251.91\nR003,2003-12-16,7.7,2.2,20295.57\nR003,2003-12-17,8.1,7.2,18000.0\nR003,2003-12-18,5.8,26.1,0.0\nR003,2003-12-19,1.0,0.3,18272.26\nR003,2003-12-20,0.6,2.7,1208.38\nR003,2003-12-21,2.0,0.0,25493.32\nR003,2003-12-22,1.6,0.0,30745.76\nR003,2003-12-23,0.1,0.0,25858.22\nR003,2003-12-24,-0.2,0.0,31233.12\nR003,2003-12-25,1.0,0.0,31268.75\nR003,2003-12-26,4.5,0.0,19106.86\nR003,2003-12-27,4.8,1.4,5751.77\nR003,2003-12-28,3.3,0.0,30245.82\nR003,2003-12-29,1.3,0.0,31429.13\nR003,2003-12-30,0.3,0.0,31474.58\nR003,2003-12-31,-0.0,0.0,30563.99\nR003,2004-01-01,1.1,0.0,29755.52\nR003,2004-01-02,1.7,0.0,28946.84\nR003,2004-01-03,2.8,0.0,31681.96\nR003,2004-01-04,4.4,1.6,18437.66\nR003,2004-01-05,6.1,7.0,0.0\nR003,2004-01-06,4.4,8.1,27163.83\nR003,2004-01-07,2.0,4.1,4349.72\nR003,2004-01-08,-0.1,11.8,12673.33\nR003,2004-01-09,-3.6,1.1,29491.63\nR003,2004-01-10,-3.8,0.0,32121.96\nR003,2004-01-11,-5.4,0.0,32190.57\nR003,2004-01-12,-0.8,0.0,28800.0\nR003,2004-01-13,3.7,0.0,32330.72\nR003,2004-01-14,2.6,0.0,32400.0\nR003,2004-01-15,3.6,5.2,6151.0\nR003,2004-01-16,7.9,13.2,4100.34\nR003,2004-01-17,3.6,23.0,5280.36\nR003,2004-01-18,0.3,0.0,32400.0\nR003,2004-01-19,1.2,0.0,32400.0\nR003,2004-01-20,3.9,0.0,27919.95\nR003,2004-01-21,5.3,0.0,17027.6\nR003,2004-01-22,6.7,0.0,18938.05\nR003,2004-01-23,7.5,6.6,29453.75\nR003,2004-01-24,4.5,14.2,21600.0\nR003,2004-01-25,2.6,5.4,31862.26\nR003,2004-01-26,3.8,0.0,32400.0\nR003,2004-01-27,3.4,0.0,24501.43\nR003,2004-01-28,3.2,0.0,32400.0\nR003,2004-01-29,6.4,0.0,26094.71\nR003,2004-01-30,6.6,0.0,32400.0\nR003,2004-01-31,5.7,2.0,18378.37\nR003,2004-02-01,4.1,13.3,1122.17\nR003,2004-02-02,2.6,0.9,10226.03\nR003,2004-02-03,1.1,1.9,15360.21\nR003,2004-02-04,0.5,12.1,17938.0\nR003,2004-02-05,-0.2,0.6,26939.87\nR003,2004-02-06,2.0,15.2,1563.89\nR003,2004-02-07,4.2,14.1,11121.27\nR003,2004-02-08,7.6,5.6,28875.83\nR003,2004-02-09,5.9,8.3,24414.3\nR003,2004-02-10,2.5,31.6,0.0\nR003,2004-02-11,-2.6,4.2,34760.7\nR003,2004-02-12,-2.8,0.0,29048.15\nR003,2004-02-13,2.8,1.9,19596.0\nR003,2004-02-14,2.1,16.9,3800.88\nR003,2004-02-15,-1.7,6.6,19210.41\nR003,2004-02-16,-4.2,2.3,33655.85\nR003,2004-02-17,-1.8,7.6,26016.52\nR003,2004-02-18,-0.8,3.1,35351.41\nR003,2004-02-19,-1.1,0.0,36389.46\nR003,2004-02-20,-0.3,12.4,955.51\nR003,2004-02-21,-1.8,16.7,2175.94\nR003,2004-02-22,-1.7,0.5,2900.3\nR003,2004-02-23,0.6,0.0,34804.37\nR003,2004-02-24,2.5,0.1,38715.35\nR003,2004-02-25,5.8,7.1,17080.41\nR003,2004-02-26,6.6,4.3,32270.75\nR003,2004-02-27,8.8,0.0,32348.99\nR003,2004-02-28,11.2,2.2,31589.06\nR003,2004-02-29,11.4,0.3,32312.43\nR003,2004-03-01,8.9,10.0,5057.54\nR003,2004-03-02,7.4,0.2,39600.0\nR003,2004-03-03,7.4,2.8,18889.5\nR003,2004-03-04,7.9,13.2,0.0\nR003,2004-03-05,5.2,59.5,0.0\nR003,2004-03-06,-0.3,9.4,27937.01\nR003,2004-03-07,-4.2,0.0,38813.61\nR003,2004-03-08,-1.3,0.0,39600.0\nR003,2004-03-09,3.9,0.0,39600.0\nR003,2004-03-10,5.8,0.0,31621.59\nR003,2004-03-11,5.9,6.9,870.91\nR003,2004-03-12,4.5,4.3,29333.06\nR003,2004-03-13,3.6,4.6,23965.03\nR003,2004-03-14,2.1,1.2,37878.95\nR003,2004-03-15,2.2,0.0,39600.0\nR003,2004-03-16,1.5,4.1,34366.15\nR003,2004-03-17,2.0,2.0,39600.0\nR003,2004-03-18,2.7,0.0,39600.0\nR003,2004-03-19,3.0,0.5,34204.36\nR003,2004-03-20,5.6,0.3,21742.2\nR003,2004-03-21,8.4,0.6,40136.67\nR003,2004-03-22,11.1,0.0,39764.69\nR003,2004-03-23,12.5,0.0,42594.08\nR003,2004-03-24,12.8,0.0,40534.09\nR003,2004-03-25,13.4,0.0,43058.52\nR003,2004-03-26,8.7,6.8,40500.67\nR003,2004-03-27,9.9,0.7,2538.81\nR003,2004-03-28,12.4,0.8,22545.2\nR003,2004-03-29,10.1,13.5,30546.99\nR003,2004-03-30,4.4,17.9,35603.93\nR003,2004-03-31,3.8,5.1,27173.19\nR003,2004-04-01,4.6,0.0,42080.0\nR003,2004-04-02,2.7,0.8,45246.31\nR003,2004-04-03,-4.7,5.4,34288.93\nR003,2004-04-04,-5.3,0.3,43450.6\nR003,2004-04-05,-1.3,0.0,44142.91\nR003,2004-04-06,4.2,0.0,44343.08\nR003,2004-04-07,9.2,0.3,44214.75\nR003,2004-04-08,11.5,0.0,46179.01\nR003,2004-04-09,14.0,0.0,46250.27\nR003,2004-04-10,14.2,0.0,44212.54\nR003,2004-04-11,16.1,0.0,39295.43\nR003,2004-04-12,18.1,0.0,46464.5\nR003,2004-04-13,15.9,0.5,19064.77\nR003,2004-04-14,15.3,4.1,5411.0\nR003,2004-04-15,8.8,30.9,10638.07\nR003,2004-04-16,8.4,0.0,46500.13\nR003,2004-04-17,6.3,11.7,2375.7\nR003,2004-04-18,7.8,0.2,45889.15\nR003,2004-04-19,9.5,0.0,46800.0\nR003,2004-04-20,11.3,0.0,46800.0\nR003,2004-04-21,12.9,1.1,44755.67\nR003,2004-04-22,12.2,12.7,23079.08\nR003,2004-04-23,11.1,13.1,12825.73\nR003,2004-04-24,9.6,8.1,41520.43\nR003,2004-04-25,9.2,3.7,32898.82\nR003,2004-04-26,11.1,0.6,44985.19\nR003,2004-04-27,15.0,0.0,47981.8\nR003,2004-04-28,15.7,4.5,44566.57\nR003,2004-04-29,13.7,6.9,45175.21\nR003,2004-04-30,11.6,20.4,0.0\nR003,2004-05-01,11.8,5.3,20113.1\nR003,2004-05-02,13.6,0.0,46800.0\nR003,2004-05-03,15.3,0.0,47112.6\nR003,2004-05-04,15.2,0.2,47388.07\nR003,2004-05-05,15.2,0.2,48366.16\nR003,2004-05-06,16.7,0.0,37706.55\nR003,2004-05-07,17.4,2.0,43610.32\nR003,2004-05-08,17.8,0.8,41313.5\nR003,2004-05-09,17.1,0.8,47243.89\nR003,2004-05-10,16.9,2.2,48554.16\nR003,2004-05-11,14.3,11.7,24448.79\nR003,2004-05-12,11.7,46.4,7279.67\nR003,2004-05-13,12.8,17.5,41938.73\nR003,2004-05-14,13.6,0.2,47985.36\nR003,2004-05-15,14.1,3.5,40725.54\nR003,2004-05-16,12.8,12.8,12991.15\nR003,2004-05-17,10.8,11.7,3208.08\nR003,2004-05-18,12.7,0.0,43625.6\nR003,2004-05-19,8.8,9.0,26944.0\nR003,2004-05-20,8.9,0.7,30110.99\nR003,2004-05-21,13.1,0.0,46036.57\nR003,2004-05-22,15.4,0.1,49128.88\nR003,2004-05-23,17.9,0.7,50605.03\nR003,2004-05-24,18.5,0.5,50936.26\nR003,2004-05-25,18.5,2.0,50273.42\nR003,2004-05-26,14.2,7.5,0.0\nR003,2004-05-27,15.5,7.9,8025.22\nR003,2004-05-28,15.5,11.9,2882.76\nR003,2004-05-29,16.5,4.7,34411.2\nR003,2004-05-30,16.3,9.4,16445.53\nR003,2004-05-31,19.0,2.8,48533.99\nR003,2004-06-01,16.3,19.4,33529.5\nR003,2004-06-02,14.9,10.5,26199.83\nR003,2004-06-03,15.8,8.3,32166.9\nR003,2004-06-04,15.6,1.0,44710.64\nR003,2004-06-05,18.0,0.0,51728.46\nR003,2004-06-06,20.9,0.3,43425.9\nR003,2004-06-07,19.3,11.4,40491.97\nR003,2004-06-08,17.9,9.5,45661.56\nR003,2004-06-09,15.8,11.3,9113.65\nR003,2004-06-10,13.9,9.4,34705.64\nR003,2004-06-11,11.5,0.7,50686.95\nR003,2004-06-12,14.6,0.0,53130.1\nR003,2004-06-13,18.7,0.0,53118.15\nR003,2004-06-14,20.7,0.0,53473.98\nR003,2004-06-15,19.7,0.8,52360.77\nR003,2004-06-16,20.6,0.5,51221.24\nR003,2004-06-17,19.7,4.0,46745.19\nR003,2004-06-18,20.1,0.1,50678.36\nR003,2004-06-19,21.7,0.6,53425.69\nR003,2004-06-20,21.4,11.3,45788.89\nR003,2004-06-21,21.2,0.9,49315.17\nR003,2004-06-22,21.0,6.8,51258.54\nR003,2004-06-23,19.9,9.2,38490.03\nR003,2004-06-24,20.3,17.2,40296.87\nR003,2004-06-25,20.3,12.3,39823.76\nR003,2004-06-26,20.8,8.7,45612.08\nR003,2004-06-27,20.5,5.2,47640.53\nR003,2004-06-28,19.3,6.2,42351.46\nR003,2004-06-29,17.1,2.5,52755.14\nR003,2004-06-30,18.0,2.2,52283.94\nR003,2004-07-01,16.3,5.9,46120.33\nR003,2004-07-02,16.0,0.0,51583.69\nR003,2004-07-03,18.2,0.0,53476.23\nR003,2004-07-04,20.2,0.8,50835.83\nR003,2004-07-05,21.3,2.3,42012.98\nR003,2004-07-06,22.1,0.0,52716.79\nR003,2004-07-07,23.1,0.2,53034.1\nR003,2004-07-08,23.0,0.4,49869.06\nR003,2004-07-09,21.2,2.5,51590.42\nR003,2004-07-10,20.5,0.0,51317.28\nR003,2004-07-11,19.8,1.0,52701.15\nR003,2004-07-12,20.8,1.9,50710.28\nR003,2004-07-13,21.7,4.5,53662.15\nR003,2004-07-14,22.7,0.8,52732.83\nR003,2004-07-15,20.3,7.0,39792.92\nR003,2004-07-16,20.5,19.4,20092.16\nR003,2004-07-17,19.3,19.3,29511.95\nR003,2004-07-18,20.0,18.4,27984.73\nR003,2004-07-19,20.4,2.3,44893.8\nR003,2004-07-20,20.9,0.0,49968.8\nR003,2004-07-21,21.1,0.3,51725.78\nR003,2004-07-22,19.5,3.6,47899.67\nR003,2004-07-23,17.6,15.4,40994.51\nR003,2004-07-24,17.5,2.9,45245.74\nR003,2004-07-25,19.3,0.0,50137.98\nR003,2004-07-26,20.4,0.0,49137.26\nR003,2004-07-27,22.0,0.4,50752.5\nR003,2004-07-28,23.2,0.0,51065.9\nR003,2004-07-29,24.1,0.0,50335.95\nR003,2004-07-30,24.5,1.1,50383.08\nR003,2004-07-31,24.8,0.3,49399.21\nR003,2004-08-01,25.3,0.2,48765.25\nR003,2004-08-02,25.3,0.9,49166.47\nR003,2004-08-03,21.9,15.0,37771.28\nR003,2004-08-04,20.4,6.3,35760.92\nR003,2004-08-05,20.9,1.6,46883.79\nR003,2004-08-06,22.5,0.8,48614.56\nR003,2004-08-07,22.8,2.3,47960.71\nR003,2004-08-08,22.0,3.7,46971.43\nR003,2004-08-09,22.0,3.5,47339.1\nR003,2004-08-10,23.1,0.0,46800.0\nR003,2004-08-11,20.7,16.9,20034.92\nR003,2004-08-12,18.7,5.9,19459.52\nR003,2004-08-13,19.9,4.8,40807.81\nR003,2004-08-14,21.8,0.0,46800.0\nR003,2004-08-15,23.1,0.0,47182.76\nR003,2004-08-16,24.0,1.3,46880.8\nR003,2004-08-17,25.6,0.3,46912.3\nR003,2004-08-18,24.0,9.5,41609.56\nR003,2004-08-19,21.3,29.1,21991.01\nR003,2004-08-20,21.2,9.1,37850.12\nR003,2004-08-21,21.1,3.6,36837.97\nR003,2004-08-22,21.3,15.3,40325.62\nR003,2004-08-23,22.1,1.1,46419.96\nR003,2004-08-24,21.2,3.3,45487.95\nR003,2004-08-25,20.2,1.2,40199.99\nR003,2004-08-26,20.7,0.0,44997.03\nR003,2004-08-27,21.7,0.2,46705.57\nR003,2004-08-28,23.8,0.0,46603.44\nR003,2004-08-29,23.0,6.6,45099.53\nR003,2004-08-30,21.2,9.2,41580.25\nR003,2004-08-31,21.3,3.2,43907.6\nR003,2004-09-01,21.2,2.6,43592.38\nR003,2004-09-02,21.5,3.2,41975.09\nR003,2004-09-03,19.9,1.9,36175.41\nR003,2004-09-04,19.3,0.4,42869.96\nR003,2004-09-05,17.2,3.0,42163.65\nR003,2004-09-06,15.0,3.0,35899.92\nR003,2004-09-07,14.1,3.0,41487.72\nR003,2004-09-08,15.0,0.0,42089.32\nR003,2004-09-09,16.9,0.9,40288.38\nR003,2004-09-10,12.3,16.1,23738.42\nR003,2004-09-11,11.8,0.8,40202.23\nR003,2004-09-12,13.3,0.0,42838.87\nR003,2004-09-13,14.1,0.0,43154.64\nR003,2004-09-14,16.1,0.0,43087.46\nR003,2004-09-15,17.1,0.0,43020.19\nR003,2004-09-16,18.4,0.0,42952.8\nR003,2004-09-17,17.7,0.5,38495.64\nR003,2004-09-18,16.6,0.4,39900.86\nR003,2004-09-19,18.3,0.0,40889.58\nR003,2004-09-20,19.9,0.0,42681.81\nR003,2004-09-21,19.2,0.0,42231.59\nR003,2004-09-22,19.6,0.0,42545.41\nR003,2004-09-23,19.3,2.9,41665.01\nR003,2004-09-24,20.0,0.1,39600.0\nR003,2004-09-25,22.9,0.0,38371.71\nR003,2004-09-26,23.4,0.0,40497.11\nR003,2004-09-27,21.6,0.0,39600.0\nR003,2004-09-28,21.1,0.0,41207.71\nR003,2004-09-29,20.2,0.0,39568.66\nR003,2004-09-30,21.4,0.0,40454.97\nR003,2004-10-01,18.9,1.4,39286.78\nR003,2004-10-02,16.1,0.0,38016.09\nR003,2004-10-03,13.0,1.2,34472.93\nR003,2004-10-04,11.7,0.0,35615.26\nR003,2004-10-05,12.0,0.0,36818.06\nR003,2004-10-06,12.5,0.0,39078.57\nR003,2004-10-07,15.8,0.0,38562.65\nR003,2004-10-08,18.1,0.0,37687.54\nR003,2004-10-09,15.9,10.0,18981.68\nR003,2004-10-10,14.8,2.9,30840.27\nR003,2004-10-11,15.7,0.0,36176.16\nR003,2004-10-12,13.5,22.2,844.48\nR003,2004-10-13,6.6,0.0,34709.62\nR003,2004-10-14,8.6,0.0,36338.06\nR003,2004-10-15,10.8,1.1,32036.47\nR003,2004-10-16,13.7,0.6,19784.65\nR003,2004-10-17,16.6,0.0,34186.62\nR003,2004-10-18,18.1,0.0,36000.0\nR003,2004-10-19,17.4,0.7,31903.68\nR003,2004-10-20,15.1,6.7,8378.47\nR003,2004-10-21,10.5,24.8,32174.23\nR003,2004-10-22,7.2,1.4,36000.0\nR003,2004-10-23,6.9,0.0,36000.0\nR003,2004-10-24,10.4,0.0,34146.07\nR003,2004-10-25,12.9,0.0,36000.0\nR003,2004-10-26,13.7,0.0,36000.0\nR003,2004-10-27,16.6,0.0,36000.0\nR003,2004-10-28,17.2,0.9,35231.3\nR003,2004-10-29,15.3,1.6,36000.0\nR003,2004-10-30,14.6,0.2,33287.45\nR003,2004-10-31,15.7,0.0,36000.0\nR003,2004-11-01,13.9,0.1,30937.67\nR003,2004-11-02,14.7,3.4,31474.01\nR003,2004-11-03,10.9,15.6,8167.77\nR003,2004-11-04,8.2,11.3,30329.43\nR003,2004-11-05,5.9,0.0,35634.23\nR003,2004-11-06,8.2,0.0,35478.27\nR003,2004-11-07,9.6,0.0,35324.3\nR003,2004-11-08,11.4,0.0,35172.51\nR003,2004-11-09,11.5,0.0,32464.88\nR003,2004-11-10,12.8,0.0,33421.33\nR003,2004-11-11,12.1,0.0,34732.18\nR003,2004-11-12,12.1,0.0,34591.09\nR003,2004-11-13,11.7,0.0,31381.0\nR003,2004-11-14,11.6,0.0,34082.0\nR003,2004-11-15,13.6,0.0,30505.91\nR003,2004-11-16,11.8,14.7,0.0\nR003,2004-11-17,8.5,17.1,666.37\nR003,2004-11-18,4.1,0.0,32235.32\nR003,2004-11-19,6.7,0.0,32594.73\nR003,2004-11-20,9.3,0.5,31216.58\nR003,2004-11-21,4.4,56.9,0.0\nR003,2004-11-22,-0.1,26.1,0.0\nR003,2004-11-23,-1.2,4.3,21231.31\nR003,2004-11-24,-0.4,4.3,19733.32\nR003,2004-11-25,3.6,15.9,0.0\nR003,2004-11-26,3.0,17.2,0.0\nR003,2004-11-27,0.1,2.8,21982.34\nR003,2004-11-28,3.5,35.2,0.0\nR003,2004-11-29,4.1,17.6,3792.67\nR003,2004-11-30,4.7,21.1,506.92\nR003,2004-12-01,2.5,13.0,28083.21\nR003,2004-12-02,1.6,0.0,30347.4\nR003,2004-12-03,4.7,15.1,3618.27\nR003,2004-12-04,7.6,25.8,10791.14\nR003,2004-12-05,3.2,1.4,28375.84\nR003,2004-12-06,1.9,0.0,22811.08\nR003,2004-12-07,2.8,2.3,27656.51\nR003,2004-12-08,1.5,0.2,27735.73\nR003,2004-12-09,0.8,12.6,1367.71\nR003,2004-12-10,0.6,6.6,25439.11\nR003,2004-12-11,-0.6,9.9,17389.47\nR003,2004-12-12,-3.7,1.5,27803.22\nR003,2004-12-13,0.2,15.7,1212.5\nR003,2004-12-14,-3.8,1.8,27745.56\nR003,2004-12-15,-7.4,0.0,30413.47\nR003,2004-12-16,-7.5,0.6,29238.54\nR003,2004-12-17,-3.9,0.1,30011.1\nR003,2004-12-18,-3.2,0.0,30783.64\nR003,2004-12-19,-2.1,0.0,30774.29\nR003,2004-12-20,-1.4,0.0,30763.48\nR003,2004-12-21,2.8,4.0,0.0\nR003,2004-12-22,4.2,0.9,12956.1\nR003,2004-12-23,4.0,6.2,17.06\nR003,2004-12-24,3.3,4.2,23738.73\nR003,2004-12-25,3.8,1.4,26802.14\nR003,2004-12-26,1.9,0.0,31329.46\nR003,2004-12-27,5.1,0.0,31035.37\nR003,2004-12-28,6.7,0.0,30620.26\nR003,2004-12-29,8.2,0.0,22522.46\nR003,2004-12-30,10.2,0.0,29005.31\nR003,2004-12-31,7.2,2.8,1409.55\nR003,2005-01-01,6.2,0.3,30900.18\nR003,2005-01-02,5.5,1.9,2761.21\nR003,2005-01-03,3.3,0.7,29185.21\nR003,2005-01-04,2.3,1.4,4531.69\nR003,2005-01-05,1.1,8.6,16622.51\nR003,2005-01-06,-0.3,2.3,23550.69\nR003,2005-01-07,-0.7,0.0,31979.14\nR003,2005-01-08,-0.4,0.1,30658.35\nR003,2005-01-09,3.5,0.0,28508.49\nR003,2005-01-10,1.7,4.2,15589.92\nR003,2005-01-11,-0.4,4.8,10763.38\nR003,2005-01-12,4.1,4.7,26413.01\nR003,2005-01-13,2.6,0.0,32377.44\nR003,2005-01-14,3.0,0.0,32400.0\nR003,2005-01-15,3.8,0.0,28800.0\nR003,2005-01-16,2.4,0.0,32400.0\nR003,2005-01-17,2.2,0.0,32400.0\nR003,2005-01-18,1.7,0.0,32400.0\nR003,2005-01-19,0.8,0.0,32400.0\nR003,2005-01-20,0.2,0.0,32400.0\nR003,2005-01-21,1.9,0.0,28800.0\nR003,2005-01-22,6.0,0.0,32400.0\nR003,2005-01-23,4.7,2.4,24795.66\nR003,2005-01-24,1.4,13.7,0.0\nR003,2005-01-25,-1.3,0.0,30714.78\nR003,2005-01-26,3.9,3.4,6725.67\nR003,2005-01-27,5.3,0.0,32400.0\nR003,2005-01-28,7.8,0.3,26635.84\nR003,2005-01-29,6.7,4.0,5490.85\nR003,2005-01-30,8.7,6.5,5074.85\nR003,2005-01-31,3.8,20.8,0.0\nR003,2005-02-01,1.0,0.0,28643.65\nR003,2005-02-02,0.4,0.0,30516.95\nR003,2005-02-03,0.3,0.0,32623.54\nR003,2005-02-04,-0.7,0.0,33644.58\nR003,2005-02-05,-2.0,0.0,33229.68\nR003,2005-02-06,-2.9,0.0,33529.68\nR003,2005-02-07,-5.8,2.1,33730.08\nR003,2005-02-08,-5.0,4.0,29651.19\nR003,2005-02-09,-5.4,0.4,34579.65\nR003,2005-02-10,-3.4,3.0,27407.61\nR003,2005-02-11,-4.0,1.3,34050.87\nR003,2005-02-12,-4.6,0.0,34297.34\nR003,2005-02-13,-3.7,0.0,36222.72\nR003,2005-02-14,0.7,0.0,30909.13\nR003,2005-02-15,6.1,0.0,36964.64\nR003,2005-02-16,8.3,0.0,35601.12\nR003,2005-02-17,10.7,0.0,35681.1\nR003,2005-02-18,10.2,7.6,21600.0\nR003,2005-02-19,6.3,24.6,0.0\nR003,2005-02-20,4.6,0.3,33247.1\nR003,2005-02-21,2.8,10.8,23547.39\nR003,2005-02-22,2.5,0.0,38487.19\nR003,2005-02-23,3.1,0.0,38661.09\nR003,2005-02-24,4.1,0.0,36763.46\nR003,2005-02-25,5.0,3.2,29349.58\nR003,2005-02-26,7.2,3.3,29981.3\nR003,2005-02-27,7.7,7.8,6349.27\nR003,2005-02-28,5.0,23.5,5919.81\nR003,2005-03-01,2.0,17.3,0.0\nR003,2005-03-02,4.6,26.5,0.0\nR003,2005-03-03,8.2,25.3,0.0\nR003,2005-03-04,4.3,16.8,13618.55\nR003,2005-03-05,5.0,0.0,36000.0\nR003,2005-03-06,7.8,2.5,13296.22\nR003,2005-03-07,7.1,8.9,34714.31\nR003,2005-03-08,6.7,10.4,8660.73\nR003,2005-03-09,3.2,23.6,22794.01\nR003,2005-03-10,0.7,6.8,36570.8\nR003,2005-03-11,-0.0,4.7,31757.4\nR003,2005-03-12,-0.3,0.0,39391.25\nR003,2005-03-13,3.7,0.0,39628.2\nR003,2005-03-14,5.5,3.8,36758.08\nR003,2005-03-15,5.2,6.3,35114.01\nR003,2005-03-16,2.9,1.9,34366.05\nR003,2005-03-17,2.9,5.0,31232.03\nR003,2005-03-18,5.8,10.5,6757.26\nR003,2005-03-19,5.6,49.3,0.0\nR003,2005-03-20,-2.4,7.7,39579.12\nR003,2005-03-21,-3.9,2.2,39949.24\nR003,2005-03-22,-2.8,0.0,39999.56\nR003,2005-03-23,-2.4,1.6,36450.34\nR003,2005-03-24,-2.7,5.8,30688.67\nR003,2005-03-25,-2.8,0.0,41809.28\nR003,2005-03-26,0.8,0.0,34645.79\nR003,2005-03-27,5.9,0.0,41728.29\nR003,2005-03-28,7.8,7.6,29566.66\nR003,2005-03-29,7.0,37.7,0.0\nR003,2005-03-30,6.9,4.7,22862.11\nR003,2005-03-31,5.3,0.2,31968.67\nR003,2005-04-01,4.5,7.1,6215.57\nR003,2005-04-02,4.7,1.7,32929.0\nR003,2005-04-03,4.5,7.5,0.0\nR003,2005-04-04,5.2,3.0,38432.79\nR003,2005-04-05,5.5,0.0,43745.0\nR003,2005-04-06,7.0,0.0,41637.86\nR003,2005-04-07,7.4,0.0,44959.15\nR003,2005-04-08,9.8,0.0,46161.83\nR003,2005-04-09,11.2,0.0,46233.38\nR003,2005-04-10,12.6,0.0,46304.92\nR003,2005-04-11,11.9,0.0,46376.45\nR003,2005-04-12,12.9,0.0,46447.94\nR003,2005-04-13,15.0,0.0,46519.39\nR003,2005-04-14,15.0,0.0,38301.38\nR003,2005-04-15,12.0,10.2,9787.07\nR003,2005-04-16,11.0,5.1,38103.6\nR003,2005-04-17,11.0,10.5,29045.47\nR003,2005-04-18,14.5,1.5,46800.0\nR003,2005-04-19,12.6,28.2,19614.83\nR003,2005-04-20,10.9,4.4,37943.08\nR003,2005-04-21,14.3,0.0,46800.0\nR003,2005-04-22,17.2,0.7,34411.29\nR003,2005-04-23,15.9,7.9,32664.37\nR003,2005-04-24,11.5,35.5,0.0\nR003,2005-04-25,10.6,0.0,2352.53\nR003,2005-04-26,12.2,0.0,41113.58\nR003,2005-04-27,13.5,0.0,46800.0\nR003,2005-04-28,11.0,4.5,24311.64\nR003,2005-04-29,9.6,7.1,39108.8\nR003,2005-04-30,12.0,3.0,43552.54\nR003,2005-05-01,11.2,6.0,42324.64\nR003,2005-05-02,10.2,1.3,33625.67\nR003,2005-05-03,9.7,6.3,25016.09\nR003,2005-05-04,9.7,10.4,0.0\nR003,2005-05-05,10.8,7.8,33329.34\nR003,2005-05-06,13.3,0.0,49356.95\nR003,2005-05-07,18.0,0.3,41010.75\nR003,2005-05-08,18.1,4.3,34189.28\nR003,2005-05-09,13.8,14.1,5782.31\nR003,2005-05-10,14.8,0.0,48559.57\nR003,2005-05-11,16.7,0.2,49904.69\nR003,2005-05-12,18.1,0.9,50266.26\nR003,2005-05-13,17.2,19.7,36224.52\nR003,2005-05-14,13.2,27.4,319.79\nR003,2005-05-15,13.7,0.5,45457.61\nR003,2005-05-16,14.7,1.1,45713.94\nR003,2005-05-17,16.3,0.1,45798.74\nR003,2005-05-18,17.7,0.6,48071.09\nR003,2005-05-19,19.7,0.9,50089.65\nR003,2005-05-20,19.8,3.9,50400.0\nR003,2005-05-21,20.9,2.4,46800.0\nR003,2005-05-22,23.3,1.6,47754.1\nR003,2005-05-23,22.5,5.7,49376.48\nR003,2005-05-24,20.7,8.6,46238.3\nR003,2005-05-25,17.9,25.8,19629.76\nR003,2005-05-26,17.6,18.5,36187.74\nR003,2005-05-27,16.9,8.6,29032.23\nR003,2005-05-28,18.5,7.9,42506.75\nR003,2005-05-29,20.5,2.4,50821.2\nR003,2005-05-30,20.9,6.8,48227.75\nR003,2005-05-31,20.1,12.8,40263.72\nR003,2005-06-01,19.7,6.4,41065.79\nR003,2005-06-02,18.1,7.0,27498.01\nR003,2005-06-03,16.4,6.3,45672.76\nR003,2005-06-04,15.4,1.2,45964.06\nR003,2005-06-05,15.7,1.8,50144.39\nR003,2005-06-06,15.5,8.7,7721.09\nR003,2005-06-07,18.1,0.6,45997.55\nR003,2005-06-08,20.4,0.0,50962.19\nR003,2005-06-09,22.5,0.4,51254.28\nR003,2005-06-10,22.5,0.7,52790.74\nR003,2005-06-11,22.6,13.6,50379.38\nR003,2005-06-12,19.3,12.8,43883.39\nR003,2005-06-13,17.4,0.0,49118.08\nR003,2005-06-14,18.8,0.8,50555.89\nR003,2005-06-15,18.9,11.2,50228.48\nR003,2005-06-16,17.4,18.1,20166.05\nR003,2005-06-17,17.0,3.6,45968.56\nR003,2005-06-18,18.8,0.0,53067.89\nR003,2005-06-19,19.1,3.2,52324.34\nR003,2005-06-20,16.1,8.9,44973.83\nR003,2005-06-21,14.7,4.8,39702.74\nR003,2005-06-22,13.8,5.9,42656.64\nR003,2005-06-23,14.3,7.2,45835.64\nR003,2005-06-24,17.1,0.0,53398.43\nR003,2005-06-25,18.3,1.9,48996.94\nR003,2005-06-26,18.8,7.9,48154.91\nR003,2005-06-27,19.0,3.0,51338.83\nR003,2005-06-28,20.3,0.6,47614.34\nR003,2005-06-29,20.3,1.9,51892.52\nR003,2005-06-30,20.7,1.1,51311.01\nR003,2005-07-01,21.4,0.0,50594.72\nR003,2005-07-02,23.0,0.9,53746.65\nR003,2005-07-03,23.5,1.9,53921.41\nR003,2005-07-04,25.4,0.3,53384.07\nR003,2005-07-05,22.4,21.0,38734.02\nR003,2005-07-06,19.5,6.0,46670.57\nR003,2005-07-07,19.8,0.0,51942.24\nR003,2005-07-08,21.7,0.0,53761.68\nR003,2005-07-09,22.8,0.2,53043.31\nR003,2005-07-10,23.0,0.6,51420.55\nR003,2005-07-11,22.1,5.2,49702.75\nR003,2005-07-12,22.1,0.7,52051.15\nR003,2005-07-13,23.2,0.1,48964.97\nR003,2005-07-14,23.9,1.2,51757.97\nR003,2005-07-15,23.0,3.2,50807.4\nR003,2005-07-16,22.1,17.0,42314.96\nR003,2005-07-17,20.2,10.5,43274.39\nR003,2005-07-18,21.1,0.0,51717.48\nR003,2005-07-19,21.0,0.0,49849.8\nR003,2005-07-20,21.9,0.1,51326.53\nR003,2005-07-21,22.7,1.0,50659.55\nR003,2005-07-22,22.9,0.5,52346.7\nR003,2005-07-23,24.1,0.6,52307.59\nR003,2005-07-24,25.1,0.6,51917.11\nR003,2005-07-25,23.1,6.4,48026.35\nR003,2005-07-26,23.0,8.3,45214.93\nR003,2005-07-27,23.8,0.0,47432.1\nR003,2005-07-28,25.1,0.0,48295.62\nR003,2005-07-29,24.9,0.2,46895.67\nR003,2005-07-30,24.9,0.2,46704.54\nR003,2005-07-31,25.0,0.4,49037.61\nR003,2005-08-01,24.3,3.4,42462.47\nR003,2005-08-02,21.9,9.2,39976.61\nR003,2005-08-03,20.9,9.5,40780.01\nR003,2005-08-04,21.4,14.2,46753.35\nR003,2005-08-05,22.6,9.5,47557.81\nR003,2005-08-06,24.5,0.0,47002.0\nR003,2005-08-07,27.0,0.1,48768.48\nR003,2005-08-08,24.5,3.0,46375.31\nR003,2005-08-09,23.8,0.0,45111.38\nR003,2005-08-10,24.5,9.1,42294.99\nR003,2005-08-11,22.7,4.9,37036.6\nR003,2005-08-12,22.5,0.3,45214.6\nR003,2005-08-13,21.9,0.8,46800.0\nR003,2005-08-14,22.5,0.7,46811.35\nR003,2005-08-15,23.9,0.0,47176.22\nR003,2005-08-16,25.7,0.0,47206.83\nR003,2005-08-17,26.4,0.0,46905.34\nR003,2005-08-18,27.2,2.2,46936.63\nR003,2005-08-19,26.0,0.3,46800.0\nR003,2005-08-20,23.0,3.4,46396.67\nR003,2005-08-21,21.7,2.7,13955.22\nR003,2005-08-22,19.9,7.7,35047.91\nR003,2005-08-23,21.7,0.1,44731.54\nR003,2005-08-24,23.5,0.1,46800.0\nR003,2005-08-25,23.7,3.2,46800.0\nR003,2005-08-26,22.5,1.8,45317.37\nR003,2005-08-27,22.8,1.2,43928.95\nR003,2005-08-28,22.1,1.5,44995.03\nR003,2005-08-29,20.9,3.6,43875.34\nR003,2005-08-30,19.5,2.5,43913.83\nR003,2005-08-31,19.8,0.6,45830.21\nR003,2005-09-01,18.8,1.7,43627.26\nR003,2005-09-02,18.0,0.7,42378.5\nR003,2005-09-03,16.8,2.9,42977.71\nR003,2005-09-04,17.1,2.9,36393.57\nR003,2005-09-05,15.7,2.2,42110.27\nR003,2005-09-06,15.7,0.2,43200.0\nR003,2005-09-07,16.6,0.4,43295.51\nR003,2005-09-08,17.5,0.6,42044.08\nR003,2005-09-09,19.0,4.3,43200.0\nR003,2005-09-10,18.6,17.1,36770.92\nR003,2005-09-11,17.7,5.4,37476.52\nR003,2005-09-12,16.9,3.6,33017.74\nR003,2005-09-13,17.7,1.4,37577.02\nR003,2005-09-14,18.1,0.7,38749.41\nR003,2005-09-15,18.5,0.0,38078.22\nR003,2005-09-16,20.2,0.0,42969.08\nR003,2005-09-17,20.7,1.4,31887.35\nR003,2005-09-18,21.1,1.6,41484.15\nR003,2005-09-19,20.0,1.8,39515.08\nR003,2005-09-20,19.6,1.3,41552.13\nR003,2005-09-21,19.3,1.9,42223.26\nR003,2005-09-22,20.1,0.0,41939.78\nR003,2005-09-23,20.5,0.2,41974.98\nR003,2005-09-24,19.3,2.0,41373.35\nR003,2005-09-25,17.6,9.6,1642.01\nR003,2005-09-26,16.6,8.1,23538.93\nR003,2005-09-27,16.8,6.3,35414.94\nR003,2005-09-28,16.3,7.5,37302.21\nR003,2005-09-29,16.6,3.0,35346.51\nR003,2005-09-30,16.1,2.4,13139.8\nR003,2005-10-01,16.1,10.8,27786.54\nR003,2005-10-02,17.1,0.0,39698.57\nR003,2005-10-03,18.3,1.6,19411.79\nR003,2005-10-04,15.5,13.3,11040.52\nR003,2005-10-05,13.0,2.9,31397.47\nR003,2005-10-06,10.8,6.8,36293.82\nR003,2005-10-07,8.8,1.0,29433.9\nR003,2005-10-08,10.1,0.3,31224.9\nR003,2005-10-09,9.8,0.0,36381.66\nR003,2005-10-10,10.8,1.7,35034.75\nR003,2005-10-11,12.6,0.0,36166.83\nR003,2005-10-12,16.3,1.1,36546.6\nR003,2005-10-13,15.8,9.4,27721.98\nR003,2005-10-14,16.2,8.4,21725.54\nR003,2005-10-15,15.1,5.7,16438.36\nR003,2005-10-16,13.3,23.7,11626.81\nR003,2005-10-17,12.2,64.5,4784.69\nR003,2005-10-18,9.2,12.2,0.0\nR003,2005-10-19,8.2,19.1,0.0\nR003,2005-10-20,7.9,0.5,26730.26\nR003,2005-10-21,6.3,4.8,36000.0\nR003,2005-10-22,6.6,0.0,36000.0\nR003,2005-10-23,9.1,0.0,36000.0\nR003,2005-10-24,10.9,0.0,36000.0\nR003,2005-10-25,11.7,3.7,26223.64\nR003,2005-10-26,9.5,0.0,36000.0\nR003,2005-10-27,11.5,7.2,31931.64\nR003,2005-10-28,8.2,13.0,31610.68\nR003,2005-10-29,5.8,3.1,30774.17\nR003,2005-10-30,6.1,0.7,0.0\nR003,2005-10-31,9.9,3.6,32582.26\nR003,2005-11-01,7.6,16.1,15610.48\nR003,2005-11-02,8.8,9.0,26109.85\nR003,2005-11-03,7.1,19.0,3242.09\nR003,2005-11-04,8.9,0.0,33169.51\nR003,2005-11-05,9.9,0.0,31425.39\nR003,2005-11-06,9.3,1.8,22073.51\nR003,2005-11-07,10.2,4.3,15721.13\nR003,2005-11-08,9.3,0.2,33469.72\nR003,2005-11-09,8.6,0.0,35070.49\nR003,2005-11-10,8.0,0.2,32358.96\nR003,2005-11-11,6.6,2.8,28895.27\nR003,2005-11-12,5.8,0.0,34636.03\nR003,2005-11-13,5.5,0.0,34493.85\nR003,2005-11-14,6.2,0.0,34353.25\nR003,2005-11-15,6.4,0.0,34214.45\nR003,2005-11-16,5.7,7.2,17437.8\nR003,2005-11-17,8.1,0.0,33338.36\nR003,2005-11-18,10.2,0.0,29586.81\nR003,2005-11-19,10.5,0.0,15839.8\nR003,2005-11-20,10.6,0.0,24921.45\nR003,2005-11-21,8.9,10.3,4532.84\nR003,2005-11-22,5.6,24.9,10107.41\nR003,2005-11-23,0.5,0.0,31058.15\nR003,2005-11-24,1.6,0.0,29377.74\nR003,2005-11-25,6.9,0.0,26767.79\nR003,2005-11-26,10.6,7.1,0.0\nR003,2005-11-27,9.1,2.0,14528.77\nR003,2005-11-28,12.8,1.2,30647.3\nR003,2005-11-29,14.1,0.6,22505.58\nR003,2005-11-30,8.9,6.4,29929.86\nR003,2005-12-01,7.2,0.0,25200.0\nR003,2005-12-02,10.2,11.2,0.0\nR003,2005-12-03,10.6,21.8,0.0\nR003,2005-12-04,9.4,0.0,29986.7\nR003,2005-12-05,8.7,9.2,22838.19\nR003,2005-12-06,7.6,0.0,30390.73\nR003,2005-12-07,10.7,0.0,14791.65\nR003,2005-12-08,13.0,0.0,24387.69\nR003,2005-12-09,8.4,5.5,10800.0\nR003,2005-12-10,5.7,0.4,25737.99\nR003,2005-12-11,7.5,6.8,0.0\nR003,2005-12-12,4.8,0.2,22264.63\nR003,2005-12-13,4.2,0.0,29644.49\nR003,2005-12-14,5.8,0.0,29642.03\nR003,2005-12-15,7.4,0.0,28859.39\nR003,2005-12-16,6.3,11.8,0.0\nR003,2005-12-17,4.6,18.0,15820.81\nR003,2005-12-18,8.4,2.2,30399.23\nR003,2005-12-19,8.1,9.2,24435.02\nR003,2005-12-20,1.9,22.1,14806.02\nR003,2005-12-21,-3.3,0.0,29974.88\nR003,2005-12-22,0.3,0.4,28689.81\nR003,2005-12-23,1.2,7.0,0.0\nR003,2005-12-24,0.5,7.7,838.3\nR003,2005-12-25,-0.1,0.0,20643.15\nR003,2005-12-26,-0.9,0.0,30668.63\nR003,2005-12-27,1.6,0.0,31367.43\nR003,2005-12-28,2.8,0.0,18594.34\nR003,2005-12-29,5.3,0.0,31460.13\nR003,2005-12-30,6.2,0.0,31509.35\nR003,2005-12-31,5.8,1.1,23197.22\nR003,2006-01-01,2.1,7.3,31612.91\nR003,2006-01-02,0.9,0.0,29705.43\nR003,2006-01-03,4.5,0.0,30816.33\nR003,2006-01-04,5.3,0.0,31779.23\nR003,2006-01-05,5.9,0.0,21274.61\nR003,2006-01-06,6.1,10.4,6234.72\nR003,2006-01-07,5.1,6.8,401.43\nR003,2006-01-08,3.2,4.2,27067.06\nR003,2006-01-09,1.8,6.4,19924.02\nR003,2006-01-10,0.1,3.4,22175.39\nR003,2006-01-11,-3.2,0.0,32216.95\nR003,2006-01-12,-0.6,0.0,32286.84\nR003,2006-01-13,0.4,0.0,32358.31\nR003,2006-01-14,0.8,1.8,29629.01\nR003,2006-01-15,-0.1,0.2,32400.0\nR003,2006-01-16,-0.5,1.2,32400.0\nR003,2006-01-17,-2.4,4.5,25056.94\nR003,2006-01-18,-3.8,2.5,32344.48\nR003,2006-01-19,-0.9,1.0,26787.13\nR003,2006-01-20,-3.4,27.5,1977.46\nR003,2006-01-21,-10.3,1.7,32400.0\nR003,2006-01-22,-4.3,10.6,1081.84\nR003,2006-01-23,-1.6,7.9,20049.44\nR003,2006-01-24,-5.9,0.0,29278.9\nR003,2006-01-25,-4.9,0.0,25532.37\nR003,2006-01-26,-0.9,6.7,0.0\nR003,2006-01-27,-2.2,5.6,26500.97\nR003,2006-01-28,-3.1,4.6,30184.82\nR003,2006-01-29,-2.1,10.6,20467.74\nR003,2006-01-30,-2.3,12.5,4552.53\nR003,2006-01-31,-1.4,36.1,615.91\nR003,2006-02-01,-0.6,24.1,897.31\nR003,2006-02-02,1.5,3.4,25761.57\nR003,2006-02-03,3.1,0.0,28077.58\nR003,2006-02-04,0.3,0.0,32938.43\nR003,2006-02-05,-0.1,0.0,30631.53\nR003,2006-02-06,3.6,0.9,12250.6\nR003,2006-02-07,3.9,25.0,0.0\nR003,2006-02-08,3.9,3.6,8301.35\nR003,2006-02-09,3.0,8.3,0.0\nR003,2006-02-10,1.8,2.1,34504.67\nR003,2006-02-11,4.3,41.7,7059.31\nR003,2006-02-12,3.9,36.1,0.0\nR003,2006-02-13,1.4,33.6,0.0\nR003,2006-02-14,0.2,9.7,539.56\nR003,2006-02-15,0.1,6.9,0.0\nR003,2006-02-16,-2.5,4.0,29472.78\nR003,2006-02-17,-5.0,0.0,36010.53\nR003,2006-02-18,1.7,5.4,30556.18\nR003,2006-02-19,3.1,9.1,29211.92\nR003,2006-02-20,4.0,0.0,32687.04\nR003,2006-02-21,6.1,0.0,38266.2\nR003,2006-02-22,5.3,0.0,38442.02\nR003,2006-02-23,6.8,0.0,36693.01\nR003,2006-02-24,7.1,0.0,38796.7\nR003,2006-02-25,7.0,2.6,36947.04\nR003,2006-02-26,7.0,11.8,26357.15\nR003,2006-02-27,5.8,20.9,14516.48\nR003,2006-02-28,5.9,2.3,34289.3\nR003,2006-03-01,5.9,7.8,17662.71\nR003,2006-03-02,4.8,2.1,37706.2\nR003,2006-03-03,4.6,23.1,15229.96\nR003,2006-03-04,-0.5,8.3,35584.22\nR003,2006-03-05,4.5,0.0,39600.0\nR003,2006-03-06,9.6,0.0,39600.0\nR003,2006-03-07,11.8,0.0,37421.92\nR003,2006-03-08,10.3,14.8,15309.44\nR003,2006-03-09,5.8,14.3,5071.29\nR003,2006-03-10,0.9,2.8,28250.76\nR003,2006-03-11,4.0,0.0,39600.0\nR003,2006-03-12,8.4,0.0,39217.01\nR003,2006-03-13,9.5,0.0,34348.92\nR003,2006-03-14,10.9,0.0,39600.0\nR003,2006-03-15,7.7,3.9,10186.45\nR003,2006-03-16,6.5,3.3,17505.64\nR003,2006-03-17,5.8,1.2,34478.21\nR003,2006-03-18,5.3,2.4,38146.36\nR003,2006-03-19,4.9,8.8,19053.98\nR003,2006-03-20,2.7,11.6,39600.0\nR003,2006-03-21,4.7,0.0,41238.53\nR003,2006-03-22,7.2,0.0,41147.36\nR003,2006-03-23,8.0,0.0,40366.64\nR003,2006-03-24,11.5,0.4,39645.41\nR003,2006-03-25,6.8,8.6,568.79\nR003,2006-03-26,5.8,7.5,7093.92\nR003,2006-03-27,6.0,0.0,18966.19\nR003,2006-03-28,7.5,0.0,40356.39\nR003,2006-03-29,7.2,0.0,42432.49\nR003,2006-03-30,9.1,0.4,37815.08\nR003,2006-03-31,8.4,6.6,9085.14\nR003,2006-04-01,8.6,0.0,43928.68\nR003,2006-04-02,7.2,20.6,8786.49\nR003,2006-04-03,6.8,15.7,0.0\nR003,2006-04-04,6.6,2.9,3790.1\nR003,2006-04-05,6.6,0.0,45931.15\nR003,2006-04-06,7.3,0.0,44678.83\nR003,2006-04-07,9.3,0.5,37238.56\nR003,2006-04-08,13.8,0.5,22595.11\nR003,2006-04-09,14.6,0.9,46186.78\nR003,2006-04-10,14.2,1.0,39600.0\nR003,2006-04-11,13.7,3.6,37199.74\nR003,2006-04-12,15.2,2.2,46097.39\nR003,2006-04-13,16.2,1.5,42136.76\nR003,2006-04-14,16.6,0.3,40622.16\nR003,2006-04-15,12.9,27.3,0.0\nR003,2006-04-16,10.6,3.5,43114.65\nR003,2006-04-17,10.3,0.0,46785.32\nR003,2006-04-18,11.8,1.5,46800.0\nR003,2006-04-19,11.9,0.0,45650.97\nR003,2006-04-20,15.3,0.0,46800.0\nR003,2006-04-21,13.6,5.6,42059.57\nR003,2006-04-22,12.1,7.3,33142.12\nR003,2006-04-23,11.4,3.0,43309.35\nR003,2006-04-24,10.4,5.9,34805.7\nR003,2006-04-25,6.6,7.6,25622.54\nR003,2006-04-26,4.5,9.5,35952.9\nR003,2006-04-27,4.1,5.2,36082.08\nR003,2006-04-28,5.9,2.0,46800.0\nR003,2006-04-29,8.8,0.0,48256.42\nR003,2006-04-30,10.6,1.4,41845.33\nR003,2006-05-01,11.2,0.0,38232.51\nR003,2006-05-02,10.9,5.2,17617.47\nR003,2006-05-03,10.2,8.2,1717.94\nR003,2006-05-04,9.6,2.0,11545.59\nR003,2006-05-05,12.8,0.9,31975.18\nR003,2006-05-06,15.6,6.0,37573.7\nR003,2006-05-07,13.6,17.0,32100.87\nR003,2006-05-08,13.3,12.9,7433.61\nR003,2006-05-09,12.2,5.5,34009.32\nR003,2006-05-10,12.7,0.0,45442.7\nR003,2006-05-11,14.6,0.0,50175.06\nR003,2006-05-12,16.2,3.6,45925.48\nR003,2006-05-13,13.8,7.6,34605.71\nR003,2006-05-14,11.9,8.2,1165.36\nR003,2006-05-15,12.4,3.8,37480.28\nR003,2006-05-16,15.2,0.3,48273.58\nR003,2006-05-17,15.7,8.9,36245.63\nR003,2006-05-18,13.5,13.8,14400.0\nR003,2006-05-19,15.2,2.8,35875.08\nR003,2006-05-20,17.0,2.4,47782.82\nR003,2006-05-21,18.0,2.8,49306.22\nR003,2006-05-22,18.7,0.7,47706.25\nR003,2006-05-23,18.6,0.6,48854.3\nR003,2006-05-24,19.3,0.2,45971.67\nR003,2006-05-25,21.2,0.0,50471.65\nR003,2006-05-26,22.5,0.0,50539.06\nR003,2006-05-27,16.9,28.1,11253.61\nR003,2006-05-28,14.1,37.7,3869.85\nR003,2006-05-29,15.7,1.4,41573.84\nR003,2006-05-30,20.5,0.0,51889.69\nR003,2006-05-31,25.3,0.4,51869.45\nR003,2006-06-01,25.5,0.6,51849.59\nR003,2006-06-02,24.4,1.1,51104.48\nR003,2006-06-03,25.7,0.9,52174.26\nR003,2006-06-04,25.6,2.9,52519.94\nR003,2006-06-05,25.1,0.6,51774.17\nR003,2006-06-06,20.5,11.7,32558.45\nR003,2006-06-07,20.0,6.9,46259.27\nR003,2006-06-08,18.8,10.1,34017.75\nR003,2006-06-09,20.0,4.5,49026.01\nR003,2006-06-10,18.1,12.3,40096.07\nR003,2006-06-11,17.4,0.9,49328.64\nR003,2006-06-12,16.8,8.5,40977.37\nR003,2006-06-13,17.7,0.7,50623.56\nR003,2006-06-14,19.2,8.2,46305.65\nR003,2006-06-15,19.6,5.9,46225.11\nR003,2006-06-16,19.6,7.1,45057.51\nR003,2006-06-17,18.4,0.8,51915.38\nR003,2006-06-18,17.8,0.0,53881.98\nR003,2006-06-19,18.6,0.1,51225.02\nR003,2006-06-20,21.0,0.5,51464.21\nR003,2006-06-21,22.6,1.1,51887.71\nR003,2006-06-22,22.8,3.7,53042.57\nR003,2006-06-23,21.8,7.3,48350.82\nR003,2006-06-24,21.3,8.1,48566.36\nR003,2006-06-25,21.2,10.7,49687.25\nR003,2006-06-26,21.0,10.6,49969.0\nR003,2006-06-27,22.1,2.2,51582.52\nR003,2006-06-28,23.9,0.9,48682.81\nR003,2006-06-29,23.9,8.7,33202.57\nR003,2006-06-30,23.7,1.2,45593.92\nR003,2006-07-01,23.0,4.8,44990.09\nR003,2006-07-02,21.2,4.5,39807.28\nR003,2006-07-03,19.8,0.2,46418.24\nR003,2006-07-04,19.3,0.1,53914.75\nR003,2006-07-05,19.8,0.7,53384.66\nR003,2006-07-06,19.2,6.7,30518.17\nR003,2006-07-07,19.2,19.1,26271.28\nR003,2006-07-08,17.3,34.3,10863.65\nR003,2006-07-09,19.6,2.3,52680.19\nR003,2006-07-10,20.9,2.1,51603.65\nR003,2006-07-11,20.9,5.7,49395.54\nR003,2006-07-12,22.2,3.2,47869.8\nR003,2006-07-13,24.1,0.6,50387.87\nR003,2006-07-14,23.0,14.3,51381.42\nR003,2006-07-15,20.3,7.9,35384.8\nR003,2006-07-16,20.6,1.3,46957.67\nR003,2006-07-17,21.8,0.7,50985.26\nR003,2006-07-18,21.3,1.2,52068.92\nR003,2006-07-19,20.8,0.3,48130.55\nR003,2006-07-20,21.9,0.0,50966.26\nR003,2006-07-21,22.6,0.0,50617.2\nR003,2006-07-22,23.4,0.0,49985.78\nR003,2006-07-23,22.8,4.9,40445.56\nR003,2006-07-24,21.3,31.9,20841.22\nR003,2006-07-25,22.0,1.9,46298.08\nR003,2006-07-26,22.1,0.7,45634.29\nR003,2006-07-27,22.7,0.9,48812.63\nR003,2006-07-28,21.3,0.3,47399.51\nR003,2006-07-29,20.0,1.3,40753.36\nR003,2006-07-30,19.8,1.7,42681.09\nR003,2006-07-31,20.9,0.0,47070.37\nR003,2006-08-01,22.1,0.7,49511.32\nR003,2006-08-02,22.5,0.9,47469.93\nR003,2006-08-03,23.2,0.5,47478.82\nR003,2006-08-04,25.1,0.0,48910.82\nR003,2006-08-05,26.6,0.2,48971.79\nR003,2006-08-06,27.0,0.0,48692.99\nR003,2006-08-07,27.0,0.4,48076.73\nR003,2006-08-08,26.8,0.2,48315.38\nR003,2006-08-09,23.9,4.5,46988.92\nR003,2006-08-10,23.9,0.0,48031.57\nR003,2006-08-11,26.5,0.3,47722.57\nR003,2006-08-12,27.8,0.3,47415.16\nR003,2006-08-13,27.4,0.0,47109.29\nR003,2006-08-14,28.3,0.4,47139.38\nR003,2006-08-15,27.5,1.4,46800.0\nR003,2006-08-16,27.6,1.8,46866.52\nR003,2006-08-17,27.8,0.3,46897.53\nR003,2006-08-18,26.8,1.1,46928.65\nR003,2006-08-19,24.8,6.2,41629.42\nR003,2006-08-20,23.8,13.4,14009.63\nR003,2006-08-21,22.3,12.3,36569.81\nR003,2006-08-22,22.4,5.3,45378.13\nR003,2006-08-23,22.5,0.3,46800.0\nR003,2006-08-24,22.4,0.3,43659.49\nR003,2006-08-25,23.4,0.0,46800.0\nR003,2006-08-26,24.7,0.0,46800.0\nR003,2006-08-27,26.6,0.0,46746.81\nR003,2006-08-28,26.5,0.5,46644.01\nR003,2006-08-29,24.9,1.9,46540.78\nR003,2006-08-30,24.3,0.0,46437.24\nR003,2006-08-31,26.2,0.0,46333.47\nR003,2006-09-01,25.1,0.8,43265.94\nR003,2006-09-02,23.3,13.1,34024.15\nR003,2006-09-03,19.4,28.2,26619.02\nR003,2006-09-04,18.8,0.0,42792.68\nR003,2006-09-05,18.0,11.1,39039.76\nR003,2006-09-06,16.1,22.5,17820.82\nR003,2006-09-07,15.1,2.4,38328.14\nR003,2006-09-08,17.0,0.0,43200.0\nR003,2006-09-09,18.5,0.0,43385.26\nR003,2006-09-10,18.7,0.3,40401.35\nR003,2006-09-11,18.6,0.0,31656.51\nR003,2006-09-12,18.3,0.4,41586.02\nR003,2006-09-13,16.6,2.1,39665.79\nR003,2006-09-14,16.1,2.5,41026.06\nR003,2006-09-15,15.5,0.8,41037.6\nR003,2006-09-16,14.6,0.2,41171.71\nR003,2006-09-17,14.8,0.0,42916.52\nR003,2006-09-18,17.0,0.0,42848.88\nR003,2006-09-19,19.9,0.0,42146.99\nR003,2006-09-20,20.8,0.5,42499.36\nR003,2006-09-21,18.1,3.3,38042.61\nR003,2006-09-22,17.1,7.4,2772.48\nR003,2006-09-23,17.3,5.1,38298.14\nR003,2006-09-24,18.1,7.1,30840.93\nR003,2006-09-25,18.2,1.5,41400.68\nR003,2006-09-26,17.8,5.6,28505.65\nR003,2006-09-27,17.1,1.9,21359.61\nR003,2006-09-28,17.9,0.1,40870.92\nR003,2006-09-29,19.6,0.0,39934.43\nR003,2006-09-30,19.0,6.5,29970.15\nR003,2006-10-01,16.3,10.2,9141.72\nR003,2006-10-02,14.9,0.0,38552.99\nR003,2006-10-03,14.9,0.0,38139.64\nR003,2006-10-04,16.0,0.0,38670.43\nR003,2006-10-05,17.3,0.0,39175.52\nR003,2006-10-06,19.4,0.0,38750.55\nR003,2006-10-07,19.4,0.1,36583.37\nR003,2006-10-08,18.0,0.0,37286.95\nR003,2006-10-09,17.3,0.0,34224.47\nR003,2006-10-10,15.4,2.2,27961.54\nR003,2006-10-11,15.1,0.2,28375.0\nR003,2006-10-12,16.1,0.0,32515.59\nR003,2006-10-13,14.9,2.4,29677.43\nR003,2006-10-14,14.1,0.1,34514.74\nR003,2006-10-15,12.5,5.0,13449.99\nR003,2006-10-16,11.2,10.5,19869.36\nR003,2006-10-17,10.0,9.1,19346.46\nR003,2006-10-18,7.8,26.7,0.0\nR003,2006-10-19,7.8,0.0,36000.0\nR003,2006-10-20,11.8,0.0,36000.0\nR003,2006-10-21,13.0,0.2,33766.94\nR003,2006-10-22,12.0,0.2,35497.3\nR003,2006-10-23,12.5,0.0,32615.01\nR003,2006-10-24,13.2,0.0,34318.75\nR003,2006-10-25,13.9,0.0,36000.0\nR003,2006-10-26,13.1,6.0,20149.83\nR003,2006-10-27,13.8,0.0,36000.0\nR003,2006-10-28,13.0,0.0,36000.0\nR003,2006-10-29,12.9,0.4,32095.33\nR003,2006-10-30,12.1,11.5,10102.57\nR003,2006-10-31,14.7,0.0,32996.47\nR003,2006-11-01,16.5,0.0,36000.0\nR003,2006-11-02,13.9,17.2,8696.66\nR003,2006-11-03,13.2,0.4,33807.96\nR003,2006-11-04,11.7,44.0,0.0\nR003,2006-11-05,6.7,31.2,0.0\nR003,2006-11-06,3.8,1.9,33271.08\nR003,2006-11-07,2.8,34.7,2438.5\nR003,2006-11-08,2.6,5.7,29760.37\nR003,2006-11-09,6.3,0.8,26579.0\nR003,2006-11-10,6.1,0.0,34959.78\nR003,2006-11-11,5.1,18.5,0.0\nR003,2006-11-12,2.3,0.5,19369.03\nR003,2006-11-13,5.4,3.0,23127.42\nR003,2006-11-14,4.4,34.2,23343.92\nR003,2006-11-15,4.0,21.7,8934.25\nR003,2006-11-16,5.8,18.8,2509.92\nR003,2006-11-17,5.8,16.4,0.0\nR003,2006-11-18,0.8,0.0,31040.14\nR003,2006-11-19,-1.4,0.0,32578.79\nR003,2006-11-20,1.7,0.0,31559.96\nR003,2006-11-21,3.7,0.0,29977.34\nR003,2006-11-22,5.4,0.0,28800.0\nR003,2006-11-23,5.3,0.0,32214.67\nR003,2006-11-24,5.0,0.0,31104.52\nR003,2006-11-25,4.7,0.0,31544.24\nR003,2006-11-26,5.1,0.0,30863.66\nR003,2006-11-27,7.9,0.0,23572.05\nR003,2006-11-28,6.4,0.0,26882.23\nR003,2006-11-29,5.1,0.0,28138.19\nR003,2006-11-30,3.5,0.3,29214.02\nR003,2006-12-01,4.0,10.0,18000.0\nR003,2006-12-02,2.3,1.6,29559.05\nR003,2006-12-03,3.5,0.0,29970.54\nR003,2006-12-04,5.1,0.0,30365.74\nR003,2006-12-05,4.7,0.0,30378.21\nR003,2006-12-06,5.1,0.0,29621.66\nR003,2006-12-07,4.2,0.0,30398.88\nR003,2006-12-08,4.0,0.0,28865.89\nR003,2006-12-09,2.8,0.0,30413.69\nR003,2006-12-10,5.2,0.0,30418.85\nR003,2006-12-11,4.3,0.0,30422.45\nR003,2006-12-12,3.5,0.0,30036.71\nR003,2006-12-13,3.3,0.6,28871.73\nR003,2006-12-14,3.3,6.9,3603.47\nR003,2006-12-15,2.6,1.6,30031.21\nR003,2006-12-16,3.7,0.1,29636.34\nR003,2006-12-17,6.6,0.0,29487.83\nR003,2006-12-18,8.1,0.0,23607.6\nR003,2006-12-19,8.2,0.0,1510.76\nR003,2006-12-20,7.8,5.5,4236.27\nR003,2006-12-21,4.5,19.5,0.0\nR003,2006-12-22,0.1,10.2,29960.91\nR003,2006-12-23,-0.4,3.9,24752.35\nR003,2006-12-24,0.5,4.8,30318.85\nR003,2006-12-25,1.4,33.7,0.0\nR003,2006-12-26,-2.7,10.1,23826.28\nR003,2006-12-27,-10.6,0.0,31360.67\nR003,2006-12-28,-6.2,0.3,28800.0\nR003,2006-12-29,0.6,5.7,2203.94\nR003,2006-12-30,0.4,10.4,20585.15\nR003,2006-12-31,-3.6,0.0,31544.18\nR003,2007-01-01,-1.2,1.7,27902.29\nR003,2007-01-02,1.7,0.3,19645.38\nR003,2007-01-03,4.5,0.0,16954.84\nR003,2007-01-04,5.3,0.0,31656.96\nR003,2007-01-05,2.9,0.0,27231.53\nR003,2007-01-06,2.3,3.9,927.5\nR003,2007-01-07,1.6,0.8,30182.61\nR003,2007-01-08,2.0,3.6,12421.81\nR003,2007-01-09,2.9,2.5,17475.61\nR003,2007-01-10,2.9,0.0,28391.45\nR003,2007-01-11,2.3,0.0,32204.68\nR003,2007-01-12,3.3,0.0,32275.44\nR003,2007-01-13,4.1,3.9,28405.54\nR003,2007-01-14,2.0,30.6,4370.78\nR003,2007-01-15,2.8,15.7,0.0\nR003,2007-01-16,-0.1,10.8,13552.12\nR003,2007-01-17,0.7,0.6,30347.87\nR003,2007-01-18,2.0,0.0,28932.65\nR003,2007-01-19,4.7,0.0,32400.0\nR003,2007-01-20,3.7,34.9,0.0\nR003,2007-01-21,1.0,2.5,32400.0\nR003,2007-01-22,3.3,1.4,18000.0\nR003,2007-01-23,5.2,0.0,32400.0\nR003,2007-01-24,7.1,0.0,29530.79\nR003,2007-01-25,9.2,0.0,32400.0\nR003,2007-01-26,8.6,0.0,32130.43\nR003,2007-01-27,8.1,1.5,2619.6\nR003,2007-01-28,4.4,37.6,0.0\nR003,2007-01-29,-4.6,1.4,26464.87\nR003,2007-01-30,-2.5,22.9,12135.33\nR003,2007-01-31,-4.5,6.5,18947.5\nR003,2007-02-01,-4.2,1.8,26941.81\nR003,2007-02-02,-0.1,6.7,6980.61\nR003,2007-02-03,0.9,1.9,30420.19\nR003,2007-02-04,1.2,1.0,29403.47\nR003,2007-02-05,0.2,7.6,26115.86\nR003,2007-02-06,0.9,2.7,14776.68\nR003,2007-02-07,2.7,0.3,31421.37\nR003,2007-02-08,3.8,0.0,26549.85\nR003,2007-02-09,5.3,0.0,35307.92\nR003,2007-02-10,4.3,0.0,35759.91\nR003,2007-02-11,5.1,0.0,35796.76\nR003,2007-02-12,5.4,0.0,32300.58\nR003,2007-02-13,5.7,0.0,35528.33\nR003,2007-02-14,5.5,0.0,36919.41\nR003,2007-02-15,5.8,0.0,37191.23\nR003,2007-02-16,4.5,0.0,37361.43\nR003,2007-02-17,3.7,2.9,21594.64\nR003,2007-02-18,1.2,0.1,35360.91\nR003,2007-02-19,1.8,0.0,37880.21\nR003,2007-02-20,2.8,6.7,588.06\nR003,2007-02-21,3.1,8.4,0.0\nR003,2007-02-22,1.8,2.2,36387.43\nR003,2007-02-23,1.0,4.9,3995.4\nR003,2007-02-24,-3.2,11.6,31988.97\nR003,2007-02-25,-7.1,0.0,38940.42\nR003,2007-02-26,-5.3,0.0,37844.18\nR003,2007-02-27,-2.9,0.0,39296.11\nR003,2007-02-28,2.6,0.0,32010.06\nR003,2007-03-01,5.3,9.1,5659.69\nR003,2007-03-02,5.6,2.7,23283.0\nR003,2007-03-03,6.2,3.2,20430.23\nR003,2007-03-04,7.0,6.9,32592.47\nR003,2007-03-05,6.2,6.1,7178.59\nR003,2007-03-06,6.7,6.8,20264.47\nR003,2007-03-07,5.4,2.1,14712.52\nR003,2007-03-08,5.7,0.0,39600.0\nR003,2007-03-09,5.5,0.0,39600.0\nR003,2007-03-10,5.5,0.0,39492.37\nR003,2007-03-11,5.7,0.0,39600.0\nR003,2007-03-12,4.7,6.2,27897.24\nR003,2007-03-13,3.9,8.0,337.82\nR003,2007-03-14,6.3,0.2,36000.0\nR003,2007-03-15,5.9,0.0,37306.08\nR003,2007-03-16,4.5,2.9,35646.75\nR003,2007-03-17,3.1,2.9,36471.2\nR003,2007-03-18,2.7,23.7,0.0\nR003,2007-03-19,1.8,0.5,39600.0\nR003,2007-03-20,4.3,0.0,41350.83\nR003,2007-03-21,6.9,0.0,41895.77\nR003,2007-03-22,7.9,0.0,42123.3\nR003,2007-03-23,6.9,0.0,42228.55\nR003,2007-03-24,8.2,11.6,13574.11\nR003,2007-03-25,6.5,9.2,33633.04\nR003,2007-03-26,3.6,14.8,695.93\nR003,2007-03-27,4.1,6.3,25712.1\nR003,2007-03-28,3.9,1.0,38600.81\nR003,2007-03-29,4.4,0.6,38501.37\nR003,2007-03-30,5.1,2.2,36883.19\nR003,2007-03-31,4.8,0.0,35110.64\nR003,2007-04-01,7.4,0.9,38156.81\nR003,2007-04-02,7.5,6.8,0.0\nR003,2007-04-03,6.1,7.5,9067.54\nR003,2007-04-04,5.2,4.8,24448.42\nR003,2007-04-05,6.9,0.0,40238.91\nR003,2007-04-06,6.0,20.5,97.64\nR003,2007-04-07,4.3,0.0,45935.01\nR003,2007-04-08,7.1,0.0,46040.52\nR003,2007-04-09,6.3,0.0,45552.22\nR003,2007-04-10,7.1,0.0,45199.98\nR003,2007-04-11,8.2,1.7,28066.11\nR003,2007-04-12,7.9,0.4,43406.38\nR003,2007-04-13,6.0,13.1,22925.16\nR003,2007-04-14,4.0,1.8,46554.68\nR003,2007-04-15,4.6,0.5,35340.87\nR003,2007-04-16,5.0,4.9,26514.15\nR003,2007-04-17,4.6,0.8,45010.87\nR003,2007-04-18,6.4,0.0,46171.1\nR003,2007-04-19,6.9,3.2,22732.96\nR003,2007-04-20,5.6,8.1,28268.7\nR003,2007-04-21,7.3,0.0,45115.46\nR003,2007-04-22,9.7,1.3,36000.0\nR003,2007-04-23,7.1,8.3,20505.08\nR003,2007-04-24,7.4,0.0,43438.26\nR003,2007-04-25,7.9,3.0,39697.93\nR003,2007-04-26,9.5,1.3,44323.57\nR003,2007-04-27,11.3,1.3,45450.77\nR003,2007-04-28,11.2,2.5,45742.58\nR003,2007-04-29,11.9,0.0,43329.38\nR003,2007-04-30,10.0,12.0,3860.69\nR003,2007-05-01,10.2,5.1,37336.96\nR003,2007-05-02,10.2,17.4,344.77\nR003,2007-05-03,9.8,16.4,43902.05\nR003,2007-05-04,9.1,0.0,49434.49\nR003,2007-05-05,11.9,0.0,49705.53\nR003,2007-05-06,14.8,0.0,49712.57\nR003,2007-05-07,17.1,0.0,49683.94\nR003,2007-05-08,19.8,0.2,44475.23\nR003,2007-05-09,15.6,15.1,42159.89\nR003,2007-05-10,14.4,2.5,46177.43\nR003,2007-05-11,14.1,0.0,42273.69\nR003,2007-05-12,16.7,0.0,50228.65\nR003,2007-05-13,16.6,1.1,40385.24\nR003,2007-05-14,15.8,0.1,45562.23\nR003,2007-05-15,14.9,0.0,50400.0\nR003,2007-05-16,17.9,0.0,50462.63\nR003,2007-05-17,21.0,0.0,50433.24\nR003,2007-05-18,22.4,0.0,50404.3\nR003,2007-05-19,22.1,0.0,50400.0\nR003,2007-05-20,21.6,0.2,50400.0\nR003,2007-05-21,21.8,2.5,50400.0\nR003,2007-05-22,22.5,1.5,50649.08\nR003,2007-05-23,23.4,0.8,50400.0\nR003,2007-05-24,24.1,0.4,48726.8\nR003,2007-05-25,23.8,1.3,50400.0\nR003,2007-05-26,25.0,0.7,50400.0\nR003,2007-05-27,24.5,3.2,32380.0\nR003,2007-05-28,23.6,4.9,43200.0\nR003,2007-05-29,23.4,2.2,51553.79\nR003,2007-05-30,22.1,5.1,43850.28\nR003,2007-05-31,21.8,8.8,47944.83\nR003,2007-06-01,21.7,10.9,48522.11\nR003,2007-06-02,20.3,2.9,49403.88\nR003,2007-06-03,19.5,10.7,26685.26\nR003,2007-06-04,18.3,14.6,21955.92\nR003,2007-06-05,18.0,22.3,3542.59\nR003,2007-06-06,19.5,0.0,49156.04\nR003,2007-06-07,20.1,0.5,51012.15\nR003,2007-06-08,18.4,9.2,13828.75\nR003,2007-06-09,18.0,19.3,21917.74\nR003,2007-06-10,17.9,12.9,48354.92\nR003,2007-06-11,18.5,3.3,50158.57\nR003,2007-06-12,18.9,3.6,50929.21\nR003,2007-06-13,18.7,1.9,51661.47\nR003,2007-06-14,21.0,0.1,53114.85\nR003,2007-06-15,22.8,1.0,53104.01\nR003,2007-06-16,23.9,0.8,53461.12\nR003,2007-06-17,22.6,4.5,52716.0\nR003,2007-06-18,20.0,14.0,46776.39\nR003,2007-06-19,19.8,8.0,35723.99\nR003,2007-06-20,18.5,7.5,34726.29\nR003,2007-06-21,17.6,5.9,47105.65\nR003,2007-06-22,17.8,0.9,51619.58\nR003,2007-06-23,20.5,0.2,53673.69\nR003,2007-06-24,21.6,0.4,51927.68\nR003,2007-06-25,18.9,0.4,53965.94\nR003,2007-06-26,18.0,0.5,53388.33\nR003,2007-06-27,21.2,0.0,53971.55\nR003,2007-06-28,23.6,0.4,53970.73\nR003,2007-06-29,23.4,1.0,53967.42\nR003,2007-06-30,19.4,8.7,47482.98\nR003,2007-07-01,19.5,3.5,53197.19\nR003,2007-07-02,20.8,1.0,53380.09\nR003,2007-07-03,21.1,0.4,52850.09\nR003,2007-07-04,22.6,0.1,53147.46\nR003,2007-07-05,24.1,0.0,53898.17\nR003,2007-07-06,25.3,0.0,53879.07\nR003,2007-07-07,19.1,25.8,19175.75\nR003,2007-07-08,18.4,0.8,53760.92\nR003,2007-07-09,18.2,0.2,53041.53\nR003,2007-07-10,19.3,0.0,53191.58\nR003,2007-07-11,22.8,0.0,53752.02\nR003,2007-07-12,25.5,1.0,53720.37\nR003,2007-07-13,25.3,1.4,53073.74\nR003,2007-07-14,20.8,14.0,29549.05\nR003,2007-07-15,20.3,4.3,45150.53\nR003,2007-07-16,19.8,0.4,51173.78\nR003,2007-07-17,21.1,0.0,52404.94\nR003,2007-07-18,22.1,0.0,53487.74\nR003,2007-07-19,22.7,0.0,53441.92\nR003,2007-07-20,24.6,0.0,53376.17\nR003,2007-07-21,25.0,0.0,53271.69\nR003,2007-07-22,26.2,0.2,52713.97\nR003,2007-07-23,25.6,0.0,52676.09\nR003,2007-07-24,25.3,0.0,51934.13\nR003,2007-07-25,26.9,0.0,52248.61\nR003,2007-07-26,27.1,0.1,51510.4\nR003,2007-07-27,24.8,0.1,50894.65\nR003,2007-07-28,23.4,0.0,49895.39\nR003,2007-07-29,23.9,0.5,49945.9\nR003,2007-07-30,25.5,0.0,51020.37\nR003,2007-07-31,27.1,0.0,50639.57\nR003,2007-08-01,27.1,0.3,48034.89\nR003,2007-08-02,23.8,11.8,46916.44\nR003,2007-08-03,21.9,15.5,32745.75\nR003,2007-08-04,22.5,2.8,46800.0\nR003,2007-08-05,24.3,1.5,46844.69\nR003,2007-08-06,27.0,0.0,48679.13\nR003,2007-08-07,28.1,2.0,48401.44\nR003,2007-08-08,26.5,1.8,47970.49\nR003,2007-08-09,24.3,7.7,46045.66\nR003,2007-08-10,23.5,1.0,47349.19\nR003,2007-08-11,24.8,2.8,47377.98\nR003,2007-08-12,26.3,0.4,47406.97\nR003,2007-08-13,27.2,0.0,47771.6\nR003,2007-08-14,23.8,6.8,39331.58\nR003,2007-08-15,23.3,15.1,44847.39\nR003,2007-08-16,22.7,4.2,46800.0\nR003,2007-08-17,22.5,4.2,31634.64\nR003,2007-08-18,22.3,3.2,46302.96\nR003,2007-08-19,22.5,5.4,46476.32\nR003,2007-08-20,23.1,4.1,46800.0\nR003,2007-08-21,23.5,4.2,46800.0\nR003,2007-08-22,22.4,11.7,46211.64\nR003,2007-08-23,21.4,6.0,36490.89\nR003,2007-08-24,23.0,3.0,39215.78\nR003,2007-08-25,23.4,8.7,39598.75\nR003,2007-08-26,22.8,2.1,46800.0\nR003,2007-08-27,22.3,4.0,38085.68\nR003,2007-08-28,21.1,6.0,37966.74\nR003,2007-08-29,20.5,2.5,30090.61\nR003,2007-08-30,21.1,0.9,40869.38\nR003,2007-08-31,21.5,0.0,43569.34\nR003,2007-09-01,22.6,0.0,46262.86\nR003,2007-09-02,23.8,0.1,45576.95\nR003,2007-09-03,24.2,0.6,45304.38\nR003,2007-09-04,25.6,0.0,45033.2\nR003,2007-09-05,25.5,0.4,44443.1\nR003,2007-09-06,24.9,1.2,44174.69\nR003,2007-09-07,24.4,1.5,43587.76\nR003,2007-09-08,23.8,3.8,43640.99\nR003,2007-09-09,20.7,17.7,35890.18\nR003,2007-09-10,18.1,20.4,34541.76\nR003,2007-09-11,18.7,0.0,42839.52\nR003,2007-09-12,19.7,0.0,43200.0\nR003,2007-09-13,19.5,12.5,43200.0\nR003,2007-09-14,14.8,14.8,34468.97\nR003,2007-09-15,15.7,0.0,41050.78\nR003,2007-09-16,14.9,6.5,30999.58\nR003,2007-09-17,11.4,4.0,20747.72\nR003,2007-09-18,15.0,0.0,38164.2\nR003,2007-09-19,17.7,0.0,42456.92\nR003,2007-09-20,19.1,0.1,42730.1\nR003,2007-09-21,18.9,0.0,42525.19\nR003,2007-09-22,19.3,0.0,42559.93\nR003,2007-09-23,19.8,1.2,39648.43\nR003,2007-09-24,18.7,6.8,19115.66\nR003,2007-09-25,17.4,20.5,12792.14\nR003,2007-09-26,17.5,0.1,38195.81\nR003,2007-09-27,17.9,0.0,41464.06\nR003,2007-09-28,18.7,0.0,41181.41\nR003,2007-09-29,18.4,0.0,41218.04\nR003,2007-09-30,18.2,0.0,40830.02\nR003,2007-10-01,18.1,1.4,40117.07\nR003,2007-10-02,17.7,0.2,40044.05\nR003,2007-10-03,17.8,0.3,39971.97\nR003,2007-10-04,19.0,0.0,38858.38\nR003,2007-10-05,19.5,0.1,38072.04\nR003,2007-10-06,18.3,1.5,37483.49\nR003,2007-10-07,16.7,0.6,36883.1\nR003,2007-10-08,16.8,1.1,26634.66\nR003,2007-10-09,16.4,0.4,35404.1\nR003,2007-10-10,16.4,0.7,36083.49\nR003,2007-10-11,16.6,1.2,36461.7\nR003,2007-10-12,16.1,0.0,36840.89\nR003,2007-10-13,17.6,0.7,36243.68\nR003,2007-10-14,16.1,13.4,20632.01\nR003,2007-10-15,11.8,32.7,168.14\nR003,2007-10-16,10.1,0.2,33719.86\nR003,2007-10-17,10.3,0.0,33603.95\nR003,2007-10-18,11.1,0.0,36000.0\nR003,2007-10-19,13.5,0.0,36000.0\nR003,2007-10-20,15.4,0.0,36000.0\nR003,2007-10-21,16.3,0.0,36000.0\nR003,2007-10-22,15.9,0.0,36000.0\nR003,2007-10-23,16.6,0.0,36000.0\nR003,2007-10-24,17.7,0.0,36000.0\nR003,2007-10-25,17.9,0.1,32038.09\nR003,2007-10-26,14.1,7.6,25488.0\nR003,2007-10-27,12.3,0.6,26569.07\nR003,2007-10-28,11.0,4.1,23703.92\nR003,2007-10-29,11.8,0.0,35938.9\nR003,2007-10-30,11.3,4.6,21600.0\nR003,2007-10-31,12.2,2.3,18394.66\nR003,2007-11-01,11.9,0.0,33555.41\nR003,2007-11-02,10.2,5.9,22946.35\nR003,2007-11-03,5.3,30.0,14400.0\nR003,2007-11-04,3.0,0.0,27409.32\nR003,2007-11-05,8.1,10.6,19236.39\nR003,2007-11-06,8.4,18.2,2697.54\nR003,2007-11-07,6.7,0.1,30195.98\nR003,2007-11-08,5.4,8.1,1380.68\nR003,2007-11-09,4.6,12.6,17955.02\nR003,2007-11-10,4.6,0.0,29661.06\nR003,2007-11-11,4.1,26.1,0.0\nR003,2007-11-12,2.5,7.8,26160.21\nR003,2007-11-13,6.0,12.5,30686.67\nR003,2007-11-14,5.7,15.7,0.0\nR003,2007-11-15,7.7,0.0,34268.18\nR003,2007-11-16,10.8,0.0,32670.8\nR003,2007-11-17,10.9,0.0,33122.12\nR003,2007-11-18,9.4,2.7,28794.31\nR003,2007-11-19,9.6,0.0,28406.85\nR003,2007-11-20,10.0,0.0,27800.42\nR003,2007-11-21,8.2,0.0,31611.56\nR003,2007-11-22,3.5,1.9,24438.67\nR003,2007-11-23,1.9,0.0,31018.59\nR003,2007-11-24,3.0,0.0,31830.18\nR003,2007-11-25,3.6,0.0,31852.98\nR003,2007-11-26,5.5,0.0,31223.22\nR003,2007-11-27,6.1,26.3,17703.93\nR003,2007-11-28,-2.4,9.0,2632.38\nR003,2007-11-29,-5.0,0.7,29900.86\nR003,2007-11-30,0.8,0.0,27145.39\nR003,2007-12-01,5.5,14.8,10905.61\nR003,2007-12-02,1.4,20.7,0.0\nR003,2007-12-03,-1.6,0.0,27915.13\nR003,2007-12-04,1.7,0.0,29982.82\nR003,2007-12-05,2.2,0.0,30377.8\nR003,2007-12-06,3.2,0.0,28800.0\nR003,2007-12-07,5.1,8.9,0.0\nR003,2007-12-08,3.9,2.3,24721.94\nR003,2007-12-09,2.4,0.0,29256.58\nR003,2007-12-10,4.2,2.2,28245.11\nR003,2007-12-11,5.4,11.3,12847.53\nR003,2007-12-12,5.1,12.6,2070.54\nR003,2007-12-13,5.7,4.4,13161.66\nR003,2007-12-14,6.2,15.7,0.0\nR003,2007-12-15,3.9,3.3,25438.47\nR003,2007-12-16,-1.3,0.0,30024.28\nR003,2007-12-17,-1.4,6.5,20628.14\nR003,2007-12-18,-3.5,6.4,24665.62\nR003,2007-12-19,-5.5,2.4,12844.08\nR003,2007-12-20,-1.6,1.9,29207.46\nR003,2007-12-21,-0.1,8.3,8134.77\nR003,2007-12-22,-1.4,0.2,29415.54\nR003,2007-12-23,0.5,2.4,29556.91\nR003,2007-12-24,-0.0,0.0,29932.05\nR003,2007-12-25,-1.3,0.0,31268.43\nR003,2007-12-26,-2.2,0.0,31305.58\nR003,2007-12-27,-1.9,0.0,29876.1\nR003,2007-12-28,0.7,1.2,17640.48\nR003,2007-12-29,0.6,3.9,14227.83\nR003,2007-12-30,0.1,0.2,30029.92\nR003,2007-12-31,-0.8,1.0,31344.41\nR003,2008-01-01,-2.4,0.0,31575.52\nR003,2008-01-02,-2.2,0.0,31629.28\nR003,2008-01-03,-1.8,0.0,30863.88\nR003,2008-01-04,0.2,4.2,394.34\nR003,2008-01-05,-0.2,10.8,2514.92\nR003,2008-01-06,-4.7,2.7,31864.93\nR003,2008-01-07,-6.4,0.7,31928.29\nR003,2008-01-08,-7.1,0.0,31993.14\nR003,2008-01-09,-5.1,0.0,32059.34\nR003,2008-01-10,-3.6,0.0,32126.76\nR003,2008-01-11,-4.4,0.4,32195.23\nR003,2008-01-12,-5.1,0.0,32264.64\nR003,2008-01-13,-3.3,0.0,32334.83\nR003,2008-01-14,-2.4,0.0,32400.0\nR003,2008-01-15,-1.2,0.0,32400.0\nR003,2008-01-16,-1.1,0.0,32400.0\nR003,2008-01-17,-0.3,0.0,32400.0\nR003,2008-01-18,0.3,0.0,32400.0\nR003,2008-01-19,0.0,0.0,32400.0\nR003,2008-01-20,-0.8,0.0,32400.0\nR003,2008-01-21,1.2,0.9,29074.0\nR003,2008-01-22,3.4,7.8,29772.64\nR003,2008-01-23,6.0,0.0,32400.0\nR003,2008-01-24,3.1,11.9,0.0\nR003,2008-01-25,-2.5,0.2,32127.48\nR003,2008-01-26,-1.5,1.5,24230.69\nR003,2008-01-27,-0.4,6.8,29774.99\nR003,2008-01-28,-0.4,25.9,0.0\nR003,2008-01-29,-1.2,2.8,1208.34\nR003,2008-01-30,-1.6,3.0,11216.41\nR003,2008-01-31,-3.2,5.2,32400.0\nR003,2008-02-01,-3.5,0.0,32114.39\nR003,2008-02-02,-1.8,0.0,32696.2\nR003,2008-02-03,-1.0,0.0,33083.25\nR003,2008-02-04,0.5,0.0,33002.0\nR003,2008-02-05,1.1,0.0,32919.72\nR003,2008-02-06,3.6,0.0,33184.67\nR003,2008-02-07,3.4,0.0,32822.88\nR003,2008-02-08,3.7,0.0,29051.14\nR003,2008-02-09,3.1,0.0,33765.86\nR003,2008-02-10,3.1,0.0,35408.64\nR003,2008-02-11,2.7,0.0,34805.34\nR003,2008-02-12,-0.1,0.0,36164.83\nR003,2008-02-13,-0.0,3.9,32285.96\nR003,2008-02-14,-2.6,8.0,30282.31\nR003,2008-02-15,-3.6,11.0,26740.8\nR003,2008-02-16,-3.2,14.2,509.74\nR003,2008-02-17,-2.6,10.4,0.0\nR003,2008-02-18,0.6,12.7,0.0\nR003,2008-02-19,-1.7,12.9,0.0\nR003,2008-02-20,-3.5,20.9,24594.44\nR003,2008-02-21,-1.7,2.1,31400.7\nR003,2008-02-22,3.2,1.0,30309.78\nR003,2008-02-23,5.4,0.5,33805.72\nR003,2008-02-24,3.2,17.1,1.55\nR003,2008-02-25,0.7,0.0,36561.14\nR003,2008-02-26,2.5,0.0,39068.54\nR003,2008-02-27,5.5,0.0,39243.44\nR003,2008-02-28,7.4,0.0,39389.83\nR003,2008-02-29,6.5,9.9,3597.81\nR003,2008-03-01,4.5,0.0,33954.1\nR003,2008-03-02,7.4,7.0,39600.0\nR003,2008-03-03,4.3,19.5,20732.21\nR003,2008-03-04,3.9,0.6,38678.8\nR003,2008-03-05,9.7,0.0,38291.04\nR003,2008-03-06,10.0,0.2,34953.74\nR003,2008-03-07,7.0,0.9,28417.72\nR003,2008-03-08,9.6,0.0,39293.43\nR003,2008-03-09,13.6,0.0,20817.37\nR003,2008-03-10,12.3,1.7,30411.91\nR003,2008-03-11,5.9,16.1,23853.41\nR003,2008-03-12,5.6,0.0,39600.0\nR003,2008-03-13,8.2,0.0,28513.17\nR003,2008-03-14,9.2,0.9,39600.0\nR003,2008-03-15,5.2,10.7,19222.98\nR003,2008-03-16,2.2,15.5,19427.65\nR003,2008-03-17,5.9,0.0,39600.0\nR003,2008-03-18,11.4,0.6,39979.45\nR003,2008-03-19,8.5,17.8,5716.15\nR003,2008-03-20,6.7,15.4,3589.09\nR003,2008-03-21,8.0,26.1,6249.35\nR003,2008-03-22,7.9,6.1,32763.33\nR003,2008-03-23,12.6,0.0,39708.47\nR003,2008-03-24,19.1,0.6,34461.58\nR003,2008-03-25,17.5,9.2,42142.28\nR003,2008-03-26,8.2,13.9,41060.56\nR003,2008-03-27,6.6,1.9,37138.63\nR003,2008-03-28,10.3,0.0,42472.22\nR003,2008-03-29,11.4,2.7,35780.59\nR003,2008-03-30,12.4,2.0,18000.0\nR003,2008-03-31,11.1,6.6,15363.09\nR003,2008-04-01,9.5,10.1,28316.49\nR003,2008-04-02,6.3,20.6,0.0\nR003,2008-04-03,7.3,0.7,42227.16\nR003,2008-04-04,8.2,0.0,43649.52\nR003,2008-04-05,8.3,3.1,42654.56\nR003,2008-04-06,10.2,0.1,33879.53\nR003,2008-04-07,12.7,5.3,38875.65\nR003,2008-04-08,10.9,11.4,37052.37\nR003,2008-04-09,10.1,1.1,37102.51\nR003,2008-04-10,11.9,1.3,44094.73\nR003,2008-04-11,13.0,0.0,41891.89\nR003,2008-04-12,13.2,1.3,37940.76\nR003,2008-04-13,15.6,0.3,40525.31\nR003,2008-04-14,18.0,0.2,42296.87\nR003,2008-04-15,20.3,0.0,43200.0\nR003,2008-04-16,18.3,3.4,40697.41\nR003,2008-04-17,9.5,24.2,9473.06\nR003,2008-04-18,10.6,0.0,46344.45\nR003,2008-04-19,12.6,0.0,33010.36\nR003,2008-04-20,12.7,0.8,46800.0\nR003,2008-04-21,14.1,1.6,46800.0\nR003,2008-04-22,15.1,0.0,46866.53\nR003,2008-04-23,20.1,2.1,19406.32\nR003,2008-04-24,16.7,3.8,24674.07\nR003,2008-04-25,13.8,16.7,13380.53\nR003,2008-04-26,16.1,1.5,37057.7\nR003,2008-04-27,18.6,5.1,10547.17\nR003,2008-04-28,16.4,1.9,45943.71\nR003,2008-04-29,15.0,14.3,24755.11\nR003,2008-04-30,12.8,16.3,23270.8\nR003,2008-05-01,12.5,4.1,32917.4\nR003,2008-05-02,12.0,6.2,39285.59\nR003,2008-05-03,12.1,7.4,30195.26\nR003,2008-05-04,12.1,7.9,38499.75\nR003,2008-05-05,10.6,13.0,12315.7\nR003,2008-05-06,11.0,8.6,18261.39\nR003,2008-05-07,9.5,9.7,31045.67\nR003,2008-05-08,8.9,3.6,47929.12\nR003,2008-05-09,9.0,5.3,29943.42\nR003,2008-05-10,8.0,13.2,0.0\nR003,2008-05-11,8.9,6.5,39927.71\nR003,2008-05-12,8.8,3.8,45785.46\nR003,2008-05-13,8.0,9.5,34469.66\nR003,2008-05-14,11.1,2.1,46641.92\nR003,2008-05-15,13.5,0.0,50400.0\nR003,2008-05-16,13.8,3.8,43705.09\nR003,2008-05-17,14.2,3.3,46800.0\nR003,2008-05-18,15.8,2.5,48656.77\nR003,2008-05-19,16.2,9.2,23260.5\nR003,2008-05-20,17.2,1.9,46477.55\nR003,2008-05-21,18.6,1.0,48410.32\nR003,2008-05-22,20.1,0.1,50629.2\nR003,2008-05-23,21.9,0.1,50959.35\nR003,2008-05-24,21.3,2.0,50897.92\nR003,2008-05-25,17.7,24.7,5819.59\nR003,2008-05-26,16.0,31.2,21346.08\nR003,2008-05-27,15.0,1.4,47812.78\nR003,2008-05-28,13.8,5.6,18452.63\nR003,2008-05-29,11.9,29.3,18377.18\nR003,2008-05-30,11.3,5.2,39031.58\nR003,2008-05-31,12.7,0.0,50769.3\nR003,2008-06-01,15.2,0.0,51836.01\nR003,2008-06-02,13.7,1.2,36691.08\nR003,2008-06-03,12.2,3.9,44502.23\nR003,2008-06-04,14.4,0.0,50502.49\nR003,2008-06-05,19.6,0.0,52859.24\nR003,2008-06-06,18.7,5.8,32770.51\nR003,2008-06-07,18.5,2.5,49975.12\nR003,2008-06-08,17.2,8.9,35494.85\nR003,2008-06-09,16.9,10.4,38797.22\nR003,2008-06-10,18.5,3.2,45725.47\nR003,2008-06-11,17.3,14.6,1122.81\nR003,2008-06-12,16.6,21.6,8909.4\nR003,2008-06-13,17.9,5.2,40135.58\nR003,2008-06-14,19.2,1.4,53105.3\nR003,2008-06-15,19.8,5.3,53094.39\nR003,2008-06-16,19.4,1.2,48113.87\nR003,2008-06-17,18.9,1.4,42399.98\nR003,2008-06-18,18.4,6.4,43059.35\nR003,2008-06-19,20.0,0.7,50363.42\nR003,2008-06-20,20.7,2.3,50736.56\nR003,2008-06-21,19.3,11.0,42488.88\nR003,2008-06-22,20.1,0.4,49976.92\nR003,2008-06-23,18.7,1.9,51804.73\nR003,2008-06-24,17.2,0.0,50217.16\nR003,2008-06-25,19.6,0.5,51002.18\nR003,2008-06-26,20.7,0.9,51265.87\nR003,2008-06-27,22.3,0.5,50629.19\nR003,2008-06-28,23.1,0.8,52283.97\nR003,2008-06-29,21.2,15.1,48741.54\nR003,2008-06-30,20.3,2.4,51135.01\nR003,2008-07-01,18.3,1.5,49888.64\nR003,2008-07-02,17.3,3.4,51687.83\nR003,2008-07-03,16.9,8.7,44151.76\nR003,2008-07-04,17.6,1.8,51421.21\nR003,2008-07-05,19.9,0.2,51685.63\nR003,2008-07-06,20.2,0.1,49359.17\nR003,2008-07-07,18.1,6.9,44559.18\nR003,2008-07-08,18.9,1.8,49047.35\nR003,2008-07-09,20.0,0.3,52627.92\nR003,2008-07-10,19.9,0.5,52613.82\nR003,2008-07-11,20.5,3.2,48631.3\nR003,2008-07-12,21.1,0.0,51137.49\nR003,2008-07-13,23.4,0.0,52721.01\nR003,2008-07-14,22.9,9.6,48628.87\nR003,2008-07-15,21.2,15.9,14719.37\nR003,2008-07-16,22.5,1.2,52401.29\nR003,2008-07-17,21.1,12.9,47816.74\nR003,2008-07-18,20.2,4.7,45853.26\nR003,2008-07-19,22.2,0.0,51738.97\nR003,2008-07-20,23.4,1.4,50275.0\nR003,2008-07-21,25.0,0.0,47844.05\nR003,2008-07-22,26.1,0.8,49652.65\nR003,2008-07-23,26.1,1.0,51593.0\nR003,2008-07-24,26.8,0.5,51555.9\nR003,2008-07-25,27.7,0.0,51518.91\nR003,2008-07-26,25.5,5.7,49140.14\nR003,2008-07-27,23.4,20.0,44126.25\nR003,2008-07-28,22.0,0.0,49589.9\nR003,2008-07-29,21.6,0.5,48253.74\nR003,2008-07-30,21.6,4.6,46732.33\nR003,2008-07-31,22.4,1.1,45479.46\nR003,2008-08-01,21.6,2.2,19985.39\nR003,2008-08-02,23.4,0.3,42459.65\nR003,2008-08-03,23.8,0.7,40893.89\nR003,2008-08-04,23.0,13.1,41451.32\nR003,2008-08-05,22.6,6.0,35187.99\nR003,2008-08-06,22.4,3.2,46800.0\nR003,2008-08-07,22.2,0.0,46943.65\nR003,2008-08-08,21.5,0.6,46973.16\nR003,2008-08-09,22.8,0.0,47002.84\nR003,2008-08-10,23.3,1.9,47707.1\nR003,2008-08-11,22.9,1.8,47062.79\nR003,2008-08-12,23.2,0.6,47092.99\nR003,2008-08-13,22.3,0.0,47123.32\nR003,2008-08-14,21.7,13.6,46508.57\nR003,2008-08-15,22.9,2.0,46851.01\nR003,2008-08-16,23.3,1.0,46800.0\nR003,2008-08-17,24.5,3.5,46800.0\nR003,2008-08-18,25.3,0.0,46945.31\nR003,2008-08-19,25.1,1.3,46800.0\nR003,2008-08-20,23.3,7.8,46800.0\nR003,2008-08-21,21.5,10.3,46800.0\nR003,2008-08-22,23.5,0.7,46800.0\nR003,2008-08-23,24.1,0.6,46800.0\nR003,2008-08-24,26.3,0.0,46800.0\nR003,2008-08-25,25.6,3.1,46800.0\nR003,2008-08-26,24.5,2.0,46575.6\nR003,2008-08-27,23.3,0.0,46701.89\nR003,2008-08-28,21.8,0.0,46599.26\nR003,2008-08-29,23.4,0.0,46495.56\nR003,2008-08-30,24.0,0.0,46115.88\nR003,2008-08-31,20.3,12.0,37134.92\nR003,2008-09-01,17.8,0.0,45567.43\nR003,2008-09-02,18.5,0.0,45616.24\nR003,2008-09-03,20.7,0.4,44703.55\nR003,2008-09-04,21.9,0.7,44113.7\nR003,2008-09-05,20.4,7.5,40460.03\nR003,2008-09-06,19.8,6.6,36250.86\nR003,2008-09-07,19.6,4.5,35928.57\nR003,2008-09-08,19.4,4.1,39193.51\nR003,2008-09-09,18.3,11.3,33660.23\nR003,2008-09-10,17.4,5.8,24061.24\nR003,2008-09-11,16.0,2.8,37918.06\nR003,2008-09-12,17.0,0.0,42846.09\nR003,2008-09-13,19.4,0.0,43152.05\nR003,2008-09-14,18.6,3.9,36300.45\nR003,2008-09-15,17.9,0.0,39428.13\nR003,2008-09-16,19.2,0.0,42950.11\nR003,2008-09-17,22.3,0.2,42882.54\nR003,2008-09-18,21.2,11.3,40939.04\nR003,2008-09-19,17.9,4.2,31258.34\nR003,2008-09-20,17.3,3.4,30474.07\nR003,2008-09-21,19.1,0.5,35141.59\nR003,2008-09-22,19.3,11.9,24513.49\nR003,2008-09-23,16.8,13.0,24857.98\nR003,2008-09-24,14.4,11.0,39088.59\nR003,2008-09-25,14.5,0.0,38697.7\nR003,2008-09-26,14.6,0.0,36461.89\nR003,2008-09-27,16.2,0.0,38965.65\nR003,2008-09-28,15.6,0.6,34761.22\nR003,2008-09-29,15.2,0.3,35014.66\nR003,2008-09-30,14.0,7.0,5067.05\nR003,2008-10-01,14.1,4.1,29933.7\nR003,2008-10-02,13.9,0.0,39195.88\nR003,2008-10-03,15.4,0.0,39924.61\nR003,2008-10-04,16.5,0.0,38926.46\nR003,2008-10-05,18.2,0.0,38110.32\nR003,2008-10-06,17.7,2.6,35239.15\nR003,2008-10-07,14.7,5.1,20480.33\nR003,2008-10-08,12.9,7.5,30840.86\nR003,2008-10-09,12.3,3.1,32824.6\nR003,2008-10-10,12.4,0.0,36124.52\nR003,2008-10-11,12.3,0.0,33764.68\nR003,2008-10-12,12.2,0.0,36558.99\nR003,2008-10-13,13.0,0.0,35633.19\nR003,2008-10-14,15.6,0.0,36341.04\nR003,2008-10-15,18.2,0.8,34268.36\nR003,2008-10-16,16.1,23.6,0.0\nR003,2008-10-17,14.3,4.3,32285.17\nR003,2008-10-18,13.1,0.0,36000.0\nR003,2008-10-19,12.3,9.5,34711.96\nR003,2008-10-20,11.3,0.0,36000.0\nR003,2008-10-21,11.0,3.1,212.58\nR003,2008-10-22,11.0,0.1,34156.74\nR003,2008-10-23,12.1,0.0,36000.0\nR003,2008-10-24,13.6,0.0,36000.0\nR003,2008-10-25,12.9,5.3,18784.83\nR003,2008-10-26,13.3,3.6,23406.87\nR003,2008-10-27,13.3,0.0,36000.0\nR003,2008-10-28,13.2,2.5,34207.73\nR003,2008-10-29,12.6,0.0,33964.07\nR003,2008-10-30,12.8,0.0,36000.0\nR003,2008-10-31,13.3,0.0,36000.0\nR003,2008-11-01,12.0,0.0,35992.79\nR003,2008-11-02,12.5,0.0,35912.63\nR003,2008-11-03,12.5,0.0,35832.38\nR003,2008-11-04,11.2,0.0,30954.89\nR003,2008-11-05,11.7,0.0,35242.59\nR003,2008-11-06,10.6,0.3,29168.36\nR003,2008-11-07,9.2,4.3,29626.73\nR003,2008-11-08,8.5,0.0,33460.04\nR003,2008-11-09,8.3,0.0,35019.93\nR003,2008-11-10,9.2,0.0,34874.01\nR003,2008-11-11,9.2,0.0,34730.99\nR003,2008-11-12,8.4,0.0,34590.9\nR003,2008-11-13,7.9,0.0,31438.4\nR003,2008-11-14,8.4,0.0,30010.18\nR003,2008-11-15,8.6,0.0,28731.06\nR003,2008-11-16,8.5,0.4,21375.69\nR003,2008-11-17,9.0,0.0,31353.92\nR003,2008-11-18,8.7,2.9,28581.47\nR003,2008-11-19,8.6,6.2,9818.78\nR003,2008-11-20,8.9,29.5,0.0\nR003,2008-11-21,9.0,10.8,19066.3\nR003,2008-11-22,10.7,0.0,31376.26\nR003,2008-11-23,9.6,30.4,0.0\nR003,2008-11-24,8.2,37.5,53.95\nR003,2008-11-25,10.1,1.8,17116.71\nR003,2008-11-26,10.2,0.0,30606.72\nR003,2008-11-27,8.6,4.1,28756.12\nR003,2008-11-28,8.1,5.2,19202.88\nR003,2008-11-29,7.6,0.0,30296.42\nR003,2008-11-30,7.6,0.0,28800.0\nR003,2008-12-01,7.9,1.4,26331.98\nR003,2008-12-02,6.4,0.0,30729.0\nR003,2008-12-03,6.7,0.0,30743.93\nR003,2008-12-04,9.5,0.0,30374.75\nR003,2008-12-05,12.4,0.0,27072.01\nR003,2008-12-06,14.3,0.0,25259.07\nR003,2008-12-07,12.3,0.0,28864.2\nR003,2008-12-08,12.7,0.0,24301.14\nR003,2008-12-09,7.2,28.5,0.0\nR003,2008-12-10,3.7,14.3,6632.47\nR003,2008-12-11,-1.1,0.0,30420.71\nR003,2008-12-12,0.6,0.0,30033.34\nR003,2008-12-13,2.3,0.0,30031.92\nR003,2008-12-14,1.9,0.0,30029.11\nR003,2008-12-15,1.0,0.0,30414.35\nR003,2008-12-16,1.9,0.0,29629.35\nR003,2008-12-17,1.8,0.0,29231.72\nR003,2008-12-18,2.8,0.0,30003.76\nR003,2008-12-19,4.0,0.0,28731.84\nR003,2008-12-20,4.3,0.1,26955.16\nR003,2008-12-21,5.2,3.2,4189.47\nR003,2008-12-22,5.7,2.3,24001.35\nR003,2008-12-23,5.5,7.5,0.0\nR003,2008-12-24,5.7,0.7,13852.91\nR003,2008-12-25,1.8,15.3,1622.28\nR003,2008-12-26,-3.0,0.0,31328.93\nR003,2008-12-27,-0.8,4.3,3183.96\nR003,2008-12-28,-1.2,2.4,15712.79\nR003,2008-12-29,-3.8,3.1,26607.83\nR003,2008-12-30,-6.3,2.4,24744.07\nR003,2008-12-31,-6.1,0.0,31567.96\nR003,2009-01-01,-5.8,0.0,31622.04\nR003,2009-01-02,-2.0,3.7,25003.57\nR003,2009-01-03,0.1,30.9,0.0\nR003,2009-01-04,0.2,4.7,23662.24\nR003,2009-01-05,3.9,31.7,0.0\nR003,2009-01-06,2.2,3.9,6248.31\nR003,2009-01-07,1.8,9.5,343.86\nR003,2009-01-08,0.2,0.9,31575.39\nR003,2009-01-09,-4.8,15.5,26242.23\nR003,2009-01-10,-4.6,1.6,21180.14\nR003,2009-01-11,-2.2,1.4,26513.16\nR003,2009-01-12,-3.1,0.0,32037.41\nR003,2009-01-13,-1.4,0.0,32377.24\nR003,2009-01-14,0.3,0.0,32400.0\nR003,2009-01-15,4.5,0.1,15269.15\nR003,2009-01-16,7.4,0.0,31837.76\nR003,2009-01-17,7.1,24.3,0.0\nR003,2009-01-18,1.4,10.6,12564.03\nR003,2009-01-19,1.3,0.4,21959.37\nR003,2009-01-20,2.5,5.5,215.46\nR003,2009-01-21,1.4,0.0,32400.0\nR003,2009-01-22,1.7,0.0,32400.0\nR003,2009-01-23,3.5,0.0,32400.0\nR003,2009-01-24,9.1,0.0,17152.6\nR003,2009-01-25,7.3,0.4,32400.0\nR003,2009-01-26,6.9,12.2,1915.94\nR003,2009-01-27,6.9,4.6,25767.47\nR003,2009-01-28,5.7,1.2,32400.0\nR003,2009-01-29,6.6,0.0,28993.45\nR003,2009-01-30,7.2,4.8,21925.11\nR003,2009-01-31,5.9,1.9,20492.93\nR003,2009-02-01,4.4,12.7,14874.63\nR003,2009-02-02,2.3,2.6,27589.42\nR003,2009-02-03,1.1,0.4,32622.27\nR003,2009-02-04,1.3,0.0,31809.77\nR003,2009-02-05,8.1,0.0,32533.61\nR003,2009-02-06,8.0,4.0,23729.64\nR003,2009-02-07,7.3,3.2,28800.0\nR003,2009-02-08,7.8,0.0,30680.64\nR003,2009-02-09,8.1,8.6,1815.3\nR003,2009-02-10,8.0,16.3,17088.35\nR003,2009-02-11,5.5,3.2,27661.61\nR003,2009-02-12,6.6,0.4,32450.35\nR003,2009-02-13,9.7,2.0,36223.18\nR003,2009-02-14,10.7,0.0,36256.81\nR003,2009-02-15,10.6,0.0,37294.31\nR003,2009-02-16,5.6,16.1,18674.21\nR003,2009-02-17,1.3,19.6,10428.25\nR003,2009-02-18,-2.7,0.0,34396.42\nR003,2009-02-19,-2.2,0.0,37083.66\nR003,2009-02-20,-0.3,0.0,35092.47\nR003,2009-02-21,2.3,0.4,24684.1\nR003,2009-02-22,1.2,2.2,26912.88\nR003,2009-02-23,1.3,1.5,36196.6\nR003,2009-02-24,3.5,0.0,23649.62\nR003,2009-02-25,4.3,9.3,1128.37\nR003,2009-02-26,3.8,0.1,38254.63\nR003,2009-02-27,4.5,0.0,39367.64\nR003,2009-02-28,5.3,9.9,0.0\nR003,2009-03-01,1.4,10.1,28164.87\nR003,2009-03-02,-2.0,13.9,31500.68\nR003,2009-03-03,-4.4,4.4,27534.63\nR003,2009-03-04,-1.3,7.0,1987.48\nR003,2009-03-05,-0.6,0.0,39600.0\nR003,2009-03-06,4.1,2.0,21377.48\nR003,2009-03-07,9.4,0.0,32734.48\nR003,2009-03-08,12.4,5.0,17807.91\nR003,2009-03-09,8.9,16.8,0.0\nR003,2009-03-10,8.7,2.9,34112.42\nR003,2009-03-11,6.5,18.1,8612.69\nR003,2009-03-12,5.6,2.9,37951.31\nR003,2009-03-13,9.6,5.3,21512.75\nR003,2009-03-14,7.0,21.5,1482.94\nR003,2009-03-15,1.9,6.8,35538.02\nR003,2009-03-16,0.7,4.0,37390.5\nR003,2009-03-17,2.7,2.6,39600.0\nR003,2009-03-18,3.1,0.1,32612.88\nR003,2009-03-19,5.5,6.1,27995.13\nR003,2009-03-20,2.7,19.3,25035.84\nR003,2009-03-21,-0.3,1.9,39944.94\nR003,2009-03-22,1.1,0.1,41044.44\nR003,2009-03-23,5.5,6.7,13936.97\nR003,2009-03-24,4.9,9.1,1729.32\nR003,2009-03-25,6.4,0.0,42122.51\nR003,2009-03-26,9.1,0.4,41397.44\nR003,2009-03-27,9.0,4.1,40259.19\nR003,2009-03-28,8.0,3.5,23446.82\nR003,2009-03-29,8.1,0.6,37035.43\nR003,2009-03-30,10.9,0.4,44052.25\nR003,2009-03-31,12.7,0.1,39303.9\nR003,2009-04-01,11.7,1.2,40551.48\nR003,2009-04-02,10.3,7.5,29467.22\nR003,2009-04-03,7.3,0.2,44851.1\nR003,2009-04-04,7.7,0.0,45687.64\nR003,2009-04-05,9.4,0.0,45341.82\nR003,2009-04-06,10.7,0.0,45629.7\nR003,2009-04-07,10.8,2.9,22100.18\nR003,2009-04-08,10.2,11.0,29777.01\nR003,2009-04-09,7.0,3.1,45158.06\nR003,2009-04-10,4.9,3.3,42708.88\nR003,2009-04-11,4.1,0.1,46380.28\nR003,2009-04-12,5.6,0.4,46451.62\nR003,2009-04-13,7.3,0.0,46522.91\nR003,2009-04-14,11.3,0.1,39600.0\nR003,2009-04-15,10.0,4.5,4982.13\nR003,2009-04-16,8.4,10.9,154.77\nR003,2009-04-17,8.5,5.2,31420.87\nR003,2009-04-18,8.9,3.0,25963.57\nR003,2009-04-19,9.8,0.4,44853.42\nR003,2009-04-20,9.5,1.3,46800.0\nR003,2009-04-21,8.7,0.0,46800.0\nR003,2009-04-22,6.2,0.7,44975.32\nR003,2009-04-23,7.3,0.0,46699.19\nR003,2009-04-24,8.4,4.3,38844.88\nR003,2009-04-25,9.9,0.9,44400.86\nR003,2009-04-26,10.2,0.6,44302.98\nR003,2009-04-27,12.0,0.4,47659.82\nR003,2009-04-28,13.0,6.0,39793.56\nR003,2009-04-29,12.7,5.8,40699.09\nR003,2009-04-30,12.4,5.5,37003.14\nR003,2009-05-01,12.4,9.8,45933.12\nR003,2009-05-02,13.7,0.9,44162.83\nR003,2009-05-03,11.7,11.4,25831.98\nR003,2009-05-04,13.4,1.2,44675.71\nR003,2009-05-05,13.2,10.7,18771.44\nR003,2009-05-06,12.0,9.0,11963.36\nR003,2009-05-07,12.1,3.7,38208.07\nR003,2009-05-08,10.9,4.6,46830.43\nR003,2009-05-09,8.6,12.9,30080.03\nR003,2009-05-10,9.0,8.2,30283.09\nR003,2009-05-11,9.4,9.9,26029.72\nR003,2009-05-12,10.7,1.9,46580.1\nR003,2009-05-13,13.0,0.9,49514.31\nR003,2009-05-14,13.7,12.2,17720.1\nR003,2009-05-15,14.3,1.9,42212.05\nR003,2009-05-16,16.9,1.0,49220.76\nR003,2009-05-17,17.8,2.4,49101.15\nR003,2009-05-18,18.9,0.1,50400.0\nR003,2009-05-19,16.8,0.3,50300.65\nR003,2009-05-20,18.1,0.0,47922.58\nR003,2009-05-21,19.9,0.2,50164.31\nR003,2009-05-22,19.2,1.4,39214.83\nR003,2009-05-23,18.7,4.3,46487.49\nR003,2009-05-24,14.8,16.0,34549.56\nR003,2009-05-25,11.3,2.1,48968.14\nR003,2009-05-26,11.7,3.2,46008.98\nR003,2009-05-27,11.0,8.6,33637.64\nR003,2009-05-28,11.8,7.9,36247.88\nR003,2009-05-29,11.8,5.0,43185.19\nR003,2009-05-30,14.9,0.0,51159.42\nR003,2009-05-31,16.3,0.3,49157.83\nR003,2009-06-01,17.9,0.0,49819.39\nR003,2009-06-02,19.6,2.4,51824.94\nR003,2009-06-03,22.0,0.0,48053.34\nR003,2009-06-04,23.9,0.2,48998.89\nR003,2009-06-05,18.7,6.1,36517.66\nR003,2009-06-06,17.4,8.5,48637.28\nR003,2009-06-07,18.8,0.1,50996.24\nR003,2009-06-08,20.7,0.7,48677.34\nR003,2009-06-09,21.2,3.0,47646.29\nR003,2009-06-10,20.2,3.3,45039.43\nR003,2009-06-11,20.1,7.6,41590.33\nR003,2009-06-12,22.2,2.0,53129.27\nR003,2009-06-13,22.4,4.7,51648.84\nR003,2009-06-14,19.6,21.8,31166.71\nR003,2009-06-15,17.9,38.0,32984.79\nR003,2009-06-16,17.8,3.0,48675.19\nR003,2009-06-17,18.8,4.9,49852.71\nR003,2009-06-18,18.3,20.9,22275.39\nR003,2009-06-19,18.7,1.2,43553.48\nR003,2009-06-20,21.0,0.0,50777.74\nR003,2009-06-21,23.0,3.3,51207.62\nR003,2009-06-22,21.5,9.8,52483.07\nR003,2009-06-23,22.9,1.3,53951.83\nR003,2009-06-24,24.4,0.0,53958.07\nR003,2009-06-25,25.0,0.5,53962.35\nR003,2009-06-26,24.2,2.3,52655.61\nR003,2009-06-27,21.7,3.2,51924.88\nR003,2009-06-28,22.3,5.4,48743.19\nR003,2009-06-29,22.7,7.2,53382.11\nR003,2009-06-30,22.4,5.9,52262.48\nR003,2009-07-01,21.4,4.8,52283.57\nR003,2009-07-02,21.6,0.8,53015.37\nR003,2009-07-03,20.8,6.7,50928.85\nR003,2009-07-04,19.6,8.7,47062.73\nR003,2009-07-05,19.8,5.0,51653.4\nR003,2009-07-06,19.4,0.6,50640.05\nR003,2009-07-07,21.6,0.3,52612.56\nR003,2009-07-08,23.2,1.5,53398.98\nR003,2009-07-09,22.6,6.5,53798.92\nR003,2009-07-10,21.9,6.7,46111.96\nR003,2009-07-11,22.1,3.1,51831.36\nR003,2009-07-12,24.3,0.4,53647.34\nR003,2009-07-13,25.3,5.1,52719.31\nR003,2009-07-14,22.5,30.6,34911.66\nR003,2009-07-15,22.2,12.1,34765.64\nR003,2009-07-16,22.8,8.7,37034.21\nR003,2009-07-17,23.6,7.4,51391.61\nR003,2009-07-18,22.6,17.6,35200.6\nR003,2009-07-19,23.1,8.6,48711.69\nR003,2009-07-20,22.4,14.7,42513.84\nR003,2009-07-21,22.7,5.6,44075.08\nR003,2009-07-22,22.7,1.8,46636.1\nR003,2009-07-23,23.5,0.3,46020.3\nR003,2009-07-24,23.1,1.0,51559.24\nR003,2009-07-25,23.7,0.0,51872.75\nR003,2009-07-26,23.5,1.2,50525.48\nR003,2009-07-27,21.6,12.8,44763.44\nR003,2009-07-28,18.4,11.7,27955.98\nR003,2009-07-29,16.0,9.7,33926.97\nR003,2009-07-30,19.2,1.5,40147.79\nR003,2009-07-31,20.1,1.7,43971.37\nR003,2009-08-01,19.7,10.8,35524.47\nR003,2009-08-02,19.3,12.2,33997.97\nR003,2009-08-03,20.6,8.6,45739.89\nR003,2009-08-04,22.1,0.7,46482.45\nR003,2009-08-05,23.2,0.5,47901.87\nR003,2009-08-06,22.7,1.6,47928.9\nR003,2009-08-07,22.2,1.3,40505.17\nR003,2009-08-08,19.3,18.0,41951.63\nR003,2009-08-09,17.5,4.0,43985.25\nR003,2009-08-10,16.0,3.8,45107.88\nR003,2009-08-11,15.1,1.1,46398.19\nR003,2009-08-12,14.7,2.2,47087.5\nR003,2009-08-13,17.1,0.0,47117.89\nR003,2009-08-14,19.2,0.0,46960.42\nR003,2009-08-15,21.0,0.2,44864.44\nR003,2009-08-16,22.1,0.1,46291.15\nR003,2009-08-17,22.7,0.1,45870.62\nR003,2009-08-18,22.3,9.0,42433.55\nR003,2009-08-19,21.6,6.0,44014.67\nR003,2009-08-20,20.5,4.7,44671.34\nR003,2009-08-21,16.8,0.1,45486.77\nR003,2009-08-22,14.4,0.0,46800.0\nR003,2009-08-23,14.0,4.3,30341.27\nR003,2009-08-24,16.8,0.0,46482.8\nR003,2009-08-25,19.5,0.8,46800.0\nR003,2009-08-26,18.8,1.0,44930.78\nR003,2009-08-27,18.3,0.0,45932.31\nR003,2009-08-28,17.8,0.0,45726.4\nR003,2009-08-29,19.1,0.0,46515.38\nR003,2009-08-30,20.0,0.1,46410.57\nR003,2009-08-31,20.0,0.4,44866.37\nR003,2009-09-01,20.0,3.8,45558.14\nR003,2009-09-02,19.0,13.0,25895.12\nR003,2009-09-03,18.3,6.1,40794.39\nR003,2009-09-04,18.6,3.7,41059.95\nR003,2009-09-05,18.5,4.4,42118.0\nR003,2009-09-06,18.4,0.0,42028.19\nR003,2009-09-07,19.8,0.5,43615.62\nR003,2009-09-08,20.7,0.7,43667.17\nR003,2009-09-09,21.5,0.0,41188.24\nR003,2009-09-10,20.1,6.4,29812.32\nR003,2009-09-11,18.6,3.8,37685.6\nR003,2009-09-12,18.7,0.0,41240.63\nR003,2009-09-13,18.4,20.8,14082.14\nR003,2009-09-14,18.7,7.4,5433.56\nR003,2009-09-15,18.1,3.4,35148.33\nR003,2009-09-16,16.5,4.6,33499.34\nR003,2009-09-17,15.9,7.9,29296.56\nR003,2009-09-18,15.6,0.7,39586.95\nR003,2009-09-19,16.0,1.8,36328.83\nR003,2009-09-20,15.1,8.9,2974.18\nR003,2009-09-21,15.6,6.0,28130.76\nR003,2009-09-22,15.4,17.5,12752.15\nR003,2009-09-23,16.1,3.0,34499.11\nR003,2009-09-24,15.2,0.0,41694.38\nR003,2009-09-25,15.8,0.2,39648.82\nR003,2009-09-26,14.9,1.2,38502.78\nR003,2009-09-27,10.8,5.1,35638.52\nR003,2009-09-28,9.7,0.6,14608.17\nR003,2009-09-29,11.0,0.3,0.0\nR003,2009-09-30,13.6,0.0,38772.64\nR003,2009-10-01,11.6,3.7,30669.46\nR003,2009-10-02,14.2,0.0,40018.54\nR003,2009-10-03,18.6,0.0,38596.06\nR003,2009-10-04,20.3,0.0,39555.2\nR003,2009-10-05,19.1,0.0,38678.44\nR003,2009-10-06,16.6,1.5,34628.79\nR003,2009-10-07,15.8,2.6,14588.05\nR003,2009-10-08,15.8,0.9,31832.72\nR003,2009-10-09,16.1,1.1,36383.97\nR003,2009-10-10,15.5,2.9,36439.18\nR003,2009-10-11,16.0,0.0,36819.19\nR003,2009-10-12,18.0,0.0,36874.2\nR003,2009-10-13,19.3,0.0,36276.53\nR003,2009-10-14,20.1,0.0,36329.56\nR003,2009-10-15,18.6,0.0,36054.82\nR003,2009-10-16,18.2,0.0,36106.04\nR003,2009-10-17,19.3,0.0,36000.0\nR003,2009-10-18,18.8,0.0,36000.0\nR003,2009-10-19,19.2,0.0,36000.0\nR003,2009-10-20,19.0,0.0,36000.0\nR003,2009-10-21,18.9,0.0,36000.0\nR003,2009-10-22,16.5,1.0,25982.9\nR003,2009-10-23,14.7,0.4,34246.93\nR003,2009-10-24,15.9,0.0,36000.0\nR003,2009-10-25,15.7,0.0,36000.0\nR003,2009-10-26,14.2,0.0,36000.0\nR003,2009-10-27,14.4,0.0,36000.0\nR003,2009-10-28,13.9,0.0,36000.0\nR003,2009-10-29,10.9,12.4,5785.43\nR003,2009-10-30,10.0,0.0,15274.39\nR003,2009-10-31,10.4,0.2,29306.89\nR003,2009-11-01,9.2,10.8,15651.54\nR003,2009-11-02,8.5,21.6,0.0\nR003,2009-11-03,6.0,17.9,2273.34\nR003,2009-11-04,6.1,3.5,4356.04\nR003,2009-11-05,10.3,21.5,5056.51\nR003,2009-11-06,10.2,0.0,35514.65\nR003,2009-11-07,12.3,0.0,29581.32\nR003,2009-11-08,14.4,0.0,34887.59\nR003,2009-11-09,13.9,0.0,33704.93\nR003,2009-11-10,13.2,0.0,34921.99\nR003,2009-11-11,13.1,0.0,34125.35\nR003,2009-11-12,13.5,0.0,28865.35\nR003,2009-11-13,11.8,1.0,28720.52\nR003,2009-11-14,10.0,19.0,0.0\nR003,2009-11-15,8.2,11.4,8059.52\nR003,2009-11-16,7.4,4.2,30828.9\nR003,2009-11-17,7.4,0.0,14143.51\nR003,2009-11-18,6.5,0.0,32153.27\nR003,2009-11-19,7.1,0.9,16861.2\nR003,2009-11-20,5.5,23.5,18873.9\nR003,2009-11-21,3.6,0.0,30091.81\nR003,2009-11-22,3.5,10.9,0.0\nR003,2009-11-23,5.2,4.8,26921.99\nR003,2009-11-24,6.7,0.0,29635.58\nR003,2009-11-25,7.3,0.2,17238.25\nR003,2009-11-26,4.3,9.7,30976.74\nR003,2009-11-27,2.6,0.0,30947.03\nR003,2009-11-28,5.3,0.0,29516.06\nR003,2009-11-29,6.1,0.0,25880.01\nR003,2009-11-30,6.3,0.0,12688.0\nR003,2009-12-01,5.6,0.0,30325.68\nR003,2009-12-02,5.5,0.0,30722.13\nR003,2009-12-03,7.1,0.0,30737.4\nR003,2009-12-04,7.5,0.0,21600.0\nR003,2009-12-05,8.9,0.0,26953.61\nR003,2009-12-06,7.5,3.2,13108.31\nR003,2009-12-07,7.7,2.8,0.0\nR003,2009-12-08,6.3,3.3,3109.3\nR003,2009-12-09,5.7,8.7,4743.32\nR003,2009-12-10,3.6,0.4,28870.37\nR003,2009-12-11,3.6,0.0,25113.59\nR003,2009-12-12,8.8,1.7,17550.86\nR003,2009-12-13,8.4,15.4,1049.88\nR003,2009-12-14,2.8,24.0,178.11\nR003,2009-12-15,-0.6,0.0,28860.08\nR003,2009-12-16,5.0,0.0,11559.27\nR003,2009-12-17,8.0,0.0,22525.52\nR003,2009-12-18,6.5,3.0,146.89\nR003,2009-12-19,4.3,11.5,4736.03\nR003,2009-12-20,6.2,0.9,28814.88\nR003,2009-12-21,9.8,16.3,0.0\nR003,2009-12-22,3.5,5.5,30742.02\nR003,2009-12-23,5.6,0.1,23362.56\nR003,2009-12-24,4.8,0.9,842.76\nR003,2009-12-25,8.5,0.0,29606.58\nR003,2009-12-26,7.2,9.1,0.0\nR003,2009-12-27,8.8,3.9,29080.62\nR003,2009-12-28,9.9,15.2,2780.39\nR003,2009-12-29,7.2,37.4,0.0\nR003,2009-12-30,2.1,3.6,31509.84\nR003,2009-12-31,2.1,0.0,31560.43\nR003,2010-01-01,7.3,10.6,7200.0\nR003,2010-01-02,9.1,18.7,3311.02\nR003,2010-01-03,5.0,29.5,2031.09\nR003,2010-01-04,2.0,23.1,0.0\nR003,2010-01-05,-2.9,0.1,31834.61\nR003,2010-01-06,-0.9,0.0,28800.0\nR003,2010-01-07,7.0,0.0,31872.43\nR003,2010-01-08,6.8,0.2,22198.55\nR003,2010-01-09,5.8,0.0,26637.56\nR003,2010-01-10,8.8,0.0,30595.35\nR003,2010-01-11,9.0,3.9,21794.6\nR003,2010-01-12,4.3,4.4,29703.56\nR003,2010-01-13,5.1,0.0,22181.71\nR003,2010-01-14,5.4,6.6,878.19\nR003,2010-01-15,6.1,14.0,0.0\nR003,2010-01-16,5.4,12.3,17920.08\nR003,2010-01-17,4.4,0.0,25344.39\nR003,2010-01-18,6.6,0.8,31042.64\nR003,2010-01-19,7.5,2.8,1550.21\nR003,2010-01-20,9.0,1.0,22061.22\nR003,2010-01-21,6.6,14.3,0.0\nR003,2010-01-22,1.1,8.3,26921.1\nR003,2010-01-23,0.9,12.1,0.0\nR003,2010-01-24,2.7,12.2,85.45\nR003,2010-01-25,-0.1,9.2,2146.05\nR003,2010-01-26,-4.0,4.9,29864.23\nR003,2010-01-27,-8.6,0.0,32400.0\nR003,2010-01-28,-4.0,3.6,2409.35\nR003,2010-01-29,2.0,3.1,10361.07\nR003,2010-01-30,5.1,12.6,13543.46\nR003,2010-01-31,6.9,0.0,32414.91\nR003,2010-02-01,5.8,9.2,11410.59\nR003,2010-02-02,4.9,0.0,32778.41\nR003,2010-02-03,5.5,0.3,32400.0\nR003,2010-02-04,2.6,8.7,5325.67\nR003,2010-02-05,1.0,2.1,20871.56\nR003,2010-02-06,0.6,4.8,18105.29\nR003,2010-02-07,-1.1,0.0,32249.96\nR003,2010-02-08,1.7,0.0,22525.04\nR003,2010-02-09,4.1,0.7,14034.48\nR003,2010-02-10,4.9,1.0,14784.36\nR003,2010-02-11,5.1,0.7,32960.36\nR003,2010-02-12,8.3,4.0,25917.22\nR003,2010-02-13,9.3,0.3,33881.58\nR003,2010-02-14,8.3,5.9,20304.92\nR003,2010-02-15,12.4,7.1,16087.93\nR003,2010-02-16,6.1,17.6,33595.78\nR003,2010-02-17,5.1,0.8,34230.45\nR003,2010-02-18,7.3,7.5,989.84\nR003,2010-02-19,8.6,12.8,5170.02\nR003,2010-02-20,10.7,1.1,35232.81\nR003,2010-02-21,10.9,18.2,0.0\nR003,2010-02-22,5.5,25.7,13572.76\nR003,2010-02-23,1.9,0.0,32429.42\nR003,2010-02-24,4.3,0.0,38804.16\nR003,2010-02-25,5.6,0.0,38982.76\nR003,2010-02-26,7.2,0.0,39161.85\nR003,2010-02-27,8.3,0.1,38899.35\nR003,2010-02-28,8.3,2.6,36575.27\nR003,2010-03-01,8.5,0.0,37934.94\nR003,2010-03-02,8.3,0.0,22279.43\nR003,2010-03-03,7.5,9.9,2405.96\nR003,2010-03-04,3.9,9.3,35777.13\nR003,2010-03-05,7.0,4.8,28372.34\nR003,2010-03-06,6.9,19.8,9700.14\nR003,2010-03-07,2.2,20.1,31159.67\nR003,2010-03-08,0.6,2.5,21696.47\nR003,2010-03-09,11.0,8.7,0.0\nR003,2010-03-10,6.0,36.7,2883.69\nR003,2010-03-11,6.7,0.5,32995.67\nR003,2010-03-12,9.6,5.1,16241.75\nR003,2010-03-13,6.2,18.6,3666.97\nR003,2010-03-14,3.8,5.9,22286.38\nR003,2010-03-15,4.6,8.3,2816.96\nR003,2010-03-16,1.1,13.1,24139.66\nR003,2010-03-17,-0.7,4.8,33316.01\nR003,2010-03-18,-0.7,3.6,39600.0\nR003,2010-03-19,0.4,14.4,14203.57\nR003,2010-03-20,1.5,0.0,40373.31\nR003,2010-03-21,4.6,0.0,41235.17\nR003,2010-03-22,9.5,1.0,28367.24\nR003,2010-03-23,10.1,0.2,36688.21\nR003,2010-03-24,11.6,0.0,42644.69\nR003,2010-03-25,12.2,0.0,41846.43\nR003,2010-03-26,10.9,0.0,41102.39\nR003,2010-03-27,11.4,0.0,42540.03\nR003,2010-03-28,12.4,0.0,43153.39\nR003,2010-03-29,12.8,3.2,42431.02\nR003,2010-03-30,10.9,6.9,37025.22\nR003,2010-03-31,8.5,4.2,6240.37\nR003,2010-04-01,9.7,0.7,40526.2\nR003,2010-04-02,10.4,1.1,44565.36\nR003,2010-04-03,9.4,0.8,18014.88\nR003,2010-04-04,7.9,9.1,14123.88\nR003,2010-04-05,9.4,0.2,45349.77\nR003,2010-04-06,11.7,0.0,42081.31\nR003,2010-04-07,14.1,2.0,38885.53\nR003,2010-04-08,9.7,12.3,24304.69\nR003,2010-04-09,8.4,3.8,31223.02\nR003,2010-04-10,7.5,6.1,27385.34\nR003,2010-04-11,8.7,0.7,45159.91\nR003,2010-04-12,9.1,5.8,18049.5\nR003,2010-04-13,7.5,12.0,0.0\nR003,2010-04-14,7.1,1.0,46575.63\nR003,2010-04-15,8.5,1.6,19562.35\nR003,2010-04-16,13.8,0.0,46717.06\nR003,2010-04-17,16.8,0.3,46787.89\nR003,2010-04-18,13.5,5.3,23584.49\nR003,2010-04-19,13.0,0.0,45117.46\nR003,2010-04-20,15.7,0.0,46800.0\nR003,2010-04-21,14.9,17.7,12835.86\nR003,2010-04-22,11.4,8.8,28360.32\nR003,2010-04-23,11.9,0.6,45182.0\nR003,2010-04-24,12.2,0.3,42656.12\nR003,2010-04-25,12.7,0.7,45928.23\nR003,2010-04-26,12.5,5.5,35093.41\nR003,2010-04-27,11.3,0.0,39978.67\nR003,2010-04-28,12.9,0.0,47626.95\nR003,2010-04-29,13.3,0.2,48254.56\nR003,2010-04-30,12.7,1.0,43091.04\nR003,2010-05-01,12.8,5.1,46017.47\nR003,2010-05-02,12.9,0.1,47135.21\nR003,2010-05-03,11.6,4.5,16518.02\nR003,2010-05-04,11.7,3.4,28007.98\nR003,2010-05-05,12.9,4.6,44530.39\nR003,2010-05-06,14.6,3.7,48346.29\nR003,2010-05-07,15.7,4.4,47933.07\nR003,2010-05-08,16.4,0.6,49647.15\nR003,2010-05-09,16.2,0.5,46454.32\nR003,2010-05-10,14.6,8.9,20960.72\nR003,2010-05-11,14.4,7.2,44514.91\nR003,2010-05-12,15.2,0.4,47826.44\nR003,2010-05-13,17.3,0.0,48487.76\nR003,2010-05-14,19.7,0.0,49694.4\nR003,2010-05-15,21.2,0.0,50400.0\nR003,2010-05-16,21.6,0.4,43200.0\nR003,2010-05-17,20.7,5.6,40413.15\nR003,2010-05-18,18.7,0.9,44906.51\nR003,2010-05-19,16.1,10.2,41040.75\nR003,2010-05-20,14.2,0.0,50400.0\nR003,2010-05-21,14.0,1.0,43131.85\nR003,2010-05-22,14.7,1.8,41313.51\nR003,2010-05-23,15.4,5.4,46039.45\nR003,2010-05-24,13.6,8.4,37683.74\nR003,2010-05-25,14.5,1.6,44078.98\nR003,2010-05-26,16.5,0.0,51256.0\nR003,2010-05-27,19.5,0.0,51233.27\nR003,2010-05-28,19.4,0.0,51210.76\nR003,2010-05-29,19.7,0.5,51188.54\nR003,2010-05-30,19.5,6.4,51166.62\nR003,2010-05-31,20.2,1.6,50783.39\nR003,2010-06-01,22.1,1.5,51848.4\nR003,2010-06-02,24.7,0.6,45032.35\nR003,2010-06-03,24.5,0.5,47701.99\nR003,2010-06-04,23.5,0.1,50699.5\nR003,2010-06-05,24.1,1.0,49862.22\nR003,2010-06-06,23.7,3.3,49890.65\nR003,2010-06-07,22.3,20.1,37207.01\nR003,2010-06-08,18.8,43.9,6909.91\nR003,2010-06-09,19.6,17.7,30354.0\nR003,2010-06-10,21.1,1.6,52424.08\nR003,2010-06-11,22.6,8.9,51677.47\nR003,2010-06-12,23.1,4.8,53131.54\nR003,2010-06-13,24.8,0.4,53769.51\nR003,2010-06-14,26.2,0.6,52742.16\nR003,2010-06-15,25.7,0.8,49192.15\nR003,2010-06-16,23.0,1.5,38995.23\nR003,2010-06-17,22.3,6.0,47467.19\nR003,2010-06-18,21.4,9.0,40850.31\nR003,2010-06-19,21.7,2.8,45713.2\nR003,2010-06-20,22.6,5.5,52107.87\nR003,2010-06-21,21.7,21.4,38916.56\nR003,2010-06-22,20.8,7.7,52710.12\nR003,2010-06-23,22.9,0.2,53442.74\nR003,2010-06-24,19.6,21.5,18719.68\nR003,2010-06-25,18.7,4.5,31616.2\nR003,2010-06-26,21.1,3.5,48541.59\nR003,2010-06-27,19.3,14.2,39088.68\nR003,2010-06-28,20.7,6.1,51732.31\nR003,2010-06-29,20.0,18.4,46540.14\nR003,2010-06-30,20.5,7.0,45046.77\nR003,2010-07-01,21.3,3.9,53340.55\nR003,2010-07-02,21.4,5.3,53013.76\nR003,2010-07-03,23.2,2.3,51083.93\nR003,2010-07-04,22.5,6.3,45054.5\nR003,2010-07-05,22.5,6.9,42236.41\nR003,2010-07-06,23.8,0.5,48701.0\nR003,2010-07-07,23.7,0.5,51949.92\nR003,2010-07-08,25.7,0.2,53399.31\nR003,2010-07-09,25.3,3.5,52189.77\nR003,2010-07-10,26.0,1.8,48615.21\nR003,2010-07-11,26.2,1.0,52697.9\nR003,2010-07-12,25.8,2.0,50620.85\nR003,2010-07-13,24.5,3.9,52201.58\nR003,2010-07-14,22.7,8.9,45967.76\nR003,2010-07-15,22.1,10.9,41542.83\nR003,2010-07-16,22.7,0.0,53468.91\nR003,2010-07-17,23.3,0.2,53124.28\nR003,2010-07-18,21.4,6.0,13199.71\nR003,2010-07-19,22.1,10.0,45333.35\nR003,2010-07-20,24.2,0.8,52068.78\nR003,2010-07-21,23.7,4.7,51676.67\nR003,2010-07-22,23.0,9.7,50339.9\nR003,2010-07-23,22.1,15.4,46662.18\nR003,2010-07-24,21.3,26.4,29442.97\nR003,2010-07-25,22.6,6.9,45822.83\nR003,2010-07-26,24.9,0.7,51846.7\nR003,2010-07-27,27.1,0.2,51461.52\nR003,2010-07-28,27.3,0.7,51077.94\nR003,2010-07-29,25.8,1.2,49263.94\nR003,2010-07-30,26.8,0.2,49664.67\nR003,2010-07-31,27.2,0.0,50629.86\nR003,2010-08-01,27.5,0.0,50116.92\nR003,2010-08-02,26.4,0.5,48795.58\nR003,2010-08-03,25.1,0.9,48511.67\nR003,2010-08-04,25.0,0.0,49257.15\nR003,2010-08-05,26.1,0.5,48975.32\nR003,2010-08-06,26.6,0.8,48696.49\nR003,2010-08-07,26.4,0.9,48420.02\nR003,2010-08-08,26.0,0.0,48485.01\nR003,2010-08-09,26.3,0.0,48006.0\nR003,2010-08-10,26.8,0.9,47696.13\nR003,2010-08-11,26.1,4.8,47051.17\nR003,2010-08-12,24.7,7.1,47081.16\nR003,2010-08-13,24.6,2.2,47111.15\nR003,2010-08-14,24.9,2.0,47141.09\nR003,2010-08-15,23.3,10.1,46837.37\nR003,2010-08-16,22.9,12.4,45258.39\nR003,2010-08-17,21.1,23.7,38313.9\nR003,2010-08-18,20.9,7.2,38687.94\nR003,2010-08-19,21.2,3.7,46800.0\nR003,2010-08-20,21.1,0.5,45567.81\nR003,2010-08-21,20.1,0.0,46398.65\nR003,2010-08-22,20.0,0.6,43336.29\nR003,2010-08-23,21.0,0.0,46800.0\nR003,2010-08-24,22.6,0.4,46800.0\nR003,2010-08-25,23.4,0.8,46800.0\nR003,2010-08-26,22.1,3.3,46800.0\nR003,2010-08-27,22.0,1.7,44990.5\nR003,2010-08-28,22.4,0.0,45316.57\nR003,2010-08-29,25.0,0.0,46536.94\nR003,2010-08-30,25.8,0.1,46433.77\nR003,2010-08-31,26.1,0.0,46330.44\nR003,2010-09-01,28.1,0.7,46227.05\nR003,2010-09-02,21.9,22.1,30815.71\nR003,2010-09-03,20.6,0.2,43752.28\nR003,2010-09-04,22.8,0.0,43601.43\nR003,2010-09-05,23.8,1.1,43816.2\nR003,2010-09-06,23.3,4.3,39659.6\nR003,2010-09-07,22.8,9.8,42201.06\nR003,2010-09-08,20.6,4.6,39363.98\nR003,2010-09-09,19.7,0.0,43200.0\nR003,2010-09-10,19.4,0.1,39501.7\nR003,2010-09-11,20.0,0.3,43200.0\nR003,2010-09-12,22.8,0.0,43200.0\nR003,2010-09-13,21.0,19.1,20399.55\nR003,2010-09-14,18.8,15.8,0.0\nR003,2010-09-15,18.1,6.4,20946.81\nR003,2010-09-16,19.2,0.0,37912.16\nR003,2010-09-17,19.9,0.0,42913.8\nR003,2010-09-18,20.5,0.2,41478.06\nR003,2010-09-19,19.8,2.1,36005.08\nR003,2010-09-20,20.3,0.0,40272.79\nR003,2010-09-21,21.9,0.0,42216.96\nR003,2010-09-22,20.3,5.8,40963.36\nR003,2010-09-23,18.3,4.8,27004.28\nR003,2010-09-24,18.8,0.0,40411.05\nR003,2010-09-25,21.2,0.0,41720.95\nR003,2010-09-26,21.8,1.2,40162.89\nR003,2010-09-27,21.2,0.4,40517.46\nR003,2010-09-28,21.4,0.5,40234.31\nR003,2010-09-29,19.4,9.7,38411.92\nR003,2010-09-30,18.6,14.1,29312.06\nR003,2010-10-01,13.7,2.6,36924.38\nR003,2010-10-02,13.3,0.9,32601.76\nR003,2010-10-03,13.1,24.8,7.11\nR003,2010-10-04,11.6,1.7,36871.57\nR003,2010-10-05,10.4,1.7,30527.45\nR003,2010-10-06,9.6,1.2,34410.38\nR003,2010-10-07,10.7,0.6,33122.56\nR003,2010-10-08,14.0,4.7,21142.44\nR003,2010-10-09,14.5,11.9,26531.66\nR003,2010-10-10,12.1,20.4,23282.55\nR003,2010-10-11,9.2,14.4,25942.18\nR003,2010-10-12,9.2,11.4,29727.52\nR003,2010-10-13,10.0,2.2,28800.0\nR003,2010-10-14,11.1,0.2,32808.79\nR003,2010-10-15,12.6,23.5,6682.91\nR003,2010-10-16,14.4,14.3,19428.63\nR003,2010-10-17,14.2,4.3,17423.82\nR003,2010-10-18,15.2,0.3,34550.73\nR003,2010-10-19,16.3,0.0,34321.69\nR003,2010-10-20,19.3,5.5,28679.92\nR003,2010-10-21,18.6,3.8,35584.28\nR003,2010-10-22,14.2,2.3,14751.74\nR003,2010-10-23,13.1,1.2,15257.66\nR003,2010-10-24,13.5,0.8,21343.93\nR003,2010-10-25,12.8,0.0,36000.0\nR003,2010-10-26,14.5,0.0,36000.0\nR003,2010-10-27,13.8,0.4,30234.01\nR003,2010-10-28,12.4,17.8,2024.11\nR003,2010-10-29,13.2,8.8,3321.99\nR003,2010-10-30,10.5,18.5,14801.79\nR003,2010-10-31,5.0,0.1,36000.0\nR003,2010-11-01,5.9,0.0,36000.0\nR003,2010-11-02,7.1,0.0,35953.15\nR003,2010-11-03,10.2,0.0,35874.7\nR003,2010-11-04,11.6,0.0,35795.67\nR003,2010-11-05,11.5,0.0,35716.13\nR003,2010-11-06,12.1,0.0,35562.33\nR003,2010-11-07,12.9,0.0,35408.51\nR003,2010-11-08,13.9,0.0,35255.8\nR003,2010-11-09,14.5,0.0,35104.39\nR003,2010-11-10,14.0,0.0,34954.45\nR003,2010-11-11,15.7,0.0,34806.17\nR003,2010-11-12,15.6,0.0,34114.72\nR003,2010-11-13,15.8,0.0,34460.22\nR003,2010-11-14,14.8,0.0,34373.14\nR003,2010-11-15,14.2,0.0,34078.08\nR003,2010-11-16,13.9,0.0,34069.31\nR003,2010-11-17,13.3,0.0,32055.31\nR003,2010-11-18,13.8,0.0,32964.17\nR003,2010-11-19,14.3,0.0,29645.82\nR003,2010-11-20,11.1,0.0,32587.44\nR003,2010-11-21,9.3,0.0,30155.66\nR003,2010-11-22,8.9,4.3,14669.92\nR003,2010-11-23,8.6,0.0,30296.31\nR003,2010-11-24,11.3,8.6,27452.31\nR003,2010-11-25,11.1,28.4,19977.18\nR003,2010-11-26,10.6,21.4,428.23\nR003,2010-11-27,11.8,11.3,5380.64\nR003,2010-11-28,11.3,27.2,0.0\nR003,2010-11-29,11.6,0.4,25500.76\nR003,2010-11-30,13.2,3.0,28800.0\nR003,2010-12-01,12.3,6.0,9515.67\nR003,2010-12-02,12.8,13.4,17500.4\nR003,2010-12-03,13.1,0.9,1534.96\nR003,2010-12-04,14.0,0.0,29219.25\nR003,2010-12-05,17.4,0.0,28846.24\nR003,2010-12-06,11.0,0.0,28181.17\nR003,2010-12-07,8.4,0.0,30014.45\nR003,2010-12-08,8.8,0.0,26791.95\nR003,2010-12-09,9.7,0.0,30027.82\nR003,2010-12-10,10.1,0.0,25062.11\nR003,2010-12-11,10.8,0.2,26821.44\nR003,2010-12-12,8.6,8.9,20687.36\nR003,2010-12-13,4.7,9.6,23644.49\nR003,2010-12-14,3.9,5.9,10171.33\nR003,2010-12-15,5.9,7.6,0.0\nR003,2010-12-16,6.0,3.6,0.0\nR003,2010-12-17,7.5,3.6,25436.76\nR003,2010-12-18,5.0,19.8,0.0\nR003,2010-12-19,3.2,0.2,30389.59\nR003,2010-12-20,6.1,5.5,30378.18\nR003,2010-12-21,7.9,0.0,30365.25\nR003,2010-12-22,9.5,0.0,30742.06\nR003,2010-12-23,8.3,0.0,31117.7\nR003,2010-12-24,6.9,0.0,31243.39\nR003,2010-12-25,6.4,0.0,31280.78\nR003,2010-12-26,9.9,0.0,20720.79\nR003,2010-12-27,12.8,0.0,19113.5\nR003,2010-12-28,12.5,0.0,16232.34\nR003,2010-12-29,10.8,0.0,31385.93\nR003,2010-12-30,7.6,0.8,20084.71\nR003,2010-12-31,5.4,8.8,0.0\nR003,2011-01-01,3.1,0.3,30026.91\nR003,2011-01-02,2.8,0.0,31643.8\nR003,2011-01-03,3.9,0.0,26729.53\nR003,2011-01-04,5.4,2.3,29488.01\nR003,2011-01-05,4.5,7.1,16865.76\nR003,2011-01-06,4.0,4.1,19912.07\nR003,2011-01-07,4.1,0.0,17777.75\nR003,2011-01-08,4.4,8.3,1202.23\nR003,2011-01-09,3.4,2.2,22744.04\nR003,2011-01-10,1.8,0.3,32137.64\nR003,2011-01-11,1.8,0.0,32207.13\nR003,2011-01-12,2.1,0.0,32277.88\nR003,2011-01-13,2.3,0.0,32349.72\nR003,2011-01-14,3.0,0.0,28800.0\nR003,2011-01-15,3.4,0.3,25483.85\nR003,2011-01-16,3.4,0.9,26914.0\nR003,2011-01-17,3.5,8.1,18346.94\nR003,2011-01-18,2.0,8.0,0.0\nR003,2011-01-19,1.6,4.0,24152.56\nR003,2011-01-20,-1.1,0.0,32400.0\nR003,2011-01-21,-0.5,0.0,26247.39\nR003,2011-01-22,0.4,0.0,32400.0\nR003,2011-01-23,4.4,0.0,30657.92\nR003,2011-01-24,5.6,26.5,13205.87\nR003,2011-01-25,7.0,9.9,24720.44\nR003,2011-01-26,5.6,18.9,1302.15\nR003,2011-01-27,2.7,10.4,1177.83\nR003,2011-01-28,0.1,2.3,32400.0\nR003,2011-01-29,-3.0,0.0,32400.0\nR003,2011-01-30,-1.6,0.0,32400.0\nR003,2011-01-31,0.6,8.0,283.25\nR003,2011-02-01,-0.7,4.1,29697.12\nR003,2011-02-02,-3.2,4.0,16135.27\nR003,2011-02-03,-2.0,0.9,32641.35\nR003,2011-02-04,-0.4,6.5,21595.91\nR003,2011-02-05,0.0,3.8,18557.87\nR003,2011-02-06,2.3,34.5,0.0\nR003,2011-02-07,0.9,4.0,32462.99\nR003,2011-02-08,2.3,6.1,1168.15\nR003,2011-02-09,3.6,6.9,24296.65\nR003,2011-02-10,-1.8,2.1,33360.85\nR003,2011-02-11,-5.1,0.1,34676.66\nR003,2011-02-12,-0.5,6.6,1234.41\nR003,2011-02-13,-2.1,25.5,10937.52\nR003,2011-02-14,-5.0,7.3,22169.69\nR003,2011-02-15,-4.6,5.6,35004.67\nR003,2011-02-16,-8.2,0.0,37318.22\nR003,2011-02-17,-7.0,0.0,37432.56\nR003,2011-02-18,-4.4,0.0,36128.56\nR003,2011-02-19,-0.7,0.0,30598.87\nR003,2011-02-20,3.1,2.8,0.0\nR003,2011-02-21,4.0,9.8,175.97\nR003,2011-02-22,3.1,0.0,38415.89\nR003,2011-02-23,5.4,0.1,25250.88\nR003,2011-02-24,7.0,0.0,38194.38\nR003,2011-02-25,7.3,0.4,37225.32\nR003,2011-02-26,6.2,0.5,23129.3\nR003,2011-02-27,3.5,7.5,0.0\nR003,2011-02-28,4.7,0.1,39427.06\nR003,2011-03-01,4.3,2.2,26746.12\nR003,2011-03-02,2.7,9.7,27948.21\nR003,2011-03-03,0.2,3.2,39600.0\nR003,2011-03-04,-0.6,1.6,39600.0\nR003,2011-03-05,-0.4,0.0,36685.4\nR003,2011-03-06,3.8,1.1,39600.0\nR003,2011-03-07,2.3,27.1,0.0\nR003,2011-03-08,2.2,2.8,22396.32\nR003,2011-03-09,5.7,0.0,39600.0\nR003,2011-03-10,5.5,4.8,19650.97\nR003,2011-03-11,4.3,1.6,31253.9\nR003,2011-03-12,2.1,3.6,33647.09\nR003,2011-03-13,3.0,2.3,36000.0\nR003,2011-03-14,5.5,0.0,39600.0\nR003,2011-03-15,8.1,0.0,39638.9\nR003,2011-03-16,10.4,0.0,39600.0\nR003,2011-03-17,8.7,0.0,39750.66\nR003,2011-03-18,8.0,4.1,16327.13\nR003,2011-03-19,7.6,11.6,6392.65\nR003,2011-03-20,8.3,0.0,39600.0\nR003,2011-03-21,7.5,7.3,6634.02\nR003,2011-03-22,5.5,9.9,3077.79\nR003,2011-03-23,5.0,3.9,32776.51\nR003,2011-03-24,3.3,5.4,22425.64\nR003,2011-03-25,3.8,12.0,21036.67\nR003,2011-03-26,6.6,0.0,43079.56\nR003,2011-03-27,8.4,0.6,42051.63\nR003,2011-03-28,9.0,0.0,42221.57\nR003,2011-03-29,8.5,0.0,43617.76\nR003,2011-03-30,8.5,0.5,32446.42\nR003,2011-03-31,9.9,2.0,39665.21\nR003,2011-04-01,7.2,1.0,42507.39\nR003,2011-04-02,9.4,0.0,25548.09\nR003,2011-04-03,10.3,3.4,14638.5\nR003,2011-04-04,9.2,0.4,23022.46\nR003,2011-04-05,7.1,12.8,0.0\nR003,2011-04-06,6.3,5.7,16838.43\nR003,2011-04-07,5.0,11.7,26077.92\nR003,2011-04-08,6.7,15.9,4638.36\nR003,2011-04-09,4.8,28.0,33388.46\nR003,2011-04-10,3.3,9.6,40034.56\nR003,2011-04-11,1.9,11.7,34715.02\nR003,2011-04-12,1.8,0.0,30086.54\nR003,2011-04-13,2.7,0.3,46060.24\nR003,2011-04-14,7.8,0.0,46558.57\nR003,2011-04-15,8.6,15.7,13769.47\nR003,2011-04-16,6.2,5.2,46701.28\nR003,2011-04-17,6.4,3.8,29317.4\nR003,2011-04-18,8.7,0.0,39600.0\nR003,2011-04-19,14.3,1.8,41952.71\nR003,2011-04-20,11.5,9.1,44033.21\nR003,2011-04-21,6.0,20.5,12468.65\nR003,2011-04-22,8.2,0.0,45017.64\nR003,2011-04-23,8.4,1.2,16331.64\nR003,2011-04-24,8.6,3.0,1834.48\nR003,2011-04-25,10.6,1.5,45545.38\nR003,2011-04-26,11.5,3.5,36088.54\nR003,2011-04-27,12.8,0.6,39842.9\nR003,2011-04-28,15.0,1.0,43050.96\nR003,2011-04-29,13.5,5.3,7283.46\nR003,2011-04-30,12.6,5.9,22185.99\nR003,2011-05-01,13.4,0.2,44270.6\nR003,2011-05-02,14.9,0.0,46811.12\nR003,2011-05-03,15.0,1.6,46976.73\nR003,2011-05-04,14.9,2.5,38070.95\nR003,2011-05-05,13.4,15.0,1647.26\nR003,2011-05-06,12.2,7.8,19271.41\nR003,2011-05-07,10.9,6.1,0.0\nR003,2011-05-08,10.0,6.1,11454.15\nR003,2011-05-09,11.3,0.3,44177.46\nR003,2011-05-10,10.5,1.9,47041.77\nR003,2011-05-11,13.0,0.6,46547.2\nR003,2011-05-12,12.5,12.5,20113.93\nR003,2011-05-13,11.7,14.8,2270.51\nR003,2011-05-14,12.9,13.5,12898.18\nR003,2011-05-15,12.5,3.7,23806.61\nR003,2011-05-16,13.8,0.3,47470.67\nR003,2011-05-17,15.8,0.0,50322.87\nR003,2011-05-18,17.6,0.0,49034.73\nR003,2011-05-19,18.0,4.8,37323.81\nR003,2011-05-20,15.3,17.9,29787.33\nR003,2011-05-21,15.7,8.7,46422.88\nR003,2011-05-22,14.4,12.8,1632.1\nR003,2011-05-23,14.4,30.3,3758.79\nR003,2011-05-24,16.9,2.5,34462.41\nR003,2011-05-25,17.6,7.4,43689.05\nR003,2011-05-26,16.5,13.4,39662.07\nR003,2011-05-27,17.7,0.4,45953.48\nR003,2011-05-28,19.7,1.1,47183.8\nR003,2011-05-29,20.0,5.4,28057.87\nR003,2011-05-30,20.7,2.6,50488.09\nR003,2011-05-31,22.4,3.8,51509.16\nR003,2011-06-01,22.8,6.7,51126.16\nR003,2011-06-02,20.3,16.1,19607.76\nR003,2011-06-03,19.6,19.9,10874.45\nR003,2011-06-04,18.2,13.3,17140.79\nR003,2011-06-05,17.7,8.0,45884.88\nR003,2011-06-06,19.0,1.4,49126.49\nR003,2011-06-07,19.2,0.9,50415.84\nR003,2011-06-08,20.4,0.9,50994.81\nR003,2011-06-09,19.7,0.0,51439.02\nR003,2011-06-10,20.4,0.0,53527.81\nR003,2011-06-11,18.2,4.3,36257.31\nR003,2011-06-12,18.1,5.2,35121.07\nR003,2011-06-13,18.4,3.9,42292.52\nR003,2011-06-14,19.2,4.9,46568.8\nR003,2011-06-15,18.6,4.4,42692.5\nR003,2011-06-16,19.5,0.0,51050.45\nR003,2011-06-17,20.6,0.9,52714.78\nR003,2011-06-18,20.8,2.2,52705.25\nR003,2011-06-19,19.8,13.2,51225.36\nR003,2011-06-20,19.3,10.6,41144.85\nR003,2011-06-21,19.3,3.6,48459.53\nR003,2011-06-22,18.3,0.2,52673.45\nR003,2011-06-23,17.1,4.0,51475.2\nR003,2011-06-24,18.3,0.1,53954.99\nR003,2011-06-25,20.4,0.0,53966.85\nR003,2011-06-26,21.0,2.3,51919.19\nR003,2011-06-27,16.5,21.3,5959.43\nR003,2011-06-28,16.9,23.6,31018.17\nR003,2011-06-29,19.5,2.3,44850.74\nR003,2011-06-30,17.7,22.6,18006.19\nR003,2011-07-01,19.0,0.0,53951.77\nR003,2011-07-02,20.1,3.7,47055.46\nR003,2011-07-03,20.9,0.5,49308.49\nR003,2011-07-04,22.1,0.9,52892.78\nR003,2011-07-05,21.1,3.1,47535.91\nR003,2011-07-06,23.0,0.0,53511.93\nR003,2011-07-07,24.7,3.4,52304.01\nR003,2011-07-08,23.6,3.7,38791.47\nR003,2011-07-09,24.1,1.3,41991.59\nR003,2011-07-10,23.6,2.6,49279.21\nR003,2011-07-11,22.6,0.3,49822.55\nR003,2011-07-12,23.1,1.5,52704.39\nR003,2011-07-13,22.4,9.3,47375.47\nR003,2011-07-14,24.2,4.1,52655.7\nR003,2011-07-15,24.3,8.3,51609.5\nR003,2011-07-16,23.0,4.6,39451.75\nR003,2011-07-17,22.7,15.7,36101.71\nR003,2011-07-18,23.4,8.0,52476.86\nR003,2011-07-19,22.9,11.1,45832.42\nR003,2011-07-20,22.4,6.3,24830.86\nR003,2011-07-21,23.1,12.6,52043.24\nR003,2011-07-22,23.9,2.4,52006.78\nR003,2011-07-23,21.7,5.1,47402.35\nR003,2011-07-24,21.8,2.6,48226.23\nR003,2011-07-25,24.3,0.9,51545.06\nR003,2011-07-26,27.5,0.3,51859.13\nR003,2011-07-27,28.3,0.1,51472.59\nR003,2011-07-28,27.0,1.2,50246.3\nR003,2011-07-29,28.3,0.0,49602.77\nR003,2011-07-30,28.5,0.3,49308.17\nR003,2011-07-31,26.3,4.1,43937.93\nR003,2011-08-01,26.2,0.0,45487.23\nR003,2011-08-02,25.8,1.7,45805.2\nR003,2011-08-03,22.6,29.0,23979.2\nR003,2011-08-04,22.2,1.3,47531.91\nR003,2011-08-05,21.3,0.0,47595.81\nR003,2011-08-06,20.1,0.0,48259.03\nR003,2011-08-07,20.9,0.8,45545.5\nR003,2011-08-08,21.6,1.1,47293.6\nR003,2011-08-09,22.3,0.0,46983.61\nR003,2011-08-10,23.1,0.0,48026.21\nR003,2011-08-11,24.1,0.0,47716.57\nR003,2011-08-12,23.4,3.5,45646.48\nR003,2011-08-13,20.5,16.6,7799.68\nR003,2011-08-14,20.4,8.2,45308.76\nR003,2011-08-15,21.5,3.7,45838.18\nR003,2011-08-16,22.0,0.9,45758.69\nR003,2011-08-17,21.2,0.0,45503.34\nR003,2011-08-18,20.6,0.9,43915.59\nR003,2011-08-19,21.2,1.2,45562.6\nR003,2011-08-20,21.1,0.0,45896.35\nR003,2011-08-21,20.5,0.0,46800.0\nR003,2011-08-22,19.1,2.2,45886.31\nR003,2011-08-23,19.5,0.1,43280.84\nR003,2011-08-24,19.4,1.1,46250.37\nR003,2011-08-25,18.9,1.7,46747.21\nR003,2011-08-26,18.3,2.5,43795.17\nR003,2011-08-27,18.4,0.0,46769.13\nR003,2011-08-28,19.3,0.0,46669.07\nR003,2011-08-29,21.3,0.0,46568.61\nR003,2011-08-30,23.6,0.0,46466.81\nR003,2011-08-31,22.7,2.9,45805.72\nR003,2011-09-01,21.1,6.3,42625.22\nR003,2011-09-02,20.4,0.3,43532.52\nR003,2011-09-03,18.7,3.7,33609.27\nR003,2011-09-04,17.6,6.6,38173.39\nR003,2011-09-05,15.6,3.4,42107.38\nR003,2011-09-06,15.7,0.1,42898.88\nR003,2011-09-07,15.5,0.0,42953.14\nR003,2011-09-08,17.0,0.0,43962.91\nR003,2011-09-09,17.3,3.3,36596.99\nR003,2011-09-10,15.8,6.8,41475.87\nR003,2011-09-11,14.8,0.0,40174.14\nR003,2011-09-12,15.6,4.7,34014.68\nR003,2011-09-13,16.4,0.0,43200.0\nR003,2011-09-14,19.0,0.0,43133.33\nR003,2011-09-15,20.1,2.2,42146.99\nR003,2011-09-16,19.6,2.0,31330.97\nR003,2011-09-17,18.2,3.6,36218.84\nR003,2011-09-18,17.5,0.0,40514.42\nR003,2011-09-19,18.1,0.0,41184.64\nR003,2011-09-20,19.2,0.0,42728.41\nR003,2011-09-21,18.9,0.4,38537.09\nR003,2011-09-22,18.9,0.0,39174.49\nR003,2011-09-23,19.4,1.1,38167.47\nR003,2011-09-24,17.8,10.6,18682.18\nR003,2011-09-25,16.6,1.7,37772.01\nR003,2011-09-26,14.6,0.2,40472.66\nR003,2011-09-27,14.0,0.0,40189.43\nR003,2011-09-28,14.6,0.0,40914.95\nR003,2011-09-29,13.8,1.1,38106.64\nR003,2011-09-30,10.7,26.7,25972.38\nR003,2011-10-01,9.3,0.3,39380.03\nR003,2011-10-02,10.2,29.0,1235.67\nR003,2011-10-03,7.2,2.9,38447.85\nR003,2011-10-04,8.6,0.0,36000.0\nR003,2011-10-05,11.1,4.4,33009.79\nR003,2011-10-06,13.4,0.2,36584.45\nR003,2011-10-07,14.9,0.0,38186.29\nR003,2011-10-08,16.1,0.0,37948.48\nR003,2011-10-09,18.4,0.0,32742.29\nR003,2011-10-10,19.4,0.6,840.35\nR003,2011-10-11,22.4,0.0,37112.53\nR003,2011-10-12,20.8,8.9,32545.76\nR003,2011-10-13,14.6,8.5,8125.71\nR003,2011-10-14,13.2,9.1,30969.88\nR003,2011-10-15,11.2,23.6,14750.43\nR003,2011-10-16,12.0,7.4,6311.76\nR003,2011-10-17,13.6,2.6,19365.04\nR003,2011-10-18,13.7,6.9,32417.12\nR003,2011-10-19,8.6,21.7,19261.45\nR003,2011-10-20,8.0,0.0,27731.68\nR003,2011-10-21,9.7,0.0,36000.0\nR003,2011-10-22,10.6,0.0,35068.98\nR003,2011-10-23,10.8,0.1,31117.04\nR003,2011-10-24,11.5,0.0,34284.66\nR003,2011-10-25,10.3,0.0,28226.63\nR003,2011-10-26,9.4,4.3,36000.0\nR003,2011-10-27,7.8,2.7,36000.0\nR003,2011-10-28,9.0,0.3,36000.0\nR003,2011-10-29,8.2,0.0,36000.0\nR003,2011-10-30,7.4,0.5,36000.0\nR003,2011-10-31,6.9,0.0,36000.0\nR003,2011-11-01,7.2,0.0,36000.0\nR003,2011-11-02,7.2,0.1,18593.02\nR003,2011-11-03,8.0,0.0,34495.19\nR003,2011-11-04,7.4,0.0,35813.37\nR003,2011-11-05,8.4,0.0,34522.36\nR003,2011-11-06,6.4,0.0,32816.31\nR003,2011-11-07,2.1,0.0,35438.3\nR003,2011-11-08,2.8,0.0,33846.79\nR003,2011-11-09,4.3,12.7,3101.62\nR003,2011-11-10,1.5,13.6,15434.3\nR003,2011-11-11,0.6,16.4,904.83\nR003,2011-11-12,-2.7,4.4,31286.21\nR003,2011-11-13,-4.3,0.0,33739.74\nR003,2011-11-14,-2.6,0.0,34404.3\nR003,2011-11-15,-0.9,0.0,34267.1\nR003,2011-11-16,1.9,0.3,31215.91\nR003,2011-11-17,3.0,1.8,23862.09\nR003,2011-11-18,1.9,0.0,32235.63\nR003,2011-11-19,1.4,0.0,32228.14\nR003,2011-11-20,0.8,0.0,32222.59\nR003,2011-11-21,1.2,0.0,30875.98\nR003,2011-11-22,1.5,0.0,30204.61\nR003,2011-11-23,2.1,1.2,15430.38\nR003,2011-11-24,-3.2,0.0,31296.12\nR003,2011-11-25,-1.0,0.1,30037.25\nR003,2011-11-26,0.1,4.0,9008.4\nR003,2011-11-27,1.3,3.9,17919.05\nR003,2011-11-28,3.1,2.2,28290.34\nR003,2011-11-29,-0.1,13.6,25546.45\nR003,2011-11-30,-1.3,15.7,16440.27\nR003,2011-12-01,-3.4,0.0,25607.79\nR003,2011-12-02,0.8,5.2,0.0\nR003,2011-12-03,1.3,1.1,29588.56\nR003,2011-12-04,1.8,0.0,30365.31\nR003,2011-12-05,3.8,0.0,30378.1\nR003,2011-12-06,6.6,0.0,29238.48\nR003,2011-12-07,9.5,0.0,13390.81\nR003,2011-12-08,7.7,1.0,19633.58\nR003,2011-12-09,5.4,17.3,0.0\nR003,2011-12-10,1.2,1.6,30031.49\nR003,2011-12-11,1.1,0.0,21790.34\nR003,2011-12-12,4.7,7.0,8903.93\nR003,2011-12-13,3.9,0.0,29162.53\nR003,2011-12-14,5.3,0.0,20882.5\nR003,2011-12-15,6.9,0.0,27308.29\nR003,2011-12-16,5.1,0.0,30412.7\nR003,2011-12-17,6.2,0.0,23799.93\nR003,2011-12-18,7.8,0.0,5946.31\nR003,2011-12-19,9.8,0.0,24059.68\nR003,2011-12-20,9.6,0.0,16134.02\nR003,2011-12-21,10.2,8.1,12743.83\nR003,2011-12-22,6.7,5.9,24229.1\nR003,2011-12-23,5.4,0.2,17975.25\nR003,2011-12-24,6.2,5.3,6637.65\nR003,2011-12-25,4.6,3.4,22645.32\nR003,2011-12-26,3.0,0.4,25887.25\nR003,2011-12-27,1.7,1.3,26880.71\nR003,2011-12-28,2.5,2.0,26151.74\nR003,2011-12-29,1.8,0.1,31429.17\nR003,2011-12-30,1.7,0.0,31476.05\nR003,2011-12-31,4.3,0.0,31525.45\nR003,2012-01-01,4.5,0.8,28625.86\nR003,2012-01-02,-0.2,12.5,29964.64\nR003,2012-01-03,-1.5,2.6,14878.38\nR003,2012-01-04,2.6,1.0,18943.92\nR003,2012-01-05,1.9,0.0,30406.62\nR003,2012-01-06,3.6,0.0,22930.44\nR003,2012-01-07,2.4,0.0,29310.15\nR003,2012-01-08,5.5,0.0,2319.22\nR003,2012-01-09,5.9,5.1,25200.0\nR003,2012-01-10,2.5,15.5,22277.8\nR003,2012-01-11,4.1,3.9,187.35\nR003,2012-01-12,4.2,11.4,593.35\nR003,2012-01-13,1.6,2.0,21312.53\nR003,2012-01-14,0.9,6.5,18347.02\nR003,2012-01-15,1.0,24.0,0.0\nR003,2012-01-16,0.4,23.2,0.0\nR003,2012-01-17,-0.9,10.2,0.0\nR003,2012-01-18,-3.4,3.1,32037.62\nR003,2012-01-19,-5.5,1.0,21247.2\nR003,2012-01-20,-7.1,0.0,32400.0\nR003,2012-01-21,-6.9,0.0,32400.0\nR003,2012-01-22,0.8,2.0,25200.0\nR003,2012-01-23,1.5,13.3,3602.67\nR003,2012-01-24,0.0,5.6,12210.84\nR003,2012-01-25,-0.8,0.0,22433.55\nR003,2012-01-26,2.3,0.8,0.0\nR003,2012-01-27,0.8,0.0,14857.72\nR003,2012-01-28,-3.4,0.0,32400.0\nR003,2012-01-29,-5.9,1.8,26895.91\nR003,2012-01-30,-6.4,0.0,32475.36\nR003,2012-01-31,-4.0,9.3,0.0\nR003,2012-02-01,-5.1,12.2,11648.37\nR003,2012-02-02,-7.4,0.4,32695.86\nR003,2012-02-03,-4.4,0.0,31901.26\nR003,2012-02-04,-3.0,0.0,33349.54\nR003,2012-02-05,1.3,0.0,33962.45\nR003,2012-02-06,1.6,0.0,33877.33\nR003,2012-02-07,-1.0,0.0,33021.26\nR003,2012-02-08,-6.3,0.0,34737.79\nR003,2012-02-09,-4.0,1.5,25501.41\nR003,2012-02-10,-4.2,1.4,29126.78\nR003,2012-02-11,-6.5,0.5,31599.83\nR003,2012-02-12,-3.2,0.0,32597.41\nR003,2012-02-13,2.0,0.0,35857.34\nR003,2012-02-14,3.6,0.5,23543.5\nR003,2012-02-15,5.3,1.5,13672.38\nR003,2012-02-16,2.0,6.0,20713.69\nR003,2012-02-17,1.5,0.5,23884.23\nR003,2012-02-18,0.8,7.9,3013.73\nR003,2012-02-19,-0.7,8.5,301.39\nR003,2012-02-20,-1.6,0.9,35751.04\nR003,2012-02-21,-0.2,0.0,35841.41\nR003,2012-02-22,-0.5,0.1,32400.0\nR003,2012-02-23,-0.8,2.3,36969.18\nR003,2012-02-24,0.7,1.0,30548.43\nR003,2012-02-25,0.8,13.6,7250.28\nR003,2012-02-26,1.0,12.6,16824.61\nR003,2012-02-27,3.9,1.0,24100.03\nR003,2012-02-28,6.3,7.4,34381.29\nR003,2012-02-29,4.5,9.6,919.48\nR003,2012-03-01,3.2,8.0,3213.51\nR003,2012-03-02,0.3,9.0,3661.63\nR003,2012-03-03,-2.0,11.3,25200.0\nR003,2012-03-04,-1.0,0.9,39600.0\nR003,2012-03-05,-1.4,7.0,32917.29\nR003,2012-03-06,-2.2,11.5,19715.3\nR003,2012-03-07,-3.7,5.1,32805.81\nR003,2012-03-08,-4.4,0.0,39600.0\nR003,2012-03-09,-3.9,0.1,39600.0\nR003,2012-03-10,-5.2,0.7,32400.0\nR003,2012-03-11,-3.6,0.0,39600.0\nR003,2012-03-12,0.4,5.4,8761.16\nR003,2012-03-13,4.3,2.5,15221.13\nR003,2012-03-14,4.0,2.1,30526.92\nR003,2012-03-15,-0.4,9.4,30303.45\nR003,2012-03-16,-3.7,1.6,37809.67\nR003,2012-03-17,-3.1,0.0,40031.43\nR003,2012-03-18,2.2,0.0,39877.6\nR003,2012-03-19,7.2,0.0,41058.19\nR003,2012-03-20,8.9,0.0,41140.77\nR003,2012-03-21,7.1,0.0,39449.27\nR003,2012-03-22,3.2,8.9,10350.67\nR003,2012-03-23,3.2,0.7,39740.7\nR003,2012-03-24,4.9,5.4,14904.2\nR003,2012-03-25,6.2,1.1,40515.98\nR003,2012-03-26,6.3,5.4,38661.25\nR003,2012-03-27,-1.0,4.4,43215.26\nR003,2012-03-28,-0.3,0.0,38198.28\nR003,2012-03-29,1.7,2.2,17465.77\nR003,2012-03-30,6.4,0.0,42697.81\nR003,2012-03-31,4.6,17.5,10699.09\nR003,2012-04-01,6.5,0.0,42439.93\nR003,2012-04-02,5.1,35.1,459.3\nR003,2012-04-03,5.8,0.0,45753.15\nR003,2012-04-04,9.9,0.0,42761.78\nR003,2012-04-05,11.8,0.0,45653.61\nR003,2012-04-06,14.2,0.0,38160.83\nR003,2012-04-07,12.6,3.5,34002.86\nR003,2012-04-08,14.0,0.6,43612.85\nR003,2012-04-09,13.8,1.0,39130.64\nR003,2012-04-10,10.8,5.0,36604.43\nR003,2012-04-11,12.1,4.3,42082.89\nR003,2012-04-12,8.1,17.1,14423.92\nR003,2012-04-13,9.9,0.6,46541.15\nR003,2012-04-14,12.0,0.0,46612.64\nR003,2012-04-15,14.6,0.0,39321.87\nR003,2012-04-16,15.3,2.9,44992.18\nR003,2012-04-17,12.7,5.5,10591.4\nR003,2012-04-18,15.1,0.0,46800.0\nR003,2012-04-19,14.8,5.7,13969.39\nR003,2012-04-20,12.1,0.0,45259.35\nR003,2012-04-21,12.9,0.2,46800.0\nR003,2012-04-22,12.8,5.1,46461.47\nR003,2012-04-23,11.7,1.7,46800.0\nR003,2012-04-24,13.5,0.0,47438.4\nR003,2012-04-25,15.6,0.0,47728.26\nR003,2012-04-26,16.6,0.5,47687.99\nR003,2012-04-27,16.2,3.7,47648.15\nR003,2012-04-28,16.2,1.0,47942.15\nR003,2012-04-29,16.1,1.3,48238.08\nR003,2012-04-30,17.2,0.0,48201.06\nR003,2012-05-01,16.6,0.0,47828.99\nR003,2012-05-02,17.7,0.0,48801.49\nR003,2012-05-03,18.3,0.5,46768.09\nR003,2012-05-04,18.1,2.9,42433.56\nR003,2012-05-05,17.2,0.2,47686.3\nR003,2012-05-06,16.1,3.6,49010.08\nR003,2012-05-07,16.1,8.2,31426.23\nR003,2012-05-08,16.0,9.8,18597.35\nR003,2012-05-09,16.4,8.8,40676.86\nR003,2012-05-10,18.2,3.3,50147.97\nR003,2012-05-11,18.9,2.0,47447.51\nR003,2012-05-12,17.5,7.5,15881.45\nR003,2012-05-13,17.3,8.9,43436.58\nR003,2012-05-14,16.8,5.6,40601.64\nR003,2012-05-15,16.3,5.9,28525.78\nR003,2012-05-16,18.4,0.8,49501.77\nR003,2012-05-17,19.3,3.0,44457.91\nR003,2012-05-18,18.4,6.4,48655.18\nR003,2012-05-19,20.7,0.8,50400.0\nR003,2012-05-20,19.7,1.7,34037.06\nR003,2012-05-21,18.2,11.5,36094.81\nR003,2012-05-22,17.4,4.0,29043.17\nR003,2012-05-23,17.1,13.1,32627.39\nR003,2012-05-24,16.1,6.9,30870.76\nR003,2012-05-25,14.4,1.0,46649.3\nR003,2012-05-26,14.2,6.3,29998.25\nR003,2012-05-27,15.9,4.9,30544.97\nR003,2012-05-28,18.5,2.9,48065.1\nR003,2012-05-29,17.9,6.2,41620.35\nR003,2012-05-30,17.4,6.5,39624.68\nR003,2012-05-31,18.3,8.4,46777.21\nR003,2012-06-01,16.5,8.9,41834.54\nR003,2012-06-02,16.6,0.1,50597.45\nR003,2012-06-03,18.9,0.0,52527.14\nR003,2012-06-04,21.5,0.0,49434.23\nR003,2012-06-05,20.5,16.1,44922.16\nR003,2012-06-06,21.7,0.0,52844.42\nR003,2012-06-07,19.8,2.7,45795.22\nR003,2012-06-08,17.1,16.5,17943.48\nR003,2012-06-09,18.7,1.0,49323.69\nR003,2012-06-10,21.5,0.0,53003.78\nR003,2012-06-11,24.4,0.0,53140.49\nR003,2012-06-12,24.4,1.8,51984.93\nR003,2012-06-13,25.4,0.6,53115.91\nR003,2012-06-14,27.2,0.6,53806.61\nR003,2012-06-15,24.8,4.3,45840.57\nR003,2012-06-16,21.5,7.8,46468.73\nR003,2012-06-17,18.2,1.7,46737.26\nR003,2012-06-18,20.7,0.1,48729.13\nR003,2012-06-19,22.0,0.0,53791.91\nR003,2012-06-20,22.3,0.0,53928.35\nR003,2012-06-21,22.3,0.2,53940.73\nR003,2012-06-22,22.7,0.6,53771.8\nR003,2012-06-23,23.0,1.2,53957.97\nR003,2012-06-24,23.5,0.1,53962.87\nR003,2012-06-25,23.7,3.3,52656.77\nR003,2012-06-26,22.2,7.0,52631.13\nR003,2012-06-27,21.2,11.3,40724.76\nR003,2012-06-28,18.3,9.6,5588.72\nR003,2012-06-29,17.7,3.7,50564.04\nR003,2012-06-30,15.7,15.2,35852.3\nR003,2012-07-01,15.1,11.8,29048.48\nR003,2012-07-02,16.9,1.1,37163.61\nR003,2012-07-03,18.8,5.1,43404.0\nR003,2012-07-04,17.8,9.4,35514.63\nR003,2012-07-05,17.9,16.0,40198.01\nR003,2012-07-06,19.3,2.9,46935.09\nR003,2012-07-07,21.0,3.3,42482.15\nR003,2012-07-08,22.2,1.6,47537.3\nR003,2012-07-09,22.3,0.7,51804.52\nR003,2012-07-10,19.7,10.1,50593.2\nR003,2012-07-11,19.9,1.7,45526.37\nR003,2012-07-12,20.9,8.8,48356.15\nR003,2012-07-13,22.9,0.3,50431.01\nR003,2012-07-14,22.7,1.8,50987.04\nR003,2012-07-15,23.9,0.1,53292.18\nR003,2012-07-16,24.4,0.0,52045.19\nR003,2012-07-17,24.4,0.0,52417.62\nR003,2012-07-18,23.1,0.2,50490.33\nR003,2012-07-19,22.3,0.2,52450.01\nR003,2012-07-20,22.3,0.0,50986.77\nR003,2012-07-21,24.6,0.0,53080.11\nR003,2012-07-22,25.7,0.0,52334.09\nR003,2012-07-23,26.9,1.3,52295.26\nR003,2012-07-24,26.6,0.0,50449.0\nR003,2012-07-25,27.1,0.0,51867.23\nR003,2012-07-26,27.6,0.2,51829.47\nR003,2012-07-27,27.1,0.1,51443.12\nR003,2012-07-28,26.2,0.8,49862.18\nR003,2012-07-29,24.3,4.6,46914.88\nR003,2012-07-30,21.7,8.2,46788.78\nR003,2012-07-31,23.0,1.8,49057.99\nR003,2012-08-01,22.5,14.0,33280.92\nR003,2012-08-02,23.2,6.8,47452.78\nR003,2012-08-03,24.0,4.6,47856.61\nR003,2012-08-04,25.2,3.4,48601.96\nR003,2012-08-05,23.8,1.4,47227.46\nR003,2012-08-06,23.7,7.4,29894.18\nR003,2012-08-07,22.7,7.4,44070.31\nR003,2012-08-08,23.1,7.5,43041.74\nR003,2012-08-09,22.4,7.1,47087.94\nR003,2012-08-10,21.6,8.1,39525.37\nR003,2012-08-11,21.4,8.9,45629.82\nR003,2012-08-12,24.0,1.5,47094.86\nR003,2012-08-13,23.4,4.9,47125.3\nR003,2012-08-14,22.8,9.1,46311.5\nR003,2012-08-15,22.4,7.7,38742.48\nR003,2012-08-16,21.1,25.0,23091.41\nR003,2012-08-17,20.4,0.2,45391.8\nR003,2012-08-18,20.1,0.4,46800.0\nR003,2012-08-19,19.1,8.8,39358.35\nR003,2012-08-20,18.7,8.4,44794.16\nR003,2012-08-21,18.3,0.0,36804.8\nR003,2012-08-22,19.6,6.2,36000.0\nR003,2012-08-23,19.4,0.0,46800.0\nR003,2012-08-24,20.8,0.0,46800.0\nR003,2012-08-25,23.1,0.0,46800.0\nR003,2012-08-26,24.4,0.0,46800.0\nR003,2012-08-27,25.1,0.0,46698.6\nR003,2012-08-28,23.7,2.9,45056.38\nR003,2012-08-29,18.7,13.4,30478.85\nR003,2012-08-30,17.4,0.0,46386.58\nR003,2012-08-31,18.7,0.0,41265.89\nR003,2012-09-01,19.9,0.9,32891.7\nR003,2012-09-02,20.6,0.0,45297.35\nR003,2012-09-03,20.9,0.0,45347.01\nR003,2012-09-04,21.0,0.0,45076.76\nR003,2012-09-05,20.4,0.0,44167.53\nR003,2012-09-06,20.1,0.0,44219.39\nR003,2012-09-07,19.8,0.0,40855.09\nR003,2012-09-08,18.6,0.0,42379.49\nR003,2012-09-09,18.0,0.0,43200.0\nR003,2012-09-10,17.2,0.0,41797.81\nR003,2012-09-11,16.8,0.0,40853.5\nR003,2012-09-12,18.8,0.0,43259.42\nR003,2012-09-13,21.4,0.0,43149.72\nR003,2012-09-14,20.7,2.4,41680.69\nR003,2012-09-15,19.7,0.2,43015.23\nR003,2012-09-16,20.6,0.0,42947.8\nR003,2012-09-17,20.7,0.0,41142.85\nR003,2012-09-18,21.3,0.0,42812.54\nR003,2012-09-19,21.1,0.0,42744.71\nR003,2012-09-20,21.0,1.6,39279.78\nR003,2012-09-21,19.6,3.7,40404.1\nR003,2012-09-22,17.8,3.4,23706.03\nR003,2012-09-23,16.1,18.4,34176.42\nR003,2012-09-24,17.0,0.0,39791.5\nR003,2012-09-25,19.7,0.0,41739.18\nR003,2012-09-26,19.5,0.0,41456.52\nR003,2012-09-27,19.3,0.0,40854.72\nR003,2012-09-28,19.5,0.0,41210.33\nR003,2012-09-29,20.0,0.0,41162.65\nR003,2012-09-30,19.5,0.0,40772.33\nR003,2012-10-01,20.0,0.0,40382.78\nR003,2012-10-02,20.4,0.0,39993.75\nR003,2012-10-03,21.2,0.0,35737.96\nR003,2012-10-04,19.7,2.7,25734.07\nR003,2012-10-05,17.3,11.4,0.0\nR003,2012-10-06,17.3,0.2,33904.9\nR003,2012-10-07,17.5,0.0,37902.98\nR003,2012-10-08,17.9,2.7,34522.49\nR003,2012-10-09,13.9,12.2,24769.02\nR003,2012-10-10,12.0,0.0,36775.99\nR003,2012-10-11,13.3,6.5,16240.69\nR003,2012-10-12,14.0,25.9,25760.95\nR003,2012-10-13,13.2,0.0,36000.0\nR003,2012-10-14,15.3,0.0,36342.83\nR003,2012-10-15,17.1,0.0,36396.03\nR003,2012-10-16,17.2,0.0,36120.87\nR003,2012-10-17,17.2,0.0,36000.0\nR003,2012-10-18,16.2,0.6,36000.0\nR003,2012-10-19,14.1,2.2,32497.92\nR003,2012-10-20,14.6,0.0,36000.0\nR003,2012-10-21,15.5,0.1,36000.0\nR003,2012-10-22,15.8,0.0,36000.0\nR003,2012-10-23,14.6,0.0,36000.0\nR003,2012-10-24,14.0,0.1,31102.23\nR003,2012-10-25,13.2,0.5,36000.0\nR003,2012-10-26,13.2,0.4,34686.15\nR003,2012-10-27,11.9,0.0,36000.0\nR003,2012-10-28,14.0,0.0,36000.0\nR003,2012-10-29,15.2,1.1,31442.68\nR003,2012-10-30,18.1,0.0,30856.23\nR003,2012-10-31,14.5,3.8,33561.65\nR003,2012-11-01,12.3,3.5,8721.11\nR003,2012-11-02,12.2,0.0,30953.06\nR003,2012-11-03,15.4,3.7,32415.95\nR003,2012-11-04,13.0,2.7,34236.24\nR003,2012-11-05,12.5,0.0,35623.3\nR003,2012-11-06,14.3,0.2,35468.39\nR003,2012-11-07,13.3,44.1,1080.72\nR003,2012-11-08,10.7,3.8,13784.4\nR003,2012-11-09,9.9,9.9,0.0\nR003,2012-11-10,11.7,7.8,0.0\nR003,2012-11-11,11.7,7.2,19603.0\nR003,2012-11-12,12.4,0.0,32309.26\nR003,2012-11-13,11.2,0.0,34091.78\nR003,2012-11-14,10.2,0.3,28189.53\nR003,2012-11-15,9.5,1.2,32647.14\nR003,2012-11-16,9.7,0.0,31242.26\nR003,2012-11-17,9.1,0.0,32970.41\nR003,2012-11-18,8.3,0.0,33329.0\nR003,2012-11-19,9.2,0.0,32959.95\nR003,2012-11-20,8.3,0.0,32590.76\nR003,2012-11-21,7.9,0.0,32588.89\nR003,2012-11-22,8.4,0.0,31378.75\nR003,2012-11-23,8.8,0.0,31447.46\nR003,2012-11-24,7.8,0.3,20547.66\nR003,2012-11-25,7.5,0.9,15421.28\nR003,2012-11-26,6.8,0.2,30231.95\nR003,2012-11-27,5.9,0.0,29878.52\nR003,2012-11-28,9.6,0.0,30653.88\nR003,2012-11-29,10.2,0.0,21824.84\nR003,2012-11-30,10.8,0.0,30315.09\nR003,2012-12-01,13.1,0.0,28800.0\nR003,2012-12-02,14.1,0.0,27801.81\nR003,2012-12-03,13.0,6.2,29216.21\nR003,2012-12-04,9.8,19.3,25200.0\nR003,2012-12-05,8.8,5.2,1687.51\nR003,2012-12-06,6.9,3.2,26162.35\nR003,2012-12-07,6.6,2.6,25200.0\nR003,2012-12-08,4.3,7.2,0.0\nR003,2012-12-09,2.3,0.0,30027.62\nR003,2012-12-10,5.5,0.0,30804.21\nR003,2012-12-11,5.4,2.3,28800.0\nR003,2012-12-12,7.2,4.7,27090.14\nR003,2012-12-13,6.3,6.6,0.0\nR003,2012-12-14,1.9,5.6,27044.9\nR003,2012-12-15,-1.4,0.4,28855.33\nR003,2012-12-16,-0.3,0.2,30018.16\nR003,2012-12-17,-2.0,0.0,30791.66\nR003,2012-12-18,-0.5,0.9,2975.37\nR003,2012-12-19,3.0,0.3,14784.43\nR003,2012-12-20,5.4,1.1,10309.13\nR003,2012-12-21,5.5,2.5,19867.31\nR003,2012-12-22,4.6,9.2,14458.83\nR003,2012-12-23,3.8,15.6,8784.22\nR003,2012-12-24,0.4,6.9,20583.32\nR003,2012-12-25,-2.8,0.8,15363.85\nR003,2012-12-26,-1.5,0.0,31333.23\nR003,2012-12-27,6.1,0.0,29076.46\nR003,2012-12-28,9.8,0.0,3533.86\nR003,2012-12-29,6.7,5.1,24879.95\nR003,2012-12-30,0.8,5.8,30752.16\nR003,2012-12-31,-3.4,0.0,31572.73\nR003,2013-01-01,-2.2,0.0,31626.64\nR003,2013-01-02,-0.5,0.0,31682.26\nR003,2013-01-03,0.9,0.0,31739.44\nR003,2013-01-04,2.5,9.4,2451.05\nR003,2013-01-05,3.1,13.2,772.95\nR003,2013-01-06,1.1,2.0,24164.46\nR003,2013-01-07,-0.1,6.8,0.0\nR003,2013-01-08,0.3,12.7,10426.63\nR003,2013-01-09,-3.6,0.0,30233.84\nR003,2013-01-10,-5.5,0.0,28898.24\nR003,2013-01-11,-2.5,0.0,27973.69\nR003,2013-01-12,2.2,2.5,78.07\nR003,2013-01-13,4.0,35.6,1185.57\nR003,2013-01-14,-2.6,0.2,32400.0\nR003,2013-01-15,-5.1,0.0,32400.0\nR003,2013-01-16,-0.7,0.0,23285.15\nR003,2013-01-17,4.2,1.3,19511.81\nR003,2013-01-18,6.0,1.8,31052.7\nR003,2013-01-19,7.3,0.6,30964.0\nR003,2013-01-20,4.5,24.7,0.0\nR003,2013-01-21,5.7,13.0,6150.16\nR003,2013-01-22,8.8,0.0,30007.47\nR003,2013-01-23,7.7,10.5,16261.34\nR003,2013-01-24,5.7,12.9,30247.46\nR003,2013-01-25,4.6,3.8,20602.07\nR003,2013-01-26,6.6,8.6,27503.55\nR003,2013-01-27,7.2,0.1,25200.0\nR003,2013-01-28,5.7,2.3,23973.15\nR003,2013-01-29,4.6,0.0,24514.28\nR003,2013-01-30,3.9,7.4,18000.0\nR003,2013-01-31,3.4,3.6,31334.34\nR003,2013-02-01,2.8,8.3,3604.79\nR003,2013-02-02,1.7,6.5,25661.03\nR003,2013-02-03,3.7,0.0,32621.53\nR003,2013-02-04,8.2,0.0,31706.21\nR003,2013-02-05,6.3,7.3,11301.9\nR003,2013-02-06,6.6,0.0,32834.07\nR003,2013-02-07,8.8,0.0,30519.25\nR003,2013-02-08,8.1,0.0,34664.52\nR003,2013-02-09,8.8,0.0,35257.65\nR003,2013-02-10,8.1,0.0,34364.43\nR003,2013-02-11,6.9,0.0,34656.2\nR003,2013-02-12,5.7,4.1,0.0\nR003,2013-02-13,4.7,0.0,35206.17\nR003,2013-02-14,5.0,0.0,32940.72\nR003,2013-02-15,7.6,0.7,34679.18\nR003,2013-02-16,5.1,10.5,16684.46\nR003,2013-02-17,1.8,8.0,31337.67\nR003,2013-02-18,0.9,1.3,34892.73\nR003,2013-02-19,1.2,1.0,33524.42\nR003,2013-02-20,1.6,0.2,24728.29\nR003,2013-02-21,4.0,5.3,24735.44\nR003,2013-02-22,4.1,0.3,26256.54\nR003,2013-02-23,6.8,7.0,527.88\nR003,2013-02-24,6.6,5.9,0.0\nR003,2013-02-25,6.8,0.0,37241.01\nR003,2013-02-26,7.4,0.0,33286.81\nR003,2013-02-27,7.6,0.1,39372.86\nR003,2013-02-28,6.6,2.9,26871.88\nR003,2013-03-01,6.3,5.9,30644.77\nR003,2013-03-02,4.9,2.7,30938.51\nR003,2013-03-03,2.2,11.9,27642.67\nR003,2013-03-04,1.9,0.7,13427.21\nR003,2013-03-05,-1.1,6.2,36878.42\nR003,2013-03-06,-2.0,0.0,39600.0\nR003,2013-03-07,4.1,0.0,39600.0\nR003,2013-03-08,9.9,0.0,39600.0\nR003,2013-03-09,7.1,2.8,3961.38\nR003,2013-03-10,7.7,10.2,25879.88\nR003,2013-03-11,8.8,15.1,23730.09\nR003,2013-03-12,9.6,6.9,18255.71\nR003,2013-03-13,8.7,31.7,15701.23\nR003,2013-03-14,9.9,0.0,39310.76\nR003,2013-03-15,16.5,0.0,37920.2\nR003,2013-03-16,14.6,23.9,3164.92\nR003,2013-03-17,7.5,40.8,0.0\nR003,2013-03-18,2.2,9.5,26118.35\nR003,2013-03-19,5.0,0.0,32015.39\nR003,2013-03-20,9.2,2.2,34956.11\nR003,2013-03-21,9.6,1.1,40259.82\nR003,2013-03-22,14.2,0.0,26735.04\nR003,2013-03-23,8.3,21.0,19721.47\nR003,2013-03-24,0.3,16.9,23351.36\nR003,2013-03-25,2.4,5.9,6429.26\nR003,2013-03-26,6.6,5.8,18587.65\nR003,2013-03-27,8.3,12.2,16683.64\nR003,2013-03-28,7.7,14.3,18518.02\nR003,2013-03-29,4.4,2.0,40065.52\nR003,2013-03-30,6.7,0.0,44691.93\nR003,2013-03-31,13.9,0.0,44763.32\nR003,2013-04-01,16.6,0.0,45153.54\nR003,2013-04-02,14.8,6.3,31706.52\nR003,2013-04-03,14.4,1.2,44612.39\nR003,2013-04-04,17.2,0.0,45691.5\nR003,2013-04-05,15.6,1.7,43743.03\nR003,2013-04-06,15.7,0.0,44349.4\nR003,2013-04-07,16.1,0.0,40633.84\nR003,2013-04-08,18.2,8.0,39163.82\nR003,2013-04-09,11.4,8.2,22813.19\nR003,2013-04-10,9.1,4.4,33240.43\nR003,2013-04-11,8.7,7.3,18104.98\nR003,2013-04-12,9.2,1.4,37375.9\nR003,2013-04-13,10.6,0.9,46382.3\nR003,2013-04-14,11.9,0.3,46594.76\nR003,2013-04-15,13.6,0.8,46665.91\nR003,2013-04-16,13.3,5.9,36647.91\nR003,2013-04-17,9.4,15.7,13590.73\nR003,2013-04-18,9.4,5.6,15637.0\nR003,2013-04-19,10.8,2.0,41298.63\nR003,2013-04-20,11.2,7.8,41465.1\nR003,2013-04-21,9.9,5.9,2786.32\nR003,2013-04-22,9.1,9.8,29821.13\nR003,2013-04-23,6.4,13.8,41939.31\nR003,2013-04-24,6.6,2.7,45897.06\nR003,2013-04-25,10.0,0.6,36801.04\nR003,2013-04-26,11.1,2.9,39425.49\nR003,2013-04-27,13.9,0.0,46800.0\nR003,2013-04-28,18.1,0.0,47951.45\nR003,2013-04-29,19.8,0.0,48580.35\nR003,2013-04-30,18.9,0.0,48208.64\nR003,2013-05-01,19.8,0.0,48842.56\nR003,2013-05-02,20.3,0.0,48807.67\nR003,2013-05-03,19.6,0.0,49110.55\nR003,2013-05-04,19.9,1.6,49077.99\nR003,2013-05-05,17.6,2.2,46613.93\nR003,2013-05-06,17.3,0.3,46941.38\nR003,2013-05-07,16.6,0.7,44966.19\nR003,2013-05-08,17.0,0.8,46800.0\nR003,2013-05-09,17.3,0.1,49269.76\nR003,2013-05-10,17.1,3.2,49570.72\nR003,2013-05-11,18.5,5.5,47831.02\nR003,2013-05-12,16.1,20.0,17338.02\nR003,2013-05-13,15.9,5.8,46229.64\nR003,2013-05-14,17.0,7.5,45266.47\nR003,2013-05-15,16.6,5.4,29754.04\nR003,2013-05-16,16.4,4.4,32011.21\nR003,2013-05-17,17.2,0.3,48151.89\nR003,2013-05-18,16.5,3.0,48411.57\nR003,2013-05-19,16.4,5.3,42901.06\nR003,2013-05-20,18.1,3.1,50543.43\nR003,2013-05-21,18.9,0.9,47767.46\nR003,2013-05-22,19.1,0.4,49106.63\nR003,2013-05-23,20.3,0.0,50963.92\nR003,2013-05-24,23.1,0.3,48928.63\nR003,2013-05-25,19.6,7.3,44450.62\nR003,2013-05-26,17.6,1.8,34487.08\nR003,2013-05-27,18.3,1.4,47514.9\nR003,2013-05-28,17.1,4.3,50400.0\nR003,2013-05-29,16.9,5.8,47271.3\nR003,2013-05-30,19.7,1.0,51522.26\nR003,2013-05-31,21.4,0.0,51864.04\nR003,2013-06-01,19.0,1.4,50400.0\nR003,2013-06-02,18.7,0.0,51462.89\nR003,2013-06-03,16.8,11.7,41481.44\nR003,2013-06-04,15.4,6.5,44961.79\nR003,2013-06-05,17.6,0.4,52499.99\nR003,2013-06-06,17.9,5.5,50659.29\nR003,2013-06-07,15.5,14.4,0.0\nR003,2013-06-08,16.3,13.4,36103.79\nR003,2013-06-09,17.3,4.9,50017.69\nR003,2013-06-10,18.1,6.6,45942.14\nR003,2013-06-11,17.4,10.6,30624.83\nR003,2013-06-12,19.3,2.7,50778.32\nR003,2013-06-13,21.8,1.4,53484.39\nR003,2013-06-14,23.9,0.1,52738.67\nR003,2013-06-15,23.0,6.3,52727.9\nR003,2013-06-16,20.5,9.1,38156.47\nR003,2013-06-17,20.3,16.3,48801.63\nR003,2013-06-18,20.2,4.7,35346.99\nR003,2013-06-19,20.6,0.6,51589.54\nR003,2013-06-20,18.4,0.3,51881.04\nR003,2013-06-21,17.7,0.6,49430.6\nR003,2013-06-22,16.9,3.6,45497.86\nR003,2013-06-23,18.7,0.7,46398.35\nR003,2013-06-24,19.9,0.1,51384.76\nR003,2013-06-25,21.0,1.1,51924.81\nR003,2013-06-26,22.2,0.1,53963.47\nR003,2013-06-27,22.8,1.2,53963.83\nR003,2013-06-28,23.9,0.2,53962.1\nR003,2013-06-29,22.8,1.3,53958.26\nR003,2013-06-30,23.3,0.1,53952.26\nR003,2013-07-01,22.5,15.0,49023.23\nR003,2013-07-02,19.3,3.4,47641.85\nR003,2013-07-03,18.9,15.1,32624.36\nR003,2013-07-04,19.1,16.5,39994.83\nR003,2013-07-05,20.9,0.4,49695.92\nR003,2013-07-06,21.2,0.6,53390.01\nR003,2013-07-07,22.6,0.7,53757.42\nR003,2013-07-08,23.1,1.4,53762.34\nR003,2013-07-09,23.6,0.0,53796.27\nR003,2013-07-10,22.1,1.0,52690.78\nR003,2013-07-11,21.7,1.2,52026.8\nR003,2013-07-12,21.7,3.7,51402.72\nR003,2013-07-13,22.4,1.1,52361.21\nR003,2013-07-14,21.6,2.2,53449.9\nR003,2013-07-15,22.3,1.5,51027.24\nR003,2013-07-16,21.8,4.3,48852.23\nR003,2013-07-17,22.3,3.0,48667.62\nR003,2013-07-18,21.8,4.4,46888.44\nR003,2013-07-19,21.9,1.1,44443.66\nR003,2013-07-20,21.7,0.7,51355.1\nR003,2013-07-21,21.4,1.7,50964.31\nR003,2013-07-22,20.5,0.2,47585.06\nR003,2013-07-23,19.8,0.0,49690.12\nR003,2013-07-24,19.9,0.0,50439.49\nR003,2013-07-25,19.5,0.1,50119.94\nR003,2013-07-26,19.5,0.7,50178.94\nR003,2013-07-27,19.1,1.8,48830.41\nR003,2013-07-28,18.4,11.5,45041.3\nR003,2013-07-29,17.0,5.8,44355.37\nR003,2013-07-30,16.8,5.2,48116.24\nR003,2013-07-31,17.9,0.0,48069.48\nR003,2013-08-01,19.4,0.0,45481.62\nR003,2013-08-02,20.0,1.6,48509.5\nR003,2013-08-03,19.3,0.6,44505.28\nR003,2013-08-04,17.4,8.5,43226.95\nR003,2013-08-05,18.0,8.0,43200.0\nR003,2013-08-06,19.7,1.1,42313.79\nR003,2013-08-07,19.6,3.2,40754.58\nR003,2013-08-08,20.4,1.4,46591.03\nR003,2013-08-09,20.6,0.0,47673.75\nR003,2013-08-10,21.1,0.0,47365.04\nR003,2013-08-11,22.1,0.0,47731.09\nR003,2013-08-12,22.9,0.3,47759.91\nR003,2013-08-13,21.0,8.8,37665.95\nR003,2013-08-14,21.9,2.6,46800.0\nR003,2013-08-15,23.4,0.8,46800.0\nR003,2013-08-16,23.2,2.7,46877.37\nR003,2013-08-17,23.0,2.8,46800.0\nR003,2013-08-18,20.7,7.3,35952.55\nR003,2013-08-19,21.2,0.0,46800.0\nR003,2013-08-20,21.0,0.3,46800.0\nR003,2013-08-21,21.1,0.1,46800.0\nR003,2013-08-22,21.2,0.0,46800.0\nR003,2013-08-23,22.3,0.1,46800.0\nR003,2013-08-24,20.1,6.7,46315.95\nR003,2013-08-25,20.0,5.5,43683.53\nR003,2013-08-26,20.8,0.2,46503.31\nR003,2013-08-27,21.8,0.5,46296.0\nR003,2013-08-28,22.0,0.4,46614.63\nR003,2013-08-29,23.7,0.0,46510.48\nR003,2013-08-30,23.7,0.2,44175.53\nR003,2013-08-31,20.3,16.9,31018.32\nR003,2013-09-01,18.3,24.7,21123.88\nR003,2013-09-02,18.4,0.0,44644.97\nR003,2013-09-03,18.3,0.0,40598.21\nR003,2013-09-04,15.7,14.3,31286.24\nR003,2013-09-05,13.0,7.1,30765.25\nR003,2013-09-06,13.1,7.3,35220.33\nR003,2013-09-07,13.9,2.1,41668.1\nR003,2013-09-08,15.1,0.1,43200.0\nR003,2013-09-09,14.5,5.4,39976.22\nR003,2013-09-10,14.4,0.0,42179.72\nR003,2013-09-11,16.1,0.0,43200.0\nR003,2013-09-12,16.4,0.0,41555.99\nR003,2013-09-13,18.3,0.0,41640.58\nR003,2013-09-14,20.5,1.5,41361.84\nR003,2013-09-15,17.0,21.9,15181.83\nR003,2013-09-16,15.2,8.7,30268.84\nR003,2013-09-17,15.7,2.7,0.0\nR003,2013-09-18,17.9,1.2,34554.41\nR003,2013-09-19,18.5,2.0,36298.91\nR003,2013-09-20,18.5,2.8,36996.63\nR003,2013-09-21,16.7,3.9,23015.52\nR003,2013-09-22,14.7,5.1,37172.56\nR003,2013-09-23,12.4,1.5,36276.02\nR003,2013-09-24,11.2,75.3,225.71\nR003,2013-09-25,9.6,29.8,19427.64\nR003,2013-09-26,10.4,0.0,41128.46\nR003,2013-09-27,14.8,0.0,41483.54\nR003,2013-09-28,11.7,6.3,6764.57\nR003,2013-09-29,8.7,10.5,24884.15\nR003,2013-09-30,10.3,2.1,24849.54\nR003,2013-10-01,15.2,2.1,32420.01\nR003,2013-10-02,16.6,2.6,29305.66\nR003,2013-10-03,11.3,40.3,0.0\nR003,2013-10-04,7.4,3.0,25755.51\nR003,2013-10-05,5.5,4.5,30658.15\nR003,2013-10-06,5.6,2.9,36542.63\nR003,2013-10-07,7.2,0.1,36922.18\nR003,2013-10-08,7.5,0.0,37351.8\nR003,2013-10-09,10.5,0.0,37770.23\nR003,2013-10-10,13.9,0.0,37540.16\nR003,2013-10-11,15.2,0.0,35850.52\nR003,2013-10-12,13.7,0.0,36550.43\nR003,2013-10-13,11.9,0.0,31714.23\nR003,2013-10-14,11.8,0.0,34313.76\nR003,2013-10-15,11.4,0.0,36055.68\nR003,2013-10-16,13.5,0.0,34557.04\nR003,2013-10-17,17.2,0.0,35869.36\nR003,2013-10-18,20.1,0.0,36000.0\nR003,2013-10-19,12.8,33.3,16420.1\nR003,2013-10-20,9.0,2.9,33920.41\nR003,2013-10-21,8.9,0.0,36000.0\nR003,2013-10-22,11.4,0.9,36000.0\nR003,2013-10-23,10.3,1.5,34389.33\nR003,2013-10-24,10.0,0.0,36000.0\nR003,2013-10-25,11.5,0.0,36000.0\nR003,2013-10-26,11.2,0.0,36000.0\nR003,2013-10-27,10.0,0.0,36000.0\nR003,2013-10-28,9.9,0.8,29823.27\nR003,2013-10-29,11.6,0.0,36000.0\nR003,2013-10-30,13.1,0.0,36000.0\nR003,2013-10-31,13.8,0.0,36000.0\nR003,2013-11-01,12.1,0.0,36000.0\nR003,2013-11-02,12.5,0.0,35928.1\nR003,2013-11-03,12.2,0.0,35849.14\nR003,2013-11-04,13.4,0.0,35770.35\nR003,2013-11-05,14.5,0.0,35301.76\nR003,2013-11-06,15.5,0.0,31050.91\nR003,2013-11-07,14.4,0.0,27451.32\nR003,2013-11-08,9.7,14.7,27109.63\nR003,2013-11-09,7.4,0.0,35066.93\nR003,2013-11-10,10.6,0.0,34919.55\nR003,2013-11-11,12.7,0.0,34773.45\nR003,2013-11-12,11.8,0.0,34628.79\nR003,2013-11-13,10.6,0.0,33172.66\nR003,2013-11-14,8.8,4.1,26840.66\nR003,2013-11-15,7.8,0.2,31709.29\nR003,2013-11-16,5.6,0.0,33706.76\nR003,2013-11-17,5.4,0.1,24934.42\nR003,2013-11-18,5.8,0.0,32600.53\nR003,2013-11-19,7.3,0.0,32959.33\nR003,2013-11-20,9.8,0.0,32955.29\nR003,2013-11-21,10.0,0.0,21596.49\nR003,2013-11-22,10.8,0.0,30987.93\nR003,2013-11-23,9.7,0.0,29944.13\nR003,2013-11-24,9.9,0.0,31128.37\nR003,2013-11-25,11.6,0.0,30202.84\nR003,2013-11-26,11.9,4.5,4160.26\nR003,2013-11-27,9.5,39.5,0.0\nR003,2013-11-28,6.7,2.2,26458.43\nR003,2013-11-29,4.2,19.9,0.0\nR003,2013-11-30,1.1,6.4,27584.08\nR003,2013-12-01,0.4,0.0,24317.2\nR003,2013-12-02,5.6,14.3,0.0\nR003,2013-12-03,-2.8,19.5,27530.63\nR003,2013-12-04,-6.9,0.0,29986.53\nR003,2013-12-05,-2.5,0.0,30380.54\nR003,2013-12-06,1.4,1.3,26921.13\nR003,2013-12-07,2.4,4.3,29245.56\nR003,2013-12-08,-0.3,14.9,24641.05\nR003,2013-12-09,-3.0,4.6,29640.82\nR003,2013-12-10,-2.2,29.5,909.0\nR003,2013-12-11,-10.0,0.0,30033.22\nR003,2013-12-12,-10.6,0.0,30810.07\nR003,2013-12-13,-6.7,0.0,30033.73\nR003,2013-12-14,-2.4,6.9,20824.67\nR003,2013-12-15,-1.2,6.9,28373.14\nR003,2013-12-16,-2.8,0.0,30023.47\nR003,2013-12-17,-3.4,0.0,30406.91\nR003,2013-12-18,-2.0,0.0,30789.12\nR003,2013-12-19,0.4,0.0,30388.83\nR003,2013-12-20,1.8,0.0,26684.47\nR003,2013-12-21,3.3,0.0,30363.95\nR003,2013-12-22,2.6,0.0,30349.06\nR003,2013-12-23,1.6,0.0,29941.33\nR003,2013-12-24,1.7,0.0,31249.38\nR003,2013-12-25,0.7,0.0,31288.23\nR003,2013-12-26,2.1,0.0,31329.26\nR003,2013-12-27,5.0,0.0,15665.4\nR003,2013-12-28,3.7,0.0,31417.44\nR003,2013-12-29,3.6,0.0,31464.36\nR003,2013-12-30,3.9,0.0,31513.01\nR003,2013-12-31,3.9,0.0,31563.28\nR003,2014-01-01,3.0,0.0,31615.06\nR003,2014-01-02,2.5,0.0,31668.21\nR003,2014-01-03,1.5,0.0,31722.64\nR003,2014-01-04,1.4,0.0,31778.21\nR003,2014-01-05,1.8,0.0,31835.07\nR003,2014-01-06,2.2,0.0,27924.45\nR003,2014-01-07,4.4,2.1,26871.97\nR003,2014-01-08,2.5,0.8,32019.15\nR003,2014-01-09,2.4,0.0,32084.67\nR003,2014-01-10,4.4,0.0,32152.03\nR003,2014-01-11,4.1,4.8,3588.61\nR003,2014-01-12,2.5,0.4,21973.33\nR003,2014-01-13,1.4,28.7,6717.06\nR003,2014-01-14,-1.6,6.1,28800.0\nR003,2014-01-15,-1.2,0.0,32400.0\nR003,2014-01-16,3.8,0.0,10729.07\nR003,2014-01-17,6.9,6.4,10916.13\nR003,2014-01-18,7.2,5.1,405.73\nR003,2014-01-19,6.8,19.6,0.0\nR003,2014-01-20,5.9,11.9,0.0\nR003,2014-01-21,6.4,8.8,25391.32\nR003,2014-01-22,6.3,10.0,21600.0\nR003,2014-01-23,4.1,24.5,3887.95\nR003,2014-01-24,2.2,20.9,0.0\nR003,2014-01-25,2.3,1.1,31480.76\nR003,2014-01-26,5.7,2.9,20583.3\nR003,2014-01-27,8.0,4.4,20359.8\nR003,2014-01-28,7.5,0.0,24855.17\nR003,2014-01-29,6.5,5.5,29128.84\nR003,2014-01-30,5.5,13.5,22829.45\nR003,2014-01-31,3.0,8.1,5341.33\nR003,2014-02-01,1.6,1.5,28800.0\nR003,2014-02-02,-1.1,0.0,32774.65\nR003,2014-02-03,-2.2,0.0,33041.92\nR003,2014-02-04,-1.7,0.0,32958.15\nR003,2014-02-05,-0.4,0.0,33567.62\nR003,2014-02-06,-0.3,0.0,33826.93\nR003,2014-02-07,1.7,0.0,34428.65\nR003,2014-02-08,2.8,0.0,34681.66\nR003,2014-02-09,4.6,0.0,23164.86\nR003,2014-02-10,7.5,0.0,32132.06\nR003,2014-02-11,6.8,0.7,29449.54\nR003,2014-02-12,7.0,0.0,34313.12\nR003,2014-02-13,9.1,0.0,36259.0\nR003,2014-02-14,10.6,0.0,29784.7\nR003,2014-02-15,9.1,0.0,36283.39\nR003,2014-02-16,6.8,0.0,35109.25\nR003,2014-02-17,5.9,0.1,33621.68\nR003,2014-02-18,6.3,20.0,12322.39\nR003,2014-02-19,4.6,1.1,35512.57\nR003,2014-02-20,7.5,0.0,31833.45\nR003,2014-02-21,7.2,10.4,24785.15\nR003,2014-02-22,6.7,0.0,33680.99\nR003,2014-02-23,8.0,0.0,38633.83\nR003,2014-02-24,8.8,0.0,35768.52\nR003,2014-02-25,10.1,0.0,34788.85\nR003,2014-02-26,7.6,5.9,4040.83\nR003,2014-02-27,5.6,5.7,27683.62\nR003,2014-02-28,5.1,3.3,36545.17\nR003,2014-03-01,6.0,0.0,39557.33\nR003,2014-03-02,8.2,0.2,34089.92\nR003,2014-03-03,10.3,4.8,39514.14\nR003,2014-03-04,11.1,0.2,21843.77\nR003,2014-03-05,9.2,3.5,34250.64\nR003,2014-03-06,8.3,0.0,36696.5\nR003,2014-03-07,8.8,3.3,9020.3\nR003,2014-03-08,8.4,0.7,23435.78\nR003,2014-03-09,11.7,0.7,39600.0\nR003,2014-03-10,10.8,0.0,39600.0\nR003,2014-03-11,8.0,13.7,2288.93\nR003,2014-03-12,4.0,14.2,0.0\nR003,2014-03-13,4.2,1.7,34969.49\nR003,2014-03-14,5.5,0.5,36445.18\nR003,2014-03-15,6.8,0.0,39600.0\nR003,2014-03-16,7.5,7.6,39600.0\nR003,2014-03-17,1.1,28.0,18135.19\nR003,2014-03-18,2.9,12.9,4149.84\nR003,2014-03-19,5.7,0.0,39600.0\nR003,2014-03-20,7.3,6.8,15819.66\nR003,2014-03-21,5.4,2.6,30394.6\nR003,2014-03-22,7.2,0.0,41776.7\nR003,2014-03-23,11.2,0.0,42575.58\nR003,2014-03-24,13.2,0.0,42647.11\nR003,2014-03-25,14.1,0.0,43035.8\nR003,2014-03-26,14.7,0.6,41958.18\nR003,2014-03-27,9.4,5.9,667.56\nR003,2014-03-28,7.9,3.0,41562.11\nR003,2014-03-29,3.4,25.9,63.88\nR003,2014-03-30,-3.3,0.6,43393.02\nR003,2014-03-31,-1.1,0.0,43784.46\nR003,2014-04-01,5.5,0.0,45299.6\nR003,2014-04-02,4.1,4.0,36924.13\nR003,2014-04-03,2.3,2.7,40963.46\nR003,2014-04-04,5.2,0.0,43759.97\nR003,2014-04-05,8.5,5.8,31038.03\nR003,2014-04-06,9.5,0.0,45637.26\nR003,2014-04-07,13.4,0.8,40668.97\nR003,2014-04-08,11.7,9.2,16191.18\nR003,2014-04-09,8.9,3.7,26454.6\nR003,2014-04-10,11.0,0.0,46155.21\nR003,2014-04-11,11.8,0.0,46363.37\nR003,2014-04-12,13.4,2.7,46434.45\nR003,2014-04-13,11.1,5.9,3810.06\nR003,2014-04-14,11.7,0.4,39742.3\nR003,2014-04-15,12.2,4.0,28210.89\nR003,2014-04-16,13.9,2.9,46717.94\nR003,2014-04-17,15.6,0.4,46789.01\nR003,2014-04-18,17.8,0.2,46800.0\nR003,2014-04-19,18.9,1.3,43200.0\nR003,2014-04-20,14.8,9.9,39824.49\nR003,2014-04-21,13.5,2.4,42836.76\nR003,2014-04-22,11.6,9.5,0.0\nR003,2014-04-23,11.2,6.9,666.09\nR003,2014-04-24,12.3,0.7,45281.12\nR003,2014-04-25,14.8,0.5,38017.66\nR003,2014-04-26,13.2,10.6,16933.55\nR003,2014-04-27,14.1,6.6,30693.01\nR003,2014-04-28,17.2,1.0,43953.1\nR003,2014-04-29,14.5,8.8,37547.97\nR003,2014-04-30,14.5,3.0,39919.15\nR003,2014-05-01,14.2,9.0,32400.0\nR003,2014-05-02,13.0,2.0,42234.15\nR003,2014-05-03,13.8,2.5,48444.35\nR003,2014-05-04,15.1,1.5,44951.98\nR003,2014-05-05,17.1,6.3,13655.91\nR003,2014-05-06,15.2,14.5,24504.09\nR003,2014-05-07,14.7,6.4,11447.94\nR003,2014-05-08,17.1,0.6,46800.0\nR003,2014-05-09,19.4,3.5,32266.64\nR003,2014-05-10,19.4,14.7,39145.9\nR003,2014-05-11,19.7,1.5,48414.12\nR003,2014-05-12,18.1,7.4,46606.92\nR003,2014-05-13,16.4,12.6,17803.71\nR003,2014-05-14,16.9,9.1,21801.17\nR003,2014-05-15,19.8,0.8,47697.6\nR003,2014-05-16,22.5,0.9,50400.0\nR003,2014-05-17,23.2,0.8,50400.0\nR003,2014-05-18,19.6,13.0,35907.92\nR003,2014-05-19,16.4,37.8,2198.6\nR003,2014-05-20,16.4,9.5,14872.67\nR003,2014-05-21,16.4,10.1,18712.04\nR003,2014-05-22,17.1,7.2,43477.81\nR003,2014-05-23,18.0,8.3,45662.46\nR003,2014-05-24,19.3,5.9,50400.0\nR003,2014-05-25,17.6,16.5,11444.6\nR003,2014-05-26,16.7,3.8,15778.78\nR003,2014-05-27,17.0,7.7,27478.93\nR003,2014-05-28,18.6,6.6,46304.62\nR003,2014-05-29,19.2,2.6,49903.02\nR003,2014-05-30,21.6,0.0,50400.0\nR003,2014-05-31,24.1,2.3,50400.0\nR003,2014-06-01,21.5,9.7,47540.05\nR003,2014-06-02,20.0,15.2,29580.25\nR003,2014-06-03,17.0,22.2,21272.09\nR003,2014-06-04,18.7,3.5,46541.59\nR003,2014-06-05,21.8,0.9,50679.62\nR003,2014-06-06,19.4,11.1,12329.91\nR003,2014-06-07,18.4,14.8,30241.29\nR003,2014-06-08,18.8,8.2,42937.36\nR003,2014-06-09,19.4,3.0,51263.55\nR003,2014-06-10,19.7,5.0,52424.31\nR003,2014-06-11,17.7,3.2,48077.89\nR003,2014-06-12,15.7,9.0,42082.1\nR003,2014-06-13,15.2,0.6,51358.98\nR003,2014-06-14,17.3,0.0,49329.45\nR003,2014-06-15,18.7,0.2,52732.8\nR003,2014-06-16,21.1,1.3,52722.88\nR003,2014-06-17,21.7,2.5,53816.44\nR003,2014-06-18,21.3,0.0,53881.05\nR003,2014-06-19,20.3,1.0,45488.08\nR003,2014-06-20,23.2,0.7,53914.38\nR003,2014-06-21,19.5,16.5,45252.22\nR003,2014-06-22,18.7,0.0,49843.34\nR003,2014-06-23,18.6,0.0,52747.99\nR003,2014-06-24,19.3,0.5,52222.7\nR003,2014-06-25,19.9,1.4,49702.35\nR003,2014-06-26,20.9,1.0,47878.96\nR003,2014-06-27,21.2,0.0,53968.55\nR003,2014-06-28,21.7,1.8,53658.05\nR003,2014-06-29,20.8,2.0,52724.83\nR003,2014-06-30,21.4,1.6,52267.24\nR003,2014-07-01,23.1,1.7,53012.83\nR003,2014-07-02,22.8,4.0,51917.78\nR003,2014-07-03,22.7,2.8,49179.8\nR003,2014-07-04,22.2,2.2,51195.98\nR003,2014-07-05,21.8,9.6,40261.63\nR003,2014-07-06,20.5,34.1,3119.69\nR003,2014-07-07,21.2,7.4,44337.14\nR003,2014-07-08,19.8,0.6,49060.38\nR003,2014-07-09,19.7,2.3,46745.22\nR003,2014-07-10,21.2,0.5,50495.4\nR003,2014-07-11,24.1,0.0,53741.62\nR003,2014-07-12,27.4,0.2,53708.84\nR003,2014-07-13,25.1,0.9,52718.16\nR003,2014-07-14,22.2,11.0,49881.33\nR003,2014-07-15,22.9,4.6,50735.76\nR003,2014-07-16,25.4,0.0,52370.05\nR003,2014-07-17,25.2,4.5,52767.79\nR003,2014-07-18,22.9,9.5,43977.31\nR003,2014-07-19,22.7,3.9,51255.29\nR003,2014-07-20,21.7,3.8,49159.86\nR003,2014-07-21,21.0,11.0,37699.87\nR003,2014-07-22,21.5,2.1,48976.94\nR003,2014-07-23,23.1,0.1,51790.95\nR003,2014-07-24,25.5,0.0,51918.53\nR003,2014-07-25,23.4,2.3,50468.26\nR003,2014-07-26,24.0,1.0,49465.74\nR003,2014-07-27,23.4,4.5,37081.68\nR003,2014-07-28,21.5,9.8,43179.39\nR003,2014-07-29,20.9,4.4,48226.2\nR003,2014-07-30,21.3,3.3,48627.58\nR003,2014-07-31,22.0,2.1,48338.16\nR003,2014-08-01,22.7,1.2,48740.8\nR003,2014-08-02,24.7,0.0,50174.4\nR003,2014-08-03,24.8,0.4,48171.42\nR003,2014-08-04,23.5,0.6,46168.02\nR003,2014-08-05,22.0,5.8,41400.92\nR003,2014-08-06,20.1,16.6,2202.46\nR003,2014-08-07,20.6,2.9,46932.43\nR003,2014-08-08,22.0,0.0,46962.26\nR003,2014-08-09,22.3,0.2,47668.64\nR003,2014-08-10,21.8,3.5,47359.53\nR003,2014-08-11,22.2,1.4,46504.5\nR003,2014-08-12,22.5,0.7,46800.0\nR003,2014-08-13,22.6,1.2,47111.71\nR003,2014-08-14,24.8,0.0,46800.0\nR003,2014-08-15,27.0,0.0,47171.75\nR003,2014-08-16,26.3,1.2,46869.0\nR003,2014-08-17,26.1,0.0,46800.0\nR003,2014-08-18,25.6,4.7,46931.4\nR003,2014-08-19,24.0,4.9,40353.68\nR003,2014-08-20,23.3,4.0,39742.09\nR003,2014-08-21,23.2,1.5,43869.64\nR003,2014-08-22,23.9,0.2,46800.0\nR003,2014-08-23,23.7,0.2,46800.0\nR003,2014-08-24,23.4,1.0,46800.0\nR003,2014-08-25,24.1,1.4,46215.64\nR003,2014-08-26,23.6,6.2,46285.55\nR003,2014-08-27,24.9,0.0,46529.92\nR003,2014-08-28,25.7,0.1,46636.99\nR003,2014-08-29,23.4,3.9,40419.12\nR003,2014-08-30,22.0,0.0,43299.1\nR003,2014-08-31,21.5,0.3,45177.31\nR003,2014-09-01,22.3,1.3,44902.72\nR003,2014-09-02,23.4,0.0,45914.0\nR003,2014-09-03,25.1,0.3,45319.68\nR003,2014-09-04,23.4,3.8,44203.33\nR003,2014-09-05,22.2,1.1,43817.35\nR003,2014-09-06,23.2,0.2,44508.83\nR003,2014-09-07,23.3,2.2,43601.99\nR003,2014-09-08,22.5,4.9,43335.86\nR003,2014-09-09,19.7,18.0,33381.19\nR003,2014-09-10,20.5,0.6,41567.21\nR003,2014-09-11,18.7,3.4,16568.72\nR003,2014-09-12,19.9,0.7,42457.23\nR003,2014-09-13,18.9,0.0,43181.06\nR003,2014-09-14,18.6,1.0,41984.14\nR003,2014-09-15,18.9,4.1,36256.17\nR003,2014-09-16,17.0,4.7,37178.79\nR003,2014-09-17,18.1,0.6,41020.37\nR003,2014-09-18,16.6,8.2,17113.46\nR003,2014-09-19,17.4,7.4,2704.31\nR003,2014-09-20,16.1,6.9,15499.85\nR003,2014-09-21,15.3,6.4,23252.83\nR003,2014-09-22,15.7,0.0,40023.9\nR003,2014-09-23,17.6,1.0,41969.22\nR003,2014-09-24,13.6,33.2,24269.55\nR003,2014-09-25,10.0,10.5,25238.85\nR003,2014-09-26,11.6,0.0,40163.64\nR003,2014-09-27,14.7,0.0,39880.21\nR003,2014-09-28,18.1,7.8,16062.02\nR003,2014-09-29,13.4,20.2,16700.23\nR003,2014-09-30,13.3,4.3,39468.02\nR003,2014-10-01,13.5,2.0,29692.75\nR003,2014-10-02,13.0,0.3,27813.75\nR003,2014-10-03,12.5,0.0,33134.03\nR003,2014-10-04,13.6,0.3,33783.56\nR003,2014-10-05,13.2,8.3,23938.58\nR003,2014-10-06,13.5,0.0,28441.69\nR003,2014-10-07,12.9,0.0,37880.81\nR003,2014-10-08,13.3,0.0,28418.15\nR003,2014-10-09,13.4,0.0,37670.11\nR003,2014-10-10,13.8,0.0,37840.94\nR003,2014-10-11,15.6,0.0,37283.72\nR003,2014-10-12,16.5,0.0,36859.15\nR003,2014-10-13,16.4,0.2,36260.6\nR003,2014-10-14,16.5,0.0,36313.04\nR003,2014-10-15,16.2,8.6,30157.29\nR003,2014-10-16,15.7,9.8,24674.13\nR003,2014-10-17,13.5,18.2,1383.68\nR003,2014-10-18,11.8,28.8,0.0\nR003,2014-10-19,7.5,7.6,34870.97\nR003,2014-10-20,6.7,0.1,34810.89\nR003,2014-10-21,9.4,0.0,36000.0\nR003,2014-10-22,13.7,1.7,33173.42\nR003,2014-10-23,15.7,0.0,36000.0\nR003,2014-10-24,15.9,0.0,36000.0\nR003,2014-10-25,11.6,17.4,2879.5\nR003,2014-10-26,10.1,0.0,36000.0\nR003,2014-10-27,11.8,0.1,36000.0\nR003,2014-10-28,12.1,0.0,36000.0\nR003,2014-10-29,12.2,0.0,36000.0\nR003,2014-10-30,12.8,0.0,36000.0\nR003,2014-10-31,11.8,1.5,21470.63\nR003,2014-11-01,11.1,4.9,14548.75\nR003,2014-11-02,4.4,4.7,25900.19\nR003,2014-11-03,1.8,0.0,35873.04\nR003,2014-11-04,3.5,1.1,33857.68\nR003,2014-11-05,6.9,0.0,35713.81\nR003,2014-11-06,10.2,0.0,35558.69\nR003,2014-11-07,10.8,0.0,35404.4\nR003,2014-11-08,10.5,0.0,35251.33\nR003,2014-11-09,11.2,0.0,35099.68\nR003,2014-11-10,12.7,0.0,34031.81\nR003,2014-11-11,11.1,0.0,25201.94\nR003,2014-11-12,10.0,0.0,30604.63\nR003,2014-11-13,9.4,0.0,34510.82\nR003,2014-11-14,10.3,0.0,33525.81\nR003,2014-11-15,11.8,0.3,26268.81\nR003,2014-11-16,11.1,0.6,30094.7\nR003,2014-11-17,9.6,0.5,30102.36\nR003,2014-11-18,9.8,0.7,28009.43\nR003,2014-11-19,9.8,0.3,31487.26\nR003,2014-11-20,9.4,0.0,31560.35\nR003,2014-11-21,10.0,0.0,16187.86\nR003,2014-11-22,8.5,0.0,30228.53\nR003,2014-11-23,4.5,0.0,31780.02\nR003,2014-11-24,4.9,0.0,30358.58\nR003,2014-11-25,4.4,0.0,30800.33\nR003,2014-11-26,5.0,0.0,28769.57\nR003,2014-11-27,6.5,0.7,9661.39\nR003,2014-11-28,5.9,5.7,16169.06\nR003,2014-11-29,3.6,0.7,23166.84\nR003,2014-11-30,4.5,25.7,0.0\nR003,2014-12-01,-0.4,0.8,28783.18\nR003,2014-12-02,-3.1,0.0,26092.97\nR003,2014-12-03,-2.5,0.0,30352.84\nR003,2014-12-04,-1.2,0.0,28800.0\nR003,2014-12-05,5.5,2.5,6958.96\nR003,2014-12-06,3.4,0.0,30390.42\nR003,2014-12-07,6.6,0.0,29630.85\nR003,2014-12-08,10.3,0.0,22767.48\nR003,2014-12-09,10.7,0.0,30028.69\nR003,2014-12-10,11.9,0.6,17838.48\nR003,2014-12-11,9.0,8.1,11252.96\nR003,2014-12-12,9.9,0.1,20116.29\nR003,2014-12-13,9.1,3.9,15480.52\nR003,2014-12-14,6.1,0.0,23336.75\nR003,2014-12-15,6.3,0.0,29640.61\nR003,2014-12-16,5.6,0.0,29245.02\nR003,2014-12-17,6.9,0.0,28847.14\nR003,2014-12-18,10.7,0.0,25105.85\nR003,2014-12-19,7.2,3.7,23359.57\nR003,2014-12-20,5.7,0.0,27224.02\nR003,2014-12-21,6.4,0.3,23075.66\nR003,2014-12-22,2.6,19.8,5152.22\nR003,2014-12-23,-1.0,6.2,1363.42\nR003,2014-12-24,5.2,32.2,0.0\nR003,2014-12-25,7.0,1.5,31282.22\nR003,2014-12-26,6.0,0.0,30670.27\nR003,2014-12-27,5.6,0.0,28940.7\nR003,2014-12-28,4.7,20.8,17004.82\nR003,2014-12-29,4.9,6.9,5290.09\nR003,2014-12-30,5.8,37.2,0.0\nR003,2014-12-31,4.9,29.1,0.0\nR003,2015-01-01,4.5,9.0,3967.65\nR003,2015-01-02,5.6,3.0,6533.18\nR003,2015-01-03,3.3,7.8,20785.96\nR003,2015-01-04,0.5,11.8,22440.76\nR003,2015-01-05,0.8,24.1,0.0\nR003,2015-01-06,1.8,16.8,0.0\nR003,2015-01-07,1.7,25.0,0.0\nR003,2015-01-08,-2.9,0.5,23433.1\nR003,2015-01-09,-6.0,0.0,27241.51\nR003,2015-01-10,-2.7,0.4,27302.41\nR003,2015-01-11,-0.3,5.2,4131.0\nR003,2015-01-12,5.3,24.0,0.0\nR003,2015-01-13,4.5,17.3,22202.13\nR003,2015-01-14,5.2,0.0,32400.0\nR003,2015-01-15,5.4,0.0,32400.0\nR003,2015-01-16,3.5,0.0,29781.39\nR003,2015-01-17,2.0,0.0,32400.0\nR003,2015-01-18,1.9,0.0,32400.0\nR003,2015-01-19,3.6,0.0,30109.61\nR003,2015-01-20,4.5,2.1,32400.0\nR003,2015-01-21,6.8,0.0,30058.74\nR003,2015-01-22,6.2,0.0,28541.86\nR003,2015-01-23,4.6,0.0,32400.0\nR003,2015-01-24,5.5,0.0,32400.0\nR003,2015-01-25,9.0,0.1,32400.0\nR003,2015-01-26,6.1,6.6,28577.8\nR003,2015-01-27,4.4,0.0,32400.0\nR003,2015-01-28,5.3,0.0,25513.12\nR003,2015-01-29,4.4,6.4,22945.22\nR003,2015-01-30,6.2,0.0,28956.37\nR003,2015-01-31,6.8,0.0,27798.05\nR003,2015-02-01,8.9,0.2,13938.4\nR003,2015-02-02,11.0,0.0,32685.24\nR003,2015-02-03,9.5,6.5,0.0\nR003,2015-02-04,4.6,2.3,32595.83\nR003,2015-02-05,2.2,4.5,22011.24\nR003,2015-02-06,6.3,0.0,32096.94\nR003,2015-02-07,6.3,3.6,12588.47\nR003,2015-02-08,2.8,5.6,33125.88\nR003,2015-02-09,3.1,7.8,8946.71\nR003,2015-02-10,3.3,0.4,31599.29\nR003,2015-02-11,6.5,0.6,30380.67\nR003,2015-02-12,5.1,6.1,20776.09\nR003,2015-02-13,5.0,2.3,19720.95\nR003,2015-02-14,3.8,1.2,29094.42\nR003,2015-02-15,2.5,0.1,34734.79\nR003,2015-02-16,1.5,8.6,30017.2\nR003,2015-02-17,0.1,10.9,16588.24\nR003,2015-02-18,0.1,0.0,36702.43\nR003,2015-02-19,1.5,0.3,37895.06\nR003,2015-02-20,0.9,3.3,24334.97\nR003,2015-02-21,0.2,0.0,35787.22\nR003,2015-02-22,3.1,0.0,38422.28\nR003,2015-02-23,5.4,0.0,38598.83\nR003,2015-02-24,6.5,0.0,35438.2\nR003,2015-02-25,7.9,0.0,38952.27\nR003,2015-02-26,7.2,0.0,39128.89\nR003,2015-02-27,8.5,0.0,38558.46\nR003,2015-02-28,8.7,0.3,25395.32\nR003,2015-03-01,6.7,8.8,1361.31\nR003,2015-03-02,5.6,2.0,38402.86\nR003,2015-03-03,6.0,0.0,39600.0\nR003,2015-03-04,3.1,6.9,14546.1\nR003,2015-03-05,1.5,3.3,38041.91\nR003,2015-03-06,2.5,0.0,36038.06\nR003,2015-03-07,6.9,5.6,20340.29\nR003,2015-03-08,6.1,0.0,37130.33\nR003,2015-03-09,6.7,0.0,39600.0\nR003,2015-03-10,10.1,0.0,39600.0\nR003,2015-03-11,13.8,0.0,39600.0\nR003,2015-03-12,13.0,3.9,13864.47\nR003,2015-03-13,10.3,7.9,31922.36\nR003,2015-03-14,10.8,0.0,36190.23\nR003,2015-03-15,9.9,7.2,36612.95\nR003,2015-03-16,8.4,9.3,13431.79\nR003,2015-03-17,5.2,9.7,30075.13\nR003,2015-03-18,6.3,0.6,36837.16\nR003,2015-03-19,8.1,0.3,36758.26\nR003,2015-03-20,7.5,7.5,14617.47\nR003,2015-03-21,6.3,5.5,19706.15\nR003,2015-03-22,8.1,1.5,37662.22\nR003,2015-03-23,5.4,0.0,33830.28\nR003,2015-03-24,4.7,0.9,39831.08\nR003,2015-03-25,8.3,0.0,40435.51\nR003,2015-03-26,12.0,0.0,43082.73\nR003,2015-03-27,14.3,0.0,40170.11\nR003,2015-03-28,15.2,2.2,5286.39\nR003,2015-03-29,14.0,10.5,37218.55\nR003,2015-03-30,11.0,6.2,10958.79\nR003,2015-03-31,10.6,0.3,37911.5\nR003,2015-04-01,9.6,4.5,21550.01\nR003,2015-04-02,9.0,18.1,29948.4\nR003,2015-04-03,4.3,4.3,43642.1\nR003,2015-04-04,3.0,12.9,4831.81\nR003,2015-04-05,3.4,0.1,41507.5\nR003,2015-04-06,9.1,0.0,42929.53\nR003,2015-04-07,11.8,8.5,30606.42\nR003,2015-04-08,6.6,17.0,19038.05\nR003,2015-04-09,7.2,23.0,2034.79\nR003,2015-04-10,5.7,28.0,4337.27\nR003,2015-04-11,6.8,0.0,38024.08\nR003,2015-04-12,7.1,3.3,2568.5\nR003,2015-04-13,8.1,0.6,38140.13\nR003,2015-04-14,8.4,2.5,46558.38\nR003,2015-04-15,4.4,18.6,12948.0\nR003,2015-04-16,6.9,0.2,43057.8\nR003,2015-04-17,12.1,0.0,46772.87\nR003,2015-04-18,13.6,3.5,35476.72\nR003,2015-04-19,10.8,31.6,17773.14\nR003,2015-04-20,3.9,4.4,46800.0\nR003,2015-04-21,4.7,0.7,38358.97\nR003,2015-04-22,4.9,5.8,44520.96\nR003,2015-04-23,6.7,0.0,46528.38\nR003,2015-04-24,7.2,0.3,47139.28\nR003,2015-04-25,10.4,0.0,46800.0\nR003,2015-04-26,13.7,0.0,44845.8\nR003,2015-04-27,13.6,3.3,39310.22\nR003,2015-04-28,13.7,2.4,47301.65\nR003,2015-04-29,16.2,0.6,43581.03\nR003,2015-04-30,14.7,4.0,44937.44\nR003,2015-05-01,13.7,3.6,44938.66\nR003,2015-05-02,12.9,0.3,46662.91\nR003,2015-05-03,14.6,0.0,43993.18\nR003,2015-05-04,12.6,6.7,8134.7\nR003,2015-05-05,11.4,8.6,43200.0\nR003,2015-05-06,10.1,2.4,36633.57\nR003,2015-05-07,10.7,3.9,20582.74\nR003,2015-05-08,12.3,2.2,38338.51\nR003,2015-05-09,12.5,2.9,37889.91\nR003,2015-05-10,11.9,0.7,45159.03\nR003,2015-05-11,12.6,1.3,45307.99\nR003,2015-05-12,13.0,2.9,42390.02\nR003,2015-05-13,12.7,6.2,40222.06\nR003,2015-05-14,12.5,7.7,43076.26\nR003,2015-05-15,14.1,0.5,46482.93\nR003,2015-05-16,14.0,10.3,17311.29\nR003,2015-05-17,13.5,8.6,36684.57\nR003,2015-05-18,13.7,0.0,50404.12\nR003,2015-05-19,15.6,0.0,50400.0\nR003,2015-05-20,18.2,0.0,50702.33\nR003,2015-05-21,21.1,0.0,50675.6\nR003,2015-05-22,21.2,5.4,50400.0\nR003,2015-05-23,19.3,7.1,38690.24\nR003,2015-05-24,17.5,17.6,37039.26\nR003,2015-05-25,19.0,3.3,46327.65\nR003,2015-05-26,19.9,3.7,49895.05\nR003,2015-05-27,20.2,1.0,50879.21\nR003,2015-05-28,18.6,7.0,48566.1\nR003,2015-05-29,17.5,3.7,48744.03\nR003,2015-05-30,17.9,9.9,43109.76\nR003,2015-05-31,16.3,15.7,32896.25\nR003,2015-06-01,17.6,5.5,51186.67\nR003,2015-06-02,19.6,0.0,50032.81\nR003,2015-06-03,20.1,4.6,51085.56\nR003,2015-06-04,19.9,10.9,48343.83\nR003,2015-06-05,20.9,2.5,47863.45\nR003,2015-06-06,21.8,3.9,39755.81\nR003,2015-06-07,21.1,0.0,43219.04\nR003,2015-06-08,22.1,0.0,52823.91\nR003,2015-06-09,21.0,8.0,48111.18\nR003,2015-06-10,21.3,2.8,52796.99\nR003,2015-06-11,19.6,6.6,52051.07\nR003,2015-06-12,18.4,10.5,48535.04\nR003,2015-06-13,19.3,3.2,49666.67\nR003,2015-06-14,19.8,7.0,50701.2\nR003,2015-06-15,18.5,19.2,41654.55\nR003,2015-06-16,18.3,16.8,14203.42\nR003,2015-06-17,18.8,9.3,38590.34\nR003,2015-06-18,20.4,0.1,52358.85\nR003,2015-06-19,21.2,4.6,53432.03\nR003,2015-06-20,21.1,9.8,46334.68\nR003,2015-06-21,20.7,8.2,49819.04\nR003,2015-06-22,21.6,7.4,51679.46\nR003,2015-06-23,22.1,11.2,51033.55\nR003,2015-06-24,22.6,16.2,47243.9\nR003,2015-06-25,20.7,16.7,32236.9\nR003,2015-06-26,20.5,27.2,17784.17\nR003,2015-06-27,19.9,4.2,43926.9\nR003,2015-06-28,19.6,8.2,48897.78\nR003,2015-06-29,19.9,16.3,46901.7\nR003,2015-06-30,20.2,3.0,49504.36\nR003,2015-07-01,20.9,0.0,50100.59\nR003,2015-07-02,22.1,1.0,49231.16\nR003,2015-07-03,20.7,10.8,31303.43\nR003,2015-07-04,22.8,1.5,45766.85\nR003,2015-07-05,23.8,1.9,41643.18\nR003,2015-07-06,22.0,11.6,7765.58\nR003,2015-07-07,22.4,1.7,50763.9\nR003,2015-07-08,22.9,0.8,53290.03\nR003,2015-07-09,25.0,1.3,53042.12\nR003,2015-07-10,23.1,25.4,26996.67\nR003,2015-07-11,21.3,5.1,47106.52\nR003,2015-07-12,20.1,0.0,51893.93\nR003,2015-07-13,19.4,0.5,50335.52\nR003,2015-07-14,20.7,0.1,52725.35\nR003,2015-07-15,19.9,5.8,39865.55\nR003,2015-07-16,20.6,0.0,50825.81\nR003,2015-07-17,20.6,0.0,49954.78\nR003,2015-07-18,19.1,0.0,52777.34\nR003,2015-07-19,20.2,0.0,53438.46\nR003,2015-07-20,20.9,0.0,52248.6\nR003,2015-07-21,22.8,0.0,52751.65\nR003,2015-07-22,22.0,1.9,49975.1\nR003,2015-07-23,21.8,0.0,47444.26\nR003,2015-07-24,24.8,0.0,52284.56\nR003,2015-07-25,26.8,0.7,51544.5\nR003,2015-07-26,26.5,0.7,48437.48\nR003,2015-07-27,23.9,8.0,49152.62\nR003,2015-07-28,23.5,3.5,47811.6\nR003,2015-07-29,24.6,0.0,50298.28\nR003,2015-07-30,25.8,0.0,48964.61\nR003,2015-07-31,28.6,0.0,50636.12\nR003,2015-08-01,27.7,1.7,50257.22\nR003,2015-08-02,26.4,3.5,48098.32\nR003,2015-08-03,25.8,7.8,44087.27\nR003,2015-08-04,25.8,3.7,48219.1\nR003,2015-08-05,25.1,2.7,48622.23\nR003,2015-08-06,24.4,0.2,46978.73\nR003,2015-08-07,24.4,0.0,48625.24\nR003,2015-08-08,24.1,0.0,47789.66\nR003,2015-08-09,23.7,0.0,47321.93\nR003,2015-08-10,25.3,0.0,48026.07\nR003,2015-08-11,26.6,0.0,47716.55\nR003,2015-08-12,26.3,1.5,47408.77\nR003,2015-08-13,24.6,3.8,46509.23\nR003,2015-08-14,25.7,1.2,46560.35\nR003,2015-08-15,25.8,0.8,46829.43\nR003,2015-08-16,23.6,5.8,46728.67\nR003,2015-08-17,24.0,0.0,46800.0\nR003,2015-08-18,23.0,0.0,46923.5\nR003,2015-08-19,24.0,0.1,46800.0\nR003,2015-08-20,24.3,0.6,46800.0\nR003,2015-08-21,24.4,0.0,46800.0\nR003,2015-08-22,24.0,0.8,46800.0\nR003,2015-08-23,22.6,4.6,45177.75\nR003,2015-08-24,19.9,13.3,19635.87\nR003,2015-08-25,19.2,2.4,35377.37\nR003,2015-08-26,19.0,10.9,38203.35\nR003,2015-08-27,19.9,0.0,46768.15\nR003,2015-08-28,19.9,0.6,42749.5\nR003,2015-08-29,20.3,0.1,46567.29\nR003,2015-08-30,20.8,0.0,46464.85\nR003,2015-08-31,20.3,0.8,45484.69\nR003,2015-09-01,20.7,0.0,46256.47\nR003,2015-09-02,22.5,0.0,46150.75\nR003,2015-09-03,23.4,0.0,45628.99\nR003,2015-09-04,24.1,0.0,45357.79\nR003,2015-09-05,24.5,0.4,44447.34\nR003,2015-09-06,24.2,1.2,43859.23\nR003,2015-09-07,24.5,0.3,43911.88\nR003,2015-09-08,22.2,1.5,38198.87\nR003,2015-09-09,21.2,5.3,33290.08\nR003,2015-09-10,22.3,0.4,38209.37\nR003,2015-09-11,23.3,1.3,42852.46\nR003,2015-09-12,22.3,0.3,41516.12\nR003,2015-09-13,21.8,4.1,29098.71\nR003,2015-09-14,20.2,6.3,40751.81\nR003,2015-09-15,20.5,1.2,39600.0\nR003,2015-09-16,20.0,0.3,38785.59\nR003,2015-09-17,19.7,0.0,42073.97\nR003,2015-09-18,20.8,0.0,42861.06\nR003,2015-09-19,22.4,0.0,42793.46\nR003,2015-09-20,24.3,0.0,42725.7\nR003,2015-09-21,25.0,0.3,42209.11\nR003,2015-09-22,24.3,0.1,40970.4\nR003,2015-09-23,23.5,0.5,40368.34\nR003,2015-09-24,22.8,2.1,40757.9\nR003,2015-09-25,22.2,0.7,41713.0\nR003,2015-09-26,21.2,0.2,41430.19\nR003,2015-09-27,20.3,1.8,40605.24\nR003,2015-09-28,19.8,2.5,39907.1\nR003,2015-09-29,19.5,0.0,40897.32\nR003,2015-09-30,19.8,1.3,36262.93\nR003,2015-10-01,17.8,7.5,11918.83\nR003,2015-10-02,18.6,0.6,28661.26\nR003,2015-10-03,17.8,1.2,34523.94\nR003,2015-10-04,16.0,3.5,38069.12\nR003,2015-10-05,15.8,1.8,37025.32\nR003,2015-10-06,15.7,0.8,38453.3\nR003,2015-10-07,15.2,0.2,29518.59\nR003,2015-10-08,12.6,0.0,36238.86\nR003,2015-10-09,10.6,0.7,36355.58\nR003,2015-10-10,9.5,0.1,36232.57\nR003,2015-10-11,11.0,1.8,34228.1\nR003,2015-10-12,13.3,55.4,180.32\nR003,2015-10-13,14.9,11.5,25155.25\nR003,2015-10-14,14.0,10.8,5609.34\nR003,2015-10-15,13.2,3.6,30243.97\nR003,2015-10-16,11.6,0.3,35264.07\nR003,2015-10-17,11.3,0.0,36132.49\nR003,2015-10-18,13.1,0.0,30316.02\nR003,2015-10-19,14.5,0.0,36000.0\nR003,2015-10-20,14.5,0.0,36000.0\nR003,2015-10-21,14.7,0.0,36000.0\nR003,2015-10-22,16.3,1.1,26682.09\nR003,2015-10-23,17.5,0.0,36000.0\nR003,2015-10-24,16.7,20.3,1460.34\nR003,2015-10-25,12.9,22.7,3243.27\nR003,2015-10-26,9.5,0.0,36000.0\nR003,2015-10-27,9.3,0.0,35002.58\nR003,2015-10-28,8.5,0.0,36000.0\nR003,2015-10-29,7.1,0.0,36000.0\nR003,2015-10-30,8.2,0.0,36000.0\nR003,2015-10-31,8.5,0.0,31002.44\nR003,2015-11-01,8.2,0.9,36000.0\nR003,2015-11-02,9.2,2.2,31756.17\nR003,2015-11-03,10.1,0.0,29512.15\nR003,2015-11-04,10.0,0.0,35810.01\nR003,2015-11-05,11.0,0.0,35729.51\nR003,2015-11-06,10.6,0.0,35588.61\nR003,2015-11-07,10.8,0.0,35432.82\nR003,2015-11-08,8.9,0.5,31342.39\nR003,2015-11-09,8.2,9.6,10850.17\nR003,2015-11-10,7.2,1.8,32858.71\nR003,2015-11-11,5.9,46.2,178.48\nR003,2015-11-12,5.3,4.4,32686.29\nR003,2015-11-13,4.7,39.0,1001.62\nR003,2015-11-14,3.2,0.0,34087.5\nR003,2015-11-15,4.1,7.0,101.28\nR003,2015-11-16,2.9,2.7,13866.09\nR003,2015-11-17,3.5,14.9,22161.37\nR003,2015-11-18,1.4,0.0,30345.86\nR003,2015-11-19,3.8,6.3,18153.22\nR003,2015-11-20,6.9,1.1,23923.24\nR003,2015-11-21,9.9,0.0,23942.9\nR003,2015-11-22,13.9,0.0,30950.19\nR003,2015-11-23,12.3,0.0,32219.36\nR003,2015-11-24,10.0,0.0,26512.3\nR003,2015-11-25,12.1,0.0,4016.45\nR003,2015-11-26,13.7,0.0,29837.18\nR003,2015-11-27,10.0,13.8,0.0\nR003,2015-11-28,10.7,0.0,29505.23\nR003,2015-11-29,9.1,12.4,1091.92\nR003,2015-11-30,7.9,9.2,17923.51\nR003,2015-12-01,6.0,17.7,25713.57\nR003,2015-12-02,2.9,44.4,0.0\nR003,2015-12-03,-2.0,0.0,30168.9\nR003,2015-12-04,-3.8,0.0,28725.49\nR003,2015-12-05,-2.3,0.0,28800.0\nR003,2015-12-06,2.2,1.7,12310.05\nR003,2015-12-07,5.3,0.6,24398.52\nR003,2015-12-08,4.8,0.5,29226.59\nR003,2015-12-09,0.0,0.0,30026.98\nR003,2015-12-10,2.7,0.0,29644.57\nR003,2015-12-11,3.7,0.0,28800.0\nR003,2015-12-12,3.6,0.0,30034.1\nR003,2015-12-13,3.0,1.7,16067.82\nR003,2015-12-14,3.8,0.4,28247.11\nR003,2015-12-15,2.6,7.1,30236.63\nR003,2015-12-16,0.6,0.0,26741.39\nR003,2015-12-17,0.3,0.0,30405.88\nR003,2015-12-18,0.2,0.0,28800.0\nR003,2015-12-19,1.7,0.0,30779.85\nR003,2015-12-20,3.3,4.9,18000.0\nR003,2015-12-21,5.7,5.5,17599.96\nR003,2015-12-22,6.6,0.0,29460.6\nR003,2015-12-23,5.7,7.2,23678.47\nR003,2015-12-24,5.5,4.3,22558.38\nR003,2015-12-25,4.5,1.8,30697.75\nR003,2015-12-26,3.3,0.0,26691.44\nR003,2015-12-27,5.8,8.1,10787.53\nR003,2015-12-28,5.6,6.5,9018.72\nR003,2015-12-29,7.0,3.6,27838.08\nR003,2015-12-30,1.9,22.0,0.0\nR003,2015-12-31,-4.8,6.3,28848.76\nR003,2016-01-01,-7.1,1.2,30284.1\nR003,2016-01-02,-11.2,0.0,31634.91\nR003,2016-01-03,-8.6,6.0,112.44\nR003,2016-01-04,-6.8,5.5,0.0\nR003,2016-01-05,2.3,23.2,0.0\nR003,2016-01-06,4.3,14.1,16988.66\nR003,2016-01-07,7.3,10.0,4693.58\nR003,2016-01-08,7.2,19.2,905.05\nR003,2016-01-09,2.8,17.4,5164.56\nR003,2016-01-10,3.1,2.6,28642.0\nR003,2016-01-11,6.2,8.4,0.0\nR003,2016-01-12,7.7,1.0,25933.62\nR003,2016-01-13,7.4,22.1,0.0\nR003,2016-01-14,4.0,8.7,22237.42\nR003,2016-01-15,0.4,0.3,32400.0\nR003,2016-01-16,1.4,0.0,32400.0\nR003,2016-01-17,8.1,0.0,21200.78\nR003,2016-01-18,9.0,3.5,23491.42\nR003,2016-01-19,3.8,23.7,118.83\nR003,2016-01-20,-1.4,5.9,31267.08\nR003,2016-01-21,-1.4,4.2,7561.56\nR003,2016-01-22,1.8,23.0,0.0\nR003,2016-01-23,0.3,10.2,0.0\nR003,2016-01-24,-2.5,5.7,0.0\nR003,2016-01-25,-5.1,1.4,20919.95\nR003,2016-01-26,-7.0,0.0,32400.0\nR003,2016-01-27,-7.8,0.0,32400.0\nR003,2016-01-28,-2.1,7.2,1265.9\nR003,2016-01-29,2.4,18.5,0.0\nR003,2016-01-30,2.6,0.4,21806.52\nR003,2016-01-31,3.3,0.0,32430.48\nR003,2016-02-01,4.5,8.7,3849.27\nR003,2016-02-02,1.9,20.7,32053.11\nR003,2016-02-03,3.9,0.0,33001.01\nR003,2016-02-04,9.7,0.0,30203.12\nR003,2016-02-05,11.2,0.0,30474.82\nR003,2016-02-06,7.7,4.1,17179.86\nR003,2016-02-07,4.0,1.9,27664.74\nR003,2016-02-08,3.5,0.8,32776.96\nR003,2016-02-09,3.4,0.0,34987.44\nR003,2016-02-10,2.6,0.0,35749.22\nR003,2016-02-11,3.6,0.3,34455.1\nR003,2016-02-12,6.9,5.4,18622.69\nR003,2016-02-13,9.9,0.0,34608.18\nR003,2016-02-14,9.2,1.6,16466.5\nR003,2016-02-15,10.8,0.5,33555.15\nR003,2016-02-16,11.9,3.4,27871.15\nR003,2016-02-17,11.3,3.5,34740.83\nR003,2016-02-18,6.4,0.2,36024.36\nR003,2016-02-19,3.7,0.0,37860.33\nR003,2016-02-20,2.0,0.5,33867.53\nR003,2016-02-21,3.4,1.2,21465.74\nR003,2016-02-22,5.0,6.6,31167.48\nR003,2016-02-23,8.8,0.0,38552.64\nR003,2016-02-24,9.8,0.0,38725.32\nR003,2016-02-25,8.3,14.0,0.0\nR003,2016-02-26,8.7,0.3,37578.58\nR003,2016-02-27,10.6,0.0,39250.93\nR003,2016-02-28,8.9,9.4,28092.31\nR003,2016-02-29,8.8,0.0,39496.9\nR003,2016-03-01,12.1,0.0,39598.65\nR003,2016-03-02,16.2,0.0,21600.0\nR003,2016-03-03,15.9,0.0,30847.08\nR003,2016-03-04,13.0,5.9,39600.0\nR003,2016-03-05,10.5,6.2,27821.5\nR003,2016-03-06,9.3,2.7,28454.25\nR003,2016-03-07,9.3,0.0,39600.0\nR003,2016-03-08,10.7,0.0,39600.0\nR003,2016-03-09,11.2,0.0,39600.0\nR003,2016-03-10,11.0,0.0,39600.0\nR003,2016-03-11,10.7,4.4,32286.44\nR003,2016-03-12,7.7,2.5,32400.0\nR003,2016-03-13,8.2,0.4,39600.0\nR003,2016-03-14,7.5,2.8,1631.71\nR003,2016-03-15,3.8,2.9,38147.14\nR003,2016-03-16,2.2,4.3,37981.78\nR003,2016-03-17,0.5,5.0,30488.2\nR003,2016-03-18,2.6,3.5,34250.63\nR003,2016-03-19,4.4,5.1,24901.52\nR003,2016-03-20,0.4,8.4,28718.41\nR003,2016-03-21,1.3,0.0,40237.26\nR003,2016-03-22,5.7,0.0,40146.1\nR003,2016-03-23,7.8,11.8,20960.39\nR003,2016-03-24,12.9,3.8,26910.08\nR003,2016-03-25,12.1,10.2,31924.47\nR003,2016-03-26,9.0,10.1,28348.77\nR003,2016-03-27,6.5,10.1,12895.06\nR003,2016-03-28,6.5,3.6,12712.32\nR003,2016-03-29,5.7,8.6,2843.85\nR003,2016-03-30,6.0,3.2,31683.1\nR003,2016-03-31,6.5,0.0,43797.81\nR003,2016-04-01,9.3,0.0,44091.03\nR003,2016-04-02,10.7,1.3,31189.22\nR003,2016-04-03,3.7,2.2,44686.04\nR003,2016-04-04,5.3,0.0,39600.0\nR003,2016-04-05,8.7,0.0,45971.42\nR003,2016-04-06,11.1,0.0,46042.74\nR003,2016-04-07,13.7,0.0,45572.98\nR003,2016-04-08,14.9,0.5,46185.54\nR003,2016-04-09,16.5,0.0,46257.01\nR003,2016-04-10,15.2,2.7,23027.07\nR003,2016-04-11,15.4,6.2,36786.66\nR003,2016-04-12,13.0,6.5,8491.7\nR003,2016-04-13,13.1,2.4,40206.44\nR003,2016-04-14,12.7,0.9,41516.1\nR003,2016-04-15,13.8,1.4,45404.93\nR003,2016-04-16,10.8,18.9,31500.56\nR003,2016-04-17,11.6,0.0,46800.0\nR003,2016-04-18,13.9,0.3,43200.0\nR003,2016-04-19,17.0,0.5,46465.57\nR003,2016-04-20,19.1,0.1,40304.83\nR003,2016-04-21,9.4,27.7,0.0\nR003,2016-04-22,6.1,1.0,46556.7\nR003,2016-04-23,12.6,0.0,43918.02\nR003,2016-04-24,16.7,0.0,32400.0\nR003,2016-04-25,16.5,5.1,40570.6\nR003,2016-04-26,15.9,3.2,45605.92\nR003,2016-04-27,12.7,17.4,17965.23\nR003,2016-04-28,13.2,0.3,47274.74\nR003,2016-04-29,12.4,1.3,44735.78\nR003,2016-04-30,13.6,0.4,39769.4\nR003,2016-05-01,12.6,2.4,35494.06\nR003,2016-05-02,13.3,0.0,46158.74\nR003,2016-05-03,15.1,0.0,49104.13\nR003,2016-05-04,12.9,10.4,8193.92\nR003,2016-05-05,13.5,8.3,37165.2\nR003,2016-05-06,12.1,9.4,13685.11\nR003,2016-05-07,12.3,6.5,21490.28\nR003,2016-05-08,13.3,1.2,46903.83\nR003,2016-05-09,14.6,1.0,47554.35\nR003,2016-05-10,16.1,0.8,45645.69\nR003,2016-05-11,12.9,24.2,27735.82\nR003,2016-05-12,14.4,0.1,48841.76\nR003,2016-05-13,15.6,2.6,32902.92\nR003,2016-05-14,18.8,2.3,48494.74\nR003,2016-05-15,18.2,0.8,48285.33\nR003,2016-05-16,13.2,21.0,5223.83\nR003,2016-05-17,16.0,0.0,48454.08\nR003,2016-05-18,16.5,3.8,46800.0\nR003,2016-05-19,14.3,9.5,31005.81\nR003,2016-05-20,14.4,0.0,49355.91\nR003,2016-05-21,16.0,0.0,50400.0\nR003,2016-05-22,18.3,0.0,50287.83\nR003,2016-05-23,17.6,12.9,34020.26\nR003,2016-05-24,14.9,28.7,356.5\nR003,2016-05-25,17.1,1.8,48142.58\nR003,2016-05-26,16.0,5.4,22257.88\nR003,2016-05-27,17.8,4.6,47261.67\nR003,2016-05-28,17.0,13.1,46637.01\nR003,2016-05-29,17.0,10.5,29987.88\nR003,2016-05-30,16.4,2.4,34593.79\nR003,2016-05-31,18.3,2.8,49016.26\nR003,2016-06-01,17.6,2.1,48336.74\nR003,2016-06-02,17.8,9.3,44362.26\nR003,2016-06-03,17.7,1.4,38178.63\nR003,2016-06-04,16.6,9.6,16264.05\nR003,2016-06-05,17.1,4.6,43660.65\nR003,2016-06-06,17.9,0.3,51217.81\nR003,2016-06-07,17.5,0.0,51446.92\nR003,2016-06-08,16.1,2.0,50086.82\nR003,2016-06-09,15.6,4.6,48536.85\nR003,2016-06-10,15.2,11.8,34311.04\nR003,2016-06-11,16.2,4.8,47553.09\nR003,2016-06-12,18.3,0.0,49813.08\nR003,2016-06-13,20.1,0.0,52928.16\nR003,2016-06-14,23.7,0.0,45892.23\nR003,2016-06-15,22.5,7.0,40986.33\nR003,2016-06-16,19.3,11.9,28318.14\nR003,2016-06-17,20.0,4.2,37366.78\nR003,2016-06-18,21.0,5.2,48645.06\nR003,2016-06-19,22.5,7.0,47035.74\nR003,2016-06-20,22.5,4.0,50002.63\nR003,2016-06-21,24.0,1.5,53778.55\nR003,2016-06-22,24.8,0.0,52654.94\nR003,2016-06-23,24.2,1.0,52665.29\nR003,2016-06-24,23.0,2.6,40040.38\nR003,2016-06-25,23.8,0.5,53025.02\nR003,2016-06-26,24.7,0.0,53964.53\nR003,2016-06-27,25.3,0.2,53753.1\nR003,2016-06-28,23.6,3.9,39306.98\nR003,2016-06-29,23.3,6.2,50683.82\nR003,2016-06-30,22.6,10.1,50089.78\nR003,2016-07-01,21.3,14.0,37725.72\nR003,2016-07-02,21.9,12.2,39812.68\nR003,2016-07-03,22.5,4.9,49054.32\nR003,2016-07-04,21.8,4.2,45331.68\nR003,2016-07-05,22.2,1.8,51366.19\nR003,2016-07-06,22.2,4.1,52895.93\nR003,2016-07-07,22.9,1.8,51716.81\nR003,2016-07-08,20.1,11.4,34868.3\nR003,2016-07-09,19.1,0.2,52813.82\nR003,2016-07-10,18.4,5.1,48486.31\nR003,2016-07-11,17.9,15.0,44408.51\nR003,2016-07-12,20.4,0.2,53694.91\nR003,2016-07-13,23.3,0.0,53659.27\nR003,2016-07-14,25.3,0.0,53451.31\nR003,2016-07-15,24.3,4.0,50860.78\nR003,2016-07-16,25.0,0.4,51972.02\nR003,2016-07-17,27.2,0.0,53488.05\nR003,2016-07-18,26.8,3.6,51367.06\nR003,2016-07-19,22.4,9.2,45353.03\nR003,2016-07-20,22.6,0.0,49925.47\nR003,2016-07-21,21.2,12.9,18515.36\nR003,2016-07-22,21.3,0.0,49304.36\nR003,2016-07-23,20.4,0.0,51811.76\nR003,2016-07-24,18.8,2.3,47994.4\nR003,2016-07-25,19.6,2.3,46474.43\nR003,2016-07-26,21.1,0.2,47299.64\nR003,2016-07-27,21.9,0.7,48846.26\nR003,2016-07-28,21.8,2.5,49593.3\nR003,2016-07-29,21.9,0.7,49992.08\nR003,2016-07-30,22.4,2.8,50042.35\nR003,2016-07-31,22.2,1.3,50095.34\nR003,2016-08-01,22.9,0.0,50492.73\nR003,2016-08-02,24.0,0.0,50200.85\nR003,2016-08-03,23.3,1.7,48542.87\nR003,2016-08-04,22.6,3.4,48260.66\nR003,2016-08-05,23.0,1.3,47299.7\nR003,2016-08-06,24.0,5.7,47705.8\nR003,2016-08-07,23.4,7.9,47963.56\nR003,2016-08-08,22.3,6.2,46783.36\nR003,2016-08-09,23.7,0.3,46800.0\nR003,2016-08-10,25.2,0.7,47708.86\nR003,2016-08-11,24.3,4.9,46800.0\nR003,2016-08-12,23.6,1.6,45388.09\nR003,2016-08-13,24.3,2.1,46800.0\nR003,2016-08-14,21.6,33.6,14606.66\nR003,2016-08-15,20.8,0.0,46800.0\nR003,2016-08-16,21.7,2.8,45998.09\nR003,2016-08-17,22.8,13.6,43710.46\nR003,2016-08-18,22.2,14.7,39787.66\nR003,2016-08-19,23.7,0.2,46800.0\nR003,2016-08-20,26.1,0.1,46800.0\nR003,2016-08-21,27.0,0.1,46800.0\nR003,2016-08-22,27.1,1.1,38405.04\nR003,2016-08-23,25.5,4.8,46800.0\nR003,2016-08-24,24.8,2.5,46378.31\nR003,2016-08-25,24.2,2.5,46037.96\nR003,2016-08-26,22.5,11.6,45938.17\nR003,2016-08-27,22.9,1.7,46694.51\nR003,2016-08-28,22.7,1.3,46591.2\nR003,2016-08-29,23.3,1.6,46396.14\nR003,2016-08-30,22.9,4.8,43864.46\nR003,2016-08-31,21.1,6.8,41002.14\nR003,2016-09-01,20.7,0.0,41928.24\nR003,2016-09-02,19.7,1.3,44978.41\nR003,2016-09-03,19.7,0.9,42650.44\nR003,2016-09-04,19.0,3.0,38593.68\nR003,2016-09-05,19.6,0.9,40400.16\nR003,2016-09-06,19.2,6.5,38439.98\nR003,2016-09-07,19.5,0.0,43953.31\nR003,2016-09-08,19.9,0.0,43367.24\nR003,2016-09-09,19.7,0.0,42930.42\nR003,2016-09-10,19.8,0.1,42808.91\nR003,2016-09-11,19.0,0.0,42984.32\nR003,2016-09-12,19.2,0.0,43200.0\nR003,2016-09-13,19.7,0.1,43148.06\nR003,2016-09-14,19.0,0.6,35580.29\nR003,2016-09-15,16.8,3.0,27687.43\nR003,2016-09-16,15.9,0.0,41428.7\nR003,2016-09-17,15.8,0.2,37644.08\nR003,2016-09-18,16.3,0.0,40540.77\nR003,2016-09-19,17.5,0.0,42166.08\nR003,2016-09-20,20.4,1.9,41526.56\nR003,2016-09-21,15.8,27.5,9355.55\nR003,2016-09-22,13.4,23.0,1784.36\nR003,2016-09-23,12.9,24.6,8686.87\nR003,2016-09-24,11.3,2.6,35545.34\nR003,2016-09-25,9.7,1.2,39525.21\nR003,2016-09-26,11.3,0.1,40819.32\nR003,2016-09-27,11.0,5.4,27141.89\nR003,2016-09-28,9.6,15.5,5062.44\nR003,2016-09-29,9.1,1.6,26524.88\nR003,2016-09-30,11.8,22.5,31190.22\nR003,2016-10-01,13.6,0.0,39742.74\nR003,2016-10-02,17.1,0.0,40313.51\nR003,2016-10-03,17.8,0.0,39602.86\nR003,2016-10-04,16.9,0.0,39532.98\nR003,2016-10-05,18.2,0.0,39025.65\nR003,2016-10-06,18.6,0.0,37507.29\nR003,2016-10-07,17.5,0.3,37601.43\nR003,2016-10-08,18.8,0.0,38021.15\nR003,2016-10-09,16.2,8.4,24752.91\nR003,2016-10-10,14.7,0.6,36778.36\nR003,2016-10-11,14.1,0.0,36000.0\nR003,2016-10-12,13.6,5.0,6582.02\nR003,2016-10-13,15.7,10.1,28627.44\nR003,2016-10-14,10.9,17.6,31636.49\nR003,2016-10-15,9.6,0.5,36070.77\nR003,2016-10-16,8.8,1.4,23496.73\nR003,2016-10-17,7.2,1.6,35309.71\nR003,2016-10-18,8.3,0.0,36000.0\nR003,2016-10-19,9.7,0.0,36000.0\nR003,2016-10-20,10.5,0.2,33907.76\nR003,2016-10-21,11.1,0.0,36000.0\nR003,2016-10-22,12.4,0.0,35384.51\nR003,2016-10-23,11.9,0.4,23745.28\nR003,2016-10-24,11.8,5.0,35419.94\nR003,2016-10-25,9.8,8.3,19560.52\nR003,2016-10-26,7.4,1.7,31651.39\nR003,2016-10-27,8.1,0.0,32051.55\nR003,2016-10-28,8.6,2.3,17320.52\nR003,2016-10-29,8.5,0.4,36000.0\nR003,2016-10-30,5.2,23.2,574.76\nR003,2016-10-31,1.9,6.9,28744.64\nR003,2016-11-01,3.2,3.5,30336.57\nR003,2016-11-02,1.6,1.9,35185.01\nR003,2016-11-03,6.5,0.0,35827.22\nR003,2016-11-04,8.1,10.6,24947.32\nR003,2016-11-05,7.5,1.5,27187.71\nR003,2016-11-06,6.8,0.0,35465.27\nR003,2016-11-07,11.0,0.2,30078.03\nR003,2016-11-08,15.5,0.1,35163.99\nR003,2016-11-09,16.3,0.0,32066.17\nR003,2016-11-10,16.4,1.1,25419.01\nR003,2016-11-11,12.5,9.3,27442.86\nR003,2016-11-12,10.1,1.8,33823.25\nR003,2016-11-13,10.7,1.7,34449.76\nR003,2016-11-14,8.7,18.0,0.0\nR003,2016-11-15,6.8,2.2,13680.06\nR003,2016-11-16,5.4,5.3,3793.8\nR003,2016-11-17,4.1,10.0,25987.24\nR003,2016-11-18,2.1,0.0,31073.33\nR003,2016-11-19,4.6,0.0,32960.23\nR003,2016-11-20,7.1,0.0,32957.41\nR003,2016-11-21,8.8,0.0,32588.77\nR003,2016-11-22,8.3,0.0,32588.37\nR003,2016-11-23,8.1,0.0,32220.43\nR003,2016-11-24,6.4,0.0,31852.39\nR003,2016-11-25,6.1,0.0,31855.35\nR003,2016-11-26,4.5,0.0,30096.76\nR003,2016-11-27,5.9,0.0,31008.58\nR003,2016-11-28,8.2,0.0,28257.8\nR003,2016-11-29,8.7,8.7,18173.8\nR003,2016-11-30,5.8,29.2,0.0\nR003,2016-12-01,2.2,8.2,7137.49\nR003,2016-12-02,0.9,9.5,0.0\nR003,2016-12-03,0.5,30.5,0.0\nR003,2016-12-04,-0.8,23.1,755.91\nR003,2016-12-05,-4.2,9.0,952.35\nR003,2016-12-06,-6.2,0.0,30394.09\nR003,2016-12-07,-2.6,13.0,0.0\nR003,2016-12-08,-7.6,0.0,27969.79\nR003,2016-12-09,-0.2,8.1,7326.5\nR003,2016-12-10,3.4,20.8,0.0\nR003,2016-12-11,4.1,30.2,0.0\nR003,2016-12-12,6.8,9.0,13971.46\nR003,2016-12-13,4.3,34.4,0.0\nR003,2016-12-14,-2.7,0.1,26998.06\nR003,2016-12-15,-2.6,1.6,28611.03\nR003,2016-12-16,-2.7,7.0,16008.18\nR003,2016-12-17,-2.9,5.8,22636.71\nR003,2016-12-18,-4.7,0.0,29222.84\nR003,2016-12-19,-3.6,4.7,26513.37\nR003,2016-12-20,-0.4,20.4,0.0\nR003,2016-12-21,0.8,2.7,25791.33\nR003,2016-12-22,-0.5,0.0,31129.41\nR003,2016-12-23,-1.0,0.0,31217.64\nR003,2016-12-24,1.6,0.0,31095.35\nR003,2016-12-25,1.6,0.8,29901.66\nR003,2016-12-26,1.5,3.5,23985.28\nR003,2016-12-27,1.8,0.9,27389.61\nR003,2016-12-28,2.3,7.9,1228.3\nR003,2016-12-29,4.1,3.9,1170.87\nR003,2016-12-30,5.7,0.7,27922.6\nR003,2016-12-31,6.7,0.0,30144.83\nR003,2017-01-01,3.7,5.0,9163.41\nR003,2017-01-02,2.0,3.7,26824.84\nR003,2017-01-03,0.3,0.0,31741.01\nR003,2017-01-04,0.3,14.4,0.0\nR003,2017-01-05,2.4,1.9,29300.59\nR003,2017-01-06,7.3,2.9,20389.33\nR003,2017-01-07,8.0,0.0,31981.29\nR003,2017-01-08,4.6,3.1,5630.56\nR003,2017-01-09,5.0,1.1,21896.36\nR003,2017-01-10,3.4,3.0,30666.0\nR003,2017-01-11,2.5,1.1,32240.22\nR003,2017-01-12,4.0,0.1,29192.19\nR003,2017-01-13,0.5,5.4,25035.7\nR003,2017-01-14,0.5,0.0,32400.0\nR003,2017-01-15,5.0,0.0,30091.42\nR003,2017-01-16,3.9,0.4,24437.44\nR003,2017-01-17,4.8,6.3,10988.49\nR003,2017-01-18,5.3,0.0,25059.17\nR003,2017-01-19,4.0,7.9,0.0\nR003,2017-01-20,1.7,8.6,32279.6\nR003,2017-01-21,0.5,1.3,28457.73\nR003,2017-01-22,-0.0,6.9,0.0\nR003,2017-01-23,-0.1,0.0,32260.48\nR003,2017-01-24,2.3,0.0,32400.0\nR003,2017-01-25,2.1,4.9,26335.89\nR003,2017-01-26,-1.4,10.6,5831.8\nR003,2017-01-27,-5.4,0.0,32400.0\nR003,2017-01-28,-6.4,0.0,32528.37\nR003,2017-01-29,-6.5,6.9,5836.14\nR003,2017-01-30,-9.6,0.5,32400.0\nR003,2017-01-31,-8.8,0.0,32753.16\nR003,2017-02-01,-4.9,0.0,32526.59\nR003,2017-02-02,-3.2,0.7,31877.16\nR003,2017-02-03,-0.0,0.0,32126.66\nR003,2017-02-04,3.2,4.7,0.0\nR003,2017-02-05,5.9,12.7,23044.26\nR003,2017-02-06,6.6,11.4,7441.88\nR003,2017-02-07,7.7,1.9,24784.26\nR003,2017-02-08,6.7,0.0,29409.73\nR003,2017-02-09,6.0,0.8,33537.78\nR003,2017-02-10,3.8,7.5,1627.61\nR003,2017-02-11,1.5,5.4,27176.02\nR003,2017-02-12,0.2,6.6,20638.65\nR003,2017-02-13,-0.9,1.2,30731.21\nR003,2017-02-14,-1.9,5.0,33148.75\nR003,2017-02-15,-2.5,20.6,9064.6\nR003,2017-02-16,-4.9,0.0,33909.95\nR003,2017-02-17,-5.9,0.0,37362.07\nR003,2017-02-18,-0.3,0.0,27219.79\nR003,2017-02-19,3.6,0.0,37979.71\nR003,2017-02-20,4.7,0.0,37913.27\nR003,2017-02-21,4.2,2.4,7845.92\nR003,2017-02-22,4.2,16.0,24674.6\nR003,2017-02-23,4.0,9.9,17828.3\nR003,2017-02-24,7.9,9.9,12046.38\nR003,2017-02-25,9.5,0.8,39031.31\nR003,2017-02-26,4.3,19.5,36604.04\nR003,2017-02-27,4.9,0.0,30062.65\nR003,2017-02-28,7.7,0.0,39478.88\nR003,2017-03-01,10.3,0.0,39581.15\nR003,2017-03-02,11.2,0.0,26223.14\nR003,2017-03-03,11.3,0.0,39600.0\nR003,2017-03-04,6.7,5.3,2894.57\nR003,2017-03-05,6.7,0.0,37141.99\nR003,2017-03-06,9.1,0.0,39600.0\nR003,2017-03-07,9.1,0.0,39600.0\nR003,2017-03-08,9.0,0.0,39600.0\nR003,2017-03-09,8.9,0.0,39600.0\nR003,2017-03-10,10.1,0.0,39600.0\nR003,2017-03-11,10.6,0.0,39600.0\nR003,2017-03-12,9.4,3.2,33944.13\nR003,2017-03-13,9.2,0.5,28230.87\nR003,2017-03-14,7.7,6.5,763.85\nR003,2017-03-15,5.3,7.8,1018.92\nR003,2017-03-16,4.5,0.9,37802.42\nR003,2017-03-17,5.8,1.2,36531.03\nR003,2017-03-18,5.0,7.1,6473.93\nR003,2017-03-19,6.2,0.0,19024.09\nR003,2017-03-20,4.6,8.0,21088.67\nR003,2017-03-21,2.1,3.4,39715.17\nR003,2017-03-22,3.1,1.2,16768.19\nR003,2017-03-23,6.1,0.0,33586.76\nR003,2017-03-24,6.7,0.0,39279.13\nR003,2017-03-25,6.8,3.3,38821.67\nR003,2017-03-26,6.1,3.4,4080.16\nR003,2017-03-27,5.1,13.3,352.3\nR003,2017-03-28,3.5,0.3,42806.37\nR003,2017-03-29,6.2,0.4,38189.8\nR003,2017-03-30,8.9,0.0,44690.18\nR003,2017-03-31,7.7,0.0,42364.51\nR003,2017-04-01,7.6,0.0,43559.02\nR003,2017-04-02,7.5,0.0,42679.13\nR003,2017-04-03,8.6,0.0,45704.28\nR003,2017-04-04,11.7,0.0,45882.34\nR003,2017-04-05,13.4,0.0,44701.09\nR003,2017-04-06,14.5,0.0,42093.37\nR003,2017-04-07,12.1,3.6,27217.41\nR003,2017-04-08,10.9,5.2,37843.96\nR003,2017-04-09,5.7,12.5,22185.87\nR003,2017-04-10,6.4,1.6,40842.18\nR003,2017-04-11,6.9,3.8,44829.04\nR003,2017-04-12,11.3,0.0,40300.7\nR003,2017-04-13,15.4,2.7,32080.39\nR003,2017-04-14,11.0,17.4,2756.9\nR003,2017-04-15,9.2,0.8,38547.81\nR003,2017-04-16,11.0,0.0,46501.09\nR003,2017-04-17,15.3,0.4,46800.0\nR003,2017-04-18,17.2,0.9,46800.0\nR003,2017-04-19,8.0,6.9,44912.78\nR003,2017-04-20,12.4,0.0,46969.29\nR003,2017-04-21,10.7,23.8,8331.25\nR003,2017-04-22,4.5,14.5,33353.54\nR003,2017-04-23,12.0,22.6,26893.61\nR003,2017-04-24,4.2,15.1,14196.17\nR003,2017-04-25,3.6,0.4,46800.0\nR003,2017-04-26,8.3,0.0,47698.27\nR003,2017-04-27,14.4,0.0,47989.93\nR003,2017-04-28,16.0,0.0,48283.54\nR003,2017-04-29,16.6,0.8,48245.24\nR003,2017-04-30,16.5,0.5,47537.99\nR003,2017-05-01,17.6,0.4,48505.99\nR003,2017-05-02,18.6,0.9,48806.71\nR003,2017-05-03,17.9,0.1,25185.87\nR003,2017-05-04,15.7,2.4,28246.58\nR003,2017-05-05,15.7,0.0,48707.07\nR003,2017-05-06,15.9,1.5,49354.24\nR003,2017-05-07,14.1,14.8,24792.32\nR003,2017-05-08,14.2,2.4,45889.95\nR003,2017-05-09,13.8,11.0,32999.15\nR003,2017-05-10,14.3,25.9,2609.85\nR003,2017-05-11,12.8,11.3,37675.57\nR003,2017-05-12,13.4,0.0,48161.78\nR003,2017-05-13,16.1,0.0,40436.38\nR003,2017-05-14,16.0,8.4,39813.57\nR003,2017-05-15,14.1,7.8,21334.91\nR003,2017-05-16,13.8,7.7,16477.56\nR003,2017-05-17,14.0,5.4,28986.65\nR003,2017-05-18,14.7,2.5,39046.52\nR003,2017-05-19,13.7,8.4,1503.76\nR003,2017-05-20,12.8,24.3,847.06\nR003,2017-05-21,12.1,13.2,28689.57\nR003,2017-05-22,11.5,11.7,23234.37\nR003,2017-05-23,11.8,8.2,24268.01\nR003,2017-05-24,10.4,16.1,1121.98\nR003,2017-05-25,12.2,0.4,39831.71\nR003,2017-05-26,16.1,0.0,50529.86\nR003,2017-05-27,16.8,2.2,44897.06\nR003,2017-05-28,16.7,5.9,50047.92\nR003,2017-05-29,17.6,2.0,49022.37\nR003,2017-05-30,19.6,2.7,50437.61\nR003,2017-05-31,17.0,9.1,19057.62\nR003,2017-06-01,16.5,6.9,33161.04\nR003,2017-06-02,16.6,4.5,46799.91\nR003,2017-06-03,16.6,1.7,44078.89\nR003,2017-06-04,18.6,0.0,51201.34\nR003,2017-06-05,21.8,0.0,52862.94\nR003,2017-06-06,23.5,2.1,51387.52\nR003,2017-06-07,19.7,29.4,13431.87\nR003,2017-06-08,21.1,2.6,44957.66\nR003,2017-06-09,22.7,0.7,51703.72\nR003,2017-06-10,20.4,6.8,33618.46\nR003,2017-06-11,18.9,13.1,42793.49\nR003,2017-06-12,18.1,0.9,48328.22\nR003,2017-06-13,18.8,0.0,52015.38\nR003,2017-06-14,19.7,0.6,50901.77\nR003,2017-06-15,16.9,4.7,47688.09\nR003,2017-06-16,14.5,4.9,47991.81\nR003,2017-06-17,15.8,0.0,53444.47\nR003,2017-06-18,18.5,1.3,48991.14\nR003,2017-06-19,17.9,9.6,19152.16\nR003,2017-06-20,16.9,29.1,8348.71\nR003,2017-06-21,18.4,16.2,44652.96\nR003,2017-06-22,18.6,6.6,45424.14\nR003,2017-06-23,18.6,4.3,52159.85\nR003,2017-06-24,20.3,0.0,53956.67\nR003,2017-06-25,20.2,0.1,53026.23\nR003,2017-06-26,22.7,0.1,53963.86\nR003,2017-06-27,24.0,2.1,53254.82\nR003,2017-06-28,24.5,1.8,53416.24\nR003,2017-06-29,22.8,1.9,53250.19\nR003,2017-06-30,22.5,2.0,48251.18\nR003,2017-07-01,24.2,0.1,52605.33\nR003,2017-07-02,27.2,0.0,53934.51\nR003,2017-07-03,28.6,0.0,53921.8\nR003,2017-07-04,27.7,3.7,50832.74\nR003,2017-07-05,22.9,8.1,41374.11\nR003,2017-07-06,22.0,2.2,49334.03\nR003,2017-07-07,20.1,13.9,47481.18\nR003,2017-07-08,18.4,1.2,49373.31\nR003,2017-07-09,19.8,0.0,51295.52\nR003,2017-07-10,20.8,0.0,53052.19\nR003,2017-07-11,21.0,0.2,50210.16\nR003,2017-07-12,22.2,0.4,52461.79\nR003,2017-07-13,21.7,7.3,52001.91\nR003,2017-07-14,20.7,6.0,49863.21\nR003,2017-07-15,21.8,5.8,51433.6\nR003,2017-07-16,23.6,1.1,53472.85\nR003,2017-07-17,25.1,1.5,53485.51\nR003,2017-07-18,25.7,0.8,53143.04\nR003,2017-07-19,25.3,1.3,50934.75\nR003,2017-07-20,22.1,18.4,30753.46\nR003,2017-07-21,23.6,1.4,46893.26\nR003,2017-07-22,23.4,5.4,48704.89\nR003,2017-07-23,22.7,9.7,32093.47\nR003,2017-07-24,23.4,0.5,48335.26\nR003,2017-07-25,22.6,6.1,45034.56\nR003,2017-07-26,23.5,0.9,48783.28\nR003,2017-07-27,25.6,0.0,51796.26\nR003,2017-07-28,28.0,0.2,51064.19\nR003,2017-07-29,22.2,28.4,33198.86\nR003,2017-07-30,21.5,0.0,48643.25\nR003,2017-07-31,21.4,0.0,50079.39\nR003,2017-08-01,23.9,0.0,48755.14\nR003,2017-08-02,25.7,0.0,50187.35\nR003,2017-08-03,26.5,0.1,49556.92\nR003,2017-08-04,26.9,0.5,49273.54\nR003,2017-08-05,27.0,0.0,48309.56\nR003,2017-08-06,27.1,0.4,48373.38\nR003,2017-08-07,27.4,2.6,48098.4\nR003,2017-08-08,27.2,1.2,47985.47\nR003,2017-08-09,26.5,0.6,47337.0\nR003,2017-08-10,26.9,0.0,47703.6\nR003,2017-08-11,26.9,0.4,47732.24\nR003,2017-08-12,27.6,1.1,47425.27\nR003,2017-08-13,27.1,3.7,47119.87\nR003,2017-08-14,25.5,2.3,45817.85\nR003,2017-08-15,22.7,13.8,38095.96\nR003,2017-08-16,22.4,1.1,46800.0\nR003,2017-08-17,22.6,10.4,46908.97\nR003,2017-08-18,21.6,7.3,46455.92\nR003,2017-08-19,22.0,3.5,45515.91\nR003,2017-08-20,22.2,3.7,46800.0\nR003,2017-08-21,24.6,0.0,46800.0\nR003,2017-08-22,24.9,3.5,38734.22\nR003,2017-08-23,20.9,12.4,40158.09\nR003,2017-08-24,20.9,0.3,44606.89\nR003,2017-08-25,22.3,0.1,45407.85\nR003,2017-08-26,22.3,0.1,43637.9\nR003,2017-08-27,23.0,0.1,46714.15\nR003,2017-08-28,23.0,1.0,46610.72\nR003,2017-08-29,21.0,10.3,33385.08\nR003,2017-08-30,20.5,13.3,39505.83\nR003,2017-08-31,17.7,33.8,13766.23\nR003,2017-09-01,18.6,1.1,42065.73\nR003,2017-09-02,19.9,0.0,40685.37\nR003,2017-09-03,20.9,0.0,45336.76\nR003,2017-09-04,23.5,0.0,45065.16\nR003,2017-09-05,23.3,0.0,44474.29\nR003,2017-09-06,21.5,13.7,40373.72\nR003,2017-09-07,17.6,23.0,27241.17\nR003,2017-09-08,17.7,0.0,43668.98\nR003,2017-09-09,20.6,0.0,43721.79\nR003,2017-09-10,21.9,0.0,43456.43\nR003,2017-09-11,21.6,0.0,43200.0\nR003,2017-09-12,23.2,0.0,43246.49\nR003,2017-09-13,23.7,0.0,43163.55\nR003,2017-09-14,23.0,0.0,43096.21\nR003,2017-09-15,23.2,0.0,43028.8\nR003,2017-09-16,23.2,0.0,42961.28\nR003,2017-09-17,22.9,0.0,42893.66\nR003,2017-09-18,22.8,0.0,42825.95\nR003,2017-09-19,22.2,0.0,42758.12\nR003,2017-09-20,23.2,0.0,42690.17\nR003,2017-09-21,23.5,0.0,42622.1\nR003,2017-09-22,23.2,0.6,42262.27\nR003,2017-09-23,20.7,6.1,31100.1\nR003,2017-09-24,19.3,3.3,31407.8\nR003,2017-09-25,18.8,0.5,38579.2\nR003,2017-09-26,19.3,0.0,40811.18\nR003,2017-09-27,17.5,0.3,37647.72\nR003,2017-09-28,14.4,1.0,39925.51\nR003,2017-09-29,14.7,0.3,37672.58\nR003,2017-09-30,15.4,0.7,37568.8\nR003,2017-10-01,14.9,0.7,34254.56\nR003,2017-10-02,13.9,0.1,35599.76\nR003,2017-10-03,11.2,6.4,31406.36\nR003,2017-10-04,10.7,0.0,39003.93\nR003,2017-10-05,12.5,0.0,39007.41\nR003,2017-10-06,14.4,0.0,38780.57\nR003,2017-10-07,16.1,0.0,36936.19\nR003,2017-10-08,18.0,0.0,38648.86\nR003,2017-10-09,16.1,2.0,35864.15\nR003,2017-10-10,13.6,7.2,33146.71\nR003,2017-10-11,12.4,1.4,36497.64\nR003,2017-10-12,11.4,0.0,36551.7\nR003,2017-10-13,11.4,0.0,36279.05\nR003,2017-10-14,10.3,7.1,32400.0\nR003,2017-10-15,9.7,10.8,28828.56\nR003,2017-10-16,8.7,22.0,30438.68\nR003,2017-10-17,8.5,0.5,27688.51\nR003,2017-10-18,11.7,0.0,30632.74\nR003,2017-10-19,12.2,0.0,36000.0\nR003,2017-10-20,13.2,0.0,36000.0\nR003,2017-10-21,13.2,0.0,36000.0\nR003,2017-10-22,11.9,9.7,22170.77\nR003,2017-10-23,11.2,1.0,33373.82\nR003,2017-10-24,14.5,0.0,32400.0\nR003,2017-10-25,17.2,0.0,36000.0\nR003,2017-10-26,14.6,8.1,23073.76\nR003,2017-10-27,11.4,1.2,34252.32\nR003,2017-10-28,13.0,0.0,36000.0\nR003,2017-10-29,12.5,20.6,313.04\nR003,2017-10-30,9.4,51.3,0.0\nR003,2017-10-31,3.5,18.5,0.0\nR003,2017-11-01,1.6,15.9,17843.29\nR003,2017-11-02,1.1,0.0,35926.65\nR003,2017-11-03,2.7,0.0,35847.91\nR003,2017-11-04,4.4,0.0,32769.74\nR003,2017-11-05,6.4,0.0,33039.97\nR003,2017-11-06,10.7,1.3,30008.06\nR003,2017-11-07,10.3,2.0,33139.85\nR003,2017-11-08,10.5,0.7,33470.05\nR003,2017-11-09,8.6,0.0,34773.93\nR003,2017-11-10,9.2,1.0,31817.7\nR003,2017-11-11,8.9,1.8,33410.87\nR003,2017-11-12,10.0,3.0,25738.86\nR003,2017-11-13,11.8,1.5,32300.62\nR003,2017-11-14,12.9,0.0,29268.44\nR003,2017-11-15,12.4,0.0,34199.63\nR003,2017-11-16,10.1,0.0,31990.43\nR003,2017-11-17,9.3,0.0,32973.27\nR003,2017-11-18,10.5,0.0,32154.11\nR003,2017-11-19,12.8,0.0,28800.0\nR003,2017-11-20,12.6,1.4,30805.91\nR003,2017-11-21,10.5,17.4,14972.31\nR003,2017-11-22,3.3,3.8,30995.63\nR003,2017-11-23,-0.5,13.6,26900.33\nR003,2017-11-24,-4.0,5.7,14106.24\nR003,2017-11-25,-3.9,0.0,31233.82\nR003,2017-11-26,-1.3,0.0,30976.27\nR003,2017-11-27,-0.2,0.0,31000.61\nR003,2017-11-28,2.8,0.0,29497.21\nR003,2017-11-29,6.3,6.2,2320.25\nR003,2017-11-30,5.8,0.5,30307.91\nR003,2017-12-01,7.1,0.0,29945.29\nR003,2017-12-02,9.5,0.0,29960.29\nR003,2017-12-03,10.9,0.0,28800.0\nR003,2017-12-04,12.9,0.0,28860.09\nR003,2017-12-05,8.2,29.9,0.0\nR003,2017-12-06,2.8,5.8,11090.14\nR003,2017-12-07,0.7,17.4,0.0\nR003,2017-12-08,3.5,16.3,0.0\nR003,2017-12-09,5.0,0.4,30413.26\nR003,2017-12-10,5.6,0.0,30417.86\nR003,2017-12-11,3.9,8.8,30033.53\nR003,2017-12-12,5.2,0.0,28800.0\nR003,2017-12-13,3.9,0.0,30422.59\nR003,2017-12-14,5.5,0.0,27873.81\nR003,2017-12-15,7.4,0.0,28421.91\nR003,2017-12-16,7.4,0.0,29050.61\nR003,2017-12-17,10.1,0.0,28800.0\nR003,2017-12-18,10.8,0.4,19492.86\nR003,2017-12-19,8.5,6.4,11814.51\nR003,2017-12-20,6.0,10.2,0.0\nR003,2017-12-21,5.4,16.2,14772.61\nR003,2017-12-22,4.4,20.6,20598.22\nR003,2017-12-23,4.2,33.2,0.0\nR003,2017-12-24,2.4,4.3,27104.84\nR003,2017-12-25,3.8,36.8,0.0\nR003,2017-12-26,6.3,8.5,10510.57\nR003,2017-12-27,8.5,0.0,28800.0\nR003,2017-12-28,10.2,0.0,31401.32\nR003,2017-12-29,9.4,0.0,30985.4\nR003,2017-12-30,9.1,0.8,29005.77\nR003,2017-12-31,8.0,4.7,28800.0\nR003,2018-01-01,6.1,11.9,0.0\nR003,2018-01-02,4.5,0.0,29638.72\nR003,2018-01-03,4.7,0.0,28896.69\nR003,2018-01-04,5.4,0.0,16896.04\nR003,2018-01-05,6.4,0.1,30002.03\nR003,2018-01-06,6.0,0.0,30913.84\nR003,2018-01-07,4.3,0.0,31956.7\nR003,2018-01-08,5.6,0.2,29123.41\nR003,2018-01-09,3.8,15.9,8159.52\nR003,2018-01-10,-3.0,0.0,32154.9\nR003,2018-01-11,1.4,0.5,28800.0\nR003,2018-01-12,5.0,4.9,14312.71\nR003,2018-01-13,3.5,11.6,630.58\nR003,2018-01-14,4.0,0.0,28800.0\nR003,2018-01-15,2.7,6.3,14429.53\nR003,2018-01-16,1.3,0.9,31840.49\nR003,2018-01-17,3.2,1.6,6478.83\nR003,2018-01-18,4.9,0.3,26602.66\nR003,2018-01-19,5.0,14.3,0.0\nR003,2018-01-20,4.5,1.3,30410.04\nR003,2018-01-21,6.2,1.6,10165.34\nR003,2018-01-22,5.7,0.0,21515.17\nR003,2018-01-23,6.5,2.1,11076.28\nR003,2018-01-24,2.4,9.9,14132.27\nR003,2018-01-25,0.8,0.0,22136.33\nR003,2018-01-26,3.0,0.0,28628.88\nR003,2018-01-27,1.6,0.5,32400.0\nR003,2018-01-28,1.9,0.0,32400.0\nR003,2018-01-29,1.8,6.8,2324.24\nR003,2018-01-30,3.0,8.6,2082.65\nR003,2018-01-31,2.4,19.1,26378.87\nR003,2018-02-01,-0.9,0.4,32719.43\nR003,2018-02-02,3.1,0.6,32496.62\nR003,2018-02-03,6.7,5.8,3808.62\nR003,2018-02-04,7.5,1.9,2875.81\nR003,2018-02-05,6.8,12.9,3227.74\nR003,2018-02-06,2.5,0.0,32495.23\nR003,2018-02-07,3.6,0.0,34080.84\nR003,2018-02-08,6.8,0.1,32950.98\nR003,2018-02-09,6.8,6.2,15782.66\nR003,2018-02-10,5.2,0.0,34621.18\nR003,2018-02-11,8.4,0.1,30041.16\nR003,2018-02-12,6.2,21.8,11084.38\nR003,2018-02-13,3.6,7.0,25695.29\nR003,2018-02-14,3.5,0.6,35235.17\nR003,2018-02-15,5.6,0.7,31618.63\nR003,2018-02-16,8.4,0.0,32400.0\nR003,2018-02-17,8.0,0.2,35527.06\nR003,2018-02-18,6.4,0.0,33747.48\nR003,2018-02-19,4.3,0.0,34093.94\nR003,2018-02-20,4.9,0.3,24042.19\nR003,2018-02-21,6.9,0.4,28195.87\nR003,2018-02-22,7.8,0.0,37821.43\nR003,2018-02-23,7.2,6.6,34260.89\nR003,2018-02-24,5.8,3.6,33638.45\nR003,2018-02-25,6.7,0.3,29939.6\nR003,2018-02-26,4.8,4.4,28544.31\nR003,2018-02-27,6.9,1.9,26335.89\nR003,2018-02-28,8.9,0.0,38166.96\nR003,2018-03-01,3.4,20.9,10749.18\nR003,2018-03-02,-0.0,0.1,36000.0\nR003,2018-03-03,7.4,5.0,12033.12\nR003,2018-03-04,7.9,19.6,30083.43\nR003,2018-03-05,4.7,14.8,23822.26\nR003,2018-03-06,4.5,0.0,39600.0\nR003,2018-03-07,11.1,3.7,24325.5\nR003,2018-03-08,10.8,7.8,17634.84\nR003,2018-03-09,9.6,2.6,33124.52\nR003,2018-03-10,4.5,9.3,27788.57\nR003,2018-03-11,5.0,7.8,30533.28\nR003,2018-03-12,5.5,0.4,39600.0\nR003,2018-03-13,9.6,0.0,39600.0\nR003,2018-03-14,9.8,8.0,22614.85\nR003,2018-03-15,8.2,5.9,33844.36\nR003,2018-03-16,8.0,5.6,37520.74\nR003,2018-03-17,9.1,0.0,36685.79\nR003,2018-03-18,14.7,0.2,28363.54\nR003,2018-03-19,14.9,1.2,36859.15\nR003,2018-03-20,13.7,4.4,36000.0\nR003,2018-03-21,9.4,6.2,39600.0\nR003,2018-03-22,7.0,17.5,1162.7\nR003,2018-03-23,10.3,5.6,10800.0\nR003,2018-03-24,11.1,22.1,0.0\nR003,2018-03-25,6.9,6.2,26395.58\nR003,2018-03-26,9.6,1.7,36846.82\nR003,2018-03-27,10.9,1.5,32074.6\nR003,2018-03-28,10.4,7.8,34485.5\nR003,2018-03-29,11.8,10.9,9499.17\nR003,2018-03-30,9.7,12.5,1740.96\nR003,2018-03-31,9.1,0.9,40659.53\nR003,2018-04-01,12.4,0.0,44178.23\nR003,2018-04-02,14.0,10.8,37798.66\nR003,2018-04-03,7.1,6.0,43908.28\nR003,2018-04-04,7.4,0.0,45678.94\nR003,2018-04-05,10.2,0.0,45937.6\nR003,2018-04-06,12.8,0.0,46009.06\nR003,2018-04-07,14.7,0.0,46080.44\nR003,2018-04-08,14.9,0.0,46151.76\nR003,2018-04-09,17.1,0.0,45863.25\nR003,2018-04-10,18.4,0.0,46294.13\nR003,2018-04-11,17.4,0.0,46365.17\nR003,2018-04-12,16.4,1.1,45772.87\nR003,2018-04-13,14.5,2.0,38830.61\nR003,2018-04-14,11.8,0.0,44614.61\nR003,2018-04-15,10.5,0.0,46648.61\nR003,2018-04-16,10.7,0.0,46719.67\nR003,2018-04-17,12.1,0.0,46790.83\nR003,2018-04-18,13.9,0.0,46800.0\nR003,2018-04-19,10.8,7.8,34572.68\nR003,2018-04-20,8.3,13.9,23123.27\nR003,2018-04-21,10.0,1.7,43200.0\nR003,2018-04-22,9.8,1.9,44177.13\nR003,2018-04-23,10.4,0.2,46800.0\nR003,2018-04-24,11.0,0.0,46800.0\nR003,2018-04-25,14.1,0.0,47085.65\nR003,2018-04-26,16.9,0.0,47374.12\nR003,2018-04-27,18.7,0.3,47664.62\nR003,2018-04-28,17.9,0.4,45600.43\nR003,2018-04-29,16.8,0.0,46800.0\nR003,2018-04-30,17.6,0.0,48548.96\nR003,2018-05-01,19.2,1.3,48512.94\nR003,2018-05-02,18.8,4.2,48477.83\nR003,2018-05-03,19.6,5.5,48780.45\nR003,2018-05-04,20.0,6.3,47735.33\nR003,2018-05-05,18.8,5.1,25596.14\nR003,2018-05-06,18.1,9.3,47668.03\nR003,2018-05-07,17.5,13.3,34740.88\nR003,2018-05-08,15.4,15.3,10542.39\nR003,2018-05-09,14.7,0.7,9288.41\nR003,2018-05-10,14.9,8.8,10442.52\nR003,2018-05-11,13.9,9.3,20843.5\nR003,2018-05-12,13.1,4.0,38466.74\nR003,2018-05-13,12.6,9.3,3231.74\nR003,2018-05-14,14.9,2.4,26076.57\nR003,2018-05-15,16.5,4.7,42863.09\nR003,2018-05-16,18.2,2.8,46775.59\nR003,2018-05-17,19.1,3.7,49176.14\nR003,2018-05-18,19.9,0.3,47533.5\nR003,2018-05-19,20.7,1.1,49204.85\nR003,2018-05-20,20.1,2.4,50269.84\nR003,2018-05-21,18.8,9.8,41511.6\nR003,2018-05-22,19.2,3.9,48085.25\nR003,2018-05-23,20.1,11.1,47458.22\nR003,2018-05-24,18.9,16.1,42335.94\nR003,2018-05-25,19.9,11.0,38436.38\nR003,2018-05-26,19.5,6.5,36056.65\nR003,2018-05-27,21.6,2.3,45783.78\nR003,2018-05-28,22.7,0.8,49637.74\nR003,2018-05-29,19.6,0.0,17822.03\nR003,2018-05-30,17.7,10.4,7535.16\nR003,2018-05-31,18.6,13.1,37730.4\nR003,2018-06-01,18.2,4.9,45395.24\nR003,2018-06-02,18.2,1.0,44544.59\nR003,2018-06-03,18.5,1.2,47163.89\nR003,2018-06-04,18.8,2.0,50273.32\nR003,2018-06-05,18.8,5.5,48410.35\nR003,2018-06-06,19.7,0.6,52483.41\nR003,2018-06-07,20.4,0.0,51371.94\nR003,2018-06-08,19.7,2.7,51355.54\nR003,2018-06-09,19.4,0.0,50420.48\nR003,2018-06-10,19.4,0.4,52790.21\nR003,2018-06-11,18.8,2.0,51677.75\nR003,2018-06-12,19.6,0.8,53499.17\nR003,2018-06-13,20.9,0.1,53121.28\nR003,2018-06-14,23.6,0.0,53793.15\nR003,2018-06-15,23.7,5.4,52732.84\nR003,2018-06-16,20.3,12.9,7966.67\nR003,2018-06-17,18.9,13.7,50699.54\nR003,2018-06-18,19.1,8.1,43527.13\nR003,2018-06-19,19.7,8.2,37829.42\nR003,2018-06-20,21.4,0.2,53569.73\nR003,2018-06-21,22.7,0.0,53928.94\nR003,2018-06-22,23.0,4.8,49887.95\nR003,2018-06-23,20.5,17.0,20774.31\nR003,2018-06-24,21.6,4.2,49875.71\nR003,2018-06-25,22.8,1.8,53964.27\nR003,2018-06-26,23.6,8.3,53755.15\nR003,2018-06-27,24.1,5.0,53968.9\nR003,2018-06-28,26.4,0.2,52931.3\nR003,2018-06-29,26.8,2.2,53964.27\nR003,2018-06-30,25.0,3.0,50816.99\nR003,2018-07-01,25.1,3.2,50860.93\nR003,2018-07-02,24.6,1.1,51629.19\nR003,2018-07-03,24.4,2.8,47919.64\nR003,2018-07-04,22.9,6.4,1238.6\nR003,2018-07-05,22.6,8.4,12032.23\nR003,2018-07-06,20.5,16.9,3155.03\nR003,2018-07-07,21.9,0.8,47428.49\nR003,2018-07-08,22.7,1.0,53762.72\nR003,2018-07-09,24.0,0.7,53799.44\nR003,2018-07-10,26.9,0.0,53770.95\nR003,2018-07-11,26.5,2.1,53059.4\nR003,2018-07-12,25.8,1.1,53428.42\nR003,2018-07-13,24.9,2.7,47261.06\nR003,2018-07-14,23.0,1.9,47939.72\nR003,2018-07-15,21.3,11.6,34615.97\nR003,2018-07-16,19.6,3.0,49558.82\nR003,2018-07-17,19.8,1.8,45453.98\nR003,2018-07-18,21.9,0.0,53469.88\nR003,2018-07-19,21.4,2.6,50548.94\nR003,2018-07-20,21.0,2.3,49306.86\nR003,2018-07-21,21.7,0.0,49116.36\nR003,2018-07-22,22.6,0.1,50693.26\nR003,2018-07-23,21.6,7.3,48849.71\nR003,2018-07-24,22.8,2.3,50076.13\nR003,2018-07-25,27.1,0.2,51881.92\nR003,2018-07-26,26.6,3.2,49814.92\nR003,2018-07-27,26.1,0.8,51460.24\nR003,2018-07-28,25.4,5.6,50727.92\nR003,2018-07-29,24.9,7.3,47479.61\nR003,2018-07-30,25.1,2.5,48283.83\nR003,2018-07-31,26.4,0.2,50068.1\nR003,2018-08-01,24.8,8.0,48480.63\nR003,2018-08-02,23.1,5.1,45792.89\nR003,2018-08-03,21.7,10.6,43589.49\nR003,2018-08-04,22.2,5.1,39969.25\nR003,2018-08-05,22.3,2.4,46295.55\nR003,2018-08-06,23.2,0.6,47913.86\nR003,2018-08-07,24.6,0.1,48292.77\nR003,2018-08-08,23.2,0.8,46718.52\nR003,2018-08-09,21.8,6.4,26264.75\nR003,2018-08-10,20.0,7.9,33205.08\nR003,2018-08-11,20.0,0.2,45905.3\nR003,2018-08-12,19.5,2.1,45413.87\nR003,2018-08-13,20.0,0.2,46800.0\nR003,2018-08-14,21.3,0.0,47476.47\nR003,2018-08-15,22.7,0.0,47172.18\nR003,2018-08-16,23.7,0.0,47202.46\nR003,2018-08-17,24.0,1.0,46900.63\nR003,2018-08-18,22.8,0.9,46800.0\nR003,2018-08-19,22.6,0.0,46800.0\nR003,2018-08-20,22.5,0.0,46800.0\nR003,2018-08-21,22.7,1.7,46800.0\nR003,2018-08-22,22.5,0.4,46800.0\nR003,2018-08-23,23.2,0.0,46800.0\nR003,2018-08-24,23.4,1.3,46800.0\nR003,2018-08-25,23.7,1.1,46800.0\nR003,2018-08-26,23.2,1.3,46482.6\nR003,2018-08-27,23.3,2.9,46735.84\nR003,2018-08-28,22.8,1.3,46634.04\nR003,2018-08-29,24.0,0.1,46532.0\nR003,2018-08-30,24.7,2.8,46429.83\nR003,2018-08-31,22.4,6.6,31283.33\nR003,2018-09-01,21.0,8.8,38470.68\nR003,2018-09-02,20.6,4.4,44630.1\nR003,2018-09-03,21.7,2.7,44466.57\nR003,2018-09-04,21.5,1.1,44408.6\nR003,2018-09-05,22.1,0.4,44459.39\nR003,2018-09-06,23.9,0.6,44168.53\nR003,2018-09-07,19.8,10.5,26779.83\nR003,2018-09-08,18.3,2.2,24484.93\nR003,2018-09-09,19.3,0.0,40886.04\nR003,2018-09-10,18.1,3.4,16695.27\nR003,2018-09-11,18.0,5.4,24467.58\nR003,2018-09-12,18.5,0.2,41874.53\nR003,2018-09-13,22.4,0.0,43180.16\nR003,2018-09-14,22.6,14.1,39512.8\nR003,2018-09-15,19.4,6.6,38709.98\nR003,2018-09-16,17.6,9.3,16612.53\nR003,2018-09-17,18.4,2.2,39600.0\nR003,2018-09-18,17.3,0.0,40843.21\nR003,2018-09-19,17.6,0.0,40876.99\nR003,2018-09-20,18.6,0.0,42707.17\nR003,2018-09-21,19.7,0.0,42639.27\nR003,2018-09-22,19.5,2.7,33734.15\nR003,2018-09-23,19.0,1.8,37306.82\nR003,2018-09-24,19.5,0.1,39600.0\nR003,2018-09-25,20.0,0.6,41722.64\nR003,2018-09-26,14.8,7.3,24430.69\nR003,2018-09-27,15.2,0.0,37166.16\nR003,2018-09-28,15.7,0.0,29451.85\nR003,2018-09-29,17.2,0.1,37551.01\nR003,2018-09-30,16.0,2.1,38829.03\nR003,2018-10-01,19.6,0.0,33464.94\nR003,2018-10-02,20.2,4.3,38700.21\nR003,2018-10-03,18.0,0.0,39117.2\nR003,2018-10-04,14.2,10.9,20754.12\nR003,2018-10-05,9.6,1.3,38735.72\nR003,2018-10-06,9.4,0.0,39404.29\nR003,2018-10-07,14.5,0.0,39027.63\nR003,2018-10-08,17.6,0.0,38302.3\nR003,2018-10-09,17.3,0.1,30606.58\nR003,2018-10-10,17.7,3.1,32762.52\nR003,2018-10-11,17.2,1.6,36805.94\nR003,2018-10-12,16.2,0.0,36859.58\nR003,2018-10-13,17.2,0.0,36913.33\nR003,2018-10-14,16.4,0.0,36640.63\nR003,2018-10-15,16.5,0.0,36366.87\nR003,2018-10-16,18.3,0.0,36091.92\nR003,2018-10-17,18.1,0.0,36143.84\nR003,2018-10-18,17.6,0.0,36000.0\nR003,2018-10-19,17.1,0.0,36000.0\nR003,2018-10-20,15.5,0.3,36000.0\nR003,2018-10-21,15.3,2.8,33775.87\nR003,2018-10-22,14.8,4.4,2130.21\nR003,2018-10-23,13.4,1.9,24600.32\nR003,2018-10-24,13.6,17.1,199.79\nR003,2018-10-25,11.4,50.0,0.0\nR003,2018-10-26,7.3,3.9,29937.57\nR003,2018-10-27,6.7,0.0,36000.0\nR003,2018-10-28,10.5,0.0,36000.0\nR003,2018-10-29,13.4,0.0,36000.0\nR003,2018-10-30,14.2,0.0,36000.0\nR003,2018-10-31,14.2,0.0,36000.0\nR003,2018-11-01,14.0,0.0,36000.0\nR003,2018-11-02,12.5,0.0,35951.96\nR003,2018-11-03,11.9,0.0,35872.6\nR003,2018-11-04,12.4,0.0,35792.79\nR003,2018-11-05,11.6,0.0,35710.22\nR003,2018-11-06,11.0,0.2,35554.3\nR003,2018-11-07,10.7,0.4,32793.18\nR003,2018-11-08,10.1,0.0,35246.12\nR003,2018-11-09,9.6,0.0,35094.21\nR003,2018-11-10,9.9,0.0,34944.0\nR003,2018-11-11,9.2,0.0,34795.68\nR003,2018-11-12,8.8,3.7,27050.61\nR003,2018-11-13,6.3,7.8,18376.64\nR003,2018-11-14,6.0,3.3,6830.44\nR003,2018-11-15,5.3,8.8,1818.44\nR003,2018-11-16,4.8,0.2,22820.29\nR003,2018-11-17,6.0,15.0,1790.66\nR003,2018-11-18,4.8,0.1,31869.01\nR003,2018-11-19,4.7,0.0,29656.54\nR003,2018-11-20,9.4,4.0,0.0\nR003,2018-11-21,6.0,0.0,30530.29\nR003,2018-11-22,7.2,0.4,19397.41\nR003,2018-11-23,7.8,2.3,30141.73\nR003,2018-11-24,6.0,1.2,30914.08\nR003,2018-11-25,5.6,0.0,30802.17\nR003,2018-11-26,5.8,0.0,29742.12\nR003,2018-11-27,8.6,27.2,17432.97\nR003,2018-11-28,10.8,34.3,13310.1\nR003,2018-11-29,8.8,36.9,0.0\nR003,2018-11-30,10.9,2.9,18514.09\nR003,2018-12-01,9.6,15.5,0.0\nR003,2018-12-02,8.4,5.2,23853.8\nR003,2018-12-03,7.0,0.0,29590.64\nR003,2018-12-04,7.6,0.0,25948.46\nR003,2018-12-05,10.6,2.3,16674.38\nR003,2018-12-06,8.7,2.0,22022.51\nR003,2018-12-07,6.6,12.4,127.14\nR003,2018-12-08,2.8,5.8,28290.34\nR003,2018-12-09,2.5,0.0,30414.92\nR003,2018-12-10,7.3,0.2,26382.93\nR003,2018-12-11,7.9,11.8,86.13\nR003,2018-12-12,5.2,9.5,1105.42\nR003,2018-12-13,3.2,4.8,19463.64\nR003,2018-12-14,1.6,2.6,29255.54\nR003,2018-12-15,3.6,0.6,26378.03\nR003,2018-12-16,6.4,0.5,29244.07\nR003,2018-12-17,7.8,0.0,26026.08\nR003,2018-12-18,7.4,0.0,30007.34\nR003,2018-12-19,6.8,0.3,19393.11\nR003,2018-12-20,8.6,0.2,26768.93\nR003,2018-12-21,6.8,3.1,23342.41\nR003,2018-12-22,5.6,5.7,11880.32\nR003,2018-12-23,5.8,9.7,8000.41\nR003,2018-12-24,2.9,9.3,27247.2\nR003,2018-12-25,3.5,17.9,0.0\nR003,2018-12-26,1.4,26.1,0.0\nR003,2018-12-27,-0.0,13.8,19082.12\nR003,2018-12-28,-1.7,0.1,29453.98\nR003,2018-12-29,0.3,0.0,27108.47\nR003,2018-12-30,1.5,1.3,16715.44\nR003,2018-12-31,-0.2,0.1,26065.59\nR003,2019-01-01,-0.4,0.4,31592.16\nR003,2019-01-02,-1.7,4.0,18000.0\nR003,2019-01-03,5.3,0.5,4752.19\nR003,2019-01-04,5.2,19.6,6003.79\nR003,2019-01-05,1.7,4.1,24618.49\nR003,2019-01-06,4.3,10.4,0.0\nR003,2019-01-07,2.3,14.9,17229.2\nR003,2019-01-08,-0.6,0.5,20912.39\nR003,2019-01-09,0.1,14.2,4461.76\nR003,2019-01-10,0.3,0.0,32145.04\nR003,2019-01-11,5.0,0.0,29105.89\nR003,2019-01-12,4.5,14.1,0.0\nR003,2019-01-13,3.7,18.4,0.0\nR003,2019-01-14,4.2,4.0,13539.52\nR003,2019-01-15,6.1,5.1,20357.11\nR003,2019-01-16,5.1,36.6,0.0\nR003,2019-01-17,0.2,4.3,29551.21\nR003,2019-01-18,-1.3,0.0,32400.0\nR003,2019-01-19,1.3,0.0,24609.87\nR003,2019-01-20,2.8,1.7,22233.48\nR003,2019-01-21,2.6,0.0,32400.0\nR003,2019-01-22,2.3,8.7,22204.27\nR003,2019-01-23,3.2,0.0,30651.24\nR003,2019-01-24,6.7,0.0,29341.51\nR003,2019-01-25,7.3,0.0,2015.19\nR003,2019-01-26,6.6,0.0,2804.35\nR003,2019-01-27,6.6,11.2,4491.17\nR003,2019-01-28,5.9,3.2,25904.36\nR003,2019-01-29,4.9,0.0,32400.0\nR003,2019-01-30,6.4,0.0,31730.12\nR003,2019-01-31,7.5,0.0,28861.09\nR003,2019-02-01,6.1,3.1,14546.34\nR003,2019-02-02,5.0,0.5,32029.18\nR003,2019-02-03,4.9,0.0,32987.29\nR003,2019-02-04,6.5,0.0,32624.1\nR003,2019-02-05,7.0,0.0,33591.65\nR003,2019-02-06,8.6,0.0,32700.96\nR003,2019-02-07,8.9,1.5,0.0\nR003,2019-02-08,6.7,0.7,24824.44\nR003,2019-02-09,6.1,3.6,4360.55\nR003,2019-02-10,5.6,8.4,1945.24\nR003,2019-02-11,5.0,0.1,33654.9\nR003,2019-02-12,5.1,0.0,34813.2\nR003,2019-02-13,7.7,0.0,31818.4\nR003,2019-02-14,5.7,0.0,25955.26\nR003,2019-02-15,4.2,0.0,12170.84\nR003,2019-02-16,3.6,2.7,24168.71\nR003,2019-02-17,3.4,1.9,33763.29\nR003,2019-02-18,3.9,6.3,11432.57\nR003,2019-02-19,1.6,0.0,37902.07\nR003,2019-02-20,2.1,0.0,38077.1\nR003,2019-02-21,4.6,0.0,36995.31\nR003,2019-02-22,2.5,2.7,35044.33\nR003,2019-02-23,5.4,1.4,21029.07\nR003,2019-02-24,4.8,0.4,38780.69\nR003,2019-02-25,2.9,0.6,27373.66\nR003,2019-02-26,1.8,7.9,3525.75\nR003,2019-02-27,1.2,14.8,119.78\nR003,2019-02-28,-0.1,0.9,39427.92\nR003,2019-03-01,0.6,19.5,3540.66\nR003,2019-03-02,0.9,7.0,8143.77\nR003,2019-03-03,-0.4,2.7,39600.0\nR003,2019-03-04,-1.4,0.0,39600.0\nR003,2019-03-05,5.6,0.0,36678.7\nR003,2019-03-06,7.0,8.5,14111.96\nR003,2019-03-07,4.9,11.7,28091.62\nR003,2019-03-08,3.9,0.0,37136.86\nR003,2019-03-09,8.5,0.0,35534.61\nR003,2019-03-10,9.7,7.4,37270.57\nR003,2019-03-11,7.6,2.8,23901.59\nR003,2019-03-12,9.3,0.9,30092.64\nR003,2019-03-13,8.1,31.1,0.0\nR003,2019-03-14,8.2,2.5,31676.47\nR003,2019-03-15,9.2,2.1,29020.02\nR003,2019-03-16,7.3,15.1,5384.83\nR003,2019-03-17,4.3,12.6,9258.89\nR003,2019-03-18,2.8,0.5,39307.92\nR003,2019-03-19,5.1,0.0,40482.17\nR003,2019-03-20,9.6,0.0,41027.97\nR003,2019-03-21,7.8,0.2,39144.05\nR003,2019-03-22,5.5,2.6,36352.08\nR003,2019-03-23,1.6,19.1,474.61\nR003,2019-03-24,-0.2,1.2,40349.25\nR003,2019-03-25,2.4,0.0,40540.1\nR003,2019-03-26,5.8,0.0,43082.97\nR003,2019-03-27,7.2,0.1,42521.05\nR003,2019-03-28,3.3,9.2,35089.76\nR003,2019-03-29,1.5,4.2,38635.22\nR003,2019-03-30,2.1,0.0,44585.03\nR003,2019-03-31,9.2,0.0,44727.3\nR003,2019-04-01,8.4,3.2,33309.17\nR003,2019-04-02,5.7,11.5,6967.71\nR003,2019-04-03,5.4,5.3,35507.85\nR003,2019-04-04,6.3,2.7,40876.66\nR003,2019-04-05,8.6,0.0,44715.84\nR003,2019-04-06,11.7,0.0,45993.69\nR003,2019-04-07,13.5,0.0,46064.88\nR003,2019-04-08,14.4,0.0,45900.63\nR003,2019-04-09,15.5,1.9,46206.93\nR003,2019-04-10,13.4,5.6,36029.49\nR003,2019-04-11,11.5,4.3,41692.14\nR003,2019-04-12,10.8,9.5,29108.4\nR003,2019-04-13,12.0,4.1,34596.22\nR003,2019-04-14,14.4,0.0,34785.85\nR003,2019-04-15,12.7,10.6,23730.56\nR003,2019-04-16,10.9,7.8,30245.74\nR003,2019-04-17,11.3,4.1,36261.02\nR003,2019-04-18,8.7,10.2,0.0\nR003,2019-04-19,7.7,11.6,21498.19\nR003,2019-04-20,7.3,5.7,23157.98\nR003,2019-04-21,7.2,3.9,36116.63\nR003,2019-04-22,8.3,2.4,25938.23\nR003,2019-04-23,9.5,3.9,32280.14\nR003,2019-04-24,10.7,0.1,45828.97\nR003,2019-04-25,11.7,0.3,47094.45\nR003,2019-04-26,13.3,0.7,47383.18\nR003,2019-04-27,14.2,0.6,47674.11\nR003,2019-04-28,15.2,0.1,47634.28\nR003,2019-04-29,14.9,0.3,47261.74\nR003,2019-04-30,17.0,0.7,46800.0\nR003,2019-05-01,18.8,0.0,47519.5\nR003,2019-05-02,17.0,6.9,36010.52\nR003,2019-05-03,11.7,15.0,22734.95\nR003,2019-05-04,12.3,1.9,47072.03\nR003,2019-05-05,14.5,0.0,48095.24\nR003,2019-05-06,16.1,1.4,41252.27\nR003,2019-05-07,19.1,0.9,49342.52\nR003,2019-05-08,16.3,14.3,44118.42\nR003,2019-05-09,14.4,11.7,31564.84\nR003,2019-05-10,14.8,3.6,43134.6\nR003,2019-05-11,15.5,0.9,45862.65\nR003,2019-05-12,14.3,13.0,34734.97\nR003,2019-05-13,15.0,2.3,45427.03\nR003,2019-05-14,16.4,0.0,50371.78\nR003,2019-05-15,19.1,0.0,50400.0\nR003,2019-05-16,22.3,0.0,50460.32\nR003,2019-05-17,23.4,0.2,50431.38\nR003,2019-05-18,24.2,0.7,50400.0\nR003,2019-05-19,20.3,12.7,44049.92\nR003,2019-05-20,17.5,8.2,17328.2\nR003,2019-05-21,18.7,2.3,47883.83\nR003,2019-05-22,16.8,7.1,34414.14\nR003,2019-05-23,16.5,1.7,50620.67\nR003,2019-05-24,18.6,0.0,48926.88\nR003,2019-05-25,17.7,13.5,34403.33\nR003,2019-05-26,16.2,1.8,44613.35\nR003,2019-05-27,17.0,2.2,41731.09\nR003,2019-05-28,18.4,0.2,49229.73\nR003,2019-05-29,21.3,0.0,51910.21\nR003,2019-05-30,22.8,0.0,51889.04\nR003,2019-05-31,22.9,0.4,51506.82\nR003,2019-06-01,22.9,2.0,51848.33\nR003,2019-06-02,24.4,1.9,50400.0\nR003,2019-06-03,23.7,5.2,43744.73\nR003,2019-06-04,22.9,2.9,33651.69\nR003,2019-06-05,22.4,7.2,45325.53\nR003,2019-06-06,22.6,1.6,51392.8\nR003,2019-06-07,22.0,4.4,48792.06\nR003,2019-06-08,21.8,1.8,51726.28\nR003,2019-06-09,22.2,0.3,52229.18\nR003,2019-06-10,24.1,1.1,52795.53\nR003,2019-06-11,24.2,5.1,52782.44\nR003,2019-06-12,22.6,12.6,52402.99\nR003,2019-06-13,20.5,5.7,31375.54\nR003,2019-06-14,21.6,4.0,38769.9\nR003,2019-06-15,23.0,3.2,48941.51\nR003,2019-06-16,23.3,4.0,53826.06\nR003,2019-06-17,22.8,4.1,53448.57\nR003,2019-06-18,22.7,1.6,53439.17\nR003,2019-06-19,23.2,0.4,52327.14\nR003,2019-06-20,23.4,0.6,48753.2\nR003,2019-06-21,22.5,1.5,33045.9\nR003,2019-06-22,21.6,7.6,49447.94\nR003,2019-06-23,22.0,9.0,53919.49\nR003,2019-06-24,22.2,9.0,49342.03\nR003,2019-06-25,24.4,0.7,49103.73\nR003,2019-06-26,24.1,2.2,53020.73\nR003,2019-06-27,24.0,2.3,52284.18\nR003,2019-06-28,22.3,3.9,35176.02\nR003,2019-06-29,19.6,6.6,36856.74\nR003,2019-06-30,15.0,8.8,48698.58\nR003,2019-07-01,16.1,0.4,53949.56\nR003,2019-07-02,21.6,0.0,53939.05\nR003,2019-07-03,23.7,0.0,53926.44\nR003,2019-07-04,22.7,2.9,53401.92\nR003,2019-07-05,21.4,6.0,48547.68\nR003,2019-07-06,21.0,3.0,53131.78\nR003,2019-07-07,21.6,0.2,51746.15\nR003,2019-07-08,22.1,3.3,52672.95\nR003,2019-07-09,21.3,0.8,49789.84\nR003,2019-07-10,21.4,0.0,51470.15\nR003,2019-07-11,20.3,5.8,47943.83\nR003,2019-07-12,19.0,41.1,27733.82\nR003,2019-07-13,17.7,1.9,50747.35\nR003,2019-07-14,18.9,0.0,42496.74\nR003,2019-07-15,18.1,12.2,36845.66\nR003,2019-07-16,18.7,4.8,48715.37\nR003,2019-07-17,20.6,0.0,52762.7\nR003,2019-07-18,20.7,19.7,45982.11\nR003,2019-07-19,21.0,13.5,47901.4\nR003,2019-07-20,23.5,0.7,47679.1\nR003,2019-07-21,22.9,5.6,35612.72\nR003,2019-07-22,22.4,1.2,33560.61\nR003,2019-07-23,22.1,7.1,26174.93\nR003,2019-07-24,22.3,7.0,25353.94\nR003,2019-07-25,21.0,16.5,33943.85\nR003,2019-07-26,19.9,0.4,48625.27\nR003,2019-07-27,20.1,0.0,50769.58\nR003,2019-07-28,21.2,1.1,47012.49\nR003,2019-07-29,22.1,0.9,45641.16\nR003,2019-07-30,24.3,0.9,51013.0\nR003,2019-07-31,23.4,0.8,49711.52\nR003,2019-08-01,22.5,0.2,45398.23\nR003,2019-08-02,22.0,1.5,46581.96\nR003,2019-08-03,21.3,0.0,47816.78\nR003,2019-08-04,21.0,6.1,42776.32\nR003,2019-08-05,18.9,1.3,47256.36\nR003,2019-08-06,18.6,0.0,48288.95\nR003,2019-08-07,19.6,0.0,47946.45\nR003,2019-08-08,21.2,0.0,48469.28\nR003,2019-08-09,23.2,0.0,48338.69\nR003,2019-08-10,22.5,0.2,48027.46\nR003,2019-08-11,22.2,0.0,47044.13\nR003,2019-08-12,21.6,0.0,47746.35\nR003,2019-08-13,22.9,0.0,47774.93\nR003,2019-08-14,25.4,0.0,47469.02\nR003,2019-08-15,27.4,0.0,47498.64\nR003,2019-08-16,25.7,5.6,42335.04\nR003,2019-08-17,21.9,33.2,17190.73\nR003,2019-08-18,22.9,3.5,35984.3\nR003,2019-08-19,24.3,0.7,43418.77\nR003,2019-08-20,26.1,0.3,46800.0\nR003,2019-08-21,27.1,2.7,46800.0\nR003,2019-08-22,24.7,2.8,43570.46\nR003,2019-08-23,24.4,5.2,36484.92\nR003,2019-08-24,23.0,7.2,16194.44\nR003,2019-08-25,22.9,11.1,36948.86\nR003,2019-08-26,22.7,0.0,45873.69\nR003,2019-08-27,22.5,0.8,46767.59\nR003,2019-08-28,21.7,0.0,46667.11\nR003,2019-08-29,22.1,0.1,46565.28\nR003,2019-08-30,22.3,2.9,46462.22\nR003,2019-08-31,20.4,0.1,44653.15\nR003,2019-09-01,20.4,0.9,46177.2\nR003,2019-09-02,20.3,2.3,44939.53\nR003,2019-09-03,20.1,0.0,45631.01\nR003,2019-09-04,20.1,1.0,45039.33\nR003,2019-09-05,20.6,2.2,43920.13\nR003,2019-09-06,19.5,9.0,41105.35\nR003,2019-09-07,18.7,3.3,38186.39\nR003,2019-09-08,18.3,0.0,40626.38\nR003,2019-09-09,18.5,0.1,43044.97\nR003,2019-09-10,18.3,3.0,38270.77\nR003,2019-09-11,18.0,1.0,39610.26\nR003,2019-09-12,18.4,0.0,43200.0\nR003,2019-09-13,18.3,0.0,42900.62\nR003,2019-09-14,18.7,0.0,43129.64\nR003,2019-09-15,17.9,0.2,42319.19\nR003,2019-09-16,18.0,0.0,42995.15\nR003,2019-09-17,17.3,4.8,32372.85\nR003,2019-09-18,18.6,0.0,42860.27\nR003,2019-09-19,18.3,0.0,42792.6\nR003,2019-09-20,17.5,4.4,39400.71\nR003,2019-09-21,11.5,10.8,36107.92\nR003,2019-09-22,9.8,0.0,42244.43\nR003,2019-09-23,13.0,0.0,42279.56\nR003,2019-09-24,13.9,0.0,33285.69\nR003,2019-09-25,16.5,2.1,6878.75\nR003,2019-09-26,16.0,11.4,13212.51\nR003,2019-09-27,17.2,3.7,34082.95\nR003,2019-09-28,16.2,27.9,25412.85\nR003,2019-09-29,15.3,0.0,38053.49\nR003,2019-09-30,15.7,0.0,40181.59\nR003,2019-10-01,16.8,0.2,40508.17\nR003,2019-10-02,18.2,0.0,40357.18\nR003,2019-10-03,20.0,0.9,37160.61\nR003,2019-10-04,20.3,0.3,39576.32\nR003,2019-10-05,21.1,0.0,38961.46\nR003,2019-10-06,16.4,22.3,22266.52\nR003,2019-10-07,12.6,2.6,38510.29\nR003,2019-10-08,15.1,6.7,32760.63\nR003,2019-10-09,14.5,9.6,31429.23\nR003,2019-10-10,15.8,0.0,37490.88\nR003,2019-10-11,17.3,0.7,34102.36\nR003,2019-10-12,14.0,5.5,33797.72\nR003,2019-10-13,14.3,0.0,36575.11\nR003,2019-10-14,17.0,0.0,36629.46\nR003,2019-10-15,17.9,0.0,36356.7\nR003,2019-10-16,17.6,0.0,36082.54\nR003,2019-10-17,18.0,0.0,36134.92\nR003,2019-10-18,17.5,0.7,36000.0\nR003,2019-10-19,17.1,1.3,36000.0\nR003,2019-10-20,15.7,4.6,36000.0\nR003,2019-10-21,15.2,0.0,36000.0\nR003,2019-10-22,15.4,0.0,36000.0\nR003,2019-10-23,14.6,0.0,32400.0\nR003,2019-10-24,14.4,0.0,36000.0\nR003,2019-10-25,14.2,0.0,36000.0\nR003,2019-10-26,13.5,0.4,36000.0\nR003,2019-10-27,13.0,0.0,36000.0\nR003,2019-10-28,12.9,0.0,36000.0\nR003,2019-10-29,13.0,0.0,36000.0\nR003,2019-10-30,11.5,13.9,10846.7\nR003,2019-10-31,8.9,5.1,1163.09\nR003,2019-11-01,8.1,5.7,2741.68\nR003,2019-11-02,6.7,0.3,34447.06\nR003,2019-11-03,7.1,0.0,35889.76\nR003,2019-11-04,9.7,0.0,35809.42\nR003,2019-11-05,12.8,0.4,31430.46\nR003,2019-11-06,14.5,0.8,35583.13\nR003,2019-11-07,14.6,0.0,35427.5\nR003,2019-11-08,15.1,0.0,35273.76\nR003,2019-11-09,14.7,0.0,35122.11\nR003,2019-11-10,13.9,0.0,34972.73\nR003,2019-11-11,14.7,0.0,34825.82\nR003,2019-11-12,13.1,0.0,34681.59\nR003,2019-11-13,12.3,0.0,33736.8\nR003,2019-11-14,13.9,0.0,34401.99\nR003,2019-11-15,15.1,0.0,34266.86\nR003,2019-11-16,12.7,0.0,34133.56\nR003,2019-11-17,10.9,0.0,34001.75\nR003,2019-11-18,10.4,0.0,33691.4\nR003,2019-11-19,9.4,0.0,33322.98\nR003,2019-11-20,7.8,1.0,31534.38\nR003,2019-11-21,7.2,11.2,8056.68\nR003,2019-11-22,6.5,0.0,32217.56\nR003,2019-11-23,7.5,0.0,32217.61\nR003,2019-11-24,7.6,0.0,32206.13\nR003,2019-11-25,8.5,0.0,30935.78\nR003,2019-11-26,9.5,0.0,29067.16\nR003,2019-11-27,11.1,4.3,0.0\nR003,2019-11-28,7.7,12.3,13793.3\nR003,2019-11-29,8.6,1.4,22786.67\nR003,2019-11-30,8.5,20.8,2579.22\nR003,2019-12-01,5.9,17.3,0.0\nR003,2019-12-02,3.0,0.7,28962.09\nR003,2019-12-03,2.7,3.3,12032.94\nR003,2019-12-04,0.5,16.1,20696.86\nR003,2019-12-05,-4.6,0.0,30379.26\nR003,2019-12-06,-1.7,0.0,30390.04\nR003,2019-12-07,0.8,0.0,30399.26\nR003,2019-12-08,2.3,0.0,29636.8\nR003,2019-12-09,4.4,0.0,29255.78\nR003,2019-12-10,3.1,0.0,30417.4\nR003,2019-12-11,1.9,0.0,30420.33\nR003,2019-12-12,3.3,0.0,30421.78\nR003,2019-12-13,5.5,0.5,24136.66\nR003,2019-12-14,9.1,0.2,27499.41\nR003,2019-12-15,10.0,8.1,23279.51\nR003,2019-12-16,8.0,3.2,27398.88\nR003,2019-12-17,5.9,0.0,25620.48\nR003,2019-12-18,6.9,0.0,30789.17\nR003,2019-12-19,8.7,0.0,30780.57\nR003,2019-12-20,7.5,0.0,30770.54\nR003,2019-12-21,8.1,0.0,29313.18\nR003,2019-12-22,9.6,0.0,28456.18\nR003,2019-12-23,12.0,0.0,29557.86\nR003,2019-12-24,12.1,0.0,22623.81\nR003,2019-12-25,10.6,1.9,0.0\nR003,2019-12-26,8.7,0.3,26297.47\nR003,2019-12-27,6.7,11.5,420.55\nR003,2019-12-28,3.9,5.8,28800.0\nR003,2019-12-29,3.6,0.0,29991.16\nR003,2019-12-30,5.3,0.4,26596.44\nR003,2019-12-31,3.7,0.0,30169.45\nR003,2020-01-01,4.2,7.3,25700.36\nR003,2020-01-02,2.1,1.7,30338.81\nR003,2020-01-03,-0.3,0.0,16973.42\nR003,2020-01-04,1.1,0.0,31750.72\nR003,2020-01-05,2.8,0.0,30015.0\nR003,2020-01-06,2.8,0.0,26611.42\nR003,2020-01-07,5.2,5.6,0.0\nR003,2020-01-08,5.3,0.2,28300.61\nR003,2020-01-09,4.0,8.8,12770.99\nR003,2020-01-10,2.0,2.1,31009.04\nR003,2020-01-11,2.7,2.1,27568.28\nR003,2020-01-12,2.9,11.7,206.16\nR003,2020-01-13,-0.4,0.0,32333.98\nR003,2020-01-14,3.4,0.0,32400.0\nR003,2020-01-15,2.7,0.0,32400.0\nR003,2020-01-16,2.3,0.0,32400.0\nR003,2020-01-17,2.4,0.0,32400.0\nR003,2020-01-18,1.4,0.3,32400.0\nR003,2020-01-19,1.5,0.0,32400.0\nR003,2020-01-20,1.7,0.0,32400.0\nR003,2020-01-21,1.5,0.0,32400.0\nR003,2020-01-22,2.7,5.2,15680.94\nR003,2020-01-23,0.8,32.2,0.0\nR003,2020-01-24,-5.3,0.0,32400.0\nR003,2020-01-25,-0.2,0.0,23199.46\nR003,2020-01-26,2.5,0.0,31634.03\nR003,2020-01-27,4.4,0.0,32400.0\nR003,2020-01-28,7.5,0.0,25030.1\nR003,2020-01-29,7.0,8.0,3259.76\nR003,2020-01-30,6.2,11.2,131.37\nR003,2020-01-31,3.6,27.6,0.0\nR003,2020-02-01,1.0,0.4,32400.0\nR003,2020-02-02,4.0,6.7,5661.47\nR003,2020-02-03,6.7,34.9,0.0\nR003,2020-02-04,3.6,35.0,25387.03\nR003,2020-02-05,4.3,7.7,0.0\nR003,2020-02-06,9.3,18.1,0.0\nR003,2020-02-07,3.0,28.9,4069.63\nR003,2020-02-08,-3.7,6.1,7567.37\nR003,2020-02-09,-11.0,0.0,30601.99\nR003,2020-02-10,-9.6,0.0,35408.44\nR003,2020-02-11,-5.2,0.0,33869.92\nR003,2020-02-12,-0.6,10.6,6396.59\nR003,2020-02-13,-0.7,7.3,22072.3\nR003,2020-02-14,-0.9,1.6,34909.18\nR003,2020-02-15,1.5,0.0,26439.29\nR003,2020-02-16,3.9,0.0,35624.23\nR003,2020-02-17,2.9,0.0,37520.59\nR003,2020-02-18,5.2,0.0,37692.63\nR003,2020-02-19,8.0,0.0,37864.93\nR003,2020-02-20,5.8,0.0,32313.5\nR003,2020-02-21,6.2,0.0,34651.42\nR003,2020-02-22,6.4,0.3,30218.83\nR003,2020-02-23,4.9,4.9,24577.09\nR003,2020-02-24,5.6,16.6,10333.71\nR003,2020-02-25,1.5,13.0,38901.93\nR003,2020-02-26,3.5,0.0,39078.02\nR003,2020-02-27,10.9,0.0,39255.32\nR003,2020-02-28,6.3,20.6,12612.9\nR003,2020-02-29,5.4,6.5,22460.07\nR003,2020-03-01,5.0,12.1,19564.23\nR003,2020-03-02,5.6,0.0,39600.0\nR003,2020-03-03,8.7,0.0,39600.0\nR003,2020-03-04,11.7,0.0,39600.0\nR003,2020-03-05,12.6,0.0,39600.0\nR003,2020-03-06,12.8,0.0,28800.0\nR003,2020-03-07,12.8,0.0,37331.4\nR003,2020-03-08,11.2,0.0,39600.0\nR003,2020-03-09,12.4,0.0,39600.0\nR003,2020-03-10,13.9,0.0,38871.19\nR003,2020-03-11,12.6,0.4,39600.0\nR003,2020-03-12,9.0,1.0,39600.0\nR003,2020-03-13,11.2,0.0,39606.96\nR003,2020-03-14,11.4,5.8,24683.88\nR003,2020-03-15,7.4,18.6,1362.59\nR003,2020-03-16,1.5,12.0,13813.81\nR003,2020-03-17,-0.2,4.4,14775.91\nR003,2020-03-18,3.3,0.5,27944.02\nR003,2020-03-19,5.6,2.0,17222.14\nR003,2020-03-20,7.2,0.0,33072.17\nR003,2020-03-21,7.4,0.7,29865.2\nR003,2020-03-22,9.0,0.0,39587.93\nR003,2020-03-23,9.5,0.0,35989.58\nR003,2020-03-24,9.0,2.1,40914.01\nR003,2020-03-25,9.5,0.9,40228.06\nR003,2020-03-26,10.2,0.0,29705.37\nR003,2020-03-27,12.3,0.1,42233.41\nR003,2020-03-28,13.2,3.9,40358.99\nR003,2020-03-29,11.5,7.2,23839.7\nR003,2020-03-30,9.0,13.9,0.0\nR003,2020-03-31,9.4,3.6,35674.17\nR003,2020-04-01,8.8,2.6,35988.17\nR003,2020-04-02,8.1,2.8,44584.52\nR003,2020-04-03,8.8,0.1,39397.55\nR003,2020-04-04,10.5,2.0,36920.02\nR003,2020-04-05,10.2,0.0,35593.85\nR003,2020-04-06,9.4,0.0,46047.11\nR003,2020-04-07,7.6,0.0,46118.57\nR003,2020-04-08,9.2,0.0,46190.05\nR003,2020-04-09,10.1,0.0,46261.55\nR003,2020-04-10,10.0,0.0,46333.04\nR003,2020-04-11,8.0,3.3,44004.18\nR003,2020-04-12,6.7,2.9,41402.39\nR003,2020-04-13,7.1,0.0,46547.41\nR003,2020-04-14,11.0,0.0,46618.78\nR003,2020-04-15,12.0,7.5,27810.06\nR003,2020-04-16,7.6,12.5,34120.34\nR003,2020-04-17,10.2,0.0,46800.0\nR003,2020-04-18,12.8,0.0,43545.56\nR003,2020-04-19,12.5,0.0,46800.0\nR003,2020-04-20,13.0,6.0,38624.54\nR003,2020-04-21,8.8,5.7,42281.87\nR003,2020-04-22,7.0,0.0,26793.31\nR003,2020-04-23,7.1,0.6,46820.23\nR003,2020-04-24,7.7,0.5,47438.36\nR003,2020-04-25,10.3,0.1,45795.67\nR003,2020-04-26,10.3,0.3,47023.24\nR003,2020-04-27,8.5,0.7,39963.31\nR003,2020-04-28,8.9,0.2,46805.43\nR003,2020-04-29,11.3,0.0,48236.67\nR003,2020-04-30,14.7,0.0,46800.0\nR003,2020-05-01,14.9,2.4,41590.97\nR003,2020-05-02,13.6,11.1,788.29\nR003,2020-05-03,13.5,5.8,42664.44\nR003,2020-05-04,13.5,9.0,33516.42\nR003,2020-05-05,14.9,4.1,43941.15\nR003,2020-05-06,13.4,8.8,13327.74\nR003,2020-05-07,14.6,0.8,43728.51\nR003,2020-05-08,10.9,8.0,26561.86\nR003,2020-05-09,8.7,4.2,39390.67\nR003,2020-05-10,8.7,1.4,49578.5\nR003,2020-05-11,11.6,0.0,49210.17\nR003,2020-05-12,15.5,0.1,44991.77\nR003,2020-05-13,17.9,2.0,50355.79\nR003,2020-05-14,16.1,0.0,50340.87\nR003,2020-05-15,19.0,1.0,41966.16\nR003,2020-05-16,19.9,0.6,47649.28\nR003,2020-05-17,16.3,0.0,47933.87\nR003,2020-05-18,17.0,1.2,49264.08\nR003,2020-05-19,15.6,0.0,50400.0\nR003,2020-05-20,17.7,0.0,49634.03\nR003,2020-05-21,22.7,0.4,50400.0\nR003,2020-05-22,18.1,24.8,34022.19\nR003,2020-05-23,10.7,47.0,27314.53\nR003,2020-05-24,10.3,0.0,41586.15\nR003,2020-05-25,11.7,6.7,21963.66\nR003,2020-05-26,15.6,0.7,45795.81\nR003,2020-05-27,20.4,0.4,51575.98\nR003,2020-05-28,23.1,0.9,51193.78\nR003,2020-05-29,18.8,14.1,41938.96\nR003,2020-05-30,15.9,3.4,48273.44\nR003,2020-05-31,17.1,0.1,52216.06\nR003,2020-06-01,17.5,4.3,44608.09\nR003,2020-06-02,18.3,7.3,48801.87\nR003,2020-06-03,16.5,14.9,43556.16\nR003,2020-06-04,16.7,5.5,51454.61\nR003,2020-06-05,18.0,0.1,50527.98\nR003,2020-06-06,20.3,0.0,52477.78\nR003,2020-06-07,23.5,0.0,53558.35\nR003,2020-06-08,24.9,0.0,53178.6\nR003,2020-06-09,25.2,0.0,53164.63\nR003,2020-06-10,24.7,3.1,53517.48\nR003,2020-06-11,25.0,2.8,51671.33\nR003,2020-06-12,21.4,17.8,34422.55\nR003,2020-06-13,20.0,20.0,42955.85\nR003,2020-06-14,19.9,15.8,50126.24\nR003,2020-06-15,21.6,4.6,45352.88\nR003,2020-06-16,23.3,1.4,52609.55\nR003,2020-06-17,23.3,3.2,51148.73\nR003,2020-06-18,22.1,2.1,53899.63\nR003,2020-06-19,20.1,11.3,26551.39\nR003,2020-06-20,19.3,18.9,30309.31\nR003,2020-06-21,19.3,11.0,35027.53\nR003,2020-06-22,21.0,2.5,51107.97\nR003,2020-06-23,23.7,0.1,53957.1\nR003,2020-06-24,22.5,4.9,51190.73\nR003,2020-06-25,21.3,1.8,50357.19\nR003,2020-06-26,22.3,0.0,53818.43\nR003,2020-06-27,21.8,1.2,53965.46\nR003,2020-06-28,21.2,0.4,53962.5\nR003,2020-06-29,21.6,0.5,53957.49\nR003,2020-06-30,22.0,0.6,53380.81\nR003,2020-07-01,21.9,5.5,53746.14\nR003,2020-07-02,23.3,1.7,53929.85\nR003,2020-07-03,24.2,1.1,53916.37\nR003,2020-07-04,24.6,3.6,53748.8\nR003,2020-07-05,24.3,4.0,53882.88\nR003,2020-07-06,24.5,3.4,53027.77\nR003,2020-07-07,25.6,1.4,53840.57\nR003,2020-07-08,23.2,2.7,51472.09\nR003,2020-07-09,22.3,1.6,47477.03\nR003,2020-07-10,22.8,0.0,48480.67\nR003,2020-07-11,23.5,0.0,51852.02\nR003,2020-07-12,23.0,8.0,46960.85\nR003,2020-07-13,20.5,17.1,12145.88\nR003,2020-07-14,21.5,0.3,49751.31\nR003,2020-07-15,21.8,0.1,51464.35\nR003,2020-07-16,23.1,0.0,53474.96\nR003,2020-07-17,22.3,7.7,49852.5\nR003,2020-07-18,23.7,0.4,48600.78\nR003,2020-07-19,25.9,0.3,52090.62\nR003,2020-07-20,27.9,0.3,51698.12\nR003,2020-07-21,22.5,14.7,35413.65\nR003,2020-07-22,22.6,1.0,48082.65\nR003,2020-07-23,22.3,0.4,51233.37\nR003,2020-07-24,22.3,1.9,49189.96\nR003,2020-07-25,24.1,1.0,51196.56\nR003,2020-07-26,25.1,5.7,50774.56\nR003,2020-07-27,25.0,1.7,47955.83\nR003,2020-07-28,22.9,35.8,40951.59\nR003,2020-07-29,21.7,6.1,46748.24\nR003,2020-07-30,22.2,0.8,47231.3\nR003,2020-07-31,23.6,1.2,49059.87\nR003,2020-08-01,21.4,5.4,47117.74\nR003,2020-08-02,20.2,6.0,45270.16\nR003,2020-08-03,20.1,0.2,48200.95\nR003,2020-08-04,20.8,0.0,49630.45\nR003,2020-08-05,21.7,0.0,49348.73\nR003,2020-08-06,22.4,0.0,48728.81\nR003,2020-08-07,24.3,0.0,48453.29\nR003,2020-08-08,25.4,0.0,48331.52\nR003,2020-08-09,24.9,0.0,48020.39\nR003,2020-08-10,23.6,1.5,47711.08\nR003,2020-08-11,21.7,6.2,44878.96\nR003,2020-08-12,21.7,1.4,45334.99\nR003,2020-08-13,20.0,2.5,45455.27\nR003,2020-08-14,19.9,0.0,46178.41\nR003,2020-08-15,19.2,0.0,46800.0\nR003,2020-08-16,18.8,0.0,46216.3\nR003,2020-08-17,19.6,0.0,46918.44\nR003,2020-08-18,20.9,0.0,46949.8\nR003,2020-08-19,22.2,0.0,46800.0\nR003,2020-08-20,23.2,0.3,46800.0\nR003,2020-08-21,20.2,9.2,37828.33\nR003,2020-08-22,19.6,2.0,42850.56\nR003,2020-08-23,19.6,0.0,46800.0\nR003,2020-08-24,19.7,0.0,46800.0\nR003,2020-08-25,21.6,0.0,46800.0\nR003,2020-08-26,22.2,0.0,46794.61\nR003,2020-08-27,21.2,0.2,46691.92\nR003,2020-08-28,20.3,0.5,43213.93\nR003,2020-08-29,21.5,0.0,46483.85\nR003,2020-08-30,23.8,0.0,46378.68\nR003,2020-08-31,24.7,0.0,46272.89\nR003,2020-09-01,25.3,0.0,46166.59\nR003,2020-09-02,25.8,0.2,45300.92\nR003,2020-09-03,26.7,0.2,45029.8\nR003,2020-09-04,27.3,0.0,42096.48\nR003,2020-09-05,25.3,2.2,40988.84\nR003,2020-09-06,20.3,15.7,31676.65\nR003,2020-09-07,21.3,0.3,39389.42\nR003,2020-09-08,21.0,3.6,39992.93\nR003,2020-09-09,21.2,1.4,40734.92\nR003,2020-09-10,21.6,0.0,41044.55\nR003,2020-09-11,23.0,0.0,43207.18\nR003,2020-09-12,23.7,0.0,43200.0\nR003,2020-09-13,24.4,0.0,43145.92\nR003,2020-09-14,24.2,0.0,43078.67\nR003,2020-09-15,22.2,0.0,40123.18\nR003,2020-09-16,21.5,0.0,42943.81\nR003,2020-09-17,22.4,0.0,42876.2\nR003,2020-09-18,21.5,1.1,40787.72\nR003,2020-09-19,18.3,0.3,39277.78\nR003,2020-09-20,16.5,0.0,40291.82\nR003,2020-09-21,17.8,0.0,42604.44\nR003,2020-09-22,19.7,0.0,42271.44\nR003,2020-09-23,20.2,0.3,41988.36\nR003,2020-09-24,20.2,0.0,41386.8\nR003,2020-09-25,20.4,0.0,41422.58\nR003,2020-09-26,20.0,0.4,40501.85\nR003,2020-09-27,21.6,0.0,41175.93\nR003,2020-09-28,21.6,1.4,40573.77\nR003,2020-09-29,21.3,0.4,40068.02\nR003,2020-09-30,20.7,0.8,33911.59\nR003,2020-10-01,18.6,2.3,30348.29\nR003,2020-10-02,16.3,11.0,106.81\nR003,2020-10-03,16.5,4.4,30470.3\nR003,2020-10-04,16.5,1.9,38057.29\nR003,2020-10-05,16.4,0.4,37469.43\nR003,2020-10-06,16.0,0.0,37526.47\nR003,2020-10-07,17.0,0.0,38575.77\nR003,2020-10-08,18.3,0.0,38347.71\nR003,2020-10-09,20.7,0.0,37796.01\nR003,2020-10-10,20.9,0.0,37105.36\nR003,2020-10-11,19.1,0.0,36510.75\nR003,2020-10-12,19.0,0.0,36891.2\nR003,2020-10-13,20.6,0.0,36619.76\nR003,2020-10-14,20.9,0.0,36346.85\nR003,2020-10-15,19.4,0.0,36072.34\nR003,2020-10-16,18.0,0.0,36124.08\nR003,2020-10-17,18.3,0.0,36175.14\nR003,2020-10-18,19.8,0.0,36000.0\nR003,2020-10-19,16.5,14.0,30588.39\nR003,2020-10-20,11.9,0.0,36000.0\nR003,2020-10-21,11.9,0.0,36000.0\nR003,2020-10-22,12.9,0.0,30949.0\nR003,2020-10-23,15.2,0.0,36000.0\nR003,2020-10-24,16.3,0.0,36000.0\nR003,2020-10-25,16.8,0.0,36000.0\nR003,2020-10-26,17.0,0.0,36000.0\nR003,2020-10-27,16.7,1.3,35580.05\nR003,2020-10-28,19.2,0.0,36000.0\nR003,2020-10-29,18.9,0.0,36000.0\nR003,2020-10-30,17.7,0.9,34768.15\nR003,2020-10-31,14.6,10.9,35859.54\nR003,2020-11-01,13.9,14.3,27830.58\nR003,2020-11-02,11.0,10.1,35186.53\nR003,2020-11-03,10.3,0.0,35746.68\nR003,2020-11-04,13.1,0.0,35746.06\nR003,2020-11-05,13.8,0.0,34940.31\nR003,2020-11-06,11.9,2.2,34505.61\nR003,2020-11-07,9.5,0.0,30585.81\nR003,2020-11-08,9.0,0.0,35162.31\nR003,2020-11-09,11.4,0.0,35016.68\nR003,2020-11-10,10.4,0.0,34872.35\nR003,2020-11-11,9.5,0.7,33400.54\nR003,2020-11-12,9.1,0.8,32308.18\nR003,2020-11-13,7.8,0.0,34446.2\nR003,2020-11-14,9.1,0.0,34080.78\nR003,2020-11-15,8.1,0.0,31193.95\nR003,2020-11-16,6.9,5.7,7742.75\nR003,2020-11-17,5.9,5.0,30043.31\nR003,2020-11-18,3.5,2.0,27913.92\nR003,2020-11-19,2.8,4.9,26111.48\nR003,2020-11-20,2.6,7.7,24736.99\nR003,2020-11-21,4.6,4.0,29785.44\nR003,2020-11-22,3.4,18.2,13267.74\nR003,2020-11-23,-0.8,0.0,31453.14\nR003,2020-11-24,3.9,0.2,16755.93\nR003,2020-11-25,3.6,1.3,30959.46\nR003,2020-11-26,2.1,5.3,12234.55\nR003,2020-11-27,2.1,0.0,30633.66\nR003,2020-11-28,6.4,0.0,31404.94\nR003,2020-11-29,6.3,0.0,31087.88\nR003,2020-11-30,5.7,0.0,30763.03\nR003,2020-12-01,6.0,0.0,30054.65\nR003,2020-12-02,6.7,0.0,27632.18\nR003,2020-12-03,7.5,0.0,29598.2\nR003,2020-12-04,6.6,0.0,29608.96\nR003,2020-12-05,6.5,0.0,30385.07\nR003,2020-12-06,4.6,0.5,29626.34\nR003,2020-12-07,3.6,0.0,30402.65\nR003,2020-12-08,4.7,0.0,30409.26\nR003,2020-12-09,5.7,3.8,11509.37\nR003,2020-12-10,5.4,0.0,30418.18\nR003,2020-12-11,6.3,0.0,26569.49\nR003,2020-12-12,8.9,0.0,24610.52\nR003,2020-12-13,10.5,0.0,27463.76\nR003,2020-12-14,10.2,0.0,29232.56\nR003,2020-12-15,10.2,0.4,28800.0\nR003,2020-12-16,10.7,1.7,22611.47\nR003,2020-12-17,6.2,3.0,24401.8\nR003,2020-12-18,5.0,0.1,29614.39\nR003,2020-12-19,4.6,0.0,27888.93\nR003,2020-12-20,3.7,4.5,3535.39\nR003,2020-12-21,3.1,13.8,23923.3\nR003,2020-12-22,1.2,7.6,15722.29\nR003,2020-12-23,-0.8,1.6,22461.37\nR003,2020-12-24,-2.8,0.0,31254.78\nR003,2020-12-25,-0.5,1.2,25200.0\nR003,2020-12-26,7.3,1.7,27193.93\nR003,2020-12-27,9.2,0.0,31030.9\nR003,2020-12-28,8.3,0.0,29833.42\nR003,2020-12-29,9.9,0.0,31473.64\nR003,2020-12-30,8.2,0.0,31523.69\nR003,2020-12-31,8.0,0.0,31575.55\nR003,2021-01-01,9.0,0.0,31629.1\nR003,2021-01-02,8.2,0.0,31684.2\nR003,2021-01-03,5.5,0.0,31740.74\nR003,2021-01-04,4.9,0.0,31798.59\nR003,2021-01-05,7.4,0.0,28838.95\nR003,2021-01-06,7.1,0.8,17827.16\nR003,2021-01-07,6.5,0.0,29921.2\nR003,2021-01-08,10.1,0.0,14012.11\nR003,2021-01-09,11.3,0.0,21440.24\nR003,2021-01-10,9.5,20.5,2760.62\nR003,2021-01-11,7.0,20.8,0.0\nR003,2021-01-12,9.0,11.8,1121.75\nR003,2021-01-13,12.1,11.3,18291.37\nR003,2021-01-14,6.0,40.9,0.0\nR003,2021-01-15,2.9,27.5,10823.24\nR003,2021-01-16,-0.2,2.0,30759.97\nR003,2021-01-17,3.6,33.2,0.0\nR003,2021-01-18,0.3,11.3,22609.88\nR003,2021-01-19,-4.0,4.4,27160.91\nR003,2021-01-20,-5.1,2.4,31736.36\nR003,2021-01-21,-4.6,0.2,32400.0\nR003,2021-01-22,1.4,0.0,32400.0\nR003,2021-01-23,2.9,0.0,32400.0\nR003,2021-01-24,4.4,0.0,30823.41\nR003,2021-01-25,9.1,0.0,32400.0\nR003,2021-01-26,7.5,0.3,28800.0\nR003,2021-01-27,9.6,0.0,25200.0\nR003,2021-01-28,7.2,10.7,4473.43\nR003,2021-01-29,1.6,4.9,3781.67\nR003,2021-01-30,1.7,0.0,32443.68\nR003,2021-01-31,5.4,2.5,16154.14\nR003,2021-02-01,7.6,0.0,31609.87\nR003,2021-02-02,10.9,0.0,32374.79\nR003,2021-02-03,11.0,0.0,32400.0\nR003,2021-02-04,11.0,0.0,32453.03\nR003,2021-02-05,5.3,24.4,14198.09\nR003,2021-02-06,2.4,9.8,1947.31\nR003,2021-02-07,5.8,20.8,0.0\nR003,2021-02-08,9.5,1.4,24478.15\nR003,2021-02-09,9.4,7.9,17938.79\nR003,2021-02-10,9.3,0.0,35155.49\nR003,2021-02-11,10.3,0.0,33969.73\nR003,2021-02-12,8.2,4.4,34831.57\nR003,2021-02-13,-1.5,21.2,2432.66\nR003,2021-02-14,0.2,10.9,2993.45\nR003,2021-02-15,2.4,32.3,0.0\nR003,2021-02-16,0.3,22.4,0.0\nR003,2021-02-17,-0.6,8.4,0.0\nR003,2021-02-18,-1.5,3.6,32400.0\nR003,2021-02-19,-1.4,5.5,13654.99\nR003,2021-02-20,-2.0,0.0,37447.76\nR003,2021-02-21,-0.8,0.0,38329.64\nR003,2021-02-22,1.9,6.5,23368.64\nR003,2021-02-23,2.8,3.9,2056.17\nR003,2021-02-24,1.7,0.0,38859.35\nR003,2021-02-25,1.6,0.1,39037.83\nR003,2021-02-26,4.4,0.0,21613.36\nR003,2021-02-27,6.0,0.0,37930.34\nR003,2021-02-28,3.3,11.6,670.7\nR003,2021-03-01,1.7,0.1,37920.38\nR003,2021-03-02,1.7,0.0,39600.0\nR003,2021-03-03,0.4,15.1,12541.09\nR003,2021-03-04,-0.1,0.0,39600.0\nR003,2021-03-05,3.6,0.0,39600.0\nR003,2021-03-06,7.3,0.6,39600.0\nR003,2021-03-07,2.4,9.7,36010.82\nR003,2021-03-08,0.2,0.0,39600.0\nR003,2021-03-09,-0.4,3.5,39123.83\nR003,2021-03-10,0.9,0.2,39600.0\nR003,2021-03-11,2.6,9.8,996.01\nR003,2021-03-12,-0.3,11.2,16423.03\nR003,2021-03-13,2.1,0.0,35632.85\nR003,2021-03-14,5.6,0.0,39600.0\nR003,2021-03-15,7.6,0.0,32633.64\nR003,2021-03-16,9.4,5.4,35059.97\nR003,2021-03-17,7.9,4.6,24237.94\nR003,2021-03-18,4.9,14.0,21772.7\nR003,2021-03-19,3.6,17.1,22165.67\nR003,2021-03-20,2.6,5.5,20300.19\nR003,2021-03-21,2.2,0.8,5023.83\nR003,2021-03-22,5.1,2.5,34844.93\nR003,2021-03-23,7.1,23.4,13792.01\nR003,2021-03-24,5.0,23.4,6006.81\nR003,2021-03-25,2.9,5.3,18642.63\nR003,2021-03-26,1.9,3.6,37327.79\nR003,2021-03-27,2.3,0.0,41619.3\nR003,2021-03-28,3.6,0.8,40456.9\nR003,2021-03-29,4.2,0.0,41126.43\nR003,2021-03-30,5.2,0.1,41995.29\nR003,2021-03-31,7.1,0.0,44124.62\nR003,2021-04-01,8.2,1.7,36340.17\nR003,2021-04-02,7.3,5.1,3087.87\nR003,2021-04-03,7.8,0.9,33152.18\nR003,2021-04-04,10.6,2.1,41138.98\nR003,2021-04-05,10.2,35.6,11382.22\nR003,2021-04-06,8.7,1.3,44026.15\nR003,2021-04-07,16.5,2.1,35503.56\nR003,2021-04-08,16.6,1.2,46172.81\nR003,2021-04-09,9.7,9.9,3447.25\nR003,2021-04-10,7.9,4.3,19801.72\nR003,2021-04-11,10.1,4.8,14404.07\nR003,2021-04-12,9.2,7.2,17433.06\nR003,2021-04-13,9.6,6.0,29452.9\nR003,2021-04-14,9.9,2.2,45279.39\nR003,2021-04-15,12.3,0.0,46647.18\nR003,2021-04-16,13.2,0.2,46742.55\nR003,2021-04-17,11.0,9.4,0.0\nR003,2021-04-18,14.3,9.8,42417.15\nR003,2021-04-19,14.4,14.2,4435.59\nR003,2021-04-20,15.5,16.2,34909.18\nR003,2021-04-21,12.3,9.8,35818.4\nR003,2021-04-22,12.1,2.0,43938.74\nR003,2021-04-23,12.6,0.0,45521.6\nR003,2021-04-24,11.7,3.2,21762.07\nR003,2021-04-25,13.2,1.9,30198.27\nR003,2021-04-26,12.5,25.2,4514.31\nR003,2021-04-27,11.0,2.6,12128.9\nR003,2021-04-28,14.9,0.3,40415.51\nR003,2021-04-29,15.6,2.3,38959.08\nR003,2021-04-30,16.7,0.7,47871.27\nR003,2021-05-01,15.5,1.6,46826.92\nR003,2021-05-02,17.0,0.1,48469.11\nR003,2021-05-03,19.8,0.0,46371.53\nR003,2021-05-04,15.4,9.7,15305.72\nR003,2021-05-05,13.7,0.0,44686.78\nR003,2021-05-06,15.4,0.0,47327.17\nR003,2021-05-07,15.7,0.0,48984.58\nR003,2021-05-08,17.7,0.1,48955.96\nR003,2021-05-09,10.6,33.3,2350.99\nR003,2021-05-10,6.1,4.0,47873.98\nR003,2021-05-11,6.7,2.7,27227.97\nR003,2021-05-12,12.8,0.0,50268.61\nR003,2021-05-13,19.2,0.0,50335.65\nR003,2021-05-14,23.1,0.3,50400.0\nR003,2021-05-15,22.2,1.0,50400.0\nR003,2021-05-16,23.7,0.9,50400.0\nR003,2021-05-17,19.0,9.7,43770.07\nR003,2021-05-18,18.2,0.1,49902.72\nR003,2021-05-19,15.4,6.1,39780.68\nR003,2021-05-20,19.8,0.0,50684.69\nR003,2021-05-21,24.6,0.0,50400.0\nR003,2021-05-22,21.2,9.5,33000.65\nR003,2021-05-23,18.3,20.2,2480.7\nR003,2021-05-24,18.2,5.8,11288.22\nR003,2021-05-25,16.1,7.9,39344.04\nR003,2021-05-26,16.4,0.4,49644.98\nR003,2021-05-27,17.7,0.9,45696.26\nR003,2021-05-28,19.9,1.3,51202.25\nR003,2021-05-29,19.1,9.4,48745.07\nR003,2021-05-30,16.0,6.5,37793.37\nR003,2021-05-31,19.5,0.0,51500.02\nR003,2021-06-01,20.5,12.8,43262.05\nR003,2021-06-02,16.3,24.7,22885.61\nR003,2021-06-03,13.7,4.9,47540.33\nR003,2021-06-04,12.7,9.9,12356.77\nR003,2021-06-05,15.0,7.4,35900.74\nR003,2021-06-06,14.0,3.4,49779.63\nR003,2021-06-07,13.9,2.3,48549.19\nR003,2021-06-08,16.4,0.0,52814.32\nR003,2021-06-09,16.1,0.8,45104.58\nR003,2021-06-10,17.6,0.4,52053.15\nR003,2021-06-11,18.3,0.0,52039.28\nR003,2021-06-12,18.9,2.8,52232.68\nR003,2021-06-13,18.6,6.4,49094.31\nR003,2021-06-14,20.9,0.1,52736.88\nR003,2021-06-15,21.6,4.5,50416.63\nR003,2021-06-16,22.2,1.0,53452.26\nR003,2021-06-17,19.7,13.1,39352.62\nR003,2021-06-18,21.3,0.0,51778.87\nR003,2021-06-19,24.6,0.1,53906.88\nR003,2021-06-20,25.2,0.9,52020.22\nR003,2021-06-21,26.0,2.3,53933.6\nR003,2021-06-22,24.7,8.3,52953.2\nR003,2021-06-23,24.8,2.6,52267.66\nR003,2021-06-24,24.3,8.8,47602.28\nR003,2021-06-25,24.0,3.8,51199.3\nR003,2021-06-26,22.7,8.5,49966.8\nR003,2021-06-27,23.1,9.1,45297.46\nR003,2021-06-28,22.9,6.3,50521.43\nR003,2021-06-29,21.8,9.8,49217.23\nR003,2021-06-30,22.6,0.0,51320.33\nR003,2021-07-01,23.8,0.0,53947.11\nR003,2021-07-02,25.0,0.7,53936.47\nR003,2021-07-03,25.6,0.1,53923.49\nR003,2021-07-04,22.5,9.9,39133.19\nR003,2021-07-05,18.7,30.3,9932.89\nR003,2021-07-06,21.1,1.9,45313.04\nR003,2021-07-07,21.3,1.5,50067.02\nR003,2021-07-08,22.3,1.1,51503.86\nR003,2021-07-09,23.2,0.3,52419.12\nR003,2021-07-10,22.2,0.0,53766.1\nR003,2021-07-11,23.0,0.3,53734.56\nR003,2021-07-12,23.7,2.9,53700.89\nR003,2021-07-13,22.7,8.7,52539.43\nR003,2021-07-14,22.6,2.5,51155.55\nR003,2021-07-15,23.3,0.0,53587.42\nR003,2021-07-16,24.1,0.0,53545.56\nR003,2021-07-17,25.2,0.0,53501.76\nR003,2021-07-18,26.0,0.0,53143.06\nR003,2021-07-19,26.8,0.0,53408.4\nR003,2021-07-20,27.0,0.0,53305.95\nR003,2021-07-21,26.2,2.5,52372.74\nR003,2021-07-22,22.8,4.6,45403.91\nR003,2021-07-23,19.8,15.9,41797.85\nR003,2021-07-24,21.8,5.8,46108.74\nR003,2021-07-25,22.4,3.0,49783.75\nR003,2021-07-26,23.7,3.1,48432.42\nR003,2021-07-27,21.7,18.7,15180.13\nR003,2021-07-28,21.2,0.0,47359.5\nR003,2021-07-29,21.9,1.7,48647.2\nR003,2021-07-30,21.8,0.8,46071.64\nR003,2021-07-31,22.9,3.3,49734.84\nR003,2021-08-01,24.0,0.2,49100.02\nR003,2021-08-02,24.2,2.3,49157.51\nR003,2021-08-03,24.4,0.5,47158.98\nR003,2021-08-04,25.1,1.0,48933.56\nR003,2021-08-05,24.8,0.8,46946.96\nR003,2021-08-06,26.1,0.3,48612.87\nR003,2021-08-07,26.1,2.2,41749.34\nR003,2021-08-08,24.5,3.9,41905.9\nR003,2021-08-09,23.4,13.8,42325.87\nR003,2021-08-10,22.8,11.0,33994.1\nR003,2021-08-11,22.8,9.6,34180.7\nR003,2021-08-12,22.3,15.0,39401.18\nR003,2021-08-13,21.7,7.6,24318.17\nR003,2021-08-14,20.8,0.3,42684.1\nR003,2021-08-15,21.0,1.4,46800.0\nR003,2021-08-16,20.4,9.4,42051.08\nR003,2021-08-17,22.1,2.3,46800.0\nR003,2021-08-18,23.8,1.0,46941.44\nR003,2021-08-19,22.3,10.6,44928.72\nR003,2021-08-20,21.1,5.6,43200.0\nR003,2021-08-21,21.9,1.3,43343.83\nR003,2021-08-22,22.2,3.2,45566.24\nR003,2021-08-23,22.0,3.3,46762.7\nR003,2021-08-24,22.3,1.6,45211.25\nR003,2021-08-25,23.4,0.0,46800.0\nR003,2021-08-26,23.4,1.4,46800.0\nR003,2021-08-27,24.3,0.2,46710.85\nR003,2021-08-28,25.5,0.1,46607.51\nR003,2021-08-29,25.9,0.0,46503.69\nR003,2021-08-30,27.6,0.0,46399.5\nR003,2021-08-31,24.0,10.3,44871.08\nR003,2021-09-01,22.6,4.2,40716.23\nR003,2021-09-02,18.3,22.2,34712.13\nR003,2021-09-03,14.6,9.3,34134.54\nR003,2021-09-04,14.9,0.0,44745.11\nR003,2021-09-05,16.2,0.0,40843.43\nR003,2021-09-06,17.5,1.5,38743.9\nR003,2021-09-07,17.7,0.3,42587.4\nR003,2021-09-08,17.4,0.3,40542.35\nR003,2021-09-09,18.2,0.2,43404.46\nR003,2021-09-10,20.1,0.0,43458.25\nR003,2021-09-11,17.9,24.2,12566.58\nR003,2021-09-12,18.1,3.3,38097.65\nR003,2021-09-13,18.3,0.3,41323.87\nR003,2021-09-14,18.9,0.0,42630.3\nR003,2021-09-15,19.9,0.0,41388.92\nR003,2021-09-16,17.7,43.9,3298.83\nR003,2021-09-17,17.4,28.4,1881.25\nR003,2021-09-18,17.8,0.3,34661.44\nR003,2021-09-19,17.0,3.0,38191.95\nR003,2021-09-20,17.9,0.0,39499.37\nR003,2021-09-21,17.7,2.9,35713.82\nR003,2021-09-22,14.3,6.1,17702.48\nR003,2021-09-23,12.7,5.4,10404.4\nR003,2021-09-24,12.9,1.4,34909.71\nR003,2021-09-25,9.2,36.4,13456.48\nR003,2021-09-26,10.8,0.4,40492.8\nR003,2021-09-27,12.7,3.0,7332.7\nR003,2021-09-28,13.3,1.2,37007.0\nR003,2021-09-29,13.8,0.4,36473.35\nR003,2021-09-30,13.9,8.4,33308.85\nR003,2021-10-01,12.9,17.7,6855.4\nR003,2021-10-02,13.6,0.3,38573.24\nR003,2021-10-03,13.3,3.3,2886.58\nR003,2021-10-04,13.1,8.7,24336.6\nR003,2021-10-05,12.1,7.1,27142.98\nR003,2021-10-06,11.2,4.0,24447.36\nR003,2021-10-07,12.4,1.4,35729.39\nR003,2021-10-08,12.4,0.4,35998.67\nR003,2021-10-09,14.3,0.0,34818.32\nR003,2021-10-10,14.6,0.0,37094.5\nR003,2021-10-11,15.2,0.0,36824.54\nR003,2021-10-12,16.5,0.0,36878.76\nR003,2021-10-13,16.7,3.3,36280.12\nR003,2021-10-14,15.7,0.1,36200.8\nR003,2021-10-15,16.4,0.0,36056.33\nR003,2021-10-16,17.1,0.4,36107.17\nR003,2021-10-17,15.6,2.3,35873.54\nR003,2021-10-18,13.7,1.9,33319.81\nR003,2021-10-19,10.2,3.4,26723.82\nR003,2021-10-20,7.7,0.0,36000.0\nR003,2021-10-21,10.1,0.0,36000.0\nR003,2021-10-22,13.9,0.0,36000.0\nR003,2021-10-23,13.4,0.0,36000.0\nR003,2021-10-24,11.8,6.3,32788.86\nR003,2021-10-25,5.2,13.3,35824.85\nR003,2021-10-26,3.7,0.0,36000.0\nR003,2021-10-27,8.1,0.0,36000.0\nR003,2021-10-28,8.3,0.0,36000.0\nR003,2021-10-29,8.1,0.0,34253.83\nR003,2021-10-30,8.2,0.0,36000.0\nR003,2021-10-31,12.1,0.0,36000.0\nR003,2021-11-01,15.0,0.1,36000.0\nR003,2021-11-02,15.3,2.0,20961.84\nR003,2021-11-03,14.1,0.3,5656.91\nR003,2021-11-04,13.3,0.0,34416.8\nR003,2021-11-05,13.7,0.0,35663.97\nR003,2021-11-06,14.1,0.0,35511.32\nR003,2021-11-07,11.6,0.0,33917.11\nR003,2021-11-08,13.9,0.0,27898.06\nR003,2021-11-09,14.6,0.0,33158.85\nR003,2021-11-10,7.0,21.5,21648.0\nR003,2021-11-11,3.3,0.0,32334.73\nR003,2021-11-12,6.0,0.0,34617.83\nR003,2021-11-13,9.1,0.0,34474.28\nR003,2021-11-14,10.0,0.0,32065.59\nR003,2021-11-15,9.7,0.0,30622.73\nR003,2021-11-16,7.0,0.0,33703.43\nR003,2021-11-17,8.1,0.0,33332.42\nR003,2021-11-18,7.8,0.0,32596.67\nR003,2021-11-19,6.5,4.0,29145.56\nR003,2021-11-20,6.3,10.5,25674.57\nR003,2021-11-21,7.2,11.6,20353.57\nR003,2021-11-22,7.7,0.0,32212.43\nR003,2021-11-23,8.8,0.3,30693.61\nR003,2021-11-24,5.5,16.5,19940.78\nR003,2021-11-25,1.7,0.0,31194.0\nR003,2021-11-26,4.3,0.0,31261.04\nR003,2021-11-27,7.1,5.4,2214.35\nR003,2021-11-28,10.7,11.4,7035.53\nR003,2021-11-29,12.7,1.6,25925.03\nR003,2021-11-30,13.1,34.3,3353.68\nR003,2021-12-01,4.2,15.9,22421.67\nR003,2021-12-02,2.0,8.8,29241.48\nR003,2021-12-03,3.7,0.0,30738.75\nR003,2021-12-04,7.3,0.0,29987.94\nR003,2021-12-05,8.7,2.3,12647.05\nR003,2021-12-06,10.1,0.0,27126.53\nR003,2021-12-07,10.7,0.0,30401.58\nR003,2021-12-08,10.8,0.3,26923.17\nR003,2021-12-09,9.1,3.2,2366.23\nR003,2021-12-10,6.9,0.0,30419.74\nR003,2021-12-11,7.9,0.0,30035.43\nR003,2021-12-12,9.1,0.0,16840.54\nR003,2021-12-13,10.4,0.0,30424.4\nR003,2021-12-14,10.5,0.0,10384.96\nR003,2021-12-15,10.7,0.0,30419.45\nR003,2021-12-16,9.8,0.0,29634.78\nR003,2021-12-17,5.8,12.4,1199.75\nR003,2021-12-18,4.4,14.0,16431.3\nR003,2021-12-19,1.2,0.0,30779.13\nR003,2021-12-20,3.8,19.2,226.51\nR003,2021-12-21,0.6,13.4,5771.96\nR003,2021-12-22,-6.0,4.7,22331.53\nR003,2021-12-23,-7.4,1.7,29157.76\nR003,2021-12-24,-7.2,0.0,28800.0\nR003,2021-12-25,0.4,6.7,0.0\nR003,2021-12-26,5.4,0.7,26665.02\nR003,2021-12-27,7.2,4.8,754.78\nR003,2021-12-28,7.5,0.3,28996.53\nR003,2021-12-29,5.3,0.0,31464.19\nR003,2021-12-30,3.8,0.0,28883.3\nR003,2021-12-31,2.7,3.7,25200.0\nR003,2022-01-01,5.1,11.9,5003.13\nR003,2022-01-02,2.8,23.8,17887.24\nR003,2022-01-03,-0.5,1.6,2033.9\nR003,2022-01-04,3.5,27.0,7657.93\nR003,2022-01-05,4.8,1.7,11065.54\nR003,2022-01-06,9.6,0.0,31892.58\nR003,2022-01-07,9.6,0.0,26122.77\nR003,2022-01-08,6.7,0.0,32019.91\nR003,2022-01-09,5.9,0.0,32086.38\nR003,2022-01-10,6.5,0.8,23034.47\nR003,2022-01-11,8.5,3.7,32224.22\nR003,2022-01-12,6.5,26.9,26.32\nR003,2022-01-13,0.9,14.3,19887.02\nR003,2022-01-14,-2.5,5.5,26674.2\nR003,2022-01-15,-1.5,25.6,19941.76\nR003,2022-01-16,-3.3,0.3,32400.0\nR003,2022-01-17,-4.5,0.0,32400.0\nR003,2022-01-18,-0.2,20.8,0.0\nR003,2022-01-19,-3.0,2.7,25200.0\nR003,2022-01-20,-4.3,3.9,27884.08\nR003,2022-01-21,0.1,3.3,24966.86\nR003,2022-01-22,3.1,21.6,0.0\nR003,2022-01-23,4.5,12.5,8977.94\nR003,2022-01-24,2.9,10.6,14506.49\nR003,2022-01-25,-0.8,3.2,30726.2\nR003,2022-01-26,-0.7,5.6,400.09\nR003,2022-01-27,-1.5,2.9,32400.0\nR003,2022-01-28,-1.7,1.8,32400.0\nR003,2022-01-29,-0.5,3.8,23149.85\nR003,2022-01-30,0.5,12.0,0.0\nR003,2022-01-31,1.6,12.6,0.0\nR003,2022-02-01,2.2,1.8,32400.0\nR003,2022-02-02,6.5,0.0,31315.89\nR003,2022-02-03,7.6,0.0,32400.0\nR003,2022-02-04,4.5,12.9,11917.76\nR003,2022-02-05,-2.4,5.7,33233.86\nR003,2022-02-06,-1.8,13.5,1174.74\nR003,2022-02-07,-0.7,0.0,25755.94\nR003,2022-02-08,4.0,4.8,5706.28\nR003,2022-02-09,5.2,37.9,0.0\nR003,2022-02-10,3.4,3.1,34373.52\nR003,2022-02-11,2.6,0.0,34745.99\nR003,2022-02-12,4.6,0.0,36183.16\nR003,2022-02-13,3.3,2.1,33887.56\nR003,2022-02-14,3.5,0.0,33320.67\nR003,2022-02-15,3.6,0.0,36621.52\nR003,2022-02-16,2.1,0.0,37406.06\nR003,2022-02-17,3.2,0.0,37593.88\nR003,2022-02-18,6.6,0.0,34348.36\nR003,2022-02-19,4.9,5.3,5944.77\nR003,2022-02-20,5.6,0.1,38105.09\nR003,2022-02-21,5.5,6.6,12147.34\nR003,2022-02-22,7.3,0.0,38468.87\nR003,2022-02-23,11.4,0.3,33387.49\nR003,2022-02-24,10.3,0.0,28989.34\nR003,2022-02-25,8.9,4.8,342.06\nR003,2022-02-26,6.9,6.2,25621.65\nR003,2022-02-27,5.4,2.1,37279.64\nR003,2022-02-28,5.7,1.4,16724.85\nR003,2022-03-01,6.0,14.9,8749.66\nR003,2022-03-02,6.3,2.7,30482.87\nR003,2022-03-03,6.7,14.2,17627.53\nR003,2022-03-04,2.6,9.3,27205.67\nR003,2022-03-05,1.9,14.7,20711.09\nR003,2022-03-06,0.5,2.1,39600.0\nR003,2022-03-07,1.7,0.0,39600.0\nR003,2022-03-08,2.0,7.9,9225.65\nR003,2022-03-09,0.6,5.4,2045.31\nR003,2022-03-10,-0.0,0.8,32400.0\nR003,2022-03-11,0.3,4.5,9735.03\nR003,2022-03-12,-0.8,4.4,39600.0\nR003,2022-03-13,-1.4,1.1,38460.04\nR003,2022-03-14,-1.1,0.0,39600.0\nR003,2022-03-15,-1.2,0.2,39600.0\nR003,2022-03-16,-1.4,7.3,29065.94\nR003,2022-03-17,-3.3,3.5,34770.39\nR003,2022-03-18,-3.4,2.2,38852.52\nR003,2022-03-19,-0.4,0.6,30965.91\nR003,2022-03-20,-0.6,0.0,40047.0\nR003,2022-03-21,0.8,0.0,41545.03\nR003,2022-03-22,2.3,0.0,41772.53\nR003,2022-03-23,3.8,0.0,42258.98\nR003,2022-03-24,3.7,5.6,4394.05\nR003,2022-03-25,3.0,3.4,34075.74\nR003,2022-03-26,3.2,4.0,34152.6\nR003,2022-03-27,2.8,13.7,24541.95\nR003,2022-03-28,0.9,3.0,43147.78\nR003,2022-03-29,7.0,0.0,43641.44\nR003,2022-03-30,10.0,1.0,15268.39\nR003,2022-03-31,10.4,0.0,44425.79\nR003,2022-04-01,14.4,0.6,34576.2\nR003,2022-04-02,16.7,0.3,38313.95\nR003,2022-04-03,13.0,5.1,40535.49\nR003,2022-04-04,6.7,9.3,29841.65\nR003,2022-04-05,5.7,8.0,11419.28\nR003,2022-04-06,8.2,1.7,36399.49\nR003,2022-04-07,8.8,0.0,46083.95\nR003,2022-04-08,8.9,8.8,14931.38\nR003,2022-04-09,10.2,3.0,44072.67\nR003,2022-04-10,12.8,0.0,38061.51\nR003,2022-04-11,15.0,0.3,46369.01\nR003,2022-04-12,7.1,33.3,0.0\nR003,2022-04-13,3.6,9.3,17283.83\nR003,2022-04-14,3.3,9.9,9847.76\nR003,2022-04-15,6.5,3.3,19735.97\nR003,2022-04-16,8.2,0.0,46724.35\nR003,2022-04-17,11.9,0.0,46795.71\nR003,2022-04-18,18.3,1.2,38703.98\nR003,2022-04-19,15.9,7.5,28428.85\nR003,2022-04-20,9.8,21.7,28800.0\nR003,2022-04-21,9.4,3.3,46800.0\nR003,2022-04-22,10.5,0.3,46800.0\nR003,2022-04-23,13.3,0.0,47172.42\nR003,2022-04-24,17.3,0.4,47458.43\nR003,2022-04-25,18.4,0.0,47415.24\nR003,2022-04-26,19.6,0.0,47704.49\nR003,2022-04-27,17.9,0.9,47663.34\nR003,2022-04-28,17.0,1.3,44830.24\nR003,2022-04-29,15.4,2.0,44454.74\nR003,2022-04-30,14.8,1.3,46689.63\nR003,2022-05-01,13.9,3.8,15363.2\nR003,2022-05-02,11.0,13.1,0.0\nR003,2022-05-03,12.3,0.4,44834.3\nR003,2022-05-04,13.1,1.4,37743.62\nR003,2022-05-05,10.5,20.9,0.0\nR003,2022-05-06,11.4,1.9,20582.83\nR003,2022-05-07,10.0,6.9,0.0\nR003,2022-05-08,11.1,5.8,12993.38\nR003,2022-05-09,11.8,4.1,40700.04\nR003,2022-05-10,10.1,4.6,34074.17\nR003,2022-05-11,9.1,3.6,41895.54\nR003,2022-05-12,9.5,0.2,48512.21\nR003,2022-05-13,13.3,0.0,50319.83\nR003,2022-05-14,17.0,0.0,50386.69\nR003,2022-05-15,17.3,0.0,50481.26\nR003,2022-05-16,13.6,1.1,47537.0\nR003,2022-05-17,12.6,0.5,50400.0\nR003,2022-05-18,14.6,0.0,49897.25\nR003,2022-05-19,10.7,20.1,29146.59\nR003,2022-05-20,8.4,6.8,32020.04\nR003,2022-05-21,12.4,0.0,50131.64\nR003,2022-05-22,16.8,0.0,50993.88\nR003,2022-05-23,13.0,1.7,33720.3\nR003,2022-05-24,10.6,3.9,41764.23\nR003,2022-05-25,11.8,4.9,43279.79\nR003,2022-05-26,13.6,0.4,49052.55\nR003,2022-05-27,15.6,0.0,47170.14\nR003,2022-05-28,18.4,0.0,51567.6\nR003,2022-05-29,20.8,0.1,51545.58\nR003,2022-05-30,22.1,0.5,51885.08\nR003,2022-05-31,23.2,1.8,50989.42\nR003,2022-06-01,22.1,2.0,51119.7\nR003,2022-06-02,22.5,0.4,52187.47\nR003,2022-06-03,22.7,0.6,50400.0\nR003,2022-06-04,24.9,0.9,51786.9\nR003,2022-06-05,23.3,4.0,45183.74\nR003,2022-06-06,21.2,3.4,48896.3\nR003,2022-06-07,20.4,1.5,49809.47\nR003,2022-06-08,20.6,5.5,50794.36\nR003,2022-06-09,18.8,12.4,49268.82\nR003,2022-06-10,19.5,6.3,44879.23\nR003,2022-06-11,20.6,1.6,53142.69\nR003,2022-06-12,21.6,1.8,52030.54\nR003,2022-06-13,19.8,3.4,49188.73\nR003,2022-06-14,18.8,11.6,32612.49\nR003,2022-06-15,18.7,10.8,47914.55\nR003,2022-06-16,19.2,5.0,49315.49\nR003,2022-06-17,19.7,1.1,53199.48\nR003,2022-06-18,20.3,0.2,53069.71\nR003,2022-06-19,20.9,1.3,53061.11\nR003,2022-06-20,22.2,0.5,53919.8\nR003,2022-06-21,21.9,1.7,52310.24\nR003,2022-06-22,19.5,13.7,51017.51\nR003,2022-06-23,15.6,8.9,42707.03\nR003,2022-06-24,15.0,4.5,13042.77\nR003,2022-06-25,17.8,4.8,45920.06\nR003,2022-06-26,17.5,25.2,0.0\nR003,2022-06-27,19.6,14.5,42182.44\nR003,2022-06-28,21.0,4.0,49778.21\nR003,2022-06-29,21.7,2.4,47435.51\nR003,2022-06-30,22.5,2.5,52441.15\nR003,2022-07-01,21.6,2.2,52477.92\nR003,2022-07-02,20.5,0.0,53597.33\nR003,2022-07-03,18.6,0.2,53927.94\nR003,2022-07-04,19.3,0.4,53912.18\nR003,2022-07-05,21.2,0.0,53894.25\nR003,2022-07-06,22.9,0.1,53874.17\nR003,2022-07-07,23.5,0.9,52304.58\nR003,2022-07-08,22.1,3.2,52313.47\nR003,2022-07-09,22.5,4.0,53801.15\nR003,2022-07-10,23.6,4.4,52688.83\nR003,2022-07-11,21.3,18.1,30508.51\nR003,2022-07-12,21.2,4.9,38008.86\nR003,2022-07-13,21.9,6.7,34837.51\nR003,2022-07-14,21.4,1.3,45965.85\nR003,2022-07-15,22.5,0.0,52639.84\nR003,2022-07-16,22.7,4.9,41671.31\nR003,2022-07-17,23.3,2.5,44261.15\nR003,2022-07-18,21.9,6.3,27246.58\nR003,2022-07-19,21.0,11.5,10373.08\nR003,2022-07-20,21.3,10.4,25331.69\nR003,2022-07-21,20.2,12.2,42886.95\nR003,2022-07-22,17.9,2.3,51752.99\nR003,2022-07-23,17.6,6.9,46159.78\nR003,2022-07-24,17.5,13.0,48193.26\nR003,2022-07-25,18.9,1.7,47603.89\nR003,2022-07-26,20.2,5.5,45998.25\nR003,2022-07-27,20.3,1.5,47257.05\nR003,2022-07-28,21.2,0.5,48558.73\nR003,2022-07-29,22.7,0.2,50659.42\nR003,2022-07-30,24.1,0.6,49671.04\nR003,2022-07-31,24.5,0.6,50415.69\nR003,2022-08-01,24.1,3.7,49779.09\nR003,2022-08-02,23.0,3.9,46791.71\nR003,2022-08-03,23.3,4.8,46556.31\nR003,2022-08-04,23.5,1.3,48921.99\nR003,2022-08-05,22.7,1.3,30835.31\nR003,2022-08-06,22.1,1.0,46894.21\nR003,2022-08-07,21.1,1.4,47954.97\nR003,2022-08-08,22.3,0.6,47304.3\nR003,2022-08-09,22.8,0.0,48347.84\nR003,2022-08-10,23.8,1.3,48036.97\nR003,2022-08-11,23.2,10.7,45959.63\nR003,2022-08-12,22.6,2.7,46800.0\nR003,2022-08-13,23.9,1.1,46800.0\nR003,2022-08-14,23.9,5.1,47116.38\nR003,2022-08-15,25.2,2.0,46800.0\nR003,2022-08-16,24.9,5.2,44427.03\nR003,2022-08-17,24.4,2.6,46412.81\nR003,2022-08-18,24.4,9.1,43286.41\nR003,2022-08-19,23.3,8.0,46069.97\nR003,2022-08-20,23.3,3.6,44069.67\nR003,2022-08-21,24.9,1.5,46718.75\nR003,2022-08-22,26.1,0.3,46800.0\nR003,2022-08-23,25.8,2.0,46800.0\nR003,2022-08-24,26.1,0.2,46800.0\nR003,2022-08-25,27.4,0.4,46800.0\nR003,2022-08-26,26.7,0.2,46800.0\nR003,2022-08-27,26.1,0.8,46734.77\nR003,2022-08-28,25.1,0.9,46633.4\nR003,2022-08-29,22.9,4.5,45407.94\nR003,2022-08-30,23.7,0.2,46430.2\nR003,2022-08-31,25.1,0.1,46328.24\nR003,2022-09-01,26.2,0.1,45224.8\nR003,2022-09-02,26.2,0.7,43988.04\nR003,2022-09-03,26.9,0.0,45642.3\nR003,2022-09-04,22.5,5.1,38568.52\nR003,2022-09-05,18.9,3.2,38241.63\nR003,2022-09-06,16.6,1.0,41104.15\nR003,2022-09-07,14.4,0.7,41849.37\nR003,2022-09-08,15.0,0.2,43659.03\nR003,2022-09-09,15.3,0.0,43712.68\nR003,2022-09-10,17.3,0.0,43447.88\nR003,2022-09-11,19.9,0.0,43200.0\nR003,2022-09-12,21.1,0.5,43200.0\nR003,2022-09-13,17.8,12.5,26503.09\nR003,2022-09-14,15.4,0.8,38081.1\nR003,2022-09-15,16.3,0.4,29112.28\nR003,2022-09-16,19.8,2.2,42583.69\nR003,2022-09-17,23.5,0.0,42084.37\nR003,2022-09-18,26.2,0.1,42839.8\nR003,2022-09-19,20.7,18.8,35229.83\nR003,2022-09-20,19.6,13.2,31111.74\nR003,2022-09-21,17.7,39.5,1681.95\nR003,2022-09-22,16.8,61.4,1002.53\nR003,2022-09-23,14.4,36.7,21600.0\nR003,2022-09-24,11.3,11.3,432.67\nR003,2022-09-25,11.4,3.5,39811.92\nR003,2022-09-26,13.9,0.0,42079.0\nR003,2022-09-27,18.3,0.0,34240.42\nR003,2022-09-28,20.3,0.4,33515.76\nR003,2022-09-29,18.1,7.6,36771.53\nR003,2022-09-30,18.4,0.0,40806.29\nR003,2022-10-01,18.8,0.0,40413.23\nR003,2022-10-02,19.5,0.5,33197.76\nR003,2022-10-03,16.5,20.0,36915.43\nR003,2022-10-04,16.0,13.0,0.0\nR003,2022-10-05,15.4,38.1,4013.59\nR003,2022-10-06,15.9,1.3,32095.29\nR003,2022-10-07,16.6,0.0,37108.9\nR003,2022-10-08,19.8,0.2,30045.05\nR003,2022-10-09,17.1,7.3,3295.26\nR003,2022-10-10,14.8,10.7,22348.49\nR003,2022-10-11,11.3,1.7,35235.13\nR003,2022-10-12,10.7,0.0,36535.17\nR003,2022-10-13,11.6,0.0,36000.0\nR003,2022-10-14,11.5,0.2,36000.0\nR003,2022-10-15,12.9,0.2,35038.46\nR003,2022-10-16,14.0,0.3,31940.16\nR003,2022-10-17,14.2,3.9,1268.56\nR003,2022-10-18,13.9,0.0,36000.0\nR003,2022-10-19,15.7,0.0,36000.0\nR003,2022-10-20,12.6,23.5,19840.45\nR003,2022-10-21,7.7,0.3,36000.0\nR003,2022-10-22,7.9,0.0,36000.0\nR003,2022-10-23,10.5,1.1,31717.76\nR003,2022-10-24,10.7,9.5,24006.04\nR003,2022-10-25,9.6,0.0,34487.92\nR003,2022-10-26,12.1,0.1,27031.01\nR003,2022-10-27,10.9,9.8,31761.39\nR003,2022-10-28,9.1,1.5,35695.13\nR003,2022-10-29,9.0,0.0,36000.0\nR003,2022-10-30,9.7,0.0,34679.21\nR003,2022-10-31,9.4,23.2,0.0\nR003,2022-11-01,7.2,0.2,33682.36\nR003,2022-11-02,6.4,2.9,28991.47\nR003,2022-11-03,9.3,9.4,24749.21\nR003,2022-11-04,9.3,8.9,29559.57\nR003,2022-11-05,8.2,0.0,34547.5\nR003,2022-11-06,6.4,0.0,35546.75\nR003,2022-11-07,8.4,0.0,35391.78\nR003,2022-11-08,8.7,0.0,35238.26\nR003,2022-11-09,8.8,1.5,19545.0\nR003,2022-11-10,8.8,0.3,34143.08\nR003,2022-11-11,8.2,0.3,34125.47\nR003,2022-11-12,7.5,0.0,34642.34\nR003,2022-11-13,8.4,0.0,34498.87\nR003,2022-11-14,7.6,1.2,33976.24\nR003,2022-11-15,6.2,0.0,33711.17\nR003,2022-11-16,9.2,0.8,30524.45\nR003,2022-11-17,9.5,1.6,31876.8\nR003,2022-11-18,10.9,6.6,18201.24\nR003,2022-11-19,12.4,3.6,20630.74\nR003,2022-11-20,14.4,3.9,29567.52\nR003,2022-11-21,14.9,0.0,9756.55\nR003,2022-11-22,14.9,3.4,16426.7\nR003,2022-11-23,11.4,1.4,30224.59\nR003,2022-11-24,12.8,0.0,30169.74\nR003,2022-11-25,13.6,0.3,26033.92\nR003,2022-11-26,11.1,7.7,12022.78\nR003,2022-11-27,10.0,10.7,17689.27\nR003,2022-11-28,7.9,0.0,30265.06\nR003,2022-11-29,8.6,0.0,27623.52\nR003,2022-11-30,10.1,0.0,30737.1\nR003,2022-12-01,10.5,0.0,30412.36\nR003,2022-12-02,9.7,3.3,2571.99\nR003,2022-12-03,9.2,3.5,4099.91\nR003,2022-12-04,8.2,2.5,21651.62\nR003,2022-12-05,7.7,2.2,26258.85\nR003,2022-12-06,5.8,0.1,29241.35\nR003,2022-12-07,5.2,0.0,29633.18\nR003,2022-12-08,6.6,0.0,21174.52\nR003,2022-12-09,8.4,0.0,26937.54\nR003,2022-12-10,9.3,0.0,30034.45\nR003,2022-12-11,9.4,0.0,30036.69\nR003,2022-12-12,10.5,2.7,12611.85\nR003,2022-12-13,10.0,1.5,25659.69\nR003,2022-12-14,8.2,23.4,0.0\nR003,2022-12-15,10.1,2.4,26318.17\nR003,2022-12-16,10.2,4.2,21771.35\nR003,2022-12-17,9.9,0.0,30406.42\nR003,2022-12-18,11.2,0.5,28836.74\nR003,2022-12-19,7.7,18.0,302.75\nR003,2022-12-20,2.4,0.1,30376.85\nR003,2022-12-21,3.1,0.0,25757.57\nR003,2022-12-22,5.4,0.0,31132.41\nR003,2022-12-23,5.6,0.0,31117.14\nR003,2022-12-24,4.0,0.0,31244.4\nR003,2022-12-25,3.7,0.0,30690.98\nR003,2022-12-26,4.7,0.4,21204.77\nR003,2022-12-27,5.2,1.0,19493.08\nR003,2022-12-28,4.1,16.3,21084.06\nR003,2022-12-29,-1.2,0.0,31444.45\nR003,2022-12-30,2.5,0.0,31490.63\nR003,2022-12-31,3.9,0.0,30945.07\nR003,2023-01-01,4.5,0.0,30916.53\nR003,2023-01-02,6.8,0.0,31644.71\nR003,2023-01-03,9.3,0.0,31700.78\nR003,2023-01-04,7.3,1.1,31758.98\nR003,2023-01-05,4.1,0.8,31819.15\nR003,2023-01-06,4.2,28.5,0.0\nR003,2023-01-07,-0.1,8.5,11073.83\nR003,2023-01-08,-4.3,0.2,29515.95\nR003,2023-01-09,-6.0,0.1,32076.86\nR003,2023-01-10,-3.2,0.0,32144.83\nR003,2023-01-11,1.3,0.0,32213.94\nR003,2023-01-12,2.0,0.0,32284.04\nR003,2023-01-13,2.0,0.0,32355.0\nR003,2023-01-14,0.1,0.0,32400.0\nR003,2023-01-15,2.0,0.0,32400.0\nR003,2023-01-16,3.7,0.0,32400.0\nR003,2023-01-17,5.6,0.0,32354.67\nR003,2023-01-18,7.8,0.0,32400.0\nR003,2023-01-19,9.0,0.0,32400.0\nR003,2023-01-20,9.8,0.0,32400.0\nR003,2023-01-21,10.7,0.0,32400.0\nR003,2023-01-22,8.1,0.0,30730.23\nR003,2023-01-23,6.2,0.0,32400.0\nR003,2023-01-24,4.4,0.0,32400.0\nR003,2023-01-25,2.9,0.0,32400.0\nR003,2023-01-26,2.6,0.0,32400.0\nR003,2023-01-27,4.0,0.0,32400.0\nR003,2023-01-28,6.6,1.9,3866.23\nR003,2023-01-29,6.2,3.7,32213.55\nR003,2023-01-30,4.2,5.0,29829.12\nR003,2023-01-31,3.6,0.3,31846.57\nR003,2023-02-01,3.6,4.1,21182.22\nR003,2023-02-02,1.9,7.1,7597.27\nR003,2023-02-03,1.0,11.2,28406.11\nR003,2023-02-04,0.5,7.6,3785.18\nR003,2023-02-05,-0.2,0.7,22481.32\nR003,2023-02-06,-0.6,2.6,15979.44\nR003,2023-02-07,-0.1,10.4,1599.47\nR003,2023-02-08,-1.3,3.9,18189.61\nR003,2023-02-09,-1.6,2.0,33584.8\nR003,2023-02-10,-0.3,2.1,26589.68\nR003,2023-02-11,0.3,2.2,33406.24\nR003,2023-02-12,-0.3,3.1,31804.8\nR003,2023-02-13,-2.8,14.1,34843.65\nR003,2023-02-14,-1.8,5.5,24629.76\nR003,2023-02-15,-1.6,14.7,12048.6\nR003,2023-02-16,-4.4,0.3,35629.77\nR003,2023-02-17,-2.5,6.0,0.0\nR003,2023-02-18,3.3,12.6,636.67\nR003,2023-02-19,4.7,33.7,0.0\nR003,2023-02-20,4.6,36.4,0.0\nR003,2023-02-21,0.6,20.9,9222.01\nR003,2023-02-22,3.2,21.4,23345.85\nR003,2023-02-23,-0.5,1.4,36511.05\nR003,2023-02-24,-1.3,0.0,38783.98\nR003,2023-02-25,6.5,0.0,36895.71\nR003,2023-02-26,9.9,0.0,38571.62\nR003,2023-02-27,11.7,0.0,36251.2\nR003,2023-02-28,12.0,0.3,34572.45\nR003,2023-03-01,8.4,11.0,0.0\nR003,2023-03-02,5.7,0.0,38070.88\nR003,2023-03-03,5.3,3.0,28800.0\nR003,2023-03-04,6.3,15.7,18869.94\nR003,2023-03-05,7.1,1.4,18787.17\nR003,2023-03-06,5.9,32.1,4047.46\nR003,2023-03-07,5.0,0.0,39600.0\nR003,2023-03-08,8.4,0.0,39600.0\nR003,2023-03-09,10.1,0.8,39600.0\nR003,2023-03-10,8.5,7.5,15913.26\nR003,2023-03-11,9.8,3.6,23504.18\nR003,2023-03-12,12.5,8.6,33752.39\nR003,2023-03-13,7.4,26.6,612.86\nR003,2023-03-14,8.9,2.2,12365.11\nR003,2023-03-15,12.0,0.2,39600.0\nR003,2023-03-16,11.5,0.8,34743.56\nR003,2023-03-17,11.0,4.6,31237.36\nR003,2023-03-18,10.6,6.0,10380.62\nR003,2023-03-19,10.9,0.0,39600.0\nR003,2023-03-20,9.7,5.5,19850.44\nR003,2023-03-21,9.1,3.3,8053.24\nR003,2023-03-22,10.8,0.0,38894.45\nR003,2023-03-23,11.0,2.4,24986.17\nR003,2023-03-24,10.1,5.4,35154.49\nR003,2023-03-25,10.7,0.6,33637.87\nR003,2023-03-26,11.9,13.2,35637.51\nR003,2023-03-27,8.7,4.3,25917.92\nR003,2023-03-28,11.7,4.7,42403.55\nR003,2023-03-29,7.0,32.5,220.96\nR003,2023-03-30,1.7,8.5,39164.32\nR003,2023-03-31,0.8,0.0,43858.85\nR003,2023-04-01,5.2,0.0,45127.33\nR003,2023-04-02,10.0,0.0,42944.53\nR003,2023-04-03,11.5,0.6,22773.54\nR003,2023-04-04,14.5,0.0,42317.98\nR003,2023-04-05,14.0,3.4,29225.3\nR003,2023-04-06,9.4,7.9,7918.06\nR003,2023-04-07,10.2,4.4,29667.48\nR003,2023-04-08,8.7,8.2,4732.39\nR003,2023-04-09,11.4,0.1,43622.8\nR003,2023-04-10,11.9,2.4,27235.47\nR003,2023-04-11,13.3,12.9,11590.59\nR003,2023-04-12,12.7,11.4,15716.63\nR003,2023-04-13,10.0,20.9,28820.1\nR003,2023-04-14,8.6,13.2,42964.21\nR003,2023-04-15,11.4,0.0,46635.81\nR003,2023-04-16,15.0,0.3,46707.41\nR003,2023-04-17,16.5,2.5,44817.96\nR003,2023-04-18,17.2,1.5,40305.14\nR003,2023-04-19,17.8,1.7,46800.0\nR003,2023-04-20,13.2,19.7,5526.16\nR003,2023-04-21,10.3,16.3,28059.83\nR003,2023-04-22,8.6,17.6,28337.67\nR003,2023-04-23,11.0,0.0,44107.92\nR003,2023-04-24,10.9,12.9,5694.97\nR003,2023-04-25,11.3,4.7,43385.72\nR003,2023-04-26,14.4,0.3,47050.14\nR003,2023-04-27,16.8,2.1,42600.92\nR003,2023-04-28,14.4,11.3,33910.6\nR003,2023-04-29,12.0,13.5,3181.06\nR003,2023-04-30,11.1,9.2,30043.6\nR003,2023-05-01,9.8,3.2,42496.62\nR003,2023-05-02,9.1,7.3,38861.95\nR003,2023-05-03,11.2,1.2,46786.4\nR003,2023-05-04,13.8,0.0,48757.53\nR003,2023-05-05,15.5,0.0,47372.32\nR003,2023-05-06,16.7,0.1,33901.77\nR003,2023-05-07,14.3,11.4,16528.77\nR003,2023-05-08,12.2,9.5,7924.95\nR003,2023-05-09,12.1,3.5,23226.64\nR003,2023-05-10,11.0,11.6,722.9\nR003,2023-05-11,10.8,8.0,30045.29\nR003,2023-05-12,14.0,0.4,40881.17\nR003,2023-05-13,14.3,17.6,1368.11\nR003,2023-05-14,12.1,15.6,10531.02\nR003,2023-05-15,13.2,0.7,48383.89\nR003,2023-05-16,15.1,0.3,50400.0\nR003,2023-05-17,15.8,0.0,50431.02\nR003,2023-05-18,16.6,0.0,50400.0\nR003,2023-05-19,17.5,2.3,48292.39\nR003,2023-05-20,17.7,5.3,44155.95\nR003,2023-05-21,16.4,15.0,28206.03\nR003,2023-05-22,17.5,9.8,47256.82\nR003,2023-05-23,19.7,4.7,50400.0\nR003,2023-05-24,20.2,4.3,50593.18\nR003,2023-05-25,19.9,3.3,48034.32\nR003,2023-05-26,18.1,7.7,50315.92\nR003,2023-05-27,18.6,2.9,49349.71\nR003,2023-05-28,19.9,3.1,44947.7\nR003,2023-05-29,19.9,5.4,34600.05\nR003,2023-05-30,18.7,16.4,28810.35\nR003,2023-05-31,18.1,4.4,48570.64\nR003,2023-06-01,17.7,5.4,45087.41\nR003,2023-06-02,18.5,0.5,51828.12\nR003,2023-06-03,21.0,1.2,51082.81\nR003,2023-06-04,20.6,8.6,39917.93\nR003,2023-06-05,19.3,5.5,41158.73\nR003,2023-06-06,18.5,4.0,30611.3\nR003,2023-06-07,19.4,3.2,46834.46\nR003,2023-06-08,20.5,0.1,51474.59\nR003,2023-06-09,22.0,1.9,39493.43\nR003,2023-06-10,19.8,19.2,23298.26\nR003,2023-06-11,21.0,3.8,47246.72\nR003,2023-06-12,20.6,9.4,37794.64\nR003,2023-06-13,20.4,19.9,43449.76\nR003,2023-06-14,20.1,19.4,29833.73\nR003,2023-06-15,19.8,13.6,42236.06\nR003,2023-06-16,19.9,3.6,46843.71\nR003,2023-06-17,20.8,0.4,50555.93\nR003,2023-06-18,20.9,2.9,53069.97\nR003,2023-06-19,19.6,14.5,47121.59\nR003,2023-06-20,19.6,9.5,38456.79\nR003,2023-06-21,20.4,0.9,49724.68\nR003,2023-06-22,20.3,0.7,46759.62\nR003,2023-06-23,20.6,1.0,53400.23\nR003,2023-06-24,21.2,5.0,51925.03\nR003,2023-06-25,22.8,0.2,53969.76\nR003,2023-06-26,23.3,2.9,53972.21\nR003,2023-06-27,21.3,5.0,46579.64\nR003,2023-06-28,21.6,0.4,53749.01\nR003,2023-06-29,21.5,2.7,53100.44\nR003,2023-06-30,18.2,1.2,52165.43\nR003,2023-07-01,18.6,1.2,53951.18\nR003,2023-07-02,20.3,0.3,50383.87\nR003,2023-07-03,22.6,0.1,53928.15\nR003,2023-07-04,24.1,0.6,53913.48\nR003,2023-07-05,24.6,0.5,53896.71\nR003,2023-07-06,25.8,0.0,53877.81\nR003,2023-07-07,25.1,14.4,47180.99\nR003,2023-07-08,22.6,12.8,35444.05\nR003,2023-07-09,20.0,17.9,3093.8\nR003,2023-07-10,19.5,33.6,8132.72\nR003,2023-07-11,17.2,3.4,48962.62\nR003,2023-07-12,18.1,1.0,50272.58\nR003,2023-07-13,21.3,0.0,53686.19\nR003,2023-07-14,22.5,0.0,52877.69\nR003,2023-07-15,22.2,1.1,49458.51\nR003,2023-07-16,20.2,0.2,53230.54\nR003,2023-07-17,19.6,0.0,51001.09\nR003,2023-07-18,20.1,0.4,52420.71\nR003,2023-07-19,20.4,0.7,52436.97\nR003,2023-07-20,21.8,0.5,50959.96\nR003,2023-07-21,22.8,2.0,51331.65\nR003,2023-07-22,24.0,1.1,48920.52\nR003,2023-07-23,23.8,1.0,50019.21\nR003,2023-07-24,22.9,4.7,46745.0\nR003,2023-07-25,22.4,0.6,49761.45\nR003,2023-07-26,23.5,0.1,51502.37\nR003,2023-07-27,26.8,0.2,51815.18\nR003,2023-07-28,20.8,19.7,29066.59\nR003,2023-07-29,19.6,0.2,48219.53\nR003,2023-07-30,21.4,0.4,47389.43\nR003,2023-07-31,23.0,0.7,50058.78\nR003,2023-08-01,23.5,1.4,48348.67\nR003,2023-08-02,23.9,1.1,46821.87\nR003,2023-08-03,23.0,6.9,47134.48\nR003,2023-08-04,22.5,0.2,47879.28\nR003,2023-08-05,24.7,0.0,48282.71\nR003,2023-08-06,27.9,0.0,48684.65\nR003,2023-08-07,27.8,0.2,48406.69\nR003,2023-08-08,26.8,0.6,47295.47\nR003,2023-08-09,26.3,1.2,47324.17\nR003,2023-08-10,27.1,1.4,47690.74\nR003,2023-08-11,27.5,0.8,47382.19\nR003,2023-08-12,26.1,3.5,42967.48\nR003,2023-08-13,26.7,1.6,46180.28\nR003,2023-08-14,26.6,0.5,46800.0\nR003,2023-08-15,27.0,0.5,46832.6\nR003,2023-08-16,27.1,0.0,47197.06\nR003,2023-08-17,26.8,0.0,47227.74\nR003,2023-08-18,25.5,0.4,46926.88\nR003,2023-08-19,24.4,1.2,46800.0\nR003,2023-08-20,24.8,0.0,46800.0\nR003,2023-08-21,24.6,0.3,46800.0\nR003,2023-08-22,24.2,4.5,46800.0\nR003,2023-08-23,23.4,0.0,46800.0\nR003,2023-08-24,23.0,0.0,46800.0\nR003,2023-08-25,21.3,0.3,46800.0\nR003,2023-08-26,21.4,0.0,46800.0\nR003,2023-08-27,22.3,0.0,46767.13\nR003,2023-08-28,23.4,0.0,46665.97\nR003,2023-08-29,23.5,0.0,46563.54\nR003,2023-08-30,23.8,0.5,46407.65\nR003,2023-08-31,25.7,0.0,46355.35\nR003,2023-09-01,26.0,0.0,45213.47\nR003,2023-09-02,24.3,0.4,44619.74\nR003,2023-09-03,21.3,9.7,6657.78\nR003,2023-09-04,21.6,14.0,27258.42\nR003,2023-09-05,22.9,1.4,43810.87\nR003,2023-09-06,23.6,1.2,43223.45\nR003,2023-09-07,22.0,5.6,34049.55\nR003,2023-09-08,20.7,0.3,40902.15\nR003,2023-09-09,17.6,6.7,35486.52\nR003,2023-09-10,14.8,0.7,42934.92\nR003,2023-09-11,15.0,1.6,41776.86\nR003,2023-09-12,16.1,1.4,41502.59\nR003,2023-09-13,15.6,1.7,40778.99\nR003,2023-09-14,15.6,0.4,43074.79\nR003,2023-09-15,17.3,0.0,43058.67\nR003,2023-09-16,17.4,2.0,40996.1\nR003,2023-09-17,17.4,0.0,38179.04\nR003,2023-09-18,16.6,0.0,42856.64\nR003,2023-09-19,17.5,0.0,42789.0\nR003,2023-09-20,19.0,0.0,42721.23\nR003,2023-09-21,19.8,0.0,42529.71\nR003,2023-09-22,20.7,0.3,42564.5\nR003,2023-09-23,20.7,0.2,41007.49\nR003,2023-09-24,20.8,0.4,41679.67\nR003,2023-09-25,19.4,1.2,41396.68\nR003,2023-09-26,18.7,0.3,39332.36\nR003,2023-09-27,19.8,0.0,41149.74\nR003,2023-09-28,20.6,1.9,40650.0\nR003,2023-09-29,20.4,0.3,39361.78\nR003,2023-09-30,20.1,0.3,40498.85\nR003,2023-10-01,18.0,4.6,36473.49\nR003,2023-10-02,17.0,8.5,4745.87\nR003,2023-10-03,15.9,2.3,37400.94\nR003,2023-10-04,14.6,0.8,35940.04\nR003,2023-10-05,14.9,0.1,38944.11\nR003,2023-10-06,13.9,0.0,37163.13\nR003,2023-10-07,13.8,0.2,37229.05\nR003,2023-10-08,15.4,0.0,37954.49\nR003,2023-10-09,10.0,26.2,29468.08\nR003,2023-10-10,7.0,4.8,9633.14\nR003,2023-10-11,8.3,4.7,32922.16\nR003,2023-10-12,7.9,0.0,37173.7\nR003,2023-10-13,10.7,0.0,36903.41\nR003,2023-10-14,13.2,0.0,36631.93\nR003,2023-10-15,15.5,0.0,36359.13\nR003,2023-10-16,15.7,2.2,36000.0\nR003,2023-10-17,10.5,10.2,35963.93\nR003,2023-10-18,9.2,4.3,11043.25\nR003,2023-10-19,10.7,2.8,28800.0\nR003,2023-10-20,13.0,0.0,36000.0\nR003,2023-10-21,17.2,0.0,36000.0\nR003,2023-10-22,17.8,0.0,36000.0\nR003,2023-10-23,18.5,0.0,33457.27\nR003,2023-10-24,17.1,1.8,31561.64\nR003,2023-10-25,14.4,0.5,28800.0\nR003,2023-10-26,16.5,6.0,10058.27\nR003,2023-10-27,17.0,5.1,14400.6\nR003,2023-10-28,18.5,0.7,28409.9\nR003,2023-10-29,16.9,2.4,31581.71\nR003,2023-10-30,15.3,0.0,36000.0\nR003,2023-10-31,16.7,0.0,36000.0\nR003,2023-11-01,18.5,0.0,32400.0\nR003,2023-11-02,17.0,2.7,28321.13\nR003,2023-11-03,15.0,12.6,15553.88\nR003,2023-11-04,15.0,0.0,35803.93\nR003,2023-11-05,17.4,0.2,35723.61\nR003,2023-11-06,14.8,17.2,15022.61\nR003,2023-11-07,12.1,0.4,34914.67\nR003,2023-11-08,11.7,0.0,33830.36\nR003,2023-11-09,12.1,18.6,11240.38\nR003,2023-11-10,6.9,0.0,34966.13\nR003,2023-11-11,10.4,0.0,34820.06\nR003,2023-11-12,15.7,5.1,33764.15\nR003,2023-11-13,12.7,21.3,2482.24\nR003,2023-11-14,11.5,31.3,13788.26\nR003,2023-11-15,10.0,0.3,19432.37\nR003,2023-11-16,10.3,55.7,0.0\nR003,2023-11-17,5.9,1.6,32608.67\nR003,2023-11-18,8.5,0.0,16772.04\nR003,2023-11-19,11.4,31.5,0.0\nR003,2023-11-20,6.6,15.7,1828.56\nR003,2023-11-21,3.8,4.2,22507.73\nR003,2023-11-22,5.9,6.2,24201.32\nR003,2023-11-23,6.1,32.6,15547.48\nR003,2023-11-24,6.9,18.3,0.0\nR003,2023-11-25,11.0,23.5,11086.13\nR003,2023-11-26,13.0,25.1,27874.36\nR003,2023-11-27,1.8,21.7,11641.47\nR003,2023-11-28,-0.1,7.2,3884.14\nR003,2023-11-29,4.3,3.7,16294.84\nR003,2023-11-30,8.1,20.7,0.0\nR003,2023-12-01,10.1,0.2,24625.98\nR003,2023-12-02,13.2,0.1,26794.11\nR003,2023-12-03,13.0,0.0,27018.45\nR003,2023-12-04,13.1,0.8,29985.78\nR003,2023-12-05,9.9,5.2,15561.41\nR003,2023-12-06,10.0,0.4,19531.55\nR003,2023-12-07,9.1,0.0,29246.63\nR003,2023-12-08,11.6,0.7,25895.46\nR003,2023-12-09,11.6,3.7,0.0\nR003,2023-12-10,9.0,13.7,3592.13\nR003,2023-12-11,9.1,0.8,20537.3\nR003,2023-12-12,8.7,11.8,3026.47\nR003,2023-12-13,6.4,7.7,2128.61\nR003,2023-12-14,5.8,0.0,26567.88\nR003,2023-12-15,10.6,0.0,11009.03\nR003,2023-12-16,10.7,3.0,21925.29\nR003,2023-12-17,4.3,24.8,17889.59\nR003,2023-12-18,2.3,7.3,5896.79\nR003,2023-12-19,4.9,4.3,13718.96\nR003,2023-12-20,5.3,0.0,30771.12\nR003,2023-12-21,8.1,0.5,20368.91\nR003,2023-12-22,9.2,5.7,4589.43\nR003,2023-12-23,4.6,36.7,367.28\nR003,2023-12-24,-0.6,27.7,4022.06\nR003,2023-12-25,1.4,22.2,1789.54\nR003,2023-12-26,7.5,14.7,6181.2\nR003,2023-12-27,8.4,6.4,16967.63\nR003,2023-12-28,1.7,2.0,31386.22\nR003,2023-12-29,1.1,0.0,31432.22\nR003,2023-12-30,7.0,0.0,31480.58\nR003,2023-12-31,6.2,0.3,26393.9\nR003,2024-01-01,5.4,0.0,31583.92\nR003,2024-01-02,7.5,1.0,31279.63\nR003,2024-01-03,5.0,13.6,20738.57\nR003,2024-01-04,5.7,29.5,0.0\nR003,2024-01-05,5.4,19.9,0.0\nR003,2024-01-06,4.6,1.6,31874.55\nR003,2024-01-07,9.0,0.0,28677.0\nR003,2024-01-08,7.5,22.1,1565.73\nR003,2024-01-09,5.0,12.4,2504.43\nR003,2024-01-10,3.0,0.3,25783.99\nR003,2024-01-11,2.7,3.2,30134.49\nR003,2024-01-12,0.8,27.1,635.64\nR003,2024-01-13,-5.9,0.9,27295.67\nR003,2024-01-14,-4.9,0.0,32400.0\nR003,2024-01-15,-0.4,0.0,31846.88\nR003,2024-01-16,4.8,17.5,15366.56\nR003,2024-01-17,4.0,7.3,25842.4\nR003,2024-01-18,4.2,0.0,32400.0\nR003,2024-01-19,7.8,5.3,17962.93\nR003,2024-01-20,7.8,12.5,24001.67\nR003,2024-01-21,5.5,39.1,0.0\nR003,2024-01-22,2.2,13.4,20449.57\nR003,2024-01-23,-0.2,0.6,32400.0\nR003,2024-01-24,-1.4,0.0,32400.0\nR003,2024-01-25,-1.1,0.2,29024.27\nR003,2024-01-26,1.8,0.7,17230.35\nR003,2024-01-27,1.2,0.8,26799.41\nR003,2024-01-28,-0.6,2.6,32400.0\nR003,2024-01-29,-1.0,0.1,32400.0\nR003,2024-01-30,-1.4,0.0,32472.53\nR003,2024-01-31,0.2,0.0,31413.18\nR003,2024-02-01,3.3,0.0,32400.0\nR003,2024-02-02,3.1,4.1,16415.48\nR003,2024-02-03,2.0,13.4,21753.81\nR003,2024-02-04,2.5,17.0,5782.66\nR003,2024-02-05,2.4,68.6,0.0\nR003,2024-02-06,4.0,27.0,5200.43\nR003,2024-02-07,7.3,3.7,28720.7\nR003,2024-02-08,8.0,4.8,22084.37\nR003,2024-02-09,9.1,7.0,28263.02\nR003,2024-02-10,9.7,0.0,32828.95\nR003,2024-02-11,11.3,0.0,19482.15\nR003,2024-02-12,12.4,0.0,32400.0\nR003,2024-02-13,11.8,0.0,25693.12\nR003,2024-02-14,11.4,0.2,29243.03\nR003,2024-02-15,8.1,4.1,12644.43\nR003,2024-02-16,8.1,3.2,28713.09\nR003,2024-02-17,7.1,5.2,28939.5\nR003,2024-02-18,6.2,5.5,30706.69\nR003,2024-02-19,4.1,2.9,29738.32\nR003,2024-02-20,3.4,1.8,34996.61\nR003,2024-02-21,3.1,0.0,37118.39\nR003,2024-02-22,3.8,2.2,23517.45\nR003,2024-02-23,5.1,3.1,36176.1\nR003,2024-02-24,5.0,0.0,38730.03\nR003,2024-02-25,5.1,0.0,38905.6\nR003,2024-02-26,5.7,0.0,39082.43\nR003,2024-02-27,5.9,0.0,39260.39\nR003,2024-02-28,4.7,0.0,39403.81\nR003,2024-02-29,4.6,0.0,39505.24\nR003,2024-03-01,4.7,0.0,39600.0\nR003,2024-03-02,7.2,0.0,37332.31\nR003,2024-03-03,8.1,0.0,27873.11\nR003,2024-03-04,6.7,0.9,3545.25\nR003,2024-03-05,8.1,0.1,37365.82\nR003,2024-03-06,7.8,6.5,6831.47\nR003,2024-03-07,6.2,15.7,2901.54\nR003,2024-03-08,4.5,8.5,24801.72\nR003,2024-03-09,4.3,1.4,39600.0\nR003,2024-03-10,4.1,0.0,39600.0\nR003,2024-03-11,4.2,0.2,38076.95\nR003,2024-03-12,6.3,0.0,39658.81\nR003,2024-03-13,11.8,0.7,21627.82\nR003,2024-03-14,10.8,2.5,19479.49\nR003,2024-03-15,8.1,3.5,24151.37\nR003,2024-03-16,8.0,4.3,24431.17\nR003,2024-03-17,6.8,3.5,38170.75\nR003,2024-03-18,8.1,0.1,39600.0\nR003,2024-03-19,8.8,0.0,36794.23\nR003,2024-03-20,7.6,0.4,38300.72\nR003,2024-03-21,7.6,0.7,36000.0\nR003,2024-03-22,6.6,10.1,0.0\nR003,2024-03-23,5.9,2.7,37370.31\nR003,2024-03-24,5.9,5.0,27192.2\nR003,2024-03-25,6.5,2.0,42442.37\nR003,2024-03-26,4.6,23.6,12842.58\nR003,2024-03-27,4.8,11.9,33656.14\nR003,2024-03-28,6.8,0.0,44571.53\nR003,2024-03-29,10.3,2.2,27061.87\nR003,2024-03-30,13.1,1.5,41649.43\nR003,2024-03-31,14.6,0.0,45068.05\nR003,2024-04-01,16.0,0.0,44863.72\nR003,2024-04-02,17.4,0.0,45573.14\nR003,2024-04-03,15.4,4.3,44048.5\nR003,2024-04-04,11.4,0.2,43407.31\nR003,2024-04-05,10.6,0.4,23679.71\nR003,2024-04-06,10.1,0.7,44658.62\nR003,2024-04-07,11.6,0.0,46118.32\nR003,2024-04-08,14.0,0.0,46189.93\nR003,2024-04-09,14.9,0.5,46261.54\nR003,2024-04-10,14.4,0.2,36368.07\nR003,2024-04-11,13.9,2.1,22302.74\nR003,2024-04-12,16.1,0.7,44788.06\nR003,2024-04-13,14.1,4.0,15602.83\nR003,2024-04-14,12.5,2.7,40114.53\nR003,2024-04-15,13.0,0.2,45788.7\nR003,2024-04-16,17.1,0.0,46761.88\nR003,2024-04-17,20.0,0.0,43200.0\nR003,2024-04-18,19.9,0.1,46800.0\nR003,2024-04-19,16.0,5.1,42302.1\nR003,2024-04-20,17.9,1.8,46800.0\nR003,2024-04-21,14.9,7.0,36657.61\nR003,2024-04-22,14.4,1.2,47192.46\nR003,2024-04-23,16.2,0.0,46204.23\nR003,2024-04-24,17.7,0.0,38630.69\nR003,2024-04-25,22.9,0.2,47063.81\nR003,2024-04-26,23.8,1.6,42371.51\nR003,2024-04-27,21.1,2.8,36434.52\nR003,2024-04-28,21.9,1.9,47272.17\nR003,2024-04-29,21.1,1.0,44690.85\nR003,2024-04-30,18.9,0.4,37647.62\nR003,2024-05-01,18.0,1.4,45231.36\nR003,2024-05-02,17.0,7.3,27118.77\nR003,2024-05-03,14.8,6.6,10172.78\nR003,2024-05-04,12.8,15.1,31621.56\nR003,2024-05-05,9.8,34.8,16138.21\nR003,2024-05-06,10.5,3.8,41028.56\nR003,2024-05-07,11.3,3.0,25637.93\nR003,2024-05-08,13.3,2.1,44309.39\nR003,2024-05-09,14.3,3.5,34620.4\nR003,2024-05-10,14.6,3.2,32942.45\nR003,2024-05-11,15.3,10.2,24554.14\nR003,2024-05-12,13.0,29.0,954.71\nR003,2024-05-13,11.7,3.9,39148.47\nR003,2024-05-14,10.3,11.8,23752.47\nR003,2024-05-15,9.4,3.5,47343.26\nR003,2024-05-16,9.9,3.2,50400.0\nR003,2024-05-17,10.2,3.9,44410.79\nR003,2024-05-18,11.6,0.8,47487.66\nR003,2024-05-19,12.4,6.4,46800.0\nR003,2024-05-20,13.4,3.4,48147.19\nR003,2024-05-21,15.8,0.8,50073.42\nR003,2024-05-22,17.8,1.0,50624.08\nR003,2024-05-23,18.7,2.7,50597.56\nR003,2024-05-24,18.0,2.8,50571.53\nR003,2024-05-25,15.4,8.1,43019.26\nR003,2024-05-26,15.4,11.9,20932.1\nR003,2024-05-27,17.1,3.2,47752.6\nR003,2024-05-28,18.3,8.5,39364.34\nR003,2024-05-29,18.4,10.4,29463.98\nR003,2024-05-30,20.6,2.3,51512.14\nR003,2024-05-31,20.1,8.8,42964.58\nR003,2024-06-01,19.4,2.4,30825.24\nR003,2024-06-02,20.7,0.1,51817.44\nR003,2024-06-03,21.6,0.7,51435.72\nR003,2024-06-04,23.3,0.4,51781.64\nR003,2024-06-05,25.3,0.0,52820.64\nR003,2024-06-06,23.9,0.3,50652.38\nR003,2024-06-07,21.7,2.8,44674.5\nR003,2024-06-08,21.2,0.8,46186.6\nR003,2024-06-09,19.2,6.1,35109.32\nR003,2024-06-10,19.5,4.8,34893.1\nR003,2024-06-11,20.6,7.8,49280.5\nR003,2024-06-12,22.6,0.8,50773.48\nR003,2024-06-13,22.7,0.6,53786.47\nR003,2024-06-14,23.4,0.4,53813.67\nR003,2024-06-15,23.1,3.5,51224.52\nR003,2024-06-16,20.9,10.6,47662.94\nR003,2024-06-17,22.3,1.0,52011.02\nR003,2024-06-18,23.5,0.9,50598.82\nR003,2024-06-19,23.8,1.1,53055.19\nR003,2024-06-20,22.9,3.6,51814.29\nR003,2024-06-21,22.3,0.2,53168.3\nR003,2024-06-22,20.2,0.2,53950.15\nR003,2024-06-23,21.8,0.0,53957.35\nR003,2024-06-24,24.5,0.0,53962.6\nR003,2024-06-25,22.9,2.0,51922.0\nR003,2024-06-26,19.1,2.5,51618.16\nR003,2024-06-27,17.9,0.6,50493.09\nR003,2024-06-28,19.6,0.1,53963.69\nR003,2024-06-29,20.7,0.0,53958.83\nR003,2024-06-30,21.7,0.0,53951.87\nR003,2024-07-01,20.9,1.7,35169.24\nR003,2024-07-02,19.6,14.8,30769.21\nR003,2024-07-03,21.0,10.4,51075.81\nR003,2024-07-04,24.2,0.6,53902.49\nR003,2024-07-05,26.3,1.8,53386.33\nR003,2024-07-06,25.5,0.5,53740.62\nR003,2024-07-07,24.9,1.1,37579.02\nR003,2024-07-08,25.3,0.6,45831.04\nR003,2024-07-09,25.1,6.2,41801.03\nR003,2024-07-10,24.2,6.3,44888.71\nR003,2024-07-11,23.8,2.5,51200.79\nR003,2024-07-12,25.1,4.3,53695.58\nR003,2024-07-13,24.9,0.0,53659.32\nR003,2024-07-14,25.2,0.0,53620.85\nR003,2024-07-15,25.2,0.0,48719.96\nR003,2024-07-16,25.9,0.2,53117.7\nR003,2024-07-17,27.0,0.7,49691.65\nR003,2024-07-18,26.0,0.2,53146.2\nR003,2024-07-19,25.1,0.0,51022.38\nR003,2024-07-20,24.3,0.3,48869.02\nR003,2024-07-21,24.0,1.7,52010.81\nR003,2024-07-22,23.6,3.0,50914.29\nR003,2024-07-23,22.8,2.2,45922.87\nR003,2024-07-24,22.4,2.7,48698.87\nR003,2024-07-25,22.7,1.5,47774.85\nR003,2024-07-26,22.4,3.5,33633.07\nR003,2024-07-27,21.8,10.3,34715.93\nR003,2024-07-28,22.6,0.0,48902.24\nR003,2024-07-29,23.0,0.0,51016.85\nR003,2024-07-30,22.5,1.9,49943.79\nR003,2024-07-31,21.3,0.2,48026.02\nR003,2024-08-01,21.4,0.8,46057.49\nR003,2024-08-02,21.8,0.6,47169.44\nR003,2024-08-03,23.7,0.4,49231.08\nR003,2024-08-04,24.3,0.2,48606.11\nR003,2024-08-05,23.3,3.8,47912.1\nR003,2024-08-06,22.0,2.6,45436.69\nR003,2024-08-07,21.9,1.1,46948.1\nR003,2024-08-08,22.0,0.8,47993.84\nR003,2024-08-09,23.0,1.0,45841.48\nR003,2024-08-10,23.5,0.8,45618.47\nR003,2024-08-11,23.5,0.9,40364.52\nR003,2024-08-12,22.4,2.4,44953.38\nR003,2024-08-13,21.0,0.0,47129.18\nR003,2024-08-14,20.5,4.9,44868.91\nR003,2024-08-15,20.3,0.0,46800.0\nR003,2024-08-16,20.9,0.0,46888.37\nR003,2024-08-17,21.2,0.0,46919.68\nR003,2024-08-18,23.2,0.0,46950.99\nR003,2024-08-19,25.2,0.0,46982.25\nR003,2024-08-20,25.3,0.6,46800.0\nR003,2024-08-21,25.9,0.3,46800.0\nR003,2024-08-22,26.0,1.0,43200.0\nR003,2024-08-23,25.1,3.0,46800.0\nR003,2024-08-24,23.1,7.7,22045.37\nR003,2024-08-25,23.5,19.3,38377.47\nR003,2024-08-26,20.5,17.0,3354.29\nR003,2024-08-27,23.0,1.0,46308.93\nR003,2024-08-28,24.3,0.0,46584.02\nR003,2024-08-29,24.0,1.1,46479.51\nR003,2024-08-30,22.8,5.0,35262.46\nR003,2024-08-31,21.7,4.8,44982.5\nR003,2024-09-01,23.4,1.3,44932.28\nR003,2024-09-02,24.0,4.9,43200.0\nR003,2024-09-03,23.3,5.1,44390.19\nR003,2024-09-04,22.3,3.4,43568.97\nR003,2024-09-05,21.8,2.8,42247.3\nR003,2024-09-06,20.6,3.2,35462.76\nR003,2024-09-07,20.9,1.5,41289.15\nR003,2024-09-08,21.6,3.9,41967.04\nR003,2024-09-09,21.5,6.3,38297.96\nR003,2024-09-10,23.5,2.7,40983.24\nR003,2024-09-11,25.2,0.0,40561.09\nR003,2024-09-12,24.6,3.9,43200.0\nR003,2024-09-13,21.8,3.2,41702.78\nR003,2024-09-14,21.6,0.7,42958.18\nR003,2024-09-15,21.7,0.0,42990.42\nR003,2024-09-16,20.1,10.4,32142.36\nR003,2024-09-17,18.9,2.5,36042.26\nR003,2024-09-18,17.8,4.6,37086.52\nR003,2024-09-19,17.5,6.8,34104.36\nR003,2024-09-20,16.7,9.3,10951.83\nR003,2024-09-21,17.1,2.9,21940.42\nR003,2024-09-22,17.7,0.7,31802.05\nR003,2024-09-23,16.6,15.7,14413.15\nR003,2024-09-24,17.0,7.0,33200.08\nR003,2024-09-25,17.2,15.7,25903.12\nR003,2024-09-26,17.3,1.1,36082.45\nR003,2024-09-27,18.3,0.1,40219.61\nR003,2024-09-28,19.8,0.0,40894.1\nR003,2024-09-29,20.4,0.0,40842.42\nR003,2024-09-30,20.0,0.2,40452.28\nR003,2024-10-01,16.2,21.2,15555.79\nR003,2024-10-02,14.7,19.0,1870.11\nR003,2024-10-03,15.5,1.9,29681.13\nR003,2024-10-04,15.3,0.0,36998.72\nR003,2024-10-05,16.6,0.0,39355.57\nR003,2024-10-06,19.6,0.0,39066.29\nR003,2024-10-07,20.9,0.0,37285.68\nR003,2024-10-08,16.5,5.8,29222.16\nR003,2024-10-09,16.3,0.0,37475.6\nR003,2024-10-10,17.6,0.6,32400.0\nR003,2024-10-11,17.0,0.6,36017.61\nR003,2024-10-12,16.7,0.0,36242.31\nR003,2024-10-13,15.4,6.1,7224.36\nR003,2024-10-14,14.4,0.6,30089.72\nR003,2024-10-15,13.8,0.0,33139.58\nR003,2024-10-16,12.9,1.5,32497.18\nR003,2024-10-17,11.4,7.8,19507.23\nR003,2024-10-18,8.7,6.8,11297.61\nR003,2024-10-19,7.2,7.7,21789.29\nR003,2024-10-20,6.5,14.4,0.0\nR003,2024-10-21,5.4,2.0,24941.27\nR003,2024-10-22,6.9,0.0,36000.0\nR003,2024-10-23,9.8,2.3,36000.0\nR003,2024-10-24,10.8,2.9,33906.65\nR003,2024-10-25,10.6,4.0,19601.06\nR003,2024-10-26,7.8,10.4,34973.52\nR003,2024-10-27,7.1,0.0,36000.0\nR003,2024-10-28,8.3,0.0,36000.0\nR003,2024-10-29,9.6,0.0,36000.0\nR003,2024-10-30,10.7,0.0,34776.45\nR003,2024-10-31,9.1,0.0,36000.0\nR003,2024-11-01,10.4,1.2,28367.1\nR003,2024-11-02,11.8,0.0,35901.77\nR003,2024-11-03,7.0,7.1,21756.96\nR003,2024-11-04,2.9,14.1,28542.86\nR003,2024-11-05,-0.4,13.7,35609.47\nR003,2024-11-06,-1.5,0.0,28968.09\nR003,2024-11-07,4.4,0.0,34542.77\nR003,2024-11-08,5.0,0.0,35159.57\nR003,2024-11-09,5.0,0.0,35013.43\nR003,2024-11-10,6.3,0.0,34868.24\nR003,2024-11-11,7.3,0.0,25792.11\nR003,2024-11-12,8.4,0.0,34581.41\nR003,2024-11-13,7.1,0.0,34440.1\nR003,2024-11-14,7.5,0.0,27659.25\nR003,2024-11-15,9.1,8.0,18591.35\nR003,2024-11-16,6.6,21.7,18000.05\nR003,2024-11-17,4.5,0.0,32968.72\nR003,2024-11-18,7.8,0.9,28553.15\nR003,2024-11-19,9.4,3.5,21584.03\nR003,2024-11-20,9.3,0.0,31854.2\nR003,2024-11-21,11.3,0.0,32050.63\nR003,2024-11-22,11.0,9.2,4984.03\nR003,2024-11-23,13.6,0.0,31084.07\nR003,2024-11-24,8.9,20.8,0.0\nR003,2024-11-25,5.5,3.0,14680.16\nR003,2024-11-26,5.5,9.8,23043.23\nR003,2024-11-27,5.1,6.9,0.0\nR003,2024-11-28,3.9,1.4,26218.71\nR003,2024-11-29,4.8,0.0,30298.63\nR003,2024-11-30,5.3,0.0,30384.62\nR003,2024-12-01,5.3,0.8,17960.2\nR003,2024-12-02,5.2,0.2,30348.52\nR003,2024-12-03,4.5,0.0,30744.13\nR003,2024-12-04,4.5,0.0,30374.76\nR003,2024-12-05,5.7,0.9,7514.0\nR003,2024-12-06,5.3,2.7,13434.28\nR003,2024-12-07,5.6,0.0,30403.45\nR003,2024-12-08,8.1,0.0,12904.0\nR003,2024-12-09,9.9,0.0,28870.26\nR003,2024-12-10,10.1,0.7,29258.42\nR003,2024-12-11,10.6,12.9,13914.56\nR003,2024-12-12,5.0,28.2,6256.76\nR003,2024-12-13,-1.1,14.7,5585.04\nR003,2024-12-14,-9.2,0.0,30419.79\nR003,2024-12-15,-2.0,1.4,25259.36\nR003,2024-12-16,2.6,18.3,9030.17\nR003,2024-12-17,3.1,60.3,0.0\nR003,2024-12-18,3.4,36.1,23872.74\nR003,2024-12-19,0.5,0.0,30777.7\nR003,2024-12-20,2.9,0.0,30766.25\nR003,2024-12-21,4.5,0.0,28800.0\nR003,2024-12-22,5.6,2.7,14116.82\nR003,2024-12-23,6.5,7.7,22113.07\nR003,2024-12-24,6.9,1.2,31094.29\nR003,2024-12-25,6.8,0.0,27083.03\nR003,2024-12-26,7.4,0.0,31337.33\nR003,2024-12-27,6.0,0.0,26635.05\nR003,2024-12-28,5.2,1.8,19288.85\nR003,2024-12-29,4.9,4.4,5843.3\nR003,2024-12-30,2.9,0.7,29071.36\nR003,2024-12-31,2.5,0.0,30924.16\nR003,2025-01-01,4.5,0.7,21178.16\nR003,2025-01-02,5.6,0.0,31685.61\nR003,2025-01-03,7.3,0.0,31741.79\nR003,2025-01-04,5.4,6.1,24391.01\nR003,2025-01-05,3.3,8.9,14964.44\nR003,2025-01-06,1.3,0.0,31917.21\nR003,2025-01-07,5.2,0.0,31977.83\nR003,2025-01-08,8.1,0.0,32040.56\nR003,2025-01-09,9.0,0.0,32105.42\nR003,2025-01-10,7.5,0.0,32172.25\nR003,2025-01-11,9.1,0.0,32240.91\nR003,2025-01-12,7.8,0.0,15436.65\nR003,2025-01-13,8.5,18.2,14488.57\nR003,2025-01-14,5.7,18.6,0.0\nR003,2025-01-15,5.4,0.3,32400.0\nR003,2025-01-16,4.1,1.9,31778.1\nR003,2025-01-17,3.0,3.9,20761.06\nR003,2025-01-18,2.5,8.2,25831.3\nR003,2025-01-19,1.9,11.3,14998.31\nR003,2025-01-20,-0.5,0.0,32400.0\nR003,2025-01-21,2.1,0.0,32400.0\nR003,2025-01-22,4.4,0.0,32400.0\nR003,2025-01-23,5.7,0.0,1271.05\nR003,2025-01-24,4.9,1.9,22277.39\nR003,2025-01-25,4.7,1.0,32400.0\nR003,2025-01-26,2.5,0.0,32400.0\nR003,2025-01-27,2.9,0.0,32400.0\nR003,2025-01-28,4.5,0.0,32529.05\nR003,2025-01-29,6.3,0.3,32400.0\nR003,2025-01-30,7.4,0.3,32443.78\nR003,2025-01-31,6.8,0.0,32752.47\nR003,2025-02-01,6.1,0.0,29605.06\nR003,2025-02-02,4.6,0.0,32663.62\nR003,2025-02-03,5.2,0.4,27904.05\nR003,2025-02-04,2.8,21.5,4573.71\nR003,2025-02-05,-1.8,0.5,28774.57\nR003,2025-02-06,1.3,0.4,28802.18\nR003,2025-02-07,1.2,0.4,29902.32\nR003,2025-02-08,0.6,0.1,33966.42\nR003,2025-02-09,-0.5,1.4,31591.94\nR003,2025-02-10,0.7,0.8,33440.97\nR003,2025-02-11,-0.1,0.0,36080.81\nR003,2025-02-12,0.7,0.0,36530.28\nR003,2025-02-13,1.5,0.1,33732.9\nR003,2025-02-14,2.3,7.4,1720.53\nR003,2025-02-15,4.0,1.4,31048.93\nR003,2025-02-16,4.4,19.0,0.0\nR003,2025-02-17,3.3,24.1,0.0\nR003,2025-02-18,2.7,23.8,0.0\nR003,2025-02-19,0.9,4.2,29497.21\nR003,2025-02-20,-2.5,1.8,35458.43\nR003,2025-02-21,-4.5,2.3,36430.45\nR003,2025-02-22,-6.2,0.0,36832.09\nR003,2025-02-23,-6.2,0.0,38687.76\nR003,2025-02-24,-4.6,7.0,15810.11\nR003,2025-02-25,-4.0,8.9,17496.25\nR003,2025-02-26,-4.6,0.6,27746.28\nR003,2025-02-27,-2.6,0.0,39386.88\nR003,2025-02-28,-1.1,0.0,39488.33\nR003,2025-03-01,-1.2,0.0,39590.31\nR003,2025-03-02,0.8,0.0,36000.0\nR003,2025-03-03,0.5,7.6,0.0\nR003,2025-03-04,1.1,1.0,39600.0\nR003,2025-03-05,3.1,0.0,31542.17\nR003,2025-03-06,5.8,0.0,30749.1\nR003,2025-03-07,9.7,0.1,39600.0\nR003,2025-03-08,9.8,0.0,39600.0\nR003,2025-03-09,11.6,0.0,39600.0\nR003,2025-03-10,10.6,0.0,39600.0\nR003,2025-03-11,11.6,0.0,39600.0\nR003,2025-03-12,13.2,0.0,39600.0\nR003,2025-03-13,16.2,0.0,39621.6\nR003,2025-03-14,15.6,0.0,20882.56\nR003,2025-03-15,14.0,0.0,39600.0\nR003,2025-03-16,17.3,0.0,39776.0\nR003,2025-03-17,15.3,1.2,36697.67\nR003,2025-03-18,8.9,23.7,3061.34\nR003,2025-03-19,3.5,9.4,9188.19\nR003,2025-03-20,2.8,0.5,33310.42\nR003,2025-03-21,4.2,1.6,12671.76\nR003,2025-03-22,4.8,4.1,22363.5\nR003,2025-03-23,5.7,5.1,31172.25\nR003,2025-03-24,8.0,0.0,41566.46\nR003,2025-03-25,11.2,1.1,36630.8\nR003,2025-03-26,12.7,6.3,20686.54\nR003,2025-03-27,13.8,0.6,37537.22\nR003,2025-03-28,14.7,0.0,43277.16\nR003,2025-03-29,17.8,0.3,43671.35\nR003,2025-03-30,16.9,3.7,27887.43\nR003,2025-03-31,12.2,25.6,14474.28\nR003,2025-04-01,13.9,0.1,44520.06\nR003,2025-04-02,13.5,3.8,21510.23\nR003,2025-04-03,11.4,8.4,815.95\nR003,2025-04-04,11.5,2.4,35311.41\nR003,2025-04-05,9.9,5.2,26156.8\nR003,2025-04-06,10.9,7.5,36225.31\nR003,2025-04-07,13.2,2.4,42536.58\nR003,2025-04-08,8.4,18.3,10424.26\nR003,2025-04-09,4.9,28.1,0.0\nR003,2025-04-10,2.4,7.0,22472.29\nR003,2025-04-11,3.5,0.3,34032.42\nR003,2025-04-12,2.8,14.6,24485.64\nR003,2025-04-13,4.3,2.0,34318.21\nR003,2025-04-14,8.8,1.3,44964.98\nR003,2025-04-15,9.9,0.7,13209.43\nR003,2025-04-16,11.4,0.0,42156.16\nR003,2025-04-17,12.3,0.0,46800.0\nR003,2025-04-18,13.9,1.1,46800.0\nR003,2025-04-19,15.7,2.0,46800.0\nR003,2025-04-20,13.6,6.0,31075.7\nR003,2025-04-21,13.3,9.0,38150.35\nR003,2025-04-22,15.1,0.2,46876.18\nR003,2025-04-23,14.6,5.4,44325.83\nR003,2025-04-24,14.1,3.3,46782.69\nR003,2025-04-25,12.9,11.5,22778.65\nR003,2025-04-26,13.8,10.8,39171.8\nR003,2025-04-27,10.8,8.0,5703.73\nR003,2025-04-28,9.6,3.0,34620.76\nR003,2025-04-29,10.6,0.0,47907.39\nR003,2025-04-30,11.8,0.0,48538.61\nR003,2025-05-01,14.8,1.5,36131.64\nR003,2025-05-02,15.4,8.7,28897.94\nR003,2025-05-03,12.5,16.4,7502.03\nR003,2025-05-04,13.9,3.5,46212.05\nR003,2025-05-05,15.0,5.4,46800.0\nR003,2025-05-06,16.1,3.5,39821.76\nR003,2025-05-07,17.4,0.0,46943.14\nR003,2025-05-08,16.1,9.3,43488.09\nR003,2025-05-09,16.9,2.6,49142.76\nR003,2025-05-10,17.7,5.1,45300.38\nR003,2025-05-11,14.3,12.5,0.0\nR003,2025-05-12,15.4,6.5,33755.45\nR003,2025-05-13,13.0,15.3,559.84\nR003,2025-05-14,12.1,4.9,46270.23\nR003,2025-05-15,10.5,5.9,15807.49\nR003,2025-05-16,14.7,0.1,50134.26\nR003,2025-05-17,21.1,0.4,50415.63\nR003,2025-05-18,17.9,8.6,40865.22\nR003,2025-05-19,13.6,10.9,45391.03\nR003,2025-05-20,14.2,0.8,47915.61\nR003,2025-05-21,13.1,0.0,48458.96\nR003,2025-05-22,16.1,0.0,50985.72\nR003,2025-05-23,19.4,0.3,50603.67\nR003,2025-05-24,20.5,0.5,50935.54\nR003,2025-05-25,20.9,0.8,50911.34\nR003,2025-05-26,22.4,0.3,50400.0\nR003,2025-05-27,21.5,5.2,44557.94\nR003,2025-05-28,19.0,12.5,38783.06\nR003,2025-05-29,17.8,15.4,27072.74\nR003,2025-05-30,17.0,21.8,34525.6\nR003,2025-05-31,17.0,9.3,39482.36\nR003,2025-06-01,18.9,4.6,46800.0\nR003,2025-06-02,17.5,15.6,0.0\nR003,2025-06-03,18.0,0.5,47839.82\nR003,2025-06-04,18.5,0.0,50749.03\nR003,2025-06-05,18.2,2.2,48350.81\nR003,2025-06-06,18.9,0.0,53209.15\nR003,2025-06-07,19.3,0.0,53559.3\nR003,2025-06-08,21.7,0.0,53545.03\nR003,2025-06-09,22.3,0.2,53165.28\nR003,2025-06-10,22.2,0.2,52419.04\nR003,2025-06-11,21.9,1.3,52405.62\nR003,2025-06-12,20.1,5.2,50925.24\nR003,2025-06-13,21.1,0.2,50759.49\nR003,2025-06-14,17.9,3.9,39660.1\nR003,2025-06-15,15.1,2.1,48077.37\nR003,2025-06-16,14.3,6.3,42114.82\nR003,2025-06-17,14.8,5.6,37768.87\nR003,2025-06-18,16.2,1.3,51004.04\nR003,2025-06-19,18.2,0.4,53906.49\nR003,2025-06-20,19.1,0.0,53921.06\nR003,2025-06-21,16.4,10.9,33782.33\nR003,2025-06-22,16.5,0.9,51936.58\nR003,2025-06-23,15.2,2.4,48697.86\nR003,2025-06-24,17.4,2.4,50911.24\nR003,2025-06-25,19.5,0.1,53965.17\nR003,2025-06-26,21.0,0.2,53967.9\nR003,2025-06-27,21.5,0.4,53144.24\nR003,2025-06-28,20.9,0.0,53966.92\nR003,2025-06-29,18.1,8.7,30687.12\nR003,2025-06-30,18.0,0.2,48849.32\nR003,2025-07-01,17.4,3.0,49892.86\nR003,2025-07-02,17.5,0.5,53057.68\nR003,2025-07-03,17.4,0.2,51264.87\nR003,2025-07-04,18.4,1.4,50070.15\nR003,2025-07-05,20.1,1.1,51668.49\nR003,2025-07-06,22.0,0.4,53870.56\nR003,2025-07-07,22.9,0.8,53757.05\nR003,2025-07-08,24.6,0.6,53822.71\nR003,2025-07-09,25.4,0.6,53795.46\nR003,2025-07-10,25.1,4.7,53766.02\nR003,2025-07-11,24.6,0.7,53421.14\nR003,2025-07-12,26.2,0.1,52172.62\nR003,2025-07-13,26.7,1.0,53665.0\nR003,2025-07-14,25.6,2.4,52014.72\nR003,2025-07-15,23.9,1.4,52175.91\nR003,2025-07-16,22.8,10.6,44621.98\nR003,2025-07-17,23.6,3.8,50745.85\nR003,2025-07-18,25.4,0.0,53456.15\nR003,2025-07-19,25.0,1.5,52801.26\nR003,2025-07-20,22.3,14.7,39066.14\nR003,2025-07-21,23.3,6.0,48902.93\nR003,2025-07-22,24.3,1.3,49299.28\nR003,2025-07-23,24.7,0.1,50045.93\nR003,2025-07-24,25.2,0.0,49458.97\nR003,2025-07-25,24.5,0.0,51185.9\nR003,2025-07-26,24.3,0.1,51828.89\nR003,2025-07-27,25.6,0.0,51793.38\nR003,2025-07-28,27.2,0.0,51409.68\nR003,2025-07-29,29.4,0.0,51375.13\nR003,2025-07-30,29.6,0.1,50994.05\nR003,2025-07-31,26.9,3.8,47440.95\nR003,2025-08-01,26.1,2.6,37369.73\nR003,2025-08-02,26.7,17.0,44842.93\nR003,2025-08-03,24.7,10.7,46076.98\nR003,2025-08-04,25.8,3.6,43624.1\nR003,2025-08-05,25.9,4.1,44420.03\nR003,2025-08-06,27.2,0.6,46904.46\nR003,2025-08-07,26.8,3.6,48300.83\nR003,2025-08-08,24.3,9.6,40432.87\nR003,2025-08-09,23.9,4.7,39831.31\nR003,2025-08-10,22.9,0.2,47032.53\nR003,2025-08-11,22.3,0.3,47735.54\nR003,2025-08-12,22.2,0.0,47092.8\nR003,2025-08-13,21.0,0.0,47122.96\nR003,2025-08-14,20.2,0.1,47487.26\nR003,2025-08-15,20.6,0.3,47183.18\nR003,2025-08-16,20.6,0.6,46880.61\nR003,2025-08-17,22.0,0.0,47243.63\nR003,2025-08-18,23.7,0.0,46942.76\nR003,2025-08-19,23.9,0.5,46800.0\nR003,2025-08-20,22.8,9.3,46534.77\nR003,2025-08-21,21.4,13.2,45497.69\nR003,2025-08-22,22.4,0.4,46800.0\nR003,2025-08-23,24.0,0.0,46800.0\nR003,2025-08-24,23.3,0.5,46800.0\nR003,2025-08-25,21.7,2.4,43333.65\nR003,2025-08-26,20.0,2.7,22522.58\nR003,2025-08-27,21.8,0.0,45330.04\nR003,2025-08-28,23.5,0.3,46603.99\nR003,2025-08-29,24.8,0.0,46500.48\nR003,2025-08-30,26.2,0.0,46396.67\nR003,2025-08-31,26.1,0.9,46292.65\nR003,2025-09-01,26.5,0.1,46188.51\nR003,2025-09-02,23.7,3.8,41107.58\nR003,2025-09-03,21.6,5.1,37928.32\nR003,2025-09-04,21.8,0.6,44085.03\nR003,2025-09-05,22.4,0.1,39461.42\nR003,2025-09-06,22.7,0.0,44206.01\nR003,2025-09-07,19.6,11.0,34170.62\nR003,2025-09-08,18.9,6.1,39554.37\nR003,2025-09-09,18.9,17.2,18311.49\nR003,2025-09-10,18.2,7.7,30364.57\nR003,2025-09-11,17.6,5.3,36116.78\nR003,2025-09-12,18.4,6.3,40279.79\nR003,2025-09-13,17.0,0.0,41007.74\nR003,2025-09-14,17.1,0.7,43089.62\nR003,2025-09-15,18.2,1.4,43022.25\nR003,2025-09-16,17.6,3.9,36963.71\nR003,2025-09-17,18.5,0.5,40821.07\nR003,2025-09-18,16.5,14.7,36299.86\nR003,2025-09-19,14.8,3.7,12564.46\nR003,2025-09-20,15.9,2.0,40604.36\nR003,2025-09-21,15.2,0.1,40002.02\nR003,2025-09-22,15.8,0.0,42265.32\nR003,2025-09-23,18.2,0.0,42300.54\nR003,2025-09-24,18.9,0.0,42336.14\nR003,2025-09-25,18.1,0.3,38232.16\nR003,2025-09-26,14.8,8.7,20569.92\nR003,2025-09-27,13.9,2.1,29971.66\nR003,2025-09-28,14.1,1.1,31674.85\nR003,2025-09-29,14.8,10.5,0.0\nR003,2025-09-30,14.4,15.0,24603.18\nR003,2025-10-01,14.1,0.4,36648.87\nR003,2025-10-02,15.2,0.8,38727.62\nR003,2025-10-03,16.8,0.0,39103.15\nR003,2025-10-04,20.7,0.0,39541.58\nR003,2025-10-05,18.5,4.4,37402.4\nR003,2025-10-06,17.0,0.0,37173.57\nR003,2025-10-07,18.8,0.0,38561.09\nR003,2025-10-08,20.9,0.1,37686.33\nR003,2025-10-09,15.8,25.1,7851.33\nR003,2025-10-10,11.9,28.1,11283.04\nR003,2025-10-11,11.1,9.6,31678.09\nR003,2025-10-12,10.4,9.1,4581.72\nR003,2025-10-13,9.9,1.4,36280.89\nR003,2025-10-14,10.9,2.8,31120.0\nR003,2025-10-15,10.9,1.0,36057.11\nR003,2025-10-16,10.5,1.3,36108.28\nR003,2025-10-17,10.5,0.0,36159.37\nR003,2025-10-18,12.2,0.0,33646.39\nR003,2025-10-19,12.7,9.3,27772.85\nR003,2025-10-20,10.2,7.1,16695.66\nR003,2025-10-21,7.7,1.3,36000.0\nR003,2025-10-22,10.2,0.0,34322.08\nR003,2025-10-23,13.3,1.5,22671.19\nR003,2025-10-24,14.1,4.0,21611.25\nR003,2025-10-25,14.8,4.3,1346.74\nR003,2025-10-26,11.9,26.8,16492.81\nR003,2025-10-27,11.4,0.9,26725.55\nR003,2025-10-28,14.9,4.4,30379.25\nR003,2025-10-29,10.6,23.2,18418.18\nR003,2025-10-30,7.8,1.5,29952.2\nR003,2025-10-31,9.5,0.0,36000.0\nR003,2025-11-01,11.1,0.0,36000.0\nR003,2025-11-02,7.8,0.0,35922.85\nR003,2025-11-03,12.3,0.0,35844.67\nR003,2025-11-04,14.2,0.0,35766.11\nR003,2025-11-05,13.5,0.0,34247.04\nR003,2025-11-06,12.4,0.0,35505.85\nR003,2025-11-07,13.7,0.0,32430.99\nR003,2025-11-08,16.0,0.3,35201.76\nR003,2025-11-09,14.7,0.3,34514.72\nR003,2025-11-10,14.3,0.0,32016.4\nR003,2025-11-11,14.3,0.0,27270.67\nR003,2025-11-12,13.4,4.1,22868.9\nR003,2025-11-13,10.8,22.2,1801.62\nR003,2025-11-14,9.6,20.6,1511.56\nR003,2025-11-15,11.4,0.5,32989.83\nR003,2025-11-16,11.5,0.0,32361.68\nR003,2025-11-17,11.1,0.0,32605.78\nR003,2025-11-18,11.6,0.0,33326.54\nR003,2025-11-19,14.8,0.0,30771.57\nR003,2025-11-20,14.8,0.0,29631.95\nR003,2025-11-21,13.4,0.0,32401.0\nR003,2025-11-22,14.5,0.0,31367.17\nR003,2025-11-23,15.2,0.0,31063.93\nR003,2025-11-24,14.1,0.0,29378.28\nR003,2025-11-25,11.8,0.0,29081.2\nR003,2025-11-26,11.1,0.0,26743.08\nR003,2025-11-27,11.0,0.0,30626.1\nR003,2025-11-28,10.5,0.0,30648.27\nR003,2025-11-29,10.9,0.0,30669.15\nR003,2025-11-30,11.7,0.6,28860.49\nR003,2025-12-01,9.6,6.5,3239.37\nR003,2025-12-02,7.8,2.1,7031.82\nR003,2025-12-03,6.2,0.0,29472.95\nR003,2025-12-04,6.1,0.0,28841.16\nR003,2025-12-05,6.7,0.0,30383.01\nR003,2025-12-06,6.7,0.0,28857.18\nR003,2025-12-07,8.4,3.9,0.0\nR003,2025-12-08,9.7,11.2,26410.28\nR003,2025-12-09,8.4,0.2,26059.71\nR003,2025-12-10,6.3,1.8,28536.46\nR003,2025-12-11,6.3,5.6,14558.37\nR003,2025-12-12,5.3,6.0,18319.67\nR003,2025-12-13,3.4,12.6,10648.35\nR003,2025-12-14,-1.6,0.0,26572.85\nR003,2025-12-15,2.7,2.2,1607.5\nR003,2025-12-16,2.1,5.8,22824.27\nR003,2025-12-17,1.1,0.0,30016.75\nR003,2025-12-18,5.1,0.0,30007.63\nR003,2025-12-19,5.7,0.0,29996.95\nR003,2025-12-20,5.0,0.0,30766.5\nR003,2025-12-21,6.3,0.0,29579.93\nR003,2025-12-22,6.8,2.2,10324.76\nR003,2025-12-23,6.5,1.9,28768.63\nR003,2025-12-24,5.7,6.9,5442.85\nR003,2025-12-25,3.5,0.0,30003.52\nR003,2025-12-26,1.9,3.5,30272.87\nR003,2025-12-27,-1.1,13.8,14790.57\nR003,2025-12-28,-2.5,18.4,12431.41\nR003,2025-12-29,-1.1,13.8,10163.35\nR003,2025-12-30,-1.0,22.6,8419.04\nR003,2025-12-31,0.7,36.5,0.0\nR004,1996-01-01,-20.0,0.0,24407.12\nR004,1996-01-02,-17.8,8.1,0.0\nR004,1996-01-03,-13.2,4.2,0.0\nR004,1996-01-04,-14.3,4.2,0.0\nR004,1996-01-05,-16.6,2.8,0.0\nR004,1996-01-06,-19.0,1.3,0.0\nR004,1996-01-07,-23.0,0.0,22875.16\nR004,1996-01-08,-14.8,5.9,0.0\nR004,1996-01-09,-13.6,7.0,1941.7\nR004,1996-01-10,-20.0,2.4,0.0\nR004,1996-01-11,-29.2,0.2,25200.0\nR004,1996-01-12,-33.9,0.0,21751.47\nR004,1996-01-13,-34.3,0.0,26586.22\nR004,1996-01-14,-27.7,0.0,456.97\nR004,1996-01-15,-23.7,0.0,19017.82\nR004,1996-01-16,-25.7,0.0,27463.65\nR004,1996-01-17,-27.9,0.0,27631.64\nR004,1996-01-18,-27.7,0.0,27692.65\nR004,1996-01-19,-29.0,0.0,27756.46\nR004,1996-01-20,-29.3,0.0,26606.34\nR004,1996-01-21,-29.9,0.0,27892.83\nR004,1996-01-22,-24.4,0.0,17223.36\nR004,1996-01-23,-21.0,0.0,27251.71\nR004,1996-01-24,-20.3,2.3,18426.3\nR004,1996-01-25,-12.9,13.5,0.0\nR004,1996-01-26,-13.6,5.8,5672.71\nR004,1996-01-27,-18.0,1.8,19564.47\nR004,1996-01-28,-24.4,0.0,30588.19\nR004,1996-01-29,-25.6,0.0,21759.1\nR004,1996-01-30,-24.6,0.0,31079.88\nR004,1996-01-31,-24.5,0.0,31294.02\nR004,1996-02-01,-20.9,0.0,29277.93\nR004,1996-02-02,-18.4,0.0,25450.68\nR004,1996-02-03,-14.5,2.0,12924.55\nR004,1996-02-04,-12.4,4.6,2914.48\nR004,1996-02-05,-15.6,0.9,19549.1\nR004,1996-02-06,-15.5,0.0,25985.15\nR004,1996-02-07,-16.1,0.0,32165.54\nR004,1996-02-08,-10.9,0.5,11131.05\nR004,1996-02-09,-4.5,0.3,24547.68\nR004,1996-02-10,-4.0,14.1,0.0\nR004,1996-02-11,-8.6,13.1,0.0\nR004,1996-02-12,-16.5,5.0,23625.73\nR004,1996-02-13,-26.9,0.0,28256.66\nR004,1996-02-14,-28.0,0.0,32498.85\nR004,1996-02-15,-26.3,0.0,32850.87\nR004,1996-02-16,-24.7,0.0,33199.27\nR004,1996-02-17,-24.1,0.0,33544.24\nR004,1996-02-18,-23.4,0.0,33885.94\nR004,1996-02-19,-20.3,0.0,34224.55\nR004,1996-02-20,-17.4,0.0,34723.01\nR004,1996-02-21,-16.6,0.0,34860.6\nR004,1996-02-22,-14.0,0.0,34447.07\nR004,1996-02-23,-14.0,0.0,27650.27\nR004,1996-02-24,-13.0,0.0,32395.54\nR004,1996-02-25,-12.2,0.0,36745.16\nR004,1996-02-26,-13.7,0.0,36415.43\nR004,1996-02-27,-14.8,0.0,36088.4\nR004,1996-02-28,-13.4,0.0,37904.75\nR004,1996-02-29,-13.2,0.0,38150.0\nR004,1996-03-01,-12.5,0.0,38396.06\nR004,1996-03-02,-10.7,0.0,32400.0\nR004,1996-03-03,-6.1,0.0,37023.68\nR004,1996-03-04,-9.2,5.4,22316.68\nR004,1996-03-05,-14.1,1.1,34909.42\nR004,1996-03-06,-14.4,0.0,39007.4\nR004,1996-03-07,-9.8,0.0,39119.2\nR004,1996-03-08,-7.9,0.0,39230.78\nR004,1996-03-09,-5.6,0.0,39342.12\nR004,1996-03-10,-4.4,0.4,34314.77\nR004,1996-03-11,-9.1,2.7,18857.73\nR004,1996-03-12,-9.9,5.0,15818.58\nR004,1996-03-13,-10.8,2.1,7834.85\nR004,1996-03-14,-12.4,0.0,38097.01\nR004,1996-03-15,-14.5,2.5,17547.03\nR004,1996-03-16,-10.1,4.4,12556.36\nR004,1996-03-17,-8.8,1.7,7240.85\nR004,1996-03-18,-9.3,2.0,26890.87\nR004,1996-03-19,-9.1,0.3,40127.56\nR004,1996-03-20,-11.4,0.0,42230.38\nR004,1996-03-21,-12.4,0.0,42371.09\nR004,1996-03-22,-12.7,0.0,42512.73\nR004,1996-03-23,-11.4,0.0,42655.18\nR004,1996-03-24,-9.9,0.0,42871.23\nR004,1996-03-25,-8.5,0.0,39452.85\nR004,1996-03-26,-5.1,0.5,39486.1\nR004,1996-03-27,-6.1,6.5,10783.71\nR004,1996-03-28,-6.6,2.7,33769.73\nR004,1996-03-29,-6.6,0.0,44152.89\nR004,1996-03-30,-4.1,0.0,44483.24\nR004,1996-03-31,-2.3,0.0,44814.49\nR004,1996-04-01,-1.4,0.0,45146.7\nR004,1996-04-02,-1.4,0.0,45480.05\nR004,1996-04-03,-4.2,0.0,45430.71\nR004,1996-04-04,-8.5,1.3,30025.82\nR004,1996-04-05,-10.4,0.1,42752.89\nR004,1996-04-06,-10.3,0.0,46010.74\nR004,1996-04-07,-9.4,0.0,46434.56\nR004,1996-04-08,-7.6,0.0,46860.01\nR004,1996-04-09,-5.6,0.0,47287.28\nR004,1996-04-10,-4.2,0.0,46388.56\nR004,1996-04-11,-1.1,0.0,47758.33\nR004,1996-04-12,1.2,0.0,45392.29\nR004,1996-04-13,1.7,0.0,39277.66\nR004,1996-04-14,0.6,9.6,0.0\nR004,1996-04-15,-0.1,0.0,40599.93\nR004,1996-04-16,-0.3,0.0,47372.54\nR004,1996-04-17,0.6,0.0,45818.69\nR004,1996-04-18,3.0,1.5,41622.55\nR004,1996-04-19,5.1,0.0,45500.68\nR004,1996-04-20,7.5,0.0,37487.91\nR004,1996-04-21,7.0,5.0,13752.29\nR004,1996-04-22,-0.8,4.2,31876.15\nR004,1996-04-23,3.7,1.3,23449.96\nR004,1996-04-24,0.0,0.9,46165.4\nR004,1996-04-25,3.9,11.9,24239.11\nR004,1996-04-26,6.2,16.7,35838.52\nR004,1996-04-27,2.9,6.2,28545.28\nR004,1996-04-28,-2.1,0.7,27518.35\nR004,1996-04-29,-3.4,3.9,33996.06\nR004,1996-04-30,-1.8,0.0,35845.17\nR004,1996-05-01,1.7,0.1,38027.43\nR004,1996-05-02,3.1,7.0,31143.0\nR004,1996-05-03,2.8,0.4,39280.53\nR004,1996-05-04,2.5,5.2,24612.35\nR004,1996-05-05,3.6,0.5,50563.68\nR004,1996-05-06,6.6,0.2,27525.86\nR004,1996-05-07,11.4,5.7,50470.02\nR004,1996-05-08,10.5,0.0,51706.11\nR004,1996-05-09,10.7,0.0,52376.22\nR004,1996-05-10,13.4,0.0,52719.9\nR004,1996-05-11,11.2,0.3,48920.45\nR004,1996-05-12,12.1,0.0,52552.08\nR004,1996-05-13,1.6,8.6,27922.77\nR004,1996-05-14,4.1,0.0,50888.91\nR004,1996-05-15,5.5,6.2,33256.96\nR004,1996-05-16,6.7,0.1,47907.73\nR004,1996-05-17,12.2,2.2,49057.19\nR004,1996-05-18,11.7,3.5,51915.05\nR004,1996-05-19,14.3,0.0,56031.53\nR004,1996-05-20,16.8,0.9,49056.72\nR004,1996-05-21,17.0,1.7,31804.82\nR004,1996-05-22,15.1,9.0,33144.26\nR004,1996-05-23,7.0,11.2,12077.6\nR004,1996-05-24,12.9,0.0,57360.24\nR004,1996-05-25,13.3,9.9,23417.96\nR004,1996-05-26,10.9,0.3,51700.02\nR004,1996-05-27,10.0,0.0,53180.21\nR004,1996-05-28,9.5,0.0,51068.33\nR004,1996-05-29,7.7,0.0,41689.46\nR004,1996-05-30,6.8,1.1,16259.92\nR004,1996-05-31,5.5,5.3,45478.45\nR004,1996-06-01,6.1,2.0,49778.0\nR004,1996-06-02,8.1,0.8,53045.76\nR004,1996-06-03,8.5,5.9,20532.08\nR004,1996-06-04,7.7,5.9,12281.01\nR004,1996-06-05,6.8,12.2,12129.38\nR004,1996-06-06,10.8,2.8,21600.0\nR004,1996-06-07,10.7,7.1,43200.0\nR004,1996-06-08,11.1,0.0,55814.23\nR004,1996-06-09,14.4,0.0,57600.0\nR004,1996-06-10,17.2,0.4,56811.42\nR004,1996-06-11,17.0,12.8,30924.36\nR004,1996-06-12,13.8,12.4,14400.0\nR004,1996-06-13,14.7,6.1,40827.33\nR004,1996-06-14,14.1,4.3,41053.86\nR004,1996-06-15,12.0,3.3,49386.54\nR004,1996-06-16,11.2,6.3,31248.81\nR004,1996-06-17,12.6,1.1,54122.02\nR004,1996-06-18,16.4,0.0,56890.0\nR004,1996-06-19,18.6,0.0,57600.0\nR004,1996-06-20,17.0,0.0,57600.0\nR004,1996-06-21,15.8,0.0,57600.0\nR004,1996-06-22,17.4,0.1,42110.42\nR004,1996-06-23,17.7,9.9,38377.76\nR004,1996-06-24,16.6,9.2,46783.96\nR004,1996-06-25,14.7,21.3,16020.61\nR004,1996-06-26,14.2,14.0,27724.61\nR004,1996-06-27,14.9,1.8,56271.12\nR004,1996-06-28,14.7,14.8,27360.55\nR004,1996-06-29,13.8,8.9,38522.55\nR004,1996-06-30,13.4,9.8,40087.33\nR004,1996-07-01,12.2,5.6,27091.62\nR004,1996-07-02,15.4,3.9,44080.44\nR004,1996-07-03,16.1,9.8,43407.3\nR004,1996-07-04,16.0,0.0,47880.12\nR004,1996-07-05,16.0,4.8,47570.4\nR004,1996-07-06,15.6,4.7,45482.57\nR004,1996-07-07,18.0,0.2,51557.98\nR004,1996-07-08,21.2,0.6,56188.06\nR004,1996-07-09,22.2,0.2,57372.67\nR004,1996-07-10,22.3,0.6,53803.78\nR004,1996-07-11,17.1,17.9,21064.14\nR004,1996-07-12,18.8,0.1,55846.44\nR004,1996-07-13,20.2,0.0,56531.19\nR004,1996-07-14,21.3,0.0,51802.15\nR004,1996-07-15,22.0,0.2,51014.57\nR004,1996-07-16,21.5,0.0,53152.18\nR004,1996-07-17,22.0,0.0,55443.06\nR004,1996-07-18,20.9,1.8,34577.13\nR004,1996-07-19,18.7,9.2,28426.88\nR004,1996-07-20,19.6,13.9,28089.04\nR004,1996-07-21,18.5,39.1,0.0\nR004,1996-07-22,19.9,2.4,25515.4\nR004,1996-07-23,20.2,6.0,44535.5\nR004,1996-07-24,17.7,24.5,0.0\nR004,1996-07-25,18.8,16.4,44281.93\nR004,1996-07-26,19.8,1.6,44181.31\nR004,1996-07-27,20.2,2.8,48606.05\nR004,1996-07-28,17.6,14.5,20415.63\nR004,1996-07-29,16.9,1.1,50326.99\nR004,1996-07-30,18.6,0.0,53924.02\nR004,1996-07-31,19.5,0.0,53823.5\nR004,1996-08-01,17.5,13.8,44158.37\nR004,1996-08-02,17.0,1.3,32339.01\nR004,1996-08-03,17.1,0.1,46463.99\nR004,1996-08-04,18.7,0.8,53137.74\nR004,1996-08-05,18.4,4.3,47204.64\nR004,1996-08-06,16.4,6.5,37598.69\nR004,1996-08-07,16.3,0.6,43660.93\nR004,1996-08-08,16.3,0.2,51551.03\nR004,1996-08-09,18.2,0.0,50400.0\nR004,1996-08-10,20.3,0.0,49400.97\nR004,1996-08-11,20.6,3.1,47964.84\nR004,1996-08-12,11.5,26.9,38227.38\nR004,1996-08-13,10.8,0.0,50400.0\nR004,1996-08-14,10.6,0.3,28252.73\nR004,1996-08-15,9.9,10.7,13916.89\nR004,1996-08-16,9.1,12.8,25861.81\nR004,1996-08-17,8.6,9.2,44802.7\nR004,1996-08-18,10.8,3.0,50398.89\nR004,1996-08-19,11.8,0.6,50297.44\nR004,1996-08-20,12.7,0.6,44547.9\nR004,1996-08-21,13.5,0.4,50094.48\nR004,1996-08-22,14.7,0.2,49993.06\nR004,1996-08-23,11.8,7.7,6973.43\nR004,1996-08-24,11.6,0.3,28514.76\nR004,1996-08-25,15.7,5.6,46934.09\nR004,1996-08-26,13.1,0.2,44726.01\nR004,1996-08-27,8.9,7.9,14801.88\nR004,1996-08-28,6.3,8.1,0.0\nR004,1996-08-29,6.4,9.7,3759.55\nR004,1996-08-30,4.9,6.9,19309.5\nR004,1996-08-31,4.1,8.9,21729.03\nR004,1996-09-01,4.4,3.2,29130.68\nR004,1996-09-02,6.0,10.3,28164.41\nR004,1996-09-03,6.0,28.4,22582.87\nR004,1996-09-04,4.9,6.2,6280.15\nR004,1996-09-05,6.7,11.8,32654.6\nR004,1996-09-06,4.5,0.6,42996.13\nR004,1996-09-07,4.8,2.8,15509.21\nR004,1996-09-08,6.3,5.3,2006.47\nR004,1996-09-09,12.0,1.4,32972.74\nR004,1996-09-10,8.9,7.2,0.0\nR004,1996-09-11,5.7,0.0,42687.84\nR004,1996-09-12,7.8,13.4,11359.47\nR004,1996-09-13,9.8,4.1,14412.36\nR004,1996-09-14,9.4,8.0,15168.55\nR004,1996-09-15,8.4,13.3,17403.16\nR004,1996-09-16,5.1,12.5,0.0\nR004,1996-09-17,4.3,3.6,16010.1\nR004,1996-09-18,6.5,11.4,3004.6\nR004,1996-09-19,8.7,5.2,3232.75\nR004,1996-09-20,11.2,1.9,17036.37\nR004,1996-09-21,10.5,20.7,16700.5\nR004,1996-09-22,8.1,0.0,42958.47\nR004,1996-09-23,8.1,0.0,43065.57\nR004,1996-09-24,10.2,4.6,27380.01\nR004,1996-09-25,6.6,11.4,0.0\nR004,1996-09-26,2.7,0.9,36892.11\nR004,1996-09-27,3.3,0.4,40020.95\nR004,1996-09-28,4.4,0.0,39626.48\nR004,1996-09-29,4.8,0.0,32427.15\nR004,1996-09-30,7.3,0.2,16569.24\nR004,1996-10-01,7.8,2.9,19663.68\nR004,1996-10-02,-0.4,3.0,20560.96\nR004,1996-10-03,0.2,0.0,17712.2\nR004,1996-10-04,-0.1,6.4,0.0\nR004,1996-10-05,2.4,4.5,0.0\nR004,1996-10-06,4.7,0.0,31654.2\nR004,1996-10-07,6.3,0.1,36000.0\nR004,1996-10-08,6.1,0.0,36000.0\nR004,1996-10-09,6.9,0.0,35263.69\nR004,1996-10-10,8.1,0.0,34816.56\nR004,1996-10-11,5.1,0.0,29620.2\nR004,1996-10-12,4.1,0.0,36000.0\nR004,1996-10-13,4.4,0.7,21983.58\nR004,1996-10-14,4.0,5.4,9414.12\nR004,1996-10-15,4.9,0.0,27057.21\nR004,1996-10-16,3.8,2.6,6630.57\nR004,1996-10-17,0.2,7.5,12238.2\nR004,1996-10-18,-1.7,0.0,11509.82\nR004,1996-10-19,-1.5,6.6,1380.5\nR004,1996-10-20,-4.9,1.7,12072.96\nR004,1996-10-21,-7.4,1.9,11951.79\nR004,1996-10-22,-8.2,2.9,16017.31\nR004,1996-10-23,-16.6,0.0,26487.39\nR004,1996-10-24,-14.3,0.1,22870.9\nR004,1996-10-25,-0.7,10.7,0.0\nR004,1996-10-26,-5.5,11.1,10990.12\nR004,1996-10-27,-14.9,0.2,31750.77\nR004,1996-10-28,-15.6,0.0,29284.73\nR004,1996-10-29,-12.8,0.0,28421.64\nR004,1996-10-30,-11.1,0.0,21029.19\nR004,1996-10-31,-8.9,0.0,25080.26\nR004,1996-11-01,-7.5,0.0,29937.8\nR004,1996-11-02,-9.8,0.0,11518.77\nR004,1996-11-03,-7.7,0.3,9107.28\nR004,1996-11-04,-7.4,0.0,26204.36\nR004,1996-11-05,-3.9,1.0,12384.27\nR004,1996-11-06,-1.6,4.3,0.0\nR004,1996-11-07,-2.7,3.5,0.0\nR004,1996-11-08,-2.8,6.5,824.07\nR004,1996-11-09,-10.1,7.1,504.43\nR004,1996-11-10,-18.9,0.0,28302.94\nR004,1996-11-11,-23.3,0.0,28821.6\nR004,1996-11-12,-22.3,0.0,28881.12\nR004,1996-11-13,-21.9,0.0,28940.1\nR004,1996-11-14,-21.9,0.0,28800.0\nR004,1996-11-15,-16.1,0.8,21942.57\nR004,1996-11-16,-4.8,3.3,0.0\nR004,1996-11-17,-3.0,1.8,0.0\nR004,1996-11-18,-4.4,11.4,0.0\nR004,1996-11-19,-4.1,15.3,0.0\nR004,1996-11-20,-3.2,7.5,5721.65\nR004,1996-11-21,-3.5,13.3,0.0\nR004,1996-11-22,-3.4,19.9,0.0\nR004,1996-11-23,-3.7,14.6,0.0\nR004,1996-11-24,-12.7,1.8,20248.1\nR004,1996-11-25,-18.6,0.8,194.06\nR004,1996-11-26,-11.0,5.4,0.0\nR004,1996-11-27,-10.5,5.6,0.0\nR004,1996-11-28,-8.9,10.0,0.0\nR004,1996-11-29,-10.1,0.0,27963.88\nR004,1996-11-30,-11.3,1.7,0.0\nR004,1996-12-01,-10.7,6.1,0.0\nR004,1996-12-02,-13.9,2.5,1275.7\nR004,1996-12-03,-19.0,0.5,24068.38\nR004,1996-12-04,-21.6,1.3,0.0\nR004,1996-12-05,-11.9,0.3,20956.27\nR004,1996-12-06,-10.7,0.4,6375.25\nR004,1996-12-07,-8.8,4.1,1732.8\nR004,1996-12-08,-5.7,8.4,0.0\nR004,1996-12-09,-6.7,13.5,0.0\nR004,1996-12-10,-6.1,4.0,0.0\nR004,1996-12-11,-3.9,9.5,0.0\nR004,1996-12-12,-3.6,20.9,0.0\nR004,1996-12-13,-5.6,10.6,0.0\nR004,1996-12-14,-19.1,2.6,1421.36\nR004,1996-12-15,-17.2,1.1,10250.62\nR004,1996-12-16,-19.9,0.0,185.47\nR004,1996-12-17,-20.4,0.0,25289.74\nR004,1996-12-18,-22.4,0.0,25328.95\nR004,1996-12-19,-15.7,1.2,21531.06\nR004,1996-12-20,-11.0,3.1,14898.32\nR004,1996-12-21,-7.7,0.1,9688.49\nR004,1996-12-22,-7.3,7.7,17908.99\nR004,1996-12-23,-13.0,1.3,23321.87\nR004,1996-12-24,-10.7,15.4,0.0\nR004,1996-12-25,-7.7,8.4,18531.19\nR004,1996-12-26,-11.8,5.3,0.0\nR004,1996-12-27,-17.2,4.0,22485.45\nR004,1996-12-28,-24.7,0.0,17829.39\nR004,1996-12-29,-25.5,0.2,23359.59\nR004,1996-12-30,-30.0,0.0,24849.3\nR004,1996-12-31,-29.8,0.0,22655.19\nR004,1997-01-01,-19.0,6.2,0.0\nR004,1997-01-02,-17.2,1.3,17484.97\nR004,1997-01-03,-24.1,0.0,22594.9\nR004,1997-01-04,-21.4,0.0,25182.26\nR004,1997-01-05,-12.6,0.9,11744.92\nR004,1997-01-06,-9.4,0.4,4969.64\nR004,1997-01-07,-12.3,0.0,21479.34\nR004,1997-01-08,-10.7,1.0,13525.81\nR004,1997-01-09,-10.9,2.0,14604.78\nR004,1997-01-10,-18.7,0.0,25450.72\nR004,1997-01-11,-21.6,0.0,13658.41\nR004,1997-01-12,-24.3,0.0,22158.82\nR004,1997-01-13,-21.5,0.2,18350.98\nR004,1997-01-14,-14.8,5.8,11028.02\nR004,1997-01-15,-21.3,1.1,17970.75\nR004,1997-01-16,-27.7,0.0,27374.85\nR004,1997-01-17,-20.9,1.9,0.0\nR004,1997-01-18,-12.8,5.6,0.0\nR004,1997-01-19,-7.9,8.4,0.0\nR004,1997-01-20,-12.4,1.0,26276.86\nR004,1997-01-21,-17.9,0.1,28147.97\nR004,1997-01-22,-13.4,1.3,0.0\nR004,1997-01-23,-9.5,1.8,0.0\nR004,1997-01-24,-7.1,0.5,0.0\nR004,1997-01-25,-7.5,2.5,0.0\nR004,1997-01-26,-5.3,8.5,0.0\nR004,1997-01-27,-3.8,1.9,1721.25\nR004,1997-01-28,-7.1,1.7,429.17\nR004,1997-01-29,-8.0,0.9,9081.02\nR004,1997-01-30,-9.8,3.9,11889.89\nR004,1997-01-31,-16.5,0.0,31433.98\nR004,1997-02-01,-14.9,0.0,4290.25\nR004,1997-02-02,-13.3,4.4,14569.95\nR004,1997-02-03,-20.6,0.0,31779.4\nR004,1997-02-04,-22.2,0.0,27053.53\nR004,1997-02-05,-18.0,1.9,13490.6\nR004,1997-02-06,-13.5,2.2,26394.8\nR004,1997-02-07,-11.8,1.5,9190.09\nR004,1997-02-08,-9.6,0.0,25043.29\nR004,1997-02-09,-12.0,0.0,21698.57\nR004,1997-02-10,-10.5,0.0,22845.64\nR004,1997-02-11,-7.8,0.8,6545.55\nR004,1997-02-12,-7.4,1.2,28337.22\nR004,1997-02-13,-13.1,0.0,23188.05\nR004,1997-02-14,-12.4,0.0,32862.25\nR004,1997-02-15,-10.2,1.7,5849.09\nR004,1997-02-16,-5.5,8.6,0.0\nR004,1997-02-17,-4.1,2.7,2102.54\nR004,1997-02-18,-3.1,0.3,19909.73\nR004,1997-02-19,-6.0,0.0,31064.95\nR004,1997-02-20,-6.9,0.0,34729.49\nR004,1997-02-21,-7.8,4.1,0.0\nR004,1997-02-22,-14.1,2.8,27359.9\nR004,1997-02-23,-15.7,2.8,2811.34\nR004,1997-02-24,-13.6,0.9,30781.71\nR004,1997-02-25,-13.5,0.0,18024.77\nR004,1997-02-26,-10.9,2.3,18423.81\nR004,1997-02-27,-10.7,4.5,9637.25\nR004,1997-02-28,-8.1,0.2,16165.25\nR004,1997-03-01,-4.8,0.0,38330.72\nR004,1997-03-02,-5.4,0.0,36312.1\nR004,1997-03-03,-4.2,0.3,23788.14\nR004,1997-03-04,-3.3,2.9,1679.41\nR004,1997-03-05,-3.7,0.6,28004.85\nR004,1997-03-06,-5.4,0.0,35172.64\nR004,1997-03-07,-3.1,4.2,1922.88\nR004,1997-03-08,-3.0,2.7,30550.51\nR004,1997-03-09,-6.0,0.0,28851.62\nR004,1997-03-10,-4.1,1.9,17819.63\nR004,1997-03-11,-3.8,1.0,23422.26\nR004,1997-03-12,-3.9,1.2,25565.57\nR004,1997-03-13,-3.6,0.2,39600.0\nR004,1997-03-14,-4.4,2.0,37900.5\nR004,1997-03-15,-4.9,1.5,26277.76\nR004,1997-03-16,-5.6,0.0,41270.31\nR004,1997-03-17,-3.2,0.0,41580.69\nR004,1997-03-18,-1.0,0.0,41901.92\nR004,1997-03-19,-1.4,0.0,41817.46\nR004,1997-03-20,-0.3,0.0,42188.58\nR004,1997-03-21,-1.2,0.0,42332.86\nR004,1997-03-22,-0.2,0.0,37932.71\nR004,1997-03-23,1.0,0.0,39177.74\nR004,1997-03-24,1.0,0.0,37646.52\nR004,1997-03-25,0.7,0.0,43326.33\nR004,1997-03-26,1.0,0.0,43418.3\nR004,1997-03-27,2.8,0.0,39706.62\nR004,1997-03-28,2.4,0.0,44217.31\nR004,1997-03-29,1.2,0.2,40265.64\nR004,1997-03-30,-1.7,0.5,570.12\nR004,1997-03-31,-1.0,0.0,44826.78\nR004,1997-04-01,1.4,0.0,41355.04\nR004,1997-04-02,4.8,1.0,41804.13\nR004,1997-04-03,2.3,1.6,43670.69\nR004,1997-04-04,3.3,3.3,1208.57\nR004,1997-04-05,-0.2,6.2,40893.89\nR004,1997-04-06,0.2,0.0,46003.49\nR004,1997-04-07,3.4,0.0,46427.54\nR004,1997-04-08,5.7,0.0,46852.94\nR004,1997-04-09,3.4,0.0,46892.13\nR004,1997-04-10,4.0,0.0,47708.12\nR004,1997-04-11,4.6,0.0,48138.21\nR004,1997-04-12,5.3,0.0,48440.19\nR004,1997-04-13,3.6,0.0,36827.48\nR004,1997-04-14,4.0,0.0,43751.6\nR004,1997-04-15,4.6,0.0,47038.79\nR004,1997-04-16,3.9,0.0,47723.74\nR004,1997-04-17,3.8,0.0,49730.24\nR004,1997-04-18,4.6,0.0,49864.47\nR004,1997-04-19,5.3,0.0,49997.51\nR004,1997-04-20,6.7,0.0,50129.24\nR004,1997-04-21,9.0,0.0,50259.5\nR004,1997-04-22,11.0,0.8,36106.48\nR004,1997-04-23,11.2,6.2,38622.52\nR004,1997-04-24,11.1,0.0,50400.0\nR004,1997-04-25,10.7,5.6,42023.99\nR004,1997-04-26,7.2,6.1,25077.66\nR004,1997-04-27,8.4,0.0,50400.0\nR004,1997-04-28,14.7,0.2,47300.68\nR004,1997-04-29,9.9,4.1,21741.96\nR004,1997-04-30,6.0,0.0,50400.0\nR004,1997-05-01,7.6,0.0,48485.84\nR004,1997-05-02,9.3,0.1,43156.39\nR004,1997-05-03,7.9,0.0,35672.73\nR004,1997-05-04,7.7,23.8,13322.85\nR004,1997-05-05,8.3,22.3,20795.21\nR004,1997-05-06,11.5,0.0,45187.89\nR004,1997-05-07,14.9,0.0,50905.8\nR004,1997-05-08,9.3,6.8,10004.45\nR004,1997-05-09,9.2,3.3,1557.77\nR004,1997-05-10,9.1,23.8,31951.5\nR004,1997-05-11,5.0,5.9,6886.91\nR004,1997-05-12,1.4,2.4,50839.44\nR004,1997-05-13,0.4,2.6,44993.34\nR004,1997-05-14,3.8,0.0,51705.44\nR004,1997-05-15,8.4,0.2,43977.01\nR004,1997-05-16,9.2,10.6,23247.51\nR004,1997-05-17,12.9,0.0,49087.44\nR004,1997-05-18,11.4,9.5,35347.44\nR004,1997-05-19,14.1,0.1,54256.28\nR004,1997-05-20,16.8,0.0,56052.5\nR004,1997-05-21,16.2,0.2,52870.94\nR004,1997-05-22,14.4,0.9,50400.0\nR004,1997-05-23,13.6,0.0,55518.51\nR004,1997-05-24,10.2,0.0,56394.81\nR004,1997-05-25,11.0,0.0,55465.12\nR004,1997-05-26,14.5,0.0,57484.85\nR004,1997-05-27,18.2,0.0,54960.13\nR004,1997-05-28,19.2,0.0,57600.0\nR004,1997-05-29,18.9,0.0,57600.0\nR004,1997-05-30,18.9,0.0,57600.0\nR004,1997-05-31,19.4,0.0,57600.0\nR004,1997-06-01,17.9,0.0,57600.0\nR004,1997-06-02,8.4,6.9,47555.07\nR004,1997-06-03,5.7,2.0,33369.29\nR004,1997-06-04,5.7,10.2,28096.62\nR004,1997-06-05,6.2,0.6,52590.63\nR004,1997-06-06,9.5,0.9,38139.12\nR004,1997-06-07,13.9,0.0,54870.32\nR004,1997-06-08,16.9,0.0,57600.0\nR004,1997-06-09,18.8,0.0,57600.0\nR004,1997-06-10,19.9,0.0,57553.55\nR004,1997-06-11,10.2,7.4,0.0\nR004,1997-06-12,8.4,3.9,48293.8\nR004,1997-06-13,7.8,5.1,51849.09\nR004,1997-06-14,12.9,0.5,20610.77\nR004,1997-06-15,15.7,15.6,39759.28\nR004,1997-06-16,16.5,0.0,52096.15\nR004,1997-06-17,20.5,0.7,56241.62\nR004,1997-06-18,16.0,7.9,36937.52\nR004,1997-06-19,10.3,0.0,56720.29\nR004,1997-06-20,15.2,0.0,57600.0\nR004,1997-06-21,17.6,0.0,52796.81\nR004,1997-06-22,17.1,0.7,35422.56\nR004,1997-06-23,19.3,0.0,55387.55\nR004,1997-06-24,20.6,2.2,57600.0\nR004,1997-06-25,18.6,4.4,41191.54\nR004,1997-06-26,12.4,36.7,0.0\nR004,1997-06-27,13.1,8.0,5329.47\nR004,1997-06-28,14.8,12.4,20459.02\nR004,1997-06-29,18.7,0.0,40851.9\nR004,1997-06-30,17.2,5.9,33862.31\nR004,1997-07-01,13.3,0.0,38249.82\nR004,1997-07-02,13.7,0.0,57278.9\nR004,1997-07-03,16.5,0.0,50400.0\nR004,1997-07-04,21.3,0.9,55535.43\nR004,1997-07-05,22.5,0.5,51649.87\nR004,1997-07-06,18.9,5.2,27574.05\nR004,1997-07-07,15.2,0.0,46982.98\nR004,1997-07-08,18.0,0.2,46690.7\nR004,1997-07-09,19.2,9.4,39081.73\nR004,1997-07-10,15.0,12.2,30857.89\nR004,1997-07-11,17.6,0.0,56943.44\nR004,1997-07-12,19.5,0.0,55190.22\nR004,1997-07-13,18.4,0.8,52143.37\nR004,1997-07-14,19.9,0.3,53965.29\nR004,1997-07-15,16.9,19.3,25542.65\nR004,1997-07-16,17.7,0.0,44260.1\nR004,1997-07-17,18.7,1.0,41371.96\nR004,1997-07-18,15.4,2.3,33352.42\nR004,1997-07-19,11.4,0.3,38325.19\nR004,1997-07-20,10.6,2.0,938.66\nR004,1997-07-21,11.4,4.9,51910.37\nR004,1997-07-22,13.0,0.0,54957.14\nR004,1997-07-23,15.1,0.0,54911.62\nR004,1997-07-24,16.0,0.7,46977.96\nR004,1997-07-25,11.6,9.5,3611.43\nR004,1997-07-26,10.0,6.5,12562.78\nR004,1997-07-27,9.5,11.2,0.0\nR004,1997-07-28,12.5,1.8,4427.87\nR004,1997-07-29,16.0,0.5,46626.91\nR004,1997-07-30,18.4,0.0,47012.36\nR004,1997-07-31,19.9,1.1,47570.58\nR004,1997-08-01,19.3,2.9,48952.35\nR004,1997-08-02,18.1,9.4,16195.22\nR004,1997-08-03,19.1,0.2,28871.75\nR004,1997-08-04,19.8,0.0,42401.39\nR004,1997-08-05,18.1,5.5,33652.81\nR004,1997-08-06,17.7,0.1,44827.51\nR004,1997-08-07,18.7,0.1,51988.92\nR004,1997-08-08,20.1,0.8,47995.04\nR004,1997-08-09,19.3,19.2,24005.21\nR004,1997-08-10,13.6,11.0,30686.29\nR004,1997-08-11,7.8,13.2,32577.41\nR004,1997-08-12,7.2,6.4,34273.08\nR004,1997-08-13,6.3,17.1,0.0\nR004,1997-08-14,12.8,12.8,22472.88\nR004,1997-08-15,16.7,0.0,38240.36\nR004,1997-08-16,18.2,0.0,50405.99\nR004,1997-08-17,20.0,0.0,39971.42\nR004,1997-08-18,17.0,13.4,598.13\nR004,1997-08-19,16.4,0.0,31017.85\nR004,1997-08-20,14.2,7.8,25658.2\nR004,1997-08-21,12.3,0.7,48331.79\nR004,1997-08-22,9.9,12.6,48553.07\nR004,1997-08-23,11.4,1.0,47453.16\nR004,1997-08-24,10.6,2.2,35730.26\nR004,1997-08-25,12.4,1.7,18745.53\nR004,1997-08-26,11.0,19.1,23290.57\nR004,1997-08-27,15.2,5.2,34831.99\nR004,1997-08-28,9.7,21.1,0.0\nR004,1997-08-29,7.6,9.4,13097.18\nR004,1997-08-30,8.9,2.0,16262.35\nR004,1997-08-31,12.0,5.5,30282.94\nR004,1997-09-01,13.2,0.0,29426.66\nR004,1997-09-02,10.3,8.0,13530.76\nR004,1997-09-03,6.0,0.1,18447.66\nR004,1997-09-04,6.3,9.7,4963.0\nR004,1997-09-05,8.2,3.4,34600.72\nR004,1997-09-06,5.8,6.8,13506.45\nR004,1997-09-07,5.1,11.0,42685.87\nR004,1997-09-08,6.1,1.1,43296.4\nR004,1997-09-09,7.0,0.1,43365.75\nR004,1997-09-10,7.6,0.0,43434.99\nR004,1997-09-11,7.4,0.0,43200.0\nR004,1997-09-12,8.5,0.2,41300.19\nR004,1997-09-13,9.7,0.0,43200.0\nR004,1997-09-14,9.6,0.0,43200.0\nR004,1997-09-15,10.0,0.0,43200.0\nR004,1997-09-16,10.9,0.0,43200.0\nR004,1997-09-17,11.0,0.0,43200.0\nR004,1997-09-18,10.4,0.0,43200.0\nR004,1997-09-19,11.4,0.3,43200.0\nR004,1997-09-20,9.8,4.1,30384.51\nR004,1997-09-21,9.9,2.2,37530.23\nR004,1997-09-22,9.7,0.0,43200.0\nR004,1997-09-23,10.9,0.0,43116.53\nR004,1997-09-24,11.8,0.0,42867.12\nR004,1997-09-25,13.0,0.0,42618.02\nR004,1997-09-26,12.1,0.0,38662.88\nR004,1997-09-27,8.5,0.0,41002.49\nR004,1997-09-28,6.0,0.0,33706.23\nR004,1997-09-29,3.0,0.0,35698.06\nR004,1997-09-30,1.2,0.0,39084.11\nR004,1997-10-01,0.5,0.0,37716.79\nR004,1997-10-02,2.8,0.0,37412.52\nR004,1997-10-03,4.4,0.0,36283.2\nR004,1997-10-04,6.7,0.0,36338.08\nR004,1997-10-05,6.3,0.0,36393.35\nR004,1997-10-06,6.9,0.0,34204.38\nR004,1997-10-07,9.9,0.0,36000.0\nR004,1997-10-08,11.5,0.0,36000.0\nR004,1997-10-09,8.0,1.0,336.71\nR004,1997-10-10,4.5,4.3,11643.06\nR004,1997-10-11,3.9,0.8,5035.3\nR004,1997-10-12,9.4,3.8,22388.33\nR004,1997-10-13,2.7,11.1,3152.1\nR004,1997-10-14,0.8,0.0,5075.02\nR004,1997-10-15,6.2,0.4,24957.97\nR004,1997-10-16,9.4,0.0,35117.54\nR004,1997-10-17,8.0,2.2,243.99\nR004,1997-10-18,10.9,3.2,0.0\nR004,1997-10-19,6.6,6.8,0.0\nR004,1997-10-20,3.6,3.0,3875.13\nR004,1997-10-21,-3.1,0.2,0.0\nR004,1997-10-22,-3.3,0.0,34140.92\nR004,1997-10-23,-1.4,0.0,33801.3\nR004,1997-10-24,-1.4,0.0,32870.02\nR004,1997-10-25,-0.7,0.0,33530.53\nR004,1997-10-26,-0.1,0.0,33184.22\nR004,1997-10-27,-0.4,0.0,32729.68\nR004,1997-10-28,-0.6,0.0,32683.42\nR004,1997-10-29,-0.9,0.0,32000.3\nR004,1997-10-30,-0.3,0.0,31713.61\nR004,1997-10-31,0.9,0.0,31144.38\nR004,1997-11-01,4.2,0.4,19358.29\nR004,1997-11-02,2.3,1.5,29559.57\nR004,1997-11-03,-1.4,0.4,29625.55\nR004,1997-11-04,-1.0,0.0,29691.53\nR004,1997-11-05,-2.8,0.0,29757.39\nR004,1997-11-06,-2.6,0.0,29823.08\nR004,1997-11-07,-0.3,6.4,0.0\nR004,1997-11-08,-5.4,0.3,15886.72\nR004,1997-11-09,-7.7,0.0,29128.91\nR004,1997-11-10,-6.4,1.2,2069.3\nR004,1997-11-11,-4.1,1.6,0.0\nR004,1997-11-12,-9.5,11.8,6167.05\nR004,1997-11-13,-18.9,0.1,16099.98\nR004,1997-11-14,-16.5,0.0,22695.76\nR004,1997-11-15,-11.1,0.0,15567.45\nR004,1997-11-16,-7.6,0.0,21991.2\nR004,1997-11-17,-8.4,0.0,28800.0\nR004,1997-11-18,-13.2,0.0,18941.69\nR004,1997-11-19,-15.7,0.0,18849.99\nR004,1997-11-20,-18.4,0.0,19522.11\nR004,1997-11-21,-10.2,2.8,0.0\nR004,1997-11-22,-9.8,8.1,0.0\nR004,1997-11-23,-28.4,0.0,22783.18\nR004,1997-11-24,-23.0,1.2,0.0\nR004,1997-11-25,-14.6,1.3,4728.42\nR004,1997-11-26,-10.9,8.6,0.0\nR004,1997-11-27,-22.1,0.8,25779.84\nR004,1997-11-28,-30.6,0.0,28230.42\nR004,1997-11-29,-34.0,0.0,28085.26\nR004,1997-11-30,-28.1,0.0,27054.96\nR004,1997-12-01,-25.1,0.0,25476.63\nR004,1997-12-02,-14.2,0.2,6501.92\nR004,1997-12-03,-15.1,4.8,3540.44\nR004,1997-12-04,-25.1,0.0,15333.8\nR004,1997-12-05,-24.4,0.0,25969.77\nR004,1997-12-06,-11.8,0.2,7042.85\nR004,1997-12-07,-14.4,0.8,22378.08\nR004,1997-12-08,-24.7,0.0,27016.89\nR004,1997-12-09,-26.4,0.0,26925.22\nR004,1997-12-10,-26.5,0.0,26839.64\nR004,1997-12-11,-24.1,0.0,24888.78\nR004,1997-12-12,-11.3,0.0,20110.71\nR004,1997-12-13,-7.2,5.8,0.0\nR004,1997-12-14,-8.7,2.1,8129.85\nR004,1997-12-15,-9.1,2.9,8297.72\nR004,1997-12-16,-15.8,3.4,18379.33\nR004,1997-12-17,-21.1,5.6,0.0\nR004,1997-12-18,-24.0,0.0,24270.66\nR004,1997-12-19,-12.0,2.1,4580.89\nR004,1997-12-20,-15.2,3.3,1436.04\nR004,1997-12-21,-15.5,1.3,24334.29\nR004,1997-12-22,-11.2,4.4,0.0\nR004,1997-12-23,-12.8,7.0,0.0\nR004,1997-12-24,-23.2,0.0,4601.78\nR004,1997-12-25,-23.3,1.5,0.0\nR004,1997-12-26,-14.8,0.3,9481.2\nR004,1997-12-27,-19.0,0.0,24670.6\nR004,1997-12-28,-13.1,2.2,0.0\nR004,1997-12-29,-7.7,4.6,0.0\nR004,1997-12-30,-7.7,8.5,0.0\nR004,1997-12-31,-8.4,9.5,0.0\nR004,1998-01-01,-24.1,0.0,17433.15\nR004,1998-01-02,-32.0,0.0,3060.64\nR004,1998-01-03,-18.8,0.0,11522.75\nR004,1998-01-04,-15.4,0.0,25144.66\nR004,1998-01-05,-24.1,0.0,25120.62\nR004,1998-01-06,-12.0,2.5,723.68\nR004,1998-01-07,-10.6,9.0,0.0\nR004,1998-01-08,-21.6,0.0,25200.0\nR004,1998-01-09,-31.3,0.0,26697.5\nR004,1998-01-10,-29.6,0.0,25200.0\nR004,1998-01-11,-27.4,0.0,25200.0\nR004,1998-01-12,-29.6,0.0,27107.12\nR004,1998-01-13,-34.2,0.0,27436.33\nR004,1998-01-14,-33.6,0.0,27488.85\nR004,1998-01-15,-33.2,0.0,27544.6\nR004,1998-01-16,-34.5,0.0,27613.15\nR004,1998-01-17,-33.3,0.0,27665.38\nR004,1998-01-18,-24.4,0.0,19490.64\nR004,1998-01-19,-17.5,0.0,24113.18\nR004,1998-01-20,-18.7,0.0,10419.58\nR004,1998-01-21,-15.1,0.0,27886.94\nR004,1998-01-22,-20.5,0.0,28632.56\nR004,1998-01-23,-21.4,0.0,27171.62\nR004,1998-01-24,-20.3,0.0,26582.57\nR004,1998-01-25,-17.7,1.4,4194.27\nR004,1998-01-26,-9.9,0.8,15651.7\nR004,1998-01-27,-15.7,0.0,6822.83\nR004,1998-01-28,-13.5,0.3,21700.54\nR004,1998-01-29,-20.4,0.0,28491.82\nR004,1998-01-30,-21.7,0.0,31177.17\nR004,1998-01-31,-21.4,0.0,31387.35\nR004,1998-02-01,-22.2,0.0,31510.09\nR004,1998-02-02,-20.2,0.0,31626.56\nR004,1998-02-03,-13.1,0.0,12279.4\nR004,1998-02-04,-11.6,1.5,1192.12\nR004,1998-02-05,-19.4,0.0,31980.11\nR004,1998-02-06,-20.7,0.0,32098.91\nR004,1998-02-07,-10.7,3.6,7295.22\nR004,1998-02-08,-10.5,1.1,10094.37\nR004,1998-02-09,-7.1,16.0,0.0\nR004,1998-02-10,-18.9,4.6,30869.88\nR004,1998-02-11,-22.9,0.6,0.0\nR004,1998-02-12,-11.1,3.5,140.33\nR004,1998-02-13,-9.9,0.0,25322.46\nR004,1998-02-14,-7.7,0.0,13928.6\nR004,1998-02-15,-5.0,0.9,18422.96\nR004,1998-02-16,-5.3,5.7,18700.89\nR004,1998-02-17,-16.2,0.9,32033.95\nR004,1998-02-18,-19.0,0.0,34260.54\nR004,1998-02-19,-15.3,0.0,32400.0\nR004,1998-02-20,-12.5,1.2,15152.54\nR004,1998-02-21,-12.6,0.0,35293.0\nR004,1998-02-22,-9.6,0.7,34945.04\nR004,1998-02-23,-12.8,1.5,33848.15\nR004,1998-02-24,-8.6,4.3,0.0\nR004,1998-02-25,-4.8,0.1,17628.67\nR004,1998-02-26,-5.3,0.0,36846.05\nR004,1998-02-27,-6.2,0.0,37713.27\nR004,1998-02-28,-5.7,0.0,37785.2\nR004,1998-03-01,-5.4,0.0,31053.82\nR004,1998-03-02,-6.5,0.0,38502.86\nR004,1998-03-03,-6.3,0.0,38616.51\nR004,1998-03-04,-5.8,0.0,38729.72\nR004,1998-03-05,-6.8,2.1,37714.04\nR004,1998-03-06,-5.9,0.6,38621.08\nR004,1998-03-07,-8.7,0.0,39066.34\nR004,1998-03-08,-7.5,0.0,37139.56\nR004,1998-03-09,-5.7,0.0,39287.9\nR004,1998-03-10,-5.9,0.0,33741.53\nR004,1998-03-11,-4.2,0.1,36252.1\nR004,1998-03-12,-7.1,0.0,40046.6\nR004,1998-03-13,-6.8,0.0,40358.85\nR004,1998-03-14,-9.0,1.7,25482.44\nR004,1998-03-15,-18.9,1.0,36156.58\nR004,1998-03-16,-22.1,0.0,39758.08\nR004,1998-03-17,-21.2,0.6,23411.43\nR004,1998-03-18,-14.7,0.0,37982.43\nR004,1998-03-19,-12.5,0.0,38172.69\nR004,1998-03-20,-9.4,0.0,39848.31\nR004,1998-03-21,-7.9,0.0,42305.59\nR004,1998-03-22,-5.7,0.0,42450.89\nR004,1998-03-23,-5.2,0.0,42595.99\nR004,1998-03-24,-1.6,0.2,31216.28\nR004,1998-03-25,-2.5,2.3,29345.11\nR004,1998-03-26,-5.4,0.7,5690.48\nR004,1998-03-27,-10.8,2.5,35339.86\nR004,1998-03-28,-12.5,0.0,43848.69\nR004,1998-03-29,-8.9,0.0,43796.56\nR004,1998-03-30,-5.7,0.0,44509.1\nR004,1998-03-31,-2.3,0.0,41373.88\nR004,1998-04-01,2.2,9.0,12715.28\nR004,1998-04-02,-0.2,5.9,3973.34\nR004,1998-04-03,0.0,0.0,44305.31\nR004,1998-04-04,1.6,1.0,4590.42\nR004,1998-04-05,3.7,0.5,33234.97\nR004,1998-04-06,2.2,10.5,5003.65\nR004,1998-04-07,-2.7,4.3,26374.19\nR004,1998-04-08,-2.0,5.5,6026.85\nR004,1998-04-09,-3.3,0.0,43880.83\nR004,1998-04-10,0.7,0.0,36765.35\nR004,1998-04-11,1.9,14.1,175.24\nR004,1998-04-12,1.0,10.4,15025.01\nR004,1998-04-13,-4.4,4.2,9179.14\nR004,1998-04-14,-1.4,11.4,21563.43\nR004,1998-04-15,-2.4,11.3,0.0\nR004,1998-04-16,5.1,24.6,5013.31\nR004,1998-04-17,-0.1,0.2,48105.1\nR004,1998-04-18,-5.4,9.8,0.0\nR004,1998-04-19,-4.1,6.1,36450.2\nR004,1998-04-20,-6.9,6.2,23909.47\nR004,1998-04-21,-8.0,0.2,42740.48\nR004,1998-04-22,-5.1,0.0,25165.41\nR004,1998-04-23,-1.4,1.5,18079.96\nR004,1998-04-24,-0.4,2.0,43364.14\nR004,1998-04-25,-0.4,0.8,36218.73\nR004,1998-04-26,2.9,2.9,37454.16\nR004,1998-04-27,2.8,5.9,6164.39\nR004,1998-04-28,5.2,6.3,34363.36\nR004,1998-04-29,2.2,4.3,20963.5\nR004,1998-04-30,6.9,0.2,40382.13\nR004,1998-05-01,5.6,0.0,47350.84\nR004,1998-05-02,3.2,3.7,24700.64\nR004,1998-05-03,1.1,9.2,14092.89\nR004,1998-05-04,1.1,7.6,27572.76\nR004,1998-05-05,-0.4,0.0,39680.16\nR004,1998-05-06,4.4,0.0,50537.81\nR004,1998-05-07,9.8,0.0,45272.26\nR004,1998-05-08,11.1,3.1,50165.05\nR004,1998-05-09,13.8,0.4,31470.6\nR004,1998-05-10,13.3,2.8,19979.79\nR004,1998-05-11,11.7,6.8,48750.27\nR004,1998-05-12,13.9,0.0,52596.92\nR004,1998-05-13,8.1,6.4,38950.07\nR004,1998-05-14,8.7,6.5,31626.33\nR004,1998-05-15,8.0,3.1,49102.19\nR004,1998-05-16,1.7,0.0,48124.71\nR004,1998-05-17,4.8,0.0,30356.4\nR004,1998-05-18,9.5,6.9,37236.96\nR004,1998-05-19,4.5,8.1,27530.47\nR004,1998-05-20,1.0,9.7,22688.97\nR004,1998-05-21,3.0,2.0,40258.44\nR004,1998-05-22,8.0,0.0,56001.76\nR004,1998-05-23,12.8,0.0,57252.19\nR004,1998-05-24,15.7,0.0,57328.05\nR004,1998-05-25,15.2,0.0,57401.24\nR004,1998-05-26,15.6,0.0,57471.67\nR004,1998-05-27,15.6,0.0,57539.22\nR004,1998-05-28,15.3,0.0,57600.0\nR004,1998-05-29,13.1,8.0,36932.61\nR004,1998-05-30,13.1,5.6,42782.05\nR004,1998-05-31,15.5,1.5,44754.19\nR004,1998-06-01,18.0,0.0,55747.06\nR004,1998-06-02,18.7,0.0,57600.0\nR004,1998-06-03,18.2,7.8,37769.28\nR004,1998-06-04,11.6,0.3,56943.64\nR004,1998-06-05,10.7,0.0,57600.0\nR004,1998-06-06,13.5,0.0,57600.0\nR004,1998-06-07,17.7,0.0,57600.0\nR004,1998-06-08,19.7,0.0,57600.0\nR004,1998-06-09,17.9,1.5,52270.3\nR004,1998-06-10,17.2,2.7,43880.13\nR004,1998-06-11,11.2,23.1,0.0\nR004,1998-06-12,12.3,0.0,50307.86\nR004,1998-06-13,14.3,2.9,57595.54\nR004,1998-06-14,15.6,0.5,47749.7\nR004,1998-06-15,14.0,0.0,53234.69\nR004,1998-06-16,9.9,3.0,27495.75\nR004,1998-06-17,14.6,0.0,57600.0\nR004,1998-06-18,14.5,0.0,57600.0\nR004,1998-06-19,16.3,0.1,55259.71\nR004,1998-06-20,15.5,14.2,46176.1\nR004,1998-06-21,10.3,4.9,22208.68\nR004,1998-06-22,7.8,8.7,27645.38\nR004,1998-06-23,8.1,6.4,49871.64\nR004,1998-06-24,9.4,0.3,49488.79\nR004,1998-06-25,9.9,2.5,37180.22\nR004,1998-06-26,12.1,1.7,45956.81\nR004,1998-06-27,14.0,1.8,51908.6\nR004,1998-06-28,16.0,0.0,57600.0\nR004,1998-06-29,17.0,1.9,45434.93\nR004,1998-06-30,19.7,1.3,50323.96\nR004,1998-07-01,16.2,5.7,14863.45\nR004,1998-07-02,13.9,9.9,40871.66\nR004,1998-07-03,15.2,0.0,50864.93\nR004,1998-07-04,18.6,0.0,54539.87\nR004,1998-07-05,19.7,0.1,57454.45\nR004,1998-07-06,21.0,0.0,55686.24\nR004,1998-07-07,21.8,0.0,57600.0\nR004,1998-07-08,22.7,0.0,57600.0\nR004,1998-07-09,23.1,0.3,56675.78\nR004,1998-07-10,23.2,0.0,57254.37\nR004,1998-07-11,22.4,0.0,41881.35\nR004,1998-07-12,17.7,7.6,4747.73\nR004,1998-07-13,17.5,4.3,32584.16\nR004,1998-07-14,17.3,15.4,17929.27\nR004,1998-07-15,19.0,1.1,51951.55\nR004,1998-07-16,19.8,0.0,55671.6\nR004,1998-07-17,20.1,0.0,54816.24\nR004,1998-07-18,19.3,1.2,40873.27\nR004,1998-07-19,16.8,15.0,35495.5\nR004,1998-07-20,16.2,0.6,43520.79\nR004,1998-07-21,15.2,12.1,27752.48\nR004,1998-07-22,16.7,3.6,31795.58\nR004,1998-07-23,17.1,13.6,8471.47\nR004,1998-07-24,18.1,0.0,48996.35\nR004,1998-07-25,17.3,0.0,54612.17\nR004,1998-07-26,17.3,0.0,54650.05\nR004,1998-07-27,17.9,0.0,54243.78\nR004,1998-07-28,17.6,0.0,51918.23\nR004,1998-07-29,17.0,0.0,49950.32\nR004,1998-07-30,17.3,0.4,43157.93\nR004,1998-07-31,18.6,0.0,53882.71\nR004,1998-08-01,18.6,0.0,53097.68\nR004,1998-08-02,17.9,0.0,53465.39\nR004,1998-08-03,16.3,0.0,43489.15\nR004,1998-08-04,18.0,0.1,53169.53\nR004,1998-08-05,19.4,0.0,51886.89\nR004,1998-08-06,19.1,0.4,47228.12\nR004,1998-08-07,19.1,0.0,52400.27\nR004,1998-08-08,19.7,0.0,52446.56\nR004,1998-08-09,21.3,0.0,52075.74\nR004,1998-08-10,22.1,0.0,51708.36\nR004,1998-08-11,22.9,0.0,50929.05\nR004,1998-08-12,23.1,0.0,50983.49\nR004,1998-08-13,22.1,0.0,50625.77\nR004,1998-08-14,21.1,0.0,46440.65\nR004,1998-08-15,22.0,0.0,50400.0\nR004,1998-08-16,21.7,3.1,39152.82\nR004,1998-08-17,16.1,44.4,21600.0\nR004,1998-08-18,16.1,1.5,50400.0\nR004,1998-08-19,17.4,1.4,44505.85\nR004,1998-08-20,15.9,3.0,43046.78\nR004,1998-08-21,12.6,8.1,13472.18\nR004,1998-08-22,11.4,0.1,21538.99\nR004,1998-08-23,12.3,0.1,44983.73\nR004,1998-08-24,12.4,2.5,36712.81\nR004,1998-08-25,12.3,6.6,47233.92\nR004,1998-08-26,13.2,0.0,46957.28\nR004,1998-08-27,15.5,0.8,45764.74\nR004,1998-08-28,15.1,0.0,48621.77\nR004,1998-08-29,16.7,2.9,42857.07\nR004,1998-08-30,15.3,12.2,44285.46\nR004,1998-08-31,14.5,0.0,37657.74\nR004,1998-09-01,15.3,4.4,25361.66\nR004,1998-09-02,14.9,6.1,30325.92\nR004,1998-09-03,15.4,24.1,20678.65\nR004,1998-09-04,13.6,1.3,23990.23\nR004,1998-09-05,15.2,0.0,30136.76\nR004,1998-09-06,11.6,2.9,43906.59\nR004,1998-09-07,5.4,2.5,33762.21\nR004,1998-09-08,4.8,0.0,44050.5\nR004,1998-09-09,5.7,0.0,43727.49\nR004,1998-09-10,7.2,0.2,29230.98\nR004,1998-09-11,7.8,1.5,37159.78\nR004,1998-09-12,7.5,0.2,26613.64\nR004,1998-09-13,8.7,0.0,43200.0\nR004,1998-09-14,11.4,0.8,36000.0\nR004,1998-09-15,8.1,11.1,16407.74\nR004,1998-09-16,0.8,2.4,29500.02\nR004,1998-09-17,3.2,0.9,32892.73\nR004,1998-09-18,8.3,0.0,42289.9\nR004,1998-09-19,11.3,0.0,43200.0\nR004,1998-09-20,10.8,0.0,39600.0\nR004,1998-09-21,9.3,0.4,34576.29\nR004,1998-09-22,4.4,2.8,6098.85\nR004,1998-09-23,0.6,0.0,7794.09\nR004,1998-09-24,1.8,0.9,12371.46\nR004,1998-09-25,11.8,0.0,33292.89\nR004,1998-09-26,13.7,1.1,33407.25\nR004,1998-09-27,10.1,0.6,30698.35\nR004,1998-09-28,2.4,8.3,29295.52\nR004,1998-09-29,-0.4,1.2,22202.64\nR004,1998-09-30,0.0,1.4,18697.69\nR004,1998-10-01,2.3,10.4,0.0\nR004,1998-10-02,6.4,2.3,65.33\nR004,1998-10-03,9.8,0.0,26851.42\nR004,1998-10-04,1.0,0.9,23381.45\nR004,1998-10-05,4.7,6.0,15673.89\nR004,1998-10-06,3.1,0.6,43.58\nR004,1998-10-07,2.8,0.0,33349.94\nR004,1998-10-08,6.6,0.0,31916.23\nR004,1998-10-09,3.7,9.8,0.0\nR004,1998-10-10,0.9,0.0,17284.99\nR004,1998-10-11,0.8,1.9,5742.2\nR004,1998-10-12,1.7,2.2,24825.72\nR004,1998-10-13,4.7,2.0,431.87\nR004,1998-10-14,3.2,11.7,0.0\nR004,1998-10-15,-4.5,0.0,28912.46\nR004,1998-10-16,0.2,13.4,0.0\nR004,1998-10-17,5.9,1.0,15875.49\nR004,1998-10-18,6.6,0.3,33707.12\nR004,1998-10-19,2.6,6.1,0.0\nR004,1998-10-20,2.1,3.4,22831.8\nR004,1998-10-21,6.6,0.4,26535.49\nR004,1998-10-22,2.4,8.2,18469.44\nR004,1998-10-23,-0.3,0.0,33783.67\nR004,1998-10-24,-0.2,0.0,33857.84\nR004,1998-10-25,0.8,6.1,1967.85\nR004,1998-10-26,-1.6,10.0,0.0\nR004,1998-10-27,-1.5,4.5,0.0\nR004,1998-10-28,-2.3,11.3,0.0\nR004,1998-10-29,-12.2,0.1,16055.84\nR004,1998-10-30,-2.6,16.5,0.0\nR004,1998-10-31,3.1,5.9,95.83\nR004,1998-11-01,3.4,0.1,4176.01\nR004,1998-11-02,2.3,3.3,23126.49\nR004,1998-11-03,-1.8,0.5,7787.64\nR004,1998-11-04,-2.0,3.1,5434.03\nR004,1998-11-05,-5.9,0.2,13017.98\nR004,1998-11-06,0.6,24.0,0.0\nR004,1998-11-07,2.0,14.0,0.0\nR004,1998-11-08,1.7,0.0,21604.12\nR004,1998-11-09,-1.9,0.0,29110.21\nR004,1998-11-10,0.2,2.3,23705.12\nR004,1998-11-11,-3.5,0.9,26103.43\nR004,1998-11-12,-10.0,1.0,0.0\nR004,1998-11-13,-11.5,2.9,0.0\nR004,1998-11-14,-17.3,0.3,9773.47\nR004,1998-11-15,-17.8,0.0,6314.54\nR004,1998-11-16,-20.8,0.0,28800.0\nR004,1998-11-17,-21.9,0.0,22187.86\nR004,1998-11-18,-13.3,1.6,13407.5\nR004,1998-11-19,-12.6,0.0,2141.92\nR004,1998-11-20,-17.3,0.1,22938.39\nR004,1998-11-21,-7.2,7.1,0.0\nR004,1998-11-22,-9.3,1.1,88.18\nR004,1998-11-23,-13.9,0.6,7066.49\nR004,1998-11-24,-15.7,1.2,0.0\nR004,1998-11-25,-17.5,0.0,10733.14\nR004,1998-11-26,-20.3,0.8,10737.9\nR004,1998-11-27,-24.4,1.5,19447.69\nR004,1998-11-28,-35.2,0.0,28279.77\nR004,1998-11-29,-33.9,0.0,28136.84\nR004,1998-11-30,-32.1,0.0,13095.71\nR004,1998-12-01,-21.2,2.5,0.0\nR004,1998-12-02,-14.7,1.3,0.0\nR004,1998-12-03,-14.7,0.3,12896.13\nR004,1998-12-04,-11.7,4.9,0.0\nR004,1998-12-05,-8.4,7.2,0.0\nR004,1998-12-06,-9.1,2.0,0.0\nR004,1998-12-07,-14.5,0.0,17117.65\nR004,1998-12-08,-15.1,0.0,12230.11\nR004,1998-12-09,-13.5,0.0,24735.38\nR004,1998-12-10,-14.4,1.1,6277.56\nR004,1998-12-11,-5.9,6.3,0.0\nR004,1998-12-12,-6.9,0.4,14102.7\nR004,1998-12-13,-7.3,0.0,20690.88\nR004,1998-12-14,-14.2,0.0,26212.13\nR004,1998-12-15,-16.8,1.4,489.56\nR004,1998-12-16,-9.3,9.4,0.0\nR004,1998-12-17,-12.4,3.8,431.23\nR004,1998-12-18,-8.9,3.4,0.0\nR004,1998-12-19,-8.5,3.6,84.64\nR004,1998-12-20,-4.2,8.1,0.0\nR004,1998-12-21,-4.7,7.2,0.0\nR004,1998-12-22,-7.0,0.0,21650.98\nR004,1998-12-23,-8.0,0.0,17286.25\nR004,1998-12-24,-12.9,0.0,25034.19\nR004,1998-12-25,-7.8,3.1,4328.47\nR004,1998-12-26,-18.6,1.5,8282.03\nR004,1998-12-27,-20.9,0.0,2742.93\nR004,1998-12-28,-21.7,0.0,15313.09\nR004,1998-12-29,-12.6,1.8,13665.52\nR004,1998-12-30,-8.9,5.0,0.0\nR004,1998-12-31,-4.5,1.5,0.0\nR004,1999-01-01,-6.0,0.0,3543.01\nR004,1999-01-02,-4.8,7.5,0.0\nR004,1999-01-03,-15.0,1.9,869.27\nR004,1999-01-04,-24.4,0.0,17548.12\nR004,1999-01-05,-21.6,0.0,5897.27\nR004,1999-01-06,-20.7,0.0,14359.57\nR004,1999-01-07,-25.1,0.0,26242.93\nR004,1999-01-08,-27.7,0.0,26728.22\nR004,1999-01-09,-26.3,0.0,25689.64\nR004,1999-01-10,-24.8,0.0,26673.5\nR004,1999-01-11,-24.5,0.0,27145.61\nR004,1999-01-12,-24.4,0.0,27111.57\nR004,1999-01-13,-24.4,0.0,27430.89\nR004,1999-01-14,-23.7,0.0,27038.04\nR004,1999-01-15,-19.6,0.1,15167.13\nR004,1999-01-16,-17.2,3.0,1669.35\nR004,1999-01-17,-19.8,0.0,17798.96\nR004,1999-01-18,-18.9,0.5,22946.65\nR004,1999-01-19,-10.8,3.3,0.0\nR004,1999-01-20,-7.6,7.8,0.0\nR004,1999-01-21,-8.1,2.0,3106.61\nR004,1999-01-22,-8.3,3.8,0.0\nR004,1999-01-23,-13.9,4.5,3811.37\nR004,1999-01-24,-18.2,9.5,0.0\nR004,1999-01-25,-10.8,8.5,0.0\nR004,1999-01-26,-14.8,1.0,19384.79\nR004,1999-01-27,-12.5,4.5,1263.72\nR004,1999-01-28,-12.1,1.9,931.99\nR004,1999-01-29,-6.8,11.6,0.0\nR004,1999-01-30,-11.6,4.2,11738.98\nR004,1999-01-31,-16.3,3.0,0.0\nR004,1999-02-01,-7.7,2.3,23650.23\nR004,1999-02-02,-11.5,0.0,31600.17\nR004,1999-02-03,-15.8,0.0,31718.61\nR004,1999-02-04,-14.9,0.0,31837.33\nR004,1999-02-05,-15.9,0.0,27035.19\nR004,1999-02-06,-12.3,0.8,9742.71\nR004,1999-02-07,-13.5,0.0,32193.92\nR004,1999-02-08,-13.8,0.0,30887.68\nR004,1999-02-09,-15.4,0.0,32400.0\nR004,1999-02-10,-15.1,0.0,31824.27\nR004,1999-02-11,-11.0,0.0,31155.45\nR004,1999-02-12,-5.0,3.6,12760.05\nR004,1999-02-13,-5.2,0.2,20361.82\nR004,1999-02-14,-7.7,0.0,25280.13\nR004,1999-02-15,-9.8,7.1,15275.93\nR004,1999-02-16,-15.2,0.4,30225.04\nR004,1999-02-17,-16.1,0.0,32400.0\nR004,1999-02-18,-15.2,0.0,33869.45\nR004,1999-02-19,-14.1,0.0,34419.23\nR004,1999-02-20,-13.2,0.0,34284.75\nR004,1999-02-21,-11.8,0.0,35281.26\nR004,1999-02-22,-11.9,0.0,35351.01\nR004,1999-02-23,-11.5,0.0,33978.17\nR004,1999-02-24,-5.0,0.0,34141.6\nR004,1999-02-25,-5.2,0.0,35966.72\nR004,1999-02-26,-6.7,0.0,36836.61\nR004,1999-02-27,-6.0,5.8,0.0\nR004,1999-02-28,-7.3,1.5,2007.39\nR004,1999-03-01,-11.1,0.0,38223.99\nR004,1999-03-02,-10.0,4.4,0.0\nR004,1999-03-03,-6.3,9.1,514.11\nR004,1999-03-04,-16.0,2.3,36283.2\nR004,1999-03-05,-23.0,0.0,38811.46\nR004,1999-03-06,-24.5,0.0,38922.77\nR004,1999-03-07,-24.6,0.0,39034.04\nR004,1999-03-08,-23.8,0.0,39145.22\nR004,1999-03-09,-22.0,0.0,39245.22\nR004,1999-03-10,-20.4,3.4,2352.52\nR004,1999-03-11,-15.1,3.8,27644.26\nR004,1999-03-12,-16.0,0.0,31647.94\nR004,1999-03-13,-15.4,0.0,30819.5\nR004,1999-03-14,-15.6,0.0,34444.43\nR004,1999-03-15,-15.4,1.4,14603.86\nR004,1999-03-16,-21.1,0.0,36458.99\nR004,1999-03-17,-23.5,0.0,40463.86\nR004,1999-03-18,-22.9,0.0,41843.76\nR004,1999-03-19,-20.3,0.0,41988.25\nR004,1999-03-20,-13.6,0.0,40626.61\nR004,1999-03-21,-8.7,0.0,42276.21\nR004,1999-03-22,-3.6,1.3,17969.28\nR004,1999-03-23,-11.8,6.7,272.71\nR004,1999-03-24,-17.4,0.0,40833.5\nR004,1999-03-25,-16.0,0.0,42844.18\nR004,1999-03-26,-10.4,0.0,39600.0\nR004,1999-03-27,-9.7,0.0,43460.49\nR004,1999-03-28,-7.4,0.0,43200.0\nR004,1999-03-29,-5.7,0.0,44192.39\nR004,1999-03-30,-6.2,0.0,29840.62\nR004,1999-03-31,-4.3,0.6,40870.83\nR004,1999-04-01,-1.3,2.6,11508.24\nR004,1999-04-02,-2.0,6.6,17968.61\nR004,1999-04-03,0.0,10.3,5106.04\nR004,1999-04-04,2.5,0.2,40446.6\nR004,1999-04-05,4.2,1.2,20851.55\nR004,1999-04-06,2.8,3.5,28048.64\nR004,1999-04-07,1.2,4.0,37170.76\nR004,1999-04-08,-2.9,0.2,46444.14\nR004,1999-04-09,-0.7,0.0,46481.99\nR004,1999-04-10,0.1,0.0,42048.1\nR004,1999-04-11,2.7,0.0,8676.62\nR004,1999-04-12,3.9,7.5,15989.36\nR004,1999-04-13,2.7,0.1,42665.89\nR004,1999-04-14,5.2,0.0,43133.95\nR004,1999-04-15,3.3,0.0,36700.18\nR004,1999-04-16,-2.2,0.0,45383.5\nR004,1999-04-17,-1.4,1.5,28979.96\nR004,1999-04-18,0.2,0.0,27617.34\nR004,1999-04-19,-5.4,1.0,49921.93\nR004,1999-04-20,-3.9,3.3,22113.38\nR004,1999-04-21,-2.2,12.8,0.0\nR004,1999-04-22,-2.8,7.0,35910.94\nR004,1999-04-23,-4.7,0.6,27574.5\nR004,1999-04-24,-2.5,4.5,0.0\nR004,1999-04-25,1.1,6.5,0.0\nR004,1999-04-26,4.8,4.3,41042.7\nR004,1999-04-27,6.2,13.9,35075.65\nR004,1999-04-28,-0.3,0.0,50400.0\nR004,1999-04-29,1.4,0.0,50466.38\nR004,1999-04-30,4.0,0.0,50118.21\nR004,1999-05-01,7.4,0.0,49148.1\nR004,1999-05-02,8.6,0.0,50737.57\nR004,1999-05-03,9.5,0.0,50690.99\nR004,1999-05-04,11.3,0.0,49939.81\nR004,1999-05-05,11.7,0.0,51020.21\nR004,1999-05-06,14.3,0.0,51442.4\nR004,1999-05-07,14.2,4.9,43200.0\nR004,1999-05-08,8.4,0.2,39776.62\nR004,1999-05-09,3.1,16.3,0.0\nR004,1999-05-10,4.2,0.0,48484.91\nR004,1999-05-11,5.8,0.0,29966.01\nR004,1999-05-12,6.3,0.0,52615.15\nR004,1999-05-13,8.6,0.0,53834.95\nR004,1999-05-14,14.4,0.0,53320.66\nR004,1999-05-15,14.5,0.0,45995.18\nR004,1999-05-16,16.6,0.0,36114.17\nR004,1999-05-17,15.7,1.2,44603.39\nR004,1999-05-18,13.3,1.4,47158.79\nR004,1999-05-19,11.7,0.1,34067.73\nR004,1999-05-20,14.0,0.0,56441.21\nR004,1999-05-21,16.5,0.0,56481.95\nR004,1999-05-22,17.0,0.0,56903.17\nR004,1999-05-23,18.4,0.0,57236.42\nR004,1999-05-24,18.2,0.0,57311.26\nR004,1999-05-25,15.3,2.1,52717.29\nR004,1999-05-26,13.6,10.0,26865.51\nR004,1999-05-27,12.4,4.6,11850.65\nR004,1999-05-28,11.8,14.0,30581.88\nR004,1999-05-29,13.7,1.6,47495.44\nR004,1999-05-30,15.9,0.0,57600.0\nR004,1999-05-31,18.4,0.2,55897.21\nR004,1999-06-01,18.8,0.5,54000.0\nR004,1999-06-02,14.1,0.1,18026.11\nR004,1999-06-03,12.5,9.6,37124.53\nR004,1999-06-04,13.1,0.2,53661.94\nR004,1999-06-05,16.2,0.3,50400.0\nR004,1999-06-06,15.0,7.4,35773.18\nR004,1999-06-07,16.1,0.0,57600.0\nR004,1999-06-08,14.6,8.0,17274.55\nR004,1999-06-09,15.3,10.1,51267.29\nR004,1999-06-10,10.4,13.4,30136.48\nR004,1999-06-11,5.0,6.3,50153.67\nR004,1999-06-12,4.4,0.1,56096.57\nR004,1999-06-13,7.5,0.0,57600.0\nR004,1999-06-14,13.7,0.0,57600.0\nR004,1999-06-15,17.1,0.0,57600.0\nR004,1999-06-16,13.8,4.6,43179.55\nR004,1999-06-17,9.2,3.7,38294.74\nR004,1999-06-18,7.6,5.6,49371.63\nR004,1999-06-19,10.1,5.1,57403.55\nR004,1999-06-20,13.1,0.0,57071.53\nR004,1999-06-21,16.4,7.5,50226.28\nR004,1999-06-22,17.1,0.4,47569.2\nR004,1999-06-23,19.3,0.5,57303.93\nR004,1999-06-24,19.4,0.0,57600.0\nR004,1999-06-25,19.3,0.0,57600.0\nR004,1999-06-26,16.7,0.3,45071.84\nR004,1999-06-27,15.7,3.2,39412.62\nR004,1999-06-28,16.7,0.0,57600.0\nR004,1999-06-29,15.0,4.3,35021.06\nR004,1999-06-30,12.3,5.9,41127.57\nR004,1999-07-01,12.2,5.2,36515.26\nR004,1999-07-02,13.7,3.0,34377.63\nR004,1999-07-03,15.5,2.4,55023.92\nR004,1999-07-04,16.9,0.0,50885.89\nR004,1999-07-05,18.1,0.0,52021.92\nR004,1999-07-06,19.5,0.0,57600.0\nR004,1999-07-07,20.5,0.0,57600.0\nR004,1999-07-08,21.5,0.0,57600.0\nR004,1999-07-09,22.4,0.0,57595.24\nR004,1999-07-10,21.3,1.9,49622.55\nR004,1999-07-11,21.0,3.8,44515.59\nR004,1999-07-12,21.0,3.9,37266.1\nR004,1999-07-13,21.3,3.5,44116.04\nR004,1999-07-14,22.2,0.9,47603.97\nR004,1999-07-15,18.9,8.2,2794.32\nR004,1999-07-16,22.7,0.0,46467.23\nR004,1999-07-17,23.5,0.0,52822.98\nR004,1999-07-18,20.9,5.4,44467.88\nR004,1999-07-19,19.3,10.4,35166.02\nR004,1999-07-20,16.4,10.5,28124.32\nR004,1999-07-21,19.3,0.0,51725.33\nR004,1999-07-22,21.2,0.0,48236.12\nR004,1999-07-23,19.5,12.2,4845.77\nR004,1999-07-24,17.3,3.2,11454.89\nR004,1999-07-25,19.3,5.2,48730.86\nR004,1999-07-26,18.5,14.7,30425.84\nR004,1999-07-27,18.2,6.2,26355.82\nR004,1999-07-28,17.2,12.4,39608.06\nR004,1999-07-29,18.7,0.0,49946.38\nR004,1999-07-30,20.0,0.0,54000.0\nR004,1999-07-31,21.6,0.0,53902.19\nR004,1999-08-01,21.6,0.0,51706.74\nR004,1999-08-02,19.7,0.1,33127.1\nR004,1999-08-03,18.9,7.3,46735.57\nR004,1999-08-04,17.0,19.1,17483.17\nR004,1999-08-05,15.7,45.0,3183.05\nR004,1999-08-06,11.7,15.3,48849.76\nR004,1999-08-07,8.4,12.9,31306.33\nR004,1999-08-08,10.5,1.8,43922.36\nR004,1999-08-09,12.0,0.0,52063.25\nR004,1999-08-10,11.7,4.3,19067.99\nR004,1999-08-11,12.1,7.3,29396.24\nR004,1999-08-12,11.3,1.7,47241.3\nR004,1999-08-13,10.8,0.0,50607.63\nR004,1999-08-14,11.0,2.8,50400.0\nR004,1999-08-15,11.7,0.0,49357.06\nR004,1999-08-16,11.4,6.3,20314.81\nR004,1999-08-17,14.4,12.5,47763.87\nR004,1999-08-18,16.5,0.0,42155.54\nR004,1999-08-19,18.0,0.0,50372.81\nR004,1999-08-20,20.5,0.0,50271.69\nR004,1999-08-21,17.2,4.0,36050.17\nR004,1999-08-22,14.7,0.2,29582.54\nR004,1999-08-23,17.2,0.0,48395.53\nR004,1999-08-24,19.6,0.0,49865.81\nR004,1999-08-25,15.5,0.0,47868.81\nR004,1999-08-26,14.0,0.0,49662.71\nR004,1999-08-27,15.8,0.0,48961.48\nR004,1999-08-28,15.7,0.9,39960.01\nR004,1999-08-29,11.3,20.1,0.0\nR004,1999-08-30,12.1,0.1,805.78\nR004,1999-08-31,13.1,0.0,44808.72\nR004,1999-09-01,13.1,0.0,46311.03\nR004,1999-09-02,13.6,0.0,41830.82\nR004,1999-09-03,15.0,0.6,45369.49\nR004,1999-09-04,13.2,4.8,36624.0\nR004,1999-09-05,9.7,2.5,24376.35\nR004,1999-09-06,11.4,3.0,25982.88\nR004,1999-09-07,8.8,14.5,18946.92\nR004,1999-09-08,5.0,10.4,10231.14\nR004,1999-09-09,4.3,6.2,28918.2\nR004,1999-09-10,3.9,4.3,24155.97\nR004,1999-09-11,5.2,0.0,38125.4\nR004,1999-09-12,5.4,0.0,43200.0\nR004,1999-09-13,8.9,0.0,43232.28\nR004,1999-09-14,9.1,0.0,43200.0\nR004,1999-09-15,8.5,0.0,43200.0\nR004,1999-09-16,6.4,0.0,43200.0\nR004,1999-09-17,7.0,0.0,43200.0\nR004,1999-09-18,10.1,0.0,43200.0\nR004,1999-09-19,10.7,0.0,43200.0\nR004,1999-09-20,12.8,0.3,39933.24\nR004,1999-09-21,11.5,7.3,28956.35\nR004,1999-09-22,6.6,21.6,28573.65\nR004,1999-09-23,6.4,1.3,18750.88\nR004,1999-09-24,5.1,11.1,21217.03\nR004,1999-09-25,4.1,3.6,18361.9\nR004,1999-09-26,2.0,6.2,19998.27\nR004,1999-09-27,1.4,3.9,25932.96\nR004,1999-09-28,2.9,0.5,29506.05\nR004,1999-09-29,2.6,14.3,0.0\nR004,1999-09-30,1.3,0.0,395.37\nR004,1999-10-01,0.6,0.0,36919.74\nR004,1999-10-02,4.0,0.0,33467.81\nR004,1999-10-03,7.0,3.0,30585.62\nR004,1999-10-04,10.1,7.0,22148.02\nR004,1999-10-05,10.0,7.9,23422.16\nR004,1999-10-06,8.5,4.2,0.0\nR004,1999-10-07,6.8,7.6,0.0\nR004,1999-10-08,5.1,23.1,0.0\nR004,1999-10-09,1.0,3.4,26221.05\nR004,1999-10-10,1.9,30.9,0.0\nR004,1999-10-11,-3.1,0.0,29589.47\nR004,1999-10-12,-4.2,0.0,31796.97\nR004,1999-10-13,-2.1,2.6,0.0\nR004,1999-10-14,-1.6,0.0,34028.49\nR004,1999-10-15,-0.5,0.0,36000.0\nR004,1999-10-16,2.8,0.0,35281.02\nR004,1999-10-17,3.6,0.0,32341.51\nR004,1999-10-18,5.9,0.0,32790.49\nR004,1999-10-19,6.2,0.0,25614.39\nR004,1999-10-20,6.2,0.0,17566.69\nR004,1999-10-21,2.7,8.1,3624.14\nR004,1999-10-22,0.0,1.2,23567.13\nR004,1999-10-23,-1.7,3.4,13138.11\nR004,1999-10-24,-3.4,2.7,371.44\nR004,1999-10-25,-2.3,0.8,0.0\nR004,1999-10-26,-7.1,7.1,14053.68\nR004,1999-10-27,-17.1,0.0,33171.76\nR004,1999-10-28,-6.3,0.0,21600.0\nR004,1999-10-29,-1.8,0.0,17581.91\nR004,1999-10-30,-4.6,0.0,30369.21\nR004,1999-10-31,-5.7,0.0,27158.43\nR004,1999-11-01,1.0,0.0,24805.52\nR004,1999-11-02,1.5,9.7,0.0\nR004,1999-11-03,0.9,7.6,0.0\nR004,1999-11-04,-0.4,1.5,15656.92\nR004,1999-11-05,-2.4,0.0,28868.81\nR004,1999-11-06,-3.5,0.0,26019.56\nR004,1999-11-07,-4.3,0.0,27170.37\nR004,1999-11-08,-2.1,0.0,23241.64\nR004,1999-11-09,-2.8,3.2,0.0\nR004,1999-11-10,-9.0,0.6,13638.93\nR004,1999-11-11,-9.9,2.0,23181.73\nR004,1999-11-12,-8.3,0.9,5464.05\nR004,1999-11-13,-8.3,0.8,26690.76\nR004,1999-11-14,-10.5,1.9,25200.0\nR004,1999-11-15,-6.6,0.3,24522.82\nR004,1999-11-16,-3.4,10.0,0.0\nR004,1999-11-17,-0.7,0.0,28647.86\nR004,1999-11-18,-9.0,0.0,28800.0\nR004,1999-11-19,-8.4,1.2,21102.17\nR004,1999-11-20,-15.4,0.1,4298.3\nR004,1999-11-21,-11.5,5.0,1053.34\nR004,1999-11-22,-14.8,6.3,8000.77\nR004,1999-11-23,-22.7,0.0,17580.65\nR004,1999-11-24,-30.3,0.0,28543.49\nR004,1999-11-25,-26.4,0.0,28477.72\nR004,1999-11-26,-23.0,0.0,28414.62\nR004,1999-11-27,-18.4,0.0,28354.27\nR004,1999-11-28,-19.0,0.0,28296.75\nR004,1999-11-29,-20.6,0.0,28168.74\nR004,1999-11-30,-16.6,0.2,20971.93\nR004,1999-12-01,-7.7,2.4,548.34\nR004,1999-12-02,-9.5,1.6,162.6\nR004,1999-12-03,-7.1,1.0,12671.09\nR004,1999-12-04,-8.4,0.0,24313.07\nR004,1999-12-05,-16.0,0.0,24398.51\nR004,1999-12-06,-10.5,0.6,15401.91\nR004,1999-12-07,-3.4,13.9,0.0\nR004,1999-12-08,-3.2,0.3,0.0\nR004,1999-12-09,-9.1,0.0,6563.44\nR004,1999-12-10,-6.9,8.2,0.0\nR004,1999-12-11,-6.3,3.1,0.0\nR004,1999-12-12,-5.5,9.9,0.0\nR004,1999-12-13,-4.4,6.7,0.0\nR004,1999-12-14,-7.7,6.6,0.0\nR004,1999-12-15,-7.9,4.2,0.0\nR004,1999-12-16,-10.3,0.0,23991.51\nR004,1999-12-17,-15.1,0.0,25254.52\nR004,1999-12-18,-20.3,0.0,25819.5\nR004,1999-12-19,-22.4,0.0,25859.81\nR004,1999-12-20,-20.5,0.0,25048.38\nR004,1999-12-21,-10.3,1.1,0.0\nR004,1999-12-22,-1.3,14.3,0.0\nR004,1999-12-23,0.3,0.5,9436.79\nR004,1999-12-24,-4.3,3.9,0.0\nR004,1999-12-25,-13.4,1.0,20193.05\nR004,1999-12-26,-11.4,6.6,0.0\nR004,1999-12-27,-10.7,6.2,0.0\nR004,1999-12-28,-14.5,1.3,11814.09\nR004,1999-12-29,-17.1,1.4,1630.63\nR004,1999-12-30,-19.4,0.8,10275.02\nR004,1999-12-31,-30.1,0.0,24762.51\nR004,2000-01-01,-34.3,0.0,12772.24\nR004,2000-01-02,-23.8,3.0,0.0\nR004,2000-01-03,-27.1,0.8,8984.23\nR004,2000-01-04,-32.6,0.0,13493.32\nR004,2000-01-05,-33.7,0.0,20344.04\nR004,2000-01-06,-30.1,0.7,0.0\nR004,2000-01-07,-22.2,2.5,0.0\nR004,2000-01-08,-19.8,0.5,20200.92\nR004,2000-01-09,-17.4,3.2,0.0\nR004,2000-01-10,-19.0,0.0,22084.3\nR004,2000-01-11,-21.2,0.0,3709.79\nR004,2000-01-12,-17.1,0.0,10037.93\nR004,2000-01-13,-13.9,3.5,0.0\nR004,2000-01-14,-10.5,1.3,16973.23\nR004,2000-01-15,-10.0,1.0,0.0\nR004,2000-01-16,-13.7,2.5,15791.65\nR004,2000-01-17,-23.9,0.0,26181.45\nR004,2000-01-18,-25.5,0.0,18733.51\nR004,2000-01-19,-17.8,1.0,0.0\nR004,2000-01-20,-14.0,1.0,0.0\nR004,2000-01-21,-18.8,0.0,28133.4\nR004,2000-01-22,-20.2,0.0,28150.04\nR004,2000-01-23,-16.6,0.0,28637.53\nR004,2000-01-24,-17.8,0.0,29123.46\nR004,2000-01-25,-18.3,0.0,29142.75\nR004,2000-01-26,-15.0,1.8,23486.69\nR004,2000-01-27,-20.5,0.0,30109.75\nR004,2000-01-28,-20.3,0.0,30590.46\nR004,2000-01-29,-19.6,0.0,30875.4\nR004,2000-01-30,-20.3,0.0,31086.85\nR004,2000-01-31,-14.4,0.5,1435.25\nR004,2000-02-01,-11.4,1.4,21882.6\nR004,2000-02-02,-18.2,0.0,31579.91\nR004,2000-02-03,-21.4,0.0,31697.55\nR004,2000-02-04,-18.1,0.8,2998.13\nR004,2000-02-05,-13.4,0.0,23383.55\nR004,2000-02-06,-9.3,0.0,24799.28\nR004,2000-02-07,-6.8,1.4,10786.7\nR004,2000-02-08,-12.4,0.0,32220.38\nR004,2000-02-09,-9.2,0.1,4853.1\nR004,2000-02-10,-8.7,6.5,3983.58\nR004,2000-02-11,-13.2,1.2,115.42\nR004,2000-02-12,-11.7,0.8,22406.37\nR004,2000-02-13,-11.2,0.3,7143.51\nR004,2000-02-14,-7.1,2.3,140.03\nR004,2000-02-15,-5.8,0.9,6632.38\nR004,2000-02-16,-5.8,4.4,11403.68\nR004,2000-02-17,-9.8,3.3,13381.26\nR004,2000-02-18,-12.2,0.0,32871.33\nR004,2000-02-19,-12.1,0.0,25294.86\nR004,2000-02-20,-13.1,0.3,4913.28\nR004,2000-02-21,-17.0,0.0,30497.29\nR004,2000-02-22,-15.3,1.4,1554.09\nR004,2000-02-23,-9.0,0.7,21559.41\nR004,2000-02-24,-9.4,0.0,32400.0\nR004,2000-02-25,-9.5,0.0,36349.27\nR004,2000-02-26,-9.0,0.0,35421.48\nR004,2000-02-27,-6.5,0.0,34435.42\nR004,2000-02-28,-9.4,0.0,26406.85\nR004,2000-02-29,-10.8,1.4,5324.1\nR004,2000-03-01,-9.9,4.6,21830.5\nR004,2000-03-02,-7.4,9.6,9467.5\nR004,2000-03-03,-8.1,0.0,36952.21\nR004,2000-03-04,-7.8,0.8,20782.58\nR004,2000-03-05,-8.0,0.0,38894.97\nR004,2000-03-06,-7.3,0.0,39007.17\nR004,2000-03-07,-8.0,0.0,39119.15\nR004,2000-03-08,-5.5,0.0,39230.89\nR004,2000-03-09,-4.7,0.0,23944.53\nR004,2000-03-10,-3.7,0.0,39453.54\nR004,2000-03-11,-4.9,0.0,39657.23\nR004,2000-03-12,-4.7,0.0,40004.3\nR004,2000-03-13,-5.4,0.0,40315.14\nR004,2000-03-14,-4.3,0.0,40625.5\nR004,2000-03-15,-3.0,0.0,37335.56\nR004,2000-03-16,-2.6,5.3,3618.59\nR004,2000-03-17,-6.2,0.0,30360.42\nR004,2000-03-18,-5.3,4.7,14681.94\nR004,2000-03-19,-8.8,1.0,34021.79\nR004,2000-03-20,-12.5,0.0,36723.17\nR004,2000-03-21,-13.6,0.9,32463.68\nR004,2000-03-22,-5.5,1.7,8799.67\nR004,2000-03-23,-2.7,0.0,42658.8\nR004,2000-03-24,-0.8,0.0,39202.51\nR004,2000-03-25,1.5,4.1,0.0\nR004,2000-03-26,1.5,0.0,26453.82\nR004,2000-03-27,2.5,10.0,679.46\nR004,2000-03-28,-1.6,3.0,43822.52\nR004,2000-03-29,-4.2,4.8,17635.23\nR004,2000-03-30,-4.5,1.0,19182.36\nR004,2000-03-31,-3.1,0.6,39600.0\nR004,2000-04-01,0.6,8.5,5188.54\nR004,2000-04-02,-0.4,3.9,41958.27\nR004,2000-04-03,-1.2,0.0,43408.24\nR004,2000-04-04,1.1,0.0,41802.29\nR004,2000-04-05,0.2,0.0,44177.9\nR004,2000-04-06,0.2,0.0,41328.1\nR004,2000-04-07,-3.0,0.6,15815.15\nR004,2000-04-08,-3.9,0.0,46860.61\nR004,2000-04-09,1.1,0.0,47288.17\nR004,2000-04-10,2.3,6.3,12292.58\nR004,2000-04-11,-3.0,0.0,42896.03\nR004,2000-04-12,-0.7,0.0,48190.84\nR004,2000-04-13,3.1,0.0,39597.35\nR004,2000-04-14,5.7,0.0,28484.26\nR004,2000-04-15,7.8,0.0,45787.47\nR004,2000-04-16,4.2,8.4,0.0\nR004,2000-04-17,3.7,3.5,39122.17\nR004,2000-04-18,5.7,7.0,25815.82\nR004,2000-04-19,3.2,2.2,50028.2\nR004,2000-04-20,3.9,0.0,42355.4\nR004,2000-04-21,7.0,0.0,50292.33\nR004,2000-04-22,8.0,0.0,50400.0\nR004,2000-04-23,4.0,2.6,48686.52\nR004,2000-04-24,9.3,0.5,45479.3\nR004,2000-04-25,12.3,0.0,48607.33\nR004,2000-04-26,6.9,16.7,16715.36\nR004,2000-04-27,1.0,6.6,3749.59\nR004,2000-04-28,8.2,4.6,43200.0\nR004,2000-04-29,12.8,0.3,50400.0\nR004,2000-04-30,4.7,6.6,31593.95\nR004,2000-05-01,1.5,0.1,48260.46\nR004,2000-05-02,2.8,11.1,0.0\nR004,2000-05-03,8.5,0.8,37273.25\nR004,2000-05-04,4.8,12.4,6552.29\nR004,2000-05-05,4.8,20.8,0.0\nR004,2000-05-06,3.4,7.5,39554.38\nR004,2000-05-07,6.2,1.1,38502.4\nR004,2000-05-08,10.7,1.4,46730.83\nR004,2000-05-09,8.1,13.0,16339.19\nR004,2000-05-10,8.6,0.1,51855.55\nR004,2000-05-11,9.4,6.2,44384.73\nR004,2000-05-12,8.4,10.8,19870.55\nR004,2000-05-13,5.5,8.6,33938.36\nR004,2000-05-14,4.9,3.2,49823.74\nR004,2000-05-15,6.9,0.4,52310.16\nR004,2000-05-16,11.2,0.0,52236.59\nR004,2000-05-17,16.7,0.0,46072.95\nR004,2000-05-18,17.4,2.4,41673.8\nR004,2000-05-19,16.9,7.3,44139.47\nR004,2000-05-20,16.5,0.6,36452.5\nR004,2000-05-21,14.5,4.3,46808.66\nR004,2000-05-22,10.7,23.6,27281.21\nR004,2000-05-23,8.9,8.4,43458.55\nR004,2000-05-24,11.3,2.7,39291.4\nR004,2000-05-25,14.7,0.0,47495.09\nR004,2000-05-26,16.7,0.0,53098.34\nR004,2000-05-27,14.8,1.5,12958.8\nR004,2000-05-28,14.0,5.8,44436.13\nR004,2000-05-29,13.9,2.9,31138.71\nR004,2000-05-30,15.2,2.2,49849.32\nR004,2000-05-31,16.0,2.2,42080.71\nR004,2000-06-01,15.5,8.3,39938.88\nR004,2000-06-02,15.3,6.4,38355.28\nR004,2000-06-03,11.4,5.6,49934.94\nR004,2000-06-04,9.8,6.5,44364.32\nR004,2000-06-05,10.7,4.3,33200.25\nR004,2000-06-06,14.1,12.0,34629.67\nR004,2000-06-07,17.7,5.7,38927.63\nR004,2000-06-08,19.3,0.7,55091.0\nR004,2000-06-09,12.9,4.0,12140.03\nR004,2000-06-10,12.0,7.0,55450.98\nR004,2000-06-11,14.0,1.2,42940.66\nR004,2000-06-12,14.6,2.4,7195.82\nR004,2000-06-13,13.5,8.1,0.0\nR004,2000-06-14,14.6,0.9,16505.57\nR004,2000-06-15,18.7,0.1,43554.12\nR004,2000-06-16,19.9,0.1,56012.96\nR004,2000-06-17,19.0,8.2,36489.92\nR004,2000-06-18,16.2,4.1,21394.63\nR004,2000-06-19,17.1,3.5,8885.3\nR004,2000-06-20,18.5,1.4,33536.71\nR004,2000-06-21,19.9,0.3,52104.93\nR004,2000-06-22,18.7,0.0,57600.0\nR004,2000-06-23,19.3,0.0,57600.0\nR004,2000-06-24,20.2,0.0,57600.0\nR004,2000-06-25,16.9,0.8,44436.21\nR004,2000-06-26,14.9,0.0,57600.0\nR004,2000-06-27,17.5,0.0,57600.0\nR004,2000-06-28,19.2,0.0,57600.0\nR004,2000-06-29,18.2,0.0,48364.39\nR004,2000-06-30,15.1,27.7,1217.86\nR004,2000-07-01,14.5,3.7,0.0\nR004,2000-07-02,15.9,0.0,51996.94\nR004,2000-07-03,17.0,0.0,25697.97\nR004,2000-07-04,17.0,7.7,21161.17\nR004,2000-07-05,17.4,0.2,53644.25\nR004,2000-07-06,19.6,0.0,57600.0\nR004,2000-07-07,20.9,0.0,57600.0\nR004,2000-07-08,18.0,15.4,30332.22\nR004,2000-07-09,15.2,0.0,52550.58\nR004,2000-07-10,16.1,0.0,56927.44\nR004,2000-07-11,18.5,0.0,48080.37\nR004,2000-07-12,16.5,23.3,18471.99\nR004,2000-07-13,15.3,9.6,0.0\nR004,2000-07-14,14.5,12.7,36161.68\nR004,2000-07-15,15.3,6.0,46800.0\nR004,2000-07-16,14.2,0.0,53356.89\nR004,2000-07-17,14.2,0.6,43912.04\nR004,2000-07-18,13.4,4.1,51797.98\nR004,2000-07-19,13.5,3.3,54080.3\nR004,2000-07-20,14.4,8.8,37653.04\nR004,2000-07-21,14.6,1.6,48735.66\nR004,2000-07-22,13.6,0.0,54514.58\nR004,2000-07-23,14.0,0.0,42513.9\nR004,2000-07-24,14.4,0.0,43682.38\nR004,2000-07-25,13.2,12.1,53898.32\nR004,2000-07-26,11.1,21.1,17237.47\nR004,2000-07-27,14.4,0.5,51397.01\nR004,2000-07-28,17.3,0.0,54000.0\nR004,2000-07-29,18.0,0.3,54000.0\nR004,2000-07-30,18.6,1.2,53922.21\nR004,2000-07-31,17.0,0.0,46252.96\nR004,2000-08-01,16.0,0.0,51357.19\nR004,2000-08-02,14.8,0.0,49606.89\nR004,2000-08-03,15.5,0.0,42117.65\nR004,2000-08-04,15.0,2.8,49088.92\nR004,2000-08-05,17.6,0.0,42443.96\nR004,2000-08-06,21.9,0.0,51111.88\nR004,2000-08-07,22.2,1.7,51160.75\nR004,2000-08-08,22.6,0.3,47295.84\nR004,2000-08-09,19.9,3.9,31385.7\nR004,2000-08-10,16.4,22.4,1205.28\nR004,2000-08-11,14.4,9.8,32390.26\nR004,2000-08-12,14.0,0.4,43296.63\nR004,2000-08-13,15.4,0.0,50651.87\nR004,2000-08-14,16.4,0.0,50400.0\nR004,2000-08-15,13.8,7.6,11208.43\nR004,2000-08-16,15.0,0.0,43220.23\nR004,2000-08-17,16.5,0.0,46800.0\nR004,2000-08-18,17.0,2.9,27798.77\nR004,2000-08-19,15.5,15.5,7667.21\nR004,2000-08-20,15.3,0.8,42906.13\nR004,2000-08-21,16.9,0.4,47280.65\nR004,2000-08-22,16.3,3.9,46929.09\nR004,2000-08-23,13.4,8.4,15971.57\nR004,2000-08-24,12.2,2.8,38986.45\nR004,2000-08-25,11.6,2.1,46340.67\nR004,2000-08-26,13.3,0.0,49587.71\nR004,2000-08-27,15.2,0.0,40220.99\nR004,2000-08-28,12.5,15.4,17430.85\nR004,2000-08-29,10.9,0.0,43351.37\nR004,2000-08-30,9.3,7.4,37699.3\nR004,2000-08-31,7.0,7.1,16901.25\nR004,2000-09-01,6.9,5.2,16590.57\nR004,2000-09-02,7.7,3.8,45605.73\nR004,2000-09-03,9.1,0.0,46160.82\nR004,2000-09-04,10.5,0.0,45957.65\nR004,2000-09-05,11.7,0.0,45416.85\nR004,2000-09-06,12.2,0.0,44714.2\nR004,2000-09-07,13.3,0.0,44398.24\nR004,2000-09-08,12.8,0.0,43732.97\nR004,2000-09-09,12.8,0.0,43386.19\nR004,2000-09-10,14.6,0.0,43200.0\nR004,2000-09-11,16.5,0.0,43200.0\nR004,2000-09-12,16.6,0.0,43200.0\nR004,2000-09-13,12.6,9.1,28800.0\nR004,2000-09-14,8.8,0.0,43200.0\nR004,2000-09-15,10.3,0.3,33448.28\nR004,2000-09-16,10.5,5.2,29546.44\nR004,2000-09-17,8.1,0.8,41388.41\nR004,2000-09-18,7.4,0.0,40902.52\nR004,2000-09-19,9.4,11.4,8603.25\nR004,2000-09-20,6.7,3.4,34722.11\nR004,2000-09-21,5.0,2.1,21227.97\nR004,2000-09-22,7.2,0.0,43200.0\nR004,2000-09-23,1.8,3.9,4479.41\nR004,2000-09-24,1.7,1.7,18493.47\nR004,2000-09-25,4.5,5.1,8637.68\nR004,2000-09-26,4.6,8.2,1834.3\nR004,2000-09-27,0.6,3.5,209.8\nR004,2000-09-28,3.6,0.0,40015.19\nR004,2000-09-29,7.6,0.0,29994.11\nR004,2000-09-30,3.6,5.9,33921.51\nR004,2000-10-01,1.6,1.5,29222.75\nR004,2000-10-02,1.8,6.8,21813.26\nR004,2000-10-03,1.5,0.5,12584.1\nR004,2000-10-04,0.4,0.0,17786.95\nR004,2000-10-05,0.3,2.1,536.98\nR004,2000-10-06,0.1,0.0,24940.33\nR004,2000-10-07,-1.1,2.7,1444.98\nR004,2000-10-08,-2.0,0.0,12468.22\nR004,2000-10-09,-3.6,0.0,18808.79\nR004,2000-10-10,-2.3,0.0,36000.0\nR004,2000-10-11,-2.2,0.0,36000.0\nR004,2000-10-12,-1.5,0.0,36000.0\nR004,2000-10-13,-1.2,0.0,36000.0\nR004,2000-10-14,0.4,0.0,31244.85\nR004,2000-10-15,3.4,3.1,0.0\nR004,2000-10-16,2.4,5.2,1406.58\nR004,2000-10-17,-0.4,1.9,22305.35\nR004,2000-10-18,-1.9,7.9,5832.39\nR004,2000-10-19,-3.8,5.0,5845.88\nR004,2000-10-20,-2.1,8.3,0.0\nR004,2000-10-21,-0.5,5.4,0.0\nR004,2000-10-22,-14.3,1.2,25910.96\nR004,2000-10-23,-12.1,0.5,24424.02\nR004,2000-10-24,-5.6,1.9,18634.7\nR004,2000-10-25,-14.0,0.0,33136.6\nR004,2000-10-26,-4.5,2.4,0.0\nR004,2000-10-27,-4.4,0.6,9525.95\nR004,2000-10-28,-7.3,1.8,0.0\nR004,2000-10-29,-0.6,4.5,117.48\nR004,2000-10-30,-1.9,0.0,29404.07\nR004,2000-10-31,-0.0,9.9,9616.34\nR004,2000-11-01,-6.8,2.3,14339.93\nR004,2000-11-02,-7.4,7.6,0.0\nR004,2000-11-03,-10.2,11.1,0.0\nR004,2000-11-04,-13.7,1.5,13021.75\nR004,2000-11-05,-21.5,0.0,28072.79\nR004,2000-11-06,-24.7,0.0,28800.0\nR004,2000-11-07,-23.3,0.0,29464.22\nR004,2000-11-08,-23.5,0.0,29528.57\nR004,2000-11-09,-17.5,3.8,0.0\nR004,2000-11-10,-9.5,1.7,4332.27\nR004,2000-11-11,-6.8,0.0,23451.8\nR004,2000-11-12,-7.0,3.4,1881.88\nR004,2000-11-13,-9.1,1.2,24992.92\nR004,2000-11-14,-12.4,0.0,28800.0\nR004,2000-11-15,-9.0,0.7,0.0\nR004,2000-11-16,-14.3,0.0,21970.53\nR004,2000-11-17,-9.0,4.1,0.0\nR004,2000-11-18,-6.1,3.1,1933.85\nR004,2000-11-19,-6.9,1.8,0.0\nR004,2000-11-20,-8.2,2.6,0.0\nR004,2000-11-21,-12.4,1.0,13709.09\nR004,2000-11-22,-15.7,0.0,6159.54\nR004,2000-11-23,-14.8,1.7,977.14\nR004,2000-11-24,-17.3,5.4,0.0\nR004,2000-11-25,-13.0,3.7,2765.2\nR004,2000-11-26,-12.4,2.1,1910.61\nR004,2000-11-27,-10.2,0.0,17359.72\nR004,2000-11-28,-10.7,0.0,15017.61\nR004,2000-11-29,-9.3,5.4,0.0\nR004,2000-11-30,-14.7,4.5,18000.0\nR004,2000-12-01,-15.6,7.9,0.0\nR004,2000-12-02,-13.8,0.0,17821.32\nR004,2000-12-03,-9.2,6.0,0.0\nR004,2000-12-04,-9.0,5.9,0.0\nR004,2000-12-05,-22.2,0.3,14038.7\nR004,2000-12-06,-18.9,3.8,0.0\nR004,2000-12-07,-25.5,0.1,10911.23\nR004,2000-12-08,-25.2,0.0,18840.46\nR004,2000-12-09,-15.2,0.0,0.0\nR004,2000-12-10,-10.2,0.0,0.0\nR004,2000-12-11,-7.3,5.2,114.34\nR004,2000-12-12,-9.6,0.3,11363.62\nR004,2000-12-13,-9.7,6.3,0.0\nR004,2000-12-14,-13.1,1.2,5285.08\nR004,2000-12-15,-8.4,3.5,0.0\nR004,2000-12-16,-6.7,7.3,0.0\nR004,2000-12-17,-9.6,0.5,13434.98\nR004,2000-12-18,-8.4,9.3,0.0\nR004,2000-12-19,-3.7,14.8,0.0\nR004,2000-12-20,-4.1,7.1,0.0\nR004,2000-12-21,-7.1,4.6,0.0\nR004,2000-12-22,-11.3,3.7,2591.64\nR004,2000-12-23,-8.8,1.7,283.38\nR004,2000-12-24,-6.7,7.8,149.82\nR004,2000-12-25,-13.1,0.0,16098.19\nR004,2000-12-26,-21.2,0.0,17299.33\nR004,2000-12-27,-16.3,3.2,0.0\nR004,2000-12-28,-9.3,11.8,0.0\nR004,2000-12-29,-12.4,1.3,4966.07\nR004,2000-12-30,-9.1,15.5,0.0\nR004,2000-12-31,-21.1,0.4,24812.3\nR004,2001-01-01,-27.7,0.0,24968.11\nR004,2001-01-02,-31.6,0.0,15701.09\nR004,2001-01-03,-27.2,0.0,21671.03\nR004,2001-01-04,-24.1,1.5,0.0\nR004,2001-01-05,-31.9,0.0,25219.55\nR004,2001-01-06,-39.4,0.0,25446.97\nR004,2001-01-07,-39.7,0.0,14414.82\nR004,2001-01-08,-33.9,2.8,0.0\nR004,2001-01-09,-21.3,6.3,0.0\nR004,2001-01-10,-17.4,2.6,647.4\nR004,2001-01-11,-18.6,2.1,5316.93\nR004,2001-01-12,-28.0,0.0,27096.75\nR004,2001-01-13,-23.7,0.0,8763.02\nR004,2001-01-14,-16.2,0.0,17241.43\nR004,2001-01-15,-12.1,4.5,408.07\nR004,2001-01-16,-11.3,1.1,717.82\nR004,2001-01-17,-11.1,0.2,18766.95\nR004,2001-01-18,-10.0,0.3,11649.73\nR004,2001-01-19,-8.0,1.9,11019.94\nR004,2001-01-20,-12.7,0.7,22447.55\nR004,2001-01-21,-18.7,0.0,28150.97\nR004,2001-01-22,-18.9,0.0,28637.87\nR004,2001-01-23,-17.6,0.0,18759.52\nR004,2001-01-24,-18.7,0.0,9063.46\nR004,2001-01-25,-16.7,0.5,5030.49\nR004,2001-01-26,-19.6,0.0,26478.62\nR004,2001-01-27,-18.5,0.0,9806.35\nR004,2001-01-28,-12.6,0.0,15696.77\nR004,2001-01-29,-15.0,1.8,5995.17\nR004,2001-01-30,-16.8,2.0,21645.53\nR004,2001-01-31,-15.7,6.8,0.0\nR004,2001-02-01,-11.8,8.0,0.0\nR004,2001-02-02,-14.2,6.0,0.0\nR004,2001-02-03,-21.7,0.6,27264.48\nR004,2001-02-04,-29.8,0.0,31894.95\nR004,2001-02-05,-30.4,0.0,32011.81\nR004,2001-02-06,-29.0,0.0,32129.31\nR004,2001-02-07,-25.4,0.0,32247.32\nR004,2001-02-08,-18.6,0.0,32365.74\nR004,2001-02-09,-18.8,0.0,32400.0\nR004,2001-02-10,-20.5,0.0,32400.0\nR004,2001-02-11,-19.2,0.0,30407.77\nR004,2001-02-12,-20.6,0.0,8197.93\nR004,2001-02-13,-14.6,2.1,2525.09\nR004,2001-02-14,-13.0,0.0,115.21\nR004,2001-02-15,-11.0,0.0,30251.63\nR004,2001-02-16,-7.0,0.0,16853.54\nR004,2001-02-17,-3.9,10.3,0.0\nR004,2001-02-18,-4.3,2.1,32653.03\nR004,2001-02-19,-7.9,6.7,19592.73\nR004,2001-02-20,-14.3,0.0,34208.13\nR004,2001-02-21,-13.4,0.0,13787.19\nR004,2001-02-22,-12.2,0.0,35787.18\nR004,2001-02-23,-10.1,0.0,34810.53\nR004,2001-02-24,-4.3,0.1,2502.62\nR004,2001-02-25,-3.9,1.1,34749.15\nR004,2001-02-26,-7.2,0.1,29174.28\nR004,2001-02-27,-4.1,4.0,19277.03\nR004,2001-02-28,-6.6,5.9,10079.28\nR004,2001-03-01,-13.2,3.1,29739.41\nR004,2001-03-02,-12.4,0.0,37888.83\nR004,2001-03-03,-9.9,3.3,29312.09\nR004,2001-03-04,-16.3,2.0,30207.94\nR004,2001-03-05,-20.7,0.0,37216.92\nR004,2001-03-06,-19.9,0.0,38981.47\nR004,2001-03-07,-14.8,0.0,39093.56\nR004,2001-03-08,-11.9,0.0,20367.19\nR004,2001-03-09,-6.9,0.0,35113.17\nR004,2001-03-10,-3.7,3.1,28910.78\nR004,2001-03-11,-4.5,6.2,19766.32\nR004,2001-03-12,-11.2,0.2,27037.05\nR004,2001-03-13,-10.3,0.0,25995.25\nR004,2001-03-14,-4.0,0.0,36269.29\nR004,2001-03-15,-3.0,0.2,38391.13\nR004,2001-03-16,-1.4,11.7,0.0\nR004,2001-03-17,0.1,0.2,21426.75\nR004,2001-03-18,1.2,3.3,18659.68\nR004,2001-03-19,1.6,2.5,33332.21\nR004,2001-03-20,-1.3,9.7,32974.24\nR004,2001-03-21,-1.0,25.6,0.0\nR004,2001-03-22,-4.8,4.5,14172.42\nR004,2001-03-23,-7.9,0.5,34882.72\nR004,2001-03-24,-6.9,0.0,35447.91\nR004,2001-03-25,-0.6,0.0,38330.01\nR004,2001-03-26,0.0,0.1,36656.05\nR004,2001-03-27,4.6,0.0,43887.81\nR004,2001-03-28,5.0,0.0,43200.0\nR004,2001-03-29,3.6,0.7,36000.0\nR004,2001-03-30,4.8,0.2,37659.6\nR004,2001-03-31,2.9,0.0,34341.15\nR004,2001-04-01,0.2,6.7,20924.9\nR004,2001-04-02,-2.6,1.8,43200.0\nR004,2001-04-03,-2.8,5.3,37050.15\nR004,2001-04-04,-1.6,0.0,36005.06\nR004,2001-04-05,0.1,8.4,0.0\nR004,2001-04-06,-1.1,7.7,5643.77\nR004,2001-04-07,-4.9,6.8,2291.95\nR004,2001-04-08,-11.1,0.0,36328.33\nR004,2001-04-09,-4.0,0.0,40496.76\nR004,2001-04-10,-0.8,0.0,32605.08\nR004,2001-04-11,1.4,0.0,46470.89\nR004,2001-04-12,2.3,0.0,45819.33\nR004,2001-04-13,3.8,0.0,48092.05\nR004,2001-04-14,7.0,1.4,47605.74\nR004,2001-04-15,4.9,2.9,13181.97\nR004,2001-04-16,4.2,14.5,28134.35\nR004,2001-04-17,-2.3,1.1,49663.56\nR004,2001-04-18,-1.1,0.0,49869.61\nR004,2001-04-19,3.0,0.0,50002.21\nR004,2001-04-20,3.5,0.0,50133.39\nR004,2001-04-21,2.4,0.0,50235.51\nR004,2001-04-22,2.8,0.0,50390.89\nR004,2001-04-23,4.1,0.0,50400.0\nR004,2001-04-24,6.8,0.0,48290.6\nR004,2001-04-25,4.1,0.8,34176.71\nR004,2001-04-26,-1.7,0.0,50400.0\nR004,2001-04-27,-3.1,0.0,48576.22\nR004,2001-04-28,-3.7,0.0,30830.87\nR004,2001-04-29,-0.1,12.0,0.0\nR004,2001-04-30,-0.4,2.8,38337.99\nR004,2001-05-01,-0.5,0.0,31351.03\nR004,2001-05-02,3.0,1.7,18304.86\nR004,2001-05-03,6.4,0.0,50666.89\nR004,2001-05-04,8.7,0.7,39369.85\nR004,2001-05-05,11.2,1.5,45346.7\nR004,2001-05-06,13.3,2.6,49708.74\nR004,2001-05-07,16.6,0.0,49991.75\nR004,2001-05-08,17.3,0.0,51288.23\nR004,2001-05-09,17.9,0.0,51535.08\nR004,2001-05-10,17.6,1.4,41870.58\nR004,2001-05-11,11.5,12.6,23905.29\nR004,2001-05-12,2.7,4.4,33660.35\nR004,2001-05-13,7.2,0.0,50498.76\nR004,2001-05-14,12.1,0.0,54844.06\nR004,2001-05-15,16.2,0.0,47795.0\nR004,2001-05-16,17.7,0.0,51697.21\nR004,2001-05-17,17.9,13.3,28489.4\nR004,2001-05-18,11.4,38.7,870.62\nR004,2001-05-19,8.6,9.5,17815.22\nR004,2001-05-20,11.5,0.4,52005.24\nR004,2001-05-21,3.0,15.4,35560.35\nR004,2001-05-22,4.1,0.0,57193.7\nR004,2001-05-23,9.6,0.0,57271.06\nR004,2001-05-24,13.9,0.0,57346.04\nR004,2001-05-25,16.9,0.0,57418.56\nR004,2001-05-26,19.1,0.0,50577.33\nR004,2001-05-27,19.9,0.0,50274.19\nR004,2001-05-28,21.0,0.0,53017.42\nR004,2001-05-29,21.0,0.0,57306.93\nR004,2001-05-30,19.6,0.4,46800.0\nR004,2001-05-31,18.2,2.7,52259.38\nR004,2001-06-01,20.0,0.0,47428.96\nR004,2001-06-02,19.4,2.2,52399.39\nR004,2001-06-03,15.1,7.9,22748.85\nR004,2001-06-04,13.2,3.8,29034.0\nR004,2001-06-05,16.5,0.0,57099.87\nR004,2001-06-06,15.0,6.9,34787.55\nR004,2001-06-07,12.0,15.6,14770.35\nR004,2001-06-08,11.7,4.4,33573.26\nR004,2001-06-09,11.1,0.7,38280.79\nR004,2001-06-10,12.2,4.1,53670.57\nR004,2001-06-11,16.5,0.5,54000.0\nR004,2001-06-12,14.8,15.7,28507.26\nR004,2001-06-13,12.5,8.5,33150.69\nR004,2001-06-14,14.8,0.3,55918.57\nR004,2001-06-15,16.9,0.0,57600.0\nR004,2001-06-16,18.5,0.0,57600.0\nR004,2001-06-17,19.4,1.4,51639.56\nR004,2001-06-18,16.8,9.8,30826.5\nR004,2001-06-19,18.2,0.0,50990.98\nR004,2001-06-20,20.2,0.0,57600.0\nR004,2001-06-21,20.0,0.3,34912.26\nR004,2001-06-22,17.5,8.4,15002.19\nR004,2001-06-23,14.7,1.9,41239.88\nR004,2001-06-24,14.6,1.7,56481.7\nR004,2001-06-25,19.0,0.0,56160.93\nR004,2001-06-26,19.2,1.9,50745.06\nR004,2001-06-27,19.2,0.7,57600.0\nR004,2001-06-28,19.6,1.4,49885.51\nR004,2001-06-29,16.7,6.2,36638.17\nR004,2001-06-30,12.9,4.3,24534.06\nR004,2001-07-01,13.4,0.0,57295.88\nR004,2001-07-02,11.7,3.4,18883.11\nR004,2001-07-03,11.3,6.4,653.5\nR004,2001-07-04,12.6,11.1,12347.68\nR004,2001-07-05,13.8,2.0,48847.99\nR004,2001-07-06,14.9,0.6,53735.68\nR004,2001-07-07,18.0,0.0,57600.0\nR004,2001-07-08,18.5,2.6,52333.3\nR004,2001-07-09,17.1,2.8,50400.0\nR004,2001-07-10,16.9,2.3,53470.68\nR004,2001-07-11,15.4,4.3,48851.07\nR004,2001-07-12,13.4,14.8,19486.82\nR004,2001-07-13,14.8,12.0,48153.85\nR004,2001-07-14,12.8,5.4,47885.36\nR004,2001-07-15,13.4,4.4,41249.4\nR004,2001-07-16,15.0,0.0,43957.06\nR004,2001-07-17,17.9,0.0,52286.39\nR004,2001-07-18,20.7,0.0,53894.37\nR004,2001-07-19,19.1,5.0,47910.89\nR004,2001-07-20,16.8,14.4,16851.69\nR004,2001-07-21,14.9,17.7,13943.25\nR004,2001-07-22,11.4,14.7,35126.21\nR004,2001-07-23,10.9,0.8,35245.64\nR004,2001-07-24,16.5,0.2,48769.63\nR004,2001-07-25,17.3,7.1,29640.25\nR004,2001-07-26,17.2,0.6,49918.01\nR004,2001-07-27,16.2,0.0,50121.79\nR004,2001-07-28,14.0,6.7,0.0\nR004,2001-07-29,13.5,0.5,13166.28\nR004,2001-07-30,13.9,0.0,50705.44\nR004,2001-07-31,16.1,0.0,45480.0\nR004,2001-08-01,17.0,0.7,38565.06\nR004,2001-08-02,15.5,0.0,53650.28\nR004,2001-08-03,16.1,0.0,53545.53\nR004,2001-08-04,17.0,0.3,52700.9\nR004,2001-08-05,19.3,0.0,52743.0\nR004,2001-08-06,21.0,0.0,50677.79\nR004,2001-08-07,20.3,1.0,50889.28\nR004,2001-08-08,21.7,0.0,50400.0\nR004,2001-08-09,19.3,0.5,51251.42\nR004,2001-08-10,18.4,11.1,46204.86\nR004,2001-08-11,15.9,27.5,41602.62\nR004,2001-08-12,16.8,0.2,46312.95\nR004,2001-08-13,14.4,28.4,4758.85\nR004,2001-08-14,13.1,9.9,28758.78\nR004,2001-08-15,14.2,1.4,34376.04\nR004,2001-08-16,16.0,0.0,47729.08\nR004,2001-08-17,17.2,0.0,50467.26\nR004,2001-08-18,19.3,0.0,50400.0\nR004,2001-08-19,19.2,1.7,42529.74\nR004,2001-08-20,16.2,1.8,29082.15\nR004,2001-08-21,15.4,1.1,50077.38\nR004,2001-08-22,15.8,0.0,50015.63\nR004,2001-08-23,14.2,3.2,38489.39\nR004,2001-08-24,15.3,0.0,48460.19\nR004,2001-08-25,17.6,0.0,42547.74\nR004,2001-08-26,17.7,4.0,41374.88\nR004,2001-08-27,14.2,2.4,18873.16\nR004,2001-08-28,13.6,0.0,49025.38\nR004,2001-08-29,15.3,0.0,46960.67\nR004,2001-08-30,15.9,0.5,26015.37\nR004,2001-08-31,13.8,0.9,43095.94\nR004,2001-09-01,11.0,1.3,33050.02\nR004,2001-09-02,9.3,11.2,21299.49\nR004,2001-09-03,11.1,0.0,44093.73\nR004,2001-09-04,11.9,0.0,46321.22\nR004,2001-09-05,10.6,12.4,18187.85\nR004,2001-09-06,11.0,0.0,43806.73\nR004,2001-09-07,13.2,0.1,40361.8\nR004,2001-09-08,13.4,8.7,26900.57\nR004,2001-09-09,10.2,14.1,22290.08\nR004,2001-09-10,8.1,12.6,13243.32\nR004,2001-09-11,8.5,4.0,35115.1\nR004,2001-09-12,7.3,4.9,39125.29\nR004,2001-09-13,5.1,0.9,8767.11\nR004,2001-09-14,5.1,0.0,35687.86\nR004,2001-09-15,5.2,0.2,39588.88\nR004,2001-09-16,5.6,0.0,43200.0\nR004,2001-09-17,8.5,0.0,43200.0\nR004,2001-09-18,8.6,3.4,28048.23\nR004,2001-09-19,7.3,2.2,2108.77\nR004,2001-09-20,8.2,1.6,14175.64\nR004,2001-09-21,5.7,5.6,28525.16\nR004,2001-09-22,5.9,4.6,28475.63\nR004,2001-09-23,7.0,6.5,9361.37\nR004,2001-09-24,8.8,0.0,41971.86\nR004,2001-09-25,9.7,0.0,42564.17\nR004,2001-09-26,8.3,0.2,41914.37\nR004,2001-09-27,9.9,0.0,40622.36\nR004,2001-09-28,11.4,0.0,36000.0\nR004,2001-09-29,6.0,4.9,6391.81\nR004,2001-09-30,0.6,0.0,16793.53\nR004,2001-10-01,1.0,0.0,37718.9\nR004,2001-10-02,4.5,0.0,37030.16\nR004,2001-10-03,2.0,3.3,14392.74\nR004,2001-10-04,1.2,0.0,36340.63\nR004,2001-10-05,1.8,1.3,19249.31\nR004,2001-10-06,4.1,4.5,1106.61\nR004,2001-10-07,6.5,5.7,20157.08\nR004,2001-10-08,6.8,11.2,2477.45\nR004,2001-10-09,8.8,5.3,18908.91\nR004,2001-10-10,3.6,7.2,21241.16\nR004,2001-10-11,0.5,1.1,16302.89\nR004,2001-10-12,2.8,1.3,910.49\nR004,2001-10-13,0.7,7.8,6483.66\nR004,2001-10-14,-3.3,0.0,15543.85\nR004,2001-10-15,-2.0,0.0,33209.02\nR004,2001-10-16,0.2,1.1,584.21\nR004,2001-10-17,4.9,4.7,34669.35\nR004,2001-10-18,2.8,4.8,18230.83\nR004,2001-10-19,1.5,5.2,18020.73\nR004,2001-10-20,-3.2,2.6,16272.45\nR004,2001-10-21,-4.5,0.0,29123.39\nR004,2001-10-22,-2.3,2.1,17197.07\nR004,2001-10-23,-2.1,1.6,8419.47\nR004,2001-10-24,-1.9,0.6,7751.85\nR004,2001-10-25,-4.2,0.0,31667.32\nR004,2001-10-26,-2.1,0.0,29943.59\nR004,2001-10-27,1.9,2.1,2012.06\nR004,2001-10-28,-2.9,7.3,14444.81\nR004,2001-10-29,-7.1,3.5,1527.12\nR004,2001-10-30,-10.7,0.0,30647.69\nR004,2001-10-31,-12.7,0.0,31147.51\nR004,2001-11-01,-10.7,0.0,1516.94\nR004,2001-11-02,-4.6,0.0,0.0\nR004,2001-11-03,-4.8,0.0,29193.55\nR004,2001-11-04,-0.8,13.7,0.0\nR004,2001-11-05,1.6,7.1,5783.99\nR004,2001-11-06,0.7,10.5,9999.06\nR004,2001-11-07,1.4,2.6,6717.85\nR004,2001-11-08,1.0,0.2,25468.43\nR004,2001-11-09,-2.6,5.0,20622.84\nR004,2001-11-10,-6.4,0.9,16223.73\nR004,2001-11-11,-3.5,2.8,2501.61\nR004,2001-11-12,-1.5,0.2,10451.32\nR004,2001-11-13,-1.6,0.0,28800.0\nR004,2001-11-14,-1.5,0.0,22582.64\nR004,2001-11-15,-1.9,1.0,28795.16\nR004,2001-11-16,3.1,1.5,0.0\nR004,2001-11-17,2.8,0.0,28800.0\nR004,2001-11-18,0.5,0.0,17829.84\nR004,2001-11-19,2.5,0.1,20884.09\nR004,2001-11-20,1.8,1.3,25200.0\nR004,2001-11-21,-2.4,8.7,1639.9\nR004,2001-11-22,-8.7,0.0,15064.52\nR004,2001-11-23,-15.2,0.0,28570.46\nR004,2001-11-24,-16.8,0.0,28501.55\nR004,2001-11-25,-13.9,0.0,21881.49\nR004,2001-11-26,-13.5,0.5,25608.67\nR004,2001-11-27,-12.6,0.4,3744.38\nR004,2001-11-28,-19.6,0.0,18985.18\nR004,2001-11-29,-23.3,0.0,24113.71\nR004,2001-11-30,-22.4,0.0,10800.0\nR004,2001-12-01,-17.5,1.5,420.94\nR004,2001-12-02,-18.8,0.0,7830.82\nR004,2001-12-03,-17.0,1.0,4548.38\nR004,2001-12-04,-13.7,3.8,0.0\nR004,2001-12-05,-13.4,2.0,0.0\nR004,2001-12-06,-14.0,3.6,0.0\nR004,2001-12-07,-13.6,2.9,0.0\nR004,2001-12-08,-13.8,3.7,0.0\nR004,2001-12-09,-23.7,1.4,0.0\nR004,2001-12-10,-34.6,0.0,20338.92\nR004,2001-12-11,-30.6,0.0,897.66\nR004,2001-12-12,-16.5,2.5,0.0\nR004,2001-12-13,-15.9,0.0,19507.91\nR004,2001-12-14,-16.2,1.3,0.0\nR004,2001-12-15,-18.4,0.0,13299.21\nR004,2001-12-16,-29.8,0.0,25769.33\nR004,2001-12-17,-32.8,0.0,25804.32\nR004,2001-12-18,-31.6,0.0,25840.35\nR004,2001-12-19,-27.1,0.0,21102.28\nR004,2001-12-20,-14.8,0.0,9541.27\nR004,2001-12-21,-11.5,0.0,8674.39\nR004,2001-12-22,-10.7,1.4,0.0\nR004,2001-12-23,-8.9,0.1,2887.85\nR004,2001-12-24,-20.3,0.0,19666.95\nR004,2001-12-25,-26.5,0.0,25092.19\nR004,2001-12-26,-23.2,0.0,22893.26\nR004,2001-12-27,-11.5,2.4,17339.18\nR004,2001-12-28,-14.5,0.0,12994.07\nR004,2001-12-29,-9.8,3.8,0.0\nR004,2001-12-30,-4.6,1.2,1339.77\nR004,2001-12-31,-9.6,0.0,21093.29\nR004,2002-01-01,-4.5,8.3,0.0\nR004,2002-01-02,-0.5,5.1,0.0\nR004,2002-01-03,0.5,1.6,13551.18\nR004,2002-01-04,-2.0,13.2,0.0\nR004,2002-01-05,-16.1,0.0,25718.88\nR004,2002-01-06,-9.3,2.6,24841.64\nR004,2002-01-07,-8.5,1.0,0.0\nR004,2002-01-08,-7.4,0.0,25201.17\nR004,2002-01-09,-1.3,16.1,0.0\nR004,2002-01-10,-5.0,6.1,16435.83\nR004,2002-01-11,-10.6,0.4,404.83\nR004,2002-01-12,-9.3,4.5,10043.13\nR004,2002-01-13,-13.2,1.2,2958.35\nR004,2002-01-14,-12.4,0.0,302.29\nR004,2002-01-15,-4.7,4.1,0.0\nR004,2002-01-16,-4.6,0.5,4381.63\nR004,2002-01-17,-9.4,0.1,26905.46\nR004,2002-01-18,-13.4,2.1,2331.07\nR004,2002-01-19,-13.1,3.9,0.0\nR004,2002-01-20,-10.4,0.6,9994.03\nR004,2002-01-21,-15.8,0.0,21189.56\nR004,2002-01-22,-14.6,0.0,8779.78\nR004,2002-01-23,-11.8,0.6,15992.52\nR004,2002-01-24,-15.4,0.0,26087.47\nR004,2002-01-25,-13.9,0.0,23599.78\nR004,2002-01-26,-14.6,0.0,29635.55\nR004,2002-01-27,-8.9,0.0,28558.0\nR004,2002-01-28,-12.5,0.0,30768.04\nR004,2002-01-29,-13.4,0.0,30971.38\nR004,2002-01-30,-13.2,0.0,23162.07\nR004,2002-01-31,-14.7,0.0,31390.66\nR004,2002-02-01,-8.5,1.0,27755.8\nR004,2002-02-02,-4.8,1.8,3108.29\nR004,2002-02-03,-5.8,1.2,7677.41\nR004,2002-02-04,-5.9,3.8,17452.99\nR004,2002-02-05,-7.5,8.1,0.0\nR004,2002-02-06,-13.1,1.1,26181.24\nR004,2002-02-07,-13.1,0.0,32222.06\nR004,2002-02-08,-8.0,6.7,0.0\nR004,2002-02-09,-6.8,7.3,4663.8\nR004,2002-02-10,-6.3,4.1,2337.3\nR004,2002-02-11,-3.7,2.1,8805.87\nR004,2002-02-12,-5.4,0.0,20869.43\nR004,2002-02-13,-3.2,0.0,32521.15\nR004,2002-02-14,-5.4,0.0,32458.26\nR004,2002-02-15,-7.2,0.0,29894.44\nR004,2002-02-16,-2.7,2.2,5293.88\nR004,2002-02-17,-0.1,2.6,26738.38\nR004,2002-02-18,-3.3,0.6,31446.56\nR004,2002-02-19,-6.3,0.0,30177.49\nR004,2002-02-20,-4.5,2.2,28800.0\nR004,2002-02-21,-8.2,0.2,32027.0\nR004,2002-02-22,-10.7,0.0,35363.9\nR004,2002-02-23,-11.7,0.0,35836.13\nR004,2002-02-24,-11.5,0.0,36307.65\nR004,2002-02-25,-8.4,0.0,36778.32\nR004,2002-02-26,-8.9,0.0,37248.26\nR004,2002-02-27,-11.0,0.0,37782.92\nR004,2002-02-28,-10.7,0.0,38034.71\nR004,2002-03-01,-9.8,0.0,38286.74\nR004,2002-03-02,-10.5,0.0,38504.82\nR004,2002-03-03,-10.2,0.0,38618.24\nR004,2002-03-04,-7.9,0.5,19487.72\nR004,2002-03-05,-2.1,1.2,3011.9\nR004,2002-03-06,0.4,11.0,0.0\nR004,2002-03-07,-3.9,8.0,27570.82\nR004,2002-03-08,-2.9,1.1,21334.0\nR004,2002-03-09,2.0,0.0,31415.98\nR004,2002-03-10,2.6,0.0,38574.72\nR004,2002-03-11,0.0,3.1,5535.6\nR004,2002-03-12,-2.0,1.0,39660.73\nR004,2002-03-13,-5.5,0.3,36797.82\nR004,2002-03-14,-3.6,2.3,26881.09\nR004,2002-03-15,-7.5,3.8,19879.83\nR004,2002-03-16,-6.0,0.0,4244.99\nR004,2002-03-17,-0.8,0.4,32782.58\nR004,2002-03-18,-0.5,12.4,5168.37\nR004,2002-03-19,-6.3,10.6,7147.49\nR004,2002-03-20,-6.5,5.9,38609.75\nR004,2002-03-21,-6.4,1.8,25297.73\nR004,2002-03-22,-1.0,4.6,20596.17\nR004,2002-03-23,0.5,0.0,42602.11\nR004,2002-03-24,-1.3,1.2,36056.64\nR004,2002-03-25,-0.8,8.5,0.0\nR004,2002-03-26,1.4,7.1,4444.66\nR004,2002-03-27,1.0,5.8,9814.13\nR004,2002-03-28,-1.2,2.1,27622.28\nR004,2002-03-29,1.8,6.7,2461.56\nR004,2002-03-30,-0.1,23.6,0.0\nR004,2002-03-31,0.0,1.6,23406.36\nR004,2002-04-01,2.5,6.4,9264.48\nR004,2002-04-02,-0.2,16.1,7229.9\nR004,2002-04-03,-6.1,0.0,43921.05\nR004,2002-04-04,-11.1,0.0,42646.58\nR004,2002-04-05,-12.8,0.0,45357.48\nR004,2002-04-06,-11.1,0.0,45994.48\nR004,2002-04-07,-7.8,0.0,46417.71\nR004,2002-04-08,-3.3,0.0,46842.28\nR004,2002-04-09,0.4,0.0,47268.42\nR004,2002-04-10,2.0,0.3,32041.22\nR004,2002-04-11,1.0,12.4,4323.69\nR004,2002-04-12,-6.6,0.0,44812.83\nR004,2002-04-13,-6.3,4.0,14250.83\nR004,2002-04-14,-2.4,5.7,34659.63\nR004,2002-04-15,-1.1,1.1,47723.99\nR004,2002-04-16,1.7,0.0,48455.62\nR004,2002-04-17,3.2,0.0,46592.05\nR004,2002-04-18,3.4,8.5,22930.0\nR004,2002-04-19,0.6,9.9,406.1\nR004,2002-04-20,-1.4,0.0,45834.05\nR004,2002-04-21,-2.0,0.0,50221.73\nR004,2002-04-22,0.4,0.0,50350.7\nR004,2002-04-23,4.2,0.0,44330.87\nR004,2002-04-24,5.9,0.0,48316.89\nR004,2002-04-25,5.1,0.7,48622.42\nR004,2002-04-26,7.5,2.6,48202.7\nR004,2002-04-27,4.7,1.4,47391.69\nR004,2002-04-28,4.3,0.2,33738.35\nR004,2002-04-29,3.3,0.0,49137.71\nR004,2002-04-30,3.3,0.7,37637.03\nR004,2002-05-01,2.7,0.9,26756.76\nR004,2002-05-02,2.2,0.0,50724.41\nR004,2002-05-03,2.5,0.0,50251.31\nR004,2002-05-04,3.7,0.0,51051.26\nR004,2002-05-05,5.5,0.0,50584.2\nR004,2002-05-06,7.8,0.0,50987.48\nR004,2002-05-07,9.9,0.0,50400.0\nR004,2002-05-08,11.4,0.0,51727.39\nR004,2002-05-09,11.5,0.5,42886.66\nR004,2002-05-10,7.5,8.4,7816.28\nR004,2002-05-11,7.0,0.7,48868.06\nR004,2002-05-12,11.0,0.0,52162.92\nR004,2002-05-13,14.0,0.0,52079.69\nR004,2002-05-14,13.5,4.1,31579.94\nR004,2002-05-15,9.5,7.4,24952.21\nR004,2002-05-16,10.2,0.0,54031.42\nR004,2002-05-17,3.5,8.2,5458.28\nR004,2002-05-18,4.8,11.5,22613.51\nR004,2002-05-19,7.9,0.0,47453.64\nR004,2002-05-20,12.2,0.0,55960.5\nR004,2002-05-21,16.4,0.0,55580.63\nR004,2002-05-22,17.4,0.0,47883.43\nR004,2002-05-23,18.4,0.0,54639.47\nR004,2002-05-24,18.1,2.4,45934.46\nR004,2002-05-25,14.5,4.0,48244.05\nR004,2002-05-26,14.6,0.0,51248.56\nR004,2002-05-27,17.1,0.0,55874.39\nR004,2002-05-28,15.4,5.7,34549.56\nR004,2002-05-29,16.0,1.3,47886.56\nR004,2002-05-30,12.4,2.0,53449.86\nR004,2002-05-31,14.3,1.0,34492.58\nR004,2002-06-01,15.1,4.5,51882.83\nR004,2002-06-02,10.8,7.2,791.84\nR004,2002-06-03,8.1,9.2,6759.99\nR004,2002-06-04,10.5,0.0,47755.88\nR004,2002-06-05,11.3,2.6,53521.05\nR004,2002-06-06,16.0,0.0,53733.39\nR004,2002-06-07,17.6,0.0,56197.23\nR004,2002-06-08,13.0,0.4,57296.14\nR004,2002-06-09,13.6,0.0,54361.03\nR004,2002-06-10,16.0,4.5,45967.49\nR004,2002-06-11,16.0,28.8,38485.59\nR004,2002-06-12,16.3,13.8,38292.19\nR004,2002-06-13,13.2,24.5,48340.81\nR004,2002-06-14,13.5,10.8,32582.98\nR004,2002-06-15,12.6,6.1,37873.6\nR004,2002-06-16,14.5,1.9,47803.94\nR004,2002-06-17,17.3,0.0,56247.83\nR004,2002-06-18,15.8,7.6,484.72\nR004,2002-06-19,13.4,16.6,0.0\nR004,2002-06-20,13.8,15.4,8263.97\nR004,2002-06-21,15.4,6.7,15683.91\nR004,2002-06-22,17.5,0.2,42968.32\nR004,2002-06-23,19.5,2.2,46200.57\nR004,2002-06-24,19.1,12.6,36043.81\nR004,2002-06-25,17.3,12.8,45076.3\nR004,2002-06-26,19.5,3.4,45227.03\nR004,2002-06-27,16.5,14.3,36850.85\nR004,2002-06-28,13.5,11.6,31900.03\nR004,2002-06-29,15.0,4.3,54470.24\nR004,2002-06-30,16.5,0.0,55456.57\nR004,2002-07-01,15.7,12.5,9639.81\nR004,2002-07-02,17.5,3.1,41279.13\nR004,2002-07-03,14.3,14.7,11032.67\nR004,2002-07-04,14.3,11.5,36261.04\nR004,2002-07-05,15.3,4.0,39821.22\nR004,2002-07-06,15.1,2.3,46926.48\nR004,2002-07-07,13.6,13.0,7571.13\nR004,2002-07-08,13.8,12.7,36523.37\nR004,2002-07-09,12.7,4.0,45580.11\nR004,2002-07-10,14.1,1.3,44838.76\nR004,2002-07-11,16.9,0.0,51321.75\nR004,2002-07-12,17.9,0.0,53467.58\nR004,2002-07-13,17.2,4.6,38822.38\nR004,2002-07-14,19.2,0.1,49948.08\nR004,2002-07-15,19.4,0.1,55643.23\nR004,2002-07-16,21.8,0.0,56132.02\nR004,2002-07-17,19.7,19.2,36238.58\nR004,2002-07-18,18.8,0.3,47512.42\nR004,2002-07-19,21.2,0.0,55305.6\nR004,2002-07-20,23.1,0.0,55338.06\nR004,2002-07-21,21.1,6.6,35379.42\nR004,2002-07-22,15.7,19.1,0.0\nR004,2002-07-23,13.2,27.5,0.0\nR004,2002-07-24,14.7,7.5,0.0\nR004,2002-07-25,14.6,5.7,667.72\nR004,2002-07-26,15.0,0.0,47248.24\nR004,2002-07-27,16.6,0.0,39670.37\nR004,2002-07-28,16.2,0.0,33336.89\nR004,2002-07-29,17.2,4.3,41234.02\nR004,2002-07-30,16.1,12.3,14168.06\nR004,2002-07-31,12.5,6.1,41163.01\nR004,2002-08-01,13.4,9.4,37254.67\nR004,2002-08-02,14.9,0.1,52611.59\nR004,2002-08-03,15.0,3.4,47773.28\nR004,2002-08-04,15.2,0.0,53459.69\nR004,2002-08-05,17.9,0.0,45258.23\nR004,2002-08-06,17.2,3.5,11111.49\nR004,2002-08-07,20.0,0.0,47407.63\nR004,2002-08-08,20.5,0.0,49728.44\nR004,2002-08-09,20.0,0.0,51659.09\nR004,2002-08-10,21.1,0.2,50876.39\nR004,2002-08-11,24.8,0.0,50930.05\nR004,2002-08-12,19.5,7.5,22967.13\nR004,2002-08-13,14.6,16.0,7702.95\nR004,2002-08-14,15.0,0.0,50400.0\nR004,2002-08-15,15.0,0.0,0.0\nR004,2002-08-16,16.5,0.1,40506.36\nR004,2002-08-17,17.3,0.1,33425.85\nR004,2002-08-18,17.4,0.0,45792.24\nR004,2002-08-19,18.6,0.0,50343.89\nR004,2002-08-20,19.5,0.0,50243.43\nR004,2002-08-21,19.4,4.7,35993.79\nR004,2002-08-22,14.3,4.7,49048.33\nR004,2002-08-23,13.8,0.7,49940.4\nR004,2002-08-24,11.2,7.1,25106.37\nR004,2002-08-25,16.5,8.7,36233.4\nR004,2002-08-26,12.5,19.8,26060.84\nR004,2002-08-27,9.5,10.9,15594.75\nR004,2002-08-28,11.7,0.1,42617.43\nR004,2002-08-29,12.1,2.7,39353.91\nR004,2002-08-30,9.5,18.1,0.0\nR004,2002-08-31,9.0,7.0,30462.04\nR004,2002-09-01,9.7,0.7,46284.99\nR004,2002-09-02,10.7,0.1,46070.56\nR004,2002-09-03,15.1,0.7,45405.77\nR004,2002-09-04,14.4,7.5,30807.64\nR004,2002-09-05,15.5,2.9,37858.33\nR004,2002-09-06,14.5,6.6,33330.27\nR004,2002-09-07,7.8,7.6,18883.1\nR004,2002-09-08,6.6,19.8,644.38\nR004,2002-09-09,8.6,1.2,13586.71\nR004,2002-09-10,11.5,0.1,43416.4\nR004,2002-09-11,12.0,0.0,43200.0\nR004,2002-09-12,12.7,0.0,43200.0\nR004,2002-09-13,14.5,0.0,43200.0\nR004,2002-09-14,11.4,5.6,27228.33\nR004,2002-09-15,-0.1,1.7,11924.26\nR004,2002-09-16,0.8,0.0,26677.29\nR004,2002-09-17,3.1,0.0,37292.45\nR004,2002-09-18,7.6,0.2,40792.73\nR004,2002-09-19,11.6,0.0,43200.0\nR004,2002-09-20,11.6,0.0,43200.0\nR004,2002-09-21,13.9,0.0,43200.0\nR004,2002-09-22,12.9,0.1,43200.0\nR004,2002-09-23,6.0,1.8,28582.68\nR004,2002-09-24,5.1,0.1,34662.73\nR004,2002-09-25,4.8,4.4,390.32\nR004,2002-09-26,11.3,3.0,14518.86\nR004,2002-09-27,8.7,2.3,1765.9\nR004,2002-09-28,10.3,9.5,12039.3\nR004,2002-09-29,3.6,5.9,16614.52\nR004,2002-09-30,3.3,1.3,32012.06\nR004,2002-10-01,6.2,6.2,6298.82\nR004,2002-10-02,3.6,5.7,32400.0\nR004,2002-10-03,2.7,0.0,36357.08\nR004,2002-10-04,4.3,0.0,32253.52\nR004,2002-10-05,8.9,3.2,14400.0\nR004,2002-10-06,8.0,10.5,0.0\nR004,2002-10-07,2.9,0.0,36000.0\nR004,2002-10-08,0.6,0.0,35332.9\nR004,2002-10-09,0.8,6.2,15950.46\nR004,2002-10-10,-2.8,5.0,695.7\nR004,2002-10-11,-2.4,2.5,11791.38\nR004,2002-10-12,3.2,22.6,0.0\nR004,2002-10-13,4.0,3.4,6260.43\nR004,2002-10-14,-0.9,11.4,4811.43\nR004,2002-10-15,-5.4,5.0,602.91\nR004,2002-10-16,-8.6,0.0,24735.38\nR004,2002-10-17,-10.8,0.0,19490.63\nR004,2002-10-18,-7.8,0.0,24010.53\nR004,2002-10-19,-6.4,0.0,35530.14\nR004,2002-10-20,-7.5,0.0,18298.07\nR004,2002-10-21,-4.6,5.7,0.0\nR004,2002-10-22,2.2,4.8,0.0\nR004,2002-10-23,0.7,0.1,30181.64\nR004,2002-10-24,-1.6,0.0,33446.48\nR004,2002-10-25,-1.3,5.5,2178.45\nR004,2002-10-26,-0.0,0.6,1185.84\nR004,2002-10-27,-3.6,0.6,2289.38\nR004,2002-10-28,-7.5,0.0,30987.25\nR004,2002-10-29,-10.1,0.0,31840.09\nR004,2002-10-30,-10.1,0.0,31692.28\nR004,2002-10-31,-9.1,0.0,31131.47\nR004,2002-11-01,-6.6,0.0,31202.93\nR004,2002-11-02,-2.3,0.0,23527.71\nR004,2002-11-03,0.6,0.0,23760.8\nR004,2002-11-04,-0.2,6.1,6055.45\nR004,2002-11-05,-4.7,4.1,698.24\nR004,2002-11-06,-8.4,0.0,21212.48\nR004,2002-11-07,-1.1,0.0,9192.61\nR004,2002-11-08,-5.0,8.8,326.45\nR004,2002-11-09,-10.6,6.0,0.0\nR004,2002-11-10,-6.8,9.4,0.0\nR004,2002-11-11,-5.9,3.7,0.0\nR004,2002-11-12,-6.1,0.0,11292.63\nR004,2002-11-13,-6.1,0.0,28800.0\nR004,2002-11-14,-9.8,0.0,25256.15\nR004,2002-11-15,-6.0,0.3,23649.67\nR004,2002-11-16,-3.3,4.5,34.79\nR004,2002-11-17,-5.7,7.6,0.0\nR004,2002-11-18,-7.6,0.9,27141.78\nR004,2002-11-19,-9.1,1.4,3089.07\nR004,2002-11-20,-3.2,6.0,0.0\nR004,2002-11-21,-4.5,0.0,10568.24\nR004,2002-11-22,-7.4,0.5,0.0\nR004,2002-11-23,-7.6,6.4,0.0\nR004,2002-11-24,-3.5,1.9,0.0\nR004,2002-11-25,0.2,7.3,0.0\nR004,2002-11-26,-5.7,9.0,564.55\nR004,2002-11-27,-18.9,0.9,20984.45\nR004,2002-11-28,-11.9,14.5,0.0\nR004,2002-11-29,-6.3,20.2,0.0\nR004,2002-11-30,-7.4,2.7,15363.44\nR004,2002-12-01,-8.6,9.5,0.0\nR004,2002-12-02,-14.8,4.1,1380.41\nR004,2002-12-03,-20.3,1.3,14390.8\nR004,2002-12-04,-14.5,6.7,0.0\nR004,2002-12-05,-19.7,0.0,21038.53\nR004,2002-12-06,-26.1,0.0,27229.26\nR004,2002-12-07,-21.8,0.0,23172.16\nR004,2002-12-08,-19.2,0.0,0.0\nR004,2002-12-09,-14.4,0.0,15480.19\nR004,2002-12-10,-8.2,5.6,0.0\nR004,2002-12-11,-14.5,6.0,7088.71\nR004,2002-12-12,-23.9,0.0,25887.06\nR004,2002-12-13,-26.9,0.0,23170.54\nR004,2002-12-14,-27.5,0.0,25686.21\nR004,2002-12-15,-21.9,2.8,0.0\nR004,2002-12-16,-12.9,2.0,1023.15\nR004,2002-12-17,-15.8,0.0,19191.23\nR004,2002-12-18,-19.0,0.0,1766.51\nR004,2002-12-19,-19.7,0.0,20067.76\nR004,2002-12-20,-20.4,0.0,24853.07\nR004,2002-12-21,-26.3,0.0,25423.23\nR004,2002-12-22,-29.0,0.0,25468.55\nR004,2002-12-23,-30.2,0.0,25515.41\nR004,2002-12-24,-30.5,0.0,25563.65\nR004,2002-12-25,-27.2,0.0,10022.69\nR004,2002-12-26,-23.7,0.0,25663.81\nR004,2002-12-27,-23.3,0.0,20003.24\nR004,2002-12-28,-20.4,0.3,10840.04\nR004,2002-12-29,-17.5,0.8,4036.86\nR004,2002-12-30,-20.2,0.0,9529.65\nR004,2002-12-31,-20.5,2.7,14063.44\nR004,2003-01-01,-26.6,0.0,18831.53\nR004,2003-01-02,-23.9,0.0,24913.72\nR004,2003-01-03,-22.5,0.0,14287.72\nR004,2003-01-04,-14.9,0.4,16855.55\nR004,2003-01-05,-21.1,0.0,26206.05\nR004,2003-01-06,-17.9,0.0,15779.02\nR004,2003-01-07,-9.4,1.1,5484.09\nR004,2003-01-08,-6.3,5.1,0.0\nR004,2003-01-09,-3.8,6.2,0.0\nR004,2003-01-10,-0.5,4.7,0.0\nR004,2003-01-11,-1.8,12.3,0.0\nR004,2003-01-12,-12.5,0.0,12360.4\nR004,2003-01-13,-13.9,2.8,0.0\nR004,2003-01-14,-10.5,2.9,0.0\nR004,2003-01-15,-15.6,0.0,21470.31\nR004,2003-01-16,-13.3,2.5,7589.11\nR004,2003-01-17,-20.0,0.0,23319.17\nR004,2003-01-18,-15.3,0.0,26871.16\nR004,2003-01-19,-7.5,4.3,0.0\nR004,2003-01-20,-5.8,1.8,14359.95\nR004,2003-01-21,-8.0,1.2,26210.44\nR004,2003-01-22,-12.0,3.5,21963.98\nR004,2003-01-23,-23.8,0.0,26737.82\nR004,2003-01-24,-28.6,0.0,28988.03\nR004,2003-01-25,-23.7,0.0,17868.06\nR004,2003-01-26,-15.8,0.0,16138.87\nR004,2003-01-27,-18.3,0.0,30112.29\nR004,2003-01-28,-18.8,0.0,30593.22\nR004,2003-01-29,-21.6,0.0,30919.51\nR004,2003-01-30,-21.0,0.0,24637.91\nR004,2003-01-31,-13.0,0.0,27775.59\nR004,2003-02-01,-11.0,0.1,24938.84\nR004,2003-02-02,-7.2,4.8,709.43\nR004,2003-02-03,-9.5,0.0,31724.38\nR004,2003-02-04,-10.5,0.0,27929.66\nR004,2003-02-05,-11.5,0.0,22029.13\nR004,2003-02-06,-13.9,1.3,23569.82\nR004,2003-02-07,-22.6,0.0,32199.61\nR004,2003-02-08,-26.3,0.0,32317.95\nR004,2003-02-09,-25.2,0.0,32400.0\nR004,2003-02-10,-23.5,0.0,32213.61\nR004,2003-02-11,-18.8,0.0,32400.0\nR004,2003-02-12,-17.3,0.0,32400.0\nR004,2003-02-13,-16.1,0.0,22595.08\nR004,2003-02-14,-9.4,2.2,9211.56\nR004,2003-02-15,-9.1,0.4,13473.86\nR004,2003-02-16,-11.5,0.5,16328.01\nR004,2003-02-17,-6.5,1.1,29826.44\nR004,2003-02-18,-9.5,1.7,0.0\nR004,2003-02-19,-16.0,0.0,34740.29\nR004,2003-02-20,-18.3,0.0,34808.7\nR004,2003-02-21,-15.4,0.0,32367.81\nR004,2003-02-22,-9.7,5.0,351.68\nR004,2003-02-23,-10.9,0.4,27862.05\nR004,2003-02-24,-10.1,0.0,13754.13\nR004,2003-02-25,-11.5,2.0,8565.3\nR004,2003-02-26,-14.0,0.2,28640.23\nR004,2003-02-27,-13.5,1.9,2248.21\nR004,2003-02-28,-13.7,0.0,30444.86\nR004,2003-03-01,-15.0,0.0,28703.76\nR004,2003-03-02,-11.8,0.0,38479.91\nR004,2003-03-03,-10.5,0.0,33221.52\nR004,2003-03-04,-12.5,1.2,6432.52\nR004,2003-03-05,-12.0,0.8,35215.64\nR004,2003-03-06,-12.9,1.4,17603.46\nR004,2003-03-07,-13.2,1.8,31360.88\nR004,2003-03-08,-15.7,0.0,33954.49\nR004,2003-03-09,-17.2,0.0,39040.9\nR004,2003-03-10,-12.1,1.1,26367.05\nR004,2003-03-11,-9.4,0.7,28534.95\nR004,2003-03-12,-12.5,0.0,39670.22\nR004,2003-03-13,-13.2,0.0,39988.11\nR004,2003-03-14,-11.6,0.0,40298.52\nR004,2003-03-15,-11.4,0.0,40608.25\nR004,2003-03-16,-9.0,0.0,41300.42\nR004,2003-03-17,-8.3,0.0,41609.07\nR004,2003-03-18,-6.4,0.0,41535.16\nR004,2003-03-19,-3.9,3.2,23403.51\nR004,2003-03-20,-5.7,0.3,41770.74\nR004,2003-03-21,-5.8,0.0,42280.1\nR004,2003-03-22,-1.1,0.0,39899.68\nR004,2003-03-23,-0.5,6.4,0.0\nR004,2003-03-24,-4.6,1.1,27331.14\nR004,2003-03-25,-3.7,0.0,42845.69\nR004,2003-03-26,-0.3,0.0,30006.45\nR004,2003-03-27,-1.2,7.0,527.42\nR004,2003-03-28,-2.1,8.4,0.0\nR004,2003-03-29,-0.9,3.1,17010.83\nR004,2003-03-30,1.1,0.3,34442.22\nR004,2003-03-31,0.5,2.2,22612.79\nR004,2003-04-01,-1.4,0.8,31009.95\nR004,2003-04-02,-2.6,0.0,40958.42\nR004,2003-04-03,-2.0,0.0,39581.96\nR004,2003-04-04,-1.2,2.9,12409.44\nR004,2003-04-05,-2.1,1.6,30198.39\nR004,2003-04-06,-1.7,2.2,45892.89\nR004,2003-04-07,-0.9,0.0,38803.71\nR004,2003-04-08,-0.3,0.0,46446.59\nR004,2003-04-09,-1.5,3.8,21265.77\nR004,2003-04-10,-5.1,0.0,43724.1\nR004,2003-04-11,-3.1,4.7,1873.38\nR004,2003-04-12,1.3,8.8,9586.71\nR004,2003-04-13,-8.1,1.3,46543.33\nR004,2003-04-14,-12.0,0.6,46530.51\nR004,2003-04-15,-13.5,0.0,33994.22\nR004,2003-04-16,-11.5,0.0,49526.39\nR004,2003-04-17,-8.5,0.0,49659.99\nR004,2003-04-18,-3.3,0.0,49793.34\nR004,2003-04-19,-0.9,0.0,49926.31\nR004,2003-04-20,-0.1,1.1,43302.98\nR004,2003-04-21,1.7,2.4,13481.61\nR004,2003-04-22,6.1,2.5,40185.91\nR004,2003-04-23,6.8,6.8,36768.26\nR004,2003-04-24,4.3,0.5,46479.67\nR004,2003-04-25,3.9,0.2,50400.0\nR004,2003-04-26,4.3,15.8,2431.33\nR004,2003-04-27,5.5,4.4,41641.9\nR004,2003-04-28,6.9,0.1,40249.45\nR004,2003-04-29,6.9,9.0,38460.49\nR004,2003-04-30,-0.2,0.3,49260.17\nR004,2003-05-01,1.0,2.7,48900.31\nR004,2003-05-02,2.0,0.0,50400.0\nR004,2003-05-03,5.4,0.0,45833.96\nR004,2003-05-04,9.4,0.0,48821.62\nR004,2003-05-05,10.1,0.0,48586.69\nR004,2003-05-06,11.5,0.2,50973.53\nR004,2003-05-07,9.6,3.5,28143.11\nR004,2003-05-08,8.1,6.0,36894.72\nR004,2003-05-09,9.1,0.0,33137.41\nR004,2003-05-10,12.6,0.0,51922.76\nR004,2003-05-11,12.0,0.3,51613.1\nR004,2003-05-12,9.5,0.0,53044.4\nR004,2003-05-13,10.8,0.0,53396.58\nR004,2003-05-14,12.4,0.0,54189.35\nR004,2003-05-15,13.7,0.0,54883.33\nR004,2003-05-16,13.4,0.0,54045.4\nR004,2003-05-17,7.6,9.4,5453.31\nR004,2003-05-18,9.8,1.7,50828.94\nR004,2003-05-19,7.7,26.2,14044.26\nR004,2003-05-20,12.2,0.2,47615.28\nR004,2003-05-21,11.0,11.7,37923.53\nR004,2003-05-22,10.5,0.6,44991.9\nR004,2003-05-23,9.3,6.9,35212.93\nR004,2003-05-24,9.6,0.0,56854.56\nR004,2003-05-25,11.7,0.0,57385.81\nR004,2003-05-26,10.7,29.0,1258.24\nR004,2003-05-27,11.5,12.2,38399.33\nR004,2003-05-28,11.3,0.4,34123.45\nR004,2003-05-29,15.9,6.9,50187.48\nR004,2003-05-30,15.1,11.6,30448.77\nR004,2003-05-31,11.6,29.8,0.0\nR004,2003-06-01,12.1,27.0,10536.55\nR004,2003-06-02,15.3,2.7,46972.28\nR004,2003-06-03,16.0,0.0,53248.4\nR004,2003-06-04,16.3,6.9,38038.8\nR004,2003-06-05,15.8,19.7,13979.77\nR004,2003-06-06,17.4,0.0,57600.0\nR004,2003-06-07,20.4,0.1,57600.0\nR004,2003-06-08,16.0,5.6,13987.07\nR004,2003-06-09,12.7,8.4,0.0\nR004,2003-06-10,15.4,0.0,29736.57\nR004,2003-06-11,19.0,0.0,57600.0\nR004,2003-06-12,20.6,0.0,43829.19\nR004,2003-06-13,18.6,8.1,34244.05\nR004,2003-06-14,16.1,8.5,467.1\nR004,2003-06-15,17.1,0.0,45031.26\nR004,2003-06-16,19.7,0.0,52712.68\nR004,2003-06-17,21.6,0.0,57600.0\nR004,2003-06-18,16.2,10.8,27006.93\nR004,2003-06-19,11.9,12.6,21509.68\nR004,2003-06-20,11.6,13.1,25468.39\nR004,2003-06-21,14.5,0.0,45694.81\nR004,2003-06-22,16.7,0.0,55316.48\nR004,2003-06-23,12.2,41.1,0.0\nR004,2003-06-24,11.1,25.2,5590.01\nR004,2003-06-25,14.5,1.2,49860.96\nR004,2003-06-26,14.7,5.0,41925.01\nR004,2003-06-27,15.3,0.0,57047.7\nR004,2003-06-28,15.5,0.0,57600.0\nR004,2003-06-29,16.1,0.0,56598.61\nR004,2003-06-30,18.0,0.0,57600.0\nR004,2003-07-01,17.9,1.2,39703.3\nR004,2003-07-02,16.4,12.9,16262.06\nR004,2003-07-03,16.6,1.2,50595.5\nR004,2003-07-04,16.5,0.0,57600.0\nR004,2003-07-05,18.5,0.0,57600.0\nR004,2003-07-06,20.1,0.0,57600.0\nR004,2003-07-07,20.4,0.0,51504.52\nR004,2003-07-08,21.8,0.0,51344.12\nR004,2003-07-09,21.3,0.5,44717.63\nR004,2003-07-10,19.7,9.7,42866.9\nR004,2003-07-11,16.1,11.5,0.0\nR004,2003-07-12,13.9,1.0,14138.0\nR004,2003-07-13,11.3,1.6,0.0\nR004,2003-07-14,13.9,0.0,41996.81\nR004,2003-07-15,15.5,0.0,41944.62\nR004,2003-07-16,16.1,3.2,49509.01\nR004,2003-07-17,15.0,10.7,27924.55\nR004,2003-07-18,16.7,9.1,34705.23\nR004,2003-07-19,15.5,10.4,41844.54\nR004,2003-07-20,13.8,6.0,42355.31\nR004,2003-07-21,17.0,0.0,52768.28\nR004,2003-07-22,18.8,0.0,47239.68\nR004,2003-07-23,16.4,18.3,4685.85\nR004,2003-07-24,16.7,6.9,34313.49\nR004,2003-07-25,9.8,19.3,0.0\nR004,2003-07-26,10.4,0.7,15072.15\nR004,2003-07-27,12.3,0.0,45851.85\nR004,2003-07-28,13.7,0.0,42481.92\nR004,2003-07-29,12.6,2.9,38250.48\nR004,2003-07-30,13.5,7.3,31141.92\nR004,2003-07-31,14.4,1.4,41712.21\nR004,2003-08-01,17.2,0.0,49745.2\nR004,2003-08-02,17.8,1.7,50039.4\nR004,2003-08-03,15.8,9.6,31889.53\nR004,2003-08-04,14.8,4.1,37210.51\nR004,2003-08-05,14.5,2.9,24626.83\nR004,2003-08-06,15.0,7.4,49293.1\nR004,2003-08-07,13.9,5.3,32688.68\nR004,2003-08-08,14.6,3.4,32960.04\nR004,2003-08-09,14.1,0.8,25209.12\nR004,2003-08-10,14.6,2.4,40896.07\nR004,2003-08-11,11.5,0.0,50400.0\nR004,2003-08-12,6.4,8.7,15016.93\nR004,2003-08-13,7.9,0.0,1127.85\nR004,2003-08-14,12.5,0.0,45919.73\nR004,2003-08-15,14.3,0.5,43200.0\nR004,2003-08-16,14.9,0.3,50162.2\nR004,2003-08-17,15.4,0.0,50400.0\nR004,2003-08-18,14.1,7.0,47367.34\nR004,2003-08-19,13.5,0.0,50368.65\nR004,2003-08-20,13.6,0.9,43067.33\nR004,2003-08-21,14.2,0.1,35758.86\nR004,2003-08-22,15.4,0.0,50064.31\nR004,2003-08-23,14.5,0.0,49962.69\nR004,2003-08-24,14.1,0.6,48044.03\nR004,2003-08-25,13.5,6.8,38935.07\nR004,2003-08-26,14.9,0.0,41256.66\nR004,2003-08-27,15.2,0.0,49556.71\nR004,2003-08-28,14.5,0.0,49398.86\nR004,2003-08-29,15.0,0.0,48637.31\nR004,2003-08-30,15.9,0.0,48155.01\nR004,2003-08-31,16.3,0.0,47674.17\nR004,2003-09-01,12.4,21.5,38683.28\nR004,2003-09-02,10.9,0.2,43127.39\nR004,2003-09-03,12.6,0.3,45759.67\nR004,2003-09-04,15.5,0.0,45872.56\nR004,2003-09-05,15.9,0.0,45214.86\nR004,2003-09-06,13.8,0.1,30417.58\nR004,2003-09-07,12.4,0.0,44344.58\nR004,2003-09-08,16.5,0.0,44029.41\nR004,2003-09-09,16.1,0.0,43715.56\nR004,2003-09-10,13.3,9.1,34854.24\nR004,2003-09-11,10.6,5.9,38232.31\nR004,2003-09-12,10.5,0.7,36000.0\nR004,2003-09-13,11.4,9.2,43200.0\nR004,2003-09-14,6.5,7.4,31433.52\nR004,2003-09-15,6.7,1.9,21973.13\nR004,2003-09-16,6.2,4.0,34193.38\nR004,2003-09-17,7.1,0.0,32754.68\nR004,2003-09-18,11.0,0.0,38094.71\nR004,2003-09-19,9.7,0.0,43200.0\nR004,2003-09-20,10.5,0.1,43200.0\nR004,2003-09-21,10.1,0.0,42357.7\nR004,2003-09-22,10.7,21.7,19778.75\nR004,2003-09-23,8.0,8.6,19996.61\nR004,2003-09-24,2.7,0.1,42994.15\nR004,2003-09-25,1.3,0.7,35041.29\nR004,2003-09-26,0.6,11.6,19855.66\nR004,2003-09-27,-0.6,0.0,18828.96\nR004,2003-09-28,0.0,2.5,0.0\nR004,2003-09-29,1.0,0.0,22121.23\nR004,2003-09-30,2.0,0.0,27376.14\nR004,2003-10-01,4.3,0.0,23434.98\nR004,2003-10-02,3.5,0.0,37359.64\nR004,2003-10-03,4.6,0.0,23717.58\nR004,2003-10-04,6.6,0.0,36315.1\nR004,2003-10-05,6.7,0.0,36000.0\nR004,2003-10-06,5.6,0.7,35227.84\nR004,2003-10-07,0.2,5.9,11937.47\nR004,2003-10-08,-4.8,0.0,36000.0\nR004,2003-10-09,-6.5,0.0,35645.58\nR004,2003-10-10,-6.3,0.0,36000.0\nR004,2003-10-11,-5.2,0.0,36000.0\nR004,2003-10-12,-3.7,0.0,35078.5\nR004,2003-10-13,-0.6,0.0,36000.0\nR004,2003-10-14,-0.3,0.0,36000.0\nR004,2003-10-15,0.2,0.0,36000.0\nR004,2003-10-16,1.0,0.0,35169.93\nR004,2003-10-17,1.7,0.0,35261.69\nR004,2003-10-18,4.9,1.6,14702.23\nR004,2003-10-19,2.1,2.5,10273.99\nR004,2003-10-20,1.6,0.0,19138.48\nR004,2003-10-21,2.7,0.4,34034.81\nR004,2003-10-22,2.4,4.8,1232.63\nR004,2003-10-23,0.7,0.0,30356.13\nR004,2003-10-24,1.2,7.6,2658.65\nR004,2003-10-25,-2.0,15.9,434.59\nR004,2003-10-26,-2.6,2.6,16804.46\nR004,2003-10-27,4.9,11.2,0.0\nR004,2003-10-28,1.4,15.7,4401.11\nR004,2003-10-29,1.2,9.8,0.0\nR004,2003-10-30,-0.4,2.4,8298.26\nR004,2003-10-31,-4.3,8.7,1112.17\nR004,2003-11-01,-4.6,2.6,4969.0\nR004,2003-11-02,-3.4,10.1,0.0\nR004,2003-11-03,-9.0,0.0,8516.38\nR004,2003-11-04,-13.0,0.0,7022.59\nR004,2003-11-05,-17.9,0.0,30159.69\nR004,2003-11-06,-19.9,0.0,29789.19\nR004,2003-11-07,-10.2,0.0,20877.75\nR004,2003-11-08,-11.4,0.0,25366.02\nR004,2003-11-09,-11.0,0.0,27495.93\nR004,2003-11-10,-11.1,0.0,23673.54\nR004,2003-11-11,-3.8,6.9,0.0\nR004,2003-11-12,-3.1,10.1,0.0\nR004,2003-11-13,-9.9,3.7,57.49\nR004,2003-11-14,-16.0,0.0,11170.63\nR004,2003-11-15,-11.1,8.0,0.0\nR004,2003-11-16,-11.2,2.6,18279.82\nR004,2003-11-17,-11.8,0.0,9255.53\nR004,2003-11-18,-10.1,2.9,0.0\nR004,2003-11-19,-18.6,0.0,25040.0\nR004,2003-11-20,-16.2,0.0,21793.55\nR004,2003-11-21,-17.1,0.0,19673.0\nR004,2003-11-22,-11.0,0.0,25846.82\nR004,2003-11-23,-11.0,4.0,0.0\nR004,2003-11-24,-3.9,6.2,0.0\nR004,2003-11-25,-4.9,6.6,0.0\nR004,2003-11-26,-14.8,2.3,0.0\nR004,2003-11-27,-14.8,0.9,3120.08\nR004,2003-11-28,-10.3,3.3,0.0\nR004,2003-11-29,-19.6,1.4,9556.15\nR004,2003-11-30,-15.9,4.4,0.0\nR004,2003-12-01,-17.6,0.2,20752.76\nR004,2003-12-02,-16.7,9.9,0.0\nR004,2003-12-03,-18.0,2.8,22702.75\nR004,2003-12-04,-28.2,0.0,24191.61\nR004,2003-12-05,-20.8,3.7,0.0\nR004,2003-12-06,-12.6,5.8,0.0\nR004,2003-12-07,-8.4,2.2,0.0\nR004,2003-12-08,-6.5,1.3,21222.01\nR004,2003-12-09,-10.3,0.0,12359.29\nR004,2003-12-10,-6.5,0.0,24197.88\nR004,2003-12-11,-6.4,1.2,4420.83\nR004,2003-12-12,-6.3,6.3,0.0\nR004,2003-12-13,-13.6,0.0,3106.73\nR004,2003-12-14,-14.4,0.0,23157.02\nR004,2003-12-15,-10.8,2.0,0.0\nR004,2003-12-16,-11.7,2.3,14558.08\nR004,2003-12-17,-20.1,0.0,22930.25\nR004,2003-12-18,-19.5,0.0,25294.8\nR004,2003-12-19,-22.3,0.0,25861.68\nR004,2003-12-20,-19.6,0.0,25903.48\nR004,2003-12-21,-8.0,0.5,23465.89\nR004,2003-12-22,-3.9,5.6,0.0\nR004,2003-12-23,-10.5,2.9,17087.85\nR004,2003-12-24,-11.5,0.0,12214.75\nR004,2003-12-25,-15.8,0.0,25604.09\nR004,2003-12-26,-18.2,0.0,25127.92\nR004,2003-12-27,-12.8,0.8,0.0\nR004,2003-12-28,-10.7,0.0,14397.45\nR004,2003-12-29,-10.9,2.0,0.0\nR004,2003-12-30,-8.9,2.1,8185.26\nR004,2003-12-31,-8.1,0.7,783.55\nR004,2004-01-01,-6.8,5.0,0.0\nR004,2004-01-02,-8.0,4.7,1992.6\nR004,2004-01-03,-8.6,3.5,0.0\nR004,2004-01-04,-11.3,0.7,0.0\nR004,2004-01-05,-13.8,1.8,17998.68\nR004,2004-01-06,-23.5,0.0,25187.59\nR004,2004-01-07,-20.6,0.0,19512.89\nR004,2004-01-08,-20.0,0.0,26222.79\nR004,2004-01-09,-20.5,0.7,16308.09\nR004,2004-01-10,-22.2,0.0,25669.04\nR004,2004-01-11,-23.6,0.0,25128.86\nR004,2004-01-12,-14.8,3.4,12259.55\nR004,2004-01-13,-12.0,0.6,463.86\nR004,2004-01-14,-17.4,0.0,14231.99\nR004,2004-01-15,-24.3,0.0,27504.82\nR004,2004-01-16,-26.5,0.0,27604.47\nR004,2004-01-17,-27.3,0.0,27632.39\nR004,2004-01-18,-26.0,0.0,27693.49\nR004,2004-01-19,-25.8,0.0,27757.94\nR004,2004-01-20,-25.5,0.0,28121.23\nR004,2004-01-21,-25.9,0.0,28135.91\nR004,2004-01-22,-24.3,0.0,25301.99\nR004,2004-01-23,-23.3,0.0,28640.23\nR004,2004-01-24,-20.6,0.0,29126.17\nR004,2004-01-25,-21.4,0.0,28212.37\nR004,2004-01-26,-16.1,1.3,8795.89\nR004,2004-01-27,-11.1,2.9,0.0\nR004,2004-01-28,-14.6,3.3,0.0\nR004,2004-01-29,-21.0,0.0,6222.35\nR004,2004-01-30,-24.8,0.0,31092.24\nR004,2004-01-31,-23.7,0.0,28273.95\nR004,2004-02-01,-21.2,1.3,16044.87\nR004,2004-02-02,-14.9,3.2,0.0\nR004,2004-02-03,-12.9,1.4,25991.7\nR004,2004-02-04,-14.9,0.0,31816.77\nR004,2004-02-05,-17.4,0.0,31933.46\nR004,2004-02-06,-18.1,0.0,32049.67\nR004,2004-02-07,-16.3,0.0,28938.8\nR004,2004-02-08,-13.9,0.4,3556.11\nR004,2004-02-09,-9.5,1.3,9512.65\nR004,2004-02-10,-4.4,0.0,6774.07\nR004,2004-02-11,-4.3,2.2,28765.67\nR004,2004-02-12,-3.9,4.5,10946.49\nR004,2004-02-13,-7.9,3.0,4505.0\nR004,2004-02-14,-8.5,1.2,24578.57\nR004,2004-02-15,-8.6,2.9,266.05\nR004,2004-02-16,-9.0,6.6,3029.88\nR004,2004-02-17,-3.8,3.9,26301.94\nR004,2004-02-18,-10.8,4.2,24796.06\nR004,2004-02-19,-12.6,0.0,32400.0\nR004,2004-02-20,-9.2,1.1,18965.0\nR004,2004-02-21,-5.4,1.8,1649.17\nR004,2004-02-22,-11.9,3.3,10724.13\nR004,2004-02-23,-7.7,12.4,0.0\nR004,2004-02-24,-6.7,3.8,0.0\nR004,2004-02-25,-3.4,2.1,30121.77\nR004,2004-02-26,-5.8,3.8,2362.76\nR004,2004-02-27,-12.2,0.0,33368.79\nR004,2004-02-28,-11.7,0.0,28308.38\nR004,2004-02-29,-14.5,0.0,9581.12\nR004,2004-03-01,-12.1,0.0,12451.9\nR004,2004-03-02,-6.1,0.0,31486.02\nR004,2004-03-03,-6.1,0.0,28155.54\nR004,2004-03-04,-7.7,0.0,38786.86\nR004,2004-03-05,-6.9,0.0,38899.3\nR004,2004-03-06,-3.9,0.0,39011.52\nR004,2004-03-07,0.3,20.6,126.94\nR004,2004-03-08,-3.8,10.6,34427.46\nR004,2004-03-09,-8.4,5.8,16969.37\nR004,2004-03-10,-7.8,7.7,614.68\nR004,2004-03-11,-7.4,3.7,31095.53\nR004,2004-03-12,-5.5,4.6,755.59\nR004,2004-03-13,-11.7,4.3,32400.0\nR004,2004-03-14,-17.4,0.0,39600.0\nR004,2004-03-15,-17.4,0.0,36831.9\nR004,2004-03-16,-18.8,0.0,41626.56\nR004,2004-03-17,-15.1,0.0,41811.76\nR004,2004-03-18,-10.4,0.9,33380.6\nR004,2004-03-19,-4.8,0.3,24223.2\nR004,2004-03-20,-7.9,0.5,42234.24\nR004,2004-03-21,-7.8,0.2,32354.41\nR004,2004-03-22,-7.3,0.0,42519.82\nR004,2004-03-23,-4.7,0.0,42650.73\nR004,2004-03-24,0.3,11.3,0.0\nR004,2004-03-25,-4.4,3.6,6358.03\nR004,2004-03-26,-12.1,0.0,39618.86\nR004,2004-03-27,-12.6,0.7,30401.48\nR004,2004-03-28,-15.5,0.0,41883.29\nR004,2004-03-29,-10.4,0.0,35811.68\nR004,2004-03-30,-4.0,0.6,13386.15\nR004,2004-03-31,-1.0,9.8,0.0\nR004,2004-04-01,3.6,0.0,34097.14\nR004,2004-04-02,5.1,0.0,39972.69\nR004,2004-04-03,1.2,0.9,63.77\nR004,2004-04-04,1.4,0.0,36000.0\nR004,2004-04-05,2.1,0.0,45974.67\nR004,2004-04-06,4.0,0.2,39380.68\nR004,2004-04-07,0.6,4.9,431.56\nR004,2004-04-08,0.1,0.0,46862.5\nR004,2004-04-09,-0.5,3.9,23485.86\nR004,2004-04-10,-5.0,1.1,36393.29\nR004,2004-04-11,-5.8,0.0,25494.34\nR004,2004-04-12,-2.9,10.2,13970.47\nR004,2004-04-13,1.4,44.4,0.0\nR004,2004-04-14,0.9,19.9,0.0\nR004,2004-04-15,5.0,27.5,0.0\nR004,2004-04-16,1.2,22.3,12198.76\nR004,2004-04-17,-2.3,0.5,14333.98\nR004,2004-04-18,-2.9,0.0,49900.79\nR004,2004-04-19,-1.0,5.6,3782.59\nR004,2004-04-20,-0.2,4.2,31856.81\nR004,2004-04-21,0.5,0.0,50297.95\nR004,2004-04-22,1.8,0.0,49296.03\nR004,2004-04-23,3.1,0.0,40742.3\nR004,2004-04-24,6.2,1.1,39539.1\nR004,2004-04-25,6.7,1.6,18898.22\nR004,2004-04-26,2.8,12.2,17375.93\nR004,2004-04-27,-1.8,2.8,43200.0\nR004,2004-04-28,-2.9,0.4,43690.65\nR004,2004-04-29,1.5,0.5,41511.51\nR004,2004-04-30,-0.5,11.9,32844.62\nR004,2004-05-01,-2.2,0.0,47291.49\nR004,2004-05-02,4.4,0.0,1640.87\nR004,2004-05-03,12.6,0.0,47128.27\nR004,2004-05-04,15.0,0.0,51028.31\nR004,2004-05-05,15.4,1.1,46855.6\nR004,2004-05-06,8.9,9.4,47915.58\nR004,2004-05-07,4.9,0.3,43333.92\nR004,2004-05-08,3.9,2.0,47493.83\nR004,2004-05-09,8.0,0.7,38535.4\nR004,2004-05-10,3.2,7.7,37311.4\nR004,2004-05-11,9.7,0.4,52627.15\nR004,2004-05-12,13.5,0.0,51245.77\nR004,2004-05-13,13.6,1.0,48225.43\nR004,2004-05-14,13.8,0.0,54429.99\nR004,2004-05-15,15.6,0.0,54060.67\nR004,2004-05-16,17.5,0.0,54096.8\nR004,2004-05-17,19.9,0.0,55474.37\nR004,2004-05-18,21.3,0.0,55974.61\nR004,2004-05-19,22.0,0.0,56033.38\nR004,2004-05-20,22.1,1.0,56488.37\nR004,2004-05-21,16.6,1.7,32192.93\nR004,2004-05-22,12.8,4.9,6970.06\nR004,2004-05-23,13.6,0.6,44884.58\nR004,2004-05-24,10.4,14.4,8972.52\nR004,2004-05-25,9.0,5.8,53641.74\nR004,2004-05-26,11.4,0.2,55432.82\nR004,2004-05-27,11.7,0.1,54000.0\nR004,2004-05-28,14.0,0.0,57600.0\nR004,2004-05-29,17.3,0.0,57600.0\nR004,2004-05-30,17.9,0.1,47330.69\nR004,2004-05-31,13.0,10.7,19749.58\nR004,2004-06-01,7.1,10.9,50445.21\nR004,2004-06-02,8.3,0.6,52556.22\nR004,2004-06-03,15.8,0.0,57600.0\nR004,2004-06-04,19.3,0.0,57600.0\nR004,2004-06-05,20.5,0.0,54451.0\nR004,2004-06-06,19.8,1.0,49462.51\nR004,2004-06-07,12.3,20.7,9779.82\nR004,2004-06-08,13.1,0.0,56893.73\nR004,2004-06-09,14.5,0.0,54292.76\nR004,2004-06-10,13.6,4.6,45114.3\nR004,2004-06-11,13.1,1.0,41028.14\nR004,2004-06-12,12.1,9.4,41702.83\nR004,2004-06-13,13.7,5.8,44161.07\nR004,2004-06-14,16.6,0.0,57413.39\nR004,2004-06-15,16.2,2.7,20551.38\nR004,2004-06-16,18.2,0.8,47855.53\nR004,2004-06-17,17.7,3.3,55667.49\nR004,2004-06-18,15.1,0.1,55548.8\nR004,2004-06-19,16.8,0.9,54736.44\nR004,2004-06-20,18.1,0.0,57600.0\nR004,2004-06-21,17.4,1.9,54327.12\nR004,2004-06-22,18.2,0.0,44643.78\nR004,2004-06-23,16.9,3.9,41061.5\nR004,2004-06-24,15.2,0.8,25506.34\nR004,2004-06-25,12.1,5.3,19328.15\nR004,2004-06-26,14.8,0.1,54000.0\nR004,2004-06-27,16.6,0.0,46883.43\nR004,2004-06-28,16.1,9.2,34410.47\nR004,2004-06-29,17.8,0.0,52212.46\nR004,2004-06-30,17.7,3.2,34922.08\nR004,2004-07-01,15.8,13.4,24251.89\nR004,2004-07-02,15.7,7.2,19117.43\nR004,2004-07-03,17.4,1.8,36409.88\nR004,2004-07-04,13.6,12.3,8630.2\nR004,2004-07-05,11.1,0.2,31971.81\nR004,2004-07-06,11.4,1.6,55729.14\nR004,2004-07-07,13.8,0.0,57193.35\nR004,2004-07-08,17.5,0.0,57600.0\nR004,2004-07-09,16.0,0.1,26254.81\nR004,2004-07-10,16.9,2.5,51857.15\nR004,2004-07-11,19.0,0.6,48579.05\nR004,2004-07-12,20.1,0.0,56508.42\nR004,2004-07-13,20.9,0.2,55690.04\nR004,2004-07-14,23.1,0.0,55632.74\nR004,2004-07-15,21.3,3.6,33212.08\nR004,2004-07-16,19.8,5.5,33704.02\nR004,2004-07-17,20.0,0.1,48418.86\nR004,2004-07-18,18.0,0.0,44839.12\nR004,2004-07-19,15.3,2.5,55319.31\nR004,2004-07-20,14.3,1.5,55160.35\nR004,2004-07-21,15.2,2.3,50516.58\nR004,2004-07-22,15.2,3.9,25891.79\nR004,2004-07-23,16.6,0.3,44779.28\nR004,2004-07-24,16.8,1.0,34483.47\nR004,2004-07-25,14.9,13.5,7522.99\nR004,2004-07-26,15.7,1.4,40951.81\nR004,2004-07-27,16.1,0.0,49842.43\nR004,2004-07-28,16.4,0.0,54304.87\nR004,2004-07-29,16.5,0.0,54000.0\nR004,2004-07-30,17.9,0.0,53919.54\nR004,2004-07-31,16.9,5.2,49643.06\nR004,2004-08-01,17.6,0.5,49825.58\nR004,2004-08-02,19.2,0.2,45636.68\nR004,2004-08-03,19.3,10.9,44061.87\nR004,2004-08-04,17.0,15.9,36702.06\nR004,2004-08-05,14.6,8.6,36504.79\nR004,2004-08-06,14.1,5.3,34509.38\nR004,2004-08-07,10.6,17.4,0.0\nR004,2004-08-08,7.9,0.9,22364.12\nR004,2004-08-09,8.7,8.0,31178.15\nR004,2004-08-10,10.7,0.0,49748.19\nR004,2004-08-11,14.5,0.0,50539.9\nR004,2004-08-12,16.9,0.7,45512.75\nR004,2004-08-13,18.0,0.0,45688.33\nR004,2004-08-14,19.2,0.0,44694.58\nR004,2004-08-15,20.1,0.0,50400.0\nR004,2004-08-16,18.5,0.0,50400.0\nR004,2004-08-17,18.4,0.0,50400.0\nR004,2004-08-18,19.1,0.0,50390.62\nR004,2004-08-19,12.8,6.5,13618.16\nR004,2004-08-20,13.2,0.0,44865.81\nR004,2004-08-21,15.3,0.2,48574.64\nR004,2004-08-22,11.8,0.7,21011.1\nR004,2004-08-23,9.6,1.9,47130.39\nR004,2004-08-24,9.1,1.4,48472.97\nR004,2004-08-25,9.8,0.2,43261.74\nR004,2004-08-26,9.9,0.0,33208.47\nR004,2004-08-27,10.9,0.0,49483.31\nR004,2004-08-28,13.3,0.0,48645.27\nR004,2004-08-29,15.1,0.0,45209.15\nR004,2004-08-30,16.7,0.0,44496.5\nR004,2004-08-31,14.1,2.9,18999.89\nR004,2004-09-01,10.6,1.3,11438.92\nR004,2004-09-02,9.1,5.4,14897.91\nR004,2004-09-03,9.9,0.0,37304.27\nR004,2004-09-04,12.0,0.6,35472.19\nR004,2004-09-05,13.8,0.0,45306.96\nR004,2004-09-06,14.7,0.0,33078.52\nR004,2004-09-07,14.9,5.1,32025.23\nR004,2004-09-08,10.4,20.3,5529.71\nR004,2004-09-09,12.0,2.8,41102.05\nR004,2004-09-10,10.9,0.0,43023.28\nR004,2004-09-11,10.7,0.0,39514.84\nR004,2004-09-12,15.4,0.0,32020.51\nR004,2004-09-13,13.2,3.9,40551.86\nR004,2004-09-14,6.5,15.3,9542.4\nR004,2004-09-15,6.7,2.3,11798.28\nR004,2004-09-16,10.1,0.7,41338.81\nR004,2004-09-17,8.9,1.4,21639.23\nR004,2004-09-18,13.2,1.2,32539.22\nR004,2004-09-19,13.8,0.0,43200.0\nR004,2004-09-20,12.7,2.2,39492.92\nR004,2004-09-21,11.9,4.5,30036.76\nR004,2004-09-22,5.8,15.2,6048.82\nR004,2004-09-23,4.4,9.4,28713.2\nR004,2004-09-24,1.4,4.9,36700.12\nR004,2004-09-25,-0.7,0.7,40396.64\nR004,2004-09-26,-0.5,0.9,35393.79\nR004,2004-09-27,-1.1,14.9,69.95\nR004,2004-09-28,-1.5,0.0,31896.41\nR004,2004-09-29,-0.2,0.0,39766.47\nR004,2004-09-30,2.2,0.0,38064.3\nR004,2004-10-01,10.3,1.5,33768.94\nR004,2004-10-02,11.1,1.1,22277.41\nR004,2004-10-03,12.9,0.0,36301.12\nR004,2004-10-04,13.5,0.0,36355.78\nR004,2004-10-05,12.3,0.0,35114.74\nR004,2004-10-06,11.4,5.3,4785.7\nR004,2004-10-07,5.1,13.2,0.0\nR004,2004-10-08,0.7,2.9,19582.28\nR004,2004-10-09,1.2,8.9,3946.08\nR004,2004-10-10,3.8,4.8,240.0\nR004,2004-10-11,6.5,0.0,35579.34\nR004,2004-10-12,5.5,0.0,36000.0\nR004,2004-10-13,7.9,0.0,36000.0\nR004,2004-10-14,9.6,0.0,35886.64\nR004,2004-10-15,3.1,1.3,13707.59\nR004,2004-10-16,0.1,1.7,13180.72\nR004,2004-10-17,1.2,4.7,8844.25\nR004,2004-10-18,-2.5,1.4,21093.46\nR004,2004-10-19,-3.9,0.2,6368.27\nR004,2004-10-20,-2.7,3.9,1893.72\nR004,2004-10-21,0.4,0.3,24917.58\nR004,2004-10-22,0.3,4.5,4345.54\nR004,2004-10-23,-4.0,0.4,22829.73\nR004,2004-10-24,-5.4,0.0,33899.46\nR004,2004-10-25,-1.3,0.0,13376.51\nR004,2004-10-26,2.0,0.0,31230.81\nR004,2004-10-27,2.2,4.9,26140.3\nR004,2004-10-28,-3.4,2.4,19047.34\nR004,2004-10-29,-3.7,0.0,29043.89\nR004,2004-10-30,-1.9,0.7,4290.16\nR004,2004-10-31,0.4,7.4,0.0\nR004,2004-11-01,-3.6,1.2,3427.02\nR004,2004-11-02,-7.0,0.8,12191.67\nR004,2004-11-03,-9.1,0.0,30079.56\nR004,2004-11-04,-5.4,10.6,0.0\nR004,2004-11-05,-4.0,2.2,21481.23\nR004,2004-11-06,-4.5,7.1,22744.4\nR004,2004-11-07,-5.2,11.3,670.06\nR004,2004-11-08,-10.7,3.0,623.82\nR004,2004-11-09,-13.3,0.0,689.61\nR004,2004-11-10,-11.9,1.8,18247.33\nR004,2004-11-11,-3.8,8.4,0.0\nR004,2004-11-12,-4.7,3.3,812.11\nR004,2004-11-13,-3.1,6.4,0.0\nR004,2004-11-14,-4.4,0.0,26371.59\nR004,2004-11-15,-9.1,0.0,28800.0\nR004,2004-11-16,-10.2,0.0,28800.0\nR004,2004-11-17,-4.7,0.0,28800.0\nR004,2004-11-18,-9.8,0.0,28800.0\nR004,2004-11-19,-12.4,0.0,28800.0\nR004,2004-11-20,-9.4,0.2,8216.9\nR004,2004-11-21,-2.6,0.1,841.45\nR004,2004-11-22,-7.6,0.0,22838.52\nR004,2004-11-23,-7.8,0.0,28556.25\nR004,2004-11-24,-14.2,0.0,28488.13\nR004,2004-11-25,-16.3,0.0,28422.84\nR004,2004-11-26,-13.7,0.0,19112.84\nR004,2004-11-27,-4.7,0.8,1730.44\nR004,2004-11-28,-6.1,0.3,16218.43\nR004,2004-11-29,-0.5,3.7,0.0\nR004,2004-11-30,-4.3,3.7,0.0\nR004,2004-12-01,-8.1,3.5,0.0\nR004,2004-12-02,-12.0,2.1,14384.08\nR004,2004-12-03,-20.9,0.0,4302.87\nR004,2004-12-04,-10.1,6.9,0.0\nR004,2004-12-05,-8.5,5.8,0.0\nR004,2004-12-06,-3.2,5.9,0.0\nR004,2004-12-07,-4.4,6.8,0.0\nR004,2004-12-08,-11.9,7.9,0.0\nR004,2004-12-09,-6.2,17.1,0.0\nR004,2004-12-10,-8.3,7.1,3239.45\nR004,2004-12-11,-13.3,0.0,16308.76\nR004,2004-12-12,-7.6,7.7,14757.94\nR004,2004-12-13,-9.0,4.0,0.0\nR004,2004-12-14,-12.0,0.0,16170.42\nR004,2004-12-15,-9.7,3.7,0.0\nR004,2004-12-16,-13.8,0.9,15255.82\nR004,2004-12-17,-18.5,0.1,10292.46\nR004,2004-12-18,-14.7,0.7,2507.41\nR004,2004-12-19,-11.8,0.0,8023.83\nR004,2004-12-20,-13.4,2.2,0.0\nR004,2004-12-21,-17.6,0.4,1119.81\nR004,2004-12-22,-22.6,0.0,20048.74\nR004,2004-12-23,-23.4,0.2,192.44\nR004,2004-12-24,-27.7,0.0,25058.2\nR004,2004-12-25,-31.8,0.0,21868.44\nR004,2004-12-26,-33.0,0.0,25679.77\nR004,2004-12-27,-33.6,0.0,25207.83\nR004,2004-12-28,-34.1,0.0,25783.51\nR004,2004-12-29,-34.4,0.0,25465.89\nR004,2004-12-30,-31.3,0.0,24893.72\nR004,2004-12-31,-23.2,1.3,14400.0\nR004,2005-01-01,-11.1,9.0,0.0\nR004,2005-01-02,-16.7,1.6,13375.41\nR004,2005-01-03,-12.6,0.5,12339.16\nR004,2005-01-04,-12.2,0.7,246.21\nR004,2005-01-05,-11.3,5.5,0.0\nR004,2005-01-06,-15.3,0.3,25230.85\nR004,2005-01-07,-22.8,0.0,25721.89\nR004,2005-01-08,-23.5,0.0,25200.0\nR004,2005-01-09,-18.3,0.0,13904.36\nR004,2005-01-10,-15.2,0.3,19336.41\nR004,2005-01-11,-17.5,0.0,26629.88\nR004,2005-01-12,-15.7,0.0,25171.56\nR004,2005-01-13,-13.5,0.0,22429.09\nR004,2005-01-14,-8.8,0.2,2301.7\nR004,2005-01-15,-9.2,0.0,4477.71\nR004,2005-01-16,-8.3,0.3,0.0\nR004,2005-01-17,-8.4,0.0,5956.0\nR004,2005-01-18,-11.8,0.0,27339.22\nR004,2005-01-19,-14.3,0.0,10464.59\nR004,2005-01-20,-17.5,0.2,10153.91\nR004,2005-01-21,-15.6,1.1,0.0\nR004,2005-01-22,-19.6,0.0,9107.93\nR004,2005-01-23,-24.3,0.0,20059.8\nR004,2005-01-24,-30.4,0.0,29141.02\nR004,2005-01-25,-32.4,0.0,29160.96\nR004,2005-01-26,-34.1,0.0,29114.96\nR004,2005-01-27,-33.0,0.0,26174.43\nR004,2005-01-28,-25.8,1.3,10085.91\nR004,2005-01-29,-28.5,0.0,31044.56\nR004,2005-01-30,-28.0,0.0,30650.69\nR004,2005-01-31,-24.0,0.0,1841.86\nR004,2005-02-01,-27.9,0.0,31549.28\nR004,2005-02-02,-27.9,0.0,25507.05\nR004,2005-02-03,-26.5,0.0,24304.8\nR004,2005-02-04,-26.7,0.0,31896.85\nR004,2005-02-05,-28.1,0.0,31625.38\nR004,2005-02-06,-28.5,0.0,32132.32\nR004,2005-02-07,-28.4,0.0,32250.81\nR004,2005-02-08,-26.5,0.0,32369.65\nR004,2005-02-09,-22.9,0.0,32400.0\nR004,2005-02-10,-21.4,0.0,29725.72\nR004,2005-02-11,-18.1,0.0,19836.66\nR004,2005-02-12,-15.7,2.5,39.88\nR004,2005-02-13,-18.5,0.1,28263.43\nR004,2005-02-14,-21.6,0.0,27756.61\nR004,2005-02-15,-25.2,0.0,32400.0\nR004,2005-02-16,-26.5,0.0,33551.95\nR004,2005-02-17,-27.2,0.0,33894.94\nR004,2005-02-18,-23.6,0.0,23945.67\nR004,2005-02-19,-15.1,3.0,4451.37\nR004,2005-02-20,-11.4,5.0,711.89\nR004,2005-02-21,-16.4,0.5,30824.62\nR004,2005-02-22,-22.2,0.0,33938.85\nR004,2005-02-23,-23.1,0.0,36254.95\nR004,2005-02-24,-21.0,0.0,35184.56\nR004,2005-02-25,-15.4,0.0,35380.94\nR004,2005-02-26,-12.7,0.0,30674.49\nR004,2005-02-27,-12.1,0.0,32841.14\nR004,2005-02-28,-10.2,0.0,38095.52\nR004,2005-03-01,-6.8,1.0,27721.89\nR004,2005-03-02,-8.6,0.0,38535.49\nR004,2005-03-03,-6.8,0.0,38648.99\nR004,2005-03-04,-5.4,0.0,38151.24\nR004,2005-03-05,-6.8,2.6,17301.54\nR004,2005-03-06,-12.6,0.0,38987.42\nR004,2005-03-07,-8.3,0.1,7760.71\nR004,2005-03-08,-8.3,8.9,12083.2\nR004,2005-03-09,-12.5,0.3,37397.61\nR004,2005-03-10,-12.7,0.0,39610.85\nR004,2005-03-11,-8.6,0.0,39649.89\nR004,2005-03-12,-7.4,1.9,4926.32\nR004,2005-03-13,-6.8,0.1,32400.0\nR004,2005-03-14,-5.4,3.8,11276.03\nR004,2005-03-15,-8.5,3.3,11153.7\nR004,2005-03-16,-9.4,0.0,41267.36\nR004,2005-03-17,-5.5,0.0,16601.0\nR004,2005-03-18,-2.1,0.0,35578.81\nR004,2005-03-19,-4.1,4.5,17969.4\nR004,2005-03-20,-3.6,4.3,742.31\nR004,2005-03-21,-3.0,0.0,42344.57\nR004,2005-03-22,-1.3,0.0,41560.56\nR004,2005-03-23,1.0,0.0,42771.89\nR004,2005-03-24,0.4,0.0,42863.57\nR004,2005-03-25,0.1,0.8,42955.26\nR004,2005-03-26,2.0,0.0,41684.39\nR004,2005-03-27,2.1,0.0,41209.4\nR004,2005-03-28,-0.9,0.4,19970.55\nR004,2005-03-29,-1.6,4.0,4496.07\nR004,2005-03-30,-1.7,0.0,44493.31\nR004,2005-03-31,0.5,0.1,30962.24\nR004,2005-04-01,0.6,4.8,9124.2\nR004,2005-04-02,-2.4,12.6,25286.14\nR004,2005-04-03,-6.9,1.5,25765.16\nR004,2005-04-04,-4.3,9.7,947.9\nR004,2005-04-05,-4.7,0.8,39739.22\nR004,2005-04-06,-5.0,5.3,30270.69\nR004,2005-04-07,-8.9,2.9,20939.95\nR004,2005-04-08,-1.4,2.3,23800.6\nR004,2005-04-09,-3.5,5.9,37854.72\nR004,2005-04-10,-8.0,6.0,21210.11\nR004,2005-04-11,1.0,5.0,25348.27\nR004,2005-04-12,3.4,0.0,48181.74\nR004,2005-04-13,2.7,0.0,49005.33\nR004,2005-04-14,5.5,0.5,45587.19\nR004,2005-04-15,1.3,7.6,33083.27\nR004,2005-04-16,0.2,18.1,13968.84\nR004,2005-04-17,-3.1,0.4,46855.54\nR004,2005-04-18,-2.7,0.0,46645.99\nR004,2005-04-19,3.9,0.0,28787.64\nR004,2005-04-20,8.3,0.0,39556.25\nR004,2005-04-21,8.4,0.3,41236.47\nR004,2005-04-22,9.6,0.0,50392.92\nR004,2005-04-23,10.5,0.0,50400.0\nR004,2005-04-24,11.7,0.0,50400.0\nR004,2005-04-25,10.8,0.5,50130.28\nR004,2005-04-26,15.2,3.1,44558.86\nR004,2005-04-27,8.6,2.9,24247.91\nR004,2005-04-28,3.5,12.2,14770.13\nR004,2005-04-29,-1.2,0.0,49429.38\nR004,2005-04-30,3.3,0.0,45259.94\nR004,2005-05-01,4.2,0.0,36000.0\nR004,2005-05-02,1.0,0.0,47990.71\nR004,2005-05-03,0.9,0.0,48149.46\nR004,2005-05-04,4.0,0.0,43200.0\nR004,2005-05-05,7.8,0.5,27010.01\nR004,2005-05-06,10.3,0.0,48984.02\nR004,2005-05-07,13.6,0.0,51328.18\nR004,2005-05-08,14.9,0.4,42534.21\nR004,2005-05-09,8.8,0.0,51959.32\nR004,2005-05-10,8.3,0.0,50639.75\nR004,2005-05-11,7.8,0.0,42122.64\nR004,2005-05-12,7.3,0.2,52135.3\nR004,2005-05-13,5.1,0.2,44766.58\nR004,2005-05-14,5.2,0.8,35894.52\nR004,2005-05-15,4.2,4.2,45180.99\nR004,2005-05-16,8.2,0.0,55401.89\nR004,2005-05-17,12.4,0.0,55264.44\nR004,2005-05-18,13.8,1.3,53431.62\nR004,2005-05-19,15.1,0.6,52480.25\nR004,2005-05-20,13.3,3.5,52132.47\nR004,2005-05-21,13.4,0.0,48943.49\nR004,2005-05-22,13.8,0.3,54648.96\nR004,2005-05-23,9.2,5.0,39837.01\nR004,2005-05-24,10.8,0.0,57151.51\nR004,2005-05-25,10.8,1.8,51894.87\nR004,2005-05-26,10.6,3.4,25992.17\nR004,2005-05-27,6.4,1.3,43830.9\nR004,2005-05-28,8.8,0.0,52016.76\nR004,2005-05-29,11.9,0.0,49572.11\nR004,2005-05-30,14.5,0.0,54000.0\nR004,2005-05-31,13.3,10.6,31689.38\nR004,2005-06-01,10.4,15.9,49982.31\nR004,2005-06-02,8.0,13.9,41057.17\nR004,2005-06-03,7.9,6.1,35937.92\nR004,2005-06-04,9.7,1.5,34189.16\nR004,2005-06-05,10.5,7.0,28800.0\nR004,2005-06-06,13.4,0.0,46082.73\nR004,2005-06-07,15.3,0.0,57512.6\nR004,2005-06-08,16.8,0.0,57600.0\nR004,2005-06-09,18.7,0.0,57600.0\nR004,2005-06-10,15.5,0.0,57600.0\nR004,2005-06-11,14.9,0.0,57600.0\nR004,2005-06-12,16.8,0.0,56413.13\nR004,2005-06-13,18.2,1.1,57113.22\nR004,2005-06-14,17.4,3.5,49211.8\nR004,2005-06-15,16.1,14.9,26678.75\nR004,2005-06-16,16.1,23.9,46531.57\nR004,2005-06-17,15.9,12.1,50973.46\nR004,2005-06-18,15.1,10.5,45587.61\nR004,2005-06-19,15.9,5.1,46191.6\nR004,2005-06-20,13.3,14.6,15897.37\nR004,2005-06-21,15.4,0.0,55348.46\nR004,2005-06-22,18.3,0.0,57600.0\nR004,2005-06-23,21.0,0.0,57600.0\nR004,2005-06-24,19.9,0.7,30540.36\nR004,2005-06-25,15.9,15.9,51917.53\nR004,2005-06-26,14.6,2.2,48278.21\nR004,2005-06-27,13.6,3.3,22846.34\nR004,2005-06-28,13.9,15.6,16630.32\nR004,2005-06-29,15.8,0.1,47646.5\nR004,2005-06-30,17.7,0.0,57324.96\nR004,2005-07-01,19.7,0.0,57600.0\nR004,2005-07-02,20.8,0.0,57401.42\nR004,2005-07-03,19.5,5.8,49902.73\nR004,2005-07-04,18.4,8.5,46203.81\nR004,2005-07-05,19.8,0.2,47088.47\nR004,2005-07-06,21.4,0.0,48179.13\nR004,2005-07-07,22.6,0.0,52738.37\nR004,2005-07-08,24.2,0.0,57600.0\nR004,2005-07-09,24.0,0.1,56386.79\nR004,2005-07-10,19.2,24.3,23148.29\nR004,2005-07-11,16.3,0.2,50747.71\nR004,2005-07-12,15.7,4.7,56035.19\nR004,2005-07-13,17.7,0.0,53754.02\nR004,2005-07-14,17.8,5.3,47854.73\nR004,2005-07-15,15.2,13.4,51820.17\nR004,2005-07-16,15.0,1.6,44387.0\nR004,2005-07-17,14.9,6.6,37679.76\nR004,2005-07-18,15.0,6.4,45687.15\nR004,2005-07-19,15.7,8.0,40970.87\nR004,2005-07-20,14.6,13.0,51467.56\nR004,2005-07-21,14.7,2.9,47973.16\nR004,2005-07-22,15.1,4.7,40602.29\nR004,2005-07-23,16.8,0.8,47491.29\nR004,2005-07-24,18.5,0.0,50253.04\nR004,2005-07-25,20.6,0.0,44811.43\nR004,2005-07-26,20.5,13.1,32400.0\nR004,2005-07-27,18.2,11.5,33414.56\nR004,2005-07-28,17.8,9.6,40851.24\nR004,2005-07-29,18.9,0.9,43418.29\nR004,2005-07-30,19.2,1.8,37713.07\nR004,2005-07-31,18.7,3.6,34987.48\nR004,2005-08-01,20.8,0.0,50252.1\nR004,2005-08-02,22.4,0.0,53647.33\nR004,2005-08-03,22.7,0.0,46806.98\nR004,2005-08-04,18.4,6.6,18070.26\nR004,2005-08-05,19.8,2.4,49093.79\nR004,2005-08-06,19.6,0.7,28156.75\nR004,2005-08-07,19.7,0.3,26851.78\nR004,2005-08-08,16.6,5.5,1848.94\nR004,2005-08-09,16.9,0.0,48537.17\nR004,2005-08-10,14.3,2.3,44397.56\nR004,2005-08-11,11.1,4.9,32203.79\nR004,2005-08-12,9.0,1.7,50586.06\nR004,2005-08-13,7.7,8.6,31945.28\nR004,2005-08-14,9.5,2.5,44877.52\nR004,2005-08-15,11.4,0.0,50400.0\nR004,2005-08-16,15.5,0.0,50408.74\nR004,2005-08-17,16.1,0.0,50400.0\nR004,2005-08-18,17.7,0.0,50400.0\nR004,2005-08-19,17.8,0.1,44065.25\nR004,2005-08-20,17.8,0.3,46773.09\nR004,2005-08-21,18.4,0.0,50114.51\nR004,2005-08-22,18.9,0.0,49161.69\nR004,2005-08-23,13.3,11.1,27051.02\nR004,2005-08-24,15.4,1.1,46902.84\nR004,2005-08-25,16.2,0.0,45992.95\nR004,2005-08-26,16.0,1.4,46996.9\nR004,2005-08-27,16.0,0.0,41993.33\nR004,2005-08-28,12.2,20.4,0.0\nR004,2005-08-29,10.4,22.6,12995.25\nR004,2005-08-30,11.8,0.0,45663.72\nR004,2005-08-31,13.6,0.4,44047.03\nR004,2005-09-01,14.6,0.8,43062.12\nR004,2005-09-02,12.9,3.4,36058.4\nR004,2005-09-03,12.8,0.0,41673.83\nR004,2005-09-04,13.6,0.0,39510.11\nR004,2005-09-05,12.7,1.1,21913.79\nR004,2005-09-06,13.7,12.0,0.0\nR004,2005-09-07,12.8,14.0,35013.44\nR004,2005-09-08,10.3,4.5,42218.14\nR004,2005-09-09,9.1,8.0,35699.12\nR004,2005-09-10,9.0,2.5,34694.88\nR004,2005-09-11,12.8,1.7,34261.7\nR004,2005-09-12,13.1,7.0,29466.88\nR004,2005-09-13,9.8,4.3,41021.94\nR004,2005-09-14,5.2,13.5,9487.32\nR004,2005-09-15,5.1,0.0,42075.61\nR004,2005-09-16,4.5,0.0,43200.0\nR004,2005-09-17,5.3,0.0,43200.0\nR004,2005-09-18,6.3,0.0,43200.0\nR004,2005-09-19,8.9,0.0,43200.0\nR004,2005-09-20,8.6,0.0,43200.0\nR004,2005-09-21,8.9,0.0,43200.0\nR004,2005-09-22,10.1,1.2,39600.0\nR004,2005-09-23,4.8,8.9,20923.04\nR004,2005-09-24,7.4,0.9,38706.54\nR004,2005-09-25,8.3,3.1,15043.52\nR004,2005-09-26,4.7,17.1,8205.61\nR004,2005-09-27,1.9,3.7,27780.1\nR004,2005-09-28,2.2,0.0,26003.92\nR004,2005-09-29,4.5,17.9,0.0\nR004,2005-09-30,3.7,0.2,20723.91\nR004,2005-10-01,5.5,0.0,37291.04\nR004,2005-10-02,6.3,0.0,37033.57\nR004,2005-10-03,5.8,4.5,34242.9\nR004,2005-10-04,2.8,5.8,18610.27\nR004,2005-10-05,2.3,0.0,36009.66\nR004,2005-10-06,3.2,0.0,36064.86\nR004,2005-10-07,7.5,0.0,36000.0\nR004,2005-10-08,6.5,4.7,19646.46\nR004,2005-10-09,4.2,9.4,7833.17\nR004,2005-10-10,-0.3,3.2,5286.84\nR004,2005-10-11,-0.8,3.1,14765.91\nR004,2005-10-12,-1.3,0.0,7808.16\nR004,2005-10-13,1.3,0.0,11905.48\nR004,2005-10-14,4.3,0.2,22051.12\nR004,2005-10-15,3.6,0.0,20812.24\nR004,2005-10-16,4.2,0.0,34184.85\nR004,2005-10-17,4.9,0.0,34572.03\nR004,2005-10-18,5.7,2.0,21521.9\nR004,2005-10-19,4.7,4.7,2145.39\nR004,2005-10-20,3.6,0.0,28508.36\nR004,2005-10-21,6.2,0.0,34480.79\nR004,2005-10-22,7.5,0.0,13696.91\nR004,2005-10-23,6.2,4.7,0.0\nR004,2005-10-24,2.9,11.8,0.0\nR004,2005-10-25,1.8,2.4,1495.28\nR004,2005-10-26,3.9,1.1,15044.02\nR004,2005-10-27,-0.0,0.0,32573.12\nR004,2005-10-28,-0.1,1.2,20306.46\nR004,2005-10-29,0.2,0.9,7059.18\nR004,2005-10-30,-1.2,4.0,11816.33\nR004,2005-10-31,-2.6,2.0,0.0\nR004,2005-11-01,-1.3,11.9,0.0\nR004,2005-11-02,-0.8,14.3,0.0\nR004,2005-11-03,-4.5,9.2,0.0\nR004,2005-11-04,-9.7,1.7,11893.76\nR004,2005-11-05,-2.5,27.3,0.0\nR004,2005-11-06,-3.4,5.6,744.75\nR004,2005-11-07,-3.0,0.6,11609.22\nR004,2005-11-08,-3.4,8.9,543.71\nR004,2005-11-09,-3.8,3.6,3437.46\nR004,2005-11-10,-2.9,6.4,0.0\nR004,2005-11-11,-7.6,0.9,9114.62\nR004,2005-11-12,-12.8,0.0,24632.86\nR004,2005-11-13,-6.3,0.1,0.0\nR004,2005-11-14,-7.3,0.0,25200.0\nR004,2005-11-15,-9.7,0.3,15365.82\nR004,2005-11-16,-7.7,3.3,11256.78\nR004,2005-11-17,-8.1,5.5,886.61\nR004,2005-11-18,-12.3,0.0,28800.0\nR004,2005-11-19,-16.0,0.0,28800.0\nR004,2005-11-20,-15.4,0.0,26320.65\nR004,2005-11-21,-14.1,0.0,28712.24\nR004,2005-11-22,-17.7,0.0,28637.58\nR004,2005-11-23,-15.8,0.0,22736.9\nR004,2005-11-24,-13.4,2.6,0.0\nR004,2005-11-25,-7.5,3.8,0.0\nR004,2005-11-26,-8.8,8.1,0.0\nR004,2005-11-27,-5.1,12.1,0.0\nR004,2005-11-28,-5.4,19.0,0.0\nR004,2005-11-29,-6.8,10.4,0.0\nR004,2005-11-30,-8.4,4.2,0.0\nR004,2005-12-01,-10.1,3.2,0.0\nR004,2005-12-02,-16.5,0.0,27677.39\nR004,2005-12-03,-21.5,0.0,27552.05\nR004,2005-12-04,-21.0,0.0,27431.73\nR004,2005-12-05,-24.7,0.0,21779.35\nR004,2005-12-06,-21.1,0.0,20907.9\nR004,2005-12-07,-16.8,0.6,4080.17\nR004,2005-12-08,-16.6,0.3,446.19\nR004,2005-12-09,-22.1,0.0,17923.75\nR004,2005-12-10,-28.0,0.0,25585.77\nR004,2005-12-11,-31.5,0.0,26659.98\nR004,2005-12-12,-32.3,0.0,26675.19\nR004,2005-12-13,-31.4,0.0,26195.94\nR004,2005-12-14,-30.1,0.0,22817.8\nR004,2005-12-15,-26.7,0.0,24430.93\nR004,2005-12-16,-14.9,0.8,0.0\nR004,2005-12-17,-9.5,2.8,0.0\nR004,2005-12-18,-13.0,0.0,16487.06\nR004,2005-12-19,-21.6,0.0,25052.96\nR004,2005-12-20,-20.7,0.0,21760.34\nR004,2005-12-21,-20.4,0.0,25431.46\nR004,2005-12-22,-20.4,0.0,24949.68\nR004,2005-12-23,-20.5,0.0,24996.38\nR004,2005-12-24,-20.7,0.0,22205.29\nR004,2005-12-25,-20.7,0.5,24482.2\nR004,2005-12-26,-15.8,0.8,20948.34\nR004,2005-12-27,-18.2,0.0,5179.31\nR004,2005-12-28,-16.9,0.0,17229.6\nR004,2005-12-29,-9.1,2.3,7921.42\nR004,2005-12-30,-12.8,2.4,14097.92\nR004,2005-12-31,-13.1,1.0,9431.07\nR004,2006-01-01,-17.9,1.9,0.0\nR004,2006-01-02,-25.2,0.4,17350.07\nR004,2006-01-03,-32.6,0.0,25086.98\nR004,2006-01-04,-25.9,0.0,0.0\nR004,2006-01-05,-18.2,0.0,10696.69\nR004,2006-01-06,-16.7,0.4,3352.73\nR004,2006-01-07,-15.4,4.2,0.0\nR004,2006-01-08,-19.8,0.0,20165.06\nR004,2006-01-09,-18.5,2.5,0.0\nR004,2006-01-10,-16.5,2.1,368.65\nR004,2006-01-11,-23.9,0.5,3441.38\nR004,2006-01-12,-21.9,5.1,0.0\nR004,2006-01-13,-22.1,4.5,865.31\nR004,2006-01-14,-22.2,6.7,0.0\nR004,2006-01-15,-13.3,3.9,0.0\nR004,2006-01-16,-10.9,3.6,590.29\nR004,2006-01-17,-10.8,0.6,5303.64\nR004,2006-01-18,-10.3,1.1,10590.35\nR004,2006-01-19,-12.3,0.0,14510.03\nR004,2006-01-20,-16.3,1.4,21523.64\nR004,2006-01-21,-26.1,0.0,27943.95\nR004,2006-01-22,-26.1,0.0,23402.8\nR004,2006-01-23,-27.2,0.0,24345.82\nR004,2006-01-24,-21.2,2.8,0.0\nR004,2006-01-25,-27.7,0.1,29151.91\nR004,2006-01-26,-25.1,2.9,0.0\nR004,2006-01-27,-27.8,0.0,17328.04\nR004,2006-01-28,-28.3,0.0,23709.98\nR004,2006-01-29,-25.4,0.0,3464.99\nR004,2006-01-30,-22.7,0.0,25641.06\nR004,2006-01-31,-28.2,0.0,31396.73\nR004,2006-02-01,-28.2,0.0,29246.5\nR004,2006-02-02,-25.0,0.0,31632.71\nR004,2006-02-03,-16.6,3.5,14400.0\nR004,2006-02-04,-12.9,0.0,23100.62\nR004,2006-02-05,-18.4,0.0,28685.93\nR004,2006-02-06,-16.2,0.4,12901.02\nR004,2006-02-07,-10.1,7.1,0.0\nR004,2006-02-08,-9.2,0.0,12613.93\nR004,2006-02-09,-6.5,0.0,494.38\nR004,2006-02-10,-4.6,3.5,719.68\nR004,2006-02-11,-3.8,19.7,0.0\nR004,2006-02-12,-14.0,5.2,0.0\nR004,2006-02-13,-15.0,1.3,14458.72\nR004,2006-02-14,-16.0,0.0,18590.99\nR004,2006-02-15,-13.7,0.1,31940.47\nR004,2006-02-16,-12.6,0.0,32400.0\nR004,2006-02-17,-11.7,0.0,33505.17\nR004,2006-02-18,-10.1,0.1,27277.2\nR004,2006-02-19,-10.3,5.9,8059.09\nR004,2006-02-20,-16.8,0.2,13536.97\nR004,2006-02-21,-15.4,0.0,12535.4\nR004,2006-02-22,-16.5,0.0,32758.44\nR004,2006-02-23,-18.6,0.0,35839.26\nR004,2006-02-24,-21.5,0.0,36711.13\nR004,2006-02-25,-19.2,0.0,35354.88\nR004,2006-02-26,-13.0,0.0,30067.55\nR004,2006-02-27,-11.3,0.0,35567.84\nR004,2006-02-28,-9.9,0.0,38046.48\nR004,2006-03-01,-11.0,0.0,35901.49\nR004,2006-03-02,-6.4,1.1,9622.65\nR004,2006-03-03,-6.9,1.0,20547.73\nR004,2006-03-04,-9.7,0.0,23788.23\nR004,2006-03-05,-6.7,0.0,3136.2\nR004,2006-03-06,-9.2,1.9,20286.13\nR004,2006-03-07,-3.3,4.9,2035.21\nR004,2006-03-08,-6.1,8.0,534.76\nR004,2006-03-09,-12.9,0.2,26803.7\nR004,2006-03-10,-16.6,0.0,39604.25\nR004,2006-03-11,-12.0,0.0,39643.39\nR004,2006-03-12,-10.3,0.0,36553.4\nR004,2006-03-13,-6.8,0.0,39970.6\nR004,2006-03-14,-5.6,0.0,34209.2\nR004,2006-03-15,-7.3,1.3,36006.54\nR004,2006-03-16,-10.7,2.6,30662.68\nR004,2006-03-17,-5.1,3.3,1843.91\nR004,2006-03-18,-4.2,0.0,19681.84\nR004,2006-03-19,-3.2,1.6,12394.3\nR004,2006-03-20,-5.4,0.9,37973.94\nR004,2006-03-21,-6.6,0.0,42318.25\nR004,2006-03-22,-5.8,0.0,42463.07\nR004,2006-03-23,-4.1,1.0,42607.49\nR004,2006-03-24,-2.0,1.4,23189.43\nR004,2006-03-25,-2.1,1.9,5208.79\nR004,2006-03-26,-0.2,0.0,43407.04\nR004,2006-03-27,-0.5,0.0,32538.18\nR004,2006-03-28,2.7,0.7,20428.61\nR004,2006-03-29,-1.0,11.4,32994.32\nR004,2006-03-30,-3.5,2.1,27067.82\nR004,2006-03-31,-0.5,17.1,0.0\nR004,2006-04-01,-0.0,10.3,11155.3\nR004,2006-04-02,-4.3,2.9,44353.1\nR004,2006-04-03,-6.9,0.0,45453.33\nR004,2006-04-04,-0.8,14.1,0.0\nR004,2006-04-05,-5.7,3.2,35002.08\nR004,2006-04-06,-8.9,0.0,45919.73\nR004,2006-04-07,-9.3,0.1,20647.09\nR004,2006-04-08,-7.0,3.2,8338.98\nR004,2006-04-09,-7.6,0.6,39399.31\nR004,2006-04-10,-8.3,0.9,36362.54\nR004,2006-04-11,-7.9,2.2,22576.73\nR004,2006-04-12,-9.4,0.0,48559.19\nR004,2006-04-13,-7.0,0.0,49032.72\nR004,2006-04-14,-3.6,2.5,34101.74\nR004,2006-04-15,2.3,18.1,20721.72\nR004,2006-04-16,-2.9,2.4,40996.65\nR004,2006-04-17,-3.5,5.6,16120.91\nR004,2006-04-18,5.8,1.4,46782.89\nR004,2006-04-19,3.2,5.1,19067.29\nR004,2006-04-20,-1.3,4.2,21546.9\nR004,2006-04-21,-0.9,0.0,46625.39\nR004,2006-04-22,2.6,1.8,35069.97\nR004,2006-04-23,2.7,4.7,26543.35\nR004,2006-04-24,1.4,4.3,39433.99\nR004,2006-04-25,2.6,0.0,48713.66\nR004,2006-04-26,3.4,2.8,31617.98\nR004,2006-04-27,1.5,9.2,0.0\nR004,2006-04-28,0.1,22.4,0.0\nR004,2006-04-29,-0.0,17.7,0.0\nR004,2006-04-30,0.3,3.7,8102.28\nR004,2006-05-01,1.4,8.5,11984.53\nR004,2006-05-02,-0.5,3.5,40088.69\nR004,2006-05-03,0.2,6.8,12812.72\nR004,2006-05-04,-0.0,7.0,38390.91\nR004,2006-05-05,-0.8,3.2,48170.49\nR004,2006-05-06,-0.9,0.2,50959.11\nR004,2006-05-07,-0.2,0.5,50915.25\nR004,2006-05-08,1.4,0.3,47622.73\nR004,2006-05-09,3.6,0.4,50027.23\nR004,2006-05-10,5.0,0.0,47190.71\nR004,2006-05-11,9.6,0.0,47186.04\nR004,2006-05-12,12.8,0.0,53455.82\nR004,2006-05-13,15.6,0.0,52507.94\nR004,2006-05-14,13.1,12.1,37119.66\nR004,2006-05-15,7.5,4.2,51464.78\nR004,2006-05-16,9.1,3.2,47121.95\nR004,2006-05-17,11.6,0.8,54395.08\nR004,2006-05-18,10.9,0.6,53137.6\nR004,2006-05-19,11.6,0.7,50327.8\nR004,2006-05-20,14.2,0.0,56056.54\nR004,2006-05-21,4.0,3.6,0.0\nR004,2006-05-22,2.7,0.0,54189.33\nR004,2006-05-23,6.6,0.0,46177.85\nR004,2006-05-24,9.6,0.0,41764.38\nR004,2006-05-25,16.1,0.0,52778.64\nR004,2006-05-26,15.8,3.3,41821.94\nR004,2006-05-27,11.9,29.0,27692.27\nR004,2006-05-28,8.1,14.6,30470.4\nR004,2006-05-29,8.6,2.5,52629.24\nR004,2006-05-30,4.5,0.0,57600.0\nR004,2006-05-31,6.5,0.0,57600.0\nR004,2006-06-01,11.1,0.0,57600.0\nR004,2006-06-02,12.1,0.2,54914.42\nR004,2006-06-03,12.9,5.0,48101.92\nR004,2006-06-04,11.4,6.1,38980.8\nR004,2006-06-05,13.8,1.0,45644.39\nR004,2006-06-06,15.8,0.0,48390.72\nR004,2006-06-07,18.1,0.0,56190.52\nR004,2006-06-08,19.7,0.0,57600.0\nR004,2006-06-09,19.4,0.0,52347.85\nR004,2006-06-10,16.6,0.0,56481.18\nR004,2006-06-11,14.0,0.0,57600.0\nR004,2006-06-12,15.9,0.0,57600.0\nR004,2006-06-13,19.2,1.2,50400.0\nR004,2006-06-14,18.7,3.2,39833.52\nR004,2006-06-15,12.4,26.8,28947.8\nR004,2006-06-16,8.3,18.1,24694.92\nR004,2006-06-17,9.7,0.0,57600.0\nR004,2006-06-18,13.9,0.0,57600.0\nR004,2006-06-19,14.6,1.1,57600.0\nR004,2006-06-20,15.8,0.0,56566.37\nR004,2006-06-21,18.8,0.0,57600.0\nR004,2006-06-22,20.3,0.0,57600.0\nR004,2006-06-23,22.0,0.0,57600.0\nR004,2006-06-24,20.3,0.7,35361.39\nR004,2006-06-25,17.3,3.3,1.46\nR004,2006-06-26,18.3,1.3,47895.91\nR004,2006-06-27,19.2,0.5,50669.23\nR004,2006-06-28,17.9,6.8,34059.13\nR004,2006-06-29,18.9,1.1,48894.47\nR004,2006-06-30,19.9,0.5,52176.04\nR004,2006-07-01,20.1,2.0,49492.16\nR004,2006-07-02,20.7,1.3,41319.56\nR004,2006-07-03,21.0,3.3,39356.86\nR004,2006-07-04,21.7,1.7,50207.06\nR004,2006-07-05,22.6,0.8,50964.02\nR004,2006-07-06,19.4,6.8,36303.91\nR004,2006-07-07,12.0,25.1,0.0\nR004,2006-07-08,13.7,1.9,49417.36\nR004,2006-07-09,14.1,6.9,45681.0\nR004,2006-07-10,10.8,5.6,17934.08\nR004,2006-07-11,13.6,0.0,37500.76\nR004,2006-07-12,15.6,0.0,56223.38\nR004,2006-07-13,16.8,0.0,56054.51\nR004,2006-07-14,18.7,0.0,56080.55\nR004,2006-07-15,17.2,10.8,4553.67\nR004,2006-07-16,17.6,3.7,35287.5\nR004,2006-07-17,17.0,2.7,40158.04\nR004,2006-07-18,17.9,0.2,49788.65\nR004,2006-07-19,18.0,0.9,45997.42\nR004,2006-07-20,16.7,5.6,24299.58\nR004,2006-07-21,17.5,3.7,42835.54\nR004,2006-07-22,17.7,10.7,32644.42\nR004,2006-07-23,19.9,0.6,54087.09\nR004,2006-07-24,13.7,8.7,31700.35\nR004,2006-07-25,12.7,0.9,21591.11\nR004,2006-07-26,14.3,0.1,50094.48\nR004,2006-07-27,15.0,2.1,48025.23\nR004,2006-07-28,15.6,5.7,53763.42\nR004,2006-07-29,16.2,0.2,50164.77\nR004,2006-07-30,19.0,0.6,40363.39\nR004,2006-07-31,17.0,10.0,21170.32\nR004,2006-08-01,18.0,1.5,50103.01\nR004,2006-08-02,18.4,0.2,51152.79\nR004,2006-08-03,18.0,0.0,48846.25\nR004,2006-08-04,18.3,1.6,47447.0\nR004,2006-08-05,14.2,17.3,20646.91\nR004,2006-08-06,12.9,10.5,21959.44\nR004,2006-08-07,12.2,12.3,46999.74\nR004,2006-08-08,12.4,11.4,46350.26\nR004,2006-08-09,12.9,2.5,38741.33\nR004,2006-08-10,11.3,31.9,1876.75\nR004,2006-08-11,10.9,9.5,11814.29\nR004,2006-08-12,12.3,0.5,50985.54\nR004,2006-08-13,14.2,0.0,51040.14\nR004,2006-08-14,16.4,1.7,50400.0\nR004,2006-08-15,13.1,12.6,33386.82\nR004,2006-08-16,9.2,9.6,22912.26\nR004,2006-08-17,9.6,2.0,1567.52\nR004,2006-08-18,11.1,5.6,31426.25\nR004,2006-08-19,12.8,4.2,41519.31\nR004,2006-08-20,13.2,0.0,39205.35\nR004,2006-08-21,13.5,0.0,50139.22\nR004,2006-08-22,17.5,4.8,45081.49\nR004,2006-08-23,14.7,16.2,32994.41\nR004,2006-08-24,12.5,0.0,41666.39\nR004,2006-08-25,12.2,0.4,47799.36\nR004,2006-08-26,6.8,2.3,39860.12\nR004,2006-08-27,7.1,0.0,41185.6\nR004,2006-08-28,8.7,0.0,40650.37\nR004,2006-08-29,9.8,0.0,45069.42\nR004,2006-08-30,10.2,8.5,29238.94\nR004,2006-08-31,6.7,1.9,16133.28\nR004,2006-09-01,7.7,4.8,16083.85\nR004,2006-09-02,9.4,0.0,35633.83\nR004,2006-09-03,10.2,0.7,39300.84\nR004,2006-09-04,12.2,5.1,26940.72\nR004,2006-09-05,7.2,0.0,39671.95\nR004,2006-09-06,6.6,0.0,45362.93\nR004,2006-09-07,7.6,0.0,44361.9\nR004,2006-09-08,10.8,0.0,44057.85\nR004,2006-09-09,13.6,0.0,43732.07\nR004,2006-09-10,15.0,0.0,43200.0\nR004,2006-09-11,15.0,0.0,34416.13\nR004,2006-09-12,13.0,1.8,41752.46\nR004,2006-09-13,13.3,0.8,43200.0\nR004,2006-09-14,11.1,8.3,20233.29\nR004,2006-09-15,9.1,5.9,38772.55\nR004,2006-09-16,11.1,1.0,41384.82\nR004,2006-09-17,11.6,0.0,43200.0\nR004,2006-09-18,11.2,0.0,14400.0\nR004,2006-09-19,6.8,4.9,0.0\nR004,2006-09-20,4.9,0.0,30840.01\nR004,2006-09-21,6.2,0.6,38814.34\nR004,2006-09-22,5.4,1.4,37003.25\nR004,2006-09-23,5.8,0.3,41249.14\nR004,2006-09-24,7.6,0.0,42918.58\nR004,2006-09-25,4.9,7.3,7200.0\nR004,2006-09-26,5.2,0.8,34429.09\nR004,2006-09-27,9.6,0.0,37102.27\nR004,2006-09-28,9.6,0.0,39565.59\nR004,2006-09-29,7.4,7.0,0.0\nR004,2006-09-30,7.7,0.0,34065.26\nR004,2006-10-01,9.7,3.6,18198.02\nR004,2006-10-02,9.0,6.3,21978.4\nR004,2006-10-03,11.1,0.0,36000.0\nR004,2006-10-04,9.3,10.9,14526.99\nR004,2006-10-05,-2.0,4.5,20160.57\nR004,2006-10-06,-2.7,0.0,29737.24\nR004,2006-10-07,-1.3,10.2,774.76\nR004,2006-10-08,1.1,16.6,0.0\nR004,2006-10-09,0.4,0.3,1669.49\nR004,2006-10-10,4.2,3.5,14678.62\nR004,2006-10-11,1.8,14.3,9194.96\nR004,2006-10-12,-1.0,1.6,24938.56\nR004,2006-10-13,-1.9,0.0,36000.0\nR004,2006-10-14,-3.4,0.0,32316.53\nR004,2006-10-15,-0.1,0.5,17324.62\nR004,2006-10-16,-0.4,8.0,12154.14\nR004,2006-10-17,0.2,7.1,12354.79\nR004,2006-10-18,-0.3,0.2,29137.58\nR004,2006-10-19,-0.8,0.0,18794.37\nR004,2006-10-20,0.5,0.0,32400.0\nR004,2006-10-21,0.3,9.2,0.0\nR004,2006-10-22,-1.8,10.0,85.71\nR004,2006-10-23,-7.8,0.1,34203.94\nR004,2006-10-24,-7.0,0.0,30553.95\nR004,2006-10-25,-0.9,6.3,0.0\nR004,2006-10-26,2.6,18.1,0.0\nR004,2006-10-27,5.0,9.6,0.0\nR004,2006-10-28,4.6,0.0,29724.34\nR004,2006-10-29,5.6,0.0,21262.01\nR004,2006-10-30,5.6,0.0,29780.35\nR004,2006-10-31,5.7,0.5,1077.63\nR004,2006-11-01,6.6,2.1,17176.3\nR004,2006-11-02,2.8,12.7,7200.0\nR004,2006-11-03,-0.6,0.8,17537.15\nR004,2006-11-04,-0.2,0.0,30111.45\nR004,2006-11-05,-1.3,0.0,29740.88\nR004,2006-11-06,-1.0,0.0,29806.25\nR004,2006-11-07,-2.6,0.0,29431.22\nR004,2006-11-08,-2.0,1.8,28800.0\nR004,2006-11-09,-0.0,0.0,26648.88\nR004,2006-11-10,2.5,5.6,0.0\nR004,2006-11-11,-0.2,1.1,17718.93\nR004,2006-11-12,-0.6,2.1,546.6\nR004,2006-11-13,-1.7,0.0,28914.01\nR004,2006-11-14,-1.9,0.0,28800.0\nR004,2006-11-15,-1.0,0.0,28381.73\nR004,2006-11-16,-1.6,0.0,11449.44\nR004,2006-11-17,-1.6,0.0,13040.62\nR004,2006-11-18,-1.7,6.9,0.0\nR004,2006-11-19,-3.0,6.9,0.0\nR004,2006-11-20,-6.3,3.8,0.0\nR004,2006-11-21,-3.1,0.1,16796.39\nR004,2006-11-22,-3.7,4.9,0.0\nR004,2006-11-23,-9.0,0.0,21218.25\nR004,2006-11-24,-11.3,0.0,17028.97\nR004,2006-11-25,-13.7,0.0,17768.99\nR004,2006-11-26,-12.9,0.6,0.0\nR004,2006-11-27,-17.3,0.0,28332.22\nR004,2006-11-28,-18.0,0.0,28274.18\nR004,2006-11-29,-18.0,0.0,28125.49\nR004,2006-11-30,-17.4,0.0,27571.53\nR004,2006-12-01,-14.4,0.0,0.0\nR004,2006-12-02,-11.7,1.6,0.0\nR004,2006-12-03,-16.7,0.0,22067.88\nR004,2006-12-04,-15.3,0.0,18350.05\nR004,2006-12-05,-10.1,0.0,24119.98\nR004,2006-12-06,-9.5,2.1,675.23\nR004,2006-12-07,-5.2,4.2,2233.02\nR004,2006-12-08,-4.0,3.0,0.0\nR004,2006-12-09,-2.6,16.1,0.0\nR004,2006-12-10,-7.0,13.1,0.0\nR004,2006-12-11,-15.7,4.2,11112.49\nR004,2006-12-12,-6.1,14.1,8788.41\nR004,2006-12-13,-7.6,6.6,0.0\nR004,2006-12-14,-5.2,8.1,0.0\nR004,2006-12-15,-5.8,0.2,23056.56\nR004,2006-12-16,-11.8,0.0,13916.73\nR004,2006-12-17,-7.5,0.8,5924.86\nR004,2006-12-18,-3.9,5.0,97.27\nR004,2006-12-19,-3.5,7.8,0.0\nR004,2006-12-20,-6.2,1.5,0.0\nR004,2006-12-21,-6.0,8.6,15772.57\nR004,2006-12-22,-10.3,5.0,0.0\nR004,2006-12-23,-8.8,6.0,10810.83\nR004,2006-12-24,-18.0,0.0,25040.61\nR004,2006-12-25,-24.8,0.0,22902.3\nR004,2006-12-26,-17.8,1.6,2334.95\nR004,2006-12-27,-12.4,0.0,21676.75\nR004,2006-12-28,-17.0,0.0,16415.49\nR004,2006-12-29,-14.2,0.0,961.25\nR004,2006-12-30,-9.1,4.4,0.0\nR004,2006-12-31,-6.5,2.2,553.56\nR004,2007-01-01,-8.6,0.0,20964.23\nR004,2007-01-02,-10.9,1.8,3138.24\nR004,2007-01-03,-17.5,0.4,14358.67\nR004,2007-01-04,-22.7,0.0,22589.41\nR004,2007-01-05,-10.0,0.0,3993.07\nR004,2007-01-06,-7.8,0.8,13262.31\nR004,2007-01-07,-13.4,0.0,1666.94\nR004,2007-01-08,-15.3,0.0,25202.95\nR004,2007-01-09,-20.2,0.0,26699.7\nR004,2007-01-10,-18.5,0.0,26671.27\nR004,2007-01-11,-15.8,0.0,21584.41\nR004,2007-01-12,-12.9,1.7,0.0\nR004,2007-01-13,-5.1,16.4,0.0\nR004,2007-01-14,-2.8,10.7,0.0\nR004,2007-01-15,-7.0,0.0,27491.26\nR004,2007-01-16,-15.7,0.0,27608.97\nR004,2007-01-17,-17.6,0.0,27654.45\nR004,2007-01-18,-16.8,0.0,27716.81\nR004,2007-01-19,-15.2,0.9,17333.03\nR004,2007-01-20,-12.3,0.0,28123.43\nR004,2007-01-21,-19.4,0.0,28137.71\nR004,2007-01-22,-21.4,0.0,28624.16\nR004,2007-01-23,-23.6,0.0,28173.0\nR004,2007-01-24,-20.5,1.6,28147.89\nR004,2007-01-25,-8.2,5.4,927.51\nR004,2007-01-26,-6.8,6.0,0.0\nR004,2007-01-27,-0.7,19.9,0.0\nR004,2007-01-28,-5.9,2.5,12337.0\nR004,2007-01-29,-8.0,0.0,21388.06\nR004,2007-01-30,-7.9,0.0,28236.16\nR004,2007-01-31,-7.2,0.0,28582.25\nR004,2007-02-01,-7.5,0.0,24292.02\nR004,2007-02-02,-8.0,1.5,24817.98\nR004,2007-02-03,-7.7,0.0,31426.05\nR004,2007-02-04,-7.9,4.8,8128.42\nR004,2007-02-05,-7.7,3.0,2177.55\nR004,2007-02-06,-12.8,0.0,32084.23\nR004,2007-02-07,-8.2,2.9,1381.21\nR004,2007-02-08,-4.5,1.1,14268.29\nR004,2007-02-09,-8.0,0.0,32248.79\nR004,2007-02-10,-8.7,2.0,15402.15\nR004,2007-02-11,-9.8,2.2,0.0\nR004,2007-02-12,-10.0,0.3,24828.73\nR004,2007-02-13,-4.3,0.9,13689.75\nR004,2007-02-14,-9.4,4.6,25972.73\nR004,2007-02-15,-17.8,0.0,32400.0\nR004,2007-02-16,-16.8,2.0,6105.8\nR004,2007-02-17,-9.5,7.4,0.0\nR004,2007-02-18,-10.9,0.8,10037.26\nR004,2007-02-19,-6.9,7.6,222.81\nR004,2007-02-20,-13.1,5.6,1426.07\nR004,2007-02-21,-11.1,4.4,734.9\nR004,2007-02-22,-8.2,4.2,0.0\nR004,2007-02-23,-7.0,2.4,18000.0\nR004,2007-02-24,-9.5,7.0,14015.85\nR004,2007-02-25,-13.7,0.2,29095.27\nR004,2007-02-26,-10.9,2.5,0.0\nR004,2007-02-27,-15.6,0.0,33728.38\nR004,2007-02-28,-18.1,0.0,35176.23\nR004,2007-03-01,-16.8,0.0,36660.51\nR004,2007-03-02,-14.3,0.0,36263.48\nR004,2007-03-03,-17.9,0.0,38594.82\nR004,2007-03-04,-15.5,0.0,37004.05\nR004,2007-03-05,-12.8,0.0,38818.58\nR004,2007-03-06,-11.5,0.0,35072.8\nR004,2007-03-07,-11.7,0.2,31419.26\nR004,2007-03-08,-14.7,0.3,37850.68\nR004,2007-03-09,-12.8,0.0,33749.44\nR004,2007-03-10,-6.7,0.0,29046.5\nR004,2007-03-11,-8.2,1.4,8481.94\nR004,2007-03-12,-12.4,0.4,31672.68\nR004,2007-03-13,-15.3,0.0,39600.76\nR004,2007-03-14,-13.0,0.0,40294.99\nR004,2007-03-15,-10.0,0.0,40221.58\nR004,2007-03-16,-8.0,0.0,41297.02\nR004,2007-03-17,-5.5,0.0,40457.86\nR004,2007-03-18,-4.2,0.0,38254.95\nR004,2007-03-19,1.7,0.0,41458.92\nR004,2007-03-20,1.5,0.4,25431.7\nR004,2007-03-21,-0.6,2.8,3931.72\nR004,2007-03-22,-4.1,4.8,101.63\nR004,2007-03-23,-2.3,9.1,0.0\nR004,2007-03-24,-0.9,11.4,28800.0\nR004,2007-03-25,-1.7,1.8,42613.84\nR004,2007-03-26,-2.2,0.0,42993.34\nR004,2007-03-27,0.4,2.9,28772.15\nR004,2007-03-28,-0.3,11.1,5084.12\nR004,2007-03-29,-5.2,2.6,28375.76\nR004,2007-03-30,-4.4,2.4,8328.12\nR004,2007-03-31,-7.7,0.9,27462.43\nR004,2007-04-01,-4.2,0.0,45180.99\nR004,2007-04-02,-1.1,0.0,32400.0\nR004,2007-04-03,2.2,4.2,2938.14\nR004,2007-04-04,2.2,0.8,36748.94\nR004,2007-04-05,2.1,0.0,45789.65\nR004,2007-04-06,3.4,0.0,45985.83\nR004,2007-04-07,5.7,0.0,43220.31\nR004,2007-04-08,5.6,0.0,45607.49\nR004,2007-04-09,5.0,0.7,46484.51\nR004,2007-04-10,4.1,0.0,47299.09\nR004,2007-04-11,3.8,2.2,40514.47\nR004,2007-04-12,3.5,3.0,22605.8\nR004,2007-04-13,2.1,0.0,48962.9\nR004,2007-04-14,3.9,0.0,47013.62\nR004,2007-04-15,3.7,5.1,24362.0\nR004,2007-04-16,8.5,1.3,34872.21\nR004,2007-04-17,8.7,8.6,16314.75\nR004,2007-04-18,6.7,22.3,23738.48\nR004,2007-04-19,3.9,1.7,42191.15\nR004,2007-04-20,5.7,0.1,38721.74\nR004,2007-04-21,6.7,0.0,50196.98\nR004,2007-04-22,8.7,0.0,50328.23\nR004,2007-04-23,10.8,0.0,50400.0\nR004,2007-04-24,11.7,0.0,50400.0\nR004,2007-04-25,10.6,0.6,48716.74\nR004,2007-04-26,11.7,0.0,45078.98\nR004,2007-04-27,14.7,0.0,30792.92\nR004,2007-04-28,7.9,8.5,36048.02\nR004,2007-04-29,10.0,3.0,30983.6\nR004,2007-04-30,11.2,0.1,33447.84\nR004,2007-05-01,12.2,0.4,49082.2\nR004,2007-05-02,14.4,0.0,39600.0\nR004,2007-05-03,15.1,0.0,50688.12\nR004,2007-05-04,14.3,0.3,50641.41\nR004,2007-05-05,7.7,9.7,22216.54\nR004,2007-05-06,6.1,0.0,50971.46\nR004,2007-05-07,9.9,0.0,50277.35\nR004,2007-05-08,9.0,2.4,33598.66\nR004,2007-05-09,4.1,11.7,6286.24\nR004,2007-05-10,3.2,18.6,9962.5\nR004,2007-05-11,10.3,3.9,36725.2\nR004,2007-05-12,15.8,0.2,47716.2\nR004,2007-05-13,9.6,32.0,25237.52\nR004,2007-05-14,7.7,15.0,27098.32\nR004,2007-05-15,11.5,19.0,12189.72\nR004,2007-05-16,15.5,0.0,44015.08\nR004,2007-05-17,12.9,6.4,2452.57\nR004,2007-05-18,9.3,0.1,51566.37\nR004,2007-05-19,11.8,5.9,43200.0\nR004,2007-05-20,11.3,18.6,21820.67\nR004,2007-05-21,6.9,12.4,46004.21\nR004,2007-05-22,5.4,1.6,45886.87\nR004,2007-05-23,7.5,9.1,39347.01\nR004,2007-05-24,10.2,18.1,13813.57\nR004,2007-05-25,7.1,13.5,11555.9\nR004,2007-05-26,10.9,5.3,50066.0\nR004,2007-05-27,10.5,18.8,25675.74\nR004,2007-05-28,9.2,1.7,49091.92\nR004,2007-05-29,7.4,3.9,4827.57\nR004,2007-05-30,12.0,51.0,0.0\nR004,2007-05-31,7.9,1.9,46800.0\nR004,2007-06-01,4.3,0.3,57600.0\nR004,2007-06-02,8.1,0.4,43846.58\nR004,2007-06-03,12.1,13.9,14915.2\nR004,2007-06-04,14.4,5.6,25691.27\nR004,2007-06-05,12.8,13.5,24993.69\nR004,2007-06-06,7.6,0.4,57600.0\nR004,2007-06-07,11.4,0.0,52631.59\nR004,2007-06-08,11.9,1.6,42252.06\nR004,2007-06-09,9.1,12.2,0.0\nR004,2007-06-10,12.6,0.0,55266.91\nR004,2007-06-11,13.7,1.0,41985.96\nR004,2007-06-12,15.7,10.2,51959.54\nR004,2007-06-13,12.9,5.0,57600.0\nR004,2007-06-14,11.5,3.6,51323.71\nR004,2007-06-15,9.9,1.7,52544.1\nR004,2007-06-16,10.9,4.0,54624.43\nR004,2007-06-17,13.7,0.0,47252.28\nR004,2007-06-18,16.0,10.6,43159.77\nR004,2007-06-19,13.1,0.0,57600.0\nR004,2007-06-20,14.6,3.4,43647.76\nR004,2007-06-21,18.2,3.2,49829.36\nR004,2007-06-22,18.6,0.0,57544.72\nR004,2007-06-23,20.7,0.0,55740.64\nR004,2007-06-24,20.9,0.0,57600.0\nR004,2007-06-25,16.6,4.2,51890.41\nR004,2007-06-26,13.3,0.1,52561.67\nR004,2007-06-27,15.0,0.6,55509.53\nR004,2007-06-28,16.1,3.3,51578.29\nR004,2007-06-29,16.4,2.2,28144.53\nR004,2007-06-30,16.4,1.6,48031.2\nR004,2007-07-01,15.5,3.2,31883.82\nR004,2007-07-02,16.4,2.4,46839.34\nR004,2007-07-03,16.3,5.7,40856.67\nR004,2007-07-04,17.6,0.0,50174.85\nR004,2007-07-05,18.5,0.2,48611.55\nR004,2007-07-06,19.0,2.5,47712.96\nR004,2007-07-07,20.7,0.0,50943.05\nR004,2007-07-08,22.0,0.0,43684.98\nR004,2007-07-09,21.9,3.7,49120.96\nR004,2007-07-10,18.9,15.4,15990.15\nR004,2007-07-11,19.0,13.2,24564.9\nR004,2007-07-12,19.3,12.7,47675.63\nR004,2007-07-13,20.9,0.0,50860.08\nR004,2007-07-14,19.9,0.0,57002.88\nR004,2007-07-15,19.5,0.0,57026.89\nR004,2007-07-16,20.0,0.0,56590.99\nR004,2007-07-17,19.4,0.0,56158.1\nR004,2007-07-18,19.5,0.0,55948.5\nR004,2007-07-19,21.0,0.0,55301.41\nR004,2007-07-20,21.8,0.0,53238.75\nR004,2007-07-21,21.1,0.0,42888.06\nR004,2007-07-22,21.2,9.0,47022.85\nR004,2007-07-23,20.9,3.5,54978.55\nR004,2007-07-24,21.0,0.0,53888.14\nR004,2007-07-25,20.7,4.2,45788.39\nR004,2007-07-26,18.9,0.0,54000.0\nR004,2007-07-27,17.1,2.2,4425.23\nR004,2007-07-28,16.1,24.2,0.0\nR004,2007-07-29,15.5,13.0,0.0\nR004,2007-07-30,15.6,8.3,27503.27\nR004,2007-07-31,16.1,3.7,53389.99\nR004,2007-08-01,17.7,0.0,47601.29\nR004,2007-08-02,19.2,0.2,49884.21\nR004,2007-08-03,18.6,0.7,43376.94\nR004,2007-08-04,17.2,3.2,49225.58\nR004,2007-08-05,15.0,15.1,27931.99\nR004,2007-08-06,14.0,17.9,43143.21\nR004,2007-08-07,14.8,0.7,50708.28\nR004,2007-08-08,16.8,5.1,41473.58\nR004,2007-08-09,19.3,0.0,51189.93\nR004,2007-08-10,18.3,1.7,49690.48\nR004,2007-08-11,13.5,21.1,165.25\nR004,2007-08-12,10.7,15.8,25354.08\nR004,2007-08-13,10.9,8.6,1948.83\nR004,2007-08-14,10.9,0.4,38590.46\nR004,2007-08-15,10.2,0.5,19094.76\nR004,2007-08-16,10.9,0.1,41498.3\nR004,2007-08-17,9.6,6.4,23498.71\nR004,2007-08-18,10.3,0.1,40626.4\nR004,2007-08-19,10.3,8.2,40203.32\nR004,2007-08-20,12.2,0.0,43643.16\nR004,2007-08-21,12.0,0.0,49710.04\nR004,2007-08-22,13.0,0.1,25417.05\nR004,2007-08-23,14.1,2.7,35374.51\nR004,2007-08-24,13.7,0.0,33806.3\nR004,2007-08-25,9.6,3.5,19137.95\nR004,2007-08-26,10.3,0.0,48133.83\nR004,2007-08-27,12.8,0.0,44642.58\nR004,2007-08-28,14.8,0.0,48222.39\nR004,2007-08-29,17.7,0.0,48636.11\nR004,2007-08-30,17.8,0.0,48154.13\nR004,2007-08-31,18.9,0.0,47672.57\nR004,2007-09-01,19.2,0.0,47484.76\nR004,2007-09-02,20.2,0.6,44491.31\nR004,2007-09-03,19.2,0.5,45377.94\nR004,2007-09-04,18.6,1.5,39897.2\nR004,2007-09-05,10.8,9.7,557.47\nR004,2007-09-06,9.3,3.3,21721.82\nR004,2007-09-07,10.7,0.0,41326.52\nR004,2007-09-08,11.1,2.2,24152.35\nR004,2007-09-09,10.9,0.0,30154.92\nR004,2007-09-10,12.8,0.0,43406.1\nR004,2007-09-11,15.5,0.0,43200.0\nR004,2007-09-12,16.1,0.0,43200.0\nR004,2007-09-13,15.7,0.0,42189.46\nR004,2007-09-14,10.9,0.5,31939.2\nR004,2007-09-15,5.9,3.2,4869.09\nR004,2007-09-16,6.4,0.0,43200.0\nR004,2007-09-17,8.0,0.0,43200.0\nR004,2007-09-18,9.7,0.0,43200.0\nR004,2007-09-19,10.4,0.0,37305.92\nR004,2007-09-20,9.9,1.6,35815.76\nR004,2007-09-21,8.1,15.3,22254.76\nR004,2007-09-22,6.0,8.3,7094.44\nR004,2007-09-23,3.9,5.6,10314.16\nR004,2007-09-24,2.4,7.8,6139.72\nR004,2007-09-25,3.8,0.3,39797.68\nR004,2007-09-26,3.9,0.0,42484.81\nR004,2007-09-27,3.8,1.8,5543.24\nR004,2007-09-28,4.4,0.0,34495.49\nR004,2007-09-29,4.5,0.0,39668.99\nR004,2007-09-30,7.2,0.0,38645.94\nR004,2007-10-01,9.0,5.7,23346.06\nR004,2007-10-02,9.5,0.5,36979.17\nR004,2007-10-03,2.5,9.4,15687.99\nR004,2007-10-04,-1.8,3.9,29462.28\nR004,2007-10-05,-0.7,1.9,29635.96\nR004,2007-10-06,0.0,0.0,36039.43\nR004,2007-10-07,0.4,0.2,36094.3\nR004,2007-10-08,4.2,3.3,18859.02\nR004,2007-10-09,-1.7,1.6,3536.46\nR004,2007-10-10,-1.6,0.0,36000.0\nR004,2007-10-11,-1.4,0.0,36000.0\nR004,2007-10-12,0.6,0.0,36000.0\nR004,2007-10-13,2.9,0.0,36000.0\nR004,2007-10-14,6.5,0.4,22375.25\nR004,2007-10-15,2.4,5.4,14802.87\nR004,2007-10-16,0.8,2.6,21580.21\nR004,2007-10-17,-0.5,0.0,35277.6\nR004,2007-10-18,3.9,0.0,34528.1\nR004,2007-10-19,5.7,3.2,3778.88\nR004,2007-10-20,5.5,5.8,10437.31\nR004,2007-10-21,4.5,4.4,0.0\nR004,2007-10-22,0.8,13.2,17252.73\nR004,2007-10-23,-2.4,0.2,13771.74\nR004,2007-10-24,-2.9,0.2,126.25\nR004,2007-10-25,-2.6,0.0,17812.04\nR004,2007-10-26,-3.3,0.4,10136.37\nR004,2007-10-27,-2.7,9.1,4320.89\nR004,2007-10-28,-2.4,23.0,14400.0\nR004,2007-10-29,-6.3,4.0,5183.53\nR004,2007-10-30,-2.8,12.7,0.0\nR004,2007-10-31,-0.4,5.2,0.0\nR004,2007-11-01,-0.1,0.1,0.0\nR004,2007-11-02,-3.1,0.0,25528.44\nR004,2007-11-03,-3.3,0.0,3159.28\nR004,2007-11-04,-3.5,0.0,30094.4\nR004,2007-11-05,-2.4,0.0,29725.52\nR004,2007-11-06,-0.9,3.6,26997.84\nR004,2007-11-07,-3.1,0.0,20255.37\nR004,2007-11-08,-0.5,0.0,0.0\nR004,2007-11-09,-2.1,4.9,0.0\nR004,2007-11-10,-12.0,0.4,0.0\nR004,2007-11-11,-14.2,0.0,28800.0\nR004,2007-11-12,-14.9,0.0,28841.69\nR004,2007-11-13,-15.8,0.0,28901.68\nR004,2007-11-14,-14.5,0.0,21787.53\nR004,2007-11-15,-9.4,3.0,17252.82\nR004,2007-11-16,-7.8,0.8,1173.97\nR004,2007-11-17,-6.8,2.4,4063.75\nR004,2007-11-18,-2.7,3.7,0.0\nR004,2007-11-19,-2.9,0.0,27498.2\nR004,2007-11-20,-6.4,0.0,28800.0\nR004,2007-11-21,-8.3,3.5,25200.0\nR004,2007-11-22,-7.8,8.8,0.0\nR004,2007-11-23,-8.5,12.7,19730.02\nR004,2007-11-24,-13.6,4.4,0.0\nR004,2007-11-25,-4.7,8.8,400.79\nR004,2007-11-26,-9.3,0.0,23724.7\nR004,2007-11-27,-11.1,0.0,28348.04\nR004,2007-11-28,-14.6,0.0,28290.14\nR004,2007-11-29,-18.7,0.0,28151.67\nR004,2007-11-30,-17.9,0.0,28004.33\nR004,2007-12-01,-16.3,0.0,27862.26\nR004,2007-12-02,-8.3,1.2,3103.32\nR004,2007-12-03,-14.5,0.0,12189.94\nR004,2007-12-04,-14.3,0.0,2410.28\nR004,2007-12-05,-9.9,0.0,21186.96\nR004,2007-12-06,-13.9,0.0,27241.36\nR004,2007-12-07,-19.8,0.0,25500.74\nR004,2007-12-08,-16.7,0.0,1107.81\nR004,2007-12-09,-16.0,0.0,16405.33\nR004,2007-12-10,-16.2,0.0,9065.39\nR004,2007-12-11,-16.9,0.0,22901.77\nR004,2007-12-12,-14.0,0.7,118.71\nR004,2007-12-13,-10.4,5.4,0.0\nR004,2007-12-14,-9.7,9.2,0.0\nR004,2007-12-15,-12.9,0.5,11986.43\nR004,2007-12-16,-10.1,4.7,0.0\nR004,2007-12-17,-9.3,0.0,7495.34\nR004,2007-12-18,-8.2,5.3,0.0\nR004,2007-12-19,-7.4,2.5,1366.71\nR004,2007-12-20,-6.6,2.0,1451.09\nR004,2007-12-21,-6.3,2.6,0.0\nR004,2007-12-22,-6.0,4.5,306.73\nR004,2007-12-23,-5.8,9.2,0.0\nR004,2007-12-24,-8.4,0.0,15259.48\nR004,2007-12-25,-13.4,1.6,17264.77\nR004,2007-12-26,-24.6,0.7,11404.95\nR004,2007-12-27,-26.0,1.6,0.0\nR004,2007-12-28,-17.7,0.3,24375.06\nR004,2007-12-29,-16.0,0.0,5204.1\nR004,2007-12-30,-23.3,0.0,25329.99\nR004,2007-12-31,-24.7,0.0,25904.77\nR004,2008-01-01,-17.6,0.0,12235.42\nR004,2008-01-02,-12.3,0.5,12581.62\nR004,2008-01-03,-20.4,0.1,24240.13\nR004,2008-01-04,-24.5,0.6,25037.52\nR004,2008-01-05,-31.0,0.4,18617.59\nR004,2008-01-06,-18.4,5.0,0.0\nR004,2008-01-07,-15.7,0.8,12883.84\nR004,2008-01-08,-19.0,0.0,15924.59\nR004,2008-01-09,-20.7,0.0,25200.0\nR004,2008-01-10,-18.5,0.0,18534.15\nR004,2008-01-11,-23.5,0.0,25200.0\nR004,2008-01-12,-27.1,0.0,25616.19\nR004,2008-01-13,-28.4,0.0,27417.59\nR004,2008-01-14,-28.6,0.0,27047.55\nR004,2008-01-15,-22.3,0.0,27009.64\nR004,2008-01-16,-21.4,1.7,0.0\nR004,2008-01-17,-20.3,4.9,0.0\nR004,2008-01-18,-27.1,0.5,25419.52\nR004,2008-01-19,-26.0,0.0,15485.93\nR004,2008-01-20,-17.2,1.9,1962.68\nR004,2008-01-21,-13.9,2.5,0.0\nR004,2008-01-22,-12.8,0.0,22642.64\nR004,2008-01-23,-16.1,0.0,27251.06\nR004,2008-01-24,-20.2,0.0,29125.75\nR004,2008-01-25,-23.7,0.0,29609.9\nR004,2008-01-26,-25.0,0.0,29166.26\nR004,2008-01-27,-24.8,0.0,30111.5\nR004,2008-01-28,-24.6,0.0,30591.87\nR004,2008-01-29,-24.3,0.0,30893.51\nR004,2008-01-30,-24.3,0.0,31104.35\nR004,2008-01-31,-23.2,0.0,31317.29\nR004,2008-02-01,-22.5,0.0,25200.0\nR004,2008-02-02,-16.5,0.0,26918.24\nR004,2008-02-03,-18.1,0.4,8249.13\nR004,2008-02-04,-20.0,0.0,25791.42\nR004,2008-02-05,-21.8,0.0,31937.44\nR004,2008-02-06,-21.9,0.0,32052.93\nR004,2008-02-07,-18.4,0.0,22946.6\nR004,2008-02-08,-13.5,2.8,1242.36\nR004,2008-02-09,-16.0,1.8,26284.21\nR004,2008-02-10,-18.2,0.0,29546.25\nR004,2008-02-11,-18.3,0.0,30144.89\nR004,2008-02-12,-13.6,0.3,10434.02\nR004,2008-02-13,-13.2,0.2,31088.11\nR004,2008-02-14,-18.5,0.0,32908.33\nR004,2008-02-15,-19.2,0.0,30308.63\nR004,2008-02-16,-13.8,0.1,18266.08\nR004,2008-02-17,-11.2,0.0,32400.0\nR004,2008-02-18,-13.6,5.7,0.0\nR004,2008-02-19,-11.9,2.6,32400.0\nR004,2008-02-20,-18.1,0.0,31907.86\nR004,2008-02-21,-17.4,0.0,20085.43\nR004,2008-02-22,-7.3,1.4,0.0\nR004,2008-02-23,-8.4,0.4,29618.67\nR004,2008-02-24,-8.3,0.0,17349.59\nR004,2008-02-25,-8.2,0.0,36349.72\nR004,2008-02-26,-8.6,0.0,35424.33\nR004,2008-02-27,-2.8,0.0,17099.72\nR004,2008-02-28,-3.7,6.9,2508.25\nR004,2008-02-29,-4.0,7.7,0.0\nR004,2008-03-01,-1.5,0.4,38412.64\nR004,2008-03-02,-4.3,0.0,28800.0\nR004,2008-03-03,-1.8,2.4,5919.54\nR004,2008-03-04,-4.2,0.0,38792.18\nR004,2008-03-05,-3.0,0.0,38904.86\nR004,2008-03-06,-0.8,0.0,32565.51\nR004,2008-03-07,-0.6,1.6,5101.54\nR004,2008-03-08,-1.5,6.4,0.0\nR004,2008-03-09,1.5,8.4,20499.45\nR004,2008-03-10,-2.8,8.6,16793.37\nR004,2008-03-11,-8.1,2.9,34819.76\nR004,2008-03-12,-10.9,0.0,33624.42\nR004,2008-03-13,-2.8,13.4,5872.79\nR004,2008-03-14,-2.9,0.8,34777.81\nR004,2008-03-15,-1.7,0.0,39194.4\nR004,2008-03-16,-8.2,2.4,28800.0\nR004,2008-03-17,-11.6,0.0,38030.94\nR004,2008-03-18,-1.4,5.8,16367.66\nR004,2008-03-19,-0.2,6.2,20649.19\nR004,2008-03-20,-0.6,8.7,7748.6\nR004,2008-03-21,-0.3,0.0,40883.29\nR004,2008-03-22,1.0,0.0,37517.1\nR004,2008-03-23,4.5,0.0,42790.06\nR004,2008-03-24,0.1,3.2,17922.06\nR004,2008-03-25,-3.5,0.5,31560.04\nR004,2008-03-26,-5.7,0.0,43445.5\nR004,2008-03-27,-4.2,0.0,39199.63\nR004,2008-03-28,1.3,0.0,24860.32\nR004,2008-03-29,-0.1,0.0,44529.36\nR004,2008-03-30,0.4,0.0,44860.08\nR004,2008-03-31,0.5,0.0,44426.36\nR004,2008-04-01,-0.3,0.0,45141.5\nR004,2008-04-02,-0.7,0.0,45475.03\nR004,2008-04-03,1.5,0.0,45041.98\nR004,2008-04-04,1.1,0.3,40096.48\nR004,2008-04-05,3.6,5.4,32797.54\nR004,2008-04-06,-4.9,3.4,42827.28\nR004,2008-04-07,-7.2,0.0,46440.38\nR004,2008-04-08,-3.8,0.9,16265.94\nR004,2008-04-09,-3.2,5.8,15028.09\nR004,2008-04-10,-1.7,0.9,39631.49\nR004,2008-04-11,3.6,0.7,33130.64\nR004,2008-04-12,9.4,5.1,41184.42\nR004,2008-04-13,9.2,5.2,47848.28\nR004,2008-04-14,11.3,0.6,42304.93\nR004,2008-04-15,1.3,3.4,42428.71\nR004,2008-04-16,1.1,12.1,19702.06\nR004,2008-04-17,2.8,7.4,16177.83\nR004,2008-04-18,-6.2,0.0,47661.88\nR004,2008-04-19,-4.3,0.0,47472.57\nR004,2008-04-20,2.1,0.4,31782.43\nR004,2008-04-21,6.0,0.0,49915.51\nR004,2008-04-22,7.3,0.0,49699.63\nR004,2008-04-23,9.0,0.0,50400.0\nR004,2008-04-24,10.3,2.7,49341.57\nR004,2008-04-25,11.6,2.0,31811.01\nR004,2008-04-26,7.3,22.5,0.0\nR004,2008-04-27,5.3,0.0,49145.06\nR004,2008-04-28,1.7,4.1,36169.25\nR004,2008-04-29,0.3,0.3,49719.25\nR004,2008-04-30,1.1,4.6,21406.33\nR004,2008-05-01,-1.1,10.9,25200.0\nR004,2008-05-02,0.9,0.0,38846.59\nR004,2008-05-03,4.0,5.2,31111.67\nR004,2008-05-04,3.1,6.3,20643.26\nR004,2008-05-05,2.1,5.4,38300.38\nR004,2008-05-06,4.2,1.1,22208.39\nR004,2008-05-07,9.0,0.0,51741.99\nR004,2008-05-08,12.0,0.0,43525.15\nR004,2008-05-09,13.7,0.0,52088.24\nR004,2008-05-10,15.6,0.0,51059.52\nR004,2008-05-11,14.9,0.5,44464.54\nR004,2008-05-12,12.0,0.1,40291.26\nR004,2008-05-13,13.0,0.0,52897.16\nR004,2008-05-14,14.8,0.0,53692.27\nR004,2008-05-15,15.8,0.0,48121.34\nR004,2008-05-16,17.0,0.0,54428.47\nR004,2008-05-17,14.6,0.6,33027.59\nR004,2008-05-18,13.2,3.8,49455.83\nR004,2008-05-19,13.7,0.0,52407.74\nR004,2008-05-20,9.2,8.0,16115.3\nR004,2008-05-21,6.1,5.9,10797.22\nR004,2008-05-22,8.5,0.0,51979.94\nR004,2008-05-23,12.1,0.0,57292.71\nR004,2008-05-24,6.1,10.6,9831.17\nR004,2008-05-25,2.5,8.1,46574.24\nR004,2008-05-26,4.1,0.0,40333.64\nR004,2008-05-27,7.8,0.7,52440.22\nR004,2008-05-28,10.8,0.2,52174.4\nR004,2008-05-29,18.1,0.8,56992.82\nR004,2008-05-30,18.3,0.2,54000.0\nR004,2008-05-31,21.1,0.0,57317.9\nR004,2008-06-01,21.8,0.0,54554.92\nR004,2008-06-02,17.9,14.8,30869.71\nR004,2008-06-03,16.8,3.5,57569.56\nR004,2008-06-04,16.9,0.3,57111.51\nR004,2008-06-05,19.6,0.0,48410.76\nR004,2008-06-06,17.4,3.8,39922.08\nR004,2008-06-07,15.5,0.0,57600.0\nR004,2008-06-08,18.4,0.0,56887.91\nR004,2008-06-09,20.3,0.2,56053.39\nR004,2008-06-10,17.9,5.5,46523.26\nR004,2008-06-11,9.5,3.8,55723.8\nR004,2008-06-12,12.2,1.7,46800.0\nR004,2008-06-13,11.0,20.8,35599.48\nR004,2008-06-14,10.3,12.3,28920.51\nR004,2008-06-15,13.3,0.0,50743.47\nR004,2008-06-16,15.9,0.0,54589.54\nR004,2008-06-17,12.3,7.6,6422.38\nR004,2008-06-18,9.8,10.4,1702.27\nR004,2008-06-19,10.3,11.5,5580.53\nR004,2008-06-20,13.3,0.7,32029.26\nR004,2008-06-21,16.5,1.9,51217.58\nR004,2008-06-22,18.0,0.0,51302.89\nR004,2008-06-23,17.3,5.6,37937.69\nR004,2008-06-24,17.6,2.0,44581.0\nR004,2008-06-25,17.3,0.4,56541.1\nR004,2008-06-26,16.7,0.0,56249.67\nR004,2008-06-27,18.1,0.0,57600.0\nR004,2008-06-28,18.2,1.4,40020.51\nR004,2008-06-29,17.3,1.7,40789.88\nR004,2008-06-30,15.5,2.4,47000.99\nR004,2008-07-01,18.3,2.2,48717.54\nR004,2008-07-02,17.6,2.7,54609.91\nR004,2008-07-03,19.8,0.0,47568.75\nR004,2008-07-04,19.0,1.8,43970.81\nR004,2008-07-05,15.5,22.8,20630.18\nR004,2008-07-06,15.3,7.4,19732.34\nR004,2008-07-07,13.4,7.5,26662.07\nR004,2008-07-08,12.9,15.3,29060.53\nR004,2008-07-09,13.2,1.8,55229.44\nR004,2008-07-10,17.2,2.7,53380.11\nR004,2008-07-11,18.0,4.7,41874.31\nR004,2008-07-12,16.8,3.5,54000.0\nR004,2008-07-13,17.3,0.0,47896.12\nR004,2008-07-14,18.3,0.0,53577.2\nR004,2008-07-15,19.7,0.0,55661.29\nR004,2008-07-16,21.0,0.1,55688.9\nR004,2008-07-17,21.4,0.0,55717.41\nR004,2008-07-18,21.4,1.0,50733.19\nR004,2008-07-19,20.9,4.5,42546.96\nR004,2008-07-20,19.4,7.9,31096.1\nR004,2008-07-21,20.6,0.6,46986.25\nR004,2008-07-22,19.6,1.4,54317.42\nR004,2008-07-23,17.7,6.9,25200.0\nR004,2008-07-24,12.5,9.7,17807.35\nR004,2008-07-25,12.3,4.8,41966.72\nR004,2008-07-26,14.4,0.0,50067.66\nR004,2008-07-27,17.7,0.0,54266.77\nR004,2008-07-28,19.9,0.2,54000.0\nR004,2008-07-29,20.9,1.7,50779.34\nR004,2008-07-30,19.6,0.0,52392.43\nR004,2008-07-31,21.2,0.0,49816.4\nR004,2008-08-01,21.4,7.4,46800.0\nR004,2008-08-02,16.1,0.4,52207.35\nR004,2008-08-03,15.8,0.7,47271.91\nR004,2008-08-04,14.3,17.0,31765.77\nR004,2008-08-05,15.3,8.2,38434.65\nR004,2008-08-06,17.9,0.0,48571.31\nR004,2008-08-07,15.9,0.0,52423.07\nR004,2008-08-08,15.8,0.0,52051.08\nR004,2008-08-09,19.0,0.0,51682.76\nR004,2008-08-10,22.6,0.0,51318.08\nR004,2008-08-11,14.6,11.6,0.0\nR004,2008-08-12,12.7,0.0,21306.58\nR004,2008-08-13,15.2,0.0,45304.26\nR004,2008-08-14,16.0,0.0,50715.06\nR004,2008-08-15,17.0,0.0,50400.0\nR004,2008-08-16,13.9,22.0,22631.12\nR004,2008-08-17,13.0,6.4,21428.68\nR004,2008-08-18,12.7,9.4,30821.36\nR004,2008-08-19,12.0,9.2,18114.14\nR004,2008-08-20,10.9,12.4,937.94\nR004,2008-08-21,8.6,6.3,38552.47\nR004,2008-08-22,7.9,0.4,36754.36\nR004,2008-08-23,8.6,1.3,28671.67\nR004,2008-08-24,9.2,7.4,33775.28\nR004,2008-08-25,10.5,0.0,38714.6\nR004,2008-08-26,10.3,16.2,15595.57\nR004,2008-08-27,10.2,8.8,25696.64\nR004,2008-08-28,9.8,2.8,31571.33\nR004,2008-08-29,15.1,0.0,48177.98\nR004,2008-08-30,16.5,0.0,45016.32\nR004,2008-08-31,14.7,9.0,36890.06\nR004,2008-09-01,12.7,1.1,30025.79\nR004,2008-09-02,13.2,0.1,43064.62\nR004,2008-09-03,13.3,0.9,44281.55\nR004,2008-09-04,10.6,2.9,38105.34\nR004,2008-09-05,10.3,0.0,44156.05\nR004,2008-09-06,11.1,0.8,20948.82\nR004,2008-09-07,8.3,5.0,964.89\nR004,2008-09-08,8.8,3.6,29883.69\nR004,2008-09-09,8.3,17.9,0.0\nR004,2008-09-10,11.0,4.4,8345.53\nR004,2008-09-11,13.5,0.4,43200.0\nR004,2008-09-12,8.1,0.0,29455.07\nR004,2008-09-13,7.1,0.0,24038.84\nR004,2008-09-14,6.7,0.0,43200.0\nR004,2008-09-15,6.2,10.1,7602.47\nR004,2008-09-16,5.7,7.1,20682.37\nR004,2008-09-17,5.0,16.8,0.0\nR004,2008-09-18,6.7,7.6,9765.56\nR004,2008-09-19,4.0,2.7,30169.92\nR004,2008-09-20,2.1,3.7,32440.05\nR004,2008-09-21,3.2,0.0,32496.37\nR004,2008-09-22,3.2,0.3,35464.19\nR004,2008-09-23,2.1,0.2,39943.68\nR004,2008-09-24,2.3,0.0,42785.31\nR004,2008-09-25,2.6,0.0,42535.39\nR004,2008-09-26,4.0,0.0,41906.06\nR004,2008-09-27,4.7,0.0,40663.11\nR004,2008-09-28,7.7,1.5,38487.96\nR004,2008-09-29,2.9,3.7,9721.5\nR004,2008-09-30,1.6,0.0,36525.35\nR004,2008-10-01,1.6,0.0,37718.76\nR004,2008-10-02,2.5,0.0,37074.49\nR004,2008-10-03,2.9,0.0,36125.04\nR004,2008-10-04,4.1,0.0,20486.59\nR004,2008-10-05,5.3,0.0,36024.68\nR004,2008-10-06,5.5,24.6,4287.07\nR004,2008-10-07,5.9,12.3,1251.45\nR004,2008-10-08,3.6,1.2,5999.2\nR004,2008-10-09,1.7,0.0,36000.0\nR004,2008-10-10,4.2,0.0,36000.0\nR004,2008-10-11,8.1,0.0,36000.0\nR004,2008-10-12,5.8,1.9,16476.4\nR004,2008-10-13,1.6,0.1,23215.4\nR004,2008-10-14,1.6,0.2,8205.68\nR004,2008-10-15,4.2,3.3,17379.55\nR004,2008-10-16,3.0,3.0,26096.43\nR004,2008-10-17,2.0,5.9,947.4\nR004,2008-10-18,2.4,5.5,3319.95\nR004,2008-10-19,-0.2,0.1,10608.53\nR004,2008-10-20,-2.2,2.1,7182.05\nR004,2008-10-21,-3.1,0.0,30256.5\nR004,2008-10-22,-1.0,0.0,30495.85\nR004,2008-10-23,-0.5,0.0,30510.9\nR004,2008-10-24,1.6,0.0,27176.28\nR004,2008-10-25,1.6,0.0,33139.41\nR004,2008-10-26,1.6,0.0,32371.84\nR004,2008-10-27,2.2,0.0,29225.52\nR004,2008-10-28,0.3,6.1,0.0\nR004,2008-10-29,3.0,2.9,952.27\nR004,2008-10-30,2.6,1.9,21036.43\nR004,2008-10-31,0.9,3.6,13878.99\nR004,2008-11-01,1.5,7.5,0.0\nR004,2008-11-02,3.8,5.6,20474.15\nR004,2008-11-03,3.3,25.9,0.0\nR004,2008-11-04,2.6,18.1,0.0\nR004,2008-11-05,2.5,6.6,11686.88\nR004,2008-11-06,0.1,2.4,23987.43\nR004,2008-11-07,2.4,0.2,27210.16\nR004,2008-11-08,2.6,8.8,1206.7\nR004,2008-11-09,-2.1,9.1,21087.09\nR004,2008-11-10,-6.2,6.0,0.0\nR004,2008-11-11,-5.8,10.3,0.0\nR004,2008-11-12,-5.3,8.6,0.0\nR004,2008-11-13,-4.2,4.2,0.0\nR004,2008-11-14,-2.9,10.6,0.0\nR004,2008-11-15,-2.5,11.2,0.0\nR004,2008-11-16,-4.7,1.7,725.07\nR004,2008-11-17,-7.5,0.0,28800.0\nR004,2008-11-18,-8.3,0.0,28800.0\nR004,2008-11-19,-10.3,0.0,28800.0\nR004,2008-11-20,-14.4,0.0,28772.8\nR004,2008-11-21,-13.0,0.0,7605.81\nR004,2008-11-22,-14.8,0.0,28623.06\nR004,2008-11-23,-10.7,0.0,15488.86\nR004,2008-11-24,-1.9,6.2,10400.35\nR004,2008-11-25,-3.2,0.0,28361.58\nR004,2008-11-26,-6.7,0.0,28356.52\nR004,2008-11-27,-5.7,0.0,3844.67\nR004,2008-11-28,-2.1,0.0,26636.24\nR004,2008-11-29,-6.8,6.4,17841.61\nR004,2008-11-30,-5.8,10.7,0.0\nR004,2008-12-01,-10.8,6.5,15174.64\nR004,2008-12-02,-25.1,0.0,22729.69\nR004,2008-12-03,-28.5,0.0,18773.06\nR004,2008-12-04,-21.5,0.0,26824.88\nR004,2008-12-05,-15.0,3.0,0.0\nR004,2008-12-06,-5.2,16.1,0.0\nR004,2008-12-07,-3.2,29.2,0.0\nR004,2008-12-08,-3.9,14.9,0.0\nR004,2008-12-09,-4.9,8.9,0.0\nR004,2008-12-10,-1.4,16.7,0.0\nR004,2008-12-11,-5.7,1.8,7892.33\nR004,2008-12-12,-9.5,0.0,3521.74\nR004,2008-12-13,-13.6,0.0,25676.54\nR004,2008-12-14,-16.1,0.2,22291.38\nR004,2008-12-15,-13.2,9.5,12130.73\nR004,2008-12-16,-18.3,0.0,16782.01\nR004,2008-12-17,-9.5,6.9,0.0\nR004,2008-12-18,-14.7,0.7,6485.37\nR004,2008-12-19,-16.7,0.0,2316.54\nR004,2008-12-20,-19.6,0.0,25407.15\nR004,2008-12-21,-24.3,0.0,25448.13\nR004,2008-12-22,-26.2,0.0,22414.82\nR004,2008-12-23,-25.8,0.0,25533.84\nR004,2008-12-24,-22.8,0.0,22368.49\nR004,2008-12-25,-20.0,0.0,15575.88\nR004,2008-12-26,-19.4,0.6,0.0\nR004,2008-12-27,-20.1,0.0,12335.16\nR004,2008-12-28,-23.6,0.0,25260.71\nR004,2008-12-29,-25.7,0.0,25837.46\nR004,2008-12-30,-20.0,3.3,0.0\nR004,2008-12-31,-16.6,0.4,24342.1\nR004,2009-01-01,-19.5,0.0,24973.89\nR004,2009-01-02,-23.7,0.0,23716.36\nR004,2009-01-03,-27.0,0.0,26127.41\nR004,2009-01-04,-23.9,0.0,11626.11\nR004,2009-01-05,-19.7,0.0,17754.85\nR004,2009-01-06,-15.8,2.0,2292.17\nR004,2009-01-07,-22.1,0.0,26229.24\nR004,2009-01-08,-17.1,1.1,0.0\nR004,2009-01-09,-18.8,0.0,22455.74\nR004,2009-01-10,-24.1,0.0,24171.82\nR004,2009-01-11,-21.3,0.0,10559.38\nR004,2009-01-12,-18.1,0.0,9669.86\nR004,2009-01-13,-16.1,0.0,26064.41\nR004,2009-01-14,-18.4,0.0,21550.45\nR004,2009-01-15,-14.6,2.4,13542.56\nR004,2009-01-16,-10.3,8.9,0.0\nR004,2009-01-17,-12.6,4.3,21600.0\nR004,2009-01-18,-10.1,12.2,0.0\nR004,2009-01-19,-12.7,9.3,0.0\nR004,2009-01-20,-18.2,2.8,14963.3\nR004,2009-01-21,-27.7,0.0,28153.15\nR004,2009-01-22,-28.8,0.0,28639.63\nR004,2009-01-23,-25.9,0.0,29124.35\nR004,2009-01-24,-27.2,0.0,29142.15\nR004,2009-01-25,-15.0,4.3,0.0\nR004,2009-01-26,-7.5,12.3,0.0\nR004,2009-01-27,-13.4,1.9,27585.26\nR004,2009-01-28,-17.3,0.0,24949.21\nR004,2009-01-29,-18.3,4.8,0.0\nR004,2009-01-30,-14.2,8.4,0.0\nR004,2009-01-31,-17.3,2.7,26943.51\nR004,2009-02-01,-23.1,0.0,17217.05\nR004,2009-02-02,-16.6,5.4,0.0\nR004,2009-02-03,-14.5,1.7,9616.48\nR004,2009-02-04,-7.1,4.8,8393.93\nR004,2009-02-05,-7.1,7.5,8541.1\nR004,2009-02-06,-5.0,6.2,6129.76\nR004,2009-02-07,-5.3,1.2,0.0\nR004,2009-02-08,-6.9,0.3,2371.45\nR004,2009-02-09,-8.3,10.5,6516.1\nR004,2009-02-10,-15.3,5.4,0.0\nR004,2009-02-11,-23.9,0.1,31383.09\nR004,2009-02-12,-28.2,0.0,23317.9\nR004,2009-02-13,-31.7,0.0,32400.0\nR004,2009-02-14,-29.2,3.9,0.0\nR004,2009-02-15,-23.1,0.3,31436.28\nR004,2009-02-16,-22.5,0.0,17089.93\nR004,2009-02-17,-25.7,0.0,32400.0\nR004,2009-02-18,-26.3,0.9,30633.18\nR004,2009-02-19,-13.1,8.5,0.0\nR004,2009-02-20,-16.1,0.0,14757.79\nR004,2009-02-21,-21.6,0.0,31486.19\nR004,2009-02-22,-20.6,1.1,13619.6\nR004,2009-02-23,-18.4,0.0,33799.32\nR004,2009-02-24,-19.5,0.0,36326.97\nR004,2009-02-25,-16.9,0.0,36796.73\nR004,2009-02-26,-15.8,0.0,35521.51\nR004,2009-02-27,-13.4,0.0,37734.7\nR004,2009-02-28,-13.4,0.0,38105.93\nR004,2009-03-01,-13.4,0.0,38358.18\nR004,2009-03-02,-12.9,0.0,38539.23\nR004,2009-03-03,-12.9,0.5,29362.31\nR004,2009-03-04,-9.9,0.0,38766.03\nR004,2009-03-05,-9.2,0.0,36518.84\nR004,2009-03-06,-4.9,0.0,32141.99\nR004,2009-03-07,-7.1,0.3,34430.31\nR004,2009-03-08,-10.7,0.0,36149.48\nR004,2009-03-09,-10.4,0.0,39003.8\nR004,2009-03-10,-10.4,0.8,4418.85\nR004,2009-03-11,-11.9,1.3,39545.89\nR004,2009-03-12,-11.4,0.0,2629.66\nR004,2009-03-13,-5.0,3.7,17218.4\nR004,2009-03-14,-2.3,9.8,101.05\nR004,2009-03-15,-2.8,4.7,15772.54\nR004,2009-03-16,-6.4,0.2,9556.15\nR004,2009-03-17,-11.9,0.0,25770.34\nR004,2009-03-18,-12.3,0.0,41500.7\nR004,2009-03-19,-11.7,2.1,647.37\nR004,2009-03-20,-9.9,0.9,31778.64\nR004,2009-03-21,-12.2,0.0,39600.0\nR004,2009-03-22,-11.1,0.0,42496.55\nR004,2009-03-23,-7.8,0.0,42641.95\nR004,2009-03-24,0.0,1.1,33073.71\nR004,2009-03-25,-0.3,3.1,26915.93\nR004,2009-03-26,1.1,0.0,38135.56\nR004,2009-03-27,3.4,0.0,43501.44\nR004,2009-03-28,2.0,0.0,44212.14\nR004,2009-03-29,3.5,0.0,44160.1\nR004,2009-03-30,1.6,0.0,44490.68\nR004,2009-03-31,0.7,0.0,30558.45\nR004,2009-04-01,0.8,0.2,28797.21\nR004,2009-04-02,2.9,0.0,37558.13\nR004,2009-04-03,3.5,0.0,45055.38\nR004,2009-04-04,5.4,0.0,30368.96\nR004,2009-04-05,2.2,6.1,13410.05\nR004,2009-04-06,2.7,2.7,25200.0\nR004,2009-04-07,2.4,7.0,3806.97\nR004,2009-04-08,-1.6,0.0,46470.64\nR004,2009-04-09,1.6,0.0,47284.3\nR004,2009-04-10,-0.3,0.4,34223.56\nR004,2009-04-11,-1.7,0.0,40093.78\nR004,2009-04-12,0.7,0.1,30274.12\nR004,2009-04-13,4.4,0.0,39201.28\nR004,2009-04-14,5.0,0.7,43332.24\nR004,2009-04-15,0.7,2.7,6629.27\nR004,2009-04-16,1.7,0.0,41900.22\nR004,2009-04-17,0.8,5.4,26055.1\nR004,2009-04-18,3.5,0.0,45011.89\nR004,2009-04-19,3.3,0.7,46042.14\nR004,2009-04-20,3.0,0.0,50139.94\nR004,2009-04-21,5.6,0.0,50267.82\nR004,2009-04-22,6.2,0.0,48789.36\nR004,2009-04-23,5.7,0.0,14452.13\nR004,2009-04-24,8.7,0.3,45432.22\nR004,2009-04-25,11.2,0.1,23051.11\nR004,2009-04-26,11.1,3.6,36958.64\nR004,2009-04-27,5.5,3.4,36958.77\nR004,2009-04-28,6.7,0.8,40368.55\nR004,2009-04-29,6.3,6.7,34469.91\nR004,2009-04-30,6.0,8.9,14059.85\nR004,2009-05-01,10.8,14.4,22541.77\nR004,2009-05-02,12.7,4.4,34795.85\nR004,2009-05-03,11.6,13.0,30655.53\nR004,2009-05-04,3.7,3.4,48379.95\nR004,2009-05-05,2.9,2.3,44614.06\nR004,2009-05-06,3.3,2.6,3748.27\nR004,2009-05-07,6.2,0.1,43169.18\nR004,2009-05-08,7.0,0.0,51711.21\nR004,2009-05-09,9.6,1.3,48046.3\nR004,2009-05-10,13.5,0.0,51630.82\nR004,2009-05-11,14.9,0.0,52648.03\nR004,2009-05-12,16.6,0.0,51699.88\nR004,2009-05-13,14.2,5.0,40454.38\nR004,2009-05-14,7.3,0.0,53978.15\nR004,2009-05-15,9.2,0.0,54909.01\nR004,2009-05-16,13.5,0.0,55406.2\nR004,2009-05-17,14.9,5.0,45599.35\nR004,2009-05-18,11.5,25.7,108.56\nR004,2009-05-19,8.8,14.6,38014.06\nR004,2009-05-20,11.1,0.0,45722.02\nR004,2009-05-21,14.5,5.7,39153.74\nR004,2009-05-22,6.7,3.4,49118.0\nR004,2009-05-23,3.7,3.8,10940.21\nR004,2009-05-24,9.2,3.0,38165.48\nR004,2009-05-25,4.9,10.3,23205.81\nR004,2009-05-26,3.7,3.6,29606.66\nR004,2009-05-27,7.4,2.9,48252.15\nR004,2009-05-28,10.5,0.0,57600.0\nR004,2009-05-29,16.5,0.0,57600.0\nR004,2009-05-30,19.2,0.6,54000.0\nR004,2009-05-31,13.8,10.7,20571.06\nR004,2009-06-01,16.2,6.1,50044.3\nR004,2009-06-02,17.4,1.4,46800.0\nR004,2009-06-03,14.5,9.8,43241.49\nR004,2009-06-04,11.1,5.6,40862.48\nR004,2009-06-05,10.0,1.0,54085.16\nR004,2009-06-06,9.7,1.0,47061.26\nR004,2009-06-07,14.3,0.0,57600.0\nR004,2009-06-08,18.9,0.0,57600.0\nR004,2009-06-09,17.4,6.5,42155.14\nR004,2009-06-10,16.3,1.6,45103.32\nR004,2009-06-11,16.5,1.3,55747.6\nR004,2009-06-12,13.6,7.2,9049.25\nR004,2009-06-13,8.1,17.9,29473.68\nR004,2009-06-14,8.9,5.4,43262.85\nR004,2009-06-15,10.1,5.2,49991.84\nR004,2009-06-16,10.3,8.4,50517.06\nR004,2009-06-17,7.0,9.3,34370.15\nR004,2009-06-18,5.4,14.3,0.0\nR004,2009-06-19,9.5,0.8,45597.94\nR004,2009-06-20,10.6,0.0,56498.97\nR004,2009-06-21,12.7,0.0,53953.85\nR004,2009-06-22,12.9,0.4,54763.52\nR004,2009-06-23,12.8,6.2,57063.98\nR004,2009-06-24,13.5,0.5,57600.0\nR004,2009-06-25,10.4,22.3,32803.37\nR004,2009-06-26,11.1,12.1,21094.08\nR004,2009-06-27,11.6,7.8,47523.58\nR004,2009-06-28,11.4,4.1,49706.07\nR004,2009-06-29,11.6,1.0,27799.32\nR004,2009-06-30,13.4,0.8,46505.36\nR004,2009-07-01,15.6,3.3,53135.95\nR004,2009-07-02,12.8,2.5,43402.31\nR004,2009-07-03,14.5,0.0,57600.0\nR004,2009-07-04,17.9,0.0,56954.84\nR004,2009-07-05,20.9,0.0,52887.79\nR004,2009-07-06,18.4,12.1,33726.62\nR004,2009-07-07,18.5,0.0,48290.85\nR004,2009-07-08,17.9,1.7,52750.58\nR004,2009-07-09,16.3,6.0,30356.36\nR004,2009-07-10,18.6,0.0,48362.58\nR004,2009-07-11,18.2,0.4,48081.55\nR004,2009-07-12,19.7,0.1,45812.33\nR004,2009-07-13,18.3,4.1,47693.4\nR004,2009-07-14,17.8,0.4,53151.78\nR004,2009-07-15,18.3,0.2,50746.9\nR004,2009-07-16,19.0,0.6,42086.52\nR004,2009-07-17,17.8,3.6,40801.46\nR004,2009-07-18,18.2,1.7,43188.25\nR004,2009-07-19,17.8,5.3,41850.68\nR004,2009-07-20,16.3,11.8,44283.76\nR004,2009-07-21,19.4,0.2,53641.94\nR004,2009-07-22,13.1,12.8,16507.96\nR004,2009-07-23,13.3,0.2,48158.95\nR004,2009-07-24,11.5,10.9,42910.31\nR004,2009-07-25,12.7,6.2,50112.88\nR004,2009-07-26,14.5,0.9,40743.69\nR004,2009-07-27,16.4,0.2,54000.0\nR004,2009-07-28,16.9,1.0,42350.97\nR004,2009-07-29,15.4,3.3,19339.36\nR004,2009-07-30,17.7,1.8,52920.83\nR004,2009-07-31,16.3,7.4,41750.74\nR004,2009-08-01,16.5,1.4,52154.49\nR004,2009-08-02,14.3,2.2,11123.35\nR004,2009-08-03,17.2,1.3,46874.55\nR004,2009-08-04,15.0,9.1,14486.7\nR004,2009-08-05,14.2,11.9,38510.89\nR004,2009-08-06,13.2,13.7,22243.6\nR004,2009-08-07,13.2,1.2,41008.23\nR004,2009-08-08,15.8,0.0,51204.25\nR004,2009-08-09,19.4,0.0,48457.55\nR004,2009-08-10,19.3,1.1,39585.21\nR004,2009-08-11,16.1,25.9,19852.32\nR004,2009-08-12,11.9,21.0,39045.44\nR004,2009-08-13,9.3,11.2,35646.06\nR004,2009-08-14,9.3,6.7,32441.1\nR004,2009-08-15,10.0,1.2,48114.85\nR004,2009-08-16,11.8,1.6,37791.69\nR004,2009-08-17,12.7,0.1,44258.55\nR004,2009-08-18,12.8,2.5,47017.76\nR004,2009-08-19,12.0,1.4,41502.94\nR004,2009-08-20,13.8,0.5,46609.12\nR004,2009-08-21,12.5,10.4,3786.08\nR004,2009-08-22,12.6,16.2,14361.1\nR004,2009-08-23,11.9,8.5,4302.98\nR004,2009-08-24,12.8,2.3,35738.61\nR004,2009-08-25,13.2,0.2,32601.63\nR004,2009-08-26,13.7,0.0,39300.15\nR004,2009-08-27,13.5,1.0,30111.02\nR004,2009-08-28,14.2,4.4,38040.88\nR004,2009-08-29,14.0,7.9,13557.24\nR004,2009-08-30,12.2,24.8,10463.99\nR004,2009-08-31,13.7,5.9,33657.06\nR004,2009-09-01,12.2,5.8,26791.57\nR004,2009-09-02,5.9,4.9,24429.33\nR004,2009-09-03,6.2,0.6,43343.37\nR004,2009-09-04,6.5,0.0,45170.87\nR004,2009-09-05,8.6,0.0,44900.05\nR004,2009-09-06,8.9,0.0,43515.72\nR004,2009-09-07,9.4,1.0,33945.88\nR004,2009-09-08,9.4,0.0,44067.4\nR004,2009-09-09,8.7,0.0,43752.08\nR004,2009-09-10,9.2,0.0,43200.0\nR004,2009-09-11,9.9,0.5,31806.26\nR004,2009-09-12,11.0,0.8,36129.82\nR004,2009-09-13,12.2,3.4,25744.35\nR004,2009-09-14,10.9,0.2,40738.01\nR004,2009-09-15,8.0,2.7,14784.23\nR004,2009-09-16,8.8,1.1,41703.6\nR004,2009-09-17,9.4,16.0,23378.64\nR004,2009-09-18,2.2,6.9,30641.09\nR004,2009-09-19,4.9,0.0,43200.0\nR004,2009-09-20,7.4,0.0,39806.17\nR004,2009-09-21,9.7,0.5,32098.64\nR004,2009-09-22,12.6,2.6,41913.64\nR004,2009-09-23,7.2,18.4,0.0\nR004,2009-09-24,4.5,0.0,21551.62\nR004,2009-09-25,7.1,0.0,31783.45\nR004,2009-09-26,10.5,0.0,30334.42\nR004,2009-09-27,12.0,0.4,38325.54\nR004,2009-09-28,10.1,10.6,27586.69\nR004,2009-09-29,7.2,0.0,27920.46\nR004,2009-09-30,7.5,0.0,38323.45\nR004,2009-10-01,7.2,0.0,37681.71\nR004,2009-10-02,8.7,0.0,37039.11\nR004,2009-10-03,10.3,3.9,26941.63\nR004,2009-10-04,8.4,11.7,0.0\nR004,2009-10-05,7.0,0.0,36000.0\nR004,2009-10-06,3.8,0.2,10075.27\nR004,2009-10-07,1.9,0.0,36000.0\nR004,2009-10-08,2.0,0.0,36000.0\nR004,2009-10-09,3.9,0.0,36000.0\nR004,2009-10-10,4.6,0.0,36000.0\nR004,2009-10-11,4.2,0.0,36000.0\nR004,2009-10-12,4.9,8.7,0.0\nR004,2009-10-13,1.1,2.6,2879.99\nR004,2009-10-14,-1.8,0.1,23754.54\nR004,2009-10-15,0.5,15.8,0.0\nR004,2009-10-16,2.7,19.3,0.0\nR004,2009-10-17,-2.1,6.3,0.0\nR004,2009-10-18,0.6,13.8,0.0\nR004,2009-10-19,0.8,10.3,111.93\nR004,2009-10-20,1.2,8.8,0.0\nR004,2009-10-21,5.0,20.4,0.0\nR004,2009-10-22,2.3,11.9,0.0\nR004,2009-10-23,-4.6,3.5,25780.6\nR004,2009-10-24,-6.6,1.9,411.63\nR004,2009-10-25,-3.3,6.9,0.0\nR004,2009-10-26,-3.2,18.4,0.0\nR004,2009-10-27,-10.4,0.5,22671.71\nR004,2009-10-28,-11.9,0.0,25514.27\nR004,2009-10-29,-8.9,0.8,17535.05\nR004,2009-10-30,-6.0,1.3,0.0\nR004,2009-10-31,-5.2,0.3,27663.91\nR004,2009-11-01,-8.2,0.0,29501.3\nR004,2009-11-02,-4.4,2.2,21641.82\nR004,2009-11-03,1.4,18.2,0.0\nR004,2009-11-04,3.2,38.0,291.76\nR004,2009-11-05,-5.1,5.8,3329.01\nR004,2009-11-06,-14.4,0.0,25635.96\nR004,2009-11-07,-20.7,0.0,29009.2\nR004,2009-11-08,-16.7,0.0,29513.22\nR004,2009-11-09,-18.2,0.0,29131.6\nR004,2009-11-10,-19.3,0.0,28463.74\nR004,2009-11-11,-20.3,0.0,28806.71\nR004,2009-11-12,-21.4,0.0,27127.39\nR004,2009-11-13,-21.0,0.0,28800.0\nR004,2009-11-14,-21.1,0.0,28800.0\nR004,2009-11-15,-18.7,0.3,13825.17\nR004,2009-11-16,-9.0,9.5,0.0\nR004,2009-11-17,-8.7,0.3,566.49\nR004,2009-11-18,-11.9,2.2,1598.23\nR004,2009-11-19,-9.2,0.2,3430.28\nR004,2009-11-20,-4.3,1.6,18000.0\nR004,2009-11-21,-6.4,0.0,25398.41\nR004,2009-11-22,-8.4,4.2,1379.4\nR004,2009-11-23,-0.7,10.2,0.0\nR004,2009-11-24,-4.0,6.6,0.0\nR004,2009-11-25,-9.6,0.0,4863.64\nR004,2009-11-26,-4.1,6.0,0.0\nR004,2009-11-27,-1.6,5.2,0.0\nR004,2009-11-28,-4.7,0.0,18370.79\nR004,2009-11-29,-8.0,4.0,3269.76\nR004,2009-11-30,-3.3,7.3,0.0\nR004,2009-12-01,-4.7,7.2,0.0\nR004,2009-12-02,-9.8,11.0,9709.1\nR004,2009-12-03,-20.8,0.0,8563.44\nR004,2009-12-04,-7.1,0.0,8145.81\nR004,2009-12-05,-4.8,12.3,0.0\nR004,2009-12-06,-7.3,6.9,13954.6\nR004,2009-12-07,-15.5,0.0,19187.01\nR004,2009-12-08,-23.0,0.0,3506.21\nR004,2009-12-09,-29.1,0.0,24691.5\nR004,2009-12-10,-25.3,0.6,0.0\nR004,2009-12-11,-18.6,1.5,1022.07\nR004,2009-12-12,-17.7,0.1,4537.7\nR004,2009-12-13,-16.7,0.8,19449.84\nR004,2009-12-14,-25.4,0.0,26226.45\nR004,2009-12-15,-26.7,0.0,23161.58\nR004,2009-12-16,-22.8,0.0,25765.71\nR004,2009-12-17,-18.5,0.4,16836.56\nR004,2009-12-18,-14.7,0.0,6133.58\nR004,2009-12-19,-14.1,0.0,0.0\nR004,2009-12-20,-10.4,5.6,0.0\nR004,2009-12-21,-8.0,7.5,0.0\nR004,2009-12-22,-21.1,0.0,13154.24\nR004,2009-12-23,-33.3,0.0,20192.5\nR004,2009-12-24,-32.5,0.0,0.0\nR004,2009-12-25,-25.3,0.6,7239.77\nR004,2009-12-26,-13.3,10.7,0.0\nR004,2009-12-27,-10.7,7.3,0.0\nR004,2009-12-28,-19.0,1.5,12734.3\nR004,2009-12-29,-24.2,1.3,9135.13\nR004,2009-12-30,-13.2,10.3,0.0\nR004,2009-12-31,-14.7,5.4,15732.08\nR004,2010-01-01,-26.9,0.3,21222.87\nR004,2010-01-02,-23.0,3.7,20812.72\nR004,2010-01-03,-27.1,0.0,26115.0\nR004,2010-01-04,-25.5,0.0,25658.58\nR004,2010-01-05,-27.6,0.0,26741.6\nR004,2010-01-06,-30.5,0.0,26257.24\nR004,2010-01-07,-36.1,0.0,26745.5\nR004,2010-01-08,-37.4,0.0,26213.56\nR004,2010-01-09,-37.0,0.0,16275.28\nR004,2010-01-10,-29.5,0.5,567.74\nR004,2010-01-11,-25.7,0.0,0.0\nR004,2010-01-12,-19.4,1.0,0.0\nR004,2010-01-13,-19.0,0.5,23972.93\nR004,2010-01-14,-19.2,1.9,1384.51\nR004,2010-01-15,-11.5,7.5,0.0\nR004,2010-01-16,-14.2,1.8,9491.44\nR004,2010-01-17,-24.5,0.0,26900.9\nR004,2010-01-18,-29.9,0.0,18654.08\nR004,2010-01-19,-32.9,0.0,27780.83\nR004,2010-01-20,-31.0,0.0,27875.69\nR004,2010-01-21,-27.7,0.0,28147.64\nR004,2010-01-22,-28.2,0.0,28633.27\nR004,2010-01-23,-27.5,0.0,28650.0\nR004,2010-01-24,-28.7,0.0,29134.5\nR004,2010-01-25,-21.5,4.0,0.0\nR004,2010-01-26,-19.6,1.6,22952.23\nR004,2010-01-27,-26.1,0.0,30118.66\nR004,2010-01-28,-25.7,0.0,30775.23\nR004,2010-01-29,-24.4,0.0,30981.69\nR004,2010-01-30,-23.2,0.0,27785.53\nR004,2010-01-31,-24.1,0.0,31402.32\nR004,2010-02-01,-22.4,0.0,31317.62\nR004,2010-02-02,-27.1,0.0,31637.13\nR004,2010-02-03,-27.1,0.0,31755.43\nR004,2010-02-04,-27.5,0.0,31874.15\nR004,2010-02-05,-26.2,0.0,26977.3\nR004,2010-02-06,-26.6,0.0,26872.4\nR004,2010-02-07,-28.4,0.0,23133.19\nR004,2010-02-08,-29.8,0.0,28757.06\nR004,2010-02-09,-32.7,0.0,32400.0\nR004,2010-02-10,-33.4,0.0,32400.0\nR004,2010-02-11,-31.7,0.0,32641.23\nR004,2010-02-12,-29.8,0.0,32579.6\nR004,2010-02-13,-27.9,0.0,32517.63\nR004,2010-02-14,-18.4,1.4,10064.08\nR004,2010-02-15,-15.3,2.1,26139.67\nR004,2010-02-16,-11.1,0.2,28650.92\nR004,2010-02-17,-9.0,0.0,26426.92\nR004,2010-02-18,-9.3,0.1,383.9\nR004,2010-02-19,-9.2,2.1,26472.48\nR004,2010-02-20,-18.6,0.0,9964.84\nR004,2010-02-21,-14.0,3.3,0.0\nR004,2010-02-22,-21.4,0.2,33561.94\nR004,2010-02-23,-27.1,0.0,35844.69\nR004,2010-02-24,-27.0,0.0,36316.4\nR004,2010-02-25,-25.8,0.0,31762.35\nR004,2010-02-26,-20.8,0.0,34455.63\nR004,2010-02-27,-20.0,0.0,18834.96\nR004,2010-02-28,-14.4,2.0,16019.81\nR004,2010-03-01,-12.3,2.1,5073.3\nR004,2010-03-02,-17.6,1.9,21047.55\nR004,2010-03-03,-21.9,0.0,34911.49\nR004,2010-03-04,-17.8,2.4,669.1\nR004,2010-03-05,-16.8,0.1,35657.73\nR004,2010-03-06,-19.2,0.0,38962.04\nR004,2010-03-07,-17.4,0.0,39072.79\nR004,2010-03-08,-14.4,0.0,37526.1\nR004,2010-03-09,-8.1,1.2,28908.27\nR004,2010-03-10,-3.1,4.2,5194.73\nR004,2010-03-11,-6.4,4.5,9662.41\nR004,2010-03-12,-9.0,2.2,12486.77\nR004,2010-03-13,-9.1,4.0,8271.94\nR004,2010-03-14,-9.9,0.0,39894.61\nR004,2010-03-15,-6.2,1.0,19337.17\nR004,2010-03-16,-1.8,2.0,24297.11\nR004,2010-03-17,-1.0,3.7,5924.05\nR004,2010-03-18,-0.9,8.8,32402.26\nR004,2010-03-19,-5.4,4.5,39383.21\nR004,2010-03-20,-5.9,5.4,0.0\nR004,2010-03-21,-9.9,0.9,30323.03\nR004,2010-03-22,-9.3,4.1,619.58\nR004,2010-03-23,-11.0,0.0,42759.95\nR004,2010-03-24,-4.7,0.3,27099.0\nR004,2010-03-25,-3.3,0.5,595.24\nR004,2010-03-26,-1.1,0.0,42594.77\nR004,2010-03-27,-3.4,3.6,20474.67\nR004,2010-03-28,-8.6,0.9,35227.33\nR004,2010-03-29,-6.0,8.4,21923.61\nR004,2010-03-30,-8.7,2.4,17855.88\nR004,2010-03-31,-6.1,0.9,21467.71\nR004,2010-04-01,-1.1,0.0,39014.19\nR004,2010-04-02,-1.5,3.2,24237.92\nR004,2010-04-03,-1.7,4.9,35716.47\nR004,2010-04-04,-1.7,1.5,35351.96\nR004,2010-04-05,1.1,3.3,7432.29\nR004,2010-04-06,-2.6,13.6,0.0\nR004,2010-04-07,-9.7,0.0,46030.99\nR004,2010-04-08,-10.7,0.5,32954.82\nR004,2010-04-09,-10.6,0.2,46574.9\nR004,2010-04-10,-10.5,0.0,46043.62\nR004,2010-04-11,-8.9,0.0,48129.43\nR004,2010-04-12,-6.2,0.0,48561.03\nR004,2010-04-13,-1.5,0.0,40966.18\nR004,2010-04-14,0.5,2.6,13783.54\nR004,2010-04-15,2.8,0.7,34860.65\nR004,2010-04-16,0.8,11.6,14400.0\nR004,2010-04-17,1.0,2.8,46322.52\nR004,2010-04-18,1.0,0.4,49620.44\nR004,2010-04-19,0.2,4.0,29109.85\nR004,2010-04-20,0.5,7.9,0.0\nR004,2010-04-21,5.0,0.0,37210.01\nR004,2010-04-22,3.9,2.3,37588.76\nR004,2010-04-23,0.3,0.2,49724.73\nR004,2010-04-24,0.3,0.0,42177.14\nR004,2010-04-25,5.2,0.0,47937.31\nR004,2010-04-26,7.9,0.0,50400.0\nR004,2010-04-27,6.8,0.0,50400.0\nR004,2010-04-28,10.0,0.7,44362.17\nR004,2010-04-29,9.1,0.1,50400.0\nR004,2010-04-30,5.9,4.4,39411.74\nR004,2010-05-01,8.0,0.2,50400.0\nR004,2010-05-02,6.4,4.6,33036.02\nR004,2010-05-03,6.5,1.9,49728.7\nR004,2010-05-04,3.7,19.4,0.0\nR004,2010-05-05,2.0,6.7,13620.56\nR004,2010-05-06,2.4,6.6,35956.83\nR004,2010-05-07,5.6,0.7,48824.15\nR004,2010-05-08,9.3,0.2,50444.4\nR004,2010-05-09,6.6,3.3,33984.6\nR004,2010-05-10,3.8,0.0,37099.58\nR004,2010-05-11,1.9,1.1,50692.12\nR004,2010-05-12,0.7,0.3,44702.81\nR004,2010-05-13,5.2,0.0,42943.79\nR004,2010-05-14,5.0,6.5,14986.45\nR004,2010-05-15,8.5,1.0,52783.28\nR004,2010-05-16,10.8,0.0,45354.89\nR004,2010-05-17,9.9,17.3,20124.22\nR004,2010-05-18,4.8,1.5,42859.91\nR004,2010-05-19,0.5,6.6,11058.88\nR004,2010-05-20,1.4,0.4,41903.38\nR004,2010-05-21,2.4,3.5,22236.28\nR004,2010-05-22,3.4,0.2,55619.38\nR004,2010-05-23,5.9,2.6,37733.39\nR004,2010-05-24,7.7,5.9,6694.75\nR004,2010-05-25,11.0,1.2,55466.88\nR004,2010-05-26,14.1,0.0,57259.96\nR004,2010-05-27,16.0,0.0,57544.62\nR004,2010-05-28,15.8,2.4,53477.19\nR004,2010-05-29,14.8,8.6,53743.23\nR004,2010-05-30,16.9,0.9,50167.81\nR004,2010-05-31,13.6,11.1,4268.34\nR004,2010-06-01,10.5,2.6,37540.98\nR004,2010-06-02,10.9,0.0,56287.32\nR004,2010-06-03,12.5,1.7,50400.0\nR004,2010-06-04,14.0,1.6,48423.24\nR004,2010-06-05,13.2,12.9,37632.89\nR004,2010-06-06,12.0,0.0,57600.0\nR004,2010-06-07,14.7,0.0,57600.0\nR004,2010-06-08,18.2,0.0,57600.0\nR004,2010-06-09,19.8,0.0,54721.42\nR004,2010-06-10,16.2,7.2,54000.0\nR004,2010-06-11,17.6,0.0,57600.0\nR004,2010-06-12,16.4,6.7,47721.29\nR004,2010-06-13,12.3,5.6,49082.14\nR004,2010-06-14,14.5,0.0,37424.17\nR004,2010-06-15,20.7,0.0,57600.0\nR004,2010-06-16,20.6,4.4,53066.09\nR004,2010-06-17,17.3,0.0,57600.0\nR004,2010-06-18,17.0,0.0,57600.0\nR004,2010-06-19,19.2,0.0,57600.0\nR004,2010-06-20,21.8,0.0,53267.49\nR004,2010-06-21,19.6,1.7,36592.11\nR004,2010-06-22,14.7,6.7,2564.51\nR004,2010-06-23,10.9,0.0,50400.0\nR004,2010-06-24,10.5,2.1,54068.25\nR004,2010-06-25,11.8,0.0,43382.53\nR004,2010-06-26,12.9,4.1,25128.18\nR004,2010-06-27,14.6,0.7,50139.25\nR004,2010-06-28,11.7,5.8,13449.05\nR004,2010-06-29,13.0,0.4,35797.29\nR004,2010-06-30,11.1,48.0,0.0\nR004,2010-07-01,13.5,14.7,15159.09\nR004,2010-07-02,16.5,0.7,51110.13\nR004,2010-07-03,14.5,12.6,42014.12\nR004,2010-07-04,14.2,0.0,56358.32\nR004,2010-07-05,17.3,0.0,56027.63\nR004,2010-07-06,18.1,1.1,57600.0\nR004,2010-07-07,15.8,15.8,35392.15\nR004,2010-07-08,14.0,17.0,48001.33\nR004,2010-07-09,13.7,1.0,45708.5\nR004,2010-07-10,14.5,3.2,44407.58\nR004,2010-07-11,13.5,8.2,22931.44\nR004,2010-07-12,13.1,1.5,35763.33\nR004,2010-07-13,14.3,2.8,49731.74\nR004,2010-07-14,16.6,0.0,54804.19\nR004,2010-07-15,19.2,0.0,56570.26\nR004,2010-07-16,20.5,0.0,55446.03\nR004,2010-07-17,21.0,0.5,43966.71\nR004,2010-07-18,19.6,19.9,22315.18\nR004,2010-07-19,16.4,23.8,9826.98\nR004,2010-07-20,16.1,0.0,26331.5\nR004,2010-07-21,17.4,2.7,47212.83\nR004,2010-07-22,17.7,2.4,14979.67\nR004,2010-07-23,17.3,6.8,28682.0\nR004,2010-07-24,17.4,0.1,45708.04\nR004,2010-07-25,20.1,0.0,43814.87\nR004,2010-07-26,20.5,5.2,50807.24\nR004,2010-07-27,15.6,11.1,32373.85\nR004,2010-07-28,9.2,11.6,14319.76\nR004,2010-07-29,10.7,0.0,38147.44\nR004,2010-07-30,13.2,0.0,53971.65\nR004,2010-07-31,14.5,0.0,53871.27\nR004,2010-08-01,13.9,2.4,27544.77\nR004,2010-08-02,11.2,15.4,20147.61\nR004,2010-08-03,11.6,0.2,52944.05\nR004,2010-08-04,14.7,0.0,53121.09\nR004,2010-08-05,17.4,0.3,53161.85\nR004,2010-08-06,14.9,0.0,52781.65\nR004,2010-08-07,13.7,0.8,25131.59\nR004,2010-08-08,17.6,9.6,50536.9\nR004,2010-08-09,15.6,20.6,24091.83\nR004,2010-08-10,13.2,0.1,27636.17\nR004,2010-08-11,10.9,3.7,39151.68\nR004,2010-08-12,10.4,3.8,33285.04\nR004,2010-08-13,12.6,0.0,44832.99\nR004,2010-08-14,14.9,0.0,50684.67\nR004,2010-08-15,13.5,8.7,45917.92\nR004,2010-08-16,10.0,1.2,27380.55\nR004,2010-08-17,9.8,0.9,24776.2\nR004,2010-08-18,11.9,0.0,50152.2\nR004,2010-08-19,12.5,2.2,32199.14\nR004,2010-08-20,13.5,0.0,50234.86\nR004,2010-08-21,16.7,0.0,50133.79\nR004,2010-08-22,20.7,0.0,50032.53\nR004,2010-08-23,18.6,2.2,12518.18\nR004,2010-08-24,12.0,0.1,23.61\nR004,2010-08-25,11.8,0.0,46128.02\nR004,2010-08-26,14.9,0.0,38168.27\nR004,2010-08-27,16.6,0.0,48190.46\nR004,2010-08-28,15.9,0.0,49019.88\nR004,2010-08-29,16.7,0.0,48605.57\nR004,2010-08-30,17.8,0.0,45751.21\nR004,2010-08-31,18.3,0.0,45468.16\nR004,2010-09-01,19.6,0.0,47547.93\nR004,2010-09-02,18.7,6.7,38350.61\nR004,2010-09-03,7.0,3.7,29646.81\nR004,2010-09-04,5.7,0.0,31465.09\nR004,2010-09-05,9.3,0.0,45251.6\nR004,2010-09-06,13.1,0.0,44596.53\nR004,2010-09-07,16.1,0.0,43979.87\nR004,2010-09-08,17.3,0.0,44062.32\nR004,2010-09-09,11.3,0.0,43735.11\nR004,2010-09-10,7.5,0.8,32622.44\nR004,2010-09-11,6.6,1.3,21455.6\nR004,2010-09-12,4.1,4.3,32886.5\nR004,2010-09-13,1.7,0.0,34156.48\nR004,2010-09-14,3.7,0.1,42280.52\nR004,2010-09-15,3.9,2.7,10526.93\nR004,2010-09-16,4.1,0.1,42688.44\nR004,2010-09-17,6.2,4.8,7268.36\nR004,2010-09-18,5.5,12.5,30683.5\nR004,2010-09-19,2.6,0.0,43200.0\nR004,2010-09-20,2.8,0.0,43200.0\nR004,2010-09-21,7.3,0.0,43200.0\nR004,2010-09-22,8.6,0.0,43200.0\nR004,2010-09-23,8.3,0.8,43157.49\nR004,2010-09-24,5.6,1.4,39824.3\nR004,2010-09-25,5.3,0.0,36217.69\nR004,2010-09-26,6.3,0.0,30785.18\nR004,2010-09-27,10.8,0.0,39824.87\nR004,2010-09-28,8.8,8.4,12021.63\nR004,2010-09-29,7.4,1.6,15516.47\nR004,2010-09-30,6.8,2.0,2008.36\nR004,2010-10-01,10.5,0.0,37652.81\nR004,2010-10-02,11.7,0.0,36994.61\nR004,2010-10-03,11.9,0.3,3604.56\nR004,2010-10-04,7.2,16.5,0.0\nR004,2010-10-05,1.2,0.9,19784.24\nR004,2010-10-06,0.8,0.0,22010.25\nR004,2010-10-07,-0.2,2.5,24417.07\nR004,2010-10-08,-0.0,8.7,4567.03\nR004,2010-10-09,0.9,15.7,0.0\nR004,2010-10-10,2.5,8.9,238.54\nR004,2010-10-11,4.0,4.2,2597.5\nR004,2010-10-12,4.1,0.1,28862.66\nR004,2010-10-13,1.8,0.0,36000.0\nR004,2010-10-14,3.4,0.0,32400.0\nR004,2010-10-15,6.3,0.0,30084.08\nR004,2010-10-16,2.9,3.0,8304.11\nR004,2010-10-17,1.2,1.9,22743.85\nR004,2010-10-18,0.4,0.7,0.0\nR004,2010-10-19,1.8,14.5,10451.32\nR004,2010-10-20,-1.7,0.6,14870.79\nR004,2010-10-21,-2.3,0.1,22494.01\nR004,2010-10-22,-1.7,0.2,14502.4\nR004,2010-10-23,-0.3,1.1,29568.56\nR004,2010-10-24,-0.3,0.0,30538.17\nR004,2010-10-25,0.6,0.0,32403.87\nR004,2010-10-26,-0.4,0.0,33180.51\nR004,2010-10-27,-0.8,0.0,32736.04\nR004,2010-10-28,0.8,0.0,32191.15\nR004,2010-10-29,2.6,0.0,31844.76\nR004,2010-10-30,2.0,0.0,27962.29\nR004,2010-10-31,0.8,0.0,25940.35\nR004,2010-11-01,2.9,0.0,25260.14\nR004,2010-11-02,4.4,0.0,2604.98\nR004,2010-11-03,4.2,0.0,24617.31\nR004,2010-11-04,1.4,4.2,13849.77\nR004,2010-11-05,-0.1,1.2,3250.07\nR004,2010-11-06,-0.8,0.0,28929.74\nR004,2010-11-07,-1.4,0.3,24247.24\nR004,2010-11-08,1.5,6.0,14309.66\nR004,2010-11-09,-1.0,16.4,5542.65\nR004,2010-11-10,-2.8,13.7,0.0\nR004,2010-11-11,-4.8,8.1,7291.2\nR004,2010-11-12,-13.4,0.0,14406.65\nR004,2010-11-13,-2.1,0.4,20064.0\nR004,2010-11-14,2.0,0.2,214.76\nR004,2010-11-15,3.7,0.0,13201.29\nR004,2010-11-16,1.6,13.9,2447.62\nR004,2010-11-17,-1.1,8.7,3144.39\nR004,2010-11-18,-1.1,31.7,3600.0\nR004,2010-11-19,-8.9,2.0,6876.33\nR004,2010-11-20,-6.1,18.9,0.0\nR004,2010-11-21,-6.3,3.1,24096.31\nR004,2010-11-22,-14.4,1.6,14952.22\nR004,2010-11-23,-23.7,0.0,19493.06\nR004,2010-11-24,-10.2,1.1,7981.77\nR004,2010-11-25,-8.5,7.7,18914.26\nR004,2010-11-26,-10.8,10.1,0.0\nR004,2010-11-27,-3.2,16.8,0.0\nR004,2010-11-28,-2.1,11.7,11173.3\nR004,2010-11-29,-2.1,19.2,0.0\nR004,2010-11-30,-4.6,14.5,2158.09\nR004,2010-12-01,-17.3,2.3,14044.1\nR004,2010-12-02,-9.0,13.7,0.0\nR004,2010-12-03,-12.7,1.6,13656.54\nR004,2010-12-04,-18.1,1.7,2383.16\nR004,2010-12-05,-12.3,4.0,264.94\nR004,2010-12-06,-8.4,13.5,0.0\nR004,2010-12-07,-4.5,2.1,0.0\nR004,2010-12-08,-4.9,4.8,0.0\nR004,2010-12-09,-15.6,1.6,22594.88\nR004,2010-12-10,-22.3,3.2,5331.88\nR004,2010-12-11,-17.9,1.6,15584.7\nR004,2010-12-12,-25.2,0.0,23223.44\nR004,2010-12-13,-31.1,0.0,26179.11\nR004,2010-12-14,-31.3,0.0,26209.93\nR004,2010-12-15,-25.6,0.0,20096.92\nR004,2010-12-16,-18.0,0.0,24449.75\nR004,2010-12-17,-9.9,6.5,0.0\nR004,2010-12-18,-6.5,20.2,0.0\nR004,2010-12-19,-12.2,1.4,8510.29\nR004,2010-12-20,-23.0,0.0,23458.66\nR004,2010-12-21,-31.2,0.0,25427.85\nR004,2010-12-22,-35.6,0.0,25473.24\nR004,2010-12-23,-36.9,0.0,25519.99\nR004,2010-12-24,-28.3,3.5,0.0\nR004,2010-12-25,-17.0,4.5,3461.06\nR004,2010-12-26,-23.1,0.0,24523.48\nR004,2010-12-27,-21.3,2.5,2418.35\nR004,2010-12-28,-23.3,0.4,0.0\nR004,2010-12-29,-26.6,0.0,25299.95\nR004,2010-12-30,-22.3,2.8,492.46\nR004,2010-12-31,-20.7,0.7,5108.88\nR004,2011-01-01,-23.1,0.9,0.0\nR004,2011-01-02,-32.1,0.0,24912.01\nR004,2011-01-03,-32.4,0.0,20113.44\nR004,2011-01-04,-36.1,0.0,25633.18\nR004,2011-01-05,-36.4,0.0,26206.08\nR004,2011-01-06,-36.4,0.0,26259.22\nR004,2011-01-07,-35.8,0.0,26238.4\nR004,2011-01-08,-33.7,0.0,26215.61\nR004,2011-01-09,-29.8,0.0,26696.81\nR004,2011-01-10,-27.0,0.0,26164.43\nR004,2011-01-11,-23.8,1.4,6075.42\nR004,2011-01-12,-20.0,2.1,4630.53\nR004,2011-01-13,-23.0,0.0,24500.32\nR004,2011-01-14,-14.0,1.3,0.0\nR004,2011-01-15,-15.0,0.0,26006.11\nR004,2011-01-16,-17.3,0.0,27611.12\nR004,2011-01-17,-18.4,0.0,27657.78\nR004,2011-01-18,-23.8,0.0,27719.99\nR004,2011-01-19,-27.6,0.0,27784.91\nR004,2011-01-20,-28.5,0.0,27852.51\nR004,2011-01-21,-27.6,0.0,28140.9\nR004,2011-01-22,-30.8,0.0,28627.47\nR004,2011-01-23,-31.6,0.0,28644.73\nR004,2011-01-24,-31.4,0.0,28664.26\nR004,2011-01-25,-29.8,0.0,28685.92\nR004,2011-01-26,-27.9,0.0,26837.51\nR004,2011-01-27,-25.7,0.0,15090.6\nR004,2011-01-28,-22.0,0.0,30598.41\nR004,2011-01-29,-21.7,0.0,30939.51\nR004,2011-01-30,-19.5,0.7,22057.96\nR004,2011-01-31,-9.7,1.5,4149.69\nR004,2011-02-01,-12.0,0.0,31497.29\nR004,2011-02-02,-10.7,0.9,5502.27\nR004,2011-02-03,-14.3,3.3,15063.21\nR004,2011-02-04,-8.2,8.1,5148.62\nR004,2011-02-05,-12.8,0.2,18078.98\nR004,2011-02-06,-9.1,3.7,0.0\nR004,2011-02-07,-13.1,0.0,32206.57\nR004,2011-02-08,-18.1,0.0,32323.81\nR004,2011-02-09,-16.5,0.0,28800.0\nR004,2011-02-10,-10.7,1.2,11654.06\nR004,2011-02-11,-16.6,0.0,32400.0\nR004,2011-02-12,-15.0,0.0,31594.52\nR004,2011-02-13,-11.1,0.1,1640.85\nR004,2011-02-14,-11.7,0.4,28302.01\nR004,2011-02-15,-15.6,0.0,30176.2\nR004,2011-02-16,-12.2,0.0,26945.86\nR004,2011-02-17,-10.7,0.0,33519.09\nR004,2011-02-18,-9.5,3.9,2403.13\nR004,2011-02-19,-9.7,3.2,0.0\nR004,2011-02-20,-12.6,1.8,4904.38\nR004,2011-02-21,-18.1,0.4,31839.28\nR004,2011-02-22,-21.4,0.0,33587.13\nR004,2011-02-23,-23.7,0.0,33761.62\nR004,2011-02-24,-20.7,0.0,35040.06\nR004,2011-02-25,-22.6,0.0,35997.43\nR004,2011-02-26,-22.4,0.0,34147.93\nR004,2011-02-27,-18.8,0.0,26006.38\nR004,2011-02-28,-17.7,0.0,37382.77\nR004,2011-03-01,-15.0,0.0,38243.6\nR004,2011-03-02,-13.8,0.0,36000.0\nR004,2011-03-03,-13.7,0.0,37063.98\nR004,2011-03-04,-13.2,0.0,38708.68\nR004,2011-03-05,-13.6,0.0,38820.73\nR004,2011-03-06,-12.3,0.0,38932.66\nR004,2011-03-07,-9.3,0.0,38383.65\nR004,2011-03-08,-4.3,0.3,13861.67\nR004,2011-03-09,-3.7,0.0,36165.9\nR004,2011-03-10,-4.7,4.8,19957.19\nR004,2011-03-11,-11.4,0.8,33935.79\nR004,2011-03-12,-12.8,0.0,35220.9\nR004,2011-03-13,-10.3,0.0,39980.96\nR004,2011-03-14,-9.5,0.0,40291.43\nR004,2011-03-15,-4.9,6.9,0.0\nR004,2011-03-16,-5.5,8.2,6587.7\nR004,2011-03-17,-7.1,9.1,0.0\nR004,2011-03-18,-10.2,3.5,32385.66\nR004,2011-03-19,-15.9,0.0,41838.79\nR004,2011-03-20,-13.3,1.2,17789.12\nR004,2011-03-21,-8.6,0.0,38404.63\nR004,2011-03-22,-8.1,0.0,42427.8\nR004,2011-03-23,-5.9,0.0,42729.62\nR004,2011-03-24,-5.3,0.0,42817.72\nR004,2011-03-25,-3.1,0.0,42852.46\nR004,2011-03-26,0.7,1.1,483.23\nR004,2011-03-27,-1.4,2.5,29510.29\nR004,2011-03-28,-1.4,0.0,43857.46\nR004,2011-03-29,-4.0,5.9,14102.62\nR004,2011-03-30,-7.7,0.0,44516.55\nR004,2011-03-31,-5.6,0.0,44847.3\nR004,2011-04-01,1.0,0.0,45178.98\nR004,2011-04-02,3.3,0.0,44745.34\nR004,2011-04-03,1.2,3.8,19207.13\nR004,2011-04-04,-4.9,3.9,41880.22\nR004,2011-04-05,-4.1,8.9,3286.35\nR004,2011-04-06,-0.4,5.2,40565.37\nR004,2011-04-07,2.0,6.5,7724.05\nR004,2011-04-08,1.1,6.6,28992.35\nR004,2011-04-09,1.6,0.0,46874.92\nR004,2011-04-10,4.4,0.0,38347.41\nR004,2011-04-11,8.3,0.0,44827.33\nR004,2011-04-12,7.7,0.0,48163.27\nR004,2011-04-13,7.1,0.0,48596.6\nR004,2011-04-14,6.0,0.0,48640.88\nR004,2011-04-15,5.9,0.0,49399.53\nR004,2011-04-16,6.9,0.0,49534.4\nR004,2011-04-17,6.1,0.8,13441.21\nR004,2011-04-18,6.1,0.0,40414.95\nR004,2011-04-19,8.1,0.0,49936.74\nR004,2011-04-20,7.1,1.1,37640.37\nR004,2011-04-21,10.3,0.0,50201.92\nR004,2011-04-22,13.9,0.0,47819.61\nR004,2011-04-23,8.8,2.2,28253.88\nR004,2011-04-24,3.7,0.1,39328.47\nR004,2011-04-25,4.4,0.0,50400.0\nR004,2011-04-26,5.3,3.3,16176.02\nR004,2011-04-27,8.6,5.4,45989.34\nR004,2011-04-28,1.7,11.0,34917.05\nR004,2011-04-29,2.8,4.1,36646.67\nR004,2011-04-30,6.0,0.0,50412.64\nR004,2011-05-01,7.6,0.0,49112.2\nR004,2011-05-02,7.6,0.2,41839.96\nR004,2011-05-03,3.9,4.3,47788.89\nR004,2011-05-04,4.0,0.0,51060.19\nR004,2011-05-05,4.5,1.6,50593.08\nR004,2011-05-06,1.6,2.2,50969.69\nR004,2011-05-07,3.7,2.8,50400.0\nR004,2011-05-08,2.3,6.5,14879.98\nR004,2011-05-09,-1.3,2.6,50396.32\nR004,2011-05-10,2.4,0.0,52344.39\nR004,2011-05-11,8.3,0.0,52688.05\nR004,2011-05-12,12.4,0.0,52604.98\nR004,2011-05-13,14.5,0.0,46548.37\nR004,2011-05-14,13.2,0.1,51570.61\nR004,2011-05-15,12.0,21.9,33100.41\nR004,2011-05-16,6.8,7.2,51158.47\nR004,2011-05-17,3.6,0.0,44074.33\nR004,2011-05-18,4.7,0.0,50599.92\nR004,2011-05-19,8.7,0.0,56444.38\nR004,2011-05-20,13.0,0.0,55106.1\nR004,2011-05-21,13.9,0.0,36525.29\nR004,2011-05-22,13.3,2.9,41876.52\nR004,2011-05-23,10.1,10.7,29507.61\nR004,2011-05-24,11.9,0.1,55952.39\nR004,2011-05-25,13.4,0.0,56183.63\nR004,2011-05-26,15.6,4.1,43712.51\nR004,2011-05-27,9.1,6.1,35362.23\nR004,2011-05-28,10.2,0.9,40716.42\nR004,2011-05-29,13.2,3.2,46984.4\nR004,2011-05-30,17.3,0.0,57600.0\nR004,2011-05-31,18.1,1.3,47876.81\nR004,2011-06-01,14.5,0.5,37599.7\nR004,2011-06-02,14.8,0.5,53283.21\nR004,2011-06-03,16.9,0.0,57600.0\nR004,2011-06-04,17.0,7.9,43993.72\nR004,2011-06-05,16.4,5.6,44022.73\nR004,2011-06-06,14.5,10.9,36801.93\nR004,2011-06-07,15.3,7.3,52216.27\nR004,2011-06-08,16.8,1.2,45772.96\nR004,2011-06-09,17.8,1.7,52457.05\nR004,2011-06-10,17.9,3.6,53438.3\nR004,2011-06-11,17.9,11.8,45569.81\nR004,2011-06-12,17.7,8.6,41627.41\nR004,2011-06-13,18.7,8.8,46385.09\nR004,2011-06-14,20.3,2.5,54762.33\nR004,2011-06-15,18.1,0.6,44189.14\nR004,2011-06-16,17.3,1.0,46800.0\nR004,2011-06-17,17.1,0.0,57600.0\nR004,2011-06-18,18.0,0.0,57600.0\nR004,2011-06-19,20.1,0.0,55025.46\nR004,2011-06-20,18.8,1.3,32212.61\nR004,2011-06-21,16.5,5.4,51869.27\nR004,2011-06-22,15.1,3.2,45909.77\nR004,2011-06-23,13.3,14.4,41947.25\nR004,2011-06-24,13.7,16.4,26711.55\nR004,2011-06-25,14.4,20.9,42893.37\nR004,2011-06-26,13.3,0.0,52805.34\nR004,2011-06-27,15.4,0.0,54316.37\nR004,2011-06-28,16.0,0.0,53576.45\nR004,2011-06-29,17.8,0.0,57600.0\nR004,2011-06-30,20.0,0.0,57600.0\nR004,2011-07-01,19.0,0.1,45160.1\nR004,2011-07-02,15.2,17.7,6189.54\nR004,2011-07-03,11.1,13.8,31122.82\nR004,2011-07-04,9.8,7.6,38994.36\nR004,2011-07-05,12.2,0.0,44923.4\nR004,2011-07-06,16.5,0.8,55231.51\nR004,2011-07-07,14.3,12.6,34483.35\nR004,2011-07-08,12.6,1.0,38865.65\nR004,2011-07-09,11.6,5.5,39932.7\nR004,2011-07-10,13.6,2.4,33320.72\nR004,2011-07-11,14.4,2.1,36061.0\nR004,2011-07-12,21.1,0.1,38840.24\nR004,2011-07-13,23.3,0.9,40027.11\nR004,2011-07-14,16.9,5.8,30008.08\nR004,2011-07-15,16.4,0.0,21780.91\nR004,2011-07-16,14.8,9.7,48199.39\nR004,2011-07-17,16.8,0.0,52044.67\nR004,2011-07-18,18.3,0.8,55729.25\nR004,2011-07-19,18.3,0.0,55302.27\nR004,2011-07-20,18.6,0.0,53543.33\nR004,2011-07-21,15.3,7.8,13205.34\nR004,2011-07-22,13.5,0.1,2640.57\nR004,2011-07-23,13.8,0.0,51092.73\nR004,2011-07-24,14.4,1.3,43342.43\nR004,2011-07-25,16.8,0.2,43376.11\nR004,2011-07-26,14.2,7.0,49774.57\nR004,2011-07-27,11.5,5.5,28844.16\nR004,2011-07-28,13.9,10.0,18112.33\nR004,2011-07-29,11.7,0.3,42442.31\nR004,2011-07-30,13.1,0.0,30458.2\nR004,2011-07-31,16.3,0.0,53889.65\nR004,2011-08-01,19.9,0.6,46822.54\nR004,2011-08-02,20.7,2.7,53034.75\nR004,2011-08-03,18.7,6.0,43915.69\nR004,2011-08-04,15.7,11.5,27823.11\nR004,2011-08-05,13.0,6.3,41531.6\nR004,2011-08-06,11.5,5.7,51077.1\nR004,2011-08-07,13.2,0.0,44757.94\nR004,2011-08-08,14.4,2.0,30761.72\nR004,2011-08-09,12.8,29.8,2789.97\nR004,2011-08-10,12.9,10.9,29442.74\nR004,2011-08-11,13.4,0.7,43973.88\nR004,2011-08-12,14.7,1.8,36288.71\nR004,2011-08-13,13.7,0.0,50504.03\nR004,2011-08-14,14.8,0.0,47891.67\nR004,2011-08-15,16.2,0.0,50400.0\nR004,2011-08-16,17.4,0.0,49763.44\nR004,2011-08-17,18.0,0.0,48216.39\nR004,2011-08-18,18.9,1.5,48113.78\nR004,2011-08-19,15.2,0.9,34391.65\nR004,2011-08-20,15.7,7.9,30988.07\nR004,2011-08-21,11.0,0.9,35581.59\nR004,2011-08-22,11.4,0.7,42913.5\nR004,2011-08-23,12.1,0.6,49952.76\nR004,2011-08-24,15.6,0.0,46101.87\nR004,2011-08-25,16.9,7.9,39588.99\nR004,2011-08-26,11.4,7.2,19109.37\nR004,2011-08-27,8.5,1.7,46800.0\nR004,2011-08-28,8.7,0.8,48224.4\nR004,2011-08-29,5.9,8.9,17586.7\nR004,2011-08-30,10.1,0.6,31728.7\nR004,2011-08-31,11.6,15.1,4054.84\nR004,2011-09-01,13.3,16.3,38493.77\nR004,2011-09-02,12.0,5.2,38535.55\nR004,2011-09-03,9.7,0.0,46226.54\nR004,2011-09-04,10.7,8.2,37226.93\nR004,2011-09-05,8.1,0.0,45312.0\nR004,2011-09-06,6.9,0.0,45340.83\nR004,2011-09-07,8.5,0.0,44735.67\nR004,2011-09-08,11.1,0.0,44421.65\nR004,2011-09-09,15.2,0.3,43722.11\nR004,2011-09-10,13.2,0.9,16806.1\nR004,2011-09-11,8.1,2.0,25670.07\nR004,2011-09-12,5.2,4.1,39847.63\nR004,2011-09-13,5.6,0.3,43200.0\nR004,2011-09-14,5.9,0.0,43200.0\nR004,2011-09-15,7.6,0.0,43200.0\nR004,2011-09-16,9.1,0.0,43200.0\nR004,2011-09-17,9.4,0.0,43200.0\nR004,2011-09-18,10.2,0.0,43200.0\nR004,2011-09-19,11.2,0.0,43200.0\nR004,2011-09-20,11.0,0.0,43200.0\nR004,2011-09-21,12.7,0.0,43200.0\nR004,2011-09-22,10.3,5.1,16848.31\nR004,2011-09-23,8.9,0.9,42349.78\nR004,2011-09-24,9.8,1.6,37445.65\nR004,2011-09-25,5.4,1.7,41569.0\nR004,2011-09-26,4.6,0.0,23425.68\nR004,2011-09-27,5.7,0.0,30098.38\nR004,2011-09-28,5.5,0.0,41464.01\nR004,2011-09-29,5.1,0.0,40058.7\nR004,2011-09-30,4.7,2.0,12302.27\nR004,2011-10-01,4.8,0.0,33792.79\nR004,2011-10-02,7.3,0.0,36984.12\nR004,2011-10-03,8.9,0.0,29735.75\nR004,2011-10-04,10.9,0.0,20888.04\nR004,2011-10-05,13.2,1.5,32400.0\nR004,2011-10-06,8.3,4.7,29952.61\nR004,2011-10-07,6.0,4.3,28914.86\nR004,2011-10-08,6.0,0.0,28175.13\nR004,2011-10-09,5.8,5.4,20819.39\nR004,2011-10-10,4.3,1.0,1110.17\nR004,2011-10-11,4.2,0.0,31493.33\nR004,2011-10-12,6.5,0.6,6465.74\nR004,2011-10-13,8.4,0.0,36000.0\nR004,2011-10-14,7.3,0.0,36000.0\nR004,2011-10-15,6.6,0.0,35975.37\nR004,2011-10-16,7.2,0.0,35319.69\nR004,2011-10-17,8.1,0.2,23105.07\nR004,2011-10-18,5.6,4.9,21471.4\nR004,2011-10-19,-0.1,0.5,18771.72\nR004,2011-10-20,-2.0,0.8,555.59\nR004,2011-10-21,0.4,0.1,3892.1\nR004,2011-10-22,-0.6,0.0,34117.33\nR004,2011-10-23,-1.0,0.0,34192.68\nR004,2011-10-24,1.5,0.0,33853.0\nR004,2011-10-25,2.1,0.6,24247.79\nR004,2011-10-26,5.3,6.7,15410.53\nR004,2011-10-27,3.4,10.0,321.91\nR004,2011-10-28,0.6,0.7,27128.61\nR004,2011-10-29,-1.5,5.5,9328.61\nR004,2011-10-30,-4.1,3.0,2057.68\nR004,2011-10-31,-6.5,0.6,24803.59\nR004,2011-11-01,-3.7,5.1,0.0\nR004,2011-11-02,-4.6,2.7,17698.31\nR004,2011-11-03,-3.3,0.5,6920.09\nR004,2011-11-04,-4.0,0.0,30096.75\nR004,2011-11-05,-3.1,0.0,28463.15\nR004,2011-11-06,-2.7,0.0,29794.21\nR004,2011-11-07,-1.9,2.1,0.0\nR004,2011-11-08,-2.2,4.5,0.0\nR004,2011-11-09,-9.9,6.4,28800.0\nR004,2011-11-10,-12.1,1.7,19085.42\nR004,2011-11-11,-13.7,2.2,24082.84\nR004,2011-11-12,-11.1,0.7,0.0\nR004,2011-11-13,-4.9,2.4,13161.04\nR004,2011-11-14,-10.3,5.5,21068.14\nR004,2011-11-15,-16.3,1.7,11502.71\nR004,2011-11-16,-9.6,0.1,13031.92\nR004,2011-11-17,-19.2,0.0,28431.1\nR004,2011-11-18,-16.3,0.0,22280.01\nR004,2011-11-19,-7.6,1.4,24069.0\nR004,2011-11-20,-7.4,6.6,746.72\nR004,2011-11-21,-18.7,0.0,28751.84\nR004,2011-11-22,-13.2,4.5,11321.71\nR004,2011-11-23,-10.4,2.2,28606.21\nR004,2011-11-24,-15.2,0.0,22546.11\nR004,2011-11-25,-15.0,0.0,23319.84\nR004,2011-11-26,-12.2,2.6,13505.33\nR004,2011-11-27,-17.9,0.0,15617.15\nR004,2011-11-28,-20.6,0.0,24549.78\nR004,2011-11-29,-15.4,0.5,6828.78\nR004,2011-11-30,-13.1,0.0,20526.34\nR004,2011-12-01,-19.8,0.0,27858.69\nR004,2011-12-02,-14.3,0.3,0.0\nR004,2011-12-03,-11.4,0.0,952.64\nR004,2011-12-04,-12.5,0.9,1290.33\nR004,2011-12-05,-11.0,4.3,190.33\nR004,2011-12-06,-10.7,4.1,4603.47\nR004,2011-12-07,-7.9,3.1,0.0\nR004,2011-12-08,-6.9,0.0,20992.98\nR004,2011-12-09,-15.8,0.0,23944.55\nR004,2011-12-10,-17.8,0.0,26086.08\nR004,2011-12-11,-19.4,0.0,26636.24\nR004,2011-12-12,-21.5,0.0,25866.53\nR004,2011-12-13,-20.2,0.0,26172.77\nR004,2011-12-14,-22.6,0.0,21600.0\nR004,2011-12-15,-19.0,0.9,0.0\nR004,2011-12-16,-15.8,0.3,24969.86\nR004,2011-12-17,-22.3,0.0,22345.66\nR004,2011-12-18,-26.0,0.0,25299.58\nR004,2011-12-19,-25.1,0.9,16412.98\nR004,2011-12-20,-13.8,8.8,0.0\nR004,2011-12-21,-13.8,0.0,0.0\nR004,2011-12-22,-12.5,2.5,0.0\nR004,2011-12-23,-10.4,2.3,0.0\nR004,2011-12-24,-14.6,0.0,23038.51\nR004,2011-12-25,-20.8,0.0,25078.4\nR004,2011-12-26,-21.8,0.0,25125.79\nR004,2011-12-27,-16.0,0.0,9672.81\nR004,2011-12-28,-13.1,2.2,15297.62\nR004,2011-12-29,-19.0,0.0,24753.89\nR004,2011-12-30,-14.0,3.6,0.0\nR004,2011-12-31,-7.8,5.0,0.0\nR004,2012-01-01,-10.3,1.2,4970.87\nR004,2012-01-02,-19.5,0.0,25500.82\nR004,2012-01-03,-21.0,0.0,25560.69\nR004,2012-01-04,-21.5,0.0,26136.03\nR004,2012-01-05,-23.1,0.0,25160.81\nR004,2012-01-06,-14.2,1.6,0.0\nR004,2012-01-07,-9.0,5.0,0.0\nR004,2012-01-08,-9.4,0.6,0.0\nR004,2012-01-09,-8.7,1.9,1260.26\nR004,2012-01-10,-11.2,0.0,22847.71\nR004,2012-01-11,-18.0,0.0,25641.29\nR004,2012-01-12,-20.8,0.0,26615.53\nR004,2012-01-13,-22.2,0.0,27081.77\nR004,2012-01-14,-19.2,0.0,17578.62\nR004,2012-01-15,-14.5,0.6,5535.07\nR004,2012-01-16,-14.1,4.8,0.0\nR004,2012-01-17,-20.3,1.2,16334.28\nR004,2012-01-18,-29.1,0.0,16406.57\nR004,2012-01-19,-33.7,0.0,26837.89\nR004,2012-01-20,-35.7,0.0,28126.6\nR004,2012-01-21,-33.6,0.0,28141.43\nR004,2012-01-22,-27.8,0.0,24428.85\nR004,2012-01-23,-23.8,0.2,18037.86\nR004,2012-01-24,-18.9,2.3,1497.37\nR004,2012-01-25,-23.6,0.0,25200.0\nR004,2012-01-26,-26.2,0.0,19744.18\nR004,2012-01-27,-26.5,0.0,19413.53\nR004,2012-01-28,-27.3,0.0,6226.86\nR004,2012-01-29,-24.9,0.0,17750.2\nR004,2012-01-30,-29.0,0.0,31112.79\nR004,2012-01-31,-32.2,0.0,31325.0\nR004,2012-02-01,-32.4,0.0,31473.55\nR004,2012-02-02,-30.4,0.0,31589.93\nR004,2012-02-03,-27.9,0.0,31706.02\nR004,2012-02-04,-26.4,0.0,29966.43\nR004,2012-02-05,-26.2,0.0,31936.97\nR004,2012-02-06,-25.9,0.0,32052.87\nR004,2012-02-07,-26.9,0.0,32169.51\nR004,2012-02-08,-24.8,0.0,32286.77\nR004,2012-02-09,-23.2,0.0,32400.0\nR004,2012-02-10,-23.0,0.0,32400.0\nR004,2012-02-11,-24.2,0.0,26926.45\nR004,2012-02-12,-25.1,0.0,32400.0\nR004,2012-02-13,-18.7,1.0,138.79\nR004,2012-02-14,-15.4,0.0,32400.0\nR004,2012-02-15,-17.8,0.0,32838.38\nR004,2012-02-16,-17.8,0.0,33186.31\nR004,2012-02-17,-17.4,0.0,33531.11\nR004,2012-02-18,-16.0,0.0,33872.81\nR004,2012-02-19,-11.3,0.0,32400.0\nR004,2012-02-20,-13.5,0.0,27951.3\nR004,2012-02-21,-15.3,0.0,34160.65\nR004,2012-02-22,-13.9,1.0,25248.62\nR004,2012-02-23,-16.0,0.0,33481.8\nR004,2012-02-24,-17.5,0.0,36284.71\nR004,2012-02-25,-18.0,0.0,36751.72\nR004,2012-02-26,-17.9,0.0,37217.86\nR004,2012-02-27,-18.0,0.0,34954.74\nR004,2012-02-28,-16.2,0.0,37920.74\nR004,2012-02-29,-16.0,0.0,38169.78\nR004,2012-03-01,-16.0,0.0,38420.04\nR004,2012-03-02,-14.9,0.0,38567.5\nR004,2012-03-03,-12.6,0.0,38173.95\nR004,2012-03-04,-10.0,0.0,36956.8\nR004,2012-03-05,-8.0,0.0,38906.07\nR004,2012-03-06,-12.0,0.0,39018.4\nR004,2012-03-07,-10.8,0.0,39130.42\nR004,2012-03-08,-9.7,0.0,39242.11\nR004,2012-03-09,-10.5,0.0,39353.43\nR004,2012-03-10,-11.1,0.0,39619.99\nR004,2012-03-11,-8.4,0.0,33918.77\nR004,2012-03-12,-4.9,0.0,39600.0\nR004,2012-03-13,-5.1,0.0,33940.97\nR004,2012-03-14,-5.0,3.7,21018.73\nR004,2012-03-15,-8.6,0.0,19200.55\nR004,2012-03-16,-10.2,0.0,41241.0\nR004,2012-03-17,-7.5,1.4,30661.02\nR004,2012-03-18,-1.1,0.0,31927.1\nR004,2012-03-19,-6.0,4.0,20293.45\nR004,2012-03-20,-5.6,1.1,4084.12\nR004,2012-03-21,-2.7,10.8,14601.59\nR004,2012-03-22,-3.0,0.2,42530.87\nR004,2012-03-23,-0.7,0.0,42796.6\nR004,2012-03-24,-0.5,0.0,42887.78\nR004,2012-03-25,2.0,1.7,32788.13\nR004,2012-03-26,3.4,4.3,31972.76\nR004,2012-03-27,-3.2,2.2,36079.08\nR004,2012-03-28,-6.4,0.0,44199.12\nR004,2012-03-29,-4.5,0.0,43764.81\nR004,2012-03-30,-0.2,1.3,1173.04\nR004,2012-03-31,-0.4,0.4,21893.71\nR004,2012-04-01,-1.5,0.0,45447.45\nR004,2012-04-02,-0.0,0.6,20675.79\nR004,2012-04-03,4.4,0.0,38039.02\nR004,2012-04-04,4.2,0.0,45765.49\nR004,2012-04-05,5.8,1.6,29028.54\nR004,2012-04-06,3.0,5.3,45976.64\nR004,2012-04-07,4.7,0.0,44460.77\nR004,2012-04-08,1.4,4.5,23590.41\nR004,2012-04-09,1.5,0.0,46905.81\nR004,2012-04-10,6.9,0.0,47334.14\nR004,2012-04-11,9.5,0.0,46596.47\nR004,2012-04-12,6.0,2.0,40053.42\nR004,2012-04-13,5.3,0.0,35768.9\nR004,2012-04-14,7.5,0.0,49038.47\nR004,2012-04-15,10.0,0.2,34747.0\nR004,2012-04-16,10.8,18.7,36103.02\nR004,2012-04-17,4.0,9.9,12637.76\nR004,2012-04-18,1.5,0.3,35069.26\nR004,2012-04-19,2.4,4.6,10.21\nR004,2012-04-20,9.4,4.0,34209.21\nR004,2012-04-21,5.5,9.0,29280.56\nR004,2012-04-22,-2.4,4.6,44990.4\nR004,2012-04-23,-0.8,0.0,43978.39\nR004,2012-04-24,4.0,0.2,29148.65\nR004,2012-04-25,5.1,6.7,29384.78\nR004,2012-04-26,1.4,0.0,50400.0\nR004,2012-04-27,3.2,0.0,50400.0\nR004,2012-04-28,2.9,0.0,38912.58\nR004,2012-04-29,3.4,0.0,50400.0\nR004,2012-04-30,4.7,2.1,40605.36\nR004,2012-05-01,9.4,18.2,12327.92\nR004,2012-05-02,14.2,2.3,50697.75\nR004,2012-05-03,15.4,0.5,50650.38\nR004,2012-05-04,13.9,2.4,45800.78\nR004,2012-05-05,9.9,6.7,36370.51\nR004,2012-05-06,6.9,1.7,40533.24\nR004,2012-05-07,3.2,5.1,45889.76\nR004,2012-05-08,3.1,4.4,35901.63\nR004,2012-05-09,-0.6,19.0,400.65\nR004,2012-05-10,1.6,0.0,50211.91\nR004,2012-05-11,5.0,0.0,43811.37\nR004,2012-05-12,6.8,0.0,54275.13\nR004,2012-05-13,10.6,0.0,54200.77\nR004,2012-05-14,15.4,0.0,54565.66\nR004,2012-05-15,17.6,0.0,54923.65\nR004,2012-05-16,13.4,7.2,49098.62\nR004,2012-05-17,3.9,0.2,55015.36\nR004,2012-05-18,6.6,0.0,44857.98\nR004,2012-05-19,7.0,9.0,39028.05\nR004,2012-05-20,3.4,2.8,51133.77\nR004,2012-05-21,5.6,0.0,57139.94\nR004,2012-05-22,9.6,0.0,56884.64\nR004,2012-05-23,13.2,0.0,53087.13\nR004,2012-05-24,16.1,1.8,50692.24\nR004,2012-05-25,7.4,13.7,0.0\nR004,2012-05-26,4.0,0.0,52352.5\nR004,2012-05-27,4.5,0.0,36266.05\nR004,2012-05-28,7.8,0.0,57598.5\nR004,2012-05-29,11.1,0.0,57600.0\nR004,2012-05-30,14.0,0.0,57600.0\nR004,2012-05-31,15.8,0.0,57600.0\nR004,2012-06-01,16.6,0.0,57600.0\nR004,2012-06-02,16.4,0.0,57600.0\nR004,2012-06-03,17.5,0.4,57600.0\nR004,2012-06-04,17.4,0.0,57600.0\nR004,2012-06-05,17.9,0.0,57600.0\nR004,2012-06-06,18.7,0.3,55152.73\nR004,2012-06-07,16.4,6.5,36520.22\nR004,2012-06-08,11.6,9.2,4983.03\nR004,2012-06-09,16.5,1.1,50762.81\nR004,2012-06-10,20.7,0.0,57600.0\nR004,2012-06-11,21.9,0.0,52983.88\nR004,2012-06-12,16.5,4.2,28178.41\nR004,2012-06-13,19.8,0.0,57600.0\nR004,2012-06-14,19.8,0.9,42490.0\nR004,2012-06-15,16.3,12.0,17571.01\nR004,2012-06-16,17.7,0.3,52770.51\nR004,2012-06-17,17.9,4.4,42212.09\nR004,2012-06-18,16.9,8.1,21077.64\nR004,2012-06-19,17.7,2.1,49216.65\nR004,2012-06-20,16.7,10.7,30438.89\nR004,2012-06-21,16.5,5.7,50735.04\nR004,2012-06-22,18.7,0.0,57600.0\nR004,2012-06-23,19.7,0.0,57600.0\nR004,2012-06-24,19.2,0.0,57664.61\nR004,2012-06-25,18.0,0.0,57664.59\nR004,2012-06-26,17.6,0.0,57600.0\nR004,2012-06-27,18.6,0.0,57600.0\nR004,2012-06-28,20.8,0.0,57600.0\nR004,2012-06-29,21.3,0.0,48117.94\nR004,2012-06-30,18.8,15.5,21437.64\nR004,2012-07-01,19.3,5.0,43892.23\nR004,2012-07-02,18.2,8.0,36036.04\nR004,2012-07-03,17.5,1.4,43368.21\nR004,2012-07-04,16.7,15.3,27196.56\nR004,2012-07-05,15.7,10.4,1336.1\nR004,2012-07-06,17.5,3.4,44127.71\nR004,2012-07-07,18.5,0.3,43718.2\nR004,2012-07-08,17.2,13.7,9191.18\nR004,2012-07-09,16.0,10.4,33352.33\nR004,2012-07-10,15.0,0.0,52575.87\nR004,2012-07-11,16.7,0.0,52444.4\nR004,2012-07-12,16.4,0.0,56975.35\nR004,2012-07-13,17.4,0.0,56997.81\nR004,2012-07-14,18.6,0.0,45309.03\nR004,2012-07-15,19.7,0.0,47442.7\nR004,2012-07-16,18.1,0.1,55230.02\nR004,2012-07-17,17.2,0.0,55718.23\nR004,2012-07-18,17.3,0.0,55747.79\nR004,2012-07-19,18.2,0.0,55778.12\nR004,2012-07-20,19.6,0.0,55355.21\nR004,2012-07-21,21.3,0.0,54935.92\nR004,2012-07-22,21.4,0.0,54971.19\nR004,2012-07-23,21.2,0.0,54557.56\nR004,2012-07-24,20.8,0.0,54000.0\nR004,2012-07-25,20.4,0.0,54634.04\nR004,2012-07-26,19.9,0.0,54228.1\nR004,2012-07-27,21.0,0.0,54269.02\nR004,2012-07-28,22.5,0.0,53822.5\nR004,2012-07-29,20.7,0.2,41712.52\nR004,2012-07-30,19.5,1.4,40379.54\nR004,2012-07-31,19.2,2.5,46389.19\nR004,2012-08-01,21.0,0.0,51829.76\nR004,2012-08-02,20.5,1.7,44743.85\nR004,2012-08-03,16.5,15.0,12434.43\nR004,2012-08-04,15.2,5.5,22913.1\nR004,2012-08-05,16.4,0.3,50238.53\nR004,2012-08-06,15.4,0.6,46800.0\nR004,2012-08-07,15.1,0.0,52425.07\nR004,2012-08-08,17.4,0.0,52053.24\nR004,2012-08-09,19.0,0.0,44716.06\nR004,2012-08-10,16.8,17.4,30654.91\nR004,2012-08-11,16.9,5.2,30782.53\nR004,2012-08-12,18.6,0.0,49114.02\nR004,2012-08-13,22.2,0.0,50659.51\nR004,2012-08-14,21.1,12.7,40862.79\nR004,2012-08-15,14.3,27.2,8512.74\nR004,2012-08-16,13.2,1.4,47323.01\nR004,2012-08-17,10.0,5.0,27061.46\nR004,2012-08-18,8.9,0.6,44053.59\nR004,2012-08-19,10.9,0.0,49910.76\nR004,2012-08-20,12.4,0.0,50178.11\nR004,2012-08-21,15.3,6.1,25856.09\nR004,2012-08-22,15.8,1.3,31186.77\nR004,2012-08-23,13.8,1.2,48830.43\nR004,2012-08-24,14.1,0.0,44918.01\nR004,2012-08-25,13.7,18.4,21812.0\nR004,2012-08-26,13.3,13.6,23668.21\nR004,2012-08-27,13.1,10.9,26160.04\nR004,2012-08-28,13.5,1.2,35072.76\nR004,2012-08-29,12.3,14.7,4022.37\nR004,2012-08-30,11.8,7.8,23433.55\nR004,2012-08-31,11.4,7.7,20185.78\nR004,2012-09-01,12.5,0.0,45771.49\nR004,2012-09-02,14.6,0.0,47018.48\nR004,2012-09-03,17.3,0.0,46246.03\nR004,2012-09-04,14.6,2.6,34349.95\nR004,2012-09-05,11.9,9.5,39206.52\nR004,2012-09-06,9.8,0.0,41018.18\nR004,2012-09-07,9.3,0.0,43252.98\nR004,2012-09-08,8.9,0.0,44090.58\nR004,2012-09-09,9.9,0.0,43776.12\nR004,2012-09-10,11.0,0.0,43462.39\nR004,2012-09-11,12.7,0.0,43200.0\nR004,2012-09-12,14.2,0.0,43219.1\nR004,2012-09-13,13.4,0.0,24024.94\nR004,2012-09-14,14.1,4.7,43200.0\nR004,2012-09-15,9.9,5.8,37356.02\nR004,2012-09-16,6.8,0.4,32937.57\nR004,2012-09-17,6.6,16.4,0.0\nR004,2012-09-18,10.5,4.8,17875.81\nR004,2012-09-19,12.0,2.6,41416.55\nR004,2012-09-20,8.3,18.8,0.0\nR004,2012-09-21,6.3,2.2,32382.65\nR004,2012-09-22,5.6,1.1,37833.6\nR004,2012-09-23,5.3,2.1,32643.05\nR004,2012-09-24,5.6,12.7,3969.4\nR004,2012-09-25,5.5,6.0,16666.97\nR004,2012-09-26,6.7,0.0,41676.36\nR004,2012-09-27,8.8,0.0,40283.45\nR004,2012-09-28,11.9,0.7,32184.85\nR004,2012-09-29,7.9,1.8,13083.88\nR004,2012-09-30,5.9,0.0,36466.93\nR004,2012-10-01,5.6,0.0,37352.78\nR004,2012-10-02,7.7,0.2,29347.62\nR004,2012-10-03,9.8,0.0,36304.22\nR004,2012-10-04,9.0,0.9,34615.26\nR004,2012-10-05,7.1,10.0,14548.06\nR004,2012-10-06,2.9,4.1,15297.24\nR004,2012-10-07,1.9,3.1,21211.76\nR004,2012-10-08,3.0,10.6,157.44\nR004,2012-10-09,7.4,9.2,10290.24\nR004,2012-10-10,4.2,7.0,19465.9\nR004,2012-10-11,-2.1,0.2,31773.86\nR004,2012-10-12,-3.4,0.0,36000.0\nR004,2012-10-13,-3.0,0.0,36000.0\nR004,2012-10-14,-2.7,0.0,36000.0\nR004,2012-10-15,-1.2,0.0,11088.07\nR004,2012-10-16,3.6,0.0,20252.07\nR004,2012-10-17,2.8,0.0,31645.1\nR004,2012-10-18,1.5,0.0,24119.19\nR004,2012-10-19,0.2,1.6,322.35\nR004,2012-10-20,1.3,8.9,0.0\nR004,2012-10-21,3.9,34.7,0.0\nR004,2012-10-22,3.6,24.7,8531.42\nR004,2012-10-23,1.2,16.7,0.0\nR004,2012-10-24,-2.1,13.2,0.0\nR004,2012-10-25,-6.5,1.1,8599.99\nR004,2012-10-26,-7.9,0.0,27476.02\nR004,2012-10-27,-7.2,0.0,31342.8\nR004,2012-10-28,-5.8,0.0,32244.94\nR004,2012-10-29,0.1,4.6,6197.59\nR004,2012-10-30,0.7,0.0,30669.86\nR004,2012-10-31,0.5,4.3,23200.77\nR004,2012-11-01,-5.0,2.8,18922.72\nR004,2012-11-02,-6.7,0.0,25304.9\nR004,2012-11-03,-2.2,0.0,21600.0\nR004,2012-11-04,-1.4,1.5,13306.04\nR004,2012-11-05,-2.2,10.6,0.0\nR004,2012-11-06,-4.5,6.2,0.0\nR004,2012-11-07,-7.4,8.1,0.0\nR004,2012-11-08,-15.7,0.1,19825.39\nR004,2012-11-09,-21.5,0.0,29152.22\nR004,2012-11-10,-13.3,0.0,4765.37\nR004,2012-11-11,-7.2,2.8,12505.72\nR004,2012-11-12,-7.9,5.5,0.0\nR004,2012-11-13,-4.9,6.3,0.0\nR004,2012-11-14,-7.5,5.3,0.0\nR004,2012-11-15,-5.9,2.5,0.0\nR004,2012-11-16,-4.9,9.2,0.0\nR004,2012-11-17,-5.8,4.7,10379.96\nR004,2012-11-18,-6.6,0.9,0.0\nR004,2012-11-19,-4.4,3.0,0.0\nR004,2012-11-20,-6.8,3.2,297.81\nR004,2012-11-21,-10.8,0.0,14700.85\nR004,2012-11-22,-8.3,5.6,0.0\nR004,2012-11-23,-8.8,6.0,0.0\nR004,2012-11-24,-11.8,8.8,0.0\nR004,2012-11-25,-5.2,14.9,0.0\nR004,2012-11-26,-9.3,7.0,10175.14\nR004,2012-11-27,-16.2,1.3,578.67\nR004,2012-11-28,-22.7,0.0,22818.04\nR004,2012-11-29,-21.1,0.0,0.0\nR004,2012-11-30,-11.6,2.3,14133.35\nR004,2012-12-01,-15.7,3.1,0.0\nR004,2012-12-02,-19.6,1.5,0.0\nR004,2012-12-03,-16.9,1.9,636.35\nR004,2012-12-04,-17.4,0.2,22829.23\nR004,2012-12-05,-13.4,7.8,0.0\nR004,2012-12-06,-16.1,0.3,24977.09\nR004,2012-12-07,-17.6,0.0,6390.16\nR004,2012-12-08,-15.6,0.0,734.47\nR004,2012-12-09,-17.1,0.0,13000.24\nR004,2012-12-10,-16.2,1.8,6053.76\nR004,2012-12-11,-23.7,0.0,20848.42\nR004,2012-12-12,-31.6,0.0,23799.71\nR004,2012-12-13,-36.0,0.0,22393.98\nR004,2012-12-14,-34.3,0.4,10632.01\nR004,2012-12-15,-32.7,0.1,24243.93\nR004,2012-12-16,-37.6,0.0,25782.92\nR004,2012-12-17,-35.3,0.3,2775.98\nR004,2012-12-18,-36.1,0.0,22915.49\nR004,2012-12-19,-36.6,0.0,25369.11\nR004,2012-12-20,-30.7,1.1,0.0\nR004,2012-12-21,-31.2,0.0,24346.66\nR004,2012-12-22,-34.5,0.0,7714.43\nR004,2012-12-23,-31.7,0.0,22725.04\nR004,2012-12-24,-20.3,4.2,0.0\nR004,2012-12-25,-15.6,1.1,1129.82\nR004,2012-12-26,-14.4,6.5,0.0\nR004,2012-12-27,-22.3,0.2,20886.5\nR004,2012-12-28,-23.7,0.0,308.73\nR004,2012-12-29,-12.6,0.8,0.0\nR004,2012-12-30,-10.3,5.4,0.0\nR004,2012-12-31,-15.7,0.0,25437.04\nR004,2013-01-01,-21.0,0.0,26013.59\nR004,2013-01-02,-26.5,0.0,26072.32\nR004,2013-01-03,-26.5,0.0,24648.72\nR004,2013-01-04,-25.9,0.0,25678.3\nR004,2013-01-05,-25.1,0.1,18065.31\nR004,2013-01-06,-20.7,0.7,18398.94\nR004,2013-01-07,-26.7,0.0,26228.19\nR004,2013-01-08,-21.2,1.2,1117.84\nR004,2013-01-09,-17.4,1.2,1235.06\nR004,2013-01-10,-19.5,3.4,2918.14\nR004,2013-01-11,-9.7,8.8,0.0\nR004,2013-01-12,-12.6,0.0,10232.26\nR004,2013-01-13,-10.6,2.1,7866.46\nR004,2013-01-14,-12.8,0.0,3088.52\nR004,2013-01-15,-13.7,0.0,7679.53\nR004,2013-01-16,-10.4,3.4,1272.01\nR004,2013-01-17,-14.0,1.6,5195.71\nR004,2013-01-18,-15.8,1.0,9243.26\nR004,2013-01-19,-15.0,1.3,1011.71\nR004,2013-01-20,-13.9,0.5,26749.45\nR004,2013-01-21,-13.1,1.4,125.81\nR004,2013-01-22,-11.1,2.6,11148.93\nR004,2013-01-23,-14.4,0.0,9159.62\nR004,2013-01-24,-5.6,8.6,0.0\nR004,2013-01-25,-4.0,3.2,0.0\nR004,2013-01-26,-1.0,3.8,7396.52\nR004,2013-01-27,-5.4,9.4,0.0\nR004,2013-01-28,-13.0,4.6,3253.85\nR004,2013-01-29,-23.0,0.0,31053.63\nR004,2013-01-30,-24.8,0.0,6041.26\nR004,2013-01-31,-19.8,0.0,21235.26\nR004,2013-02-01,-16.0,0.0,19457.26\nR004,2013-02-02,-12.8,0.0,24878.83\nR004,2013-02-03,-17.2,0.0,29987.59\nR004,2013-02-04,-16.5,0.0,28680.89\nR004,2013-02-05,-12.4,0.0,21899.04\nR004,2013-02-06,-10.7,4.6,0.0\nR004,2013-02-07,-16.1,0.0,25693.92\nR004,2013-02-08,-19.6,0.0,32378.81\nR004,2013-02-09,-21.7,0.0,32400.0\nR004,2013-02-10,-19.3,0.6,26179.43\nR004,2013-02-11,-9.8,16.1,0.0\nR004,2013-02-12,-12.8,2.4,18001.76\nR004,2013-02-13,-15.7,0.0,16032.47\nR004,2013-02-14,-12.6,3.9,0.0\nR004,2013-02-15,-13.2,3.9,5160.67\nR004,2013-02-16,-20.9,0.2,21723.44\nR004,2013-02-17,-19.6,0.0,17136.62\nR004,2013-02-18,-16.0,1.5,22243.37\nR004,2013-02-19,-13.2,0.0,26282.58\nR004,2013-02-20,-12.6,0.0,33272.77\nR004,2013-02-21,-12.7,0.0,32198.38\nR004,2013-02-22,-8.8,1.1,12643.29\nR004,2013-02-23,-7.3,3.5,17203.8\nR004,2013-02-24,-8.4,2.3,13038.66\nR004,2013-02-25,-13.4,4.2,0.0\nR004,2013-02-26,-15.6,2.3,20600.16\nR004,2013-02-27,-15.9,0.5,35047.7\nR004,2013-02-28,-15.5,0.0,32201.66\nR004,2013-03-01,-9.7,2.4,7345.41\nR004,2013-03-02,-8.0,6.8,0.0\nR004,2013-03-03,-2.7,10.2,9374.42\nR004,2013-03-04,-0.5,4.8,3236.84\nR004,2013-03-05,-4.8,4.6,17060.26\nR004,2013-03-06,-10.7,8.7,8661.23\nR004,2013-03-07,-4.6,17.9,0.0\nR004,2013-03-08,-10.0,3.3,23358.45\nR004,2013-03-09,-5.0,0.0,36280.68\nR004,2013-03-10,-8.3,3.4,24447.55\nR004,2013-03-11,-14.7,0.0,39546.83\nR004,2013-03-12,-12.9,0.0,29148.84\nR004,2013-03-13,-4.3,0.0,39947.08\nR004,2013-03-14,0.2,0.0,40258.76\nR004,2013-03-15,2.8,8.2,1725.9\nR004,2013-03-16,-4.4,0.1,29374.95\nR004,2013-03-17,-2.6,9.7,515.84\nR004,2013-03-18,-0.6,0.2,25745.95\nR004,2013-03-19,1.0,0.0,42067.73\nR004,2013-03-20,-4.1,0.0,28779.74\nR004,2013-03-21,-4.4,2.0,24990.74\nR004,2013-03-22,-3.7,1.2,25754.18\nR004,2013-03-23,-2.6,3.4,31130.27\nR004,2013-03-24,-1.9,2.5,17542.37\nR004,2013-03-25,-2.9,1.0,41393.48\nR004,2013-03-26,-5.3,0.0,43437.2\nR004,2013-03-27,-4.7,1.7,19085.38\nR004,2013-03-28,-0.6,1.1,40938.96\nR004,2013-03-29,-0.1,0.7,43324.45\nR004,2013-03-30,0.5,0.0,43343.99\nR004,2013-03-31,4.4,0.0,42402.24\nR004,2013-04-01,-3.6,11.1,15602.5\nR004,2013-04-02,-7.8,0.0,34138.73\nR004,2013-04-03,-7.1,1.2,39642.43\nR004,2013-04-04,-6.2,0.0,39600.0\nR004,2013-04-05,-0.8,2.8,26927.87\nR004,2013-04-06,-3.9,1.8,20154.79\nR004,2013-04-07,-3.3,0.1,43143.79\nR004,2013-04-08,3.9,0.6,42563.07\nR004,2013-04-09,5.1,0.0,46895.02\nR004,2013-04-10,6.5,0.0,44884.01\nR004,2013-04-11,6.6,1.5,42001.92\nR004,2013-04-12,3.0,0.0,45513.9\nR004,2013-04-13,4.1,0.0,46491.64\nR004,2013-04-14,1.2,7.5,260.43\nR004,2013-04-15,1.0,1.5,37401.56\nR004,2013-04-16,-0.7,0.0,48928.92\nR004,2013-04-17,0.7,0.0,47294.44\nR004,2013-04-18,1.0,3.0,31726.18\nR004,2013-04-19,3.1,2.4,33523.97\nR004,2013-04-20,2.2,2.3,46800.0\nR004,2013-04-21,1.1,9.0,9799.78\nR004,2013-04-22,4.4,0.6,48318.92\nR004,2013-04-23,7.6,0.0,50400.0\nR004,2013-04-24,10.0,0.7,27167.65\nR004,2013-04-25,12.6,8.2,46507.0\nR004,2013-04-26,7.5,0.9,44952.03\nR004,2013-04-27,9.3,0.0,50400.0\nR004,2013-04-28,13.0,0.0,50400.0\nR004,2013-04-29,10.8,4.5,29685.27\nR004,2013-04-30,13.9,1.7,36000.0\nR004,2013-05-01,7.4,4.2,29261.87\nR004,2013-05-02,6.7,0.9,33855.18\nR004,2013-05-03,7.3,15.9,11174.78\nR004,2013-05-04,2.9,6.5,10474.78\nR004,2013-05-05,5.9,3.0,39640.73\nR004,2013-05-06,3.4,11.6,36253.02\nR004,2013-05-07,6.9,0.1,39581.8\nR004,2013-05-08,15.0,0.0,50857.39\nR004,2013-05-09,16.3,0.0,50400.0\nR004,2013-05-10,11.6,3.0,36530.83\nR004,2013-05-11,3.7,9.0,32326.64\nR004,2013-05-12,5.3,0.0,26817.9\nR004,2013-05-13,2.2,10.3,39662.01\nR004,2013-05-14,4.1,0.0,47834.69\nR004,2013-05-15,7.2,10.6,40188.73\nR004,2013-05-16,5.6,18.4,44266.54\nR004,2013-05-17,3.1,4.1,32663.05\nR004,2013-05-18,6.3,11.0,29168.77\nR004,2013-05-19,7.8,5.1,36760.92\nR004,2013-05-20,7.5,0.7,46666.55\nR004,2013-05-21,5.3,6.0,3651.86\nR004,2013-05-22,7.5,1.2,40063.5\nR004,2013-05-23,8.2,11.1,18288.08\nR004,2013-05-24,6.6,11.4,27636.61\nR004,2013-05-25,4.6,7.2,5854.63\nR004,2013-05-26,4.8,0.1,49777.55\nR004,2013-05-27,8.7,0.0,57214.24\nR004,2013-05-28,12.4,20.2,19026.71\nR004,2013-05-29,8.6,1.3,41827.53\nR004,2013-05-30,6.0,2.0,45068.3\nR004,2013-05-31,10.6,1.8,50475.98\nR004,2013-06-01,7.6,7.4,34196.87\nR004,2013-06-02,7.3,0.7,50373.88\nR004,2013-06-03,8.4,0.0,45414.3\nR004,2013-06-04,9.1,1.5,44013.6\nR004,2013-06-05,10.1,0.3,43410.4\nR004,2013-06-06,10.6,2.8,51231.67\nR004,2013-06-07,11.0,1.0,56237.14\nR004,2013-06-08,14.7,0.0,57600.0\nR004,2013-06-09,17.4,0.0,55052.87\nR004,2013-06-10,17.0,1.8,39625.58\nR004,2013-06-11,13.5,21.0,27236.28\nR004,2013-06-12,13.0,4.4,55176.48\nR004,2013-06-13,13.3,6.4,45533.31\nR004,2013-06-14,13.9,1.3,46999.96\nR004,2013-06-15,15.7,0.1,57063.78\nR004,2013-06-16,15.5,2.7,53984.42\nR004,2013-06-17,17.8,0.0,49719.65\nR004,2013-06-18,11.0,2.0,155.15\nR004,2013-06-19,14.7,0.0,52797.95\nR004,2013-06-20,17.7,0.0,34599.3\nR004,2013-06-21,15.9,11.4,1701.64\nR004,2013-06-22,17.0,1.1,31628.64\nR004,2013-06-23,16.2,1.1,51483.21\nR004,2013-06-24,12.8,1.0,49101.05\nR004,2013-06-25,11.6,1.6,54329.52\nR004,2013-06-26,12.8,0.0,57600.0\nR004,2013-06-27,13.5,0.0,57600.0\nR004,2013-06-28,13.6,0.0,45604.73\nR004,2013-06-29,12.6,3.3,0.0\nR004,2013-06-30,13.6,6.8,39464.43\nR004,2013-07-01,11.6,5.8,47777.1\nR004,2013-07-02,13.5,0.1,54609.39\nR004,2013-07-03,14.0,0.9,33234.22\nR004,2013-07-04,16.1,7.9,40208.77\nR004,2013-07-05,14.1,12.2,36103.3\nR004,2013-07-06,14.1,9.3,51142.6\nR004,2013-07-07,14.4,10.0,32847.52\nR004,2013-07-08,16.6,0.0,49378.82\nR004,2013-07-09,13.9,11.7,30793.66\nR004,2013-07-10,15.2,0.0,55522.67\nR004,2013-07-11,14.7,1.3,52423.97\nR004,2013-07-12,15.3,0.0,56966.31\nR004,2013-07-13,17.1,0.0,55598.42\nR004,2013-07-14,18.5,0.0,57012.31\nR004,2013-07-15,20.1,0.0,56575.38\nR004,2013-07-16,17.9,3.0,36381.88\nR004,2013-07-17,18.1,1.2,37013.58\nR004,2013-07-18,18.6,0.0,54370.43\nR004,2013-07-19,17.5,3.4,44375.86\nR004,2013-07-20,18.4,0.5,39907.44\nR004,2013-07-21,18.9,0.0,46445.56\nR004,2013-07-22,18.8,6.2,38349.83\nR004,2013-07-23,18.3,15.3,50584.48\nR004,2013-07-24,18.4,2.5,33188.68\nR004,2013-07-25,16.2,12.9,3006.04\nR004,2013-07-26,16.4,5.3,34175.93\nR004,2013-07-27,17.2,2.5,47227.05\nR004,2013-07-28,17.3,2.8,34334.65\nR004,2013-07-29,16.5,10.6,39670.98\nR004,2013-07-30,15.9,17.4,17268.97\nR004,2013-07-31,16.2,5.9,46380.41\nR004,2013-08-01,15.4,5.5,27554.74\nR004,2013-08-02,16.7,0.0,50038.74\nR004,2013-08-03,15.7,8.9,32904.81\nR004,2013-08-04,15.6,14.4,37051.21\nR004,2013-08-05,16.5,13.1,44522.55\nR004,2013-08-06,16.5,10.1,18197.35\nR004,2013-08-07,17.9,8.4,45472.84\nR004,2013-08-08,17.4,3.0,42074.37\nR004,2013-08-09,17.8,9.0,42469.52\nR004,2013-08-10,18.2,0.5,51023.05\nR004,2013-08-11,19.1,0.2,45757.88\nR004,2013-08-12,16.4,16.1,26539.71\nR004,2013-08-13,16.9,3.7,47513.99\nR004,2013-08-14,14.5,16.4,13982.11\nR004,2013-08-15,14.1,21.0,4260.15\nR004,2013-08-16,13.3,4.3,16302.22\nR004,2013-08-17,13.6,0.0,24962.9\nR004,2013-08-18,13.2,0.0,13680.44\nR004,2013-08-19,14.1,1.5,23999.21\nR004,2013-08-20,14.6,0.0,44434.14\nR004,2013-08-21,14.7,7.2,42906.46\nR004,2013-08-22,13.5,0.1,41865.05\nR004,2013-08-23,12.9,0.0,48169.84\nR004,2013-08-24,14.7,0.0,48861.26\nR004,2013-08-25,15.3,5.8,36910.52\nR004,2013-08-26,14.4,8.8,9434.87\nR004,2013-08-27,15.6,0.6,41716.57\nR004,2013-08-28,16.1,0.0,43205.5\nR004,2013-08-29,18.5,0.0,48581.74\nR004,2013-08-30,15.0,0.7,18505.12\nR004,2013-08-31,9.0,8.1,4315.49\nR004,2013-09-01,10.0,1.4,27467.19\nR004,2013-09-02,12.4,0.8,25815.16\nR004,2013-09-03,14.2,0.0,46557.04\nR004,2013-09-04,14.9,0.0,45946.14\nR004,2013-09-05,15.9,0.0,44132.15\nR004,2013-09-06,13.9,0.9,43933.55\nR004,2013-09-07,8.0,1.3,19854.35\nR004,2013-09-08,6.9,0.0,44095.92\nR004,2013-09-09,8.2,0.0,43370.31\nR004,2013-09-10,9.6,2.9,24049.6\nR004,2013-09-11,8.7,16.3,22183.93\nR004,2013-09-12,6.3,15.9,16942.53\nR004,2013-09-13,4.7,0.0,43200.0\nR004,2013-09-14,7.1,0.0,43200.0\nR004,2013-09-15,8.1,13.5,17510.0\nR004,2013-09-16,5.9,7.1,32133.03\nR004,2013-09-17,4.2,2.3,25007.76\nR004,2013-09-18,2.3,1.4,43200.0\nR004,2013-09-19,2.5,0.0,32307.01\nR004,2013-09-20,1.8,4.6,33874.14\nR004,2013-09-21,1.2,2.9,29238.05\nR004,2013-09-22,1.9,0.0,42943.84\nR004,2013-09-23,4.0,0.0,34829.56\nR004,2013-09-24,7.4,0.0,42834.62\nR004,2013-09-25,10.9,0.0,42588.44\nR004,2013-09-26,10.9,0.2,745.02\nR004,2013-09-27,10.5,0.2,32865.43\nR004,2013-09-28,7.4,13.9,24011.93\nR004,2013-09-29,4.5,12.1,0.0\nR004,2013-09-30,6.3,0.0,36000.0\nR004,2013-10-01,8.9,0.0,37684.54\nR004,2013-10-02,5.9,14.7,0.0\nR004,2013-10-03,3.2,1.5,35743.17\nR004,2013-10-04,1.7,8.3,13417.78\nR004,2013-10-05,-0.8,1.1,30052.01\nR004,2013-10-06,0.7,0.0,35606.3\nR004,2013-10-07,5.5,3.2,1208.45\nR004,2013-10-08,2.3,2.5,23218.22\nR004,2013-10-09,5.1,0.0,36000.0\nR004,2013-10-10,2.1,6.4,0.0\nR004,2013-10-11,-1.6,3.1,29540.6\nR004,2013-10-12,-4.3,0.0,22911.94\nR004,2013-10-13,-3.8,0.0,28877.33\nR004,2013-10-14,1.3,0.0,36000.0\nR004,2013-10-15,3.1,0.0,32137.73\nR004,2013-10-16,0.7,8.7,1346.46\nR004,2013-10-17,-0.6,0.5,34544.06\nR004,2013-10-18,2.4,0.0,35827.01\nR004,2013-10-19,2.8,4.5,28800.0\nR004,2013-10-20,-4.4,0.0,33747.06\nR004,2013-10-21,-3.8,0.0,34484.54\nR004,2013-10-22,-0.1,2.5,12502.3\nR004,2013-10-23,0.3,3.6,0.0\nR004,2013-10-24,-0.9,7.0,3983.35\nR004,2013-10-25,-1.2,1.5,14380.74\nR004,2013-10-26,0.7,2.9,1631.67\nR004,2013-10-27,3.5,3.7,7704.05\nR004,2013-10-28,2.1,3.7,0.0\nR004,2013-10-29,2.6,12.4,0.0\nR004,2013-10-30,3.9,5.0,11619.06\nR004,2013-10-31,4.4,0.0,23096.31\nR004,2013-11-01,4.9,8.4,3080.77\nR004,2013-11-02,5.7,0.3,30000.85\nR004,2013-11-03,1.4,5.2,13083.89\nR004,2013-11-04,-0.8,4.2,0.0\nR004,2013-11-05,-0.1,6.8,5220.29\nR004,2013-11-06,0.7,31.4,0.0\nR004,2013-11-07,-2.5,8.0,4370.32\nR004,2013-11-08,-9.8,0.2,28800.0\nR004,2013-11-09,-11.1,0.0,29132.47\nR004,2013-11-10,-9.1,0.0,29194.35\nR004,2013-11-11,-8.7,0.0,26007.02\nR004,2013-11-12,-6.4,2.4,0.0\nR004,2013-11-13,-0.5,7.3,3873.57\nR004,2013-11-14,-2.0,11.6,0.0\nR004,2013-11-15,-1.6,3.0,1311.55\nR004,2013-11-16,-2.5,0.0,28800.0\nR004,2013-11-17,-3.4,0.0,9396.09\nR004,2013-11-18,-0.2,5.1,0.0\nR004,2013-11-19,-1.1,1.7,8131.51\nR004,2013-11-20,0.3,9.8,4161.78\nR004,2013-11-21,-4.7,2.1,12752.0\nR004,2013-11-22,-6.1,2.7,0.0\nR004,2013-11-23,-5.6,3.5,10174.94\nR004,2013-11-24,-7.3,2.8,1540.5\nR004,2013-11-25,-6.0,2.2,0.0\nR004,2013-11-26,-12.7,0.0,28028.62\nR004,2013-11-27,-16.1,0.0,28311.52\nR004,2013-11-28,-10.2,1.4,4954.81\nR004,2013-11-29,-1.9,0.1,19176.59\nR004,2013-11-30,1.6,6.5,1518.32\nR004,2013-12-01,-0.6,4.4,765.84\nR004,2013-12-02,-1.6,5.0,0.0\nR004,2013-12-03,-8.2,0.0,20684.79\nR004,2013-12-04,-12.2,0.0,10879.7\nR004,2013-12-05,-5.4,2.1,0.0\nR004,2013-12-06,-5.3,1.3,18553.67\nR004,2013-12-07,-11.7,0.0,21051.93\nR004,2013-12-08,-15.2,0.0,6716.01\nR004,2013-12-09,-18.9,0.0,26907.51\nR004,2013-12-10,-19.0,0.0,25272.85\nR004,2013-12-11,-18.0,0.0,26373.2\nR004,2013-12-12,-16.7,0.0,20521.14\nR004,2013-12-13,-9.4,0.2,23980.37\nR004,2013-12-14,-8.8,0.0,11762.21\nR004,2013-12-15,-14.1,0.0,15836.85\nR004,2013-12-16,-9.2,0.0,2667.16\nR004,2013-12-17,-6.1,2.5,648.94\nR004,2013-12-18,-12.5,0.7,17068.38\nR004,2013-12-19,-15.5,0.8,23126.61\nR004,2013-12-20,-11.3,1.0,14340.47\nR004,2013-12-21,-8.4,3.0,0.0\nR004,2013-12-22,-8.5,3.5,935.75\nR004,2013-12-23,-19.5,0.0,21010.02\nR004,2013-12-24,-27.4,0.0,24524.53\nR004,2013-12-25,-25.7,0.0,21845.9\nR004,2013-12-26,-21.1,0.0,24534.9\nR004,2013-12-27,-13.3,0.0,6392.83\nR004,2013-12-28,-8.9,1.4,731.53\nR004,2013-12-29,-6.9,6.6,0.0\nR004,2013-12-30,-6.4,20.1,0.0\nR004,2013-12-31,-6.1,0.7,15988.25\nR004,2014-01-01,-7.2,8.7,0.0\nR004,2014-01-02,-7.5,10.8,0.0\nR004,2014-01-03,-9.0,3.0,17538.69\nR004,2014-01-04,-11.8,5.4,0.0\nR004,2014-01-05,-24.9,0.4,7036.1\nR004,2014-01-06,-28.7,0.0,17619.07\nR004,2014-01-07,-19.7,1.9,0.0\nR004,2014-01-08,-19.2,0.0,11047.85\nR004,2014-01-09,-17.7,0.8,38.52\nR004,2014-01-10,-11.9,2.7,6737.41\nR004,2014-01-11,-9.4,2.3,9282.89\nR004,2014-01-12,-12.9,0.0,26598.65\nR004,2014-01-13,-19.6,0.0,27061.82\nR004,2014-01-14,-21.5,0.1,23060.95\nR004,2014-01-15,-13.6,1.3,21887.79\nR004,2014-01-16,-13.7,0.7,6722.74\nR004,2014-01-17,-11.0,1.7,13487.54\nR004,2014-01-18,-10.6,2.5,12510.5\nR004,2014-01-19,-18.4,0.0,12440.38\nR004,2014-01-20,-15.6,0.0,9141.62\nR004,2014-01-21,-7.3,6.4,2823.86\nR004,2014-01-22,-15.5,0.8,5149.21\nR004,2014-01-23,-16.7,1.6,468.36\nR004,2014-01-24,-20.9,0.0,29136.65\nR004,2014-01-25,-17.6,3.1,0.0\nR004,2014-01-26,-23.0,0.1,29148.29\nR004,2014-01-27,-20.3,5.0,308.2\nR004,2014-01-28,-17.5,2.5,7361.71\nR004,2014-01-29,-20.0,0.0,21030.45\nR004,2014-01-30,-22.5,0.0,25755.48\nR004,2014-01-31,-25.8,0.3,9214.7\nR004,2014-02-01,-24.7,0.1,1159.15\nR004,2014-02-02,-23.9,0.0,27713.98\nR004,2014-02-03,-28.3,0.0,31759.42\nR004,2014-02-04,-28.5,0.0,31878.3\nR004,2014-02-05,-30.1,0.0,26299.36\nR004,2014-02-06,-25.7,0.0,32116.59\nR004,2014-02-07,-24.3,0.0,32235.74\nR004,2014-02-08,-23.4,0.0,26878.04\nR004,2014-02-09,-19.2,0.0,20689.98\nR004,2014-02-10,-17.8,0.0,16048.38\nR004,2014-02-11,-18.6,0.0,19372.93\nR004,2014-02-12,-21.0,0.0,30028.04\nR004,2014-02-13,-21.8,0.0,15933.36\nR004,2014-02-14,-20.5,0.1,9615.57\nR004,2014-02-15,-20.0,0.1,25039.47\nR004,2014-02-16,-19.6,0.3,14444.58\nR004,2014-02-17,-17.5,0.0,32400.0\nR004,2014-02-18,-10.1,1.8,1840.41\nR004,2014-02-19,-6.6,2.4,6542.14\nR004,2014-02-20,-3.5,3.7,2601.8\nR004,2014-02-21,-3.3,2.9,21424.69\nR004,2014-02-22,-4.9,0.5,18009.65\nR004,2014-02-23,-7.5,6.9,0.0\nR004,2014-02-24,-22.2,0.0,33336.27\nR004,2014-02-25,-18.8,0.0,32589.63\nR004,2014-02-26,-17.1,0.0,35668.11\nR004,2014-02-27,-14.8,0.0,32400.0\nR004,2014-02-28,-12.1,0.1,19802.77\nR004,2014-03-01,-8.3,2.6,2666.01\nR004,2014-03-02,-8.7,2.6,15816.0\nR004,2014-03-03,-11.7,0.4,7577.14\nR004,2014-03-04,-11.0,0.7,26719.2\nR004,2014-03-05,-10.6,0.0,35734.9\nR004,2014-03-06,-9.4,0.0,38965.06\nR004,2014-03-07,-8.1,0.0,37975.51\nR004,2014-03-08,-5.7,5.6,0.0\nR004,2014-03-09,-10.6,0.5,33483.8\nR004,2014-03-10,-11.5,0.0,35424.79\nR004,2014-03-11,-9.9,1.7,19206.29\nR004,2014-03-12,-3.4,1.5,29116.65\nR004,2014-03-13,-2.8,0.0,39600.0\nR004,2014-03-14,-1.7,0.7,23056.06\nR004,2014-03-15,-0.6,5.5,13381.0\nR004,2014-03-16,-0.6,0.0,34668.43\nR004,2014-03-17,-0.1,1.5,34318.84\nR004,2014-03-18,1.4,0.0,38296.18\nR004,2014-03-19,1.8,0.0,42041.06\nR004,2014-03-20,2.0,1.5,34985.74\nR004,2014-03-21,0.9,0.9,42330.06\nR004,2014-03-22,-0.1,0.0,42284.1\nR004,2014-03-23,2.2,1.7,26780.93\nR004,2014-03-24,0.2,1.2,14143.85\nR004,2014-03-25,-0.6,0.0,43322.05\nR004,2014-03-26,-0.4,1.0,15692.02\nR004,2014-03-27,3.9,0.0,43497.58\nR004,2014-03-28,-0.5,2.6,35393.91\nR004,2014-03-29,0.4,0.5,29084.26\nR004,2014-03-30,3.0,0.0,40730.81\nR004,2014-03-31,5.3,2.4,32505.0\nR004,2014-04-01,6.5,0.0,44018.43\nR004,2014-04-02,7.6,0.0,45499.02\nR004,2014-04-03,8.9,0.0,45450.44\nR004,2014-04-04,9.9,0.0,44248.65\nR004,2014-04-05,6.3,5.4,19059.07\nR004,2014-04-06,3.8,3.5,0.0\nR004,2014-04-07,6.0,0.0,44022.45\nR004,2014-04-08,4.1,1.1,41777.48\nR004,2014-04-09,6.0,0.0,46883.63\nR004,2014-04-10,8.7,0.0,46922.7\nR004,2014-04-11,6.5,12.3,0.0\nR004,2014-04-12,-2.1,6.8,4356.19\nR004,2014-04-13,-0.6,0.0,48994.46\nR004,2014-04-14,-0.2,0.0,13369.82\nR004,2014-04-15,2.4,0.0,10696.79\nR004,2014-04-16,3.4,0.3,40895.59\nR004,2014-04-17,4.8,0.0,34643.68\nR004,2014-04-18,7.7,0.0,42002.45\nR004,2014-04-19,9.1,0.0,46423.38\nR004,2014-04-20,9.5,5.7,44323.16\nR004,2014-04-21,9.8,0.9,45363.54\nR004,2014-04-22,5.7,8.6,13994.22\nR004,2014-04-23,-3.4,1.1,39191.44\nR004,2014-04-24,1.4,0.0,50400.0\nR004,2014-04-25,9.1,0.0,50400.0\nR004,2014-04-26,8.3,1.4,18330.05\nR004,2014-04-27,7.0,0.0,50400.0\nR004,2014-04-28,2.1,8.7,16126.65\nR004,2014-04-29,-0.1,0.8,40665.14\nR004,2014-04-30,2.1,8.9,25590.63\nR004,2014-05-01,9.2,3.1,48903.65\nR004,2014-05-02,10.7,0.0,50719.22\nR004,2014-05-03,12.7,0.0,48320.81\nR004,2014-05-04,14.0,1.1,44102.11\nR004,2014-05-05,13.3,1.7,51002.64\nR004,2014-05-06,5.0,0.0,47687.12\nR004,2014-05-07,1.9,0.0,45525.26\nR004,2014-05-08,6.1,0.0,52072.56\nR004,2014-05-09,10.3,0.0,50701.14\nR004,2014-05-10,8.6,0.7,42578.4\nR004,2014-05-11,2.7,0.0,50606.39\nR004,2014-05-12,4.5,0.0,48636.49\nR004,2014-05-13,7.6,9.4,25136.88\nR004,2014-05-14,7.7,18.6,2260.36\nR004,2014-05-15,5.9,3.5,20258.43\nR004,2014-05-16,10.1,0.0,54461.04\nR004,2014-05-17,13.5,0.3,51167.83\nR004,2014-05-18,2.7,8.2,37634.99\nR004,2014-05-19,2.7,0.0,55590.51\nR004,2014-05-20,5.3,0.2,35500.91\nR004,2014-05-21,5.0,5.0,27768.8\nR004,2014-05-22,6.5,2.9,40145.27\nR004,2014-05-23,6.0,3.9,27041.89\nR004,2014-05-24,13.1,1.3,37600.23\nR004,2014-05-25,9.0,24.7,15853.13\nR004,2014-05-26,4.4,26.5,0.0\nR004,2014-05-27,9.4,13.5,36923.22\nR004,2014-05-28,9.6,14.9,34017.2\nR004,2014-05-29,6.9,17.0,251.59\nR004,2014-05-30,6.7,13.3,34989.77\nR004,2014-05-31,4.9,5.4,44933.03\nR004,2014-06-01,5.7,4.8,46878.63\nR004,2014-06-02,4.5,9.7,30371.48\nR004,2014-06-03,6.0,0.5,53197.12\nR004,2014-06-04,8.1,11.0,0.0\nR004,2014-06-05,7.8,1.1,52730.33\nR004,2014-06-06,8.4,0.1,53794.14\nR004,2014-06-07,9.3,5.3,13251.44\nR004,2014-06-08,7.1,8.7,15164.72\nR004,2014-06-09,9.1,3.1,50854.16\nR004,2014-06-10,11.1,0.4,57600.0\nR004,2014-06-11,11.8,4.5,50394.67\nR004,2014-06-12,16.5,0.0,57121.22\nR004,2014-06-13,16.1,0.0,57600.0\nR004,2014-06-14,17.2,0.0,57600.0\nR004,2014-06-15,18.5,0.0,57600.0\nR004,2014-06-16,19.5,0.0,57600.0\nR004,2014-06-17,19.3,0.0,57048.09\nR004,2014-06-18,16.5,6.5,28361.1\nR004,2014-06-19,17.0,3.3,35220.91\nR004,2014-06-20,15.5,6.0,43524.94\nR004,2014-06-21,15.5,0.2,45623.47\nR004,2014-06-22,17.5,5.0,45352.87\nR004,2014-06-23,18.1,4.8,38064.83\nR004,2014-06-24,19.8,0.1,53689.65\nR004,2014-06-25,21.7,0.7,48116.28\nR004,2014-06-26,18.4,18.4,19968.61\nR004,2014-06-27,19.6,6.5,40643.6\nR004,2014-06-28,20.9,0.0,55196.06\nR004,2014-06-29,19.1,1.5,28754.56\nR004,2014-06-30,16.9,4.6,31682.0\nR004,2014-07-01,16.6,0.0,57600.0\nR004,2014-07-02,17.8,8.4,40019.97\nR004,2014-07-03,16.7,6.4,51362.44\nR004,2014-07-04,14.4,8.5,42420.11\nR004,2014-07-05,14.2,6.9,38544.96\nR004,2014-07-06,15.2,2.1,34799.77\nR004,2014-07-07,17.3,1.6,49045.32\nR004,2014-07-08,19.8,0.0,57362.42\nR004,2014-07-09,19.7,0.0,57569.73\nR004,2014-07-10,20.0,0.1,55293.25\nR004,2014-07-11,21.5,0.0,57405.91\nR004,2014-07-12,23.3,0.3,52497.84\nR004,2014-07-13,24.3,0.2,46352.24\nR004,2014-07-14,19.2,7.8,34367.02\nR004,2014-07-15,17.9,1.2,48464.88\nR004,2014-07-16,13.4,10.6,37120.67\nR004,2014-07-17,11.4,13.1,7606.41\nR004,2014-07-18,14.3,2.8,43384.35\nR004,2014-07-19,17.1,0.0,55312.05\nR004,2014-07-20,19.1,0.0,55344.17\nR004,2014-07-21,20.6,0.0,55376.67\nR004,2014-07-22,19.8,4.4,50300.39\nR004,2014-07-23,16.9,1.1,32056.09\nR004,2014-07-24,15.3,10.8,28156.38\nR004,2014-07-25,13.5,9.2,28994.05\nR004,2014-07-26,14.4,2.4,36676.08\nR004,2014-07-27,16.0,1.3,48330.58\nR004,2014-07-28,16.0,13.1,38752.08\nR004,2014-07-29,13.4,10.7,37294.94\nR004,2014-07-30,12.3,15.6,7728.23\nR004,2014-07-31,12.7,5.2,36508.17\nR004,2014-08-01,12.4,7.8,15233.72\nR004,2014-08-02,13.2,28.2,11015.35\nR004,2014-08-03,15.6,0.6,48190.17\nR004,2014-08-04,17.0,0.0,53123.09\nR004,2014-08-05,18.3,0.0,50624.87\nR004,2014-08-06,16.8,8.0,24419.59\nR004,2014-08-07,14.9,8.4,6661.05\nR004,2014-08-08,14.4,10.7,2171.71\nR004,2014-08-09,13.3,0.0,52081.65\nR004,2014-08-10,15.4,0.0,51713.84\nR004,2014-08-11,17.5,0.0,51349.25\nR004,2014-08-12,18.9,0.0,50987.78\nR004,2014-08-13,20.3,1.5,46800.0\nR004,2014-08-14,19.6,0.9,46800.0\nR004,2014-08-15,20.1,0.2,41159.83\nR004,2014-08-16,19.5,1.4,33816.43\nR004,2014-08-17,15.8,7.3,17683.21\nR004,2014-08-18,16.7,0.0,47681.98\nR004,2014-08-19,18.2,0.0,50332.2\nR004,2014-08-20,17.5,7.4,36789.51\nR004,2014-08-21,12.7,0.0,49126.01\nR004,2014-08-22,12.2,0.0,47222.04\nR004,2014-08-23,14.2,0.0,49926.71\nR004,2014-08-24,16.0,0.0,44725.39\nR004,2014-08-25,16.5,0.0,49723.36\nR004,2014-08-26,18.6,0.4,47761.99\nR004,2014-08-27,11.6,10.5,0.0\nR004,2014-08-28,5.5,9.6,18891.65\nR004,2014-08-29,8.0,0.0,41774.38\nR004,2014-08-30,9.2,0.0,48091.66\nR004,2014-08-31,13.8,0.0,48028.96\nR004,2014-09-01,17.5,0.0,47548.3\nR004,2014-09-02,12.3,2.7,33006.83\nR004,2014-09-03,7.3,0.0,43092.63\nR004,2014-09-04,7.4,0.0,45527.88\nR004,2014-09-05,7.8,0.0,45255.23\nR004,2014-09-06,8.5,0.0,44298.29\nR004,2014-09-07,9.5,0.0,41856.27\nR004,2014-09-08,11.7,1.0,19076.41\nR004,2014-09-09,7.6,21.7,14683.03\nR004,2014-09-10,6.0,3.9,34469.09\nR004,2014-09-11,4.1,5.3,14608.8\nR004,2014-09-12,3.6,4.3,23009.85\nR004,2014-09-13,7.6,0.5,42238.43\nR004,2014-09-14,8.0,0.0,43200.0\nR004,2014-09-15,9.4,1.5,37435.29\nR004,2014-09-16,9.2,2.0,21505.62\nR004,2014-09-17,10.9,0.3,34798.67\nR004,2014-09-18,13.2,0.4,39284.06\nR004,2014-09-19,5.0,11.6,2163.1\nR004,2014-09-20,2.5,1.1,24427.16\nR004,2014-09-21,5.2,8.3,8552.45\nR004,2014-09-22,4.9,6.0,36000.0\nR004,2014-09-23,4.9,0.0,43153.41\nR004,2014-09-24,5.5,0.0,42907.35\nR004,2014-09-25,6.3,0.0,41942.83\nR004,2014-09-26,2.6,0.5,14343.54\nR004,2014-09-27,0.8,0.0,42025.88\nR004,2014-09-28,1.6,0.3,31859.02\nR004,2014-09-29,5.3,2.3,14050.91\nR004,2014-09-30,7.5,0.9,23458.94\nR004,2014-10-01,6.6,2.2,23337.71\nR004,2014-10-02,4.8,5.7,16857.05\nR004,2014-10-03,3.0,0.0,36665.24\nR004,2014-10-04,2.2,5.3,19820.48\nR004,2014-10-05,1.7,0.5,20945.59\nR004,2014-10-06,2.4,2.8,31080.83\nR004,2014-10-07,2.4,1.4,36000.0\nR004,2014-10-08,4.0,3.3,25028.82\nR004,2014-10-09,-2.0,6.6,4080.68\nR004,2014-10-10,-4.5,0.3,18131.97\nR004,2014-10-11,-1.3,0.9,16299.05\nR004,2014-10-12,1.4,11.0,0.0\nR004,2014-10-13,-1.8,7.7,7012.79\nR004,2014-10-14,-1.3,15.7,1525.59\nR004,2014-10-15,5.2,26.3,0.0\nR004,2014-10-16,3.8,27.0,548.07\nR004,2014-10-17,-0.4,26.8,0.0\nR004,2014-10-18,-6.3,7.0,25657.75\nR004,2014-10-19,-8.4,0.0,27669.98\nR004,2014-10-20,-3.7,2.9,25169.82\nR004,2014-10-21,2.6,5.9,0.0\nR004,2014-10-22,-1.4,2.3,6760.6\nR004,2014-10-23,0.1,5.6,17938.57\nR004,2014-10-24,1.5,6.4,0.0\nR004,2014-10-25,2.4,3.4,25009.77\nR004,2014-10-26,-0.3,12.3,0.0\nR004,2014-10-27,-6.3,9.9,13853.72\nR004,2014-10-28,-13.4,0.0,25604.55\nR004,2014-10-29,-11.7,0.0,29409.82\nR004,2014-10-30,-4.9,6.8,0.0\nR004,2014-10-31,-1.6,10.1,0.0\nR004,2014-11-01,-0.4,2.0,16144.84\nR004,2014-11-02,-5.4,0.0,29980.41\nR004,2014-11-03,-3.0,0.0,25220.46\nR004,2014-11-04,-3.3,0.0,21711.14\nR004,2014-11-05,-5.1,0.0,29743.73\nR004,2014-11-06,-4.5,0.0,28800.0\nR004,2014-11-07,-3.3,0.0,14400.0\nR004,2014-11-08,-2.4,3.1,0.0\nR004,2014-11-09,-2.9,1.5,2881.65\nR004,2014-11-10,-7.8,0.0,21866.27\nR004,2014-11-11,-14.4,0.0,28800.0\nR004,2014-11-12,-13.1,0.0,28857.38\nR004,2014-11-13,-13.2,0.0,28917.67\nR004,2014-11-14,-5.6,4.1,0.0\nR004,2014-11-15,-3.1,9.7,0.0\nR004,2014-11-16,-10.4,0.3,23789.53\nR004,2014-11-17,-8.3,8.1,0.0\nR004,2014-11-18,-5.9,6.2,0.0\nR004,2014-11-19,-10.8,10.1,0.0\nR004,2014-11-20,-9.6,5.3,399.21\nR004,2014-11-21,-5.7,8.6,0.0\nR004,2014-11-22,-5.9,8.9,0.0\nR004,2014-11-23,-7.1,1.1,10776.31\nR004,2014-11-24,-9.7,11.6,0.0\nR004,2014-11-25,-23.1,0.5,16455.89\nR004,2014-11-26,-14.6,8.8,0.0\nR004,2014-11-27,-11.0,7.5,0.0\nR004,2014-11-28,-20.3,2.1,17909.68\nR004,2014-11-29,-27.8,0.0,20681.58\nR004,2014-11-30,-30.5,0.0,23466.89\nR004,2014-12-01,-30.3,0.0,27834.55\nR004,2014-12-02,-26.9,0.0,27699.5\nR004,2014-12-03,-22.6,0.0,24265.72\nR004,2014-12-04,-14.1,0.7,9810.83\nR004,2014-12-05,-12.2,1.4,4397.57\nR004,2014-12-06,-15.5,0.2,845.36\nR004,2014-12-07,-12.4,2.0,0.0\nR004,2014-12-08,-14.2,2.3,12140.92\nR004,2014-12-09,-14.7,2.0,11155.27\nR004,2014-12-10,-12.7,1.2,21324.85\nR004,2014-12-11,-13.2,0.0,2026.28\nR004,2014-12-12,-16.8,0.1,21923.35\nR004,2014-12-13,-16.8,0.3,16419.09\nR004,2014-12-14,-20.7,0.0,22928.94\nR004,2014-12-15,-19.9,0.0,24948.92\nR004,2014-12-16,-19.5,0.0,22584.62\nR004,2014-12-17,-12.1,2.5,692.75\nR004,2014-12-18,-13.2,0.0,12358.67\nR004,2014-12-19,-21.9,0.0,25343.2\nR004,2014-12-20,-23.1,0.0,25385.75\nR004,2014-12-21,-21.8,0.0,3138.88\nR004,2014-12-22,-15.6,0.0,23419.64\nR004,2014-12-23,-16.2,0.0,24996.39\nR004,2014-12-24,-20.4,0.0,25569.84\nR004,2014-12-25,-16.7,0.2,8265.7\nR004,2014-12-26,-9.2,0.7,7648.46\nR004,2014-12-27,-6.3,5.0,0.0\nR004,2014-12-28,-8.6,4.9,20761.51\nR004,2014-12-29,-16.3,3.1,2688.73\nR004,2014-12-30,-23.7,0.0,14015.0\nR004,2014-12-31,-15.7,0.0,1945.56\nR004,2015-01-01,-15.4,0.0,24848.12\nR004,2015-01-02,-8.6,3.4,3167.44\nR004,2015-01-03,-16.0,2.5,0.0\nR004,2015-01-04,-16.5,0.0,20002.77\nR004,2015-01-05,-18.3,0.0,26207.87\nR004,2015-01-06,-13.0,2.8,0.0\nR004,2015-01-07,-6.7,2.4,21600.0\nR004,2015-01-08,-5.7,2.6,0.0\nR004,2015-01-09,-8.9,0.0,26694.63\nR004,2015-01-10,-7.0,4.5,0.0\nR004,2015-01-11,-14.2,0.0,26636.19\nR004,2015-01-12,-18.0,0.0,12204.33\nR004,2015-01-13,-13.7,0.7,3596.27\nR004,2015-01-14,-16.5,0.0,25200.0\nR004,2015-01-15,-14.5,0.0,1567.53\nR004,2015-01-16,-8.7,5.7,0.0\nR004,2015-01-17,-4.3,2.9,0.0\nR004,2015-01-18,-6.1,4.4,0.0\nR004,2015-01-19,-8.5,0.2,11667.58\nR004,2015-01-20,-8.6,1.2,14382.62\nR004,2015-01-21,-6.5,9.8,0.0\nR004,2015-01-22,-11.5,1.3,9347.47\nR004,2015-01-23,-13.3,2.6,0.0\nR004,2015-01-24,-22.5,0.3,26133.37\nR004,2015-01-25,-28.5,0.0,29618.19\nR004,2015-01-26,-31.2,0.0,30100.83\nR004,2015-01-27,-29.3,0.3,24830.51\nR004,2015-01-28,-16.6,0.9,16808.23\nR004,2015-01-29,-13.5,0.0,14963.81\nR004,2015-01-30,-17.0,0.0,28828.39\nR004,2015-01-31,-20.0,0.0,31376.67\nR004,2015-02-01,-23.8,0.0,31500.53\nR004,2015-02-02,-22.0,0.0,31618.55\nR004,2015-02-03,-19.5,0.0,31736.65\nR004,2015-02-04,-20.0,0.0,31854.68\nR004,2015-02-05,-19.8,0.0,31972.54\nR004,2015-02-06,-18.4,0.0,32090.07\nR004,2015-02-07,-18.7,0.0,32207.17\nR004,2015-02-08,-13.9,0.0,21134.99\nR004,2015-02-09,-12.7,0.0,31657.6\nR004,2015-02-10,-8.3,2.4,10059.87\nR004,2015-02-11,-10.0,0.0,5832.3\nR004,2015-02-12,-4.9,11.5,0.0\nR004,2015-02-13,-7.8,4.8,0.0\nR004,2015-02-14,-8.5,0.8,0.0\nR004,2015-02-15,-8.8,2.8,21751.11\nR004,2015-02-16,-14.8,0.0,32400.0\nR004,2015-02-17,-11.3,0.5,9670.77\nR004,2015-02-18,-9.4,7.3,0.0\nR004,2015-02-19,-22.1,0.1,31504.51\nR004,2015-02-20,-25.2,0.0,34823.09\nR004,2015-02-21,-23.8,0.0,35296.43\nR004,2015-02-22,-18.8,1.2,1744.57\nR004,2015-02-23,-14.9,0.6,28800.0\nR004,2015-02-24,-8.7,0.9,20666.23\nR004,2015-02-25,-6.7,2.0,0.0\nR004,2015-02-26,-7.2,5.0,7049.74\nR004,2015-02-27,-7.1,1.7,184.58\nR004,2015-02-28,-5.5,15.1,0.0\nR004,2015-03-01,-6.6,3.8,31283.87\nR004,2015-03-02,-9.5,0.0,38244.52\nR004,2015-03-03,-10.7,0.0,34760.73\nR004,2015-03-04,-11.3,0.0,25457.96\nR004,2015-03-05,-9.4,1.7,10551.89\nR004,2015-03-06,-10.2,1.3,24739.74\nR004,2015-03-07,-14.2,0.0,39047.44\nR004,2015-03-08,-13.4,0.0,39159.18\nR004,2015-03-09,-13.0,0.0,35460.92\nR004,2015-03-10,-4.7,0.2,16767.27\nR004,2015-03-11,-2.1,4.5,31571.08\nR004,2015-03-12,-4.1,3.6,16904.87\nR004,2015-03-13,-4.0,7.4,5831.31\nR004,2015-03-14,-4.4,8.3,25428.21\nR004,2015-03-15,-6.7,1.1,15093.74\nR004,2015-03-16,-6.5,2.9,9768.64\nR004,2015-03-17,-5.3,6.7,11339.31\nR004,2015-03-18,-5.4,8.1,10592.93\nR004,2015-03-19,-3.9,9.9,0.0\nR004,2015-03-20,-3.2,4.4,17813.65\nR004,2015-03-21,-4.0,0.0,42288.43\nR004,2015-03-22,-0.7,0.0,42429.12\nR004,2015-03-23,1.0,0.8,22378.47\nR004,2015-03-24,3.0,0.9,30473.93\nR004,2015-03-25,2.4,5.8,26340.59\nR004,2015-03-26,-4.0,2.8,29181.71\nR004,2015-03-27,-2.4,4.2,0.0\nR004,2015-03-28,-4.7,4.8,6264.98\nR004,2015-03-29,-5.8,0.0,38636.83\nR004,2015-03-30,-5.0,0.0,44515.88\nR004,2015-03-31,-6.2,0.0,19290.37\nR004,2015-04-01,-6.0,8.1,0.0\nR004,2015-04-02,-7.1,3.2,1481.62\nR004,2015-04-03,-5.4,0.0,26463.79\nR004,2015-04-04,-3.0,0.0,45688.35\nR004,2015-04-05,-2.4,0.0,45794.73\nR004,2015-04-06,0.4,0.0,45985.94\nR004,2015-04-07,0.9,0.0,46409.17\nR004,2015-04-08,1.9,0.0,46447.19\nR004,2015-04-09,1.1,0.1,24600.71\nR004,2015-04-10,2.6,0.2,42786.89\nR004,2015-04-11,5.2,0.0,42828.58\nR004,2015-04-12,7.5,0.0,45882.8\nR004,2015-04-13,2.8,8.6,2943.66\nR004,2015-04-14,0.5,2.9,42374.1\nR004,2015-04-15,1.1,0.0,32231.98\nR004,2015-04-16,2.5,10.7,0.0\nR004,2015-04-17,1.9,2.9,44702.94\nR004,2015-04-18,2.9,0.0,48323.27\nR004,2015-04-19,4.3,3.6,27294.44\nR004,2015-04-20,6.9,0.0,43200.0\nR004,2015-04-21,7.8,0.0,49861.35\nR004,2015-04-22,8.0,0.0,50338.18\nR004,2015-04-23,9.7,0.8,38298.45\nR004,2015-04-24,13.3,4.0,31911.39\nR004,2015-04-25,9.7,3.2,50400.0\nR004,2015-04-26,6.5,5.0,8764.38\nR004,2015-04-27,8.3,11.1,24903.2\nR004,2015-04-28,4.4,7.9,39660.83\nR004,2015-04-29,2.2,7.0,39715.06\nR004,2015-04-30,3.2,1.8,22531.93\nR004,2015-05-01,5.4,3.8,49111.68\nR004,2015-05-02,7.5,0.0,50733.16\nR004,2015-05-03,6.0,1.8,42549.76\nR004,2015-05-04,6.8,0.0,50159.23\nR004,2015-05-05,7.4,0.3,18435.1\nR004,2015-05-06,8.1,2.0,43408.29\nR004,2015-05-07,6.4,1.6,14767.14\nR004,2015-05-08,8.2,0.0,50762.21\nR004,2015-05-09,13.3,0.0,50712.75\nR004,2015-05-10,12.5,1.4,41364.27\nR004,2015-05-11,12.8,5.2,43015.24\nR004,2015-05-12,9.7,0.3,22752.52\nR004,2015-05-13,10.0,0.0,47454.41\nR004,2015-05-14,10.2,0.0,39384.4\nR004,2015-05-15,12.9,0.0,54013.45\nR004,2015-05-16,13.9,0.0,54945.2\nR004,2015-05-17,13.9,9.4,30326.99\nR004,2015-05-18,15.2,8.4,50851.3\nR004,2015-05-19,17.2,0.0,54223.06\nR004,2015-05-20,14.3,4.0,53429.58\nR004,2015-05-21,14.8,0.0,45997.58\nR004,2015-05-22,10.4,6.2,33240.26\nR004,2015-05-23,10.3,8.1,44126.94\nR004,2015-05-24,8.4,12.7,34487.17\nR004,2015-05-25,9.5,1.1,51763.57\nR004,2015-05-26,9.6,11.0,33195.46\nR004,2015-05-27,8.6,13.1,20539.48\nR004,2015-05-28,9.6,6.5,30340.21\nR004,2015-05-29,8.3,14.6,28644.45\nR004,2015-05-30,3.6,27.8,0.0\nR004,2015-05-31,7.4,0.3,41840.99\nR004,2015-06-01,11.5,0.0,52112.15\nR004,2015-06-02,12.6,1.9,27181.22\nR004,2015-06-03,14.2,12.6,21671.38\nR004,2015-06-04,15.1,14.8,38083.31\nR004,2015-06-05,14.1,0.0,57600.0\nR004,2015-06-06,15.9,0.0,57600.0\nR004,2015-06-07,16.9,0.0,57600.0\nR004,2015-06-08,17.0,0.0,57600.0\nR004,2015-06-09,16.5,7.6,46800.0\nR004,2015-06-10,13.7,6.4,35297.75\nR004,2015-06-11,16.7,0.0,57600.0\nR004,2015-06-12,17.5,1.2,57600.0\nR004,2015-06-13,16.8,6.3,42881.42\nR004,2015-06-14,18.8,0.0,46876.74\nR004,2015-06-15,16.2,11.3,23613.26\nR004,2015-06-16,13.6,3.5,26391.98\nR004,2015-06-17,14.6,0.1,54974.82\nR004,2015-06-18,16.6,0.0,52422.9\nR004,2015-06-19,19.1,0.0,57600.0\nR004,2015-06-20,19.4,6.0,43939.84\nR004,2015-06-21,18.0,18.4,14510.33\nR004,2015-06-22,17.8,0.0,52964.19\nR004,2015-06-23,17.5,0.0,57600.0\nR004,2015-06-24,20.0,0.0,57600.0\nR004,2015-06-25,15.8,1.1,48487.48\nR004,2015-06-26,13.8,0.0,57600.0\nR004,2015-06-27,15.6,0.0,57600.0\nR004,2015-06-28,17.8,0.0,57600.0\nR004,2015-06-29,18.9,0.0,57600.0\nR004,2015-06-30,20.7,0.0,57600.0\nR004,2015-07-01,22.5,0.0,57600.0\nR004,2015-07-02,22.9,0.2,50400.0\nR004,2015-07-03,19.4,10.9,42435.71\nR004,2015-07-04,16.9,23.7,30287.46\nR004,2015-07-05,13.6,17.8,31269.77\nR004,2015-07-06,8.5,13.5,26479.27\nR004,2015-07-07,13.2,0.8,57174.47\nR004,2015-07-08,18.0,6.5,43291.04\nR004,2015-07-09,12.3,6.4,34635.06\nR004,2015-07-10,14.6,0.6,50275.4\nR004,2015-07-11,13.5,15.0,33614.06\nR004,2015-07-12,14.1,4.7,46088.99\nR004,2015-07-13,16.7,0.0,48163.98\nR004,2015-07-14,21.0,0.0,49895.3\nR004,2015-07-15,18.2,6.5,24318.18\nR004,2015-07-16,17.8,0.0,51531.55\nR004,2015-07-17,19.4,0.0,56160.13\nR004,2015-07-18,20.8,0.0,55729.89\nR004,2015-07-19,22.9,0.0,54016.28\nR004,2015-07-20,19.6,21.6,25905.36\nR004,2015-07-21,19.1,0.0,46632.44\nR004,2015-07-22,21.8,0.7,41719.48\nR004,2015-07-23,21.0,12.1,53887.78\nR004,2015-07-24,20.0,0.1,54117.9\nR004,2015-07-25,19.8,0.0,53061.96\nR004,2015-07-26,15.4,20.1,1395.11\nR004,2015-07-27,17.2,0.0,54237.39\nR004,2015-07-28,16.0,11.5,43151.73\nR004,2015-07-29,15.7,1.9,54000.0\nR004,2015-07-30,15.1,6.1,44082.48\nR004,2015-07-31,14.6,7.0,32285.03\nR004,2015-08-01,16.3,0.0,53783.72\nR004,2015-08-02,16.6,0.8,45195.98\nR004,2015-08-03,16.9,4.7,36304.76\nR004,2015-08-04,17.5,1.3,44546.65\nR004,2015-08-05,19.0,0.0,52306.31\nR004,2015-08-06,17.0,19.8,43200.0\nR004,2015-08-07,18.6,2.3,51552.12\nR004,2015-08-08,16.3,7.6,11395.26\nR004,2015-08-09,12.7,0.0,34291.13\nR004,2015-08-10,13.0,0.0,49371.15\nR004,2015-08-11,13.2,0.0,51333.49\nR004,2015-08-12,12.5,3.0,10407.87\nR004,2015-08-13,14.3,13.8,36985.93\nR004,2015-08-14,12.2,20.0,25781.32\nR004,2015-08-15,10.9,22.5,28341.02\nR004,2015-08-16,11.2,0.0,46607.42\nR004,2015-08-17,13.3,0.0,50442.1\nR004,2015-08-18,16.2,0.0,50503.74\nR004,2015-08-19,18.2,0.0,50355.46\nR004,2015-08-20,19.8,0.0,49877.47\nR004,2015-08-21,20.3,0.7,44020.26\nR004,2015-08-22,15.2,2.8,26873.64\nR004,2015-08-23,11.0,2.9,0.0\nR004,2015-08-24,12.9,0.0,44460.39\nR004,2015-08-25,14.2,0.0,49279.23\nR004,2015-08-26,15.8,5.4,39229.26\nR004,2015-08-27,11.0,4.4,34395.91\nR004,2015-08-28,10.4,0.0,49403.23\nR004,2015-08-29,11.0,0.0,49026.63\nR004,2015-08-30,11.8,0.0,48542.38\nR004,2015-08-31,14.3,0.0,48058.68\nR004,2015-09-01,15.4,0.0,47575.56\nR004,2015-09-02,16.8,0.0,46049.17\nR004,2015-09-03,17.7,0.0,46611.33\nR004,2015-09-04,11.0,12.5,0.0\nR004,2015-09-05,8.9,4.3,17268.43\nR004,2015-09-06,8.6,5.5,44669.94\nR004,2015-09-07,8.4,5.2,39494.64\nR004,2015-09-08,8.2,0.0,42249.26\nR004,2015-09-09,9.9,0.0,43200.0\nR004,2015-09-10,12.7,0.0,43411.66\nR004,2015-09-11,14.9,0.0,43200.0\nR004,2015-09-12,17.7,0.0,43200.0\nR004,2015-09-13,8.3,17.3,1384.36\nR004,2015-09-14,3.6,8.9,22068.63\nR004,2015-09-15,1.5,4.2,21720.48\nR004,2015-09-16,4.3,2.0,15240.15\nR004,2015-09-17,3.8,3.0,41222.64\nR004,2015-09-18,3.9,0.0,5496.91\nR004,2015-09-19,6.0,0.1,42691.81\nR004,2015-09-20,6.4,6.9,22027.5\nR004,2015-09-21,8.6,7.4,12416.54\nR004,2015-09-22,3.7,13.6,13215.87\nR004,2015-09-23,2.7,6.4,16066.98\nR004,2015-09-24,1.0,12.3,22245.21\nR004,2015-09-25,0.2,5.8,16295.07\nR004,2015-09-26,0.1,2.0,32621.23\nR004,2015-09-27,0.2,2.5,21904.43\nR004,2015-09-28,0.6,0.7,20202.48\nR004,2015-09-29,0.6,5.7,17011.64\nR004,2015-09-30,5.8,15.1,1491.56\nR004,2015-10-01,11.1,4.8,36859.0\nR004,2015-10-02,9.7,3.4,27463.55\nR004,2015-10-03,10.2,4.8,24732.91\nR004,2015-10-04,9.9,7.9,18000.0\nR004,2015-10-05,4.8,27.2,7200.0\nR004,2015-10-06,1.0,6.9,26658.93\nR004,2015-10-07,2.6,0.0,36000.0\nR004,2015-10-08,4.1,0.0,36000.0\nR004,2015-10-09,5.8,0.0,36000.0\nR004,2015-10-10,8.5,0.0,29140.55\nR004,2015-10-11,10.4,0.0,35154.09\nR004,2015-10-12,9.6,1.3,22258.79\nR004,2015-10-13,6.5,4.2,15275.77\nR004,2015-10-14,0.8,6.0,1182.19\nR004,2015-10-15,-3.4,2.1,19852.29\nR004,2015-10-16,-3.2,0.0,22808.88\nR004,2015-10-17,-2.4,0.0,8007.35\nR004,2015-10-18,-0.3,3.1,323.8\nR004,2015-10-19,1.1,5.1,4031.97\nR004,2015-10-20,2.4,7.8,0.0\nR004,2015-10-21,3.2,0.1,0.0\nR004,2015-10-22,4.1,0.0,0.0\nR004,2015-10-23,1.0,3.2,0.0\nR004,2015-10-24,-2.6,0.9,15217.15\nR004,2015-10-25,-2.3,5.1,623.06\nR004,2015-10-26,-1.9,10.5,0.0\nR004,2015-10-27,-1.9,2.3,2671.24\nR004,2015-10-28,-2.1,0.0,32229.27\nR004,2015-10-29,-2.7,0.0,32110.36\nR004,2015-10-30,0.6,0.0,17653.79\nR004,2015-10-31,1.3,5.1,26190.12\nR004,2015-11-01,-1.8,6.5,0.0\nR004,2015-11-02,-6.5,5.7,0.0\nR004,2015-11-03,-8.3,1.7,6373.37\nR004,2015-11-04,-7.8,0.0,25284.1\nR004,2015-11-05,-12.1,0.0,29293.46\nR004,2015-11-06,-9.1,0.0,16881.71\nR004,2015-11-07,-7.7,0.0,29422.75\nR004,2015-11-08,-4.8,3.1,20066.96\nR004,2015-11-09,-6.6,2.6,26606.22\nR004,2015-11-10,-9.3,3.0,25685.26\nR004,2015-11-11,-12.2,0.4,128.76\nR004,2015-11-12,-16.6,0.0,28548.31\nR004,2015-11-13,-16.1,0.0,28903.79\nR004,2015-11-14,-17.4,0.0,20684.05\nR004,2015-11-15,-13.1,6.1,0.0\nR004,2015-11-16,-16.2,0.0,28800.0\nR004,2015-11-17,-22.8,0.0,28800.0\nR004,2015-11-18,-18.2,0.0,15267.51\nR004,2015-11-19,-23.0,0.0,28800.0\nR004,2015-11-20,-22.9,0.0,28800.0\nR004,2015-11-21,-18.6,0.0,11748.67\nR004,2015-11-22,-17.1,0.0,11059.08\nR004,2015-11-23,-18.8,0.0,10041.86\nR004,2015-11-24,-15.4,8.8,0.0\nR004,2015-11-25,-1.3,20.5,3600.0\nR004,2015-11-26,-2.7,9.9,0.0\nR004,2015-11-27,-2.9,12.5,0.0\nR004,2015-11-28,-5.6,0.0,28286.54\nR004,2015-11-29,-4.9,8.0,0.0\nR004,2015-11-30,-4.2,12.9,0.0\nR004,2015-12-01,-3.4,1.6,7.27\nR004,2015-12-02,-6.8,0.0,27719.88\nR004,2015-12-03,-12.4,0.0,20852.36\nR004,2015-12-04,-14.8,0.0,27369.55\nR004,2015-12-05,-11.4,0.0,13965.7\nR004,2015-12-06,-5.3,7.1,5302.39\nR004,2015-12-07,-6.5,1.6,2781.82\nR004,2015-12-08,-8.7,1.1,24080.63\nR004,2015-12-09,-3.7,4.9,1582.63\nR004,2015-12-10,-2.9,4.6,11785.11\nR004,2015-12-11,-5.8,2.6,20792.41\nR004,2015-12-12,-9.5,0.8,0.0\nR004,2015-12-13,-5.4,10.8,0.0\nR004,2015-12-14,-9.0,1.3,23980.66\nR004,2015-12-15,-14.8,0.0,23875.46\nR004,2015-12-16,-15.2,0.0,24969.84\nR004,2015-12-17,-14.6,0.0,8677.63\nR004,2015-12-18,-9.4,1.2,1384.52\nR004,2015-12-19,-10.0,3.2,0.0\nR004,2015-12-20,-14.4,0.0,0.0\nR004,2015-12-21,-17.5,0.0,7582.47\nR004,2015-12-22,-21.8,0.0,21653.87\nR004,2015-12-23,-19.0,4.5,0.0\nR004,2015-12-24,-4.5,16.3,0.0\nR004,2015-12-25,-0.6,15.7,277.48\nR004,2015-12-26,-0.6,1.1,3268.72\nR004,2015-12-27,-1.3,5.0,14644.63\nR004,2015-12-28,-3.1,5.7,0.0\nR004,2015-12-29,-2.2,6.4,0.0\nR004,2015-12-30,-6.4,0.7,14821.05\nR004,2015-12-31,-11.2,0.2,21696.04\nR004,2016-01-01,-14.4,0.1,12795.8\nR004,2016-01-02,-22.7,0.0,25504.36\nR004,2016-01-03,-26.0,0.0,26080.34\nR004,2016-01-04,-25.7,0.0,26140.26\nR004,2016-01-05,-22.1,0.0,25688.24\nR004,2016-01-06,-19.4,0.0,23778.6\nR004,2016-01-07,-23.2,0.0,26240.72\nR004,2016-01-08,-22.6,0.0,24629.82\nR004,2016-01-09,-17.6,0.0,3881.39\nR004,2016-01-10,-19.1,0.0,22313.13\nR004,2016-01-11,-22.2,0.0,27148.24\nR004,2016-01-12,-23.3,0.0,27115.68\nR004,2016-01-13,-22.8,0.0,27421.43\nR004,2016-01-14,-17.8,1.3,2024.36\nR004,2016-01-15,-19.0,0.0,12647.91\nR004,2016-01-16,-22.9,0.0,26474.61\nR004,2016-01-17,-24.5,0.0,27638.74\nR004,2016-01-18,-18.7,0.6,6152.52\nR004,2016-01-19,-19.3,0.0,21757.91\nR004,2016-01-20,-24.4,0.0,24566.5\nR004,2016-01-21,-24.4,0.0,27910.06\nR004,2016-01-22,-22.8,0.0,28159.23\nR004,2016-01-23,-21.6,0.0,28646.52\nR004,2016-01-24,-22.0,0.0,29132.04\nR004,2016-01-25,-19.9,0.0,29615.8\nR004,2016-01-26,-16.1,0.0,30097.61\nR004,2016-01-27,-20.0,0.0,30116.73\nR004,2016-01-28,-20.7,0.0,30703.22\nR004,2016-01-29,-23.1,0.0,30910.59\nR004,2016-01-30,-25.2,0.0,31120.04\nR004,2016-01-31,-23.1,0.0,27586.02\nR004,2016-02-01,-18.6,0.0,26918.91\nR004,2016-02-02,-18.8,0.0,31592.22\nR004,2016-02-03,-21.3,0.0,31707.5\nR004,2016-02-04,-21.6,0.0,31822.44\nR004,2016-02-05,-18.7,3.6,14796.44\nR004,2016-02-06,-4.6,24.0,0.0\nR004,2016-02-07,-1.4,13.9,141.74\nR004,2016-02-08,-1.9,0.7,0.0\nR004,2016-02-09,-4.9,9.8,0.0\nR004,2016-02-10,-14.7,0.0,14261.74\nR004,2016-02-11,-16.4,0.0,22902.67\nR004,2016-02-12,-14.8,0.0,9414.79\nR004,2016-02-13,-17.0,0.0,32548.05\nR004,2016-02-14,-19.8,0.0,32901.38\nR004,2016-02-15,-19.4,0.0,33251.29\nR004,2016-02-16,-17.5,0.0,32769.01\nR004,2016-02-17,-13.6,0.8,19731.76\nR004,2016-02-18,-12.9,3.1,0.0\nR004,2016-02-19,-10.0,0.0,30198.89\nR004,2016-02-20,-10.3,5.8,5520.83\nR004,2016-02-21,-14.5,0.0,29381.68\nR004,2016-02-22,-16.9,0.0,35346.79\nR004,2016-02-23,-15.3,0.0,32614.18\nR004,2016-02-24,-9.3,0.0,34166.98\nR004,2016-02-25,-1.4,0.5,12257.16\nR004,2016-02-26,-0.5,0.0,36302.67\nR004,2016-02-27,-0.6,0.0,36889.57\nR004,2016-02-28,-0.5,4.7,5764.5\nR004,2016-02-29,-2.6,0.0,21000.71\nR004,2016-03-01,-2.6,0.6,6966.39\nR004,2016-03-02,-3.5,8.1,0.0\nR004,2016-03-03,-6.6,5.7,9285.16\nR004,2016-03-04,-8.2,1.1,27230.06\nR004,2016-03-05,-9.1,0.8,26481.9\nR004,2016-03-06,-13.8,0.0,39023.25\nR004,2016-03-07,-16.2,0.0,39135.36\nR004,2016-03-08,-14.2,0.0,39247.1\nR004,2016-03-09,-14.1,0.0,39358.43\nR004,2016-03-10,-12.2,0.0,33495.86\nR004,2016-03-11,-7.0,0.0,39579.78\nR004,2016-03-12,-5.5,0.0,39994.28\nR004,2016-03-13,-3.9,0.0,40306.27\nR004,2016-03-14,-2.2,0.9,6450.4\nR004,2016-03-15,-2.2,0.8,1034.51\nR004,2016-03-16,-5.5,0.3,35675.81\nR004,2016-03-17,-3.2,0.8,12168.15\nR004,2016-03-18,-0.8,2.2,34272.33\nR004,2016-03-19,-0.4,0.0,37202.56\nR004,2016-03-20,0.3,0.0,42246.2\nR004,2016-03-21,-0.1,0.0,35923.24\nR004,2016-03-22,1.3,0.5,36212.51\nR004,2016-03-23,0.3,1.9,28800.0\nR004,2016-03-24,0.6,0.0,42890.51\nR004,2016-03-25,2.7,0.0,31764.37\nR004,2016-03-26,0.2,6.9,4788.93\nR004,2016-03-27,0.5,0.7,43261.49\nR004,2016-03-28,1.7,0.0,40201.78\nR004,2016-03-29,1.1,9.9,0.0\nR004,2016-03-30,0.7,0.4,860.97\nR004,2016-03-31,0.3,0.0,44807.26\nR004,2016-04-01,1.3,0.0,45139.83\nR004,2016-04-02,2.4,0.0,45090.01\nR004,2016-04-03,2.8,3.4,13362.3\nR004,2016-04-04,1.7,2.7,20287.46\nR004,2016-04-05,1.9,0.0,15355.88\nR004,2016-04-06,2.0,0.0,46017.24\nR004,2016-04-07,1.6,2.6,21190.2\nR004,2016-04-08,4.5,1.3,32709.09\nR004,2016-04-09,1.6,9.1,17337.23\nR004,2016-04-10,0.5,5.5,24369.65\nR004,2016-04-11,0.9,25.9,850.33\nR004,2016-04-12,1.4,8.3,6509.07\nR004,2016-04-13,0.7,13.0,20747.71\nR004,2016-04-14,-1.0,1.8,28790.08\nR004,2016-04-15,2.0,0.0,45778.11\nR004,2016-04-16,7.1,0.0,49647.85\nR004,2016-04-17,7.9,0.0,46645.66\nR004,2016-04-18,9.3,0.1,49916.64\nR004,2016-04-19,11.3,0.0,50049.31\nR004,2016-04-20,13.1,0.2,48943.33\nR004,2016-04-21,12.3,0.0,47969.02\nR004,2016-04-22,12.5,0.1,46045.63\nR004,2016-04-23,10.9,13.2,5343.6\nR004,2016-04-24,10.8,7.0,24207.64\nR004,2016-04-25,9.5,12.4,5706.76\nR004,2016-04-26,8.5,6.7,36591.27\nR004,2016-04-27,5.1,6.5,43944.31\nR004,2016-04-28,6.0,0.3,50400.0\nR004,2016-04-29,3.8,5.4,13280.88\nR004,2016-04-30,4.6,3.3,37508.83\nR004,2016-05-01,8.4,0.0,50400.0\nR004,2016-05-02,8.5,0.3,45029.71\nR004,2016-05-03,0.9,2.1,46143.37\nR004,2016-05-04,2.9,0.0,51022.94\nR004,2016-05-05,7.7,0.0,46849.99\nR004,2016-05-06,8.6,0.3,23701.11\nR004,2016-05-07,11.2,0.0,50887.77\nR004,2016-05-08,11.7,5.1,28592.08\nR004,2016-05-09,3.5,11.6,27343.6\nR004,2016-05-10,1.2,0.6,47341.22\nR004,2016-05-11,6.6,0.0,19484.77\nR004,2016-05-12,4.9,16.9,25603.4\nR004,2016-05-13,6.5,5.2,47463.39\nR004,2016-05-14,13.7,0.0,54870.16\nR004,2016-05-15,15.6,0.6,44521.08\nR004,2016-05-16,7.1,18.8,241.39\nR004,2016-05-17,4.1,10.4,24170.45\nR004,2016-05-18,2.2,6.9,51754.41\nR004,2016-05-19,4.4,1.2,55151.83\nR004,2016-05-20,8.1,0.0,56934.64\nR004,2016-05-21,10.6,0.0,54953.3\nR004,2016-05-22,11.2,0.0,48491.04\nR004,2016-05-23,13.3,0.0,54704.08\nR004,2016-05-24,14.8,1.5,51259.16\nR004,2016-05-25,14.4,0.0,57440.54\nR004,2016-05-26,14.7,1.8,54518.12\nR004,2016-05-27,12.6,0.0,47735.51\nR004,2016-05-28,13.0,0.0,57600.0\nR004,2016-05-29,10.1,10.6,28800.0\nR004,2016-05-30,8.9,0.0,57600.0\nR004,2016-05-31,13.0,7.2,56741.53\nR004,2016-06-01,8.2,18.2,849.87\nR004,2016-06-02,8.4,0.0,50751.17\nR004,2016-06-03,11.3,0.0,57600.0\nR004,2016-06-04,14.5,0.0,57600.0\nR004,2016-06-05,17.2,0.0,55202.71\nR004,2016-06-06,16.6,17.0,45585.45\nR004,2016-06-07,16.7,4.3,45616.39\nR004,2016-06-08,17.5,0.0,49168.7\nR004,2016-06-09,18.6,0.0,50361.12\nR004,2016-06-10,19.3,0.0,57600.0\nR004,2016-06-11,17.1,0.0,54947.58\nR004,2016-06-12,17.5,0.0,57600.0\nR004,2016-06-13,20.6,0.0,54650.82\nR004,2016-06-14,20.5,0.4,49526.91\nR004,2016-06-15,17.9,7.7,57412.87\nR004,2016-06-16,18.5,0.5,55672.16\nR004,2016-06-17,17.3,0.0,46455.32\nR004,2016-06-18,16.5,0.1,52760.37\nR004,2016-06-19,13.1,7.5,36033.4\nR004,2016-06-20,13.1,8.6,29799.19\nR004,2016-06-21,14.5,0.8,57119.64\nR004,2016-06-22,15.3,0.0,53871.21\nR004,2016-06-23,16.4,2.8,42112.69\nR004,2016-06-24,15.3,5.0,13659.13\nR004,2016-06-25,15.1,14.5,8267.38\nR004,2016-06-26,18.1,2.5,57600.0\nR004,2016-06-27,19.6,0.0,54052.39\nR004,2016-06-28,17.4,8.2,31987.66\nR004,2016-06-29,17.9,0.9,53329.83\nR004,2016-06-30,17.8,3.0,51103.3\nR004,2016-07-01,17.9,7.5,55685.62\nR004,2016-07-02,16.7,7.0,48713.64\nR004,2016-07-03,15.7,6.6,43186.66\nR004,2016-07-04,17.5,0.0,54577.32\nR004,2016-07-05,19.2,0.8,54535.24\nR004,2016-07-06,19.8,0.1,44507.4\nR004,2016-07-07,18.9,4.8,29076.35\nR004,2016-07-08,18.3,7.9,41526.63\nR004,2016-07-09,18.6,1.7,41755.68\nR004,2016-07-10,20.2,0.1,49414.35\nR004,2016-07-11,19.2,2.8,47171.82\nR004,2016-07-12,18.9,7.9,33903.47\nR004,2016-07-13,19.3,2.1,54441.89\nR004,2016-07-14,20.3,0.0,53312.01\nR004,2016-07-15,21.3,0.0,51757.8\nR004,2016-07-16,18.3,27.0,25521.27\nR004,2016-07-17,18.2,2.1,53575.98\nR004,2016-07-18,17.2,5.2,48450.74\nR004,2016-07-19,16.8,13.7,32141.9\nR004,2016-07-20,17.2,5.8,33154.72\nR004,2016-07-21,18.6,0.4,46778.9\nR004,2016-07-22,20.0,0.0,54521.79\nR004,2016-07-23,20.6,2.0,44401.81\nR004,2016-07-24,18.1,23.1,5646.4\nR004,2016-07-25,17.7,11.3,41795.63\nR004,2016-07-26,18.2,3.3,45053.6\nR004,2016-07-27,16.6,5.6,33169.31\nR004,2016-07-28,17.9,0.0,45011.65\nR004,2016-07-29,18.0,7.3,30105.28\nR004,2016-07-30,17.5,8.4,27801.6\nR004,2016-07-31,15.5,5.7,45091.82\nR004,2016-08-01,17.0,0.0,33813.15\nR004,2016-08-02,15.6,15.1,4576.63\nR004,2016-08-03,16.6,0.9,42827.52\nR004,2016-08-04,17.7,0.0,51868.99\nR004,2016-08-05,18.4,0.0,51914.04\nR004,2016-08-06,19.2,4.0,45771.76\nR004,2016-08-07,17.4,9.5,25200.0\nR004,2016-08-08,15.1,13.9,0.0\nR004,2016-08-09,12.3,15.0,4274.69\nR004,2016-08-10,12.5,8.2,25173.05\nR004,2016-08-11,13.0,0.0,44225.96\nR004,2016-08-12,14.6,0.0,50400.0\nR004,2016-08-13,15.2,0.2,50662.13\nR004,2016-08-14,12.5,8.6,42571.45\nR004,2016-08-15,12.4,4.6,29378.6\nR004,2016-08-16,13.2,4.8,37993.91\nR004,2016-08-17,14.3,0.0,44073.3\nR004,2016-08-18,15.7,0.1,50378.8\nR004,2016-08-19,15.9,0.0,50278.12\nR004,2016-08-20,14.3,6.7,38010.11\nR004,2016-08-21,10.0,0.0,32598.99\nR004,2016-08-22,8.7,1.5,34333.35\nR004,2016-08-23,10.1,0.0,46084.61\nR004,2016-08-24,9.6,0.0,31253.32\nR004,2016-08-25,12.5,0.0,47335.03\nR004,2016-08-26,13.4,0.0,49575.48\nR004,2016-08-27,14.8,0.0,46609.68\nR004,2016-08-28,17.3,0.0,48257.3\nR004,2016-08-29,14.7,0.7,45128.19\nR004,2016-08-30,15.9,0.1,47685.2\nR004,2016-08-31,9.7,17.5,4935.34\nR004,2016-09-01,8.3,1.3,43823.67\nR004,2016-09-02,8.3,0.0,47013.93\nR004,2016-09-03,9.4,0.0,46532.3\nR004,2016-09-04,11.6,0.0,45976.3\nR004,2016-09-05,13.1,0.0,45320.35\nR004,2016-09-06,13.9,0.0,44339.44\nR004,2016-09-07,14.1,0.0,38504.98\nR004,2016-09-08,15.4,0.0,43708.08\nR004,2016-09-09,15.5,0.5,37957.24\nR004,2016-09-10,14.7,0.1,43200.0\nR004,2016-09-11,14.2,0.0,43200.0\nR004,2016-09-12,14.0,0.9,36000.0\nR004,2016-09-13,15.1,0.0,43200.0\nR004,2016-09-14,12.2,6.9,0.0\nR004,2016-09-15,12.3,0.2,16950.99\nR004,2016-09-16,12.7,0.0,43200.0\nR004,2016-09-17,13.8,0.0,43200.0\nR004,2016-09-18,15.3,0.0,43200.0\nR004,2016-09-19,13.7,0.0,29850.57\nR004,2016-09-20,15.0,0.0,43200.0\nR004,2016-09-21,14.3,2.9,32043.68\nR004,2016-09-22,10.4,18.4,446.75\nR004,2016-09-23,7.0,5.2,0.0\nR004,2016-09-24,7.1,0.0,42621.3\nR004,2016-09-25,7.2,0.0,42519.19\nR004,2016-09-26,6.6,0.0,42270.57\nR004,2016-09-27,6.1,0.0,41438.53\nR004,2016-09-28,6.0,0.0,40031.77\nR004,2016-09-29,5.3,0.0,36858.47\nR004,2016-09-30,6.2,0.0,36912.97\nR004,2016-10-01,3.9,18.5,7629.13\nR004,2016-10-02,2.5,22.1,0.0\nR004,2016-10-03,1.4,0.9,30084.74\nR004,2016-10-04,-0.4,0.0,36359.93\nR004,2016-10-05,-1.3,0.0,35796.56\nR004,2016-10-06,0.1,0.0,36000.0\nR004,2016-10-07,-0.2,1.7,4239.39\nR004,2016-10-08,-0.7,5.6,15476.44\nR004,2016-10-09,-1.9,1.3,17708.57\nR004,2016-10-10,-0.8,1.6,10186.2\nR004,2016-10-11,0.3,6.1,7992.41\nR004,2016-10-12,-0.5,12.9,0.0\nR004,2016-10-13,-2.2,8.9,26.34\nR004,2016-10-14,-3.4,9.8,12817.9\nR004,2016-10-15,-7.4,0.0,3958.31\nR004,2016-10-16,-2.8,1.1,5407.18\nR004,2016-10-17,-3.9,0.0,21224.89\nR004,2016-10-18,-7.4,0.0,29771.71\nR004,2016-10-19,-4.4,6.8,2592.54\nR004,2016-10-20,-5.9,1.1,18286.16\nR004,2016-10-21,-9.8,0.0,18955.6\nR004,2016-10-22,-8.4,0.0,17950.73\nR004,2016-10-23,-9.2,0.0,6577.0\nR004,2016-10-24,-10.4,2.4,1529.42\nR004,2016-10-25,-8.2,8.6,0.0\nR004,2016-10-26,-8.4,5.4,764.5\nR004,2016-10-27,-8.1,9.4,0.0\nR004,2016-10-28,-7.5,0.4,0.0\nR004,2016-10-29,-9.1,0.0,17495.61\nR004,2016-10-30,-16.9,0.0,31527.79\nR004,2016-10-31,-11.4,0.0,6217.47\nR004,2016-11-01,-3.0,0.0,27817.26\nR004,2016-11-02,-0.1,12.3,0.0\nR004,2016-11-03,-1.5,1.7,0.0\nR004,2016-11-04,-4.4,1.1,0.0\nR004,2016-11-05,-12.1,2.6,18073.43\nR004,2016-11-06,-14.5,0.0,23643.54\nR004,2016-11-07,-10.4,0.0,21922.57\nR004,2016-11-08,-5.7,5.0,2226.03\nR004,2016-11-09,-7.8,5.4,1460.0\nR004,2016-11-10,-7.5,11.8,15163.58\nR004,2016-11-11,-12.0,4.1,2124.86\nR004,2016-11-12,-10.7,5.4,0.0\nR004,2016-11-13,-12.3,2.2,15587.75\nR004,2016-11-14,-16.7,5.0,0.0\nR004,2016-11-15,-15.2,2.1,0.0\nR004,2016-11-16,-23.9,0.0,26036.35\nR004,2016-11-17,-28.8,0.0,28800.0\nR004,2016-11-18,-31.0,0.0,28800.0\nR004,2016-11-19,-31.7,0.0,28800.0\nR004,2016-11-20,-31.4,0.0,28768.43\nR004,2016-11-21,-28.0,0.0,28692.02\nR004,2016-11-22,-23.0,0.0,4382.58\nR004,2016-11-23,-12.9,1.6,0.0\nR004,2016-11-24,-9.0,5.3,0.0\nR004,2016-11-25,-8.1,11.6,0.0\nR004,2016-11-26,-5.8,5.2,989.2\nR004,2016-11-27,-8.6,0.0,25750.43\nR004,2016-11-28,-10.2,1.0,11293.71\nR004,2016-11-29,-6.2,5.7,0.0\nR004,2016-11-30,-8.2,0.0,18692.3\nR004,2016-12-01,-8.1,0.0,27762.58\nR004,2016-12-02,-6.1,9.1,0.0\nR004,2016-12-03,-11.6,0.0,27513.83\nR004,2016-12-04,-15.2,0.5,24323.25\nR004,2016-12-05,-6.7,7.3,7682.32\nR004,2016-12-06,-7.5,3.6,8103.97\nR004,2016-12-07,-10.3,0.7,16420.92\nR004,2016-12-08,-14.3,0.0,20106.5\nR004,2016-12-09,-13.9,1.2,11832.56\nR004,2016-12-10,-15.1,0.6,22136.85\nR004,2016-12-11,-17.6,0.2,26142.95\nR004,2016-12-12,-18.1,0.0,26173.2\nR004,2016-12-13,-12.8,0.5,5986.75\nR004,2016-12-14,-13.5,0.0,24402.63\nR004,2016-12-15,-9.8,0.0,21122.3\nR004,2016-12-16,-5.3,10.4,0.0\nR004,2016-12-17,-5.3,2.4,3745.01\nR004,2016-12-18,-11.0,2.2,0.0\nR004,2016-12-19,-13.7,1.1,10969.58\nR004,2016-12-20,-15.0,0.0,8324.8\nR004,2016-12-21,-11.4,2.5,0.0\nR004,2016-12-22,-14.6,2.4,0.0\nR004,2016-12-23,-11.9,5.9,0.0\nR004,2016-12-24,-17.4,1.1,0.0\nR004,2016-12-25,-16.6,0.1,21600.0\nR004,2016-12-26,-11.0,5.9,666.82\nR004,2016-12-27,-17.2,0.0,24689.3\nR004,2016-12-28,-18.3,0.0,3532.16\nR004,2016-12-29,-14.8,0.3,16482.11\nR004,2016-12-30,-11.6,1.8,1425.08\nR004,2016-12-31,-8.2,2.2,3084.85\nR004,2017-01-01,-7.3,1.9,0.0\nR004,2017-01-02,-4.9,15.4,0.0\nR004,2017-01-03,-3.7,4.7,8490.51\nR004,2017-01-04,-2.4,3.0,2997.2\nR004,2017-01-05,-2.8,0.9,17984.96\nR004,2017-01-06,-4.7,2.7,8138.64\nR004,2017-01-07,-12.1,0.0,21208.54\nR004,2017-01-08,-11.5,3.6,0.0\nR004,2017-01-09,-7.7,3.9,963.02\nR004,2017-01-10,-11.4,1.3,145.03\nR004,2017-01-11,-15.6,0.0,19375.85\nR004,2017-01-12,-13.9,1.5,1737.28\nR004,2017-01-13,-15.6,0.1,22910.43\nR004,2017-01-14,-15.5,0.5,16186.42\nR004,2017-01-15,-13.5,4.0,4943.33\nR004,2017-01-16,-16.2,0.7,2391.0\nR004,2017-01-17,-16.8,1.3,16674.92\nR004,2017-01-18,-22.2,0.0,27758.5\nR004,2017-01-19,-19.7,0.0,11810.16\nR004,2017-01-20,-20.2,0.0,21886.12\nR004,2017-01-21,-14.9,0.1,17857.49\nR004,2017-01-22,-11.0,1.4,24957.79\nR004,2017-01-23,-12.0,0.7,23798.17\nR004,2017-01-24,-10.0,1.4,11229.53\nR004,2017-01-25,-12.1,0.4,17491.45\nR004,2017-01-26,-8.5,11.9,0.0\nR004,2017-01-27,-18.6,0.0,30601.91\nR004,2017-01-28,-21.4,0.0,28552.78\nR004,2017-01-29,-18.6,0.0,29645.06\nR004,2017-01-30,-13.0,0.0,30571.94\nR004,2017-01-31,-17.1,0.0,31438.9\nR004,2017-02-01,-15.3,0.0,31554.61\nR004,2017-02-02,-13.8,0.5,14651.53\nR004,2017-02-03,-14.0,0.2,31788.55\nR004,2017-02-04,-13.6,3.3,0.0\nR004,2017-02-05,-14.1,1.5,24003.32\nR004,2017-02-06,-17.8,0.0,28706.84\nR004,2017-02-07,-20.5,0.0,31849.22\nR004,2017-02-08,-18.1,0.0,29779.44\nR004,2017-02-09,-16.5,0.0,32243.84\nR004,2017-02-10,-12.4,0.7,15754.12\nR004,2017-02-11,-16.8,0.2,30640.13\nR004,2017-02-12,-20.7,0.0,25931.05\nR004,2017-02-13,-21.0,0.0,29697.92\nR004,2017-02-14,-19.7,0.0,32400.0\nR004,2017-02-15,-21.5,0.0,33201.42\nR004,2017-02-16,-22.4,0.0,27300.2\nR004,2017-02-17,-14.8,8.2,0.0\nR004,2017-02-18,-12.6,2.1,31762.34\nR004,2017-02-19,-16.9,1.6,16708.07\nR004,2017-02-20,-9.7,1.3,29442.82\nR004,2017-02-21,-5.6,4.3,19961.63\nR004,2017-02-22,-2.3,1.3,19202.59\nR004,2017-02-23,-4.2,0.0,35863.04\nR004,2017-02-24,-7.7,0.0,36334.11\nR004,2017-02-25,-6.2,0.0,36405.63\nR004,2017-02-26,-4.7,0.0,30211.63\nR004,2017-02-27,-2.2,0.0,5133.36\nR004,2017-02-28,0.6,0.0,38124.96\nR004,2017-03-01,-2.4,0.0,38377.34\nR004,2017-03-02,-3.8,4.1,0.0\nR004,2017-03-03,-7.1,0.0,6487.26\nR004,2017-03-04,-6.7,0.0,38317.52\nR004,2017-03-05,-3.8,0.0,38885.06\nR004,2017-03-06,-4.4,0.0,38997.16\nR004,2017-03-07,-4.3,0.0,39108.71\nR004,2017-03-08,-5.3,0.0,39219.69\nR004,2017-03-09,-6.9,0.0,39001.33\nR004,2017-03-10,-7.1,0.2,39439.86\nR004,2017-03-11,-9.0,0.0,39653.83\nR004,2017-03-12,-6.9,0.0,40017.6\nR004,2017-03-13,-7.0,0.0,40329.66\nR004,2017-03-14,-4.5,0.0,40640.66\nR004,2017-03-15,-7.5,0.4,25240.04\nR004,2017-03-16,-9.1,0.0,37302.45\nR004,2017-03-17,-9.9,0.0,41782.77\nR004,2017-03-18,-8.6,0.0,41927.49\nR004,2017-03-19,-7.0,0.0,42072.55\nR004,2017-03-20,-6.5,0.0,42217.81\nR004,2017-03-21,-6.0,0.0,42363.14\nR004,2017-03-22,-4.4,0.0,42508.42\nR004,2017-03-23,-2.9,0.0,42784.85\nR004,2017-03-24,-2.7,0.0,42875.93\nR004,2017-03-25,-2.0,0.0,29439.89\nR004,2017-03-26,0.3,1.3,19301.76\nR004,2017-03-27,-1.5,4.3,12309.13\nR004,2017-03-28,0.6,1.2,32442.46\nR004,2017-03-29,2.4,0.0,43776.51\nR004,2017-03-30,4.6,0.0,40162.28\nR004,2017-03-31,1.7,1.3,44820.88\nR004,2017-04-01,1.7,1.3,43200.0\nR004,2017-04-02,-5.2,1.2,35861.18\nR004,2017-04-03,-6.6,0.0,45634.06\nR004,2017-04-04,-7.0,0.0,38487.04\nR004,2017-04-05,-5.3,2.2,9452.09\nR004,2017-04-06,-6.1,0.6,15061.67\nR004,2017-04-07,0.4,0.9,18099.37\nR004,2017-04-08,1.9,0.0,45300.46\nR004,2017-04-09,2.8,0.0,28697.51\nR004,2017-04-10,3.6,0.0,46935.35\nR004,2017-04-11,4.0,0.0,47752.95\nR004,2017-04-12,6.3,0.0,37067.82\nR004,2017-04-13,5.2,0.0,40133.25\nR004,2017-04-14,4.9,8.0,14822.83\nR004,2017-04-15,4.5,3.9,40653.25\nR004,2017-04-16,1.7,0.0,25169.66\nR004,2017-04-17,4.5,5.3,40161.83\nR004,2017-04-18,0.9,8.4,7227.83\nR004,2017-04-19,3.2,0.0,50012.99\nR004,2017-04-20,4.1,2.1,28467.76\nR004,2017-04-21,2.3,0.0,49437.94\nR004,2017-04-22,4.4,0.0,50397.83\nR004,2017-04-23,5.2,9.5,8891.17\nR004,2017-04-24,6.5,0.4,38919.63\nR004,2017-04-25,6.8,5.5,37416.39\nR004,2017-04-26,11.4,0.0,50400.0\nR004,2017-04-27,14.8,0.0,50400.0\nR004,2017-04-28,16.7,0.3,50400.0\nR004,2017-04-29,11.1,1.7,19261.88\nR004,2017-04-30,7.0,9.8,23061.93\nR004,2017-05-01,1.5,7.5,37151.62\nR004,2017-05-02,-1.1,6.6,24992.83\nR004,2017-05-03,-4.3,3.2,48958.51\nR004,2017-05-04,-2.8,1.1,46800.0\nR004,2017-05-05,13.4,0.8,50565.76\nR004,2017-05-06,16.8,0.0,50874.08\nR004,2017-05-07,9.2,6.0,30933.7\nR004,2017-05-08,10.5,2.4,31864.16\nR004,2017-05-09,7.7,6.3,48070.2\nR004,2017-05-10,8.6,0.0,48911.53\nR004,2017-05-11,11.4,8.3,26461.52\nR004,2017-05-12,13.5,6.5,26075.81\nR004,2017-05-13,16.7,0.0,53353.85\nR004,2017-05-14,16.7,0.0,50659.65\nR004,2017-05-15,17.1,0.3,49255.18\nR004,2017-05-16,11.5,13.0,34511.63\nR004,2017-05-17,4.9,9.0,37095.79\nR004,2017-05-18,2.0,0.0,54199.69\nR004,2017-05-19,4.6,0.0,56477.21\nR004,2017-05-20,9.5,0.2,52818.53\nR004,2017-05-21,11.8,1.4,41856.43\nR004,2017-05-22,14.4,0.0,50370.98\nR004,2017-05-23,18.4,0.0,57281.05\nR004,2017-05-24,17.4,0.3,39418.91\nR004,2017-05-25,12.1,5.5,25597.85\nR004,2017-05-26,10.4,0.9,57497.36\nR004,2017-05-27,10.8,0.0,57563.83\nR004,2017-05-28,14.8,0.0,57600.0\nR004,2017-05-29,15.3,0.3,36000.0\nR004,2017-05-30,15.9,2.1,40567.2\nR004,2017-05-31,10.2,4.5,44646.12\nR004,2017-06-01,12.4,0.0,57600.0\nR004,2017-06-02,13.5,5.0,54776.61\nR004,2017-06-03,13.5,0.0,57600.0\nR004,2017-06-04,12.9,0.0,57600.0\nR004,2017-06-05,14.2,0.0,57600.0\nR004,2017-06-06,15.5,0.0,57600.0\nR004,2017-06-07,16.9,0.0,49771.38\nR004,2017-06-08,11.6,10.5,7259.05\nR004,2017-06-09,13.9,0.0,41636.44\nR004,2017-06-10,16.9,0.2,53242.03\nR004,2017-06-11,14.0,4.3,33717.28\nR004,2017-06-12,11.8,2.4,55845.08\nR004,2017-06-13,14.9,0.0,54941.97\nR004,2017-06-14,17.8,0.0,52086.34\nR004,2017-06-15,20.6,0.0,57600.0\nR004,2017-06-16,21.3,0.0,56741.67\nR004,2017-06-17,22.1,0.0,57600.0\nR004,2017-06-18,24.2,0.0,57600.0\nR004,2017-06-19,22.8,1.7,56662.51\nR004,2017-06-20,23.9,0.0,56383.81\nR004,2017-06-21,19.4,6.7,25986.02\nR004,2017-06-22,20.7,3.1,38188.95\nR004,2017-06-23,22.1,0.0,53626.45\nR004,2017-06-24,22.5,0.4,52488.13\nR004,2017-06-25,18.5,14.6,21739.0\nR004,2017-06-26,18.7,1.0,53084.3\nR004,2017-06-27,21.3,0.0,57600.0\nR004,2017-06-28,21.1,0.0,54000.0\nR004,2017-06-29,22.5,0.0,48444.61\nR004,2017-06-30,17.7,13.3,1645.19\nR004,2017-07-01,19.0,9.0,35819.72\nR004,2017-07-02,14.9,21.3,17882.85\nR004,2017-07-03,14.8,4.0,44528.95\nR004,2017-07-04,17.1,0.1,51044.74\nR004,2017-07-05,16.9,4.8,20929.75\nR004,2017-07-06,12.0,8.6,3676.6\nR004,2017-07-07,15.5,4.7,53663.34\nR004,2017-07-08,18.7,18.0,55404.3\nR004,2017-07-09,17.0,0.1,33303.46\nR004,2017-07-10,16.6,12.3,45380.03\nR004,2017-07-11,14.8,22.6,32163.06\nR004,2017-07-12,16.3,0.0,56038.07\nR004,2017-07-13,14.6,9.3,24547.22\nR004,2017-07-14,14.8,2.0,44481.61\nR004,2017-07-15,14.9,6.8,49996.96\nR004,2017-07-16,14.2,4.8,48823.44\nR004,2017-07-17,15.9,0.0,51161.55\nR004,2017-07-18,15.8,2.7,50843.37\nR004,2017-07-19,13.6,7.9,38998.65\nR004,2017-07-20,14.0,8.2,29753.11\nR004,2017-07-21,15.9,1.0,51594.41\nR004,2017-07-22,16.7,0.2,47066.6\nR004,2017-07-23,15.1,16.7,36350.62\nR004,2017-07-24,16.7,1.2,45918.49\nR004,2017-07-25,19.3,0.0,53429.69\nR004,2017-07-26,19.4,0.1,54221.17\nR004,2017-07-27,20.9,0.0,54260.97\nR004,2017-07-28,22.3,0.0,52535.35\nR004,2017-07-29,17.5,15.5,35578.12\nR004,2017-07-30,17.5,2.6,42672.67\nR004,2017-07-31,15.6,13.6,24925.42\nR004,2017-08-01,15.8,4.3,43335.64\nR004,2017-08-02,16.3,1.1,44028.14\nR004,2017-08-03,17.7,1.5,40582.54\nR004,2017-08-04,20.4,0.2,51010.92\nR004,2017-08-05,21.3,0.0,50565.79\nR004,2017-08-06,20.8,1.1,51529.63\nR004,2017-08-07,19.9,2.7,45907.28\nR004,2017-08-08,14.4,14.7,29733.27\nR004,2017-08-09,13.6,3.5,39864.02\nR004,2017-08-10,16.2,5.9,22402.35\nR004,2017-08-11,18.3,4.2,38165.79\nR004,2017-08-12,16.4,4.8,9721.78\nR004,2017-08-13,11.3,10.2,15774.53\nR004,2017-08-14,11.2,0.0,44733.51\nR004,2017-08-15,12.4,0.0,47698.3\nR004,2017-08-16,13.1,0.7,50197.82\nR004,2017-08-17,13.6,0.1,47667.31\nR004,2017-08-18,14.3,0.0,36290.88\nR004,2017-08-19,12.9,8.1,19524.06\nR004,2017-08-20,10.0,16.5,29527.14\nR004,2017-08-21,8.9,4.1,45701.79\nR004,2017-08-22,11.1,0.4,49574.05\nR004,2017-08-23,10.7,4.9,17154.9\nR004,2017-08-24,12.8,0.0,41359.37\nR004,2017-08-25,15.5,0.5,46734.65\nR004,2017-08-26,14.5,2.2,40585.76\nR004,2017-08-27,12.2,0.0,49382.03\nR004,2017-08-28,14.1,0.0,49032.01\nR004,2017-08-29,16.3,0.0,48578.18\nR004,2017-08-30,16.7,0.0,48093.23\nR004,2017-08-31,15.3,0.1,45112.84\nR004,2017-09-01,13.1,7.6,43909.2\nR004,2017-09-02,10.5,6.0,2654.98\nR004,2017-09-03,12.7,0.5,42398.37\nR004,2017-09-04,13.7,0.0,45562.33\nR004,2017-09-05,12.6,0.0,44904.77\nR004,2017-09-06,11.3,0.0,44318.74\nR004,2017-09-07,11.0,6.8,32712.58\nR004,2017-09-08,6.7,15.9,29839.42\nR004,2017-09-09,6.3,5.9,26645.03\nR004,2017-09-10,7.2,0.8,34271.23\nR004,2017-09-11,12.0,7.8,18358.05\nR004,2017-09-12,12.4,6.2,37929.43\nR004,2017-09-13,6.2,18.4,9585.14\nR004,2017-09-14,6.2,3.1,28686.59\nR004,2017-09-15,8.0,0.7,38188.66\nR004,2017-09-16,12.2,0.1,43200.0\nR004,2017-09-17,13.2,0.2,40252.74\nR004,2017-09-18,13.4,1.1,33575.46\nR004,2017-09-19,10.4,22.6,31805.72\nR004,2017-09-20,5.8,12.2,1347.19\nR004,2017-09-21,6.5,18.1,0.0\nR004,2017-09-22,6.2,14.2,1462.81\nR004,2017-09-23,3.4,6.7,1893.48\nR004,2017-09-24,0.0,4.1,15442.33\nR004,2017-09-25,0.1,0.2,11672.88\nR004,2017-09-26,-0.2,0.6,4630.39\nR004,2017-09-27,0.8,0.0,34625.15\nR004,2017-09-28,1.6,1.0,24997.74\nR004,2017-09-29,0.3,7.8,0.0\nR004,2017-09-30,1.3,0.2,26606.86\nR004,2017-10-01,1.5,0.5,18286.45\nR004,2017-10-02,2.3,0.0,31572.89\nR004,2017-10-03,4.1,1.6,34087.53\nR004,2017-10-04,4.2,1.0,18461.24\nR004,2017-10-05,2.5,2.8,15444.92\nR004,2017-10-06,-1.3,2.5,15062.41\nR004,2017-10-07,-1.0,0.0,26956.02\nR004,2017-10-08,-0.1,0.0,36000.0\nR004,2017-10-09,-1.0,0.0,36000.0\nR004,2017-10-10,-1.1,0.0,24229.52\nR004,2017-10-11,1.0,2.3,2046.65\nR004,2017-10-12,0.2,2.4,25393.08\nR004,2017-10-13,-1.2,0.0,36000.0\nR004,2017-10-14,-2.1,0.0,36000.0\nR004,2017-10-15,0.2,0.0,36000.0\nR004,2017-10-16,4.6,0.6,15682.6\nR004,2017-10-17,3.3,0.0,31340.01\nR004,2017-10-18,3.4,0.0,30648.31\nR004,2017-10-19,2.2,3.7,7478.81\nR004,2017-10-20,2.9,3.9,17386.56\nR004,2017-10-21,5.1,7.3,0.0\nR004,2017-10-22,6.3,8.1,25275.58\nR004,2017-10-23,0.1,5.6,10966.64\nR004,2017-10-24,-0.7,0.0,31114.27\nR004,2017-10-25,-1.6,6.3,6483.63\nR004,2017-10-26,-3.7,1.0,12547.23\nR004,2017-10-27,-3.0,0.1,30856.0\nR004,2017-10-28,-3.5,0.0,32676.24\nR004,2017-10-29,-1.4,6.0,0.0\nR004,2017-10-30,-2.2,4.0,15651.59\nR004,2017-10-31,-4.0,0.0,23594.06\nR004,2017-11-01,-3.3,0.0,30797.71\nR004,2017-11-02,0.4,0.2,14663.74\nR004,2017-11-03,5.1,0.0,14782.06\nR004,2017-11-04,1.6,1.3,7199.93\nR004,2017-11-05,-0.7,4.4,12617.99\nR004,2017-11-06,-1.6,6.4,0.0\nR004,2017-11-07,-0.4,12.5,0.0\nR004,2017-11-08,-3.5,5.5,13743.51\nR004,2017-11-09,-4.2,0.0,28800.0\nR004,2017-11-10,-4.2,7.2,0.0\nR004,2017-11-11,-8.3,5.9,15873.01\nR004,2017-11-12,-15.2,0.3,21600.0\nR004,2017-11-13,-10.6,4.3,0.0\nR004,2017-11-14,-6.9,2.9,0.0\nR004,2017-11-15,-8.1,2.1,9456.35\nR004,2017-11-16,-20.0,0.0,28800.0\nR004,2017-11-17,-18.8,0.0,25774.3\nR004,2017-11-18,-14.0,0.0,26950.96\nR004,2017-11-19,-9.5,1.4,21489.38\nR004,2017-11-20,-6.2,7.7,0.0\nR004,2017-11-21,-5.6,5.9,0.0\nR004,2017-11-22,-5.7,11.2,0.0\nR004,2017-11-23,-4.3,1.0,24960.35\nR004,2017-11-24,-8.4,0.0,28493.58\nR004,2017-11-25,-10.1,0.0,28430.33\nR004,2017-11-26,-11.2,0.0,230.86\nR004,2017-11-27,-23.1,0.0,21411.69\nR004,2017-11-28,-14.9,0.9,26423.05\nR004,2017-11-29,-18.7,0.0,28081.13\nR004,2017-11-30,-18.1,0.0,27940.1\nR004,2017-12-01,-15.3,0.0,12049.47\nR004,2017-12-02,-14.4,0.0,26940.04\nR004,2017-12-03,-13.7,0.4,0.0\nR004,2017-12-04,-12.7,0.0,24664.16\nR004,2017-12-05,-10.2,0.1,755.67\nR004,2017-12-06,-10.4,0.0,26536.41\nR004,2017-12-07,-16.3,0.0,27092.97\nR004,2017-12-08,-19.8,0.0,24497.96\nR004,2017-12-09,-15.6,0.0,26903.18\nR004,2017-12-10,-16.1,0.0,23705.33\nR004,2017-12-11,-10.8,1.1,23386.12\nR004,2017-12-12,-11.6,2.6,0.0\nR004,2017-12-13,-17.4,0.0,325.44\nR004,2017-12-14,-19.6,0.0,15513.32\nR004,2017-12-15,-23.7,0.0,21405.33\nR004,2017-12-16,-23.7,0.0,22474.15\nR004,2017-12-17,-21.1,0.0,25272.38\nR004,2017-12-18,-17.8,0.0,25310.1\nR004,2017-12-19,-21.7,0.0,25350.02\nR004,2017-12-20,-21.9,0.0,25391.97\nR004,2017-12-21,-20.5,0.0,22924.12\nR004,2017-12-22,-13.8,1.6,719.35\nR004,2017-12-23,-9.4,2.3,11034.18\nR004,2017-12-24,-12.9,0.0,24448.79\nR004,2017-12-25,-17.2,0.0,7807.85\nR004,2017-12-26,-5.8,8.2,0.0\nR004,2017-12-27,-4.3,3.0,0.0\nR004,2017-12-28,-9.6,0.5,7363.11\nR004,2017-12-29,-10.1,2.4,0.0\nR004,2017-12-30,-5.4,13.5,0.0\nR004,2017-12-31,-13.2,1.3,8044.94\nR004,2018-01-01,-22.5,0.0,19402.27\nR004,2018-01-02,-27.5,0.0,26059.62\nR004,2018-01-03,-28.3,0.0,26115.41\nR004,2018-01-04,-29.1,0.0,26171.11\nR004,2018-01-05,-29.7,0.0,26228.12\nR004,2018-01-06,-28.5,0.0,25511.16\nR004,2018-01-07,-19.0,0.0,18749.4\nR004,2018-01-08,-17.5,0.0,26210.15\nR004,2018-01-09,-18.8,0.0,26690.16\nR004,2018-01-10,-10.8,0.5,2011.29\nR004,2018-01-11,-10.2,0.6,8949.7\nR004,2018-01-12,-10.8,0.6,21451.24\nR004,2018-01-13,-13.9,0.0,9843.31\nR004,2018-01-14,-19.9,0.0,15002.9\nR004,2018-01-15,-18.7,2.8,0.0\nR004,2018-01-16,-14.1,7.7,0.0\nR004,2018-01-17,-11.8,8.5,0.0\nR004,2018-01-18,-13.4,4.7,322.2\nR004,2018-01-19,-14.6,0.0,15037.29\nR004,2018-01-20,-22.2,0.0,26529.96\nR004,2018-01-21,-30.6,0.0,28147.16\nR004,2018-01-22,-32.3,0.0,28633.06\nR004,2018-01-23,-34.9,0.0,21510.54\nR004,2018-01-24,-34.3,0.0,26564.61\nR004,2018-01-25,-36.3,0.0,29154.79\nR004,2018-01-26,-36.4,0.0,29638.74\nR004,2018-01-27,-32.7,0.0,30581.34\nR004,2018-01-28,-30.7,0.0,30602.11\nR004,2018-01-29,-18.7,0.9,6556.76\nR004,2018-01-30,-13.0,1.4,2962.94\nR004,2018-01-31,-18.7,0.0,31410.41\nR004,2018-02-01,-21.8,0.0,31528.02\nR004,2018-02-02,-22.4,0.0,31646.17\nR004,2018-02-03,-20.7,0.0,31764.7\nR004,2018-02-04,-17.6,0.0,17626.55\nR004,2018-02-05,-16.7,0.0,22876.62\nR004,2018-02-06,-18.6,0.0,32121.38\nR004,2018-02-07,-18.6,1.2,19778.76\nR004,2018-02-08,-13.0,1.9,23400.79\nR004,2018-02-09,-12.3,0.0,31874.5\nR004,2018-02-10,-15.2,0.0,32400.0\nR004,2018-02-11,-14.5,0.0,32400.0\nR004,2018-02-12,-14.8,0.0,32400.0\nR004,2018-02-13,-15.0,0.0,22886.38\nR004,2018-02-14,-15.5,0.0,32870.94\nR004,2018-02-15,-12.4,0.0,15967.4\nR004,2018-02-16,-13.3,3.0,13616.71\nR004,2018-02-17,-19.7,0.0,33087.05\nR004,2018-02-18,-20.3,0.0,33018.55\nR004,2018-02-19,-18.7,0.5,25196.2\nR004,2018-02-20,-11.5,0.0,21758.37\nR004,2018-02-21,-10.2,0.0,28800.0\nR004,2018-02-22,-8.9,1.7,379.87\nR004,2018-02-23,-10.3,0.0,13874.34\nR004,2018-02-24,-8.8,1.4,7275.47\nR004,2018-02-25,-11.3,1.9,31186.52\nR004,2018-02-26,-13.0,0.0,9783.35\nR004,2018-02-27,-14.9,0.0,37363.38\nR004,2018-02-28,-17.6,0.0,4975.49\nR004,2018-03-01,-9.3,8.8,0.0\nR004,2018-03-02,-17.2,1.9,24212.57\nR004,2018-03-03,-16.1,0.4,27159.73\nR004,2018-03-04,-14.2,0.0,38744.02\nR004,2018-03-05,-12.6,0.0,10060.76\nR004,2018-03-06,-8.5,0.0,38966.53\nR004,2018-03-07,-7.2,2.6,5803.24\nR004,2018-03-08,-8.2,3.6,16209.28\nR004,2018-03-09,-10.8,0.0,17114.51\nR004,2018-03-10,-4.4,5.6,0.0\nR004,2018-03-11,-3.4,5.8,11745.27\nR004,2018-03-12,-9.8,0.0,27537.47\nR004,2018-03-13,-13.3,0.0,39962.03\nR004,2018-03-14,-13.6,0.0,36591.69\nR004,2018-03-15,-6.7,0.0,39600.0\nR004,2018-03-16,-4.5,0.0,40891.22\nR004,2018-03-17,-2.1,0.0,41200.0\nR004,2018-03-18,-2.2,0.0,41508.6\nR004,2018-03-19,-2.0,0.0,42045.58\nR004,2018-03-20,-2.5,2.8,36097.84\nR004,2018-03-21,-2.7,4.3,4061.51\nR004,2018-03-22,2.9,4.2,18560.27\nR004,2018-03-23,3.2,4.6,31966.23\nR004,2018-03-24,2.0,3.5,39160.81\nR004,2018-03-25,-2.5,4.3,9496.32\nR004,2018-03-26,-1.4,34.3,0.0\nR004,2018-03-27,-1.9,0.7,25308.03\nR004,2018-03-28,0.9,0.0,35672.93\nR004,2018-03-29,2.5,0.2,40129.63\nR004,2018-03-30,0.7,14.8,8036.22\nR004,2018-03-31,-6.2,19.2,26371.19\nR004,2018-04-01,-10.5,0.2,42652.93\nR004,2018-04-02,-4.8,0.8,41194.38\nR004,2018-04-03,-5.1,2.2,23366.5\nR004,2018-04-04,-6.3,2.9,40366.23\nR004,2018-04-05,-5.3,0.0,45961.87\nR004,2018-04-06,0.1,0.0,42022.62\nR004,2018-04-07,3.3,0.3,16550.69\nR004,2018-04-08,2.4,3.0,29959.38\nR004,2018-04-09,1.5,2.4,36567.68\nR004,2018-04-10,0.1,1.9,28354.7\nR004,2018-04-11,0.2,1.6,44401.93\nR004,2018-04-12,1.3,0.4,12950.27\nR004,2018-04-13,6.5,0.0,39789.14\nR004,2018-04-14,3.0,5.8,1200.11\nR004,2018-04-15,0.7,0.0,43289.36\nR004,2018-04-16,1.7,1.3,45399.61\nR004,2018-04-17,1.7,8.0,1393.32\nR004,2018-04-18,-2.4,2.6,29044.71\nR004,2018-04-19,-2.0,0.1,34312.53\nR004,2018-04-20,0.7,0.2,45823.35\nR004,2018-04-21,2.0,0.0,50236.87\nR004,2018-04-22,5.6,0.0,50367.36\nR004,2018-04-23,8.4,0.0,44165.95\nR004,2018-04-24,9.3,1.8,35144.08\nR004,2018-04-25,7.0,2.0,9060.93\nR004,2018-04-26,5.3,0.4,22158.38\nR004,2018-04-27,4.1,2.4,49370.77\nR004,2018-04-28,4.4,0.4,10862.36\nR004,2018-04-29,8.1,0.1,46002.15\nR004,2018-04-30,11.1,0.0,47320.26\nR004,2018-05-01,10.2,7.4,21535.25\nR004,2018-05-02,6.3,6.8,42179.43\nR004,2018-05-03,4.4,7.4,18272.21\nR004,2018-05-04,4.0,1.2,33357.42\nR004,2018-05-05,4.9,38.6,0.0\nR004,2018-05-06,1.4,5.4,46933.9\nR004,2018-05-07,1.3,2.2,44055.04\nR004,2018-05-08,5.1,1.7,25294.69\nR004,2018-05-09,6.8,5.2,26283.78\nR004,2018-05-10,7.1,1.3,38572.68\nR004,2018-05-11,8.7,0.0,39507.59\nR004,2018-05-12,8.5,13.0,336.97\nR004,2018-05-13,6.4,0.2,41890.44\nR004,2018-05-14,2.8,9.8,29405.7\nR004,2018-05-15,6.7,5.7,37773.71\nR004,2018-05-16,5.8,2.6,39570.67\nR004,2018-05-17,4.2,5.1,24619.55\nR004,2018-05-18,1.7,0.2,37615.56\nR004,2018-05-19,0.0,6.8,26788.17\nR004,2018-05-20,1.0,2.1,51051.7\nR004,2018-05-21,6.6,2.9,34446.93\nR004,2018-05-22,11.7,7.5,26078.31\nR004,2018-05-23,9.6,26.7,16343.69\nR004,2018-05-24,2.8,9.1,27623.8\nR004,2018-05-25,3.9,8.9,27382.07\nR004,2018-05-26,11.0,18.0,15906.91\nR004,2018-05-27,19.6,0.0,51981.56\nR004,2018-05-28,16.5,0.0,53151.79\nR004,2018-05-29,9.7,3.3,28800.0\nR004,2018-05-30,15.4,0.0,48997.95\nR004,2018-05-31,10.1,4.7,19562.98\nR004,2018-06-01,11.7,0.0,57600.0\nR004,2018-06-02,17.2,0.0,57600.0\nR004,2018-06-03,13.8,3.8,36558.56\nR004,2018-06-04,11.9,0.0,57600.0\nR004,2018-06-05,13.7,2.9,27261.85\nR004,2018-06-06,18.0,0.4,57052.86\nR004,2018-06-07,18.8,0.0,57600.0\nR004,2018-06-08,18.7,0.0,57600.0\nR004,2018-06-09,19.2,0.0,57600.0\nR004,2018-06-10,19.8,0.3,48705.64\nR004,2018-06-11,18.3,0.9,56165.75\nR004,2018-06-12,16.8,8.7,43353.65\nR004,2018-06-13,18.5,0.5,45337.7\nR004,2018-06-14,15.6,9.3,35907.85\nR004,2018-06-15,13.0,3.4,53301.37\nR004,2018-06-16,15.3,0.0,57600.0\nR004,2018-06-17,18.9,0.0,57600.0\nR004,2018-06-18,20.2,0.0,36346.5\nR004,2018-06-19,21.8,0.8,49293.98\nR004,2018-06-20,19.2,0.4,51394.65\nR004,2018-06-21,16.9,3.1,57600.0\nR004,2018-06-22,19.8,0.0,57600.0\nR004,2018-06-23,21.2,0.9,53401.37\nR004,2018-06-24,21.0,1.5,55669.51\nR004,2018-06-25,23.9,0.0,56127.55\nR004,2018-06-26,20.8,1.5,41718.49\nR004,2018-06-27,20.5,9.7,39176.37\nR004,2018-06-28,21.7,1.1,53298.98\nR004,2018-06-29,20.3,3.3,46950.1\nR004,2018-06-30,18.8,0.0,56617.59\nR004,2018-07-01,20.2,0.0,53057.72\nR004,2018-07-02,19.6,2.5,48358.13\nR004,2018-07-03,18.7,8.2,44353.69\nR004,2018-07-04,11.8,6.8,8238.65\nR004,2018-07-05,11.5,1.1,55716.98\nR004,2018-07-06,12.6,4.4,48994.4\nR004,2018-07-07,15.2,0.3,50841.69\nR004,2018-07-08,15.8,0.0,57600.0\nR004,2018-07-09,16.5,0.0,57568.01\nR004,2018-07-10,17.3,0.0,57504.53\nR004,2018-07-11,18.3,0.0,57406.64\nR004,2018-07-12,20.6,0.0,57370.64\nR004,2018-07-13,21.4,0.8,41837.36\nR004,2018-07-14,19.2,5.4,17481.06\nR004,2018-07-15,19.8,0.4,52087.68\nR004,2018-07-16,19.6,0.0,53520.48\nR004,2018-07-17,17.0,0.0,29424.45\nR004,2018-07-18,15.3,0.4,22196.9\nR004,2018-07-19,14.8,1.3,23583.49\nR004,2018-07-20,15.8,0.0,24885.7\nR004,2018-07-21,17.6,0.4,44830.16\nR004,2018-07-22,19.0,0.5,42679.05\nR004,2018-07-23,20.5,2.0,38763.66\nR004,2018-07-24,16.4,5.7,14002.1\nR004,2018-07-25,13.0,14.0,0.0\nR004,2018-07-26,12.9,0.0,30093.88\nR004,2018-07-27,13.4,7.1,31611.87\nR004,2018-07-28,12.7,10.4,31716.85\nR004,2018-07-29,12.9,4.8,37442.1\nR004,2018-07-30,15.7,0.0,53963.21\nR004,2018-07-31,16.1,0.0,46515.38\nR004,2018-08-01,16.0,0.0,53759.95\nR004,2018-08-02,16.3,0.0,53045.57\nR004,2018-08-03,16.1,0.4,51575.43\nR004,2018-08-04,17.1,0.3,49262.21\nR004,2018-08-05,17.3,4.6,46681.6\nR004,2018-08-06,15.1,0.0,53206.26\nR004,2018-08-07,15.7,0.0,52407.81\nR004,2018-08-08,16.4,0.6,30805.24\nR004,2018-08-09,19.3,0.0,52082.33\nR004,2018-08-10,19.8,0.0,51298.03\nR004,2018-08-11,20.4,0.1,43200.0\nR004,2018-08-12,14.4,8.6,7302.83\nR004,2018-08-13,11.1,11.4,15072.65\nR004,2018-08-14,12.2,0.9,20162.32\nR004,2018-08-15,14.2,0.0,50744.9\nR004,2018-08-16,16.1,0.0,50400.0\nR004,2018-08-17,16.5,0.0,25837.69\nR004,2018-08-18,17.7,0.5,27355.89\nR004,2018-08-19,19.2,0.0,43996.53\nR004,2018-08-20,18.4,4.3,39218.56\nR004,2018-08-21,18.3,0.5,30372.09\nR004,2018-08-22,15.3,3.1,39890.93\nR004,2018-08-23,14.6,0.6,49604.6\nR004,2018-08-24,15.7,1.7,49248.45\nR004,2018-08-25,15.6,1.3,47430.64\nR004,2018-08-26,16.5,0.0,49620.15\nR004,2018-08-27,14.8,1.3,14213.87\nR004,2018-08-28,13.6,0.0,46005.47\nR004,2018-08-29,15.2,0.0,45648.23\nR004,2018-08-30,14.6,1.0,35961.42\nR004,2018-08-31,13.0,3.2,37265.27\nR004,2018-09-01,12.7,3.8,25657.34\nR004,2018-09-02,13.5,1.7,43208.12\nR004,2018-09-03,7.6,14.1,33298.72\nR004,2018-09-04,5.4,10.4,2624.42\nR004,2018-09-05,9.2,0.1,33451.13\nR004,2018-09-06,10.7,0.0,45373.31\nR004,2018-09-07,11.6,0.0,44369.15\nR004,2018-09-08,12.6,7.2,27134.99\nR004,2018-09-09,10.0,1.0,39412.26\nR004,2018-09-10,9.5,1.7,25702.42\nR004,2018-09-11,5.6,13.1,10291.84\nR004,2018-09-12,1.6,5.6,32790.13\nR004,2018-09-13,3.3,0.9,4813.36\nR004,2018-09-14,9.2,0.0,39249.21\nR004,2018-09-15,10.1,2.7,21389.47\nR004,2018-09-16,9.9,6.9,35506.32\nR004,2018-09-17,7.5,9.4,32639.33\nR004,2018-09-18,5.4,0.2,33301.99\nR004,2018-09-19,5.6,0.0,43200.0\nR004,2018-09-20,6.4,0.0,43200.0\nR004,2018-09-21,8.6,0.0,43200.0\nR004,2018-09-22,11.4,0.0,34186.95\nR004,2018-09-23,10.6,1.0,36187.94\nR004,2018-09-24,9.4,8.9,26966.81\nR004,2018-09-25,7.7,2.8,19410.16\nR004,2018-09-26,9.5,1.2,2695.66\nR004,2018-09-27,11.7,0.0,30226.04\nR004,2018-09-28,10.7,0.0,39962.95\nR004,2018-09-29,9.7,0.0,37573.08\nR004,2018-09-30,9.2,0.0,38686.65\nR004,2018-10-01,10.3,0.0,37661.7\nR004,2018-10-02,7.7,10.8,20238.39\nR004,2018-10-03,2.8,4.9,24222.64\nR004,2018-10-04,3.6,0.2,30628.58\nR004,2018-10-05,4.6,0.0,36003.1\nR004,2018-10-06,6.3,0.0,36058.03\nR004,2018-10-07,6.9,0.0,36000.0\nR004,2018-10-08,4.9,0.0,16369.12\nR004,2018-10-09,4.3,0.0,36000.0\nR004,2018-10-10,6.4,0.0,34440.34\nR004,2018-10-11,3.7,0.0,28493.02\nR004,2018-10-12,2.7,0.0,36000.0\nR004,2018-10-13,5.0,0.0,36000.0\nR004,2018-10-14,3.4,4.0,175.47\nR004,2018-10-15,0.5,0.0,28092.7\nR004,2018-10-16,2.4,0.1,17714.62\nR004,2018-10-17,2.3,2.7,0.0\nR004,2018-10-18,0.5,0.1,23825.63\nR004,2018-10-19,-0.3,0.0,28800.0\nR004,2018-10-20,0.1,0.5,31835.55\nR004,2018-10-21,-0.5,7.1,4242.03\nR004,2018-10-22,2.0,13.5,0.0\nR004,2018-10-23,-2.2,2.2,9793.46\nR004,2018-10-24,-2.2,0.2,6738.87\nR004,2018-10-25,0.2,0.0,26898.55\nR004,2018-10-26,1.3,0.0,29875.25\nR004,2018-10-27,0.2,0.0,32736.47\nR004,2018-10-28,1.3,0.0,32690.96\nR004,2018-10-29,1.6,2.6,16029.12\nR004,2018-10-30,1.2,10.3,0.0\nR004,2018-10-31,-3.4,3.3,23314.5\nR004,2018-11-01,-7.9,0.0,23928.2\nR004,2018-11-02,-8.5,0.4,3898.52\nR004,2018-11-03,-4.9,5.0,0.0\nR004,2018-11-04,-3.9,5.0,3600.0\nR004,2018-11-05,-3.2,0.0,1451.65\nR004,2018-11-06,-2.3,12.7,0.0\nR004,2018-11-07,-1.1,5.3,6738.5\nR004,2018-11-08,-1.4,7.8,0.0\nR004,2018-11-09,-2.6,11.3,0.0\nR004,2018-11-10,-17.0,0.0,7384.92\nR004,2018-11-11,-20.3,0.0,23491.87\nR004,2018-11-12,-26.5,0.0,28800.0\nR004,2018-11-13,-15.5,1.9,0.0\nR004,2018-11-14,-14.0,0.0,28800.0\nR004,2018-11-15,-14.1,0.7,23806.19\nR004,2018-11-16,-9.1,10.8,0.0\nR004,2018-11-17,-12.5,0.0,9716.34\nR004,2018-11-18,-13.0,0.0,0.0\nR004,2018-11-19,-13.5,0.0,23718.36\nR004,2018-11-20,-11.5,0.0,780.33\nR004,2018-11-21,-13.4,0.0,4307.14\nR004,2018-11-22,-11.9,0.0,437.17\nR004,2018-11-23,-5.8,4.4,2745.85\nR004,2018-11-24,-2.8,6.6,12607.59\nR004,2018-11-25,-9.1,8.7,9570.22\nR004,2018-11-26,-10.8,4.5,0.0\nR004,2018-11-27,-11.6,1.5,28138.08\nR004,2018-11-28,-7.2,3.9,0.0\nR004,2018-11-29,-4.8,5.8,0.0\nR004,2018-11-30,-2.9,23.5,0.0\nR004,2018-12-01,-18.2,3.5,1118.83\nR004,2018-12-02,-19.2,0.0,4132.12\nR004,2018-12-03,-17.3,2.8,345.81\nR004,2018-12-04,-21.1,0.1,18616.98\nR004,2018-12-05,-28.0,0.0,25476.12\nR004,2018-12-06,-31.3,0.0,27211.25\nR004,2018-12-07,-31.5,0.0,27106.98\nR004,2018-12-08,-25.7,0.0,15376.73\nR004,2018-12-09,-20.9,0.0,26072.08\nR004,2018-12-10,-23.2,0.0,26526.99\nR004,2018-12-11,-23.5,0.0,26644.46\nR004,2018-12-12,-23.5,0.0,25882.02\nR004,2018-12-13,-16.8,2.2,0.0\nR004,2018-12-14,-10.5,3.1,645.91\nR004,2018-12-15,-6.6,4.9,0.0\nR004,2018-12-16,-7.0,2.6,11156.59\nR004,2018-12-17,-5.1,6.7,3634.04\nR004,2018-12-18,-9.5,1.2,6929.55\nR004,2018-12-19,-10.8,2.6,444.06\nR004,2018-12-20,-13.8,5.6,10488.3\nR004,2018-12-21,-20.7,1.7,1533.51\nR004,2018-12-22,-15.3,2.4,16363.58\nR004,2018-12-23,-17.0,0.0,0.0\nR004,2018-12-24,-20.1,0.0,22683.52\nR004,2018-12-25,-25.6,0.0,25619.88\nR004,2018-12-26,-26.4,0.0,25668.96\nR004,2018-12-27,-30.3,0.0,25718.8\nR004,2018-12-28,-32.7,0.0,25769.11\nR004,2018-12-29,-32.8,0.0,25819.81\nR004,2018-12-30,-30.0,0.0,25870.9\nR004,2018-12-31,-26.4,0.0,809.02\nR004,2019-01-01,-22.3,0.0,12417.89\nR004,2019-01-02,-25.0,0.0,21708.5\nR004,2019-01-03,-27.4,0.0,26091.35\nR004,2019-01-04,-29.0,0.0,26150.12\nR004,2019-01-05,-26.0,0.0,20518.84\nR004,2019-01-06,-17.4,3.3,852.91\nR004,2019-01-07,-19.7,0.0,25225.98\nR004,2019-01-08,-19.3,0.0,5808.19\nR004,2019-01-09,-15.6,0.3,0.0\nR004,2019-01-10,-9.2,5.9,0.0\nR004,2019-01-11,-16.5,0.3,25200.0\nR004,2019-01-12,-17.4,1.6,0.0\nR004,2019-01-13,-18.0,0.0,17932.96\nR004,2019-01-14,-20.8,0.0,27031.63\nR004,2019-01-15,-22.9,0.0,27543.5\nR004,2019-01-16,-18.2,0.1,18775.05\nR004,2019-01-17,-11.7,1.3,447.37\nR004,2019-01-18,-13.2,0.0,25296.91\nR004,2019-01-19,-16.8,0.0,17281.76\nR004,2019-01-20,-17.8,0.0,24042.22\nR004,2019-01-21,-16.0,0.0,26944.74\nR004,2019-01-22,-9.7,1.3,7436.59\nR004,2019-01-23,-13.0,0.0,28647.2\nR004,2019-01-24,-15.9,0.0,27555.79\nR004,2019-01-25,-15.3,2.0,8330.5\nR004,2019-01-26,-18.2,0.0,25973.75\nR004,2019-01-27,-16.0,2.6,0.0\nR004,2019-01-28,-22.1,0.1,4566.02\nR004,2019-01-29,-20.0,0.0,22240.47\nR004,2019-01-30,-18.4,0.0,28698.74\nR004,2019-01-31,-16.4,0.0,4604.36\nR004,2019-02-01,-24.2,0.0,30895.35\nR004,2019-02-02,-29.7,0.0,30981.5\nR004,2019-02-03,-25.7,0.2,18778.85\nR004,2019-02-04,-29.3,0.0,31860.54\nR004,2019-02-05,-31.7,0.0,31977.89\nR004,2019-02-06,-28.0,0.0,29804.56\nR004,2019-02-07,-26.3,0.0,31276.33\nR004,2019-02-08,-30.3,0.0,32327.12\nR004,2019-02-09,-31.0,0.0,32400.0\nR004,2019-02-10,-27.1,0.0,32400.0\nR004,2019-02-11,-25.1,0.0,32400.0\nR004,2019-02-12,-20.8,0.0,32400.0\nR004,2019-02-13,-18.6,0.2,25399.0\nR004,2019-02-14,-13.2,1.8,1802.58\nR004,2019-02-15,-12.2,0.0,32821.74\nR004,2019-02-16,-13.5,0.0,31790.68\nR004,2019-02-17,-9.8,0.0,30577.51\nR004,2019-02-18,-10.3,0.0,28882.63\nR004,2019-02-19,-6.1,0.0,18376.77\nR004,2019-02-20,-6.8,0.0,33300.93\nR004,2019-02-21,-6.9,1.2,21284.21\nR004,2019-02-22,-9.4,0.2,33987.68\nR004,2019-02-23,-8.6,0.0,35838.58\nR004,2019-02-24,-9.7,0.0,36308.68\nR004,2019-02-25,-9.0,0.0,35579.41\nR004,2019-02-26,-9.5,0.0,30654.65\nR004,2019-02-27,-6.1,0.1,23648.9\nR004,2019-02-28,-3.0,1.2,14456.33\nR004,2019-03-01,-2.8,3.0,25801.25\nR004,2019-03-02,-3.2,0.0,25196.61\nR004,2019-03-03,-2.6,0.0,36000.0\nR004,2019-03-04,-0.3,0.0,34460.82\nR004,2019-03-05,-3.0,1.7,12397.41\nR004,2019-03-06,-3.2,0.0,37185.12\nR004,2019-03-07,-5.4,0.0,38475.9\nR004,2019-03-08,-6.4,0.0,39163.67\nR004,2019-03-09,-7.4,0.5,20776.32\nR004,2019-03-10,-7.6,0.0,39386.73\nR004,2019-03-11,-7.5,0.0,39634.57\nR004,2019-03-12,-4.0,0.0,25857.22\nR004,2019-03-13,-0.2,0.0,39975.53\nR004,2019-03-14,0.8,0.0,39902.76\nR004,2019-03-15,0.5,0.0,40596.47\nR004,2019-03-16,-0.3,1.9,23822.73\nR004,2019-03-17,-6.1,1.9,22298.98\nR004,2019-03-18,-9.3,0.2,35052.71\nR004,2019-03-19,-8.9,0.0,36592.3\nR004,2019-03-20,-6.7,0.0,37819.75\nR004,2019-03-21,-6.8,0.0,42288.69\nR004,2019-03-22,-3.3,0.0,42430.25\nR004,2019-03-23,0.1,0.0,42730.4\nR004,2019-03-24,1.2,0.0,40668.63\nR004,2019-03-25,-0.1,8.2,25960.82\nR004,2019-03-26,-2.0,0.0,43381.82\nR004,2019-03-27,-2.2,0.0,39961.09\nR004,2019-03-28,-0.9,0.0,42638.09\nR004,2019-03-29,0.3,0.0,37427.48\nR004,2019-03-30,1.9,4.9,37572.33\nR004,2019-03-31,3.4,1.8,24829.12\nR004,2019-04-01,2.1,1.7,34481.85\nR004,2019-04-02,2.5,0.0,44741.91\nR004,2019-04-03,3.3,0.0,35550.68\nR004,2019-04-04,1.5,1.7,36909.25\nR004,2019-04-05,0.4,11.2,0.0\nR004,2019-04-06,0.8,13.8,0.0\nR004,2019-04-07,-0.7,4.7,24932.9\nR004,2019-04-08,-1.9,0.0,46452.19\nR004,2019-04-09,-0.8,0.0,46878.38\nR004,2019-04-10,1.0,0.0,47306.44\nR004,2019-04-11,-0.4,0.0,47736.08\nR004,2019-04-12,0.5,0.0,48167.4\nR004,2019-04-13,4.6,0.0,48600.54\nR004,2019-04-14,4.9,5.3,48253.73\nR004,2019-04-15,0.3,17.5,0.0\nR004,2019-04-16,-0.6,9.6,15154.79\nR004,2019-04-17,-0.8,0.0,49485.73\nR004,2019-04-18,-3.4,9.1,22532.28\nR004,2019-04-19,-6.5,0.0,46346.14\nR004,2019-04-20,-2.9,1.7,1463.49\nR004,2019-04-21,0.1,3.3,35288.02\nR004,2019-04-22,0.4,0.2,47851.16\nR004,2019-04-23,1.0,0.0,50400.0\nR004,2019-04-24,3.1,0.0,44319.79\nR004,2019-04-25,1.5,1.0,19439.29\nR004,2019-04-26,3.9,5.7,12613.93\nR004,2019-04-27,1.5,20.8,2955.89\nR004,2019-04-28,1.4,3.2,19541.55\nR004,2019-04-29,7.9,0.6,44367.93\nR004,2019-04-30,13.8,0.0,50411.07\nR004,2019-05-01,3.1,6.2,12213.66\nR004,2019-05-02,1.0,0.0,47619.98\nR004,2019-05-03,3.7,0.0,51102.96\nR004,2019-05-04,6.7,0.0,51057.25\nR004,2019-05-05,8.9,0.0,51102.66\nR004,2019-05-06,11.3,0.0,51427.33\nR004,2019-05-07,12.9,0.0,51346.03\nR004,2019-05-08,13.9,0.2,50450.13\nR004,2019-05-09,10.6,0.0,51259.73\nR004,2019-05-10,4.7,0.0,52504.9\nR004,2019-05-11,5.6,0.0,41472.31\nR004,2019-05-12,5.4,9.1,14934.06\nR004,2019-05-13,1.7,2.3,48294.22\nR004,2019-05-14,0.6,1.0,50316.16\nR004,2019-05-15,0.4,3.3,36169.26\nR004,2019-05-16,1.5,2.6,48749.05\nR004,2019-05-17,3.2,0.8,53085.66\nR004,2019-05-18,6.5,0.0,37288.92\nR004,2019-05-19,12.3,0.0,51532.97\nR004,2019-05-20,13.1,2.8,34895.16\nR004,2019-05-21,10.4,17.0,0.0\nR004,2019-05-22,8.6,4.2,34193.6\nR004,2019-05-23,4.9,12.5,24065.54\nR004,2019-05-24,6.5,3.5,52471.34\nR004,2019-05-25,11.1,0.0,51998.86\nR004,2019-05-26,8.8,4.4,31793.86\nR004,2019-05-27,6.0,0.4,49236.05\nR004,2019-05-28,9.1,0.0,57589.75\nR004,2019-05-29,14.4,0.0,57544.71\nR004,2019-05-30,15.8,0.4,43200.0\nR004,2019-05-31,11.0,4.5,36074.22\nR004,2019-06-01,11.5,9.7,29530.52\nR004,2019-06-02,9.6,5.9,39141.11\nR004,2019-06-03,10.1,0.0,57600.0\nR004,2019-06-04,10.5,9.1,50323.73\nR004,2019-06-05,13.8,0.3,55619.74\nR004,2019-06-06,15.9,0.0,53848.28\nR004,2019-06-07,13.8,6.4,36141.29\nR004,2019-06-08,13.7,3.8,35369.21\nR004,2019-06-09,14.2,3.6,31979.98\nR004,2019-06-10,13.5,6.3,34036.7\nR004,2019-06-11,14.8,0.0,44545.3\nR004,2019-06-12,17.0,0.0,57600.0\nR004,2019-06-13,18.0,0.0,48511.26\nR004,2019-06-14,17.5,1.6,57600.0\nR004,2019-06-15,16.1,4.7,34870.69\nR004,2019-06-16,16.4,3.6,44909.05\nR004,2019-06-17,10.4,6.3,49795.53\nR004,2019-06-18,14.6,0.0,52420.79\nR004,2019-06-19,14.1,3.2,44762.87\nR004,2019-06-20,15.0,0.2,57600.0\nR004,2019-06-21,16.7,0.0,57600.0\nR004,2019-06-22,16.7,2.9,49836.32\nR004,2019-06-23,14.2,8.8,40727.79\nR004,2019-06-24,10.7,8.4,51948.53\nR004,2019-06-25,10.5,5.7,36186.27\nR004,2019-06-26,13.3,0.0,55501.22\nR004,2019-06-27,18.3,0.0,57600.0\nR004,2019-06-28,20.1,0.0,55667.57\nR004,2019-06-29,20.6,0.0,52263.54\nR004,2019-06-30,12.9,7.7,0.0\nR004,2019-07-01,10.3,9.7,0.0\nR004,2019-07-02,11.9,1.4,20548.52\nR004,2019-07-03,14.2,0.0,29086.8\nR004,2019-07-04,16.9,0.0,44509.42\nR004,2019-07-05,18.6,5.8,51564.65\nR004,2019-07-06,18.7,0.3,56109.95\nR004,2019-07-07,19.3,0.7,49805.38\nR004,2019-07-08,19.6,0.0,55390.35\nR004,2019-07-09,19.7,0.1,54085.32\nR004,2019-07-10,16.6,7.0,25239.18\nR004,2019-07-11,16.3,0.0,48001.79\nR004,2019-07-12,17.7,0.0,53504.33\nR004,2019-07-13,17.2,1.7,27042.98\nR004,2019-07-14,18.4,0.0,40227.05\nR004,2019-07-15,17.3,0.5,54765.55\nR004,2019-07-16,14.5,1.8,49350.93\nR004,2019-07-17,15.0,0.7,27797.28\nR004,2019-07-18,16.4,6.0,42053.9\nR004,2019-07-19,15.4,14.1,19514.64\nR004,2019-07-20,16.5,7.1,45905.64\nR004,2019-07-21,15.7,2.1,41630.2\nR004,2019-07-22,15.0,0.0,53987.67\nR004,2019-07-23,16.1,0.0,42785.49\nR004,2019-07-24,15.4,6.3,19722.65\nR004,2019-07-25,15.8,0.3,39743.06\nR004,2019-07-26,16.1,1.9,52607.95\nR004,2019-07-27,15.4,0.7,42586.52\nR004,2019-07-28,16.3,0.1,49928.37\nR004,2019-07-29,17.7,0.0,52240.91\nR004,2019-07-30,19.7,0.0,50400.0\nR004,2019-07-31,19.5,5.4,34008.1\nR004,2019-08-01,19.4,0.0,48124.65\nR004,2019-08-02,21.3,0.0,50722.84\nR004,2019-08-03,22.2,0.0,53500.67\nR004,2019-08-04,22.7,0.0,53113.29\nR004,2019-08-05,18.2,3.5,31801.14\nR004,2019-08-06,16.4,2.5,2376.53\nR004,2019-08-07,18.2,0.4,41479.43\nR004,2019-08-08,19.4,0.2,47054.52\nR004,2019-08-09,16.1,13.9,23427.02\nR004,2019-08-10,14.1,2.5,42481.12\nR004,2019-08-11,13.3,6.5,48991.39\nR004,2019-08-12,13.1,0.0,38400.0\nR004,2019-08-13,14.2,0.0,51029.43\nR004,2019-08-14,15.5,0.0,43200.0\nR004,2019-08-15,19.1,0.0,50400.0\nR004,2019-08-16,14.1,21.3,6640.73\nR004,2019-08-17,14.9,0.0,44659.86\nR004,2019-08-18,15.1,0.7,48520.07\nR004,2019-08-19,15.3,1.8,29682.43\nR004,2019-08-20,13.5,13.0,102.95\nR004,2019-08-21,13.5,0.3,46091.86\nR004,2019-08-22,14.5,0.1,45088.07\nR004,2019-08-23,11.4,19.5,7200.0\nR004,2019-08-24,11.3,9.7,19133.07\nR004,2019-08-25,14.7,0.4,40177.15\nR004,2019-08-26,16.6,0.0,49644.53\nR004,2019-08-27,17.3,0.0,49362.42\nR004,2019-08-28,15.4,3.1,24720.14\nR004,2019-08-29,16.6,0.0,48241.28\nR004,2019-08-30,18.6,0.2,37636.11\nR004,2019-08-31,18.8,1.0,47665.36\nR004,2019-09-01,14.6,2.7,45163.55\nR004,2019-09-02,14.3,1.4,46049.95\nR004,2019-09-03,12.9,2.6,44229.4\nR004,2019-09-04,9.7,1.9,23709.89\nR004,2019-09-05,9.7,0.0,45232.63\nR004,2019-09-06,10.0,0.0,43808.34\nR004,2019-09-07,10.0,0.2,4399.28\nR004,2019-09-08,10.8,0.0,37561.35\nR004,2019-09-09,10.5,2.4,32400.0\nR004,2019-09-10,9.7,5.6,40243.63\nR004,2019-09-11,8.1,13.3,14566.62\nR004,2019-09-12,5.7,7.5,11871.94\nR004,2019-09-13,6.7,9.9,7274.79\nR004,2019-09-14,7.3,9.9,14332.21\nR004,2019-09-15,10.1,5.6,27382.11\nR004,2019-09-16,11.2,3.2,43200.0\nR004,2019-09-17,10.5,0.2,43200.0\nR004,2019-09-18,11.1,8.4,6038.6\nR004,2019-09-19,14.4,0.0,38234.61\nR004,2019-09-20,14.9,0.4,43200.0\nR004,2019-09-21,11.3,0.5,38796.63\nR004,2019-09-22,10.5,0.0,43200.0\nR004,2019-09-23,11.9,1.3,31781.86\nR004,2019-09-24,10.6,0.0,41544.89\nR004,2019-09-25,8.7,1.7,41952.07\nR004,2019-09-26,4.7,8.4,14288.74\nR004,2019-09-27,3.6,0.0,36397.71\nR004,2019-09-28,5.7,3.4,0.0\nR004,2019-09-29,4.0,3.6,11317.41\nR004,2019-09-30,0.9,3.4,33922.54\nR004,2019-10-01,0.1,0.1,22166.71\nR004,2019-10-02,0.5,0.0,37762.6\nR004,2019-10-03,1.5,0.0,37119.14\nR004,2019-10-04,2.8,0.0,36708.26\nR004,2019-10-05,6.6,0.0,34889.33\nR004,2019-10-06,9.8,0.0,36000.0\nR004,2019-10-07,8.8,0.0,36000.0\nR004,2019-10-08,8.3,0.0,36000.0\nR004,2019-10-09,6.1,0.0,36000.0\nR004,2019-10-10,1.4,0.0,35903.81\nR004,2019-10-11,0.5,0.0,36000.0\nR004,2019-10-12,1.5,0.0,36000.0\nR004,2019-10-13,3.0,0.0,36000.0\nR004,2019-10-14,4.0,0.0,36000.0\nR004,2019-10-15,7.4,0.5,35683.93\nR004,2019-10-16,7.4,1.3,22206.86\nR004,2019-10-17,4.7,18.7,0.0\nR004,2019-10-18,0.0,7.8,10004.96\nR004,2019-10-19,-2.0,11.9,659.95\nR004,2019-10-20,-3.8,6.0,2150.99\nR004,2019-10-21,-6.7,2.0,585.3\nR004,2019-10-22,-11.9,0.0,31303.99\nR004,2019-10-23,-7.6,0.0,15663.42\nR004,2019-10-24,-1.7,0.0,13328.32\nR004,2019-10-25,0.6,17.7,0.0\nR004,2019-10-26,-1.8,16.7,0.0\nR004,2019-10-27,0.6,5.1,1029.51\nR004,2019-10-28,6.6,8.2,19763.47\nR004,2019-10-29,8.8,14.7,3600.0\nR004,2019-10-30,5.6,7.6,0.0\nR004,2019-10-31,5.7,0.0,29401.7\nR004,2019-11-01,6.1,3.0,23212.38\nR004,2019-11-02,2.4,7.7,0.0\nR004,2019-11-03,-4.8,2.6,28706.85\nR004,2019-11-04,-6.3,0.0,25200.0\nR004,2019-11-05,-1.9,0.9,0.0\nR004,2019-11-06,-0.9,0.0,24442.0\nR004,2019-11-07,-2.0,0.0,25200.0\nR004,2019-11-08,0.1,2.4,11080.72\nR004,2019-11-09,-1.7,13.1,0.0\nR004,2019-11-10,-13.7,0.0,16700.77\nR004,2019-11-11,-15.5,0.3,1717.25\nR004,2019-11-12,-18.1,4.8,0.0\nR004,2019-11-13,-5.7,21.5,0.0\nR004,2019-11-14,-9.1,8.8,0.0\nR004,2019-11-15,-19.2,0.0,22608.68\nR004,2019-11-16,-24.0,0.0,28519.77\nR004,2019-11-17,-16.6,1.6,8033.82\nR004,2019-11-18,-7.9,16.3,0.0\nR004,2019-11-19,-5.0,11.3,0.0\nR004,2019-11-20,-16.7,4.2,1697.29\nR004,2019-11-21,-26.2,0.0,26006.82\nR004,2019-11-22,-22.7,0.2,7.89\nR004,2019-11-23,-18.0,0.0,24940.65\nR004,2019-11-24,-19.1,0.0,18639.77\nR004,2019-11-25,-14.0,0.4,28028.9\nR004,2019-11-26,-18.5,0.0,26586.65\nR004,2019-11-27,-17.0,2.4,3665.27\nR004,2019-11-28,-13.9,0.9,5239.25\nR004,2019-11-29,-12.2,3.5,13763.36\nR004,2019-11-30,-9.0,3.1,0.0\nR004,2019-12-01,-8.5,0.0,25422.87\nR004,2019-12-02,-16.7,0.0,27714.85\nR004,2019-12-03,-18.2,0.0,27587.17\nR004,2019-12-04,-14.3,1.5,0.0\nR004,2019-12-05,-12.9,0.0,27352.46\nR004,2019-12-06,-8.4,4.3,1851.23\nR004,2019-12-07,-6.3,13.7,0.0\nR004,2019-12-08,-3.0,20.4,0.0\nR004,2019-12-09,-2.8,2.1,0.0\nR004,2019-12-10,-4.8,0.0,22652.19\nR004,2019-12-11,-8.1,0.0,17803.4\nR004,2019-12-12,-11.6,0.0,25343.48\nR004,2019-12-13,-13.4,0.7,11304.77\nR004,2019-12-14,-6.1,12.5,0.0\nR004,2019-12-15,-9.4,0.1,3676.37\nR004,2019-12-16,-9.7,0.0,1917.19\nR004,2019-12-17,-11.2,3.4,0.0\nR004,2019-12-18,-12.4,0.0,7897.78\nR004,2019-12-19,-16.2,0.0,22382.95\nR004,2019-12-20,-14.1,2.6,13192.8\nR004,2019-12-21,-11.8,0.0,0.0\nR004,2019-12-22,-7.5,6.3,4014.97\nR004,2019-12-23,-12.4,0.0,24389.76\nR004,2019-12-24,-15.6,1.0,8541.34\nR004,2019-12-25,-8.7,17.8,0.0\nR004,2019-12-26,-8.3,15.8,0.0\nR004,2019-12-27,-15.7,1.7,20491.21\nR004,2019-12-28,-26.4,0.0,25744.84\nR004,2019-12-29,-28.1,0.0,22819.72\nR004,2019-12-30,-17.5,3.9,0.0\nR004,2019-12-31,-11.5,6.9,0.0\nR004,2020-01-01,-10.4,3.6,0.0\nR004,2020-01-02,-8.5,0.8,2349.67\nR004,2020-01-03,-11.3,0.0,25565.46\nR004,2020-01-04,-8.7,1.3,0.0\nR004,2020-01-05,-7.3,0.0,18317.06\nR004,2020-01-06,-13.6,0.0,26259.99\nR004,2020-01-07,-18.9,0.0,26240.43\nR004,2020-01-08,-16.5,0.0,0.0\nR004,2020-01-09,-16.6,0.3,16268.72\nR004,2020-01-10,-19.0,0.0,16403.03\nR004,2020-01-11,-21.6,0.0,26143.29\nR004,2020-01-12,-22.2,0.0,13951.65\nR004,2020-01-13,-16.0,1.0,108.47\nR004,2020-01-14,-11.4,2.4,4156.43\nR004,2020-01-15,-11.5,0.0,6368.36\nR004,2020-01-16,-10.2,0.1,11012.31\nR004,2020-01-17,-8.9,2.8,13985.77\nR004,2020-01-18,-7.7,12.9,1993.41\nR004,2020-01-19,-10.3,6.4,0.0\nR004,2020-01-20,-14.3,1.6,18849.73\nR004,2020-01-21,-18.2,0.0,0.0\nR004,2020-01-22,-15.5,0.3,8833.84\nR004,2020-01-23,-10.1,4.1,0.0\nR004,2020-01-24,-5.7,2.2,0.0\nR004,2020-01-25,-8.0,1.6,379.64\nR004,2020-01-26,-6.9,5.4,2793.89\nR004,2020-01-27,-14.1,0.0,29194.93\nR004,2020-01-28,-11.0,1.0,4106.54\nR004,2020-01-29,-11.0,2.6,9548.78\nR004,2020-01-30,-18.3,0.1,27739.68\nR004,2020-01-31,-19.9,0.0,31334.94\nR004,2020-02-01,-22.0,0.0,31479.07\nR004,2020-02-02,-20.9,0.0,31594.04\nR004,2020-02-03,-20.1,0.0,31708.71\nR004,2020-02-04,-13.6,3.6,0.0\nR004,2020-02-05,-8.9,1.3,3570.65\nR004,2020-02-06,-9.8,0.0,32057.39\nR004,2020-02-07,-4.9,5.1,0.0\nR004,2020-02-08,-1.6,2.2,6578.6\nR004,2020-02-09,2.7,0.0,10653.98\nR004,2020-02-10,-1.8,8.0,3050.24\nR004,2020-02-11,-11.4,0.8,19554.2\nR004,2020-02-12,-15.3,0.0,32400.0\nR004,2020-02-13,-13.5,0.0,32545.85\nR004,2020-02-14,-11.4,0.0,32899.12\nR004,2020-02-15,-12.1,0.0,29870.93\nR004,2020-02-16,-13.4,0.0,31842.03\nR004,2020-02-17,-12.3,0.0,25849.0\nR004,2020-02-18,-10.2,0.2,11409.42\nR004,2020-02-19,-9.0,1.9,0.0\nR004,2020-02-20,-5.0,6.3,0.0\nR004,2020-02-21,-2.3,0.8,21908.85\nR004,2020-02-22,-3.0,2.9,0.0\nR004,2020-02-23,-5.8,2.1,7018.29\nR004,2020-02-24,-8.2,0.0,29587.06\nR004,2020-02-25,-9.6,0.0,26838.85\nR004,2020-02-26,-9.4,0.0,37218.76\nR004,2020-02-27,-10.3,2.6,4028.11\nR004,2020-02-28,-3.8,11.6,4471.11\nR004,2020-02-29,-2.2,6.3,0.0\nR004,2020-03-01,-1.2,0.0,26357.31\nR004,2020-03-02,-4.3,0.0,36181.69\nR004,2020-03-03,-3.7,0.0,6983.45\nR004,2020-03-04,-3.4,6.5,12803.86\nR004,2020-03-05,-3.6,0.7,11739.85\nR004,2020-03-06,-6.7,2.7,18495.4\nR004,2020-03-07,-12.9,0.8,32873.71\nR004,2020-03-08,-12.2,4.5,0.0\nR004,2020-03-09,-6.0,5.6,655.42\nR004,2020-03-10,-4.9,11.8,1077.59\nR004,2020-03-11,-8.2,0.4,32769.59\nR004,2020-03-12,-6.7,0.0,39992.88\nR004,2020-03-13,-3.8,0.0,40305.27\nR004,2020-03-14,-2.1,0.0,24000.15\nR004,2020-03-15,1.8,4.6,0.0\nR004,2020-03-16,3.9,0.0,15471.16\nR004,2020-03-17,1.0,0.0,34062.89\nR004,2020-03-18,0.7,3.2,18850.42\nR004,2020-03-19,-2.0,4.0,11833.65\nR004,2020-03-20,-1.3,1.1,6644.61\nR004,2020-03-21,0.1,0.0,42395.45\nR004,2020-03-22,0.4,2.4,1410.16\nR004,2020-03-23,-2.3,0.5,39521.34\nR004,2020-03-24,-9.7,0.1,42095.15\nR004,2020-03-25,-11.9,0.0,42985.4\nR004,2020-03-26,-7.0,0.0,43839.8\nR004,2020-03-27,-3.9,0.0,43200.0\nR004,2020-03-28,-2.0,3.1,18612.99\nR004,2020-03-29,-3.1,0.5,43760.6\nR004,2020-03-30,-1.6,0.0,44473.28\nR004,2020-03-31,-0.3,0.0,44804.82\nR004,2020-04-01,1.6,0.0,45455.14\nR004,2020-04-02,2.1,0.0,45561.02\nR004,2020-04-03,5.3,0.0,38816.15\nR004,2020-04-04,4.1,0.0,40708.5\nR004,2020-04-05,2.3,0.0,45984.08\nR004,2020-04-06,3.8,0.0,46407.83\nR004,2020-04-07,4.8,0.0,46090.73\nR004,2020-04-08,2.4,1.2,34803.29\nR004,2020-04-09,0.2,6.0,10720.95\nR004,2020-04-10,-0.8,7.3,18635.65\nR004,2020-04-11,-1.1,0.0,48158.34\nR004,2020-04-12,1.1,0.0,48590.74\nR004,2020-04-13,2.7,0.7,42685.37\nR004,2020-04-14,4.4,0.0,38991.24\nR004,2020-04-15,5.1,1.4,44029.44\nR004,2020-04-16,4.3,2.4,46881.66\nR004,2020-04-17,4.1,0.2,46726.27\nR004,2020-04-18,2.9,0.0,49920.82\nR004,2020-04-19,5.3,0.0,50053.04\nR004,2020-04-20,6.2,0.0,50183.81\nR004,2020-04-21,8.9,0.0,50313.0\nR004,2020-04-22,9.7,0.0,50400.0\nR004,2020-04-23,13.4,0.0,50400.0\nR004,2020-04-24,18.1,0.0,49141.82\nR004,2020-04-25,15.6,0.0,50400.0\nR004,2020-04-26,7.9,3.6,806.49\nR004,2020-04-27,10.2,0.1,33394.29\nR004,2020-04-28,12.1,0.7,29919.23\nR004,2020-04-29,16.7,2.2,43766.2\nR004,2020-04-30,14.3,3.8,43688.8\nR004,2020-05-01,9.4,6.4,32692.64\nR004,2020-05-02,12.2,1.0,50348.86\nR004,2020-05-03,11.4,0.0,45319.4\nR004,2020-05-04,7.7,4.8,26959.42\nR004,2020-05-05,3.1,2.6,44017.26\nR004,2020-05-06,4.6,1.5,49308.07\nR004,2020-05-07,8.5,8.5,33064.46\nR004,2020-05-08,9.1,0.2,48501.85\nR004,2020-05-09,10.2,0.0,52356.23\nR004,2020-05-10,13.6,0.0,53132.01\nR004,2020-05-11,15.5,0.0,38394.48\nR004,2020-05-12,17.4,0.4,46947.98\nR004,2020-05-13,17.1,0.0,52024.33\nR004,2020-05-14,14.6,1.2,52465.01\nR004,2020-05-15,12.0,0.0,54928.43\nR004,2020-05-16,11.7,1.5,26587.41\nR004,2020-05-17,16.3,0.0,55483.94\nR004,2020-05-18,17.8,0.0,55986.1\nR004,2020-05-19,17.7,0.0,52568.2\nR004,2020-05-20,17.4,0.8,55333.93\nR004,2020-05-21,15.1,0.3,44504.91\nR004,2020-05-22,11.6,3.5,0.0\nR004,2020-05-23,13.6,1.4,35785.49\nR004,2020-05-24,15.4,0.0,53528.38\nR004,2020-05-25,16.5,0.4,47343.01\nR004,2020-05-26,20.9,1.1,48257.2\nR004,2020-05-27,10.6,4.5,31212.64\nR004,2020-05-28,7.5,0.0,22250.32\nR004,2020-05-29,10.2,0.0,57600.0\nR004,2020-05-30,10.9,1.0,26661.79\nR004,2020-05-31,12.3,1.4,20975.35\nR004,2020-06-01,13.6,0.7,47276.92\nR004,2020-06-02,15.1,4.8,41520.77\nR004,2020-06-03,11.9,0.3,48197.37\nR004,2020-06-04,9.9,1.0,49370.42\nR004,2020-06-05,8.8,5.0,49847.68\nR004,2020-06-06,7.8,10.4,11422.66\nR004,2020-06-07,9.4,19.2,35808.11\nR004,2020-06-08,12.8,4.8,43446.59\nR004,2020-06-09,12.2,11.7,38462.9\nR004,2020-06-10,11.6,0.4,47291.74\nR004,2020-06-11,13.0,0.0,57600.0\nR004,2020-06-12,16.8,0.0,57600.0\nR004,2020-06-13,18.8,0.0,57600.0\nR004,2020-06-14,20.1,0.4,57600.0\nR004,2020-06-15,14.5,7.5,37551.69\nR004,2020-06-16,13.4,0.0,56021.23\nR004,2020-06-17,15.3,0.0,57600.0\nR004,2020-06-18,13.4,9.3,32645.44\nR004,2020-06-19,14.2,4.1,47876.67\nR004,2020-06-20,9.3,17.1,0.0\nR004,2020-06-21,7.1,14.0,0.0\nR004,2020-06-22,9.7,0.5,15029.6\nR004,2020-06-23,15.6,0.0,57600.0\nR004,2020-06-24,17.7,0.9,47265.94\nR004,2020-06-25,17.0,8.1,32642.67\nR004,2020-06-26,18.3,0.0,47979.17\nR004,2020-06-27,19.9,0.0,57600.0\nR004,2020-06-28,16.7,1.3,38138.58\nR004,2020-06-29,12.1,8.2,16011.77\nR004,2020-06-30,14.3,0.3,30000.46\nR004,2020-07-01,16.8,2.5,31670.12\nR004,2020-07-02,19.0,0.4,45724.23\nR004,2020-07-03,19.6,0.2,54625.78\nR004,2020-07-04,17.6,2.3,53338.1\nR004,2020-07-05,18.4,0.0,46383.19\nR004,2020-07-06,17.7,1.4,44522.69\nR004,2020-07-07,16.2,10.4,32959.66\nR004,2020-07-08,16.3,5.1,30400.15\nR004,2020-07-09,16.1,4.0,24849.42\nR004,2020-07-10,17.7,14.9,28973.71\nR004,2020-07-11,15.0,0.2,40347.33\nR004,2020-07-12,16.9,0.0,47762.06\nR004,2020-07-13,17.5,0.0,35857.53\nR004,2020-07-14,19.0,0.0,45640.42\nR004,2020-07-15,19.2,0.2,56123.44\nR004,2020-07-16,16.9,0.0,54374.76\nR004,2020-07-17,20.0,0.0,54804.28\nR004,2020-07-18,19.0,0.0,55750.28\nR004,2020-07-19,17.9,0.0,55780.9\nR004,2020-07-20,18.7,0.0,55358.44\nR004,2020-07-21,19.1,0.6,50986.2\nR004,2020-07-22,16.4,0.2,53045.08\nR004,2020-07-23,16.4,0.3,54000.0\nR004,2020-07-24,17.3,0.0,52649.94\nR004,2020-07-25,17.3,18.0,41973.15\nR004,2020-07-26,16.9,14.6,38522.26\nR004,2020-07-27,16.2,6.2,45711.28\nR004,2020-07-28,16.7,1.3,53109.19\nR004,2020-07-29,16.3,2.2,40399.91\nR004,2020-07-30,14.2,10.2,23531.77\nR004,2020-07-31,14.5,0.1,48325.25\nR004,2020-08-01,15.3,0.1,42667.36\nR004,2020-08-02,15.6,0.0,47880.32\nR004,2020-08-03,17.0,0.0,52268.92\nR004,2020-08-04,19.4,0.0,53140.89\nR004,2020-08-05,17.6,0.1,51492.04\nR004,2020-08-06,16.2,0.1,52381.98\nR004,2020-08-07,15.9,3.6,50794.12\nR004,2020-08-08,16.5,0.0,44856.52\nR004,2020-08-09,18.5,0.0,51271.65\nR004,2020-08-10,18.8,0.6,45494.13\nR004,2020-08-11,18.0,1.0,50400.0\nR004,2020-08-12,17.3,7.6,40055.88\nR004,2020-08-13,17.7,2.3,50663.35\nR004,2020-08-14,16.1,2.9,50190.12\nR004,2020-08-15,14.0,9.1,1465.91\nR004,2020-08-16,15.9,3.0,46173.3\nR004,2020-08-17,17.5,0.3,47560.62\nR004,2020-08-18,18.5,1.4,43054.21\nR004,2020-08-19,15.6,12.6,38022.06\nR004,2020-08-20,16.7,0.3,50172.73\nR004,2020-08-21,16.3,0.4,42057.87\nR004,2020-08-22,14.6,2.2,49973.03\nR004,2020-08-23,12.5,5.7,38498.2\nR004,2020-08-24,13.1,3.0,48478.32\nR004,2020-08-25,12.4,8.8,26392.6\nR004,2020-08-26,12.2,19.3,28837.81\nR004,2020-08-27,13.7,0.7,44281.81\nR004,2020-08-28,12.5,13.9,4681.32\nR004,2020-08-29,13.2,9.6,27854.33\nR004,2020-08-30,13.6,0.0,41470.83\nR004,2020-08-31,14.4,0.2,46383.36\nR004,2020-09-01,15.2,0.0,44234.01\nR004,2020-09-02,15.7,4.2,33370.36\nR004,2020-09-03,17.9,1.4,43934.4\nR004,2020-09-04,11.5,14.9,0.0\nR004,2020-09-05,7.3,3.0,43887.98\nR004,2020-09-06,7.8,0.8,23665.48\nR004,2020-09-07,8.0,5.9,29509.37\nR004,2020-09-08,6.4,2.8,14938.62\nR004,2020-09-09,5.9,5.4,27393.75\nR004,2020-09-10,5.2,7.3,37583.35\nR004,2020-09-11,6.1,5.4,40262.29\nR004,2020-09-12,6.7,0.0,43221.08\nR004,2020-09-13,7.5,0.0,43200.0\nR004,2020-09-14,8.2,0.0,43200.0\nR004,2020-09-15,8.9,0.0,40933.3\nR004,2020-09-16,11.4,0.0,43200.0\nR004,2020-09-17,12.5,0.8,15249.09\nR004,2020-09-18,10.6,3.4,18253.0\nR004,2020-09-19,7.9,4.0,20783.88\nR004,2020-09-20,10.9,0.0,43200.0\nR004,2020-09-21,10.9,0.0,43200.0\nR004,2020-09-22,9.7,0.5,32532.02\nR004,2020-09-23,5.1,7.5,10472.02\nR004,2020-09-24,4.7,7.7,3078.72\nR004,2020-09-25,3.1,9.3,4381.95\nR004,2020-09-26,3.8,6.0,10890.16\nR004,2020-09-27,3.9,0.0,38385.37\nR004,2020-09-28,3.7,0.0,39650.56\nR004,2020-09-29,3.7,1.6,18488.09\nR004,2020-09-30,2.2,10.1,0.0\nR004,2020-10-01,0.3,4.9,0.0\nR004,2020-10-02,1.4,0.0,22008.28\nR004,2020-10-03,0.6,0.0,36822.49\nR004,2020-10-04,2.9,0.0,36362.2\nR004,2020-10-05,4.2,7.0,6854.15\nR004,2020-10-06,-1.3,7.6,6622.95\nR004,2020-10-07,-0.0,2.5,17930.59\nR004,2020-10-08,-0.4,10.4,0.0\nR004,2020-10-09,-3.9,1.0,9506.2\nR004,2020-10-10,-3.1,2.2,2963.71\nR004,2020-10-11,-0.5,7.6,0.0\nR004,2020-10-12,0.4,3.0,98.36\nR004,2020-10-13,2.1,21.3,0.0\nR004,2020-10-14,2.7,0.9,8108.18\nR004,2020-10-15,4.7,0.0,16643.46\nR004,2020-10-16,5.4,5.6,0.0\nR004,2020-10-17,6.6,0.0,33286.06\nR004,2020-10-18,6.5,6.3,2657.26\nR004,2020-10-19,-1.3,1.1,22474.28\nR004,2020-10-20,0.9,0.0,7552.5\nR004,2020-10-21,4.6,0.0,31270.32\nR004,2020-10-22,4.7,1.9,32520.68\nR004,2020-10-23,1.7,3.5,9908.95\nR004,2020-10-24,-1.5,4.2,5369.27\nR004,2020-10-25,-3.9,0.8,30443.81\nR004,2020-10-26,-0.0,2.7,0.0\nR004,2020-10-27,2.5,0.0,18957.18\nR004,2020-10-28,0.0,2.5,1694.15\nR004,2020-10-29,-2.9,2.7,22283.51\nR004,2020-10-30,-2.3,12.6,0.0\nR004,2020-10-31,-1.1,7.9,0.0\nR004,2020-11-01,-2.3,0.0,29952.48\nR004,2020-11-02,-3.7,0.0,24011.67\nR004,2020-11-03,-1.4,4.9,0.0\nR004,2020-11-04,-3.8,3.6,10389.32\nR004,2020-11-05,-2.2,9.9,0.0\nR004,2020-11-06,0.8,1.3,14400.0\nR004,2020-11-07,2.5,1.3,23001.6\nR004,2020-11-08,0.4,0.0,29091.61\nR004,2020-11-09,1.0,3.0,247.14\nR004,2020-11-10,0.3,4.3,854.81\nR004,2020-11-11,-0.4,0.6,28800.0\nR004,2020-11-12,0.1,5.1,0.0\nR004,2020-11-13,-4.0,1.9,24036.43\nR004,2020-11-14,-6.2,0.0,28277.24\nR004,2020-11-15,-6.7,0.0,28800.0\nR004,2020-11-16,-6.6,2.5,16786.36\nR004,2020-11-17,-8.1,2.2,19257.82\nR004,2020-11-18,-13.9,0.0,25835.17\nR004,2020-11-19,-14.6,0.0,5338.67\nR004,2020-11-20,-9.1,0.4,19568.66\nR004,2020-11-21,-7.0,4.2,0.0\nR004,2020-11-22,-3.9,1.5,11858.41\nR004,2020-11-23,-5.4,0.4,301.65\nR004,2020-11-24,-6.4,0.2,6551.71\nR004,2020-11-25,-10.8,0.0,22867.88\nR004,2020-11-26,-15.0,0.0,28348.65\nR004,2020-11-27,-15.6,0.0,28290.79\nR004,2020-11-28,-18.0,0.0,28170.57\nR004,2020-11-29,-18.3,0.0,28028.98\nR004,2020-11-30,-11.3,0.0,900.0\nR004,2020-12-01,-13.1,0.0,0.0\nR004,2020-12-02,-15.8,0.0,12329.98\nR004,2020-12-03,-20.9,0.0,26992.55\nR004,2020-12-04,-21.9,0.0,24895.84\nR004,2020-12-05,-20.3,0.0,24960.8\nR004,2020-12-06,-19.7,0.0,23344.55\nR004,2020-12-07,-19.8,0.0,24478.22\nR004,2020-12-08,-11.2,10.4,0.0\nR004,2020-12-09,-12.7,4.5,15801.86\nR004,2020-12-10,-12.9,1.7,0.0\nR004,2020-12-11,-11.0,5.7,1129.01\nR004,2020-12-12,-13.9,0.0,19247.07\nR004,2020-12-13,-14.2,2.4,2616.24\nR004,2020-12-14,-10.3,2.1,0.0\nR004,2020-12-15,-7.8,3.8,0.0\nR004,2020-12-16,-11.3,3.4,1557.62\nR004,2020-12-17,-9.6,3.7,0.0\nR004,2020-12-18,-8.5,0.0,18292.23\nR004,2020-12-19,-11.5,0.0,15814.72\nR004,2020-12-20,-14.1,0.7,0.0\nR004,2020-12-21,-11.1,4.1,4929.98\nR004,2020-12-22,-13.6,3.0,1866.68\nR004,2020-12-23,-13.1,8.9,0.0\nR004,2020-12-24,-8.8,16.4,0.0\nR004,2020-12-25,-12.3,8.1,155.94\nR004,2020-12-26,-27.5,0.0,24542.01\nR004,2020-12-27,-34.6,0.0,25213.94\nR004,2020-12-28,-37.9,0.0,25790.68\nR004,2020-12-29,-32.8,0.0,25325.0\nR004,2020-12-30,-25.4,0.0,1754.1\nR004,2020-12-31,-18.1,1.6,0.0\nR004,2021-01-01,-21.3,0.0,65.4\nR004,2021-01-02,-25.8,0.0,23660.09\nR004,2021-01-03,-31.5,0.0,22341.65\nR004,2021-01-04,-29.2,0.0,0.0\nR004,2021-01-05,-27.8,0.0,25739.69\nR004,2021-01-06,-29.0,0.0,26248.93\nR004,2021-01-07,-26.1,0.0,25200.0\nR004,2021-01-08,-27.8,0.0,26711.64\nR004,2021-01-09,-28.7,0.0,26684.78\nR004,2021-01-10,-18.9,1.7,0.0\nR004,2021-01-11,-10.0,16.2,0.0\nR004,2021-01-12,-12.5,2.4,0.0\nR004,2021-01-13,-13.8,3.2,0.0\nR004,2021-01-14,-22.9,0.0,20922.12\nR004,2021-01-15,-23.9,0.0,27014.95\nR004,2021-01-16,-24.0,0.0,1846.98\nR004,2021-01-17,-15.6,0.0,20383.21\nR004,2021-01-18,-9.8,0.3,13146.65\nR004,2021-01-19,-12.5,1.0,24407.63\nR004,2021-01-20,-21.3,0.0,25200.0\nR004,2021-01-21,-14.5,4.4,0.0\nR004,2021-01-22,-9.6,7.9,0.0\nR004,2021-01-23,-14.9,2.6,16985.54\nR004,2021-01-24,-16.5,0.6,0.0\nR004,2021-01-25,-26.7,0.0,28292.19\nR004,2021-01-26,-24.9,0.0,22556.35\nR004,2021-01-27,-13.5,12.1,0.0\nR004,2021-01-28,-6.9,8.3,0.0\nR004,2021-01-29,-4.6,9.2,0.0\nR004,2021-01-30,-7.3,5.7,683.2\nR004,2021-01-31,-18.1,0.0,31440.46\nR004,2021-02-01,-21.1,0.0,25880.88\nR004,2021-02-02,-18.5,0.0,31674.01\nR004,2021-02-03,-8.6,9.1,606.04\nR004,2021-02-04,-3.0,20.9,0.0\nR004,2021-02-05,-8.5,0.0,19342.03\nR004,2021-02-06,-6.6,2.3,0.0\nR004,2021-02-07,-6.5,0.7,23948.09\nR004,2021-02-08,-3.2,1.2,23658.03\nR004,2021-02-09,-8.9,7.3,22605.54\nR004,2021-02-10,-16.8,0.2,24434.01\nR004,2021-02-11,-22.4,0.5,28800.0\nR004,2021-02-12,-25.7,0.0,32558.28\nR004,2021-02-13,-24.4,0.0,27110.66\nR004,2021-02-14,-15.3,0.6,19390.53\nR004,2021-02-15,-7.8,0.0,31203.03\nR004,2021-02-16,-6.6,0.2,30359.61\nR004,2021-02-17,-2.4,8.1,0.0\nR004,2021-02-18,-6.3,16.0,0.0\nR004,2021-02-19,-3.2,10.9,0.0\nR004,2021-02-20,-13.5,4.5,19360.68\nR004,2021-02-21,-15.2,0.0,34919.65\nR004,2021-02-22,-10.8,4.7,0.0\nR004,2021-02-23,-14.7,0.0,31527.99\nR004,2021-02-24,-14.8,1.0,5955.17\nR004,2021-02-25,-13.2,5.6,1218.59\nR004,2021-02-26,-20.3,1.3,17577.66\nR004,2021-02-27,-20.6,0.0,29067.99\nR004,2021-02-28,-18.7,0.0,33639.35\nR004,2021-03-01,-10.3,13.3,0.0\nR004,2021-03-02,-4.2,7.2,16438.06\nR004,2021-03-03,-7.1,0.5,17944.9\nR004,2021-03-04,-8.7,0.0,38776.04\nR004,2021-03-05,-8.7,1.5,9069.01\nR004,2021-03-06,-4.0,0.0,31346.36\nR004,2021-03-07,-4.0,0.8,9604.86\nR004,2021-03-08,-4.5,0.7,39222.1\nR004,2021-03-09,-4.1,2.9,33872.31\nR004,2021-03-10,-3.9,0.0,32242.14\nR004,2021-03-11,-4.1,2.8,223.07\nR004,2021-03-12,-6.5,4.5,15177.23\nR004,2021-03-13,-10.9,12.8,10800.0\nR004,2021-03-14,-15.8,0.0,40255.14\nR004,2021-03-15,-13.3,0.0,28136.16\nR004,2021-03-16,-13.1,0.0,36569.12\nR004,2021-03-17,-12.7,0.0,41788.43\nR004,2021-03-18,-12.3,0.6,26867.97\nR004,2021-03-19,-8.0,0.4,37795.17\nR004,2021-03-20,-7.2,0.0,42223.85\nR004,2021-03-21,-2.6,0.0,11728.29\nR004,2021-03-22,1.6,1.4,31364.53\nR004,2021-03-23,1.9,0.4,17292.68\nR004,2021-03-24,-1.2,2.5,18193.42\nR004,2021-03-25,-2.2,0.0,36000.0\nR004,2021-03-26,-2.9,6.4,22650.11\nR004,2021-03-27,-3.1,0.0,43200.0\nR004,2021-03-28,-0.1,0.3,872.77\nR004,2021-03-29,-0.2,5.3,18581.76\nR004,2021-03-30,-3.9,4.3,31811.41\nR004,2021-03-31,-3.9,1.7,34361.05\nR004,2021-04-01,-4.8,0.0,45151.26\nR004,2021-04-02,-3.5,0.0,45484.55\nR004,2021-04-03,-0.2,0.0,39755.52\nR004,2021-04-04,1.1,0.0,45745.61\nR004,2021-04-05,1.7,0.0,45974.97\nR004,2021-04-06,3.2,0.8,44570.67\nR004,2021-04-07,2.4,5.9,260.04\nR004,2021-04-08,1.2,0.1,36773.38\nR004,2021-04-09,1.1,0.2,46279.1\nR004,2021-04-10,1.2,0.0,47327.01\nR004,2021-04-11,1.9,0.0,48145.49\nR004,2021-04-12,3.4,0.0,42448.64\nR004,2021-04-13,1.2,1.7,23192.65\nR004,2021-04-14,0.8,1.4,30263.21\nR004,2021-04-15,4.5,0.0,47024.67\nR004,2021-04-16,4.9,0.0,49622.03\nR004,2021-04-17,0.9,2.6,41015.74\nR004,2021-04-18,-0.6,0.0,39791.62\nR004,2021-04-19,0.5,3.9,25916.41\nR004,2021-04-20,-6.3,0.0,50143.2\nR004,2021-04-21,-6.6,0.0,39981.42\nR004,2021-04-22,-2.7,0.0,47160.62\nR004,2021-04-23,-0.3,0.0,40568.87\nR004,2021-04-24,-0.1,0.0,48011.81\nR004,2021-04-25,1.7,0.6,38243.2\nR004,2021-04-26,3.3,0.0,47353.24\nR004,2021-04-27,4.3,9.7,24309.65\nR004,2021-04-28,6.7,16.2,1578.9\nR004,2021-04-29,6.9,0.9,37466.94\nR004,2021-04-30,5.1,13.3,0.0\nR004,2021-05-01,3.2,1.4,26626.13\nR004,2021-05-02,2.3,3.3,31959.95\nR004,2021-05-03,3.5,0.0,50656.86\nR004,2021-05-04,4.5,3.2,37647.28\nR004,2021-05-05,4.8,0.0,46800.0\nR004,2021-05-06,11.9,0.0,44044.99\nR004,2021-05-07,13.0,1.5,3600.0\nR004,2021-05-08,10.7,0.0,51354.32\nR004,2021-05-09,10.5,0.9,52097.44\nR004,2021-05-10,7.9,0.0,52297.51\nR004,2021-05-11,10.8,0.0,43858.46\nR004,2021-05-12,11.4,0.0,32204.74\nR004,2021-05-13,11.4,0.0,51177.59\nR004,2021-05-14,9.8,5.3,50881.71\nR004,2021-05-15,11.3,0.2,51884.86\nR004,2021-05-16,15.2,1.0,51156.69\nR004,2021-05-17,13.0,12.9,36491.42\nR004,2021-05-18,13.2,7.7,33981.32\nR004,2021-05-19,10.3,11.0,12884.91\nR004,2021-05-20,3.9,8.7,33070.13\nR004,2021-05-21,3.7,0.2,48692.73\nR004,2021-05-22,5.7,1.0,22776.75\nR004,2021-05-23,12.0,0.3,38547.94\nR004,2021-05-24,15.1,4.6,26446.04\nR004,2021-05-25,15.3,0.3,49162.37\nR004,2021-05-26,19.2,0.3,55889.53\nR004,2021-05-27,18.4,1.5,49644.44\nR004,2021-05-28,11.6,7.9,52186.67\nR004,2021-05-29,7.9,0.7,54306.26\nR004,2021-05-30,10.0,0.0,54221.71\nR004,2021-05-31,14.4,0.0,57600.0\nR004,2021-06-01,15.1,0.0,57534.58\nR004,2021-06-02,17.7,0.0,57600.0\nR004,2021-06-03,19.9,0.8,51422.79\nR004,2021-06-04,16.2,21.5,25063.48\nR004,2021-06-05,8.0,27.2,27997.26\nR004,2021-06-06,7.3,3.1,22124.95\nR004,2021-06-07,14.8,0.0,57600.0\nR004,2021-06-08,17.3,0.7,56928.78\nR004,2021-06-09,16.3,0.4,57337.33\nR004,2021-06-10,12.0,0.0,57187.31\nR004,2021-06-11,12.6,0.0,52059.95\nR004,2021-06-12,10.5,0.1,18809.87\nR004,2021-06-13,12.0,0.2,55019.59\nR004,2021-06-14,11.1,6.1,16587.89\nR004,2021-06-15,13.0,0.2,45273.29\nR004,2021-06-16,15.3,0.2,57600.0\nR004,2021-06-17,17.5,0.0,53825.23\nR004,2021-06-18,19.7,0.0,57600.0\nR004,2021-06-19,20.2,1.2,56894.08\nR004,2021-06-20,13.0,3.3,9640.49\nR004,2021-06-21,11.7,0.0,39681.24\nR004,2021-06-22,12.7,1.2,30572.4\nR004,2021-06-23,11.1,4.4,50509.62\nR004,2021-06-24,10.3,5.1,28134.19\nR004,2021-06-25,10.4,2.5,54176.31\nR004,2021-06-26,12.6,1.0,48927.9\nR004,2021-06-27,12.9,0.3,53973.39\nR004,2021-06-28,14.0,0.0,38619.6\nR004,2021-06-29,15.0,3.5,54748.64\nR004,2021-06-30,16.2,0.0,48809.28\nR004,2021-07-01,16.0,5.7,31714.26\nR004,2021-07-02,16.7,0.0,57600.0\nR004,2021-07-03,18.0,0.0,57600.0\nR004,2021-07-04,20.9,0.0,57600.0\nR004,2021-07-05,23.4,0.0,57600.0\nR004,2021-07-06,23.0,0.7,41380.29\nR004,2021-07-07,19.5,4.8,47166.23\nR004,2021-07-08,14.3,0.0,56860.37\nR004,2021-07-09,15.4,0.0,55944.68\nR004,2021-07-10,16.9,0.0,46218.72\nR004,2021-07-11,17.3,3.7,38267.73\nR004,2021-07-12,15.6,7.7,28745.43\nR004,2021-07-13,17.6,2.4,49133.28\nR004,2021-07-14,18.9,0.3,43458.48\nR004,2021-07-15,18.5,3.7,55656.31\nR004,2021-07-16,17.1,6.2,47851.22\nR004,2021-07-17,14.7,3.6,35988.81\nR004,2021-07-18,18.0,0.9,54831.11\nR004,2021-07-19,17.2,9.3,52678.98\nR004,2021-07-20,17.2,0.1,51684.99\nR004,2021-07-21,17.0,1.9,32070.85\nR004,2021-07-22,19.7,0.0,50331.96\nR004,2021-07-23,19.2,0.0,18881.45\nR004,2021-07-24,19.2,1.7,30448.35\nR004,2021-07-25,18.0,16.7,24566.11\nR004,2021-07-26,20.2,2.6,45222.17\nR004,2021-07-27,16.3,13.6,28697.03\nR004,2021-07-28,15.2,0.0,52089.56\nR004,2021-07-29,15.5,0.1,50855.4\nR004,2021-07-30,15.7,0.3,38285.01\nR004,2021-07-31,16.8,0.0,53840.02\nR004,2021-08-01,16.6,0.0,53737.31\nR004,2021-08-02,18.7,0.0,53053.88\nR004,2021-08-03,18.4,2.9,47365.12\nR004,2021-08-04,16.5,8.2,39864.35\nR004,2021-08-05,15.5,2.5,45285.45\nR004,2021-08-06,14.1,11.0,22511.63\nR004,2021-08-07,16.4,3.6,40319.88\nR004,2021-08-08,17.4,0.0,51438.91\nR004,2021-08-09,19.1,0.0,47067.58\nR004,2021-08-10,17.6,0.1,37536.41\nR004,2021-08-11,16.7,1.9,50471.73\nR004,2021-08-12,16.5,5.2,45024.8\nR004,2021-08-13,15.2,0.6,14927.44\nR004,2021-08-14,14.8,4.6,14970.93\nR004,2021-08-15,14.5,0.0,9002.77\nR004,2021-08-16,15.9,0.0,39687.22\nR004,2021-08-17,16.4,0.0,43276.87\nR004,2021-08-18,17.1,0.0,50400.0\nR004,2021-08-19,16.8,0.4,50301.33\nR004,2021-08-20,12.6,14.1,27095.76\nR004,2021-08-21,8.2,6.6,34321.01\nR004,2021-08-22,9.8,0.0,49999.17\nR004,2021-08-23,11.4,0.0,49897.98\nR004,2021-08-24,13.7,0.0,49796.63\nR004,2021-08-25,15.5,0.0,49695.16\nR004,2021-08-26,16.4,0.0,49593.61\nR004,2021-08-27,16.3,0.0,49492.03\nR004,2021-08-28,16.0,0.0,48390.11\nR004,2021-08-29,16.5,0.0,47793.16\nR004,2021-08-30,16.0,3.0,42726.87\nR004,2021-08-31,12.4,2.4,1999.87\nR004,2021-09-01,12.0,0.0,21011.85\nR004,2021-09-02,11.1,0.0,42567.02\nR004,2021-09-03,10.7,0.0,46225.71\nR004,2021-09-04,11.8,0.0,45951.23\nR004,2021-09-05,13.1,0.0,45677.13\nR004,2021-09-06,14.8,0.0,44704.44\nR004,2021-09-07,16.6,0.0,44387.1\nR004,2021-09-08,15.8,0.0,43229.71\nR004,2021-09-09,10.9,14.7,15172.84\nR004,2021-09-10,8.2,0.0,43444.49\nR004,2021-09-11,8.1,0.8,5766.75\nR004,2021-09-12,11.0,4.5,4996.77\nR004,2021-09-13,6.5,0.9,43200.0\nR004,2021-09-14,7.1,1.3,42929.73\nR004,2021-09-15,10.6,0.3,28800.0\nR004,2021-09-16,11.9,0.0,36086.81\nR004,2021-09-17,12.8,0.0,43200.0\nR004,2021-09-18,10.3,4.0,6403.14\nR004,2021-09-19,10.2,0.0,27098.0\nR004,2021-09-20,5.6,10.6,20076.48\nR004,2021-09-21,2.3,7.8,26083.92\nR004,2021-09-22,2.6,3.1,18832.85\nR004,2021-09-23,1.7,4.6,2338.61\nR004,2021-09-24,1.4,2.3,35344.92\nR004,2021-09-25,1.8,0.8,17630.19\nR004,2021-09-26,0.8,7.8,10806.23\nR004,2021-09-27,-0.9,2.3,8034.82\nR004,2021-09-28,0.9,0.0,35992.41\nR004,2021-09-29,0.6,2.2,20068.56\nR004,2021-09-30,4.2,8.5,11541.16\nR004,2021-10-01,2.4,10.1,1270.15\nR004,2021-10-02,1.5,5.8,6756.07\nR004,2021-10-03,0.0,1.8,33546.53\nR004,2021-10-04,0.9,0.1,18746.85\nR004,2021-10-05,2.0,0.5,26323.27\nR004,2021-10-06,2.9,0.9,30385.24\nR004,2021-10-07,-0.1,5.3,0.0\nR004,2021-10-08,-0.3,10.6,0.0\nR004,2021-10-09,4.2,0.7,34157.21\nR004,2021-10-10,5.1,0.0,36000.0\nR004,2021-10-11,3.0,4.5,0.0\nR004,2021-10-12,1.6,9.4,0.0\nR004,2021-10-13,-0.8,0.8,7275.09\nR004,2021-10-14,0.2,0.0,10798.07\nR004,2021-10-15,-0.0,0.0,36000.0\nR004,2021-10-16,-0.3,0.0,35849.82\nR004,2021-10-17,0.4,0.0,34714.9\nR004,2021-10-18,1.5,0.0,33422.83\nR004,2021-10-19,0.7,0.0,35153.33\nR004,2021-10-20,0.2,0.0,34822.68\nR004,2021-10-21,2.2,0.0,34078.46\nR004,2021-10-22,2.4,0.4,29510.66\nR004,2021-10-23,2.0,0.2,32400.0\nR004,2021-10-24,4.2,5.0,0.0\nR004,2021-10-25,6.0,0.1,19087.26\nR004,2021-10-26,5.1,0.0,32067.45\nR004,2021-10-27,1.1,2.3,1814.01\nR004,2021-10-28,-3.9,0.0,30194.9\nR004,2021-10-29,-1.2,0.0,27077.27\nR004,2021-10-30,1.4,1.9,14180.05\nR004,2021-10-31,0.3,4.5,1838.41\nR004,2021-11-01,-2.2,5.8,3468.37\nR004,2021-11-02,-2.2,20.6,0.0\nR004,2021-11-03,-18.1,0.4,24232.54\nR004,2021-11-04,-23.1,0.0,29699.19\nR004,2021-11-05,-20.0,0.0,28791.97\nR004,2021-11-06,-6.8,1.2,0.0\nR004,2021-11-07,-5.4,0.0,27428.06\nR004,2021-11-08,-1.3,0.0,29073.35\nR004,2021-11-09,-3.9,0.0,28800.0\nR004,2021-11-10,-6.4,0.0,29197.95\nR004,2021-11-11,-2.5,0.2,5310.65\nR004,2021-11-12,-1.9,0.0,397.55\nR004,2021-11-13,-0.8,2.5,347.95\nR004,2021-11-14,-2.2,3.1,1634.53\nR004,2021-11-15,-1.8,9.5,5328.67\nR004,2021-11-16,-6.4,2.6,9346.39\nR004,2021-11-17,-10.4,1.4,15381.36\nR004,2021-11-18,-8.9,5.1,893.24\nR004,2021-11-19,-13.0,1.9,0.0\nR004,2021-11-20,-14.6,0.1,5477.77\nR004,2021-11-21,-12.1,0.0,19738.31\nR004,2021-11-22,-10.9,0.2,26071.36\nR004,2021-11-23,-3.4,5.3,0.0\nR004,2021-11-24,-4.1,8.7,23968.22\nR004,2021-11-25,-2.1,7.6,0.0\nR004,2021-11-26,-7.5,13.4,3081.97\nR004,2021-11-27,-13.7,0.8,20358.82\nR004,2021-11-28,-6.3,3.3,0.0\nR004,2021-11-29,-10.3,0.0,22891.96\nR004,2021-11-30,-14.9,0.0,27051.08\nR004,2021-12-01,-13.9,0.0,27798.1\nR004,2021-12-02,-16.7,0.0,27665.78\nR004,2021-12-03,-13.8,0.3,21109.64\nR004,2021-12-04,-2.3,7.9,890.65\nR004,2021-12-05,-6.9,0.0,23860.08\nR004,2021-12-06,-12.4,0.0,23472.42\nR004,2021-12-07,-7.5,3.8,0.0\nR004,2021-12-08,-11.2,2.8,13154.71\nR004,2021-12-09,-18.4,0.0,19262.28\nR004,2021-12-10,-18.0,0.2,16402.02\nR004,2021-12-11,-14.7,0.0,22971.24\nR004,2021-12-12,-14.5,1.9,20668.74\nR004,2021-12-13,-14.1,3.4,2561.8\nR004,2021-12-14,-15.8,2.1,3159.92\nR004,2021-12-15,-18.5,0.4,20177.75\nR004,2021-12-16,-15.1,6.7,0.0\nR004,2021-12-17,-6.1,6.2,0.0\nR004,2021-12-18,-4.0,8.0,0.0\nR004,2021-12-19,-6.4,4.8,0.0\nR004,2021-12-20,-6.5,7.8,0.0\nR004,2021-12-21,-4.5,5.5,0.0\nR004,2021-12-22,-7.4,0.0,19862.39\nR004,2021-12-23,-13.3,0.0,25003.33\nR004,2021-12-24,-16.7,0.0,25577.67\nR004,2021-12-25,-15.3,0.0,25103.23\nR004,2021-12-26,-16.8,0.0,9856.73\nR004,2021-12-27,-16.5,0.1,12311.23\nR004,2021-12-28,-14.9,0.9,24637.02\nR004,2021-12-29,-17.5,0.0,22187.74\nR004,2021-12-30,-13.8,3.4,2613.98\nR004,2021-12-31,-20.2,0.0,23819.48\nR004,2022-01-01,-22.9,0.0,10805.26\nR004,2022-01-02,-23.3,0.0,26055.55\nR004,2022-01-03,-16.3,0.0,22560.07\nR004,2022-01-04,-18.2,0.0,26166.73\nR004,2022-01-05,-11.0,2.3,0.0\nR004,2022-01-06,-8.3,2.6,6431.28\nR004,2022-01-07,-13.2,0.0,25723.26\nR004,2022-01-08,-10.2,0.7,15195.12\nR004,2022-01-09,-12.3,1.0,22512.33\nR004,2022-01-10,-10.9,3.5,0.0\nR004,2022-01-11,-16.8,0.0,25200.0\nR004,2022-01-12,-19.8,0.0,20894.41\nR004,2022-01-13,-15.9,2.1,5019.43\nR004,2022-01-14,-20.1,0.0,24752.47\nR004,2022-01-15,-20.4,0.4,27474.13\nR004,2022-01-16,-11.7,2.1,23807.15\nR004,2022-01-17,-6.1,4.5,13608.95\nR004,2022-01-18,-6.2,3.4,12169.32\nR004,2022-01-19,-11.1,0.8,22162.63\nR004,2022-01-20,-16.2,0.0,27242.19\nR004,2022-01-21,-17.4,0.0,20319.53\nR004,2022-01-22,-21.4,0.0,28636.15\nR004,2022-01-23,-23.0,0.0,28653.58\nR004,2022-01-24,-21.3,1.4,19534.47\nR004,2022-01-25,-22.9,0.0,19663.42\nR004,2022-01-26,-20.9,0.0,3044.69\nR004,2022-01-27,-16.1,0.3,24470.64\nR004,2022-01-28,-19.8,0.0,30791.34\nR004,2022-01-29,-22.7,0.0,28282.77\nR004,2022-01-30,-22.4,0.0,31213.88\nR004,2022-01-31,-21.7,0.0,31411.54\nR004,2022-02-01,-17.4,0.0,19176.7\nR004,2022-02-02,-19.1,0.0,31647.98\nR004,2022-02-03,-18.0,0.0,31766.74\nR004,2022-02-04,-19.4,0.0,31885.68\nR004,2022-02-05,-19.8,0.0,32004.67\nR004,2022-02-06,-18.4,0.0,32123.58\nR004,2022-02-07,-17.3,0.0,32242.29\nR004,2022-02-08,-15.7,1.0,26241.95\nR004,2022-02-09,-15.3,1.0,32400.0\nR004,2022-02-10,-21.0,0.0,32400.0\nR004,2022-02-11,-25.2,0.0,32638.04\nR004,2022-02-12,-25.9,0.0,32577.45\nR004,2022-02-13,-22.8,0.0,32811.97\nR004,2022-02-14,-15.5,0.4,12083.95\nR004,2022-02-15,-14.1,1.5,13052.21\nR004,2022-02-16,-14.2,2.8,4595.17\nR004,2022-02-17,-14.9,1.7,22444.98\nR004,2022-02-18,-10.5,3.1,0.0\nR004,2022-02-19,-9.8,0.0,5499.59\nR004,2022-02-20,-11.8,0.0,34839.32\nR004,2022-02-21,-13.3,0.0,35312.99\nR004,2022-02-22,-11.5,0.0,33953.17\nR004,2022-02-23,-7.7,2.6,8425.07\nR004,2022-02-24,-4.9,1.0,22958.62\nR004,2022-02-25,-5.6,0.9,17856.43\nR004,2022-02-26,-5.0,8.2,14560.7\nR004,2022-02-27,-7.5,0.3,31457.63\nR004,2022-02-28,-7.7,0.0,35784.62\nR004,2022-03-01,-6.9,0.0,38329.27\nR004,2022-03-02,-7.6,7.4,2724.63\nR004,2022-03-03,-8.7,1.3,33836.67\nR004,2022-03-04,-8.0,0.0,32815.52\nR004,2022-03-05,-4.2,0.2,28137.91\nR004,2022-03-06,-3.7,0.4,9616.4\nR004,2022-03-07,0.1,6.0,0.0\nR004,2022-03-08,-2.8,7.4,5488.33\nR004,2022-03-09,-11.5,0.0,39302.48\nR004,2022-03-10,-12.7,0.0,34206.7\nR004,2022-03-11,-10.9,0.2,27480.77\nR004,2022-03-12,-12.9,0.0,30887.04\nR004,2022-03-13,-12.3,0.0,36532.61\nR004,2022-03-14,-10.7,1.7,30945.6\nR004,2022-03-15,-10.8,0.0,36575.97\nR004,2022-03-16,-9.2,1.9,11602.81\nR004,2022-03-17,-9.1,0.6,39600.0\nR004,2022-03-18,-11.1,0.0,41906.8\nR004,2022-03-19,-11.1,0.0,42050.94\nR004,2022-03-20,-10.2,0.0,39787.23\nR004,2022-03-21,-8.4,0.0,40174.63\nR004,2022-03-22,-5.8,2.9,28472.56\nR004,2022-03-23,-5.4,6.0,26643.32\nR004,2022-03-24,-5.7,4.5,4601.34\nR004,2022-03-25,-4.6,6.2,35371.13\nR004,2022-03-26,-3.5,0.0,41353.38\nR004,2022-03-27,-4.6,0.0,26585.1\nR004,2022-03-28,-3.2,2.3,13751.03\nR004,2022-03-29,0.6,0.4,42999.53\nR004,2022-03-30,-1.4,1.7,18909.56\nR004,2022-03-31,0.3,1.1,156.41\nR004,2022-04-01,1.7,0.0,35986.24\nR004,2022-04-02,3.6,1.6,43200.0\nR004,2022-04-03,1.9,18.1,0.0\nR004,2022-04-04,3.1,4.4,1174.7\nR004,2022-04-05,4.7,0.0,44194.16\nR004,2022-04-06,6.5,0.3,41581.06\nR004,2022-04-07,4.8,5.0,39763.53\nR004,2022-04-08,0.4,0.9,3338.17\nR004,2022-04-09,0.2,6.1,42054.81\nR004,2022-04-10,-0.4,0.7,29859.35\nR004,2022-04-11,-1.9,0.1,47355.61\nR004,2022-04-12,-1.7,0.0,48175.97\nR004,2022-04-13,0.4,0.0,48609.51\nR004,2022-04-14,2.0,0.0,49293.56\nR004,2022-04-15,2.3,0.0,49446.03\nR004,2022-04-16,5.8,0.0,49579.45\nR004,2022-04-17,6.2,0.0,49712.7\nR004,2022-04-18,9.7,0.0,49845.67\nR004,2022-04-19,3.0,5.8,23530.83\nR004,2022-04-20,1.2,1.6,37180.06\nR004,2022-04-21,2.3,18.2,9372.64\nR004,2022-04-22,1.2,9.8,26236.05\nR004,2022-04-23,-1.8,0.7,42541.0\nR004,2022-04-24,-1.0,0.2,49612.68\nR004,2022-04-25,1.9,0.0,50400.0\nR004,2022-04-26,4.3,0.0,50400.0\nR004,2022-04-27,6.8,0.4,41913.67\nR004,2022-04-28,9.1,0.0,50400.0\nR004,2022-04-29,12.5,0.0,50400.0\nR004,2022-04-30,14.7,0.0,43102.86\nR004,2022-05-01,16.7,0.0,50400.0\nR004,2022-05-02,15.2,2.2,31474.86\nR004,2022-05-03,9.3,0.3,50669.8\nR004,2022-05-04,5.1,0.0,50623.44\nR004,2022-05-05,7.9,0.0,32562.69\nR004,2022-05-06,10.8,0.0,51400.84\nR004,2022-05-07,8.4,3.8,21600.0\nR004,2022-05-08,2.9,0.1,51292.15\nR004,2022-05-09,3.4,0.0,46903.63\nR004,2022-05-10,5.1,0.0,53177.37\nR004,2022-05-11,7.2,0.0,53525.13\nR004,2022-05-12,12.1,0.0,53878.26\nR004,2022-05-13,13.3,0.0,53800.5\nR004,2022-05-14,15.5,0.0,54159.74\nR004,2022-05-15,17.2,0.1,44876.48\nR004,2022-05-16,15.4,6.6,51824.86\nR004,2022-05-17,16.9,0.0,55022.99\nR004,2022-05-18,18.8,0.0,52995.43\nR004,2022-05-19,19.8,0.0,56024.06\nR004,2022-05-20,15.9,3.8,51046.85\nR004,2022-05-21,16.9,0.0,56470.87\nR004,2022-05-22,18.1,0.1,35281.6\nR004,2022-05-23,13.8,1.8,47987.65\nR004,2022-05-24,14.7,0.0,57298.84\nR004,2022-05-25,20.1,0.0,55919.37\nR004,2022-05-26,21.2,0.0,57479.92\nR004,2022-05-27,18.4,0.2,56330.24\nR004,2022-05-28,17.2,0.0,57600.0\nR004,2022-05-29,16.8,4.6,41660.29\nR004,2022-05-30,16.2,0.4,51018.51\nR004,2022-05-31,18.1,0.0,53621.97\nR004,2022-06-01,13.2,11.2,6854.26\nR004,2022-06-02,7.3,16.6,13079.01\nR004,2022-06-03,8.7,2.0,49799.58\nR004,2022-06-04,6.8,12.6,992.21\nR004,2022-06-05,8.3,7.6,31294.78\nR004,2022-06-06,7.3,7.4,41907.63\nR004,2022-06-07,9.0,0.0,57354.19\nR004,2022-06-08,11.0,0.6,57600.0\nR004,2022-06-09,11.3,0.0,57600.0\nR004,2022-06-10,16.4,0.0,57600.0\nR004,2022-06-11,20.0,0.0,54001.27\nR004,2022-06-12,19.4,0.3,54668.34\nR004,2022-06-13,18.1,4.5,32620.83\nR004,2022-06-14,17.7,8.1,20861.56\nR004,2022-06-15,18.1,12.9,47372.22\nR004,2022-06-16,13.4,13.1,13106.62\nR004,2022-06-17,12.9,7.5,52502.75\nR004,2022-06-18,14.4,5.2,49964.35\nR004,2022-06-19,14.7,7.0,54549.81\nR004,2022-06-20,13.6,11.9,26631.35\nR004,2022-06-21,16.0,0.5,49919.99\nR004,2022-06-22,18.6,2.3,47051.19\nR004,2022-06-23,21.2,0.0,55755.59\nR004,2022-06-24,18.3,0.2,53416.94\nR004,2022-06-25,19.3,0.0,56580.63\nR004,2022-06-26,22.8,0.1,54506.24\nR004,2022-06-27,22.9,0.6,41772.07\nR004,2022-06-28,17.1,5.7,42722.67\nR004,2022-06-29,17.4,0.0,57062.53\nR004,2022-06-30,17.1,1.1,51872.92\nR004,2022-07-01,16.3,2.5,49557.46\nR004,2022-07-02,14.6,0.1,43983.16\nR004,2022-07-03,13.3,7.9,51689.54\nR004,2022-07-04,15.4,0.5,56780.44\nR004,2022-07-05,16.5,3.1,45119.22\nR004,2022-07-06,15.2,8.9,46580.16\nR004,2022-07-07,14.1,23.9,26356.01\nR004,2022-07-08,16.6,0.2,45579.99\nR004,2022-07-09,18.3,0.0,51717.74\nR004,2022-07-10,19.1,0.0,57499.71\nR004,2022-07-11,19.3,0.3,57408.85\nR004,2022-07-12,18.6,0.7,55188.43\nR004,2022-07-13,14.7,1.0,52887.9\nR004,2022-07-14,12.8,1.0,54000.0\nR004,2022-07-15,12.9,0.1,44743.39\nR004,2022-07-16,15.6,0.6,51372.58\nR004,2022-07-17,14.7,16.7,22748.28\nR004,2022-07-18,13.9,6.1,44733.43\nR004,2022-07-19,12.8,7.3,41245.86\nR004,2022-07-20,14.3,0.5,54892.05\nR004,2022-07-21,14.5,1.1,20263.77\nR004,2022-07-22,15.4,11.9,28995.84\nR004,2022-07-23,14.0,25.4,42355.49\nR004,2022-07-24,13.4,10.1,39164.85\nR004,2022-07-25,12.9,4.0,41096.46\nR004,2022-07-26,14.3,0.4,47988.78\nR004,2022-07-27,16.2,0.0,50400.0\nR004,2022-07-28,16.6,0.0,54290.81\nR004,2022-07-29,18.0,0.0,54332.24\nR004,2022-07-30,17.9,1.4,51291.51\nR004,2022-07-31,18.3,2.7,43267.77\nR004,2022-08-01,16.5,16.2,33772.38\nR004,2022-08-02,15.2,10.8,34721.14\nR004,2022-08-03,14.9,0.0,52493.65\nR004,2022-08-04,13.1,0.0,52700.22\nR004,2022-08-05,12.4,0.0,52728.3\nR004,2022-08-06,14.4,0.0,52784.89\nR004,2022-08-07,15.0,0.0,52828.38\nR004,2022-08-08,16.6,0.0,50400.0\nR004,2022-08-09,18.1,0.0,47053.37\nR004,2022-08-10,18.0,3.3,46843.56\nR004,2022-08-11,17.4,1.1,36531.47\nR004,2022-08-12,16.6,4.7,26459.72\nR004,2022-08-13,14.4,19.0,31106.5\nR004,2022-08-14,15.6,8.4,29784.9\nR004,2022-08-15,13.1,1.4,50400.0\nR004,2022-08-16,10.3,3.9,50400.0\nR004,2022-08-17,8.5,3.7,44790.4\nR004,2022-08-18,8.9,5.9,42841.14\nR004,2022-08-19,6.5,10.8,24976.84\nR004,2022-08-20,7.9,4.3,43460.65\nR004,2022-08-21,10.6,0.4,45658.48\nR004,2022-08-22,10.7,3.6,26895.47\nR004,2022-08-23,11.3,2.6,40967.54\nR004,2022-08-24,11.1,0.0,41524.09\nR004,2022-08-25,12.9,0.0,49714.96\nR004,2022-08-26,13.6,0.1,49613.55\nR004,2022-08-27,12.9,3.2,38696.84\nR004,2022-08-28,9.6,0.4,46688.19\nR004,2022-08-29,11.2,0.0,48602.79\nR004,2022-08-30,14.0,0.0,48120.56\nR004,2022-08-31,14.9,0.0,47249.94\nR004,2022-09-01,16.3,0.1,47159.46\nR004,2022-09-02,15.7,0.1,39592.84\nR004,2022-09-03,12.4,0.0,39411.37\nR004,2022-09-04,12.9,0.1,40938.29\nR004,2022-09-05,14.4,0.1,43456.98\nR004,2022-09-06,15.7,0.0,44687.65\nR004,2022-09-07,21.2,0.1,43986.86\nR004,2022-09-08,14.4,0.4,32062.42\nR004,2022-09-09,10.0,0.2,43359.39\nR004,2022-09-10,9.1,0.0,41623.22\nR004,2022-09-11,9.8,2.4,27016.18\nR004,2022-09-12,9.3,16.5,32400.0\nR004,2022-09-13,7.0,2.7,21604.81\nR004,2022-09-14,5.5,7.5,2829.48\nR004,2022-09-15,5.3,0.4,34871.9\nR004,2022-09-16,6.4,0.8,42791.57\nR004,2022-09-17,6.8,0.0,40954.13\nR004,2022-09-18,7.8,0.0,43200.0\nR004,2022-09-19,8.3,0.0,42225.64\nR004,2022-09-20,8.4,0.0,25620.31\nR004,2022-09-21,9.8,0.0,43200.0\nR004,2022-09-22,10.7,0.0,43200.0\nR004,2022-09-23,10.9,0.0,43141.67\nR004,2022-09-24,7.4,0.0,40600.05\nR004,2022-09-25,7.6,1.8,28563.67\nR004,2022-09-26,8.0,13.2,0.0\nR004,2022-09-27,6.8,17.0,4818.27\nR004,2022-09-28,5.8,8.3,2979.41\nR004,2022-09-29,-0.5,6.1,0.0\nR004,2022-09-30,-2.9,1.9,32783.01\nR004,2022-10-01,-1.7,2.4,32395.37\nR004,2022-10-02,-0.3,0.2,37025.4\nR004,2022-10-03,-0.6,0.0,37155.04\nR004,2022-10-04,-0.9,0.0,36724.66\nR004,2022-10-05,0.1,5.9,14706.79\nR004,2022-10-06,1.0,8.4,1047.14\nR004,2022-10-07,1.1,0.0,33590.95\nR004,2022-10-08,2.7,0.0,36000.0\nR004,2022-10-09,7.4,0.1,36000.0\nR004,2022-10-10,4.6,6.9,13049.2\nR004,2022-10-11,5.1,1.6,10984.88\nR004,2022-10-12,5.6,8.1,0.0\nR004,2022-10-13,2.3,1.9,29028.19\nR004,2022-10-14,1.5,0.0,36000.0\nR004,2022-10-15,0.0,0.0,36000.0\nR004,2022-10-16,0.0,0.0,32664.32\nR004,2022-10-17,1.6,3.9,0.0\nR004,2022-10-18,2.8,5.1,27113.28\nR004,2022-10-19,1.0,2.6,27234.01\nR004,2022-10-20,0.1,1.0,11985.08\nR004,2022-10-21,2.7,29.8,0.0\nR004,2022-10-22,7.2,0.8,25105.56\nR004,2022-10-23,6.3,2.9,0.0\nR004,2022-10-24,-2.0,3.3,29913.27\nR004,2022-10-25,-3.3,0.0,33533.9\nR004,2022-10-26,-3.9,0.0,33187.56\nR004,2022-10-27,1.0,1.0,19466.67\nR004,2022-10-28,-0.4,6.0,20083.3\nR004,2022-10-29,-3.3,0.8,2409.8\nR004,2022-10-30,-3.8,0.0,31278.72\nR004,2022-10-31,-3.1,2.8,9479.06\nR004,2022-11-01,-0.8,4.7,205.39\nR004,2022-11-02,-1.7,0.0,30415.68\nR004,2022-11-03,-0.9,0.0,25508.42\nR004,2022-11-04,0.2,0.0,29246.44\nR004,2022-11-05,-0.7,0.0,8432.08\nR004,2022-11-06,-0.7,5.5,13616.14\nR004,2022-11-07,-2.9,0.7,10998.79\nR004,2022-11-08,-3.7,1.1,570.74\nR004,2022-11-09,-3.0,1.8,1116.4\nR004,2022-11-10,-4.6,0.2,15419.59\nR004,2022-11-11,-4.3,3.6,0.0\nR004,2022-11-12,-3.1,4.4,0.0\nR004,2022-11-13,-0.9,14.0,0.0\nR004,2022-11-14,-0.4,1.9,28800.0\nR004,2022-11-15,-0.1,4.1,130.95\nR004,2022-11-16,-1.2,3.5,23028.65\nR004,2022-11-17,-7.7,4.1,533.86\nR004,2022-11-18,-10.1,1.6,2254.5\nR004,2022-11-19,-17.9,0.1,25255.48\nR004,2022-11-20,-10.9,5.4,3651.62\nR004,2022-11-21,-15.8,0.0,13684.98\nR004,2022-11-22,-10.1,2.7,22002.11\nR004,2022-11-23,-11.1,4.3,13716.26\nR004,2022-11-24,-21.1,0.3,27001.93\nR004,2022-11-25,-24.5,1.4,2819.36\nR004,2022-11-26,-23.2,2.0,19358.46\nR004,2022-11-27,-29.8,0.0,17200.85\nR004,2022-11-28,-31.1,0.0,28255.39\nR004,2022-11-29,-28.4,0.0,23916.85\nR004,2022-11-30,-29.3,0.0,27564.13\nR004,2022-12-01,-29.6,0.0,27819.4\nR004,2022-12-02,-25.0,0.0,24705.56\nR004,2022-12-03,-21.5,0.0,26989.93\nR004,2022-12-04,-28.1,0.0,24781.25\nR004,2022-12-05,-25.4,0.0,24553.38\nR004,2022-12-06,-21.8,0.0,25534.36\nR004,2022-12-07,-23.7,0.0,19037.37\nR004,2022-12-08,-24.8,0.0,23652.97\nR004,2022-12-09,-28.5,0.0,26917.84\nR004,2022-12-10,-22.7,0.2,15774.3\nR004,2022-12-11,-16.3,4.0,0.0\nR004,2022-12-12,-22.1,0.2,25622.3\nR004,2022-12-13,-16.6,10.3,0.0\nR004,2022-12-14,-19.8,1.3,20844.34\nR004,2022-12-15,-28.7,0.0,24667.13\nR004,2022-12-16,-23.7,0.0,25754.9\nR004,2022-12-17,-21.9,0.0,25792.73\nR004,2022-12-18,-18.3,0.3,7554.05\nR004,2022-12-19,-16.1,0.5,984.97\nR004,2022-12-20,-6.4,23.7,0.0\nR004,2022-12-21,-9.5,0.0,25432.71\nR004,2022-12-22,-10.1,1.0,20146.95\nR004,2022-12-23,-17.1,0.0,22146.52\nR004,2022-12-24,-22.8,0.7,168.0\nR004,2022-12-25,-14.8,1.6,20364.09\nR004,2022-12-26,-10.6,1.1,1176.03\nR004,2022-12-27,-8.0,0.0,24669.83\nR004,2022-12-28,-17.4,0.0,25243.5\nR004,2022-12-29,-11.8,5.5,0.0\nR004,2022-12-30,-11.9,2.1,10665.8\nR004,2022-12-31,-13.6,0.6,12122.61\nR004,2023-01-01,-15.7,1.2,7435.83\nR004,2023-01-02,-16.0,0.4,21260.04\nR004,2023-01-03,-21.1,0.0,6399.44\nR004,2023-01-04,-16.1,2.6,19633.8\nR004,2023-01-05,-8.1,4.5,0.0\nR004,2023-01-06,-1.6,23.9,0.0\nR004,2023-01-07,-3.6,4.4,0.0\nR004,2023-01-08,-7.9,0.8,17888.59\nR004,2023-01-09,-7.1,5.6,0.0\nR004,2023-01-10,-18.1,0.7,20427.21\nR004,2023-01-11,-25.3,0.2,7484.81\nR004,2023-01-12,-25.1,0.0,24206.22\nR004,2023-01-13,-21.1,2.3,0.0\nR004,2023-01-14,-14.8,3.8,0.0\nR004,2023-01-15,-12.3,5.2,0.0\nR004,2023-01-16,-12.0,1.9,19259.25\nR004,2023-01-17,-11.0,8.3,0.0\nR004,2023-01-18,-10.4,7.3,8309.86\nR004,2023-01-19,-12.3,3.2,0.0\nR004,2023-01-20,-12.8,5.9,708.88\nR004,2023-01-21,-21.2,0.4,25980.68\nR004,2023-01-22,-29.8,0.0,28634.76\nR004,2023-01-23,-21.0,1.6,0.0\nR004,2023-01-24,-13.7,4.2,6220.17\nR004,2023-01-25,-11.0,5.9,0.0\nR004,2023-01-26,-8.4,5.1,0.0\nR004,2023-01-27,-9.1,0.0,28424.08\nR004,2023-01-28,-7.5,0.4,24712.82\nR004,2023-01-29,-10.6,0.0,30171.15\nR004,2023-01-30,-10.1,2.4,0.0\nR004,2023-01-31,-12.5,0.0,31389.32\nR004,2023-02-01,-15.0,0.0,22006.86\nR004,2023-02-02,-17.7,0.0,31624.34\nR004,2023-02-03,-17.5,0.0,31741.76\nR004,2023-02-04,-15.7,0.0,27228.43\nR004,2023-02-05,-16.1,0.0,31975.73\nR004,2023-02-06,-17.5,0.0,32092.02\nR004,2023-02-07,-11.7,2.5,13270.33\nR004,2023-02-08,-10.4,1.2,12228.75\nR004,2023-02-09,-14.7,0.0,28984.16\nR004,2023-02-10,-18.4,0.0,32400.0\nR004,2023-02-11,-20.7,0.0,32400.0\nR004,2023-02-12,-20.4,0.0,28994.75\nR004,2023-02-13,-14.8,0.4,3306.11\nR004,2023-02-14,-16.2,0.0,32400.0\nR004,2023-02-15,-18.6,0.0,33233.98\nR004,2023-02-16,-18.5,0.0,33579.92\nR004,2023-02-17,-17.6,0.0,33509.9\nR004,2023-02-18,-15.8,0.0,34262.23\nR004,2023-02-19,-10.0,0.7,13583.41\nR004,2023-02-20,-8.7,0.3,34661.8\nR004,2023-02-21,-9.8,0.5,22616.67\nR004,2023-02-22,-4.7,0.8,22906.98\nR004,2023-02-23,-4.8,0.0,19264.61\nR004,2023-02-24,-3.7,0.0,35214.91\nR004,2023-02-25,-3.1,9.9,0.0\nR004,2023-02-26,-10.1,3.1,29374.29\nR004,2023-02-27,-7.9,1.1,29755.0\nR004,2023-02-28,-4.7,0.0,34165.02\nR004,2023-03-01,-1.3,0.0,3966.67\nR004,2023-03-02,1.7,0.0,16708.46\nR004,2023-03-03,1.3,3.4,0.0\nR004,2023-03-04,-5.1,3.1,21035.2\nR004,2023-03-05,-6.4,0.0,27077.66\nR004,2023-03-06,-3.4,10.3,14028.83\nR004,2023-03-07,-2.9,5.2,0.0\nR004,2023-03-08,0.3,12.0,17124.32\nR004,2023-03-09,-5.6,4.8,28317.1\nR004,2023-03-10,-5.8,2.6,11513.24\nR004,2023-03-11,-5.4,0.0,29408.7\nR004,2023-03-12,-1.0,13.6,0.0\nR004,2023-03-13,-4.1,1.2,39401.51\nR004,2023-03-14,-3.7,0.0,40283.18\nR004,2023-03-15,-3.1,0.0,40593.7\nR004,2023-03-16,-1.9,0.0,40444.22\nR004,2023-03-17,-0.1,0.0,33928.72\nR004,2023-03-18,-0.8,0.0,41869.68\nR004,2023-03-19,-2.2,3.7,34810.06\nR004,2023-03-20,-12.2,1.7,37092.23\nR004,2023-03-21,-13.2,0.0,28819.91\nR004,2023-03-22,-11.8,0.0,42434.56\nR004,2023-03-23,-9.4,0.0,42577.7\nR004,2023-03-24,-2.7,0.0,26511.1\nR004,2023-03-25,-2.0,0.0,42916.1\nR004,2023-03-26,1.5,0.0,43010.17\nR004,2023-03-27,3.5,0.0,43479.87\nR004,2023-03-28,3.1,0.0,15657.07\nR004,2023-03-29,2.9,0.0,44182.69\nR004,2023-03-30,-0.5,5.2,17910.63\nR004,2023-03-31,-7.9,1.8,37032.19\nR004,2023-04-01,-10.6,0.1,41480.2\nR004,2023-04-02,-11.6,0.0,45125.0\nR004,2023-04-03,-6.9,0.0,45458.68\nR004,2023-04-04,-1.7,0.0,45241.79\nR004,2023-04-05,-0.0,0.0,45801.01\nR004,2023-04-06,-0.1,0.0,45988.39\nR004,2023-04-07,5.0,0.2,43735.83\nR004,2023-04-08,-3.8,5.0,0.0\nR004,2023-04-09,-6.7,2.6,12178.82\nR004,2023-04-10,-2.8,0.0,46529.28\nR004,2023-04-11,0.8,2.0,32860.61\nR004,2023-04-12,-3.9,0.7,37493.44\nR004,2023-04-13,-1.3,2.2,32739.92\nR004,2023-04-14,4.1,7.0,30689.42\nR004,2023-04-15,-4.8,12.4,7464.13\nR004,2023-04-16,-5.8,2.6,29526.83\nR004,2023-04-17,-6.6,10.4,5510.28\nR004,2023-04-18,-7.8,2.4,37249.05\nR004,2023-04-19,-6.9,4.4,19208.97\nR004,2023-04-20,-5.7,0.0,44170.03\nR004,2023-04-21,-2.8,0.0,46945.2\nR004,2023-04-22,-0.8,0.0,50348.82\nR004,2023-04-23,0.6,0.0,50400.0\nR004,2023-04-24,4.1,0.6,33246.97\nR004,2023-04-25,1.3,10.1,26959.25\nR004,2023-04-26,-1.3,1.9,33623.52\nR004,2023-04-27,2.6,1.0,26050.39\nR004,2023-04-28,10.6,0.0,44441.78\nR004,2023-04-29,8.9,0.3,45221.62\nR004,2023-04-30,3.4,0.2,43209.35\nR004,2023-05-01,-1.7,1.3,48711.95\nR004,2023-05-02,-1.1,1.6,49781.75\nR004,2023-05-03,-0.0,0.2,46624.87\nR004,2023-05-04,3.4,0.2,46089.78\nR004,2023-05-05,5.4,0.0,51009.44\nR004,2023-05-06,9.8,0.1,50545.2\nR004,2023-05-07,12.1,0.0,51343.89\nR004,2023-05-08,13.7,0.0,51726.94\nR004,2023-05-09,15.2,0.5,48567.07\nR004,2023-05-10,8.6,4.5,21316.26\nR004,2023-05-11,8.1,1.6,32794.65\nR004,2023-05-12,10.8,0.0,51730.4\nR004,2023-05-13,6.5,0.0,50519.73\nR004,2023-05-14,4.6,3.1,22553.76\nR004,2023-05-15,7.9,4.7,45532.83\nR004,2023-05-16,8.5,0.1,24538.08\nR004,2023-05-17,4.5,3.5,36849.29\nR004,2023-05-18,3.8,0.0,51879.87\nR004,2023-05-19,7.1,0.0,56446.33\nR004,2023-05-20,9.2,0.1,56951.65\nR004,2023-05-21,10.7,1.9,55346.81\nR004,2023-05-22,10.9,1.5,45953.24\nR004,2023-05-23,12.4,0.5,56427.05\nR004,2023-05-24,10.7,0.0,55052.76\nR004,2023-05-25,8.4,0.0,57390.89\nR004,2023-05-26,7.2,0.7,57460.86\nR004,2023-05-27,8.1,0.5,57528.37\nR004,2023-05-28,11.1,0.2,38482.82\nR004,2023-05-29,7.7,9.6,22011.18\nR004,2023-05-30,8.6,7.7,39150.34\nR004,2023-05-31,10.0,3.2,35765.45\nR004,2023-06-01,12.9,0.4,51000.35\nR004,2023-06-02,16.1,0.0,57600.0\nR004,2023-06-03,18.5,0.0,55916.76\nR004,2023-06-04,20.9,0.1,57600.0\nR004,2023-06-05,20.7,0.0,57600.0\nR004,2023-06-06,22.4,0.0,57600.0\nR004,2023-06-07,23.3,0.1,50465.44\nR004,2023-06-08,22.2,0.9,47157.16\nR004,2023-06-09,21.3,1.5,57600.0\nR004,2023-06-10,18.6,6.5,37874.59\nR004,2023-06-11,12.6,3.8,3432.96\nR004,2023-06-12,14.7,0.0,54697.22\nR004,2023-06-13,15.8,0.0,51338.36\nR004,2023-06-14,9.7,17.9,40403.84\nR004,2023-06-15,10.9,0.0,56374.42\nR004,2023-06-16,15.1,0.0,57600.0\nR004,2023-06-17,17.0,0.0,57537.78\nR004,2023-06-18,19.0,0.0,57600.0\nR004,2023-06-19,20.5,0.0,53686.3\nR004,2023-06-20,19.7,1.1,46247.19\nR004,2023-06-21,16.1,9.5,40196.42\nR004,2023-06-22,16.1,3.3,57490.69\nR004,2023-06-23,13.2,3.2,56340.62\nR004,2023-06-24,11.3,6.3,30443.08\nR004,2023-06-25,11.6,1.7,53194.69\nR004,2023-06-26,12.6,0.6,49495.57\nR004,2023-06-27,15.3,0.3,52454.93\nR004,2023-06-28,16.2,3.6,52198.68\nR004,2023-06-29,13.6,13.9,48329.26\nR004,2023-06-30,13.2,4.2,47246.85\nR004,2023-07-01,12.8,12.3,40229.39\nR004,2023-07-02,13.4,3.9,52484.28\nR004,2023-07-03,13.9,2.7,43338.67\nR004,2023-07-04,14.5,0.1,57600.0\nR004,2023-07-05,16.7,0.1,49099.82\nR004,2023-07-06,18.9,0.9,47028.55\nR004,2023-07-07,19.6,1.9,44199.63\nR004,2023-07-08,17.5,17.5,42820.43\nR004,2023-07-09,17.8,4.4,55023.61\nR004,2023-07-10,15.3,10.9,50169.12\nR004,2023-07-11,16.8,0.3,41630.0\nR004,2023-07-12,18.7,6.7,45370.45\nR004,2023-07-13,21.7,0.6,55888.05\nR004,2023-07-14,22.0,0.0,56545.26\nR004,2023-07-15,23.9,0.2,47930.73\nR004,2023-07-16,17.0,8.6,41678.19\nR004,2023-07-17,15.4,9.3,34592.21\nR004,2023-07-18,18.0,0.0,55730.72\nR004,2023-07-19,20.1,0.0,54391.08\nR004,2023-07-20,20.0,0.8,38807.28\nR004,2023-07-21,18.8,7.3,53241.67\nR004,2023-07-22,17.2,8.8,46473.58\nR004,2023-07-23,13.9,15.9,24086.34\nR004,2023-07-24,15.8,5.3,47642.1\nR004,2023-07-25,18.5,1.4,53260.77\nR004,2023-07-26,18.7,0.5,54000.0\nR004,2023-07-27,18.0,0.0,39841.63\nR004,2023-07-28,19.2,1.1,42151.95\nR004,2023-07-29,16.2,41.6,8742.92\nR004,2023-07-30,18.2,1.8,52548.87\nR004,2023-07-31,20.4,0.6,53824.51\nR004,2023-08-01,19.8,1.5,51086.46\nR004,2023-08-02,20.1,0.2,46791.06\nR004,2023-08-03,19.2,2.3,35827.56\nR004,2023-08-04,17.3,17.7,30526.62\nR004,2023-08-05,16.9,14.5,507.34\nR004,2023-08-06,16.2,3.2,24489.76\nR004,2023-08-07,15.9,0.8,46866.23\nR004,2023-08-08,17.2,0.0,50761.72\nR004,2023-08-09,17.0,0.0,50400.0\nR004,2023-08-10,17.6,0.1,49527.8\nR004,2023-08-11,16.1,9.9,18417.64\nR004,2023-08-12,14.3,6.5,28625.16\nR004,2023-08-13,13.2,1.1,36476.39\nR004,2023-08-14,10.4,10.2,21342.24\nR004,2023-08-15,11.6,1.0,50400.0\nR004,2023-08-16,12.3,1.4,50400.0\nR004,2023-08-17,11.9,18.0,16669.22\nR004,2023-08-18,12.9,8.7,39316.35\nR004,2023-08-19,12.3,0.3,27057.81\nR004,2023-08-20,13.5,16.0,10756.77\nR004,2023-08-21,15.9,4.4,43722.87\nR004,2023-08-22,16.2,0.1,44775.36\nR004,2023-08-23,17.5,0.0,49941.07\nR004,2023-08-24,18.7,0.0,49840.31\nR004,2023-08-25,19.2,1.3,42606.9\nR004,2023-08-26,15.7,12.0,21872.63\nR004,2023-08-27,17.9,0.0,45318.55\nR004,2023-08-28,20.4,1.9,30944.65\nR004,2023-08-29,15.3,12.7,166.67\nR004,2023-08-30,14.9,0.0,27058.65\nR004,2023-08-31,17.9,0.0,45759.09\nR004,2023-09-01,15.0,9.7,30581.09\nR004,2023-09-02,14.2,0.1,42046.42\nR004,2023-09-03,13.8,0.3,44234.66\nR004,2023-09-04,14.6,1.6,30135.01\nR004,2023-09-05,12.4,12.5,36240.72\nR004,2023-09-06,11.2,5.7,37531.9\nR004,2023-09-07,8.7,7.1,33150.42\nR004,2023-09-08,9.4,0.3,42874.44\nR004,2023-09-09,9.6,0.8,43345.61\nR004,2023-09-10,9.6,0.0,42386.89\nR004,2023-09-11,10.6,0.0,42477.56\nR004,2023-09-12,10.2,8.2,880.4\nR004,2023-09-13,11.6,0.4,43200.0\nR004,2023-09-14,11.2,2.0,35749.9\nR004,2023-09-15,10.5,1.4,32464.62\nR004,2023-09-16,10.2,1.0,26229.41\nR004,2023-09-17,11.2,0.3,30187.83\nR004,2023-09-18,12.4,0.0,35562.72\nR004,2023-09-19,10.9,5.1,39393.93\nR004,2023-09-20,10.8,1.6,20057.95\nR004,2023-09-21,10.0,0.0,32775.94\nR004,2023-09-22,10.1,5.8,39039.52\nR004,2023-09-23,7.9,10.6,22244.81\nR004,2023-09-24,2.6,6.1,38043.67\nR004,2023-09-25,0.7,20.6,2561.02\nR004,2023-09-26,1.4,1.6,37621.39\nR004,2023-09-27,2.2,0.0,41730.56\nR004,2023-09-28,3.5,0.0,30779.69\nR004,2023-09-29,5.0,7.0,0.0\nR004,2023-09-30,6.9,7.8,16613.73\nR004,2023-10-01,5.1,1.4,17609.83\nR004,2023-10-02,4.1,0.0,36608.95\nR004,2023-10-03,3.4,0.0,37122.57\nR004,2023-10-04,2.2,0.0,36862.86\nR004,2023-10-05,3.2,0.0,25392.31\nR004,2023-10-06,4.8,0.0,36044.83\nR004,2023-10-07,4.5,0.0,36000.0\nR004,2023-10-08,6.3,0.7,34119.06\nR004,2023-10-09,9.2,1.1,33228.86\nR004,2023-10-10,11.0,0.0,36000.0\nR004,2023-10-11,10.0,0.0,36000.0\nR004,2023-10-12,8.7,0.7,31469.02\nR004,2023-10-13,2.9,9.4,0.0\nR004,2023-10-14,1.8,0.0,31444.92\nR004,2023-10-15,2.9,5.6,0.0\nR004,2023-10-16,3.1,5.6,3955.48\nR004,2023-10-17,3.0,3.5,0.0\nR004,2023-10-18,7.2,0.0,34766.45\nR004,2023-10-19,5.8,0.0,35530.85\nR004,2023-10-20,5.0,0.0,32493.04\nR004,2023-10-21,3.9,2.5,29182.02\nR004,2023-10-22,2.9,0.3,17447.58\nR004,2023-10-23,3.0,10.5,12224.61\nR004,2023-10-24,-1.9,2.8,10176.1\nR004,2023-10-25,1.4,5.4,0.0\nR004,2023-10-26,5.4,0.9,5945.43\nR004,2023-10-27,-1.3,8.0,21425.42\nR004,2023-10-28,-5.9,0.5,18075.3\nR004,2023-10-29,-6.4,0.0,18741.96\nR004,2023-10-30,-3.7,7.0,113.32\nR004,2023-10-31,1.2,16.3,0.0\nR004,2023-11-01,-1.2,8.1,0.0\nR004,2023-11-02,-1.0,13.1,195.86\nR004,2023-11-03,-8.2,9.4,16604.49\nR004,2023-11-04,-16.3,0.0,19247.4\nR004,2023-11-05,-5.5,11.3,0.0\nR004,2023-11-06,-6.6,2.6,6238.04\nR004,2023-11-07,-5.3,7.3,0.0\nR004,2023-11-08,-5.0,0.8,4166.01\nR004,2023-11-09,-6.1,0.0,9891.82\nR004,2023-11-10,-7.8,0.0,22840.31\nR004,2023-11-11,-4.6,0.0,12475.14\nR004,2023-11-12,-6.0,0.0,28754.99\nR004,2023-11-13,-1.8,0.9,27391.28\nR004,2023-11-14,-0.8,7.6,2080.78\nR004,2023-11-15,-1.2,0.1,20702.54\nR004,2023-11-16,-5.6,0.0,28800.0\nR004,2023-11-17,-1.2,8.3,0.0\nR004,2023-11-18,1.3,6.0,674.0\nR004,2023-11-19,2.8,1.6,15402.97\nR004,2023-11-20,-8.0,1.0,27116.4\nR004,2023-11-21,-15.5,0.4,25806.1\nR004,2023-11-22,-21.1,0.0,28670.37\nR004,2023-11-23,-15.2,0.0,3732.83\nR004,2023-11-24,-5.6,2.4,19540.89\nR004,2023-11-25,-5.0,5.4,0.0\nR004,2023-11-26,-4.5,7.3,0.0\nR004,2023-11-27,-2.4,4.1,30.83\nR004,2023-11-28,-9.3,0.0,28277.78\nR004,2023-11-29,-12.0,0.0,28126.97\nR004,2023-11-30,-13.4,0.0,24006.1\nR004,2023-12-01,-6.1,0.5,0.0\nR004,2023-12-02,-5.0,11.7,0.0\nR004,2023-12-03,-0.3,10.9,0.0\nR004,2023-12-04,-6.4,14.0,0.0\nR004,2023-12-05,-12.4,9.2,0.0\nR004,2023-12-06,-2.6,31.0,0.0\nR004,2023-12-07,-14.5,4.0,0.0\nR004,2023-12-08,-21.9,0.1,21701.76\nR004,2023-12-09,-27.2,0.0,24272.65\nR004,2023-12-10,-32.7,0.0,26866.31\nR004,2023-12-11,-34.6,0.0,25592.86\nR004,2023-12-12,-34.4,0.0,26669.91\nR004,2023-12-13,-36.0,0.0,25916.17\nR004,2023-12-14,-31.7,1.1,2555.28\nR004,2023-12-15,-22.1,0.7,20090.34\nR004,2023-12-16,-21.5,0.8,14679.0\nR004,2023-12-17,-22.7,1.0,17999.06\nR004,2023-12-18,-31.8,0.0,25829.05\nR004,2023-12-19,-31.8,0.0,25037.61\nR004,2023-12-20,-23.0,0.0,10637.4\nR004,2023-12-21,-14.0,2.0,15957.52\nR004,2023-12-22,-7.3,4.2,0.0\nR004,2023-12-23,-8.0,0.0,21668.37\nR004,2023-12-24,-13.2,0.0,22397.32\nR004,2023-12-25,-11.9,0.8,695.66\nR004,2023-12-26,-9.6,0.0,22370.29\nR004,2023-12-27,-7.1,2.3,23519.35\nR004,2023-12-28,-6.3,3.6,0.0\nR004,2023-12-29,-5.6,0.3,2145.39\nR004,2023-12-30,-4.0,4.4,9129.31\nR004,2023-12-31,-4.6,3.6,0.0\nR004,2024-01-01,-3.9,7.5,0.0\nR004,2024-01-02,-5.2,4.3,0.0\nR004,2024-01-03,-6.4,12.4,0.0\nR004,2024-01-04,-15.7,2.0,22764.07\nR004,2024-01-05,-20.8,0.0,19678.16\nR004,2024-01-06,-11.2,8.3,0.0\nR004,2024-01-07,-7.6,5.0,1573.4\nR004,2024-01-08,-9.8,1.6,0.0\nR004,2024-01-09,-9.1,4.5,15156.74\nR004,2024-01-10,-17.5,1.0,16862.15\nR004,2024-01-11,-10.9,4.1,20030.0\nR004,2024-01-12,-16.0,3.5,25200.0\nR004,2024-01-13,-21.4,0.0,25546.99\nR004,2024-01-14,-12.8,1.9,16058.85\nR004,2024-01-15,-16.0,5.3,15417.98\nR004,2024-01-16,-22.0,0.0,27612.02\nR004,2024-01-17,-26.4,0.0,27643.67\nR004,2024-01-18,-28.1,0.0,27707.53\nR004,2024-01-19,-26.0,0.0,27774.48\nR004,2024-01-20,-24.7,0.0,27844.37\nR004,2024-01-21,-25.7,0.0,28146.32\nR004,2024-01-22,-25.9,0.0,28633.08\nR004,2024-01-23,-24.6,0.0,28649.97\nR004,2024-01-24,-23.6,0.0,28202.4\nR004,2024-01-25,-22.8,0.0,29153.92\nR004,2024-01-26,-23.7,0.0,30099.89\nR004,2024-01-27,-22.4,0.0,29197.1\nR004,2024-01-28,-20.3,0.0,30598.29\nR004,2024-01-29,-19.1,0.5,14264.5\nR004,2024-01-30,-18.4,0.3,27212.37\nR004,2024-01-31,-19.0,0.0,28852.0\nR004,2024-02-01,-16.3,0.0,15165.09\nR004,2024-02-02,-16.4,0.0,11322.82\nR004,2024-02-03,-12.6,0.4,27562.09\nR004,2024-02-04,-11.4,0.0,25812.81\nR004,2024-02-05,-7.1,0.1,29485.68\nR004,2024-02-06,-10.6,0.0,32057.29\nR004,2024-02-07,-14.1,0.0,32175.38\nR004,2024-02-08,-14.5,0.7,4043.07\nR004,2024-02-09,-7.9,5.3,0.0\nR004,2024-02-10,2.1,15.9,9409.66\nR004,2024-02-11,-8.4,13.9,13197.37\nR004,2024-02-12,-15.3,2.0,12606.37\nR004,2024-02-13,-15.3,1.2,26738.94\nR004,2024-02-14,-10.8,4.0,15664.49\nR004,2024-02-15,-14.0,5.4,29759.05\nR004,2024-02-16,-25.5,0.3,25433.97\nR004,2024-02-17,-28.0,0.6,7702.16\nR004,2024-02-18,-25.4,0.2,20193.89\nR004,2024-02-19,-27.6,0.0,34745.36\nR004,2024-02-20,-28.1,0.0,34811.14\nR004,2024-02-21,-27.5,0.0,34786.38\nR004,2024-02-22,-24.7,0.0,35641.05\nR004,2024-02-23,-24.0,0.0,35816.77\nR004,2024-02-24,-21.4,0.0,32713.29\nR004,2024-02-25,-18.5,0.0,29224.23\nR004,2024-02-26,-15.0,2.3,13374.07\nR004,2024-02-27,-16.6,0.1,34543.22\nR004,2024-02-28,-17.9,0.0,37761.16\nR004,2024-02-29,-13.1,0.3,28919.77\nR004,2024-03-01,-7.5,3.5,0.0\nR004,2024-03-02,-7.4,3.2,0.0\nR004,2024-03-03,-9.1,0.0,9760.66\nR004,2024-03-04,-9.1,1.0,13222.02\nR004,2024-03-05,-6.5,2.5,8767.93\nR004,2024-03-06,-3.3,0.0,39029.26\nR004,2024-03-07,-4.6,0.0,39141.21\nR004,2024-03-08,-5.2,1.3,21600.0\nR004,2024-03-09,-6.6,0.1,39363.75\nR004,2024-03-10,-3.6,0.4,26398.01\nR004,2024-03-11,-0.6,3.2,6904.25\nR004,2024-03-12,-1.7,1.7,37124.95\nR004,2024-03-13,-3.2,5.7,16815.52\nR004,2024-03-14,-8.7,1.4,36555.79\nR004,2024-03-15,-9.9,0.8,31994.87\nR004,2024-03-16,-7.0,1.0,36140.29\nR004,2024-03-17,-4.4,8.0,0.0\nR004,2024-03-18,-1.1,0.0,40327.26\nR004,2024-03-19,-0.1,0.2,35051.39\nR004,2024-03-20,0.2,7.4,11552.78\nR004,2024-03-21,-4.3,1.4,34332.13\nR004,2024-03-22,-5.7,0.0,40321.99\nR004,2024-03-23,-3.0,0.0,36478.92\nR004,2024-03-24,-0.8,0.0,42900.61\nR004,2024-03-25,-0.1,5.7,22780.31\nR004,2024-03-26,-0.8,3.5,30165.7\nR004,2024-03-27,-0.7,0.0,35585.53\nR004,2024-03-28,2.8,0.7,35063.2\nR004,2024-03-29,2.7,3.1,22320.37\nR004,2024-03-30,3.3,4.7,40682.48\nR004,2024-03-31,0.2,4.9,20494.4\nR004,2024-04-01,-0.2,3.6,0.0\nR004,2024-04-02,2.6,0.0,40665.7\nR004,2024-04-03,2.6,1.1,20305.22\nR004,2024-04-04,1.9,0.4,41451.04\nR004,2024-04-05,5.7,0.0,44948.99\nR004,2024-04-06,4.5,0.0,46019.62\nR004,2024-04-07,4.6,0.0,46353.37\nR004,2024-04-08,3.0,0.0,46197.98\nR004,2024-04-09,3.3,3.9,24083.35\nR004,2024-04-10,3.0,8.9,0.0\nR004,2024-04-11,0.8,5.1,17983.33\nR004,2024-04-12,-4.4,4.2,26840.23\nR004,2024-04-13,-2.3,0.0,38573.57\nR004,2024-04-14,7.1,2.3,47072.7\nR004,2024-04-15,7.1,1.8,34997.6\nR004,2024-04-16,2.4,0.0,42358.21\nR004,2024-04-17,2.6,0.0,49793.75\nR004,2024-04-18,5.8,2.7,29515.43\nR004,2024-04-19,6.9,13.1,15723.04\nR004,2024-04-20,3.6,0.0,48281.1\nR004,2024-04-21,3.3,0.0,50316.9\nR004,2024-04-22,5.7,5.4,44377.15\nR004,2024-04-23,0.6,4.3,39600.0\nR004,2024-04-24,-3.1,1.1,36610.36\nR004,2024-04-25,-3.0,0.0,48133.9\nR004,2024-04-26,0.6,0.0,50400.0\nR004,2024-04-27,4.7,0.0,46889.4\nR004,2024-04-28,7.1,0.0,48077.67\nR004,2024-04-29,8.7,6.4,10565.05\nR004,2024-04-30,3.3,13.8,29951.21\nR004,2024-05-01,0.7,8.9,0.0\nR004,2024-05-02,3.8,2.1,24110.76\nR004,2024-05-03,9.8,3.0,29630.79\nR004,2024-05-04,9.9,3.4,36496.94\nR004,2024-05-05,16.5,0.3,43200.0\nR004,2024-05-06,9.6,9.7,20186.19\nR004,2024-05-07,10.1,0.0,49221.51\nR004,2024-05-08,11.7,0.0,44429.53\nR004,2024-05-09,12.6,0.0,50669.13\nR004,2024-05-10,15.6,0.0,41795.4\nR004,2024-05-11,14.7,6.1,3117.89\nR004,2024-05-12,12.5,6.1,41169.98\nR004,2024-05-13,13.0,0.0,54196.71\nR004,2024-05-14,13.7,0.0,46235.39\nR004,2024-05-15,13.5,8.7,0.0\nR004,2024-05-16,7.5,10.0,34974.09\nR004,2024-05-17,9.1,0.0,55928.94\nR004,2024-05-18,12.8,0.0,55545.59\nR004,2024-05-19,12.5,4.2,23850.34\nR004,2024-05-20,15.5,6.8,39262.13\nR004,2024-05-21,14.0,0.8,39730.98\nR004,2024-05-22,11.0,11.4,24946.44\nR004,2024-05-23,12.1,4.7,54063.94\nR004,2024-05-24,14.1,0.0,57376.21\nR004,2024-05-25,14.2,5.3,31502.76\nR004,2024-05-26,15.3,4.4,33305.63\nR004,2024-05-27,9.1,20.6,32938.49\nR004,2024-05-28,8.9,8.4,30986.92\nR004,2024-05-29,6.8,5.5,26858.92\nR004,2024-05-30,5.5,14.3,2714.49\nR004,2024-05-31,5.2,6.7,36008.34\nR004,2024-06-01,6.8,0.0,56906.12\nR004,2024-06-02,9.9,0.0,57600.0\nR004,2024-06-03,9.7,3.3,19406.53\nR004,2024-06-04,10.0,8.3,27726.13\nR004,2024-06-05,8.3,0.1,31282.06\nR004,2024-06-06,13.3,0.4,49171.43\nR004,2024-06-07,17.1,7.4,53230.99\nR004,2024-06-08,16.3,7.0,33844.56\nR004,2024-06-09,16.4,17.9,34856.04\nR004,2024-06-10,16.2,0.2,54687.07\nR004,2024-06-11,16.2,0.2,49423.46\nR004,2024-06-12,17.0,5.6,38531.13\nR004,2024-06-13,18.6,0.3,40509.27\nR004,2024-06-14,21.2,0.0,52717.01\nR004,2024-06-15,21.6,0.0,57600.0\nR004,2024-06-16,19.9,0.6,55113.37\nR004,2024-06-17,18.5,6.5,37064.81\nR004,2024-06-18,16.3,27.0,13772.21\nR004,2024-06-19,11.2,2.7,56712.62\nR004,2024-06-20,12.0,0.6,56046.6\nR004,2024-06-21,15.2,0.1,53233.28\nR004,2024-06-22,21.1,0.0,57600.0\nR004,2024-06-23,22.1,0.5,51039.56\nR004,2024-06-24,23.3,0.0,54352.9\nR004,2024-06-25,23.5,0.0,57600.0\nR004,2024-06-26,19.7,15.6,33314.97\nR004,2024-06-27,20.8,0.9,50983.71\nR004,2024-06-28,22.0,0.0,54835.51\nR004,2024-06-29,24.0,0.0,57600.0\nR004,2024-06-30,21.9,0.5,35353.33\nR004,2024-07-01,19.6,8.2,46757.74\nR004,2024-07-02,18.4,8.4,33967.34\nR004,2024-07-03,16.1,14.9,21817.44\nR004,2024-07-04,17.9,1.0,56851.6\nR004,2024-07-05,19.2,0.2,55685.3\nR004,2024-07-06,19.3,1.4,47458.52\nR004,2024-07-07,18.3,6.6,35053.28\nR004,2024-07-08,19.2,2.5,46546.06\nR004,2024-07-09,18.0,14.1,32325.18\nR004,2024-07-10,16.2,3.1,31009.89\nR004,2024-07-11,16.1,1.0,54889.87\nR004,2024-07-12,17.7,1.7,56047.85\nR004,2024-07-13,19.8,0.0,56535.76\nR004,2024-07-14,20.7,0.2,56098.25\nR004,2024-07-15,21.9,0.0,51301.64\nR004,2024-07-16,21.8,0.0,38935.31\nR004,2024-07-17,21.6,2.3,33895.85\nR004,2024-07-18,19.9,4.4,28981.95\nR004,2024-07-19,19.2,4.3,51717.19\nR004,2024-07-20,20.0,1.2,43700.88\nR004,2024-07-21,17.4,12.0,21657.36\nR004,2024-07-22,16.6,8.3,45505.43\nR004,2024-07-23,17.6,10.2,27880.95\nR004,2024-07-24,20.2,0.0,50593.0\nR004,2024-07-25,20.4,0.0,41815.43\nR004,2024-07-26,22.4,0.1,49137.56\nR004,2024-07-27,22.3,4.2,51061.25\nR004,2024-07-28,22.5,2.4,53158.35\nR004,2024-07-29,19.0,14.5,29897.51\nR004,2024-07-30,19.9,0.0,44373.13\nR004,2024-07-31,18.4,3.4,53785.59\nR004,2024-08-01,18.6,0.0,50951.71\nR004,2024-08-02,20.0,0.0,53606.57\nR004,2024-08-03,20.9,0.0,43538.05\nR004,2024-08-04,17.5,16.6,21111.98\nR004,2024-08-05,16.5,8.1,32636.75\nR004,2024-08-06,13.9,14.2,33475.82\nR004,2024-08-07,15.8,4.0,49624.84\nR004,2024-08-08,17.0,7.6,23064.45\nR004,2024-08-09,17.8,2.7,36109.97\nR004,2024-08-10,19.6,6.4,43664.88\nR004,2024-08-11,14.9,9.2,21836.06\nR004,2024-08-12,14.5,7.0,43685.65\nR004,2024-08-13,15.7,0.0,50665.44\nR004,2024-08-14,18.3,0.0,50723.35\nR004,2024-08-15,19.7,0.0,47815.84\nR004,2024-08-16,21.0,0.7,47349.39\nR004,2024-08-17,18.0,8.8,48876.58\nR004,2024-08-18,15.5,17.3,21726.49\nR004,2024-08-19,14.2,13.9,36702.15\nR004,2024-08-20,16.1,0.0,50169.98\nR004,2024-08-21,16.3,0.2,44634.35\nR004,2024-08-22,14.4,9.8,41975.75\nR004,2024-08-23,15.5,0.0,48040.75\nR004,2024-08-24,17.6,1.3,49270.99\nR004,2024-08-25,16.5,12.0,45956.52\nR004,2024-08-26,13.6,1.5,43760.46\nR004,2024-08-27,12.7,1.8,34156.17\nR004,2024-08-28,12.9,1.1,44418.17\nR004,2024-08-29,12.8,0.0,47654.22\nR004,2024-08-30,13.3,6.5,35962.74\nR004,2024-08-31,12.4,12.1,20042.86\nR004,2024-09-01,11.2,22.8,0.0\nR004,2024-09-02,8.5,16.7,20560.04\nR004,2024-09-03,7.1,12.2,0.0\nR004,2024-09-04,9.2,13.3,1376.84\nR004,2024-09-05,12.1,0.1,16595.17\nR004,2024-09-06,11.4,1.3,37793.04\nR004,2024-09-07,9.2,3.2,43257.38\nR004,2024-09-08,7.4,7.9,0.0\nR004,2024-09-09,10.0,5.5,9147.54\nR004,2024-09-10,10.6,2.7,28288.91\nR004,2024-09-11,9.1,0.0,39418.67\nR004,2024-09-12,8.5,0.0,43223.0\nR004,2024-09-13,8.7,0.5,31229.66\nR004,2024-09-14,7.2,8.3,2.31\nR004,2024-09-15,6.1,15.0,0.0\nR004,2024-09-16,2.2,4.7,42118.19\nR004,2024-09-17,3.1,0.3,30461.32\nR004,2024-09-18,3.3,0.0,43200.0\nR004,2024-09-19,6.9,0.0,43200.0\nR004,2024-09-20,8.3,0.0,43200.0\nR004,2024-09-21,10.6,1.5,40541.2\nR004,2024-09-22,11.0,0.0,42322.14\nR004,2024-09-23,12.1,0.1,42998.81\nR004,2024-09-24,7.5,2.7,33634.61\nR004,2024-09-25,2.4,2.2,42502.55\nR004,2024-09-26,-0.1,0.7,38036.6\nR004,2024-09-27,-0.3,3.5,13453.44\nR004,2024-09-28,0.9,1.5,31597.49\nR004,2024-09-29,0.9,0.0,39397.01\nR004,2024-09-30,2.5,0.0,364.91\nR004,2024-10-01,10.3,0.0,29861.37\nR004,2024-10-02,8.2,0.0,31674.7\nR004,2024-10-03,2.3,0.0,34704.67\nR004,2024-10-04,1.7,0.0,36363.88\nR004,2024-10-05,3.9,0.0,36031.51\nR004,2024-10-06,4.5,0.0,33605.29\nR004,2024-10-07,10.5,4.1,24565.18\nR004,2024-10-08,3.3,4.1,15889.52\nR004,2024-10-09,8.6,3.3,5806.34\nR004,2024-10-10,4.1,3.2,31251.65\nR004,2024-10-11,2.2,0.1,29120.68\nR004,2024-10-12,0.7,11.6,0.0\nR004,2024-10-13,-2.4,7.4,76.41\nR004,2024-10-14,-1.9,0.0,36000.0\nR004,2024-10-15,-1.4,0.0,35966.33\nR004,2024-10-16,-2.2,0.0,35618.28\nR004,2024-10-17,-0.5,0.0,35213.86\nR004,2024-10-18,-0.9,0.0,33913.11\nR004,2024-10-19,-0.9,0.0,0.0\nR004,2024-10-20,0.4,0.0,16596.84\nR004,2024-10-21,3.4,4.0,6587.31\nR004,2024-10-22,2.4,11.3,0.0\nR004,2024-10-23,2.3,2.2,416.68\nR004,2024-10-24,3.8,8.1,476.61\nR004,2024-10-25,5.2,11.8,0.0\nR004,2024-10-26,6.0,1.7,0.0\nR004,2024-10-27,5.5,7.2,7025.98\nR004,2024-10-28,4.4,4.9,17718.89\nR004,2024-10-29,1.2,9.3,0.0\nR004,2024-10-30,-0.8,2.6,7416.93\nR004,2024-10-31,-1.2,10.6,0.0\nR004,2024-11-01,-2.6,2.5,15194.35\nR004,2024-11-02,-3.4,0.0,30023.04\nR004,2024-11-03,-0.9,4.4,0.0\nR004,2024-11-04,2.6,0.0,28148.36\nR004,2024-11-05,0.9,8.4,0.0\nR004,2024-11-06,-0.3,1.7,21888.43\nR004,2024-11-07,-1.0,9.1,8258.5\nR004,2024-11-08,-4.1,1.4,763.07\nR004,2024-11-09,-4.6,1.2,28800.0\nR004,2024-11-10,-5.9,5.5,498.78\nR004,2024-11-11,-1.5,10.4,0.0\nR004,2024-11-12,-8.6,5.9,16551.56\nR004,2024-11-13,-13.6,0.2,16468.41\nR004,2024-11-14,-14.1,2.4,0.0\nR004,2024-11-15,-7.3,12.7,0.0\nR004,2024-11-16,-5.3,4.1,0.0\nR004,2024-11-17,-10.0,0.0,24389.81\nR004,2024-11-18,-6.8,0.0,28800.0\nR004,2024-11-19,-5.6,0.0,27747.45\nR004,2024-11-20,-11.3,0.0,28758.3\nR004,2024-11-21,-12.3,0.3,15990.29\nR004,2024-11-22,-4.0,4.9,11217.9\nR004,2024-11-23,-11.1,0.3,0.0\nR004,2024-11-24,-9.2,3.7,0.0\nR004,2024-11-25,-8.6,0.0,24574.65\nR004,2024-11-26,-15.1,0.0,28345.15\nR004,2024-11-27,-15.4,0.0,28287.92\nR004,2024-11-28,-15.1,0.0,28168.85\nR004,2024-11-29,-14.5,0.0,23194.25\nR004,2024-11-30,-6.0,0.0,16668.9\nR004,2024-12-01,-8.7,0.2,26474.68\nR004,2024-12-02,-4.1,6.9,0.0\nR004,2024-12-03,-8.5,2.7,0.0\nR004,2024-12-04,-6.8,3.7,0.0\nR004,2024-12-05,-5.2,0.9,1886.5\nR004,2024-12-06,-9.1,0.0,17375.12\nR004,2024-12-07,-13.2,0.2,12223.08\nR004,2024-12-08,-8.5,3.9,0.0\nR004,2024-12-09,-15.0,1.0,17315.55\nR004,2024-12-10,-24.3,0.0,14405.12\nR004,2024-12-11,-23.8,0.0,22978.82\nR004,2024-12-12,-10.1,0.0,1802.29\nR004,2024-12-13,-12.8,0.0,25430.42\nR004,2024-12-14,-16.8,0.0,11457.14\nR004,2024-12-15,-15.6,0.0,21405.69\nR004,2024-12-16,-9.8,4.9,0.0\nR004,2024-12-17,-22.9,0.0,24759.52\nR004,2024-12-18,-24.3,0.0,25322.31\nR004,2024-12-19,-21.7,0.0,19020.27\nR004,2024-12-20,-19.7,0.0,22402.14\nR004,2024-12-21,-16.5,0.8,0.0\nR004,2024-12-22,-16.2,2.2,1269.24\nR004,2024-12-23,-16.6,0.0,5964.4\nR004,2024-12-24,-17.7,0.1,18.0\nR004,2024-12-25,-24.5,0.0,25109.96\nR004,2024-12-26,-24.8,0.0,25162.38\nR004,2024-12-27,-24.4,0.0,5882.47\nR004,2024-12-28,-12.1,7.7,0.0\nR004,2024-12-29,-6.8,4.1,16229.74\nR004,2024-12-30,-8.3,6.2,28.15\nR004,2024-12-31,-15.3,0.2,24924.49\nR004,2025-01-01,-24.9,0.0,26018.95\nR004,2025-01-02,-17.4,1.0,17577.51\nR004,2025-01-03,-7.9,2.8,11717.13\nR004,2025-01-04,-3.2,13.7,0.0\nR004,2025-01-05,-6.1,0.3,25183.52\nR004,2025-01-06,-14.6,0.0,26249.1\nR004,2025-01-07,-15.0,0.0,9776.48\nR004,2025-01-08,-12.4,0.9,0.0\nR004,2025-01-09,-10.5,1.5,12184.05\nR004,2025-01-10,-8.6,8.3,0.0\nR004,2025-01-11,-5.4,5.1,424.09\nR004,2025-01-12,-7.7,3.4,9863.18\nR004,2025-01-13,-12.5,0.5,20493.91\nR004,2025-01-14,-16.7,0.0,27518.26\nR004,2025-01-15,-14.0,0.0,27011.09\nR004,2025-01-16,-17.2,0.0,27420.18\nR004,2025-01-17,-7.6,6.2,0.0\nR004,2025-01-18,-6.0,8.3,0.0\nR004,2025-01-19,-8.4,0.7,19310.22\nR004,2025-01-20,-4.6,4.8,0.0\nR004,2025-01-21,-5.7,1.0,107.37\nR004,2025-01-22,-4.8,6.0,0.0\nR004,2025-01-23,-14.8,1.7,26178.12\nR004,2025-01-24,-23.5,0.4,16094.75\nR004,2025-01-25,-25.6,0.0,10372.65\nR004,2025-01-26,-17.4,0.1,19292.81\nR004,2025-01-27,-15.5,0.0,7380.73\nR004,2025-01-28,-14.8,0.7,1203.45\nR004,2025-01-29,-11.5,3.1,8906.91\nR004,2025-01-30,-11.3,1.4,417.27\nR004,2025-01-31,-11.1,2.2,5552.43\nR004,2025-02-01,-13.7,2.6,2278.08\nR004,2025-02-02,-15.5,0.0,4512.31\nR004,2025-02-03,-12.7,0.2,18212.09\nR004,2025-02-04,-10.5,0.0,28312.37\nR004,2025-02-05,-13.4,0.0,32031.35\nR004,2025-02-06,-13.8,0.0,32150.56\nR004,2025-02-07,-15.0,0.0,32269.89\nR004,2025-02-08,-16.9,0.0,26636.81\nR004,2025-02-09,-15.7,5.2,228.03\nR004,2025-02-10,-12.1,5.6,0.0\nR004,2025-02-11,-6.5,1.3,20115.55\nR004,2025-02-12,-6.7,5.0,0.0\nR004,2025-02-13,-5.9,7.7,0.0\nR004,2025-02-14,-8.3,6.4,18387.92\nR004,2025-02-15,-10.7,0.1,32400.0\nR004,2025-02-16,-11.7,0.0,19805.34\nR004,2025-02-17,-10.7,0.0,32652.63\nR004,2025-02-18,-13.6,0.0,34229.38\nR004,2025-02-19,-14.0,0.0,34787.37\nR004,2025-02-20,-13.7,0.0,35259.61\nR004,2025-02-21,-13.4,0.0,35327.95\nR004,2025-02-22,-10.5,0.0,32454.18\nR004,2025-02-23,-8.5,0.0,22016.86\nR004,2025-02-24,-11.1,0.0,36061.93\nR004,2025-02-25,-10.0,0.0,37211.33\nR004,2025-02-26,-8.5,0.0,37640.5\nR004,2025-02-27,-7.0,3.3,10319.76\nR004,2025-02-28,-8.5,4.3,10037.74\nR004,2025-03-01,-9.9,2.4,15218.2\nR004,2025-03-02,-10.0,0.0,38552.63\nR004,2025-03-03,-9.7,0.0,38665.93\nR004,2025-03-04,-9.6,0.0,33795.85\nR004,2025-03-05,-8.8,0.5,30774.52\nR004,2025-03-06,-9.5,0.0,36503.24\nR004,2025-03-07,-7.0,6.0,0.0\nR004,2025-03-08,-4.2,1.6,20217.86\nR004,2025-03-09,-5.9,1.7,21106.99\nR004,2025-03-10,-7.0,3.7,29240.5\nR004,2025-03-11,-7.4,3.1,7057.79\nR004,2025-03-12,-7.4,4.2,32400.0\nR004,2025-03-13,-12.1,0.1,39600.0\nR004,2025-03-14,-9.2,0.0,39600.0\nR004,2025-03-15,-7.2,0.0,40944.96\nR004,2025-03-16,-5.1,0.0,41636.73\nR004,2025-03-17,-3.8,0.0,41795.6\nR004,2025-03-18,0.4,1.5,3104.85\nR004,2025-03-19,6.0,1.9,38115.28\nR004,2025-03-20,4.2,0.0,42231.13\nR004,2025-03-21,6.2,1.8,42376.21\nR004,2025-03-22,-1.5,6.6,39600.0\nR004,2025-03-23,-6.2,0.0,42665.41\nR004,2025-03-24,-3.9,4.0,6788.02\nR004,2025-03-25,-9.4,5.9,2559.1\nR004,2025-03-26,-8.9,1.2,40224.11\nR004,2025-03-27,-9.3,0.0,43878.86\nR004,2025-03-28,-5.2,0.0,44208.42\nR004,2025-03-29,-1.4,0.0,44156.58\nR004,2025-03-30,-0.3,0.0,43340.19\nR004,2025-03-31,1.8,0.0,45201.21\nR004,2025-04-01,1.8,2.0,43200.0\nR004,2025-04-02,0.8,8.6,17318.76\nR004,2025-04-03,-0.1,0.0,40671.45\nR004,2025-04-04,4.9,0.0,44232.55\nR004,2025-04-05,3.3,4.4,35962.16\nR004,2025-04-06,3.6,0.0,44820.44\nR004,2025-04-07,5.4,0.9,43964.14\nR004,2025-04-08,4.9,1.4,35504.76\nR004,2025-04-09,3.0,4.9,652.33\nR004,2025-04-10,0.4,0.1,45647.27\nR004,2025-04-11,2.7,0.0,47364.44\nR004,2025-04-12,10.3,0.0,47015.19\nR004,2025-04-13,13.1,0.0,49127.82\nR004,2025-04-14,12.7,3.0,40005.32\nR004,2025-04-15,4.6,7.6,32573.71\nR004,2025-04-16,6.1,0.0,43200.0\nR004,2025-04-17,6.7,0.5,46355.8\nR004,2025-04-18,-0.2,6.3,37720.45\nR004,2025-04-19,-3.3,2.8,35166.39\nR004,2025-04-20,-1.8,3.3,36761.64\nR004,2025-04-21,3.7,41.4,9634.09\nR004,2025-04-22,1.9,0.0,50400.0\nR004,2025-04-23,6.3,0.2,45578.49\nR004,2025-04-24,3.3,4.4,36557.1\nR004,2025-04-25,3.4,0.0,47421.55\nR004,2025-04-26,9.7,0.0,50400.0\nR004,2025-04-27,14.8,0.0,46800.0\nR004,2025-04-28,13.5,18.4,8369.66\nR004,2025-04-29,16.3,4.6,49299.35\nR004,2025-04-30,8.5,8.6,46800.0\nR004,2025-05-01,6.3,0.0,50750.02\nR004,2025-05-02,8.4,0.0,45673.24\nR004,2025-05-03,5.1,5.6,47430.69\nR004,2025-05-04,2.7,0.3,43701.14\nR004,2025-05-05,8.5,1.1,45356.56\nR004,2025-05-06,7.0,7.0,41871.18\nR004,2025-05-07,9.2,0.3,48363.87\nR004,2025-05-08,13.0,0.3,49060.78\nR004,2025-05-09,16.9,0.0,49611.69\nR004,2025-05-10,14.0,10.7,16752.55\nR004,2025-05-11,11.3,15.2,0.0\nR004,2025-05-12,7.0,0.0,52990.69\nR004,2025-05-13,5.2,5.4,0.0\nR004,2025-05-14,11.2,2.1,40160.66\nR004,2025-05-15,8.6,0.0,51729.25\nR004,2025-05-16,7.3,4.2,26495.31\nR004,2025-05-17,11.9,0.7,49470.78\nR004,2025-05-18,16.7,4.6,41444.68\nR004,2025-05-19,11.7,5.9,15515.66\nR004,2025-05-20,10.9,0.0,41095.63\nR004,2025-05-21,15.3,0.0,55235.03\nR004,2025-05-22,17.6,0.0,55988.28\nR004,2025-05-23,18.2,0.0,51660.81\nR004,2025-05-24,15.0,1.9,25097.65\nR004,2025-05-25,13.9,0.7,49618.71\nR004,2025-05-26,9.9,14.6,18542.99\nR004,2025-05-27,2.3,10.9,45321.93\nR004,2025-05-28,0.7,9.3,20247.96\nR004,2025-05-29,2.9,2.4,46943.59\nR004,2025-05-30,6.3,0.4,54797.83\nR004,2025-05-31,10.3,9.9,47621.23\nR004,2025-06-01,14.8,0.2,57600.0\nR004,2025-06-02,17.7,0.0,57126.26\nR004,2025-06-03,15.1,1.3,26835.31\nR004,2025-06-04,17.2,0.1,56098.04\nR004,2025-06-05,17.8,0.0,56847.3\nR004,2025-06-06,16.9,9.4,54693.85\nR004,2025-06-07,18.5,0.7,54101.29\nR004,2025-06-08,18.1,0.6,46563.71\nR004,2025-06-09,15.2,38.7,9519.54\nR004,2025-06-10,14.4,15.6,41245.17\nR004,2025-06-11,14.8,6.7,48157.96\nR004,2025-06-12,18.3,2.3,49060.46\nR004,2025-06-13,19.6,0.0,57600.0\nR004,2025-06-14,20.1,0.0,51912.3\nR004,2025-06-15,19.2,5.9,44545.62\nR004,2025-06-16,16.8,20.8,1682.6\nR004,2025-06-17,18.9,1.2,50058.99\nR004,2025-06-18,20.8,0.0,57600.0\nR004,2025-06-19,22.5,0.4,52292.05\nR004,2025-06-20,22.6,0.6,52285.36\nR004,2025-06-21,17.8,10.1,33831.68\nR004,2025-06-22,19.0,0.0,57600.0\nR004,2025-06-23,20.4,0.0,57600.0\nR004,2025-06-24,16.2,4.0,29783.51\nR004,2025-06-25,16.6,1.9,51925.63\nR004,2025-06-26,14.8,16.5,33110.27\nR004,2025-06-27,15.2,0.0,54000.0\nR004,2025-06-28,16.4,10.4,46800.0\nR004,2025-06-29,16.4,4.7,42283.87\nR004,2025-06-30,16.2,5.3,47894.97\nR004,2025-07-01,14.8,15.1,28202.43\nR004,2025-07-02,14.5,13.7,10121.82\nR004,2025-07-03,17.4,0.5,46007.77\nR004,2025-07-04,19.7,0.0,57600.0\nR004,2025-07-05,21.3,0.0,54001.33\nR004,2025-07-06,17.7,7.7,41040.6\nR004,2025-07-07,17.4,0.8,42642.58\nR004,2025-07-08,15.5,13.2,3804.14\nR004,2025-07-09,16.0,6.7,10325.99\nR004,2025-07-10,17.7,0.0,35976.58\nR004,2025-07-11,19.5,4.4,42438.61\nR004,2025-07-12,16.2,23.0,28953.01\nR004,2025-07-13,17.1,7.2,48031.46\nR004,2025-07-14,18.2,1.5,52915.09\nR004,2025-07-15,16.0,6.9,41221.89\nR004,2025-07-16,18.0,0.6,54470.89\nR004,2025-07-17,18.8,0.0,52911.57\nR004,2025-07-18,19.6,0.8,50679.65\nR004,2025-07-19,21.5,2.9,44100.73\nR004,2025-07-20,15.4,9.6,36369.95\nR004,2025-07-21,13.7,4.8,51885.37\nR004,2025-07-22,10.3,2.7,42977.5\nR004,2025-07-23,9.7,0.0,35552.04\nR004,2025-07-24,12.0,0.0,55041.69\nR004,2025-07-25,15.9,0.0,54184.13\nR004,2025-07-26,18.4,0.0,54668.43\nR004,2025-07-27,19.3,0.0,54000.0\nR004,2025-07-28,18.5,0.3,32507.67\nR004,2025-07-29,17.2,7.1,19047.17\nR004,2025-07-30,18.4,4.2,45151.98\nR004,2025-07-31,18.8,1.1,46007.96\nR004,2025-08-01,16.2,9.4,27170.33\nR004,2025-08-02,17.9,0.0,47222.79\nR004,2025-08-03,14.6,24.5,1044.62\nR004,2025-08-04,10.8,3.9,44810.83\nR004,2025-08-05,9.1,4.3,46626.64\nR004,2025-08-06,10.3,0.4,33436.88\nR004,2025-08-07,9.7,4.8,27993.94\nR004,2025-08-08,10.2,2.6,24866.9\nR004,2025-08-09,11.4,5.8,13024.28\nR004,2025-08-10,10.0,18.5,15025.56\nR004,2025-08-11,12.1,9.2,44003.13\nR004,2025-08-12,15.2,0.9,39600.0\nR004,2025-08-13,13.9,9.0,32542.6\nR004,2025-08-14,15.9,4.9,39723.93\nR004,2025-08-15,15.0,5.2,39344.01\nR004,2025-08-16,15.1,2.6,43655.33\nR004,2025-08-17,15.7,1.9,41254.96\nR004,2025-08-18,15.1,4.7,49633.59\nR004,2025-08-19,13.2,6.7,34676.17\nR004,2025-08-20,12.4,2.5,41591.81\nR004,2025-08-21,14.3,0.0,44743.57\nR004,2025-08-22,15.7,0.0,49994.04\nR004,2025-08-23,15.5,9.1,47006.74\nR004,2025-08-24,14.2,11.1,40387.24\nR004,2025-08-25,13.1,23.2,0.0\nR004,2025-08-26,14.1,6.2,35502.72\nR004,2025-08-27,12.6,4.2,35336.81\nR004,2025-08-28,13.6,2.4,47468.29\nR004,2025-08-29,14.4,0.0,44787.59\nR004,2025-08-30,13.5,2.4,30141.87\nR004,2025-08-31,11.6,11.3,20742.45\nR004,2025-09-01,9.5,19.4,444.03\nR004,2025-09-02,10.5,0.0,46646.46\nR004,2025-09-03,9.4,6.0,3175.28\nR004,2025-09-04,9.2,14.2,4832.89\nR004,2025-09-05,9.5,2.8,25868.32\nR004,2025-09-06,9.4,0.0,39618.86\nR004,2025-09-07,9.2,0.1,42020.95\nR004,2025-09-08,10.0,1.5,36032.9\nR004,2025-09-09,11.0,24.8,17406.01\nR004,2025-09-10,10.5,12.6,32111.52\nR004,2025-09-11,10.0,0.0,38687.15\nR004,2025-09-12,10.5,3.1,41254.99\nR004,2025-09-13,7.6,10.0,1369.82\nR004,2025-09-14,5.9,0.9,10619.57\nR004,2025-09-15,6.5,0.0,31632.0\nR004,2025-09-16,7.4,0.0,43200.0\nR004,2025-09-17,7.9,0.0,39531.85\nR004,2025-09-18,9.9,0.0,43200.0\nR004,2025-09-19,10.5,0.1,41647.73\nR004,2025-09-20,11.2,0.1,22854.83\nR004,2025-09-21,10.8,0.8,27931.51\nR004,2025-09-22,11.1,0.1,41493.24\nR004,2025-09-23,10.8,1.1,20757.63\nR004,2025-09-24,10.7,3.2,19494.29\nR004,2025-09-25,13.4,0.1,38913.55\nR004,2025-09-26,14.9,0.0,42286.62\nR004,2025-09-27,13.5,0.0,39872.58\nR004,2025-09-28,14.3,0.4,33673.36\nR004,2025-09-29,4.1,8.5,0.0\nR004,2025-09-30,1.8,0.1,205.6\nR004,2025-10-01,0.8,1.2,10750.0\nR004,2025-10-02,1.5,0.6,28838.5\nR004,2025-10-03,1.3,12.5,0.0\nR004,2025-10-04,-0.7,2.6,219.81\nR004,2025-10-05,-1.3,9.3,5423.95\nR004,2025-10-06,-2.7,1.7,30071.88\nR004,2025-10-07,-1.2,7.7,0.0\nR004,2025-10-08,-2.4,0.1,30761.47\nR004,2025-10-09,-1.3,0.8,29710.83\nR004,2025-10-10,-1.9,2.8,31616.65\nR004,2025-10-11,-2.0,0.0,31131.34\nR004,2025-10-12,-2.6,0.0,36000.0\nR004,2025-10-13,-3.0,0.5,20820.4\nR004,2025-10-14,-2.9,1.0,9873.42\nR004,2025-10-15,-3.6,0.0,33424.8\nR004,2025-10-16,-4.6,0.0,35857.74\nR004,2025-10-17,-3.4,0.0,35857.58\nR004,2025-10-18,-2.2,0.0,35855.5\nR004,2025-10-19,-2.4,0.0,35565.15\nR004,2025-10-20,-0.2,0.0,31178.79\nR004,2025-10-21,-0.8,0.4,29483.49\nR004,2025-10-22,0.1,0.0,34157.68\nR004,2025-10-23,-0.1,0.0,33405.7\nR004,2025-10-24,0.2,0.0,33478.54\nR004,2025-10-25,-0.6,0.0,33550.99\nR004,2025-10-26,0.4,0.9,16406.27\nR004,2025-10-27,2.1,0.4,6013.49\nR004,2025-10-28,0.6,0.3,1999.23\nR004,2025-10-29,0.7,0.2,2150.31\nR004,2025-10-30,2.0,0.0,30662.51\nR004,2025-10-31,2.2,0.0,26479.8\nR004,2025-11-01,2.1,0.0,29508.03\nR004,2025-11-02,2.0,3.3,19485.66\nR004,2025-11-03,0.3,2.2,0.0\nR004,2025-11-04,1.6,7.1,3814.91\nR004,2025-11-05,-1.2,2.6,0.0\nR004,2025-11-06,-5.9,0.2,24641.97\nR004,2025-11-07,-5.9,2.1,438.04\nR004,2025-11-08,-2.0,8.0,230.22\nR004,2025-11-09,-0.1,13.8,0.0\nR004,2025-11-10,-0.5,11.3,0.0\nR004,2025-11-11,-6.7,4.2,326.61\nR004,2025-11-12,-2.1,9.7,0.0\nR004,2025-11-13,-2.7,17.9,10800.0\nR004,2025-11-14,-8.9,4.4,7106.25\nR004,2025-11-15,-11.7,0.6,26374.01\nR004,2025-11-16,-16.5,0.0,28800.0\nR004,2025-11-17,-14.2,0.0,28800.0\nR004,2025-11-18,-5.9,1.0,1191.56\nR004,2025-11-19,-8.0,0.0,22268.52\nR004,2025-11-20,-4.1,0.0,28772.16\nR004,2025-11-21,-1.2,18.3,20.12\nR004,2025-11-22,-6.9,2.7,21741.69\nR004,2025-11-23,-9.6,0.7,1121.92\nR004,2025-11-24,-2.5,20.7,0.0\nR004,2025-11-25,-9.1,2.9,0.0\nR004,2025-11-26,-4.5,7.9,0.0\nR004,2025-11-27,-0.8,27.7,0.0\nR004,2025-11-28,-6.9,7.1,4065.37\nR004,2025-11-29,-18.2,0.9,24444.02\nR004,2025-11-30,-28.1,0.0,27928.37\nR004,2025-12-01,-19.4,1.1,5554.2\nR004,2025-12-02,-10.0,2.4,0.0\nR004,2025-12-03,-6.9,0.5,23272.51\nR004,2025-12-04,-7.4,6.9,0.0\nR004,2025-12-05,-3.6,18.9,0.0\nR004,2025-12-06,-10.7,1.9,10921.63\nR004,2025-12-07,-8.7,12.7,0.0\nR004,2025-12-08,-8.3,5.3,0.0\nR004,2025-12-09,-14.1,1.1,18070.09\nR004,2025-12-10,-21.3,0.0,25583.43\nR004,2025-12-11,-19.2,0.0,5288.72\nR004,2025-12-12,-18.5,0.0,17114.82\nR004,2025-12-13,-16.3,1.1,21280.47\nR004,2025-12-14,-6.5,10.3,0.0\nR004,2025-12-15,-5.7,2.1,19367.2\nR004,2025-12-16,-9.0,2.4,10834.07\nR004,2025-12-17,-12.6,6.8,0.0\nR004,2025-12-18,-16.0,1.6,17454.4\nR004,2025-12-19,-13.1,7.5,0.0\nR004,2025-12-20,-9.2,1.8,2554.8\nR004,2025-12-21,-8.5,8.1,24.78\nR004,2025-12-22,-14.8,2.1,0.0\nR004,2025-12-23,-8.4,4.8,21.3\nR004,2025-12-24,-12.1,0.0,22385.01\nR004,2025-12-25,-11.8,0.0,13169.0\nR004,2025-12-26,-10.4,4.0,2924.86\nR004,2025-12-27,-16.4,1.3,13161.34\nR004,2025-12-28,-15.5,0.2,24742.12\nR004,2025-12-29,-20.8,0.0,25318.24\nR004,2025-12-30,-24.2,0.0,25892.92\nR004,2025-12-31,-22.3,0.0,25427.93\n'))

# Отдельный, более поздний снимок инфраструктуры.
infrastructure = pd.read_json(StringIO('[{"resort_id": "R001", "name_ru": "Роза Хутор", "food_objects_6km": 107, "lodging_objects_6km": 86, "osm_radius_km": 6, "osm_data_timestamp": "2026-09-14T12:12:58Z", "latitude": 43.64639, "longitude": 40.33222, "osm_overlap_warning": "Possible overlap with Газпром Поляна (6.0 km), Красная Поляна (7.3 km)"}, {"resort_id": "R002", "name_ru": "Красная Поляна", "food_objects_6km": 259, "lodging_objects_6km": 255, "osm_radius_km": 6, "osm_data_timestamp": "2026-09-14T12:12:58Z", "latitude": 43.68611, "longitude": 40.26, "osm_overlap_warning": "Possible overlap with Газпром Поляна (1.6 km), Роза Хутор (7.3 km)"}, {"resort_id": "R003", "name_ru": "Газпром Поляна", "food_objects_6km": 243, "lodging_objects_6km": 223, "osm_radius_km": 6, "osm_data_timestamp": "2026-09-14T12:12:58Z", "latitude": 43.6845194, "longitude": 40.2801722, "osm_overlap_warning": "Possible overlap with Красная Поляна (1.6 km), Роза Хутор (6.0 km)"}, {"resort_id": "R004", "name_ru": "Шерегеш", "food_objects_6km": 57, "lodging_objects_6km": 233, "osm_radius_km": 6, "osm_data_timestamp": "2026-09-14T12:12:58Z", "latitude": 52.95337967, "longitude": 87.95548035, "osm_overlap_warning": ""}, {"resort_id": "R005", "name_ru": "Архыз", "food_objects_6km": 69, "lodging_objects_6km": 187, "osm_radius_km": 6, "osm_data_timestamp": "2026-09-14T12:15:00Z", "latitude": 43.54333, "longitude": 41.18167, "osm_overlap_warning": ""}, {"resort_id": "R006", "name_ru": "Домбай", "food_objects_6km": 51, "lodging_objects_6km": 85, "osm_radius_km": 6, "osm_data_timestamp": "2026-09-14T12:15:00Z", "latitude": 43.289843, "longitude": 41.624341, "osm_overlap_warning": ""}, {"resort_id": "R007", "name_ru": "Эльбрус", "food_objects_6km": 88, "lodging_objects_6km": 93, "osm_radius_km": 6, "osm_data_timestamp": "2026-09-14T12:15:00Z", "latitude": 43.255775, "longitude": 42.513551, "osm_overlap_warning": "Possible overlap with Чегет (1.4 km)"}, {"resort_id": "R008", "name_ru": "Чегет", "food_objects_6km": 82, "lodging_objects_6km": 94, "osm_radius_km": 6, "osm_data_timestamp": "2026-09-14T12:15:00Z", "latitude": 43.24472, "longitude": 42.52139, "osm_overlap_warning": "Possible overlap with Эльбрус (1.4 km)"}, {"resort_id": "R009", "name_ru": "Мамисон", "food_objects_6km": 0, "lodging_objects_6km": 0, "osm_radius_km": 6, "osm_data_timestamp": "2026-09-14T12:15:00Z", "latitude": 42.6659, "longitude": 43.8494, "osm_overlap_warning": "Possible overlap with Цей (11.5 km)"}, {"resort_id": "R010", "name_ru": "Цей", "food_objects_6km": 8, "lodging_objects_6km": 6, "osm_radius_km": 6, "osm_data_timestamp": "2026-09-14T12:15:00Z", "latitude": 42.7667, "longitude": 43.8833, "osm_overlap_warning": "Possible overlap with Мамисон (11.5 km)"}, {"resort_id": "R011", "name_ru": "Манжерок", "food_objects_6km": 35, "lodging_objects_6km": 57, "osm_radius_km": 6, "osm_data_timestamp": "2026-09-14T12:15:00Z", "latitude": 51.811, "longitude": 85.801, "osm_overlap_warning": ""}, {"resort_id": "R012", "name_ru": "Телецкий", "food_objects_6km": 25, "lodging_objects_6km": 43, "osm_radius_km": 6, "osm_data_timestamp": "2026-09-14T12:15:00Z", "latitude": 51.771, "longitude": 87.23, "osm_overlap_warning": ""}, {"resort_id": "R013", "name_ru": "Большой Вудъявр", "food_objects_6km": 37, "lodging_objects_6km": 14, "osm_radius_km": 6, "osm_data_timestamp": "2026-09-14T12:16:01Z", "latitude": 67.61, "longitude": 33.69, "osm_overlap_warning": "Possible overlap with КолаСпортЛэнд (1.7 km), Кукисвумчорр (6.7 km)"}, {"resort_id": "R014", "name_ru": "Кукисвумчорр", "food_objects_6km": 8, "lodging_objects_6km": 4, "osm_radius_km": 6, "osm_data_timestamp": "2026-09-14T12:16:01Z", "latitude": 67.669, "longitude": 33.72, "osm_overlap_warning": "Possible overlap with Большой Вудъявр (6.7 km), КолаСпортЛэнд (8.1 km)"}, {"resort_id": "R015", "name_ru": "КолаСпортЛэнд", "food_objects_6km": 36, "lodging_objects_6km": 12, "osm_radius_km": 6, "osm_data_timestamp": "2026-09-14T12:16:01Z", "latitude": 67.6, "longitude": 33.66, "osm_overlap_warning": "Possible overlap with Большой Вудъявр (1.7 km), Кукисвумчорр (8.1 km)"}, {"resort_id": "R016", "name_ru": "Горный воздух", "food_objects_6km": 146, "lodging_objects_6km": 51, "osm_radius_km": 6, "osm_data_timestamp": "2026-09-14T12:16:01Z", "latitude": 46.955, "longitude": 142.789, "osm_overlap_warning": ""}, {"resort_id": "R017", "name_ru": "Холдоми", "food_objects_6km": 0, "lodging_objects_6km": 0, "osm_radius_km": 6, "osm_data_timestamp": "2026-09-14T12:16:01Z", "latitude": 50.72, "longitude": 136.41, "osm_overlap_warning": ""}, {"resort_id": "R018", "name_ru": "Морозная гора", "food_objects_6km": 31, "lodging_objects_6km": 18, "osm_radius_km": 6, "osm_data_timestamp": "2026-09-14T12:16:01Z", "latitude": 53.183, "longitude": 158.451, "osm_overlap_warning": ""}, {"resort_id": "R019", "name_ru": "Бобровый лог", "food_objects_6km": 71, "lodging_objects_6km": 36, "osm_radius_km": 6, "osm_data_timestamp": "2026-09-14T12:16:01Z", "latitude": 55.96, "longitude": 92.793, "osm_overlap_warning": ""}, {"resort_id": "R020", "name_ru": "Соболиная гора", "food_objects_6km": 29, "lodging_objects_6km": 18, "osm_radius_km": 6, "osm_data_timestamp": "2026-09-14T12:16:01Z", "latitude": 51.5, "longitude": 104.15, "osm_overlap_warning": ""}, {"resort_id": "R021", "name_ru": "Гора Белая", "food_objects_6km": 2, "lodging_objects_6km": 3, "osm_radius_km": 6, "osm_data_timestamp": "2026-09-14T12:16:01Z", "latitude": 57.66, "longitude": 59.7, "osm_overlap_warning": ""}, {"resort_id": "R022", "name_ru": "Евразия", "food_objects_6km": 47, "lodging_objects_6km": 7, "osm_radius_km": 6, "osm_data_timestamp": "2026-09-14T12:16:01Z", "latitude": 55.109, "longitude": 59.732, "osm_overlap_warning": ""}, {"resort_id": "R023", "name_ru": "Аджигардак", "food_objects_6km": 11, "lodging_objects_6km": 1, "osm_radius_km": 6, "osm_data_timestamp": "2026-09-14T12:16:01Z", "latitude": 54.98, "longitude": 57.27, "osm_overlap_warning": ""}, {"resort_id": "R024", "name_ru": "Солнечная долина", "food_objects_6km": 5, "lodging_objects_6km": 14, "osm_radius_km": 6, "osm_data_timestamp": "2026-09-14T12:16:01Z", "latitude": 55.04, "longitude": 59.96, "osm_overlap_warning": ""}, {"resort_id": "R025", "name_ru": "Завьялиха", "food_objects_6km": 3, "lodging_objects_6km": 1, "osm_radius_km": 6, "osm_data_timestamp": "2026-09-14T12:17:00Z", "latitude": 54.81, "longitude": 58.62, "osm_overlap_warning": ""}, {"resort_id": "R026", "name_ru": "Абзаково", "food_objects_6km": 15, "lodging_objects_6km": 55, "osm_radius_km": 6, "osm_data_timestamp": "2026-09-14T12:17:00Z", "latitude": 53.8, "longitude": 58.63, "osm_overlap_warning": ""}, {"resort_id": "R027", "name_ru": "Мраткино", "food_objects_6km": 34, "lodging_objects_6km": 4, "osm_radius_km": 6, "osm_data_timestamp": "2026-09-14T12:17:00Z", "latitude": 53.96, "longitude": 58.39, "osm_overlap_warning": ""}, {"resort_id": "R028", "name_ru": "Губаха", "food_objects_6km": 22, "lodging_objects_6km": 17, "osm_radius_km": 6, "osm_data_timestamp": "2026-09-14T12:17:00Z", "latitude": 58.84, "longitude": 57.56, "osm_overlap_warning": ""}, {"resort_id": "R029", "name_ru": "Игора", "food_objects_6km": 28, "lodging_objects_6km": 39, "osm_radius_km": 6, "osm_data_timestamp": "2026-09-14T12:17:00Z", "latitude": 60.51444444, "longitude": 30.21025, "osm_overlap_warning": ""}, {"resort_id": "R030", "name_ru": "Сорочаны", "food_objects_6km": 34, "lodging_objects_6km": 30, "osm_radius_km": 6, "osm_data_timestamp": "2026-09-14T12:17:00Z", "latitude": 56.276, "longitude": 37.5645, "osm_overlap_warning": ""}, {"resort_id": "R031", "name_ru": "Красная Глинка", "food_objects_6km": 24, "lodging_objects_6km": 8, "osm_radius_km": 6, "osm_data_timestamp": "2026-09-14T12:17:00Z", "latitude": 53.3757028, "longitude": 50.171333, "osm_overlap_warning": ""}, {"resort_id": "R032", "name_ru": "Гладенькая", "food_objects_6km": 10, "lodging_objects_6km": 3, "osm_radius_km": 6, "osm_data_timestamp": "2026-09-14T12:17:00Z", "latitude": 53.1, "longitude": 91.4, "osm_overlap_warning": ""}]'))

# Метаданные источника: период, модель, координаты точки и ячейки, высота.
history_metadata = {'R001': {'file': 'R001_1996_2025.csv', 'start': '1996-01-01', 'end': '2025-12-31', 'years': 30, 'days': 10958, 'model': 'ERA5', 'timezone': 'Europe/Moscow', 'requested_latitude': 43.64639, 'requested_longitude': 40.33222, 'grid_latitude': 43.75, 'grid_longitude': 40.25, 'elevation_m': 950.0, 'units': {'time': 'iso8601', 'temperature_2m_mean': '°C', 'precipitation_sum': 'mm', 'rain_sum': 'mm', 'snowfall_sum': 'cm', 'sunshine_duration': 's', 'wind_speed_10m_mean': 'm/s', 'wind_gusts_10m_max': 'm/s', 'relative_humidity_2m_mean': '%', 'cloud_cover_mean': '%'}, 'source_url': 'https://archive-api.open-meteo.com/v1/archive?latitude=43.64639&longitude=40.33222&start_date=1996-01-01&end_date=2025-12-31&daily=temperature_2m_mean%2Cprecipitation_sum%2Crain_sum%2Csnowfall_sum%2Csunshine_duration%2Cwind_speed_10m_mean%2Cwind_gusts_10m_max%2Crelative_humidity_2m_mean%2Ccloud_cover_mean&models=era5&timezone=auto&wind_speed_unit=ms', 'collected_at': '2026-10-13T08:11:59.648682+00:00', 'visibility_status': 'not_available_in_selected_archive'}, 'R002': {'file': 'R002_1996_2025.csv', 'start': '1996-01-01', 'end': '2025-12-31', 'years': 30, 'days': 10958, 'model': 'ERA5', 'timezone': 'Europe/Moscow', 'requested_latitude': 43.68611, 'requested_longitude': 40.26, 'grid_latitude': 43.75, 'grid_longitude': 40.25, 'elevation_m': 513.0, 'units': {'time': 'iso8601', 'temperature_2m_mean': '°C', 'precipitation_sum': 'mm', 'rain_sum': 'mm', 'snowfall_sum': 'cm', 'sunshine_duration': 's', 'wind_speed_10m_mean': 'm/s', 'wind_gusts_10m_max': 'm/s', 'relative_humidity_2m_mean': '%', 'cloud_cover_mean': '%'}, 'source_url': 'https://archive-api.open-meteo.com/v1/archive?latitude=43.68611&longitude=40.26&start_date=1996-01-01&end_date=2025-12-31&daily=temperature_2m_mean%2Cprecipitation_sum%2Crain_sum%2Csnowfall_sum%2Csunshine_duration%2Cwind_speed_10m_mean%2Cwind_gusts_10m_max%2Crelative_humidity_2m_mean%2Ccloud_cover_mean&models=era5&timezone=auto&wind_speed_unit=ms', 'collected_at': '2026-10-13T08:16:36.345423+00:00', 'visibility_status': 'not_available_in_selected_archive'}, 'R003': {'file': 'R003_1996_2025.csv', 'start': '1996-01-01', 'end': '2025-12-31', 'years': 30, 'days': 10958, 'model': 'ERA5', 'timezone': 'Europe/Moscow', 'requested_latitude': 43.6845194, 'requested_longitude': 40.2801722, 'grid_latitude': 43.75, 'grid_longitude': 40.25, 'elevation_m': 546.0, 'units': {'time': 'iso8601', 'temperature_2m_mean': '°C', 'precipitation_sum': 'mm', 'rain_sum': 'mm', 'snowfall_sum': 'cm', 'sunshine_duration': 's', 'wind_speed_10m_mean': 'm/s', 'wind_gusts_10m_max': 'm/s', 'relative_humidity_2m_mean': '%', 'cloud_cover_mean': '%'}, 'source_url': 'https://archive-api.open-meteo.com/v1/archive?latitude=43.6845194&longitude=40.2801722&start_date=1996-01-01&end_date=2025-12-31&daily=temperature_2m_mean%2Cprecipitation_sum%2Crain_sum%2Csnowfall_sum%2Csunshine_duration%2Cwind_speed_10m_mean%2Cwind_gusts_10m_max%2Crelative_humidity_2m_mean%2Ccloud_cover_mean&models=era5&timezone=auto&wind_speed_unit=ms', 'collected_at': '2026-10-13T08:23:32.777997+00:00', 'visibility_status': 'not_available_in_selected_archive'}, 'R004': {'file': 'R004_1996_2025.csv', 'start': '1996-01-01', 'end': '2025-12-31', 'years': 30, 'days': 10958, 'model': 'ERA5', 'timezone': 'Asia/Novokuznetsk', 'requested_latitude': 52.95337967, 'requested_longitude': 87.95548035, 'grid_latitude': 53.0, 'grid_longitude': 88.0, 'elevation_m': 695.0, 'units': {'time': 'iso8601', 'temperature_2m_mean': '°C', 'precipitation_sum': 'mm', 'rain_sum': 'mm', 'snowfall_sum': 'cm', 'sunshine_duration': 's', 'wind_speed_10m_mean': 'm/s', 'wind_gusts_10m_max': 'm/s', 'relative_humidity_2m_mean': '%', 'cloud_cover_mean': '%'}, 'source_url': 'https://archive-api.open-meteo.com/v1/archive?latitude=52.95337967&longitude=87.95548035&start_date=1996-01-01&end_date=2025-12-31&daily=temperature_2m_mean%2Cprecipitation_sum%2Crain_sum%2Csnowfall_sum%2Csunshine_duration%2Cwind_speed_10m_mean%2Cwind_gusts_10m_max%2Crelative_humidity_2m_mean%2Ccloud_cover_mean&models=era5&timezone=auto&wind_speed_unit=ms', 'collected_at': '2026-10-13T08:29:15.818705+00:00', 'visibility_status': 'not_available_in_selected_archive'}}


*Подготовка и валидация данных*

На этом этапе мы приводим типы данных к корректным форматам и проверяем целостность архива погоды.

- Даты конвертируем в тип datetime, чтобы корректно фильтровать периоды.

- Числовые колонки (температура, осадки, солнечное сияние) переводим в числа, заменяя некорректные значения на NaN.

- Проверки через assert гарантируют, что в данных нет дублей, отрицательных значений и других логических ошибок, которые исказили бы итоговый рейтинг курортов.

In [3]:
# Приведение типов
weather['date'] = pd.to_datetime(weather['date']).dt.normalize()
for column in ['temperature_c', 'precipitation_mm', 'sunshine_seconds']:
    weather[column] = pd.to_numeric(weather[column], errors='coerce')

# Проверяем целостность архива: уникальность дней, корректность типов и физическую допустимость значений. Ошибки здесь привели бы к искажению рейтинга курортов
assert not weather.duplicated(['resort_id', 'date']).any()
assert infrastructure['resort_id'].is_unique
assert not (weather['precipitation_mm'] < 0).any()
assert not (weather['sunshine_seconds'] < 0).any()

first_date = weather['date'].min().date()
last_date = weather['date'].max().date()

# Проверим реальный период по каждому загруженному курорту
coverage = weather.groupby('resort_id').agg(
    first_date=('date', 'min'), last_date=('date', 'max'), days=('date', 'nunique')
)
print('Архив:', first_date, '—', last_date)
print('Строк дневной погоды:', len(weather))
print('Курортов с дневной историей:', weather['resort_id'].nunique())
display(coverage)

# Дату по умолчанию тоже берем из доступных данных
default_start = pd.Timestamp('2025-01-15').date()
default_end = pd.Timestamp('2025-01-21').date()


Архив: 1996-01-01 — 2025-12-31
Строк дневной погоды: 43832
Курортов с дневной историей: 4


,first_date,last_date,days
resort_id,,,
R001,1996-01-01,2025-12-31,10958
R002,1996-01-01,2025-12-31,10958
R003,1996-01-01,2025-12-31,10958
R004,1996-01-01,2025-12-31,10958


In [4]:
# Проверка данных: пропуски и разумные диапазоны

# Сколько «пустых» значений появилось после приведения типов?
print('Пустых значений по колонкам:')
print(weather.isna().sum())

# Существуют ли пропуски в самой календарной сетке
days_per_resort = weather.groupby('resort_id')['date'].nunique()
expected_days = (last_date - first_date).days + 1
print('\nОжидается дней:', expected_days)
print(days_per_resort)

# Проверка здравого смысла: температура должна быть в разумных пределах
print('\nТемпература: мин', weather['temperature_c'].min(),
      'макс', weather['temperature_c'].max())

# Солнечное сияние за день не может превышать длину светового дня.
#    Долгота дня на широте 43° летом — максимум ~15.5 часов, это ~55 800 секунд.
print('Макс. солнечное сияние, сек:', weather['sunshine_seconds'].max(),
      '=', round(weather['sunshine_seconds'].max() / 3600, 1), 'часов')


Пустых значений по колонкам:
resort_id           0
date                0
temperature_c       0
precipitation_mm    0
sunshine_seconds    0
dtype: int64

Ожидается дней: 10958
resort_id
R001    10958
R002    10958
R003    10958
R004    10958
Name: date, dtype: int64

Температура: мин -39.7 макс 30.0
Макс. солнечное сияние, сек: 57664.61 = 16.0 часов


Результаты проверок подтверждают целостность и пригодность архива для расчёта рейтинга:

- Нет пропусков. 

- Полное покрытие по дням. 

- Диапазон температур реалистичен.

- Солнечное сияние в пределах здравого смысла.

## 1. Выбираем параметры отдыха

Галочки означают повышенную важность, а не обязательные условия. Доступны комфортная температура, меньше осадков, больше солнца, выбор мест питания и выбор жилья.

Температура - среднесуточная из архива. Солнце - продолжительность солнечного сияния за день. Это не длина светового дня и не расчет из облачности.

Питание и жилье - число объектов OSM в радиусе 6 км вокруг исходных точек курортов, не оценки качества и не цены.

In [5]:
parameter_names = {
    'temperature': 'Комфортная температура',
    'precipitation': 'Меньше осадков',
    'sunshine': 'Больше солнца',
    'food': 'Где поесть: выбор мест питания',
    'lodging': 'Выбор жилья'
}

# Один Checkbox — один параметр. Можно выбрать несколько.
checkboxes = {}
for key, label in parameter_names.items():
    checkboxes[key] = widgets.Checkbox(
        value=False, description=label, indent=False,
        layout=widgets.Layout(width='auto')
    )

unavailable = [
    widgets.Checkbox(description='Трассы по сложности — нужны данные', disabled=True, indent=False),
    widgets.Checkbox(description='Бюджет — нужны цены', disabled=True, indent=False),
    widgets.Checkbox(description='Качество ресторанов — нужны оценки', disabled=True, indent=False)
]

display(widgets.VBox(list(checkboxes.values()) + unavailable))

Выбор параметров

Пользователь отмечает, какие условия для отдыха ему важны. Галочка означает повышенную важность параметра, а не фильтр «только такие курорты». Три критерия (трассы, бюджет, качество ресторанов) показаны, но отключены - для них нет проверенных данных.

In [6]:
# Контроль: что выбрал пользователь
selected = [key for key, cb in checkboxes.items() if cb.value]
print('Выбрано параметров:', len(selected))
for key in selected:
    print(' -', key, '→', parameter_names[key])


Выбрано параметров: 0


### Шаг 2. Преобразование приоритетов в веса

Мы превращаем выбор пользователя (галочки) в числовые веса для расчёта рейтинга.

**Логика расчёта:** выбранный параметр получает 4 условные доли, невыбранный - 1. Затем доли нормализуются так, чтобы сумма весов была равна 100%.

Если параметров N, а выбрано k, сумма долей равна 4k + (N − k). Тогда:

- каждый выбранный получает 4 / (4k + N − k);
- каждый невыбранный - 1 / (4k + N − k);
- общий вес выбранных - 4k / (4k + N − k);
- остаток для всех невыбранных - (N − k) / (4k + N − k).

Пример: выбрали два из пяти параметров - всего 11 долей. Выбранные получают по 36,36%, остальные - по 9,09%. Чем больше параметров выбрано, тем меньше достаётся каждому: при выборе всех пяти вес каждого - ровно 20%.

Все выбранные параметры равноправны: порядок кликов ничего не меняет. Такой подход делает выбранные критерии заметно важнее, но сохраняет баланс между всеми параметрами.

**Если ничего не выбрано** - используются базовые веса, заданные куратором: осадки 30%, температура 25%, солнце 10%, питание 20%, жильё 15%. Это гарантирует, что модель всегда выдаст осмысленный рейтинг.

Важно: веса - это учебное решение, а не статистически доказанная важность параметров.


In [7]:
# Логика весов: 4 к 1. Если выбрано, то 4 доли, иначе 1
# Если ничего не выбрано, используем базовые веса
base_weights = {
    'temperature': 0.25,
    'precipitation': 0.30,
    'sunshine': 0.10,
    'food': 0.20,
    'lodging': 0.15
}


def make_weights(selected):
    # Если ничего не выбрано, возвращаем обычные веса
    if len(selected) == 0:
        return base_weights.copy()

    # Сначала раздаем доли: 4 выбранному, 1 остальным
    parts = {}
    for key in parameter_names:
        parts[key] = 4 if key in selected else 1

    # Потом превращаем доли в веса с общей суммой 1
    total_parts = sum(parts.values())
    weights = {}
    for key, value in parts.items():
        weights[key] = value / total_parts
    return weights


weights_output = widgets.Output()
display(weights_output)


def show_weights():
    selected = [key for key, checkbox in checkboxes.items() if checkbox.value]
    weights = make_weights(selected)
    with weights_output:
        clear_output(wait=True)
        rows = []
        for key in parameter_names:
            rows.append({
                'Параметр': parameter_names[key],
                'Выбран': key in selected,
                'Вес, %': weights[key] * 100
            })
        display(pd.DataFrame(rows).round(2))
        print('Общая сумма:', round(sum(weights.values()) * 100, 2), '%')
    return weights


show_weights()

Output()

{'temperature': 0.25,
 'precipitation': 0.3,
 'sunshine': 0.1,
 'food': 0.2,
 'lodging': 0.15}

In [8]:
# Проверка крайних случаев
cases = [
    [],                         # ничего не выбрано
    list(parameter_names.keys())  # все выбрано
]

for case in cases:
    w = make_weights(case)
    print('Выбор:', case if case else 'ничего')
    for k, v in w.items():
        print(f'  {k}: {v:.2%}')
    print()


Выбор: ничего
  temperature: 25.00%
  precipitation: 30.00%
  sunshine: 10.00%
  food: 20.00%
  lodging: 15.00%

Выбор: ['temperature', 'precipitation', 'sunshine', 'food', 'lodging']
  temperature: 20.00%
  precipitation: 20.00%
  sunshine: 20.00%
  food: 20.00%
  lodging: 20.00%



## Шаг 3. Выбор временного окна

Пользователь выбирает конкретный исторический период для сравнения курортов.

**Ограничения:**
- Календарь привязан к реальным границам архива: 01.01.1996 — 31.12.2025.
- Обе границы периода включаются в расчёт (15–21.01 = 7 дней).

**Проверки и логика:**
- Если даты не заданы, конец раньше начала или период вне архива — расчёт останавливается.
- Для каждого курорта проверяется наличие всех дней и показателей в периоде. При любых пропусках курорт исключается из сравнения целиком (без усечённых периодов).

**Важно:** мы сравниваем конкретные исторические даты, а не усредняем по месяцам. Климатический рейтинг по месяцам будет реализован отдельным блоком агрегации.


In [9]:
start_picker = widgets.DatePicker(
    description='Начало:', value=default_start, min=first_date, max=last_date
)
end_picker = widgets.DatePicker(
    description='Конец:', value=default_end, min=first_date, max=last_date
)
display(widgets.VBox([start_picker, end_picker]))

## Шаг 4. Переводим историю в баллы

Чтобы сравнивать курорты честно, каждый показатель переведем в единую шкалу 0–100 - только так их можно смешивать с весами из Шага 2:

| Показатель | 100 баллов | Частичный балл | 0 баллов |
| --- | --- | --- | --- |
| Температура | −5…+5 °C | −10…+10 → 70; −15…+15 → 30 | вне диапазона |
| Осадки | до 1 мм | до 5 мм → 60 | больше 5 мм → 20 |
| Солнце | ≥ 6 часов | 3 часа → 50 | 0 |
| Питание | ≥ 20 объектов | линейно до порога | 0 |
| Жильё | ≥ 30 объектов | линейно до порога | 0 |

**Несколько важных решений модели:**

- **Температура** - ступенчатая шкала для зимних прогулок, построенная на настоящей среднесуточной температуре архива (а не на среднем из минимума и максимума дня).
- **Осадки** - чем меньше, тем выше балл. Тип осадков не учитываем: это не рейтинг свежего снега, а сравнение условий для прогулок. Даже в самый дождливый день курорт не получает 0, чтобы один ливень не «обнулял» остальные преимущества места.
- **Солнце** - секунды из архива делим на 3600, получаем часы. Порог «6 часов = 100 баллов» — учебный: абсолютное число часов зависит от сезона и широты, и при сравнении разных сезонов это нужно помнить. Долю солнца от длины светового дня здесь не считаем.
- **Питание и жильё** - считаются от числа объектов OSM в радиусе 6 км. 20 и 30 объектов соответственно дают 100 баллов, а больше объектов балл не повышает - потолок защиты от «перекоса» в мегаполисах.

**Порядок расчёта:** сначала баллы каждого дня → затем средние за выбранный период → и только потом веса. Так веса применяются к характеристикам периода в целом, а не к отдельным дням.


In [10]:
def temperature_score(value):
    if pd.isna(value):
        return float('nan')
    if -5 <= value <= 5:
        return 100
    if -10 <= value <= 10:
        return 70
    if -15 <= value <= 15:
        return 30
    return 0


def precipitation_score(value):
    if pd.isna(value):
        return float('nan')
    if value <= 1:
        return 100
    if value <= 5:
        return 60
    return 20


weather_scores = weather.copy()
weather_scores['sunshine_h'] = weather_scores['sunshine_seconds'] / 3600
weather_scores['temperature'] = weather_scores['temperature_c'].apply(temperature_score)
weather_scores['precipitation'] = weather_scores['precipitation_mm'].apply(precipitation_score)
weather_scores['sunshine'] = (weather_scores['sunshine_h'] / 6).clip(upper=1) * 100

features = infrastructure.copy().rename(columns={'name_ru': 'resort'})
for column in ['food_objects_6km', 'lodging_objects_6km']:
    features[column] = pd.to_numeric(features[column], errors='coerce')
    assert not (features[column] < 0).any()
features['food'] = (features['food_objects_6km'] / 20).clip(upper=1) * 100
features['lodging'] = (features['lodging_objects_6km'] / 30).clip(upper=1) * 100

In [11]:
# Проверяем функции баллов на контрольных значениях
print('Температура:')
for t in [0, 5, 7, -12, 20, -20, None]:
    print(f'  {t} °C → {temperature_score(t)}')

print('Осадки:')
for p in [0, 0.9, 1, 3, 5.5, None]:
    print(f'  {p} мм → {precipitation_score(p)}')

# Солнце: проверим линейную шкалу
for h in [0, 3, 6, 10]:
    print(f'  {h} ч солнца → {min(h / 6, 1) * 100:.0f} баллов')


Температура:
  0 °C → 100
  5 °C → 100
  7 °C → 70
  -12 °C → 30
  20 °C → 0
  -20 °C → 0
  None °C → nan
Осадки:
  0 мм → 100
  0.9 мм → 100
  1 мм → 100
  3 мм → 60
  5.5 мм → 20
  None мм → nan
  0 ч солнца → 0 баллов
  3 ч солнца → 50 баллов
  6 ч солнца → 100 баллов
  10 ч солнца → 100 баллов


### Проверка логики начисления баллов (тестовые значения)

Перед запуском основного расчёта мы протестировали функции на контрольных точках, чтобы убедиться, что пороги и границы работают корректно.

**Что проверено:**
- **Границы диапазонов:** значения ровно на пороге (5 °C, 1 мм) попадают в нужную категорию.
- **Пропущенные данные:** `None` превращается в `NaN`, а не в 0 — это защищает среднее от искажений.
- **Потолки:** баллы не превышают 100 даже при экстремальных значениях (например, 10 часов солнца).

Результаты тестов подтверждают, что логика начисления баллов соответствует требованиям модели.


## Шаг 5. Расчёт рейтинга на выбранный период

Здесь соединяются все предыдущие шаги: баллы за каждый день (Шаг 4) усредняются за выбранный период и смешиваются по весам (Шаг 2).

**Логика расчёта:**
- Формула: `балл температуры * ее вес + балл осадков * их вес + ...`.
- Курорт участвует, только если в периоде есть все дни и все пять показателей. Веса при этом не перенормируются: иначе место с пропусками могло бы получить несправедливое преимущество.
- Курорты с неполными данными выводятся в отдельную таблицу «исключённые» - с указанием причины.

**Правила определения мест:**
- Места считаются до округления баллов.
- Одинаковые оценки получают одинаковое место: 1, 1, 3 (как в спорте).
- При равенстве баллов курорты сортируются по ID - это только порядок отображения, а не доказательство превосходства.


In [12]:
def rank_resorts(start, end, weights):
    if start is None or end is None:
        raise ValueError('Заполни обе даты.')
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    if end < start:
        raise ValueError('Конец периода не может быть раньше начала.')
    if start < pd.Timestamp(first_date) or end > pd.Timestamp(last_date):
        raise ValueError('Для этих дат нет скачанной дневной истории. Выбери период внутри архива.')
    if set(weights) != set(parameter_names):
        raise ValueError('Набор весов не совпадает с параметрами.')
    if any(value < 0 for value in weights.values()) or abs(sum(weights.values()) - 1) > 0.000001:
        raise ValueError('Веса должны быть неотрицательными и давать сумму 1.')

    period = weather_scores[weather_scores['date'].between(start, end)].copy()
    days_needed = (end - start).days + 1

    # Средние баллы и заполненность считаем для каждого курорта отдельно.
    means = period.groupby('resort_id')[
        ['temperature', 'precipitation', 'sunshine']
    ].mean()
    counts = period.groupby('resort_id')[
        ['temperature', 'precipitation', 'sunshine']
    ].count()
    valid_ids = counts.index[counts.eq(days_needed).all(axis=1)]
    means.loc[~means.index.isin(valid_ids), :] = float('nan')

    result = features.merge(means, on='resort_id', how='left', validate='one_to_one')
    columns = list(parameter_names)
    complete = result[columns].notna().all(axis=1)
    excluded = result.loc[~complete, ['resort_id', 'resort']].copy()
    excluded['reason'] = 'Нет всех дневных значений или показателей за выбранный период'
    result = result.loc[complete].copy()

    # Видно вклад каждого параметра: балл умножаем на вес.
    result['score'] = 0.0
    for key in parameter_names:
        result[key + '_contribution'] = result[key] * weights[key]
        result['score'] += result[key + '_contribution']

    result['rank'] = result['score'].rank(ascending=False, method='min').astype('Int64')
    result = result.sort_values(['score', 'resort_id'], ascending=[False, True]).reset_index(drop=True)
    return result, excluded, period


# Пример без интерфейса: обычный вызов функции для проверки понимания.
example_rating, example_excluded, example_period = rank_resorts(
    start_picker.value, end_picker.value, make_weights([])
)
example_rating[['resort', 'rank', 'score']].round(2).head(10)

,resort,rank,score
0,Роза Хутор,1,89.22
1,Красная Поляна,2,88.15
2,Газпром Поляна,2,88.15
3,Шерегеш,4,75.21


In [13]:
# Контроль: из чего сложился балл победителя
w = make_weights([])  # базовые веса, как в примере
top = example_rating.iloc[0]
print('Победитель:', top['resort'])
for key in parameter_names:
    print(f"  {parameter_names[key]:32s} балл {top[key]:6.2f} × вес {w[key]:.2f} = вклад {top[key] * w[key]:5.2f}")
print(f"  Итого: {top['score']:.2f}")


Победитель: Роза Хутор
  Комфортная температура           балл 100.00 × вес 0.25 = вклад 25.00
  Меньше осадков                   балл  65.71 × вес 0.30 = вклад 19.71
  Больше солнца                    балл  95.08 × вес 0.10 = вклад  9.51
  Где поесть: выбор мест питания   балл 100.00 × вес 0.20 = вклад 20.00
  Выбор жилья                      балл 100.00 × вес 0.15 = вклад 15.00
  Итого: 89.22


## Шаг 6. Карточка лидера за исторический период

Итог шага - наглядная карточка победителя рейтинга: даты, итоговый балл, выбранные приоритеты и вклад каждого параметра в результат.

**Что показываем:**
- Название курорта, итоговый балл (X из 100) и даты периода (обе включены).
- Приоритеты пользователя или пометку «Базовые веса», если ничего не выбрано.
- Таблицу вкладов: балл каждого параметра, его вес и вклад в итог - видно, из чего сложилась победа.
- Метаданные точки: запрошенные координаты и ячейка сетки ERA5 - погода привязана к конкретной точке, а не к «курорту в целом».

**Честные оговорки:**
- Если первое место делят несколько курортов, карточка прямо об этом сообщает.
- Карточка описывает прошлую погоду в сочетании со снимком инфраструктуры 2026 года. Она не восстанавливает цены, открытые трассы и состав ресторанов выбранного прошлого года.
- Количество объектов не равно их качеству.

При длинном периоде показываются первые 14 дневных строк; полная таблица хранится в переменной.


In [14]:
def make_top_card(rating, period, weights, selected, start, end):
    top = rating.iloc[0]
    tied_count = (rating['score'] == top['score']).sum()
    top_weather = period[period['resort_id'] == top['resort_id']].copy()
    point = history_metadata[str(top['resort_id'])]
    place = f"точка {point['requested_latitude']}, {point['requested_longitude']}; сетка ERA5 {point['grid_latitude']}, {point['grid_longitude']}"
    priority_text = ', '.join(parameter_names[key] for key in selected) if selected else 'Базовые веса'
    tie_text = f'Первое место делят {tied_count} курорта. Карточка одного из них.' if tied_count > 1 else 'Первое место в этом расчете.'

    # escape защищает HTML-разметку от текста в названиях исходных данных.
    details = []
    for key in parameter_names:
        details.append({
            'Параметр': parameter_names[key],
            'Балл': top[key],
            'Вес, %': weights[key] * 100,
            'Вклад в итог': top[key + '_contribution']
        })
    details_html = pd.DataFrame(details).round(2).to_html(index=False, escape=True)
    card = f'''
    <section style="border:1px solid #999;padding:20px;max-width:900px;color:#111;background:#fff">
      <p>УЧЕБНЫЙ РЕЙТИНГ · ИСТОРИЧЕСКАЯ ПОГОДА</p>
      <h2>{escape(str(top['resort']))}</h2>
      <p><b>{top['score']:.2f} из 100</b> · {escape(tie_text)}</p>
      <p>Даты: {start:%d.%m.%Y} — {end:%d.%m.%Y}, обе даты включены.</p>
      <p>Приоритеты: {escape(priority_text)}</p>
      <p>Мест питания в снимке OSM 2026 года: {int(top['food_objects_6km'])}; объектов проживания: {int(top['lodging_objects_6km'])}. Радиус: 6 км.</p>
      <p>Архивная погода для точки: {escape(str(place))}. Объекты инфраструктуры считались вокруг исходной точки курорта.</p>
      {details_html}
      <p>Количество объектов не равно их качеству. Это не проверка открытых трасс, цен или безопасности катания.</p>
      <p>Источник: Open-Meteo, OpenStreetMap contributors. Погода ERA5 за выбранные даты; инфраструктура из снимка 2026 года, а не из выбранного прошлого года.</p>
    </section>'''
    return HTML(card), top_weather


In [15]:
# Контроль: карточка лидера при разных приоритетах
card, top_days = make_top_card(
    example_rating, example_period, make_weights([]),
    [], start_picker.value, end_picker.value
)
display(card)
print('Дней в периоде у лидера:', len(top_days))
print('Мин. балл за день:', top_days[['temperature', 'precipitation', 'sunshine']].min().to_dict())


Параметр,Балл,"Вес, %",Вклад в итог
Комфортная температура,100.00,25.0,25.00
Меньше осадков,65.71,30.0,19.71
Больше солнца,95.08,10.0,9.51
Где поесть: выбор мест питания,100.00,20.0,20.00
Выбор жилья,100.00,15.0,15.00


Дней в периоде у лидера: 7
Мин. балл за день: {'temperature': 100.0, 'precipitation': 20.0, 'sunshine': 69.43662037037038}


## Шаг 7. Связываем интерфейс с расчётом

Финальная сборка: галочки приоритетов и календарь дат соединены с моделью в один живой инструмент.

**Как это устроено:**
- Изменение галочки обновляет и таблицу весов, и рейтинг.
- Изменение даты обновляет только рейтинг: веса от дат не зависят.
- Старый результат очищается **до** нового расчёта - при невалидных датах на экране не остаётся карточка от прошлого расчёта.
- Ошибки ввода показываются как понятные сообщения, а не как падение программы.

**Важно:** сама модель осталась в функциях `make_weights()` и `rank_resorts()`. Их можно вызывать отдельно, без виджетов - интерфейс и расчётная логика разделены.


In [16]:
result_output = widgets.Output()
display(result_output)


def show_result():
    selected = [key for key, checkbox in checkboxes.items() if checkbox.value]
    weights = make_weights(selected)
    with result_output:
        clear_output(wait=True)
        try:
            rating, excluded, period = rank_resorts(
                start_picker.value, end_picker.value, weights
            )
            if rating.empty:
                print('Для этого периода нет курортов с полными данными.')
                display(excluded)
                return

            print('Скачанная дневная история:', first_date, '—', last_date)
            print('Даты сравнения:', start_picker.value, '—', end_picker.value)
            print('Курортов в рейтинге:', len(rating))
            display(rating[['resort', 'rank', 'score']].round(2))
            if not excluded.empty:
                print('Не хватает данных для следующих курортов:')
                display(excluded)

            card, top_weather = make_top_card(
                rating, period, weights, selected,
                start_picker.value, end_picker.value
            )
            display(card)
            print('Погода лидера из исторического реанализа, первые 14 дней:')
            display(top_weather[[
                'date', 'temperature_c', 'precipitation_mm', 'sunshine_h'
            ]].head(14))
            print('Всего дней в выбранной истории лидера:', len(top_weather))
        except ValueError as error:
            print(str(error))


def priorities_changed(change):
    show_weights()
    show_result()


for checkbox in checkboxes.values():
    checkbox.observe(priorities_changed, names='value')
start_picker.observe(lambda change: show_result(), names='value')
end_picker.observe(lambda change: show_result(), names='value')

# Первый результат появляется сразу, даже без выбранных галочек.
show_result()

Output()

In [17]:
# Контроль: три сценария работы готового инструмента
print('=== Сценарий 1: базовые веса (ничего не выбрано) ===')
rating, excl, _ = rank_resorts(start_picker.value, end_picker.value, make_weights([]))
display(rating[['resort', 'rank', 'score']].round(2))

print('\n=== Сценарий 2: выбраны «питание» и «жильё» ===')
rating2, _, _ = rank_resorts(start_picker.value, end_picker.value, make_weights(['food', 'lodging']))
display(rating2[['resort', 'rank', 'score']].round(2))

print('\n=== Сценарий 3: выбраны все параметры (каждому по 20%) ===')
rating3, _, _ = rank_resorts(start_picker.value, end_picker.value, make_weights(list(parameter_names)))
display(rating3[['resort', 'rank', 'score']].round(2))


=== Сценарий 1: базовые веса (ничего не выбрано) ===


,resort,rank,score
0,Роза Хутор,1,89.22
1,Красная Поляна,2,88.15
2,Газпром Поляна,2,88.15
3,Шерегеш,4,75.21



=== Сценарий 2: выбраны «питание» и «жильё» ===


,resort,rank,score
0,Роза Хутор,1,96.44
1,Красная Поляна,2,96.05
2,Газпром Поляна,2,96.05
3,Шерегеш,4,88.31



=== Сценарий 3: выбраны все параметры (каждому по 20%) ===


,resort,rank,score
0,Роза Хутор,1,92.16
1,Красная Поляна,2,91.30
2,Газпром Поляна,2,91.30
3,Шерегеш,4,74.28


*«Стесс-тест» интерфейса*

In [18]:
# Контроль: как интерфейс реагирует на ошибку
try:
    rank_resorts(pd.Timestamp('2025-01-21'), pd.Timestamp('2025-01-15'), make_weights([]))
except ValueError as error:
    print('Перехвачено:', error)


Перехвачено: Конец периода не может быть раньше начала.


## Заключение: итоги проекта Travel Designer

Разработан аналитический инструмент для сравнительной оценки горнолыжных курортов, объединяющий погодные данные и показатели инфраструктуры в единый рейтинг.

**Ключевые возможности:**
- **Гибкая приоритизация:** рейтинг пересчитывается в реальном времени с учётом пользовательских весов по пяти параметрам (температура, осадки, солнце, питание, жильё).  
- **Прозрачность результата:** карточка лидера детализирует вклад каждого параметра в итоговый балл - это позволяет обосновать позицию курорта количественно.  
- **Контроль качества данных:** курорты с неполной историей за выбранный период исключаются из рейтинга и выводятся отдельно - это исключает искажение результатов из‑за пропусков.  
- **Архитектурная надёжность:** расчётное ядро отделено от интерфейса, что обеспечивает возможность повторного использования логики в других сценариях.

**Ограничения модели:**
- Инфраструктурные данные взяты из снимка OpenStreetMap за 2026 год: количество объектов не отражает их качество, цены или фактическую доступность в выбранном историческом периоде.  
- Рейтинг не учитывает операционные параметры (открытые трассы, загруженность, стоимость услуг).  

Инструмент демонстрирует полный цикл аналитической работы: от валидации входных данных до прозрачной модели оценки и удобного интерфейса для интерпретации результатов.


In [19]:
# ============================================================
# Экспорт данных для дашборда в DataLens
# Готовит плоские таблицы: DataLens не запускает Python,
# поэтому вся логика модели остаётся здесь, а дашборд — витрина.
# ============================================================

EXPORT_DIR = 'datalens_export'
os.makedirs(EXPORT_DIR, exist_ok=True)

# --- 1. Дневные баллы: основа для интерактивного рейтинга ---
daily_scores = weather_scores[[
    'resort_id', 'date',
    'temperature_c', 'precipitation_mm', 'sunshine_h',
    'temperature', 'precipitation', 'sunshine'
]].copy()
daily_scores['date'] = pd.to_datetime(daily_scores['date']).dt.date
daily_scores.to_csv(f'{EXPORT_DIR}/daily_scores.csv', index=False, date_format='%Y-%m-%d')

# --- 2. Инфраструктура: по строке на курорт (все 32 из справочника) ---
features_export = features[[
    'resort_id', 'resort',
    'food_objects_6km', 'lodging_objects_6km',
    'food', 'lodging'
]].copy()
features_export.to_csv(f'{EXPORT_DIR}/features.csv', index=False)

# --- 3. Пример рейтинга по базовым весам: витрина «как в ноутбуке» ---
example_rating, example_excluded, _ = rank_resorts(
    start_picker.value, end_picker.value, make_weights([])
)
example_rating[[
    'resort_id', 'resort', 'rank', 'score',
    'temperature_contribution', 'precipitation_contribution',
    'sunshine_contribution', 'food_contribution', 'lodging_contribution'
]].round(2).to_csv(f'{EXPORT_DIR}/rating_example.csv', index=False)

# --- 4. Справочник курортов: и видно, кто остался без истории ---
catalog = features_export[['resort_id', 'resort']].copy()
catalog['has_daily_history'] = catalog['resort_id'].isin(
    weather_scores['resort_id'].unique()
)
catalog.to_csv(f'{EXPORT_DIR}/resorts_catalog.csv', index=False)

# --- Контроль перед выгрузкой ---
print('Файлы сохранены в папку', EXPORT_DIR + ':')
for name in sorted(os.listdir(EXPORT_DIR)):
    path = os.path.join(EXPORT_DIR, name)
    print(f'  {name}: {os.path.getsize(path) / 1024:.1f} КБ')
print('\nКонтроль:')
print('  дневных строк:', len(daily_scores),
      '(ожидаем 43 832)')
print('  курортов в дневной истории:', daily_scores["resort_id"].nunique())
print('  курортов в инфраструктуре:', features_export['resort_id'].nunique())
print('  пустых дат в дневной истории:', daily_scores["date"].isna().sum())


Файлы сохранены в папку datalens_export:
  .ipynb_checkpoints: 0.0 КБ
  daily_scores.csv: 2405.4 КБ
  features.csv: 1.5 КБ
  rating_example.csv: 0.4 КБ
  resorts_catalog.csv: 1.0 КБ

Контроль:
  дневных строк: 43832 (ожидаем 43 832)
  курортов в дневной истории: 4
  курортов в инфраструктуре: 32
  пустых дат в дневной истории: 0


## Попытка сделать дашборд в Yandex DataLens:

https://datalens.yandex/x8udfi3j8brog?_share_link=public 